## Set coding environment

In [26]:
import os

# Fix Windows OpenMP duplicate-runtime crash
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"
os.environ["OMP_NUM_THREADS"] = "1"

print("OpenMP environment configured")

OpenMP environment configured


In [27]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print(
        "GPU memory:",
        round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2),
        "GB"
    )

PyTorch: 2.13.0+cu126
CUDA available: True
GPU: NVIDIA GeForce RTX 3050 Ti Laptop GPU
GPU memory: 4.0 GB


In [1]:
from pathlib import Path

LOCAL_ROOT = Path(
    r"C:\Users\varda\Documents\_My Computer\COMMON_space\Learning\Upgrad_EPGP_MS\upgrad_IIITB_MS_course\MS_LJMU_Material\Topic Data\SortWaste\dataset\dataset"
)

RESULTS_ROOT = Path(
    r"C:\Users\varda\Documents\_My Computer\COMMON_space\Learning\Upgrad_EPGP_MS\upgrad_IIITB_MS_course\MS_LJMU_Material\Thesis_Code\results"
)

for split in ["train", "val", "test"]:

    coco_images = (
        LOCAL_ROOT /
        "splited_all_dataset_coco" /
        split /
        "images"
    )

    yolo_labels = (
        LOCAL_ROOT /
        "splited_all_dataset_coco" /
        split /
        "labels"
    )

    print(
        f"{split.upper():5s} | "
        f"images: {len(list(coco_images.iterdir())):5d} | "
        f"YOLO labels: {len(list(yolo_labels.glob('*.txt'))):5d}"
    )

TRAIN | images:  3705 | YOLO labels:  3705
VAL   | images:   780 | YOLO labels:   780
TEST  | images:   776 | YOLO labels:   776


## Bring in data

In [ ]:
## bring in sortwaste data from LOCAL
from pathlib import Path

#DATA_ROOT = Path("/content/drive/MyDrive/LJMU_research_data/dataset")

YOLO_ROOT_colab = LOCAL_ROOT / "splited_all_dataset_coco"
COCO_ROOT_colab = LOCAL_ROOT / "splited_all_dataset_coco"

print("YOLO:", YOLO_ROOT_colab)
print("COCO:", COCO_ROOT_colab)

YOLO: C:\Users\varda\Documents\_My Computer\COMMON_space\Learning\Upgrad_EPGP_MS\upgrad_IIITB_MS_course\MS_LJMU_Material\Topic Data\SortWaste\dataset\dataset\splited_all_dataset_coco
COCO: C:\Users\varda\Documents\_My Computer\COMMON_space\Learning\Upgrad_EPGP_MS\upgrad_IIITB_MS_course\MS_LJMU_Material\Topic Data\SortWaste\dataset\dataset\splited_all_dataset_coco


In [ ]:
from pathlib import Path

for split in ["train", "val", "test"]:
    print(f"\n===== {split.upper()} =====")

    coco_images = COCO_ROOT_colab / split / "images"
    yolo_labels = YOLO_ROOT_colab / split / "labels"

    print("COCO images :", coco_images)
    print("Exists      :", coco_images.exists())

    print("YOLO labels :", yolo_labels)
    print("Exists      :", yolo_labels.exists())

    if coco_images.exists():
        print("Images:", len(list(coco_images.glob("*"))))

    if yolo_labels.exists():
        print("Labels:", len(list(yolo_labels.glob("*.txt"))))


===== TRAIN =====
COCO images : C:\Users\varda\Documents\_My Computer\COMMON_space\Learning\Upgrad_EPGP_MS\upgrad_IIITB_MS_course\MS_LJMU_Material\Topic Data\SortWaste\dataset\dataset\splited_all_dataset_coco\train\images
Exists      : True
YOLO labels : C:\Users\varda\Documents\_My Computer\COMMON_space\Learning\Upgrad_EPGP_MS\upgrad_IIITB_MS_course\MS_LJMU_Material\Topic Data\SortWaste\dataset\dataset\splited_all_dataset_coco\train\labels
Exists      : True
Images: 3705
Labels: 3705

===== VAL =====
COCO images : C:\Users\varda\Documents\_My Computer\COMMON_space\Learning\Upgrad_EPGP_MS\upgrad_IIITB_MS_course\MS_LJMU_Material\Topic Data\SortWaste\dataset\dataset\splited_all_dataset_coco\val\images
Exists      : True
YOLO labels : C:\Users\varda\Documents\_My Computer\COMMON_space\Learning\Upgrad_EPGP_MS\upgrad_IIITB_MS_course\MS_LJMU_Material\Topic Data\SortWaste\dataset\dataset\splited_all_dataset_coco\val\labels
Exists      : True
Images: 780
Labels: 780

===== TEST =====
COCO ima

## Run the baseline models - YOLO and Faster R-CNN

## Model E1 : YOLO only

In [ ]:
!pip install -q ultralytics

In [ ]:
import ultralytics

print("Ultralytics:", ultralytics.__version__)

Ultralytics: 8.4.126


In [ ]:
import yaml

DATA_YAML = YOLO_ROOT_colab / "data_pc.yaml"

with open(DATA_YAML, "r") as f:
    data_config = yaml.safe_load(f)

print(data_config)

{'train': 'C:\\Users\\varda\\Documents\\_My Computer\\COMMON_space\\Learning\\Upgrad_EPGP_MS\\upgrad_IIITB_MS_course\\MS_LJMU_Material\\Topic Data\\SortWaste\\dataset\\dataset\\splited_all_dataset_coco\\train\\images', 'val': 'C:\\Users\\varda\\Documents\\_My Computer\\COMMON_space\\Learning\\Upgrad_EPGP_MS\\upgrad_IIITB_MS_course\\MS_LJMU_Material\\Topic Data\\SortWaste\\dataset\\dataset\\splited_all_dataset_coco\\val\\images', 'test': 'C:\\Users\\varda\\Documents\\_My Computer\\COMMON_space\\Learning\\Upgrad_EPGP_MS\\upgrad_IIITB_MS_course\\MS_LJMU_Material\\Topic Data\\SortWaste\\dataset\\dataset\\splited_all_dataset_coco\\test\\images', 'nc': 8, 'names': ['0 pet', '1 hdpe', '2 mixed_plastic_soft', '3 ecal', '4 metal', '5 cardboard', '6 mixed_plastic_rigid', '7 pet_oil']}


In [ ]:
## model initialization
from ultralytics import YOLO

yolo_model = YOLO("yolo11n.pt")

In [ ]:
## model training / run begin

from ultralytics import YOLO
import torch
import os

# ---------------------------------------------------------
# 2. Paths
# ---------------------------------------------------------
model_path = r"C:\Users\varda\Documents\_My Computer\COMMON_space\Learning\Upgrad_EPGP_MS\upgrad_IIITB_MS_course\MS_LJMU_Material\Thesis_Code\runs\detect\runs\sortwaste\yolo11n_baseline-2\weights\last.pt"

data_yaml = DATA_YAML

# ---------------------------------------------------------
# 3. Load learned weights
#    IMPORTANT: Do NOT use resume=True
# ---------------------------------------------------------
model = YOLO(model_path)

# ---------------------------------------------------------
# 4. Continue training using the learned weights
# ---------------------------------------------------------
results = model.train(
    data=data_yaml,

    epochs=13,          # Epoch 37 -> approximately 50
    imgsz=640,

    batch=4,            # safer for 4 GB RTX 3050 Ti
    device=0,
    workers=0,          # safer in Windows + Jupyter

    amp=True,

    # Start a NEW run
    project=r"C:\Users\varda\Documents\_My Computer\COMMON_space\Learning\Upgrad_EPGP_MS\upgrad_IIITB_MS_course\MS_LJMU_Material\Thesis_Code\runs\sortwaste",
    name="YOLO11n_baseline_pc",

    # Keep your experiment deterministic
    seed=42,

    # Don't use Colab paths
    exist_ok=True
)

New https://pypi.org/project/ultralytics/8.4.127 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.126  Python-3.11.15 torch-2.13.0+cu126 CUDA:0 (NVIDIA GeForce RTX 3050 Ti Laptop GPU, 4096MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=4, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=False, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=C:\Users\varda\Documents\_My Computer\COMMON_space\Learning\Upgrad_EPGP_MS\upgrad_IIITB_MS_course\MS_LJMU_Material\Topic Data\SortWaste\dataset\dataset\splited_all_dataset_coco\data_pc.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=13, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fract

## Model E2 : Faster R-CNN baseline

In [ ]:
## data check
from pathlib import Path

root = Path(r"C:\Users\varda\Documents\_My Computer\COMMON_space\Learning\Upgrad_EPGP_MS\upgrad_IIITB_MS_course\MS_LJMU_Material\Topic Data\SortWaste\dataset\dataset\splited_all_dataset_coco")

for split in ["train", "val", "test"]:
    ann_dir = root / split / "annotations"
    print(f"\n{split.upper()}:")
    for f in ann_dir.glob("*.json"):
        print(f.name)


TRAIN:
train_coco.json

VAL:
val_coco.json

TEST:
test_coco.json


In [ ]:
import json

for split in ["train", "val", "test"]:
    ann_dir = root / split / "annotations"
    json_files = list(ann_dir.glob("*.json"))

    for f in json_files:
        with open(f, "r", encoding="utf-8") as fh:
            data = json.load(fh)

        print(f"\n{split.upper()} - {f.name}")
        print("Images:", len(data["images"]))
        print("Annotations:", len(data["annotations"]))
        print("Categories:", data["categories"])


TRAIN - train_coco.json
Images: 3705
Annotations: 61842
Categories: [{'id': 1, 'name': 'pet', 'supercategory': ''}, {'id': 2, 'name': 'hdpe', 'supercategory': ''}, {'id': 3, 'name': 'mixed_plastic_soft', 'supercategory': ''}, {'id': 4, 'name': 'ecal', 'supercategory': ''}, {'id': 5, 'name': 'metal', 'supercategory': ''}, {'id': 6, 'name': 'cardboard', 'supercategory': ''}, {'id': 7, 'name': 'mixed_plastic_rigid', 'supercategory': ''}, {'id': 8, 'name': 'pet_oil', 'supercategory': ''}]

VAL - val_coco.json
Images: 780
Annotations: 13065
Categories: [{'id': 1, 'name': 'pet', 'supercategory': ''}, {'id': 2, 'name': 'hdpe', 'supercategory': ''}, {'id': 3, 'name': 'mixed_plastic_soft', 'supercategory': ''}, {'id': 4, 'name': 'ecal', 'supercategory': ''}, {'id': 5, 'name': 'metal', 'supercategory': ''}, {'id': 6, 'name': 'cardboard', 'supercategory': ''}, {'id': 7, 'name': 'mixed_plastic_rigid', 'supercategory': ''}, {'id': 8, 'name': 'pet_oil', 'supercategory': ''}]

TEST - test_coco.json


In [ ]:
### model training run
# ============================================================
# E2: Faster R-CNN ResNet-50-FPN Baseline - SortWaste
# ============================================================

import os
import json
import time
import csv
import math
from pathlib import Path

import torch
import torchvision
import numpy as np
from PIL import Image

from torch.utils.data import Dataset, DataLoader
from torchvision.transforms import functional as F
from torchvision.models.detection import (
    fasterrcnn_resnet50_fpn,
    FasterRCNN_ResNet50_FPN_Weights
)
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor

# COCO evaluation
from pycocotools.cocoeval import COCOeval
from pycocotools.coco import COCO


# ============================================================
# 1. CONFIGURATION
# ============================================================

ROOT = Path(
    r"C:\Users\varda\Documents\_My Computer\COMMON_space\Learning\Upgrad_EPGP_MS"
    r"\upgrad_IIITB_MS_course\MS_LJMU_Material\Topic Data\SortWaste"
    r"\dataset\dataset\splited_all_dataset_coco"
)

TRAIN_IMAGES = ROOT / "train" / "images"
VAL_IMAGES   = ROOT / "val" / "images"

TRAIN_JSON = ROOT / "train" / "annotations" / "train_coco.json"
VAL_JSON   = ROOT / "val" / "annotations" / "val_coco.json"

# Test set deliberately NOT used during training
TEST_JSON = ROOT / "test" / "annotations" / "test_coco.json"


# Output directory
OUTPUT_DIR = Path(
    r"C:\Users\varda\Documents\_My Computer\COMMON_space\Learning"
    r"\Upgrad_EPGP_MS\upgrad_IIITB_MS_course\MS_LJMU_Material"
    r"\Thesis_Code\runs\sortwaste\faster_rcnn_baseline"
)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

CHECKPOINT_DIR = OUTPUT_DIR / "checkpoints"
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

BEST_MODEL = OUTPUT_DIR / "best_model.pth"
LAST_MODEL = OUTPUT_DIR / "last_model.pth"
RESULTS_CSV = OUTPUT_DIR / "results.csv"


# Training parameters
NUM_CLASSES = 9          # 8 waste classes + background
NUM_EPOCHS = 30
BATCH_SIZE = 4
NUM_WORKERS = 0

LEARNING_RATE = 0.005
MOMENTUM = 0.9
WEIGHT_DECAY = 0.0005

# Keep resolution controlled because GPU has only 4 GB
MIN_SIZE = 640
MAX_SIZE = 640

# Validation frequency
VAL_EVERY = 1

# Reproducibility
SEED = 42


# ============================================================
# 2. REPRODUCIBILITY
# ============================================================

torch.manual_seed(SEED)
np.random.seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False


# ============================================================
# 3. DEVICE
# ============================================================

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("=" * 70)
print("E2 - Faster R-CNN ResNet-50-FPN Baseline")
print("=" * 70)
print(f"PyTorch       : {torch.__version__}")
print(f"Torchvision   : {torchvision.__version__}")
print(f"Device        : {DEVICE}")

if torch.cuda.is_available():
    print(f"GPU           : {torch.cuda.get_device_name(0)}")
    print(
        f"GPU memory    : "
        f"{torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB"
    )

print(f"Train images  : {TRAIN_IMAGES}")
print(f"Train JSON    : {TRAIN_JSON}")
print(f"Val images    : {VAL_IMAGES}")
print(f"Val JSON      : {VAL_JSON}")
print(f"Epochs        : {NUM_EPOCHS}")
print(f"Batch size    : {BATCH_SIZE}")
print(f"Image size    : {MIN_SIZE} x {MAX_SIZE}")
print("=" * 70)


# ============================================================
# 4. CLASS MAPPING
# ============================================================

CLASS_NAMES = {
    1: "pet",
    2: "hdpe",
    3: "mixed_plastic_soft",
    4: "ecal",
    5: "metal",
    6: "cardboard",
    7: "mixed_plastic_rigid",
    8: "pet_oil",
}

print("\nClasses:")
for k, v in CLASS_NAMES.items():
    print(f"  {k}: {v}")


# ============================================================
# 5. COCO DATASET
# ============================================================

class SortWasteCocoDataset(Dataset):

    def __init__(
        self,
        image_dir,
        annotation_file,
        train=False
    ):

        self.image_dir = Path(image_dir)
        self.annotation_file = Path(annotation_file)
        self.train = train

        with open(self.annotation_file, "r", encoding="utf-8") as f:
            self.coco_data = json.load(f)

        self.images = self.coco_data["images"]
        self.annotations = self.coco_data["annotations"]

        # Image ID -> annotations
        self.image_to_annotations = {}

        for ann in self.annotations:
            image_id = ann["image_id"]

            if image_id not in self.image_to_annotations:
                self.image_to_annotations[image_id] = []

            self.image_to_annotations[image_id].append(ann)

        print(
            f"\nLoaded {self.annotation_file.name}: "
            f"{len(self.images)} images, "
            f"{len(self.annotations)} annotations"
        )

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):

        image_info = self.images[idx]

        image_id = image_info["id"]
        file_name = image_info["file_name"]

        image_path = self.image_dir / file_name

        # ----------------------------------------------------
        # Load image
        # ----------------------------------------------------

        image = Image.open(image_path).convert("RGB")

        width, height = image.size

        # ----------------------------------------------------
        # Get annotations
        # ----------------------------------------------------

        anns = self.image_to_annotations.get(image_id, [])

        boxes = []
        labels = []
        areas = []
        iscrowd = []

        for ann in anns:

            # COCO bbox:
            # [x_min, y_min, width, height]

            x, y, w, h = ann["bbox"]

            # Ignore invalid boxes
            if w <= 0 or h <= 0:
                continue

            x1 = max(0, x)
            y1 = max(0, y)

            x2 = min(width, x + w)
            y2 = min(height, y + h)

            if x2 <= x1 or y2 <= y1:
                continue

            boxes.append([x1, y1, x2, y2])

            # Category IDs are already 1-8
            labels.append(int(ann["category_id"]))

            areas.append(float((x2 - x1) * (y2 - y1)))

            iscrowd.append(int(ann.get("iscrowd", 0)))

        # ----------------------------------------------------
        # Convert to tensors
        # ----------------------------------------------------

        if len(boxes) == 0:

            boxes = torch.zeros((0, 4), dtype=torch.float32)
            labels = torch.zeros((0,), dtype=torch.int64)
            areas = torch.zeros((0,), dtype=torch.float32)
            iscrowd = torch.zeros((0,), dtype=torch.int64)

        else:

            boxes = torch.tensor(boxes, dtype=torch.float32)
            labels = torch.tensor(labels, dtype=torch.int64)
            areas = torch.tensor(areas, dtype=torch.float32)
            iscrowd = torch.tensor(iscrowd, dtype=torch.int64)

        # ----------------------------------------------------
        # Target expected by Faster R-CNN
        # ----------------------------------------------------

        target = {
            "boxes": boxes,
            "labels": labels,
            "image_id": torch.tensor([image_id]),
            "area": areas,
            "iscrowd": iscrowd,
        }

        # ----------------------------------------------------
        # Convert image to tensor
        # ----------------------------------------------------

        image = F.to_tensor(image)

        # ----------------------------------------------------
        # Simple horizontal flip augmentation
        # ----------------------------------------------------

        if self.train:

            if torch.rand(1).item() < 0.5:

                image = torch.flip(image, dims=[2])

                if len(boxes) > 0:

                    boxes = target["boxes"].clone()

                    old_x1 = boxes[:, 0].clone()
                    old_x2 = boxes[:, 2].clone()

                    boxes[:, 0] = width - old_x2
                    boxes[:, 2] = width - old_x1

                    target["boxes"] = boxes

        return image, target


# ============================================================
# 6. COLLATE FUNCTION
# ============================================================

def collate_fn(batch):

    images, targets = zip(*batch)

    return list(images), list(targets)


# ============================================================
# 7. DATASETS
# ============================================================

train_dataset = SortWasteCocoDataset(
    TRAIN_IMAGES,
    TRAIN_JSON,
    train=True
)

val_dataset = SortWasteCocoDataset(
    VAL_IMAGES,
    VAL_JSON,
    train=False
)


# ============================================================
# 8. DATALOADERS
# ============================================================

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    collate_fn=collate_fn,
    pin_memory=torch.cuda.is_available()
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    collate_fn=collate_fn,
    pin_memory=torch.cuda.is_available()
)

print("\nDataloaders ready.")
print(f"Training batches  : {len(train_loader)}")
print(f"Validation batches: {len(val_loader)}")


# ============================================================
# 9. CREATE FASTER R-CNN
# ============================================================

print("\nLoading pretrained Faster R-CNN ResNet-50-FPN...")

weights = FasterRCNN_ResNet50_FPN_Weights.DEFAULT

model = fasterrcnn_resnet50_fpn(
    weights=weights,

    # Control memory consumption
    min_size=MIN_SIZE,
    max_size=MAX_SIZE,

    # Keep standard RPN settings
    rpn_pre_nms_top_n_train=2000,
    rpn_post_nms_top_n_train=1000,
    rpn_pre_nms_top_n_test=1000,
    rpn_post_nms_top_n_test=300,

    box_detections_per_img=100
)

# ------------------------------------------------------------
# Replace COCO classifier with 8-class SortWaste classifier
# ------------------------------------------------------------

in_features = model.roi_heads.box_predictor.cls_score.in_features

model.roi_heads.box_predictor = FastRCNNPredictor(
    in_features,
    NUM_CLASSES
)

model.to(DEVICE)

print("Faster R-CNN model ready.")


# ============================================================
# 10. OPTIMIZER
# ============================================================

params = [
    p for p in model.parameters()
    if p.requires_grad
]

optimizer = torch.optim.SGD(
    params,
    lr=LEARNING_RATE,
    momentum=MOMENTUM,
    weight_decay=WEIGHT_DECAY
)

# Reduce LR during training
lr_scheduler = torch.optim.lr_scheduler.StepLR(
    optimizer,
    step_size=10,
    gamma=0.1
)


# ============================================================
# 11. AMP
# ============================================================

scaler = torch.cuda.amp.GradScaler(
    enabled=torch.cuda.is_available()
)


# ============================================================
# 12. COCO EVALUATION FUNCTION
# ============================================================

def evaluate_coco(model, data_loader, coco_gt):

    model.eval()

    results = []

    inference_start = time.time()

    with torch.no_grad():

        for images, targets in data_loader:

            images = [
                image.to(DEVICE)
                for image in images
            ]

            with torch.cuda.amp.autocast(
                enabled=torch.cuda.is_available()
            ):

                outputs = model(images)

            for target, output in zip(targets, outputs):

                image_id = int(
                    target["image_id"].item()
                )

                boxes = output["boxes"].detach().cpu().numpy()
                scores = output["scores"].detach().cpu().numpy()
                labels = output["labels"].detach().cpu().numpy()

                for box, score, label in zip(
                    boxes,
                    scores,
                    labels
                ):

                    x1, y1, x2, y2 = box

                    width = x2 - x1
                    height = y2 - y1

                    results.append({
                        "image_id": image_id,
                        "category_id": int(label),
                        "bbox": [
                            float(x1),
                            float(y1),
                            float(width),
                            float(height)
                        ],
                        "score": float(score)
                    })

    inference_time = time.time() - inference_start

    if len(results) == 0:

        print("WARNING: No predictions generated.")

        return {
            "precision": 0.0,
            "recall": 0.0,
            "map50": 0.0,
            "map5095": 0.0,
            "class_ap": {},
            "inference_time": inference_time
        }

    coco_dt = coco_gt.loadRes(results)

    coco_eval = COCOeval(
        coco_gt,
        coco_dt,
        "bbox"
    )

    coco_eval.evaluate()
    coco_eval.accumulate()
    coco_eval.summarize()

    # --------------------------------------------------------
    # Standard COCO metrics
    # --------------------------------------------------------

    map5095 = float(coco_eval.stats[0])
    map50 = float(coco_eval.stats[1])

    # Approximate overall precision/recall at IoU=0.50
    precision_tensor = coco_eval.eval["precision"]

    # IoU index 0 corresponds to IoU=0.50
    # Shape:
    # [IoU, Recall, Classes, Area, MaxDets]

    precision_values = precision_tensor[0]

    precision_values = precision_values[
        precision_values > -1
    ]

    precision = (
        float(np.mean(precision_values))
        if len(precision_values) > 0
        else 0.0
    )

    # Recall
    recall_tensor = coco_eval.eval["recall"]

    recall_values = recall_tensor[0]

    recall_values = recall_values[
        recall_values > -1
    ]

    recall = (
        float(np.mean(recall_values))
        if len(recall_values) > 0
        else 0.0
    )

    # --------------------------------------------------------
    # Class-wise AP@50
    # --------------------------------------------------------

    class_ap = {}

    for class_index, class_id in enumerate(
        coco_eval.params.catIds
    ):

        precision_class = precision_tensor[
            0, :, class_index, 0, -1
        ]

        precision_class = precision_class[
            precision_class > -1
        ]

        if len(precision_class) > 0:

            ap = float(
                np.mean(precision_class)
            )

        else:

            ap = 0.0

        class_ap[
            CLASS_NAMES.get(
                class_id,
                str(class_id)
            )
        ] = ap

    return {
        "precision": precision,
        "recall": recall,
        "map50": map50,
        "map5095": map5095,
        "class_ap": class_ap,
        "inference_time": inference_time
    }


# ============================================================
# 13. COCO GROUND TRUTH
# ============================================================

coco_val = COCO(str(VAL_JSON))


# ============================================================
# 14. CSV INITIALIZATION
# ============================================================

csv_fields = [
    "epoch",
    "train_loss",
    "learning_rate",
    "precision",
    "recall",
    "mAP50",
    "mAP50_95",
    "epoch_time"
]

with open(
    RESULTS_CSV,
    "w",
    newline="",
    encoding="utf-8"
) as f:

    writer = csv.DictWriter(
        f,
        fieldnames=csv_fields
    )

    writer.writeheader()


# ============================================================
# 15. TRAINING
# ============================================================

best_map5095 = -1.0

print("\n")
print("=" * 70)
print("STARTING E2 TRAINING")
print("=" * 70)


for epoch in range(1, NUM_EPOCHS + 1):

    epoch_start = time.time()

    model.train()

    running_loss = 0.0

    num_batches = len(train_loader)

    print(
        f"\nEpoch {epoch}/{NUM_EPOCHS}"
    )

    # --------------------------------------------------------
    # Training
    # --------------------------------------------------------

    for batch_idx, (images, targets) in enumerate(
        train_loader,
        start=1
    ):

        images = [
            image.to(DEVICE)
            for image in images
        ]

        targets = [
            {
                k: v.to(DEVICE)
                for k, v in target.items()
            }
            for target in targets
        ]

        optimizer.zero_grad(
            set_to_none=True
        )

        try:

            with torch.cuda.amp.autocast(
                enabled=torch.cuda.is_available()
            ):

                loss_dict = model(
                    images,
                    targets
                )

                losses = sum(
                    loss for loss in loss_dict.values()
                )

            # ------------------------------------------------
            # Backpropagation
            # ------------------------------------------------

            scaler.scale(losses).backward()

            scaler.step(optimizer)

            scaler.update()

            loss_value = losses.item()

            running_loss += loss_value

        except RuntimeError as e:

            if "out of memory" in str(e).lower():

                print(
                    "\nCUDA OUT OF MEMORY."
                )

                print(
                    "Clearing GPU cache..."
                )

                optimizer.zero_grad(
                    set_to_none=True
                )

                if torch.cuda.is_available():
                    torch.cuda.empty_cache()

                raise e

            else:
                raise e

        # ----------------------------------------------------
        # Progress
        # ----------------------------------------------------

        if (
            batch_idx == 1
            or batch_idx % 50 == 0
            or batch_idx == num_batches
        ):

            avg_loss = (
                running_loss / batch_idx
            )

            if torch.cuda.is_available():

                gpu_mem = (
                    torch.cuda.memory_allocated()
                    / 1024**3
                )

            else:

                gpu_mem = 0

            print(
                f"  Batch {batch_idx}/{num_batches} "
                f"| Loss: {loss_value:.4f} "
                f"| Avg: {avg_loss:.4f} "
                f"| GPU: {gpu_mem:.2f} GB"
            )

    # --------------------------------------------------------
    # Epoch statistics
    # --------------------------------------------------------

    train_loss = (
        running_loss / num_batches
    )

    current_lr = optimizer.param_groups[0]["lr"]

    # --------------------------------------------------------
    # Validation
    # --------------------------------------------------------

    print(
        f"\nValidation after epoch {epoch}..."
    )

    metrics = evaluate_coco(
        model,
        val_loader,
        coco_val
    )

    epoch_time = time.time() - epoch_start

    print("\nEpoch summary:")
    print(
        f"  Train Loss : {train_loss:.4f}"
    )
    print(
        f"  Precision  : {metrics['precision']:.4f}"
    )
    print(
        f"  Recall     : {metrics['recall']:.4f}"
    )
    print(
        f"  mAP@50     : {metrics['map50']:.4f}"
    )
    print(
        f"  mAP@50:95  : {metrics['map5095']:.4f}"
    )
    print(
        f"  Time       : {epoch_time / 60:.2f} min"
    )

    # --------------------------------------------------------
    # Class AP
    # --------------------------------------------------------

    print("\nClass AP@50:")

    for class_name, ap in metrics["class_ap"].items():

        print(
            f"  {class_name:<22} {ap:.4f}"
        )

    # --------------------------------------------------------
    # Save epoch checkpoint
    # --------------------------------------------------------

    epoch_checkpoint = CHECKPOINT_DIR / (
        f"epoch_{epoch:03d}.pth"
    )

    torch.save(
        {
            "epoch": epoch,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "scheduler_state_dict": lr_scheduler.state_dict(),
            "scaler_state_dict": scaler.state_dict(),
            "train_loss": train_loss,
            "metrics": metrics,
            "class_names": CLASS_NAMES
        },
        epoch_checkpoint
    )

    # --------------------------------------------------------
    # Save last model
    # --------------------------------------------------------

    torch.save(
        {
            "epoch": epoch,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "scheduler_state_dict": lr_scheduler.state_dict(),
            "scaler_state_dict": scaler.state_dict(),
            "train_loss": train_loss,
            "metrics": metrics,
            "class_names": CLASS_NAMES
        },
        LAST_MODEL
    )

    # --------------------------------------------------------
    # Save best model
    # --------------------------------------------------------

    if metrics["map5095"] > best_map5095:

        best_map5095 = metrics["map5095"]

        torch.save(
            {
                "epoch": epoch,
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "scheduler_state_dict": lr_scheduler.state_dict(),
                "scaler_state_dict": scaler.state_dict(),
                "train_loss": train_loss,
                "metrics": metrics,
                "class_names": CLASS_NAMES
            },
            BEST_MODEL
        )

        print(
            f"\n*** NEW BEST MODEL ***"
        )

        print(
            f"Best mAP@50:95 = "
            f"{best_map5095:.4f}"
        )

    # --------------------------------------------------------
    # CSV
    # --------------------------------------------------------

    with open(
        RESULTS_CSV,
        "a",
        newline="",
        encoding="utf-8"
    ) as f:

        writer = csv.DictWriter(
            f,
            fieldnames=csv_fields
        )

        writer.writerow(
            {
                "epoch": epoch,
                "train_loss": train_loss,
                "learning_rate": current_lr,
                "precision": metrics["precision"],
                "recall": metrics["recall"],
                "mAP50": metrics["map50"],
                "mAP50_95": metrics["map5095"],
                "epoch_time": epoch_time
            }
        )

    # --------------------------------------------------------
    # LR scheduler
    # --------------------------------------------------------

    lr_scheduler.step()

    # --------------------------------------------------------
    # Free memory
    # --------------------------------------------------------

    if torch.cuda.is_available():
        torch.cuda.empty_cache()


# ============================================================
# 16. TRAINING COMPLETE
# ============================================================

print("\n")
print("=" * 70)
print("E2 TRAINING COMPLETE")
print("=" * 70)

print(
    f"Best mAP@50:95 : {best_map5095:.4f}"
)

print(
    f"Best model     : {BEST_MODEL}"
)

print(
    f"Last model     : {LAST_MODEL}"
)

print(
    f"Results CSV    : {RESULTS_CSV}"
)

print("=" * 70)

E2 - Faster R-CNN ResNet-50-FPN Baseline
PyTorch       : 2.13.0+cu126
Torchvision   : 0.28.0+cu126
Device        : cuda
GPU           : NVIDIA GeForce RTX 3050 Ti Laptop GPU
GPU memory    : 4.00 GB
Train images  : C:\Users\varda\Documents\_My Computer\COMMON_space\Learning\Upgrad_EPGP_MS\upgrad_IIITB_MS_course\MS_LJMU_Material\Topic Data\SortWaste\dataset\dataset\splited_all_dataset_coco\train\images
Train JSON    : C:\Users\varda\Documents\_My Computer\COMMON_space\Learning\Upgrad_EPGP_MS\upgrad_IIITB_MS_course\MS_LJMU_Material\Topic Data\SortWaste\dataset\dataset\splited_all_dataset_coco\train\annotations\train_coco.json
Val images    : C:\Users\varda\Documents\_My Computer\COMMON_space\Learning\Upgrad_EPGP_MS\upgrad_IIITB_MS_course\MS_LJMU_Material\Topic Data\SortWaste\dataset\dataset\splited_all_dataset_coco\val\images
Val JSON      : C:\Users\varda\Documents\_My Computer\COMMON_space\Learning\Upgrad_EPGP_MS\upgrad_IIITB_MS_course\MS_LJMU_Material\Topic Data\SortWaste\dataset\datas

C:\Users\varda\AppData\Local\Temp\ipykernel_6612\2332079180.py:437: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(
C:\Users\varda\AppData\Local\Temp\ipykernel_6612\2332079180.py:706: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(


  Batch 1/927 | Loss: 3.6573 | Avg: 3.6573 | GPU: 0.47 GB
  Batch 50/927 | Loss: 1.4338 | Avg: 1.6867 | GPU: 0.63 GB
  Batch 100/927 | Loss: 1.0854 | Avg: 1.4641 | GPU: 0.63 GB
  Batch 150/927 | Loss: 1.0410 | Avg: 1.3527 | GPU: 0.63 GB
  Batch 200/927 | Loss: 1.1849 | Avg: 1.2801 | GPU: 0.63 GB
  Batch 250/927 | Loss: 1.0407 | Avg: 1.2326 | GPU: 0.63 GB
  Batch 300/927 | Loss: 0.9272 | Avg: 1.1901 | GPU: 0.63 GB
  Batch 350/927 | Loss: 1.1383 | Avg: 1.1560 | GPU: 0.63 GB
  Batch 400/927 | Loss: 1.0106 | Avg: 1.1255 | GPU: 0.63 GB
  Batch 450/927 | Loss: 0.9131 | Avg: 1.1001 | GPU: 0.63 GB
  Batch 500/927 | Loss: 0.8815 | Avg: 1.0771 | GPU: 0.63 GB
  Batch 550/927 | Loss: 0.8969 | Avg: 1.0592 | GPU: 0.63 GB
  Batch 600/927 | Loss: 0.8245 | Avg: 1.0424 | GPU: 0.63 GB
  Batch 650/927 | Loss: 0.7457 | Avg: 1.0268 | GPU: 0.63 GB
  Batch 700/927 | Loss: 0.9258 | Avg: 1.0123 | GPU: 0.63 GB
  Batch 750/927 | Loss: 0.8057 | Avg: 0.9972 | GPU: 0.63 GB
  Batch 800/927 | Loss: 0.9028 | Avg: 0.984

C:\Users\varda\AppData\Local\Temp\ipykernel_6612\2332079180.py:463: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(


Loading and preparing results...
DONE (t=0.03s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=3.48s).
Accumulating evaluation results...
DONE (t=0.50s).
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.309
 Average Precision  (AP) @[ IoU=0.50      | area=   all | maxDets=100 ] = 0.451
 Average Precision  (AP) @[ IoU=0.75      | area=   all | maxDets=100 ] = 0.363
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= small | maxDets=100 ] = -1.000
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=medium | maxDets=100 ] = 0.180
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= large | maxDets=100 ] = 0.313
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets=  1 ] = 0.220
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets= 10 ] = 0.549
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.570
 Average Recall     (AR) @[ IoU=0.50:0.95 | area= small | maxDets=1

## Model E3 : Yolov11n + MobileNetV3-large 

## E3A: Classification Model 3A : Mobilenet with realistic augmentation

In [ ]:
# Packages import and GPU check
import torch
import torchvision
import ultralytics
import cv2

print("PyTorch:", torch.__version__)
print("Torchvision:", torchvision.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

print("Ultralytics:", ultralytics.__version__)

PyTorch: 2.13.0+cu126
Torchvision: 0.28.0+cu126
CUDA available: True
GPU: NVIDIA GeForce RTX 3050 Ti Laptop GPU
Ultralytics: 8.4.126


In [ ]:
from pathlib import Path

train_root = Path(
    r"C:\Users\varda\Documents\_My Computer\COMMON_space\Learning\Upgrad_EPGP_MS\upgrad_IIITB_MS_course\MS_LJMU_Material\Topic Data\SortWaste\dataset\dataset\splited_all_dataset_coco\train"
)

print("Train directory exists:", train_root.exists())
print("\nContents:")

for item in train_root.iterdir():
    print(item)

Train directory exists: True

Contents:
C:\Users\varda\Documents\_My Computer\COMMON_space\Learning\Upgrad_EPGP_MS\upgrad_IIITB_MS_course\MS_LJMU_Material\Topic Data\SortWaste\dataset\dataset\splited_all_dataset_coco\train\annotations
C:\Users\varda\Documents\_My Computer\COMMON_space\Learning\Upgrad_EPGP_MS\upgrad_IIITB_MS_course\MS_LJMU_Material\Topic Data\SortWaste\dataset\dataset\splited_all_dataset_coco\train\images
C:\Users\varda\Documents\_My Computer\COMMON_space\Learning\Upgrad_EPGP_MS\upgrad_IIITB_MS_course\MS_LJMU_Material\Topic Data\SortWaste\dataset\dataset\splited_all_dataset_coco\train\labels
C:\Users\varda\Documents\_My Computer\COMMON_space\Learning\Upgrad_EPGP_MS\upgrad_IIITB_MS_course\MS_LJMU_Material\Topic Data\SortWaste\dataset\dataset\splited_all_dataset_coco\train\labels.cache


In [ ]:
# data check
import json
from pathlib import Path

json_path = Path(
    r"C:\Users\varda\Documents\_My Computer\COMMON_space\Learning\Upgrad_EPGP_MS\upgrad_IIITB_MS_course\MS_LJMU_Material\Topic Data\SortWaste\dataset\dataset\splited_all_dataset_coco\train\annotations\train_coco.json"
)

with open(json_path, "r", encoding="utf-8") as f:
    coco = json.load(f)

print("Number of images:", len(coco["images"]))
print("Number of annotations:", len(coco["annotations"]))
print("Number of categories:", len(coco["categories"]))

print("\nFirst image entry:")
print(coco["images"][0])

print("\nFirst annotation:")
print(coco["annotations"][0])

print("\nCategories:")
for category in coco["categories"]:
    print(category)

Number of images: 3705
Number of annotations: 61842
Number of categories: 8

First image entry:
{'id': 3005, 'width': 1920, 'height': 1080, 'file_name': '0019_0026_5.png', 'license': 0, 'flickr_url': '', 'coco_url': '', 'date_captured': 0}

First annotation:
{'id': 40, 'image_id': 5, 'category_id': 4, 'segmentation': [], 'area': 20841.662400000012, 'bbox': [90.74, 382.4, 118.56, 175.79], 'iscrowd': 0, 'attributes': {'occluded': False, 'rotation': 0.0, 'track_id': 2, 'keyframe': True}}

Categories:
{'id': 1, 'name': 'pet', 'supercategory': ''}
{'id': 2, 'name': 'hdpe', 'supercategory': ''}
{'id': 3, 'name': 'mixed_plastic_soft', 'supercategory': ''}
{'id': 4, 'name': 'ecal', 'supercategory': ''}
{'id': 5, 'name': 'metal', 'supercategory': ''}
{'id': 6, 'name': 'cardboard', 'supercategory': ''}
{'id': 7, 'name': 'mixed_plastic_rigid', 'supercategory': ''}
{'id': 8, 'name': 'pet_oil', 'supercategory': ''}


In [ ]:
## creating TRAINING cropped data

import json
from pathlib import Path
from PIL import Image
from tqdm import tqdm


# ============================================================
# PATHS
# ============================================================

TRAIN_ROOT = Path(
    r"C:\Users\varda\Documents\_My Computer\COMMON_space\Learning\Upgrad_EPGP_MS\upgrad_IIITB_MS_course\MS_LJMU_Material\Topic Data\SortWaste\dataset\dataset\splited_all_dataset_coco\train"
)

ANNOTATION_FILE = (
    TRAIN_ROOT
    / "annotations"
    / "train_coco.json"
)

IMAGE_DIR = (
    TRAIN_ROOT
    / "images"
)

# Existing cropped-data folder
OUTPUT_DIR = Path(
    r"C:\Users\varda\Documents\_My Computer\COMMON_space\Learning\Upgrad_EPGP_MS\upgrad_IIITB_MS_course\MS_LJMU_Material\Topic Data\SortWaste\dataset\dataset\cropped_data\train"
)


# ============================================================
# SETTINGS
# ============================================================

# Small amount of context around each bounding box
PADDING = 0.05


# ============================================================
# LOAD COCO JSON
# ============================================================

with open(
    ANNOTATION_FILE,
    "r",
    encoding="utf-8"
) as f:

    coco = json.load(f)


# ============================================================
# CATEGORY MAPPING
# ============================================================

category_id_to_name = {
    category["id"]: category["name"]
    for category in coco["categories"]
}

print("Category mapping:")

for category_id, category_name in category_id_to_name.items():

    print(
        f"  {category_id}: {category_name}"
    )


# ============================================================
# IMAGE LOOKUP
# ============================================================

images_by_id = {
    image["id"]: image
    for image in coco["images"]
}


# ============================================================
# CREATE CLASS FOLDERS
# ============================================================

for class_name in category_id_to_name.values():

    class_dir = OUTPUT_DIR / class_name

    class_dir.mkdir(
        parents=True,
        exist_ok=True
    )


# ============================================================
# CREATE CROPS
# ============================================================

created = 0
skipped = 0

for annotation in tqdm(
    coco["annotations"],
    desc="Creating training crops"
):

    image_id = annotation["image_id"]
    category_id = annotation["category_id"]


    # --------------------------------------------------------
    # Find image
    # --------------------------------------------------------

    if image_id not in images_by_id:

        skipped += 1
        continue

    image_info = images_by_id[image_id]

    image_filename = image_info["file_name"]

    image_path = IMAGE_DIR / image_filename


    if not image_path.exists():

        print(
            f"\nWARNING: Image not found:"
            f"\n{image_path}"
        )

        skipped += 1
        continue


    # --------------------------------------------------------
    # Find class
    # --------------------------------------------------------

    if category_id not in category_id_to_name:

        skipped += 1
        continue

    class_name = category_id_to_name[category_id]


    # --------------------------------------------------------
    # Open image
    # --------------------------------------------------------

    try:

        image = Image.open(
            image_path
        ).convert("RGB")

    except Exception as e:

        print(
            f"\nWARNING: Could not open "
            f"{image_path}: {e}"
        )

        skipped += 1
        continue


    # --------------------------------------------------------
    # COCO bounding box
    #
    # [x, y, width, height]
    # --------------------------------------------------------

    x, y, width, height = annotation["bbox"]


    # --------------------------------------------------------
    # Add 5% padding
    # --------------------------------------------------------

    pad_x = width * PADDING
    pad_y = height * PADDING

    x1 = max(
        0,
        int(x - pad_x)
    )

    y1 = max(
        0,
        int(y - pad_y)
    )

    x2 = min(
        image.width,
        int(x + width + pad_x)
    )

    y2 = min(
        image.height,
        int(y + height + pad_y)
    )


    # --------------------------------------------------------
    # Validate crop
    # --------------------------------------------------------

    if x2 <= x1 or y2 <= y1:

        skipped += 1
        continue


    # --------------------------------------------------------
    # Crop object
    # --------------------------------------------------------

    crop = image.crop(
        (x1, y1, x2, y2)
    )


    # --------------------------------------------------------
    # Save crop
    # --------------------------------------------------------

    crop_filename = (
        f"{Path(image_filename).stem}"
        f"_ann_{annotation['id']}.jpg"
    )

    crop_path = (
        OUTPUT_DIR
        / class_name
        / crop_filename
    )

    crop.save(
        crop_path,
        quality=95
    )

    created += 1


# ============================================================
# FINAL SUMMARY
# ============================================================

print("\n" + "=" * 60)
print("TRAINING CROP CREATION COMPLETE")
print("=" * 60)

print(
    f"COCO annotations : {len(coco['annotations'])}"
)

print(
    f"Crops created    : {created}"
)

print(
    f"Skipped           : {skipped}"
)

print(
    f"\nOutput directory:"
    f"\n{OUTPUT_DIR}"
)

Category mapping:
  1: pet
  2: hdpe
  3: mixed_plastic_soft
  4: ecal
  5: metal
  6: cardboard
  7: mixed_plastic_rigid
  8: pet_oil


Creating training crops: 100%|██████████| 61842/61842 [33:57<00:00, 30.35it/s] 


TRAINING CROP CREATION COMPLETE
COCO annotations : 61842
Crops created    : 61842
Skipped           : 0

Output directory:
C:\Users\varda\Documents\_My Computer\COMMON_space\Learning\Upgrad_EPGP_MS\upgrad_IIITB_MS_course\MS_LJMU_Material\Topic Data\SortWaste\dataset\dataset\cropped_data\train


In [ ]:
# check distribution
from pathlib import Path

TRAIN_CROPS = Path(
    r"C:\Users\varda\Documents\_My Computer\COMMON_space\Learning\Upgrad_EPGP_MS\upgrad_IIITB_MS_course\MS_LJMU_Material\Topic Data\SortWaste\dataset\dataset\cropped_data\train"
)

print("TRAINING CROP COUNTS")
print("=" * 50)

total = 0

for class_dir in sorted(TRAIN_CROPS.iterdir()):

    if class_dir.is_dir():

        count = len(list(class_dir.glob("*.jpg")))

        print(f"{class_dir.name:25s}: {count:,}")

        total += count

print("=" * 50)
print(f"{'TOTAL':25s}: {total:,}")

TRAINING CROP COUNTS
cardboard                : 1,524
ecal                     : 13,649
hdpe                     : 16,803
metal                    : 945
mixed_plastic_rigid      : 7,066
mixed_plastic_soft       : 9,077
pet                      : 11,976
pet_oil                  : 802
TOTAL                    : 61,842


In [ ]:
from pathlib import Path
import os

# Find the cropped_data folder automatically
base = Path(
    r"C:\Users\varda\Documents\_My Computer\COMMON_space\Learning\Upgrad_EPGP_MS"
)

matches = list(
    base.rglob("cropped_data")
)

print("Found cropped_data folders:")

for m in matches:
    print(m)

TRAIN_CROPS = matches[0] / "train"

pet_folder = TRAIN_CROPS / "pet"

image_path = next(pet_folder.glob("*.jpg"))

print("\nOpening:")
print(image_path)

print("File size:", image_path.stat().st_size, "bytes")

os.startfile(image_path)

Found cropped_data folders:
C:\Users\varda\Documents\_My Computer\COMMON_space\Learning\Upgrad_EPGP_MS\upgrad_IIITB_MS_course\MS_LJMU_Material\Topic Data\SortWaste\dataset\dataset\cropped_data

Opening:
C:\Users\varda\Documents\_My Computer\COMMON_space\Learning\Upgrad_EPGP_MS\upgrad_IIITB_MS_course\MS_LJMU_Material\Topic Data\SortWaste\dataset\dataset\cropped_data\train\pet\0008_0005_5_ann_48.jpg
File size: 4996 bytes


In [ ]:
from pathlib import Path
import json

VAL_ROOT = Path(
    r"C:\Users\varda\Documents\_My Computer\COMMON_space\Learning\Upgrad_EPGP_MS\upgrad_IIITB_MS_course\MS_LJMU_Material\Topic Data\SortWaste\dataset\dataset\splited_all_dataset_coco\val"
)

print("Validation directory exists:", VAL_ROOT.exists())

print("\nContents:")
for item in VAL_ROOT.iterdir():
    print(item)

VAL_JSON = VAL_ROOT / "annotations" / "val_coco.json"

print("\nValidation JSON exists:", VAL_JSON.exists())

if VAL_JSON.exists():
    with open(VAL_JSON, "r", encoding="utf-8") as f:
        coco_val = json.load(f)

    print("Number of images:", len(coco_val["images"]))
    print("Number of annotations:", len(coco_val["annotations"]))
    print("Number of categories:", len(coco_val["categories"]))

Validation directory exists: True

Contents:
C:\Users\varda\Documents\_My Computer\COMMON_space\Learning\Upgrad_EPGP_MS\upgrad_IIITB_MS_course\MS_LJMU_Material\Topic Data\SortWaste\dataset\dataset\splited_all_dataset_coco\val\annotations
C:\Users\varda\Documents\_My Computer\COMMON_space\Learning\Upgrad_EPGP_MS\upgrad_IIITB_MS_course\MS_LJMU_Material\Topic Data\SortWaste\dataset\dataset\splited_all_dataset_coco\val\images
C:\Users\varda\Documents\_My Computer\COMMON_space\Learning\Upgrad_EPGP_MS\upgrad_IIITB_MS_course\MS_LJMU_Material\Topic Data\SortWaste\dataset\dataset\splited_all_dataset_coco\val\labels
C:\Users\varda\Documents\_My Computer\COMMON_space\Learning\Upgrad_EPGP_MS\upgrad_IIITB_MS_course\MS_LJMU_Material\Topic Data\SortWaste\dataset\dataset\splited_all_dataset_coco\val\labels.cache

Validation JSON exists: True
Number of images: 780
Number of annotations: 13065
Number of categories: 8


In [ ]:
## Creating VALIDATION cropped data
import json
from pathlib import Path
from PIL import Image
from tqdm import tqdm


# ============================================================
# PATHS
# ============================================================

VAL_ROOT = Path(
    r"C:\Users\varda\Documents\_My Computer\COMMON_space\Learning\Upgrad_EPGP_MS\upgrad_IIITB_MS_course\MS_LJMU_Material\Topic Data\SortWaste\dataset\dataset\splited_all_dataset_coco\val"
)

ANNOTATION_FILE = (
    VAL_ROOT
    / "annotations"
    / "val_coco.json"
)

IMAGE_DIR = (
    VAL_ROOT
    / "images"
)

OUTPUT_DIR = Path(
    r"C:\Users\varda\Documents\_My Computer\COMMON_space\Learning\Upgrad_EPGP_MS\upgrad_IIITB_MS_course\MS_LJMU_Material\Topic Data\SortWaste\dataset\dataset\cropped_data\val"
)


# ============================================================
# SETTINGS
# ============================================================

PADDING = 0.05


# ============================================================
# LOAD COCO JSON
# ============================================================

with open(
    ANNOTATION_FILE,
    "r",
    encoding="utf-8"
) as f:

    coco = json.load(f)


# ============================================================
# CATEGORY MAPPING
# ============================================================

category_id_to_name = {
    category["id"]: category["name"]
    for category in coco["categories"]
}

print("Category mapping:")

for category_id, category_name in category_id_to_name.items():
    print(f"  {category_id}: {category_name}")


# ============================================================
# IMAGE LOOKUP
# ============================================================

images_by_id = {
    image["id"]: image
    for image in coco["images"]
}


# ============================================================
# CREATE CLASS DIRECTORIES
# ============================================================

for class_name in category_id_to_name.values():

    class_dir = OUTPUT_DIR / class_name

    class_dir.mkdir(
        parents=True,
        exist_ok=True
    )


# ============================================================
# CREATE CROPS
# ============================================================

created = 0
skipped = 0

for annotation in tqdm(
    coco["annotations"],
    desc="Creating validation crops"
):

    image_id = annotation["image_id"]
    category_id = annotation["category_id"]


    # --------------------------------------------------------
    # Find image
    # --------------------------------------------------------

    if image_id not in images_by_id:

        skipped += 1
        continue

    image_info = images_by_id[image_id]

    image_filename = image_info["file_name"]

    image_path = IMAGE_DIR / image_filename


    if not image_path.exists():

        print(
            f"\nWARNING: Image not found:"
            f"\n{image_path}"
        )

        skipped += 1
        continue


    # --------------------------------------------------------
    # Find class
    # --------------------------------------------------------

    if category_id not in category_id_to_name:

        skipped += 1
        continue

    class_name = category_id_to_name[category_id]


    # --------------------------------------------------------
    # Open image
    # --------------------------------------------------------

    try:

        image = Image.open(
            image_path
        ).convert("RGB")

    except Exception as e:

        print(
            f"\nWARNING: Could not open "
            f"{image_path}: {e}"
        )

        skipped += 1
        continue


    # --------------------------------------------------------
    # COCO bounding box
    # [x, y, width, height]
    # --------------------------------------------------------

    x, y, width, height = annotation["bbox"]


    # --------------------------------------------------------
    # Add 5% padding
    # --------------------------------------------------------

    pad_x = width * PADDING
    pad_y = height * PADDING

    x1 = max(
        0,
        int(x - pad_x)
    )

    y1 = max(
        0,
        int(y - pad_y)
    )

    x2 = min(
        image.width,
        int(x + width + pad_x)
    )

    y2 = min(
        image.height,
        int(y + height + pad_y)
    )


    # --------------------------------------------------------
    # Validate bounding box
    # --------------------------------------------------------

    if x2 <= x1 or y2 <= y1:

        skipped += 1
        continue


    # --------------------------------------------------------
    # Crop
    # --------------------------------------------------------

    crop = image.crop(
        (x1, y1, x2, y2)
    )


    # --------------------------------------------------------
    # Save crop
    # --------------------------------------------------------

    crop_filename = (
        f"{Path(image_filename).stem}"
        f"_ann_{annotation['id']}.jpg"
    )

    crop_path = (
        OUTPUT_DIR
        / class_name
        / crop_filename
    )

    crop.save(
        crop_path,
        quality=95
    )

    created += 1


# ============================================================
# SUMMARY
# ============================================================

print("\n" + "=" * 60)
print("VALIDATION CROP CREATION COMPLETE")
print("=" * 60)

print(
    f"COCO annotations : {len(coco['annotations'])}"
)

print(
    f"Crops created    : {created}"
)

print(
    f"Skipped           : {skipped}"
)

print(
    f"\nOutput directory:"
    f"\n{OUTPUT_DIR}"
)

Category mapping:
  1: pet
  2: hdpe
  3: mixed_plastic_soft
  4: ecal
  5: metal
  6: cardboard
  7: mixed_plastic_rigid
  8: pet_oil


Creating validation crops: 100%|██████████| 13065/13065 [07:06<00:00, 30.61it/s]


VALIDATION CROP CREATION COMPLETE
COCO annotations : 13065
Crops created    : 13065
Skipped           : 0

Output directory:
C:\Users\varda\Documents\_My Computer\COMMON_space\Learning\Upgrad_EPGP_MS\upgrad_IIITB_MS_course\MS_LJMU_Material\Topic Data\SortWaste\dataset\dataset\cropped_data\val


In [ ]:
from pathlib import Path

VAL_CROPS = Path(
    r"C:\Users\varda\Documents\_My Computer\COMMON_space\Learning\Upgrad_EPGP_MS\upgrad_IIITB_MS_course\MS_LJMU_Material\Topic Data\SortWaste\dataset\dataset\cropped_data\val"
)

print("VALIDATION CROP COUNTS")
print("=" * 50)

total = 0

for class_dir in sorted(VAL_CROPS.iterdir()):

    if class_dir.is_dir():

        count = len(
            list(class_dir.glob("*.jpg"))
        )

        print(
            f"{class_dir.name:25s}: {count:,}"
        )

        total += count

print("=" * 50)
print(f"{'TOTAL':25s}: {total:,}")

VALIDATION CROP COUNTS
cardboard                : 425
ecal                     : 2,552
hdpe                     : 4,972
metal                    : 277
mixed_plastic_rigid      : 1,120
mixed_plastic_soft       : 1,443
pet                      : 2,108
pet_oil                  : 168
TOTAL                    : 13,065


In [ ]:
## set up check
import torch
import torchvision

print("PyTorch version :", torch.__version__)
print("Torchvision version :", torchvision.__version__)
print("CUDA available :", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU :", torch.cuda.get_device_name(0))
    print("CUDA version :", torch.version.cuda)

PyTorch version : 2.13.0+cu126
Torchvision version : 0.28.0+cu126
CUDA available : True
GPU : NVIDIA GeForce RTX 3050 Ti Laptop GPU
CUDA version : 12.6


In [ ]:
## load mobilenet large parameters
import torch
from torchvision.models import (
    mobilenet_v3_large,
    MobileNet_V3_Large_Weights
)

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

weights = MobileNet_V3_Large_Weights.DEFAULT

model = mobilenet_v3_large(
    weights=weights
)

print("MobileNetV3-Large loaded successfully")
print("Device:", device)
print(
    "Number of parameters:",
    sum(p.numel() for p in model.parameters())
)

Downloading: "https://download.pytorch.org/models/mobilenet_v3_large-5c1a4163.pth" to C:\Users\varda/.cache\torch\hub\checkpoints\mobilenet_v3_large-5c1a4163.pth


100%|██████████| 21.1M/21.1M [00:00<00:00, 28.2MB/s]


MobileNetV3-Large loaded successfully
Device: cuda
Number of parameters: 5483032


In [ ]:
## set up mobilenet with augmentation
import torch
from pathlib import Path

from torchvision import datasets
from torchvision.transforms import v2


# ============================================================
# PATHS
# ============================================================

DATA_ROOT = Path(
    r"C:\Users\varda\Documents\_My Computer\COMMON_space\Learning\Upgrad_EPGP_MS\upgrad_IIITB_MS_course\MS_LJMU_Material\Topic Data\SortWaste\dataset\dataset\cropped_data"
)

TRAIN_DIR = DATA_ROOT / "train"
VAL_DIR = DATA_ROOT / "val"


# ============================================================
# IMAGE SIZE
# ============================================================

IMAGE_SIZE = 224


# ============================================================
# IMAGENET NORMALIZATION
# ============================================================

IMAGENET_MEAN = [
    0.485,
    0.456,
    0.406
]

IMAGENET_STD = [
    0.229,
    0.224,
    0.225
]


# ============================================================
# VALIDATION TRANSFORM
# ============================================================

val_transform = v2.Compose([
    v2.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    v2.ToImage(),
    v2.ToDtype(
        dtype=torch.float32,
        scale=True
    ),
    v2.Normalize(
        mean=IMAGENET_MEAN,
        std=IMAGENET_STD
    )
])


# ============================================================
# CREATE VALIDATION DATASET
# ============================================================

val_dataset = datasets.ImageFolder(
    root=VAL_DIR,
    transform=val_transform
)


# ============================================================
# CHECK DATASET
# ============================================================

print("Validation samples:", len(val_dataset))

print("\nClass names:")
print(val_dataset.classes)

print("\nClass → index mapping:")
print(val_dataset.class_to_idx)

Validation samples: 13065

Class names:
['cardboard', 'ecal', 'hdpe', 'metal', 'mixed_plastic_rigid', 'mixed_plastic_soft', 'pet', 'pet_oil']

Class → index mapping:
{'cardboard': 0, 'ecal': 1, 'hdpe': 2, 'metal': 3, 'mixed_plastic_rigid': 4, 'mixed_plastic_soft': 5, 'pet': 6, 'pet_oil': 7}


In [ ]:
## augmentations
import torch
from pathlib import Path

from torchvision import datasets
from torchvision.transforms import v2


# ============================================================
# PATHS
# ============================================================

DATA_ROOT = Path(
    r"C:\Users\varda\Documents\_My Computer\COMMON_space\Learning\Upgrad_EPGP_MS\upgrad_IIITB_MS_course\MS_LJMU_Material\Topic Data\SortWaste\dataset\dataset\cropped_data"
)

TRAIN_DIR = DATA_ROOT / "train"
VAL_DIR = DATA_ROOT / "val"


# ============================================================
# SETTINGS
# ============================================================

IMAGE_SIZE = 224

IMAGENET_MEAN = [
    0.485,
    0.456,
    0.406
]

IMAGENET_STD = [
    0.229,
    0.224,
    0.225
]


# ============================================================
# TRAINING TRANSFORM
# ============================================================

train_transform = v2.Compose([

    # Convert PIL image to torchvision image
    v2.ToImage(),

    # Mild scale/crop variation
    v2.RandomResizedCrop(
        size=(IMAGE_SIZE, IMAGE_SIZE),
        scale=(0.85, 1.0),
        ratio=(0.8, 1.25)
    ),

    # Objects can appear in different orientations
    v2.RandomHorizontalFlip(
        p=0.5
    ),

    # Small realistic rotation
    v2.RandomRotation(
        degrees=10
    ),

    # Mild lighting and colour variation
    v2.ColorJitter(
        brightness=0.2,
        contrast=0.2,
        saturation=0.15,
        hue=0.03
    ),

    # Convert to float tensor [0, 1]
    v2.ToDtype(
        torch.float32,
        scale=True
    ),

    # ImageNet normalization
    v2.Normalize(
        mean=IMAGENET_MEAN,
        std=IMAGENET_STD
    )
])


# ============================================================
# CREATE TRAINING DATASET
# ============================================================

train_dataset = datasets.ImageFolder(
    root=TRAIN_DIR,
    transform=train_transform
)


# ============================================================
# CHECK DATASET
# ============================================================

print("Training samples:", len(train_dataset))

print("\nClass names:")
print(train_dataset.classes)

print("\nClass → index mapping:")
print(train_dataset.class_to_idx)

Training samples: 61842

Class names:
['cardboard', 'ecal', 'hdpe', 'metal', 'mixed_plastic_rigid', 'mixed_plastic_soft', 'pet', 'pet_oil']

Class → index mapping:
{'cardboard': 0, 'ecal': 1, 'hdpe': 2, 'metal': 3, 'mixed_plastic_rigid': 4, 'mixed_plastic_soft': 5, 'pet': 6, 'pet_oil': 7}


In [ ]:
## model set up
from torch.utils.data import DataLoader


# ============================================================
# DATALOADER SETTINGS
# ============================================================

BATCH_SIZE = 32

NUM_WORKERS = 2


# ============================================================
# TRAIN DATALOADER
# ============================================================

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=True
)


# ============================================================
# VALIDATION DATALOADER
# ============================================================

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True
)


# ============================================================
# CHECK
# ============================================================

print("Batch size:", BATCH_SIZE)

print(
    "Training batches:",
    len(train_loader)
)

print(
    "Validation batches:",
    len(val_loader)
)

Batch size: 32
Training batches: 1933
Validation batches: 409


In [ ]:
## checking tensors
# ============================================================
# SANITY CHECK — ONE TRAINING BATCH
# ============================================================

images, labels = next(iter(train_loader))

print("Image tensor shape :", images.shape)
print("Label tensor shape :", labels.shape)

print("Image dtype        :", images.dtype)
print("Label dtype        :", labels.dtype)

print("Image device       :", images.device)

print("Image min          :", images.min().item())
print("Image max          :", images.max().item())

print("First 10 labels    :", labels[:10].tolist())

Image tensor shape : torch.Size([32, 3, 224, 224])
Label tensor shape : torch.Size([32])
Image dtype        : torch.float32
Label dtype        : torch.int64
Image device       : cpu
Image min          : -2.1179039478302
Image max          : 2.640000104904175
First 10 labels    : [6, 1, 2, 2, 4, 5, 4, 2, 5, 1]


In [ ]:
## check if augmentation is working
from pathlib import Path
from PIL import Image
import torch


# ============================================================
# PATHS
# ============================================================

TRAIN_CROPS = Path(
    r"C:\Users\varda\Documents\_My Computer\COMMON_space\Learning\Upgrad_EPGP_MS\upgrad_IIITB_MS_course\MS_LJMU_Material\Topic Data\SortWaste\dataset\dataset\cropped_data\train"
)

TEST_AUG_DIR = TRAIN_CROPS.parent / "augmentation_test"

TEST_AUG_DIR.mkdir(
    parents=True,
    exist_ok=True
)


# ============================================================
# LOAD ONE ORIGINAL CROP
# ============================================================

image_path = next(
    (TRAIN_CROPS / "pet").glob("*.jpg")
)

original_image = Image.open(
    image_path
).convert("RGB")

print("Original:")
print(image_path)
print("Size:", original_image.size)


# ============================================================
# UN-NORMALIZATION VALUES
# ============================================================

mean = torch.tensor(
    IMAGENET_MEAN
).view(3, 1, 1)

std = torch.tensor(
    IMAGENET_STD
).view(3, 1, 1)


# ============================================================
# GENERATE 5 AUGMENTED IMAGES
# ============================================================

for i in range(5):

    augmented = train_transform(
        original_image
    )

    # Undo normalization for saving
    augmented = (
        augmented * std + mean
    )

    augmented = augmented.clamp(
        0,
        1
    )

    # Convert tensor → PIL
    augmented = (
        augmented
        .mul(255)
        .byte()
        .permute(1, 2, 0)
        .cpu()
        .numpy()
    )

    augmented_image = Image.fromarray(
        augmented
    )

    output_path = (
        TEST_AUG_DIR
        / f"pet_augmented_{i+1}.jpg"
    )

    augmented_image.save(
        output_path,
        quality=95
    )

    print(
        "Saved:",
        output_path
    )


print("\nDone.")

Original:
C:\Users\varda\Documents\_My Computer\COMMON_space\Learning\Upgrad_EPGP_MS\upgrad_IIITB_MS_course\MS_LJMU_Material\Topic Data\SortWaste\dataset\dataset\cropped_data\train\pet\0008_0005_5_ann_48.jpg
Size: (94, 146)
Saved: C:\Users\varda\Documents\_My Computer\COMMON_space\Learning\Upgrad_EPGP_MS\upgrad_IIITB_MS_course\MS_LJMU_Material\Topic Data\SortWaste\dataset\dataset\cropped_data\augmentation_test\pet_augmented_1.jpg
Saved: C:\Users\varda\Documents\_My Computer\COMMON_space\Learning\Upgrad_EPGP_MS\upgrad_IIITB_MS_course\MS_LJMU_Material\Topic Data\SortWaste\dataset\dataset\cropped_data\augmentation_test\pet_augmented_2.jpg
Saved: C:\Users\varda\Documents\_My Computer\COMMON_space\Learning\Upgrad_EPGP_MS\upgrad_IIITB_MS_course\MS_LJMU_Material\Topic Data\SortWaste\dataset\dataset\cropped_data\augmentation_test\pet_augmented_3.jpg
Saved: C:\Users\varda\Documents\_My Computer\COMMON_space\Learning\Upgrad_EPGP_MS\upgrad_IIITB_MS_course\MS_LJMU_Material\Topic Data\SortWaste\dat

In [ ]:
## model initialization
import torch

from torchvision.models import (
    mobilenet_v3_large,
    MobileNet_V3_Large_Weights
)


# ============================================================
# DEVICE
# ============================================================

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Using device:", device)


# ============================================================
# LOAD PRETRAINED MOBILENETV3-LARGE
# ============================================================

weights = MobileNet_V3_Large_Weights.DEFAULT

model = mobilenet_v3_large(
    weights=weights
)


# ============================================================
# REPLACE IMAGENET CLASSIFIER
# ============================================================

num_classes = 8

input_features = (
    model.classifier[-1].in_features
)

model.classifier[-1] = torch.nn.Linear(
    input_features,
    num_classes
)


# ============================================================
# MOVE MODEL TO GPU
# ============================================================

model = model.to(device)


# ============================================================
# CHECK
# ============================================================

print("\nModel:", model.__class__.__name__)

print(
    "Classifier:",
    model.classifier
)

print(
    "\nNumber of output classes:",
    num_classes
)

print(
    "Model device:",
    next(model.parameters()).device
)

Using device: cuda

Model: MobileNetV3
Classifier: Sequential(
  (0): Linear(in_features=960, out_features=1280, bias=True)
  (1): Hardswish()
  (2): Dropout(p=0.2, inplace=True)
  (3): Linear(in_features=1280, out_features=8, bias=True)
)

Number of output classes: 8
Model device: cuda:0


In [ ]:
import torch


# ============================================================
# LOSS FUNCTION
# ============================================================

criterion = torch.nn.CrossEntropyLoss()


# ============================================================
# OPTIMIZER
# ============================================================

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=1e-4,
    weight_decay=1e-4
)


# ============================================================
# CHECK
# ============================================================

print("Loss function:")
print(criterion)

print("\nOptimizer:")
print(optimizer)

Loss function:
CrossEntropyLoss()

Optimizer:
AdamW (
Parameter Group 0
    amsgrad: False
    betas: (0.9, 0.999)
    capturable: False
    decoupled_weight_decay: True
    differentiable: False
    eps: 1e-08
    foreach: None
    fused: None
    lr: 0.0001
    maximize: False
    weight_decay: 0.0001
)


In [ ]:
## dry run
import time
import torch


# ============================================================
# ONE-EPOCH TRAINING TEST
# ============================================================

model.train()

train_loss = 0.0
correct = 0
total = 0

start_time = time.time()


for batch_idx, (images, labels) in enumerate(train_loader):

    # --------------------------------------------------------
    # Move data to GPU
    # --------------------------------------------------------

    images = images.to(device, non_blocking=True)
    labels = labels.to(device, non_blocking=True)

    # --------------------------------------------------------
    # Clear previous gradients
    # --------------------------------------------------------

    optimizer.zero_grad()

    # --------------------------------------------------------
    # Forward pass
    # --------------------------------------------------------

    outputs = model(images)

    # --------------------------------------------------------
    # Calculate loss
    # --------------------------------------------------------

    loss = criterion(outputs, labels)

    # --------------------------------------------------------
    # Backpropagation
    # --------------------------------------------------------

    loss.backward()

    # --------------------------------------------------------
    # Update model weights
    # --------------------------------------------------------

    optimizer.step()

    # --------------------------------------------------------
    # Statistics
    # --------------------------------------------------------

    train_loss += loss.item()

    _, predicted = torch.max(outputs, 1)

    total += labels.size(0)

    correct += (
        predicted == labels
    ).sum().item()

    # --------------------------------------------------------
    # Progress
    # --------------------------------------------------------

    if (batch_idx + 1) % 100 == 0:

        print(
            f"Batch [{batch_idx + 1}/{len(train_loader)}] "
            f"Loss: {loss.item():.4f}"
        )


# ============================================================
# TRAINING RESULTS
# ============================================================

epoch_loss = train_loss / len(train_loader)

epoch_accuracy = (
    100.0 * correct / total
)

elapsed = time.time() - start_time


print("\n" + "=" * 60)
print("ONE-EPOCH TRAINING TEST COMPLETE")
print("=" * 60)

print(f"Training loss     : {epoch_loss:.4f}")
print(f"Training accuracy : {epoch_accuracy:.2f}%")
print(f"Images processed  : {total}")
print(f"Time taken        : {elapsed / 60:.2f} minutes")

Batch [100/1933] Loss: 1.1838
Batch [200/1933] Loss: 0.9387
Batch [300/1933] Loss: 0.7679
Batch [400/1933] Loss: 0.8055
Batch [500/1933] Loss: 0.3399
Batch [600/1933] Loss: 0.4961
Batch [700/1933] Loss: 0.7721
Batch [800/1933] Loss: 0.5504
Batch [900/1933] Loss: 0.4857
Batch [1000/1933] Loss: 0.5646
Batch [1100/1933] Loss: 0.6245
Batch [1200/1933] Loss: 0.2192
Batch [1300/1933] Loss: 0.3131
Batch [1400/1933] Loss: 0.5410
Batch [1500/1933] Loss: 0.2926
Batch [1600/1933] Loss: 0.1418
Batch [1700/1933] Loss: 0.3177
Batch [1800/1933] Loss: 0.2091
Batch [1900/1933] Loss: 0.1864

ONE-EPOCH TRAINING TEST COMPLETE
Training loss     : 0.6099
Training accuracy : 78.34%
Images processed  : 61842
Time taken        : 8.39 minutes


In [ ]:
# VALIDATION — AFTER ONE TRAINING EPOCH
# ============================================================

import torch
from sklearn.metrics import (
    accuracy_score,
    f1_score
)


# ============================================================
# EVALUATION MODE
# ============================================================

model.eval()


val_loss = 0.0

all_predictions = []
all_labels = []


# ============================================================
# VALIDATION LOOP
# ============================================================

with torch.no_grad():

    for images, labels in val_loader:

        # Move data to GPU
        images = images.to(
            device,
            non_blocking=True
        )

        labels = labels.to(
            device,
            non_blocking=True
        )

        # Forward pass
        outputs = model(images)

        # Validation loss
        loss = criterion(
            outputs,
            labels
        )

        val_loss += loss.item()

        # Predictions
        predictions = torch.argmax(
            outputs,
            dim=1
        )

        # Move predictions/labels back to CPU
        all_predictions.extend(
            predictions.cpu().numpy()
        )

        all_labels.extend(
            labels.cpu().numpy()
        )


# ============================================================
# CALCULATE METRICS
# ============================================================

val_loss = (
    val_loss / len(val_loader)
)

val_accuracy = accuracy_score(
    all_labels,
    all_predictions
)

val_macro_f1 = f1_score(
    all_labels,
    all_predictions,
    average="macro"
)

val_weighted_f1 = f1_score(
    all_labels,
    all_predictions,
    average="weighted"
)


# ============================================================
# RESULTS
# ============================================================

print("\n" + "=" * 60)
print("VALIDATION RESULTS — AFTER ONE EPOCH")
print("=" * 60)

print(
    f"Validation loss     : {val_loss:.4f}"
)

print(
    f"Validation accuracy : {val_accuracy * 100:.2f}%"
)

print(
    f"Validation Macro F1  : {val_macro_f1:.4f}"
)

print(
    f"Validation Weighted F1: {val_weighted_f1:.4f}"
)

print(
    f"Validation samples  : {len(all_labels)}"
)


VALIDATION RESULTS — AFTER ONE EPOCH
Validation loss     : 0.7317
Validation accuracy : 80.09%
Validation Macro F1  : 0.6753
Validation Weighted F1: 0.7917
Validation samples  : 13065


In [ ]:
from sklearn.metrics import classification_report


# ============================================================
# CLASS NAMES
# ============================================================

class_names = val_dataset.classes


# ============================================================
# PER-CLASS CLASSIFICATION REPORT
# ============================================================

print("\n" + "=" * 80)
print("PER-CLASS VALIDATION RESULTS — AFTER ONE EPOCH")
print("=" * 80)

print(
    classification_report(
        all_labels,
        all_predictions,
        target_names=class_names,
        digits=4
    )
)


PER-CLASS VALIDATION RESULTS — AFTER ONE EPOCH
                     precision    recall  f1-score   support

          cardboard     0.5583    0.1576    0.2459       425
               ecal     0.7864    0.9173    0.8468      2552
               hdpe     0.8609    0.8737    0.8672      4972
              metal     0.7514    0.4693    0.5778       277
mixed_plastic_rigid     0.6500    0.6286    0.6391      1120
 mixed_plastic_soft     0.6713    0.6369    0.6536      1443
                pet     0.8548    0.8800    0.8672      2108
            pet_oil     0.8189    0.6190    0.7051       168

           accuracy                         0.8009     13065
          macro avg     0.7440    0.6478    0.6753     13065
       weighted avg     0.7936    0.8009    0.7917     13065



In [ ]:
## Train/run the final model
import torch
from torchvision.models import (
    mobilenet_v3_large,
    MobileNet_V3_Large_Weights
)


# ============================================================
# REPRODUCIBILITY
# ============================================================

torch.manual_seed(42)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)


# ============================================================
# DEVICE
# ============================================================

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device:", device)


# ============================================================
# LOAD FRESH IMAGENET-PRETRAINED MODEL
# ============================================================

weights = MobileNet_V3_Large_Weights.DEFAULT

model = mobilenet_v3_large(
    weights=weights
)


# ============================================================
# REPLACE FINAL CLASSIFIER
# ============================================================

num_classes = 8

input_features = model.classifier[-1].in_features

model.classifier[-1] = torch.nn.Linear(
    input_features,
    num_classes
)


# ============================================================
# MOVE TO GPU
# ============================================================

model = model.to(device)


# ============================================================
# LOSS
# ============================================================

criterion = torch.nn.CrossEntropyLoss()


# ============================================================
# OPTIMIZER
# ============================================================

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=1e-4,
    weight_decay=1e-4
)


# ============================================================
# CHECK
# ============================================================

print("\nFresh MobileNetV3-Large loaded.")
print("Output classes:", num_classes)
print("Device:", next(model.parameters()).device)
print("Learning rate:", optimizer.param_groups[0]["lr"])

Device: cuda

Fresh MobileNetV3-Large loaded.
Output classes: 8
Device: cuda:0
Learning rate: 0.0001


In [ ]:
##### Training mobilenetv3 large for 20 epochs #####
import time
import copy
import pandas as pd

from sklearn.metrics import (
    accuracy_score,
    f1_score
)


# ============================================================
# TRAINING SETTINGS
# ============================================================

NUM_EPOCHS = 20

BEST_MACRO_F1 = -1.0

BEST_MODEL_PATH = (
    DATA_ROOT / "mobilenet_v3_large_E3_best.pth"
)


# ============================================================
# HISTORY
# ============================================================

history = []


# ============================================================
# TRAINING LOOP
# ============================================================

for epoch in range(NUM_EPOCHS):

    epoch_start = time.time()

    print("\n" + "=" * 80)
    print(
        f"EPOCH {epoch + 1}/{NUM_EPOCHS}"
    )
    print("=" * 80)


    # ========================================================
    # TRAINING
    # ========================================================

    model.train()

    running_train_loss = 0.0

    train_correct = 0
    train_total = 0


    for batch_idx, (images, labels) in enumerate(
        train_loader
    ):

        # ----------------------------------------------------
        # Move to GPU
        # ----------------------------------------------------

        images = images.to(
            device,
            non_blocking=True
        )

        labels = labels.to(
            device,
            non_blocking=True
        )


        # ----------------------------------------------------
        # Clear gradients
        # ----------------------------------------------------

        optimizer.zero_grad(
            set_to_none=True
        )


        # ----------------------------------------------------
        # Forward
        # ----------------------------------------------------

        outputs = model(images)


        # ----------------------------------------------------
        # Loss
        # ----------------------------------------------------

        loss = criterion(
            outputs,
            labels
        )


        # ----------------------------------------------------
        # Backpropagation
        # ----------------------------------------------------

        loss.backward()


        # ----------------------------------------------------
        # Update weights
        # ----------------------------------------------------

        optimizer.step()


        # ----------------------------------------------------
        # Statistics
        # ----------------------------------------------------

        running_train_loss += loss.item()

        predictions = torch.argmax(
            outputs,
            dim=1
        )

        train_total += labels.size(0)

        train_correct += (
            predictions == labels
        ).sum().item()


        # ----------------------------------------------------
        # Progress
        # ----------------------------------------------------

        if (batch_idx + 1) % 200 == 0:

            print(
                f"Batch [{batch_idx + 1}/{len(train_loader)}] "
                f"Loss: {loss.item():.4f}"
            )


    # ========================================================
    # TRAINING METRICS
    # ========================================================

    train_loss = (
        running_train_loss
        / len(train_loader)
    )

    train_accuracy = (
        train_correct
        / train_total
    )


    # ========================================================
    # VALIDATION
    # ========================================================

    model.eval()

    running_val_loss = 0.0

    all_predictions = []
    all_labels = []


    with torch.no_grad():

        for images, labels in val_loader:

            images = images.to(
                device,
                non_blocking=True
            )

            labels = labels.to(
                device,
                non_blocking=True
            )


            # Forward pass
            outputs = model(images)


            # Validation loss
            loss = criterion(
                outputs,
                labels
            )

            running_val_loss += loss.item()


            # Predictions
            predictions = torch.argmax(
                outputs,
                dim=1
            )


            all_predictions.extend(
                predictions.cpu().numpy()
            )

            all_labels.extend(
                labels.cpu().numpy()
            )


    # ========================================================
    # VALIDATION METRICS
    # ========================================================

    val_loss = (
        running_val_loss
        / len(val_loader)
    )

    val_accuracy = accuracy_score(
        all_labels,
        all_predictions
    )

    val_macro_f1 = f1_score(
        all_labels,
        all_predictions,
        average="macro"
    )

    val_weighted_f1 = f1_score(
        all_labels,
        all_predictions,
        average="weighted"
    )


    # ========================================================
    # SAVE BEST MODEL
    # ========================================================

    is_best = (
        val_macro_f1 > BEST_MACRO_F1
    )


    if is_best:

        BEST_MACRO_F1 = val_macro_f1

        torch.save(
            {
                "epoch": epoch + 1,
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "val_macro_f1": val_macro_f1,
                "val_accuracy": val_accuracy,
                "val_loss": val_loss,
                "class_names": val_dataset.classes,
                "image_size": IMAGE_SIZE,
            },
            BEST_MODEL_PATH
        )


    # ========================================================
    # RECORD HISTORY
    # ========================================================

    epoch_time = (
        time.time()
        - epoch_start
    )

    history.append(
        {
            "epoch": epoch + 1,
            "train_loss": train_loss,
            "train_accuracy": train_accuracy,
            "val_loss": val_loss,
            "val_accuracy": val_accuracy,
            "val_macro_f1": val_macro_f1,
            "val_weighted_f1": val_weighted_f1,
            "epoch_time_minutes": (
                epoch_time / 60
            ),
            "best_model": is_best,
        }
    )


    # ========================================================
    # PRINT RESULTS
    # ========================================================

    print("\nEpoch results:")

    print(
        f"Train Loss       : {train_loss:.4f}"
    )

    print(
        f"Train Accuracy   : "
        f"{train_accuracy * 100:.2f}%"
    )

    print(
        f"Val Loss         : {val_loss:.4f}"
    )

    print(
        f"Val Accuracy     : "
        f"{val_accuracy * 100:.2f}%"
    )

    print(
        f"Val Macro F1     : "
        f"{val_macro_f1:.4f}"
    )

    print(
        f"Val Weighted F1  : "
        f"{val_weighted_f1:.4f}"
    )

    print(
        f"Epoch Time       : "
        f"{epoch_time / 60:.2f} min"
    )

    if is_best:
        print(
            "★ NEW BEST MODEL — saved"
        )
    else:
        print(
            f"Best Macro F1 so far: "
            f"{BEST_MACRO_F1:.4f}"
        )


# ============================================================
# FINAL HISTORY
# ============================================================

history_df = pd.DataFrame(history)


print("\n" + "=" * 80)
print("20-EPOCH TRAINING COMPLETE")
print("=" * 80)

print(
    f"Best Validation Macro F1: "
    f"{BEST_MACRO_F1:.4f}"
)

print(
    f"Best model saved to:\n"
    f"{BEST_MODEL_PATH}"
)


print("\nTraining history:")
print(history_df.to_string(index=False))


EPOCH 1/20
Batch [200/1933] Loss: 0.8148
Batch [400/1933] Loss: 0.7815
Batch [600/1933] Loss: 0.7616
Batch [800/1933] Loss: 0.6677
Batch [1000/1933] Loss: 0.4598
Batch [1200/1933] Loss: 0.5729
Batch [1400/1933] Loss: 0.4470
Batch [1600/1933] Loss: 0.3551
Batch [1800/1933] Loss: 0.5986

Epoch results:
Train Loss       : 0.6104
Train Accuracy   : 78.36%
Val Loss         : 0.6985
Val Accuracy     : 78.52%
Val Macro F1     : 0.6734
Val Weighted F1  : 0.7802
Epoch Time       : 5.09 min
★ NEW BEST MODEL — saved

EPOCH 2/20
Batch [200/1933] Loss: 0.6652
Batch [400/1933] Loss: 0.2127
Batch [600/1933] Loss: 0.2400
Batch [800/1933] Loss: 0.2837
Batch [1000/1933] Loss: 0.5784
Batch [1200/1933] Loss: 0.0826
Batch [1400/1933] Loss: 0.1905
Batch [1600/1933] Loss: 0.0984
Batch [1800/1933] Loss: 0.1855

Epoch results:
Train Loss       : 0.2309
Train Accuracy   : 92.14%
Val Loss         : 0.9553
Val Accuracy     : 76.52%
Val Macro F1     : 0.6705
Val Weighted F1  : 0.7651
Epoch Time       : 5.12 min
B

In [ ]:
#### VALIDATION
# ============================================================
# E3 — BEST CHECKPOINT EVALUATION
# MobileNetV3-Large — Epoch 8
# ============================================================

import torch
import numpy as np
import matplotlib.pyplot as plt

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay
)

from torchvision.models import (
    mobilenet_v3_large
)


# ============================================================
# PATH TO BEST CHECKPOINT
# ============================================================

BEST_MODEL_PATH = (
    DATA_ROOT / "mobilenet_v3_large_E3_best.pth"
)

print("Checkpoint:")
print(BEST_MODEL_PATH)


# ============================================================
# LOAD FRESH MODEL ARCHITECTURE
# ============================================================

model = mobilenet_v3_large(
    weights=None
)


# Replace final classifier
num_classes = 8

input_features = model.classifier[-1].in_features

model.classifier[-1] = torch.nn.Linear(
    input_features,
    num_classes
)


# ============================================================
# LOAD CHECKPOINT
# ============================================================

checkpoint = torch.load(
    BEST_MODEL_PATH,
    map_location=device
)


model.load_state_dict(
    checkpoint["model_state_dict"]
)

model = model.to(device)

model.eval()


# ============================================================
# CHECKPOINT INFORMATION
# ============================================================

print("\n" + "=" * 70)
print("BEST CHECKPOINT INFORMATION")
print("=" * 70)

print(
    "Saved epoch       :",
    checkpoint["epoch"]
)

print(
    "Validation Macro F1:",
    f"{checkpoint['val_macro_f1']:.4f}"
)

print(
    "Validation accuracy:",
    f"{checkpoint['val_accuracy'] * 100:.2f}%"
)

print(
    "Validation loss    :",
    f"{checkpoint['val_loss']:.4f}"
)


# ============================================================
# CLASS MAPPING
# ============================================================

class_names = val_dataset.classes

print("\n" + "=" * 70)
print("MOBILENET CLASS MAPPING")
print("=" * 70)

for idx, class_name in enumerate(class_names):

    print(
        f"{idx} -> {class_name}"
    )


# ============================================================
# VALIDATION
# ============================================================

all_predictions = []
all_labels = []

running_val_loss = 0.0


with torch.no_grad():

    for images, labels in val_loader:

        images = images.to(
            device,
            non_blocking=True
        )

        labels = labels.to(
            device,
            non_blocking=True
        )

        outputs = model(images)

        loss = criterion(
            outputs,
            labels
        )

        running_val_loss += loss.item()

        predictions = torch.argmax(
            outputs,
            dim=1
        )

        all_predictions.extend(
            predictions.cpu().numpy()
        )

        all_labels.extend(
            labels.cpu().numpy()
        )


# ============================================================
# NUMPY ARRAYS
# ============================================================

all_labels = np.array(
    all_labels
)

all_predictions = np.array(
    all_predictions
)


# ============================================================
# OVERALL METRICS
# ============================================================

val_loss = (
    running_val_loss
    / len(val_loader)
)

val_accuracy = accuracy_score(
    all_labels,
    all_predictions
)

macro_f1 = f1_score(
    all_labels,
    all_predictions,
    average="macro"
)

weighted_f1 = f1_score(
    all_labels,
    all_predictions,
    average="weighted"
)


# ============================================================
# OVERALL RESULTS
# ============================================================

print("\n" + "=" * 70)
print("BEST CHECKPOINT — OVERALL VALIDATION RESULTS")
print("=" * 70)

print(
    f"Epoch              : {checkpoint['epoch']}"
)

print(
    f"Validation loss    : {val_loss:.4f}"
)

print(
    f"Validation accuracy: {val_accuracy * 100:.2f}%"
)

print(
    f"Macro F1           : {macro_f1:.4f}"
)

print(
    f"Weighted F1        : {weighted_f1:.4f}"
)

print(
    f"Validation samples : {len(all_labels)}"
)


# ============================================================
# PER-CLASS CLASSIFICATION REPORT
# ============================================================

print("\n" + "=" * 80)
print("PER-CLASS VALIDATION RESULTS — BEST CHECKPOINT")
print("=" * 80)

print(
    classification_report(
        all_labels,
        all_predictions,
        target_names=class_names,
        digits=4
    )
)


# ============================================================
# CONFUSION MATRIX
# ============================================================

cm = confusion_matrix(
    all_labels,
    all_predictions
)


print("\n" + "=" * 80)
print("CONFUSION MATRIX")
print("=" * 80)

print(cm)


# ============================================================
# DISPLAY CONFUSION MATRIX
# ============================================================

fig, ax = plt.subplots(
    figsize=(12, 10)
)

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=class_names
)

disp.plot(
    ax=ax,
    xticks_rotation=45
)

plt.title(
    "MobileNetV3-Large — E3 Best Checkpoint\n"
    f"Epoch {checkpoint['epoch']}"
)

plt.tight_layout()

plt.show()

Checkpoint:
C:\Users\varda\Documents\_My Computer\COMMON_space\Learning\Upgrad_EPGP_MS\upgrad_IIITB_MS_course\MS_LJMU_Material\Topic Data\SortWaste\dataset\dataset\cropped_data\mobilenet_v3_large_E3_best.pth

BEST CHECKPOINT INFORMATION
Saved epoch       : 8
Validation Macro F1: 0.7100
Validation accuracy: 80.24%
Validation loss    : 1.0887

MOBILENET CLASS MAPPING
0 -> cardboard
1 -> ecal
2 -> hdpe
3 -> metal
4 -> mixed_plastic_rigid
5 -> mixed_plastic_soft
6 -> pet
7 -> pet_oil

BEST CHECKPOINT — OVERALL VALIDATION RESULTS
Epoch              : 8
Validation loss    : 1.0887
Validation accuracy: 80.24%
Macro F1           : 0.7100
Weighted F1        : 0.7966
Validation samples : 13065

PER-CLASS VALIDATION RESULTS — BEST CHECKPOINT
                     precision    recall  f1-score   support

          cardboard     0.5911    0.3435    0.4345       425
               ecal     0.8205    0.8797    0.8491      2552
               hdpe     0.8432    0.8890    0.8655      4972
              

<Figure size 1200x1000 with 2 Axes>

## E3B: Classification Model E3B : Mobilenet with realistic augmentation + imbalance handling

In [ ]:
## Class weights calculation
# ============================================================
# E3-B — CLASS WEIGHTS
# ============================================================

import torch
import numpy as np


# ============================================================
# TRAINING CLASS COUNTS
# ============================================================

class_counts = np.array([
    1524,    # cardboard
    13649,   # ecal
    16803,   # hdpe
    945,     # metal
    7066,    # mixed_plastic_rigid
    9077,    # mixed_plastic_soft
    11976,   # pet
    802      # pet_oil
])


# ============================================================
# CHECK TOTAL
# ============================================================

print("Total training samples:",
      class_counts.sum())


# ============================================================
# MODERATED INVERSE-FREQUENCY WEIGHTS
# ============================================================

total_samples = class_counts.sum()

class_weights = np.sqrt(
    total_samples / class_counts
)


# ============================================================
# NORMALIZE WEIGHTS
# ============================================================

class_weights = (
    class_weights / class_weights.mean()
)


# ============================================================
# DISPLAY
# ============================================================

print("\n" + "=" * 70)
print("CLASS WEIGHTS")
print("=" * 70)

for idx, (name, count, weight) in enumerate(
    zip(
        train_dataset.classes,
        class_counts,
        class_weights
    )
):

    print(
        f"{idx:>2}  "
        f"{name:<25} "
        f"count={count:>6}  "
        f"weight={weight:.4f}"
    )


# ============================================================
# CONVERT TO GPU TENSOR
# ============================================================

class_weights_tensor = torch.tensor(
    class_weights,
    dtype=torch.float32,
    device=device
)


print("\nClass weights tensor:")
print(class_weights_tensor)

Total training samples: 61842

CLASS WEIGHTS
 0  cardboard                 count=  1524  weight=1.4507
 1  ecal                      count= 13649  weight=0.4847
 2  hdpe                      count= 16803  weight=0.4369
 3  metal                     count=   945  weight=1.8423
 4  mixed_plastic_rigid       count=  7066  weight=0.6737
 5  mixed_plastic_soft        count=  9077  weight=0.5944
 6  pet                       count= 11976  weight=0.5175
 7  pet_oil                   count=   802  weight=1.9998

Class weights tensor:
tensor([1.4507, 0.4847, 0.4369, 1.8423, 0.6737, 0.5944, 0.5175, 1.9998], device='cuda:0')


In [ ]:
## model set-up
# ============================================================
# E3-B — FRESH MOBILENETV3-LARGE
# CLASS-WEIGHTED LOSS
# ============================================================

import torch

from torchvision.models import (
    mobilenet_v3_large,
    MobileNet_V3_Large_Weights
)


# ============================================================
# REPRODUCIBILITY
# ============================================================

torch.manual_seed(42)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)


# ============================================================
# DEVICE
# ============================================================

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device:", device)


# ============================================================
# LOAD FRESH IMAGENET-PRETRAINED MODEL
# ============================================================

weights = MobileNet_V3_Large_Weights.DEFAULT

model = mobilenet_v3_large(
    weights=weights
)


# ============================================================
# REPLACE FINAL CLASSIFIER
# ============================================================

num_classes = 8

input_features = (
    model.classifier[-1].in_features
)

model.classifier[-1] = torch.nn.Linear(
    input_features,
    num_classes
)


# ============================================================
# MOVE MODEL TO DEVICE
# ============================================================

model = model.to(device)


# ============================================================
# CLASS-WEIGHTED CROSS ENTROPY
# ============================================================

criterion = torch.nn.CrossEntropyLoss(
    weight=class_weights_tensor
)


# ============================================================
# OPTIMIZER
# ============================================================

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=1e-4,
    weight_decay=1e-4
)


# ============================================================
# CHECK
# ============================================================

print("\n" + "=" * 70)
print("E3-B MODEL CONFIGURATION")
print("=" * 70)

print(
    "Model              : MobileNetV3-Large"
)

print(
    "Number of classes  :",
    num_classes
)

print(
    "Loss               : "
    "Weighted CrossEntropyLoss"
)

print(
    "Optimizer          : AdamW"
)

print(
    "Learning rate      : 0.0001"
)

print(
    "Weight decay       : 0.0001"
)

print(
    "Device             :",
    next(model.parameters()).device
)

print(
    "\nClass weights:"
)

for name, weight in zip(
    train_dataset.classes,
    class_weights_tensor.cpu().numpy()
):

    print(
        f"{name:<25} {weight:.4f}"
    )

Device: cuda

E3-B MODEL CONFIGURATION
Model              : MobileNetV3-Large
Number of classes  : 8
Loss               : Weighted CrossEntropyLoss
Optimizer          : AdamW
Learning rate      : 0.0001
Weight decay       : 0.0001
Device             : cuda:0

Class weights:
cardboard                 1.4507
ecal                      0.4847
hdpe                      0.4369
metal                     1.8423
mixed_plastic_rigid       0.6737
mixed_plastic_soft        0.5944
pet                       0.5175
pet_oil                   1.9998


In [ ]:
######## Training model execution #######
# ============================================================
# E3-B — 20 EPOCH TRAINING
# CLASS-WEIGHTED CROSS ENTROPY
# ============================================================

import time
import copy
import pandas as pd

from sklearn.metrics import (
    accuracy_score,
    f1_score
)


# ============================================================
# CONFIGURATION
# ============================================================

NUM_EPOCHS = 20

best_macro_f1 = -1.0
best_epoch = 0

best_model_state = None

training_history = []


# ============================================================
# TRAINING LOOP
# ============================================================

for epoch in range(1, NUM_EPOCHS + 1):

    epoch_start = time.time()

    # --------------------------------------------------------
    # TRAINING
    # --------------------------------------------------------

    model.train()

    running_loss = 0.0

    all_train_predictions = []
    all_train_labels = []

    print("\n" + "=" * 70)
    print(f"EPOCH {epoch}/{NUM_EPOCHS}")
    print("=" * 70)

    for batch_idx, (images, labels) in enumerate(
        train_loader,
        start=1
    ):

        images = images.to(device)
        labels = labels.to(device)

        # Clear gradients
        optimizer.zero_grad()

        # Forward pass
        outputs = model(images)

        # Weighted loss
        loss = criterion(
            outputs,
            labels
        )

        # Backpropagation
        loss.backward()

        # Update weights
        optimizer.step()

        # ----------------------------------------------------
        # TRACK LOSS
        # ----------------------------------------------------

        running_loss += (
            loss.item() * images.size(0)
        )

        # ----------------------------------------------------
        # TRACK PREDICTIONS
        # ----------------------------------------------------

        predictions = torch.argmax(
            outputs,
            dim=1
        )

        all_train_predictions.extend(
            predictions.detach().cpu().numpy()
        )

        all_train_labels.extend(
            labels.detach().cpu().numpy()
        )

        # ----------------------------------------------------
        # PROGRESS
        # ----------------------------------------------------

        if batch_idx % 100 == 0:

            print(
                f"Batch [{batch_idx}/{len(train_loader)}] "
                f"Loss: {loss.item():.4f}"
            )


    # ========================================================
    # TRAINING METRICS
    # ========================================================

    train_loss = (
        running_loss /
        len(train_loader.dataset)
    )

    train_accuracy = accuracy_score(
        all_train_labels,
        all_train_predictions
    )


    # ========================================================
    # VALIDATION
    # ========================================================

    model.eval()

    running_val_loss = 0.0

    all_val_predictions = []
    all_val_labels = []


    with torch.no_grad():

        for images, labels in val_loader:

            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)

            loss = criterion(
                outputs,
                labels
            )

            running_val_loss += (
                loss.item() * images.size(0)
            )

            predictions = torch.argmax(
                outputs,
                dim=1
            )

            all_val_predictions.extend(
                predictions.cpu().numpy()
            )

            all_val_labels.extend(
                labels.cpu().numpy()
            )


    # ========================================================
    # VALIDATION METRICS
    # ========================================================

    val_loss = (
        running_val_loss /
        len(val_loader.dataset)
    )

    val_accuracy = accuracy_score(
        all_val_labels,
        all_val_predictions
    )

    val_macro_f1 = f1_score(
        all_val_labels,
        all_val_predictions,
        average="macro"
    )

    val_weighted_f1 = f1_score(
        all_val_labels,
        all_val_predictions,
        average="weighted"
    )


    # ========================================================
    # EPOCH TIME
    # ========================================================

    epoch_time = (
        time.time() -
        epoch_start
    ) / 60


    # ========================================================
    # SAVE BEST MODEL
    # ========================================================

    is_best = False

    if val_macro_f1 > best_macro_f1:

        best_macro_f1 = val_macro_f1

        best_epoch = epoch

        best_model_state = copy.deepcopy(
            model.state_dict()
        )

        is_best = True


    # ========================================================
    # SAVE HISTORY
    # ========================================================

    training_history.append({

        "epoch": epoch,

        "train_loss": train_loss,

        "train_accuracy": train_accuracy,

        "val_loss": val_loss,

        "val_accuracy": val_accuracy,

        "val_macro_f1": val_macro_f1,

        "val_weighted_f1": val_weighted_f1,

        "epoch_time_minutes": epoch_time,

        "best_model": is_best
    })


    # ========================================================
    # PRINT EPOCH RESULTS
    # ========================================================

    print("\n" + "-" * 70)

    print(
        f"Epoch {epoch} Results"
    )

    print(
        f"Training Loss     : "
        f"{train_loss:.4f}"
    )

    print(
        f"Training Accuracy : "
        f"{train_accuracy * 100:.2f}%"
    )

    print(
        f"Validation Loss   : "
        f"{val_loss:.4f}"
    )

    print(
        f"Validation Accuracy: "
        f"{val_accuracy * 100:.2f}%"
    )

    print(
        f"Validation Macro F1: "
        f"{val_macro_f1:.4f}"
    )

    print(
        f"Validation Weighted F1: "
        f"{val_weighted_f1:.4f}"
    )

    print(
        f"Epoch Time       : "
        f"{epoch_time:.2f} minutes"
    )

    if is_best:

        print(
            "★ NEW BEST MODEL"
        )

    print("-" * 70)


# ============================================================
# RESTORE BEST MODEL
# ============================================================

model.load_state_dict(
    best_model_state
)


# ============================================================
# CREATE HISTORY DATAFRAME
# ============================================================

history_df = pd.DataFrame(
    training_history
)


# ============================================================
# FINAL SUMMARY
# ============================================================

print("\n" + "=" * 70)
print("E3-B TRAINING COMPLETE")
print("=" * 70)

print(
    f"Best Epoch       : {best_epoch}"
)

print(
    f"Best Macro F1    : "
    f"{best_macro_f1:.4f}"
)

print(
    f"Total Epochs     : {NUM_EPOCHS}"
)

print("\nTraining history:")
print(
    history_df.to_string(
        index=False
    )
)


EPOCH 1/20
Batch [100/1933] Loss: 1.3359
Batch [200/1933] Loss: 0.9518
Batch [300/1933] Loss: 1.0667
Batch [400/1933] Loss: 0.8462
Batch [500/1933] Loss: 0.4056
Batch [600/1933] Loss: 0.8912
Batch [700/1933] Loss: 0.4788
Batch [800/1933] Loss: 0.5715
Batch [900/1933] Loss: 0.5615
Batch [1000/1933] Loss: 0.5475
Batch [1100/1933] Loss: 0.5013
Batch [1200/1933] Loss: 0.4965
Batch [1300/1933] Loss: 0.5531
Batch [1400/1933] Loss: 0.4156
Batch [1500/1933] Loss: 0.4908
Batch [1600/1933] Loss: 0.3874
Batch [1700/1933] Loss: 0.3382
Batch [1800/1933] Loss: 0.6992
Batch [1900/1933] Loss: 0.3330

----------------------------------------------------------------------
Epoch 1 Results
Training Loss     : 0.6717
Training Accuracy : 77.31%
Validation Loss   : 0.7076
Validation Accuracy: 78.82%
Validation Macro F1: 0.6895
Validation Weighted F1: 0.7852
Epoch Time       : 10.23 minutes
★ NEW BEST MODEL
----------------------------------------------------------------------

EPOCH 2/20
Batch [100/1933] Lo

In [ ]:
### per class numbers
# ============================================================
# E3-B — DETAILED PER-CLASS VALIDATION
# BEST CHECKPOINT
# ============================================================

import torch
import numpy as np

from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score,
    f1_score
)


# ============================================================
# RESTORE BEST E3-B MODEL
# ============================================================

model.load_state_dict(best_model_state)

model.eval()


# ============================================================
# VALIDATION PREDICTIONS
# ============================================================

all_val_predictions = []
all_val_labels = []


with torch.no_grad():

    for images, labels in val_loader:

        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)

        predictions = torch.argmax(
            outputs,
            dim=1
        )

        all_val_predictions.extend(
            predictions.cpu().numpy()
        )

        all_val_labels.extend(
            labels.cpu().numpy()
        )


# Convert to NumPy arrays

all_val_predictions = np.array(
    all_val_predictions
)

all_val_labels = np.array(
    all_val_labels
)


# ============================================================
# CLASS NAMES
# ============================================================

class_names = val_dataset.classes


# ============================================================
# OVERALL METRICS
# ============================================================

accuracy = accuracy_score(
    all_val_labels,
    all_val_predictions
)

macro_f1 = f1_score(
    all_val_labels,
    all_val_predictions,
    average="macro"
)

weighted_f1 = f1_score(
    all_val_labels,
    all_val_predictions,
    average="weighted"
)


# ============================================================
# CLASSIFICATION REPORT
# ============================================================

print("\n")
print("=" * 80)
print("E3-B — PER-CLASS VALIDATION RESULTS")
print("BEST CHECKPOINT")
print("=" * 80)

print(
    f"\nAccuracy       : {accuracy:.4f}"
)

print(
    f"Accuracy (%)   : {accuracy * 100:.2f}%"
)

print(
    f"Macro F1       : {macro_f1:.4f}"
)

print(
    f"Weighted F1    : {weighted_f1:.4f}"
)

print(
    f"Samples        : {len(all_val_labels)}"
)


print("\n")
print("=" * 80)
print("CLASSIFICATION REPORT")
print("=" * 80)

print(
    classification_report(
        all_val_labels,
        all_val_predictions,
        target_names=class_names,
        digits=4
    )
)


# ============================================================
# CONFUSION MATRIX
# ============================================================

cm = confusion_matrix(
    all_val_labels,
    all_val_predictions
)


print("\n")
print("=" * 80)
print("CONFUSION MATRIX")
print("=" * 80)

print(
    "Rows    = Actual class"
)

print(
    "Columns = Predicted class\n"
)

print(
    "Class order:"
)

for i, class_name in enumerate(class_names):

    print(
        f"{i} -> {class_name}"
    )


print("\n")

print(cm)



E3-B — PER-CLASS VALIDATION RESULTS
BEST CHECKPOINT

Accuracy       : 0.7978
Accuracy (%)   : 79.78%
Macro F1       : 0.7024
Weighted F1    : 0.7942
Samples        : 13065


CLASSIFICATION REPORT
                     precision    recall  f1-score   support

          cardboard     0.6069    0.3741    0.4629       425
               ecal     0.8079    0.8997    0.8513      2552
               hdpe     0.8880    0.8479    0.8675      4972
              metal     0.6386    0.5740    0.6046       277
mixed_plastic_rigid     0.6511    0.5464    0.5942      1120
 mixed_plastic_soft     0.6280    0.6681    0.6474      1443
                pet     0.8175    0.8966    0.8552      2108
            pet_oil     0.7175    0.7560    0.7362       168

           accuracy                         0.7978     13065
          macro avg     0.7194    0.6953    0.7024     13065
       weighted avg     0.7953    0.7978    0.7942     13065



CONFUSION MATRIX
Rows    = Actual class
Columns = Predicted class

## E3P: Classification Model E3P : Plastic-focused MobileNetV3-Large with realistic augmentation

In [ ]:
## creating 3P data - combining cardboard + metal = non-plastic
from pathlib import Path
import os


# ============================================================
# PATHS
# ============================================================

SOURCE_ROOT = Path(
    r"C:\Users\varda\Documents\_My Computer\COMMON_space\Learning\Upgrad_EPGP_MS\upgrad_IIITB_MS_course\MS_LJMU_Material\Topic Data\SortWaste\dataset\dataset\cropped_data"
)

OUTPUT_ROOT = Path(
    r"C:\Users\varda\Documents\_My Computer\COMMON_space\Learning\Upgrad_EPGP_MS\upgrad_IIITB_MS_course\MS_LJMU_Material\Topic Data\SortWaste\dataset\dataset\cropped_data_plastic"
)


# ============================================================
# E3-P CLASS MAPPING
# ============================================================

CLASS_MAPPING = {
    "pet": "pet",
    "hdpe": "hdpe",
    "ecal": "ecal",
    "pet_oil": "pet_oil",
    "mixed_plastic_soft": "mixed_plastic_soft",
    "mixed_plastic_rigid": "mixed_plastic_rigid",

    # Cardboard + Metal → Non-plastic
    "cardboard": "non_plastic",
    "metal": "non_plastic"
}


# ============================================================
# CREATE DIRECTORIES
# ============================================================

NEW_CLASSES = sorted(set(CLASS_MAPPING.values()))

for split in ["train", "val"]:

    for class_name in NEW_CLASSES:

        directory = OUTPUT_ROOT / split / class_name

        directory.mkdir(
            parents=True,
            exist_ok=True
        )


# ============================================================
# CREATE HARD LINKS
# ============================================================

for split in ["train", "val"]:

    print("\n" + "=" * 70)
    print(f"CREATING E3-P {split.upper()} DATA")
    print("=" * 70)

    counts = {
        class_name: 0
        for class_name in NEW_CLASSES
    }

    for original_class, new_class in CLASS_MAPPING.items():

        source_dir = (
            SOURCE_ROOT /
            split /
            original_class
        )

        destination_dir = (
            OUTPUT_ROOT /
            split /
            new_class
        )

        if not source_dir.exists():

            print(
                f"WARNING: Source directory not found:\n"
                f"{source_dir}"
            )

            continue


        files = [
            f for f in source_dir.iterdir()
            if f.is_file()
        ]


        print(
            f"{original_class:<25} → "
            f"{new_class:<15} : "
            f"{len(files):>6}"
        )


        for source_file in files:

            # Prefix original class to prevent
            # filename collisions.
            destination_file = (
                destination_dir /
                f"{original_class}_{source_file.name}"
            )

            # Avoid recreating existing links
            if destination_file.exists():
                continue

            try:

                os.link(
                    source_file,
                    destination_file
                )

                counts[new_class] += 1

            except OSError as e:

                print(
                    f"\nERROR creating hard link:"
                )

                print(
                    f"Source: {source_file}"
                )

                print(
                    f"Destination: {destination_file}"
                )

                print(e)

                raise


    # ========================================================
    # SUMMARY
    # ========================================================

    print("\n" + "-" * 70)
    print(f"E3-P {split.upper()} CLASS COUNTS")
    print("-" * 70)

    total = 0

    for class_name in NEW_CLASSES:

        count = counts[class_name]

        print(
            f"{class_name:<25}: {count:>6}"
        )

        total += count

    print("-" * 70)

    print(
        f"{'TOTAL':<25}: {total:>6}"
    )


# ============================================================
# FINAL MESSAGE
# ============================================================

print("\n" + "=" * 70)
print("E3-P DATASET CREATION COMPLETE")
print("=" * 70)

print(f"\nLocation:")
print(OUTPUT_ROOT)

print("\nOriginal cropped_data was NOT modified.")

print("\nE3-P classes:")

for i, class_name in enumerate(NEW_CLASSES):

    print(
        f"{i}: {class_name}"
    )


CREATING E3-P TRAIN DATA
pet                       → pet             :  11976
hdpe                      → hdpe            :  16803
ecal                      → ecal            :  13649
pet_oil                   → pet_oil         :    802
mixed_plastic_soft        → mixed_plastic_soft :   9077
mixed_plastic_rigid       → mixed_plastic_rigid :   7066
cardboard                 → non_plastic     :   1524
metal                     → non_plastic     :    945

----------------------------------------------------------------------
E3-P TRAIN CLASS COUNTS
----------------------------------------------------------------------
ecal                     :  13649
hdpe                     :  16803
mixed_plastic_rigid      :   7066
mixed_plastic_soft       :   9077
non_plastic              :   2469
pet                      :  11976
pet_oil                  :    802
----------------------------------------------------------------------
TOTAL                    :  61842

CREATING E3-P VAL DATA
pet      

In [ ]:
## augmentation
from pathlib import Path
import torch

from torchvision import datasets
from torchvision.transforms import v2


# ============================================================
# E3-P DATASET PATHS
# ============================================================

DATA_ROOT = Path(
    r"C:\Users\varda\Documents\_My Computer\COMMON_space\Learning\Upgrad_EPGP_MS\upgrad_IIITB_MS_course\MS_LJMU_Material\Topic Data\SortWaste\dataset\dataset\cropped_data_plastic"
)

TRAIN_DIR = DATA_ROOT / "train"
VAL_DIR = DATA_ROOT / "val"


# ============================================================
# IMAGE CONFIGURATION
# ============================================================

IMAGE_SIZE = 224


# ============================================================
# IMAGENET NORMALIZATION
# ============================================================

IMAGENET_MEAN = [
    0.485,
    0.456,
    0.406
]

IMAGENET_STD = [
    0.229,
    0.224,
    0.225
]


# ============================================================
# E3-P TRAINING TRANSFORMS
# ============================================================
# Realistic augmentation for waste-object classification.
#
# IMPORTANT:
# These transformations are applied ONLY to training images.
# ============================================================

train_transform = v2.Compose([

    # Convert PIL image to tensor-compatible format
    v2.ToImage(),

    # Mild geometric variation
    v2.RandomResizedCrop(
        size=(IMAGE_SIZE, IMAGE_SIZE),
        scale=(0.85, 1.0),
        ratio=(0.90, 1.10)
    ),

    # Waste objects can appear in different orientations
    v2.RandomHorizontalFlip(
        p=0.5
    ),

    # Small rotation only
    v2.RandomRotation(
        degrees=10
    ),

    # Realistic lighting/camera variation
    v2.ColorJitter(
        brightness=0.20,
        contrast=0.20,
        saturation=0.15,
        hue=0.03
    ),

    # Mild blur occasionally
    v2.RandomApply(
        [
            v2.GaussianBlur(
                kernel_size=3,
                sigma=(0.1, 1.0)
            )
        ],
        p=0.10
    ),

    # Convert to float [0,1]
    v2.ToDtype(
        dtype=torch.float32,
        scale=True
    ),

    # ImageNet normalization
    v2.Normalize(
        mean=IMAGENET_MEAN,
        std=IMAGENET_STD
    )
])


# ============================================================
# VALIDATION TRANSFORMS
# ============================================================
# NO augmentation here.
#
# Validation must represent the natural/unmodified
# validation distribution.
# ============================================================

val_transform = v2.Compose([

    v2.ToImage(),

    v2.Resize(
        size=(IMAGE_SIZE, IMAGE_SIZE)
    ),

    v2.ToDtype(
        dtype=torch.float32,
        scale=True
    ),

    v2.Normalize(
        mean=IMAGENET_MEAN,
        std=IMAGENET_STD
    )
])


# ============================================================
# LOAD TRAINING DATASET
# ============================================================

train_dataset = datasets.ImageFolder(
    root=TRAIN_DIR,
    transform=train_transform
)


# ============================================================
# LOAD VALIDATION DATASET
# ============================================================

val_dataset = datasets.ImageFolder(
    root=VAL_DIR,
    transform=val_transform
)


# ============================================================
# DATASET VERIFICATION
# ============================================================

print("=" * 70)
print("E3-P DATASET VERIFICATION")
print("=" * 70)

print("\nTraining samples :", len(train_dataset))
print("Validation samples:", len(val_dataset))


print("\n" + "=" * 70)
print("CLASS NAMES")
print("=" * 70)

print(train_dataset.classes)


print("\n" + "=" * 70)
print("CLASS → INDEX MAPPING")
print("=" * 70)

for class_name, class_index in train_dataset.class_to_idx.items():

    print(
        f"{class_index} → {class_name}"
    )


# ============================================================
# VERIFY TRAIN / VAL CLASS MAPPINGS ARE IDENTICAL
# ============================================================

print("\n" + "=" * 70)
print("MAPPING CONSISTENCY CHECK")
print("=" * 70)

if train_dataset.class_to_idx == val_dataset.class_to_idx:

    print("✓ Train and validation mappings are identical.")

else:

    print("✗ WARNING: Train and validation mappings differ!")

    print("\nTrain:")
    print(train_dataset.class_to_idx)

    print("\nValidation:")
    print(val_dataset.class_to_idx)


# ============================================================
# COUNT SAMPLES PER CLASS
# ============================================================

from collections import Counter


train_counts = Counter(
    train_dataset.targets
)

val_counts = Counter(
    val_dataset.targets
)


print("\n" + "=" * 70)
print("TRAINING CLASS COUNTS")
print("=" * 70)

for class_index, class_name in enumerate(train_dataset.classes):

    print(
        f"{class_name:<25}: "
        f"{train_counts[class_index]:>6}"
    )


print("\n" + "=" * 70)
print("VALIDATION CLASS COUNTS")
print("=" * 70)

for class_index, class_name in enumerate(val_dataset.classes):

    print(
        f"{class_name:<25}: "
        f"{val_counts[class_index]:>6}"
    )


# ============================================================
# TRANSFORM SUMMARY
# ============================================================

print("\n" + "=" * 70)
print("TRANSFORM CONFIGURATION")
print("=" * 70)

print("\nTRAIN:")
print(train_transform)

print("\nVALIDATION:")
print(val_transform)

E3-P DATASET VERIFICATION

Training samples : 61842
Validation samples: 13065

CLASS NAMES
['ecal', 'hdpe', 'mixed_plastic_rigid', 'mixed_plastic_soft', 'non_plastic', 'pet', 'pet_oil']

CLASS → INDEX MAPPING
0 → ecal
1 → hdpe
2 → mixed_plastic_rigid
3 → mixed_plastic_soft
4 → non_plastic
5 → pet
6 → pet_oil

MAPPING CONSISTENCY CHECK
✓ Train and validation mappings are identical.

TRAINING CLASS COUNTS
ecal                     :  13649
hdpe                     :  16803
mixed_plastic_rigid      :   7066
mixed_plastic_soft       :   9077
non_plastic              :   2469
pet                      :  11976
pet_oil                  :    802

VALIDATION CLASS COUNTS
ecal                     :   2552
hdpe                     :   4972
mixed_plastic_rigid      :   1120
mixed_plastic_soft       :   1443
non_plastic              :    702
pet                      :   2108
pet_oil                  :    168

TRANSFORM CONFIGURATION

TRAIN:
Compose(
      ToImage()
      RandomResizedCrop(size=(224,

In [ ]:
### model set-up
from torch.utils.data import DataLoader


# ============================================================
# E3-P DATALOADER CONFIGURATION
# ============================================================

BATCH_SIZE = 32

NUM_WORKERS = 0
PIN_MEMORY = True


# ============================================================
# CREATE TRAINING DATALOADER
# ============================================================

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY
)


# ============================================================
# CREATE VALIDATION DATALOADER
# ============================================================

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY
)


# ============================================================
# DATALOADER INFORMATION
# ============================================================

print("=" * 70)
print("E3-P DATALOADER CONFIGURATION")
print("=" * 70)

print(f"\nBatch size       : {BATCH_SIZE}")
print(f"Training samples : {len(train_dataset)}")
print(f"Validation samples: {len(val_dataset)}")

print(
    f"\nTraining batches : {len(train_loader)}"
)

print(
    f"Validation batches: {len(val_loader)}"
)

print(
    f"Num workers      : {NUM_WORKERS}"
)

print(
    f"Pin memory       : {PIN_MEMORY}"
)


# ============================================================
# GET ONE TRAINING BATCH
# ============================================================

train_images, train_labels = next(
    iter(train_loader)
)


# ============================================================
# GET ONE VALIDATION BATCH
# ============================================================

val_images, val_labels = next(
    iter(val_loader)
)


# ============================================================
# TRAINING BATCH CHECK
# ============================================================

print("\n" + "=" * 70)
print("TRAINING BATCH CHECK")
print("=" * 70)

print(
    "Image tensor shape :",
    train_images.shape
)

print(
    "Label tensor shape :",
    train_labels.shape
)

print(
    "Image dtype        :",
    train_images.dtype
)

print(
    "Label dtype        :",
    train_labels.dtype
)

print(
    "Image min          :",
    train_images.min().item()
)

print(
    "Image max          :",
    train_images.max().item()
)

print(
    "First 10 labels    :",
    train_labels[:10].tolist()
)


# ============================================================
# VALIDATION BATCH CHECK
# ============================================================

print("\n" + "=" * 70)
print("VALIDATION BATCH CHECK")
print("=" * 70)

print(
    "Image tensor shape :",
    val_images.shape
)

print(
    "Label tensor shape :",
    val_labels.shape
)

print(
    "Image dtype        :",
    val_images.dtype
)

print(
    "Label dtype        :",
    val_labels.dtype
)

print(
    "Image min          :",
    val_images.min().item()
)

print(
    "Image max          :",
    val_images.max().item()
)

print(
    "First 10 labels    :",
    val_labels[:10].tolist()
)

E3-P DATALOADER CONFIGURATION

Batch size       : 32
Training samples : 61842
Validation samples: 13065

Training batches : 1933
Validation batches: 409
Num workers      : 0
Pin memory       : True

TRAINING BATCH CHECK
Image tensor shape : torch.Size([32, 3, 224, 224])
Label tensor shape : torch.Size([32])
Image dtype        : torch.float32
Label dtype        : torch.int64
Image min          : -2.1179039478302
Image max          : 2.640000104904175
First 10 labels    : [0, 5, 3, 3, 2, 3, 3, 5, 0, 2]

VALIDATION BATCH CHECK
Image tensor shape : torch.Size([32, 3, 224, 224])
Label tensor shape : torch.Size([32])
Image dtype        : torch.float32
Label dtype        : torch.int64
Image min          : -2.1179039478302
Image max          : 2.6225709915161133
First 10 labels    : [0, 0, 0, 0, 0, 0, 0, 0, 0, 0]


In [ ]:
## initialize model
import torch
import torch.nn as nn

from torchvision.models import (
    mobilenet_v3_large,
    MobileNet_V3_Large_Weights
)


# ============================================================
# DEVICE
# ============================================================

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device:", device)


# ============================================================
# NUMBER OF CLASSES
# ============================================================

NUM_CLASSES = 7


# ============================================================
# LOAD PRETRAINED MOBILENETV3-LARGE
# ============================================================

weights = MobileNet_V3_Large_Weights.DEFAULT

model = mobilenet_v3_large(
    weights=weights
)


# ============================================================
# REPLACE CLASSIFIER
# ============================================================

# Original classifier:
#
# Linear(960 → 1280)
# Hardswish
# Dropout
# Linear(1280 → 1000)
#
# We replace only the final layer.

model.classifier[3] = nn.Linear(
    in_features=1280,
    out_features=NUM_CLASSES
)


# ============================================================
# MOVE MODEL TO GPU
# ============================================================

model = model.to(device)


# ============================================================
# MODEL INFORMATION
# ============================================================

print("\n" + "=" * 70)
print("E3-P MOBILENETV3-LARGE")
print("=" * 70)

print(
    "\nModel:",
    model.__class__.__name__
)

print(
    "\nClassifier:"
)

print(
    model.classifier
)

print(
    "\nNumber of output classes:",
    NUM_CLASSES
)

print(
    "Model device:",
    next(model.parameters()).device
)


# ============================================================
# PARAMETER COUNT
# ============================================================

total_parameters = sum(
    p.numel()
    for p in model.parameters()
)

trainable_parameters = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)


print(
    "\nTotal parameters:",
    f"{total_parameters:,}"
)

print(
    "Trainable parameters:",
    f"{trainable_parameters:,}"
)


# ============================================================
# FINAL OUTPUT SHAPE TEST
# ============================================================

model.eval()

with torch.no_grad():

    test_images = torch.randn(
        2,
        3,
        224,
        224
    ).to(device)

    test_outputs = model(
        test_images
    )


print(
    "\nTest input shape:",
    test_images.shape
)

print(
    "Test output shape:",
    test_outputs.shape
)

print(
    "\nExpected output shape:",
    "(2, 7)"
)


# ============================================================
# VERIFY
# ============================================================

assert test_outputs.shape == (
    2,
    NUM_CLASSES
)

print(
    "\n✓ Model verification successful."
)

Device: cuda

E3-P MOBILENETV3-LARGE

Model: MobileNetV3

Classifier:
Sequential(
  (0): Linear(in_features=960, out_features=1280, bias=True)
  (1): Hardswish()
  (2): Dropout(p=0.2, inplace=True)
  (3): Linear(in_features=1280, out_features=7, bias=True)
)

Number of output classes: 7
Model device: cuda:0

Total parameters: 4,210,999
Trainable parameters: 4,210,999

Test input shape: torch.Size([2, 3, 224, 224])
Test output shape: torch.Size([2, 7])

Expected output shape: (2, 7)

✓ Model verification successful.


In [ ]:
## training  config
import torch
import torch.nn as nn


# ============================================================
# E3-P TRAINING CONFIGURATION
# ============================================================

LEARNING_RATE = 0.0001
WEIGHT_DECAY = 0.0001


# ============================================================
# LOSS FUNCTION
# ============================================================

criterion = nn.CrossEntropyLoss()


# ============================================================
# OPTIMIZER
# ============================================================

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY
)


# ============================================================
# VERIFY CONFIGURATION
# ============================================================

print("=" * 70)
print("E3-P TRAINING CONFIGURATION")
print("=" * 70)

print(
    "\nModel              : MobileNetV3-Large"
)

print(
    "Number of classes  : 7"
)

print(
    "Loss               : CrossEntropyLoss"
)

print(
    "Optimizer          : AdamW"
)

print(
    f"Learning rate      : {LEARNING_RATE}"
)

print(
    f"Weight decay       : {WEIGHT_DECAY}"
)

print(
    "Device             :",
    device
)


print("\nLoss function:")
print(criterion)


print("\nOptimizer:")
print(optimizer)

E3-P TRAINING CONFIGURATION

Model              : MobileNetV3-Large
Number of classes  : 7
Loss               : CrossEntropyLoss
Optimizer          : AdamW
Learning rate      : 0.0001
Weight decay       : 0.0001
Device             : cuda

Loss function:
CrossEntropyLoss()

Optimizer:
AdamW (
Parameter Group 0
    amsgrad: False
    betas: (0.9, 0.999)
    capturable: False
    decoupled_weight_decay: True
    differentiable: False
    eps: 1e-08
    foreach: None
    fused: None
    lr: 0.0001
    maximize: False
    weight_decay: 0.0001
)


In [ ]:
########### Training - running model training for 20 epochs ###############
import time
import copy
import pandas as pd

from sklearn.metrics import (
    accuracy_score,
    f1_score
)


# ============================================================
# TRAINING CONFIGURATION
# ============================================================

NUM_EPOCHS = 20

BEST_MODEL_PATH = (
    DATA_ROOT /
    "mobilenet_v3_large_E3P_best.pth"
)


# ============================================================
# HISTORY
# ============================================================

history = []

best_macro_f1 = -1.0
best_epoch = None


# ============================================================
# TRAINING LOOP
# ============================================================

for epoch in range(1, NUM_EPOCHS + 1):

    epoch_start = time.time()

    # ========================================================
    # TRAINING
    # ========================================================

    model.train()

    running_loss = 0.0
    train_correct = 0
    train_total = 0

    for batch_idx, (images, labels) in enumerate(
        train_loader,
        start=1
    ):

        # Move data to GPU
        images = images.to(
            device,
            non_blocking=True
        )

        labels = labels.to(
            device,
            non_blocking=True
        )

        # ----------------------------------------------------
        # Forward pass
        # ----------------------------------------------------

        optimizer.zero_grad(
            set_to_none=True
        )

        outputs = model(images)

        loss = criterion(
            outputs,
            labels
        )

        # ----------------------------------------------------
        # Backpropagation
        # ----------------------------------------------------

        loss.backward()

        optimizer.step()

        # ----------------------------------------------------
        # Statistics
        # ----------------------------------------------------

        batch_size = labels.size(0)

        running_loss += (
            loss.item() * batch_size
        )

        predictions = outputs.argmax(
            dim=1
        )

        train_correct += (
            predictions == labels
        ).sum().item()

        train_total += batch_size

        # ----------------------------------------------------
        # Progress
        # ----------------------------------------------------

        if batch_idx % 100 == 0:

            print(
                f"Epoch [{epoch}/{NUM_EPOCHS}] "
                f"Batch [{batch_idx}/{len(train_loader)}] "
                f"Loss: {loss.item():.4f}"
            )


    # ========================================================
    # TRAINING METRICS
    # ========================================================

    train_loss = (
        running_loss /
        train_total
    )

    train_accuracy = (
        train_correct /
        train_total
    )


    # ========================================================
    # VALIDATION
    # ========================================================

    model.eval()

    val_running_loss = 0.0

    val_predictions = []
    val_targets = []

    with torch.no_grad():

        for images, labels in val_loader:

            images = images.to(
                device,
                non_blocking=True
            )

            labels = labels.to(
                device,
                non_blocking=True
            )

            outputs = model(images)

            loss = criterion(
                outputs,
                labels
            )

            batch_size = labels.size(0)

            val_running_loss += (
                loss.item() * batch_size
            )

            predictions = outputs.argmax(
                dim=1
            )

            val_predictions.extend(
                predictions.cpu().numpy()
            )

            val_targets.extend(
                labels.cpu().numpy()
            )


    # ========================================================
    # VALIDATION METRICS
    # ========================================================

    val_loss = (
        val_running_loss /
        len(val_dataset)
    )

    val_accuracy = accuracy_score(
        val_targets,
        val_predictions
    )

    val_macro_f1 = f1_score(
        val_targets,
        val_predictions,
        average="macro"
    )

    val_weighted_f1 = f1_score(
        val_targets,
        val_predictions,
        average="weighted"
    )


    # ========================================================
    # EPOCH TIME
    # ========================================================

    epoch_time = (
        time.time() -
        epoch_start
    ) / 60


    # ========================================================
    # CHECK FOR BEST MODEL
    # ========================================================

    is_best = (
        val_macro_f1 >
        best_macro_f1
    )

    if is_best:

        best_macro_f1 = val_macro_f1

        best_epoch = epoch

        torch.save(
            {
                "epoch": epoch,

                "model_state_dict":
                    model.state_dict(),

                "optimizer_state_dict":
                    optimizer.state_dict(),

                "val_loss":
                    val_loss,

                "val_accuracy":
                    val_accuracy,

                "val_macro_f1":
                    val_macro_f1,

                "val_weighted_f1":
                    val_weighted_f1,

                "class_to_idx":
                    train_dataset.class_to_idx,

                "num_classes":
                    NUM_CLASSES,

                "image_size":
                    IMAGE_SIZE
            },
            BEST_MODEL_PATH
        )


    # ========================================================
    # STORE HISTORY
    # ========================================================

    history.append(
        {
            "epoch": epoch,

            "train_loss":
                train_loss,

            "train_accuracy":
                train_accuracy,

            "val_loss":
                val_loss,

            "val_accuracy":
                val_accuracy,

            "val_macro_f1":
                val_macro_f1,

            "val_weighted_f1":
                val_weighted_f1,

            "epoch_time_minutes":
                epoch_time,

            "best_model":
                is_best
        }
    )


    # ========================================================
    # PRINT EPOCH SUMMARY
    # ========================================================

    print("\n" + "=" * 70)

    print(
        f"EPOCH {epoch}/{NUM_EPOCHS} COMPLETE"
    )

    print("=" * 70)

    print(
        f"Training loss      : {train_loss:.4f}"
    )

    print(
        f"Training accuracy  : "
        f"{train_accuracy * 100:.2f}%"
    )

    print(
        f"Validation loss    : {val_loss:.4f}"
    )

    print(
        f"Validation accuracy: "
        f"{val_accuracy * 100:.2f}%"
    )

    print(
        f"Validation Macro F1: "
        f"{val_macro_f1:.4f}"
    )

    print(
        f"Validation Weighted F1: "
        f"{val_weighted_f1:.4f}"
    )

    print(
        f"Epoch time         : "
        f"{epoch_time:.2f} minutes"
    )

    if is_best:

        print(
            "✓ NEW BEST MODEL SAVED"
        )

    print(
        f"Best Macro F1 so far: "
        f"{best_macro_f1:.4f}"
    )


# ============================================================
# SAVE TRAINING HISTORY
# ============================================================

history_df = pd.DataFrame(
    history
)


HISTORY_PATH = (
    DATA_ROOT /
    "mobilenet_v3_large_E3P_training_history.csv"
)


history_df.to_csv(
    HISTORY_PATH,
    index=False
)


# ============================================================
# FINAL SUMMARY
# ============================================================

print("\n" + "=" * 70)
print("E3-P TRAINING COMPLETE")
print("=" * 70)

print(
    f"\nBest epoch       : {best_epoch}"
)

print(
    f"Best Macro F1    : "
    f"{best_macro_f1:.4f}"
)

print(
    f"\nBest checkpoint:"
)

print(
    BEST_MODEL_PATH
)

print(
    f"\nTraining history:"
)

print(
    HISTORY_PATH
)

Epoch [1/20] Batch [100/1933] Loss: 1.1100
Epoch [1/20] Batch [200/1933] Loss: 1.0088
Epoch [1/20] Batch [300/1933] Loss: 0.7554
Epoch [1/20] Batch [400/1933] Loss: 0.7330
Epoch [1/20] Batch [500/1933] Loss: 0.9615
Epoch [1/20] Batch [600/1933] Loss: 0.6060
Epoch [1/20] Batch [700/1933] Loss: 0.5352
Epoch [1/20] Batch [800/1933] Loss: 0.8858
Epoch [1/20] Batch [900/1933] Loss: 0.7254
Epoch [1/20] Batch [1000/1933] Loss: 0.3804
Epoch [1/20] Batch [1100/1933] Loss: 0.3640
Epoch [1/20] Batch [1200/1933] Loss: 0.5854
Epoch [1/20] Batch [1300/1933] Loss: 0.5481
Epoch [1/20] Batch [1400/1933] Loss: 0.9684
Epoch [1/20] Batch [1500/1933] Loss: 0.6194
Epoch [1/20] Batch [1600/1933] Loss: 0.5426
Epoch [1/20] Batch [1700/1933] Loss: 0.4630
Epoch [1/20] Batch [1800/1933] Loss: 0.1907
Epoch [1/20] Batch [1900/1933] Loss: 0.4535

EPOCH 1/20 COMPLETE
Training loss      : 0.6343
Training accuracy  : 77.19%
Validation loss    : 0.7005
Validation accuracy: 78.42%
Validation Macro F1: 0.7022
Validation W

In [ ]:
### Per class results
# ================================================================
# E3-P — DETAILED PER-CLASS VALIDATION EVALUATION
# Best Checkpoint: Epoch 13
# ================================================================

import torch
import numpy as np
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score,
    f1_score
)

# ----------------------------------------------------------------
# 1. CONFIGURATION
# ----------------------------------------------------------------

BEST_CHECKPOINT = r"C:\Users\varda\Documents\_My Computer\COMMON_space\Learning\Upgrad_EPGP_MS\upgrad_IIITB_MS_course\MS_LJMU_Material\Topic Data\SortWaste\dataset\dataset\cropped_data_plastic\mobilenet_v3_large_E3P_best.pth"

class_names = [
    "ecal",
    "hdpe",
    "mixed_plastic_rigid",
    "mixed_plastic_soft",
    "non_plastic",
    "pet",
    "pet_oil"
]

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("=" * 80)
print("E3-P — DETAILED PER-CLASS VALIDATION EVALUATION")
print("=" * 80)

print(f"Checkpoint : {BEST_CHECKPOINT}")
print(f"Device     : {device}")
print(f"Classes    : {len(class_names)}")


# ----------------------------------------------------------------
# 2. LOAD BEST CHECKPOINT
# ----------------------------------------------------------------

checkpoint = torch.load(
    BEST_CHECKPOINT,
    map_location=device
)

# Handle both possible checkpoint formats:
#   A) complete checkpoint dictionary
#   B) direct state_dict

if isinstance(checkpoint, dict) and "model_state_dict" in checkpoint:

    model.load_state_dict(checkpoint["model_state_dict"])

    saved_epoch = checkpoint.get("epoch", "Unknown")
    saved_macro_f1 = checkpoint.get("val_macro_f1", "Unknown")
    saved_accuracy = checkpoint.get("val_accuracy", "Unknown")

else:
    model.load_state_dict(checkpoint)

    saved_epoch = "Unknown"
    saved_macro_f1 = "Unknown"
    saved_accuracy = "Unknown"


model = model.to(device)
model.eval()

print("\nCheckpoint loaded successfully.")

if saved_epoch != "Unknown":
    print(f"Saved epoch       : {saved_epoch}")
if saved_macro_f1 != "Unknown":
    print(f"Saved Macro F1    : {saved_macro_f1}")
if saved_accuracy != "Unknown":
    print(f"Saved accuracy    : {saved_accuracy}")


# ----------------------------------------------------------------
# 3. RUN VALIDATION INFERENCE
# ----------------------------------------------------------------

all_labels = []
all_predictions = []

with torch.no_grad():

    for images, labels in val_loader:

        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)

        predictions = torch.argmax(outputs, dim=1)

        all_labels.extend(labels.cpu().numpy())
        all_predictions.extend(predictions.cpu().numpy())


all_labels = np.array(all_labels)
all_predictions = np.array(all_predictions)


# ----------------------------------------------------------------
# 4. OVERALL METRICS
# ----------------------------------------------------------------

accuracy = accuracy_score(
    all_labels,
    all_predictions
)

macro_f1 = f1_score(
    all_labels,
    all_predictions,
    average="macro"
)

weighted_f1 = f1_score(
    all_labels,
    all_predictions,
    average="weighted"
)

print("\n" + "=" * 80)
print("E3-P — OVERALL VALIDATION RESULTS")
print("=" * 80)

print(f"Accuracy       : {accuracy:.4f}")
print(f"Accuracy (%)   : {accuracy * 100:.2f}%")
print(f"Macro F1       : {macro_f1:.4f}")
print(f"Weighted F1    : {weighted_f1:.4f}")
print(f"Samples        : {len(all_labels)}")


# ----------------------------------------------------------------
# 5. CLASSIFICATION REPORT
# ----------------------------------------------------------------

print("\n" + "=" * 80)
print("CLASSIFICATION REPORT")
print("=" * 80)

report = classification_report(
    all_labels,
    all_predictions,
    labels=np.arange(len(class_names)),
    target_names=class_names,
    digits=4,
    zero_division=0
)

print(report)


# ----------------------------------------------------------------
# 6. CONFUSION MATRIX
# ----------------------------------------------------------------

cm = confusion_matrix(
    all_labels,
    all_predictions,
    labels=np.arange(len(class_names))
)

print("\n" + "=" * 80)
print("CONFUSION MATRIX")
print("=" * 80)

print("Rows    = Actual class")
print("Columns = Predicted class")
print("\nClass order:")

for i, name in enumerate(class_names):
    print(f"{i} -> {name}")

print("\n")
print(cm)


# ----------------------------------------------------------------
# 7. PER-CLASS CORRECT / INCORRECT COUNTS
# ----------------------------------------------------------------

print("\n" + "=" * 80)
print("PER-CLASS PREDICTION COUNTS")
print("=" * 80)

print(
    f"{'Class':25s}"
    f"{'Actual':>10s}"
    f"{'Correct':>10s}"
    f"{'Incorrect':>12s}"
    f"{'Recall':>10s}"
)

for i, name in enumerate(class_names):

    actual = np.sum(all_labels == i)
    correct = cm[i, i]
    incorrect = actual - correct

    recall = correct / actual if actual > 0 else 0

    print(
        f"{name:25s}"
        f"{actual:10d}"
        f"{correct:10d}"
        f"{incorrect:12d}"
        f"{recall:10.4f}"
    )


# ----------------------------------------------------------------
# 8. PLASTIC-ONLY SUMMARY
# ----------------------------------------------------------------
# ECAL is treated as PLASTIC for the purpose of the research
# interpretation. non_plastic is excluded here.

plastic_classes = [
    "ecal",
    "hdpe",
    "mixed_plastic_rigid",
    "mixed_plastic_soft",
    "pet",
    "pet_oil"
]

plastic_indices = [
    class_names.index(x)
    for x in plastic_classes
]

plastic_mask = np.isin(all_labels, plastic_indices)

plastic_labels = all_labels[plastic_mask]
plastic_predictions = all_predictions[plastic_mask]

plastic_accuracy = accuracy_score(
    plastic_labels,
    plastic_predictions
)

plastic_macro_f1 = f1_score(
    plastic_labels,
    plastic_predictions,
    labels=plastic_indices,
    average="macro"
)

plastic_weighted_f1 = f1_score(
    plastic_labels,
    plastic_predictions,
    labels=plastic_indices,
    average="weighted"
)

print("\n" + "=" * 80)
print("PLASTIC-ONLY CLASSIFICATION SUMMARY")
print("=" * 80)

print("Plastic classes:")
for name in plastic_classes:
    print(f"  - {name}")

print(f"\nPlastic samples : {len(plastic_labels)}")
print(f"Plastic accuracy: {plastic_accuracy:.4f}")
print(f"Plastic accuracy: {plastic_accuracy * 100:.2f}%")
print(f"Plastic Macro F1: {plastic_macro_f1:.4f}")
print(f"Plastic Weighted F1: {plastic_weighted_f1:.4f}")


print("\n" + "=" * 80)
print("E3-P EVALUATION COMPLETE")
print("=" * 80)

E3-P — DETAILED PER-CLASS VALIDATION EVALUATION
Checkpoint : C:\Users\varda\Documents\_My Computer\COMMON_space\Learning\Upgrad_EPGP_MS\upgrad_IIITB_MS_course\MS_LJMU_Material\Topic Data\SortWaste\dataset\dataset\cropped_data_plastic\mobilenet_v3_large_E3P_best.pth
Device     : cuda
Classes    : 7

Checkpoint loaded successfully.
Saved epoch       : 13
Saved Macro F1    : 0.7110381270121937
Saved accuracy    : 0.7870646766169154

E3-P — OVERALL VALIDATION RESULTS
Accuracy       : 0.7871
Accuracy (%)   : 78.71%
Macro F1       : 0.7110
Weighted F1    : 0.7811
Samples        : 13065

CLASSIFICATION REPORT
                     precision    recall  f1-score   support

               ecal     0.8722    0.8025    0.8359      2552
               hdpe     0.8037    0.9008    0.8495      4972
mixed_plastic_rigid     0.6593    0.5598    0.6055      1120
 mixed_plastic_soft     0.6595    0.5988    0.6277      1443
        non_plastic     0.6878    0.4487    0.5431       702
                pet    

# E3Y : YOLO + Mobilenet
E3Y - E3-P pipeline = pretrained/best YOLOv11 detector + trained MobileNetV3 classifier

## E3Y-A : E1 YOLO + Train Mobilenet on Yolo detections
Crop retained only when YOLO prediction matched a GT object at IoU ≥ 0.5

In [ ]:
## load best YOLO model - epoch 18 and created cropped detections
from pathlib import Path
import csv
import shutil
import cv2
import torch
from ultralytics import YOLO

# ============================================================
# E3Y — YOLO → MOBILENET CROP GENERATION
# ============================================================

print("=" * 80)
print("E3Y — YOLO → MOBILENET CROP GENERATION")
print("=" * 80)

# ------------------------------------------------------------
# 1. PATHS
# ------------------------------------------------------------

DATASET_ROOT = Path(
    r"C:\Users\varda\Documents\_My Computer\COMMON_space\Learning"
    r"\Upgrad_EPGP_MS\upgrad_IIITB_MS_course\MS_LJMU_Material"
    r"\Topic Data\SortWaste\dataset\dataset"
)

YOLO_CHECKPOINT = Path(
    r"C:\Users\varda\Documents\_My Computer\COMMON_space\Learning"
    r"\Upgrad_EPGP_MS\upgrad_IIITB_MS_course\MS_LJMU_Material"
    r"\Thesis_Code\runs\detect\runs\sortwaste\yolo11n_baseline-2"
    r"\weights\best.pt"
)

OUTPUT_ROOT = DATASET_ROOT / "yolo_mobilenet_crops_E3Y"

# Exact source directories used by the YOLO training
SPLIT_ROOT = DATASET_ROOT / "splited_all_dataset_coco"

TRAIN_IMAGES = SPLIT_ROOT / "train" / "images"
TRAIN_LABELS = SPLIT_ROOT / "train" / "labels"

VAL_IMAGES = SPLIT_ROOT / "val" / "images"
VAL_LABELS = SPLIT_ROOT / "val" / "labels"


# ------------------------------------------------------------
# 2. MOBILENET CLASSES
# ------------------------------------------------------------

MOBILENET_CLASSES = [
    "ecal",
    "hdpe",
    "mixed_plastic_rigid",
    "mixed_plastic_soft",
    "non_plastic",
    "pet",
    "pet_oil",
]

CLASS_TO_ID = {
    name: i
    for i, name in enumerate(MOBILENET_CLASSES)
}

# ------------------------------------------------------------
# 3. YOLO → MOBILENET MAPPING
# ------------------------------------------------------------
#
# YOLO YAML:
#
# 0 = pet
# 1 = hdpe
# 2 = mixed_plastic_soft
# 3 = ecal
# 4 = metal
# 5 = cardboard
# 6 = mixed_plastic_rigid
# 7 = pet_oil
#
# Metal + cardboard are merged into NON-PLASTIC.
# ------------------------------------------------------------

YOLO_TO_MOBILENET = {
    0: "pet",
    1: "hdpe",
    2: "mixed_plastic_soft",
    3: "ecal",
    4: "non_plastic",          # metal
    5: "non_plastic",          # cardboard
    6: "mixed_plastic_rigid",
    7: "pet_oil",
}


# ------------------------------------------------------------
# 4. CHECK PATHS
# ------------------------------------------------------------

print("\nChecking dataset paths...")

paths_to_check = {
    "Dataset root": DATASET_ROOT,
    "Split root": SPLIT_ROOT,
    "Train images": TRAIN_IMAGES,
    "Train labels": TRAIN_LABELS,
    "Val images": VAL_IMAGES,
    "Val labels": VAL_LABELS,
    "YOLO checkpoint": YOLO_CHECKPOINT,
}

for name, path in paths_to_check.items():
    print(f"{name:20s}: {path}")
    print(f"{'':20s}  Exists: {path.exists()}")

if not YOLO_CHECKPOINT.exists():
    raise FileNotFoundError(
        f"\nYOLO checkpoint not found:\n{YOLO_CHECKPOINT}"
    )

if not TRAIN_IMAGES.exists():
    raise FileNotFoundError(
        f"\nTRAIN images directory not found:\n{TRAIN_IMAGES}"
    )

if not TRAIN_LABELS.exists():
    raise FileNotFoundError(
        f"\nTRAIN labels directory not found:\n{TRAIN_LABELS}"
    )

if not VAL_IMAGES.exists():
    raise FileNotFoundError(
        f"\nVAL images directory not found:\n{VAL_IMAGES}"
    )

if not VAL_LABELS.exists():
    raise FileNotFoundError(
        f"\nVAL labels directory not found:\n{VAL_LABELS}"
    )


# ------------------------------------------------------------
# 5. DISPLAY CLASS MAPPING
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("YOLO → MOBILENET CLASS MAPPING")
print("=" * 80)

original_names = {
    0: "pet",
    1: "hdpe",
    2: "mixed_plastic_soft",
    3: "ecal",
    4: "metal",
    5: "cardboard",
    6: "mixed_plastic_rigid",
    7: "pet_oil",
}

for yolo_id in range(8):
    print(
        f"{yolo_id} -> "
        f"{original_names[yolo_id]:25s} -> "
        f"{YOLO_TO_MOBILENET[yolo_id]}"
    )


# ------------------------------------------------------------
# 6. LOAD YOLO
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("Loading YOLO11n Epoch-18 checkpoint...")
print("=" * 80)

device = 0 if torch.cuda.is_available() else "cpu"

model = YOLO(str(YOLO_CHECKPOINT))

print("Model loaded successfully.")
print(f"Checkpoint: {YOLO_CHECKPOINT}")
print(f"Device: {device}")


# ------------------------------------------------------------
# 7. RESET OUTPUT DIRECTORY
# ------------------------------------------------------------

if OUTPUT_ROOT.exists():
    print("\nRemoving previous E3Y output...")
    shutil.rmtree(OUTPUT_ROOT)

OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

print(f"Output directory:\n{OUTPUT_ROOT}")


# ------------------------------------------------------------
# 8. IMAGE EXTENSIONS
# ------------------------------------------------------------

IMAGE_EXTENSIONS = {
    ".jpg",
    ".jpeg",
    ".png",
    ".bmp",
    ".webp",
}


# ------------------------------------------------------------
# 9. READ YOLO GROUND-TRUTH LABEL
# ------------------------------------------------------------

def read_yolo_labels(label_path):
    """
    Reads YOLO-format labels.

    Each line:
    class_id x_center y_center width height

    Coordinates are normalized [0,1].
    """

    labels = []

    if not label_path.exists():
        return labels

    with open(label_path, "r") as f:

        for line in f:

            parts = line.strip().split()

            if len(parts) < 5:
                continue

            class_id = int(parts[0])

            x_center = float(parts[1])
            y_center = float(parts[2])
            width = float(parts[3])
            height = float(parts[4])

            labels.append(
                {
                    "class_id": class_id,
                    "x_center": x_center,
                    "y_center": y_center,
                    "width": width,
                    "height": height,
                }
            )

    return labels


# ------------------------------------------------------------
# 10. IoU FUNCTION
# ------------------------------------------------------------

def calculate_iou(box1, box2):

    x1 = max(box1[0], box2[0])
    y1 = max(box1[1], box2[1])

    x2 = min(box1[2], box2[2])
    y2 = min(box1[3], box2[3])

    intersection_width = max(0, x2 - x1)
    intersection_height = max(0, y2 - y1)

    intersection = (
        intersection_width *
        intersection_height
    )

    area1 = (
        max(0, box1[2] - box1[0]) *
        max(0, box1[3] - box1[1])
    )

    area2 = (
        max(0, box2[2] - box2[0]) *
        max(0, box2[3] - box2[1])
    )

    union = area1 + area2 - intersection

    if union <= 0:
        return 0.0

    return intersection / union


# ------------------------------------------------------------
# 11. CONVERT YOLO GT BOX TO PIXEL BOX
# ------------------------------------------------------------

def yolo_to_pixel_box(label, image_width, image_height):

    xc = label["x_center"] * image_width
    yc = label["y_center"] * image_height

    w = label["width"] * image_width
    h = label["height"] * image_height

    x1 = int(xc - w / 2)
    y1 = int(yc - h / 2)

    x2 = int(xc + w / 2)
    y2 = int(yc + h / 2)

    x1 = max(0, min(x1, image_width - 1))
    y1 = max(0, min(y1, image_height - 1))

    x2 = max(0, min(x2, image_width))
    y2 = max(0, min(y2, image_height))

    return [x1, y1, x2, y2]


# ------------------------------------------------------------
# 12. PROCESS SPLIT
# ------------------------------------------------------------

def process_split(
    split_name,
    image_dir,
    label_dir,
):

    print("\n" + "=" * 80)
    print(f"PROCESSING {split_name.upper()}")
    print("=" * 80)

    output_dir = OUTPUT_ROOT / split_name

    output_dir.mkdir(
        parents=True,
        exist_ok=True
    )

    # --------------------------------------------------------
    # Find images
    # --------------------------------------------------------

    image_paths = sorted(
        [
            p for p in image_dir.iterdir()
            if p.is_file()
            and p.suffix.lower() in IMAGE_EXTENSIONS
        ]
    )

    print(f"Image directory : {image_dir}")
    print(f"Label directory : {label_dir}")
    print(f"Images          : {len(image_paths)}")

    if len(image_paths) == 0:
        raise RuntimeError(
            f"\nZERO IMAGES FOUND.\n"
            f"Expected images in:\n{image_dir}"
        )

    # --------------------------------------------------------
    # Create class folders
    # --------------------------------------------------------

    for class_name in MOBILENET_CLASSES:

        (output_dir / class_name).mkdir(
            parents=True,
            exist_ok=True
        )

    # --------------------------------------------------------
    # Counters
    # --------------------------------------------------------

    total_detections = 0
    matched_detections = 0
    saved_crops = 0

    class_counts = {
        class_name: 0
        for class_name in MOBILENET_CLASSES
    }

    metadata_rows = []

    # --------------------------------------------------------
    # Process images
    # --------------------------------------------------------

    for image_index, image_path in enumerate(
        image_paths,
        start=1
    ):

        image = cv2.imread(str(image_path))

        if image is None:
            print(
                f"WARNING: Could not read {image_path}"
            )
            continue

        image_height, image_width = image.shape[:2]

        # ----------------------------------------------------
        # Ground truth
        # ----------------------------------------------------

        label_path = (
            label_dir /
            f"{image_path.stem}.txt"
        )

        gt_labels = read_yolo_labels(label_path)

        gt_boxes = []

        for gt in gt_labels:

            if gt["class_id"] not in YOLO_TO_MOBILENET:
                continue

            gt_box = yolo_to_pixel_box(
                gt,
                image_width,
                image_height
            )

            gt_boxes.append(
                {
                    "box": gt_box,
                    "class_id": gt["class_id"],
                }
            )

        # ----------------------------------------------------
        # YOLO prediction
        # ----------------------------------------------------

        results = model.predict(
            source=str(image_path),
            device=device,
            verbose=False,
            conf=0.001,
            iou=0.7,
        )

        result = results[0]

        if result.boxes is None:
            continue

        predictions = result.boxes

        total_detections += len(predictions)

        # ----------------------------------------------------
        # Match prediction to GT
        # ----------------------------------------------------

        used_gt = set()

        for pred_index in range(len(predictions)):

            pred_box = predictions.xyxy[
                pred_index
            ].cpu().numpy()

            pred_class = int(
                predictions.cls[
                    pred_index
                ].item()
            )

            # We only care about the 8 original classes
            if pred_class not in YOLO_TO_MOBILENET:
                continue

            best_iou = 0.0
            best_gt_index = None

            for gt_index, gt in enumerate(gt_boxes):

                if gt_index in used_gt:
                    continue

                iou = calculate_iou(
                    pred_box,
                    gt["box"]
                )

                if iou > best_iou:
                    best_iou = iou
                    best_gt_index = gt_index

            # ------------------------------------------------
            # Match threshold
            # ------------------------------------------------

            if (
                best_gt_index is None
                or best_iou < 0.5
            ):
                continue

            gt = gt_boxes[best_gt_index]

            used_gt.add(best_gt_index)

            matched_detections += 1

            # ------------------------------------------------
            # IMPORTANT:
            # MobileNet target comes from GT class,
            # NOT YOLO predicted class.
            #
            # This ensures:
            # metal + cardboard -> non_plastic
            # ------------------------------------------------

            gt_class_id = gt["class_id"]

            mobilenet_class = (
                YOLO_TO_MOBILENET[
                    gt_class_id
                ]
            )

            # ------------------------------------------------
            # Crop using YOLO predicted bounding box
            # ------------------------------------------------

            x1 = max(
                0,
                int(pred_box[0])
            )

            y1 = max(
                0,
                int(pred_box[1])
            )

            x2 = min(
                image_width,
                int(pred_box[2])
            )

            y2 = min(
                image_height,
                int(pred_box[3])
            )

            if x2 <= x1 or y2 <= y1:
                continue

            crop = image[
                y1:y2,
                x1:x2
            ]

            if crop.size == 0:
                continue

            # ------------------------------------------------
            # Save crop
            # ------------------------------------------------

            crop_id = (
                f"{image_path.stem}"
                f"_det{pred_index:04d}"
            )

            crop_path = (
                output_dir /
                mobilenet_class /
                f"{crop_id}.jpg"
            )

            cv2.imwrite(
                str(crop_path),
                crop
            )

            saved_crops += 1
            class_counts[mobilenet_class] += 1

            # ------------------------------------------------
            # Metadata
            # ------------------------------------------------

            metadata_rows.append(
                {
                    "crop_path": str(
                        crop_path.relative_to(
                            OUTPUT_ROOT
                        )
                    ),
                    "source_image": str(
                        image_path
                    ),
                    "split": split_name,
                    "yolo_gt_class_id": gt_class_id,
                    "yolo_gt_class_name":
                        original_names[
                            gt_class_id
                        ],
                    "mobilenet_class":
                        mobilenet_class,
                    "mobilenet_class_id":
                        CLASS_TO_ID[
                            mobilenet_class
                        ],
                    "yolo_pred_class_id":
                        pred_class,
                    "yolo_pred_class_name":
                        original_names.get(
                            pred_class,
                            "unknown"
                        ),
                    "iou_with_gt":
                        best_iou,
                    "x1": x1,
                    "y1": y1,
                    "x2": x2,
                    "y2": y2,
                }
            )

        # ----------------------------------------------------
        # Progress
        # ----------------------------------------------------

        if (
            image_index % 100 == 0
            or image_index == len(image_paths)
        ):

            print(
                f"Processed {image_index}/{len(image_paths)} | "
                f"Detections: {total_detections} | "
                f"Matched: {matched_detections} | "
                f"Crops: {saved_crops}"
            )

    # --------------------------------------------------------
    # Save metadata
    # --------------------------------------------------------

    metadata_path = (
        output_dir /
        "metadata.csv"
    )

    fieldnames = [
        "crop_path",
        "source_image",
        "split",
        "yolo_gt_class_id",
        "yolo_gt_class_name",
        "mobilenet_class",
        "mobilenet_class_id",
        "yolo_pred_class_id",
        "yolo_pred_class_name",
        "iou_with_gt",
        "x1",
        "y1",
        "x2",
        "y2",
    ]

    with open(
        metadata_path,
        "w",
        newline="",
        encoding="utf-8"
    ) as f:

        writer = csv.DictWriter(
            f,
            fieldnames=fieldnames
        )

        writer.writeheader()
        writer.writerows(metadata_rows)

    # --------------------------------------------------------
    # Summary
    # --------------------------------------------------------

    print("\n" + "-" * 70)
    print(f"{split_name.upper()} COMPLETE")
    print("-" * 70)

    print(
        f"Images processed       : {len(image_paths)}"
    )

    print(
        f"YOLO detections        : {total_detections}"
    )

    print(
        f"GT-matched detections  : {matched_detections}"
    )

    print(
        f"Saved crops            : {saved_crops}"
    )

    print(
        f"Metadata rows          : {len(metadata_rows)}"
    )

    print("\nClass distribution:")

    for class_name in MOBILENET_CLASSES:

        print(
            f"{class_name:25s}: "
            f"{class_counts[class_name]}"
        )

    # --------------------------------------------------------
    # Integrity checks
    # --------------------------------------------------------

    saved_files = sum(
        1
        for class_name in MOBILENET_CLASSES
        for p in (
            output_dir / class_name
        ).iterdir()
        if p.is_file()
    )

    print("\nIntegrity checks:")

    print(
        "sum(class counts) == metadata rows:",
        sum(class_counts.values())
        == len(metadata_rows)
    )

    print(
        "metadata rows == saved crops:",
        len(metadata_rows)
        == saved_files
    )

    print(
        "metadata file:",
        metadata_path
    )

    return {
        "images": len(image_paths),
        "detections": total_detections,
        "matched": matched_detections,
        "crops": saved_crops,
        "class_counts": class_counts,
    }


# ============================================================
# 13. PROCESS TRAIN
# ============================================================

train_results = process_split(
    "train",
    TRAIN_IMAGES,
    TRAIN_LABELS,
)


# ============================================================
# 14. PROCESS VAL
# ============================================================

val_results = process_split(
    "val",
    VAL_IMAGES,
    VAL_LABELS,
)


# ============================================================
# 15. FINAL SUMMARY
# ============================================================

print("\n" + "=" * 80)
print("E3Y — CROP GENERATION COMPLETE")
print("=" * 80)

print(f"\nOutput directory:")
print(OUTPUT_ROOT)

print("\nTRAIN:")
print(
    f"  Images       : {train_results['images']}"
)
print(
    f"  Detections   : {train_results['detections']}"
)
print(
    f"  GT matched   : {train_results['matched']}"
)
print(
    f"  Crops        : {train_results['crops']}"
)

print("\nVAL:")
print(
    f"  Images       : {val_results['images']}"
)
print(
    f"  Detections   : {val_results['detections']}"
)
print(
    f"  GT matched   : {val_results['matched']}"
)
print(
    f"  Crops        : {val_results['crops']}"
)

print("\n7-class MobileNet distribution:")

for class_name in MOBILENET_CLASSES:

    train_count = (
        train_results["class_counts"]
        [class_name]
    )

    val_count = (
        val_results["class_counts"]
        [class_name]
    )

    print(
        f"{class_name:25s} "
        f"Train: {train_count:8d} | "
        f"Val: {val_count:8d}"
    )

print("\n" + "=" * 80)
print("IMPORTANT")
print("=" * 80)

print(
    "Metal and cardboard are both mapped to NON-PLASTIC."
)

print(
    "Do NOT train MobileNet until the distributions "
    "and integrity checks have been reviewed."
)

E3Y — YOLO → MOBILENET CROP GENERATION

Checking dataset paths...
Dataset root        : C:\Users\varda\Documents\_My Computer\COMMON_space\Learning\Upgrad_EPGP_MS\upgrad_IIITB_MS_course\MS_LJMU_Material\Topic Data\SortWaste\dataset\dataset
                      Exists: True
Split root          : C:\Users\varda\Documents\_My Computer\COMMON_space\Learning\Upgrad_EPGP_MS\upgrad_IIITB_MS_course\MS_LJMU_Material\Topic Data\SortWaste\dataset\dataset\splited_all_dataset_coco
                      Exists: True
Train images        : C:\Users\varda\Documents\_My Computer\COMMON_space\Learning\Upgrad_EPGP_MS\upgrad_IIITB_MS_course\MS_LJMU_Material\Topic Data\SortWaste\dataset\dataset\splited_all_dataset_coco\train\images
                      Exists: True
Train labels        : C:\Users\varda\Documents\_My Computer\COMMON_space\Learning\Upgrad_EPGP_MS\upgrad_IIITB_MS_course\MS_LJMU_Material\Topic Data\SortWaste\dataset\dataset\splited_all_dataset_coco\train\labels
                      Exists: Tr

In [ ]:
## model trainig
# =============================================================================
# E3Y — YOLO11n → MobileNet-V3-Large BASELINE
# =============================================================================
#
# Experiment:
#   YOLO11n Epoch-18 matched crops → MobileNet-V3-Large
#
# Purpose:
#   Establish the baseline second-stage classifier for the proposed
#   YOLO11n + MobileNet pipeline.
#
# IMPORTANT:
#   - NO class weights
#   - NO oversampling
#   - NO focal loss
#   - NO class-specific augmentation
#   - Standard augmentation only
#
# Best model selected using VALIDATION MACRO F1.
#
# =============================================================================

import os
import random
import json
import time
import copy
import numpy as np
import pandas as pd
from pathlib import Path

import torch
import torch.nn as nn
import torch.optim as optim

from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from PIL import Image

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    classification_report,
    confusion_matrix
)

import matplotlib.pyplot as plt


# =============================================================================
# 1. CONFIGURATION
# =============================================================================

SEED = 42

# -------------------------------------------------------------------------
# Dataset
# -------------------------------------------------------------------------

DATASET_ROOT = Path(
    r"C:\Users\varda\Documents\_My Computer\COMMON_space\Learning\Upgrad_EPGP_MS"
    r"\upgrad_IIITB_MS_course\MS_LJMU_Material\Topic Data\SortWaste\dataset\dataset"
)

E3Y_ROOT = DATASET_ROOT / "yolo_mobilenet_crops_E3Y"

TRAIN_DIR = E3Y_ROOT / "train"
VAL_DIR = E3Y_ROOT / "val"

TRAIN_METADATA = TRAIN_DIR / "metadata.csv"
VAL_METADATA = VAL_DIR / "metadata.csv"


# -------------------------------------------------------------------------
# Results
# -------------------------------------------------------------------------

RESULTS_DIR = (
    DATASET_ROOT
    / "mobilenet_results"
    / "E3Y_baseline"
)

RESULTS_DIR.mkdir(
    parents=True,
    exist_ok=True
)


# -------------------------------------------------------------------------
# Classes
# -------------------------------------------------------------------------

CLASS_NAMES = [
    "ecal",
    "hdpe",
    "mixed_plastic_rigid",
    "mixed_plastic_soft",
    "non_plastic",
    "pet",
    "pet_oil"
]

NUM_CLASSES = len(CLASS_NAMES)

CLASS_TO_IDX = {
    name: idx
    for idx, name in enumerate(CLASS_NAMES)
}


# -------------------------------------------------------------------------
# Training
# -------------------------------------------------------------------------

IMAGE_SIZE = 224

BATCH_SIZE = 64

NUM_EPOCHS = 30

LEARNING_RATE = 1e-4

WEIGHT_DECAY = 1e-4

NUM_WORKERS = 0

PATIENCE = 7

MIN_DELTA = 1e-4


# =============================================================================
# 2. REPRODUCIBILITY
# =============================================================================

def set_seed(seed=42):

    random.seed(seed)
    np.random.seed(seed)

    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)

    # Reproducibility
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


set_seed(SEED)


# =============================================================================
# 3. DEVICE
# =============================================================================

DEVICE = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

print("=" * 80)
print("E3Y — YOLO11n → MobileNet-V3-Large BASELINE")
print("=" * 80)

print(f"Device       : {DEVICE}")

if torch.cuda.is_available():
    print(f"GPU          : {torch.cuda.get_device_name(0)}")
    print(
        f"CUDA         : "
        f"{torch.version.cuda}"
    )

print(f"Classes      : {NUM_CLASSES}")
print(f"Batch size   : {BATCH_SIZE}")
print(f"Epochs       : {NUM_EPOCHS}")
print(f"Learning rate: {LEARNING_RATE}")
print(f"Seed         : {SEED}")

print()


# =============================================================================
# 4. CHECK PATHS
# =============================================================================

print("=" * 80)
print("CHECKING DATASET")
print("=" * 80)

required_paths = [
    E3Y_ROOT,
    TRAIN_DIR,
    VAL_DIR,
    TRAIN_METADATA,
    VAL_METADATA
]

for path in required_paths:

    print(
        f"{str(path):<100} "
        f"Exists: {path.exists()}"
    )

    if not path.exists():
        raise FileNotFoundError(
            f"Required path does not exist:\n{path}"
        )

print()


# =============================================================================
# 5. LOAD METADATA
# =============================================================================

print("=" * 80)
print("LOADING METADATA")
print("=" * 80)

train_df = pd.read_csv(TRAIN_METADATA)
val_df = pd.read_csv(VAL_METADATA)

print(f"Training metadata rows  : {len(train_df)}")
print(f"Validation metadata rows: {len(val_df)}")

print()


# =============================================================================
# 6. IDENTIFY METADATA COLUMNS
# =============================================================================
#
# We expect the crop-generation script to contain a path column and a
# MobileNet class column.
#
# The code below handles common column names automatically.
# =============================================================================

def find_column(df, candidates):

    lower_map = {
        col.lower(): col
        for col in df.columns
    }

    for candidate in candidates:

        if candidate.lower() in lower_map:
            return lower_map[candidate.lower()]

    return None


PATH_COLUMN = find_column(
    train_df,
    [
        "crop_path",
        "image_path",
        "path",
        "filepath",
        "file_path",
        "crop"
    ]
)

CLASS_COLUMN = find_column(
    train_df,
    [
        "class_name",
        "class",
        "mobile_net_class",
        "mobilenet_class",
        "target_class",
        "label"
    ]
)


print("Metadata columns:")
print(list(train_df.columns))
print()

print(f"Detected path column : {PATH_COLUMN}")
print(f"Detected class column: {CLASS_COLUMN}")

if PATH_COLUMN is None:
    raise ValueError(
        "Could not identify crop path column in metadata.csv.\n"
        f"Available columns: {list(train_df.columns)}"
    )

if CLASS_COLUMN is None:
    raise ValueError(
        "Could not identify class column in metadata.csv.\n"
        f"Available columns: {list(train_df.columns)}"
    )

print()


# =============================================================================
# 7. NORMALIZE CLASS LABELS
# =============================================================================

def normalize_class_name(x):

    x = str(x).strip().lower()

    # Handle possible naming variations
    replacements = {
        "mixed rigid plastic": "mixed_plastic_rigid",
        "mixed soft plastic": "mixed_plastic_soft",
        "mixed_plastic_rigid": "mixed_plastic_rigid",
        "mixed_plastic_soft": "mixed_plastic_soft",
        "non plastic": "non_plastic",
        "non-plastic": "non_plastic",
        "non_plastic": "non_plastic",
        "pet oil": "pet_oil",
        "pet_oil": "pet_oil"
    }

    return replacements.get(x, x)


train_df["class_name_normalized"] = (
    train_df[CLASS_COLUMN]
    .apply(normalize_class_name)
)

val_df["class_name_normalized"] = (
    val_df[CLASS_COLUMN]
    .apply(normalize_class_name)
)


# =============================================================================
# 8. VALIDATE CLASSES
# =============================================================================

print("=" * 80)
print("CLASS VALIDATION")
print("=" * 80)

train_classes = set(
    train_df["class_name_normalized"]
)

val_classes = set(
    val_df["class_name_normalized"]
)

expected_classes = set(CLASS_NAMES)

print("Expected classes:")
print(CLASS_NAMES)

print()

print("Training classes:")
print(sorted(train_classes))

print()

print("Validation classes:")
print(sorted(val_classes))

print()

unknown_train = train_classes - expected_classes
unknown_val = val_classes - expected_classes

if unknown_train:
    raise ValueError(
        f"Unknown training classes found: {unknown_train}"
    )

if unknown_val:
    raise ValueError(
        f"Unknown validation classes found: {unknown_val}"
    )

print("Class validation passed.")
print()


# =============================================================================
# 9. CREATE NUMERIC LABELS
# =============================================================================

train_df["label"] = train_df[
    "class_name_normalized"
].map(CLASS_TO_IDX)

val_df["label"] = val_df[
    "class_name_normalized"
].map(CLASS_TO_IDX)


if train_df["label"].isna().any():
    raise ValueError(
        "Some training samples have invalid labels."
    )

if val_df["label"].isna().any():
    raise ValueError(
        "Some validation samples have invalid labels."
    )

train_df["label"] = train_df["label"].astype(int)
val_df["label"] = val_df["label"].astype(int)


# =============================================================================
# 10. RESOLVE CROP PATHS
# =============================================================================

def resolve_image_path(path_value, split_dir):

    path_value = str(path_value)

    # Absolute path
    p = Path(path_value)

    if p.is_absolute() and p.exists():
        return p

    # Relative path from split directory
    p = split_dir / path_value

    if p.exists():
        return p

    # Relative path from E3Y root
    p = E3Y_ROOT / path_value

    if p.exists():
        return p

    # Filename only — recursive search
    filename = Path(path_value).name

    matches = list(
        split_dir.rglob(filename)
    )

    if len(matches) > 0:
        return matches[0]

    return None


print("=" * 80)
print("VALIDATING IMAGE PATHS")
print("=" * 80)

train_df["resolved_path"] = train_df[
    PATH_COLUMN
].apply(
    lambda x: resolve_image_path(
        x,
        TRAIN_DIR
    )
)

val_df["resolved_path"] = val_df[
    PATH_COLUMN
].apply(
    lambda x: resolve_image_path(
        x,
        VAL_DIR
    )
)


train_missing = train_df[
    train_df["resolved_path"].isna()
]

val_missing = val_df[
    val_df["resolved_path"].isna()
]

print(
    f"Missing training images  : {len(train_missing)}"
)

print(
    f"Missing validation images: {len(val_missing)}"
)

if len(train_missing) > 0:

    print("\nExample missing training paths:")
    print(
        train_missing[PATH_COLUMN]
        .head(10)
        .to_string(index=False)
    )

    raise FileNotFoundError(
        "Training crop paths could not be resolved."
    )


if len(val_missing) > 0:

    print("\nExample missing validation paths:")
    print(
        val_missing[PATH_COLUMN]
        .head(10)
        .to_string(index=False)
    )

    raise FileNotFoundError(
        "Validation crop paths could not be resolved."
    )

print("All crop paths successfully resolved.")
print()


# =============================================================================
# 11. CLASS DISTRIBUTION
# =============================================================================

print("=" * 80)
print("CLASS DISTRIBUTION")
print("=" * 80)

train_counts = (
    train_df["class_name_normalized"]
    .value_counts()
    .reindex(CLASS_NAMES)
    .fillna(0)
    .astype(int)
)

val_counts = (
    val_df["class_name_normalized"]
    .value_counts()
    .reindex(CLASS_NAMES)
    .fillna(0)
    .astype(int)
)

distribution_df = pd.DataFrame({
    "class": CLASS_NAMES,
    "train": train_counts.values,
    "validation": val_counts.values
})

print(
    distribution_df.to_string(
        index=False
    )
)

print()

print(
    "IMPORTANT: No class balancing is being applied."
)

print()


# =============================================================================
# 12. DATASET CLASS
# =============================================================================

class E3YDataset(Dataset):

    def __init__(
        self,
        dataframe,
        transform=None
    ):

        self.df = dataframe.reset_index(
            drop=True
        )

        self.transform = transform

    def __len__(self):

        return len(self.df)

    def __getitem__(self, idx):

        row = self.df.iloc[idx]

        image_path = row["resolved_path"]

        label = int(row["label"])

        try:

            image = Image.open(
                image_path
            ).convert("RGB")

        except Exception as e:

            raise RuntimeError(
                f"Could not load image:\n"
                f"{image_path}\n"
                f"Error: {e}"
            )

        if self.transform is not None:

            image = self.transform(image)

        return image, label


# =============================================================================
# 13. TRANSFORMS
# =============================================================================
#
# Standard augmentation only.
#
# These augmentations are NOT targeted at minority classes.
# =============================================================================

train_transform = transforms.Compose([

    transforms.Resize(
        (IMAGE_SIZE, IMAGE_SIZE)
    ),

    transforms.RandomHorizontalFlip(
        p=0.5
    ),

    transforms.RandomRotation(
        degrees=10
    ),

    transforms.ColorJitter(
        brightness=0.15,
        contrast=0.15,
        saturation=0.10,
        hue=0.02
    ),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=[
            0.485,
            0.456,
            0.406
        ],
        std=[
            0.229,
            0.224,
            0.225
        ]
    )
])


val_transform = transforms.Compose([

    transforms.Resize(
        (IMAGE_SIZE, IMAGE_SIZE)
    ),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=[
            0.485,
            0.456,
            0.406
        ],
        std=[
            0.229,
            0.224,
            0.225
        ]
    )
])


# =============================================================================
# 14. CREATE DATASETS
# =============================================================================

train_dataset = E3YDataset(
    train_df,
    transform=train_transform
)

val_dataset = E3YDataset(
    val_df,
    transform=val_transform
)


# =============================================================================
# 15. CREATE DATALOADERS
# =============================================================================

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available()
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available()
)


print("=" * 80)
print("DATALOADERS")
print("=" * 80)

print(
    f"Training batches  : {len(train_loader)}"
)

print(
    f"Validation batches: {len(val_loader)}"
)

print()


# =============================================================================
# 16. LOAD MOBILENET-V3-LARGE
# =============================================================================

print("=" * 80)
print("LOADING MOBILENET-V3-LARGE")
print("=" * 80)

weights = (
    models.MobileNet_V3_Large_Weights.DEFAULT
)

model = models.mobilenet_v3_large(
    weights=weights
)


# Replace classifier
in_features = model.classifier[-1].in_features

model.classifier[-1] = nn.Linear(
    in_features,
    NUM_CLASSES
)

model = model.to(DEVICE)


print(model.classifier)
print()


# =============================================================================
# 17. LOSS / OPTIMIZER
# =============================================================================

criterion = nn.CrossEntropyLoss()

optimizer = optim.AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY
)

scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="max",
    factor=0.5,
    patience=2
)


# =============================================================================
# 18. TRAINING FUNCTION
# =============================================================================

def train_one_epoch(
    model,
    loader,
    criterion,
    optimizer,
    device
):

    model.train()

    running_loss = 0.0

    all_predictions = []
    all_targets = []

    for images, targets in loader:

        images = images.to(
            device,
            non_blocking=True
        )

        targets = targets.to(
            device,
            non_blocking=True
        )

        optimizer.zero_grad(
            set_to_none=True
        )

        outputs = model(images)

        loss = criterion(
            outputs,
            targets
        )

        loss.backward()

        optimizer.step()

        running_loss += (
            loss.item()
            * images.size(0)
        )

        predictions = (
            outputs.argmax(
                dim=1
            )
        )

        all_predictions.extend(
            predictions.detach()
            .cpu()
            .numpy()
        )

        all_targets.extend(
            targets.detach()
            .cpu()
            .numpy()
        )

    epoch_loss = (
        running_loss
        / len(loader.dataset)
    )

    accuracy = accuracy_score(
        all_targets,
        all_predictions
    )

    macro_f1 = f1_score(
        all_targets,
        all_predictions,
        average="macro",
        zero_division=0
    )

    weighted_f1 = f1_score(
        all_targets,
        all_predictions,
        average="weighted",
        zero_division=0
    )

    return (
        epoch_loss,
        accuracy,
        macro_f1,
        weighted_f1
    )


# =============================================================================
# 19. VALIDATION FUNCTION
# =============================================================================

def evaluate(
    model,
    loader,
    criterion,
    device
):

    model.eval()

    running_loss = 0.0

    all_predictions = []
    all_targets = []

    with torch.no_grad():

        for images, targets in loader:

            images = images.to(
                device,
                non_blocking=True
            )

            targets = targets.to(
                device,
                non_blocking=True
            )

            outputs = model(images)

            loss = criterion(
                outputs,
                targets
            )

            running_loss += (
                loss.item()
                * images.size(0)
            )

            predictions = (
                outputs.argmax(
                    dim=1
                )
            )

            all_predictions.extend(
                predictions.cpu().numpy()
            )

            all_targets.extend(
                targets.cpu().numpy()
            )

    epoch_loss = (
        running_loss
        / len(loader.dataset)
    )

    accuracy = accuracy_score(
        all_targets,
        all_predictions
    )

    macro_f1 = f1_score(
        all_targets,
        all_predictions,
        average="macro",
        zero_division=0
    )

    weighted_f1 = f1_score(
        all_targets,
        all_predictions,
        average="weighted",
        zero_division=0
    )

    return (
        epoch_loss,
        accuracy,
        macro_f1,
        weighted_f1,
        all_targets,
        all_predictions
    )


# =============================================================================
# 20. TRAINING LOOP
# =============================================================================

print("=" * 80)
print("STARTING E3Y BASELINE TRAINING")
print("=" * 80)

history = []

best_macro_f1 = -1.0

best_epoch = 0

best_state = None

epochs_without_improvement = 0

training_start = time.time()


for epoch in range(
    1,
    NUM_EPOCHS + 1
):

    epoch_start = time.time()

    (
        train_loss,
        train_acc,
        train_macro_f1,
        train_weighted_f1
    ) = train_one_epoch(
        model,
        train_loader,
        criterion,
        optimizer,
        DEVICE
    )

    (
        val_loss,
        val_acc,
        val_macro_f1,
        val_weighted_f1,
        val_targets,
        val_predictions
    ) = evaluate(
        model,
        val_loader,
        criterion,
        DEVICE
    )

    scheduler.step(
        val_macro_f1
    )

    current_lr = optimizer.param_groups[0]["lr"]

    epoch_time = (
        time.time()
        - epoch_start
    )

    history.append({

        "epoch": epoch,

        "train_loss": train_loss,
        "train_accuracy": train_acc,
        "train_macro_f1": train_macro_f1,
        "train_weighted_f1": train_weighted_f1,

        "val_loss": val_loss,
        "val_accuracy": val_acc,
        "val_macro_f1": val_macro_f1,
        "val_weighted_f1": val_weighted_f1,

        "learning_rate": current_lr,

        "epoch_time_sec": epoch_time

    })


    print(
        f"\nEpoch {epoch:02d}/{NUM_EPOCHS}"
    )

    print(
        f"Train | "
        f"Loss: {train_loss:.4f} | "
        f"Acc: {train_acc:.4f} | "
        f"Macro F1: {train_macro_f1:.4f} | "
        f"Weighted F1: {train_weighted_f1:.4f}"
    )

    print(
        f"Val   | "
        f"Loss: {val_loss:.4f} | "
        f"Acc: {val_acc:.4f} | "
        f"Macro F1: {val_macro_f1:.4f} | "
        f"Weighted F1: {val_weighted_f1:.4f}"
    )

    print(
        f"LR: {current_lr:.8f} | "
        f"Time: {epoch_time:.1f}s"
    )


    # ---------------------------------------------------------------------
    # BEST MODEL
    # ---------------------------------------------------------------------

    if val_macro_f1 > (
        best_macro_f1 + MIN_DELTA
    ):

        best_macro_f1 = val_macro_f1

        best_epoch = epoch

        best_state = copy.deepcopy(
            model.state_dict()
        )

        epochs_without_improvement = 0

        best_checkpoint = {

            "epoch": epoch,

            "model_state_dict":
                model.state_dict(),

            "optimizer_state_dict":
                optimizer.state_dict(),

            "best_macro_f1":
                best_macro_f1,

            "val_accuracy":
                val_acc,

            "val_weighted_f1":
                val_weighted_f1,

            "class_names":
                CLASS_NAMES,

            "seed":
                SEED,

            "experiment":
                "E3Y_YOLO11n_Epoch18_MobileNetV3Large_baseline"

        }

        torch.save(
            best_checkpoint,
            RESULTS_DIR
            / "E3Y_MobileNetV3Large_best.pth"
        )

        print(
            f"*** NEW BEST MODEL — "
            f"Macro F1 = {best_macro_f1:.4f}"
        )

    else:

        epochs_without_improvement += 1

        print(
            f"No improvement "
            f"({epochs_without_improvement}/{PATIENCE})"
        )


    # ---------------------------------------------------------------------
    # EARLY STOPPING
    # ---------------------------------------------------------------------

    if epochs_without_improvement >= PATIENCE:

        print(
            "\nEarly stopping triggered."
        )

        break


training_time = (
    time.time()
    - training_start
)


# =============================================================================
# 21. RESTORE BEST MODEL
# =============================================================================

print()
print("=" * 80)
print("RESTORING BEST MODEL")
print("=" * 80)

if best_state is None:

    raise RuntimeError(
        "No best model was saved."
    )

model.load_state_dict(
    best_state
)

print(
    f"Best epoch   : {best_epoch}"
)

print(
    f"Best Macro F1: {best_macro_f1:.4f}"
)

print()


# =============================================================================
# 22. FINAL VALIDATION
# =============================================================================

print("=" * 80)
print("FINAL E3Y VALIDATION")
print("=" * 80)

(
    final_loss,
    final_accuracy,
    final_macro_f1,
    final_weighted_f1,
    final_targets,
    final_predictions
) = evaluate(
    model,
    val_loader,
    criterion,
    DEVICE
)


print(
    f"Accuracy       : {final_accuracy:.4f}"
)

print(
    f"Accuracy (%)   : {final_accuracy * 100:.2f}%"
)

print(
    f"Macro F1       : {final_macro_f1:.4f}"
)

print(
    f"Weighted F1    : {final_weighted_f1:.4f}"
)

print(
    f"Validation loss: {final_loss:.4f}"
)

print(
    f"Samples        : {len(final_targets)}"
)

print()


# =============================================================================
# 23. CLASSIFICATION REPORT
# =============================================================================

print("=" * 80)
print("CLASSIFICATION REPORT")
print("=" * 80)

report_dict = classification_report(
    final_targets,
    final_predictions,
    labels=list(range(NUM_CLASSES)),
    target_names=CLASS_NAMES,
    output_dict=True,
    zero_division=0
)

report_text = classification_report(
    final_targets,
    final_predictions,
    labels=list(range(NUM_CLASSES)),
    target_names=CLASS_NAMES,
    zero_division=0
)

print(report_text)


# Save report
report_df = pd.DataFrame(
    report_dict
).transpose()

report_df.to_csv(
    RESULTS_DIR
    / "classification_report.csv"
)


# =============================================================================
# 24. CONFUSION MATRIX
# =============================================================================

cm = confusion_matrix(
    final_targets,
    final_predictions,
    labels=list(range(NUM_CLASSES))
)

cm_df = pd.DataFrame(
    cm,
    index=CLASS_NAMES,
    columns=CLASS_NAMES
)

cm_df.to_csv(
    RESULTS_DIR
    / "confusion_matrix.csv"
)


print()
print("=" * 80)
print("CONFUSION MATRIX")
print("=" * 80)

print(cm_df)


# =============================================================================
# 25. PER-CLASS SUMMARY
# =============================================================================

per_class_rows = []

for i, class_name in enumerate(
    CLASS_NAMES
):

    tp = cm[i, i]

    actual = cm[i, :].sum()

    predicted = cm[:, i].sum()

    incorrect = actual - tp

    recall = (
        tp / actual
        if actual > 0
        else 0
    )

    precision = (
        tp / predicted
        if predicted > 0
        else 0
    )

    if (
        precision + recall
    ) > 0:

        f1 = (
            2
            * precision
            * recall
            / (precision + recall)
        )

    else:

        f1 = 0

    per_class_rows.append({

        "class": class_name,

        "actual": actual,

        "correct": tp,

        "incorrect": incorrect,

        "precision": precision,

        "recall": recall,

        "f1": f1

    })


per_class_df = pd.DataFrame(
    per_class_rows
)

per_class_df.to_csv(
    RESULTS_DIR
    / "per_class_results.csv",
    index=False
)


print()
print("=" * 80)
print("PER-CLASS RESULTS")
print("=" * 80)

print(
    per_class_df.to_string(
        index=False
    )
)


# =============================================================================
# 26. PLASTIC-ONLY RESULTS
# =============================================================================
#
# Exclude non_plastic.
#
# This is particularly important for your thesis because the research focus
# is plastic-type classification.
# =============================================================================

plastic_indices = [
    CLASS_TO_IDX["ecal"],
    CLASS_TO_IDX["hdpe"],
    CLASS_TO_IDX["mixed_plastic_rigid"],
    CLASS_TO_IDX["mixed_plastic_soft"],
    CLASS_TO_IDX["pet"],
    CLASS_TO_IDX["pet_oil"]
]


plastic_mask = np.isin(
    np.array(final_targets),
    plastic_indices
)

plastic_targets = np.array(
    final_targets
)[plastic_mask]

plastic_predictions = np.array(
    final_predictions
)[plastic_mask]


plastic_accuracy = accuracy_score(
    plastic_targets,
    plastic_predictions
)

plastic_macro_f1 = f1_score(
    plastic_targets,
    plastic_predictions,
    labels=plastic_indices,
    average="macro",
    zero_division=0
)

plastic_weighted_f1 = f1_score(
    plastic_targets,
    plastic_predictions,
    labels=plastic_indices,
    average="weighted",
    zero_division=0
)


print()
print("=" * 80)
print("PLASTIC-ONLY CLASSIFICATION")
print("=" * 80)

print(
    f"Plastic samples     : "
    f"{len(plastic_targets)}"
)

print(
    f"Plastic accuracy    : "
    f"{plastic_accuracy:.4f}"
)

print(
    f"Plastic accuracy    : "
    f"{plastic_accuracy * 100:.2f}%"
)

print(
    f"Plastic Macro F1    : "
    f"{plastic_macro_f1:.4f}"
)

print(
    f"Plastic Weighted F1 : "
    f"{plastic_weighted_f1:.4f}"
)


# =============================================================================
# 27. SAVE TRAINING HISTORY
# =============================================================================

history_df = pd.DataFrame(
    history
)

history_df.to_csv(
    RESULTS_DIR
    / "training_history.csv",
    index=False
)


# =============================================================================
# 28. TRAINING CURVES
# =============================================================================

# -------------------------------------------------------------------------
# Loss
# -------------------------------------------------------------------------

plt.figure(
    figsize=(8, 5)
)

plt.plot(
    history_df["epoch"],
    history_df["train_loss"],
    label="Train Loss"
)

plt.plot(
    history_df["epoch"],
    history_df["val_loss"],
    label="Validation Loss"
)

plt.xlabel("Epoch")
plt.ylabel("Loss")

plt.title(
    "E3Y MobileNet-V3-Large Loss"
)

plt.legend()

plt.grid(
    True,
    alpha=0.3
)

plt.tight_layout()

plt.savefig(
    RESULTS_DIR
    / "loss_curve.png",
    dpi=300
)

plt.close()


# -------------------------------------------------------------------------
# Macro F1
# -------------------------------------------------------------------------

plt.figure(
    figsize=(8, 5)
)

plt.plot(
    history_df["epoch"],
    history_df["train_macro_f1"],
    label="Train Macro F1"
)

plt.plot(
    history_df["epoch"],
    history_df["val_macro_f1"],
    label="Validation Macro F1"
)

plt.axvline(
    best_epoch,
    linestyle="--",
    label=f"Best Epoch ({best_epoch})"
)

plt.xlabel("Epoch")
plt.ylabel("Macro F1")

plt.title(
    "E3Y MobileNet-V3-Large Macro F1"
)

plt.legend()

plt.grid(
    True,
    alpha=0.3
)

plt.tight_layout()

plt.savefig(
    RESULTS_DIR
    / "macro_f1_curve.png",
    dpi=300
)

plt.close()


# -------------------------------------------------------------------------
# Accuracy
# -------------------------------------------------------------------------

plt.figure(
    figsize=(8, 5)
)

plt.plot(
    history_df["epoch"],
    history_df["train_accuracy"],
    label="Train Accuracy"
)

plt.plot(
    history_df["epoch"],
    history_df["val_accuracy"],
    label="Validation Accuracy"
)

plt.xlabel("Epoch")
plt.ylabel("Accuracy")

plt.title(
    "E3Y MobileNet-V3-Large Accuracy"
)

plt.legend()

plt.grid(
    True,
    alpha=0.3
)

plt.tight_layout()

plt.savefig(
    RESULTS_DIR
    / "accuracy_curve.png",
    dpi=300
)

plt.close()


# =============================================================================
# 29. SAVE FINAL CHECKPOINT
# =============================================================================

final_checkpoint = {

    "experiment":
        "E3Y_YOLO11n_Epoch18_MobileNetV3Large_baseline",

    "model":
        "MobileNetV3-Large",

    "detector":
        "YOLO11n",

    "detector_epoch":
        18,

    "num_classes":
        NUM_CLASSES,

    "class_names":
        CLASS_NAMES,

    "best_epoch":
        best_epoch,

    "best_macro_f1":
        best_macro_f1,

    "final_accuracy":
        final_accuracy,

    "final_macro_f1":
        final_macro_f1,

    "final_weighted_f1":
        final_weighted_f1,

    "plastic_accuracy":
        plastic_accuracy,

    "plastic_macro_f1":
        plastic_macro_f1,

    "plastic_weighted_f1":
        plastic_weighted_f1,

    "model_state_dict":
        model.state_dict(),

    "seed":
        SEED

}


torch.save(
    final_checkpoint,
    RESULTS_DIR
    / "E3Y_MobileNetV3Large_final.pth"
)


# =============================================================================
# 30. SAVE EXPERIMENT SUMMARY
# =============================================================================

summary = {

    "experiment":
        "E3Y — YOLO11n Epoch-18 + MobileNet-V3-Large",

    "detector":
        "YOLO11n",

    "detector_epoch":
        18,

    "classifier":
        "MobileNet-V3-Large",

    "num_classes":
        NUM_CLASSES,

    "classes":
        CLASS_NAMES,

    "train_samples":
        len(train_df),

    "validation_samples":
        len(val_df),

    "best_epoch":
        best_epoch,

    "best_validation_macro_f1":
        best_macro_f1,

    "validation_accuracy":
        final_accuracy,

    "validation_macro_f1":
        final_macro_f1,

    "validation_weighted_f1":
        final_weighted_f1,

    "plastic_samples":
        len(plastic_targets),

    "plastic_accuracy":
        plastic_accuracy,

    "plastic_macro_f1":
        plastic_macro_f1,

    "plastic_weighted_f1":
        plastic_weighted_f1,

    "epochs_completed":
        len(history),

    "training_time_minutes":
        training_time / 60,

    "class_balancing":
        False,

    "class_weights":
        False,

    "oversampling":
        False,

    "focal_loss":
        False,

    "seed":
        SEED

}


with open(
    RESULTS_DIR / "experiment_summary.json",
    "w"
) as f:

    json.dump(
        summary,
        f,
        indent=4
    )


# =============================================================================
# 31. FINAL OUTPUT
# =============================================================================

print()
print("=" * 80)
print("E3Y — MOBILE NET BASELINE COMPLETE")
print("=" * 80)

print(
    f"Best epoch             : {best_epoch}"
)

print(
    f"Best Validation Macro F1: "
    f"{best_macro_f1:.4f}"
)

print(
    f"Validation Accuracy     : "
    f"{final_accuracy:.4f} "
    f"({final_accuracy * 100:.2f}%)"
)

print(
    f"Validation Macro F1     : "
    f"{final_macro_f1:.4f}"
)

print(
    f"Validation Weighted F1  : "
    f"{final_weighted_f1:.4f}"
)

print()

print(
    f"Plastic Accuracy        : "
    f"{plastic_accuracy:.4f} "
    f"({plastic_accuracy * 100:.2f}%)"
)

print(
    f"Plastic Macro F1        : "
    f"{plastic_macro_f1:.4f}"
)

print(
    f"Plastic Weighted F1     : "
    f"{plastic_weighted_f1:.4f}"
)

print()

print(
    "Best checkpoint:"
)

print(
    RESULTS_DIR
    / "E3Y_MobileNetV3Large_best.pth"
)

print()

print(
    "Results directory:"
)

print(
    RESULTS_DIR
)

print("=" * 80)

E3Y — YOLO11n → MobileNet-V3-Large BASELINE
Device       : cuda
GPU          : NVIDIA GeForce RTX 3050 Ti Laptop GPU
CUDA         : 12.6
Classes      : 7
Batch size   : 64
Epochs       : 30
Learning rate: 0.0001
Seed         : 42

CHECKING DATASET
C:\Users\varda\Documents\_My Computer\COMMON_space\Learning\Upgrad_EPGP_MS\upgrad_IIITB_MS_course\MS_LJMU_Material\Topic Data\SortWaste\dataset\dataset\yolo_mobilenet_crops_E3Y Exists: True
C:\Users\varda\Documents\_My Computer\COMMON_space\Learning\Upgrad_EPGP_MS\upgrad_IIITB_MS_course\MS_LJMU_Material\Topic Data\SortWaste\dataset\dataset\yolo_mobilenet_crops_E3Y\train Exists: True
C:\Users\varda\Documents\_My Computer\COMMON_space\Learning\Upgrad_EPGP_MS\upgrad_IIITB_MS_course\MS_LJMU_Material\Topic Data\SortWaste\dataset\dataset\yolo_mobilenet_crops_E3Y\val Exists: True
C:\Users\varda\Documents\_My Computer\COMMON_space\Learning\Upgrad_EPGP_MS\upgrad_IIITB_MS_course\MS_LJMU_Material\Topic Data\SortWaste\dataset\dataset\yolo_mobilenet_crops

In [ ]:
## End-to-end evaluating E3Y- A on all the validation detection crops
# ============================================================
# E3Y-A — YOLO11n → MobileNet-V3-Large
# CORRECTED FINAL END-TO-END COCO EVALUATION
#
# E3Y-A uses:
#   YOLO11n validation detections
#       ↓
#   In-memory bounding-box crops
#       ↓
#   MobileNet-V3-Large classification
#       ↓
#   Final confidence =
#       YOLO confidence × MobileNet class probability
#
# Primary evaluation:
#   COCOeval
#
# Diagnostic evaluation:
#   Proper one-to-one IoU matching at IoU >= 0.50
#
# Taxonomy:
#   8 original SortWaste classes
#       ↓
#   7 evaluation classes
#
#   metal + cardboard → non_plastic
#
# IMPORTANT:
#   - Uses ALL YOLO validation detections above conf=0.001
#   - Does NOT use GT-matched crops for inference
#   - Does NOT save new crops
#   - YOLO boxes are cropped in memory
#   - MobileNet supplies the final class
#   - COCO JSON is the authoritative validation GT
#   - COCOeval is the primary end-to-end metric
#   - No random augmentation is applied during evaluation
# ============================================================

from pathlib import Path
import csv
import json

import cv2
import torch
import numpy as np

from ultralytics import YOLO
from torchvision import models, transforms
from PIL import Image

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix,
)

from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval


# ============================================================
# 1. PATHS
# ============================================================

DATASET_ROOT = Path(
    r"C:\Users\varda\Documents\_My Computer\COMMON_space\Learning"
    r"\Upgrad_EPGP_MS\upgrad_IIITB_MS_course\MS_LJMU_Material"
    r"\Topic Data\SortWaste\dataset\dataset"
)

VAL_IMAGES = (
    DATASET_ROOT
    / "splited_all_dataset_coco"
    / "val"
    / "images"
)

VAL_LABELS = (
    DATASET_ROOT
    / "splited_all_dataset_coco"
    / "val"
    / "labels"
)

VAL_COCO_JSON = (
    DATASET_ROOT
    / "splited_all_dataset_coco"
    / "val"
    / "annotations"
    / "val_coco.json"
)

YOLO_CHECKPOINT = Path(
    r"C:\Users\varda\Documents\_My Computer\COMMON_space\Learning"
    r"\Upgrad_EPGP_MS\upgrad_IIITB_MS_course\MS_LJMU_Material"
    r"\Thesis_Code\runs\detect\runs\sortwaste\yolo11n_baseline-2"
    r"\weights\best.pt"
)

# E3Y-A MobileNet checkpoint
MOBILENET_CHECKPOINT = Path(
    r"C:\Users\varda\Documents\_My Computer\COMMON_space\Learning"
    r"\Upgrad_EPGP_MS\upgrad_IIITB_MS_course\MS_LJMU_Material"
    r"\Topic Data\SortWaste\dataset\dataset"
    r"\yolo_mobilenet_crops_E3Y"
    r"\mobilenet_results"
    r"\E3Y_A_baseline"
    r"\E3Y_MobileNetV3Large_best.pth"
)

RESULTS_DIR = (
    DATASET_ROOT
    / "yolo_mobilenet_crops_E3Y"
    / "mobilenet_results"
    / "E3Y_A_COCO_evaluation_corrected"
)

RESULTS_DIR.mkdir(
    parents=True,
    exist_ok=True
)

RAW_RESULTS_CSV = (
    RESULTS_DIR
    / "E3Y_A_COCO_all_detections.csv"
)

COCO_PREDICTIONS_JSON = (
    RESULTS_DIR
    / "E3Y_A_COCO_predictions.json"
)

COCO_RESULTS_CSV = (
    RESULTS_DIR
    / "E3Y_A_COCO_results.csv"
)

CLASS_RESULTS_CSV = (
    RESULTS_DIR
    / "E3Y_A_class_results.csv"
)

MATCHED_RESULTS_CSV = (
    RESULTS_DIR
    / "E3Y_A_matched_classification.csv"
)

SUMMARY_TXT = (
    RESULTS_DIR
    / "E3Y_A_COCO_summary.txt"
)


# ============================================================
# 2. CONFIGURATION
# ============================================================

YOLO_CONF = 0.001
YOLO_NMS_IOU = 0.7

DIAGNOSTIC_MATCH_IOU = 0.50

MOBILENET_IMAGE_SIZE = 224
BATCH_SIZE = 64

DEVICE = (
    torch.device("cuda")
    if torch.cuda.is_available()
    else torch.device("cpu")
)


# ============================================================
# 3. CLASS DEFINITIONS
# ============================================================

# Final 7-class evaluation taxonomy
MOBILENET_CLASSES = [
    "ecal",
    "hdpe",
    "mixed_plastic_rigid",
    "mixed_plastic_soft",
    "non_plastic",
    "pet",
    "pet_oil",
]

CLASS_TO_ID = {
    name: i
    for i, name in enumerate(MOBILENET_CLASSES)
}

ID_TO_CLASS = {
    i: name
    for i, name in enumerate(MOBILENET_CLASSES)
}


# Original SortWaste YOLO classes
ORIGINAL_NAMES = {
    0: "pet",
    1: "hdpe",
    2: "mixed_plastic_soft",
    3: "ecal",
    4: "metal",
    5: "cardboard",
    6: "mixed_plastic_rigid",
    7: "pet_oil",
}


# Original YOLO class → final 7-class taxonomy
YOLO_TO_MOBILENET = {
    0: "pet",
    1: "hdpe",
    2: "mixed_plastic_soft",
    3: "ecal",
    4: "non_plastic",
    5: "non_plastic",
    6: "mixed_plastic_rigid",
    7: "pet_oil",
}


# COCO category IDs used in the evaluation JSON
#
# IMPORTANT:
# These correspond to the corrected evaluation taxonomy.
#
# COCO ID 1 → ecal
# COCO ID 2 → hdpe
# COCO ID 3 → mixed_plastic_rigid
# COCO ID 4 → mixed_plastic_soft
# COCO ID 5 → non_plastic
# COCO ID 6 → pet
# COCO ID 7 → pet_oil
#
# We verify this from the JSON rather than blindly assuming it.

EXPECTED_COCO_CATEGORIES = {
    1: "ecal",
    2: "hdpe",
    3: "mixed_plastic_rigid",
    4: "mixed_plastic_soft",
    5: "non_plastic",
    6: "pet",
    7: "pet_oil",
}


# ============================================================
# 4. HEADER
# ============================================================

print("=" * 80)
print("E3Y-A — YOLO11n -> MobileNet-V3-Large")
print("CORRECTED FINAL END-TO-END COCO EVALUATION")
print("=" * 80)

print()
print(f"Device: {DEVICE}")

if torch.cuda.is_available():
    print(
        f"GPU: {torch.cuda.get_device_name(0)}"
    )
    print(
        f"CUDA: {torch.version.cuda}"
    )

print()


# ============================================================
# 5. PATH CHECKS
# ============================================================

print("=" * 80)
print("CHECKING PATHS")
print("=" * 80)

paths = {
    "Dataset root": DATASET_ROOT,
    "Validation images": VAL_IMAGES,
    "Validation labels": VAL_LABELS,
    "Validation COCO JSON": VAL_COCO_JSON,
    "YOLO checkpoint": YOLO_CHECKPOINT,
    "MobileNet checkpoint": MOBILENET_CHECKPOINT,
}

for name, path in paths.items():

    print(
        f"{name:25s}: {path}"
    )

    print(
        f"{'':25s}  Exists: {path.exists()}"
    )

for name, path in paths.items():

    if not path.exists():

        raise FileNotFoundError(
            f"\n{name} not found:\n{path}"
        )


# ============================================================
# 6. CLASS MAPPING
# ============================================================

print()
print("=" * 80)
print("CLASS MAPPING")
print("=" * 80)

print()
print("Original SortWaste classes -> evaluation classes:")

for class_id in range(8):

    print(
        f"YOLO {class_id} "
        f"({ORIGINAL_NAMES[class_id]:25s}) "
        f"-> "
        f"{YOLO_TO_MOBILENET[class_id]}"
    )

print()
print("Expected COCO evaluation categories:")

for coco_id, class_name in EXPECTED_COCO_CATEGORIES.items():

    print(
        f"COCO ID {coco_id} -> {class_name}"
    )


# ============================================================
# 7. LOAD COCO VALIDATION ANNOTATIONS
# ============================================================

print()
print("=" * 80)
print("LOADING COCO VALIDATION ANNOTATIONS")
print("=" * 80)

coco_gt = COCO(
    str(VAL_COCO_JSON)
)

print(
    f"Original COCO images: "
    f"{len(coco_gt.imgs)}"
)

print(
    f"Original COCO annotations: "
    f"{len(coco_gt.anns)}"
)


# ============================================================
# 8. VERIFY AND BUILD 7-CLASS EVALUATION COCO
# ============================================================

print()
print("=" * 80)
print("VERIFYING AND BUILDING 7-CLASS COCO EVALUATION")
print("=" * 80)

# ------------------------------------------------------------
# Original COCO categories
# ------------------------------------------------------------

actual_categories = {}

for category_id, category in coco_gt.cats.items():

    actual_categories[int(category_id)] = category["name"]

    print(
        f"Original COCO ID {category_id} "
        f"-> {category['name']}"
    )

EXPECTED_ORIGINAL_COCO_CATEGORIES = {
    1: "pet",
    2: "hdpe",
    3: "mixed_plastic_soft",
    4: "ecal",
    5: "metal",
    6: "cardboard",
    7: "mixed_plastic_rigid",
    8: "pet_oil",
}

if actual_categories != EXPECTED_ORIGINAL_COCO_CATEGORIES:

    raise ValueError(
        "\nOriginal COCO category mapping does not match "
        "the expected SortWaste taxonomy.\n"
        f"Expected: {EXPECTED_ORIGINAL_COCO_CATEGORIES}\n"
        f"Actual:   {actual_categories}"
    )

print(
    "\nOriginal COCO category mapping verified."
)


# ------------------------------------------------------------
# Original COCO category ID -> E3Y-A 7-class category ID
# ------------------------------------------------------------

ORIGINAL_COCO_TO_E3Y = {

    1: 6,  # pet
    2: 2,  # hdpe
    3: 4,  # mixed_plastic_soft
    4: 1,  # ecal
    5: 5,  # metal -> non_plastic
    6: 5,  # cardboard -> non_plastic
    7: 3,  # mixed_plastic_rigid
    8: 7,  # pet_oil

}


print()
print("Original COCO -> E3Y-A evaluation mapping:")

for original_id, e3y_id in ORIGINAL_COCO_TO_E3Y.items():

    print(
        f"COCO {original_id} "
        f"({actual_categories[original_id]}) "
        f"-> "
        f"E3Y-A {e3y_id} "
        f"({EXPECTED_COCO_CATEGORIES[e3y_id]})"
    )


# ------------------------------------------------------------
# Build a new COCO dictionary in memory
# ------------------------------------------------------------

with open(
    VAL_COCO_JSON,
    "r",
    encoding="utf-8"
) as f:

    original_coco_data = json.load(f)


evaluation_coco_data = {

    "info":
        original_coco_data.get(
            "info",
            {}
        ),

    "licenses":
        original_coco_data.get(
            "licenses",
            []
        ),

    "images":
        original_coco_data["images"],

    "annotations":
        [],

    "categories": [

        {
            "id": 1,
            "name": "ecal",
            "supercategory": "waste"
        },

        {
            "id": 2,
            "name": "hdpe",
            "supercategory": "waste"
        },

        {
            "id": 3,
            "name": "mixed_plastic_rigid",
            "supercategory": "waste"
        },

        {
            "id": 4,
            "name": "mixed_plastic_soft",
            "supercategory": "waste"
        },

        {
            "id": 5,
            "name": "non_plastic",
            "supercategory": "waste"
        },

        {
            "id": 6,
            "name": "pet",
            "supercategory": "waste"
        },

        {
            "id": 7,
            "name": "pet_oil",
            "supercategory": "waste"
        },

    ],
}


# ------------------------------------------------------------
# Remap annotations
# ------------------------------------------------------------

for annotation in original_coco_data["annotations"]:

    original_category_id = int(
        annotation["category_id"]
    )

    if (
        original_category_id
        not in ORIGINAL_COCO_TO_E3Y
    ):

        continue

    new_annotation = dict(
        annotation
    )

    new_annotation["category_id"] = (
        ORIGINAL_COCO_TO_E3Y[
            original_category_id
        ]
    )

    evaluation_coco_data[
        "annotations"
    ].append(
        new_annotation
    )


# ------------------------------------------------------------
# Write temporary 7-class COCO JSON
# ------------------------------------------------------------

EVALUATION_COCO_JSON = (
    RESULTS_DIR
    / "E3Y_A_7class_validation_gt.json"
)

with open(
    EVALUATION_COCO_JSON,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        evaluation_coco_data,
        f
    )


print()
print(
    "7-class evaluation COCO JSON created:"
)

print(
    EVALUATION_COCO_JSON
)


# ------------------------------------------------------------
# Load remapped COCO GT
# ------------------------------------------------------------

coco_gt = COCO(
    str(EVALUATION_COCO_JSON)
)

print()
print(
    f"7-class COCO images: "
    f"{len(coco_gt.imgs)}"
)

print(
    f"7-class COCO annotations: "
    f"{len(coco_gt.anns)}"
)


# ------------------------------------------------------------
# Final verification
# ------------------------------------------------------------

actual_evaluation_categories = {

    int(category_id):
        category["name"]

    for category_id, category
    in coco_gt.cats.items()

}

if (
    actual_evaluation_categories
    != EXPECTED_COCO_CATEGORIES
):

    raise ValueError(
        "\n7-class evaluation COCO mapping is incorrect.\n"
        f"Expected: {EXPECTED_COCO_CATEGORIES}\n"
        f"Actual:   {actual_evaluation_categories}"
    )

print(
    "\n7-class evaluation COCO mapping verified."
)


# ============================================================
# 9. VALIDATION DATA
# ============================================================

IMAGE_EXTENSIONS = {
    ".jpg",
    ".jpeg",
    ".png",
    ".bmp",
    ".webp",
}

image_paths = sorted([
    p
    for p in VAL_IMAGES.iterdir()
    if (
        p.is_file()
        and p.suffix.lower()
        in IMAGE_EXTENSIONS
    )
])

print()
print("=" * 80)
print("VALIDATION DATA")
print("=" * 80)

print(
    f"Validation images: "
    f"{len(image_paths)}"
)


# ============================================================
# 10. BUILD COCO IMAGE LOOKUP
# ============================================================

print()
print("=" * 80)
print("BUILDING COCO IMAGE LOOKUP")
print("=" * 80)

coco_image_by_filename = {}

for image_id, image_info in coco_gt.imgs.items():

    file_name = Path(
        image_info["file_name"]
    ).name

    coco_image_by_filename[
        file_name
    ] = {
        "id": int(image_id),
        "width": image_info.get("width"),
        "height": image_info.get("height"),
    }

print(
    f"COCO image lookup entries: "
    f"{len(coco_image_by_filename)}"
)


# ============================================================
# 11. LOAD YOLO
# ============================================================

print()
print("=" * 80)
print("LOADING E1 YOLO11n")
print("=" * 80)

yolo_model = YOLO(
    str(YOLO_CHECKPOINT)
)

print(
    "YOLO loaded successfully."
)


# ============================================================
# 12. LOAD MOBILENET
# ============================================================

print()
print("=" * 80)
print("LOADING E3Y-A MOBILENET-V3-LARGE")
print("=" * 80)

mobilenet = models.mobilenet_v3_large(
    weights=None
)

mobilenet.classifier[3] = torch.nn.Linear(
    mobilenet.classifier[3].in_features,
    len(MOBILENET_CLASSES)
)

checkpoint = torch.load(
    MOBILENET_CHECKPOINT,
    map_location=DEVICE
)

if isinstance(checkpoint, dict):

    if "model_state_dict" in checkpoint:

        state_dict = checkpoint[
            "model_state_dict"
        ]

    elif "state_dict" in checkpoint:

        state_dict = checkpoint[
            "state_dict"
        ]

    else:

        state_dict = checkpoint

else:

    state_dict = checkpoint


clean_state_dict = {}

for key, value in state_dict.items():

    if key.startswith("module."):

        clean_state_dict[
            key[len("module."):]
        ] = value

    else:

        clean_state_dict[key] = value


mobilenet.load_state_dict(
    clean_state_dict
)

mobilenet = mobilenet.to(
    DEVICE
)

mobilenet.eval()

print(
    "MobileNet loaded successfully."
)


# ============================================================
# 13. MOBILENET TRANSFORM
# ============================================================

mobilenet_transform = transforms.Compose([

    transforms.Resize(
        (
            MOBILENET_IMAGE_SIZE,
            MOBILENET_IMAGE_SIZE
        )
    ),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=[
            0.485,
            0.456,
            0.406
        ],
        std=[
            0.229,
            0.224,
            0.225
        ]
    ),
])


# ============================================================
# 14. READ YOLO LABELS
# ============================================================

def read_yolo_labels(label_path):

    labels = []

    if not label_path.exists():

        return labels

    with open(
        label_path,
        "r",
        encoding="utf-8"
    ) as f:

        for line in f:

            parts = line.strip().split()

            if len(parts) < 5:
                continue

            class_id = int(parts[0])

            xc = float(parts[1])
            yc = float(parts[2])
            w = float(parts[3])
            h = float(parts[4])

            labels.append({

                "class_id": class_id,

                "xc": xc,
                "yc": yc,
                "w": w,
                "h": h,

            })

    return labels


# ============================================================
# 15. YOLO BOX → PIXEL BOX
# ============================================================

def yolo_to_pixel_box(
    label,
    image_width,
    image_height
):

    xc = label["xc"] * image_width
    yc = label["yc"] * image_height

    w = label["w"] * image_width
    h = label["h"] * image_height

    x1 = int(xc - w / 2)
    y1 = int(yc - h / 2)

    x2 = int(xc + w / 2)
    y2 = int(yc + h / 2)

    x1 = max(
        0,
        min(
            x1,
            image_width - 1
        )
    )

    y1 = max(
        0,
        min(
            y1,
            image_height - 1
        )
    )

    x2 = max(
        0,
        min(
            x2,
            image_width
        )
    )

    y2 = max(
        0,
        min(
            y2,
            image_height
        )
    )

    return [
        x1,
        y1,
        x2,
        y2
    ]


# ============================================================
# 16. IoU
# ============================================================

def calculate_iou(
    box1,
    box2
):

    x1 = max(
        box1[0],
        box2[0]
    )

    y1 = max(
        box1[1],
        box2[1]
    )

    x2 = min(
        box1[2],
        box2[2]
    )

    y2 = min(
        box1[3],
        box2[3]
    )

    intersection_width = max(
        0,
        x2 - x1
    )

    intersection_height = max(
        0,
        y2 - y1
    )

    intersection = (
        intersection_width
        *
        intersection_height
    )

    area1 = (
        max(
            0,
            box1[2] - box1[0]
        )
        *
        max(
            0,
            box1[3] - box1[1]
        )
    )

    area2 = (
        max(
            0,
            box2[2] - box2[0]
        )
        *
        max(
            0,
            box2[3] - box2[1]
        )
    )

    union = (
        area1
        + area2
        - intersection
    )

    if union <= 0:

        return 0.0

    return intersection / union


# ============================================================
# 17. PROPER ONE-TO-ONE GT MATCHING
# ============================================================

def match_prediction_to_gt(
    pred_box,
    gt_boxes,
    used_gt_indices
):

    best_iou = 0.0
    best_index = None

    for index, gt in enumerate(
        gt_boxes
    ):

        if index in used_gt_indices:
            continue

        iou = calculate_iou(
            pred_box,
            gt["box"]
        )

        if iou > best_iou:

            best_iou = iou
            best_index = index

    return best_index, best_iou


# ============================================================
# 18. STORAGE
# ============================================================

all_results = []

matched_true = []
matched_pred = []

gt_objects_total = 0
total_yolo_detections = 0
valid_mobilenet = 0
matched_predictions = 0
correct_predictions = 0

coco_predictions = []

images_without_coco_id = 0
invalid_prediction_boxes = 0


# ============================================================
# 19. START END-TO-END EVALUATION
# ============================================================

print()
print("=" * 80)
print("STARTING E3Y-A END-TO-END EVALUATION")
print("=" * 80)

print()
print(
    f"YOLO confidence threshold: "
    f"{YOLO_CONF}"
)

print(
    f"YOLO NMS IoU: "
    f"{YOLO_NMS_IOU}"
)

print()
print(
    "Final confidence formula:"
)

print(
    "YOLO confidence x "
    "MobileNet class probability"
)

print()
print(
    f"Diagnostic matching IoU threshold: "
    f"{DIAGNOSTIC_MATCH_IOU}"
)


# ============================================================
# 20. PROCESS VALIDATION IMAGES
# ============================================================

for image_index, image_path in enumerate(
    image_paths,
    start=1
):

    image = cv2.imread(
        str(image_path)
    )

    if image is None:

        print(
            f"WARNING: Could not read "
            f"{image_path}"
        )

        continue

    image_height, image_width = (
        image.shape[:2]
    )

    # --------------------------------------------------------
    # COCO IMAGE ID
    # --------------------------------------------------------

    filename = image_path.name

    coco_image_info = (
        coco_image_by_filename.get(
            filename
        )
    )

    if coco_image_info is None:

        images_without_coco_id += 1

        continue

    coco_image_id = (
        coco_image_info["id"]
    )

    # --------------------------------------------------------
    # Ground truth
    #
    # For diagnostic classification only.
    # COCO JSON remains authoritative for COCOeval.
    # --------------------------------------------------------

    label_path = (
        VAL_LABELS
        / f"{image_path.stem}.txt"
    )

    raw_gt = read_yolo_labels(
        label_path
    )

    gt_boxes = []

    for gt in raw_gt:

        if (
            gt["class_id"]
            not in YOLO_TO_MOBILENET
        ):

            continue

        pixel_box = yolo_to_pixel_box(
            gt,
            image_width,
            image_height
        )

        gt_boxes.append({

            "box": pixel_box,

            "class_id":
                gt["class_id"],

            "mobilenet_class":
                YOLO_TO_MOBILENET[
                    gt["class_id"]
                ],

        })

    gt_objects_total += len(
        gt_boxes
    )

    # --------------------------------------------------------
    # YOLO inference
    # --------------------------------------------------------

    results = yolo_model.predict(

        source=str(
            image_path
        ),

        device=DEVICE,

        verbose=False,

        conf=YOLO_CONF,

        iou=YOLO_NMS_IOU,
    )

    result = results[0]

    if (
        result.boxes is None
        or len(result.boxes) == 0
    ):

        continue

    boxes = result.boxes

    total_yolo_detections += len(
        boxes
    )

    # --------------------------------------------------------
    # Prepare MobileNet crops
    # --------------------------------------------------------

    crop_tensors = []
    crop_info = []

    for pred_index in range(
        len(boxes)
    ):

        pred_box = (
            boxes.xyxy[
                pred_index
            ]
            .cpu()
            .numpy()
        )

        yolo_class_id = int(
            boxes.cls[
                pred_index
            ].item()
        )

        yolo_confidence = float(
            boxes.conf[
                pred_index
            ].item()
        )

        if (
            yolo_class_id
            not in YOLO_TO_MOBILENET
        ):

            continue

        x1 = max(
            0,
            int(pred_box[0])
        )

        y1 = max(
            0,
            int(pred_box[1])
        )

        x2 = min(
            image_width,
            int(pred_box[2])
        )

        y2 = min(
            image_height,
            int(pred_box[3])
        )

        if (
            x2 <= x1
            or y2 <= y1
        ):

            invalid_prediction_boxes += 1

            continue

        crop = image[
            y1:y2,
            x1:x2
        ]

        if crop.size == 0:

            invalid_prediction_boxes += 1

            continue

        crop_rgb = cv2.cvtColor(
            crop,
            cv2.COLOR_BGR2RGB
        )

        pil_image = Image.fromarray(
            crop_rgb
        )

        tensor = mobilenet_transform(
            pil_image
        )

        crop_tensors.append(
            tensor
        )

        crop_info.append({

            "pred_index":
                pred_index,

            "box": [
                x1,
                y1,
                x2,
                y2
            ],

            "yolo_class_id":
                yolo_class_id,

            "yolo_class_name":
                ORIGINAL_NAMES[
                    yolo_class_id
                ],

            "yolo_confidence":
                yolo_confidence,

        })

    # --------------------------------------------------------
    # MobileNet inference
    # --------------------------------------------------------

    if len(crop_tensors) == 0:

        continue

    for batch_start in range(
        0,
        len(crop_tensors),
        BATCH_SIZE
    ):

        batch_tensors = torch.stack(
            crop_tensors[
                batch_start:
                batch_start + BATCH_SIZE
            ]
        ).to(DEVICE)

        with torch.no_grad():

            logits = mobilenet(
                batch_tensors
            )

            probabilities = (
                torch.softmax(
                    logits,
                    dim=1
                )
            )

            mobile_preds = (
                torch.argmax(
                    probabilities,
                    dim=1
                )
                .cpu()
                .numpy()
            )

            mobile_class_probabilities = (
                probabilities
                .cpu()
                .numpy()
            )

        for local_index, mobile_pred in enumerate(
            mobile_preds
        ):

            global_index = (
                batch_start
                + local_index
            )

            info = crop_info[
                global_index
            ]

            mobile_pred = int(
                mobile_pred
            )

            final_class = ID_TO_CLASS[
                mobile_pred
            ]

            mobile_class_probability = float(
                mobile_class_probabilities[
                    local_index,
                    mobile_pred
                ]
            )

            yolo_confidence = (
                info["yolo_confidence"]
            )

            # ------------------------------------------------
            # CORRECTED FINAL CONFIDENCE
            # ------------------------------------------------

            final_confidence = (
                yolo_confidence
                *
                mobile_class_probability
            )

            pred_box = info["box"]

            # ------------------------------------------------
            # COCO CATEGORY ID
            # ------------------------------------------------

            coco_category_id = (
                CLASS_TO_ID[
                    final_class
                ] + 1
            )

            # ------------------------------------------------
            # COCO PREDICTION
            #
            # COCO uses:
            # [x, y, width, height]
            # ------------------------------------------------

            x1, y1, x2, y2 = pred_box

            bbox_width = (
                x2 - x1
            )

            bbox_height = (
                y2 - y1
            )

            coco_predictions.append({

                "image_id":
                    coco_image_id,

                "category_id":
                    coco_category_id,

                "bbox": [
                    float(x1),
                    float(y1),
                    float(bbox_width),
                    float(bbox_height),
                ],

                "score":
                    float(final_confidence),

            })

            # ------------------------------------------------
            # Diagnostic one-to-one matching
            # ------------------------------------------------
            #
            # IMPORTANT:
            # Matching is done independently for each
            # prediction. To make this proper one-to-one,
            # we maintain used GTs per image.
            #
            # This section is handled below through
            # image-level matching storage.
            # ------------------------------------------------

            valid_mobilenet += 1

            all_results.append({

                "image":
                    str(image_path),

                "image_id":
                    coco_image_id,

                "pred_index":
                    info["pred_index"],

                "x1":
                    x1,

                "y1":
                    y1,

                "x2":
                    x2,

                "y2":
                    y2,

                "yolo_class_id":
                    info["yolo_class_id"],

                "yolo_class_name":
                    info["yolo_class_name"],

                "yolo_confidence":
                    yolo_confidence,

                "mobilenet_class":
                    final_class,

                "mobilenet_class_id":
                    mobile_pred,

                "mobilenet_confidence":
                    mobile_class_probability,

                "final_confidence":
                    final_confidence,

            })

    # --------------------------------------------------------
    # Progress
    # --------------------------------------------------------

    if (
        image_index % 50 == 0
        or image_index == len(
            image_paths
        )
    ):

        print(
            f"Processed "
            f"{image_index}/"
            f"{len(image_paths)} | "
            f"YOLO detections: "
            f"{total_yolo_detections:,} | "
            f"MobileNet predictions: "
            f"{valid_mobilenet:,}"
        )


# ============================================================
# 21. PRE-COCO SANITY CHECKS
# ============================================================

print()
print("=" * 80)
print("PRE-COCO SANITY CHECKS")
print("=" * 80)

print(
    f"Validation images: "
    f"{len(image_paths)}"
)

print(
    f"COCO images: "
    f"{len(coco_gt.imgs)}"
)

print(
    f"Ground-truth objects: "
    f"{len(coco_gt.anns)}"
)

print(
    f"YOLO detections: "
    f"{total_yolo_detections}"
)

print(
    f"MobileNet predictions: "
    f"{valid_mobilenet}"
)

print(
    f"COCO predictions: "
    f"{len(coco_predictions)}"
)

print(
    f"Images without COCO ID: "
    f"{images_without_coco_id}"
)

print(
    f"Invalid prediction boxes: "
    f"{invalid_prediction_boxes}"
)

if images_without_coco_id != 0:

    raise RuntimeError(
        "Some validation images could not "
        "be mapped to COCO image IDs."
    )

if len(coco_predictions) == 0:

    raise RuntimeError(
        "No COCO predictions were generated."
    )

print(
    "\nSanity checks completed."
)


# ============================================================
# 22. SAVE RAW DETECTION RESULTS
# ============================================================

print()
print("=" * 80)
print("SAVING RAW DETECTION RESULTS")
print("=" * 80)

raw_fieldnames = [

    "image",
    "image_id",

    "pred_index",

    "x1",
    "y1",
    "x2",
    "y2",

    "yolo_class_id",
    "yolo_class_name",

    "yolo_confidence",

    "mobilenet_class",
    "mobilenet_class_id",
    "mobilenet_confidence",

    "final_confidence",

]

with open(
    RAW_RESULTS_CSV,
    "w",
    newline="",
    encoding="utf-8"
) as f:

    writer = csv.DictWriter(
        f,
        fieldnames=raw_fieldnames
    )

    writer.writeheader()

    writer.writerows(
        all_results
    )

print(
    f"Raw detection results saved:\n"
    f"{RAW_RESULTS_CSV}"
)


# ============================================================
# 23. SAVE COCO PREDICTIONS JSON
# ============================================================

with open(
    COCO_PREDICTIONS_JSON,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        coco_predictions,
        f
    )

print(
    f"COCO predictions saved:\n"
    f"{COCO_PREDICTIONS_JSON}"
)


# ============================================================
# 24. COCO EVALUATION
# ============================================================

print()
print("=" * 80)
print("COCO EVALUATION")
print("=" * 80)

print(
    f"COCO predictions: "
    f"{len(coco_predictions)}"
)

coco_dt = coco_gt.loadRes(
    str(COCO_PREDICTIONS_JSON)
)

coco_eval = COCOeval(
    coco_gt,
    coco_dt,
    "bbox"
)

# Evaluate all validation images
coco_eval.params.imgIds = sorted(
    coco_gt.imgs.keys()
)

# Explicitly evaluate all 7 categories
coco_eval.params.catIds = sorted(
    EXPECTED_COCO_CATEGORIES.keys()
)

coco_eval.evaluate()
coco_eval.accumulate()
coco_eval.summarize()


# ============================================================
# 25. EXTRACT OVERALL COCO METRICS
# ============================================================

mAP50_95 = float(
    coco_eval.stats[0]
)

mAP50 = float(
    coco_eval.stats[1]
)

mAP75 = float(
    coco_eval.stats[2]
)

AR1 = float(
    coco_eval.stats[6]
)

AR10 = float(
    coco_eval.stats[7]
)

AR100 = float(
    coco_eval.stats[8]
)

print()
print("=" * 80)
print("FINAL E3Y-A COCO METRICS")
print("=" * 80)

print(
    f"mAP50-95 : "
    f"{mAP50_95:.6f}"
)

print(
    f"mAP50-95 : "
    f"{mAP50_95 * 100:.2f}%"
)

print(
    f"mAP50    : "
    f"{mAP50:.6f}"
)

print(
    f"mAP50    : "
    f"{mAP50 * 100:.2f}%"
)

print(
    f"mAP75    : "
    f"{mAP75:.6f}"
)

print(
    f"AR@1     : "
    f"{AR1:.6f}"
)

print(
    f"AR@10    : "
    f"{AR10:.6f}"
)

print(
    f"AR@100   : "
    f"{AR100:.6f}"
)


# ============================================================
# 26. PER-CLASS COCO AP
# ============================================================

print()
print("=" * 80)
print("PER-CLASS COCO AP")
print("=" * 80)

class_metrics = []

precision = coco_eval.eval["precision"]

# precision dimensions:
#
# [IoU, Recall, Category, Area, MaxDets]
#
# Category order follows coco_eval.params.catIds

for category_position, category_id in enumerate(
    coco_eval.params.catIds
):

    class_name = EXPECTED_COCO_CATEGORIES[
        int(category_id)
    ]

    # AP50
    ap50_values = precision[
        0,
        :,
        category_position,
        0,
        2
    ]

    ap50_values = ap50_values[
        ap50_values > -1
    ]

    ap50 = (
        float(np.mean(ap50_values))
        if len(ap50_values) > 0
        else 0.0
    )

    # AP50:95
    ap_values = precision[
        :,
        :,
        category_position,
        0,
        2
    ]

    ap_values = ap_values[
        ap_values > -1
    ]

    ap50_95 = (
        float(np.mean(ap_values))
        if len(ap_values) > 0
        else 0.0
    )

    gt_count = len(
        coco_gt.getAnnIds(
            catIds=[int(category_id)]
        )
    )

    pred_count = sum(
        1
        for prediction
        in coco_predictions
        if prediction[
            "category_id"
        ]
        == int(category_id)
    )

    class_metrics.append({

        "class":
            class_name,

        "COCO_category_id":
            int(category_id),

        "AP50":
            ap50,

        "AP50_95":
            ap50_95,

        "GT":
            gt_count,

        "Pred":
            pred_count,

    })

    print(
        f"{class_name:25s} "
        f"AP50={ap50:.6f} "
        f"AP50-95={ap50_95:.6f} "
        f"GT={gt_count:6d} "
        f"Pred={pred_count:6d}"
    )


# ============================================================
# 27. SAVE COCO SUMMARY
# ============================================================

with open(
    COCO_RESULTS_CSV,
    "w",
    newline="",
    encoding="utf-8"
) as f:

    writer = csv.writer(f)

    writer.writerow([
        "metric",
        "value",
    ])

    writer.writerow([
        "mAP50-95",
        mAP50_95,
    ])

    writer.writerow([
        "mAP50",
        mAP50,
    ])

    writer.writerow([
        "mAP75",
        mAP75,
    ])

    writer.writerow([
        "AR@1",
        AR1,
    ])

    writer.writerow([
        "AR@10",
        AR10,
    ])

    writer.writerow([
        "AR@100",
        AR100,
    ])


with open(
    CLASS_RESULTS_CSV,
    "w",
    newline="",
    encoding="utf-8"
) as f:

    writer = csv.DictWriter(
        f,
        fieldnames=[
            "class",
            "COCO_category_id",
            "AP50",
            "AP50_95",
            "GT",
            "Pred",
        ]
    )

    writer.writeheader()

    writer.writerows(
        class_metrics
    )


print()
print(
    f"COCO summary saved:\n"
    f"{COCO_RESULTS_CSV}"
)

print(
    f"Per-class results saved:\n"
    f"{CLASS_RESULTS_CSV}"
)


# ============================================================
# 28. PROPER ONE-TO-ONE DIAGNOSTIC MATCHING
# ============================================================
#
# This is NOT the primary detection metric.
#
# It answers:
#
# "When a YOLO+MobileNet prediction sufficiently overlaps
#  a real object, how accurately does the pipeline assign
#  the final class?"
#
# Each GT can be matched at most once per image.
# ============================================================

print()
print("=" * 80)
print("PROPER ONE-TO-ONE DIAGNOSTIC MATCHING")
print("=" * 80)

# Group predictions by image
predictions_by_image = {}

for row in all_results:

    image_key = row["image"]

    predictions_by_image.setdefault(
        image_key,
        []
    ).append(row)


for image_path in image_paths:

    image_key = str(
        image_path
    )

    gt_path = (
        VAL_LABELS
        / f"{image_path.stem}.txt"
    )

    raw_gt = read_yolo_labels(
        gt_path
    )

    image = cv2.imread(
        str(image_path)
    )

    if image is None:
        continue

    h, w = image.shape[:2]

    gt_boxes = []

    for gt in raw_gt:

        if (
            gt["class_id"]
            not in YOLO_TO_MOBILENET
        ):
            continue

        gt_boxes.append({

            "box":
                yolo_to_pixel_box(
                    gt,
                    w,
                    h
                ),

            "class":
                YOLO_TO_MOBILENET[
                    gt["class_id"]
                ],

        })

    predictions = predictions_by_image.get(
        image_key,
        []
    )

    if len(predictions) == 0:
        continue

    # Sort predictions by final confidence
    # so higher-confidence predictions get
    # first opportunity to claim GT objects.
    predictions = sorted(
        predictions,
        key=lambda x:
            float(
                x["final_confidence"]
            ),
        reverse=True
    )

    used_gt_indices = set()

    for prediction in predictions:

        pred_box = [
            float(prediction["x1"]),
            float(prediction["y1"]),
            float(prediction["x2"]),
            float(prediction["y2"]),
        ]

        gt_index, best_iou = (
            match_prediction_to_gt(
                pred_box,
                gt_boxes,
                used_gt_indices
            )
        )

        matched = (
            gt_index is not None
            and best_iou >= DIAGNOSTIC_MATCH_IOU
        )

        if not matched:
            continue

        used_gt_indices.add(
            gt_index
        )

        gt_class = gt_boxes[
            gt_index
        ]["class"]

        predicted_class = prediction[
            "mobilenet_class"
        ]

        matched_true.append(
            gt_class
        )

        matched_pred.append(
            predicted_class
        )

        matched_predictions += 1

        if predicted_class == gt_class:

            correct_predictions += 1


# ============================================================
# 29. MATCHED CLASSIFICATION METRICS
# ============================================================

if len(matched_true) > 0:

    matched_accuracy = accuracy_score(
        matched_true,
        matched_pred
    )

    matched_macro_f1 = f1_score(
        matched_true,
        matched_pred,
        labels=MOBILENET_CLASSES,
        average="macro",
        zero_division=0
    )

    matched_weighted_f1 = f1_score(
        matched_true,
        matched_pred,
        labels=MOBILENET_CLASSES,
        average="weighted",
        zero_division=0
    )

    print(
        f"Matched samples : "
        f"{len(matched_true):,}"
    )

    print(
        f"Accuracy        : "
        f"{matched_accuracy:.6f}"
    )

    print(
        f"Macro F1        : "
        f"{matched_macro_f1:.6f}"
    )

    print(
        f"Weighted F1     : "
        f"{matched_weighted_f1:.6f}"
    )

    print()
    print(
        "Classification report:"
    )

    print(
        classification_report(
            matched_true,
            matched_pred,
            labels=MOBILENET_CLASSES,
            zero_division=0,
            digits=2
        )
    )

    print(
        "Confusion matrix:"
    )

    cm = confusion_matrix(
        matched_true,
        matched_pred,
        labels=MOBILENET_CLASSES
    )

    print(
        " " * 30
        + " ".join(
            f"{c[:10]:>12s}"
            for c in MOBILENET_CLASSES
        )
    )

    for i, class_name in enumerate(
        MOBILENET_CLASSES
    ):

        print(
            f"{class_name:30s}"
            + " ".join(
                f"{value:12d}"
                for value in cm[i]
            )
        )

else:

    matched_accuracy = 0.0
    matched_macro_f1 = 0.0
    matched_weighted_f1 = 0.0

    print(
        "No matched detections."
    )


# ============================================================
# 30. SAVE MATCHED DIAGNOSTIC RESULTS
# ============================================================

matched_rows = []

# Reconstruct the diagnostic matching table
# from the matched arrays for traceability.

for true_class, predicted_class in zip(
    matched_true,
    matched_pred
):

    matched_rows.append({

        "gt_class":
            true_class,

        "predicted_class":
            predicted_class,

        "correct":
            (
                true_class
                == predicted_class
            ),

    })


with open(
    MATCHED_RESULTS_CSV,
    "w",
    newline="",
    encoding="utf-8"
) as f:

    writer = csv.DictWriter(
        f,
        fieldnames=[
            "gt_class",
            "predicted_class",
            "correct",
        ]
    )

    writer.writeheader()

    writer.writerows(
        matched_rows
    )

print(
    f"\nMatched diagnostic results saved:\n"
    f"{MATCHED_RESULTS_CSV}"
)


# ============================================================
# 31. PLASTIC-ONLY END-TO-END CLASSIFICATION
# ============================================================

PLASTIC_CLASSES = [

    "ecal",
    "hdpe",
    "mixed_plastic_rigid",
    "mixed_plastic_soft",
    "pet",
    "pet_oil",

]

plastic_true = []
plastic_pred = []

for true_class, predicted_class in zip(
    matched_true,
    matched_pred
):

    if true_class in PLASTIC_CLASSES:

        plastic_true.append(
            true_class
        )

        plastic_pred.append(
            predicted_class
        )


if len(plastic_true) > 0:

    plastic_accuracy = (
        accuracy_score(
            plastic_true,
            plastic_pred
        )
    )

    plastic_macro_f1 = (
        f1_score(
            plastic_true,
            plastic_pred,
            labels=PLASTIC_CLASSES,
            average="macro",
            zero_division=0
        )
    )

    plastic_weighted_f1 = (
        f1_score(
            plastic_true,
            plastic_pred,
            labels=PLASTIC_CLASSES,
            average="weighted",
            zero_division=0
        )
    )

else:

    plastic_accuracy = 0.0
    plastic_macro_f1 = 0.0
    plastic_weighted_f1 = 0.0


print()
print("=" * 80)
print("PLASTIC-ONLY END-TO-END CLASSIFICATION")
print("=" * 80)

print(
    f"Plastic samples     : "
    f"{len(plastic_true):,}"
)

print(
    f"Plastic accuracy    : "
    f"{plastic_accuracy:.6f}"
)

print(
    f"Plastic Macro F1    : "
    f"{plastic_macro_f1:.6f}"
)

print(
    f"Plastic Weighted F1 : "
    f"{plastic_weighted_f1:.6f}"
)


# ============================================================
# 32. FINAL SUMMARY
# ============================================================

print()
print("=" * 80)
print("E3Y-A FINAL SUMMARY")
print("=" * 80)

print()
print(
    f"Validation images       : "
    f"{len(image_paths)}"
)

print(
    f"Ground-truth objects    : "
    f"{len(coco_gt.anns):,}"
)

print(
    f"YOLO detections         : "
    f"{total_yolo_detections:,}"
)

print(
    f"MobileNet predictions   : "
    f"{valid_mobilenet:,}"
)

print(
    f"COCO predictions        : "
    f"{len(coco_predictions):,}"
)

print(
    f"Proper GT matches       : "
    f"{matched_predictions:,}"
)

print(
    f"Correct final classes   : "
    f"{correct_predictions:,}"
)

print()
print(
    f"E3Y-A mAP50             : "
    f"{mAP50:.6f}"
)

print(
    f"E3Y-A mAP50             : "
    f"{mAP50 * 100:.2f}%"
)

print()
print(
    f"E3Y-A mAP50-95         : "
    f"{mAP50_95:.6f}"
)

print(
    f"E3Y-A mAP50-95         : "
    f"{mAP50_95 * 100:.2f}%"
)

print()
print(
    f"Matched classification accuracy : "
    f"{matched_accuracy:.6f}"
)

print(
    f"Matched Macro F1                : "
    f"{matched_macro_f1:.6f}"
)

print(
    f"Matched Weighted F1             : "
    f"{matched_weighted_f1:.6f}"
)

print()
print(
    f"Plastic-only accuracy           : "
    f"{plastic_accuracy:.6f}"
)

print(
    f"Plastic-only Macro F1           : "
    f"{plastic_macro_f1:.6f}"
)

print(
    f"Plastic-only Weighted F1        : "
    f"{plastic_weighted_f1:.6f}"
)

print()
print(
    f"Raw detection CSV:\n"
    f"{RAW_RESULTS_CSV}"
)

print(
    f"\nCOCO predictions JSON:\n"
    f"{COCO_PREDICTIONS_JSON}"
)

print(
    f"\nCOCO results CSV:\n"
    f"{COCO_RESULTS_CSV}"
)

print(
    f"\nPer-class results CSV:\n"
    f"{CLASS_RESULTS_CSV}"
)

print(
    f"\nMatched classification CSV:\n"
    f"{MATCHED_RESULTS_CSV}"
)

print(
    f"\nSummary:\n"
    f"{SUMMARY_TXT}"
)


# ============================================================
# 33. SAVE TEXT SUMMARY
# ============================================================

with open(
    SUMMARY_TXT,
    "w",
    encoding="utf-8"
) as f:

    f.write(
        "E3Y-A — YOLO11n -> MobileNet-V3-Large\n"
    )

    f.write(
        "CORRECTED FINAL END-TO-END COCO EVALUATION\n\n"
    )

    f.write(
        f"YOLO confidence: "
        f"{YOLO_CONF}\n"
    )

    f.write(
        f"YOLO NMS IoU: "
        f"{YOLO_NMS_IOU}\n"
    )

    f.write(
        f"Diagnostic matching IoU: "
        f"{DIAGNOSTIC_MATCH_IOU}\n\n"
    )

    f.write(
        "Final confidence = "
        "YOLO confidence x "
        "MobileNet predicted-class probability\n\n"
    )

    f.write(
        f"Validation images: "
        f"{len(image_paths)}\n"
    )

    f.write(
        f"COCO GT objects: "
        f"{len(coco_gt.anns)}\n"
    )

    f.write(
        f"YOLO detections: "
        f"{total_yolo_detections}\n"
    )

    f.write(
        f"MobileNet predictions: "
        f"{valid_mobilenet}\n"
    )

    f.write(
        f"COCO predictions: "
        f"{len(coco_predictions)}\n"
    )

    f.write(
        f"Proper GT matches: "
        f"{matched_predictions}\n"
    )

    f.write(
        f"Correct final classes: "
        f"{correct_predictions}\n\n"
    )

    f.write(
        f"mAP50: "
        f"{mAP50:.6f}\n"
    )

    f.write(
        f"mAP50-95: "
        f"{mAP50_95:.6f}\n"
    )

    f.write(
        f"mAP75: "
        f"{mAP75:.6f}\n"
    )

    f.write(
        f"AR@1: "
        f"{AR1:.6f}\n"
    )

    f.write(
        f"AR@10: "
        f"{AR10:.6f}\n"
    )

    f.write(
        f"AR@100: "
        f"{AR100:.6f}\n\n"
    )

    f.write(
        f"Matched Accuracy: "
        f"{matched_accuracy:.6f}\n"
    )

    f.write(
        f"Matched Macro F1: "
        f"{matched_macro_f1:.6f}\n"
    )

    f.write(
        f"Matched Weighted F1: "
        f"{matched_weighted_f1:.6f}\n\n"
    )

    f.write(
        f"Plastic-only Accuracy: "
        f"{plastic_accuracy:.6f}\n"
    )

    f.write(
        f"Plastic-only Macro F1: "
        f"{plastic_macro_f1:.6f}\n"
    )

    f.write(
        f"Plastic-only Weighted F1: "
        f"{plastic_weighted_f1:.6f}\n\n"
    )

    f.write(
        "CLASS-WISE COCO RESULTS\n"
    )

    f.write(
        "-" * 80
        + "\n"
    )

    for row in class_metrics:

        f.write(
            f"{row['class']:25s} "
            f"AP50={row['AP50']:.6f} "
            f"AP50-95={row['AP50_95']:.6f} "
            f"GT={row['GT']} "
            f"Pred={row['Pred']}\n"
        )


# ============================================================
# 34. IMPORTANT METHODOLOGICAL NOTES
# ============================================================

print()
print("=" * 80)
print("IMPORTANT METHODOLOGICAL NOTES")
print("=" * 80)

print(
    "1. E3Y-A uses all YOLO validation detections "
    "above conf=0.001."
)

print(
    "2. YOLO supplies the bounding box."
)

print(
    "3. MobileNet supplies the final class."
)

print(
    "4. Final confidence = YOLO confidence x "
    "MobileNet predicted-class probability."
)

print(
    "5. COCOeval is the primary end-to-end "
    "detection evaluation."
)

print(
    "6. COCO validation JSON is the authoritative "
    "ground-truth source for COCOeval."
)

print(
    "7. COCO image IDs are taken directly from "
    "the validation COCO JSON."
)

print(
    "8. COCO categories use the corrected 7-class taxonomy."
)

print(
    "9. Metal + cardboard -> non_plastic."
)

print(
    "10. Diagnostic classification uses proper "
    "one-to-one IoU matching at IoU >= 0.50."
)

print(
    "11. No random augmentation is applied during evaluation."
)

print(
    "12. mAP50-95 is the primary COCO detection metric."
)

print(
    "13. AP50 is reported separately for comparison "
    "with detector results."
)

print("=" * 80)
print(
    "E3Y-A corrected evaluation complete."
)
print("=" * 80)

E3Y-A — YOLO11n -> MobileNet-V3-Large
CORRECTED FINAL END-TO-END COCO EVALUATION

Device: cuda
GPU: NVIDIA GeForce RTX 3050 Ti Laptop GPU
CUDA: 12.6

CHECKING PATHS
Dataset root             : C:\Users\varda\Documents\_My Computer\COMMON_space\Learning\Upgrad_EPGP_MS\upgrad_IIITB_MS_course\MS_LJMU_Material\Topic Data\SortWaste\dataset\dataset
                           Exists: True
Validation images        : C:\Users\varda\Documents\_My Computer\COMMON_space\Learning\Upgrad_EPGP_MS\upgrad_IIITB_MS_course\MS_LJMU_Material\Topic Data\SortWaste\dataset\dataset\splited_all_dataset_coco\val\images
                           Exists: True
Validation labels        : C:\Users\varda\Documents\_My Computer\COMMON_space\Learning\Upgrad_EPGP_MS\upgrad_IIITB_MS_course\MS_LJMU_Material\Topic Data\SortWaste\dataset\dataset\splited_all_dataset_coco\val\labels
                           Exists: True
Validation COCO JSON     : C:\Users\varda\Documents\_My Computer\COMMON_space\Learning\Upgrad_EPGP_MS\upgr

## E3Y - P : E1 YOLOv11 + E3P MobileNet-V3-Large (augmentation-only)

In [ ]:
### END-TO-END COCO EVALUATION on all validation crops
# E3Y-P: YOLO11n -> MobileNet-V3-Large
# CORRECTED FINAL END-TO-END COCO EVALUATION
# ============================================================
#
# Pipeline:
#
#   YOLO11n
#       |
#       | bounding box + YOLO confidence
#       v
#   Object crop
#       |
#       v
#   MobileNet-V3-Large
#       |
#       | predicted class + class probability
#       v
#   Final prediction
#
#
# OPTION B FINAL CONFIDENCE:
#
#   final_confidence =
#       YOLO_confidence * MobileNet_class_probability
#
#
# PRIMARY EVALUATION:
#
#   COCOeval
#
#   mAP50
#   mAP50-95
#   mAP75
#   AR@1
#   AR@10
#   AR@100
#
#
# DIAGNOSTIC CLASSIFICATION:
#
#   Proper one-to-one IoU matching at IoU >= 0.50
#
#   Accuracy
#   Macro F1
#   Weighted F1
#   Confusion matrix
#
#
# IMPORTANT:
#
# Original SortWaste taxonomy:
#
#   0 = pet
#   1 = hdpe
#   2 = mixed_plastic_soft
#   3 = ecal
#   4 = metal
#   5 = cardboard
#   6 = mixed_plastic_rigid
#   7 = pet_oil
#
#
# Evaluation taxonomy:
#
#   1 = ecal
#   2 = hdpe
#   3 = mixed_plastic_rigid
#   4 = mixed_plastic_soft
#   5 = non_plastic
#   6 = pet
#   7 = pet_oil
#
#
# Mapping:
#
#   metal      -> non_plastic
#   cardboard  -> non_plastic
#
# The mapping is applied consistently to:
#
#   1. COCO ground truth
#   2. YOLO detections
#   3. MobileNet predictions
#
# ============================================================


from pathlib import Path
import csv
import json
import time

import cv2
import torch
import numpy as np

from ultralytics import YOLO
from torchvision import models, transforms
from PIL import Image

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix,
)

from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval


# ============================================================
# 1. PATHS
# ============================================================

DATASET_ROOT = Path(
    r"C:\Users\varda\Documents\_My Computer\COMMON_space"
    r"\Learning\Upgrad_EPGP_MS\upgrad_IIITB_MS_course"
    r"\MS_LJMU_Material\Topic Data\SortWaste\dataset\dataset"
)

SPLIT_ROOT = (
    DATASET_ROOT
    / "splited_all_dataset_coco"
)

VAL_IMAGES = (
    SPLIT_ROOT
    / "val"
    / "images"
)

VAL_LABELS = (
    SPLIT_ROOT
    / "val"
    / "labels"
)

VAL_JSON = (
    SPLIT_ROOT
    / "val"
    / "annotations"
    / "val_coco.json"
)


# ============================================================
# YOLO CHECKPOINT
# ============================================================

YOLO_CHECKPOINT = Path(
    r"C:\Users\varda\Documents\_My Computer\COMMON_space"
    r"\Learning\Upgrad_EPGP_MS\upgrad_IIITB_MS_course"
    r"\MS_LJMU_Material\Thesis_Code"
    r"\runs\detect\runs\sortwaste\yolo11n_baseline-2"
    r"\weights\best.pt"
)


# ============================================================
# MOBILENET CHECKPOINT
# ============================================================

MOBILENET_CHECKPOINT = Path(
    r"C:\Users\varda\Documents\_My Computer\COMMON_space"
    r"\Learning\Upgrad_EPGP_MS\upgrad_IIITB_MS_course"
    r"\MS_LJMU_Material\Topic Data\SortWaste\dataset\dataset"
    r"\cropped_data_plastic"
    r"\mobilenet_v3_large_E3P_best.pth"
)


# ============================================================
# OUTPUT DIRECTORY
# ============================================================

OUTPUT_ROOT = (
    DATASET_ROOT
    / "cropped_data_plastic"
    / "mobilenet_results"
    / "E3Y_P_COCO_evaluation_corrected"
)

OUTPUT_ROOT.mkdir(
    parents=True,
    exist_ok=True
)


RAW_RESULTS_CSV = (
    OUTPUT_ROOT
    / "E3Y_P_all_detections.csv"
)

COCO_RESULTS_CSV = (
    OUTPUT_ROOT
    / "E3Y_P_COCO_results.csv"
)

CLASS_RESULTS_CSV = (
    OUTPUT_ROOT
    / "E3Y_P_class_results.csv"
)

CONFUSION_MATRIX_CSV = (
    OUTPUT_ROOT
    / "E3Y_P_confusion_matrix.csv"
)

MATCHED_RESULTS_CSV = (
    OUTPUT_ROOT
    / "E3Y_P_matched_classification.csv"
)

SUMMARY_TXT = (
    OUTPUT_ROOT
    / "E3Y_P_summary.txt"
)


# ============================================================
# 2. CONFIGURATION
# ============================================================

# ------------------------------------------------------------
# YOLO
# ------------------------------------------------------------

YOLO_CONF = 0.001
YOLO_NMS_IOU = 0.70


# ------------------------------------------------------------
# MobileNet
# ------------------------------------------------------------

IMAGE_SIZE = 224
BATCH_SIZE = 64


# ------------------------------------------------------------
# Taxonomy
# ------------------------------------------------------------

NUM_YOLO_CLASSES = 8
NUM_EVAL_CLASSES = 7


MOBILENET_CLASSES = [
    "ecal",
    "hdpe",
    "mixed_plastic_rigid",
    "mixed_plastic_soft",
    "non_plastic",
    "pet",
    "pet_oil",
]


CLASS_TO_ID = {
    name: index + 1
    for index, name in enumerate(
        MOBILENET_CLASSES
    )
}


ID_TO_CLASS = {
    index: name
    for name, index in CLASS_TO_ID.items()
}


# ------------------------------------------------------------
# Diagnostic matching
# ------------------------------------------------------------

MATCH_IOU_THRESHOLD = 0.50


# ============================================================
# 3. ORIGINAL SORTWASTE CLASS MAPPING
# ============================================================

ORIGINAL_YOLO_NAMES = {

    0: "pet",

    1: "hdpe",

    2: "mixed_plastic_soft",

    3: "ecal",

    4: "metal",

    5: "cardboard",

    6: "mixed_plastic_rigid",

    7: "pet_oil",
}


ORIGINAL_COCO_NAMES = {

    1: "pet",

    2: "hdpe",

    3: "mixed_plastic_soft",

    4: "ecal",

    5: "metal",

    6: "cardboard",

    7: "mixed_plastic_rigid",

    8: "pet_oil",
}


# ------------------------------------------------------------
# YOLO -> evaluation taxonomy
# ------------------------------------------------------------

YOLO_TO_EVAL = {

    0: "pet",

    1: "hdpe",

    2: "mixed_plastic_soft",

    3: "ecal",

    4: "non_plastic",

    5: "non_plastic",

    6: "mixed_plastic_rigid",

    7: "pet_oil",
}


# ------------------------------------------------------------
# Original COCO -> evaluation taxonomy
# ------------------------------------------------------------

COCO_TO_EVAL = {

    1: "pet",

    2: "hdpe",

    3: "mixed_plastic_soft",

    4: "ecal",

    5: "non_plastic",

    6: "non_plastic",

    7: "mixed_plastic_rigid",

    8: "pet_oil",
}


# ------------------------------------------------------------
# Evaluation COCO IDs
# ------------------------------------------------------------

EVAL_CLASS_TO_COCO_ID = {

    "ecal": 1,

    "hdpe": 2,

    "mixed_plastic_rigid": 3,

    "mixed_plastic_soft": 4,

    "non_plastic": 5,

    "pet": 6,

    "pet_oil": 7,
}


EVAL_COCO_ID_TO_CLASS = {
    value: key
    for key, value in EVAL_CLASS_TO_COCO_ID.items()
}


# ============================================================
# 4. DEVICE
# ============================================================

DEVICE = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)


print("=" * 80)
print(
    "E3Y-P — YOLO11n -> MobileNet-V3-Large"
)
print(
    "CORRECTED FINAL END-TO-END COCO EVALUATION"
)
print("=" * 80)

print(
    "\nDevice:",
    DEVICE
)


if torch.cuda.is_available():

    print(
        "GPU:",
        torch.cuda.get_device_name(0)
    )

    print(
        "CUDA:",
        torch.version.cuda
    )


# ============================================================
# 5. PATH VALIDATION
# ============================================================

print("\n" + "=" * 80)
print("CHECKING PATHS")
print("=" * 80)


paths_to_check = {

    "Dataset root":
        DATASET_ROOT,

    "Validation images":
        VAL_IMAGES,

    "Validation labels":
        VAL_LABELS,

    "Validation COCO JSON":
        VAL_JSON,

    "YOLO checkpoint":
        YOLO_CHECKPOINT,

    "MobileNet checkpoint":
        MOBILENET_CHECKPOINT,
}


for name, path in paths_to_check.items():

    print(
        f"{name:25s}: {path}"
    )

    print(
        f"{'':25s}  Exists: {path.exists()}"
    )

    if not path.exists():

        raise FileNotFoundError(
            f"\nRequired path does not exist:\n{path}"
        )


# ============================================================
# 6. CLASS MAPPING DISPLAY
# ============================================================

print("\n" + "=" * 80)
print("CLASS MAPPING")
print("=" * 80)


print(
    "\nOriginal SortWaste classes -> evaluation classes:"
)


for yolo_id in range(
    NUM_YOLO_CLASSES
):

    print(
        f"YOLO {yolo_id} "
        f"({ORIGINAL_YOLO_NAMES[yolo_id]:25s}) "
        f"-> "
        f"{YOLO_TO_EVAL[yolo_id]}"
    )


print(
    "\nEvaluation COCO categories:"
)


for class_name in MOBILENET_CLASSES:

    print(
        f"COCO ID "
        f"{EVAL_CLASS_TO_COCO_ID[class_name]} "
        f"-> "
        f"{class_name}"
    )


# ============================================================
# 7. LOAD YOLO
# ============================================================

print("\n" + "=" * 80)
print("LOADING E1 YOLO11n")
print("=" * 80)


yolo_model = YOLO(
    str(YOLO_CHECKPOINT)
)


print(
    "YOLO loaded successfully."
)


# ============================================================
# 8. LOAD MOBILENET
# ============================================================

print("\n" + "=" * 80)
print("LOADING E3P MOBILENET-V3-LARGE")
print("=" * 80)


mobilenet_model = models.mobilenet_v3_large(
    weights=None
)


in_features = (
    mobilenet_model.classifier[-1].in_features
)


mobilenet_model.classifier[-1] = (
    torch.nn.Linear(
        in_features,
        NUM_EVAL_CLASSES
    )
)


checkpoint = torch.load(
    MOBILENET_CHECKPOINT,
    map_location=DEVICE
)


if isinstance(checkpoint, dict):

    if "model_state_dict" in checkpoint:

        state_dict = (
            checkpoint["model_state_dict"]
        )

    elif "state_dict" in checkpoint:

        state_dict = (
            checkpoint["state_dict"]
        )

    else:

        state_dict = checkpoint

else:

    state_dict = checkpoint


clean_state_dict = {}


for key, value in state_dict.items():

    if key.startswith("module."):

        key = key[len("module."):]

    clean_state_dict[key] = value


missing_keys, unexpected_keys = (
    mobilenet_model.load_state_dict(
        clean_state_dict,
        strict=False
    )
)


if missing_keys:

    print(
        "\nWARNING — Missing MobileNet keys:"
    )

    for key in missing_keys:

        print(
            " ",
            key
        )


if unexpected_keys:

    print(
        "\nWARNING — Unexpected MobileNet keys:"
    )

    for key in unexpected_keys:

        print(
            " ",
            key
        )


mobilenet_model = (
    mobilenet_model.to(DEVICE)
)

mobilenet_model.eval()


print(
    "MobileNet loaded successfully."
)


# ============================================================
# 9. MOBILE NET EVALUATION TRANSFORM
# ============================================================

eval_transform = transforms.Compose(
    [

        transforms.Resize(
            (
                IMAGE_SIZE,
                IMAGE_SIZE
            )
        ),

        transforms.ToTensor(),

        transforms.Normalize(

            mean=[
                0.485,
                0.456,
                0.406
            ],

            std=[
                0.229,
                0.224,
                0.225
            ]
        ),
    ]
)


# ============================================================
# 10. IMAGE EXTENSIONS
# ============================================================

IMAGE_EXTENSIONS = {

    ".jpg",
    ".jpeg",
    ".png",
    ".bmp",
    ".webp",
}


# ============================================================
# 11. IoU
# ============================================================

def calculate_iou(
    box1,
    box2
):

    x1 = max(
        box1[0],
        box2[0]
    )

    y1 = max(
        box1[1],
        box2[1]
    )

    x2 = min(
        box1[2],
        box2[2]
    )

    y2 = min(
        box1[3],
        box2[3]
    )


    intersection_width = max(
        0.0,
        x2 - x1
    )

    intersection_height = max(
        0.0,
        y2 - y1
    )


    intersection = (
        intersection_width
        *
        intersection_height
    )


    area1 = (
        max(
            0.0,
            box1[2] - box1[0]
        )
        *
        max(
            0.0,
            box1[3] - box1[1]
        )
    )


    area2 = (
        max(
            0.0,
            box2[2] - box2[0]
        )
        *
        max(
            0.0,
            box2[3] - box2[1]
        )
    )


    union = (
        area1
        +
        area2
        -
        intersection
    )


    if union <= 0:

        return 0.0


    return (
        intersection / union
    )


# ============================================================
# 12. MOBILE NET BATCH CLASSIFICATION
# ============================================================

def classify_crops(
    crops
):

    if len(crops) == 0:

        return [], []


    tensors = []

    valid_indices = []


    for index, crop in enumerate(
        crops
    ):

        if crop is None:

            continue


        if crop.size == 0:

            continue


        crop_rgb = cv2.cvtColor(
            crop,
            cv2.COLOR_BGR2RGB
        )


        pil_image = Image.fromarray(
            crop_rgb
        )


        tensor = eval_transform(
            pil_image
        )


        tensors.append(
            tensor
        )

        valid_indices.append(
            index
        )


    if len(tensors) == 0:

        return (
            [None] * len(crops),
            [None] * len(crops)
        )


    batch = torch.stack(
        tensors
    ).to(DEVICE)


    with torch.no_grad():

        outputs = mobilenet_model(
            batch
        )


        probabilities = torch.softmax(
            outputs,
            dim=1
        )


        predictions = torch.argmax(
            probabilities,
            dim=1
        )


        prediction_probabilities = (
            probabilities[
                torch.arange(
                    len(predictions),
                    device=DEVICE
                ),
                predictions
            ]
        )


    final_predictions = (
        [None] * len(crops)
    )

    final_probabilities = (
        [None] * len(crops)
    )


    for (
        local_index,
        original_index
    ) in enumerate(
        valid_indices
    ):

        final_predictions[
            original_index
        ] = int(
            predictions[
                local_index
            ].item()
        )


        final_probabilities[
            original_index
        ] = float(
            prediction_probabilities[
                local_index
            ].item()
        )


    return (
        final_predictions,
        final_probabilities
    )


# ============================================================
# 13. LOAD COCO VALIDATION ANNOTATIONS
# ============================================================

print("\n" + "=" * 80)
print("LOADING COCO VALIDATION ANNOTATIONS")
print("=" * 80)


with open(
    VAL_JSON,
    "r",
    encoding="utf-8"
) as f:

    original_coco_data = json.load(f)


print(
    "Original COCO images:",
    len(
        original_coco_data["images"]
    )
)

print(
    "Original COCO annotations:",
    len(
        original_coco_data["annotations"]
    )
)


# ============================================================
# 14. BUILD IMAGE ID MAPPING
# ============================================================

filename_to_image_id = {}

image_id_to_filename = {}


for image_info in (
    original_coco_data["images"]
):

    image_id = int(
        image_info["id"]
    )

    file_name = Path(
        image_info["file_name"]
    ).name


    filename_to_image_id[
        file_name
    ] = image_id


    image_id_to_filename[
        image_id
    ] = file_name


# ============================================================
# 15. BUILD 7-CLASS COCO GROUND TRUTH
# ============================================================

print("\n" + "=" * 80)
print("BUILDING 7-CLASS COCO GROUND TRUTH")
print("=" * 80)


evaluation_categories = []


for class_name in MOBILENET_CLASSES:

    evaluation_categories.append(
        {
            "id":
                EVAL_CLASS_TO_COCO_ID[
                    class_name
                ],

            "name":
                class_name,

            "supercategory":
                "waste",
        }
    )


evaluation_images = []


for image_info in (
    original_coco_data["images"]
):

    evaluation_images.append(
        {
            "id":
                int(
                    image_info["id"]
                ),

            "file_name":
                image_info["file_name"],

            "width":
                int(
                    image_info["width"]
                ),

            "height":
                int(
                    image_info["height"]
                ),
        }
    )


evaluation_annotations = []


new_annotation_id = 1


for ann in (
    original_coco_data["annotations"]
):

    original_category_id = int(
        ann["category_id"]
    )


    if (
        original_category_id
        not in COCO_TO_EVAL
    ):

        continue


    evaluation_class = (
        COCO_TO_EVAL[
            original_category_id
        ]
    )


    evaluation_category_id = (
        EVAL_CLASS_TO_COCO_ID[
            evaluation_class
        ]
    )


    bbox = ann["bbox"]


    if len(bbox) != 4:

        continue


    x, y, width, height = (
        bbox
    )


    x = float(x)
    y = float(y)
    width = float(width)
    height = float(height)


    if (
        width <= 0
        or height <= 0
    ):

        continue


    evaluation_annotations.append(
        {
            "id":
                new_annotation_id,

            "image_id":
                int(
                    ann["image_id"]
                ),

            "category_id":
                int(
                    evaluation_category_id
                ),

            "bbox":
                [
                    x,
                    y,
                    width,
                    height
                ],

            "area":
                float(
                    ann.get(
                        "area",
                        width * height
                    )
                ),

            "iscrowd":
                int(
                    ann.get(
                        "iscrowd",
                        0
                    )
                ),
        }
    )


    new_annotation_id += 1


evaluation_coco_dict = {

    "images":
        evaluation_images,

    "annotations":
        evaluation_annotations,

    "categories":
        evaluation_categories,
}


coco_gt = COCO()

coco_gt.dataset = (
    evaluation_coco_dict
)

coco_gt.createIndex()


print(
    "Evaluation GT annotations:",
    len(
        evaluation_annotations
    )
)

print(
    "Evaluation classes:",
    len(
        evaluation_categories
    )
)


# ============================================================
# 16. VALIDATION IMAGES
# ============================================================

image_paths = sorted(

    [

        p

        for p in VAL_IMAGES.iterdir()

        if (
            p.is_file()
            and p.suffix.lower()
            in IMAGE_EXTENSIONS
        )
    ]
)


print("\n" + "=" * 80)
print("VALIDATION DATA")
print("=" * 80)


print(
    "Validation images:",
    len(image_paths)
)


if len(image_paths) == 0:

    raise RuntimeError(
        "No validation images found."
    )


# ------------------------------------------------------------
# IMPORTANT IMAGE COUNT CHECK
# ------------------------------------------------------------

if (
    len(image_paths)
    != len(original_coco_data["images"])
):

    print(
        "\nWARNING:"
    )

    print(
        "Number of files in validation image "
        "directory does not equal number of images "
        "in COCO JSON."
    )

    print(
        "Directory images:",
        len(image_paths)
    )

    print(
        "COCO images:",
        len(
            original_coco_data["images"]
        )
    )


# ============================================================
# 17. STORAGE
# ============================================================

all_records = []

coco_predictions = []

matched_true = []
matched_pred = []

matched_records = []


gt_object_count = 0

yolo_detection_count = 0

mobilenet_prediction_count = 0

matched_detection_count = 0

correct_final_class_count = 0

images_without_coco_id = 0

invalid_prediction_boxes = 0


# ============================================================
# 18. END-TO-END EVALUATION
# ============================================================

print("\n" + "=" * 80)
print("STARTING E3Y-P END-TO-END EVALUATION")
print("=" * 80)


print(
    "\nYOLO confidence threshold:",
    YOLO_CONF
)

print(
    "YOLO NMS IoU:",
    YOLO_NMS_IOU
)

print(
    "\nFinal confidence formula:"
)

print(
    "YOLO confidence x MobileNet class probability"
)


print(
    "\nDiagnostic matching IoU threshold:",
    MATCH_IOU_THRESHOLD
)


evaluation_start = time.time()


for (
    image_index,
    image_path
) in enumerate(
    image_paths,
    start=1
):


    # ========================================================
    # READ IMAGE
    # ========================================================

    image = cv2.imread(
        str(image_path)
    )


    if image is None:

        print(
            "WARNING: Could not read:",
            image_path
        )

        continue


    image_height, image_width = (
        image.shape[:2]
    )


    image_name = image_path.name


    # ========================================================
    # COCO IMAGE ID
    # ========================================================

    image_id = (
        filename_to_image_id.get(
            image_name
        )
    )


    if image_id is None:

        images_without_coco_id += 1

        print(
            "WARNING: Image not found "
            "in COCO annotations:",
            image_name
        )

        continue


    # ========================================================
    # GROUND TRUTH FROM COCO
    # ========================================================
    #
    # IMPORTANT:
    #
    # Primary COCO evaluation uses COCO annotations
    # directly, rather than rebuilding GT from YOLO labels.
    #
    # ========================================================

    coco_gt_annotations = (
        coco_gt.imgToAnns.get(
            image_id,
            []
        )
    )


    gt_boxes = []


    for ann in coco_gt_annotations:

        bbox = ann["bbox"]


        x = float(bbox[0])
        y = float(bbox[1])

        width = float(bbox[2])
        height = float(bbox[3])


        gt_box = [

            x,

            y,

            x + width,

            y + height,
        ]


        gt_boxes.append(
            {
                "annotation_id":
                    int(
                        ann["id"]
                    ),

                "box":
                    gt_box,

                "class_name":
                    EVAL_COCO_ID_TO_CLASS[
                        int(
                            ann["category_id"]
                        )
                    ],

                "eval_category_id":
                    int(
                        ann["category_id"]
                    ),
            }
        )


    gt_object_count += len(
        gt_boxes
    )


    # ========================================================
    # YOLO DETECTION
    # ========================================================

    results = yolo_model.predict(

        source=str(
            image_path
        ),

        device=DEVICE,

        verbose=False,

        conf=YOLO_CONF,

        iou=YOLO_NMS_IOU,
    )


    result = results[0]


    if (
        result.boxes is None
        or len(result.boxes) == 0
    ):

        if (
            image_index % 50 == 0
            or image_index
            == len(image_paths)
        ):

            print(
                f"Processed "
                f"{image_index}/"
                f"{len(image_paths)} | "
                f"YOLO detections: "
                f"{yolo_detection_count:,} | "
                f"MobileNet predictions: "
                f"{mobilenet_prediction_count:,}"
            )

        continue


    predictions = result.boxes


    yolo_detection_count += len(
        predictions
    )


    # ========================================================
    # PREPARE CROPS
    # ========================================================

    detection_records = []

    crops = []


    for pred_index in range(
        len(predictions)
    ):


        # ----------------------------------------------------
        # YOLO bbox
        # ----------------------------------------------------

        pred_box = (
            predictions.xyxy[
                pred_index
            ]
            .detach()
            .cpu()
            .numpy()
            .astype(
                float
            )
        )


        pred_class = int(
            predictions.cls[
                pred_index
            ].item()
        )


        yolo_confidence = float(
            predictions.conf[
                pred_index
            ].item()
        )


        if (
            pred_class
            not in YOLO_TO_EVAL
        ):

            continue


        # ----------------------------------------------------
        # Clip bbox
        # ----------------------------------------------------

        x1 = max(
            0.0,
            min(
                pred_box[0],
                image_width - 1
            )
        )

        y1 = max(
            0.0,
            min(
                pred_box[1],
                image_height - 1
            )
        )

        x2 = max(
            0.0,
            min(
                pred_box[2],
                image_width
            )
        )

        y2 = max(
            0.0,
            min(
                pred_box[3],
                image_height
            )
        )


        if (
            x2 <= x1
            or y2 <= y1
        ):

            invalid_prediction_boxes += 1

            continue


        # ----------------------------------------------------
        # Crop using current detection bbox
        # ----------------------------------------------------

        crop = image[
            int(y1):int(y2),
            int(x1):int(x2)
        ]


        if crop.size == 0:

            invalid_prediction_boxes += 1

            continue


        # ----------------------------------------------------
        # STORE CURRENT DETECTION BBOX
        #
        # This is the critical correction.
        #
        # ----------------------------------------------------

        detection_record = {

            "image":
                image_name,

            "source_image":
                str(
                    image_path
                ),

            "image_id":
                int(
                    image_id
                ),

            "detection_index":
                int(
                    pred_index
                ),

            "yolo_pred_class_id":
                int(
                    pred_class
                ),

            "yolo_pred_class_name":
                ORIGINAL_YOLO_NAMES[
                    pred_class
                ],

            "yolo_eval_class_name":
                YOLO_TO_EVAL[
                    pred_class
                ],

            "yolo_confidence":
                yolo_confidence,

            "x1":
                x1,

            "y1":
                y1,

            "x2":
                x2,

            "y2":
                y2,

            "bbox_width":
                x2 - x1,

            "bbox_height":
                y2 - y1,
        }


        # ----------------------------------------------------
        # Diagnostic best IoU
        #
        # This is NOT used by COCOeval.
        # ----------------------------------------------------

        best_iou = 0.0

        best_gt_index = None


        for (
            gt_index,
            gt
        ) in enumerate(
            gt_boxes
        ):

            iou = calculate_iou(

                [
                    x1,
                    y1,
                    x2,
                    y2
                ],

                gt["box"]
            )


            if iou > best_iou:

                best_iou = iou

                best_gt_index = (
                    gt_index
                )


        detection_record[
            "best_gt_iou"
        ] = best_iou


        detection_record[
            "best_gt_index"
        ] = best_gt_index


        detection_records.append(
            detection_record
        )


        crops.append(
            crop
        )


    # ========================================================
    # MOBILENET CLASSIFICATION
    # ========================================================

    predictions_mobilenet = []

    probabilities_mobilenet = []


    for start in range(

        0,

        len(crops),

        BATCH_SIZE
    ):


        batch_crops = crops[

            start:
            start + BATCH_SIZE
        ]


        (
            batch_predictions,
            batch_probabilities
        ) = classify_crops(
            batch_crops
        )


        predictions_mobilenet.extend(
            batch_predictions
        )

        probabilities_mobilenet.extend(
            batch_probabilities
        )


    # ========================================================
    # ATTACH MOBILENET RESULTS
    # ========================================================

    for (

        record,

        mobile_prediction,

        mobile_probability

    ) in zip(

        detection_records,

        predictions_mobilenet,

        probabilities_mobilenet
    ):


        if mobile_prediction is None:

            continue


        if mobile_probability is None:

            continue


        mobilenet_prediction_count += 1


        # ----------------------------------------------------
        # MobileNet prediction
        # ----------------------------------------------------

        mobile_class_name = (
            MOBILENET_CLASSES[
                mobile_prediction
            ]
        )


        mobile_probability = float(
            mobile_probability
        )


        # ----------------------------------------------------
        # OPTION B
        # ----------------------------------------------------

        yolo_confidence = float(
            record[
                "yolo_confidence"
            ]
        )


        final_confidence = (

            yolo_confidence

            *
            mobile_probability
        )


        final_eval_category_id = (
            EVAL_CLASS_TO_COCO_ID[
                mobile_class_name
            ]
        )


        # ----------------------------------------------------
        # Store MobileNet results
        # ----------------------------------------------------

        record[
            "mobilenet_pred_class_id"
        ] = int(
            mobile_prediction
        )


        record[
            "mobilenet_pred_class_name"
        ] = mobile_class_name


        record[
            "mobilenet_probability"
        ] = mobile_probability


        record[
            "final_class_name"
        ] = mobile_class_name


        record[
            "final_eval_category_id"
        ] = int(
            final_eval_category_id
        )


        record[
            "final_confidence"
        ] = float(
            final_confidence
        )


        # ====================================================
        # COCO PREDICTION
        #
        # CRITICAL:
        #
        # bbox comes from THIS record.
        #
        # We do NOT use stale x1/y1/x2/y2 variables.
        # ====================================================

        pred_x1 = float(
            record["x1"]
        )

        pred_y1 = float(
            record["y1"]
        )

        pred_x2 = float(
            record["x2"]
        )

        pred_y2 = float(
            record["y2"]
        )


        pred_width = (
            pred_x2
            - pred_x1
        )

        pred_height = (
            pred_y2
            - pred_y1
        )


        if (
            pred_width <= 0
            or pred_height <= 0
        ):

            invalid_prediction_boxes += 1

            continue


        coco_prediction = {

            "image_id":
                int(
                    image_id
                ),

            "category_id":
                int(
                    final_eval_category_id
                ),

            "bbox":
                [
                    pred_x1,
                    pred_y1,
                    pred_width,
                    pred_height
                ],

            "score":
                float(
                    final_confidence
                ),
        }


        coco_predictions.append(
            coco_prediction
        )


        # ====================================================
        # Store record
        # ====================================================

        all_records.append(
            record
        )


    # ========================================================
    # PROGRESS
    # ========================================================

    if (

        image_index % 50 == 0

        or image_index
        == len(image_paths)
    ):

        print(

            f"Processed "
            f"{image_index}/"
            f"{len(image_paths)} | "

            f"YOLO detections: "
            f"{yolo_detection_count:,} | "

            f"MobileNet predictions: "
            f"{mobilenet_prediction_count:,}"
        )


# ============================================================
# 19. SANITY CHECKS BEFORE COCO EVALUATION
# ============================================================

print("\n" + "=" * 80)
print("PRE-COCO SANITY CHECKS")
print("=" * 80)


print(
    "Validation images:",
    len(image_paths)
)

print(
    "COCO images:",
    len(
        original_coco_data["images"]
    )
)

print(
    "Ground-truth objects:",
    gt_object_count
)

print(
    "YOLO detections:",
    yolo_detection_count
)

print(
    "MobileNet predictions:",
    mobilenet_prediction_count
)

print(
    "COCO predictions:",
    len(coco_predictions)
)

print(
    "Images without COCO ID:",
    images_without_coco_id
)

print(
    "Invalid prediction boxes:",
    invalid_prediction_boxes
)


if (
    images_without_coco_id > 0
):

    raise RuntimeError(
        "Some validation images could not be "
        "mapped to COCO image IDs."
    )


if (
    len(coco_predictions)
    != mobilenet_prediction_count
):

    print(
        "\nWARNING:"
    )

    print(
        "COCO prediction count differs from "
        "MobileNet prediction count."
    )


# ------------------------------------------------------------
# Check all prediction category IDs
# ------------------------------------------------------------

invalid_category_ids = [

    pred["category_id"]

    for pred in coco_predictions

    if (
        pred["category_id"]
        not in EVAL_COCO_ID_TO_CLASS
    )
]


if invalid_category_ids:

    raise RuntimeError(
        "Invalid COCO category IDs detected."
    )


# ------------------------------------------------------------
# Check all prediction bboxes
# ------------------------------------------------------------

invalid_boxes = []


for pred in coco_predictions:

    bbox = pred["bbox"]


    if (

        len(bbox) != 4

        or bbox[2] <= 0

        or bbox[3] <= 0

    ):

        invalid_boxes.append(
            pred
        )


if invalid_boxes:

    raise RuntimeError(
        "Invalid COCO prediction bounding boxes detected."
    )


print(
    "\nSanity checks completed."
)


# ============================================================
# 20. SAVE RAW DETECTION RESULTS
# ============================================================

print("\n" + "=" * 80)
print("SAVING RAW DETECTION RESULTS")
print("=" * 80)


if all_records:

    fieldnames = list(
        all_records[0].keys()
    )


    with open(

        RAW_RESULTS_CSV,

        "w",

        newline="",

        encoding="utf-8"

    ) as f:


        writer = csv.DictWriter(

            f,

            fieldnames=fieldnames
        )


        writer.writeheader()


        writer.writerows(
            all_records
        )


    print(
        "Raw detection results saved:"
    )

    print(
        RAW_RESULTS_CSV
    )


# ============================================================
# 21. COCO EVALUATION
# ============================================================

print("\n" + "=" * 80)
print("COCO EVALUATION")
print("=" * 80)


if len(coco_predictions) == 0:

    raise RuntimeError(
        "No valid E3Y-P predictions were generated."
    )


print(
    "COCO predictions:",
    len(coco_predictions)
)


# ------------------------------------------------------------
# Load predictions
# ------------------------------------------------------------

coco_dt = coco_gt.loadRes(
    coco_predictions
)


# ------------------------------------------------------------
# COCO evaluator
# ------------------------------------------------------------

coco_eval = COCOeval(

    coco_gt,

    coco_dt,

    "bbox"
)


coco_eval.evaluate()

coco_eval.accumulate()

coco_eval.summarize()


# ============================================================
# 22. STANDARD COCO METRICS
# ============================================================

map5095 = float(
    coco_eval.stats[0]
)

map50 = float(
    coco_eval.stats[1]
)

map75 = float(
    coco_eval.stats[2]
)

mar1 = float(
    coco_eval.stats[6]
)

mar10 = float(
    coco_eval.stats[7]
)

mar100 = float(
    coco_eval.stats[8]
)


print("\n" + "=" * 80)
print("FINAL E3Y-P COCO METRICS")
print("=" * 80)


print(
    f"mAP50-95 : {map5095:.6f}"
)

print(
    f"mAP50-95 : {map5095 * 100:.2f}%"
)


print(
    f"mAP50    : {map50:.6f}"
)

print(
    f"mAP50    : {map50 * 100:.2f}%"
)


print(
    f"mAP75    : {map75:.6f}"
)


print(
    f"AR@1     : {mar1:.6f}"
)

print(
    f"AR@10    : {mar10:.6f}"
)

print(
    f"AR@100   : {mar100:.6f}"
)


# ============================================================
# 23. PER-CLASS COCO AP
# ============================================================

print("\n" + "=" * 80)
print("PER-CLASS COCO AP")
print("=" * 80)


precision_tensor = (
    coco_eval.eval[
        "precision"
    ]
)


class_results = []


for class_index, category_id in enumerate(
    coco_eval.params.catIds
):


    class_name = (
        EVAL_COCO_ID_TO_CLASS[
            category_id
        ]
    )


    # --------------------------------------------------------
    # AP50
    # --------------------------------------------------------

    precision_ap50 = (
        precision_tensor[
            0,
            :,
            class_index,
            0,
            -1
        ]
    )


    precision_ap50 = (
        precision_ap50[
            precision_ap50 > -1
        ]
    )


    if len(precision_ap50) > 0:

        class_ap50 = float(
            np.mean(
                precision_ap50
            )
        )

    else:

        class_ap50 = 0.0


    # --------------------------------------------------------
    # AP50-95
    # --------------------------------------------------------

    precision_ap5095 = (
        precision_tensor[
            :,
            :,
            class_index,
            0,
            -1
        ]
    )


    precision_ap5095 = (
        precision_ap5095[
            precision_ap5095 > -1
        ]
    )


    if len(precision_ap5095) > 0:

        class_ap5095 = float(
            np.mean(
                precision_ap5095
            )
        )

    else:

        class_ap5095 = 0.0


    # --------------------------------------------------------
    # GT count
    # --------------------------------------------------------

    gt_count = sum(

        1

        for ann
        in evaluation_annotations

        if (
            ann["category_id"]
            == category_id
        )
    )


    # --------------------------------------------------------
    # Prediction count
    # --------------------------------------------------------

    prediction_count = sum(

        1

        for pred
        in coco_predictions

        if (
            pred["category_id"]
            == category_id
        )
    )


    class_results.append(

        {

            "class":
                class_name,

            "COCO_category_id":
                category_id,

            "AP50":
                class_ap50,

            "AP50_95":
                class_ap5095,

            "GT_count":
                gt_count,

            "predictions":
                prediction_count,
        }
    )


    print(

        f"{class_name:25s} "

        f"AP50={class_ap50:.6f} "

        f"AP50-95={class_ap5095:.6f} "

        f"GT={gt_count:6d} "

        f"Pred={prediction_count:6d}"
    )


# ============================================================
# 24. SAVE COCO RESULTS
# ============================================================

with open(

    COCO_RESULTS_CSV,

    "w",

    newline="",

    encoding="utf-8"

) as f:


    writer = csv.writer(
        f
    )


    writer.writerow(
        [
            "metric",
            "value"
        ]
    )


    writer.writerow(
        [
            "mAP50",
            map50
        ]
    )


    writer.writerow(
        [
            "mAP50_95",
            map5095
        ]
    )


    writer.writerow(
        [
            "mAP75",
            map75
        ]
    )


    writer.writerow(
        [
            "AR@1",
            mar1
        ]
    )


    writer.writerow(
        [
            "AR@10",
            mar10
        ]
    )


    writer.writerow(
        [
            "AR@100",
            mar100
        ]
    )


print(
    "\nCOCO summary saved:"
)

print(
    COCO_RESULTS_CSV
)


# ============================================================
# 25. SAVE PER-CLASS RESULTS
# ============================================================

with open(

    CLASS_RESULTS_CSV,

    "w",

    newline="",

    encoding="utf-8"

) as f:


    writer = csv.DictWriter(

        f,

        fieldnames=[

            "class",

            "COCO_category_id",

            "AP50",

            "AP50_95",

            "GT_count",

            "predictions",
        ]
    )


    writer.writeheader()


    writer.writerows(
        class_results
    )


print(
    "Per-class results saved:"
)

print(
    CLASS_RESULTS_CSV
)


# ============================================================
# 26. PROPER ONE-TO-ONE MATCHING
# ============================================================
#
# This is ONLY for diagnostic classification metrics.
#
# COCOeval remains the authoritative detection evaluation.
#
# Matching strategy:
#
#   1. Sort predictions by final confidence descending.
#   2. For each prediction:
#        - find unmatched GT with highest IoU
#        - require IoU >= 0.50
#   3. Once a GT is matched, it cannot be matched again.
#
# This prevents:
#
#   one GT -> many predictions
#
# which caused the previous:
#
#   30,030 matched predictions
#   vs
#   13,065 GT objects
#
# ============================================================

print("\n" + "=" * 80)
print("PROPER ONE-TO-ONE DIAGNOSTIC MATCHING")
print("=" * 80)


# ------------------------------------------------------------
# Group detection records by image
# ------------------------------------------------------------

records_by_image = {}


for record in all_records:

    image_id = int(
        record["image_id"]
    )


    records_by_image.setdefault(
        image_id,
        []
    ).append(
        record
    )


# ------------------------------------------------------------
# Build GT by image
# ------------------------------------------------------------

gt_by_image = {}


for gt in evaluation_annotations:

    image_id = int(
        gt["image_id"]
    )


    bbox = gt["bbox"]


    gt_box = [

        float(bbox[0]),

        float(bbox[1]),

        float(
            bbox[0] + bbox[2]
        ),

        float(
            bbox[1] + bbox[3]
        ),
    ]


    gt_record = {

        "annotation_id":
            int(
                gt["id"]
            ),

        "box":
            gt_box,

        "category_id":
            int(
                gt["category_id"]
            ),

        "class_name":
            EVAL_COCO_ID_TO_CLASS[
                int(
                    gt["category_id"]
                )
            ],
    }


    gt_by_image.setdefault(
        image_id,
        []
    ).append(
        gt_record
    )


# ------------------------------------------------------------
# Match
# ------------------------------------------------------------

for image_id in (
    sorted(
        set(
            list(
                records_by_image.keys()
            )
            +
            list(
                gt_by_image.keys()
            )
        )
    )
):


    image_records = (
        records_by_image.get(
            image_id,
            []
        )
    )


    image_gt = (
        gt_by_image.get(
            image_id,
            []
        )
    )


    # Sort by confidence
    image_records = sorted(

        image_records,

        key=lambda r:
            float(
                r[
                    "final_confidence"
                ]
            ),

        reverse=True
    )


    matched_gt_indices = set()


    for record in image_records:


        pred_box = [

            float(
                record["x1"]
            ),

            float(
                record["y1"]
            ),

            float(
                record["x2"]
            ),

            float(
                record["y2"]
            ),
        ]


        best_iou = 0.0

        best_gt_index = None


        for gt_index, gt in enumerate(
            image_gt
        ):

            if gt_index in matched_gt_indices:

                continue


            iou = calculate_iou(

                pred_box,

                gt["box"]
            )


            if iou > best_iou:

                best_iou = iou

                best_gt_index = (
                    gt_index
                )


        # ----------------------------------------------------
        # Successful one-to-one match
        # ----------------------------------------------------

        if (

            best_gt_index is not None

            and best_iou
            >= MATCH_IOU_THRESHOLD

        ):


            matched_gt_indices.add(
                best_gt_index
            )


            gt = image_gt[
                best_gt_index
            ]


            true_class_id = int(
                gt["category_id"]
            )


            pred_class_id = int(
                record[
                    "final_eval_category_id"
                ]
            )


            matched_true.append(
                true_class_id
            )


            matched_pred.append(
                pred_class_id
            )


            matched_record = {

                "image":
                    record["image"],

                "image_id":
                    image_id,

                "detection_index":
                    record[
                        "detection_index"
                    ],

                "IoU":
                    best_iou,

                "GT_class":
                    gt[
                        "class_name"
                    ],

                "Predicted_class":
                    record[
                        "final_class_name"
                    ],

                "YOLO_confidence":
                    record[
                        "yolo_confidence"
                    ],

                "MobileNet_probability":
                    record[
                        "mobilenet_probability"
                    ],

                "Final_confidence":
                    record[
                        "final_confidence"
                    ],

                "Correct":
                    (
                        true_class_id
                        ==
                        pred_class_id
                    ),
            }


            matched_records.append(
                matched_record
            )


# ============================================================
# 27. MATCHED CLASSIFICATION METRICS
# ============================================================

matched_accuracy = 0.0

matched_macro_f1 = 0.0

matched_weighted_f1 = 0.0


if len(matched_true) > 0:


    matched_accuracy = (
        accuracy_score(
            matched_true,
            matched_pred
        )
    )


    matched_macro_f1 = (
        f1_score(

            matched_true,

            matched_pred,

            labels=list(
                range(
                    1,
                    NUM_EVAL_CLASSES + 1
                )
            ),

            average="macro",

            zero_division=0
        )
    )


    matched_weighted_f1 = (
        f1_score(

            matched_true,

            matched_pred,

            labels=list(
                range(
                    1,
                    NUM_EVAL_CLASSES + 1
                )
            ),

            average="weighted",

            zero_division=0
        )
    )


    print(
        f"Matched samples : "
        f"{len(matched_true):,}"
    )


    print(
        f"Accuracy        : "
        f"{matched_accuracy:.6f}"
    )


    print(
        f"Macro F1        : "
        f"{matched_macro_f1:.6f}"
    )


    print(
        f"Weighted F1     : "
        f"{matched_weighted_f1:.6f}"
    )


    print(
        "\nClassification report:"
    )


    report = classification_report(

        matched_true,

        matched_pred,

        labels=list(
            range(
                1,
                NUM_EVAL_CLASSES + 1
            )
        ),

        target_names=
            MOBILENET_CLASSES,

        zero_division=0
    )


    print(
        report
    )


    # --------------------------------------------------------
    # Confusion matrix
    # --------------------------------------------------------

    cm = confusion_matrix(

        matched_true,

        matched_pred,

        labels=list(
            range(
                1,
                NUM_EVAL_CLASSES + 1
            )
        )
    )


    print(
        "\nConfusion matrix:"
    )


    print(
        " " * 25,

        *[
            f"{name:20s}"

            for name
            in MOBILENET_CLASSES
        ]
    )


    for i, row in enumerate(cm):

        print(

            f"{MOBILENET_CLASSES[i]:25s}",

            *[
                f"{value:20d}"

                for value
                in row
            ]
        )


    # --------------------------------------------------------
    # Save confusion matrix
    # --------------------------------------------------------

    with open(

        CONFUSION_MATRIX_CSV,

        "w",

        newline="",

        encoding="utf-8"

    ) as f:


        writer = csv.writer(
            f
        )


        writer.writerow(

            [
                "actual \\ predicted"
            ]

            +
            MOBILENET_CLASSES
        )


        for i, row in enumerate(cm):

            writer.writerow(

                [
                    MOBILENET_CLASSES[i]
                ]

                +
                list(row)
            )


# ============================================================
# 28. SAVE MATCHED RESULTS
# ============================================================

if matched_records:

    with open(

        MATCHED_RESULTS_CSV,

        "w",

        newline="",

        encoding="utf-8"

    ) as f:


        fieldnames = list(
            matched_records[0].keys()
        )


        writer = csv.DictWriter(

            f,

            fieldnames=fieldnames
        )


        writer.writeheader()


        writer.writerows(
            matched_records
        )


    print(
        "\nMatched diagnostic results saved:"
    )

    print(
        MATCHED_RESULTS_CSV
    )


# ============================================================
# 29. PLASTIC-ONLY CLASSIFICATION
# ============================================================

print("\n" + "=" * 80)
print("PLASTIC-ONLY END-TO-END CLASSIFICATION")
print("=" * 80)


plastic_class_ids = {

    CLASS_TO_ID["ecal"],

    CLASS_TO_ID["hdpe"],

    CLASS_TO_ID[
        "mixed_plastic_rigid"
    ],

    CLASS_TO_ID[
        "mixed_plastic_soft"
    ],

    CLASS_TO_ID["pet"],

    CLASS_TO_ID["pet_oil"],
}


plastic_true = []

plastic_pred = []


for true, pred in zip(

    matched_true,

    matched_pred

):


    if true in plastic_class_ids:

        plastic_true.append(
            true
        )

        plastic_pred.append(
            pred
        )


plastic_accuracy = 0.0

plastic_macro_f1 = 0.0

plastic_weighted_f1 = 0.0


if len(plastic_true) > 0:


    plastic_accuracy = (
        accuracy_score(

            plastic_true,

            plastic_pred
        )
    )


    plastic_macro_f1 = (
        f1_score(

            plastic_true,

            plastic_pred,

            labels=sorted(
                plastic_class_ids
            ),

            average="macro",

            zero_division=0
        )
    )


    plastic_weighted_f1 = (
        f1_score(

            plastic_true,

            plastic_pred,

            labels=sorted(
                plastic_class_ids
            ),

            average="weighted",

            zero_division=0
        )
    )


    print(
        f"Plastic samples     : "
        f"{len(plastic_true):,}"
    )


    print(
        f"Plastic accuracy    : "
        f"{plastic_accuracy:.6f}"
    )


    print(
        f"Plastic Macro F1    : "
        f"{plastic_macro_f1:.6f}"
    )


    print(
        f"Plastic Weighted F1 : "
        f"{plastic_weighted_f1:.6f}"
    )


# ============================================================
# 30. FINAL COUNTS
# ============================================================

evaluation_time = (
    time.time()
    -
    evaluation_start
)


print("\n" + "=" * 80)
print("E3Y-P FINAL SUMMARY")
print("=" * 80)


print(
    f"\nValidation images       : "
    f"{len(image_paths):,}"
)


print(
    f"Ground-truth objects    : "
    f"{gt_object_count:,}"
)


print(
    f"YOLO detections         : "
    f"{yolo_detection_count:,}"
)


print(
    f"MobileNet predictions   : "
    f"{mobilenet_prediction_count:,}"
)


print(
    f"COCO predictions        : "
    f"{len(coco_predictions):,}"
)


print(
    f"Proper GT matches       : "
    f"{len(matched_true):,}"
)


print(f"Correct final classes: {sum(1 for t, p in zip(matched_true, matched_pred) if t == p):,}")



print(
    f"\nE3Y-P mAP50             : "
    f"{map50:.6f}"
)


print(
    f"E3Y-P mAP50             : "
    f"{map50 * 100:.2f}%"
)


print(
    f"\nE3Y-P mAP50-95          : "
    f"{map5095:.6f}"
)


print(
    f"E3Y-P mAP50-95          : "
    f"{map5095 * 100:.2f}%"
)


if len(matched_true) > 0:

    print(
        f"\nMatched classification accuracy : "
        f"{matched_accuracy:.6f}"
    )

    print(
        f"Matched Macro F1                : "
        f"{matched_macro_f1:.6f}"
    )

    print(
        f"Matched Weighted F1             : "
        f"{matched_weighted_f1:.6f}"
    )


if len(plastic_true) > 0:

    print(
        f"\nPlastic-only accuracy           : "
        f"{plastic_accuracy:.6f}"
    )

    print(
        f"Plastic-only Macro F1           : "
        f"{plastic_macro_f1:.6f}"
    )

    print(
        f"Plastic-only Weighted F1        : "
        f"{plastic_weighted_f1:.6f}"
    )


print(
    f"\nEvaluation time        : "
    f"{evaluation_time / 60:.2f} minutes"
)


print(
    "\nResults directory:"
)

print(
    OUTPUT_ROOT
)


# ============================================================
# 31. SAVE FINAL SUMMARY
# ============================================================

with open(

    SUMMARY_TXT,

    "w",

    encoding="utf-8"

) as f:


    f.write(
        "E3Y-P — CORRECTED FINAL END-TO-END "
        "COCO EVALUATION\n"
    )

    f.write(
        "=" * 70
        +
        "\n\n"
    )


    f.write(
        "PIPELINE\n"
    )

    f.write(
        "YOLO11n -> object crop -> "
        "MobileNet-V3-Large\n\n"
    )


    f.write(
        "OPTION B CONFIDENCE\n"
    )

    f.write(
        "Final confidence = "
        "YOLO confidence x "
        "MobileNet predicted-class probability\n\n"
    )


    f.write(
        "EVALUATION TAXONOMY\n"
    )

    f.write(
        "Original 8 SortWaste classes -> "
        "7 evaluation classes\n"
    )

    f.write(
        "Metal + cardboard -> non_plastic\n\n"
    )


    f.write(
        "DATASET\n"
    )

    f.write(
        f"Validation images: "
        f"{len(image_paths)}\n"
    )

    f.write(
        f"Ground truth objects: "
        f"{gt_object_count}\n"
    )


    f.write(
        "\nPREDICTIONS\n"
    )

    f.write(
        f"YOLO detections: "
        f"{yolo_detection_count}\n"
    )

    f.write(
        f"MobileNet predictions: "
        f"{mobilenet_prediction_count}\n"
    )

    f.write(
        f"COCO predictions: "
        f"{len(coco_predictions)}\n"
    )


    f.write(
        "\nPRIMARY COCO DETECTION METRICS\n"
    )

    f.write(
        f"mAP50: "
        f"{map50:.6f}\n"
    )

    f.write(
        f"mAP50-95: "
        f"{map5095:.6f}\n"
    )

    f.write(
        f"mAP75: "
        f"{map75:.6f}\n"
    )

    f.write(
        f"AR@1: "
        f"{mar1:.6f}\n"
    )

    f.write(
        f"AR@10: "
        f"{mar10:.6f}\n"
    )

    f.write(
        f"AR@100: "
        f"{mar100:.6f}\n"
    )


    f.write(
        "\nDIAGNOSTIC ONE-TO-ONE CLASSIFICATION\n"
    )

    f.write(
        f"IoU threshold: "
        f"{MATCH_IOU_THRESHOLD:.2f}\n"
    )

    f.write(
        f"Matched samples: "
        f"{len(matched_true)}\n"
    )

    f.write(
        f"Accuracy: "
        f"{matched_accuracy:.6f}\n"
    )

    f.write(
        f"Macro F1: "
        f"{matched_macro_f1:.6f}\n"
    )

    f.write(
        f"Weighted F1: "
        f"{matched_weighted_f1:.6f}\n"
    )


    f.write(
        "\nPLASTIC-ONLY CLASSIFICATION\n"
    )

    f.write(
        f"Plastic samples: "
        f"{len(plastic_true)}\n"
    )

    f.write(
        f"Plastic accuracy: "
        f"{plastic_accuracy:.6f}\n"
    )

    f.write(
        f"Plastic Macro F1: "
        f"{plastic_macro_f1:.6f}\n"
    )

    f.write(
        f"Plastic Weighted F1: "
        f"{plastic_weighted_f1:.6f}\n"
    )


    f.write(
        "\nSANITY CHECKS\n"
    )

    f.write(
        f"Images without COCO ID: "
        f"{images_without_coco_id}\n"
    )

    f.write(
        f"Invalid prediction boxes: "
        f"{invalid_prediction_boxes}\n"
    )


    f.write(
        "\nMETHODOLOGICAL NOTES\n"
    )

    f.write(
        "1. YOLO provides the bounding boxes.\n"
    )

    f.write(
        "2. MobileNet provides the final class.\n"
    )

    f.write(
        "3. Final confidence uses Option B.\n"
    )

    f.write(
        "4. COCO annotations are used as the "
        "primary ground truth source.\n"
    )

    f.write(
        "5. Original 8-class COCO annotations "
        "are remapped to the 7-class evaluation "
        "taxonomy.\n"
    )

    f.write(
        "6. Metal and cardboard are mapped to "
        "non_plastic.\n"
    )

    f.write(
        "7. No random augmentation is used "
        "during evaluation.\n"
    )

    f.write(
        "8. COCOeval is the primary end-to-end "
        "detection evaluation.\n"
    )

    f.write(
        "9. Classification metrics use "
        "one-to-one IoU matching and are "
        "diagnostic.\n"
    )

    f.write(
        "10. Each GT object can be matched to "
        "at most one prediction.\n"
    )


# ============================================================
# 32. FINAL
# ============================================================

print("\n" + "=" * 80)
print("IMPORTANT METHODOLOGICAL NOTES")
print("=" * 80)


print(
    "\n1. E3Y-P uses all YOLO validation detections "
    "above conf=0.001."
)


print(
    "2. YOLO supplies the bounding box."
)


print(
    "3. MobileNet supplies the final class."
)


print(
    "4. Final confidence = YOLO confidence "
    "x MobileNet predicted-class probability."
)


print(
    "5. COCOeval is the primary end-to-end "
    "detection evaluation."
)


print(
    "6. COCO GT is remapped from 8 original "
    "classes to 7 evaluation classes."
)


print(
    "7. Metal + cardboard -> non_plastic."
)


print(
    "8. Diagnostic classification uses proper "
    "one-to-one IoU matching."
)


print(
    "9. No random augmentation is applied "
    "during evaluation."
)


print(
    "\nE3Y-P corrected evaluation complete."
)


print("=" * 80)

E3Y-P — YOLO11n -> MobileNet-V3-Large
CORRECTED FINAL END-TO-END COCO EVALUATION

Device: cuda
GPU: NVIDIA GeForce RTX 3050 Ti Laptop GPU
CUDA: 12.6

CHECKING PATHS
Dataset root             : C:\Users\varda\Documents\_My Computer\COMMON_space\Learning\Upgrad_EPGP_MS\upgrad_IIITB_MS_course\MS_LJMU_Material\Topic Data\SortWaste\dataset\dataset
                           Exists: True
Validation images        : C:\Users\varda\Documents\_My Computer\COMMON_space\Learning\Upgrad_EPGP_MS\upgrad_IIITB_MS_course\MS_LJMU_Material\Topic Data\SortWaste\dataset\dataset\splited_all_dataset_coco\val\images
                           Exists: True
Validation labels        : C:\Users\varda\Documents\_My Computer\COMMON_space\Learning\Upgrad_EPGP_MS\upgrad_IIITB_MS_course\MS_LJMU_Material\Topic Data\SortWaste\dataset\dataset\splited_all_dataset_coco\val\labels
                           Exists: True
Validation COCO JSON     : C:\Users\varda\Documents\_My Computer\COMMON_space\Learning\Upgrad_EPGP_MS\upgr

## E3Y-B : E1 YOLO + Train Mobilenet on Yolo detections + class imbalance handling = (E3Y-A + class imbalance handling)
Crop retained only when YOLO prediction matched a GT object at IoU ≥ 0.5

In [ ]:
#### Training the model
# E3Y-B — YOLO11n → MobileNet-V3-Large + Class-Imbalance Handling

# =============================================================================
# E3Y-B — YOLO11n -> MobileNet-V3-Large + CLASS-WEIGHTED LOSS
# =============================================================================
#
# Experiment:
#   YOLO11n Epoch-18 matched crops -> MobileNet-V3-Large
#   + class-weighted CrossEntropyLoss
#
# Purpose:
#   Evaluate whether class-imbalance handling improves the second-stage
#   MobileNet-V3-Large classifier while keeping the dataset, augmentation,
#   architecture and training procedure identical to E3Y-A.
#
# IMPORTANT EXPERIMENTAL DESIGN:
#
#   E3Y-B = E3Y-A + CLASS-WEIGHTED LOSS
#
#   SAME:
#       - YOLO11n detector
#       - YOLO Epoch-18 detections
#       - GT matching IoU >= 0.5
#       - Training crops
#       - Validation crops
#       - MobileNet-V3-Large
#       - Image size
#       - Standard augmentation
#       - Optimizer
#       - Learning rate
#       - Weight decay
#       - Scheduler
#       - Early stopping
#       - Random seed
#
#   ONLY CHANGE:
#       - CrossEntropyLoss -> weighted CrossEntropyLoss
#
# NO:
#   - Oversampling
#   - SMOTE
#   - Focal loss
#   - Class-specific augmentation
#   - Synthetic data generation
#
# Class weights:
#   Inverse-frequency weighting calculated ONLY from the training crops.
#
# Best model:
#   Selected using VALIDATION MACRO F1.
#
# =============================================================================


# =============================================================================
# 1. IMPORTS
# =============================================================================

import os
import random
import json
import time
import copy

import numpy as np
import pandas as pd

from pathlib import Path

import torch
import torch.nn as nn
import torch.optim as optim

from torch.utils.data import Dataset, DataLoader

from torchvision import models, transforms

from PIL import Image

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    classification_report,
    confusion_matrix
)

import matplotlib.pyplot as plt


# =============================================================================
# 2. CONFIGURATION
# =============================================================================

SEED = 42


# -----------------------------------------------------------------------------
# Dataset
# -----------------------------------------------------------------------------

DATASET_ROOT = Path(
    r"C:\Users\varda\Documents\_My Computer\COMMON_space\Learning\Upgrad_EPGP_MS"
    r"\upgrad_IIITB_MS_course\MS_LJMU_Material\Topic Data\SortWaste\dataset\dataset"
)

# IMPORTANT:
# This MUST be the SAME matched-crop dataset used by E3Y-A.

E3Y_ROOT = DATASET_ROOT / "yolo_mobilenet_crops_E3Y"

TRAIN_DIR = E3Y_ROOT / "train"

VAL_DIR = E3Y_ROOT / "val"

TRAIN_METADATA = TRAIN_DIR / "metadata.csv"

VAL_METADATA = VAL_DIR / "metadata.csv"


# -----------------------------------------------------------------------------
# Results
# -----------------------------------------------------------------------------

RESULTS_DIR = (
    E3Y_ROOT
    / "mobilenet_results"
    / "E3Y_B_class_weighted"
)

RESULTS_DIR.mkdir(
    parents=True,
    exist_ok=True
)


# -----------------------------------------------------------------------------
# Classes
# -----------------------------------------------------------------------------

CLASS_NAMES = [
    "ecal",
    "hdpe",
    "mixed_plastic_rigid",
    "mixed_plastic_soft",
    "non_plastic",
    "pet",
    "pet_oil"
]

NUM_CLASSES = len(CLASS_NAMES)

CLASS_TO_IDX = {
    name: idx
    for idx, name in enumerate(CLASS_NAMES)
}


# -----------------------------------------------------------------------------
# Training
# -----------------------------------------------------------------------------

IMAGE_SIZE = 224

BATCH_SIZE = 64

NUM_EPOCHS = 30

LEARNING_RATE = 1e-4

WEIGHT_DECAY = 1e-4

NUM_WORKERS = 0

PATIENCE = 7

MIN_DELTA = 1e-4


# =============================================================================
# 3. REPRODUCIBILITY
# =============================================================================

def set_seed(seed=42):

    random.seed(seed)

    np.random.seed(seed)

    torch.manual_seed(seed)

    if torch.cuda.is_available():

        torch.cuda.manual_seed(seed)

        torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True

    torch.backends.cudnn.benchmark = False


set_seed(SEED)


# =============================================================================
# 4. DEVICE
# =============================================================================

DEVICE = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)


print("=" * 80)
print("E3Y-B — YOLO11n -> MobileNet-V3-Large")
print("CLASS-WEIGHTED LOSS")
print("=" * 80)

print(f"Device       : {DEVICE}")

if torch.cuda.is_available():

    print(
        f"GPU          : "
        f"{torch.cuda.get_device_name(0)}"
    )

    print(
        f"CUDA         : "
        f"{torch.version.cuda}"
    )

print(f"Classes      : {NUM_CLASSES}")

print(f"Batch size   : {BATCH_SIZE}")

print(f"Epochs       : {NUM_EPOCHS}")

print(f"Learning rate: {LEARNING_RATE}")

print(f"Weight decay : {WEIGHT_DECAY}")

print(f"Seed         : {SEED}")

print()


# =============================================================================
# 5. EXPERIMENT DEFINITION
# =============================================================================

print("=" * 80)
print("EXPERIMENT DEFINITION")
print("=" * 80)

print()
print("E3Y-B = E3Y-A + CLASS-WEIGHTED CROSS-ENTROPY LOSS")
print()

print("YOLO detector          : YOLO11n")

print("Detector checkpoint    : Epoch-18 matched detections")

print("Crop matching          : GT IoU >= 0.5")

print("Class balancing        : CLASS WEIGHTS")

print("Oversampling            : NO")

print("SMOTE                   : NO")

print("Focal loss              : NO")

print("Class-specific augment. : NO")

print()


# =============================================================================
# 6. CHECK PATHS
# =============================================================================

print("=" * 80)
print("CHECKING DATASET")
print("=" * 80)


required_paths = [

    E3Y_ROOT,

    TRAIN_DIR,

    VAL_DIR,

    TRAIN_METADATA,

    VAL_METADATA

]


for path in required_paths:

    print(
        f"{str(path):<100} "
        f"Exists: {path.exists()}"
    )

    if not path.exists():

        raise FileNotFoundError(
            f"Required path does not exist:\n{path}"
        )


print()


# =============================================================================
# 7. LOAD METADATA
# =============================================================================

print("=" * 80)
print("LOADING METADATA")
print("=" * 80)


train_df = pd.read_csv(
    TRAIN_METADATA
)

val_df = pd.read_csv(
    VAL_METADATA
)


print(
    f"Training metadata rows  : "
    f"{len(train_df)}"
)

print(
    f"Validation metadata rows: "
    f"{len(val_df)}"
)

print()


# =============================================================================
# 8. IDENTIFY METADATA COLUMNS
# =============================================================================

def find_column(
    df,
    candidates
):

    lower_map = {
        col.lower(): col
        for col in df.columns
    }

    for candidate in candidates:

        if candidate.lower() in lower_map:

            return lower_map[
                candidate.lower()
            ]

    return None


PATH_COLUMN = find_column(
    train_df,
    [
        "crop_path",
        "image_path",
        "path",
        "filepath",
        "file_path",
        "crop"
    ]
)


CLASS_COLUMN = find_column(
    train_df,
    [
        "class_name",
        "class",
        "mobile_net_class",
        "mobilenet_class",
        "target_class",
        "label"
    ]
)


print("Metadata columns:")

print(
    list(train_df.columns)
)

print()

print(
    f"Detected path column : "
    f"{PATH_COLUMN}"
)

print(
    f"Detected class column: "
    f"{CLASS_COLUMN}"
)


if PATH_COLUMN is None:

    raise ValueError(
        "Could not identify crop path column."
    )


if CLASS_COLUMN is None:

    raise ValueError(
        "Could not identify class column."
    )


print()


# =============================================================================
# 9. NORMALIZE CLASS LABELS
# =============================================================================

def normalize_class_name(x):

    x = str(x).strip().lower()

    replacements = {

        "mixed rigid plastic":
            "mixed_plastic_rigid",

        "mixed soft plastic":
            "mixed_plastic_soft",

        "mixed_plastic_rigid":
            "mixed_plastic_rigid",

        "mixed_plastic_soft":
            "mixed_plastic_soft",

        "non plastic":
            "non_plastic",

        "non-plastic":
            "non_plastic",

        "non_plastic":
            "non_plastic",

        "pet oil":
            "pet_oil",

        "pet_oil":
            "pet_oil"

    }

    return replacements.get(
        x,
        x
    )


train_df[
    "class_name_normalized"
] = train_df[
    CLASS_COLUMN
].apply(
    normalize_class_name
)


val_df[
    "class_name_normalized"
] = val_df[
    CLASS_COLUMN
].apply(
    normalize_class_name
)


# =============================================================================
# 10. VALIDATE CLASSES
# =============================================================================

print("=" * 80)
print("CLASS VALIDATION")
print("=" * 80)


train_classes = set(
    train_df[
        "class_name_normalized"
    ]
)


val_classes = set(
    val_df[
        "class_name_normalized"
    ]
)


expected_classes = set(
    CLASS_NAMES
)


print("Expected classes:")

print(
    CLASS_NAMES
)

print()

print("Training classes:")

print(
    sorted(train_classes)
)

print()

print("Validation classes:")

print(
    sorted(val_classes)
)

print()


unknown_train = (
    train_classes
    - expected_classes
)

unknown_val = (
    val_classes
    - expected_classes
)


if unknown_train:

    raise ValueError(
        f"Unknown training classes: "
        f"{unknown_train}"
    )


if unknown_val:

    raise ValueError(
        f"Unknown validation classes: "
        f"{unknown_val}"
    )


print(
    "Class validation passed."
)

print()


# =============================================================================
# 11. CREATE NUMERIC LABELS
# =============================================================================

train_df["label"] = train_df[
    "class_name_normalized"
].map(
    CLASS_TO_IDX
)


val_df["label"] = val_df[
    "class_name_normalized"
].map(
    CLASS_TO_IDX
)


if train_df["label"].isna().any():

    raise ValueError(
        "Some training samples have invalid labels."
    )


if val_df["label"].isna().any():

    raise ValueError(
        "Some validation samples have invalid labels."
    )


train_df["label"] = (
    train_df["label"]
    .astype(int)
)


val_df["label"] = (
    val_df["label"]
    .astype(int)
)


# =============================================================================
# 12. RESOLVE CROP PATHS
# =============================================================================

def resolve_image_path(
    path_value,
    split_dir
):

    path_value = str(
        path_value
    )

    p = Path(
        path_value
    )


    # Absolute path

    if (
        p.is_absolute()
        and p.exists()
    ):

        return p


    # Relative path from split directory

    p = (
        split_dir
        / path_value
    )

    if p.exists():

        return p


    # Relative path from E3Y root

    p = (
        E3Y_ROOT
        / path_value
    )

    if p.exists():

        return p


    # Filename-only recursive search

    filename = Path(
        path_value
    ).name


    matches = list(
        split_dir.rglob(
            filename
        )
    )


    if len(matches) > 0:

        return matches[0]


    return None


print("=" * 80)
print("VALIDATING IMAGE PATHS")
print("=" * 80)


train_df[
    "resolved_path"
] = train_df[
    PATH_COLUMN
].apply(
    lambda x:
        resolve_image_path(
            x,
            TRAIN_DIR
        )
)


val_df[
    "resolved_path"
] = val_df[
    PATH_COLUMN
].apply(
    lambda x:
        resolve_image_path(
            x,
            VAL_DIR
        )
)


train_missing = train_df[
    train_df[
        "resolved_path"
    ].isna()
]


val_missing = val_df[
    val_df[
        "resolved_path"
    ].isna()
]


print(
    f"Missing training images  : "
    f"{len(train_missing)}"
)

print(
    f"Missing validation images: "
    f"{len(val_missing)}"
)


if len(train_missing) > 0:

    print(
        "\nExample missing training paths:"
    )

    print(
        train_missing[
            PATH_COLUMN
        ]
        .head(10)
        .to_string(
            index=False
        )
    )

    raise FileNotFoundError(
        "Training crop paths could not be resolved."
    )


if len(val_missing) > 0:

    print(
        "\nExample missing validation paths:"
    )

    print(
        val_missing[
            PATH_COLUMN
        ]
        .head(10)
        .to_string(
            index=False
        )
    )

    raise FileNotFoundError(
        "Validation crop paths could not be resolved."
    )


print(
    "All crop paths successfully resolved."
)

print()


# =============================================================================
# 13. CLASS DISTRIBUTION
# =============================================================================

print("=" * 80)
print("CLASS DISTRIBUTION")
print("=" * 80)


train_counts = (
    train_df[
        "class_name_normalized"
    ]
    .value_counts()
    .reindex(
        CLASS_NAMES
    )
    .fillna(0)
    .astype(int)
)


val_counts = (
    val_df[
        "class_name_normalized"
    ]
    .value_counts()
    .reindex(
        CLASS_NAMES
    )
    .fillna(0)
    .astype(int)
)


distribution_df = pd.DataFrame({

    "class":
        CLASS_NAMES,

    "train":
        train_counts.values,

    "validation":
        val_counts.values

})


print(
    distribution_df.to_string(
        index=False
    )
)

print()


# =============================================================================
# 14. CALCULATE CLASS WEIGHTS
# =============================================================================
#
# IMPORTANT:
#
# Class weights are calculated ONLY from the training data.
#
# Formula:
#
#       weight_c = N / (K * n_c)
#
# where:
#
#       N = total number of training samples
#       K = number of classes
#       n_c = number of samples in class c
#
# This produces weights whose mean is approximately 1.
#
# No validation information is used.
#
# =============================================================================

print("=" * 80)
print("CALCULATING CLASS WEIGHTS")
print("=" * 80)


total_train_samples = len(
    train_df
)


class_weights = []


for class_name in CLASS_NAMES:

    class_count = train_counts[
        class_name
    ]

    if class_count <= 0:

        raise ValueError(
            f"Class '{class_name}' "
            f"has zero training samples."
        )


    weight = (
        total_train_samples
        /
        (
            NUM_CLASSES
            * class_count
        )
    )


    class_weights.append(
        weight
    )


class_weights = np.array(
    class_weights,
    dtype=np.float32
)


class_weight_df = pd.DataFrame({

    "class":
        CLASS_NAMES,

    "training_samples":
        [
            train_counts[c]
            for c in CLASS_NAMES
        ],

    "class_weight":
        class_weights

})


print()

print(
    class_weight_df.to_string(
        index=False
    )
)

print()

print(
    f"Mean class weight: "
    f"{class_weights.mean():.6f}"
)

print()

print(
    "Class weights calculated "
    "from TRAINING data only."
)

print()


# Save class weights

class_weight_df.to_csv(
    RESULTS_DIR
    / "class_weights.csv",
    index=False
)


# Convert to PyTorch tensor

class_weights_tensor = torch.tensor(
    class_weights,
    dtype=torch.float32,
    device=DEVICE
)


# =============================================================================
# 15. DATASET CLASS
# =============================================================================

class E3YDataset(
    Dataset
):

    def __init__(
        self,
        dataframe,
        transform=None
    ):

        self.df = dataframe.reset_index(
            drop=True
        )

        self.transform = transform


    def __len__(self):

        return len(
            self.df
        )


    def __getitem__(
        self,
        idx
    ):

        row = self.df.iloc[
            idx
        ]


        image_path = row[
            "resolved_path"
        ]


        label = int(
            row["label"]
        )


        try:

            image = Image.open(
                image_path
            ).convert(
                "RGB"
            )


        except Exception as e:

            raise RuntimeError(

                f"Could not load image:\n"
                f"{image_path}\n"
                f"Error: {e}"

            )


        if self.transform is not None:

            image = self.transform(
                image
            )


        return image, label


# =============================================================================
# 16. TRANSFORMS
# =============================================================================
#
# IMPORTANT:
#
# These transforms are IDENTICAL to E3Y-A.
#
# Therefore the only intended experimental difference between E3Y-A and
# E3Y-B is the class-weighted loss.
#
# =============================================================================

train_transform = transforms.Compose([

    transforms.Resize(
        (IMAGE_SIZE, IMAGE_SIZE)
    ),

    transforms.RandomHorizontalFlip(
        p=0.5
    ),

    transforms.RandomRotation(
        degrees=10
    ),

    transforms.ColorJitter(
        brightness=0.15,
        contrast=0.15,
        saturation=0.10,
        hue=0.02
    ),

    transforms.ToTensor(),

    transforms.Normalize(

        mean=[
            0.485,
            0.456,
            0.406
        ],

        std=[
            0.229,
            0.224,
            0.225
        ]

    )

])


val_transform = transforms.Compose([

    transforms.Resize(
        (IMAGE_SIZE, IMAGE_SIZE)
    ),

    transforms.ToTensor(),

    transforms.Normalize(

        mean=[
            0.485,
            0.456,
            0.406
        ],

        std=[
            0.229,
            0.224,
            0.225
        ]

    )

])


# =============================================================================
# 17. CREATE DATASETS
# =============================================================================

train_dataset = E3YDataset(

    train_df,

    transform=train_transform

)


val_dataset = E3YDataset(

    val_df,

    transform=val_transform

)


# =============================================================================
# 18. CREATE DATALOADERS
# =============================================================================

train_loader = DataLoader(

    train_dataset,

    batch_size=BATCH_SIZE,

    shuffle=True,

    num_workers=NUM_WORKERS,

    pin_memory=torch.cuda.is_available()

)


val_loader = DataLoader(

    val_dataset,

    batch_size=BATCH_SIZE,

    shuffle=False,

    num_workers=NUM_WORKERS,

    pin_memory=torch.cuda.is_available()

)


print("=" * 80)
print("DATALOADERS")
print("=" * 80)


print(
    f"Training samples  : "
    f"{len(train_dataset)}"
)

print(
    f"Validation samples: "
    f"{len(val_dataset)}"
)

print(
    f"Training batches  : "
    f"{len(train_loader)}"
)

print(
    f"Validation batches: "
    f"{len(val_loader)}"
)

print()


# =============================================================================
# 19. LOAD MOBILENET-V3-LARGE
# =============================================================================

print("=" * 80)
print("LOADING MOBILENET-V3-LARGE")
print("=" * 80)


weights = (
    models.MobileNet_V3_Large_Weights.DEFAULT
)


model = models.mobilenet_v3_large(
    weights=weights
)


in_features = (
    model.classifier[-1]
    .in_features
)


model.classifier[-1] = nn.Linear(

    in_features,

    NUM_CLASSES

)


model = model.to(
    DEVICE
)


print(
    model.classifier
)

print()


# =============================================================================
# 20. LOSS / OPTIMIZER
# =============================================================================
#
# THIS IS THE MAIN DIFFERENCE FROM E3Y-A.
#
# E3Y-A:
#
#     CrossEntropyLoss()
#
# E3Y-B:
#
#     CrossEntropyLoss(weight=class_weights)
#
# =============================================================================

criterion = nn.CrossEntropyLoss(

    weight=class_weights_tensor

)


optimizer = optim.AdamW(

    model.parameters(),

    lr=LEARNING_RATE,

    weight_decay=WEIGHT_DECAY

)


scheduler = optim.lr_scheduler.ReduceLROnPlateau(

    optimizer,

    mode="max",

    factor=0.5,

    patience=2

)


print(
    "Loss function: "
    "Class-weighted CrossEntropyLoss"
)

print()


# =============================================================================
# 21. TRAINING FUNCTION
# =============================================================================

def train_one_epoch(

    model,

    loader,

    criterion,

    optimizer,

    device

):

    model.train()


    running_loss = 0.0


    all_predictions = []

    all_targets = []


    for images, targets in loader:

        images = images.to(

            device,

            non_blocking=True

        )


        targets = targets.to(

            device,

            non_blocking=True

        )


        optimizer.zero_grad(

            set_to_none=True

        )


        outputs = model(
            images
        )


        loss = criterion(

            outputs,

            targets

        )


        loss.backward()


        optimizer.step()


        running_loss += (

            loss.item()
            *
            images.size(0)

        )


        predictions = (

            outputs.argmax(
                dim=1
            )

        )


        all_predictions.extend(

            predictions.detach()
            .cpu()
            .numpy()

        )


        all_targets.extend(

            targets.detach()
            .cpu()
            .numpy()

        )


    epoch_loss = (

        running_loss
        /
        len(loader.dataset)

    )


    accuracy = accuracy_score(

        all_targets,

        all_predictions

    )


    macro_f1 = f1_score(

        all_targets,

        all_predictions,

        average="macro",

        zero_division=0

    )


    weighted_f1 = f1_score(

        all_targets,

        all_predictions,

        average="weighted",

        zero_division=0

    )


    return (

        epoch_loss,

        accuracy,

        macro_f1,

        weighted_f1

    )


# =============================================================================
# 22. VALIDATION FUNCTION
# =============================================================================

def evaluate(

    model,

    loader,

    criterion,

    device

):

    model.eval()


    running_loss = 0.0


    all_predictions = []

    all_targets = []


    with torch.no_grad():

        for images, targets in loader:

            images = images.to(

                device,

                non_blocking=True

            )


            targets = targets.to(

                device,

                non_blocking=True

            )


            outputs = model(
                images
            )


            loss = criterion(

                outputs,

                targets

            )


            running_loss += (

                loss.item()
                *
                images.size(0)

            )


            predictions = (

                outputs.argmax(
                    dim=1
                )

            )


            all_predictions.extend(

                predictions.cpu()
                .numpy()

            )


            all_targets.extend(

                targets.cpu()
                .numpy()

            )


    epoch_loss = (

        running_loss
        /
        len(loader.dataset)

    )


    accuracy = accuracy_score(

        all_targets,

        all_predictions

    )


    macro_f1 = f1_score(

        all_targets,

        all_predictions,

        average="macro",

        zero_division=0

    )


    weighted_f1 = f1_score(

        all_targets,

        all_predictions,

        average="weighted",

        zero_division=0

    )


    return (

        epoch_loss,

        accuracy,

        macro_f1,

        weighted_f1,

        all_targets,

        all_predictions

    )


# =============================================================================
# 23. TRAINING LOOP
# =============================================================================

print("=" * 80)
print("STARTING E3Y-B TRAINING")
print("=" * 80)


history = []


best_macro_f1 = -1.0

best_epoch = 0

best_state = None

epochs_without_improvement = 0


training_start = time.time()


for epoch in range(

    1,

    NUM_EPOCHS + 1

):

    epoch_start = time.time()


    (

        train_loss,

        train_acc,

        train_macro_f1,

        train_weighted_f1

    ) = train_one_epoch(

        model,

        train_loader,

        criterion,

        optimizer,

        DEVICE

    )


    (

        val_loss,

        val_acc,

        val_macro_f1,

        val_weighted_f1,

        val_targets,

        val_predictions

    ) = evaluate(

        model,

        val_loader,

        criterion,

        DEVICE

    )


    scheduler.step(

        val_macro_f1

    )


    current_lr = (
        optimizer
        .param_groups[0]["lr"]
    )


    epoch_time = (

        time.time()
        -
        epoch_start

    )


    history.append({

        "epoch":
            epoch,

        "train_loss":
            train_loss,

        "train_accuracy":
            train_acc,

        "train_macro_f1":
            train_macro_f1,

        "train_weighted_f1":
            train_weighted_f1,

        "val_loss":
            val_loss,

        "val_accuracy":
            val_acc,

        "val_macro_f1":
            val_macro_f1,

        "val_weighted_f1":
            val_weighted_f1,

        "learning_rate":
            current_lr,

        "epoch_time_sec":
            epoch_time

    })


    print(
        f"\nEpoch {epoch:02d}/{NUM_EPOCHS}"
    )


    print(

        f"Train | "
        f"Loss: {train_loss:.4f} | "
        f"Acc: {train_acc:.4f} | "
        f"Macro F1: {train_macro_f1:.4f} | "
        f"Weighted F1: {train_weighted_f1:.4f}"

    )


    print(

        f"Val   | "
        f"Loss: {val_loss:.4f} | "
        f"Acc: {val_acc:.4f} | "
        f"Macro F1: {val_macro_f1:.4f} | "
        f"Weighted F1: {val_weighted_f1:.4f}"

    )


    print(

        f"LR: {current_lr:.8f} | "
        f"Time: {epoch_time:.1f}s"

    )


    # -------------------------------------------------------------------------
    # BEST MODEL
    # -------------------------------------------------------------------------

    if val_macro_f1 > (

        best_macro_f1
        +
        MIN_DELTA

    ):

        best_macro_f1 = (
            val_macro_f1
        )


        best_epoch = epoch


        best_state = copy.deepcopy(

            model.state_dict()

        )


        epochs_without_improvement = 0


        best_checkpoint = {

            "epoch":
                epoch,

            "model_state_dict":
                model.state_dict(),

            "optimizer_state_dict":
                optimizer.state_dict(),

            "best_macro_f1":
                best_macro_f1,

            "val_accuracy":
                val_acc,

            "val_weighted_f1":
                val_weighted_f1,

            "class_names":
                CLASS_NAMES,

            "class_weights":
                class_weights.tolist(),

            "seed":
                SEED,

            "experiment":
                "E3Y_B_YOLO11n_Epoch18_MobileNetV3Large_ClassWeighted"

        }


        torch.save(

            best_checkpoint,

            RESULTS_DIR
            /
            "E3Y_B_MobileNetV3Large_best.pth"

        )


        print(

            f"*** NEW BEST MODEL — "
            f"Macro F1 = {best_macro_f1:.4f}"

        )


    else:

        epochs_without_improvement += 1


        print(

            f"No improvement "
            f"({epochs_without_improvement}/{PATIENCE})"

        )


    # -------------------------------------------------------------------------
    # EARLY STOPPING
    # -------------------------------------------------------------------------

    if (

        epochs_without_improvement
        >=
        PATIENCE

    ):

        print(
            "\nEarly stopping triggered."
        )

        break


training_time = (

    time.time()
    -
    training_start

)


# =============================================================================
# 24. RESTORE BEST MODEL
# =============================================================================

print()

print("=" * 80)

print("RESTORING BEST MODEL")

print("=" * 80)


if best_state is None:

    raise RuntimeError(
        "No best model was saved."
    )


model.load_state_dict(
    best_state
)


print(
    f"Best epoch   : "
    f"{best_epoch}"
)

print(
    f"Best Macro F1: "
    f"{best_macro_f1:.4f}"
)

print()


# =============================================================================
# 25. FINAL VALIDATION
# =============================================================================

print("=" * 80)

print("FINAL E3Y-B VALIDATION")

print("=" * 80)


(

    final_loss,

    final_accuracy,

    final_macro_f1,

    final_weighted_f1,

    final_targets,

    final_predictions

) = evaluate(

    model,

    val_loader,

    criterion,

    DEVICE

)


print(

    f"Accuracy       : "
    f"{final_accuracy:.4f}"

)

print(

    f"Accuracy (%)   : "
    f"{final_accuracy * 100:.2f}%"

)

print(

    f"Macro F1       : "
    f"{final_macro_f1:.4f}"

)

print(

    f"Weighted F1    : "
    f"{final_weighted_f1:.4f}"

)

print(

    f"Validation loss: "
    f"{final_loss:.4f}"

)

print(

    f"Samples        : "
    f"{len(final_targets)}"

)

print()


# =============================================================================
# 26. CLASSIFICATION REPORT
# =============================================================================

print("=" * 80)

print("CLASSIFICATION REPORT")

print("=" * 80)


report_dict = classification_report(

    final_targets,

    final_predictions,

    labels=list(
        range(NUM_CLASSES)
    ),

    target_names=CLASS_NAMES,

    output_dict=True,

    zero_division=0

)


report_text = classification_report(

    final_targets,

    final_predictions,

    labels=list(
        range(NUM_CLASSES)
    ),

    target_names=CLASS_NAMES,

    zero_division=0

)


print(
    report_text
)


report_df = pd.DataFrame(
    report_dict
).transpose()


report_df.to_csv(

    RESULTS_DIR
    /
    "classification_report.csv"

)


# =============================================================================
# 27. CONFUSION MATRIX
# =============================================================================

cm = confusion_matrix(

    final_targets,

    final_predictions,

    labels=list(
        range(NUM_CLASSES)
    )

)


cm_df = pd.DataFrame(

    cm,

    index=CLASS_NAMES,

    columns=CLASS_NAMES

)


cm_df.to_csv(

    RESULTS_DIR
    /
    "confusion_matrix.csv"

)


print()

print("=" * 80)

print("CONFUSION MATRIX")

print("=" * 80)


print(
    cm_df
)


# =============================================================================
# 28. PER-CLASS SUMMARY
# =============================================================================

per_class_rows = []


for i, class_name in enumerate(
    CLASS_NAMES
):

    tp = cm[i, i]

    actual = cm[i, :].sum()

    predicted = cm[:, i].sum()

    incorrect = actual - tp


    recall = (

        tp / actual

        if actual > 0

        else 0

    )


    precision = (

        tp / predicted

        if predicted > 0

        else 0

    )


    if (

        precision
        +
        recall

    ) > 0:

        f1 = (

            2
            *
            precision
            *
            recall
            /
            (
                precision
                +
                recall
            )

        )

    else:

        f1 = 0


    per_class_rows.append({

        "class":
            class_name,

        "actual":
            actual,

        "correct":
            tp,

        "incorrect":
            incorrect,

        "precision":
            precision,

        "recall":
            recall,

        "f1":
            f1

    })


per_class_df = pd.DataFrame(
    per_class_rows
)


per_class_df.to_csv(

    RESULTS_DIR
    /
    "per_class_results.csv",

    index=False

)


print()

print("=" * 80)

print("PER-CLASS RESULTS")

print("=" * 80)


print(

    per_class_df.to_string(
        index=False
    )

)


# =============================================================================
# 29. PLASTIC-ONLY RESULTS
# =============================================================================
#
# Exclude non_plastic.
#
# Research focus:
#   Plastic-type classification.
#
# =============================================================================

plastic_indices = [

    CLASS_TO_IDX["ecal"],

    CLASS_TO_IDX["hdpe"],

    CLASS_TO_IDX[
        "mixed_plastic_rigid"
    ],

    CLASS_TO_IDX[
        "mixed_plastic_soft"
    ],

    CLASS_TO_IDX["pet"],

    CLASS_TO_IDX["pet_oil"]

]


plastic_mask = np.isin(

    np.array(final_targets),

    plastic_indices

)


plastic_targets = np.array(
    final_targets
)[
    plastic_mask
]


plastic_predictions = np.array(
    final_predictions
)[
    plastic_mask
]


plastic_accuracy = accuracy_score(

    plastic_targets,

    plastic_predictions

)


plastic_macro_f1 = f1_score(

    plastic_targets,

    plastic_predictions,

    labels=plastic_indices,

    average="macro",

    zero_division=0

)


plastic_weighted_f1 = f1_score(

    plastic_targets,

    plastic_predictions,

    labels=plastic_indices,

    average="weighted",

    zero_division=0

)


print()

print("=" * 80)

print("PLASTIC-ONLY CLASSIFICATION")

print("=" * 80)


print(

    f"Plastic samples     : "
    f"{len(plastic_targets)}"

)


print(

    f"Plastic accuracy    : "
    f"{plastic_accuracy:.4f}"

)


print(

    f"Plastic accuracy    : "
    f"{plastic_accuracy * 100:.2f}%"

)


print(

    f"Plastic Macro F1    : "
    f"{plastic_macro_f1:.4f}"

)


print(

    f"Plastic Weighted F1 : "
    f"{plastic_weighted_f1:.4f}"

)


# =============================================================================
# 30. SAVE TRAINING HISTORY
# =============================================================================

history_df = pd.DataFrame(
    history
)


history_df.to_csv(

    RESULTS_DIR
    /
    "training_history.csv",

    index=False

)


# =============================================================================
# 31. TRAINING CURVES
# =============================================================================

# -----------------------------------------------------------------------------
# Loss
# -----------------------------------------------------------------------------

plt.figure(
    figsize=(8, 5)
)


plt.plot(

    history_df["epoch"],

    history_df["train_loss"],

    label="Train Loss"

)


plt.plot(

    history_df["epoch"],

    history_df["val_loss"],

    label="Validation Loss"

)


plt.xlabel(
    "Epoch"
)

plt.ylabel(
    "Loss"
)


plt.title(
    "E3Y-B MobileNet-V3-Large Loss"
)


plt.legend()


plt.grid(
    True,
    alpha=0.3
)


plt.tight_layout()


plt.savefig(

    RESULTS_DIR
    /
    "loss_curve.png",

    dpi=300

)


plt.close()


# -----------------------------------------------------------------------------
# Macro F1
# -----------------------------------------------------------------------------

plt.figure(
    figsize=(8, 5)
)


plt.plot(

    history_df["epoch"],

    history_df["train_macro_f1"],

    label="Train Macro F1"

)


plt.plot(

    history_df["epoch"],

    history_df["val_macro_f1"],

    label="Validation Macro F1"

)


plt.axvline(

    best_epoch,

    linestyle="--",

    label=f"Best Epoch ({best_epoch})"

)


plt.xlabel(
    "Epoch"
)

plt.ylabel(
    "Macro F1"
)


plt.title(
    "E3Y-B MobileNet-V3-Large Macro F1"
)


plt.legend()


plt.grid(
    True,
    alpha=0.3
)


plt.tight_layout()


plt.savefig(

    RESULTS_DIR
    /
    "macro_f1_curve.png",

    dpi=300

)


plt.close()


# -----------------------------------------------------------------------------
# Accuracy
# -----------------------------------------------------------------------------

plt.figure(
    figsize=(8, 5)
)


plt.plot(

    history_df["epoch"],

    history_df["train_accuracy"],

    label="Train Accuracy"

)


plt.plot(

    history_df["epoch"],

    history_df["val_accuracy"],

    label="Validation Accuracy"

)


plt.xlabel(
    "Epoch"
)

plt.ylabel(
    "Accuracy"
)


plt.title(
    "E3Y-B MobileNet-V3-Large Accuracy"
)


plt.legend()


plt.grid(
    True,
    alpha=0.3
)


plt.tight_layout()


plt.savefig(

    RESULTS_DIR
    /
    "accuracy_curve.png",

    dpi=300

)


plt.close()


# =============================================================================
# 32. SAVE FINAL CHECKPOINT
# =============================================================================

final_checkpoint = {

    "experiment":
        "E3Y_B_YOLO11n_Epoch18_MobileNetV3Large_ClassWeighted",

    "model":
        "MobileNetV3-Large",

    "detector":
        "YOLO11n",

    "detector_epoch":
        18,

    "num_classes":
        NUM_CLASSES,

    "class_names":
        CLASS_NAMES,

    "class_weights":
        class_weights.tolist(),

    "best_epoch":
        best_epoch,

    "best_macro_f1":
        best_macro_f1,

    "final_accuracy":
        final_accuracy,

    "final_macro_f1":
        final_macro_f1,

    "final_weighted_f1":
        final_weighted_f1,

    "plastic_accuracy":
        plastic_accuracy,

    "plastic_macro_f1":
        plastic_macro_f1,

    "plastic_weighted_f1":
        plastic_weighted_f1,

    "model_state_dict":
        model.state_dict(),

    "seed":
        SEED

}


torch.save(

    final_checkpoint,

    RESULTS_DIR
    /
    "E3Y_B_MobileNetV3Large_final.pth"

)


# =============================================================================
# 33. SAVE EXPERIMENT SUMMARY
# =============================================================================

summary = {

    "experiment":
        "E3Y-B — YOLO11n Epoch-18 + MobileNet-V3-Large + Class Weighting",

    "detector":
        "YOLO11n",

    "detector_epoch":
        18,

    "classifier":
        "MobileNet-V3-Large",

    "num_classes":
        NUM_CLASSES,

    "classes":
        CLASS_NAMES,

    "train_samples":
        len(train_df),

    "validation_samples":
        len(val_df),

    "best_epoch":
        best_epoch,

    "best_validation_macro_f1":
        best_macro_f1,

    "validation_accuracy":
        final_accuracy,

    "validation_macro_f1":
        final_macro_f1,

    "validation_weighted_f1":
        final_weighted_f1,

    "plastic_samples":
        len(plastic_targets),

    "plastic_accuracy":
        plastic_accuracy,

    "plastic_macro_f1":
        plastic_macro_f1,

    "plastic_weighted_f1":
        plastic_weighted_f1,

    "epochs_completed":
        len(history),

    "training_time_minutes":
        training_time / 60,

    "class_balancing":
        True,

    "class_balancing_method":
        "inverse_frequency_class_weighted_cross_entropy",

    "class_weights":
        {
            CLASS_NAMES[i]:
                float(class_weights[i])
            for i in range(NUM_CLASSES)
        },

    "oversampling":
        False,

    "smote":
        False,

    "focal_loss":
        False,

    "class_specific_augmentation":
        False,

    "crop_matching_iou":
        0.5,

    "seed":
        SEED

}


with open(

    RESULTS_DIR
    /
    "experiment_summary.json",

    "w"

) as f:

    json.dump(

        summary,

        f,

        indent=4

    )


# =============================================================================
# 34. FINAL OUTPUT
# =============================================================================

print()

print("=" * 80)

print("E3Y-B — MOBILE NET CLASS-WEIGHTED TRAINING COMPLETE")

print("=" * 80)


print(

    f"Best epoch              : "
    f"{best_epoch}"

)


print(

    f"Best Validation Macro F1: "
    f"{best_macro_f1:.4f}"

)


print(

    f"Validation Accuracy     : "
    f"{final_accuracy:.4f} "
    f"({final_accuracy * 100:.2f}%)"

)


print(

    f"Validation Macro F1     : "
    f"{final_macro_f1:.4f}"

)


print(

    f"Validation Weighted F1  : "
    f"{final_weighted_f1:.4f}"

)


print()


print(

    f"Plastic Accuracy        : "
    f"{plastic_accuracy:.4f} "
    f"({plastic_accuracy * 100:.2f}%)"

)


print(

    f"Plastic Macro F1        : "
    f"{plastic_macro_f1:.4f}"

)


print(

    f"Plastic Weighted F1     : "
    f"{plastic_weighted_f1:.4f}"

)


print()

print(
    "Class weights:"
)


for class_name, weight in zip(

    CLASS_NAMES,

    class_weights

):

    print(

        f"  {class_name:<25} "
        f"{weight:.6f}"

    )


print()

print(
    "Best checkpoint:"
)


print(

    RESULTS_DIR
    /
    "E3Y_B_MobileNetV3Large_best.pth"

)


print()

print(
    "Results directory:"
)


print(
    RESULTS_DIR
)


print()

print("=" * 80)



E3Y-B — YOLO11n -> MobileNet-V3-Large
CLASS-WEIGHTED LOSS
Device       : cuda
GPU          : NVIDIA GeForce RTX 3050 Ti Laptop GPU
CUDA         : 12.6
Classes      : 7
Batch size   : 64
Epochs       : 30
Learning rate: 0.0001
Weight decay : 0.0001
Seed         : 42

EXPERIMENT DEFINITION

E3Y-B = E3Y-A + CLASS-WEIGHTED CROSS-ENTROPY LOSS

YOLO detector          : YOLO11n
Detector checkpoint    : Epoch-18 matched detections
Crop matching          : GT IoU >= 0.5
Class balancing        : CLASS WEIGHTS
Oversampling            : NO
SMOTE                   : NO
Focal loss              : NO
Class-specific augment. : NO

CHECKING DATASET
C:\Users\varda\Documents\_My Computer\COMMON_space\Learning\Upgrad_EPGP_MS\upgrad_IIITB_MS_course\MS_LJMU_Material\Topic Data\SortWaste\dataset\dataset\yolo_mobilenet_crops_E3Y Exists: True
C:\Users\varda\Documents\_My Computer\COMMON_space\Learning\Upgrad_EPGP_MS\upgrad_IIITB_MS_course\MS_LJMU_Material\Topic Data\SortWaste\dataset\dataset\yolo_mobilenet_crop

In [ ]:
#### E3Y-B end-to-end COCO evaluation script
# ============================================================
# END-TO-END COCO EVALUATION ON ALL VALIDATION IMAGES
# E3Y-B: YOLO11n -> MobileNet-V3-Large
#
# CORRECTED FINAL END-TO-END COCO EVALUATION
# ============================================================
#
# Pipeline:
#
#   YOLO11n
#       |
#       | bounding box + YOLO confidence
#       v
#   Object crop
#       |
#       v
#   MobileNet-V3-Large
#       |
#       | predicted class + class probability
#       v
#   Final prediction
#
#
# OPTION B FINAL CONFIDENCE:
#
#   final_confidence =
#       YOLO_confidence * MobileNet_class_probability
#
#
# PRIMARY EVALUATION:
#
#   COCOeval
#
#   mAP50
#   mAP50-95
#   mAP75
#   AR@1
#   AR@10
#   AR@100
#
#
# DIAGNOSTIC CLASSIFICATION:
#
#   Proper one-to-one IoU matching at IoU >= 0.50
#
#   Accuracy
#   Macro F1
#   Weighted F1
#   Confusion matrix
#
#
# IMPORTANT:
#
# Original SortWaste taxonomy:
#
#   0 = pet
#   1 = hdpe
#   2 = mixed_plastic_soft
#   3 = ecal
#   4 = metal
#   5 = cardboard
#   6 = mixed_plastic_rigid
#   7 = pet_oil
#
#
# Evaluation taxonomy:
#
#   1 = ecal
#   2 = hdpe
#   3 = mixed_plastic_rigid
#   4 = mixed_plastic_soft
#   5 = non_plastic
#   6 = pet
#   7 = pet_oil
#
#
# Mapping:
#
#   metal      -> non_plastic
#   cardboard  -> non_plastic
#
# The mapping is applied consistently to:
#
#   1. COCO ground truth
#   2. YOLO detections
#   3. MobileNet predictions
#
#
# E3Y-B DIFFERENCE FROM E3Y-P:
#
#   ONLY the MobileNet checkpoint changes.
#
#   E3Y-B:
#       MobileNet-V3-Large trained using class-weighted loss.
#
# All evaluation settings remain identical to E3Y-P.
#
# ============================================================


from pathlib import Path
import csv
import json
import time

import cv2
import torch
import numpy as np

from ultralytics import YOLO
from torchvision import models, transforms
from PIL import Image

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix,
)

from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval


# ============================================================
# 1. PATHS
# ============================================================

DATASET_ROOT = Path(
    r"C:\Users\varda\Documents\_My Computer\COMMON_space"
    r"\Learning\Upgrad_EPGP_MS\upgrad_IIITB_MS_course"
    r"\MS_LJMU_Material\Topic Data\SortWaste\dataset\dataset"
)


SPLIT_ROOT = (
    DATASET_ROOT
    / "splited_all_dataset_coco"
)


VAL_IMAGES = (
    SPLIT_ROOT
    / "val"
    / "images"
)


VAL_LABELS = (
    SPLIT_ROOT
    / "val"
    / "labels"
)


VAL_JSON = (
    SPLIT_ROOT
    / "val"
    / "annotations"
    / "val_coco.json"
)


# ============================================================
# YOLO CHECKPOINT
# ============================================================

YOLO_CHECKPOINT = Path(
    r"C:\Users\varda\Documents\_My Computer\COMMON_space"
    r"\Learning\Upgrad_EPGP_MS\upgrad_IIITB_MS_course"
    r"\MS_LJMU_Material\Thesis_Code"
    r"\runs\detect\runs\sortwaste\yolo11n_baseline-2"
    r"\weights\best.pt"
)


# ============================================================
# MOBILENET CHECKPOINT
# ============================================================
#
# E3Y-B = class-weighted MobileNet-V3-Large
#
# IMPORTANT:
# This is the ONLY model checkpoint changed from E3Y-P.
#
# ============================================================

MOBILENET_CHECKPOINT = Path(
    r"C:\Users\varda\Documents\_My Computer\COMMON_space"
    r"\Learning\Upgrad_EPGP_MS\upgrad_IIITB_MS_course"
    r"\MS_LJMU_Material\Topic Data\SortWaste\dataset\dataset"
    r"\yolo_mobilenet_crops_E3Y"
    r"\mobilenet_results"
    r"\E3Y_B_class_weighted"
    r"\E3Y_B_MobileNetV3Large_best.pth"
)


# ============================================================
# OUTPUT DIRECTORY
# ============================================================

OUTPUT_ROOT = (
    DATASET_ROOT
    / "yolo_mobilenet_crops_E3Y"
    / "mobilenet_results"
    / "E3Y_B_COCO_evaluation_corrected"
)


OUTPUT_ROOT.mkdir(
    parents=True,
    exist_ok=True
)


RAW_RESULTS_CSV = (
    OUTPUT_ROOT
    / "E3Y_B_all_detections.csv"
)


COCO_RESULTS_CSV = (
    OUTPUT_ROOT
    / "E3Y_B_COCO_results.csv"
)


CLASS_RESULTS_CSV = (
    OUTPUT_ROOT
    / "E3Y_B_class_results.csv"
)


CONFUSION_MATRIX_CSV = (
    OUTPUT_ROOT
    / "E3Y_B_confusion_matrix.csv"
)


MATCHED_RESULTS_CSV = (
    OUTPUT_ROOT
    / "E3Y_B_matched_classification.csv"
)


SUMMARY_TXT = (
    OUTPUT_ROOT
    / "E3Y_B_summary.txt"
)


# ============================================================
# 2. CONFIGURATION
# ============================================================

# ------------------------------------------------------------
# YOLO
# ------------------------------------------------------------

YOLO_CONF = 0.001
YOLO_NMS_IOU = 0.70


# ------------------------------------------------------------
# MobileNet
# ------------------------------------------------------------

IMAGE_SIZE = 224
BATCH_SIZE = 64


# ------------------------------------------------------------
# Taxonomy
# ------------------------------------------------------------

NUM_YOLO_CLASSES = 8
NUM_EVAL_CLASSES = 7


MOBILENET_CLASSES = [
    "ecal",
    "hdpe",
    "mixed_plastic_rigid",
    "mixed_plastic_soft",
    "non_plastic",
    "pet",
    "pet_oil",
]


CLASS_TO_ID = {
    name: index + 1
    for index, name in enumerate(
        MOBILENET_CLASSES
    )
}


ID_TO_CLASS = {
    index: name
    for name, index in CLASS_TO_ID.items()
}


# ------------------------------------------------------------
# Diagnostic matching
# ------------------------------------------------------------

MATCH_IOU_THRESHOLD = 0.50


# ============================================================
# 3. ORIGINAL SORTWASTE CLASS MAPPING
# ============================================================

ORIGINAL_YOLO_NAMES = {

    0: "pet",

    1: "hdpe",

    2: "mixed_plastic_soft",

    3: "ecal",

    4: "metal",

    5: "cardboard",

    6: "mixed_plastic_rigid",

    7: "pet_oil",
}


ORIGINAL_COCO_NAMES = {

    1: "pet",

    2: "hdpe",

    3: "mixed_plastic_soft",

    4: "ecal",

    5: "metal",

    6: "cardboard",

    7: "mixed_plastic_rigid",

    8: "pet_oil",
}


# ------------------------------------------------------------
# YOLO -> evaluation taxonomy
# ------------------------------------------------------------

YOLO_TO_EVAL = {

    0: "pet",

    1: "hdpe",

    2: "mixed_plastic_soft",

    3: "ecal",

    4: "non_plastic",

    5: "non_plastic",

    6: "mixed_plastic_rigid",

    7: "pet_oil",
}


# ------------------------------------------------------------
# Original COCO -> evaluation taxonomy
# ------------------------------------------------------------

COCO_TO_EVAL = {

    1: "pet",

    2: "hdpe",

    3: "mixed_plastic_soft",

    4: "ecal",

    5: "non_plastic",

    6: "non_plastic",

    7: "mixed_plastic_rigid",

    8: "pet_oil",
}


# ------------------------------------------------------------
# Evaluation COCO IDs
# ------------------------------------------------------------

EVAL_CLASS_TO_COCO_ID = {

    "ecal": 1,

    "hdpe": 2,

    "mixed_plastic_rigid": 3,

    "mixed_plastic_soft": 4,

    "non_plastic": 5,

    "pet": 6,

    "pet_oil": 7,
}


EVAL_COCO_ID_TO_CLASS = {
    value: key
    for key, value in EVAL_CLASS_TO_COCO_ID.items()
}


# ============================================================
# 4. DEVICE
# ============================================================

DEVICE = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)


print("=" * 80)

print(
    "E3Y-B — YOLO11n -> MobileNet-V3-Large"
)

print(
    "CORRECTED FINAL END-TO-END COCO EVALUATION"
)

print("=" * 80)


print(
    "\nDevice:",
    DEVICE
)


if torch.cuda.is_available():

    print(
        "GPU:",
        torch.cuda.get_device_name(0)
    )

    print(
        "CUDA:",
        torch.version.cuda
    )


# ============================================================
# 5. PATH VALIDATION
# ============================================================

print("\n" + "=" * 80)
print("CHECKING PATHS")
print("=" * 80)


paths_to_check = {

    "Dataset root":
        DATASET_ROOT,

    "Validation images":
        VAL_IMAGES,

    "Validation labels":
        VAL_LABELS,

    "Validation COCO JSON":
        VAL_JSON,

    "YOLO checkpoint":
        YOLO_CHECKPOINT,

    "MobileNet checkpoint":
        MOBILENET_CHECKPOINT,
}


for name, path in paths_to_check.items():

    print(
        f"{name:25s}: {path}"
    )

    print(
        f"{'':25s}  Exists: {path.exists()}"
    )

    if not path.exists():

        raise FileNotFoundError(
            f"\nRequired path does not exist:\n{path}"
        )


# ============================================================
# 6. CLASS MAPPING DISPLAY
# ============================================================

print("\n" + "=" * 80)
print("CLASS MAPPING")
print("=" * 80)


print(
    "\nOriginal SortWaste classes -> evaluation classes:"
)


for yolo_id in range(
    NUM_YOLO_CLASSES
):

    print(
        f"YOLO {yolo_id} "
        f"({ORIGINAL_YOLO_NAMES[yolo_id]:25s}) "
        f"-> "
        f"{YOLO_TO_EVAL[yolo_id]}"
    )


print(
    "\nEvaluation COCO categories:"
)


for class_name in MOBILENET_CLASSES:

    print(
        f"COCO ID "
        f"{EVAL_CLASS_TO_COCO_ID[class_name]} "
        f"-> "
        f"{class_name}"
    )


# ============================================================
# 7. LOAD YOLO
# ============================================================

print("\n" + "=" * 80)
print("LOADING YOLO11n")
print("=" * 80)


yolo_model = YOLO(
    str(YOLO_CHECKPOINT)
)


print(
    "YOLO loaded successfully."
)


# ============================================================
# 8. LOAD MOBILENET
# ============================================================

print("\n" + "=" * 80)
print("LOADING E3Y-B MOBILENET-V3-LARGE")
print("=" * 80)


print(
    "\nMobileNet checkpoint:"
)

print(
    MOBILENET_CHECKPOINT
)


mobilenet_model = models.mobilenet_v3_large(
    weights=None
)


in_features = (
    mobilenet_model.classifier[-1].in_features
)


mobilenet_model.classifier[-1] = (
    torch.nn.Linear(
        in_features,
        NUM_EVAL_CLASSES
    )
)


checkpoint = torch.load(
    MOBILENET_CHECKPOINT,
    map_location=DEVICE
)


if isinstance(checkpoint, dict):

    if "model_state_dict" in checkpoint:

        state_dict = (
            checkpoint["model_state_dict"]
        )

    elif "state_dict" in checkpoint:

        state_dict = (
            checkpoint["state_dict"]
        )

    else:

        state_dict = checkpoint

else:

    state_dict = checkpoint


clean_state_dict = {}


for key, value in state_dict.items():

    if key.startswith("module."):

        key = key[len("module."):]

    clean_state_dict[key] = value


missing_keys, unexpected_keys = (
    mobilenet_model.load_state_dict(
        clean_state_dict,
        strict=False
    )
)


if missing_keys:

    print(
        "\nWARNING — Missing MobileNet keys:"
    )

    for key in missing_keys:

        print(
            " ",
            key
        )


if unexpected_keys:

    print(
        "\nWARNING — Unexpected MobileNet keys:"
    )

    for key in unexpected_keys:

        print(
            " ",
            key
        )


mobilenet_model = (
    mobilenet_model.to(DEVICE)
)


mobilenet_model.eval()


print(
    "MobileNet loaded successfully."
)


# ============================================================
# 9. MOBILE NET EVALUATION TRANSFORM
# ============================================================

eval_transform = transforms.Compose(
    [

        transforms.Resize(
            (
                IMAGE_SIZE,
                IMAGE_SIZE
            )
        ),

        transforms.ToTensor(),

        transforms.Normalize(

            mean=[
                0.485,
                0.456,
                0.406
            ],

            std=[
                0.229,
                0.224,
                0.225
            ]
        ),
    ]
)


# ============================================================
# 10. IMAGE EXTENSIONS
# ============================================================

IMAGE_EXTENSIONS = {

    ".jpg",
    ".jpeg",
    ".png",
    ".bmp",
    ".webp",
}


# ============================================================
# 11. IoU
# ============================================================

def calculate_iou(
    box1,
    box2
):

    x1 = max(
        box1[0],
        box2[0]
    )

    y1 = max(
        box1[1],
        box2[1]
    )

    x2 = min(
        box1[2],
        box2[2]
    )

    y2 = min(
        box1[3],
        box2[3]
    )


    intersection_width = max(
        0.0,
        x2 - x1
    )

    intersection_height = max(
        0.0,
        y2 - y1
    )


    intersection = (
        intersection_width
        *
        intersection_height
    )


    area1 = (
        max(
            0.0,
            box1[2] - box1[0]
        )
        *
        max(
            0.0,
            box1[3] - box1[1]
        )
    )


    area2 = (
        max(
            0.0,
            box2[2] - box2[0]
        )
        *
        max(
            0.0,
            box2[3] - box2[1]
        )
    )


    union = (
        area1
        +
        area2
        -
        intersection
    )


    if union <= 0:

        return 0.0


    return (
        intersection / union
    )


# ============================================================
# 12. MOBILE NET BATCH CLASSIFICATION
# ============================================================

def classify_crops(
    crops
):

    if len(crops) == 0:

        return [], []


    tensors = []

    valid_indices = []


    for index, crop in enumerate(
        crops
    ):

        if crop is None:

            continue


        if crop.size == 0:

            continue


        crop_rgb = cv2.cvtColor(
            crop,
            cv2.COLOR_BGR2RGB
        )


        pil_image = Image.fromarray(
            crop_rgb
        )


        tensor = eval_transform(
            pil_image
        )


        tensors.append(
            tensor
        )

        valid_indices.append(
            index
        )


    if len(tensors) == 0:

        return (
            [None] * len(crops),
            [None] * len(crops)
        )


    batch = torch.stack(
        tensors
    ).to(DEVICE)


    with torch.no_grad():

        outputs = mobilenet_model(
            batch
        )


        probabilities = torch.softmax(
            outputs,
            dim=1
        )


        predictions = torch.argmax(
            probabilities,
            dim=1
        )


        prediction_probabilities = (
            probabilities[
                torch.arange(
                    len(predictions),
                    device=DEVICE
                ),
                predictions
            ]
        )


    final_predictions = (
        [None] * len(crops)
    )


    final_probabilities = (
        [None] * len(crops)
    )


    for (
        local_index,
        original_index
    ) in enumerate(
        valid_indices
    ):

        final_predictions[
            original_index
        ] = int(
            predictions[
                local_index
            ].item()
        )


        final_probabilities[
            original_index
        ] = float(
            prediction_probabilities[
                local_index
            ].item()
        )


    return (
        final_predictions,
        final_probabilities
    )


# ============================================================
# 13. LOAD COCO VALIDATION ANNOTATIONS
# ============================================================

print("\n" + "=" * 80)
print("LOADING COCO VALIDATION ANNOTATIONS")
print("=" * 80)


with open(
    VAL_JSON,
    "r",
    encoding="utf-8"
) as f:

    original_coco_data = json.load(f)


print(
    "Original COCO images:",
    len(
        original_coco_data["images"]
    )
)


print(
    "Original COCO annotations:",
    len(
        original_coco_data["annotations"]
    )
)


# ============================================================
# 14. BUILD IMAGE ID MAPPING
# ============================================================

filename_to_image_id = {}

image_id_to_filename = {}


for image_info in (
    original_coco_data["images"]
):

    image_id = int(
        image_info["id"]
    )


    file_name = Path(
        image_info["file_name"]
    ).name


    filename_to_image_id[
        file_name
    ] = image_id


    image_id_to_filename[
        image_id
    ] = file_name


# ============================================================
# 15. BUILD 7-CLASS COCO GROUND TRUTH
# ============================================================

print("\n" + "=" * 80)
print("BUILDING 7-CLASS COCO GROUND TRUTH")
print("=" * 80)


evaluation_categories = []


for class_name in MOBILENET_CLASSES:

    evaluation_categories.append(
        {
            "id":
                EVAL_CLASS_TO_COCO_ID[
                    class_name
                ],

            "name":
                class_name,

            "supercategory":
                "waste",
        }
    )


evaluation_images = []


for image_info in (
    original_coco_data["images"]
):

    evaluation_images.append(
        {
            "id":
                int(
                    image_info["id"]
                ),

            "file_name":
                image_info["file_name"],

            "width":
                int(
                    image_info["width"]
                ),

            "height":
                int(
                    image_info["height"]
                ),
        }
    )


evaluation_annotations = []


new_annotation_id = 1


for ann in (
    original_coco_data["annotations"]
):

    original_category_id = int(
        ann["category_id"]
    )


    if (
        original_category_id
        not in COCO_TO_EVAL
    ):

        continue


    evaluation_class = (
        COCO_TO_EVAL[
            original_category_id
        ]
    )


    evaluation_category_id = (
        EVAL_CLASS_TO_COCO_ID[
            evaluation_class
        ]
    )


    bbox = ann["bbox"]


    if len(bbox) != 4:

        continue


    x, y, width, height = (
        bbox
    )


    x = float(x)
    y = float(y)
    width = float(width)
    height = float(height)


    if (
        width <= 0
        or height <= 0
    ):

        continue


    evaluation_annotations.append(
        {
            "id":
                new_annotation_id,

            "image_id":
                int(
                    ann["image_id"]
                ),

            "category_id":
                int(
                    evaluation_category_id
                ),

            "bbox":
                [
                    x,
                    y,
                    width,
                    height
                ],

            "area":
                float(
                    ann.get(
                        "area",
                        width * height
                    )
                ),

            "iscrowd":
                int(
                    ann.get(
                        "iscrowd",
                        0
                    )
                ),
        }
    )


    new_annotation_id += 1


evaluation_coco_dict = {

    "images":
        evaluation_images,

    "annotations":
        evaluation_annotations,

    "categories":
        evaluation_categories,
}


coco_gt = COCO()


coco_gt.dataset = (
    evaluation_coco_dict
)


coco_gt.createIndex()


print(
    "Evaluation GT annotations:",
    len(
        evaluation_annotations
    )
)


print(
    "Evaluation classes:",
    len(
        evaluation_categories
    )
)


# ============================================================
# 16. VALIDATION IMAGES
# ============================================================

image_paths = sorted(

    [

        p

        for p in VAL_IMAGES.iterdir()

        if (
            p.is_file()
            and p.suffix.lower()
            in IMAGE_EXTENSIONS
        )
    ]
)


print("\n" + "=" * 80)
print("VALIDATION DATA")
print("=" * 80)


print(
    "Validation images:",
    len(image_paths)
)


if len(image_paths) == 0:

    raise RuntimeError(
        "No validation images found."
    )


# ------------------------------------------------------------
# IMPORTANT IMAGE COUNT CHECK
# ------------------------------------------------------------

if (
    len(image_paths)
    != len(original_coco_data["images"])
):

    print(
        "\nWARNING:"
    )


    print(
        "Number of files in validation image "
        "directory does not equal number of images "
        "in COCO JSON."
    )


    print(
        "Directory images:",
        len(image_paths)
    )


    print(
        "COCO images:",
        len(
            original_coco_data["images"]
        )
    )


# ============================================================
# 17. STORAGE
# ============================================================

all_records = []

coco_predictions = []

matched_true = []

matched_pred = []

matched_records = []


gt_object_count = 0

yolo_detection_count = 0

mobilenet_prediction_count = 0

matched_detection_count = 0

correct_final_class_count = 0

images_without_coco_id = 0

invalid_prediction_boxes = 0


# ============================================================
# 18. END-TO-END EVALUATION
# ============================================================

print("\n" + "=" * 80)
print("STARTING E3Y-B END-TO-END EVALUATION")
print("=" * 80)


print(
    "\nYOLO confidence threshold:",
    YOLO_CONF
)


print(
    "YOLO NMS IoU:",
    YOLO_NMS_IOU
)


print(
    "\nFinal confidence formula:"
)


print(
    "YOLO confidence x MobileNet class probability"
)


print(
    "\nDiagnostic matching IoU threshold:",
    MATCH_IOU_THRESHOLD
)


evaluation_start = time.time()


for (
    image_index,
    image_path
) in enumerate(
    image_paths,
    start=1
):


    # ========================================================
    # READ IMAGE
    # ========================================================

    image = cv2.imread(
        str(image_path)
    )


    if image is None:

        print(
            "WARNING: Could not read:",
            image_path
        )

        continue


    image_height, image_width = (
        image.shape[:2]
    )


    image_name = image_path.name


    # ========================================================
    # COCO IMAGE ID
    # ========================================================

    image_id = (
        filename_to_image_id.get(
            image_name
        )
    )


    if image_id is None:

        images_without_coco_id += 1

        print(
            "WARNING: Image not found "
            "in COCO annotations:",
            image_name
        )

        continue


    # ========================================================
    # GROUND TRUTH FROM COCO
    # ========================================================
    #
    # Primary COCO evaluation uses COCO annotations
    # directly, rather than rebuilding GT from YOLO labels.
    #
    # ========================================================

    coco_gt_annotations = (
        coco_gt.imgToAnns.get(
            image_id,
            []
        )
    )


    gt_boxes = []


    for ann in coco_gt_annotations:

        bbox = ann["bbox"]


        x = float(bbox[0])
        y = float(bbox[1])

        width = float(bbox[2])
        height = float(bbox[3])


        gt_box = [

            x,

            y,

            x + width,

            y + height,
        ]


        gt_boxes.append(
            {
                "annotation_id":
                    int(
                        ann["id"]
                    ),

                "box":
                    gt_box,

                "class_name":
                    EVAL_COCO_ID_TO_CLASS[
                        int(
                            ann["category_id"]
                        )
                    ],

                "eval_category_id":
                    int(
                        ann["category_id"]
                    ),
            }
        )


    gt_object_count += len(
        gt_boxes
    )


    # ========================================================
    # YOLO DETECTION
    # ========================================================

    results = yolo_model.predict(

        source=str(
            image_path
        ),

        device=DEVICE,

        verbose=False,

        conf=YOLO_CONF,

        iou=YOLO_NMS_IOU,
    )


    result = results[0]


    if (
        result.boxes is None
        or len(result.boxes) == 0
    ):

        if (
            image_index % 50 == 0
            or image_index
            == len(image_paths)
        ):

            print(
                f"Processed "
                f"{image_index}/"
                f"{len(image_paths)} | "
                f"YOLO detections: "
                f"{yolo_detection_count:,} | "
                f"MobileNet predictions: "
                f"{mobilenet_prediction_count:,}"
            )

        continue


    predictions = result.boxes


    yolo_detection_count += len(
        predictions
    )


    # ========================================================
    # PREPARE CROPS
    # ========================================================

    detection_records = []

    crops = []


    for pred_index in range(
        len(predictions)
    ):


        # ----------------------------------------------------
        # YOLO bbox
        # ----------------------------------------------------

        pred_box = (
            predictions.xyxy[
                pred_index
            ]
            .detach()
            .cpu()
            .numpy()
            .astype(
                float
            )
        )


        pred_class = int(
            predictions.cls[
                pred_index
            ].item()
        )


        yolo_confidence = float(
            predictions.conf[
                pred_index
            ].item()
        )


        if (
            pred_class
            not in YOLO_TO_EVAL
        ):

            continue


        # ----------------------------------------------------
        # Clip bbox
        # ----------------------------------------------------

        x1 = max(
            0.0,
            min(
                pred_box[0],
                image_width - 1
            )
        )


        y1 = max(
            0.0,
            min(
                pred_box[1],
                image_height - 1
            )
        )


        x2 = max(
            0.0,
            min(
                pred_box[2],
                image_width
            )
        )


        y2 = max(
            0.0,
            min(
                pred_box[3],
                image_height
            )
        )


        if (
            x2 <= x1
            or y2 <= y1
        ):

            invalid_prediction_boxes += 1

            continue


        # ----------------------------------------------------
        # Crop using current detection bbox
        # ----------------------------------------------------

        crop = image[
            int(y1):int(y2),
            int(x1):int(x2)
        ]


        if crop.size == 0:

            invalid_prediction_boxes += 1

            continue


        # ----------------------------------------------------
        # STORE CURRENT DETECTION BBOX
        #
        # This is the critical correction.
        # ----------------------------------------------------

        detection_record = {

            "image":
                image_name,

            "source_image":
                str(
                    image_path
                ),

            "image_id":
                int(
                    image_id
                ),

            "detection_index":
                int(
                    pred_index
                ),

            "yolo_pred_class_id":
                int(
                    pred_class
                ),

            "yolo_pred_class_name":
                ORIGINAL_YOLO_NAMES[
                    pred_class
                ],

            "yolo_eval_class_name":
                YOLO_TO_EVAL[
                    pred_class
                ],

            "yolo_confidence":
                yolo_confidence,

            "x1":
                x1,

            "y1":
                y1,

            "x2":
                x2,

            "y2":
                y2,

            "bbox_width":
                x2 - x1,

            "bbox_height":
                y2 - y1,
        }


        # ----------------------------------------------------
        # Diagnostic best IoU
        #
        # This is NOT used by COCOeval.
        # ----------------------------------------------------

        best_iou = 0.0

        best_gt_index = None


        for (
            gt_index,
            gt
        ) in enumerate(
            gt_boxes
        ):

            iou = calculate_iou(

                [
                    x1,
                    y1,
                    x2,
                    y2
                ],

                gt["box"]
            )


            if iou > best_iou:

                best_iou = iou

                best_gt_index = (
                    gt_index
                )


        detection_record[
            "best_gt_iou"
        ] = best_iou


        detection_record[
            "best_gt_index"
        ] = best_gt_index


        detection_records.append(
            detection_record
        )


        crops.append(
            crop
        )


    # ========================================================
    # MOBILENET CLASSIFICATION
    # ========================================================

    predictions_mobilenet = []

    probabilities_mobilenet = []


    for start in range(

        0,

        len(crops),

        BATCH_SIZE
    ):


        batch_crops = crops[

            start:
            start + BATCH_SIZE
        ]


        (
            batch_predictions,
            batch_probabilities
        ) = classify_crops(
            batch_crops
        )


        predictions_mobilenet.extend(
            batch_predictions
        )


        probabilities_mobilenet.extend(
            batch_probabilities
        )


    # ========================================================
    # ATTACH MOBILENET RESULTS
    # ========================================================

    for (

        record,

        mobile_prediction,

        mobile_probability

    ) in zip(

        detection_records,

        predictions_mobilenet,

        probabilities_mobilenet
    ):


        if mobile_prediction is None:

            continue


        if mobile_probability is None:

            continue


        mobilenet_prediction_count += 1


        # ----------------------------------------------------
        # MobileNet prediction
        # ----------------------------------------------------

        mobile_class_name = (
            MOBILENET_CLASSES[
                mobile_prediction
            ]
        )


        mobile_probability = float(
            mobile_probability
        )


        # ----------------------------------------------------
        # OPTION B
        # ----------------------------------------------------

        yolo_confidence = float(
            record[
                "yolo_confidence"
            ]
        )


        final_confidence = (

            yolo_confidence

            *

            mobile_probability
        )


        final_eval_category_id = (
            EVAL_CLASS_TO_COCO_ID[
                mobile_class_name
            ]
        )


        # ----------------------------------------------------
        # Store MobileNet results
        # ----------------------------------------------------

        record[
            "mobilenet_pred_class_id"
        ] = int(
            mobile_prediction
        )


        record[
            "mobilenet_pred_class_name"
        ] = mobile_class_name


        record[
            "mobilenet_probability"
        ] = mobile_probability


        record[
            "final_class_name"
        ] = mobile_class_name


        record[
            "final_eval_category_id"
        ] = int(
            final_eval_category_id
        )


        record[
            "final_confidence"
        ] = float(
            final_confidence
        )


        # ====================================================
        # COCO PREDICTION
        #
        # CRITICAL:
        #
        # bbox comes from THIS record.
        #
        # We do NOT use stale x1/y1/x2/y2 variables.
        # ====================================================

        pred_x1 = float(
            record["x1"]
        )


        pred_y1 = float(
            record["y1"]
        )


        pred_x2 = float(
            record["x2"]
        )


        pred_y2 = float(
            record["y2"]
        )


        pred_width = (
            pred_x2
            -
            pred_x1
        )


        pred_height = (
            pred_y2
            -
            pred_y1
        )


        if (
            pred_width <= 0
            or pred_height <= 0
        ):

            invalid_prediction_boxes += 1

            continue


        coco_prediction = {

            "image_id":
                int(
                    image_id
                ),

            "category_id":
                int(
                    final_eval_category_id
                ),

            "bbox":
                [
                    pred_x1,
                    pred_y1,
                    pred_width,
                    pred_height
                ],

            "score":
                float(
                    final_confidence
                ),
        }


        coco_predictions.append(
            coco_prediction
        )


        # ====================================================
        # Store record
        # ====================================================

        all_records.append(
            record
        )


    # ========================================================
    # PROGRESS
    # ========================================================

    if (

        image_index % 50 == 0

        or image_index
        == len(image_paths)
    ):

        print(

            f"Processed "
            f"{image_index}/"
            f"{len(image_paths)} | "

            f"YOLO detections: "
            f"{yolo_detection_count:,} | "

            f"MobileNet predictions: "
            f"{mobilenet_prediction_count:,}"
        )


# ============================================================
# 19. SANITY CHECKS BEFORE COCO EVALUATION
# ============================================================

print("\n" + "=" * 80)
print("PRE-COCO SANITY CHECKS")
print("=" * 80)


print(
    "Validation images:",
    len(image_paths)
)


print(
    "COCO images:",
    len(
        original_coco_data["images"]
    )
)


print(
    "Ground-truth objects:",
    gt_object_count
)


print(
    "YOLO detections:",
    yolo_detection_count
)


print(
    "MobileNet predictions:",
    mobilenet_prediction_count
)


print(
    "COCO predictions:",
    len(coco_predictions)
)


print(
    "Images without COCO ID:",
    images_without_coco_id
)


print(
    "Invalid prediction boxes:",
    invalid_prediction_boxes
)


if (
    images_without_coco_id > 0
):

    raise RuntimeError(
        "Some validation images could not be "
        "mapped to COCO image IDs."
    )


if (
    len(coco_predictions)
    != mobilenet_prediction_count
):

    print(
        "\nWARNING:"
    )


    print(
        "COCO prediction count differs from "
        "MobileNet prediction count."
    )


# ------------------------------------------------------------
# Check all prediction category IDs
# ------------------------------------------------------------

invalid_category_ids = [

    pred["category_id"]

    for pred in coco_predictions

    if (
        pred["category_id"]
        not in EVAL_COCO_ID_TO_CLASS
    )
]


if invalid_category_ids:

    raise RuntimeError(
        "Invalid COCO category IDs detected."
    )


# ------------------------------------------------------------
# Check all prediction bboxes
# ------------------------------------------------------------

invalid_boxes = []


for pred in coco_predictions:

    bbox = pred["bbox"]


    if (

        len(bbox) != 4

        or bbox[2] <= 0

        or bbox[3] <= 0

    ):

        invalid_boxes.append(
            pred
        )


if invalid_boxes:

    raise RuntimeError(
        "Invalid COCO prediction bounding boxes detected."
    )


print(
    "\nSanity checks completed."
)


# ============================================================
# 20. SAVE RAW DETECTION RESULTS
# ============================================================

print("\n" + "=" * 80)
print("SAVING RAW DETECTION RESULTS")
print("=" * 80)


if all_records:

    fieldnames = list(
        all_records[0].keys()
    )


    with open(

        RAW_RESULTS_CSV,

        "w",

        newline="",

        encoding="utf-8"

    ) as f:


        writer = csv.DictWriter(

            f,

            fieldnames=fieldnames
        )


        writer.writeheader()


        writer.writerows(
            all_records
        )


    print(
        "Raw detection results saved:"
    )


    print(
        RAW_RESULTS_CSV
    )


# ============================================================
# 21. COCO EVALUATION
# ============================================================

print("\n" + "=" * 80)
print("COCO EVALUATION")
print("=" * 80)


if len(coco_predictions) == 0:

    raise RuntimeError(
        "No valid E3Y-B predictions were generated."
    )


print(
    "COCO predictions:",
    len(coco_predictions)
)


# ------------------------------------------------------------
# Load predictions
# ------------------------------------------------------------

coco_dt = coco_gt.loadRes(
    coco_predictions
)


# ------------------------------------------------------------
# COCO evaluator
# ------------------------------------------------------------

coco_eval = COCOeval(

    coco_gt,

    coco_dt,

    "bbox"
)


coco_eval.evaluate()

coco_eval.accumulate()

coco_eval.summarize()


# ============================================================
# 22. STANDARD COCO METRICS
# ============================================================

map5095 = float(
    coco_eval.stats[0]
)


map50 = float(
    coco_eval.stats[1]
)


map75 = float(
    coco_eval.stats[2]
)


mar1 = float(
    coco_eval.stats[6]
)


mar10 = float(
    coco_eval.stats[7]
)


mar100 = float(
    coco_eval.stats[8]
)


print("\n" + "=" * 80)
print("FINAL E3Y-B COCO METRICS")
print("=" * 80)


print(
    f"mAP50-95 : {map5095:.6f}"
)


print(
    f"mAP50-95 : {map5095 * 100:.2f}%"
)


print(
    f"mAP50    : {map50:.6f}"
)


print(
    f"mAP50    : {map50 * 100:.2f}%"
)


print(
    f"mAP75    : {map75:.6f}"
)


print(
    f"AR@1     : {mar1:.6f}"
)


print(
    f"AR@10    : {mar10:.6f}"
)


print(
    f"AR@100   : {mar100:.6f}"
)


# ============================================================
# 23. PER-CLASS COCO AP
# ============================================================

print("\n" + "=" * 80)
print("PER-CLASS COCO AP")
print("=" * 80)


precision_tensor = (
    coco_eval.eval[
        "precision"
    ]
)


class_results = []


for class_index, category_id in enumerate(
    coco_eval.params.catIds
):


    class_name = (
        EVAL_COCO_ID_TO_CLASS[
            category_id
        ]
    )


    # --------------------------------------------------------
    # AP50
    # --------------------------------------------------------

    precision_ap50 = (
        precision_tensor[
            0,
            :,
            class_index,
            0,
            -1
        ]
    )


    precision_ap50 = (
        precision_ap50[
            precision_ap50 > -1
        ]
    )


    if len(precision_ap50) > 0:

        class_ap50 = float(
            np.mean(
                precision_ap50
            )
        )

    else:

        class_ap50 = 0.0


    # --------------------------------------------------------
    # AP50-95
    # --------------------------------------------------------

    precision_ap5095 = (
        precision_tensor[
            :,
            :,
            class_index,
            0,
            -1
        ]
    )


    precision_ap5095 = (
        precision_ap5095[
            precision_ap5095 > -1
        ]
    )


    if len(precision_ap5095) > 0:

        class_ap5095 = float(
            np.mean(
                precision_ap5095
            )
        )

    else:

        class_ap5095 = 0.0


    # --------------------------------------------------------
    # GT count
    # --------------------------------------------------------

    gt_count = sum(

        1

        for ann
        in evaluation_annotations

        if (
            ann["category_id"]
            == category_id
        )
    )


    # --------------------------------------------------------
    # Prediction count
    # --------------------------------------------------------

    prediction_count = sum(

        1

        for pred
        in coco_predictions

        if (
            pred["category_id"]
            == category_id
        )
    )


    class_results.append(

        {

            "class":
                class_name,

            "COCO_category_id":
                category_id,

            "AP50":
                class_ap50,

            "AP50_95":
                class_ap5095,

            "GT_count":
                gt_count,

            "predictions":
                prediction_count,
        }
    )


    print(

        f"{class_name:25s} "

        f"AP50={class_ap50:.6f} "

        f"AP50-95={class_ap5095:.6f} "

        f"GT={gt_count:6d} "

        f"Pred={prediction_count:6d}"
    )


# ============================================================
# 24. SAVE COCO RESULTS
# ============================================================

with open(

    COCO_RESULTS_CSV,

    "w",

    newline="",

    encoding="utf-8"

) as f:


    writer = csv.writer(
        f
    )


    writer.writerow(
        [
            "metric",
            "value"
        ]
    )


    writer.writerow(
        [
            "mAP50",
            map50
        ]
    )


    writer.writerow(
        [
            "mAP50_95",
            map5095
        ]
    )


    writer.writerow(
        [
            "mAP75",
            map75
        ]
    )


    writer.writerow(
        [
            "AR@1",
            mar1
        ]
    )


    writer.writerow(
        [
            "AR@10",
            mar10
        ]
    )


    writer.writerow(
        [
            "AR@100",
            mar100
        ]
    )


print(
    "\nCOCO summary saved:"
)


print(
    COCO_RESULTS_CSV
)


# ============================================================
# 25. SAVE PER-CLASS RESULTS
# ============================================================

with open(

    CLASS_RESULTS_CSV,

    "w",

    newline="",

    encoding="utf-8"

) as f:


    writer = csv.DictWriter(

        f,

        fieldnames=[

            "class",

            "COCO_category_id",

            "AP50",

            "AP50_95",

            "GT_count",

            "predictions",
        ]
    )


    writer.writeheader()


    writer.writerows(
        class_results
    )


print(
    "Per-class results saved:"
)


print(
    CLASS_RESULTS_CSV
)


# ============================================================
# 26. PROPER ONE-TO-ONE MATCHING
# ============================================================
#
# This is ONLY for diagnostic classification metrics.
#
# COCOeval remains the authoritative detection evaluation.
#
# Matching strategy:
#
#   1. Sort predictions by final confidence descending.
#   2. For each prediction:
#        - find unmatched GT with highest IoU
#        - require IoU >= 0.50
#   3. Once a GT is matched, it cannot be matched again.
#
# ============================================================

print("\n" + "=" * 80)
print("PROPER ONE-TO-ONE DIAGNOSTIC MATCHING")
print("=" * 80)


# ------------------------------------------------------------
# Group detection records by image
# ------------------------------------------------------------

records_by_image = {}


for record in all_records:

    image_id = int(
        record["image_id"]
    )


    records_by_image.setdefault(
        image_id,
        []
    ).append(
        record
    )


# ------------------------------------------------------------
# Build GT by image
# ------------------------------------------------------------

gt_by_image = {}


for gt in evaluation_annotations:

    image_id = int(
        gt["image_id"]
    )


    bbox = gt["bbox"]


    gt_box = [

        float(bbox[0]),

        float(bbox[1]),

        float(
            bbox[0] + bbox[2]
        ),

        float(
            bbox[1] + bbox[3]
        ),
    ]


    gt_record = {

        "annotation_id":
            int(
                gt["id"]
            ),

        "box":
            gt_box,

        "category_id":
            int(
                gt["category_id"]
            ),

        "class_name":
            EVAL_COCO_ID_TO_CLASS[
                int(
                    gt["category_id"]
                )
            ],
    }


    gt_by_image.setdefault(
        image_id,
        []
    ).append(
        gt_record
    )


# ------------------------------------------------------------
# Match
# ------------------------------------------------------------

for image_id in (
    sorted(
        set(
            list(
                records_by_image.keys()
            )
            +
            list(
                gt_by_image.keys()
            )
        )
    )
):


    image_records = (
        records_by_image.get(
            image_id,
            []
        )
    )


    image_gt = (
        gt_by_image.get(
            image_id,
            []
        )
    )


    # --------------------------------------------------------
    # Sort by confidence
    # --------------------------------------------------------

    image_records = sorted(

        image_records,

        key=lambda r:
            float(
                r[
                    "final_confidence"
                ]
            ),

        reverse=True
    )


    matched_gt_indices = set()


    for record in image_records:


        pred_box = [

            float(
                record["x1"]
            ),

            float(
                record["y1"]
            ),

            float(
                record["x2"]
            ),

            float(
                record["y2"]
            ),
        ]


        best_iou = 0.0

        best_gt_index = None


        for gt_index, gt in enumerate(
            image_gt
        ):

            if gt_index in matched_gt_indices:

                continue


            iou = calculate_iou(

                pred_box,

                gt["box"]
            )


            if iou > best_iou:

                best_iou = iou

                best_gt_index = (
                    gt_index
                )


        # ----------------------------------------------------
        # Successful one-to-one match
        # ----------------------------------------------------

        if (

            best_gt_index is not None

            and best_iou
            >= MATCH_IOU_THRESHOLD

        ):


            matched_gt_indices.add(
                best_gt_index
            )


            gt = image_gt[
                best_gt_index
            ]


            true_class_id = int(
                gt["category_id"]
            )


            pred_class_id = int(
                record[
                    "final_eval_category_id"
                ]
            )


            matched_true.append(
                true_class_id
            )


            matched_pred.append(
                pred_class_id
            )


            matched_record = {

                "image":
                    record["image"],

                "image_id":
                    image_id,

                "detection_index":
                    record[
                        "detection_index"
                    ],

                "IoU":
                    best_iou,

                "GT_class":
                    gt[
                        "class_name"
                    ],

                "Predicted_class":
                    record[
                        "final_class_name"
                    ],

                "YOLO_confidence":
                    record[
                        "yolo_confidence"
                    ],

                "MobileNet_probability":
                    record[
                        "mobilenet_probability"
                    ],

                "Final_confidence":
                    record[
                        "final_confidence"
                    ],

                "Correct":
                    (
                        true_class_id
                        ==
                        pred_class_id
                    ),
            }


            matched_records.append(
                matched_record
            )


# ============================================================
# 27. MATCHED CLASSIFICATION METRICS
# ============================================================

matched_accuracy = 0.0

matched_macro_f1 = 0.0

matched_weighted_f1 = 0.0


if len(matched_true) > 0:


    matched_accuracy = (
        accuracy_score(
            matched_true,
            matched_pred
        )
    )


    matched_macro_f1 = (
        f1_score(

            matched_true,

            matched_pred,

            labels=list(
                range(
                    1,
                    NUM_EVAL_CLASSES + 1
                )
            ),

            average="macro",

            zero_division=0
        )
    )


    matched_weighted_f1 = (
        f1_score(

            matched_true,

            matched_pred,

            labels=list(
                range(
                    1,
                    NUM_EVAL_CLASSES + 1
                )
            ),

            average="weighted",

            zero_division=0
        )
    )


    print(
        f"Matched samples : "
        f"{len(matched_true):,}"
    )


    print(
        f"Accuracy        : "
        f"{matched_accuracy:.6f}"
    )


    print(
        f"Macro F1        : "
        f"{matched_macro_f1:.6f}"
    )


    print(
        f"Weighted F1     : "
        f"{matched_weighted_f1:.6f}"
    )


    print(
        "\nClassification report:"
    )


    report = classification_report(

        matched_true,

        matched_pred,

        labels=list(
            range(
                1,
                NUM_EVAL_CLASSES + 1
            )
        ),

        target_names=
            MOBILENET_CLASSES,

        zero_division=0
    )


    print(
        report
    )


    # --------------------------------------------------------
    # Confusion matrix
    # --------------------------------------------------------

    cm = confusion_matrix(

        matched_true,

        matched_pred,

        labels=list(
            range(
                1,
                NUM_EVAL_CLASSES + 1
            )
        )
    )


    print(
        "\nConfusion matrix:"
    )


    print(
        " " * 25,

        *[
            f"{name:20s}"

            for name
            in MOBILENET_CLASSES
        ]
    )


    for i, row in enumerate(cm):

        print(

            f"{MOBILENET_CLASSES[i]:25s}",

            *[
                f"{value:20d}"

                for value
                in row
            ]
        )


    # --------------------------------------------------------
    # Save confusion matrix
    # --------------------------------------------------------

    with open(

        CONFUSION_MATRIX_CSV,

        "w",

        newline="",

        encoding="utf-8"

    ) as f:


        writer = csv.writer(
            f
        )


        writer.writerow(

            [
                "actual \\ predicted"
            ]

            +
            MOBILENET_CLASSES
        )


        for i, row in enumerate(cm):

            writer.writerow(

                [
                    MOBILENET_CLASSES[i]
                ]

                +
                list(row)
            )


# ============================================================
# 28. SAVE MATCHED RESULTS
# ============================================================

if matched_records:

    with open(

        MATCHED_RESULTS_CSV,

        "w",

        newline="",

        encoding="utf-8"

    ) as f:


        fieldnames = list(
            matched_records[0].keys()
        )


        writer = csv.DictWriter(

            f,

            fieldnames=fieldnames
        )


        writer.writeheader()


        writer.writerows(
            matched_records
        )


    print(
        "\nMatched diagnostic results saved:"
    )


    print(
        MATCHED_RESULTS_CSV
    )


# ============================================================
# 29. PLASTIC-ONLY CLASSIFICATION
# ============================================================

print("\n" + "=" * 80)
print("PLASTIC-ONLY END-TO-END CLASSIFICATION")
print("=" * 80)


plastic_class_ids = {

    CLASS_TO_ID["ecal"],

    CLASS_TO_ID["hdpe"],

    CLASS_TO_ID[
        "mixed_plastic_rigid"
    ],

    CLASS_TO_ID[
        "mixed_plastic_soft"
    ],

    CLASS_TO_ID["pet"],

    CLASS_TO_ID["pet_oil"],
}


plastic_true = []

plastic_pred = []


for true, pred in zip(

    matched_true,

    matched_pred

):


    if true in plastic_class_ids:

        plastic_true.append(
            true
        )

        plastic_pred.append(
            pred
        )


plastic_accuracy = 0.0

plastic_macro_f1 = 0.0

plastic_weighted_f1 = 0.0


if len(plastic_true) > 0:


    plastic_accuracy = (
        accuracy_score(

            plastic_true,

            plastic_pred
        )
    )


    plastic_macro_f1 = (
        f1_score(

            plastic_true,

            plastic_pred,

            labels=sorted(
                plastic_class_ids
            ),

            average="macro",

            zero_division=0
        )
    )


    plastic_weighted_f1 = (
        f1_score(

            plastic_true,

            plastic_pred,

            labels=sorted(
                plastic_class_ids
            ),

            average="weighted",

            zero_division=0
        )
    )


    print(
        f"Plastic samples     : "
        f"{len(plastic_true):,}"
    )


    print(
        f"Plastic accuracy    : "
        f"{plastic_accuracy:.6f}"
    )


    print(
        f"Plastic Macro F1    : "
        f"{plastic_macro_f1:.6f}"
    )


    print(
        f"Plastic Weighted F1 : "
        f"{plastic_weighted_f1:.6f}"
    )


# ============================================================
# 30. FINAL COUNTS
# ============================================================

evaluation_time = (
    time.time()
    -
    evaluation_start
)


print("\n" + "=" * 80)
print("E3Y-B FINAL SUMMARY")
print("=" * 80)


print(
    f"\nValidation images       : "
    f"{len(image_paths):,}"
)


print(
    f"Ground-truth objects    : "
    f"{gt_object_count:,}"
)


print(
    f"YOLO detections         : "
    f"{yolo_detection_count:,}"
)


print(
    f"MobileNet predictions   : "
    f"{mobilenet_prediction_count:,}"
)


print(
    f"COCO predictions        : "
    f"{len(coco_predictions):,}"
)


print(
    f"Proper GT matches       : "
    f"{len(matched_true):,}"
)


print(
    f"Correct final classes   : "
    f"{sum(1 for t, p in zip(matched_true, matched_pred) if t == p):,}"
)


print(
    f"\nE3Y-B mAP50             : "
    f"{map50:.6f}"
)


print(
    f"E3Y-B mAP50             : "
    f"{map50 * 100:.2f}%"
)


print(
    f"\nE3Y-B mAP50-95          : "
    f"{map5095:.6f}"
)


print(
    f"E3Y-B mAP50-95          : "
    f"{map5095 * 100:.2f}%"
)


if len(matched_true) > 0:

    print(
        f"\nMatched classification accuracy : "
        f"{matched_accuracy:.6f}"
    )


    print(
        f"Matched Macro F1                : "
        f"{matched_macro_f1:.6f}"
    )


    print(
        f"Matched Weighted F1             : "
        f"{matched_weighted_f1:.6f}"
    )


if len(plastic_true) > 0:

    print(
        f"\nPlastic-only accuracy           : "
        f"{plastic_accuracy:.6f}"
    )


    print(
        f"Plastic-only Macro F1           : "
        f"{plastic_macro_f1:.6f}"
    )


    print(
        f"Plastic-only Weighted F1        : "
        f"{plastic_weighted_f1:.6f}"
    )


print(
    f"\nEvaluation time        : "
    f"{evaluation_time / 60:.2f} minutes"
)


print(
    "\nResults directory:"
)


print(
    OUTPUT_ROOT
)


# ============================================================
# 31. SAVE FINAL SUMMARY
# ============================================================

with open(

    SUMMARY_TXT,

    "w",

    encoding="utf-8"

) as f:


    f.write(
        "E3Y-B — CORRECTED FINAL END-TO-END "
        "COCO EVALUATION\n"
    )


    f.write(
        "=" * 70
        +
        "\n\n"
    )


    f.write(
        "PIPELINE\n"
    )


    f.write(
        "YOLO11n -> object crop -> "
        "MobileNet-V3-Large\n\n"
    )


    f.write(
        "EXPERIMENT\n"
    )


    f.write(
        "E3Y-B — MobileNet-V3-Large "
        "with class-weighted loss\n\n"
    )


    f.write(
        "OPTION B CONFIDENCE\n"
    )


    f.write(
        "Final confidence = "
        "YOLO confidence x "
        "MobileNet predicted-class probability\n\n"
    )


    f.write(
        "EVALUATION TAXONOMY\n"
    )


    f.write(
        "Original 8 SortWaste classes -> "
        "7 evaluation classes\n"
    )


    f.write(
        "Metal + cardboard -> non_plastic\n\n"
    )


    f.write(
        "DATASET\n"
    )


    f.write(
        f"Validation images: "
        f"{len(image_paths)}\n"
    )


    f.write(
        f"Ground truth objects: "
        f"{gt_object_count}\n"
    )


    f.write(
        "\nPREDICTIONS\n"
    )


    f.write(
        f"YOLO detections: "
        f"{yolo_detection_count}\n"
    )


    f.write(
        f"MobileNet predictions: "
        f"{mobilenet_prediction_count}\n"
    )


    f.write(
        f"COCO predictions: "
        f"{len(coco_predictions)}\n"
    )


    f.write(
        "\nPRIMARY COCO DETECTION METRICS\n"
    )


    f.write(
        f"mAP50: "
        f"{map50:.6f}\n"
    )


    f.write(
        f"mAP50-95: "
        f"{map5095:.6f}\n"
    )


    f.write(
        f"mAP75: "
        f"{map75:.6f}\n"
    )


    f.write(
        f"AR@1: "
        f"{mar1:.6f}\n"
    )


    f.write(
        f"AR@10: "
        f"{mar10:.6f}\n"
    )


    f.write(
        f"AR@100: "
        f"{mar100:.6f}\n"
    )


    f.write(
        "\nDIAGNOSTIC ONE-TO-ONE CLASSIFICATION\n"
    )


    f.write(
        f"IoU threshold: "
        f"{MATCH_IOU_THRESHOLD:.2f}\n"
    )


    f.write(
        f"Matched samples: "
        f"{len(matched_true)}\n"
    )


    f.write(
        f"Accuracy: "
        f"{matched_accuracy:.6f}\n"
    )


    f.write(
        f"Macro F1: "
        f"{matched_macro_f1:.6f}\n"
    )


    f.write(
        f"Weighted F1: "
        f"{matched_weighted_f1:.6f}\n"
    )


    f.write(
        "\nPLASTIC-ONLY CLASSIFICATION\n"
    )


    f.write(
        f"Plastic samples: "
        f"{len(plastic_true)}\n"
    )


    f.write(
        f"Plastic accuracy: "
        f"{plastic_accuracy:.6f}\n"
    )


    f.write(
        f"Plastic Macro F1: "
        f"{plastic_macro_f1:.6f}\n"
    )


    f.write(
        f"Plastic Weighted F1: "
        f"{plastic_weighted_f1:.6f}\n"
    )


    f.write(
        "\nSANITY CHECKS\n"
    )


    f.write(
        f"Images without COCO ID: "
        f"{images_without_coco_id}\n"
    )


    f.write(
        f"Invalid prediction boxes: "
        f"{invalid_prediction_boxes}\n"
    )


    f.write(
        "\nMETHODOLOGICAL NOTES\n"
    )


    f.write(
        "1. YOLO provides the bounding boxes.\n"
    )


    f.write(
        "2. MobileNet provides the final class.\n"
    )


    f.write(
        "3. Final confidence uses Option B.\n"
    )


    f.write(
        "4. COCO annotations are used as the "
        "primary ground truth source.\n"
    )


    f.write(
        "5. Original 8-class COCO annotations "
        "are remapped to the 7-class evaluation "
        "taxonomy.\n"
    )


    f.write(
        "6. Metal and cardboard are mapped to "
        "non_plastic.\n"
    )


    f.write(
        "7. No random augmentation is used "
        "during evaluation.\n"
    )


    f.write(
        "8. COCOeval is the primary end-to-end "
        "detection evaluation.\n"
    )


    f.write(
        "9. Classification metrics use "
        "one-to-one IoU matching and are "
        "diagnostic.\n"
    )


    f.write(
        "10. Each GT object can be matched to "
        "at most one prediction.\n"
    )


    f.write(
        "11. E3Y-B uses the class-weighted "
        "MobileNet-V3-Large checkpoint.\n"
    )


# ============================================================
# 32. FINAL
# ============================================================

print("\n" + "=" * 80)
print("IMPORTANT METHODOLOGICAL NOTES")
print("=" * 80)


print(
    "\n1. E3Y-B uses all YOLO validation detections "
    "above conf=0.001."
)


print(
    "2. YOLO supplies the bounding box."
)


print(
    "3. MobileNet supplies the final class."
)


print(
    "4. Final confidence = YOLO confidence "
    "x MobileNet predicted-class probability."
)


print(
    "5. COCOeval is the primary end-to-end "
    "detection evaluation."
)


print(
    "6. COCO GT is remapped from 8 original "
    "classes to 7 evaluation classes."
)


print(
    "7. Metal + cardboard -> non_plastic."
)


print(
    "8. Diagnostic classification uses proper "
    "one-to-one IoU matching."
)


print(
    "9. No random augmentation is applied "
    "during evaluation."
)


print(
    "10. E3Y-B uses the class-weighted "
    "MobileNet-V3-Large checkpoint."
)


print(
    "\nE3Y-B corrected evaluation complete."
)


print("=" * 80)

E3Y-B — YOLO11n -> MobileNet-V3-Large
CORRECTED FINAL END-TO-END COCO EVALUATION

Device: cuda
GPU: NVIDIA GeForce RTX 3050 Ti Laptop GPU
CUDA: 12.6

CHECKING PATHS
Dataset root             : C:\Users\varda\Documents\_My Computer\COMMON_space\Learning\Upgrad_EPGP_MS\upgrad_IIITB_MS_course\MS_LJMU_Material\Topic Data\SortWaste\dataset\dataset
                           Exists: True
Validation images        : C:\Users\varda\Documents\_My Computer\COMMON_space\Learning\Upgrad_EPGP_MS\upgrad_IIITB_MS_course\MS_LJMU_Material\Topic Data\SortWaste\dataset\dataset\splited_all_dataset_coco\val\images
                           Exists: True
Validation labels        : C:\Users\varda\Documents\_My Computer\COMMON_space\Learning\Upgrad_EPGP_MS\upgrad_IIITB_MS_course\MS_LJMU_Material\Topic Data\SortWaste\dataset\dataset\splited_all_dataset_coco\val\labels
                           Exists: True
Validation COCO JSON     : C:\Users\varda\Documents\_My Computer\COMMON_space\Learning\Upgrad_EPGP_MS\upgr

# Model E4 — Faster R-CNN + MobileNet-V3-Large.

## E4F-A : Faster R-CNN crops → MobileNet baseline

In [ ]:
### GT-matched Faster R-CNN crops
# ============================================================
# E4-A — FASTER R-CNN -> MOBILENET
# STAGE 1: FASTER R-CNN DETECTION CROP CREATION
# ============================================================
#
# Purpose:
#   1. Load the trained Faster R-CNN detector
#   2. Run inference on SortWaste train/validation images
#   3. Match Faster R-CNN detections to COCO GT objects
#   4. Create object crops from Faster R-CNN bounding boxes
#   5. Assign the GT class to each matched crop
#   6. Merge Metal + Cardboard -> non_plastic
#
# IMPORTANT:
#   MobileNet will NOT be trained in this script.
#
#   This script ONLY creates the Faster R-CNN crops.
#
# Pipeline:
#
#   SortWaste image
#          |
#          v
#   Faster R-CNN
#          |
#          v
#   Detection bounding box
#          |
#          v
#   IoU matching against COCO GT
#          |
#          v
#   GT class label
#          |
#          v
#   Object crop
#          |
#          v
#   class folder
#
# ============================================================


# ============================================================
# 1. IMPORTS
# ============================================================

import os
import json
import csv
import time
import shutil
import random

from pathlib import Path
from collections import defaultdict

import numpy as np
from PIL import Image

import torch
import torchvision

from torchvision.models.detection import fasterrcnn_resnet50_fpn

from tqdm import tqdm


# ============================================================
# 2. CONFIGURATION
# ============================================================

print("\n" + "=" * 80)
print("E4-A — FASTER R-CNN CROP CREATION")
print("=" * 80)


# ------------------------------------------------------------
# Dataset root
# ------------------------------------------------------------

DATASET_ROOT = Path(
    r"C:\Users\varda\Documents\_My Computer\COMMON_space\Learning\Upgrad_EPGP_MS\upgrad_IIITB_MS_course\MS_LJMU_Material\Topic Data\SortWaste\dataset\dataset"
)


# ------------------------------------------------------------
# SortWaste split locations
# ------------------------------------------------------------

TRAIN_IMAGES_DIR = (
    DATASET_ROOT
    / "splited_all_dataset_coco"
    / "train"
    / "images"
)

VAL_IMAGES_DIR = (
    DATASET_ROOT
    / "splited_all_dataset_coco"
    / "val"
    / "images"
)


TRAIN_COCO_JSON = (
    DATASET_ROOT
    / "splited_all_dataset_coco"
    / "train"
    / "annotations"
    / "train_coco.json"
)

VAL_COCO_JSON = (
    DATASET_ROOT
    / "splited_all_dataset_coco"
    / "val"
    / "annotations"
    / "val_coco.json"
)


# ============================================================
# 3. FASTER R-CNN CHECKPOINT
# ============================================================
#
# IMPORTANT:
# Replace this path with the actual Faster R-CNN best checkpoint
# from your E2 baseline training.
#
# The model is assumed to have:
#
#   8 foreground classes
#   + background
#
# Therefore:
#
#   NUM_CLASSES = 9
#
# ============================================================

FASTER_RCNN_CHECKPOINT = Path(
    r"C:\Users\varda\Documents\_My Computer\COMMON_space\Learning\Upgrad_EPGP_MS\upgrad_IIITB_MS_course\MS_LJMU_Material\Thesis_Code\runs\sortwaste\faster_rcnn_baseline\best_model.pth"
)


# ============================================================
# 4. OUTPUT DIRECTORY
# ============================================================

OUTPUT_ROOT = (
    DATASET_ROOT
    / "faster_rcnn_mobilenet_crops_E4F"
)


TRAIN_CROPS_DIR = (
    OUTPUT_ROOT
    / "train"
)

VAL_CROPS_DIR = (
    OUTPUT_ROOT
    / "val"
)


TRAIN_MANIFEST = (
    OUTPUT_ROOT
    / "E4A_train_crop_manifest.csv"
)

VAL_MANIFEST = (
    OUTPUT_ROOT
    / "E4A_val_crop_manifest.csv"
)


SUMMARY_FILE = (
    OUTPUT_ROOT
    / "E4A_crop_creation_summary.txt"
)


# ============================================================
# 5. MODEL CONFIGURATION
# ============================================================

NUM_ORIGINAL_CLASSES = 8

NUM_MODEL_CLASSES = (
    NUM_ORIGINAL_CLASSES + 1
)


# Faster R-CNN confidence threshold.
#
# 0.30 is deliberately used for crop creation.
#
# We want reasonably good detections while still giving
# MobileNet enough examples.
#
DETECTION_CONF_THRESHOLD = 0.30


# IoU required for assigning a detector crop to GT.
#
# A detector prediction must overlap a GT object by at least
# this amount.
#
MATCH_IOU_THRESHOLD = 0.50


# Minimum crop dimensions.
#
# Extremely tiny crops are not useful for MobileNet.
MIN_CROP_WIDTH = 10
MIN_CROP_HEIGHT = 10


# Padding around detector bounding box.
#
# 0.05 means 5% padding on each side.
#
CROP_PADDING_RATIO = 0.05


# Maximum number of detections per image.
#
# Faster R-CNN normally returns 100.
MAX_DETECTIONS_PER_IMAGE = 100


# ============================================================
# 6. CLASS MAPPING
# ============================================================

# Original SortWaste class names.
#
# These correspond to the 8 original SortWaste classes.

ORIGINAL_CLASSES = {

    1: "pet",

    2: "hdpe",

    3: "mixed_plastic_soft",

    4: "ecal",

    5: "metal",

    6: "cardboard",

    7: "mixed_plastic_rigid",

    8: "pet_oil",
}


# ------------------------------------------------------------
# Evaluation taxonomy
# ------------------------------------------------------------
#
# Metal + cardboard are deliberately combined into
# non_plastic.
#
# This is the same taxonomy used in E3Y-B.
# ------------------------------------------------------------

EVAL_CLASSES = {

    1: "ecal",

    2: "hdpe",

    3: "mixed_plastic_rigid",

    4: "mixed_plastic_soft",

    5: "non_plastic",

    6: "pet",

    7: "pet_oil",
}


# ------------------------------------------------------------
# Original COCO category ID -> evaluation class
# ------------------------------------------------------------

ORIGINAL_COCO_TO_EVAL = {

    1: "pet",

    2: "hdpe",

    3: "mixed_plastic_soft",

    4: "ecal",

    5: "non_plastic",

    6: "non_plastic",

    7: "mixed_plastic_rigid",

    8: "pet_oil",
}


# ------------------------------------------------------------
# Evaluation class -> MobileNet integer label
# ------------------------------------------------------------

EVAL_CLASS_TO_ID = {

    "ecal": 1,

    "hdpe": 2,

    "mixed_plastic_rigid": 3,

    "mixed_plastic_soft": 4,

    "non_plastic": 5,

    "pet": 6,

    "pet_oil": 7,
}


EVAL_ID_TO_CLASS = {

    value: key

    for key, value
    in EVAL_CLASS_TO_ID.items()
}


# ============================================================
# 7. RANDOM SEED
# ============================================================

RANDOM_SEED = 42

random.seed(
    RANDOM_SEED
)

np.random.seed(
    RANDOM_SEED
)

torch.manual_seed(
    RANDOM_SEED
)


# ============================================================
# 8. DEVICE
# ============================================================

DEVICE = torch.device(

    "cuda"

    if torch.cuda.is_available()

    else "cpu"
)


print(
    f"\nDevice: {DEVICE}"
)


if torch.cuda.is_available():

    print(
        f"GPU: "
        f"{torch.cuda.get_device_name(0)}"
    )

    print(
        f"CUDA: "
        f"{torch.version.cuda}"
    )


# ============================================================
# 9. CHECK PATHS
# ============================================================

print("\n" + "=" * 80)
print("CHECKING PATHS")
print("=" * 80)


paths_to_check = {

    "Dataset root":
        DATASET_ROOT,

    "Train images":
        TRAIN_IMAGES_DIR,

    "Validation images":
        VAL_IMAGES_DIR,

    "Train COCO JSON":
        TRAIN_COCO_JSON,

    "Validation COCO JSON":
        VAL_COCO_JSON,

    "Faster R-CNN checkpoint":
        FASTER_RCNN_CHECKPOINT,
}


for name, path in paths_to_check.items():

    print(
        f"{name:30s}: {path}"
    )

    print(
        f"{'':30s}  Exists: {path.exists()}"
    )


missing_paths = [

    path

    for path
    in paths_to_check.values()

    if not path.exists()
]


if missing_paths:

    raise FileNotFoundError(

        "\nOne or more required paths do not exist."
        "\nPlease correct the configuration section."
    )


# ============================================================
# 10. CREATE OUTPUT DIRECTORIES
# ============================================================

OUTPUT_ROOT.mkdir(

    parents=True,

    exist_ok=True
)


TRAIN_CROPS_DIR.mkdir(

    parents=True,

    exist_ok=True
)


VAL_CROPS_DIR.mkdir(

    parents=True,

    exist_ok=True
)


# ------------------------------------------------------------
# Create class folders
# ------------------------------------------------------------

for class_name in EVAL_CLASS_TO_ID.keys():

    (
        TRAIN_CROPS_DIR
        / class_name
    ).mkdir(
        parents=True,
        exist_ok=True
    )

    (
        VAL_CROPS_DIR
        / class_name
    ).mkdir(
        parents=True,
        exist_ok=True
    )


# ============================================================
# 11. BUILD FASTER R-CNN MODEL
# ============================================================

print("\n" + "=" * 80)
print("BUILDING FASTER R-CNN")
print("=" * 80)


model = fasterrcnn_resnet50_fpn(

    weights=None,

    weights_backbone=None,

    num_classes=NUM_MODEL_CLASSES,

)


# ============================================================
# 12. LOAD CHECKPOINT
# ============================================================

print("\nLoading Faster R-CNN checkpoint...")


checkpoint = torch.load(

    FASTER_RCNN_CHECKPOINT,

    map_location="cpu"
)


# ------------------------------------------------------------
# Handle common checkpoint formats
# ------------------------------------------------------------

if isinstance(
    checkpoint,
    dict
):

    if "model_state_dict" in checkpoint:

        state_dict = checkpoint[
            "model_state_dict"
        ]

    elif "state_dict" in checkpoint:

        state_dict = checkpoint[
            "state_dict"
        ]

    else:

        # Assume checkpoint itself is a
        # state_dict.
        state_dict = checkpoint

else:

    state_dict = checkpoint


# ------------------------------------------------------------
# Remove possible DataParallel prefix
# ------------------------------------------------------------

clean_state_dict = {}


for key, value in state_dict.items():

    new_key = key

    if new_key.startswith(
        "module."
    ):

        new_key = new_key[
            len("module.") :
        ]

    clean_state_dict[
        new_key
    ] = value


# ------------------------------------------------------------
# Load weights
# ------------------------------------------------------------

missing_keys, unexpected_keys = (
    model.load_state_dict(
        clean_state_dict,
        strict=False
    )
)


if missing_keys:

    print(
        "\nWARNING — Missing checkpoint keys:"
    )

    for key in missing_keys:

        print(
            key
        )


if unexpected_keys:

    print(
        "\nWARNING — Unexpected checkpoint keys:"
    )

    for key in unexpected_keys:

        print(
            key
        )


model.to(
    DEVICE
)


model.eval()


print(
    "\nFaster R-CNN loaded successfully."
)


# ============================================================
# 13. IoU FUNCTION
# ============================================================

def calculate_iou(
    box_a,
    box_b
):

    """
    Calculate IoU between two boxes.

    Box format:

        [x1, y1, x2, y2]
    """

    ax1, ay1, ax2, ay2 = box_a

    bx1, by1, bx2, by2 = box_b


    intersection_x1 = max(
        ax1,
        bx1
    )

    intersection_y1 = max(
        ay1,
        by1
    )

    intersection_x2 = min(
        ax2,
        bx2
    )

    intersection_y2 = min(
        ay2,
        by2
    )


    intersection_width = max(

        0.0,

        intersection_x2
        -
        intersection_x1
    )


    intersection_height = max(

        0.0,

        intersection_y2
        -
        intersection_y1
    )


    intersection_area = (

        intersection_width
        *
        intersection_height
    )


    area_a = (

        max(
            0.0,
            ax2 - ax1
        )
        *
        max(
            0.0,
            ay2 - ay1
        )
    )


    area_b = (

        max(
            0.0,
            bx2 - bx1
        )
        *
        max(
            0.0,
            by2 - by1
        )
    )


    union_area = (

        area_a
        +
        area_b
        -
        intersection_area
    )


    if union_area <= 0:

        return 0.0


    return (
        intersection_area
        /
        union_area
    )


# ============================================================
# 14. LOAD COCO JSON
# ============================================================

def load_coco_annotations(
    coco_json_path
):

    with open(

        coco_json_path,

        "r",

        encoding="utf-8"

    ) as f:

        coco = json.load(f)


    images = coco[
        "images"
    ]

    annotations = coco[
        "annotations"
    ]


    images_by_id = {

        int(image["id"]):
            image

        for image
        in images
    }


    annotations_by_image = defaultdict(
        list
    )


    for ann in annotations:

        annotations_by_image[
            int(
                ann["image_id"]
            )
        ].append(
            ann
        )


    return (
        images_by_id,
        annotations_by_image
    )


# ============================================================
# 15. CONVERT COCO GT BOX
# ============================================================

def coco_bbox_to_xyxy(
    bbox
):

    x, y, width, height = (

        float(bbox[0]),

        float(bbox[1]),

        float(bbox[2]),

        float(bbox[3])
    )


    return [

        x,

        y,

        x + width,

        y + height,
    ]


# ============================================================
# 16. MAP GT CATEGORY TO EVALUATION CLASS
# ============================================================

def category_to_eval_class(
    category_id
):

    category_id = int(
        category_id
    )


    if category_id not in (
        ORIGINAL_COCO_TO_EVAL
    ):

        raise ValueError(

            f"Unknown COCO category ID: "
            f"{category_id}"
        )


    return (
        ORIGINAL_COCO_TO_EVAL[
            category_id
        ]
    )


# ============================================================
# 17. ADD CROP PADDING
# ============================================================

def add_padding_to_box(

    box,

    image_width,

    image_height,

    padding_ratio

):

    x1, y1, x2, y2 = box


    width = (
        x2 - x1
    )

    height = (
        y2 - y1
    )


    pad_x = (
        width
        *
        padding_ratio
    )

    pad_y = (
        height
        *
        padding_ratio
    )


    x1 = max(
        0.0,
        x1 - pad_x
    )

    y1 = max(
        0.0,
        y1 - pad_y
    )

    x2 = min(
        float(image_width),
        x2 + pad_x
    )

    y2 = min(
        float(image_height),
        y2 + pad_y
    )


    return [

        x1,

        y1,

        x2,

        y2,
    ]


# ============================================================
# 18. MATCH DETECTION TO GT
# ============================================================

def find_best_gt_match(

    detection_box,

    gt_objects

):

    best_iou = 0.0

    best_gt = None


    for gt in gt_objects:

        iou = calculate_iou(

            detection_box,

            gt["box"]
        )


        if iou > best_iou:

            best_iou = iou

            best_gt = gt


    if (

        best_gt is not None

        and best_iou
        >= MATCH_IOU_THRESHOLD

    ):

        return (
            best_gt,
            best_iou
        )


    return (
        None,
        best_iou
    )


# ============================================================
# 19. IMAGE TRANSFORM
# ============================================================

from torchvision.transforms.functional import (
    to_tensor
)


# ============================================================
# 20. CREATE CROPS FOR ONE SPLIT
# ============================================================

def create_crops_for_split(

    split_name,

    images_dir,

    coco_json_path,

    output_dir,

    manifest_path

):

    print("\n" + "=" * 80)

    print(
        f"CREATING {split_name.upper()} FASTER R-CNN CROPS"
    )

    print("=" * 80)


    # --------------------------------------------------------
    # Load COCO
    # --------------------------------------------------------

    (
        images_by_id,
        annotations_by_image
    ) = load_coco_annotations(
        coco_json_path
    )


    print(
        f"COCO images: "
        f"{len(images_by_id):,}"
    )


    total_annotations = sum(

        len(value)

        for value
        in annotations_by_image.values()
    )


    print(
        f"GT annotations: "
        f"{total_annotations:,}"
    )


    # --------------------------------------------------------
    # Statistics
    # --------------------------------------------------------

    statistics = {

        "images": 0,

        "images_with_detections": 0,

        "total_detections": 0,

        "matched_detections": 0,

        "unmatched_detections": 0,

        "crops_saved": 0,

        "invalid_crops": 0,
    }


    class_counts = {

        class_name: 0

        for class_name
        in EVAL_CLASS_TO_ID.keys()
    }


    manifest_rows = []


    # --------------------------------------------------------
    # Iterate through images
    # --------------------------------------------------------

    image_items = list(
        images_by_id.items()
    )


    progress = tqdm(

        image_items,

        desc=f"{split_name} crop creation",

        unit="image"
    )


    for image_id, image_info in progress:

        statistics[
            "images"
        ] += 1


        # ----------------------------------------------------
        # Image filename
        # ----------------------------------------------------

        image_filename = image_info[
            "file_name"
        ]


        image_path = (
            images_dir
            / Path(image_filename).name
        )


        if not image_path.exists():

            # Some COCO files may contain
            # directory prefixes.
            #
            # Search by basename.

            candidates = list(

                images_dir.glob(
                    Path(
                        image_filename
                    ).name
                )
            )


            if not candidates:

                print(
                    f"\nWARNING: image not found:"
                    f" {image_filename}"
                )

                continue


            image_path = candidates[0]


        # ----------------------------------------------------
        # Load image
        # ----------------------------------------------------

        try:

            image = Image.open(
                image_path
            ).convert(
                "RGB"
            )

        except Exception as e:

            print(
                f"\nWARNING: could not load "
                f"{image_path}: {e}"
            )

            continue


        image_width, image_height = (
            image.size
        )


        # ----------------------------------------------------
        # Build GT objects
        # ----------------------------------------------------

        gt_objects = []


        for ann in annotations_by_image.get(
            int(image_id),
            []
        ):

            # Ignore crowd annotations.
            #
            # SortWaste normally should not contain
            # problematic crowd annotations, but this
            # protects the matching process.

            if ann.get(
                "iscrowd",
                0
            ) == 1:

                continue


            category_id = int(
                ann[
                    "category_id"
                ]
            )


            if category_id not in (
                ORIGINAL_COCO_TO_EVAL
            ):

                continue


            gt_box = (
                coco_bbox_to_xyxy(
                    ann[
                        "bbox"
                    ]
                )
            )


            gt_objects.append(

                {

                    "annotation_id":
                        int(
                            ann["id"]
                        ),

                    "category_id":
                        category_id,

                    "class_name":
                        category_to_eval_class(
                            category_id
                        ),

                    "box":
                        gt_box,
                }
            )


        # ----------------------------------------------------
        # Faster R-CNN inference
        # ----------------------------------------------------

        image_tensor = to_tensor(
            image
        ).to(
            DEVICE
        )


        with torch.no_grad():

            prediction = model(
                [
                    image_tensor
                ]
            )[0]


        boxes = (
            prediction[
                "boxes"
            ]
            .detach()
            .cpu()
            .numpy()
        )


        labels = (
            prediction[
                "labels"
            ]
            .detach()
            .cpu()
            .numpy()
        )


        scores = (
            prediction[
                "scores"
            ]
            .detach()
            .cpu()
            .numpy()
        )


        # ----------------------------------------------------
        # Filter detections
        # ----------------------------------------------------

        keep = (

            scores
            >= DETECTION_CONF_THRESHOLD
        )


        boxes = boxes[
            keep
        ]

        labels = labels[
            keep
        ]

        scores = scores[
            keep
        ]


        # ----------------------------------------------------
        # Limit number of detections
        # ----------------------------------------------------

        if len(scores) > MAX_DETECTIONS_PER_IMAGE:

            order = np.argsort(
                scores
            )[
                ::-1
            ][
                :MAX_DETECTIONS_PER_IMAGE
            ]

            boxes = boxes[
                order
            ]

            labels = labels[
                order
            ]

            scores = scores[
                order
            ]


        detection_count = len(
            boxes
        )


        statistics[
            "total_detections"
        ] += detection_count


        if detection_count > 0:

            statistics[
                "images_with_detections"
            ] += 1


        # ----------------------------------------------------
        # Match detections to GT
        # ----------------------------------------------------
        #
        # IMPORTANT:
        #
        # Each GT object can only be assigned to ONE crop.
        #
        # This prevents multiple detector boxes from creating
        # many copies of the same training object.
        #
        # We process detections in descending confidence order.
        # ----------------------------------------------------

        detection_indices = sorted(

            range(
                detection_count
            ),

            key=lambda i:
                float(
                    scores[i]
                ),

            reverse=True
        )


        matched_gt_indices = set()


        for detection_index in detection_indices:

            detection_box = [

                float(
                    boxes[
                        detection_index
                    ][0]
                ),

                float(
                    boxes[
                        detection_index
                    ][1]
                ),

                float(
                    boxes[
                        detection_index
                    ][2]
                ),

                float(
                    boxes[
                        detection_index
                    ][3]
                ),
            ]


            detection_score = float(
                scores[
                    detection_index
                ]
            )


            detector_class_id = int(
                labels[
                    detection_index
                ]
            )


            # ------------------------------------------------
            # Find best unmatched GT
            # ------------------------------------------------

            best_iou = 0.0

            best_gt_index = None


            for gt_index, gt in enumerate(
                gt_objects
            ):

                if (
                    gt_index
                    in matched_gt_indices
                ):

                    continue


                iou = calculate_iou(

                    detection_box,

                    gt[
                        "box"
                    ]
                )


                if iou > best_iou:

                    best_iou = iou

                    best_gt_index = (
                        gt_index
                    )


            # ------------------------------------------------
            # No valid GT match
            # ------------------------------------------------

            if (

                best_gt_index is None

                or best_iou
                < MATCH_IOU_THRESHOLD

            ):

                statistics[
                    "unmatched_detections"
                ] += 1

                continue


            # ------------------------------------------------
            # Mark GT as matched
            # ------------------------------------------------

            matched_gt_indices.add(
                best_gt_index
            )


            gt = gt_objects[
                best_gt_index
            ]


            statistics[
                "matched_detections"
            ] += 1


            # ------------------------------------------------
            # IMPORTANT:
            #
            # The crop gets the GT class, NOT the
            # Faster R-CNN predicted class.
            #
            # This means MobileNet receives a clean
            # supervised label.
            # ------------------------------------------------

            final_class_name = (
                gt[
                    "class_name"
                ]
            )


            # ------------------------------------------------
            # Crop box
            # ------------------------------------------------

            crop_box = add_padding_to_box(

                detection_box,

                image_width,

                image_height,

                CROP_PADDING_RATIO
            )


            crop_x1 = int(
                round(
                    crop_box[0]
                )
            )

            crop_y1 = int(
                round(
                    crop_box[1]
                )
            )

            crop_x2 = int(
                round(
                    crop_box[2]
                )
            )

            crop_y2 = int(
                round(
                    crop_box[3]
                )
            )


            # ------------------------------------------------
            # Validate crop
            # ------------------------------------------------

            crop_width = (
                crop_x2
                -
                crop_x1
            )

            crop_height = (
                crop_y2
                -
                crop_y1
            )


            if (

                crop_width
                < MIN_CROP_WIDTH

                or crop_height
                < MIN_CROP_HEIGHT

            ):

                statistics[
                    "invalid_crops"
                ] += 1

                continue


            # ------------------------------------------------
            # Extract crop
            # ------------------------------------------------

            crop = image.crop(

                (

                    crop_x1,

                    crop_y1,

                    crop_x2,

                    crop_y2,
                )
            )


            # ------------------------------------------------
            # Create unique filename
            # ------------------------------------------------

            original_stem = (
                Path(
                    image_filename
                ).stem
            )


            crop_filename = (

                f"{original_stem}"

                f"_det{detection_index:03d}"

                f"_gt{gt['annotation_id']}"

                f"_iou{best_iou:.3f}"

                f".jpg"
            )


            crop_path = (

                output_dir
                /
                final_class_name
                /
                crop_filename
            )


            # ------------------------------------------------
            # Save crop
            # ------------------------------------------------

            crop.save(

                crop_path,

                quality=95
            )


            statistics[
                "crops_saved"
            ] += 1


            class_counts[
                final_class_name
            ] += 1


            # ------------------------------------------------
            # Save manifest information
            # ------------------------------------------------

            manifest_rows.append(

                {

                    "crop_path":
                        str(
                            crop_path
                        ),

                    "image":
                        image_filename,

                    "image_id":
                        int(
                            image_id
                        ),

                    "gt_annotation_id":
                        int(
                            gt[
                                "annotation_id"
                            ]
                        ),

                    "gt_class":
                        final_class_name,

                    "gt_category_id":
                        int(
                            gt[
                                "category_id"
                            ]
                        ),

                    "detector_class_id":
                        detector_class_id,

                    "detector_confidence":
                        detection_score,

                    "IoU":
                        best_iou,

                    "det_x1":
                        detection_box[0],

                    "det_y1":
                        detection_box[1],

                    "det_x2":
                        detection_box[2],

                    "det_y2":
                        detection_box[3],

                    "crop_x1":
                        crop_x1,

                    "crop_y1":
                        crop_y1,

                    "crop_x2":
                        crop_x2,

                    "crop_y2":
                        crop_y2,
                }
            )


        # ----------------------------------------------------
        # Progress information
        # ----------------------------------------------------

        progress.set_postfix(

            detections=(
                statistics[
                    "total_detections"
                ]
            ),

            matched=(
                statistics[
                    "matched_detections"
                ]
            ),

            crops=(
                statistics[
                    "crops_saved"
                ]
            )
        )


    # ========================================================
    # SAVE MANIFEST
    # ========================================================

    print(
        "\nSaving crop manifest..."
    )


    manifest_fields = [

        "crop_path",

        "image",

        "image_id",

        "gt_annotation_id",

        "gt_class",

        "gt_category_id",

        "detector_class_id",

        "detector_confidence",

        "IoU",

        "det_x1",

        "det_y1",

        "det_x2",

        "det_y2",

        "crop_x1",

        "crop_y1",

        "crop_x2",

        "crop_y2",
    ]


    with open(

        manifest_path,

        "w",

        newline="",

        encoding="utf-8"

    ) as f:

        writer = csv.DictWriter(

            f,

            fieldnames=manifest_fields
        )


        writer.writeheader()


        writer.writerows(
            manifest_rows
        )


    # ========================================================
    # PRINT SUMMARY
    # ========================================================

    print("\n" + "-" * 80)

    print(
        f"{split_name.upper()} SUMMARY"
    )

    print("-" * 80)


    print(
        f"Images                  : "
        f"{statistics['images']:,}"
    )


    print(
        f"Images with detections  : "
        f"{statistics['images_with_detections']:,}"
    )


    print(
        f"Faster R-CNN detections : "
        f"{statistics['total_detections']:,}"
    )


    print(
        f"Matched detections      : "
        f"{statistics['matched_detections']:,}"
    )


    print(
        f"Unmatched detections    : "
        f"{statistics['unmatched_detections']:,}"
    )


    print(
        f"Invalid crops           : "
        f"{statistics['invalid_crops']:,}"
    )


    print(
        f"Crops saved             : "
        f"{statistics['crops_saved']:,}"
    )


    print(
        "\nClass distribution:"
    )


    for class_name in EVAL_CLASS_TO_ID.keys():

        print(

            f"{class_name:25s}: "
            f"{class_counts[class_name]:,}"
        )


    return (
        statistics,
        class_counts,
        manifest_rows
    )


# ============================================================
# 21. CREATE TRAIN CROPS
# ============================================================

evaluation_start = time.time()


(
    train_statistics,
    train_class_counts,
    train_manifest
) = create_crops_for_split(

    split_name="train",

    images_dir=TRAIN_IMAGES_DIR,

    coco_json_path=TRAIN_COCO_JSON,

    output_dir=TRAIN_CROPS_DIR,

    manifest_path=TRAIN_MANIFEST,
)


# ============================================================
# 22. CREATE VALIDATION CROPS
# ============================================================

(
    val_statistics,
    val_class_counts,
    val_manifest
) = create_crops_for_split(

    split_name="val",

    images_dir=VAL_IMAGES_DIR,

    coco_json_path=VAL_COCO_JSON,

    output_dir=VAL_CROPS_DIR,

    manifest_path=VAL_MANIFEST,
)


# ============================================================
# 23. FINAL SUMMARY
# ============================================================

elapsed_minutes = (

    time.time()
    -
    evaluation_start
) / 60.0


print("\n" + "=" * 80)

print(
    "E4-A — FASTER R-CNN CROP CREATION COMPLETE"
)

print("=" * 80)


print(
    f"\nOutput directory:"
)

print(
    OUTPUT_ROOT
)


print(
    "\nTRAIN"
)

print(
    f"Images                  : "
    f"{train_statistics['images']:,}"
)

print(
    f"GT-matched crops        : "
    f"{train_statistics['crops_saved']:,}"
)


print(
    "\nVALIDATION"
)

print(
    f"Images                  : "
    f"{val_statistics['images']:,}"
)

print(
    f"GT-matched crops        : "
    f"{val_statistics['crops_saved']:,}"
)


print(
    "\n" + "-" * 80
)

print(
    "TRAIN CLASS DISTRIBUTION"
)

print(
    "-" * 80
)


for class_name in EVAL_CLASS_TO_ID.keys():

    print(

        f"{class_name:25s}: "
        f"{train_class_counts[class_name]:,}"
    )


print(
    "\n" + "-" * 80
)

print(
    "VALIDATION CLASS DISTRIBUTION"
)

print(
    "-" * 80
)


for class_name in EVAL_CLASS_TO_ID.keys():

    print(

        f"{class_name:25s}: "
        f"{val_class_counts[class_name]:,}"
    )


# ============================================================
# 24. SAVE SUMMARY FILE
# ============================================================

with open(

    SUMMARY_FILE,

    "w",

    encoding="utf-8"

) as f:

    f.write(
        "E4-A — FASTER R-CNN CROP CREATION\n"
    )

    f.write(
        "=" * 70
        +
        "\n\n"
    )


    f.write(
        "PIPELINE\n"
    )

    f.write(
        "Faster R-CNN -> detection -> "
        "GT IoU matching -> object crop\n\n"
    )


    f.write(
        "CLASSIFICATION TAXONOMY\n"
    )

    f.write(
        "7 evaluation classes\n"
    )

    f.write(
        "Metal + Cardboard -> non_plastic\n\n"
    )


    f.write(
        "CONFIGURATION\n"
    )

    f.write(
        f"Detection confidence threshold: "
        f"{DETECTION_CONF_THRESHOLD}\n"
    )

    f.write(
        f"IoU matching threshold: "
        f"{MATCH_IOU_THRESHOLD}\n"
    )

    f.write(
        f"Crop padding ratio: "
        f"{CROP_PADDING_RATIO}\n"
    )


    f.write(
        "\nTRAIN\n"
    )

    f.write(
        f"Images: "
        f"{train_statistics['images']}\n"
    )

    f.write(
        f"Faster R-CNN detections: "
        f"{train_statistics['total_detections']}\n"
    )

    f.write(
        f"Matched detections: "
        f"{train_statistics['matched_detections']}\n"
    )

    f.write(
        f"Crops saved: "
        f"{train_statistics['crops_saved']}\n"
    )


    f.write(
        "\nTRAIN CLASS COUNTS\n"
    )


    for class_name in EVAL_CLASS_TO_ID.keys():

        f.write(

            f"{class_name}: "
            f"{train_class_counts[class_name]}\n"
        )


    f.write(
        "\nVALIDATION\n"
    )

    f.write(
        f"Images: "
        f"{val_statistics['images']}\n"
    )

    f.write(
        f"Faster R-CNN detections: "
        f"{val_statistics['total_detections']}\n"
    )

    f.write(
        f"Matched detections: "
        f"{val_statistics['matched_detections']}\n"
    )

    f.write(
        f"Crops saved: "
        f"{val_statistics['crops_saved']}\n"
    )


    f.write(
        "\nVALIDATION CLASS COUNTS\n"
    )


    for class_name in EVAL_CLASS_TO_ID.keys():

        f.write(

            f"{class_name}: "
            f"{val_class_counts[class_name]}\n"
        )


    f.write(
        "\nMETHODOLOGICAL NOTES\n"
    )


    f.write(
        "1. Faster R-CNN supplies the object bounding box.\n"
    )

    f.write(
        "2. Detection crops are matched to COCO GT using IoU.\n"
    )

    f.write(
        "3. Each GT object can generate at most one crop.\n"
    )

    f.write(
        "4. MobileNet training labels come from GT, not "
        "Faster R-CNN predicted classes.\n"
    )

    f.write(
        "5. Metal and cardboard are mapped to non_plastic.\n"
    )

    f.write(
        "6. Train and validation crops are kept separate.\n"
    )

    f.write(
        "7. MobileNet training is NOT performed in this script.\n"
    )


# ============================================================
# 25. FINISHED
# ============================================================

print(
    f"\nSummary saved:"
)

print(
    SUMMARY_FILE
)


print(
    f"\nTotal execution time: "
    f"{elapsed_minutes:.2f} minutes"
)


print("\n" + "=" * 80)

print(
    "NEXT STEP:"
)

print(
    "Use the generated TRAIN crops to train "
    "MobileNet-V3-Large."
)

print(
    "Use the generated VAL crops for MobileNet "
    "validation."
)

print("=" * 80)


E4-A — FASTER R-CNN CROP CREATION

Device: cuda
GPU: NVIDIA GeForce RTX 3050 Ti Laptop GPU
CUDA: 12.6

CHECKING PATHS
Dataset root                  : C:\Users\varda\Documents\_My Computer\COMMON_space\Learning\Upgrad_EPGP_MS\upgrad_IIITB_MS_course\MS_LJMU_Material\Topic Data\SortWaste\dataset\dataset
                                Exists: True
Train images                  : C:\Users\varda\Documents\_My Computer\COMMON_space\Learning\Upgrad_EPGP_MS\upgrad_IIITB_MS_course\MS_LJMU_Material\Topic Data\SortWaste\dataset\dataset\splited_all_dataset_coco\train\images
                                Exists: True
Validation images             : C:\Users\varda\Documents\_My Computer\COMMON_space\Learning\Upgrad_EPGP_MS\upgrad_IIITB_MS_course\MS_LJMU_Material\Topic Data\SortWaste\dataset\dataset\splited_all_dataset_coco\val\images
                                Exists: True
Train COCO JSON               : C:\Users\varda\Documents\_My Computer\COMMON_space\Learning\Upgrad_EPGP_MS\upgrad_IIITB_

train crop creation: 100%|██████████| 3705/3705 [29:17<00:00,  2.11image/s, crops=56171, detections=118629, matched=56171]



Saving crop manifest...

--------------------------------------------------------------------------------
TRAIN SUMMARY
--------------------------------------------------------------------------------
Images                  : 3,705
Images with detections  : 3,701
Faster R-CNN detections : 118,629
Matched detections      : 56,171
Unmatched detections    : 62,458
Invalid crops           : 0
Crops saved             : 56,171

Class distribution:
ecal                     : 12,376
hdpe                     : 15,632
mixed_plastic_rigid      : 6,454
mixed_plastic_soft       : 8,015
non_plastic              : 2,204
pet                      : 10,772
pet_oil                  : 718

CREATING VAL FASTER R-CNN CROPS
COCO images: 780
GT annotations: 13,065


val crop creation: 100%|██████████| 780/780 [06:25<00:00,  2.02image/s, crops=11831, detections=25112, matched=11831]



Saving crop manifest...

--------------------------------------------------------------------------------
VAL SUMMARY
--------------------------------------------------------------------------------
Images                  : 780
Images with detections  : 780
Faster R-CNN detections : 25,112
Matched detections      : 11,831
Unmatched detections    : 13,281
Invalid crops           : 0
Crops saved             : 11,831

Class distribution:
ecal                     : 2,239
hdpe                     : 4,530
mixed_plastic_rigid      : 1,033
mixed_plastic_soft       : 1,314
non_plastic              : 632
pet                      : 1,929
pet_oil                  : 154

E4-A — FASTER R-CNN CROP CREATION COMPLETE

Output directory:
C:\Users\varda\Documents\_My Computer\COMMON_space\Learning\Upgrad_EPGP_MS\upgrad_IIITB_MS_course\MS_LJMU_Material\Topic Data\SortWaste\dataset\dataset\faster_rcnn_mobilenet_crops_E4F

TRAIN
Images                  : 3,705
GT-matched crops        : 56,171

VALIDATION
I

In [ ]:
### Train the mobilenet on faster R-CNN detected crops
# ============================================================
# E4F-A — MOBILENET-V3-LARGE TRAINING
# ============================================================
#
# Pipeline:
#
# Faster R-CNN detections
#        ↓
# Object crops
#        ↓
# MobileNet-V3-Large
#
# E4F-A evaluation taxonomy:
#
#   ecal
#   hdpe
#   mixed_plastic_rigid
#   mixed_plastic_soft
#   non_plastic       <- metal + cardboard
#   pet
#   pet_oil
#
# IMPORTANT:
# MobileNet is trained using CROPS CREATED BY FASTER R-CNN.
#
# This script does NOT perform Faster R-CNN detection.
# It only trains MobileNet on the already-created crops.
#
# ============================================================


# ============================================================
# 1. IMPORTS
# ============================================================

import os
import json
import csv
import random
import shutil
import time
from pathlib import Path

import numpy as np
import pandas as pd

from PIL import Image

import torch
import torch.nn as nn
import torch.optim as optim

from torch.utils.data import (
    Dataset,
    DataLoader
)

from torchvision import transforms
from torchvision.models import (
    mobilenet_v3_large,
    MobileNet_V3_Large_Weights
)

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix
)


# ============================================================
# 2. REPRODUCIBILITY
# ============================================================

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)


# ============================================================
# 3. CONFIGURATION
# ============================================================

# ------------------------------------------------------------
# Faster R-CNN crop directory
# ------------------------------------------------------------
#
# This MUST match the directory created by the previous
# Faster R-CNN crop-generation script.
#

CROP_ROOT = Path(
    r"C:\Users\varda\Documents\_My Computer\COMMON_space\Learning\Upgrad_EPGP_MS\upgrad_IIITB_MS_course\MS_LJMU_Material\Topic Data\SortWaste\dataset\dataset\faster_rcnn_mobilenet_crops_E4F"
)


# ------------------------------------------------------------
# MobileNet output directory
# ------------------------------------------------------------

OUTPUT_ROOT = (
    CROP_ROOT /
    "mobilenet_results" /
    "E4F-A_MobileNetV3Large"
)

OUTPUT_ROOT.mkdir(
    parents=True,
    exist_ok=True
)


# ------------------------------------------------------------
# Model checkpoint
# ------------------------------------------------------------

BEST_MODEL_PATH = (
    OUTPUT_ROOT /
    "E4FA_MobileNetV3Large_best.pth"
)

LAST_MODEL_PATH = (
    OUTPUT_ROOT /
    "E4FA_MobileNetV3Large_last.pth"
)


# ------------------------------------------------------------
# Result files
# ------------------------------------------------------------

TRAIN_HISTORY_CSV = (
    OUTPUT_ROOT /
    "training_history.csv"
)

CLASS_RESULTS_CSV = (
    OUTPUT_ROOT /
    "classification_results.csv"
)

CONFUSION_MATRIX_CSV = (
    OUTPUT_ROOT /
    "confusion_matrix.csv"
)

SUMMARY_TXT = (
    OUTPUT_ROOT /
    "training_summary.txt"
)


# ============================================================
# 4. CLASS DEFINITIONS
# ============================================================

MOBILENET_CLASSES = [

    "ecal",

    "hdpe",

    "mixed_plastic_rigid",

    "mixed_plastic_soft",

    "non_plastic",

    "pet",

    "pet_oil",
]


NUM_CLASSES = len(
    MOBILENET_CLASSES
)


CLASS_TO_INDEX = {

    class_name: index

    for index, class_name
    in enumerate(
        MOBILENET_CLASSES
    )
}


INDEX_TO_CLASS = {

    index: class_name

    for class_name, index
    in CLASS_TO_INDEX.items()
}


print("=" * 80)
print("E4-A — MOBILENET-V3-LARGE TRAINING")
print("=" * 80)

print(
    "\nEvaluation classes:"
)

for index, class_name in INDEX_TO_CLASS.items():

    print(
        f"{index}: {class_name}"
    )


# ============================================================
# 5. DEVICE
# ============================================================

DEVICE = torch.device(

    "cuda"

    if torch.cuda.is_available()

    else "cpu"
)


print(
    f"\nDevice: {DEVICE}"
)


if torch.cuda.is_available():

    print(
        f"GPU: "
        f"{torch.cuda.get_device_name(0)}"
    )

    print(
        f"CUDA: "
        f"{torch.version.cuda}"
    )


# ============================================================
# 6. TRAINING PARAMETERS
# ============================================================

BATCH_SIZE = 32

NUM_EPOCHS = 30

LEARNING_RATE = 1e-4

WEIGHT_DECAY = 1e-4

NUM_WORKERS = 0

IMAGE_SIZE = 224

PATIENCE = 7


# ============================================================
# 7. CHECK CROP DIRECTORY
# ============================================================

print("\n" + "=" * 80)
print("CHECKING FASTER R-CNN CROP DIRECTORY")
print("=" * 80)

print(
    f"\nCrop root:\n{CROP_ROOT}"
)

print(
    f"Exists: "
    f"{CROP_ROOT.exists()}"
)


if not CROP_ROOT.exists():

    raise FileNotFoundError(

        f"\nFaster R-CNN crop directory "
        f"does not exist:\n{CROP_ROOT}\n\n"
        f"Run the Faster R-CNN crop creation "
        f"script first."
    )


# ============================================================
# 8. EXPECTED CROP STRUCTURE
# ============================================================
#
# The crop-generation script creates a structure similar to:
#
# faster_rcnn_mobilenet_crops_E4A/
#
#     train/
#         ecal/
#         hdpe/
#         mixed_plastic_rigid/
#         mixed_plastic_soft/
#         non_plastic/
#         pet/
#         pet_oil/
#
#     val/
#         ecal/
#         hdpe/
#         mixed_plastic_rigid/
#         mixed_plastic_soft/
#         non_plastic/
#         pet/
#         pet_oil/
#
# ============================================================


TRAIN_ROOT = (
    CROP_ROOT /
    "train"
)

VAL_ROOT = (
    CROP_ROOT /
    "val"
)


print(
    "\nTrain crop directory:"
)

print(
    TRAIN_ROOT
)

print(
    f"Exists: {TRAIN_ROOT.exists()}"
)


print(
    "\nValidation crop directory:"
)

print(
    VAL_ROOT
)

print(
    f"Exists: {VAL_ROOT.exists()}"
)


if not TRAIN_ROOT.exists():

    raise FileNotFoundError(
        f"Training crop directory missing:\n"
        f"{TRAIN_ROOT}"
    )


if not VAL_ROOT.exists():

    raise FileNotFoundError(
        f"Validation crop directory missing:\n"
        f"{VAL_ROOT}"
    )


# ============================================================
# 9. CHECK CLASS DIRECTORIES
# ============================================================

print("\n" + "=" * 80)
print("CHECKING CLASS DIRECTORIES")
print("=" * 80)


for split_root, split_name in [

    (TRAIN_ROOT, "TRAIN"),

    (VAL_ROOT, "VALIDATION"),

]:

    print(
        f"\n{split_name}"
    )

    for class_name in MOBILENET_CLASSES:

        class_dir = (
            split_root /
            class_name
        )

        print(
            f"{class_name:25s} "
            f"Exists={class_dir.exists()}"
        )

        if not class_dir.exists():

            raise FileNotFoundError(

                f"\nMissing class directory:\n"
                f"{class_dir}"
            )


# ============================================================
# 10. DATASET CLASS
# ============================================================

class CropDataset(Dataset):

    def __init__(
        self,
        root_dir,
        transform=None
    ):

        self.root_dir = Path(
            root_dir
        )

        self.transform = transform

        self.samples = []

        valid_extensions = {

            ".jpg",
            ".jpeg",
            ".png",
            ".bmp",
            ".webp",
        }


        # ----------------------------------------------------
        # Load all crops
        # ----------------------------------------------------

        for class_name in MOBILENET_CLASSES:

            class_dir = (
                self.root_dir /
                class_name
            )

            class_index = (
                CLASS_TO_INDEX[
                    class_name
                ]
            )


            for image_path in sorted(
                class_dir.rglob("*")
            ):

                if not image_path.is_file():

                    continue


                if (
                    image_path.suffix.lower()
                    not in valid_extensions
                ):

                    continue


                self.samples.append(

                    (
                        str(image_path),
                        class_index
                    )
                )


        if len(self.samples) == 0:

            raise RuntimeError(

                f"No valid image crops found "
                f"in {self.root_dir}"
            )


    def __len__(self):

        return len(
            self.samples
        )


    def __getitem__(
        self,
        index
    ):

        image_path, label = (
            self.samples[index]
        )


        try:

            image = Image.open(
                image_path
            ).convert("RGB")


        except Exception as e:

            raise RuntimeError(

                f"Unable to load image:\n"
                f"{image_path}\n"
                f"{e}"
            )


        if self.transform is not None:

            image = self.transform(
                image
            )


        return image, label


# ============================================================
# 11. TRANSFORMS
# ============================================================

# ------------------------------------------------------------
# Training augmentation
# ------------------------------------------------------------
#
# These are realistic image-level transformations.
#
# No aggressive geometric distortion is used because the
# object crop itself has already been generated by Faster R-CNN.
#
# ------------------------------------------------------------

train_transform = transforms.Compose([

    transforms.Resize(
        (IMAGE_SIZE, IMAGE_SIZE)
    ),

    transforms.RandomHorizontalFlip(
        p=0.5
    ),

    transforms.RandomRotation(
        degrees=10
    ),

    transforms.ColorJitter(
        brightness=0.15,
        contrast=0.15,
        saturation=0.15,
        hue=0.03
    ),

    transforms.ToTensor(),

    transforms.Normalize(

        mean=[
            0.485,
            0.456,
            0.406
        ],

        std=[
            0.229,
            0.224,
            0.225
        ]
    ),
])


# ------------------------------------------------------------
# Validation transform
# ------------------------------------------------------------

val_transform = transforms.Compose([

    transforms.Resize(
        (IMAGE_SIZE, IMAGE_SIZE)
    ),

    transforms.ToTensor(),

    transforms.Normalize(

        mean=[
            0.485,
            0.456,
            0.406
        ],

        std=[
            0.229,
            0.224,
            0.225
        ]
    ),
])


# ============================================================
# 12. CREATE DATASETS
# ============================================================

print("\n" + "=" * 80)
print("LOADING CROPS")
print("=" * 80)


train_dataset = CropDataset(

    TRAIN_ROOT,

    transform=train_transform
)


val_dataset = CropDataset(

    VAL_ROOT,

    transform=val_transform
)


print(
    f"\nTraining crops: "
    f"{len(train_dataset):,}"
)


print(
    f"Validation crops: "
    f"{len(val_dataset):,}"
)


# ============================================================
# 13. DATASET CLASS DISTRIBUTION
# ============================================================

def get_class_counts(
    dataset
):

    counts = {
        class_name: 0
        for class_name
        in MOBILENET_CLASSES
    }


    for _, label in dataset.samples:

        class_name = (
            INDEX_TO_CLASS[
                label
            ]
        )

        counts[class_name] += 1


    return counts


train_counts = (
    get_class_counts(
        train_dataset
    )
)


val_counts = (
    get_class_counts(
        val_dataset
    )
)


print("\n" + "=" * 80)
print("TRAINING CLASS DISTRIBUTION")
print("=" * 80)


for class_name in MOBILENET_CLASSES:

    print(
        f"{class_name:25s} "
        f"{train_counts[class_name]:8,d}"
    )


print("\n" + "=" * 80)
print("VALIDATION CLASS DISTRIBUTION")
print("=" * 80)


for class_name in MOBILENET_CLASSES:

    print(
        f"{class_name:25s} "
        f"{val_counts[class_name]:8,d}"
    )


# ============================================================
# 14. CREATE CLASS WEIGHTS
# ============================================================
#
# E3Y-B used a class-weighted MobileNet model.
#
# We maintain the same approach for E4-A so that the
# comparison between E3Y-B and E4-A remains methodologically
# meaningful.
#
# Weight:
#
#       total_samples
#       -------------
#       num_classes * class_count
#
# ============================================================

total_train_samples = (
    len(train_dataset)
)


class_weights = []


for class_name in MOBILENET_CLASSES:

    count = (
        train_counts[
            class_name
        ]
    )


    if count == 0:

        weight = 0.0

    else:

        weight = (

            total_train_samples
            /
            (
                NUM_CLASSES
                *
                count
            )
        )


    class_weights.append(
        weight
    )


class_weights_tensor = torch.tensor(

    class_weights,

    dtype=torch.float32,

    device=DEVICE
)


print("\n" + "=" * 80)
print("CLASS WEIGHTS")
print("=" * 80)


for class_name, weight in zip(

    MOBILENET_CLASSES,

    class_weights

):

    print(
        f"{class_name:25s} "
        f"{weight:.6f}"
    )


# ============================================================
# 15. DATA LOADERS
# ============================================================

train_loader = DataLoader(

    train_dataset,

    batch_size=BATCH_SIZE,

    shuffle=True,

    num_workers=NUM_WORKERS,

    pin_memory=(
        DEVICE.type == "cuda"
    )
)


val_loader = DataLoader(

    val_dataset,

    batch_size=BATCH_SIZE,

    shuffle=False,

    num_workers=NUM_WORKERS,

    pin_memory=(
        DEVICE.type == "cuda"
    )
)


# ============================================================
# 16. LOAD MOBILENET-V3-LARGE
# ============================================================

print("\n" + "=" * 80)
print("LOADING MOBILENET-V3-LARGE")
print("=" * 80)


weights = (
    MobileNet_V3_Large_Weights.DEFAULT
)


model = mobilenet_v3_large(
    weights=weights
)


# ------------------------------------------------------------
# Replace classifier
# ------------------------------------------------------------

in_features = (
    model.classifier[-1].in_features
)


model.classifier[-1] = nn.Linear(

    in_features,

    NUM_CLASSES
)


model = model.to(
    DEVICE
)


print(
    "\nMobileNet-V3-Large loaded."
)


print(
    f"Number of output classes: "
    f"{NUM_CLASSES}"
)


# ============================================================
# 17. LOSS FUNCTION
# ============================================================

criterion = nn.CrossEntropyLoss(

    weight=class_weights_tensor
)


# ============================================================
# 18. OPTIMIZER
# ============================================================

optimizer = optim.AdamW(

    model.parameters(),

    lr=LEARNING_RATE,

    weight_decay=WEIGHT_DECAY
)


# ============================================================
# 19. LEARNING RATE SCHEDULER
# ============================================================

scheduler = optim.lr_scheduler.ReduceLROnPlateau(

    optimizer,

    mode="max",

    factor=0.5,

    patience=2
)


# ============================================================
# 20. TRAINING FUNCTIONS
# ============================================================

def train_one_epoch(

    model,

    loader,

    criterion,

    optimizer,

    device

):

    model.train()


    running_loss = 0.0

    all_true = []

    all_pred = []


    for images, labels in loader:

        images = images.to(
            device,
            non_blocking=True
        )

        labels = labels.to(
            device,
            non_blocking=True
        )


        optimizer.zero_grad(
            set_to_none=True
        )


        outputs = model(
            images
        )


        loss = criterion(
            outputs,
            labels
        )


        loss.backward()


        optimizer.step()


        running_loss += (

            loss.item()
            *
            images.size(0)
        )


        predictions = (
            outputs.argmax(
                dim=1
            )
        )


        all_true.extend(
            labels.detach()
            .cpu()
            .numpy()
            .tolist()
        )


        all_pred.extend(
            predictions.detach()
            .cpu()
            .numpy()
            .tolist()
        )


    epoch_loss = (

        running_loss
        /
        len(loader.dataset)
    )


    epoch_accuracy = (
        accuracy_score(
            all_true,
            all_pred
        )
    )


    epoch_macro_f1 = (
        f1_score(

            all_true,

            all_pred,

            labels=list(
                range(
                    NUM_CLASSES
                )
            ),

            average="macro",

            zero_division=0
        )
    )


    return (

        epoch_loss,

        epoch_accuracy,

        epoch_macro_f1
    )


# ============================================================
# 21. VALIDATION FUNCTION
# ============================================================

def validate(

    model,

    loader,

    criterion,

    device

):

    model.eval()


    running_loss = 0.0

    all_true = []

    all_pred = []


    with torch.no_grad():

        for images, labels in loader:

            images = images.to(
                device,
                non_blocking=True
            )

            labels = labels.to(
                device,
                non_blocking=True
            )


            outputs = model(
                images
            )


            loss = criterion(
                outputs,
                labels
            )


            running_loss += (

                loss.item()
                *
                images.size(0)
            )


            predictions = (
                outputs.argmax(
                    dim=1
                )
            )


            all_true.extend(
                labels.cpu()
                .numpy()
                .tolist()
            )


            all_pred.extend(
                predictions.cpu()
                .numpy()
                .tolist()
            )


    epoch_loss = (

        running_loss
        /
        len(loader.dataset)
    )


    epoch_accuracy = (
        accuracy_score(
            all_true,
            all_pred
        )
    )


    epoch_macro_f1 = (
        f1_score(

            all_true,

            all_pred,

            labels=list(
                range(
                    NUM_CLASSES
                )
            ),

            average="macro",

            zero_division=0
        )
    )


    return (

        epoch_loss,

        epoch_accuracy,

        epoch_macro_f1,

        all_true,

        all_pred
    )


# ============================================================
# 22. TRAINING LOOP
# ============================================================

print("\n" + "=" * 80)
print("STARTING E4-A MOBILENET TRAINING")
print("=" * 80)


print(
    f"\nEpochs       : {NUM_EPOCHS}"
)

print(
    f"Batch size   : {BATCH_SIZE}"
)

print(
    f"Learning rate: {LEARNING_RATE}"
)

print(
    f"Weight decay : {WEIGHT_DECAY}"
)

print(
    f"Patience     : {PATIENCE}"
)

print(
    "\nLoss: weighted cross entropy"
)


history = []


best_val_f1 = -1.0

best_epoch = 0

epochs_without_improvement = 0


training_start = time.time()


for epoch in range(

    1,

    NUM_EPOCHS + 1

):


    epoch_start = time.time()


    # --------------------------------------------------------
    # TRAIN
    # --------------------------------------------------------

    train_loss, train_accuracy, train_f1 = (

        train_one_epoch(

            model,

            train_loader,

            criterion,

            optimizer,

            DEVICE
        )
    )


    # --------------------------------------------------------
    # VALIDATION
    # --------------------------------------------------------

    (
        val_loss,
        val_accuracy,
        val_f1,
        val_true,
        val_pred
    ) = validate(

        model,

        val_loader,

        criterion,

        DEVICE
    )


    # --------------------------------------------------------
    # Scheduler
    # --------------------------------------------------------

    scheduler.step(
        val_f1
    )


    current_lr = (
        optimizer.param_groups[0]["lr"]
    )


    epoch_time = (

        time.time()
        -
        epoch_start
    )


    # --------------------------------------------------------
    # Store history
    # --------------------------------------------------------

    history.append(

        {

            "epoch":
                epoch,

            "train_loss":
                train_loss,

            "train_accuracy":
                train_accuracy,

            "train_macro_f1":
                train_f1,

            "val_loss":
                val_loss,

            "val_accuracy":
                val_accuracy,

            "val_macro_f1":
                val_f1,

            "learning_rate":
                current_lr,

            "epoch_time_seconds":
                epoch_time,
        }
    )


    # --------------------------------------------------------
    # Print
    # --------------------------------------------------------

    print(

        f"\nEpoch "
        f"{epoch:02d}/{NUM_EPOCHS}"
    )


    print(

        f"Train Loss: "
        f"{train_loss:.6f} | "

        f"Train Acc: "
        f"{train_accuracy:.6f} | "

        f"Train F1: "
        f"{train_f1:.6f}"
    )


    print(

        f"Val Loss:   "
        f"{val_loss:.6f} | "

        f"Val Acc:   "
        f"{val_accuracy:.6f} | "

        f"Val F1:    "
        f"{val_f1:.6f}"
    )


    print(

        f"LR: "
        f"{current_lr:.8f} | "

        f"Time: "
        f"{epoch_time:.1f}s"
    )


    # --------------------------------------------------------
    # Save best model
    # --------------------------------------------------------

    if val_f1 > best_val_f1:

        best_val_f1 = val_f1

        best_epoch = epoch

        epochs_without_improvement = 0


        torch.save(

            {

                "epoch":
                    epoch,

                "model_state_dict":
                    model.state_dict(),

                "optimizer_state_dict":
                    optimizer.state_dict(),

                "scheduler_state_dict":
                    scheduler.state_dict(),

                "best_val_f1":
                    best_val_f1,

                "classes":
                    MOBILENET_CLASSES,

                "class_to_index":
                    CLASS_TO_INDEX,

                "class_weights":
                    class_weights,

            },

            BEST_MODEL_PATH
        )


        print(
            "✓ Best model saved."
        )


    else:

        epochs_without_improvement += 1


    # --------------------------------------------------------
    # Early stopping
    # --------------------------------------------------------

    if (
        epochs_without_improvement
        >= PATIENCE
    ):

        print(

            f"\nEarly stopping triggered "
            f"after {PATIENCE} epochs "
            f"without improvement."
        )

        break


# ============================================================
# 23. SAVE LAST MODEL
# ============================================================

torch.save(

    {

        "epoch":
            epoch,

        "model_state_dict":
            model.state_dict(),

        "optimizer_state_dict":
            optimizer.state_dict(),

        "scheduler_state_dict":
            scheduler.state_dict(),

        "best_val_f1":
            best_val_f1,

        "classes":
            MOBILENET_CLASSES,

        "class_to_index":
            CLASS_TO_INDEX,

        "class_weights":
            class_weights,

    },

    LAST_MODEL_PATH
)


# ============================================================
# 24. SAVE TRAINING HISTORY
# ============================================================

history_df = pd.DataFrame(
    history
)


history_df.to_csv(

    TRAIN_HISTORY_CSV,

    index=False
)


print(
    "\nTraining history saved:"
)

print(
    TRAIN_HISTORY_CSV
)


# ============================================================
# 25. LOAD BEST MODEL
# ============================================================

print("\n" + "=" * 80)
print("LOADING BEST MODEL")
print("=" * 80)


checkpoint = torch.load(

    BEST_MODEL_PATH,

    map_location=DEVICE
)


model.load_state_dict(

    checkpoint[
        "model_state_dict"
    ]
)


model.eval()


print(
    f"\nBest epoch: "
    f"{checkpoint['epoch']}"
)


print(
    f"Best validation Macro F1: "
    f"{checkpoint['best_val_f1']:.6f}"
)


# ============================================================
# 26. FINAL VALIDATION
# ============================================================

print("\n" + "=" * 80)
print("FINAL VALIDATION")
print("=" * 80)


(
    final_val_loss,
    final_val_accuracy,
    final_val_macro_f1,
    final_true,
    final_pred
) = validate(

    model,

    val_loader,

    criterion,

    DEVICE
)


final_weighted_f1 = (
    f1_score(

        final_true,

        final_pred,

        labels=list(
            range(
                NUM_CLASSES
            )
        ),

        average="weighted",

        zero_division=0
    )
)


print(
    f"\nValidation loss: "
    f"{final_val_loss:.6f}"
)


print(
    f"Validation accuracy: "
    f"{final_val_accuracy:.6f}"
)


print(
    f"Validation Macro F1: "
    f"{final_val_macro_f1:.6f}"
)


print(
    f"Validation Weighted F1: "
    f"{final_weighted_f1:.6f}"
)


# ============================================================
# 27. CLASSIFICATION REPORT
# ============================================================

print("\n" + "=" * 80)
print("FINAL CLASSIFICATION REPORT")
print("=" * 80)


report = classification_report(

    final_true,

    final_pred,

    labels=list(
        range(
            NUM_CLASSES
        )
    ),

    target_names=MOBILENET_CLASSES,

    zero_division=0
)


print(
    report
)


# ============================================================
# 28. CONFUSION MATRIX
# ============================================================

cm = confusion_matrix(

    final_true,

    final_pred,

    labels=list(
        range(
            NUM_CLASSES
        )
    )
)


print("\n" + "=" * 80)
print("CONFUSION MATRIX")
print("=" * 80)


print(
    f"{'Actual / Predicted':25s}",

    *[
        f"{name:20s}"
        for name
        in MOBILENET_CLASSES
    ]
)


for i, row in enumerate(cm):

    print(

        f"{MOBILENET_CLASSES[i]:25s}",

        *[
            f"{value:20d}"
            for value
            in row
        ]
    )


# ============================================================
# 29. SAVE CONFUSION MATRIX
# ============================================================

with open(

    CONFUSION_MATRIX_CSV,

    "w",

    newline="",

    encoding="utf-8"

) as f:


    writer = csv.writer(
        f
    )


    writer.writerow(

        [
            "actual \\ predicted"
        ]

        +

        MOBILENET_CLASSES
    )


    for i, row in enumerate(cm):

        writer.writerow(

            [
                MOBILENET_CLASSES[i]
            ]

            +

            list(row)
        )


print(
    "\nConfusion matrix saved:"
)

print(
    CONFUSION_MATRIX_CSV
)


# ============================================================
# 30. PER-CLASS METRICS
# ============================================================

from sklearn.metrics import precision_recall_fscore_support


precision, recall, f1, support = (

    precision_recall_fscore_support(

        final_true,

        final_pred,

        labels=list(
            range(
                NUM_CLASSES
            )
        ),

        zero_division=0
    )
)


class_results = []


print("\n" + "=" * 80)
print("PER-CLASS RESULTS")
print("=" * 80)


for i, class_name in enumerate(

    MOBILENET_CLASSES

):


    result = {

        "class":
            class_name,

        "class_index":
            i,

        "precision":
            float(
                precision[i]
            ),

        "recall":
            float(
                recall[i]
            ),

        "f1":
            float(
                f1[i]
            ),

        "support":
            int(
                support[i]
            ),
    }


    class_results.append(
        result
    )


    print(

        f"{class_name:25s} "

        f"Precision="
        f"{precision[i]:.6f} "

        f"Recall="
        f"{recall[i]:.6f} "

        f"F1="
        f"{f1[i]:.6f} "

        f"Support="
        f"{support[i]:6d}"
    )


# ============================================================
# 31. SAVE PER-CLASS RESULTS
# ============================================================

with open(

    CLASS_RESULTS_CSV,

    "w",

    newline="",

    encoding="utf-8"

) as f:


    writer = csv.DictWriter(

        f,

        fieldnames=[

            "class",

            "class_index",

            "precision",

            "recall",

            "f1",

            "support",

        ]
    )


    writer.writeheader()


    writer.writerows(
        class_results
    )


print(
    "\nPer-class results saved:"
)

print(
    CLASS_RESULTS_CSV
)


# ============================================================
# 32. PLASTIC-ONLY CLASSIFICATION
# ============================================================
#
# The research focus is plastic classification.
#
# Plastic classes:
#
#   ecal
#   hdpe
#   mixed_plastic_rigid
#   mixed_plastic_soft
#   pet
#   pet_oil
#
# non_plastic is excluded from this metric.
#
# ============================================================

print("\n" + "=" * 80)
print("PLASTIC-ONLY CLASSIFICATION")
print("=" * 80)


plastic_class_names = [

    "ecal",

    "hdpe",

    "mixed_plastic_rigid",

    "mixed_plastic_soft",

    "pet",

    "pet_oil",
]


plastic_class_indices = [

    CLASS_TO_INDEX[
        class_name
    ]

    for class_name
    in plastic_class_names
]


plastic_true = []

plastic_pred = []


for true_label, pred_label in zip(

    final_true,

    final_pred

):


    if true_label in plastic_class_indices:

        plastic_true.append(
            true_label
        )

        plastic_pred.append(
            pred_label
        )


plastic_accuracy = (
    accuracy_score(

        plastic_true,

        plastic_pred
    )
)


plastic_macro_f1 = (
    f1_score(

        plastic_true,

        plastic_pred,

        labels=plastic_class_indices,

        average="macro",

        zero_division=0
    )
)


plastic_weighted_f1 = (
    f1_score(

        plastic_true,

        plastic_pred,

        labels=plastic_class_indices,

        average="weighted",

        zero_division=0
    )
)


print(
    f"\nPlastic samples: "
    f"{len(plastic_true):,}"
)


print(
    f"Plastic accuracy: "
    f"{plastic_accuracy:.6f}"
)


print(
    f"Plastic Macro F1: "
    f"{plastic_macro_f1:.6f}"
)


print(
    f"Plastic Weighted F1: "
    f"{plastic_weighted_f1:.6f}"
)


# ============================================================
# 33. TRAINING TIME
# ============================================================

training_time = (

    time.time()
    -
    training_start
)


# ============================================================
# 34. FINAL SUMMARY
# ============================================================

print("\n" + "=" * 80)
print("E4-A MOBILENET FINAL SUMMARY")
print("=" * 80)


print(

    f"\nTraining crops       : "
    f"{len(train_dataset):,}"
)


print(

    f"Validation crops     : "
    f"{len(val_dataset):,}"
)


print(

    f"Best epoch           : "
    f"{best_epoch}"
)


print(

    f"Best validation F1   : "
    f"{best_val_f1:.6f}"
)


print(

    f"\nValidation accuracy  : "
    f"{final_val_accuracy:.6f}"
)


print(

    f"Validation Macro F1  : "
    f"{final_val_macro_f1:.6f}"
)


print(

    f"Validation Weighted F1: "
    f"{final_weighted_f1:.6f}"
)


print(

    f"\nPlastic accuracy     : "
    f"{plastic_accuracy:.6f}"
)


print(

    f"Plastic Macro F1     : "
    f"{plastic_macro_f1:.6f}"
)


print(

    f"Plastic Weighted F1  : "
    f"{plastic_weighted_f1:.6f}"
)


print(

    f"\nTraining time        : "
    f"{training_time / 60:.2f} minutes"
)


print(
    "\nBest model:"
)

print(
    BEST_MODEL_PATH
)


# ============================================================
# 35. SAVE FINAL SUMMARY
# ============================================================

with open(

    SUMMARY_TXT,

    "w",

    encoding="utf-8"

) as f:


    f.write(

        "E4-A — FASTER R-CNN + MOBILENET-V3-LARGE\n"
    )

    f.write(

        "MOBILENET TRAINING RESULTS\n"
    )

    f.write(

        "=" * 70
        +
        "\n\n"
    )


    f.write(
        "PIPELINE\n"
    )

    f.write(

        "Faster R-CNN detections -> "
        "object crops -> "
        "MobileNet-V3-Large\n\n"
    )


    f.write(
        "EVALUATION TAXONOMY\n"
    )

    f.write(

        "Original SortWaste 8 classes -> "
        "7 evaluation classes\n"
    )

    f.write(

        "Metal + cardboard -> non_plastic\n\n"
    )


    f.write(
        "CLASSES\n"
    )

    for i, class_name in enumerate(

        MOBILENET_CLASSES

    ):

        f.write(

            f"{i}: {class_name}\n"
        )


    f.write(
        "\nDATASET\n"
    )

    f.write(

        f"Training crops: "
        f"{len(train_dataset)}\n"
    )

    f.write(

        f"Validation crops: "
        f"{len(val_dataset)}\n"
    )


    f.write(
        "\nTRAINING CONFIGURATION\n"
    )

    f.write(

        f"Batch size: "
        f"{BATCH_SIZE}\n"
    )

    f.write(

        f"Epochs: "
        f"{NUM_EPOCHS}\n"
    )

    f.write(

        f"Learning rate: "
        f"{LEARNING_RATE}\n"
    )

    f.write(

        f"Weight decay: "
        f"{WEIGHT_DECAY}\n"
    )

    f.write(

        f"Image size: "
        f"{IMAGE_SIZE}\n"
    )

    f.write(

        "Loss: Weighted Cross Entropy\n"
    )


    f.write(
        "\nBEST MODEL\n"
    )

    f.write(

        f"Best epoch: "
        f"{best_epoch}\n"
    )

    f.write(

        f"Best validation Macro F1: "
        f"{best_val_f1:.6f}\n"
    )


    f.write(
        "\nFINAL VALIDATION RESULTS\n"
    )

    f.write(

        f"Validation loss: "
        f"{final_val_loss:.6f}\n"
    )

    f.write(

        f"Validation accuracy: "
        f"{final_val_accuracy:.6f}\n"
    )

    f.write(

        f"Validation Macro F1: "
        f"{final_val_macro_f1:.6f}\n"
    )

    f.write(

        f"Validation Weighted F1: "
        f"{final_weighted_f1:.6f}\n"
    )


    f.write(
        "\nPLASTIC-ONLY RESULTS\n"
    )

    f.write(

        f"Plastic samples: "
        f"{len(plastic_true)}\n"
    )

    f.write(

        f"Plastic accuracy: "
        f"{plastic_accuracy:.6f}\n"
    )

    f.write(

        f"Plastic Macro F1: "
        f"{plastic_macro_f1:.6f}\n"
    )

    f.write(

        f"Plastic Weighted F1: "
        f"{plastic_weighted_f1:.6f}\n"
    )


    f.write(
        "\nMETHODOLOGICAL NOTES\n"
    )

    f.write(

        "1. MobileNet-V3-Large is trained on "
        "Faster R-CNN-generated object crops.\n"
    )

    f.write(

        "2. The Faster R-CNN detector is not "
        "trained by this script.\n"
    )

    f.write(

        "3. Metal and cardboard are mapped to "
        "non_plastic.\n"
    )

    f.write(

        "4. Seven evaluation classes are used.\n"
    )

    f.write(

        "5. Class-weighted cross entropy is used "
        "to address class imbalance.\n"
    )

    f.write(

        "6. Training augmentation consists of "
        "horizontal flipping, small rotation, "
        "and mild colour variation.\n"
    )

    f.write(

        "7. No augmentation is applied to the "
        "validation crops.\n"
    )

    f.write(

        "8. The best checkpoint is selected using "
        "validation Macro F1.\n"
    )

    f.write(

        "9. Plastic-only metrics exclude "
        "non_plastic samples.\n"
    )


# ============================================================
# 36. FINAL
# ============================================================

print("\n" + "=" * 80)
print("E4-A MOBILENET TRAINING COMPLETE")
print("=" * 80)


print(
    "\nBest checkpoint:"
)

print(
    BEST_MODEL_PATH
)


print(
    "\nTraining summary:"
)

print(
    SUMMARY_TXT
)


print(
    "\nNext stage:"
)

print(
    "Faster R-CNN detections "
    "-> MobileNet classification "
    "-> corrected end-to-end COCO evaluation"
)


print("=" * 80)

E4-A — MOBILENET-V3-LARGE TRAINING

Evaluation classes:
0: ecal
1: hdpe
2: mixed_plastic_rigid
3: mixed_plastic_soft
4: non_plastic
5: pet
6: pet_oil

Device: cuda
GPU: NVIDIA GeForce RTX 3050 Ti Laptop GPU
CUDA: 12.6

CHECKING FASTER R-CNN CROP DIRECTORY

Crop root:
C:\Users\varda\Documents\_My Computer\COMMON_space\Learning\Upgrad_EPGP_MS\upgrad_IIITB_MS_course\MS_LJMU_Material\Topic Data\SortWaste\dataset\dataset\faster_rcnn_mobilenet_crops_E4F
Exists: True

Train crop directory:
C:\Users\varda\Documents\_My Computer\COMMON_space\Learning\Upgrad_EPGP_MS\upgrad_IIITB_MS_course\MS_LJMU_Material\Topic Data\SortWaste\dataset\dataset\faster_rcnn_mobilenet_crops_E4F\train
Exists: True

Validation crop directory:
C:\Users\varda\Documents\_My Computer\COMMON_space\Learning\Upgrad_EPGP_MS\upgrad_IIITB_MS_course\MS_LJMU_Material\Topic Data\SortWaste\dataset\dataset\faster_rcnn_mobilenet_crops_E4F\val
Exists: True

CHECKING CLASS DIRECTORIES

TRAIN
ecal                      Exists=True
hdpe   

In [ ]:
## End to end Evaluation
# ============================================================
# E4F-A
# Faster R-CNN -> MobileNet-V3-Large
# CLASS-WEIGHTED END-TO-END EVALUATION
#
# Detector:
#   Faster R-CNN
#
# Classifier:
#   MobileNet-V3-Large
#
# Evaluation taxonomy:
#   Original 8 SortWaste classes
#       ->
#   7 evaluation classes
#
# Metal + cardboard -> non_plastic
#
# Final confidence:
#   Faster R-CNN confidence
#       x
#   MobileNet predicted-class probability
#
# COCOeval:
#   Primary end-to-end detection evaluation
#
# Diagnostic classification:
#   One-to-one IoU matching at IoU >= 0.50
#
# ============================================================


# ============================================================
# 1. IMPORTS
# ============================================================

import os
import csv
import json
import time
import random

import numpy as np
import pandas as pd
from PIL import Image

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

import torchvision
from torchvision.models import (
    mobilenet_v3_large,
    MobileNet_V3_Large_Weights
)
from torchvision.transforms import (
    Compose,
    Resize,
    ToTensor,
    Normalize
)

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix
)

from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval


# ============================================================
# 2. CONFIGURATION
# ============================================================

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)


# ------------------------------------------------------------
# DEVICE
# ------------------------------------------------------------

DEVICE = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)


# ------------------------------------------------------------
# DATASET ROOT
# ------------------------------------------------------------

DATASET_ROOT = (
    r"C:\Users\varda\Documents\_My Computer"
    r"\COMMON_space\Learning\Upgrad_EPGP_MS"
    r"\upgrad_IIITB_MS_course\MS_LJMU_Material"
    r"\Topic Data\SortWaste\dataset\dataset"
)


# ------------------------------------------------------------
# VALIDATION DATA
# ------------------------------------------------------------

VAL_IMAGES_DIR = os.path.join(

    DATASET_ROOT,

    "splited_all_dataset_coco",

    "val",

    "images"
)


VAL_LABELS_DIR = os.path.join(

    DATASET_ROOT,

    "splited_all_dataset_coco",

    "val",

    "labels"
)


VAL_COCO_JSON = os.path.join(

    DATASET_ROOT,

    "splited_all_dataset_coco",

    "val",

    "annotations",

    "val_coco.json"
)


# ------------------------------------------------------------
# FASTER R-CNN CHECKPOINT
#
# IMPORTANT:
# Replace this with the exact checkpoint you used
# for the Faster R-CNN baseline / crop generation.
# ------------------------------------------------------------

FASTER_RCNN_CHECKPOINT = (
    r"C:\Users\varda\Documents\_My Computer\COMMON_space\Learning\Upgrad_EPGP_MS\upgrad_IIITB_MS_course\MS_LJMU_Material\Thesis_Code\runs\sortwaste\faster_rcnn_baseline"
    r"\best_model.pth"
)


# ------------------------------------------------------------
# E4F-A MOBILENET CHECKPOINT
#
# IMPORTANT:
# Replace this with the exact E4F-A checkpoint.
# ------------------------------------------------------------

MOBILENET_CHECKPOINT = (
    r"C:\Users\varda\Documents\_My Computer"
    r"\COMMON_space\Learning\Upgrad_EPGP_MS"
    r"\upgrad_IIITB_MS_course\MS_LJMU_Material"
    r"\Topic Data\SortWaste\dataset\dataset"
    r"\faster_rcnn_mobilenet_crops_E4F\mobilenet_results\E4F-A_MobileNetV3Large"
    r"\E4FA_MobileNetV3Large_best.pth"
)


# ------------------------------------------------------------
# OUTPUT
# ------------------------------------------------------------

OUTPUT_ROOT = os.path.join(
    DATASET_ROOT,
    "faster_rcnn_mobilenet_crops_E4F",
    "mobilenet_results",
    "E4F_A_endtoend_eval"
)


os.makedirs(
    OUTPUT_ROOT,
    exist_ok=True
)


RAW_DETECTIONS_CSV = os.path.join(

    OUTPUT_ROOT,

    "E4F_A_all_detections.csv"
)


COCO_RESULTS_CSV = os.path.join(

    OUTPUT_ROOT,

    "E4F_A_COCO_results.csv"
)


CLASS_RESULTS_CSV = os.path.join(

    OUTPUT_ROOT,

    "E4F_A_class_results.csv"
)


MATCHED_RESULTS_CSV = os.path.join(

    OUTPUT_ROOT,

    "E4F_A_matched_classification.csv"
)


CONFUSION_MATRIX_CSV = os.path.join(

    OUTPUT_ROOT,

    "E4F_A_confusion_matrix.csv"
)


SUMMARY_TXT = os.path.join(

    OUTPUT_ROOT,

    "E4F_A_final_summary.txt"
)


# ============================================================
# 3. EVALUATION PARAMETERS
# ============================================================

# Faster R-CNN detection threshold.
#
# IMPORTANT:
# We use a very low threshold so that COCOeval can evaluate
# the detector/classifier combination across the confidence
# ranking rather than prematurely removing detections.
#
# This should correspond conceptually to the conf=0.001
# strategy used for E3Y-B.
# ------------------------------------------------------------

DETECTOR_CONF_THRESHOLD = 0.001


# ------------------------------------------------------------
# ONE-TO-ONE DIAGNOSTIC MATCHING
# ------------------------------------------------------------

MATCH_IOU_THRESHOLD = 0.50


# ------------------------------------------------------------
# IMAGE SIZE
# ------------------------------------------------------------

IMAGE_SIZE = 224


# ------------------------------------------------------------
# DATALOADER
# ------------------------------------------------------------

BATCH_SIZE = 1


# ============================================================
# 4. CLASS DEFINITIONS
# ============================================================

# Original SortWaste YOLO class order

YOLO_CLASSES = [

    "pet",

    "hdpe",

    "mixed_plastic_soft",

    "ecal",

    "metal",

    "cardboard",

    "mixed_plastic_rigid",

    "pet_oil"
]


# ------------------------------------------------------------
# Evaluation classes
# ------------------------------------------------------------

EVAL_CLASSES = [

    "ecal",

    "hdpe",

    "mixed_plastic_rigid",

    "mixed_plastic_soft",

    "non_plastic",

    "pet",

    "pet_oil"
]


NUM_EVAL_CLASSES = len(
    EVAL_CLASSES
)


# ------------------------------------------------------------
# Evaluation class IDs
# ------------------------------------------------------------

CLASS_TO_ID = {

    name: index + 1

    for index, name
    in enumerate(EVAL_CLASSES)
}


EVAL_ID_TO_CLASS = {

    index: name

    for name, index
    in CLASS_TO_ID.items()
}


# ------------------------------------------------------------
# Original YOLO -> evaluation class
# ------------------------------------------------------------

YOLO_TO_EVAL = {

    0: "pet",

    1: "hdpe",

    2: "mixed_plastic_soft",

    3: "ecal",

    4: "non_plastic",

    5: "non_plastic",

    6: "mixed_plastic_rigid",

    7: "pet_oil"
}


# ------------------------------------------------------------
# MobileNet class order
#
# This MUST match the class order used during E4F-A
# MobileNet training.
# ------------------------------------------------------------

MOBILENET_CLASSES = [

    "ecal",

    "hdpe",

    "mixed_plastic_rigid",

    "mixed_plastic_soft",

    "non_plastic",

    "pet",

    "pet_oil"
]


MOBILENET_CLASS_TO_INDEX = {

    name: index

    for index, name
    in enumerate(MOBILENET_CLASSES)
}


MOBILENET_INDEX_TO_CLASS = {
    index: name
    for index, name in enumerate(MOBILENET_CLASSES)
}


# ============================================================
# 5. FASTER R-CNN CLASS MAPPING
# ============================================================
#
# IMPORTANT:
#
# This assumes Faster R-CNN was trained using the original
# SortWaste 8-class YOLO/COCO ordering:
#
#   1 -> pet
#   2 -> hdpe
#   3 -> mixed_plastic_soft
#   4 -> ecal
#   5 -> metal
#   6 -> cardboard
#   7 -> mixed_plastic_rigid
#   8 -> pet_oil
#
# Faster R-CNN label 0 is background.
#
# If your Faster R-CNN training script used a different
# label mapping, CHANGE THIS SECTION accordingly.
# ============================================================


FASTER_RCNN_LABEL_TO_YOLO_CLASS = {

    1: 0,   # pet

    2: 1,   # hdpe

    3: 2,   # mixed_plastic_soft

    4: 3,   # ecal

    5: 4,   # metal

    6: 5,   # cardboard

    7: 6,   # mixed_plastic_rigid

    8: 7    # pet_oil
}


# ============================================================
# 6. HELPER FUNCTIONS
# ============================================================

def calculate_iou(
    box_a,
    box_b
):

    ax1, ay1, ax2, ay2 = box_a

    bx1, by1, bx2, by2 = box_b


    inter_x1 = max(
        ax1,
        bx1
    )

    inter_y1 = max(
        ay1,
        by1
    )

    inter_x2 = min(
        ax2,
        bx2
    )

    inter_y2 = min(
        ay2,
        by2
    )


    inter_width = max(
        0.0,
        inter_x2 - inter_x1
    )

    inter_height = max(
        0.0,
        inter_y2 - inter_y1
    )


    intersection = (
        inter_width
        *
        inter_height
    )


    area_a = (

        max(
            0.0,
            ax2 - ax1
        )

        *

        max(
            0.0,
            ay2 - ay1
        )
    )


    area_b = (

        max(
            0.0,
            bx2 - bx1
        )

        *

        max(
            0.0,
            by2 - by1
        )
    )


    union = (

        area_a
        +
        area_b
        -
        intersection
    )


    if union <= 0:

        return 0.0


    return (
        intersection
        /
        union
    )


# ============================================================
# 7. CHECK PATHS
# ============================================================

print("\n" + "=" * 80)

print(
    "E4F-A — Faster R-CNN -> MobileNet-V3-Large"
)

print(
    "CORRECTED FINAL END-TO-END COCO EVALUATION"
)

print("=" * 80)


print(
    f"\nDevice: {DEVICE}"
)


if torch.cuda.is_available():

    print(
        "GPU:",
        torch.cuda.get_device_name(0)
    )

    print(
        "CUDA:",
        torch.version.cuda
    )


print("\n" + "=" * 80)
print("CHECKING PATHS")
print("=" * 80)


paths_to_check = {

    "Dataset root":
        DATASET_ROOT,

    "Validation images":
        VAL_IMAGES_DIR,

    "Validation labels":
        VAL_LABELS_DIR,

    "Validation COCO JSON":
        VAL_COCO_JSON,

    "Faster R-CNN checkpoint":
        FASTER_RCNN_CHECKPOINT,

    "MobileNet checkpoint":
        MOBILENET_CHECKPOINT
}


for name, path in paths_to_check.items():

    print(
        f"{name:30s}: {path}"
    )

    print(
        f"{'':30s}  Exists: {os.path.exists(path)}"
    )


missing_paths = [

    name

    for name, path
    in paths_to_check.items()

    if not os.path.exists(path)
]


if missing_paths:

    raise FileNotFoundError(

        "Missing required paths:\n"
        +
        "\n".join(
            missing_paths
        )

    )


# ============================================================
# 8. CLASS MAPPING DISPLAY
# ============================================================

print("\n" + "=" * 80)
print("CLASS MAPPING")
print("=" * 80)


print(
    "\nOriginal SortWaste classes -> evaluation classes:"
)


for yolo_id, class_name in enumerate(
    YOLO_CLASSES
):

    print(

        f"YOLO {yolo_id} "
        f"({class_name:25s}) -> "
        f"{YOLO_TO_EVAL[yolo_id]}"
    )


print(
    "\nEvaluation COCO categories:"
)


for class_name, category_id in (
    CLASS_TO_ID.items()
):

    print(

        f"COCO ID {category_id} -> "
        f"{class_name}"
    )


# ============================================================
# 9. LOAD FASTER R-CNN
# ============================================================

print("\n" + "=" * 80)
print("LOADING FASTER R-CNN")
print("=" * 80)


# ------------------------------------------------------------
# IMPORTANT:
#
# This loader assumes the checkpoint contains a complete
# torchvision Faster R-CNN model or a state_dict compatible
# with the architecture below.
#
# If your Faster R-CNN training script used a custom
# architecture, the architecture section must match it.
# ------------------------------------------------------------


NUM_FASTER_RCNN_CLASSES = 9
# 8 foreground classes + background


faster_rcnn = (
    torchvision.models.detection
    .fasterrcnn_resnet50_fpn(
        weights=None,
        weights_backbone=None,
        num_classes=NUM_FASTER_RCNN_CLASSES
    )
)


checkpoint = torch.load(

    FASTER_RCNN_CHECKPOINT,

    map_location=DEVICE,

    weights_only=False

)


if isinstance(
    checkpoint,
    dict
):

    if "model_state_dict" in checkpoint:

        state_dict = (
            checkpoint[
                "model_state_dict"
            ]
        )

    elif "state_dict" in checkpoint:

        state_dict = (
            checkpoint[
                "state_dict"
            ]
        )

    else:

        state_dict = checkpoint

else:

    state_dict = checkpoint.state_dict()


# Remove possible DataParallel prefix

clean_state_dict = {}

for key, value in state_dict.items():

    if key.startswith("module."):

        key = key[
            len("module.") :
        ]

    clean_state_dict[key] = value


missing_keys, unexpected_keys = (
    faster_rcnn.load_state_dict(
        clean_state_dict,
        strict=False
    )
)


if missing_keys:

    print(
        "\nWARNING — Missing Faster R-CNN keys:"
    )

    print(
        missing_keys
    )


if unexpected_keys:

    print(
        "\nWARNING — Unexpected Faster R-CNN keys:"
    )

    print(
        unexpected_keys
    )


faster_rcnn = (
    faster_rcnn
    .to(DEVICE)
)


faster_rcnn.eval()


print(
    "\nFaster R-CNN loaded successfully."
)


# ============================================================
# 10. LOAD MOBILENET-V3-LARGE
# ============================================================

print("\n" + "=" * 80)
print("LOADING E4F-A MOBILENET-V3-LARGE")
print("=" * 80)


mobilenet = (
    mobilenet_v3_large(
        weights=None
    )
)


mobilenet.classifier[3] = (
    nn.Linear(
        mobilenet.classifier[3].in_features,
        NUM_EVAL_CLASSES
    )
)


mobilenet_checkpoint = torch.load(

    MOBILENET_CHECKPOINT,

    map_location=DEVICE,

    weights_only=False

)


if isinstance(
    mobilenet_checkpoint,
    dict
):

    if "model_state_dict" in (
        mobilenet_checkpoint
    ):

        mobilenet_state_dict = (
            mobilenet_checkpoint[
                "model_state_dict"
            ]
        )

    elif "state_dict" in (
        mobilenet_checkpoint
    ):

        mobilenet_state_dict = (
            mobilenet_checkpoint[
                "state_dict"
            ]
        )

    else:

        mobilenet_state_dict = (
            mobilenet_checkpoint
        )

else:

    mobilenet_state_dict = (
        mobilenet_checkpoint.state_dict()
    )


clean_mobilenet_state_dict = {}


for key, value in (
    mobilenet_state_dict.items()
):

    if key.startswith("module."):

        key = key[
            len("module.") :
        ]

    clean_mobilenet_state_dict[
        key
    ] = value


missing_keys, unexpected_keys = (
    mobilenet.load_state_dict(
        clean_mobilenet_state_dict,
        strict=False
    )
)


if missing_keys:

    print(
        "\nWARNING — Missing MobileNet keys:"
    )

    print(
        missing_keys
    )


if unexpected_keys:

    print(
        "\nWARNING — Unexpected MobileNet keys:"
    )

    print(
        unexpected_keys
    )


mobilenet = (
    mobilenet
    .to(DEVICE)
)


mobilenet.eval()


print(
    "\nMobileNet checkpoint:"
)

print(
    MOBILENET_CHECKPOINT
)

print(
    "\nMobileNet loaded successfully."
)


# ============================================================
# 11. MOBILENET TRANSFORM
# ============================================================

mobilenet_transform = Compose([

    Resize(
        (
            IMAGE_SIZE,
            IMAGE_SIZE
        )
    ),

    ToTensor(),

    Normalize(

        mean=[
            0.485,
            0.456,
            0.406
        ],

        std=[
            0.229,
            0.224,
            0.225
        ]
    )
])


# ============================================================
# 12. LOAD COCO VALIDATION ANNOTATIONS
# ============================================================

print("\n" + "=" * 80)
print("LOADING COCO VALIDATION ANNOTATIONS")
print("=" * 80)


coco_gt_original = COCO(
    VAL_COCO_JSON
)


print(
    "Original COCO images:",
    len(
        coco_gt_original.imgs
    )
)


print(
    "Original COCO annotations:",
    len(
        coco_gt_original.anns
    )
)


# ============================================================
# 13. BUILD 7-CLASS COCO GROUND TRUTH
# ============================================================

print("\n" + "=" * 80)
print("BUILDING 7-CLASS COCO GROUND TRUTH")
print("=" * 80)


# Original COCO IDs from the SortWaste dataset:
#
# 1 -> ecal
# 2 -> hdpe
# 3 -> mixed_plastic_rigid
# 4 -> mixed_plastic_soft
# 5 -> non_plastic
# 6 -> pet
# 7 -> pet_oil
#
# We determine the mapping from the original category names
# to avoid relying blindly on numeric IDs.


original_categories = (
    coco_gt_original
    .loadCats(
        coco_gt_original.getCatIds()
    )
)


original_name_to_id = {

    cat["name"].lower():
        cat["id"]

    for cat in original_categories
}


print(
    "\nOriginal COCO categories:"
)

for cat in original_categories:

    print(
        cat["id"],
        "->",
        cat["name"]
    )


# ------------------------------------------------------------
# Find original COCO IDs
# ------------------------------------------------------------

def find_original_category_id(
    possible_names
):

    for name in possible_names:

        name_lower = name.lower()

        if (
            name_lower
            in original_name_to_id
        ):

            return original_name_to_id[
                name_lower
            ]

    return None


original_to_eval_category = {}


for eval_class in EVAL_CLASSES:

    if eval_class == "non_plastic":

        # Metal + cardboard

        metal_id = (
            find_original_category_id(
                [
                    "metal"
                ]
            )
        )

        cardboard_id = (
            find_original_category_id(
                [
                    "cardboard"
                ]
            )
        )

        if metal_id is not None:

            original_to_eval_category[
                metal_id
            ] = CLASS_TO_ID[
                "non_plastic"
            ]

        if cardboard_id is not None:

            original_to_eval_category[
                cardboard_id
            ] = CLASS_TO_ID[
                "non_plastic"
            ]

    else:

        original_id = (
            find_original_category_id(
                [
                    eval_class
                ]
            )
        )

        if original_id is not None:

            original_to_eval_category[
                original_id
            ] = CLASS_TO_ID[
                eval_class
            ]


print(
    "\nOriginal COCO -> evaluation COCO mapping:"
)

for original_id, eval_id in (
    original_to_eval_category.items()
):

    print(
        f"{original_id} -> "
        f"{eval_id} "
        f"({EVAL_ID_TO_CLASS[eval_id]})"
    )


# ------------------------------------------------------------
# Build evaluation annotations
# ------------------------------------------------------------

evaluation_annotations = []


for ann_id, ann in (
    coco_gt_original.anns.items()
):

    original_category_id = (
        int(
            ann["category_id"]
        )
    )


    if (
        original_category_id
        not in original_to_eval_category
    ):

        continue


    new_ann = dict(
        ann
    )


    new_ann[
        "category_id"
    ] = (
        original_to_eval_category[
            original_category_id
        ]
    )


    evaluation_annotations.append(
        new_ann
    )


# ------------------------------------------------------------
# Build evaluation categories
# ------------------------------------------------------------

evaluation_categories = [

    {
        "id":
            CLASS_TO_ID[
                class_name
            ],

        "name":
            class_name,

        "supercategory":
            "waste"
    }

    for class_name
    in EVAL_CLASSES
]


# ------------------------------------------------------------
# Build COCO-format dictionary
# ------------------------------------------------------------

evaluation_coco_dict = {

    "images":
        list(
            coco_gt_original.imgs.values()
        ),

    "annotations":
        evaluation_annotations,

    "categories":
        evaluation_categories
}


EVAL_COCO_JSON = os.path.join(

    OUTPUT_ROOT,

    "E4F_A_eval_gt_7class.json"
)


with open(

    EVAL_COCO_JSON,

    "w",

    encoding="utf-8"

) as f:

    json.dump(

        evaluation_coco_dict,

        f
    )


coco_gt = COCO(
    EVAL_COCO_JSON
)


print(
    "\nEvaluation GT annotations:",
    len(
        evaluation_annotations
    )
)


print(
    "Evaluation classes:",
    len(
        EVAL_CLASSES
    )
)


# ============================================================
# 14. BUILD VALIDATION IMAGE LIST
# ============================================================

print("\n" + "=" * 80)
print("VALIDATION DATA")
print("=" * 80)


image_ids = sorted(

    coco_gt.imgs.keys()

)


image_paths = []


for image_id in image_ids:

    image_info = (
        coco_gt.imgs[
            image_id
        ]
    )


    file_name = (
        image_info[
            "file_name"
        ]
    )


    image_path = os.path.join(

        VAL_IMAGES_DIR,

        os.path.basename(
            file_name
        )
    )


    if not os.path.exists(
        image_path
    ):

        # Try the relative path

        image_path = os.path.join(

            VAL_IMAGES_DIR,

            file_name
        )


    if not os.path.exists(
        image_path
    ):

        raise FileNotFoundError(

            f"Validation image not found: "
            f"{file_name}"
        )


    image_paths.append(
        (
            image_id,
            image_path
        )
    )


print(
    "Validation images:",
    len(image_paths)
)


# ============================================================
# 15. START EVALUATION
# ============================================================

evaluation_start = time.time()


print("\n" + "=" * 80)
print("STARTING E4F-A END-TO-END EVALUATION")
print("=" * 80)


print(
    "\nFaster R-CNN confidence threshold:",
    DETECTOR_CONF_THRESHOLD
)


print(
    "Final confidence formula:"
)


print(
    "Faster R-CNN confidence x "
    "MobileNet class probability"
)


print(
    "\nDiagnostic matching IoU threshold:",
    MATCH_IOU_THRESHOLD
)


# ============================================================
# 16. STORAGE
# ============================================================

all_records = []

coco_predictions = []

matched_true = []

matched_pred = []

matched_records = []


faster_rcnn_detection_count = 0

mobilenet_prediction_count = 0


images_without_coco_id = 0

invalid_prediction_boxes = 0


gt_object_count = len(
    evaluation_annotations
)


# ============================================================
# 17. END-TO-END INFERENCE
# ============================================================

with torch.no_grad():

    for image_number, (
        image_id,
        image_path
    ) in enumerate(
        image_paths,
        start=1
    ):

        # ----------------------------------------------------
        # Load image
        # ----------------------------------------------------

        image = Image.open(
            image_path
        ).convert(
            "RGB"
        )


        image_width, image_height = (
            image.size
        )


        image_tensor = (
            ToTensor()(
                image
            )
            .to(DEVICE)
        )


        # ----------------------------------------------------
        # Faster R-CNN inference
        # ----------------------------------------------------

        outputs = faster_rcnn(
            [
                image_tensor
            ]
        )


        output = outputs[0]


        boxes = (
            output[
                "boxes"
            ]
            .detach()
            .cpu()
            .numpy()
        )


        scores = (
            output[
                "scores"
            ]
            .detach()
            .cpu()
            .numpy()
        )


        labels = (
            output[
                "labels"
            ]
            .detach()
            .cpu()
            .numpy()
        )


        # ----------------------------------------------------
        # Process detections
        # ----------------------------------------------------

        for detection_index in range(
            len(boxes)
        ):

            detector_confidence = float(
                scores[
                    detection_index
                ]
            )


            if (
                detector_confidence
                < DETECTOR_CONF_THRESHOLD
            ):

                continue


            faster_rcnn_label = int(
                labels[
                    detection_index
                ]
            )


            if (
                faster_rcnn_label
                not in
                FASTER_RCNN_LABEL_TO_YOLO_CLASS
            ):

                continue


            yolo_class_id = (
                FASTER_RCNN_LABEL_TO_YOLO_CLASS[
                    faster_rcnn_label
                ]
            )


            detector_class_name = (
                YOLO_CLASSES[
                    yolo_class_id
                ]
            )


            # ------------------------------------------------
            # Box
            # ------------------------------------------------

            box = (
                boxes[
                    detection_index
                ]
            )


            x1 = float(
                box[0]
            )

            y1 = float(
                box[1]
            )

            x2 = float(
                box[2]
            )

            y2 = float(
                box[3]
            )


            # ------------------------------------------------
            # Clip box
            # ------------------------------------------------

            x1 = max(
                0.0,
                min(
                    x1,
                    image_width
                )
            )

            y1 = max(
                0.0,
                min(
                    y1,
                    image_height
                )
            )

            x2 = max(
                0.0,
                min(
                    x2,
                    image_width
                )
            )

            y2 = max(
                0.0,
                min(
                    y2,
                    image_height
                )
            )


            if (
                x2 <= x1
                or
                y2 <= y1
            ):

                invalid_prediction_boxes += 1

                continue


            faster_rcnn_detection_count += 1


            # ------------------------------------------------
            # Crop detected object
            # ------------------------------------------------

            crop = image.crop(

                (
                    int(
                        x1
                    ),

                    int(
                        y1
                    ),

                    int(
                        x2
                    ),

                    int(
                        y2
                    )
                )
            )


            # ------------------------------------------------
            # MobileNet preprocessing
            # ------------------------------------------------

            crop_tensor = (
                mobilenet_transform(
                    crop
                )
                .unsqueeze(0)
                .to(DEVICE)
            )


            # ------------------------------------------------
            # MobileNet prediction
            # ------------------------------------------------

            mobilenet_logits = (
                mobilenet(
                    crop_tensor
                )
            )


            mobilenet_probabilities = (
                torch.softmax(
                    mobilenet_logits,
                    dim=1
                )
            )


            mobilenet_probability, \
            mobilenet_class_index = (
                torch.max(
                    mobilenet_probabilities,
                    dim=1
                )
            )


            mobilenet_probability = float(
                mobilenet_probability[
                    0
                ]
            )


            mobilenet_class_index = int(
                mobilenet_class_index[
                    0
                ]
            )


            final_class_name = (
                MOBILENET_INDEX_TO_CLASS[
                    mobilenet_class_index
                ]
            )


            final_eval_category_id = (
                CLASS_TO_ID[
                    final_class_name
                ]
            )


            # ------------------------------------------------
            # Final confidence
            # ------------------------------------------------

            final_confidence = (

                detector_confidence

                *

                mobilenet_probability
            )


            # ------------------------------------------------
            # Store COCO prediction
            # ------------------------------------------------

            coco_prediction = {

                "image_id":
                    int(
                        image_id
                    ),

                "category_id":
                    int(
                        final_eval_category_id
                    ),

                "bbox": [

                    x1,

                    y1,

                    x2 - x1,

                    y2 - y1
                ],

                "score":
                    float(
                        final_confidence
                    )
            }


            coco_predictions.append(
                coco_prediction
            )


            mobilenet_prediction_count += 1


            # ------------------------------------------------
            # Store detailed record
            # ------------------------------------------------

            record = {

                "image":
                    os.path.basename(
                        image_path
                    ),

                "image_id":
                    int(
                        image_id
                    ),

                "detection_index":
                    int(
                        detection_index
                    ),

                "x1":
                    x1,

                "y1":
                    y1,

                "x2":
                    x2,

                "y2":
                    y2,

                "faster_rcnn_label":
                    faster_rcnn_label,

                "detector_class_name":
                    detector_class_name,

                "faster_rcnn_confidence":
                    detector_confidence,

                "mobilenet_class_index":
                    mobilenet_class_index,

                "final_class_name":
                    final_class_name,

                "mobilenet_probability":
                    mobilenet_probability,

                "final_eval_category_id":
                    final_eval_category_id,

                "final_confidence":
                    final_confidence
            }


            all_records.append(
                record
            )


        # ----------------------------------------------------
        # Progress
        # ----------------------------------------------------

        if (

            image_number % 50 == 0

            or

            image_number
            == len(image_paths)

        ):

            print(

                f"Processed "
                f"{image_number}/"
                f"{len(image_paths)} "
                f"| Faster R-CNN detections: "
                f"{faster_rcnn_detection_count:,} "
                f"| MobileNet predictions: "
                f"{mobilenet_prediction_count:,}"
            )


# ============================================================
# 18. PRE-COCO SANITY CHECKS
# ============================================================

print("\n" + "=" * 80)
print("PRE-COCO SANITY CHECKS")
print("=" * 80)


print(
    "Validation images:",
    len(image_paths)
)


print(
    "COCO images:",
    len(coco_gt.imgs)
)


print(
    "Ground-truth objects:",
    gt_object_count
)


print(
    "Faster R-CNN detections:",
    faster_rcnn_detection_count
)


print(
    "MobileNet predictions:",
    mobilenet_prediction_count
)


print(
    "COCO predictions:",
    len(coco_predictions)
)


print(
    "Images without COCO ID:",
    images_without_coco_id
)


print(
    "Invalid prediction boxes:",
    invalid_prediction_boxes
)


if (
    len(coco_predictions)
    != mobilenet_prediction_count
):

    raise RuntimeError(

        "Mismatch between MobileNet predictions "
        "and COCO predictions."
    )


print(
    "\nSanity checks completed."
)


# ============================================================
# 19. SAVE RAW DETECTION RESULTS
# ============================================================

print("\n" + "=" * 80)
print("SAVING RAW DETECTION RESULTS")
print("=" * 80)


if all_records:

    with open(

        RAW_DETECTIONS_CSV,

        "w",

        newline="",

        encoding="utf-8"

    ) as f:

        writer = csv.DictWriter(

            f,

            fieldnames=list(
                all_records[0].keys()
            )
        )


        writer.writeheader()


        writer.writerows(
            all_records
        )


print(
    "Raw detection results saved:"
)


print(
    RAW_DETECTIONS_CSV
)


# ============================================================
# 20. COCO EVALUATION
# ============================================================

print("\n" + "=" * 80)
print("COCO EVALUATION")
print("=" * 80)


if len(coco_predictions) == 0:

    raise RuntimeError(
        "No COCO predictions generated."
    )


coco_prediction_results = (
    coco_gt.loadRes(
        coco_predictions
    )
)


coco_eval = COCOeval(

    coco_gt,

    coco_prediction_results,

    "bbox"
)


coco_eval.params.imgIds = (
    image_ids
)


coco_eval.evaluate()

coco_eval.accumulate()

coco_eval.summarize()


# ============================================================
# 21. STANDARD COCO METRICS
# ============================================================

map5095 = float(
    coco_eval.stats[0]
)


map50 = float(
    coco_eval.stats[1]
)


map75 = float(
    coco_eval.stats[2]
)


mar1 = float(
    coco_eval.stats[6]
)


mar10 = float(
    coco_eval.stats[7]
)


mar100 = float(
    coco_eval.stats[8]
)


print("\n" + "=" * 80)
print("FINAL E4F-A COCO METRICS")
print("=" * 80)


print(
    f"mAP50-95 : {map5095:.6f}"
)


print(
    f"mAP50-95 : {map5095 * 100:.2f}%"
)


print(
    f"mAP50    : {map50:.6f}"
)


print(
    f"mAP50    : {map50 * 100:.2f}%"
)


print(
    f"mAP75    : {map75:.6f}"
)


print(
    f"AR@1     : {mar1:.6f}"
)


print(
    f"AR@10    : {mar10:.6f}"
)


print(
    f"AR@100   : {mar100:.6f}"
)


# ============================================================
# 22. PER-CLASS COCO AP
# ============================================================

print("\n" + "=" * 80)
print("PER-CLASS COCO AP")
print("=" * 80)


precision_tensor = (
    coco_eval.eval[
        "precision"
    ]
)


class_results = []


for class_index, category_id in enumerate(
    coco_eval.params.catIds
):


    class_name = (
        EVAL_ID_TO_CLASS[
            category_id
        ]
    )


    # --------------------------------------------------------
    # AP50
    # --------------------------------------------------------

    precision_ap50 = (
        precision_tensor[
            0,
            :,
            class_index,
            0,
            -1
        ]
    )


    precision_ap50 = (
        precision_ap50[
            precision_ap50 > -1
        ]
    )


    if len(
        precision_ap50
    ) > 0:

        class_ap50 = float(
            np.mean(
                precision_ap50
            )
        )

    else:

        class_ap50 = 0.0


    # --------------------------------------------------------
    # AP50-95
    # --------------------------------------------------------

    precision_ap5095 = (
        precision_tensor[
            :,
            :,
            class_index,
            0,
            -1
        ]
    )


    precision_ap5095 = (
        precision_ap5095[
            precision_ap5095 > -1
        ]
    )


    if len(
        precision_ap5095
    ) > 0:

        class_ap5095 = float(
            np.mean(
                precision_ap5095
            )
        )

    else:

        class_ap5095 = 0.0


    # --------------------------------------------------------
    # GT count
    # --------------------------------------------------------

    gt_count = sum(

        1

        for ann
        in evaluation_annotations

        if (
            ann["category_id"]
            ==
            category_id
        )
    )


    # --------------------------------------------------------
    # Prediction count
    # --------------------------------------------------------

    prediction_count = sum(

        1

        for pred
        in coco_predictions

        if (
            pred["category_id"]
            ==
            category_id
        )
    )


    class_results.append({

        "class":
            class_name,

        "COCO_category_id":
            category_id,

        "AP50":
            class_ap50,

        "AP50_95":
            class_ap5095,

        "GT_count":
            gt_count,

        "predictions":
            prediction_count

    })


    print(

        f"{class_name:25s} "

        f"AP50={class_ap50:.6f} "

        f"AP50-95={class_ap5095:.6f} "

        f"GT={gt_count:6d} "

        f"Pred={prediction_count:6d}"
    )


# ============================================================
# 23. SAVE COCO RESULTS
# ============================================================

with open(

    COCO_RESULTS_CSV,

    "w",

    newline="",

    encoding="utf-8"

) as f:

    writer = csv.writer(
        f
    )


    writer.writerow([
        "metric",
        "value"
    ])


    writer.writerow([
        "mAP50",
        map50
    ])


    writer.writerow([
        "mAP50_95",
        map5095
    ])


    writer.writerow([
        "mAP75",
        map75
    ])


    writer.writerow([
        "AR@1",
        mar1
    ])


    writer.writerow([
        "AR@10",
        mar10
    ])


    writer.writerow([
        "AR@100",
        mar100
    ])


print(
    "\nCOCO summary saved:"
)


print(
    COCO_RESULTS_CSV
)


# ============================================================
# 24. SAVE PER-CLASS RESULTS
# ============================================================

with open(

    CLASS_RESULTS_CSV,

    "w",

    newline="",

    encoding="utf-8"

) as f:

    writer = csv.DictWriter(

        f,

        fieldnames=[

            "class",

            "COCO_category_id",

            "AP50",

            "AP50_95",

            "GT_count",

            "predictions"
        ]
    )


    writer.writeheader()


    writer.writerows(
        class_results
    )


print(
    "Per-class results saved:"
)


print(
    CLASS_RESULTS_CSV
)


# ============================================================
# 25. PROPER ONE-TO-ONE MATCHING
# ============================================================

print("\n" + "=" * 80)
print("PROPER ONE-TO-ONE DIAGNOSTIC MATCHING")
print("=" * 80)


# ------------------------------------------------------------
# Group predictions by image
# ------------------------------------------------------------

records_by_image = {}


for record in all_records:

    image_id = int(
        record["image_id"]
    )


    records_by_image.setdefault(

        image_id,

        []

    ).append(
        record
    )


# ------------------------------------------------------------
# Build GT by image
# ------------------------------------------------------------

gt_by_image = {}


for gt in evaluation_annotations:

    image_id = int(
        gt["image_id"]
    )


    bbox = gt["bbox"]


    gt_box = [

        float(
            bbox[0]
        ),

        float(
            bbox[1]
        ),

        float(
            bbox[0] + bbox[2]
        ),

        float(
            bbox[1] + bbox[3]
        )
    ]


    gt_record = {

        "annotation_id":
            int(
                gt["id"]
            ),

        "box":
            gt_box,

        "category_id":
            int(
                gt["category_id"]
            ),

        "class_name":
            EVAL_ID_TO_CLASS[
                int(
                    gt["category_id"]
                )
            ]
    }


    gt_by_image.setdefault(

        image_id,

        []

    ).append(
        gt_record
    )


# ------------------------------------------------------------
# Match
# ------------------------------------------------------------

for image_id in sorted(

    set(

        list(
            records_by_image.keys()
        )

        +

        list(
            gt_by_image.keys()
        )
    )
):


    image_records = (
        records_by_image.get(
            image_id,
            []
        )
    )


    image_gt = (
        gt_by_image.get(
            image_id,
            []
        )
    )


    # --------------------------------------------------------
    # Sort predictions by final confidence
    # --------------------------------------------------------

    image_records = sorted(

        image_records,

        key=lambda r:
            float(
                r[
                    "final_confidence"
                ]
            ),

        reverse=True
    )


    matched_gt_indices = set()


    # --------------------------------------------------------
    # Match predictions
    # --------------------------------------------------------

    for record in image_records:


        pred_box = [

            float(
                record["x1"]
            ),

            float(
                record["y1"]
            ),

            float(
                record["x2"]
            ),

            float(
                record["y2"]
            )
        ]


        best_iou = 0.0

        best_gt_index = None


        for gt_index, gt in enumerate(
            image_gt
        ):


            if (
                gt_index
                in
                matched_gt_indices
            ):

                continue


            iou = calculate_iou(

                pred_box,

                gt["box"]
            )


            if iou > best_iou:

                best_iou = iou

                best_gt_index = (
                    gt_index
                )


        # ----------------------------------------------------
        # Successful match
        # ----------------------------------------------------

        if (

            best_gt_index is not None

            and

            best_iou
            >= MATCH_IOU_THRESHOLD

        ):


            matched_gt_indices.add(
                best_gt_index
            )


            gt = image_gt[
                best_gt_index
            ]


            true_class_id = int(
                gt[
                    "category_id"
                ]
            )


            pred_class_id = int(
                record[
                    "final_eval_category_id"
                ]
            )


            matched_true.append(
                true_class_id
            )


            matched_pred.append(
                pred_class_id
            )


            matched_record = {

                "image":
                    record[
                        "image"
                    ],

                "image_id":
                    image_id,

                "detection_index":
                    record[
                        "detection_index"
                    ],

                "IoU":
                    best_iou,

                "GT_class":
                    gt[
                        "class_name"
                    ],

                "Predicted_class":
                    record[
                        "final_class_name"
                    ],

                "FasterRCNN_confidence":
                    record[
                        "faster_rcnn_confidence"
                    ],

                "MobileNet_probability":
                    record[
                        "mobilenet_probability"
                    ],

                "Final_confidence":
                    record[
                        "final_confidence"
                    ],

                "Correct":
                    (
                        true_class_id
                        ==
                        pred_class_id
                    )
            }


            matched_records.append(
                matched_record
            )


# ============================================================
# 26. MATCHED CLASSIFICATION METRICS
# ============================================================

matched_accuracy = 0.0

matched_macro_f1 = 0.0

matched_weighted_f1 = 0.0


if len(
    matched_true
) > 0:


    matched_accuracy = (
        accuracy_score(

            matched_true,

            matched_pred
        )
    )


    matched_macro_f1 = (
        f1_score(

            matched_true,

            matched_pred,

            labels=list(
                range(
                    1,
                    NUM_EVAL_CLASSES + 1
                )
            ),

            average="macro",

            zero_division=0
        )
    )


    matched_weighted_f1 = (
        f1_score(

            matched_true,

            matched_pred,

            labels=list(
                range(
                    1,
                    NUM_EVAL_CLASSES + 1
                )
            ),

            average="weighted",

            zero_division=0
        )
    )


    print(
        f"Matched samples : "
        f"{len(matched_true):,}"
    )


    print(
        f"Accuracy        : "
        f"{matched_accuracy:.6f}"
    )


    print(
        f"Macro F1        : "
        f"{matched_macro_f1:.6f}"
    )


    print(
        f"Weighted F1     : "
        f"{matched_weighted_f1:.6f}"
    )


    print(
        "\nClassification report:"
    )


    report = classification_report(

        matched_true,

        matched_pred,

        labels=list(
            range(
                1,
                NUM_EVAL_CLASSES + 1
            )
        ),

        target_names=
            MOBILENET_CLASSES,

        zero_division=0
    )


    print(
        report
    )


    # --------------------------------------------------------
    # Confusion matrix
    # --------------------------------------------------------

    cm = confusion_matrix(

        matched_true,

        matched_pred,

        labels=list(
            range(
                1,
                NUM_EVAL_CLASSES + 1
            )
        )
    )


    print(
        "\nConfusion matrix:"
    )


    print(

        " " * 25,

        *[
            f"{name:20s}"

            for name
            in MOBILENET_CLASSES
        ]
    )


    for i, row in enumerate(cm):

        print(

            f"{MOBILENET_CLASSES[i]:25s}",

            *[
                f"{value:20d}"

                for value
                in row
            ]
        )


    # --------------------------------------------------------
    # Save confusion matrix
    # --------------------------------------------------------

    with open(

        CONFUSION_MATRIX_CSV,

        "w",

        newline="",

        encoding="utf-8"

    ) as f:

        writer = csv.writer(
            f
        )


        writer.writerow(

            [
                "actual \\ predicted"
            ]

            +

            MOBILENET_CLASSES
        )


        for i, row in enumerate(cm):

            writer.writerow(

                [
                    MOBILENET_CLASSES[i]
                ]

                +

                list(row)
            )


# ============================================================
# 27. SAVE MATCHED RESULTS
# ============================================================

if matched_records:

    with open(

        MATCHED_RESULTS_CSV,

        "w",

        newline="",

        encoding="utf-8"

    ) as f:


        fieldnames = list(
            matched_records[0].keys()
        )


        writer = csv.DictWriter(

            f,

            fieldnames=fieldnames
        )


        writer.writeheader()


        writer.writerows(
            matched_records
        )


    print(
        "\nMatched diagnostic results saved:"
    )


    print(
        MATCHED_RESULTS_CSV
    )


# ============================================================
# 28. PLASTIC-ONLY CLASSIFICATION
# ============================================================

print("\n" + "=" * 80)
print("PLASTIC-ONLY END-TO-END CLASSIFICATION")
print("=" * 80)


plastic_class_ids = {

    CLASS_TO_ID[
        "ecal"
    ],

    CLASS_TO_ID[
        "hdpe"
    ],

    CLASS_TO_ID[
        "mixed_plastic_rigid"
    ],

    CLASS_TO_ID[
        "mixed_plastic_soft"
    ],

    CLASS_TO_ID[
        "pet"
    ],

    CLASS_TO_ID[
        "pet_oil"
    ]
}


plastic_true = []

plastic_pred = []


for true, pred in zip(

    matched_true,

    matched_pred

):


    if true in plastic_class_ids:

        plastic_true.append(
            true
        )

        plastic_pred.append(
            pred
        )


plastic_accuracy = 0.0

plastic_macro_f1 = 0.0

plastic_weighted_f1 = 0.0


if len(
    plastic_true
) > 0:


    plastic_accuracy = (
        accuracy_score(

            plastic_true,

            plastic_pred
        )
    )


    plastic_macro_f1 = (
        f1_score(

            plastic_true,

            plastic_pred,

            labels=sorted(
                plastic_class_ids
            ),

            average="macro",

            zero_division=0
        )
    )


    plastic_weighted_f1 = (
        f1_score(

            plastic_true,

            plastic_pred,

            labels=sorted(
                plastic_class_ids
            ),

            average="weighted",

            zero_division=0
        )
    )


    print(
        f"Plastic samples     : "
        f"{len(plastic_true):,}"
    )


    print(
        f"Plastic accuracy    : "
        f"{plastic_accuracy:.6f}"
    )


    print(
        f"Plastic Macro F1    : "
        f"{plastic_macro_f1:.6f}"
    )


    print(
        f"Plastic Weighted F1 : "
        f"{plastic_weighted_f1:.6f}"
    )


# ============================================================
# 29. FINAL COUNTS
# ============================================================

evaluation_time = (

    time.time()

    -

    evaluation_start
)


print("\n" + "=" * 80)
print("E4F-A FINAL SUMMARY")
print("=" * 80)


print(

    f"\nValidation images       : "
    f"{len(image_paths):,}"
)


print(

    f"Ground-truth objects    : "
    f"{gt_object_count:,}"
)


print(

    f"Faster R-CNN detections : "
    f"{faster_rcnn_detection_count:,}"
)


print(

    f"MobileNet predictions   : "
    f"{mobilenet_prediction_count:,}"
)


print(

    f"COCO predictions        : "
    f"{len(coco_predictions):,}"
)


print(

    f"Proper GT matches       : "
    f"{len(matched_true):,}"
)


correct_classes = sum(1 for t, p in zip(matched_true, matched_pred) if t == p)
print(f"Correct final classes   : {correct_classes:,}")



print(

    f"\nE4F-A mAP50             : "
    f"{map50:.6f}"
)


print(

    f"E4F-A mAP50             : "
    f"{map50 * 100:.2f}%"
)


print(

    f"\nE4F-A mAP50-95          : "
    f"{map5095:.6f}"
)


print(

    f"E4F-A mAP50-95          : "
    f"{map5095 * 100:.2f}%"
)


if len(
    matched_true
) > 0:

    print(

        f"\nMatched classification accuracy : "
        f"{matched_accuracy:.6f}"
    )


    print(

        f"Matched Macro F1                : "
        f"{matched_macro_f1:.6f}"
    )


    print(

        f"Matched Weighted F1             : "
        f"{matched_weighted_f1:.6f}"
    )


if len(
    plastic_true
) > 0:

    print(

        f"\nPlastic-only accuracy           : "
        f"{plastic_accuracy:.6f}"
    )


    print(

        f"Plastic-only Macro F1           : "
        f"{plastic_macro_f1:.6f}"
    )


    print(

        f"Plastic-only Weighted F1        : "
        f"{plastic_weighted_f1:.6f}"
    )


print(

    f"\nEvaluation time        : "
    f"{evaluation_time / 60:.2f} minutes"
)


print(
    "\nResults directory:"
)


print(
    OUTPUT_ROOT
)


# ============================================================
# 30. SAVE FINAL SUMMARY
# ============================================================

with open(

    SUMMARY_TXT,

    "w",

    encoding="utf-8"

) as f:


    f.write(

        "E4F-A — CORRECTED FINAL END-TO-END "
        "COCO EVALUATION\n"
    )


    f.write(
        "=" * 70
        +
        "\n\n"
    )


    f.write(
        "PIPELINE\n"
    )


    f.write(

        "Faster R-CNN -> object crop -> "
        "MobileNet-V3-Large\n\n"
    )


    f.write(
        "EXPERIMENT\n"
    )


    f.write(

        "E4F-A uses the class-weighted "
        "MobileNet-V3-Large checkpoint.\n\n"
    )


    f.write(
        "CONFIDENCE\n"
    )


    f.write(

        "Final confidence = "
        "Faster R-CNN confidence x "
        "MobileNet predicted-class probability\n\n"
    )


    f.write(
        "EVALUATION TAXONOMY\n"
    )


    f.write(

        "Original 8 SortWaste classes -> "
        "7 evaluation classes\n"
    )


    f.write(

        "Metal + cardboard -> non_plastic\n\n"
    )


    f.write(
        "DATASET\n"
    )


    f.write(

        f"Validation images: "
        f"{len(image_paths)}\n"
    )


    f.write(

        f"Ground truth objects: "
        f"{gt_object_count}\n"
    )


    f.write(
        "\nPREDICTIONS\n"
    )


    f.write(

        f"Faster R-CNN detections: "
        f"{faster_rcnn_detection_count}\n"
    )


    f.write(

        f"MobileNet predictions: "
        f"{mobilenet_prediction_count}\n"
    )


    f.write(

        f"COCO predictions: "
        f"{len(coco_predictions)}\n"
    )


    f.write(
        "\nPRIMARY COCO DETECTION METRICS\n"
    )


    f.write(

        f"mAP50: "
        f"{map50:.6f}\n"
    )


    f.write(

        f"mAP50-95: "
        f"{map5095:.6f}\n"
    )


    f.write(

        f"mAP75: "
        f"{map75:.6f}\n"
    )


    f.write(

        f"AR@1: "
        f"{mar1:.6f}\n"
    )


    f.write(

        f"AR@10: "
        f"{mar10:.6f}\n"
    )


    f.write(

        f"AR@100: "
        f"{mar100:.6f}\n"
    )


    f.write(
        "\nDIAGNOSTIC ONE-TO-ONE CLASSIFICATION\n"
    )


    f.write(

        f"IoU threshold: "
        f"{MATCH_IOU_THRESHOLD:.2f}\n"
    )


    f.write(

        f"Matched samples: "
        f"{len(matched_true)}\n"
    )


    f.write(

        f"Accuracy: "
        f"{matched_accuracy:.6f}\n"
    )


    f.write(

        f"Macro F1: "
        f"{matched_macro_f1:.6f}\n"
    )


    f.write(

        f"Weighted F1: "
        f"{matched_weighted_f1:.6f}\n"
    )


    f.write(
        "\nPLASTIC-ONLY CLASSIFICATION\n"
    )


    f.write(

        f"Plastic samples: "
        f"{len(plastic_true)}\n"
    )


    f.write(

        f"Plastic accuracy: "
        f"{plastic_accuracy:.6f}\n"
    )


    f.write(

        f"Plastic Macro F1: "
        f"{plastic_macro_f1:.6f}\n"
    )


    f.write(

        f"Plastic Weighted F1: "
        f"{plastic_weighted_f1:.6f}\n"
    )


    f.write(
        "\nSANITY CHECKS\n"
    )


    f.write(

        f"Images without COCO ID: "
        f"{images_without_coco_id}\n"
    )


    f.write(

        f"Invalid prediction boxes: "
        f"{invalid_prediction_boxes}\n"
    )


    f.write(
        "\nMETHODOLOGICAL NOTES\n"
    )


    f.write(
        "1. Faster R-CNN provides the bounding boxes.\n"
    )


    f.write(
        "2. MobileNet provides the final class.\n"
    )


    f.write(
        "3. Final confidence uses detector confidence "
        "x MobileNet predicted-class probability.\n"
    )


    f.write(
        "4. COCO annotations are used as the primary "
        "ground truth source.\n"
    )


    f.write(
        "5. Original 8-class COCO annotations are "
        "remapped to the 7-class evaluation taxonomy.\n"
    )


    f.write(
        "6. Metal and cardboard are mapped to "
        "non_plastic.\n"
    )


    f.write(
        "7. No random augmentation is used during "
        "evaluation.\n"
    )


    f.write(
        "8. COCOeval is the primary end-to-end "
        "detection evaluation.\n"
    )


    f.write(
        "9. Classification metrics use one-to-one "
        "IoU matching and are diagnostic.\n"
    )


    f.write(
        "10. Each GT object can be matched to at "
        "most one prediction.\n"
    )


    f.write(
        "11. MobileNet was trained using class-weighted "
        "cross-entropy for E4F-A.\n"
    )


# ============================================================
# 31. FINAL
# ============================================================

print("\n" + "=" * 80)
print("IMPORTANT METHODOLOGICAL NOTES")
print("=" * 80)


print(

    "\n1. E4F-A uses all Faster R-CNN validation "
    "detections above conf=0.001."
)


print(
    "2. Faster R-CNN supplies the bounding box."
)


print(
    "3. MobileNet supplies the final class."
)


print(

    "4. Final confidence = Faster R-CNN confidence "
    "x MobileNet predicted-class probability."
)


print(
    "5. COCOeval is the primary end-to-end detection evaluation."
)


print(

    "6. COCO GT is remapped from 8 original classes "
    "to 7 evaluation classes."
)


print(
    "7. Metal + cardboard -> non_plastic."
)


print(
    "8. Diagnostic classification uses proper "
    "one-to-one IoU matching."
)


print(
    "9. No random augmentation is applied during evaluation."
)


print(
    "10. E4F-A uses the class-weighted MobileNet-V3-Large checkpoint."
)


print(
    "\nE4F-A corrected evaluation complete."
)


print("=" * 80)


E4F-A — Faster R-CNN -> MobileNet-V3-Large
CORRECTED FINAL END-TO-END COCO EVALUATION

Device: cuda
GPU: NVIDIA GeForce RTX 3050 Ti Laptop GPU
CUDA: 12.6

CHECKING PATHS
Dataset root                  : C:\Users\varda\Documents\_My Computer\COMMON_space\Learning\Upgrad_EPGP_MS\upgrad_IIITB_MS_course\MS_LJMU_Material\Topic Data\SortWaste\dataset\dataset
                                Exists: True
Validation images             : C:\Users\varda\Documents\_My Computer\COMMON_space\Learning\Upgrad_EPGP_MS\upgrad_IIITB_MS_course\MS_LJMU_Material\Topic Data\SortWaste\dataset\dataset\splited_all_dataset_coco\val\images
                                Exists: True
Validation labels             : C:\Users\varda\Documents\_My Computer\COMMON_space\Learning\Upgrad_EPGP_MS\upgrad_IIITB_MS_course\MS_LJMU_Material\Topic Data\SortWaste\dataset\dataset\splited_all_dataset_coco\val\labels
                                Exists: True
Validation COCO JSON          : C:\Users\varda\Documents\_My Computer\

# Model E5 - YOLO11s, 7-class plastic-focused taxonomy, realistic augmentation, and class-aware oversampling @640

In [ ]:
# E5 — YOLO11s + 7-Class Taxonomy + Realistic Augmentation + Class-Aware Oversampling
#
# LJMU Thesis — SortWaste
# ======================================================================
#
# FINAL E5 DEFINITION
#
# Model:
#     YOLO11s pretrained
#
# Taxonomy:
#     0 = ecal
#     1 = hdpe
#     2 = mixed_plastic_rigid
#     3 = mixed_plastic_soft
#     4 = non_plastic
#     5 = pet
#     6 = pet_oil
#
# Original SortWaste:
#     metal + cardboard -> non_plastic
#
# Class-aware oversampling:
#     pet_oil      -> 3x training exposure
#     non_plastic  -> 2x training exposure
#     all plastics -> 1x
#
# IMPORTANT:
#     - Original dataset is NEVER modified.
#     - Validation/test are NEVER oversampled.
#     - Derived 7-class labels are created separately.
#     - Images are hard-linked, NOT copied.
#     - Hard links consume essentially no duplicate image storage.
#
# ======================================================================


import os
import sys
import random
import shutil
from pathlib import Path
from collections import Counter, defaultdict

import torch
import yaml

from ultralytics import YOLO


# ======================================================================
# 1. GLOBAL SETTINGS
# ======================================================================

SEED = 42

random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)


# ======================================================================
# 2. YOUR ORIGINAL SORTWASTE DATA YAML
# ======================================================================

DATA_YAML = Path(
    r"C:\Users\varda\Documents\_My Computer\COMMON_space\Learning"
    r"\Upgrad_EPGP_MS\upgrad_IIITB_MS_course\MS_LJMU_Material"
    r"\Topic Data\SortWaste\dataset\dataset"
    r"\splited_all_dataset_coco\data_pc.yaml"
)


# ======================================================================
# 3. THESIS OUTPUT ROOT
# ======================================================================

PROJECT_ROOT = Path(
    r"C:\Users\varda\Documents\_My Computer\COMMON_space\Learning"
    r"\Upgrad_EPGP_MS\upgrad_IIITB_MS_course"
    r"\MS_LJMU_Material\Thesis_Code"
    r"\runs\sortwaste"
)


EXPERIMENT_NAME = "E5_yolo11s_7class_aug_classbalance_640"


# ======================================================================
# 4. E5 DERIVED DATASET LOCATION
# ======================================================================

E5_DATASET_ROOT = (
    PROJECT_ROOT
    / "E5_7class_dataset"
)


E5_DATA_YAML = (
    E5_DATASET_ROOT
    / "E5_7class.yaml"
)


# ======================================================================
# 5. ORIGINAL 8-CLASS SORTWASTE TAXONOMY
# ======================================================================
#
# Original YOLO class indices:
#
# 0 pet
# 1 hdpe
# 2 mixed_plastic_soft
# 3 ecal
# 4 metal
# 5 cardboard
# 6 mixed_plastic_rigid
# 7 pet_oil
#
# ======================================================================

ORIGINAL_NAMES = {
    0: "pet",
    1: "hdpe",
    2: "mixed_plastic_soft",
    3: "ecal",
    4: "metal",
    5: "cardboard",
    6: "mixed_plastic_rigid",
    7: "pet_oil",
}


# ======================================================================
# 6. NEW E5 7-CLASS TAXONOMY
# ======================================================================

E5_NAMES = {
    0: "ecal",
    1: "hdpe",
    2: "mixed_plastic_rigid",
    3: "mixed_plastic_soft",
    4: "non_plastic",
    5: "pet",
    6: "pet_oil",
}


# ======================================================================
# 7. ORIGINAL CLASS -> E5 CLASS
# ======================================================================

CLASS_MAPPING = {

    # pet -> pet
    0: 5,

    # hdpe -> hdpe
    1: 1,

    # mixed soft -> mixed soft
    2: 3,

    # ecal -> ecal
    3: 0,

    # metal -> non_plastic
    4: 4,

    # cardboard -> non_plastic
    5: 4,

    # mixed rigid -> mixed rigid
    6: 2,

    # pet oil -> pet oil
    7: 6,
}


# ======================================================================
# 8. CLASS-AWARE OVERSAMPLING
# ======================================================================
#
# IMPORTANT:
#
# non_plastic = original metal + cardboard
#
# combined original train objects:
#
#     945 + 1524 = 2469
#
# PET Oil:
#
#     802
#
# We therefore use MODERATE oversampling only.
#
# ======================================================================

REPEAT_FACTOR = {

    0: 1,    # ecal

    1: 1,    # hdpe

    2: 1,    # mixed rigid

    3: 1,    # mixed soft

    4: 2,    # non_plastic

    5: 1,    # pet

    6: 3,    # pet_oil
}


# ======================================================================
# 9. IMAGE EXTENSIONS
# ======================================================================

IMAGE_EXTENSIONS = {
    ".jpg",
    ".jpeg",
    ".png",
    ".bmp",
    ".tif",
    ".tiff",
    ".webp",
}


# ======================================================================
# 10. ENVIRONMENT INFORMATION
# ======================================================================

print("=" * 90)
print("E5 — YOLO11s / 7 CLASS / AUGMENTATION / CLASS BALANCING")
print("=" * 90)

print("Python          :", sys.version.split()[0])
print("PyTorch         :", torch.__version__)
print("CUDA available  :", torch.cuda.is_available())


try:
    import ultralytics

    print(
        "Ultralytics    :",
        ultralytics.__version__
    )

except Exception:
    pass


if torch.cuda.is_available():

    DEVICE = 0

    print(
        "GPU             :",
        torch.cuda.get_device_name(0)
    )

    gpu_memory = (
        torch.cuda.get_device_properties(0).total_memory
        / 1024 ** 3
    )

    print(
        "GPU memory      :",
        f"{gpu_memory:.2f} GB"
    )

else:

    DEVICE = "cpu"

    print(
        "\nWARNING: CUDA unavailable."
    )

    print(
        "YOLO11s training on CPU will be very slow."
    )


# ======================================================================
# 11. VERIFY DATA YAML
# ======================================================================

print("\n" + "=" * 90)
print("SOURCE DATASET CHECK")
print("=" * 90)


print(
    "Original YAML:"
)

print(
    DATA_YAML
)


print(
    "Exists:",
    DATA_YAML.exists()
)


if not DATA_YAML.exists():

    raise FileNotFoundError(
        f"\nCould not find dataset YAML:\n{DATA_YAML}"
    )


# ======================================================================
# 12. READ ORIGINAL YAML
# ======================================================================

with open(
    DATA_YAML,
    "r",
    encoding="utf-8"
) as f:

    original_cfg = yaml.safe_load(f)


print(
    "\nOriginal YAML contents:"
)


for key, value in original_cfg.items():

    print(
        f"{key}: {value}"
    )


# ======================================================================
# 13. RESOLVE DATASET ROOT
# ======================================================================

yaml_parent = DATA_YAML.parent


yaml_root = original_cfg.get(
    "path",
    None
)


if yaml_root is None:

    SOURCE_DATASET_ROOT = (
        yaml_parent.resolve()
    )

else:

    root_path = Path(
        str(yaml_root)
    )


    if root_path.is_absolute():

        SOURCE_DATASET_ROOT = (
            root_path.resolve()
        )

    else:

        SOURCE_DATASET_ROOT = (
            yaml_parent
            / root_path
        ).resolve()


print(
    "\nResolved source dataset root:"
)

print(
    SOURCE_DATASET_ROOT
)


# ======================================================================
# 14. SPLIT PATH RESOLUTION
# ======================================================================

def resolve_split(entry):

    if entry is None:

        return None


    p = Path(
        str(entry)
    )


    if p.is_absolute():

        return p.resolve()


    candidate = (
        SOURCE_DATASET_ROOT
        / p
    ).resolve()


    if candidate.exists():

        return candidate


    candidate = (
        yaml_parent
        / p
    ).resolve()


    return candidate


TRAIN_SOURCE = resolve_split(
    original_cfg.get("train")
)


VAL_SOURCE = resolve_split(
    original_cfg.get("val")
)


TEST_SOURCE = resolve_split(
    original_cfg.get("test")
)


if TRAIN_SOURCE is None:

    raise RuntimeError(
        "No train split found in data_pc.yaml"
    )


if VAL_SOURCE is None:

    raise RuntimeError(
        "No val split found in data_pc.yaml"
    )


print("\nResolved splits:")

print("TRAIN :", TRAIN_SOURCE)
print("VAL   :", VAL_SOURCE)
print("TEST  :", TEST_SOURCE)


# ======================================================================
# 15. LOAD IMAGE PATHS
# ======================================================================

def load_image_paths(source):

    source = Path(
        source
    )


    # --------------------------------------------------------------
    # Normal image directory
    # --------------------------------------------------------------

    if source.is_dir():

        return sorted(

            p.resolve()

            for p in source.rglob("*")

            if p.is_file()

            and p.suffix.lower()
            in IMAGE_EXTENSIONS
        )


    # --------------------------------------------------------------
    # Image-list TXT
    # --------------------------------------------------------------

    if source.is_file():

        image_paths = []


        with open(
            source,
            "r",
            encoding="utf-8"
        ) as f:

            for line in f:

                line = line.strip()


                if not line:

                    continue


                p = Path(
                    line
                )


                if not p.is_absolute():

                    p = (
                        source.parent
                        / p
                    ).resolve()


                image_paths.append(
                    p
                )


        return image_paths


    raise FileNotFoundError(
        f"Split source does not exist:\n{source}"
    )


train_images = load_image_paths(
    TRAIN_SOURCE
)


val_images = load_image_paths(
    VAL_SOURCE
)


if TEST_SOURCE is not None:

    test_images = load_image_paths(
        TEST_SOURCE
    )

else:

    test_images = []


print("\nImage counts:")

print(
    "Train:",
    len(train_images)
)

print(
    "Val  :",
    len(val_images)
)

print(
    "Test :",
    len(test_images)
)


if len(train_images) == 0:

    raise RuntimeError(
        "No training images found."
    )


if len(val_images) == 0:

    raise RuntimeError(
        "No validation images found."
    )


# ======================================================================
# 16. CONVERT IMAGE PATH -> ORIGINAL YOLO LABEL PATH
# ======================================================================

def image_to_label_path(image_path):

    image_path = Path(
        image_path
    )


    parts = list(
        image_path.parts
    )


    image_indexes = [

        i

        for i, part
        in enumerate(parts)

        if part.lower() == "images"
    ]


    if not image_indexes:

        raise RuntimeError(
            "\nUnable to infer label path from image:"
            f"\n{image_path}"
            "\nExpected an 'images' directory."
        )


    idx = image_indexes[-1]


    parts[idx] = "labels"


    return Path(
        *parts
    ).with_suffix(
        ".txt"
    )


# ======================================================================
# 17. CHECK ORIGINAL LABEL DISTRIBUTION
# ======================================================================

original_train_counts = Counter()

mapped_train_counts = Counter()

train_images_per_new_class = Counter()

missing_labels = []

invalid_rows = []


for image_path in train_images:

    label_path = image_to_label_path(
        image_path
    )


    if not label_path.exists():

        missing_labels.append(
            label_path
        )

        continue


    classes_present = set()


    with open(
        label_path,
        "r",
        encoding="utf-8"
    ) as f:

        for row_number, line in enumerate(
            f,
            start=1
        ):

            line = line.strip()


            if not line:

                continue


            parts = line.split()


            if len(parts) < 5:

                invalid_rows.append(
                    (
                        label_path,
                        row_number,
                        line
                    )
                )

                continue


            try:

                old_class = int(
                    float(parts[0])
                )

            except Exception:

                invalid_rows.append(
                    (
                        label_path,
                        row_number,
                        line
                    )
                )

                continue


            if old_class not in CLASS_MAPPING:

                raise RuntimeError(
                    "\nUnexpected source class:"
                    f"\nClass ID: {old_class}"
                    f"\nFile: {label_path}"
                )


            new_class = CLASS_MAPPING[
                old_class
            ]


            original_train_counts[
                old_class
            ] += 1


            mapped_train_counts[
                new_class
            ] += 1


            classes_present.add(
                new_class
            )


    for cls in classes_present:

        train_images_per_new_class[
            cls
        ] += 1


# ======================================================================
# 18. PRINT ORIGINAL 8-CLASS COUNTS
# ======================================================================

print("\n" + "=" * 90)
print("ORIGINAL 8-CLASS TRAIN DISTRIBUTION")
print("=" * 90)


for cls in range(8):

    print(
        f"{cls:<2d} "
        f"{ORIGINAL_NAMES[cls]:25s} "
        f"{original_train_counts[cls]:8d}"
    )


print(
    "\nMissing label files:",
    len(missing_labels)
)


print(
    "Invalid label rows:",
    len(invalid_rows)
)


if missing_labels:

    print(
        "\nFirst missing label files:"
    )

    for p in missing_labels[:10]:

        print(p)


if len(missing_labels) > 0:

    raise RuntimeError(
        "\nSome source images have no YOLO labels."
        "\nE5 dataset generation stopped."
    )


# ======================================================================
# 19. PRINT REMAPPED 7-CLASS COUNTS
# ======================================================================

print("\n" + "=" * 90)
print("E5 7-CLASS TRAIN DISTRIBUTION — BEFORE OVERSAMPLING")
print("=" * 90)


for cls in range(7):

    print(
        f"{cls:<2d} "
        f"{E5_NAMES[cls]:25s} "
        f"objects = {mapped_train_counts[cls]:7d} | "
        f"images = {train_images_per_new_class[cls]:5d} | "
        f"repeat = {REPEAT_FACTOR[cls]}x"
    )


print(
    "\nExpected non_plastic count approximately:"
)

print(
    "metal 945 + cardboard 1524 = 2469"
)


# ======================================================================
# 20. OPTIONAL CLEAN REBUILD
# ======================================================================
#
# Set True when first creating E5.
#
# If you rerun this script from scratch, True ensures no stale
# generated labels/hardlinks remain.
#
# ======================================================================

REBUILD_E5_DATASET = True


if (
    REBUILD_E5_DATASET
    and E5_DATASET_ROOT.exists()
):

    print(
        "\nRemoving previous generated E5 dataset..."
    )

    shutil.rmtree(
        E5_DATASET_ROOT
    )


# ======================================================================
# 21. CREATE E5 DIRECTORY STRUCTURE
# ======================================================================

for split in [
    "train",
    "val",
    "test"
]:

    (
        E5_DATASET_ROOT
        / split
        / "images"
    ).mkdir(
        parents=True,
        exist_ok=True
    )


    (
        E5_DATASET_ROOT
        / split
        / "labels"
    ).mkdir(
        parents=True,
        exist_ok=True
    )


# ======================================================================
# 22. HARD-LINK HELPER
# ======================================================================
#
# Windows NTFS hard links:
#
#     Different filename
#     Same underlying image data
#
# Therefore:
#
#     NO full image copy
#     NO major additional SSD usage
#
# ======================================================================

def create_hardlink(source, destination):

    source = Path(
        source
    )


    destination = Path(
        destination
    )


    destination.parent.mkdir(
        parents=True,
        exist_ok=True
    )


    if destination.exists():

        destination.unlink()


    try:

        os.link(
            source,
            destination
        )


    except OSError as exc:

        raise RuntimeError(
            "\nCould not create NTFS hard link."
            "\n"
            "\nSource:"
            f"\n{source}"
            "\n"
            "\nDestination:"
            f"\n{destination}"
            "\n"
            "\nThis script deliberately does NOT fall back "
            "to copying images because that could consume "
            "significant SSD space."
            "\n"
            f"\nOriginal error: {exc}"
        )


# ======================================================================
# 23. REMAP A YOLO LABEL FILE
# ======================================================================

def remap_label_file(
    source_label,
    destination_label
):

    source_label = Path(
        source_label
    )


    destination_label = Path(
        destination_label
    )


    destination_label.parent.mkdir(
        parents=True,
        exist_ok=True
    )


    output_rows = []


    with open(
        source_label,
        "r",
        encoding="utf-8"
    ) as f:

        for line in f:

            line = line.strip()


            if not line:

                continue


            parts = line.split()


            old_class = int(
                float(parts[0])
            )


            new_class = CLASS_MAPPING[
                old_class
            ]


            parts[0] = str(
                new_class
            )


            output_rows.append(
                " ".join(parts)
            )


    with open(
        destination_label,
        "w",
        encoding="utf-8"
    ) as f:

        if output_rows:

            f.write(
                "\n".join(output_rows)
            )

            f.write(
                "\n"
            )


# ======================================================================
# 24. UNIQUE OUTPUT NAME
# ======================================================================
#
# A sequential prefix prevents accidental filename collisions.
#
# Example:
#
#     000001_originalname.jpg
#
# Oversampled aliases:
#
#     000001_originalname_os2.jpg
#     000001_originalname_os3.jpg
#
# ======================================================================

def make_base_filename(
    index,
    image_path
):

    image_path = Path(
        image_path
    )


    return (
        f"{index:06d}_"
        f"{image_path.stem}"
    )


# ======================================================================
# 25. BUILD VALIDATION / TEST SET
# ======================================================================
#
# No oversampling.
#
# ======================================================================

def build_eval_split(
    split_name,
    images
):

    print(
        f"\nBuilding {split_name}..."
    )


    for index, source_image in enumerate(
        images,
        start=1
    ):

        source_label = image_to_label_path(
            source_image
        )


        if not source_label.exists():

            raise RuntimeError(
                f"Missing label:\n{source_label}"
            )


        base_name = make_base_filename(
            index,
            source_image
        )


        destination_image = (
            E5_DATASET_ROOT
            / split_name
            / "images"
            / (
                base_name
                + source_image.suffix.lower()
            )
        )


        destination_label = (
            E5_DATASET_ROOT
            / split_name
            / "labels"
            / (
                base_name
                + ".txt"
            )
        )


        create_hardlink(
            source_image,
            destination_image
        )


        remap_label_file(
            source_label,
            destination_label
        )


    print(
        f"{split_name} created:",
        len(images),
        "images"
    )


# ======================================================================
# 26. BUILD TRAIN SET WITH CLASS-AWARE OVERSAMPLING
# ======================================================================

print("\n" + "=" * 90)
print("BUILDING E5 TRAIN SET")
print("=" * 90)


repeat_distribution = Counter()

effective_training_entries = 0

effective_object_counts = Counter()


for index, source_image in enumerate(
    train_images,
    start=1
):

    source_label = image_to_label_path(
        source_image
    )


    # --------------------------------------------------------------
    # Read classes appearing in this image
    # --------------------------------------------------------------

    new_classes_in_image = set()

    objects_in_image = Counter()


    with open(
        source_label,
        "r",
        encoding="utf-8"
    ) as f:

        for line in f:

            line = line.strip()


            if not line:

                continue


            parts = line.split()


            old_class = int(
                float(parts[0])
            )


            new_class = CLASS_MAPPING[
                old_class
            ]


            new_classes_in_image.add(
                new_class
            )


            objects_in_image[
                new_class
            ] += 1


    # --------------------------------------------------------------
    # Image receives maximum class repeat factor
    # --------------------------------------------------------------

    if new_classes_in_image:

        repeat = max(

            REPEAT_FACTOR[
                cls
            ]

            for cls
            in new_classes_in_image
        )

    else:

        repeat = 1


    repeat_distribution[
        repeat
    ] += 1


    base_name = make_base_filename(
        index,
        source_image
    )


    # --------------------------------------------------------------
    # Create 1x / 2x / 3x unique hard-linked examples
    # --------------------------------------------------------------

    for repetition in range(
        1,
        repeat + 1
    ):

        if repetition == 1:

            alias = base_name

        else:

            alias = (
                base_name
                + f"_os{repetition}"
            )


        destination_image = (
            E5_DATASET_ROOT
            / "train"
            / "images"
            / (
                alias
                + source_image.suffix.lower()
            )
        )


        destination_label = (
            E5_DATASET_ROOT
            / "train"
            / "labels"
            / (
                alias
                + ".txt"
            )
        )


        create_hardlink(
            source_image,
            destination_image
        )


        remap_label_file(
            source_label,
            destination_label
        )


        effective_training_entries += 1


        for cls, count in objects_in_image.items():

            effective_object_counts[
                cls
            ] += count


# ======================================================================
# 27. BUILD VAL + TEST
# ======================================================================

build_eval_split(
    "val",
    val_images
)


if test_images:

    build_eval_split(
        "test",
        test_images
    )


# ======================================================================
# 28. OVERSAMPLING SUMMARY
# ======================================================================

dataset_multiplier = (

    effective_training_entries
    / len(train_images)
)


print("\n" + "=" * 90)
print("E5 OVERSAMPLING SUMMARY")
print("=" * 90)


print(
    "Original training images  :",
    len(train_images)
)


print(
    "Effective training images :",
    effective_training_entries
)


print(
    "Dataset multiplier        :",
    f"{dataset_multiplier:.3f}x"
)


print(
    "\nImage repeat distribution:"
)


for repeat in sorted(
    repeat_distribution
):

    print(
        f"{repeat}x : "
        f"{repeat_distribution[repeat]} original images"
    )


# ======================================================================
# 29. EFFECTIVE OBJECT EXPOSURE AFTER OVERSAMPLING
# ======================================================================

print("\n" + "=" * 90)
print("EFFECTIVE OBJECT EXPOSURE AFTER OVERSAMPLING")
print("=" * 90)


for cls in range(7):

    original_count = mapped_train_counts[
        cls
    ]


    effective_count = effective_object_counts[
        cls
    ]


    exposure_ratio = (

        effective_count
        / original_count

        if original_count > 0

        else 0
    )


    print(
        f"{cls:<2d} "
        f"{E5_NAMES[cls]:25s} "
        f"original={original_count:7d} | "
        f"effective={effective_count:7d} | "
        f"exposure={exposure_ratio:.3f}x"
    )


# ======================================================================
# 30. OVERSAMPLING SAFETY GUARD
# ======================================================================

if dataset_multiplier > 1.8:

    raise RuntimeError(
        "\nE5 oversampling exceeded 1.8x."
        "\nThis is more aggressive than intended."
        "\nTraining has NOT started."
    )


# ======================================================================
# 31. CREATE E5 YOLO YAML
# ======================================================================

e5_yaml = {

    "path": str(
        E5_DATASET_ROOT.resolve()
    ),

    "train": "train/images",

    "val": "val/images",

    "nc": 7,

    "names": [

        "ecal",
        "hdpe",
        "mixed_plastic_rigid",
        "mixed_plastic_soft",
        "non_plastic",
        "pet",
        "pet_oil",
    ],
}


if test_images:

    e5_yaml[
        "test"
    ] = "test/images"


with open(
    E5_DATA_YAML,
    "w",
    encoding="utf-8"
) as f:

    yaml.safe_dump(
        e5_yaml,
        f,
        sort_keys=False
    )


print("\n" + "=" * 90)
print("E5 DATA YAML")
print("=" * 90)


print(
    E5_DATA_YAML
)


print()


for key, value in e5_yaml.items():

    print(
        f"{key}: {value}"
    )


# ======================================================================
# 32. VERIFY GENERATED DATASET COUNTS
# ======================================================================

generated_train_images = [

    p

    for p in (
        E5_DATASET_ROOT
        / "train"
        / "images"
    ).iterdir()

    if p.suffix.lower()
    in IMAGE_EXTENSIONS
]


generated_train_labels = list(

    (
        E5_DATASET_ROOT
        / "train"
        / "labels"
    ).glob(
        "*.txt"
    )
)


generated_val_images = [

    p

    for p in (
        E5_DATASET_ROOT
        / "val"
        / "images"
    ).iterdir()

    if p.suffix.lower()
    in IMAGE_EXTENSIONS
]


generated_val_labels = list(

    (
        E5_DATASET_ROOT
        / "val"
        / "labels"
    ).glob(
        "*.txt"
    )
)


print("\n" + "=" * 90)
print("GENERATED DATASET VERIFICATION")
print("=" * 90)


print(
    "Train images:",
    len(generated_train_images)
)

print(
    "Train labels:",
    len(generated_train_labels)
)

print(
    "Val images  :",
    len(generated_val_images)
)

print(
    "Val labels  :",
    len(generated_val_labels)
)


if (
    len(generated_train_images)
    != len(generated_train_labels)
):

    raise RuntimeError(
        "Generated train image/label counts do not match."
    )


if (
    len(generated_val_images)
    != len(generated_val_labels)
):

    raise RuntimeError(
        "Generated validation image/label counts do not match."
    )


# ======================================================================
# 33. CLEAR CUDA CACHE
# ======================================================================

if torch.cuda.is_available():

    torch.cuda.empty_cache()


# ======================================================================
# 34. LOAD PRETRAINED YOLO11s
# ======================================================================

print("\n" + "=" * 90)
print("LOADING YOLO11s")
print("=" * 90)


model = YOLO(
    "yolo11s.pt"
)


print(
    "YOLO11s loaded successfully."
)


# ======================================================================
# 35. DISPLAY FINAL E5 CONFIGURATION
# ======================================================================

print("\n" + "=" * 90)
print("FINAL E5 TRAINING CONFIGURATION")
print("=" * 90)


print(
    """
Experiment:
    E5

Model:
    YOLO11s

Taxonomy:
    7 classes

    ecal
    hdpe
    mixed_plastic_rigid
    mixed_plastic_soft
    non_plastic
    pet
    pet_oil

Non-plastic definition:
    original metal + cardboard

Input resolution:
    640 x 640

Optimizer:
    AdamW

Maximum epochs:
    150

Early stopping:
    patience = 15

Batch:
    4

Augmentation:
    Mosaic             = 0.80
    MixUp              = 0.05
    Rotation           = +/- 5 degrees
    Translation        = 0.10
    Scale              = 0.30
    Horizontal flip    = 0.50
    Mild HSV variation

Oversampling:
    non_plastic        = 2x exposure
    pet_oil            = 3x exposure

Other classes:
    no targeted oversampling

Final epochs:
    Mosaic disabled for final 10 epochs
"""
)


# ======================================================================
# 36. TRAIN E5
# ======================================================================

train_results = model.train(

    # Dataset
    data=str(
        E5_DATA_YAML
    ),

    # Resolution
    imgsz=640,

    # Training length
    epochs=150,

    patience=15,

    # Hardware
    batch=4,

    device=DEVICE,

    workers=0,

    cache=False,

    # Optimization
    optimizer="AdamW",

    cos_lr=True,

    amp=True,

    pretrained=True,

    # --------------------------------------------------------------
    # Realistic augmentation
    # --------------------------------------------------------------

    mosaic=0.80,

    mixup=0.05,

    degrees=5.0,

    translate=0.10,

    scale=0.30,

    flipud=0.0,

    fliplr=0.50,

    hsv_h=0.015,

    hsv_s=0.40,

    hsv_v=0.30,

    # Avoid unrealistic industrial-waste distortions
    shear=0.0,

    perspective=0.0,

    # Disable mosaic toward end
    close_mosaic=10,

    # Reproducibility
    seed=SEED,

    deterministic=True,

    # Validation during training
    val=True,

    # Outputs
    project=str(
        PROJECT_ROOT
    ),

    name=EXPERIMENT_NAME,

    exist_ok=False,

    plots=True,

    verbose=True,
)


# ======================================================================
# 37. LOCATE BEST CHECKPOINT
# ======================================================================

RUN_DIR = (
    PROJECT_ROOT
    / EXPERIMENT_NAME
)


BEST_MODEL = (
    RUN_DIR
    / "weights"
    / "best.pt"
)


LAST_MODEL = (
    RUN_DIR
    / "weights"
    / "last.pt"
)


print("\n" + "=" * 90)
print("E5 TRAINING COMPLETE")
print("=" * 90)


print(
    "Best checkpoint:"
)

print(
    BEST_MODEL
)


print(
    "Exists:",
    BEST_MODEL.exists()
)


print(
    "\nLast checkpoint:"
)

print(
    LAST_MODEL
)


if not BEST_MODEL.exists():

    raise FileNotFoundError(
        f"\nBest E5 checkpoint not found:\n{BEST_MODEL}"
    )


# ======================================================================
# 38. FINAL VALIDATION
# ======================================================================
#
# IMPORTANT:
#
# We evaluate on the ORIGINAL validation images,
# but with E5's corresponding REMAPPED 7-class labels.
#
# There is NO validation oversampling.
#
# ======================================================================

print("\n" + "=" * 90)
print("E5 FINAL 7-CLASS VALIDATION")
print("=" * 90)


best_model = YOLO(
    str(BEST_MODEL)
)


val_results = best_model.val(

    data=str(
        E5_DATA_YAML
    ),

    split="val",

    imgsz=640,

    batch=4,

    device=DEVICE,

    workers=0,

    # Low confidence floor for AP computation
    conf=0.001,

    # Starting NMS setting.
    # E6 will tune this separately.
    iou=0.70,

    max_det=100,

    plots=True,

    project=str(
        PROJECT_ROOT
    ),

    name=(
        EXPERIMENT_NAME
        + "_final_val"
    ),

    exist_ok=True,
)


# ======================================================================
# 39. OVERALL RESULTS
# ======================================================================

print("\n" + "=" * 90)
print("E5 OVERALL RESULTS — 7 CLASS")
print("=" * 90)


print(
    f"mAP50-95 : "
    f"{val_results.box.map:.6f} "
    f"= {val_results.box.map * 100:.2f}%"
)


print(
    f"mAP50    : "
    f"{val_results.box.map50:.6f} "
    f"= {val_results.box.map50 * 100:.2f}%"
)


print(
    f"mAP75    : "
    f"{val_results.box.map75:.6f} "
    f"= {val_results.box.map75 * 100:.2f}%"
)


# ======================================================================
# 40. PER-CLASS RESULTS
# ======================================================================

print("\n" + "=" * 90)
print("E5 PER-CLASS RESULTS")
print("=" * 90)


class_names = best_model.names


print(
    f"{'Class':25s}"
    f"{'Precision':>12s}"
    f"{'Recall':>12s}"
    f"{'AP50':>12s}"
    f"{'AP50-95':>12s}"
)


print(
    "-" * 73
)


for class_id in range(
    len(class_names)
):

    result = val_results.box.class_result(
        class_id
    )


    precision = result[0]

    recall = result[1]

    ap50 = result[2]

    ap5095 = result[3]


    print(
        f"{class_names[class_id]:25s}"
        f"{precision:12.4f}"
        f"{recall:12.4f}"
        f"{ap50:12.4f}"
        f"{ap5095:12.4f}"
    )


# ======================================================================
# 41. SAVE E5 SUMMARY
# ======================================================================

SUMMARY_FILE = (
    RUN_DIR
    / "E5_7class_final_summary.txt"
)


with open(
    SUMMARY_FILE,
    "w",
    encoding="utf-8"
) as f:

    f.write(
        "E5 — YOLO11s + 7-class taxonomy + realistic augmentation "
        "+ class-aware oversampling\n"
    )

    f.write(
        "=" * 80
        + "\n\n"
    )


    f.write(
        "Taxonomy:\n"
    )

    for cls in range(7):

        f.write(
            f"{cls}: {E5_NAMES[cls]}\n"
        )


    f.write(
        "\nmetal + cardboard -> non_plastic\n\n"
    )


    f.write(
        f"Original training images  = {len(train_images)}\n"
    )

    f.write(
        f"Effective training images = {effective_training_entries}\n"
    )

    f.write(
        f"Dataset multiplier        = {dataset_multiplier:.3f}x\n\n"
    )


    f.write(
        f"mAP50-95 = {val_results.box.map:.6f}\n"
    )

    f.write(
        f"mAP50    = {val_results.box.map50:.6f}\n"
    )

    f.write(
        f"mAP75    = {val_results.box.map75:.6f}\n\n"
    )


    f.write(
        "Per-class results\n"
    )

    f.write(
        "-" * 80
        + "\n"
    )


    for class_id in range(
        len(class_names)
    ):

        result = val_results.box.class_result(
            class_id
        )


        f.write(
            f"{class_names[class_id]} | "
            f"P={result[0]:.6f} | "
            f"R={result[1]:.6f} | "
            f"AP50={result[2]:.6f} | "
            f"AP50-95={result[3]:.6f}\n"
        )


print("\n" + "=" * 90)
print("E5 COMPLETE")
print("=" * 90)


print(
    "\nSummary:"
)

print(
    SUMMARY_FILE
)


print(
    "\nBest model:"
)

print(
    BEST_MODEL
)


print(
    "\nNext experiment:"
)

print(
    "E6 = NMS IoU tuning using E5 best.pt."
)

E5 — YOLO11s / 7 CLASS / AUGMENTATION / CLASS BALANCING
Python          : 3.11.15
PyTorch         : 2.13.0+cu126
CUDA available  : True
Ultralytics    : 8.4.126
GPU             : NVIDIA GeForce RTX 3050 Ti Laptop GPU
GPU memory      : 4.00 GB

SOURCE DATASET CHECK
Original YAML:
C:\Users\varda\Documents\_My Computer\COMMON_space\Learning\Upgrad_EPGP_MS\upgrad_IIITB_MS_course\MS_LJMU_Material\Topic Data\SortWaste\dataset\dataset\splited_all_dataset_coco\data_pc.yaml
Exists: True

Original YAML contents:
train: C:\Users\varda\Documents\_My Computer\COMMON_space\Learning\Upgrad_EPGP_MS\upgrad_IIITB_MS_course\MS_LJMU_Material\Topic Data\SortWaste\dataset\dataset\splited_all_dataset_coco\train\images
val: C:\Users\varda\Documents\_My Computer\COMMON_space\Learning\Upgrad_EPGP_MS\upgrad_IIITB_MS_course\MS_LJMU_Material\Topic Data\SortWaste\dataset\dataset\splited_all_dataset_coco\val\images
test: C:\Users\varda\Documents\_My Computer\COMMON_space\Learning\Upgrad_EPGP_MS\upgrad_IIITB_MS_cours

# Model E6 — YOLO11s + 7-class taxonomy + realistic augmentation @640, no oversampling

In [ ]:
## Training and Validation
# ======================================================================
# E6 — YOLO11s + 7-Class Taxonomy + Realistic Augmentation @640
#      NO CLASS-AWARE OVERSAMPLING
#
# LJMU Thesis — SortWaste
# ======================================================================
#
# E6 purpose:
#     Ablation of E5 to isolate the effect of class-aware oversampling.
#
# E5:
#     YOLO11s + 7-class + augmentation + oversampling
#
# E6:
#     YOLO11s + 7-class + augmentation ONLY
#
# IMPORTANT:
#     - Original SortWaste dataset is NEVER modified.
#     - Metal + Cardboard -> non_plastic
#     - Training uses each original image ONCE.
#     - Validation/test are unchanged apart from taxonomy remapping.
# ======================================================================


import os
import sys
import random
import shutil
from pathlib import Path
from collections import Counter

import torch
import yaml

from ultralytics import YOLO


# ======================================================================
# 1. GLOBAL SETTINGS
# ======================================================================

SEED = 42

random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)


# ======================================================================
# 2. ORIGINAL SORTWASTE YAML
# ======================================================================

DATA_YAML = Path(
    r"C:\Users\varda\Documents\_My Computer\COMMON_space\Learning"
    r"\Upgrad_EPGP_MS\upgrad_IIITB_MS_course\MS_LJMU_Material"
    r"\Topic Data\SortWaste\dataset\dataset"
    r"\splited_all_dataset_coco\data_pc.yaml"
)


# ======================================================================
# 3. OUTPUT ROOT
# ======================================================================

PROJECT_ROOT = Path(
    r"C:\Users\varda\Documents\_My Computer\COMMON_space\Learning"
    r"\Upgrad_EPGP_MS\upgrad_IIITB_MS_course"
    r"\MS_LJMU_Material\Thesis_Code"
    r"\runs\sortwaste"
)


EXPERIMENT_NAME = "E6_yolo11s_7class_aug_no_oversampling_640"


# ======================================================================
# 4. E6 DERIVED DATASET
# ======================================================================

E6_DATASET_ROOT = (
    PROJECT_ROOT
    / "E6_7class_dataset"
)


E6_DATA_YAML = (
    E6_DATASET_ROOT
    / "E6_7class.yaml"
)


# ======================================================================
# 5. ORIGINAL SORTWASTE 8-CLASS TAXONOMY
# ======================================================================

ORIGINAL_NAMES = {
    0: "pet",
    1: "hdpe",
    2: "mixed_plastic_soft",
    3: "ecal",
    4: "metal",
    5: "cardboard",
    6: "mixed_plastic_rigid",
    7: "pet_oil",
}


# ======================================================================
# 6. E6 7-CLASS TAXONOMY
# ======================================================================

E6_NAMES = {
    0: "ecal",
    1: "hdpe",
    2: "mixed_plastic_rigid",
    3: "mixed_plastic_soft",
    4: "non_plastic",
    5: "pet",
    6: "pet_oil",
}


# ======================================================================
# 7. ORIGINAL -> E6 CLASS MAPPING
# ======================================================================

CLASS_MAPPING = {
    0: 5,   # pet -> pet
    1: 1,   # hdpe -> hdpe
    2: 3,   # mixed soft -> mixed soft
    3: 0,   # ecal -> ecal
    4: 4,   # metal -> non_plastic
    5: 4,   # cardboard -> non_plastic
    6: 2,   # mixed rigid -> mixed rigid
    7: 6,   # pet oil -> pet oil
}


IMAGE_EXTENSIONS = {
    ".jpg",
    ".jpeg",
    ".png",
    ".bmp",
    ".tif",
    ".tiff",
    ".webp",
}


# ======================================================================
# 8. ENVIRONMENT
# ======================================================================

print("=" * 90)
print("E6 — YOLO11s / 7 CLASS / AUGMENTATION / NO OVERSAMPLING")
print("=" * 90)

print("Python          :", sys.version.split()[0])
print("PyTorch         :", torch.__version__)
print("CUDA available  :", torch.cuda.is_available())

try:
    import ultralytics
    print("Ultralytics     :", ultralytics.__version__)
except Exception:
    pass


if torch.cuda.is_available():

    DEVICE = 0

    print(
        "GPU             :",
        torch.cuda.get_device_name(0)
    )

    gpu_memory = (
        torch.cuda.get_device_properties(0).total_memory
        / 1024 ** 3
    )

    print(
        "GPU memory      :",
        f"{gpu_memory:.2f} GB"
    )

else:

    DEVICE = "cpu"

    print("\nWARNING: CUDA unavailable.")


# ======================================================================
# 9. VERIFY ORIGINAL YAML
# ======================================================================

print("\n" + "=" * 90)
print("SOURCE DATASET CHECK")
print("=" * 90)

print("Original YAML:")
print(DATA_YAML)

print("Exists:", DATA_YAML.exists())


if not DATA_YAML.exists():

    raise FileNotFoundError(
        f"\nCould not find:\n{DATA_YAML}"
    )


with open(
    DATA_YAML,
    "r",
    encoding="utf-8"
) as f:

    original_cfg = yaml.safe_load(f)


print("\nOriginal YAML contents:")

for key, value in original_cfg.items():
    print(f"{key}: {value}")


# ======================================================================
# 10. RESOLVE DATASET ROOT
# ======================================================================

yaml_parent = DATA_YAML.parent

yaml_root = original_cfg.get(
    "path",
    None
)


if yaml_root is None:

    SOURCE_DATASET_ROOT = (
        yaml_parent.resolve()
    )

else:

    root_path = Path(
        str(yaml_root)
    )

    if root_path.is_absolute():

        SOURCE_DATASET_ROOT = (
            root_path.resolve()
        )

    else:

        SOURCE_DATASET_ROOT = (
            yaml_parent
            / root_path
        ).resolve()


print("\nResolved source dataset root:")
print(SOURCE_DATASET_ROOT)


# ======================================================================
# 11. SPLIT RESOLUTION
# ======================================================================

def resolve_split(entry):

    if entry is None:
        return None

    p = Path(
        str(entry)
    )

    if p.is_absolute():
        return p.resolve()

    candidate = (
        SOURCE_DATASET_ROOT
        / p
    ).resolve()

    if candidate.exists():
        return candidate

    return (
        yaml_parent
        / p
    ).resolve()


TRAIN_SOURCE = resolve_split(
    original_cfg.get("train")
)

VAL_SOURCE = resolve_split(
    original_cfg.get("val")
)

TEST_SOURCE = resolve_split(
    original_cfg.get("test")
)


print("\nResolved splits:")
print("TRAIN :", TRAIN_SOURCE)
print("VAL   :", VAL_SOURCE)
print("TEST  :", TEST_SOURCE)


if TRAIN_SOURCE is None:
    raise RuntimeError("Train split not found.")

if VAL_SOURCE is None:
    raise RuntimeError("Validation split not found.")


# ======================================================================
# 12. LOAD IMAGE PATHS
# ======================================================================

def load_image_paths(source):

    source = Path(source)

    if source.is_dir():

        return sorted(
            p.resolve()
            for p in source.rglob("*")
            if p.is_file()
            and p.suffix.lower() in IMAGE_EXTENSIONS
        )

    if source.is_file():

        image_paths = []

        with open(
            source,
            "r",
            encoding="utf-8"
        ) as f:

            for line in f:

                line = line.strip()

                if not line:
                    continue

                p = Path(line)

                if not p.is_absolute():
                    p = (
                        source.parent
                        / p
                    ).resolve()

                image_paths.append(p)

        return image_paths

    raise FileNotFoundError(
        f"Source does not exist:\n{source}"
    )


train_images = load_image_paths(
    TRAIN_SOURCE
)

val_images = load_image_paths(
    VAL_SOURCE
)


if TEST_SOURCE is not None:
    test_images = load_image_paths(
        TEST_SOURCE
    )
else:
    test_images = []


print("\nImage counts:")
print("Train:", len(train_images))
print("Val  :", len(val_images))
print("Test :", len(test_images))


# ======================================================================
# 13. IMAGE -> LABEL PATH
# ======================================================================

def image_to_label_path(image_path):

    image_path = Path(
        image_path
    )

    parts = list(
        image_path.parts
    )

    indexes = [
        i
        for i, part in enumerate(parts)
        if part.lower() == "images"
    ]

    if not indexes:

        raise RuntimeError(
            f"Could not infer label path from:\n{image_path}"
        )

    idx = indexes[-1]

    parts[idx] = "labels"

    return Path(
        *parts
    ).with_suffix(
        ".txt"
    )


# ======================================================================
# 14. VERIFY ORIGINAL COUNTS + 7-CLASS REMAPPING
# ======================================================================

original_counts = Counter()
mapped_counts = Counter()

missing_labels = []


for image_path in train_images:

    label_path = image_to_label_path(
        image_path
    )

    if not label_path.exists():

        missing_labels.append(
            label_path
        )

        continue

    with open(
        label_path,
        "r",
        encoding="utf-8"
    ) as f:

        for line in f:

            line = line.strip()

            if not line:
                continue

            parts = line.split()

            old_class = int(
                float(parts[0])
            )

            if old_class not in CLASS_MAPPING:

                raise RuntimeError(
                    f"Unexpected class {old_class}"
                    f"\nin {label_path}"
                )

            new_class = CLASS_MAPPING[
                old_class
            ]

            original_counts[
                old_class
            ] += 1

            mapped_counts[
                new_class
            ] += 1


print("\n" + "=" * 90)
print("ORIGINAL 8-CLASS TRAIN DISTRIBUTION")
print("=" * 90)

for cls in range(8):

    print(
        f"{cls:<2d} "
        f"{ORIGINAL_NAMES[cls]:25s} "
        f"{original_counts[cls]:8d}"
    )


print(
    "\nMissing labels:",
    len(missing_labels)
)


if missing_labels:

    raise RuntimeError(
        "Missing YOLO labels detected."
    )


print("\n" + "=" * 90)
print("E6 7-CLASS TRAIN DISTRIBUTION")
print("=" * 90)

for cls in range(7):

    print(
        f"{cls:<2d} "
        f"{E6_NAMES[cls]:25s} "
        f"{mapped_counts[cls]:8d}"
    )


print(
    "\nExpected non_plastic:"
)

print(
    "945 metal + 1524 cardboard = 2469"
)


# ======================================================================
# 15. CLEAN / REBUILD E6 DATASET
# ======================================================================

REBUILD_E6_DATASET = True


if (
    REBUILD_E6_DATASET
    and E6_DATASET_ROOT.exists()
):

    print(
        "\nRemoving previous E6 generated dataset..."
    )

    shutil.rmtree(
        E6_DATASET_ROOT
    )


# ======================================================================
# 16. CREATE DIRECTORY STRUCTURE
# ======================================================================

for split in [
    "train",
    "val",
    "test"
]:

    (
        E6_DATASET_ROOT
        / split
        / "images"
    ).mkdir(
        parents=True,
        exist_ok=True
    )

    (
        E6_DATASET_ROOT
        / split
        / "labels"
    ).mkdir(
        parents=True,
        exist_ok=True
    )


# ======================================================================
# 17. HARD LINK HELPER
# ======================================================================

def create_hardlink(
    source,
    destination
):

    source = Path(source)
    destination = Path(destination)

    destination.parent.mkdir(
        parents=True,
        exist_ok=True
    )

    if destination.exists():
        destination.unlink()

    try:

        os.link(
            source,
            destination
        )

    except OSError as exc:

        raise RuntimeError(
            "\nCould not create hard link."
            "\nSource:"
            f"\n{source}"
            "\nDestination:"
            f"\n{destination}"
            f"\nError: {exc}"
        )


# ======================================================================
# 18. LABEL REMAPPING
# ======================================================================

def remap_label_file(
    source_label,
    destination_label
):

    output_rows = []

    with open(
        source_label,
        "r",
        encoding="utf-8"
    ) as f:

        for line in f:

            line = line.strip()

            if not line:
                continue

            parts = line.split()

            old_class = int(
                float(parts[0])
            )

            new_class = CLASS_MAPPING[
                old_class
            ]

            parts[0] = str(
                new_class
            )

            output_rows.append(
                " ".join(parts)
            )

    with open(
        destination_label,
        "w",
        encoding="utf-8"
    ) as f:

        if output_rows:

            f.write(
                "\n".join(output_rows)
            )

            f.write("\n")


# ======================================================================
# 19. UNIQUE OUTPUT NAME
# ======================================================================

def make_base_filename(
    index,
    image_path
):

    image_path = Path(
        image_path
    )

    return (
        f"{index:06d}_"
        f"{image_path.stem}"
    )


# ======================================================================
# 20. BUILD SPLIT — NO OVERSAMPLING
# ======================================================================

def build_split(
    split_name,
    images
):

    print(
        f"\nBuilding {split_name}..."
    )

    for index, source_image in enumerate(
        images,
        start=1
    ):

        source_label = image_to_label_path(
            source_image
        )

        if not source_label.exists():

            raise RuntimeError(
                f"Missing label:\n{source_label}"
            )

        base_name = make_base_filename(
            index,
            source_image
        )

        destination_image = (
            E6_DATASET_ROOT
            / split_name
            / "images"
            / (
                base_name
                + source_image.suffix.lower()
            )
        )

        destination_label = (
            E6_DATASET_ROOT
            / split_name
            / "labels"
            / (
                base_name
                + ".txt"
            )
        )

        # Every image appears EXACTLY ONCE
        create_hardlink(
            source_image,
            destination_image
        )

        remap_label_file(
            source_label,
            destination_label
        )

    print(
        f"{split_name} created:",
        len(images),
        "images"
    )


# ======================================================================
# 21. BUILD E6 DATASET
# ======================================================================

print("\n" + "=" * 90)
print("BUILDING E6 7-CLASS DATASET — NO OVERSAMPLING")
print("=" * 90)


build_split(
    "train",
    train_images
)

build_split(
    "val",
    val_images
)


if test_images:

    build_split(
        "test",
        test_images
    )


# ======================================================================
# 22. CREATE E6 YAML
# ======================================================================

e6_yaml = {

    "path": str(
        E6_DATASET_ROOT.resolve()
    ),

    "train": "train/images",

    "val": "val/images",

    "nc": 7,

    "names": [
        "ecal",
        "hdpe",
        "mixed_plastic_rigid",
        "mixed_plastic_soft",
        "non_plastic",
        "pet",
        "pet_oil",
    ],
}


if test_images:
    e6_yaml["test"] = "test/images"


with open(
    E6_DATA_YAML,
    "w",
    encoding="utf-8"
) as f:

    yaml.safe_dump(
        e6_yaml,
        f,
        sort_keys=False
    )


print("\n" + "=" * 90)
print("E6 DATA YAML")
print("=" * 90)

print(E6_DATA_YAML)

for key, value in e6_yaml.items():
    print(f"{key}: {value}")


# ======================================================================
# 23. VERIFY GENERATED DATASET
# ======================================================================

generated_train_images = [
    p
    for p in (
        E6_DATASET_ROOT
        / "train"
        / "images"
    ).iterdir()
    if p.suffix.lower() in IMAGE_EXTENSIONS
]


generated_train_labels = list(
    (
        E6_DATASET_ROOT
        / "train"
        / "labels"
    ).glob("*.txt")
)


generated_val_images = [
    p
    for p in (
        E6_DATASET_ROOT
        / "val"
        / "images"
    ).iterdir()
    if p.suffix.lower() in IMAGE_EXTENSIONS
]


generated_val_labels = list(
    (
        E6_DATASET_ROOT
        / "val"
        / "labels"
    ).glob("*.txt")
)


print("\n" + "=" * 90)
print("GENERATED DATASET VERIFICATION")
print("=" * 90)

print(
    "Train images:",
    len(generated_train_images)
)

print(
    "Train labels:",
    len(generated_train_labels)
)

print(
    "Val images  :",
    len(generated_val_images)
)

print(
    "Val labels  :",
    len(generated_val_labels)
)


if len(generated_train_images) != len(train_images):

    raise RuntimeError(
        "\nE6 training image count is incorrect."
        "\nExpected exactly one training entry per original image."
    )


if len(generated_train_images) != len(generated_train_labels):

    raise RuntimeError(
        "Train image/label counts do not match."
    )


if len(generated_val_images) != len(generated_val_labels):

    raise RuntimeError(
        "Validation image/label counts do not match."
    )


print(
    "\nE6 CHECK PASSED:"
)

print(
    f"Original train images = {len(train_images)}"
)

print(
    f"E6 train images       = {len(generated_train_images)}"
)

print(
    "Dataset multiplier    = 1.000x"
)


# ======================================================================
# 24. CLEAR CUDA CACHE
# ======================================================================

if torch.cuda.is_available():
    torch.cuda.empty_cache()


# ======================================================================
# 25. LOAD YOLO11s
# ======================================================================

print("\n" + "=" * 90)
print("LOADING YOLO11s")
print("=" * 90)


model = YOLO(
    "yolo11s.pt"
)


print(
    "YOLO11s loaded successfully."
)


# ======================================================================
# 26. FINAL E6 CONFIG
# ======================================================================

print("\n" + "=" * 90)
print("FINAL E6 TRAINING CONFIGURATION")
print("=" * 90)


print(
    """
Experiment:
    E6

Model:
    YOLO11s

Taxonomy:
    7 classes

    ecal
    hdpe
    mixed_plastic_rigid
    mixed_plastic_soft
    non_plastic
    pet
    pet_oil

Metal + Cardboard:
    merged into non_plastic

Input:
    640 x 640

Training images:
    original 3705 images

Oversampling:
    NONE

Optimizer:
    AdamW

Maximum epochs:
    150

Early stopping:
    patience = 15

Batch:
    4

Augmentation:
    Mosaic          = 0.80
    MixUp           = 0.05
    Rotation        = +/- 5 degrees
    Translation     = 0.10
    Scale           = 0.30
    Horizontal flip = 0.50
    HSV             = mild

Final 10 epochs:
    mosaic disabled
"""
)


# ======================================================================
# 27. TRAIN E6
# ======================================================================

train_results = model.train(

    data=str(
        E6_DATA_YAML
    ),

    imgsz=640,

    epochs=150,

    patience=15,

    batch=4,

    device=DEVICE,

    workers=0,

    cache=False,

    optimizer="AdamW",

    cos_lr=True,

    amp=True,

    pretrained=True,

    # --------------------------------------------------------------
    # SAME REALISTIC AUGMENTATION AS E5
    # --------------------------------------------------------------

    mosaic=0.80,

    mixup=0.05,

    degrees=5.0,

    translate=0.10,

    scale=0.30,

    flipud=0.0,

    fliplr=0.50,

    hsv_h=0.015,

    hsv_s=0.40,

    hsv_v=0.30,

    shear=0.0,

    perspective=0.0,

    close_mosaic=10,

    # --------------------------------------------------------------
    # Reproducibility
    # --------------------------------------------------------------

    seed=SEED,

    deterministic=True,

    val=True,

    project=str(
        PROJECT_ROOT
    ),

    name=EXPERIMENT_NAME,

    exist_ok=False,

    plots=True,

    verbose=True,
)


# ======================================================================
# 28. LOCATE BEST CHECKPOINT
# ======================================================================

RUN_DIR = (
    PROJECT_ROOT
    / EXPERIMENT_NAME
)


BEST_MODEL = (
    RUN_DIR
    / "weights"
    / "best.pt"
)


LAST_MODEL = (
    RUN_DIR
    / "weights"
    / "last.pt"
)


print("\n" + "=" * 90)
print("E6 TRAINING COMPLETE")
print("=" * 90)


print("Best checkpoint:")
print(BEST_MODEL)

print("Exists:", BEST_MODEL.exists())


print("\nLast checkpoint:")
print(LAST_MODEL)


if not BEST_MODEL.exists():

    raise FileNotFoundError(
        f"\nBest checkpoint missing:\n{BEST_MODEL}"
    )


# ======================================================================
# 29. STANDARDIZED FINAL VALIDATION
# ======================================================================
#
# Must stay identical to E5 final validation.
#
# ======================================================================

print("\n" + "=" * 90)
print("E6 FINAL 7-CLASS VALIDATION")
print("=" * 90)


best_model = YOLO(
    str(BEST_MODEL)
)


val_results = best_model.val(

    data=str(
        E6_DATA_YAML
    ),

    split="val",

    imgsz=640,

    batch=4,

    device=DEVICE,

    workers=0,

    conf=0.001,

    iou=0.70,

    max_det=100,

    plots=True,

    project=str(
        PROJECT_ROOT
    ),

    name=(
        EXPERIMENT_NAME
        + "_final_val"
    ),

    exist_ok=True,
)


# ======================================================================
# 30. OVERALL RESULTS
# ======================================================================

print("\n" + "=" * 90)
print("E6 OVERALL RESULTS — 7 CLASS")
print("=" * 90)


print(
    f"mAP50-95 : "
    f"{val_results.box.map:.6f} "
    f"= {val_results.box.map * 100:.2f}%"
)


print(
    f"mAP50    : "
    f"{val_results.box.map50:.6f} "
    f"= {val_results.box.map50 * 100:.2f}%"
)


print(
    f"mAP75    : "
    f"{val_results.box.map75:.6f} "
    f"= {val_results.box.map75 * 100:.2f}%"
)


# ======================================================================
# 31. PER-CLASS RESULTS
# ======================================================================

print("\n" + "=" * 90)
print("E6 PER-CLASS RESULTS")
print("=" * 90)


class_names = best_model.names


print(
    f"{'Class':25s}"
    f"{'Precision':>12s}"
    f"{'Recall':>12s}"
    f"{'AP50':>12s}"
    f"{'AP50-95':>12s}"
)


print("-" * 73)


for class_id in range(
    len(class_names)
):

    result = val_results.box.class_result(
        class_id
    )

    precision = result[0]
    recall = result[1]
    ap50 = result[2]
    ap5095 = result[3]

    print(
        f"{class_names[class_id]:25s}"
        f"{precision:12.4f}"
        f"{recall:12.4f}"
        f"{ap50:12.4f}"
        f"{ap5095:12.4f}"
    )


# ======================================================================
# 32. SAVE SUMMARY
# ======================================================================

SUMMARY_FILE = (
    RUN_DIR
    / "E6_7class_final_summary.txt"
)


with open(
    SUMMARY_FILE,
    "w",
    encoding="utf-8"
) as f:

    f.write(
        "E6 — YOLO11s + 7-class taxonomy + realistic augmentation "
        "+ NO oversampling\n"
    )

    f.write(
        "=" * 80
        + "\n\n"
    )

    f.write(
        "Purpose: E5 oversampling ablation\n\n"
    )

    f.write(
        "metal + cardboard -> non_plastic\n"
    )

    f.write(
        "Oversampling = NONE\n\n"
    )

    f.write(
        f"Training images = {len(train_images)}\n"
    )

    f.write(
        "Dataset multiplier = 1.000x\n\n"
    )

    f.write(
        f"mAP50-95 = {val_results.box.map:.6f}\n"
    )

    f.write(
        f"mAP50    = {val_results.box.map50:.6f}\n"
    )

    f.write(
        f"mAP75    = {val_results.box.map75:.6f}\n\n"
    )

    f.write(
        "Per-class results\n"
    )

    f.write(
        "-" * 80
        + "\n"
    )

    for class_id in range(
        len(class_names)
    ):

        result = val_results.box.class_result(
            class_id
        )

        f.write(
            f"{class_names[class_id]} | "
            f"P={result[0]:.6f} | "
            f"R={result[1]:.6f} | "
            f"AP50={result[2]:.6f} | "
            f"AP50-95={result[3]:.6f}\n"
        )


print("\n" + "=" * 90)
print("E6 COMPLETE")
print("=" * 90)


print("\nSummary:")
print(SUMMARY_FILE)

print("\nBest model:")
print(BEST_MODEL)

print("\nNext step:")
print(
    "Compare E6 directly with E5. "
    "Then perform NMS tuning on whichever checkpoint performs better."
)

E6 — YOLO11s / 7 CLASS / AUGMENTATION / NO OVERSAMPLING
Python          : 3.11.15
PyTorch         : 2.13.0+cu126
CUDA available  : True
Ultralytics     : 8.4.126
GPU             : NVIDIA GeForce RTX 3050 Ti Laptop GPU
GPU memory      : 4.00 GB

SOURCE DATASET CHECK
Original YAML:
C:\Users\varda\Documents\_My Computer\COMMON_space\Learning\Upgrad_EPGP_MS\upgrad_IIITB_MS_course\MS_LJMU_Material\Topic Data\SortWaste\dataset\dataset\splited_all_dataset_coco\data_pc.yaml
Exists: True

Original YAML contents:
train: C:\Users\varda\Documents\_My Computer\COMMON_space\Learning\Upgrad_EPGP_MS\upgrad_IIITB_MS_course\MS_LJMU_Material\Topic Data\SortWaste\dataset\dataset\splited_all_dataset_coco\train\images
val: C:\Users\varda\Documents\_My Computer\COMMON_space\Learning\Upgrad_EPGP_MS\upgrad_IIITB_MS_course\MS_LJMU_Material\Topic Data\SortWaste\dataset\dataset\splited_all_dataset_coco\val\images
test: C:\Users\varda\Documents\_My Computer\COMMON_space\Learning\Upgrad_EPGP_MS\upgrad_IIITB_MS_cour

# Model E7 - NMS IoU tuning on E5 best.pt

In [ ]:
# E7 — NMS IoU Tuning on Best E5 Detector
#
# LJMU Thesis — SortWaste
#
# Purpose:
#     Tune inference-time NMS IoU threshold using the best E5 checkpoint.
#
# IMPORTANT:
#     - NO retraining
#     - Same E5 model
#     - Same 7-class dataset
#     - Same validation split
#     - Same confidence threshold
#     - Same max_det
#     - ONLY NMS IoU changes
# ======================================================================

from pathlib import Path
import csv

import torch
from ultralytics import YOLO


# ======================================================================
# 1. PATHS
# ======================================================================

PROJECT_ROOT = Path(
    r"C:\Users\varda\Documents\_My Computer\COMMON_space\Learning"
    r"\Upgrad_EPGP_MS\upgrad_IIITB_MS_course"
    r"\MS_LJMU_Material\Thesis_Code"
    r"\runs\sortwaste"
)

E5_RUN_DIR = (
    PROJECT_ROOT
    / "E5_yolo11s_7class_aug_classbalance_640"
)

BEST_MODEL = (
    E5_RUN_DIR
    / "weights"
    / "best.pt"
)

E5_DATA_YAML = (
    PROJECT_ROOT
    / "E5_7class_dataset"
    / "E5_7class.yaml"
)

E7_OUTPUT_ROOT = (
    PROJECT_ROOT
    / "E7_nms_tuning_E5"
)

E7_OUTPUT_ROOT.mkdir(
    parents=True,
    exist_ok=True
)


# ======================================================================
# 2. SETTINGS
# ======================================================================

NMS_IOU_VALUES = [
    0.50,
    0.60,
    0.70,
    0.80,
]

CONF_THRESHOLD = 0.001
MAX_DET = 100
IMG_SIZE = 640
BATCH_SIZE = 4

DEVICE = 0 if torch.cuda.is_available() else "cpu"


# ======================================================================
# 3. BASIC CHECKS
# ======================================================================

print("=" * 90)
print("E7 — NMS IoU TUNING ON E5")
print("=" * 90)

print("PyTorch        :", torch.__version__)
print("CUDA available :", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU            :", torch.cuda.get_device_name(0))

print("\nE5 checkpoint:")
print(BEST_MODEL)
print("Exists:", BEST_MODEL.exists())

print("\nE5 dataset YAML:")
print(E5_DATA_YAML)
print("Exists:", E5_DATA_YAML.exists())


if not BEST_MODEL.exists():
    raise FileNotFoundError(
        f"E5 best checkpoint not found:\n{BEST_MODEL}"
    )

if not E5_DATA_YAML.exists():
    raise FileNotFoundError(
        f"E5 dataset YAML not found:\n{E5_DATA_YAML}"
    )


# ======================================================================
# 4. LOAD MODEL
# ======================================================================

print("\n" + "=" * 90)
print("LOADING E5 BEST MODEL")
print("=" * 90)

model = YOLO(
    str(BEST_MODEL)
)

print("Model loaded successfully.")


# ======================================================================
# 5. RUN E7 NMS SWEEP
# ======================================================================

results_summary = []


for nms_iou in NMS_IOU_VALUES:

    print("\n" + "=" * 90)
    print(
        f"E7 VALIDATION — NMS IoU = {nms_iou:.2f}"
    )
    print("=" * 90)

    run_name = (
        f"E7_E5_nms_iou_{nms_iou:.2f}"
        .replace(".", "_")
    )

    val_results = model.val(

        data=str(
            E5_DATA_YAML
        ),

        split="val",

        imgsz=IMG_SIZE,

        batch=BATCH_SIZE,

        device=DEVICE,

        workers=0,

        conf=CONF_THRESHOLD,

        iou=nms_iou,

        max_det=MAX_DET,

        plots=True,

        project=str(
            E7_OUTPUT_ROOT
        ),

        name=run_name,

        exist_ok=True,

        verbose=True,
    )


    # --------------------------------------------------------------
    # OVERALL METRICS
    # --------------------------------------------------------------

    map5095 = float(
        val_results.box.map
    )

    map50 = float(
        val_results.box.map50
    )

    map75 = float(
        val_results.box.map75
    )

    precision = float(
        val_results.box.mp
    )

    recall = float(
        val_results.box.mr
    )


    print("\nOverall:")

    print(
        f"Precision : {precision:.6f}"
        f" = {precision * 100:.2f}%"
    )

    print(
        f"Recall    : {recall:.6f}"
        f" = {recall * 100:.2f}%"
    )

    print(
        f"mAP50-95  : {map5095:.6f}"
        f" = {map5095 * 100:.2f}%"
    )

    print(
        f"mAP50     : {map50:.6f}"
        f" = {map50 * 100:.2f}%"
    )

    print(
        f"mAP75     : {map75:.6f}"
        f" = {map75 * 100:.2f}%"
    )


    # --------------------------------------------------------------
    # PER-CLASS METRICS
    # --------------------------------------------------------------

    class_metrics = {}

    class_names = model.names

    print("\nPer-class:")
    print(
        f"{'Class':25s}"
        f"{'P':>10s}"
        f"{'R':>10s}"
        f"{'AP50':>10s}"
        f"{'AP50-95':>12s}"
    )

    print("-" * 67)


    for class_id in range(
        len(class_names)
    ):

        class_result = (
            val_results.box.class_result(
                class_id
            )
        )

        cls_p = float(
            class_result[0]
        )

        cls_r = float(
            class_result[1]
        )

        cls_ap50 = float(
            class_result[2]
        )

        cls_ap5095 = float(
            class_result[3]
        )

        class_name = class_names[
            class_id
        ]

        class_metrics[
            class_name
        ] = {
            "precision": cls_p,
            "recall": cls_r,
            "ap50": cls_ap50,
            "ap5095": cls_ap5095,
        }

        print(
            f"{class_name:25s}"
            f"{cls_p:10.4f}"
            f"{cls_r:10.4f}"
            f"{cls_ap50:10.4f}"
            f"{cls_ap5095:12.4f}"
        )


    # --------------------------------------------------------------
    # STORE RESULTS
    # --------------------------------------------------------------

    results_summary.append(
        {
            "nms_iou": nms_iou,
            "precision": precision,
            "recall": recall,
            "map50": map50,
            "map75": map75,
            "map5095": map5095,
            "class_metrics": class_metrics,
        }
    )


# ======================================================================
# 6. FINAL COMPARISON TABLE
# ======================================================================

print("\n" + "=" * 90)
print("E7 FINAL NMS COMPARISON")
print("=" * 90)

print(
    f"{'NMS IoU':>10s}"
    f"{'Precision':>12s}"
    f"{'Recall':>12s}"
    f"{'mAP50':>12s}"
    f"{'mAP75':>12s}"
    f"{'mAP50-95':>14s}"
)

print("-" * 72)


for result in results_summary:

    print(
        f"{result['nms_iou']:10.2f}"
        f"{result['precision']:12.4f}"
        f"{result['recall']:12.4f}"
        f"{result['map50']:12.4f}"
        f"{result['map75']:12.4f}"
        f"{result['map5095']:14.4f}"
    )


# ======================================================================
# 7. SELECT BEST BY PRIMARY METRIC
# ======================================================================

best_result = max(
    results_summary,
    key=lambda x: x["map5095"]
)


print("\n" + "=" * 90)
print("BEST E7 CONFIGURATION")
print("=" * 90)

print(
    f"Best NMS IoU : "
    f"{best_result['nms_iou']:.2f}"
)

print(
    f"mAP50-95     : "
    f"{best_result['map5095']:.6f}"
    f" = {best_result['map5095'] * 100:.2f}%"
)

print(
    f"mAP50        : "
    f"{best_result['map50']:.6f}"
    f" = {best_result['map50'] * 100:.2f}%"
)

print(
    f"mAP75        : "
    f"{best_result['map75']:.6f}"
    f" = {best_result['map75'] * 100:.2f}%"
)

print(
    f"Precision    : "
    f"{best_result['precision']:.6f}"
)

print(
    f"Recall       : "
    f"{best_result['recall']:.6f}"
)


# ======================================================================
# 8. SAVE CSV SUMMARY
# ======================================================================

CSV_FILE = (
    E7_OUTPUT_ROOT
    / "E7_nms_tuning_summary.csv"
)


with open(
    CSV_FILE,
    "w",
    newline="",
    encoding="utf-8"
) as f:

    writer = csv.writer(f)

    writer.writerow(
        [
            "nms_iou",
            "precision",
            "recall",
            "mAP50",
            "mAP75",
            "mAP50_95",
        ]
    )

    for result in results_summary:

        writer.writerow(
            [
                result["nms_iou"],
                result["precision"],
                result["recall"],
                result["map50"],
                result["map75"],
                result["map5095"],
            ]
        )


# ======================================================================
# 9. SAVE DETAILED TEXT SUMMARY
# ======================================================================

SUMMARY_FILE = (
    E7_OUTPUT_ROOT
    / "E7_nms_tuning_summary.txt"
)


with open(
    SUMMARY_FILE,
    "w",
    encoding="utf-8"
) as f:

    f.write(
        "E7 — NMS IoU Tuning on E5 Best Detector\n"
    )

    f.write(
        "=" * 80
        + "\n\n"
    )

    f.write(
        f"Checkpoint:\n{BEST_MODEL}\n\n"
    )

    f.write(
        f"Dataset:\n{E5_DATA_YAML}\n\n"
    )

    f.write(
        f"imgsz = {IMG_SIZE}\n"
    )

    f.write(
        f"conf = {CONF_THRESHOLD}\n"
    )

    f.write(
        f"max_det = {MAX_DET}\n\n"
    )


    for result in results_summary:

        f.write(
            "-" * 80
            + "\n"
        )

        f.write(
            f"NMS IoU = "
            f"{result['nms_iou']:.2f}\n"
        )

        f.write(
            f"Precision = "
            f"{result['precision']:.6f}\n"
        )

        f.write(
            f"Recall = "
            f"{result['recall']:.6f}\n"
        )

        f.write(
            f"mAP50 = "
            f"{result['map50']:.6f}\n"
        )

        f.write(
            f"mAP75 = "
            f"{result['map75']:.6f}\n"
        )

        f.write(
            f"mAP50-95 = "
            f"{result['map5095']:.6f}\n\n"
        )

        f.write(
            "Per-class AP50 / AP50-95\n"
        )

        for class_name, metrics in (
            result["class_metrics"].items()
        ):

            f.write(
                f"{class_name}: "
                f"AP50={metrics['ap50']:.6f}, "
                f"AP50-95={metrics['ap5095']:.6f}\n"
            )

        f.write("\n")


    f.write(
        "=" * 80
        + "\n"
    )

    f.write(
        "BEST CONFIGURATION\n"
    )

    f.write(
        "=" * 80
        + "\n"
    )

    f.write(
        f"NMS IoU = "
        f"{best_result['nms_iou']:.2f}\n"
    )

    f.write(
        f"mAP50-95 = "
        f"{best_result['map5095']:.6f}\n"
    )

    f.write(
        f"mAP50 = "
        f"{best_result['map50']:.6f}\n"
    )

    f.write(
        f"mAP75 = "
        f"{best_result['map75']:.6f}\n"
    )


# ======================================================================
# 10. FINISH
# ======================================================================

print("\n" + "=" * 90)
print("E7 COMPLETE")
print("=" * 90)

print("\nCSV summary:")
print(CSV_FILE)

print("\nDetailed summary:")
print(SUMMARY_FILE)

print("\nBest NMS IoU:")
print(
    f"{best_result['nms_iou']:.2f}"
)

E7 — NMS IoU TUNING ON E5
PyTorch        : 2.13.0+cu126
CUDA available : True
GPU            : NVIDIA GeForce RTX 3050 Ti Laptop GPU

E5 checkpoint:
C:\Users\varda\Documents\_My Computer\COMMON_space\Learning\Upgrad_EPGP_MS\upgrad_IIITB_MS_course\MS_LJMU_Material\Thesis_Code\runs\sortwaste\E5_yolo11s_7class_aug_classbalance_640\weights\best.pt
Exists: True

E5 dataset YAML:
C:\Users\varda\Documents\_My Computer\COMMON_space\Learning\Upgrad_EPGP_MS\upgrad_IIITB_MS_course\MS_LJMU_Material\Thesis_Code\runs\sortwaste\E5_7class_dataset\E5_7class.yaml
Exists: True

LOADING E5 BEST MODEL
Model loaded successfully.

E7 VALIDATION — NMS IoU = 0.50
Ultralytics 8.4.126  Python-3.11.15 torch-2.13.0+cu126 CUDA:0 (NVIDIA GeForce RTX 3050 Ti Laptop GPU, 4096MiB)
YOLO11s summary (fused): 101 layers, 9,415,509 parameters, 0 gradients, 21.4 GFLOPs
val: Fast image access  (ping: 0.10.0 ms, read: 2302.7912.0 MB/s, size: 2061.0 KB)
val: Scanning C:\Users\varda\Documents\_My Computer\COMMON_space\Learning\U

# Model E8 : YOLO11s + MobileNetV3-Large, class-weighted, using E5 detector and E7 NMS

In [ ]:
# E8 — YOLO11s + MobileNetV3-Large END-TO-END EVALUATION
#
# LJMU Thesis — SortWaste
#
# Detector:
#     E5 YOLO11s best.pt
#
# Detector inference configuration:
#     imgsz   = 640
#     conf    = 0.001
#     NMS IoU = 0.60       <- selected by E7
#     max_det = 100
#
# Classifier:
#     Existing E3Y-B class-weighted MobileNetV3-Large
#
# Evaluation taxonomy:
#     0 ecal
#     1 hdpe
#     2 mixed_plastic_rigid
#     3 mixed_plastic_soft
#     4 non_plastic
#     5 pet
#     6 pet_oil
#
# Final prediction:
#     bbox       = YOLO11s detection
#     class      = MobileNet prediction
#     confidence = YOLO confidence × MobileNet predicted-class probability
#
# Primary evaluation:
#     COCO-style AP:
#         mAP50-95
#         mAP50
#         mAP75
#         AR1 / AR10 / AR100
#
# Secondary diagnostic evaluation:
#     one-to-one GT/prediction matching at IoU >= 0.50
#     accuracy
#     macro-F1
#     weighted-F1
#     plastic-only metrics
#
# IMPORTANT:
#     - NO training
#     - NO classifier filtering
#     - MobileNet classifies every valid YOLO crop
#     - Metal + cardboard GT are merged to non_plastic
# ==================================================================================================

from pathlib import Path
from collections import defaultdict
import json
import math
import csv

import numpy as np
from PIL import Image

import torch
import torch.nn as nn
from torchvision.models import mobilenet_v3_large
from torchvision.models import MobileNet_V3_Large_Weights
from torchvision.transforms import v2

from ultralytics import YOLO

from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    confusion_matrix,
    classification_report,
)

from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval


# ==================================================================================================
# 1. PATHS
# ==================================================================================================

MS_ROOT = Path(
    r"C:\Users\varda\Documents\_My Computer\COMMON_space\Learning"
    r"\Upgrad_EPGP_MS\upgrad_IIITB_MS_course\MS_LJMU_Material"
)

DATASET_ROOT = (
    MS_ROOT
    / "Topic Data"
    / "SortWaste"
    / "dataset"
    / "dataset"
)

COCO_DATASET_ROOT = (
    DATASET_ROOT
    / "splited_all_dataset_coco"
)

VAL_IMAGES_DIR = (
    COCO_DATASET_ROOT
    / "val"
    / "images"
)

# ------------------------------------------------------------------
# IMPORTANT:
# Adjust this annotation filename ONLY if your actual val JSON uses
# a different filename.
# ------------------------------------------------------------------

POSSIBLE_VAL_JSONS = [
    COCO_DATASET_ROOT / "val" / "_annotations.coco.json",
    COCO_DATASET_ROOT / "val" / "annotations.json",
    COCO_DATASET_ROOT / "val" / "instances_val.json",
    COCO_DATASET_ROOT / "val" / "instances_default.json",
]


THESIS_RUNS = (
    MS_ROOT
    / "Thesis_Code"
    / "runs"
    / "sortwaste"
)

YOLO_CHECKPOINT = (
    THESIS_RUNS
    / "E5_yolo11s_7class_aug_classbalance_640"
    / "weights"
    / "best.pt"
)

MOBILENET_CHECKPOINT = (
    DATASET_ROOT
    / "yolo_mobilenet_crops_E3Y"
    / "mobilenet_results"
    / "E3Y_B_class_weighted"
    / "E3Y_B_MobileNetV3Large_best.pth"
)

OUTPUT_DIR = (
    THESIS_RUNS
    / "E8_yolo11s_mobilenet_endtoend"
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)


# ==================================================================================================
# 2. E8 CONFIGURATION
# ==================================================================================================

IMG_SIZE_YOLO = 640

YOLO_CONF = 0.001

# Best NMS threshold selected in E7
YOLO_NMS_IOU = 0.60

YOLO_MAX_DET = 100

MOBILENET_IMG_SIZE = 224

MATCH_IOU_THRESHOLD = 0.50

DEVICE = (
    torch.device("cuda")
    if torch.cuda.is_available()
    else torch.device("cpu")
)

YOLO_DEVICE = (
    0
    if torch.cuda.is_available()
    else "cpu"
)


CLASS_NAMES = [
    "ecal",
    "hdpe",
    "mixed_plastic_rigid",
    "mixed_plastic_soft",
    "non_plastic",
    "pet",
    "pet_oil",
]

NUM_CLASSES = len(CLASS_NAMES)

CLASS_ID_TO_NAME = {
    i: name
    for i, name in enumerate(CLASS_NAMES)
}

NAME_TO_CLASS_ID = {
    name: i
    for i, name in CLASS_ID_TO_NAME.items()
}


# ==================================================================================================
# 3. ORIGINAL COCO -> E8 7-CLASS MAPPING
#
# Original SortWaste COCO:
#
#   1 pet
#   2 hdpe
#   3 mixed_plastic_soft
#   4 ecal
#   5 metal
#   6 cardboard
#   7 mixed_plastic_rigid
#   8 pet_oil
#
# E8 internal class IDs:
#
#   0 ecal
#   1 hdpe
#   2 mixed_plastic_rigid
#   3 mixed_plastic_soft
#   4 non_plastic
#   5 pet
#   6 pet_oil
# ==================================================================================================

ORIGINAL_COCO_TO_E8 = {
    1: 5,   # pet
    2: 1,   # hdpe
    3: 3,   # mixed soft
    4: 0,   # ecal
    5: 4,   # metal -> non_plastic
    6: 4,   # cardboard -> non_plastic
    7: 2,   # mixed rigid
    8: 6,   # pet oil
}


# ==================================================================================================
# 4. FIND COCO VALIDATION JSON
# ==================================================================================================

VAL_IMAGES_DIR = (
    COCO_DATASET_ROOT
    / "val"
    / "images"
)

VAL_COCO_JSON = (
    COCO_DATASET_ROOT
    / "val"
    / "annotations"
    / "val_coco.json"
)


# ==================================================================================================
# 5. ENVIRONMENT / PATH CHECK
# ==================================================================================================

print("=" * 100)
print("E8 — YOLO11s + MobileNetV3-Large END-TO-END EVALUATION")
print("=" * 100)

print("\nPyTorch          :", torch.__version__)
print("CUDA available   :", torch.cuda.is_available())

if torch.cuda.is_available():

    print(
        "GPU              :",
        torch.cuda.get_device_name(0)
    )

    print(
        "GPU memory       :",
        f"{torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB"
    )

print("\nValidation images:")
print(VAL_IMAGES_DIR)
print("Exists:", VAL_IMAGES_DIR.exists())

print("\nValidation COCO JSON:")
print(VAL_COCO_JSON)
print("Exists:", VAL_COCO_JSON.exists())

print("\nYOLO11s checkpoint:")
print(YOLO_CHECKPOINT)
print("Exists:", YOLO_CHECKPOINT.exists())

print("\nMobileNet checkpoint:")
print(MOBILENET_CHECKPOINT)
print("Exists:", MOBILENET_CHECKPOINT.exists())

print("\nOutput:")
print(OUTPUT_DIR)


if not VAL_IMAGES_DIR.exists():
    raise FileNotFoundError(VAL_IMAGES_DIR)

if not VAL_COCO_JSON.exists():
    raise FileNotFoundError(VAL_COCO_JSON)

if not YOLO_CHECKPOINT.exists():
    raise FileNotFoundError(YOLO_CHECKPOINT)

if not MOBILENET_CHECKPOINT.exists():
    raise FileNotFoundError(MOBILENET_CHECKPOINT)


# ==================================================================================================
# 6. LOAD ORIGINAL COCO GROUND TRUTH
# ==================================================================================================

print("\n" + "=" * 100)
print("LOADING ORIGINAL COCO VALIDATION GROUND TRUTH")
print("=" * 100)

with open(
    VAL_COCO_JSON,
    "r",
    encoding="utf-8"
) as f:

    original_coco = json.load(f)


print(
    "Images      :",
    len(original_coco["images"])
)

print(
    "Annotations :",
    len(original_coco["annotations"])
)

print(
    "Categories  :",
    len(original_coco["categories"])
)


# ==================================================================================================
# 7. CREATE REMAPPED 7-CLASS COCO GT
#
# pycocotools COCO category IDs should be positive.
#
# Internal E8 IDs:
#     0..6
#
# COCO evaluation IDs:
#     1..7
#
# So:
#     coco_category_id = internal_id + 1
# ==================================================================================================

print("\n" + "=" * 100)
print("CREATING E8 7-CLASS COCO GROUND TRUTH")
print("=" * 100)


e8_coco_gt_dict = {
    "info": original_coco.get(
        "info",
        {}
    ),

    "licenses": original_coco.get(
        "licenses",
        []
    ),

    "images": original_coco[
        "images"
    ],

    "categories": [
        {
            "id": i + 1,
            "name": class_name,
            "supercategory": "waste",
        }
        for i, class_name in enumerate(
            CLASS_NAMES
        )
    ],

    "annotations": [],
}


new_annotation_id = 1

gt_by_image = defaultdict(list)


for ann in original_coco["annotations"]:

    original_category_id = int(
        ann["category_id"]
    )

    if original_category_id not in ORIGINAL_COCO_TO_E8:
        continue

    e8_class_id = ORIGINAL_COCO_TO_E8[
        original_category_id
    ]

    new_ann = dict(ann)

    new_ann["id"] = new_annotation_id

    new_ann["category_id"] = (
        e8_class_id + 1
    )

    # COCO bbox:
    # x, y, width, height

    x, y, w, h = [
        float(v)
        for v in new_ann["bbox"]
    ]

    # Some original files may have area already.
    # Recompute consistently.

    new_ann["area"] = float(
        max(0.0, w)
        *
        max(0.0, h)
    )

    new_ann["iscrowd"] = int(
        new_ann.get(
            "iscrowd",
            0
        )
    )

    if "segmentation" not in new_ann:
        new_ann["segmentation"] = []

    e8_coco_gt_dict[
        "annotations"
    ].append(
        new_ann
    )

    gt_by_image[
        int(new_ann["image_id"])
    ].append(
        {
            "bbox_xywh": [
                x,
                y,
                w,
                h
            ],

            "bbox_xyxy": [
                x,
                y,
                x + w,
                y + h
            ],

            "class_id": e8_class_id,

            "annotation_id": new_annotation_id,
        }
    )

    new_annotation_id += 1


print(
    "E8 GT images      :",
    len(e8_coco_gt_dict["images"])
)

print(
    "E8 GT annotations :",
    len(e8_coco_gt_dict["annotations"])
)

print(
    "E8 GT categories  :",
    len(e8_coco_gt_dict["categories"])
)


# Validate expected counts

if len(e8_coco_gt_dict["images"]) != 780:

    print(
        "\nWARNING:"
        f" expected 780 val images but found "
        f"{len(e8_coco_gt_dict['images'])}"
    )


if len(e8_coco_gt_dict["annotations"]) != 13065:

    print(
        "\nWARNING:"
        f" expected 13065 GT annotations but found "
        f"{len(e8_coco_gt_dict['annotations'])}"
    )


E8_GT_JSON = (
    OUTPUT_DIR
    / "E8_7class_ground_truth.json"
)

with open(
    E8_GT_JSON,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        e8_coco_gt_dict,
        f
    )


print("\nSaved remapped GT:")
print(E8_GT_JSON)


# ==================================================================================================
# 8. IMAGE ID / FILE NAME LOOKUPS
# ==================================================================================================

image_id_to_info = {
    int(img["id"]): img
    for img in e8_coco_gt_dict[
        "images"
    ]
}

filename_to_image_id = {
    Path(img["file_name"]).name:
        int(img["id"])
    for img in e8_coco_gt_dict[
        "images"
    ]
}


# ==================================================================================================
# 9. LOAD YOLO11s DETECTOR
# ==================================================================================================

print("\n" + "=" * 100)
print("LOADING E5 YOLO11s DETECTOR")
print("=" * 100)

detector = YOLO(
    str(YOLO_CHECKPOINT)
)

print("YOLO11s loaded successfully.")

print("\nDetector class names:")
print(detector.names)


# ==================================================================================================
# 10. LOAD E3Y-B MOBILENET
# ==================================================================================================

print("\n" + "=" * 100)
print("LOADING E3Y-B MOBILENETV3-LARGE")
print("=" * 100)


# Build architecture exactly as used in training.
mobilenet = mobilenet_v3_large(
    weights=None
)

in_features = (
    mobilenet.classifier[-1].in_features
)

mobilenet.classifier[-1] = nn.Linear(
    in_features,
    NUM_CLASSES
)


# ------------------------------------------------------------------
# Load checkpoint robustly.
#
# Handles:
#   raw state_dict
#   {"model_state_dict": ...}
#   {"state_dict": ...}
#   {"model": ...}
# ------------------------------------------------------------------

checkpoint = torch.load(
    MOBILENET_CHECKPOINT,
    map_location=DEVICE,
    weights_only=False
)


if isinstance(
    checkpoint,
    dict
):

    if "model_state_dict" in checkpoint:

        state_dict = checkpoint[
            "model_state_dict"
        ]

    elif "state_dict" in checkpoint:

        state_dict = checkpoint[
            "state_dict"
        ]

    elif "model" in checkpoint and isinstance(
        checkpoint["model"],
        dict
    ):

        state_dict = checkpoint[
            "model"
        ]

    else:

        # May already be a raw state dict
        state_dict = checkpoint

else:

    state_dict = checkpoint


# Strip optional "module." prefix.

clean_state_dict = {}

for key, value in state_dict.items():

    new_key = key

    if new_key.startswith(
        "module."
    ):
        new_key = new_key[
            len("module.") :
        ]

    clean_state_dict[
        new_key
    ] = value


mobilenet.load_state_dict(
    clean_state_dict,
    strict=True
)

mobilenet = mobilenet.to(
    DEVICE
)

mobilenet.eval()

print(
    "MobileNetV3-Large loaded successfully."
)


# ==================================================================================================
# 11. MOBILENET VALIDATION TRANSFORM
#
# NO RANDOM AUGMENTATION DURING EVALUATION.
# ==================================================================================================

mobilenet_transform = v2.Compose(
    [
        v2.Resize(
            (
                224,
                224
            )
        ),

        v2.ToImage(),

        v2.ToDtype(
            torch.float32,
            scale=True
        ),

        v2.Normalize(
            mean=[
                0.485,
                0.456,
                0.406
            ],

            std=[
                0.229,
                0.224,
                0.225
            ]
        ),
    ]
)


# ==================================================================================================
# 12. HELPERS
# ==================================================================================================

def clamp_box_xyxy(
    box,
    width,
    height
):

    x1, y1, x2, y2 = [
        float(v)
        for v in box
    ]

    x1 = max(
        0.0,
        min(
            x1,
            width
        )
    )

    y1 = max(
        0.0,
        min(
            y1,
            height
        )
    )

    x2 = max(
        0.0,
        min(
            x2,
            width
        )
    )

    y2 = max(
        0.0,
        min(
            y2,
            height
        )
    )

    return (
        x1,
        y1,
        x2,
        y2
    )


def xyxy_to_xywh(
    box
):

    x1, y1, x2, y2 = box

    return [
        float(x1),
        float(y1),
        float(
            x2 - x1
        ),
        float(
            y2 - y1
        ),
    ]


def bbox_iou_xyxy(
    box_a,
    box_b
):

    ax1, ay1, ax2, ay2 = box_a

    bx1, by1, bx2, by2 = box_b

    inter_x1 = max(
        ax1,
        bx1
    )

    inter_y1 = max(
        ay1,
        by1
    )

    inter_x2 = min(
        ax2,
        bx2
    )

    inter_y2 = min(
        ay2,
        by2
    )

    inter_w = max(
        0.0,
        inter_x2 - inter_x1
    )

    inter_h = max(
        0.0,
        inter_y2 - inter_y1
    )

    intersection = (
        inter_w
        *
        inter_h
    )

    area_a = max(
        0.0,
        ax2 - ax1
    ) * max(
        0.0,
        ay2 - ay1
    )

    area_b = max(
        0.0,
        bx2 - bx1
    ) * max(
        0.0,
        by2 - by1
    )

    union = (
        area_a
        +
        area_b
        -
        intersection
    )

    if union <= 0:

        return 0.0

    return float(
        intersection
        /
        union
    )


def classify_crop(
    pil_crop
):

    tensor = mobilenet_transform(
        pil_crop
    )

    tensor = tensor.unsqueeze(
        0
    ).to(
        DEVICE
    )

    with torch.no_grad():

        logits = mobilenet(
            tensor
        )

        probabilities = torch.softmax(
            logits,
            dim=1
        )

        probability, class_id = torch.max(
            probabilities,
            dim=1
        )

    return (
        int(
            class_id.item()
        ),

        float(
            probability.item()
        )
    )


# ==================================================================================================
# 13. RUN YOLO11s + MOBILENET END-TO-END
# ==================================================================================================

print("\n" + "=" * 100)
print("RUNNING E8 END-TO-END INFERENCE")
print("=" * 100)

print(
    "\nDetector configuration:"
)

print(
    "imgsz       :",
    IMG_SIZE_YOLO
)

print(
    "conf        :",
    YOLO_CONF
)

print(
    "NMS IoU     :",
    YOLO_NMS_IOU
)

print(
    "max_det     :",
    YOLO_MAX_DET
)

print(
    "\nFinal confidence = "
    "YOLO confidence × MobileNet class probability"
)


# COCO result predictions
coco_predictions = []

# Diagnostic predictions grouped by image
predictions_by_image = defaultdict(
    list
)

total_yolo_detections = 0

valid_crops = 0

invalid_boxes = 0

missing_image_ids = 0


# ------------------------------------------------------------------
# stream=True prevents storing all result objects simultaneously.
# ------------------------------------------------------------------

results_stream = detector.predict(

    source=str(
        VAL_IMAGES_DIR
    ),

    imgsz=IMG_SIZE_YOLO,

    conf=YOLO_CONF,

    iou=YOLO_NMS_IOU,

    max_det=YOLO_MAX_DET,

    device=YOLO_DEVICE,

    stream=True,

    verbose=False,

    save=False,
)


for image_index, result in enumerate(
    results_stream,
    start=1
):

    image_path = Path(
        result.path
    )

    image_name = image_path.name


    if image_name not in filename_to_image_id:

        print(
            "\nWARNING: image not found in COCO GT:",
            image_name
        )

        missing_image_ids += 1

        continue


    image_id = filename_to_image_id[
        image_name
    ]


    # Read with PIL for consistent RGB crop handling.

    image = Image.open(
        image_path
    ).convert(
        "RGB"
    )

    image_width, image_height = image.size


    if result.boxes is None:

        continue


    boxes_xyxy = (
        result.boxes.xyxy
        .detach()
        .cpu()
        .numpy()
    )

    detector_confidences = (
        result.boxes.conf
        .detach()
        .cpu()
        .numpy()
    )


    total_yolo_detections += len(
        boxes_xyxy
    )


    for det_index in range(
        len(boxes_xyxy)
    ):

        raw_box = boxes_xyxy[
            det_index
        ]

        detector_conf = float(
            detector_confidences[
                det_index
            ]
        )


        x1, y1, x2, y2 = clamp_box_xyxy(

            raw_box,

            image_width,

            image_height
        )


        # Reject invalid/zero-area boxes.

        if (
            x2 <= x1
            or
            y2 <= y1
        ):

            invalid_boxes += 1

            continue


        # PIL crop accepts float coordinates,
        # but integer boundaries are cleaner.

        crop_left = int(
            math.floor(
                x1
            )
        )

        crop_top = int(
            math.floor(
                y1
            )
        )

        crop_right = int(
            math.ceil(
                x2
            )
        )

        crop_bottom = int(
            math.ceil(
                y2
            )
        )


        crop_left = max(
            0,
            min(
                crop_left,
                image_width - 1
            )
        )

        crop_top = max(
            0,
            min(
                crop_top,
                image_height - 1
            )
        )

        crop_right = max(
            crop_left + 1,
            min(
                crop_right,
                image_width
            )
        )

        crop_bottom = max(
            crop_top + 1,
            min(
                crop_bottom,
                image_height
            )
        )


        crop = image.crop(
            (
                crop_left,
                crop_top,
                crop_right,
                crop_bottom
            )
        )


        if (
            crop.width <= 0
            or
            crop.height <= 0
        ):

            invalid_boxes += 1

            continue


        final_class_id, mobilenet_prob = classify_crop(
            crop
        )


        final_confidence = (
            detector_conf
            *
            mobilenet_prob
        )


        bbox_xyxy = [
            float(x1),
            float(y1),
            float(x2),
            float(y2),
        ]


        bbox_xywh = xyxy_to_xywh(
            bbox_xyxy
        )


        prediction_record = {
            "image_id": int(
                image_id
            ),

            # COCO category IDs = internal ID + 1

            "category_id": int(
                final_class_id + 1
            ),

            "bbox": [
                float(v)
                for v in bbox_xywh
            ],

            "score": float(
                final_confidence
            ),
        }


        coco_predictions.append(
            prediction_record
        )


        predictions_by_image[
            image_id
        ].append(
            {
                "bbox_xyxy": bbox_xyxy,

                "class_id": final_class_id,

                "detector_conf": detector_conf,

                "mobilenet_prob": mobilenet_prob,

                "score": final_confidence,
            }
        )


        valid_crops += 1


    if (
        image_index % 50 == 0
        or
        image_index == len(
            image_id_to_info
        )
    ):

        print(
            f"Processed "
            f"{image_index}/"
            f"{len(image_id_to_info)} images"
        )


print("\n" + "=" * 100)
print("E8 INFERENCE COMPLETE")
print("=" * 100)

print(
    "YOLO detections           :",
    total_yolo_detections
)

print(
    "MobileNet valid crops     :",
    valid_crops
)

print(
    "COCO predictions generated:",
    len(coco_predictions)
)

print(
    "Invalid boxes             :",
    invalid_boxes
)

print(
    "Missing COCO image IDs    :",
    missing_image_ids
)


# ==================================================================================================
# 14. SAVE COCO PREDICTIONS
# ==================================================================================================

E8_PREDICTIONS_JSON = (
    OUTPUT_DIR
    / "E8_predictions.json"
)

with open(
    E8_PREDICTIONS_JSON,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        coco_predictions,
        f
    )


print("\nSaved predictions:")
print(E8_PREDICTIONS_JSON)


if len(
    coco_predictions
) == 0:

    raise RuntimeError(
        "No predictions were generated."
    )


# ==================================================================================================
# 15. COCO EVALUATION
# ==================================================================================================

print("\n" + "=" * 100)
print("E8 COCO EVALUATION")
print("=" * 100)


coco_gt = COCO(
    str(E8_GT_JSON)
)

coco_dt = coco_gt.loadRes(
    str(E8_PREDICTIONS_JSON)
)

coco_eval = COCOeval(
    coco_gt,
    coco_dt,
    "bbox"
)


# Match previous standardized protocol.
coco_eval.params.maxDets = [
    1,
    10,
    100,
]


coco_eval.evaluate()

coco_eval.accumulate()

coco_eval.summarize()


stats = coco_eval.stats


map5095 = float(
    stats[0]
)

map50 = float(
    stats[1]
)

map75 = float(
    stats[2]
)

ar1 = float(
    stats[6]
)

ar10 = float(
    stats[7]
)

ar100 = float(
    stats[8]
)


print("\n" + "=" * 100)
print("E8 PRIMARY RESULTS")
print("=" * 100)

print(
    f"mAP50-95 : "
    f"{map5095:.6f}"
    f" = {map5095 * 100:.2f}%"
)

print(
    f"mAP50    : "
    f"{map50:.6f}"
    f" = {map50 * 100:.2f}%"
)

print(
    f"mAP75    : "
    f"{map75:.6f}"
    f" = {map75 * 100:.2f}%"
)

print(
    f"AR1      : "
    f"{ar1:.6f}"
)

print(
    f"AR10     : "
    f"{ar10:.6f}"
)

print(
    f"AR100    : "
    f"{ar100:.6f}"
)


# ==================================================================================================
# 16. PER-CLASS COCO AP
# ==================================================================================================

print("\n" + "=" * 100)
print("E8 PER-CLASS COCO RESULTS")
print("=" * 100)


# COCO precision shape:
#
# [T, R, K, A, M]
#
# T = IoU thresholds
# R = recall thresholds
# K = classes
# A = area range
# M = maxDets settings

precision = coco_eval.eval[
    "precision"
]


iou_thresholds = coco_eval.params.iouThrs

idx_iou_50 = int(
    np.argmin(
        np.abs(
            iou_thresholds
            -
            0.50
        )
    )
)


per_class_results = {}


print(
    f"{'Class':25s}"
    f"{'AP50':>12s}"
    f"{'AP50-95':>14s}"
)

print("-" * 51)


for class_index, class_name in enumerate(
    CLASS_NAMES
):

    # area=all => index 0
    # maxDets=100 => last index

    class_precision_all = precision[
        :,
        :,
        class_index,
        0,
        -1
    ]

    valid_all = (
        class_precision_all
        >
        -1
    )


    if np.any(
        valid_all
    ):

        class_ap5095 = float(
            np.mean(
                class_precision_all[
                    valid_all
                ]
            )
        )

    else:

        class_ap5095 = float(
            "nan"
        )


    class_precision_50 = precision[
        idx_iou_50,
        :,
        class_index,
        0,
        -1
    ]

    valid_50 = (
        class_precision_50
        >
        -1
    )


    if np.any(
        valid_50
    ):

        class_ap50 = float(
            np.mean(
                class_precision_50[
                    valid_50
                ]
            )
        )

    else:

        class_ap50 = float(
            "nan"
        )


    per_class_results[
        class_name
    ] = {
        "AP50": class_ap50,
        "AP50-95": class_ap5095,
    }


    print(
        f"{class_name:25s}"
        f"{class_ap50:12.6f}"
        f"{class_ap5095:14.6f}"
    )


# ==================================================================================================
# 17. ONE-TO-ONE MATCHING AT IoU >= 0.50
#
# Diagnostic classification evaluation.
#
# Matching strategy:
#     For each image:
#         calculate candidate GT/pred pairs with IoU >= 0.50
#         sort candidate pairs by IoU descending
#         greedily match one-to-one
#
# IMPORTANT:
#     Matching is geometry-only.
#     Class agreement is NOT required for matching.
# ==================================================================================================

print("\n" + "=" * 100)
print(
    "E8 MATCHED CLASSIFICATION DIAGNOSTICS "
    f"(IoU >= {MATCH_IOU_THRESHOLD:.2f})"
)
print("=" * 100)


matched_true = []

matched_pred = []

matched_ious = []

matched_scores = []


for image_id in image_id_to_info.keys():

    gt_objects = gt_by_image.get(
        image_id,
        []
    )

    pred_objects = predictions_by_image.get(
        image_id,
        []
    )


    candidate_pairs = []


    for gt_index, gt_obj in enumerate(
        gt_objects
    ):

        for pred_index, pred_obj in enumerate(
            pred_objects
        ):

            iou = bbox_iou_xyxy(
                gt_obj[
                    "bbox_xyxy"
                ],

                pred_obj[
                    "bbox_xyxy"
                ]
            )


            if iou >= MATCH_IOU_THRESHOLD:

                candidate_pairs.append(
                    (
                        iou,
                        gt_index,
                        pred_index
                    )
                )


    candidate_pairs.sort(
        key=lambda x: x[0],
        reverse=True
    )


    matched_gt_indices = set()

    matched_pred_indices = set()


    for (
        iou,
        gt_index,
        pred_index
    ) in candidate_pairs:

        if (
            gt_index
            in matched_gt_indices
        ):
            continue

        if (
            pred_index
            in matched_pred_indices
        ):
            continue


        gt_obj = gt_objects[
            gt_index
        ]

        pred_obj = pred_objects[
            pred_index
        ]


        matched_true.append(
            int(
                gt_obj[
                    "class_id"
                ]
            )
        )

        matched_pred.append(
            int(
                pred_obj[
                    "class_id"
                ]
            )
        )

        matched_ious.append(
            float(
                iou
            )
        )

        matched_scores.append(
            float(
                pred_obj[
                    "score"
                ]
            )
        )


        matched_gt_indices.add(
            gt_index
        )

        matched_pred_indices.add(
            pred_index
        )


print(
    "Proper one-to-one GT matches:",
    len(matched_true)
)

print(
    "Correct final classes       :",
    sum(
        int(a == b)
        for a, b in zip(
            matched_true,
            matched_pred
        )
    )
)


if len(
    matched_true
) == 0:

    raise RuntimeError(
        "No IoU >= 0.50 matches found."
    )


# ==================================================================================================
# 18. MATCHED CLASSIFICATION METRICS
# ==================================================================================================

matched_accuracy = accuracy_score(
    matched_true,
    matched_pred
)


matched_precision_macro, \
matched_recall_macro, \
matched_f1_macro, _ = (
    precision_recall_fscore_support(

        matched_true,

        matched_pred,

        labels=list(
            range(
                NUM_CLASSES
            )
        ),

        average="macro",

        zero_division=0,
    )
)


matched_precision_weighted, \
matched_recall_weighted, \
matched_f1_weighted, _ = (
    precision_recall_fscore_support(

        matched_true,

        matched_pred,

        labels=list(
            range(
                NUM_CLASSES
            )
        ),

        average="weighted",

        zero_division=0,
    )
)


print("\nMatched classification:")

print(
    f"Accuracy     : "
    f"{matched_accuracy:.6f}"
    f" = {matched_accuracy * 100:.2f}%"
)

print(
    f"Macro-F1     : "
    f"{matched_f1_macro:.6f}"
    f" = {matched_f1_macro * 100:.2f}%"
)

print(
    f"Weighted-F1  : "
    f"{matched_f1_weighted:.6f}"
    f" = {matched_f1_weighted * 100:.2f}%"
)


# ==================================================================================================
# 19. PER-CLASS CLASSIFICATION METRICS
# ==================================================================================================

class_precision, \
class_recall, \
class_f1, \
class_support = (
    precision_recall_fscore_support(

        matched_true,

        matched_pred,

        labels=list(
            range(
                NUM_CLASSES
            )
        ),

        zero_division=0,
    )
)


print("\n" + "=" * 100)
print("E8 MATCHED PER-CLASS CLASSIFICATION")
print("=" * 100)

print(
    f"{'Class':25s}"
    f"{'Precision':>12s}"
    f"{'Recall':>12s}"
    f"{'F1':>12s}"
    f"{'Support':>12s}"
)

print("-" * 73)


matched_classification_results = {}


for class_id, class_name in enumerate(
    CLASS_NAMES
):

    matched_classification_results[
        class_name
    ] = {
        "precision": float(
            class_precision[
                class_id
            ]
        ),

        "recall": float(
            class_recall[
                class_id
            ]
        ),

        "f1": float(
            class_f1[
                class_id
            ]
        ),

        "support": int(
            class_support[
                class_id
            ]
        ),
    }


    print(
        f"{class_name:25s}"
        f"{class_precision[class_id]:12.4f}"
        f"{class_recall[class_id]:12.4f}"
        f"{class_f1[class_id]:12.4f}"
        f"{int(class_support[class_id]):12d}"
    )


# ==================================================================================================
# 20. PLASTIC-ONLY CLASSIFICATION METRICS
#
# Exclude class 4 = non_plastic.
#
# IMPORTANT:
# Keep only GT samples belonging to plastic classes.
# Prediction may still be non_plastic, which correctly counts as an error.
# ==================================================================================================

PLASTIC_CLASS_IDS = [
    0,   # ecal
    1,   # hdpe
    2,   # mixed rigid
    3,   # mixed soft
    5,   # pet
    6,   # pet oil
]


plastic_true = []

plastic_pred = []


for true_class, pred_class in zip(
    matched_true,
    matched_pred
):

    if true_class in PLASTIC_CLASS_IDS:

        plastic_true.append(
            true_class
        )

        plastic_pred.append(
            pred_class
        )


plastic_accuracy = accuracy_score(
    plastic_true,
    plastic_pred
)


_, _, plastic_macro_f1, _ = (
    precision_recall_fscore_support(

        plastic_true,

        plastic_pred,

        labels=PLASTIC_CLASS_IDS,

        average="macro",

        zero_division=0,
    )
)


_, _, plastic_weighted_f1, _ = (
    precision_recall_fscore_support(

        plastic_true,

        plastic_pred,

        labels=PLASTIC_CLASS_IDS,

        average="weighted",

        zero_division=0,
    )
)


print("\n" + "=" * 100)
print("E8 PLASTIC-ONLY MATCHED CLASSIFICATION")
print("=" * 100)

print(
    "Plastic samples:",
    len(plastic_true)
)

print(
    f"Plastic accuracy    : "
    f"{plastic_accuracy:.6f}"
    f" = {plastic_accuracy * 100:.2f}%"
)

print(
    f"Plastic macro-F1    : "
    f"{plastic_macro_f1:.6f}"
    f" = {plastic_macro_f1 * 100:.2f}%"
)

print(
    f"Plastic weighted-F1 : "
    f"{plastic_weighted_f1:.6f}"
    f" = {plastic_weighted_f1 * 100:.2f}%"
)


# ==================================================================================================
# 21. CONFUSION MATRIX
# ==================================================================================================

cm = confusion_matrix(

    matched_true,

    matched_pred,

    labels=list(
        range(
            NUM_CLASSES
        )
    )
)


CM_CSV = (
    OUTPUT_DIR
    / "E8_confusion_matrix.csv"
)


with open(
    CM_CSV,
    "w",
    newline="",
    encoding="utf-8"
) as f:

    writer = csv.writer(
        f
    )

    writer.writerow(
        [
            "actual/predicted"
        ]
        +
        CLASS_NAMES
    )


    for i, row in enumerate(
        cm
    ):

        writer.writerow(
            [
                CLASS_NAMES[
                    i
                ]
            ]
            +
            row.tolist()
        )


print("\nConfusion matrix saved:")
print(CM_CSV)


# ==================================================================================================
# 22. CLASSIFICATION REPORT
# ==================================================================================================

classification_report_text = classification_report(

    matched_true,

    matched_pred,

    labels=list(
        range(
            NUM_CLASSES
        )
    ),

    target_names=CLASS_NAMES,

    digits=4,

    zero_division=0,
)


REPORT_FILE = (
    OUTPUT_DIR
    / "E8_classification_report.txt"
)


with open(
    REPORT_FILE,
    "w",
    encoding="utf-8"
) as f:

    f.write(
        classification_report_text
    )


print("\nClassification report:")
print(classification_report_text)


# ==================================================================================================
# 23. SAVE DETAILED SUMMARY
# ==================================================================================================

SUMMARY_FILE = (
    OUTPUT_DIR
    / "E8_endtoend_summary.txt"
)


with open(
    SUMMARY_FILE,
    "w",
    encoding="utf-8"
) as f:

    f.write(
        "E8 — YOLO11s + MobileNetV3-Large End-to-End Evaluation\n"
    )

    f.write(
        "=" * 90
        + "\n\n"
    )


    f.write(
        "Detector checkpoint:\n"
        f"{YOLO_CHECKPOINT}\n\n"
    )

    f.write(
        "MobileNet checkpoint:\n"
        f"{MOBILENET_CHECKPOINT}\n\n"
    )

    f.write(
        "Detector inference configuration\n"
    )

    f.write(
        f"imgsz   = {IMG_SIZE_YOLO}\n"
    )

    f.write(
        f"conf    = {YOLO_CONF}\n"
    )

    f.write(
        f"NMS IoU = {YOLO_NMS_IOU}\n"
    )

    f.write(
        f"max_det = {YOLO_MAX_DET}\n\n"
    )


    f.write(
        "Counts\n"
    )

    f.write(
        f"Images = {len(image_id_to_info)}\n"
    )

    f.write(
        f"GT annotations = "
        f"{len(e8_coco_gt_dict['annotations'])}\n"
    )

    f.write(
        f"YOLO detections = "
        f"{total_yolo_detections}\n"
    )

    f.write(
        f"MobileNet predictions = "
        f"{valid_crops}\n"
    )

    f.write(
        f"Invalid boxes = "
        f"{invalid_boxes}\n\n"
    )


    f.write(
        "PRIMARY COCO METRICS\n"
    )

    f.write(
        f"mAP50-95 = {map5095:.6f}\n"
    )

    f.write(
        f"mAP50 = {map50:.6f}\n"
    )

    f.write(
        f"mAP75 = {map75:.6f}\n"
    )

    f.write(
        f"AR1 = {ar1:.6f}\n"
    )

    f.write(
        f"AR10 = {ar10:.6f}\n"
    )

    f.write(
        f"AR100 = {ar100:.6f}\n\n"
    )


    f.write(
        "PER-CLASS COCO AP\n"
    )

    for class_name in CLASS_NAMES:

        values = per_class_results[
            class_name
        ]

        f.write(
            f"{class_name}: "
            f"AP50={values['AP50']:.6f}, "
            f"AP50-95={values['AP50-95']:.6f}\n"
        )


    f.write(
        "\nMATCHED CLASSIFICATION DIAGNOSTICS\n"
    )

    f.write(
        f"IoU threshold = "
        f"{MATCH_IOU_THRESHOLD:.2f}\n"
    )

    f.write(
        f"Matches = "
        f"{len(matched_true)}\n"
    )

    f.write(
        f"Correct classes = "
        f"{sum(int(a == b) for a, b in zip(matched_true, matched_pred))}\n"
    )

    f.write(
        f"Accuracy = "
        f"{matched_accuracy:.6f}\n"
    )

    f.write(
        f"Macro-F1 = "
        f"{matched_f1_macro:.6f}\n"
    )

    f.write(
        f"Weighted-F1 = "
        f"{matched_f1_weighted:.6f}\n\n"
    )


    f.write(
        "PLASTIC-ONLY MATCHED CLASSIFICATION\n"
    )

    f.write(
        f"Samples = "
        f"{len(plastic_true)}\n"
    )

    f.write(
        f"Accuracy = "
        f"{plastic_accuracy:.6f}\n"
    )

    f.write(
        f"Macro-F1 = "
        f"{plastic_macro_f1:.6f}\n"
    )

    f.write(
        f"Weighted-F1 = "
        f"{plastic_weighted_f1:.6f}\n"
    )


# ==================================================================================================
# 24. FINAL CONSOLE SUMMARY
# ==================================================================================================

print("\n" + "=" * 100)
print("E8 FINAL RESULTS")
print("=" * 100)

print("\nPRIMARY COCO METRICS")

print(
    f"mAP50-95 : "
    f"{map5095:.6f}"
    f" = {map5095 * 100:.2f}%"
)

print(
    f"mAP50    : "
    f"{map50:.6f}"
    f" = {map50 * 100:.2f}%"
)

print(
    f"mAP75    : "
    f"{map75:.6f}"
    f" = {map75 * 100:.2f}%"
)

print(
    f"AR1      : "
    f"{ar1:.6f}"
)

print(
    f"AR10     : "
    f"{ar10:.6f}"
)

print(
    f"AR100    : "
    f"{ar100:.6f}"
)


print("\nMATCHED CLASSIFICATION")

print(
    f"Matches       : "
    f"{len(matched_true)}"
)

print(
    f"Accuracy      : "
    f"{matched_accuracy:.6f}"
    f" = {matched_accuracy * 100:.2f}%"
)

print(
    f"Macro-F1      : "
    f"{matched_f1_macro:.6f}"
    f" = {matched_f1_macro * 100:.2f}%"
)

print(
    f"Weighted-F1   : "
    f"{matched_f1_weighted:.6f}"
    f" = {matched_f1_weighted * 100:.2f}%"
)


print("\nPLASTIC-ONLY")

print(
    f"Samples       : "
    f"{len(plastic_true)}"
)

print(
    f"Accuracy      : "
    f"{plastic_accuracy:.6f}"
    f" = {plastic_accuracy * 100:.2f}%"
)

print(
    f"Macro-F1      : "
    f"{plastic_macro_f1:.6f}"
    f" = {plastic_macro_f1 * 100:.2f}%"
)

print(
    f"Weighted-F1   : "
    f"{plastic_weighted_f1:.6f}"
    f" = {plastic_weighted_f1 * 100:.2f}%"
)


print("\nPER-CLASS COCO RESULTS")

print(
    f"{'Class':25s}"
    f"{'AP50':>12s}"
    f"{'AP50-95':>14s}"
)

print("-" * 51)

for class_name in CLASS_NAMES:

    values = per_class_results[
        class_name
    ]

    print(
        f"{class_name:25s}"
        f"{values['AP50']:12.6f}"
        f"{values['AP50-95']:14.6f}"
    )


print("\n" + "=" * 100)
print("E8 COMPLETE")
print("=" * 100)

print("\nSummary:")
print(SUMMARY_FILE)

print("\nPredictions:")
print(E8_PREDICTIONS_JSON)

print("\nConfusion matrix:")
print(CM_CSV)

print("\nClassification report:")
print(REPORT_FILE)

E8 — YOLO11s + MobileNetV3-Large END-TO-END EVALUATION

PyTorch          : 2.13.0+cu126
CUDA available   : True
GPU              : NVIDIA GeForce RTX 3050 Ti Laptop GPU
GPU memory       : 4.00 GB

Validation images:
C:\Users\varda\Documents\_My Computer\COMMON_space\Learning\Upgrad_EPGP_MS\upgrad_IIITB_MS_course\MS_LJMU_Material\Topic Data\SortWaste\dataset\dataset\splited_all_dataset_coco\val\images
Exists: True

Validation COCO JSON:
C:\Users\varda\Documents\_My Computer\COMMON_space\Learning\Upgrad_EPGP_MS\upgrad_IIITB_MS_course\MS_LJMU_Material\Topic Data\SortWaste\dataset\dataset\splited_all_dataset_coco\val\annotations\val_coco.json
Exists: True

YOLO11s checkpoint:
C:\Users\varda\Documents\_My Computer\COMMON_space\Learning\Upgrad_EPGP_MS\upgrad_IIITB_MS_course\MS_LJMU_Material\Thesis_Code\runs\sortwaste\E5_yolo11s_7class_aug_classbalance_640\weights\best.pt
Exists: True

MobileNet checkpoint:
C:\Users\varda\Documents\_My Computer\COMMON_space\Learning\Upgrad_EPGP_MS\upgrad_IIIT

# Model E9 — YOLO11s-matched MobileNetV3-Large retraining + end-to-end evaluation

In [ ]:
# E9 — CREATE YOLO11s-MATCHED CROPS FOR MOBILENET
#
# Detector:
#   E5 YOLO11s best.pt
#
# Purpose:
#   Generate detector-matched object crops for MobileNet retraining.
#
# Matching:
#   YOLO detections matched one-to-one with GT using IoU >= 0.50.
#
# IMPORTANT:
#   - YOLO is frozen
#   - GT class determines crop label
#   - No MobileNet inference here
#   - No training here
# ==================================================================================================

from pathlib import Path
from collections import defaultdict
import json
import math
import shutil

from PIL import Image
import torch
from ultralytics import YOLO


# ==================================================================================================
# 1. PATHS
# ==================================================================================================

MS_ROOT = Path(
    r"C:\Users\varda\Documents\_My Computer\COMMON_space\Learning"
    r"\Upgrad_EPGP_MS\upgrad_IIITB_MS_course\MS_LJMU_Material"
)

DATASET_ROOT = (
    MS_ROOT
    / "Topic Data"
    / "SortWaste"
    / "dataset"
    / "dataset"
)

COCO_ROOT = (
    DATASET_ROOT
    / "splited_all_dataset_coco"
)

TRAIN_IMAGES = COCO_ROOT / "train" / "images"
VAL_IMAGES = COCO_ROOT / "val" / "images"

TRAIN_JSON = (
    COCO_ROOT
    / "train"
    / "annotations"
    / "train_coco.json"
)

VAL_JSON = (
    COCO_ROOT
    / "val"
    / "annotations"
    / "val_coco.json"
)

RUNS_ROOT = (
    MS_ROOT
    / "Thesis_Code"
    / "runs"
    / "sortwaste"
)

YOLO_CHECKPOINT = (
    RUNS_ROOT
    / "E5_yolo11s_7class_aug_classbalance_640"
    / "weights"
    / "best.pt"
)

OUTPUT_ROOT = (
    DATASET_ROOT
    / "yolo11s_mobilenet_crops_E9"
)


# ==================================================================================================
# 2. SETTINGS
# ==================================================================================================

YOLO_IMGSZ = 640
YOLO_CONF = 0.001
YOLO_NMS_IOU = 0.60
YOLO_MAX_DET = 100

MATCH_IOU = 0.50

DEVICE = 0 if torch.cuda.is_available() else "cpu"

CLASS_NAMES = [
    "ecal",
    "hdpe",
    "mixed_plastic_rigid",
    "mixed_plastic_soft",
    "non_plastic",
    "pet",
    "pet_oil",
]


# Original COCO -> E9 internal 0-based IDs
COCO_TO_E9 = {
    1: 5,  # pet
    2: 1,  # hdpe
    3: 3,  # mixed soft
    4: 0,  # ecal
    5: 4,  # metal -> non_plastic
    6: 4,  # cardboard -> non_plastic
    7: 2,  # mixed rigid
    8: 6,  # pet oil
}


# ==================================================================================================
# 3. HELPERS
# ==================================================================================================

def bbox_iou_xyxy(a, b):

    ax1, ay1, ax2, ay2 = a
    bx1, by1, bx2, by2 = b

    ix1 = max(ax1, bx1)
    iy1 = max(ay1, by1)
    ix2 = min(ax2, bx2)
    iy2 = min(ay2, by2)

    iw = max(0.0, ix2 - ix1)
    ih = max(0.0, iy2 - iy1)

    inter = iw * ih

    area_a = max(0.0, ax2 - ax1) * max(0.0, ay2 - ay1)
    area_b = max(0.0, bx2 - bx1) * max(0.0, by2 - by1)

    union = area_a + area_b - inter

    if union <= 0:
        return 0.0

    return inter / union


def clamp_box(box, w, h):

    x1, y1, x2, y2 = map(float, box)

    x1 = max(0.0, min(x1, w))
    y1 = max(0.0, min(y1, h))
    x2 = max(0.0, min(x2, w))
    y2 = max(0.0, min(y2, h))

    return [x1, y1, x2, y2]


def load_gt(json_path):

    with open(json_path, "r", encoding="utf-8") as f:
        coco = json.load(f)

    image_lookup = {
        int(x["id"]): x
        for x in coco["images"]
    }

    filename_to_id = {
        Path(x["file_name"]).name: int(x["id"])
        for x in coco["images"]
    }

    gt_by_image = defaultdict(list)

    for ann in coco["annotations"]:

        original_cls = int(ann["category_id"])

        if original_cls not in COCO_TO_E9:
            continue

        cls_id = COCO_TO_E9[original_cls]

        x, y, w, h = map(float, ann["bbox"])

        gt_by_image[int(ann["image_id"])].append(
            {
                "class_id": cls_id,
                "bbox": [x, y, x + w, y + h],
                "ann_id": int(ann["id"]),
            }
        )

    return coco, image_lookup, filename_to_id, gt_by_image


# ==================================================================================================
# 4. PREPARE OUTPUT
# ==================================================================================================

print("=" * 100)
print("E9-A — YOLO11s MATCHED CROP GENERATION")
print("=" * 100)

print("YOLO checkpoint:", YOLO_CHECKPOINT)
print("Exists:", YOLO_CHECKPOINT.exists())

print("Train JSON:", TRAIN_JSON)
print("Exists:", TRAIN_JSON.exists())

print("Val JSON:", VAL_JSON)
print("Exists:", VAL_JSON.exists())

if not YOLO_CHECKPOINT.exists():
    raise FileNotFoundError(YOLO_CHECKPOINT)

if not TRAIN_JSON.exists():
    raise FileNotFoundError(TRAIN_JSON)

if not VAL_JSON.exists():
    raise FileNotFoundError(VAL_JSON)


if OUTPUT_ROOT.exists():
    print("\nRemoving old E9 crop directory:")
    print(OUTPUT_ROOT)
    shutil.rmtree(OUTPUT_ROOT)


for split in ["train", "val"]:
    for cls in CLASS_NAMES:
        (
            OUTPUT_ROOT
            / split
            / cls
        ).mkdir(
            parents=True,
            exist_ok=True
        )


# ==================================================================================================
# 5. LOAD YOLO
# ==================================================================================================

print("\nLoading E5 YOLO11s...")

model = YOLO(
    str(YOLO_CHECKPOINT)
)

print("Loaded successfully.")


# ==================================================================================================
# 6. PROCESS SPLIT
# ==================================================================================================

def process_split(
    split_name,
    images_dir,
    json_path
):

    print("\n" + "=" * 100)
    print(f"PROCESSING {split_name.upper()}")
    print("=" * 100)

    coco, image_lookup, filename_to_id, gt_by_image = load_gt(
        json_path
    )

    counts = {
        cls: 0
        for cls in CLASS_NAMES
    }

    total_detections = 0
    total_gt = len(coco["annotations"])
    total_matches = 0
    invalid_crops = 0

    results = model.predict(

        source=str(images_dir),

        imgsz=YOLO_IMGSZ,

        conf=YOLO_CONF,

        iou=YOLO_NMS_IOU,

        max_det=YOLO_MAX_DET,

        device=DEVICE,

        stream=True,

        verbose=False,
    )


    for image_number, result in enumerate(
        results,
        start=1
    ):

        image_path = Path(result.path)
        filename = image_path.name

        if filename not in filename_to_id:
            continue

        image_id = filename_to_id[
            filename
        ]

        gt_objects = gt_by_image.get(
            image_id,
            []
        )

        if result.boxes is None:
            continue

        pred_boxes = (
            result.boxes.xyxy
            .detach()
            .cpu()
            .numpy()
        )

        total_detections += len(
            pred_boxes
        )

        # ----------------------------------------------------------
        # Build all geometry-only candidate matches.
        # ----------------------------------------------------------

        candidates = []

        for gt_idx, gt in enumerate(
            gt_objects
        ):

            for pred_idx, pred_box in enumerate(
                pred_boxes
            ):

                iou = bbox_iou_xyxy(
                    gt["bbox"],
                    pred_box
                )

                if iou >= MATCH_IOU:

                    candidates.append(
                        (
                            iou,
                            gt_idx,
                            pred_idx
                        )
                    )


        # Highest IoU first.
        candidates.sort(
            reverse=True,
            key=lambda x: x[0]
        )

        used_gt = set()
        used_pred = set()

        matches = []

        for iou, gt_idx, pred_idx in candidates:

            if gt_idx in used_gt:
                continue

            if pred_idx in used_pred:
                continue

            used_gt.add(gt_idx)
            used_pred.add(pred_idx)

            matches.append(
                (
                    gt_idx,
                    pred_idx,
                    iou
                )
            )


        if len(matches) == 0:
            continue


        image = Image.open(
            image_path
        ).convert(
            "RGB"
        )

        width, height = image.size


        for gt_idx, pred_idx, iou in matches:

            gt = gt_objects[
                gt_idx
            ]

            pred_box = pred_boxes[
                pred_idx
            ]

            box = clamp_box(
                pred_box,
                width,
                height
            )

            x1, y1, x2, y2 = box

            if x2 <= x1 or y2 <= y1:

                invalid_crops += 1
                continue


            left = max(
                0,
                int(
                    math.floor(
                        x1
                    )
                )
            )

            top = max(
                0,
                int(
                    math.floor(
                        y1
                    )
                )
            )

            right = min(
                width,
                int(
                    math.ceil(
                        x2
                    )
                )
            )

            bottom = min(
                height,
                int(
                    math.ceil(
                        y2
                    )
                )
            )

            if right <= left or bottom <= top:

                invalid_crops += 1
                continue


            crop = image.crop(
                (
                    left,
                    top,
                    right,
                    bottom
                )
            )

            cls_id = int(
                gt["class_id"]
            )

            cls_name = CLASS_NAMES[
                cls_id
            ]


            # Unique filename:
            # original image + GT annotation + pred index.
            crop_name = (
                f"{image_path.stem}"
                f"_ann{gt['ann_id']}"
                f"_pred{pred_idx}"
                f".jpg"
            )


            output_path = (
                OUTPUT_ROOT
                / split_name
                / cls_name
                / crop_name
            )


            crop.save(
                output_path,
                quality=95
            )

            counts[
                cls_name
            ] += 1

            total_matches += 1


        if (
            image_number % 100 == 0
        ):

            print(
                f"{split_name}: "
                f"processed {image_number}/"
                f"{len(image_lookup)} images"
            )


    print("\n" + "-" * 100)
    print(f"{split_name.upper()} SUMMARY")
    print("-" * 100)

    print(
        "Images       :",
        len(image_lookup)
    )

    print(
        "GT objects   :",
        total_gt
    )

    print(
        "Detections   :",
        total_detections
    )

    print(
        "Matched crops:",
        total_matches
    )

    print(
        "Invalid crops:",
        invalid_crops
    )


    print("\nClass distribution:")

    for cls in CLASS_NAMES:

        print(
            f"{cls:25s}: "
            f"{counts[cls]}"
        )


    return {
        "images": len(image_lookup),
        "gt": total_gt,
        "detections": total_detections,
        "matches": total_matches,
        "invalid": invalid_crops,
        "counts": counts,
    }


# ==================================================================================================
# 7. RUN TRAIN AND VAL
# ==================================================================================================

train_summary = process_split(
    "train",
    TRAIN_IMAGES,
    TRAIN_JSON
)

val_summary = process_split(
    "val",
    VAL_IMAGES,
    VAL_JSON
)


# ==================================================================================================
# 8. SAVE SUMMARY
# ==================================================================================================

SUMMARY_FILE = (
    OUTPUT_ROOT
    / "E9_crop_generation_summary.txt"
)


with open(
    SUMMARY_FILE,
    "w",
    encoding="utf-8"
) as f:

    f.write(
        "E9-A — YOLO11s-Matched Crop Generation\n"
    )

    f.write(
        "=" * 80
        + "\n\n"
    )

    f.write(
        f"Detector: {YOLO_CHECKPOINT}\n"
    )

    f.write(
        f"imgsz={YOLO_IMGSZ}\n"
    )

    f.write(
        f"conf={YOLO_CONF}\n"
    )

    f.write(
        f"NMS IoU={YOLO_NMS_IOU}\n"
    )

    f.write(
        f"max_det={YOLO_MAX_DET}\n"
    )

    f.write(
        f"match IoU={MATCH_IOU}\n\n"
    )


    for split_name, summary in [
        ("train", train_summary),
        ("val", val_summary),
    ]:

        f.write(
            f"{split_name.upper()}\n"
        )

        f.write(
            "-" * 40
            + "\n"
        )

        f.write(
            f"Images={summary['images']}\n"
        )

        f.write(
            f"GT={summary['gt']}\n"
        )

        f.write(
            f"Detections={summary['detections']}\n"
        )

        f.write(
            f"Matches={summary['matches']}\n"
        )

        f.write(
            f"Invalid={summary['invalid']}\n"
        )

        for cls, count in summary[
            "counts"
        ].items():

            f.write(
                f"{cls}={count}\n"
            )

        f.write("\n")


print("\n" + "=" * 100)
print("E9-A COMPLETE")
print("=" * 100)

print("\nOutput crops:")
print(OUTPUT_ROOT)

print("\nSummary:")
print(SUMMARY_FILE)

E9-A — YOLO11s MATCHED CROP GENERATION
YOLO checkpoint: C:\Users\varda\Documents\_My Computer\COMMON_space\Learning\Upgrad_EPGP_MS\upgrad_IIITB_MS_course\MS_LJMU_Material\Thesis_Code\runs\sortwaste\E5_yolo11s_7class_aug_classbalance_640\weights\best.pt
Exists: True
Train JSON: C:\Users\varda\Documents\_My Computer\COMMON_space\Learning\Upgrad_EPGP_MS\upgrad_IIITB_MS_course\MS_LJMU_Material\Topic Data\SortWaste\dataset\dataset\splited_all_dataset_coco\train\annotations\train_coco.json
Exists: True
Val JSON: C:\Users\varda\Documents\_My Computer\COMMON_space\Learning\Upgrad_EPGP_MS\upgrad_IIITB_MS_course\MS_LJMU_Material\Topic Data\SortWaste\dataset\dataset\splited_all_dataset_coco\val\annotations\val_coco.json
Exists: True

Loading E5 YOLO11s...
Loaded successfully.

PROCESSING TRAIN
train: processed 100/3705 images
train: processed 200/3705 images
train: processed 300/3705 images
train: processed 400/3705 images
train: processed 500/3705 images
train: processed 600/3705 images
train: p

In [ ]:
# E9— RETRAIN MOBILENETV3-LARGE ON YOLO11s-MATCHED CROPS
#
# Dataset:
#   Generated by E9-A:
#       yolo11s_mobilenet_crops_E9/train/<class>
#       yolo11s_mobilenet_crops_E9/val/<class>
#
# Model:
#   MobileNetV3-Large pretrained on ImageNet
#
# Purpose:
#   Align second-stage classifier training with YOLO11s crop distribution.
#
# Loss:
#   Class-weighted CrossEntropyLoss
#
# Primary classifier selection metric:
#   Validation macro-F1
# ==================================================================================================

from pathlib import Path
import json
import copy
import time

import numpy as np

import torch
import torch.nn as nn

from torch.utils.data import DataLoader
from torchvision.datasets import ImageFolder
from torchvision.transforms import v2
from torchvision.models import (
    mobilenet_v3_large,
    MobileNet_V3_Large_Weights,
)

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_recall_fscore_support,
    confusion_matrix,
    classification_report,
)


# ==================================================================================================
# 1. PATHS
# ==================================================================================================

MS_ROOT = Path(
    r"C:\Users\varda\Documents\_My Computer\COMMON_space\Learning"
    r"\Upgrad_EPGP_MS\upgrad_IIITB_MS_course\MS_LJMU_Material"
)

DATASET_ROOT = (
    MS_ROOT
    / "Topic Data"
    / "SortWaste"
    / "dataset"
    / "dataset"
)

CROP_ROOT = (
    DATASET_ROOT
    / "yolo11s_mobilenet_crops_E9"
)

TRAIN_DIR = (
    CROP_ROOT
    / "train"
)

VAL_DIR = (
    CROP_ROOT
    / "val"
)

OUTPUT_DIR = (
    CROP_ROOT
    / "mobilenet_results"
    / "E9_YOLO11s_class_weighted"
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

BEST_MODEL_PATH = (
    OUTPUT_DIR
    / "E9_MobileNetV3Large_best.pth"
)

HISTORY_PATH = (
    OUTPUT_DIR
    / "E9_training_history.json"
)

REPORT_PATH = (
    OUTPUT_DIR
    / "E9_validation_report.txt"
)


# ==================================================================================================
# 2. SETTINGS
# ==================================================================================================

IMAGE_SIZE = 224

BATCH_SIZE = 32

NUM_WORKERS = 0

MAX_EPOCHS = 25

EARLY_STOPPING_PATIENCE = 6

LEARNING_RATE = 1e-4

WEIGHT_DECAY = 1e-4

SEED = 42


torch.manual_seed(
    SEED
)

np.random.seed(
    SEED
)


DEVICE = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)


# ==================================================================================================
# 3. TRANSFORMS
# ==================================================================================================

train_transform = v2.Compose(
    [
        v2.RandomResizedCrop(
            size=IMAGE_SIZE,
            scale=(0.85, 1.0),
            ratio=(0.90, 1.10),
        ),

        v2.RandomHorizontalFlip(
            p=0.5
        ),

        v2.RandomRotation(
            degrees=10
        ),

        v2.ColorJitter(
            brightness=0.15,
            contrast=0.15,
            saturation=0.15,
            hue=0.03,
        ),

        v2.ToImage(),

        v2.ToDtype(
            torch.float32,
            scale=True
        ),

        v2.Normalize(
            mean=[
                0.485,
                0.456,
                0.406
            ],

            std=[
                0.229,
                0.224,
                0.225
            ]
        ),
    ]
)


val_transform = v2.Compose(
    [
        v2.Resize(
            (
                IMAGE_SIZE,
                IMAGE_SIZE
            )
        ),

        v2.ToImage(),

        v2.ToDtype(
            torch.float32,
            scale=True
        ),

        v2.Normalize(
            mean=[
                0.485,
                0.456,
                0.406
            ],

            std=[
                0.229,
                0.224,
                0.225
            ]
        ),
    ]
)


# ==================================================================================================
# 4. DATASETS
# ==================================================================================================

print("=" * 100)
print("E9-B — MOBILENETV3-LARGE TRAINING")
print("=" * 100)

print("\nTrain directory:")
print(TRAIN_DIR)
print("Exists:", TRAIN_DIR.exists())

print("\nVal directory:")
print(VAL_DIR)
print("Exists:", VAL_DIR.exists())


if not TRAIN_DIR.exists():
    raise FileNotFoundError(
        TRAIN_DIR
    )

if not VAL_DIR.exists():
    raise FileNotFoundError(
        VAL_DIR
    )


train_dataset = ImageFolder(
    TRAIN_DIR,
    transform=train_transform
)

val_dataset = ImageFolder(
    VAL_DIR,
    transform=val_transform
)


print("\nClasses:")
print(train_dataset.classes)

print(
    "Train samples:",
    len(train_dataset)
)

print(
    "Val samples  :",
    len(val_dataset)
)


if train_dataset.classes != val_dataset.classes:

    raise RuntimeError(
        "Train and validation class ordering differs."
    )


CLASS_NAMES = train_dataset.classes

NUM_CLASSES = len(
    CLASS_NAMES
)


EXPECTED_CLASSES = [
    "ecal",
    "hdpe",
    "mixed_plastic_rigid",
    "mixed_plastic_soft",
    "non_plastic",
    "pet",
    "pet_oil",
]


if CLASS_NAMES != EXPECTED_CLASSES:

    raise RuntimeError(
        f"Unexpected class ordering.\n"
        f"Expected: {EXPECTED_CLASSES}\n"
        f"Found: {CLASS_NAMES}"
    )


# ==================================================================================================
# 5. CLASS COUNTS / WEIGHTS
# ==================================================================================================

targets = np.array(
    train_dataset.targets
)

class_counts = np.bincount(
    targets,
    minlength=NUM_CLASSES
)


print("\nTraining class counts:")

for idx, cls in enumerate(
    CLASS_NAMES
):

    print(
        f"{cls:25s}: "
        f"{class_counts[idx]}"
    )


# Balanced class-weight formula:
#
# weight_c = N / (K * n_c)
#
# This has mean weight around 1 and avoids extreme raw inverse-count magnitudes.

total_samples = len(
    train_dataset
)

class_weights = (
    total_samples
    /
    (
        NUM_CLASSES
        *
        class_counts
    )
)


class_weights_tensor = torch.tensor(
    class_weights,
    dtype=torch.float32,
    device=DEVICE
)


print("\nClass weights:")

for idx, cls in enumerate(
    CLASS_NAMES
):

    print(
        f"{cls:25s}: "
        f"{class_weights[idx]:.4f}"
    )


# ==================================================================================================
# 6. LOADERS
# ==================================================================================================

train_loader = DataLoader(

    train_dataset,

    batch_size=BATCH_SIZE,

    shuffle=True,

    num_workers=NUM_WORKERS,

    pin_memory=torch.cuda.is_available(),
)


val_loader = DataLoader(

    val_dataset,

    batch_size=BATCH_SIZE,

    shuffle=False,

    num_workers=NUM_WORKERS,

    pin_memory=torch.cuda.is_available(),
)


print(
    "\nTrain batches:",
    len(train_loader)
)

print(
    "Val batches  :",
    len(val_loader)
)


# ==================================================================================================
# 7. MODEL
# ==================================================================================================

print("\nLoading pretrained MobileNetV3-Large...")

weights = (
    MobileNet_V3_Large_Weights.DEFAULT
)

model = mobilenet_v3_large(
    weights=weights
)

in_features = (
    model.classifier[-1].in_features
)

model.classifier[-1] = nn.Linear(
    in_features,
    NUM_CLASSES
)

model = model.to(
    DEVICE
)


print(
    "Parameters:",
    f"{sum(p.numel() for p in model.parameters()):,}"
)


# ==================================================================================================
# 8. LOSS / OPTIMIZER
# ==================================================================================================

criterion = nn.CrossEntropyLoss(
    weight=class_weights_tensor
)


optimizer = torch.optim.AdamW(

    model.parameters(),

    lr=LEARNING_RATE,

    weight_decay=WEIGHT_DECAY,
)


# ==================================================================================================
# 9. EVALUATION FUNCTION
# ==================================================================================================

def evaluate(
    model,
    loader
):

    model.eval()

    running_loss = 0.0

    all_true = []
    all_pred = []


    with torch.no_grad():

        for images, labels in loader:

            images = images.to(
                DEVICE,
                non_blocking=True
            )

            labels = labels.to(
                DEVICE,
                non_blocking=True
            )


            logits = model(
                images
            )

            loss = criterion(
                logits,
                labels
            )


            running_loss += (
                loss.item()
                *
                images.size(0)
            )


            preds = torch.argmax(
                logits,
                dim=1
            )


            all_true.extend(
                labels.cpu().numpy().tolist()
            )

            all_pred.extend(
                preds.cpu().numpy().tolist()
            )


    avg_loss = (
        running_loss
        /
        len(loader.dataset)
    )

    accuracy = accuracy_score(
        all_true,
        all_pred
    )

    macro_f1 = f1_score(
        all_true,
        all_pred,
        average="macro",
        zero_division=0
    )

    weighted_f1 = f1_score(
        all_true,
        all_pred,
        average="weighted",
        zero_division=0
    )


    return {
        "loss": avg_loss,
        "accuracy": accuracy,
        "macro_f1": macro_f1,
        "weighted_f1": weighted_f1,
        "true": all_true,
        "pred": all_pred,
    }


# ==================================================================================================
# 10. TRAIN LOOP
# ==================================================================================================

history = []

best_macro_f1 = -1.0

best_epoch = -1

epochs_without_improvement = 0


print("\n" + "=" * 100)
print("TRAINING")
print("=" * 100)


for epoch in range(
    1,
    MAX_EPOCHS + 1
):

    epoch_start = time.time()

    model.train()

    running_loss = 0.0

    train_true = []
    train_pred = []


    for images, labels in train_loader:

        images = images.to(
            DEVICE,
            non_blocking=True
        )

        labels = labels.to(
            DEVICE,
            non_blocking=True
        )


        optimizer.zero_grad(
            set_to_none=True
        )


        logits = model(
            images
        )


        loss = criterion(
            logits,
            labels
        )


        loss.backward()

        optimizer.step()


        running_loss += (
            loss.item()
            *
            images.size(0)
        )


        preds = torch.argmax(
            logits,
            dim=1
        )


        train_true.extend(
            labels.cpu().numpy().tolist()
        )

        train_pred.extend(
            preds.detach().cpu().numpy().tolist()
        )


    train_loss = (
        running_loss
        /
        len(train_loader.dataset)
    )

    train_acc = accuracy_score(
        train_true,
        train_pred
    )


    val_metrics = evaluate(
        model,
        val_loader
    )


    epoch_time = (
        time.time()
        -
        epoch_start
    )


    print(
        f"Epoch {epoch:02d}/{MAX_EPOCHS} | "
        f"train_loss={train_loss:.4f} | "
        f"train_acc={train_acc*100:.2f}% | "
        f"val_loss={val_metrics['loss']:.4f} | "
        f"val_acc={val_metrics['accuracy']*100:.2f}% | "
        f"macroF1={val_metrics['macro_f1']:.4f} | "
        f"weightedF1={val_metrics['weighted_f1']:.4f} | "
        f"{epoch_time:.1f}s"
    )


    history.append(
        {
            "epoch": epoch,
            "train_loss": train_loss,
            "train_accuracy": train_acc,
            "val_loss": val_metrics["loss"],
            "val_accuracy": val_metrics["accuracy"],
            "val_macro_f1": val_metrics["macro_f1"],
            "val_weighted_f1": val_metrics["weighted_f1"],
        }
    )


    # ------------------------------------------------------------------
    # Select checkpoint by macro-F1 because class imbalance is important.
    # ------------------------------------------------------------------

    if (
        val_metrics["macro_f1"]
        >
        best_macro_f1
    ):

        best_macro_f1 = val_metrics[
            "macro_f1"
        ]

        best_epoch = epoch

        epochs_without_improvement = 0


        torch.save(
            {
                "epoch": epoch,

                "model_state_dict": model.state_dict(),

                "optimizer_state_dict": optimizer.state_dict(),

                "best_macro_f1": best_macro_f1,

                "classes": CLASS_NAMES,

                "class_weights": class_weights.tolist(),

                "image_size": IMAGE_SIZE,
            },

            BEST_MODEL_PATH
        )


        print(
            f"  -> saved new best model "
            f"(macro-F1={best_macro_f1:.4f})"
        )


    else:

        epochs_without_improvement += 1


    if (
        epochs_without_improvement
        >=
        EARLY_STOPPING_PATIENCE
    ):

        print(
            f"\nEarly stopping triggered "
            f"after {epoch} epochs."
        )

        break


# ==================================================================================================
# 11. SAVE HISTORY
# ==================================================================================================

with open(
    HISTORY_PATH,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        history,
        f,
        indent=2
    )


# ==================================================================================================
# 12. LOAD BEST MODEL
# ==================================================================================================

print("\n" + "=" * 100)
print("BEST MODEL EVALUATION")
print("=" * 100)

checkpoint = torch.load(
    BEST_MODEL_PATH,
    map_location=DEVICE,
    weights_only=False
)


model.load_state_dict(
    checkpoint[
        "model_state_dict"
    ]
)

model.eval()


best_val = evaluate(
    model,
    val_loader
)


print(
    "Best epoch       :",
    best_epoch
)

print(
    f"Val loss         : "
    f"{best_val['loss']:.4f}"
)

print(
    f"Val accuracy     : "
    f"{best_val['accuracy']*100:.2f}%"
)

print(
    f"Val macro-F1     : "
    f"{best_val['macro_f1']:.4f}"
)

print(
    f"Val weighted-F1  : "
    f"{best_val['weighted_f1']:.4f}"
)


# ==================================================================================================
# 13. PER-CLASS METRICS
# ==================================================================================================

precision, recall, f1, support = (
    precision_recall_fscore_support(

        best_val["true"],

        best_val["pred"],

        labels=list(
            range(
                NUM_CLASSES
            )
        ),

        zero_division=0,
    )
)


print("\nPer-class:")

print(
    f"{'Class':25s}"
    f"{'Precision':>12s}"
    f"{'Recall':>12s}"
    f"{'F1':>12s}"
    f"{'Support':>10s}"
)

print("-" * 71)


for idx, cls in enumerate(
    CLASS_NAMES
):

    print(
        f"{cls:25s}"
        f"{precision[idx]:12.4f}"
        f"{recall[idx]:12.4f}"
        f"{f1[idx]:12.4f}"
        f"{int(support[idx]):10d}"
    )


# ==================================================================================================
# 14. PLASTIC-ONLY METRICS
# ==================================================================================================

PLASTIC_IDS = [
    0,
    1,
    2,
    3,
    5,
    6,
]


plastic_true = []

plastic_pred = []


for t, p in zip(
    best_val["true"],
    best_val["pred"]
):

    if t in PLASTIC_IDS:

        plastic_true.append(
            t
        )

        plastic_pred.append(
            p
        )


plastic_acc = accuracy_score(
    plastic_true,
    plastic_pred
)


plastic_macro_f1 = f1_score(
    plastic_true,
    plastic_pred,
    labels=PLASTIC_IDS,
    average="macro",
    zero_division=0
)


plastic_weighted_f1 = f1_score(
    plastic_true,
    plastic_pred,
    labels=PLASTIC_IDS,
    average="weighted",
    zero_division=0
)


print("\nPlastic-only:")

print(
    "Samples:",
    len(plastic_true)
)

print(
    f"Accuracy    : "
    f"{plastic_acc*100:.2f}%"
)

print(
    f"Macro-F1    : "
    f"{plastic_macro_f1:.4f}"
)

print(
    f"Weighted-F1 : "
    f"{plastic_weighted_f1:.4f}"
)


# ==================================================================================================
# 15. SAVE REPORT
# ==================================================================================================

report = classification_report(

    best_val["true"],

    best_val["pred"],

    labels=list(
        range(
            NUM_CLASSES
        )
    ),

    target_names=CLASS_NAMES,

    digits=4,

    zero_division=0,
)


with open(
    REPORT_PATH,
    "w",
    encoding="utf-8"
) as f:

    f.write(
        "E9-B — MobileNetV3-Large Validation Report\n"
    )

    f.write(
        "=" * 80
        + "\n\n"
    )

    f.write(
        f"Best epoch: {best_epoch}\n"
    )

    f.write(
        f"Val accuracy: {best_val['accuracy']:.6f}\n"
    )

    f.write(
        f"Val macro-F1: {best_val['macro_f1']:.6f}\n"
    )

    f.write(
        f"Val weighted-F1: {best_val['weighted_f1']:.6f}\n"
    )

    f.write(
        f"Plastic accuracy: {plastic_acc:.6f}\n"
    )

    f.write(
        f"Plastic macro-F1: {plastic_macro_f1:.6f}\n"
    )

    f.write(
        f"Plastic weighted-F1: {plastic_weighted_f1:.6f}\n\n"
    )

    f.write(
        report
    )


print("\n" + "=" * 100)
print("E9-B COMPLETE")
print("=" * 100)

print("\nBest checkpoint:")
print(BEST_MODEL_PATH)

print("\nTraining history:")
print(HISTORY_PATH)

print("\nValidation report:")
print(REPORT_PATH)

E9-B — MOBILENETV3-LARGE TRAINING

Train directory:
C:\Users\varda\Documents\_My Computer\COMMON_space\Learning\Upgrad_EPGP_MS\upgrad_IIITB_MS_course\MS_LJMU_Material\Topic Data\SortWaste\dataset\dataset\yolo11s_mobilenet_crops_E9\train
Exists: True

Val directory:
C:\Users\varda\Documents\_My Computer\COMMON_space\Learning\Upgrad_EPGP_MS\upgrad_IIITB_MS_course\MS_LJMU_Material\Topic Data\SortWaste\dataset\dataset\yolo11s_mobilenet_crops_E9\val
Exists: True

Classes:
['ecal', 'hdpe', 'mixed_plastic_rigid', 'mixed_plastic_soft', 'non_plastic', 'pet', 'pet_oil']
Train samples: 61541
Val samples  : 12714

Training class counts:
ecal                     : 13608
hdpe                     : 16683
mixed_plastic_rigid      : 7053
mixed_plastic_soft       : 9035
non_plastic              : 2455
pet                      : 11912
pet_oil                  : 795

Class weights:
ecal                     : 0.6461
hdpe                     : 0.5270
mixed_plastic_rigid      : 1.2465
mixed_plastic_soft     

In [ ]:
# E9— YOLO11s + YOLO11s-MATCHED MOBILENETV3-LARGE
#        END-TO-END COCO EVALUATION
#
# Detector:
#   E5 YOLO11s best.pt
#
# Classifier:
#   E9-B MobileNetV3-Large trained on YOLO11s-matched crops
#
# Evaluation:
#   - Validation split
#   - 7-class taxonomy
#   - YOLO bbox retained
#   - MobileNet supplies final class
#   - Final confidence = YOLO confidence * MobileNet class probability
#   - COCO mAP50-95 / mAP50 / mAP75
#   - Geometry-only one-to-one matching at IoU >= 0.50
#   - Matched classification diagnostics
#   - Plastic-only diagnostics
#
# IMPORTANT:
#   This intentionally keeps the E8 protocol unchanged so that E8 vs E9-C
#   isolates the effect of retraining MobileNet on YOLO11s-matched crops.
# ==================================================================================================

from pathlib import Path
from collections import defaultdict
import json
import math
import csv

import numpy as np
from PIL import Image

import torch
import torch.nn as nn

from ultralytics import YOLO

from torchvision.models import (
    mobilenet_v3_large,
    MobileNet_V3_Large_Weights,
)
from torchvision.transforms import v2

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_recall_fscore_support,
    confusion_matrix,
    classification_report,
)

from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval


# ==================================================================================================
# 1. PATHS
# ==================================================================================================

MS_ROOT = Path(
    r"C:\Users\varda\Documents\_My Computer\COMMON_space\Learning"
    r"\Upgrad_EPGP_MS\upgrad_IIITB_MS_course\MS_LJMU_Material"
)

DATASET_ROOT = (
    MS_ROOT
    / "Topic Data"
    / "SortWaste"
    / "dataset"
    / "dataset"
)

COCO_ROOT = (
    DATASET_ROOT
    / "splited_all_dataset_coco"
)

VAL_IMAGES_DIR = (
    COCO_ROOT
    / "val"
    / "images"
)

VAL_COCO_JSON = (
    COCO_ROOT
    / "val"
    / "annotations"
    / "val_coco.json"
)

RUNS_ROOT = (
    MS_ROOT
    / "Thesis_Code"
    / "runs"
    / "sortwaste"
)

YOLO_CHECKPOINT = (
    RUNS_ROOT
    / "E5_yolo11s_7class_aug_classbalance_640"
    / "weights"
    / "best.pt"
)

MOBILENET_CHECKPOINT = (
    DATASET_ROOT
    / "yolo11s_mobilenet_crops_E9"
    / "mobilenet_results"
    / "E9_YOLO11s_class_weighted"
    / "E9_MobileNetV3Large_best.pth"
)

OUTPUT_DIR = (
    RUNS_ROOT
    / "E9_yolo11s_mobilenet_endtoend"
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

GT_7CLASS_JSON = (
    OUTPUT_DIR
    / "E9_7class_ground_truth.json"
)

PREDICTIONS_JSON = (
    OUTPUT_DIR
    / "E9_predictions.json"
)

CONFUSION_MATRIX_CSV = (
    OUTPUT_DIR
    / "E9_confusion_matrix.csv"
)

CLASSIFICATION_REPORT_TXT = (
    OUTPUT_DIR
    / "E9_classification_report.txt"
)

SUMMARY_TXT = (
    OUTPUT_DIR
    / "E9_endtoend_summary.txt"
)


# ==================================================================================================
# 2. SETTINGS
# ==================================================================================================

YOLO_IMGSZ = 640

YOLO_CONF = 0.001

YOLO_NMS_IOU = 0.60

YOLO_MAX_DET = 100

MATCH_IOU = 0.50

MOBILENET_IMAGE_SIZE = 224

DEVICE = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

YOLO_DEVICE = (
    0
    if torch.cuda.is_available()
    else "cpu"
)


CLASS_NAMES = [
    "ecal",
    "hdpe",
    "mixed_plastic_rigid",
    "mixed_plastic_soft",
    "non_plastic",
    "pet",
    "pet_oil",
]

NUM_CLASSES = len(
    CLASS_NAMES
)


# E9 internal 0-based ID -> COCO evaluation category ID
#
# COCO evaluation IDs will be 1..7.
E9_TO_COCO = {
    0: 1,  # ecal
    1: 2,  # hdpe
    2: 3,  # mixed rigid
    3: 4,  # mixed soft
    4: 5,  # non plastic
    5: 6,  # pet
    6: 7,  # pet oil
}


# Original SortWaste COCO category ID -> new 7-class COCO category ID
ORIGINAL_COCO_TO_7CLASS = {
    1: 6,  # pet
    2: 2,  # hdpe
    3: 4,  # mixed soft
    4: 1,  # ecal
    5: 5,  # metal -> non_plastic
    6: 5,  # cardboard -> non_plastic
    7: 3,  # mixed rigid
    8: 7,  # pet oil
}


PLASTIC_CLASS_IDS_0BASED = [
    0,  # ecal
    1,  # hdpe
    2,  # mixed rigid
    3,  # mixed soft
    5,  # pet
    6,  # pet oil
]


# ==================================================================================================
# 3. BASIC CHECKS
# ==================================================================================================

print("=" * 100)
print("E9-C — YOLO11s + YOLO11s-MATCHED MOBILENET END-TO-END EVALUATION")
print("=" * 100)

print()
print("PyTorch          :", torch.__version__)
print("CUDA available   :", torch.cuda.is_available())

if torch.cuda.is_available():

    print(
        "GPU              :",
        torch.cuda.get_device_name(0)
    )

    gpu_mem_gb = (
        torch.cuda.get_device_properties(0).total_memory
        /
        1024**3
    )

    print(
        "GPU memory       :",
        f"{gpu_mem_gb:.2f} GB"
    )


print("\nValidation images:")
print(VAL_IMAGES_DIR)
print("Exists:", VAL_IMAGES_DIR.exists())

print("\nValidation COCO JSON:")
print(VAL_COCO_JSON)
print("Exists:", VAL_COCO_JSON.exists())

print("\nYOLO11s checkpoint:")
print(YOLO_CHECKPOINT)
print("Exists:", YOLO_CHECKPOINT.exists())

print("\nMobileNet checkpoint:")
print(MOBILENET_CHECKPOINT)
print("Exists:", MOBILENET_CHECKPOINT.exists())

print("\nOutput:")
print(OUTPUT_DIR)


if not VAL_IMAGES_DIR.exists():

    raise FileNotFoundError(
        VAL_IMAGES_DIR
    )

if not VAL_COCO_JSON.exists():

    raise FileNotFoundError(
        VAL_COCO_JSON
    )

if not YOLO_CHECKPOINT.exists():

    raise FileNotFoundError(
        YOLO_CHECKPOINT
    )

if not MOBILENET_CHECKPOINT.exists():

    raise FileNotFoundError(
        MOBILENET_CHECKPOINT
    )


# ==================================================================================================
# 4. LOAD ORIGINAL COCO VALIDATION GT
# ==================================================================================================

print("\n" + "=" * 100)
print("LOADING ORIGINAL COCO VALIDATION GROUND TRUTH")
print("=" * 100)

with open(
    VAL_COCO_JSON,
    "r",
    encoding="utf-8"
) as f:

    original_coco = json.load(
        f
    )


print(
    "Images      :",
    len(original_coco["images"])
)

print(
    "Annotations :",
    len(original_coco["annotations"])
)

print(
    "Categories  :",
    len(original_coco["categories"])
)


# ==================================================================================================
# 5. CREATE 7-CLASS COCO GROUND TRUTH
# ==================================================================================================

print("\n" + "=" * 100)
print("CREATING E9 7-CLASS COCO GROUND TRUTH")
print("=" * 100)


gt7 = {
    "images": original_coco["images"],
    "annotations": [],
    "categories": [
        {
            "id": idx + 1,
            "name": name,
            "supercategory": "waste",
        }
        for idx, name in enumerate(
            CLASS_NAMES
        )
    ],
}


for ann in original_coco[
    "annotations"
]:

    original_cat = int(
        ann["category_id"]
    )

    if original_cat not in ORIGINAL_COCO_TO_7CLASS:
        continue

    new_ann = dict(
        ann
    )

    new_ann[
        "category_id"
    ] = ORIGINAL_COCO_TO_7CLASS[
        original_cat
    ]

    gt7[
        "annotations"
    ].append(
        new_ann
    )


# Keep optional COCO metadata if present.
for optional_key in [
    "info",
    "licenses",
]:

    if optional_key in original_coco:

        gt7[
            optional_key
        ] = original_coco[
            optional_key
        ]


with open(
    GT_7CLASS_JSON,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        gt7,
        f
    )


print(
    "E9 GT images      :",
    len(gt7["images"])
)

print(
    "E9 GT annotations :",
    len(gt7["annotations"])
)

print(
    "E9 GT categories  :",
    len(gt7["categories"])
)

print("\nSaved remapped GT:")
print(GT_7CLASS_JSON)


# ==================================================================================================
# 6. LOOKUPS FOR MATCHED DIAGNOSTICS
# ==================================================================================================

filename_to_image_id = {
    Path(
        img["file_name"]
    ).name: int(
        img["id"]
    )
    for img in original_coco["images"]
}


gt_by_image = defaultdict(
    list
)


for ann in gt7[
    "annotations"
]:

    x, y, w, h = map(
        float,
        ann["bbox"]
    )

    gt_by_image[
        int(
            ann["image_id"]
        )
    ].append(
        {
            # convert 1..7 to 0..6
            "class_id": int(
                ann["category_id"]
            ) - 1,

            "bbox": [
                x,
                y,
                x + w,
                y + h,
            ],

            "ann_id": int(
                ann["id"]
            ),
        }
    )


# ==================================================================================================
# 7. LOAD YOLO11s
# ==================================================================================================

print("\n" + "=" * 100)
print("LOADING E5 YOLO11s DETECTOR")
print("=" * 100)

detector = YOLO(
    str(
        YOLO_CHECKPOINT
    )
)

print(
    "YOLO11s loaded successfully."
)

print("\nDetector class names:")
print(
    detector.names
)


# ==================================================================================================
# 8. LOAD E9 MOBILENET
# ==================================================================================================

print("\n" + "=" * 100)
print("LOADING E9-B MOBILENETV3-LARGE")
print("=" * 100)


mobilenet = mobilenet_v3_large(
    weights=None
)

mobilenet.classifier[-1] = nn.Linear(
    mobilenet.classifier[-1].in_features,
    NUM_CLASSES
)


checkpoint = torch.load(
    MOBILENET_CHECKPOINT,
    map_location=DEVICE,
    weights_only=False
)


# Support both raw state_dict and saved checkpoint dictionary.
if isinstance(
    checkpoint,
    dict
):

    if (
        "model_state_dict"
        in checkpoint
    ):

        state_dict = checkpoint[
            "model_state_dict"
        ]

    elif (
        "state_dict"
        in checkpoint
    ):

        state_dict = checkpoint[
            "state_dict"
        ]

    else:

        # If every value is tensor-like, assume raw state dict.
        if all(
            isinstance(
                v,
                torch.Tensor
            )
            for v in checkpoint.values()
        ):

            state_dict = checkpoint

        else:

            raise RuntimeError(
                "Could not identify MobileNet model state dictionary "
                "inside checkpoint."
            )

else:

    raise RuntimeError(
        "Unexpected MobileNet checkpoint format."
    )


# Remove DataParallel prefix if present.
clean_state_dict = {}

for key, value in state_dict.items():

    if key.startswith(
        "module."
    ):

        key = key[
            len("module.") :
        ]

    clean_state_dict[
        key
    ] = value


mobilenet.load_state_dict(
    clean_state_dict
)

mobilenet = mobilenet.to(
    DEVICE
)

mobilenet.eval()


print(
    "MobileNetV3-Large loaded successfully."
)


# ==================================================================================================
# 9. MOBILENET EVAL TRANSFORM
# ==================================================================================================

mobilenet_transform = v2.Compose(
    [
        v2.ToImage(),

        v2.Resize(
            (
                MOBILENET_IMAGE_SIZE,
                MOBILENET_IMAGE_SIZE
            )
        ),

        v2.ToDtype(
            torch.float32,
            scale=True
        ),

        v2.Normalize(
            mean=[
                0.485,
                0.456,
                0.406
            ],

            std=[
                0.229,
                0.224,
                0.225
            ]
        ),
    ]
)


# ==================================================================================================
# 10. HELPERS
# ==================================================================================================

def bbox_iou_xyxy(
    a,
    b
):

    ax1, ay1, ax2, ay2 = map(
        float,
        a
    )

    bx1, by1, bx2, by2 = map(
        float,
        b
    )


    ix1 = max(
        ax1,
        bx1
    )

    iy1 = max(
        ay1,
        by1
    )

    ix2 = min(
        ax2,
        bx2
    )

    iy2 = min(
        ay2,
        by2
    )


    iw = max(
        0.0,
        ix2 - ix1
    )

    ih = max(
        0.0,
        iy2 - iy1
    )

    intersection = (
        iw
        *
        ih
    )


    area_a = (
        max(
            0.0,
            ax2 - ax1
        )
        *
        max(
            0.0,
            ay2 - ay1
        )
    )

    area_b = (
        max(
            0.0,
            bx2 - bx1
        )
        *
        max(
            0.0,
            by2 - by1
        )
    )


    union = (
        area_a
        +
        area_b
        -
        intersection
    )


    if union <= 0:

        return 0.0


    return (
        intersection
        /
        union
    )


def clamp_crop_box(
    box,
    width,
    height
):

    x1, y1, x2, y2 = map(
        float,
        box
    )


    x1 = max(
        0.0,
        min(
            x1,
            width
        )
    )

    y1 = max(
        0.0,
        min(
            y1,
            height
        )
    )

    x2 = max(
        0.0,
        min(
            x2,
            width
        )
    )

    y2 = max(
        0.0,
        min(
            y2,
            height
        )
    )


    left = max(
        0,
        int(
            math.floor(
                x1
            )
        )
    )

    top = max(
        0,
        int(
            math.floor(
                y1
            )
        )
    )

    right = min(
        width,
        int(
            math.ceil(
                x2
            )
        )
    )

    bottom = min(
        height,
        int(
            math.ceil(
                y2
            )
        )
    )


    return (
        left,
        top,
        right,
        bottom
    )


# ==================================================================================================
# 11. RUN END-TO-END INFERENCE
# ==================================================================================================

print("\n" + "=" * 100)
print("RUNNING E9-C END-TO-END INFERENCE")
print("=" * 100)

print("\nDetector configuration:")

print(
    "imgsz       :",
    YOLO_IMGSZ
)

print(
    "conf        :",
    YOLO_CONF
)

print(
    "NMS IoU     :",
    YOLO_NMS_IOU
)

print(
    "max_det     :",
    YOLO_MAX_DET
)

print(
    "\nFinal confidence = "
    "YOLO confidence × MobileNet class probability"
)


predictions = []

# For geometry-only classification diagnostics.
predictions_by_image = defaultdict(
    list
)

total_yolo_detections = 0

valid_crops = 0

invalid_boxes = 0

missing_image_ids = 0


results = detector.predict(

    source=str(
        VAL_IMAGES_DIR
    ),

    imgsz=YOLO_IMGSZ,

    conf=YOLO_CONF,

    iou=YOLO_NMS_IOU,

    max_det=YOLO_MAX_DET,

    device=YOLO_DEVICE,

    stream=True,

    verbose=False,
)


with torch.no_grad():

    for image_number, result in enumerate(
        results,
        start=1
    ):

        image_path = Path(
            result.path
        )

        filename = image_path.name


        if filename not in filename_to_image_id:

            missing_image_ids += 1
            continue


        image_id = filename_to_image_id[
            filename
        ]


        if result.boxes is None:

            if (
                image_number % 50 == 0
                or image_number == 780
            ):

                print(
                    f"Processed "
                    f"{image_number}/780 images"
                )

            continue


        boxes_xyxy = (
            result.boxes.xyxy
            .detach()
            .cpu()
            .numpy()
        )

        detector_confidences = (
            result.boxes.conf
            .detach()
            .cpu()
            .numpy()
        )


        total_yolo_detections += len(
            boxes_xyxy
        )


        image = Image.open(
            image_path
        ).convert(
            "RGB"
        )

        width, height = image.size


        crop_tensors = []

        crop_metadata = []


        for pred_idx, (
            box,
            yolo_conf
        ) in enumerate(
            zip(
                boxes_xyxy,
                detector_confidences
            )
        ):

            left, top, right, bottom = (
                clamp_crop_box(
                    box,
                    width,
                    height
                )
            )


            if (
                right <= left
                or bottom <= top
            ):

                invalid_boxes += 1
                continue


            crop = image.crop(
                (
                    left,
                    top,
                    right,
                    bottom
                )
            )


            tensor = mobilenet_transform(
                crop
            )


            crop_tensors.append(
                tensor
            )


            crop_metadata.append(
                {
                    "box_xyxy": [
                        float(
                            box[0]
                        ),
                        float(
                            box[1]
                        ),
                        float(
                            box[2]
                        ),
                        float(
                            box[3]
                        ),
                    ],

                    "yolo_conf": float(
                        yolo_conf
                    ),
                }
            )


        if len(
            crop_tensors
        ) > 0:

            crop_batch = torch.stack(
                crop_tensors
            ).to(
                DEVICE
            )


            logits = mobilenet(
                crop_batch
            )


            probabilities = torch.softmax(
                logits,
                dim=1
            )


            predicted_classes = torch.argmax(
                probabilities,
                dim=1
            )


            predicted_probs = probabilities[
                torch.arange(
                    len(
                        predicted_classes
                    ),
                    device=DEVICE
                ),
                predicted_classes
            ]


            predicted_classes = (
                predicted_classes
                .detach()
                .cpu()
                .numpy()
            )

            predicted_probs = (
                predicted_probs
                .detach()
                .cpu()
                .numpy()
            )


            for metadata, cls_id, class_prob in zip(
                crop_metadata,
                predicted_classes,
                predicted_probs
            ):

                cls_id = int(
                    cls_id
                )

                class_prob = float(
                    class_prob
                )

                yolo_conf = metadata[
                    "yolo_conf"
                ]


                final_score = (
                    yolo_conf
                    *
                    class_prob
                )


                x1, y1, x2, y2 = metadata[
                    "box_xyxy"
                ]


                bbox_xywh = [
                    float(
                        x1
                    ),

                    float(
                        y1
                    ),

                    float(
                        max(
                            0.0,
                            x2 - x1
                        )
                    ),

                    float(
                        max(
                            0.0,
                            y2 - y1
                        )
                    ),
                ]


                coco_prediction = {
                    "image_id": int(
                        image_id
                    ),

                    "category_id": int(
                        E9_TO_COCO[
                            cls_id
                        ]
                    ),

                    "bbox": bbox_xywh,

                    "score": float(
                        final_score
                    ),
                }


                predictions.append(
                    coco_prediction
                )


                predictions_by_image[
                    image_id
                ].append(
                    {
                        "class_id": cls_id,

                        "bbox": [
                            x1,
                            y1,
                            x2,
                            y2,
                        ],

                        "score": final_score,

                        "yolo_conf": yolo_conf,

                        "mobilenet_prob": class_prob,
                    }
                )


                valid_crops += 1


        if (
            image_number % 50 == 0
            or image_number == 780
        ):

            print(
                f"Processed "
                f"{image_number}/780 images"
            )


# ==================================================================================================
# 12. SAVE PREDICTIONS
# ==================================================================================================

print("\n" + "=" * 100)
print("E9-C INFERENCE COMPLETE")
print("=" * 100)

print(
    "YOLO detections           :",
    total_yolo_detections
)

print(
    "MobileNet valid crops     :",
    valid_crops
)

print(
    "COCO predictions generated:",
    len(
        predictions
    )
)

print(
    "Invalid boxes             :",
    invalid_boxes
)

print(
    "Missing COCO image IDs    :",
    missing_image_ids
)


with open(
    PREDICTIONS_JSON,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        predictions,
        f
    )


print("\nSaved predictions:")
print(PREDICTIONS_JSON)


# ==================================================================================================
# 13. COCO EVALUATION
# ==================================================================================================

print("\n" + "=" * 100)
print("E9-C COCO EVALUATION")
print("=" * 100)


coco_gt = COCO(
    str(
        GT_7CLASS_JSON
    )
)

coco_dt = coco_gt.loadRes(
    str(
        PREDICTIONS_JSON
    )
)


coco_eval = COCOeval(
    coco_gt,
    coco_dt,
    "bbox"
)


coco_eval.params.maxDets = [
    1,
    10,
    100,
]


coco_eval.evaluate()

coco_eval.accumulate()

coco_eval.summarize()


mAP_50_95 = float(
    coco_eval.stats[0]
)

mAP_50 = float(
    coco_eval.stats[1]
)

mAP_75 = float(
    coco_eval.stats[2]
)

AR1 = float(
    coco_eval.stats[6]
)

AR10 = float(
    coco_eval.stats[7]
)

AR100 = float(
    coco_eval.stats[8]
)


print("\n" + "=" * 100)
print("E9-C PRIMARY RESULTS")
print("=" * 100)

print(
    f"mAP50-95 : "
    f"{mAP_50_95:.6f} = "
    f"{mAP_50_95 * 100:.2f}%"
)

print(
    f"mAP50    : "
    f"{mAP_50:.6f} = "
    f"{mAP_50 * 100:.2f}%"
)

print(
    f"mAP75    : "
    f"{mAP_75:.6f} = "
    f"{mAP_75 * 100:.2f}%"
)

print(
    f"AR1      : "
    f"{AR1:.6f}"
)

print(
    f"AR10     : "
    f"{AR10:.6f}"
)

print(
    f"AR100    : "
    f"{AR100:.6f}"
)


# ==================================================================================================
# 14. PER-CLASS COCO METRICS
# ==================================================================================================

# precision shape:
# [IoU thresholds, recall thresholds, categories, area ranges, maxDets]
precision_tensor = (
    coco_eval.eval[
        "precision"
    ]
)

iou_thresholds = (
    coco_eval.params.iouThrs
)

max_det_index = (
    list(
        coco_eval.params.maxDets
    ).index(
        100
    )
)

area_index = 0


def mean_valid_precision(
    values
):

    valid = values[
        values > -1
    ]

    if len(
        valid
    ) == 0:

        return float(
            "nan"
        )

    return float(
        np.mean(
            valid
        )
    )


per_class_coco = {}


# Locate IoU=.50 index.
iou50_idx = int(
    np.argmin(
        np.abs(
            iou_thresholds
            -
            0.50
        )
    )
)


for class_idx, class_name in enumerate(
    CLASS_NAMES
):

    ap50_values = precision_tensor[
        iou50_idx,
        :,
        class_idx,
        area_index,
        max_det_index,
    ]


    ap_all_values = precision_tensor[
        :,
        :,
        class_idx,
        area_index,
        max_det_index,
    ]


    ap50 = mean_valid_precision(
        ap50_values
    )

    ap5095 = mean_valid_precision(
        ap_all_values
    )


    per_class_coco[
        class_name
    ] = {
        "AP50": ap50,
        "AP50-95": ap5095,
    }


print("\n" + "=" * 100)
print("E9-C PER-CLASS COCO RESULTS")
print("=" * 100)

print(
    f"{'Class':25s}"
    f"{'AP50':>14s}"
    f"{'AP50-95':>14s}"
)

print(
    "-" * 53
)


for class_name in CLASS_NAMES:

    metrics = per_class_coco[
        class_name
    ]

    print(
        f"{class_name:25s}"
        f"{metrics['AP50']:14.6f}"
        f"{metrics['AP50-95']:14.6f}"
    )


# ==================================================================================================
# 15. GEOMETRY-ONLY ONE-TO-ONE MATCHING FOR CLASSIFICATION DIAGNOSTICS
# ==================================================================================================

all_gt_classes = []

all_pred_classes = []

total_matches = 0


for image_id in filename_to_image_id.values():

    gt_objects = gt_by_image.get(
        image_id,
        []
    )

    pred_objects = predictions_by_image.get(
        image_id,
        []
    )


    candidates = []


    for gt_idx, gt in enumerate(
        gt_objects
    ):

        for pred_idx, pred in enumerate(
            pred_objects
        ):

            iou = bbox_iou_xyxy(
                gt["bbox"],
                pred["bbox"]
            )


            if iou >= MATCH_IOU:

                candidates.append(
                    (
                        iou,
                        gt_idx,
                        pred_idx
                    )
                )


    candidates.sort(
        key=lambda x: x[0],
        reverse=True
    )


    used_gt = set()

    used_pred = set()


    for iou, gt_idx, pred_idx in candidates:

        if gt_idx in used_gt:
            continue

        if pred_idx in used_pred:
            continue


        used_gt.add(
            gt_idx
        )

        used_pred.add(
            pred_idx
        )


        all_gt_classes.append(
            int(
                gt_objects[
                    gt_idx
                ][
                    "class_id"
                ]
            )
        )


        all_pred_classes.append(
            int(
                pred_objects[
                    pred_idx
                ][
                    "class_id"
                ]
            )
        )


        total_matches += 1


correct_final_classes = sum(
    int(
        t == p
    )
    for t, p in zip(
        all_gt_classes,
        all_pred_classes
    )
)


matched_accuracy = accuracy_score(
    all_gt_classes,
    all_pred_classes
)

matched_macro_f1 = f1_score(
    all_gt_classes,
    all_pred_classes,
    labels=list(
        range(
            NUM_CLASSES
        )
    ),
    average="macro",
    zero_division=0,
)

matched_weighted_f1 = f1_score(
    all_gt_classes,
    all_pred_classes,
    labels=list(
        range(
            NUM_CLASSES
        )
    ),
    average="weighted",
    zero_division=0,
)


print("\n" + "=" * 100)
print("E9-C MATCHED CLASSIFICATION DIAGNOSTICS (IoU >= 0.50)")
print("=" * 100)

print(
    "Proper one-to-one GT matches:",
    total_matches
)

print(
    "Correct final classes       :",
    correct_final_classes
)

print("\nMatched classification:")

print(
    f"Accuracy     : "
    f"{matched_accuracy:.6f} = "
    f"{matched_accuracy * 100:.2f}%"
)

print(
    f"Macro-F1     : "
    f"{matched_macro_f1:.6f} = "
    f"{matched_macro_f1 * 100:.2f}%"
)

print(
    f"Weighted-F1  : "
    f"{matched_weighted_f1:.6f} = "
    f"{matched_weighted_f1 * 100:.2f}%"
)


# ==================================================================================================
# 16. PER-CLASS MATCHED CLASSIFICATION
# ==================================================================================================

precision_cls, recall_cls, f1_cls, support_cls = (
    precision_recall_fscore_support(
        all_gt_classes,
        all_pred_classes,
        labels=list(
            range(
                NUM_CLASSES
            )
        ),
        zero_division=0,
    )
)


print("\n" + "=" * 100)
print("E9-C MATCHED PER-CLASS CLASSIFICATION")
print("=" * 100)

print(
    f"{'Class':25s}"
    f"{'Precision':>12s}"
    f"{'Recall':>12s}"
    f"{'F1':>12s}"
    f"{'Support':>12s}"
)

print(
    "-" * 73
)


for idx, class_name in enumerate(
    CLASS_NAMES
):

    print(
        f"{class_name:25s}"
        f"{precision_cls[idx]:12.4f}"
        f"{recall_cls[idx]:12.4f}"
        f"{f1_cls[idx]:12.4f}"
        f"{int(support_cls[idx]):12d}"
    )


# ==================================================================================================
# 17. PLASTIC-ONLY MATCHED CLASSIFICATION
# ==================================================================================================

plastic_true = []

plastic_pred = []


for gt_cls, pred_cls in zip(
    all_gt_classes,
    all_pred_classes
):

    if gt_cls in PLASTIC_CLASS_IDS_0BASED:

        plastic_true.append(
            gt_cls
        )

        # Leave prediction unchanged.
        #
        # If MobileNet predicts non_plastic for a plastic GT,
        # it still counts as an error.
        plastic_pred.append(
            pred_cls
        )


plastic_accuracy = accuracy_score(
    plastic_true,
    plastic_pred
)


plastic_macro_f1 = f1_score(
    plastic_true,
    plastic_pred,
    labels=PLASTIC_CLASS_IDS_0BASED,
    average="macro",
    zero_division=0,
)


plastic_weighted_f1 = f1_score(
    plastic_true,
    plastic_pred,
    labels=PLASTIC_CLASS_IDS_0BASED,
    average="weighted",
    zero_division=0,
)


print("\n" + "=" * 100)
print("E9-C PLASTIC-ONLY MATCHED CLASSIFICATION")
print("=" * 100)

print(
    "Plastic samples:",
    len(
        plastic_true
    )
)

print(
    f"Plastic accuracy    : "
    f"{plastic_accuracy:.6f} = "
    f"{plastic_accuracy * 100:.2f}%"
)

print(
    f"Plastic macro-F1    : "
    f"{plastic_macro_f1:.6f} = "
    f"{plastic_macro_f1 * 100:.2f}%"
)

print(
    f"Plastic weighted-F1 : "
    f"{plastic_weighted_f1:.6f} = "
    f"{plastic_weighted_f1 * 100:.2f}%"
)


# ==================================================================================================
# 18. CONFUSION MATRIX
# ==================================================================================================

cm = confusion_matrix(
    all_gt_classes,
    all_pred_classes,
    labels=list(
        range(
            NUM_CLASSES
        )
    ),
)


with open(
    CONFUSION_MATRIX_CSV,
    "w",
    newline="",
    encoding="utf-8"
) as f:

    writer = csv.writer(
        f
    )


    writer.writerow(
        [
            "actual/predicted",
            *CLASS_NAMES
        ]
    )


    for idx, class_name in enumerate(
        CLASS_NAMES
    ):

        writer.writerow(
            [
                class_name,
                *cm[
                    idx
                ].tolist()
            ]
        )


print("\nConfusion matrix saved:")
print(
    CONFUSION_MATRIX_CSV
)


# ==================================================================================================
# 19. CLASSIFICATION REPORT
# ==================================================================================================

report = classification_report(
    all_gt_classes,
    all_pred_classes,
    labels=list(
        range(
            NUM_CLASSES
        )
    ),
    target_names=CLASS_NAMES,
    digits=4,
    zero_division=0,
)


with open(
    CLASSIFICATION_REPORT_TXT,
    "w",
    encoding="utf-8"
) as f:

    f.write(
        report
    )


print("\nClassification report:")
print(
    report
)


# ==================================================================================================
# 20. SAVE FULL SUMMARY
# ==================================================================================================

with open(
    SUMMARY_TXT,
    "w",
    encoding="utf-8"
) as f:

    f.write(
        "E9-C — YOLO11s + YOLO11s-Matched MobileNetV3-Large\n"
    )

    f.write(
        "=" * 90
        + "\n\n"
    )


    f.write(
        "CONFIGURATION\n"
    )

    f.write(
        "-" * 50
        + "\n"
    )

    f.write(
        f"YOLO checkpoint: "
        f"{YOLO_CHECKPOINT}\n"
    )

    f.write(
        f"MobileNet checkpoint: "
        f"{MOBILENET_CHECKPOINT}\n"
    )

    f.write(
        f"Validation JSON: "
        f"{VAL_COCO_JSON}\n"
    )

    f.write(
        f"imgsz={YOLO_IMGSZ}\n"
    )

    f.write(
        f"conf={YOLO_CONF}\n"
    )

    f.write(
        f"NMS IoU={YOLO_NMS_IOU}\n"
    )

    f.write(
        f"max_det={YOLO_MAX_DET}\n"
    )

    f.write(
        f"match IoU={MATCH_IOU}\n"
    )

    f.write(
        "final score = YOLO confidence * MobileNet probability\n\n"
    )


    f.write(
        "INFERENCE COUNTS\n"
    )

    f.write(
        "-" * 50
        + "\n"
    )

    f.write(
        f"YOLO detections={total_yolo_detections}\n"
    )

    f.write(
        f"Valid MobileNet crops={valid_crops}\n"
    )

    f.write(
        f"Predictions={len(predictions)}\n"
    )

    f.write(
        f"Invalid boxes={invalid_boxes}\n"
    )

    f.write(
        f"Missing image IDs={missing_image_ids}\n\n"
    )


    f.write(
        "PRIMARY COCO METRICS\n"
    )

    f.write(
        "-" * 50
        + "\n"
    )

    f.write(
        f"mAP50-95={mAP_50_95:.6f}\n"
    )

    f.write(
        f"mAP50={mAP_50:.6f}\n"
    )

    f.write(
        f"mAP75={mAP_75:.6f}\n"
    )

    f.write(
        f"AR1={AR1:.6f}\n"
    )

    f.write(
        f"AR10={AR10:.6f}\n"
    )

    f.write(
        f"AR100={AR100:.6f}\n\n"
    )


    f.write(
        "PER-CLASS COCO RESULTS\n"
    )

    f.write(
        "-" * 50
        + "\n"
    )


    for class_name in CLASS_NAMES:

        f.write(
            f"{class_name}: "
            f"AP50="
            f"{per_class_coco[class_name]['AP50']:.6f}, "
            f"AP50-95="
            f"{per_class_coco[class_name]['AP50-95']:.6f}\n"
        )


    f.write(
        "\nMATCHED CLASSIFICATION\n"
    )

    f.write(
        "-" * 50
        + "\n"
    )

    f.write(
        f"Matches={total_matches}\n"
    )

    f.write(
        f"Correct classes={correct_final_classes}\n"
    )

    f.write(
        f"Accuracy={matched_accuracy:.6f}\n"
    )

    f.write(
        f"Macro-F1={matched_macro_f1:.6f}\n"
    )

    f.write(
        f"Weighted-F1={matched_weighted_f1:.6f}\n\n"
    )


    f.write(
        "PER-CLASS MATCHED CLASSIFICATION\n"
    )

    f.write(
        "-" * 50
        + "\n"
    )


    for idx, class_name in enumerate(
        CLASS_NAMES
    ):

        f.write(
            f"{class_name}: "
            f"precision={precision_cls[idx]:.6f}, "
            f"recall={recall_cls[idx]:.6f}, "
            f"F1={f1_cls[idx]:.6f}, "
            f"support={int(support_cls[idx])}\n"
        )


    f.write(
        "\nPLASTIC-ONLY\n"
    )

    f.write(
        "-" * 50
        + "\n"
    )

    f.write(
        f"Samples={len(plastic_true)}\n"
    )

    f.write(
        f"Accuracy={plastic_accuracy:.6f}\n"
    )

    f.write(
        f"Macro-F1={plastic_macro_f1:.6f}\n"
    )

    f.write(
        f"Weighted-F1={plastic_weighted_f1:.6f}\n"
    )


# ==================================================================================================
# 21. FINAL PRINT
# ==================================================================================================

print("\n" + "=" * 100)
print("E9-C FINAL RESULTS")
print("=" * 100)

print("\nPRIMARY COCO METRICS")

print(
    f"mAP50-95 : "
    f"{mAP_50_95:.6f} = "
    f"{mAP_50_95 * 100:.2f}%"
)

print(
    f"mAP50    : "
    f"{mAP_50:.6f} = "
    f"{mAP_50 * 100:.2f}%"
)

print(
    f"mAP75    : "
    f"{mAP_75:.6f} = "
    f"{mAP_75 * 100:.2f}%"
)

print(
    f"AR1      : "
    f"{AR1:.6f}"
)

print(
    f"AR10     : "
    f"{AR10:.6f}"
)

print(
    f"AR100    : "
    f"{AR100:.6f}"
)


print("\nMATCHED CLASSIFICATION")

print(
    "Matches       :",
    total_matches
)

print(
    f"Accuracy      : "
    f"{matched_accuracy:.6f} = "
    f"{matched_accuracy * 100:.2f}%"
)

print(
    f"Macro-F1      : "
    f"{matched_macro_f1:.6f} = "
    f"{matched_macro_f1 * 100:.2f}%"
)

print(
    f"Weighted-F1   : "
    f"{matched_weighted_f1:.6f} = "
    f"{matched_weighted_f1 * 100:.2f}%"
)


print("\nPLASTIC-ONLY")

print(
    "Samples       :",
    len(
        plastic_true
    )
)

print(
    f"Accuracy      : "
    f"{plastic_accuracy:.6f} = "
    f"{plastic_accuracy * 100:.2f}%"
)

print(
    f"Macro-F1      : "
    f"{plastic_macro_f1:.6f} = "
    f"{plastic_macro_f1 * 100:.2f}%"
)

print(
    f"Weighted-F1   : "
    f"{plastic_weighted_f1:.6f} = "
    f"{plastic_weighted_f1 * 100:.2f}%"
)


print("\nPER-CLASS COCO RESULTS")

print(
    f"{'Class':25s}"
    f"{'AP50':>14s}"
    f"{'AP50-95':>14s}"
)

print(
    "-" * 53
)


for class_name in CLASS_NAMES:

    metrics = per_class_coco[
        class_name
    ]

    print(
        f"{class_name:25s}"
        f"{metrics['AP50']:14.6f}"
        f"{metrics['AP50-95']:14.6f}"
    )


print("\n" + "=" * 100)
print("E9-C COMPLETE")
print("=" * 100)

print("\nSummary:")
print(
    SUMMARY_TXT
)

print("\nPredictions:")
print(
    PREDICTIONS_JSON
)

print("\nConfusion matrix:")
print(
    CONFUSION_MATRIX_CSV
)

print("\nClassification report:")
print(
    CLASSIFICATION_REPORT_TXT
)

E9-C — YOLO11s + YOLO11s-MATCHED MOBILENET END-TO-END EVALUATION

PyTorch          : 2.13.0+cu126
CUDA available   : True
GPU              : NVIDIA GeForce RTX 3050 Ti Laptop GPU
GPU memory       : 4.00 GB

Validation images:
C:\Users\varda\Documents\_My Computer\COMMON_space\Learning\Upgrad_EPGP_MS\upgrad_IIITB_MS_course\MS_LJMU_Material\Topic Data\SortWaste\dataset\dataset\splited_all_dataset_coco\val\images
Exists: True

Validation COCO JSON:
C:\Users\varda\Documents\_My Computer\COMMON_space\Learning\Upgrad_EPGP_MS\upgrad_IIITB_MS_course\MS_LJMU_Material\Topic Data\SortWaste\dataset\dataset\splited_all_dataset_coco\val\annotations\val_coco.json
Exists: True

YOLO11s checkpoint:
C:\Users\varda\Documents\_My Computer\COMMON_space\Learning\Upgrad_EPGP_MS\upgrad_IIITB_MS_course\MS_LJMU_Material\Thesis_Code\runs\sortwaste\E5_yolo11s_7class_aug_classbalance_640\weights\best.pt
Exists: True

MobileNet checkpoint:
C:\Users\varda\Documents\_My Computer\COMMON_space\Learning\Upgrad_EPGP_MS\u

# Model E10 — YOLO11s-Matched Crops + MobileNetV3-Large Using the Controlled E3Y-B Training Protocol

E10 uses the already-generated YOLO11s-matched E9 crops, while reproducing the MobileNetV3-Large architecture, augmentation, class-weighted loss, optimizer, learning-rate scheduler, batch size, early stopping criterion, checkpoint selection criterion, and random seed used in E3Y-B. Thus, the intended experimental difference between E3Y-B and E10 is the detector used to generate the classifier training crops: YOLO11n versus YOLO11s.

In [ ]:
# E10 — YOLO11s-MATCHED CROPS + MOBILENET-V3-LARGE CONTROLLED E3Y-B TRAINING PROTOCOL
# =============================================================================
#
# Official experiment:
#
#   E10 — YOLO11s-Matched Crops + MobileNetV3-Large
#          Using the Controlled E3Y-B Training Protocol
#
# PURPOSE
# -------
# Perform one final controlled MobileNet retraining experiment using the
# already-created E9 YOLO11s-matched crops.
#
# The MobileNet training procedure reproduces E3Y-B as closely as possible.
#
#
# CONTROLLED COMPARISON
# ---------------------
#
# E3Y-B:
#
#   YOLO11n matched crops
#       ->
#   MobileNetV3-Large
#       ->
#   E3Y-B training protocol
#
#
# E10:
#
#   YOLO11s matched crops
#       ->
#   MobileNetV3-Large
#       ->
#   SAME E3Y-B training protocol
#
#
# INTENDED MAIN DIFFERENCE
# ------------------------
#
#   E3Y-B crop detector : YOLO11n
#   E10 crop detector   : YOLO11s
#
#
# SAME TRAINING SETTINGS AS E3Y-B
# --------------------------------
#
#   Image size          : 224 x 224
#   Batch size          : 64
#   Epochs              : 30
#   Learning rate       : 1e-4
#   Weight decay        : 1e-4
#   Optimizer           : AdamW
#   Loss                : inverse-frequency class-weighted CrossEntropyLoss
#
#   Augmentation:
#       Resize(224,224)
#       RandomHorizontalFlip(p=0.5)
#       RandomRotation(10 degrees)
#       ColorJitter(
#           brightness=0.15,
#           contrast=0.15,
#           saturation=0.10,
#           hue=0.02
#       )
#
#   IMPORTANT:
#       NO RandomResizedCrop
#
#   Scheduler:
#       ReduceLROnPlateau(
#           mode="max",
#           factor=0.5,
#           patience=2
#       )
#
#   Early stopping      : validation Macro-F1
#   Early stop patience : 7
#   Minimum delta       : 1e-4
#   Best checkpoint     : validation Macro-F1
#   Seed                : 42
#
#
# DATA
# ----
#
# Reuse the already-created E9 crops:
#
#   yolo11s_mobilenet_crops_E9/train
#   yolo11s_mobilenet_crops_E9/val
#
# Expected E9 counts:
#
# Train = 61,541
# Val   = 12,714
#
#
# NO:
# ---
#   - New crop generation
#   - Oversampling
#   - SMOTE
#   - Focal loss
#   - RandomResizedCrop
#   - Class-specific augmentation
#   - Synthetic data
#   - New hyperparameter tuning
#
# =============================================================================


# =============================================================================
# 1. IMPORTS
# =============================================================================

import random
import json
import time
import copy

import numpy as np
import pandas as pd

from pathlib import Path

import torch
import torch.nn as nn
import torch.optim as optim

from torch.utils.data import Dataset, DataLoader

from torchvision import models, transforms

from PIL import Image

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix
)

import matplotlib.pyplot as plt


# =============================================================================
# 2. CONFIGURATION
# =============================================================================

SEED = 42


# -----------------------------------------------------------------------------
# Dataset root
# -----------------------------------------------------------------------------

DATASET_ROOT = Path(
    r"C:\Users\varda\Documents\_My Computer\COMMON_space\Learning\Upgrad_EPGP_MS"
    r"\upgrad_IIITB_MS_course\MS_LJMU_Material\Topic Data\SortWaste\dataset\dataset"
)


# -----------------------------------------------------------------------------
# IMPORTANT:
#
# REUSE THE EXISTING E9 YOLO11s-MATCHED CROPS.
#
# DO NOT GENERATE CROPS AGAIN.
# -----------------------------------------------------------------------------

E9_ROOT = (
    DATASET_ROOT
    / "yolo11s_mobilenet_crops_E9"
)

TRAIN_DIR = (
    E9_ROOT
    / "train"
)

VAL_DIR = (
    E9_ROOT
    / "val"
)


# -----------------------------------------------------------------------------
# Results
# -----------------------------------------------------------------------------

RESULTS_DIR = (
    E9_ROOT
    / "mobilenet_results"
    / "E10_controlled_E3YB_protocol"
)

RESULTS_DIR.mkdir(
    parents=True,
    exist_ok=True
)


# -----------------------------------------------------------------------------
# Classes
# -----------------------------------------------------------------------------

CLASS_NAMES = [
    "ecal",
    "hdpe",
    "mixed_plastic_rigid",
    "mixed_plastic_soft",
    "non_plastic",
    "pet",
    "pet_oil"
]

NUM_CLASSES = len(
    CLASS_NAMES
)

CLASS_TO_IDX = {
    name: idx
    for idx, name in enumerate(
        CLASS_NAMES
    )
}


# -----------------------------------------------------------------------------
# Training configuration
#
# THESE VALUES COPY E3Y-B.
# -----------------------------------------------------------------------------

IMAGE_SIZE = 224

BATCH_SIZE = 64

NUM_EPOCHS = 30

LEARNING_RATE = 1e-4

WEIGHT_DECAY = 1e-4

NUM_WORKERS = 0

PATIENCE = 7

MIN_DELTA = 1e-4


# =============================================================================
# 3. REPRODUCIBILITY
# =============================================================================

def set_seed(seed=42):

    random.seed(
        seed
    )

    np.random.seed(
        seed
    )

    torch.manual_seed(
        seed
    )

    if torch.cuda.is_available():

        torch.cuda.manual_seed(
            seed
        )

        torch.cuda.manual_seed_all(
            seed
        )

    torch.backends.cudnn.deterministic = True

    torch.backends.cudnn.benchmark = False


set_seed(
    SEED
)


# =============================================================================
# 4. DEVICE
# =============================================================================

DEVICE = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)


print("=" * 90)

print(
    "E10 — YOLO11s-MATCHED CROPS + MOBILENET-V3-LARGE"
)

print(
    "CONTROLLED E3Y-B TRAINING PROTOCOL"
)

print("=" * 90)

print()

print(
    f"Device         : {DEVICE}"
)

if torch.cuda.is_available():

    print(
        f"GPU            : "
        f"{torch.cuda.get_device_name(0)}"
    )

    print(
        f"CUDA           : "
        f"{torch.version.cuda}"
    )

    gpu_memory = (
        torch.cuda.get_device_properties(
            0
        ).total_memory
        /
        1024**3
    )

    print(
        f"GPU memory     : "
        f"{gpu_memory:.2f} GB"
    )


print(
    f"Classes        : {NUM_CLASSES}"
)

print(
    f"Batch size     : {BATCH_SIZE}"
)

print(
    f"Epochs         : {NUM_EPOCHS}"
)

print(
    f"Learning rate  : {LEARNING_RATE}"
)

print(
    f"Weight decay   : {WEIGHT_DECAY}"
)

print(
    f"Early patience : {PATIENCE}"
)

print(
    f"Seed           : {SEED}"
)

print()


# =============================================================================
# 5. EXPERIMENT DEFINITION
# =============================================================================

print("=" * 90)
print("EXPERIMENT DEFINITION")
print("=" * 90)

print()

print(
    "E10 = EXISTING E9 YOLO11s-MATCHED CROPS "
    "+ EXACT E3Y-B MOBILENET TRAINING RECIPE"
)

print()

print(
    "Crop-producing detector : YOLO11s"
)

print(
    "Crop dataset            : Existing E9 crops"
)

print(
    "Classifier              : MobileNetV3-Large"
)

print(
    "Class balancing         : Class-weighted CrossEntropyLoss"
)

print(
    "Best model criterion    : Validation Macro-F1"
)

print(
    "Scheduler               : ReduceLROnPlateau"
)

print()

print(
    "RandomResizedCrop       : NO"
)

print(
    "Oversampling            : NO"
)

print(
    "SMOTE                   : NO"
)

print(
    "Focal loss              : NO"
)

print(
    "Class-specific augment. : NO"
)

print()


# =============================================================================
# 6. CHECK DATASET PATHS
# =============================================================================

print("=" * 90)
print("CHECKING E9 CROP DATASET")
print("=" * 90)


for path in [
    E9_ROOT,
    TRAIN_DIR,
    VAL_DIR
]:

    print(
        f"{str(path):<110} "
        f"Exists: {path.exists()}"
    )

    if not path.exists():

        raise FileNotFoundError(
            f"Required path does not exist:\n"
            f"{path}"
        )


print()


# =============================================================================
# 7. BUILD DATAFRAME FROM EXISTING E9 CLASS FOLDERS
# =============================================================================
#
# E9 crops are already organized into class directories.
#
# We explicitly walk CLASS_NAMES in the required class order rather than
# allowing automatic alphabetical mappings to define the labels.
#
# =============================================================================

VALID_EXTENSIONS = {
    ".jpg",
    ".jpeg",
    ".png",
    ".bmp",
    ".webp"
}


def build_dataframe(
    root_dir
):

    rows = []

    for class_name in CLASS_NAMES:

        class_dir = (
            root_dir
            / class_name
        )

        if not class_dir.exists():

            raise FileNotFoundError(
                f"Missing expected class directory:\n"
                f"{class_dir}"
            )

        image_paths = sorted(
            [
                p
                for p in class_dir.rglob("*")
                if (
                    p.is_file()
                    and
                    p.suffix.lower()
                    in VALID_EXTENSIONS
                )
            ]
        )

        for image_path in image_paths:

            rows.append(
                {
                    "resolved_path":
                        image_path,

                    "class_name_normalized":
                        class_name,

                    "label":
                        CLASS_TO_IDX[
                            class_name
                        ]
                }
            )

    return pd.DataFrame(
        rows
    )


train_df = build_dataframe(
    TRAIN_DIR
)

val_df = build_dataframe(
    VAL_DIR
)


print(
    f"Training samples   : "
    f"{len(train_df)}"
)

print(
    f"Validation samples : "
    f"{len(val_df)}"
)

print()


# =============================================================================
# 8. SANITY CHECK EXPECTED E9 COUNTS
# =============================================================================

EXPECTED_TRAIN = 61541

EXPECTED_VAL = 12714


print("=" * 90)
print("E9 DATASET COUNT CHECK")
print("=" * 90)


print(
    f"Expected train : {EXPECTED_TRAIN}"
)

print(
    f"Actual train   : {len(train_df)}"
)

print()

print(
    f"Expected val   : {EXPECTED_VAL}"
)

print(
    f"Actual val     : {len(val_df)}"
)

print()


if len(
    train_df
) != EXPECTED_TRAIN:

    print(
        "WARNING: Training sample count differs "
        "from the recorded E9 count."
    )


if len(
    val_df
) != EXPECTED_VAL:

    print(
        "WARNING: Validation sample count differs "
        "from the recorded E9 count."
    )


print()


# =============================================================================
# 9. CLASS DISTRIBUTION
# =============================================================================

print("=" * 90)
print("CLASS DISTRIBUTION")
print("=" * 90)


train_counts = (
    train_df[
        "class_name_normalized"
    ]
    .value_counts()
    .reindex(
        CLASS_NAMES
    )
    .fillna(
        0
    )
    .astype(
        int
    )
)


val_counts = (
    val_df[
        "class_name_normalized"
    ]
    .value_counts()
    .reindex(
        CLASS_NAMES
    )
    .fillna(
        0
    )
    .astype(
        int
    )
)


distribution_df = pd.DataFrame(
    {
        "class":
            CLASS_NAMES,

        "train":
            train_counts.values,

        "validation":
            val_counts.values
    }
)


print(
    distribution_df.to_string(
        index=False
    )
)

print()


distribution_df.to_csv(
    RESULTS_DIR
    / "class_distribution.csv",
    index=False
)


# =============================================================================
# 10. CHECK RECORDED E9 CLASS COUNTS
# =============================================================================

EXPECTED_TRAIN_COUNTS = {
    "ecal": 13608,
    "hdpe": 16683,
    "mixed_plastic_rigid": 7053,
    "mixed_plastic_soft": 9035,
    "non_plastic": 2455,
    "pet": 11912,
    "pet_oil": 795,
}


EXPECTED_VAL_COUNTS = {
    "ecal": 2515,
    "hdpe": 4801,
    "mixed_plastic_rigid": 1084,
    "mixed_plastic_soft": 1405,
    "non_plastic": 681,
    "pet": 2061,
    "pet_oil": 167,
}


print("=" * 90)
print("RECORDED E9 CLASS COUNT VERIFICATION")
print("=" * 90)


count_mismatch = False


for class_name in CLASS_NAMES:

    actual_train = int(
        train_counts[
            class_name
        ]
    )

    expected_train = (
        EXPECTED_TRAIN_COUNTS[
            class_name
        ]
    )

    actual_val = int(
        val_counts[
            class_name
        ]
    )

    expected_val = (
        EXPECTED_VAL_COUNTS[
            class_name
        ]
    )


    train_ok = (
        actual_train
        ==
        expected_train
    )

    val_ok = (
        actual_val
        ==
        expected_val
    )


    print(
        f"{class_name:<25}"
        f" Train "
        f"{actual_train:>6}/{expected_train:<6} "
        f"{'OK' if train_ok else 'MISMATCH':<10}"
        f" Val "
        f"{actual_val:>5}/{expected_val:<5} "
        f"{'OK' if val_ok else 'MISMATCH'}"
    )


    if (
        not train_ok
        or
        not val_ok
    ):

        count_mismatch = True


print()


if count_mismatch:

    print(
        "WARNING: One or more E9 class counts "
        "do not match the recorded E9 crop generation."
    )

else:

    print(
        "All E9 crop counts match the recorded E9 dataset."
    )


print()


# =============================================================================
# 11. CALCULATE CLASS WEIGHTS
# =============================================================================
#
# EXACT E3Y-B FORMULA:
#
#       weight_c = N / (K * n_c)
#
# N   = total training samples
# K   = number of classes
# n_c = training samples for class c
#
# =============================================================================

print("=" * 90)
print("CALCULATING CLASS WEIGHTS")
print("=" * 90)


total_train_samples = len(
    train_df
)


class_weights = []


for class_name in CLASS_NAMES:

    class_count = int(
        train_counts[
            class_name
        ]
    )

    if class_count <= 0:

        raise ValueError(
            f"Class '{class_name}' "
            f"has zero training samples."
        )


    weight = (
        total_train_samples
        /
        (
            NUM_CLASSES
            *
            class_count
        )
    )


    class_weights.append(
        weight
    )


class_weights = np.array(
    class_weights,
    dtype=np.float32
)


class_weight_df = pd.DataFrame(
    {
        "class":
            CLASS_NAMES,

        "training_samples":
            [
                int(
                    train_counts[c]
                )
                for c in CLASS_NAMES
            ],

        "class_weight":
            class_weights
    }
)


print()

print(
    class_weight_df.to_string(
        index=False
    )
)

print()


class_weight_df.to_csv(
    RESULTS_DIR
    / "class_weights.csv",
    index=False
)


class_weights_tensor = torch.tensor(
    class_weights,
    dtype=torch.float32,
    device=DEVICE
)


# =============================================================================
# 12. DATASET CLASS
# =============================================================================

class E10Dataset(
    Dataset
):

    def __init__(
        self,
        dataframe,
        transform=None
    ):

        self.df = dataframe.reset_index(
            drop=True
        )

        self.transform = transform


    def __len__(
        self
    ):

        return len(
            self.df
        )


    def __getitem__(
        self,
        idx
    ):

        row = self.df.iloc[
            idx
        ]


        image_path = row[
            "resolved_path"
        ]


        label = int(
            row[
                "label"
            ]
        )


        try:

            image = Image.open(
                image_path
            ).convert(
                "RGB"
            )

        except Exception as e:

            raise RuntimeError(
                f"Could not load image:\n"
                f"{image_path}\n"
                f"Error: {e}"
            )


        if self.transform is not None:

            image = self.transform(
                image
            )


        return (
            image,
            label
        )


# =============================================================================
# 13. TRANSFORMS
# =============================================================================
#
# IMPORTANT:
#
# COPY OF E3Y-B TRANSFORMS.
#
# DO NOT REPLACE Resize WITH RandomResizedCrop.
#
# =============================================================================

train_transform = transforms.Compose(
    [

        transforms.Resize(
            (
                IMAGE_SIZE,
                IMAGE_SIZE
            )
        ),

        transforms.RandomHorizontalFlip(
            p=0.5
        ),

        transforms.RandomRotation(
            degrees=10
        ),

        transforms.ColorJitter(
            brightness=0.15,
            contrast=0.15,
            saturation=0.10,
            hue=0.02
        ),

        transforms.ToTensor(),

        transforms.Normalize(
            mean=[
                0.485,
                0.456,
                0.406
            ],

            std=[
                0.229,
                0.224,
                0.225
            ]
        )
    ]
)


val_transform = transforms.Compose(
    [

        transforms.Resize(
            (
                IMAGE_SIZE,
                IMAGE_SIZE
            )
        ),

        transforms.ToTensor(),

        transforms.Normalize(
            mean=[
                0.485,
                0.456,
                0.406
            ],

            std=[
                0.229,
                0.224,
                0.225
            ]
        )
    ]
)


# =============================================================================
# 14. CREATE DATASETS
# =============================================================================

train_dataset = E10Dataset(
    train_df,
    transform=train_transform
)


val_dataset = E10Dataset(
    val_df,
    transform=val_transform
)


# =============================================================================
# 15. CREATE DATALOADERS
# =============================================================================
#
# COPY E3Y-B:
#
# BATCH_SIZE = 64
# shuffle train = True
# shuffle val   = False
# workers       = 0
#
# =============================================================================

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available()
)


val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available()
)


print("=" * 90)
print("DATALOADERS")
print("=" * 90)


print(
    f"Training samples   : "
    f"{len(train_dataset)}"
)

print(
    f"Validation samples : "
    f"{len(val_dataset)}"
)

print(
    f"Training batches   : "
    f"{len(train_loader)}"
)

print(
    f"Validation batches : "
    f"{len(val_loader)}"
)

print()


# =============================================================================
# 16. LOAD MOBILENET-V3-LARGE
# =============================================================================
#
# COPY E3Y-B:
#
# ImageNet pretrained MobileNetV3-Large
#
# =============================================================================

print("=" * 90)
print("LOADING MOBILENET-V3-LARGE")
print("=" * 90)


weights = (
    models
    .MobileNet_V3_Large_Weights
    .DEFAULT
)


model = models.mobilenet_v3_large(
    weights=weights
)


in_features = (
    model
    .classifier[-1]
    .in_features
)


model.classifier[-1] = nn.Linear(
    in_features,
    NUM_CLASSES
)


model = model.to(
    DEVICE
)


print(
    model.classifier
)

print()


total_parameters = sum(
    p.numel()
    for p in model.parameters()
)


trainable_parameters = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)


print(
    f"Total parameters     : "
    f"{total_parameters:,}"
)

print(
    f"Trainable parameters : "
    f"{trainable_parameters:,}"
)

print()


# =============================================================================
# 17. LOSS
# =============================================================================
#
# COPY E3Y-B:
#
# inverse-frequency weighted CrossEntropyLoss
#
# =============================================================================

criterion = nn.CrossEntropyLoss(
    weight=class_weights_tensor
)


# =============================================================================
# 18. OPTIMIZER
# =============================================================================
#
# COPY E3Y-B:
#
# AdamW
# lr = 1e-4
# weight_decay = 1e-4
#
# =============================================================================

optimizer = optim.AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY
)


# =============================================================================
# 19. LR SCHEDULER
# =============================================================================
#
# COPY E3Y-B EXACTLY:
#
# ReduceLROnPlateau
# mode = max
# factor = 0.5
# patience = 2
#
# =============================================================================

scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="max",
    factor=0.5,
    patience=2
)


print(
    "Loss function : "
    "Class-weighted CrossEntropyLoss"
)

print(
    "Optimizer     : AdamW"
)

print(
    "Scheduler     : ReduceLROnPlateau"
)

print()


# =============================================================================
# 20. TRAIN ONE EPOCH
# =============================================================================

def train_one_epoch(
    model,
    loader,
    criterion,
    optimizer,
    device
):

    model.train()


    running_loss = 0.0

    all_predictions = []

    all_targets = []


    for images, targets in loader:

        images = images.to(
            device,
            non_blocking=True
        )


        targets = targets.to(
            device,
            non_blocking=True
        )


        optimizer.zero_grad(
            set_to_none=True
        )


        outputs = model(
            images
        )


        loss = criterion(
            outputs,
            targets
        )


        loss.backward()


        optimizer.step()


        running_loss += (
            loss.item()
            *
            images.size(
                0
            )
        )


        predictions = outputs.argmax(
            dim=1
        )


        all_predictions.extend(
            predictions
            .detach()
            .cpu()
            .numpy()
        )


        all_targets.extend(
            targets
            .detach()
            .cpu()
            .numpy()
        )


    epoch_loss = (
        running_loss
        /
        len(
            loader.dataset
        )
    )


    accuracy = accuracy_score(
        all_targets,
        all_predictions
    )


    macro_f1 = f1_score(
        all_targets,
        all_predictions,
        average="macro",
        zero_division=0
    )


    weighted_f1 = f1_score(
        all_targets,
        all_predictions,
        average="weighted",
        zero_division=0
    )


    return (
        epoch_loss,
        accuracy,
        macro_f1,
        weighted_f1
    )


# =============================================================================
# 21. VALIDATION
# =============================================================================

def evaluate(
    model,
    loader,
    criterion,
    device
):

    model.eval()


    running_loss = 0.0

    all_predictions = []

    all_targets = []


    with torch.no_grad():

        for images, targets in loader:

            images = images.to(
                device,
                non_blocking=True
            )


            targets = targets.to(
                device,
                non_blocking=True
            )


            outputs = model(
                images
            )


            loss = criterion(
                outputs,
                targets
            )


            running_loss += (
                loss.item()
                *
                images.size(
                    0
                )
            )


            predictions = outputs.argmax(
                dim=1
            )


            all_predictions.extend(
                predictions
                .cpu()
                .numpy()
            )


            all_targets.extend(
                targets
                .cpu()
                .numpy()
            )


    epoch_loss = (
        running_loss
        /
        len(
            loader.dataset
        )
    )


    accuracy = accuracy_score(
        all_targets,
        all_predictions
    )


    macro_f1 = f1_score(
        all_targets,
        all_predictions,
        average="macro",
        zero_division=0
    )


    weighted_f1 = f1_score(
        all_targets,
        all_predictions,
        average="weighted",
        zero_division=0
    )


    return (
        epoch_loss,
        accuracy,
        macro_f1,
        weighted_f1,
        all_targets,
        all_predictions
    )


# =============================================================================
# 22. TRAINING LOOP
# =============================================================================

print("=" * 90)
print("STARTING E10 TRAINING")
print("=" * 90)

print()

print(
    "This run intentionally copies the E3Y-B training recipe."
)

print(
    "The existing E9 YOLO11s crops are the new input dataset."
)

print()


history = []


best_macro_f1 = -1.0

best_epoch = 0

best_state = None

epochs_without_improvement = 0


training_start = time.time()


for epoch in range(
    1,
    NUM_EPOCHS + 1
):

    epoch_start = time.time()


    (
        train_loss,
        train_acc,
        train_macro_f1,
        train_weighted_f1

    ) = train_one_epoch(
        model,
        train_loader,
        criterion,
        optimizer,
        DEVICE
    )


    (
        val_loss,
        val_acc,
        val_macro_f1,
        val_weighted_f1,
        val_targets,
        val_predictions

    ) = evaluate(
        model,
        val_loader,
        criterion,
        DEVICE
    )


    # -------------------------------------------------------------------------
    # COPY E3Y-B scheduler behaviour
    # -------------------------------------------------------------------------

    scheduler.step(
        val_macro_f1
    )


    current_lr = (
        optimizer
        .param_groups[0][
            "lr"
        ]
    )


    epoch_time = (
        time.time()
        -
        epoch_start
    )


    history.append(
        {
            "epoch":
                epoch,

            "train_loss":
                train_loss,

            "train_accuracy":
                train_acc,

            "train_macro_f1":
                train_macro_f1,

            "train_weighted_f1":
                train_weighted_f1,

            "val_loss":
                val_loss,

            "val_accuracy":
                val_acc,

            "val_macro_f1":
                val_macro_f1,

            "val_weighted_f1":
                val_weighted_f1,

            "learning_rate":
                current_lr,

            "epoch_time_sec":
                epoch_time
        }
    )


    print(
        f"\nEpoch "
        f"{epoch:02d}/{NUM_EPOCHS}"
    )


    print(
        f"Train | "
        f"Loss: {train_loss:.4f} | "
        f"Acc: {train_acc:.4f} | "
        f"Macro F1: {train_macro_f1:.4f} | "
        f"Weighted F1: {train_weighted_f1:.4f}"
    )


    print(
        f"Val   | "
        f"Loss: {val_loss:.4f} | "
        f"Acc: {val_acc:.4f} | "
        f"Macro F1: {val_macro_f1:.4f} | "
        f"Weighted F1: {val_weighted_f1:.4f}"
    )


    print(
        f"LR: {current_lr:.8f} | "
        f"Time: {epoch_time:.1f}s"
    )


    # -------------------------------------------------------------------------
    # BEST MODEL
    #
    # COPY E3Y-B:
    #
    # Validation Macro-F1 with MIN_DELTA = 1e-4
    # -------------------------------------------------------------------------

    if val_macro_f1 > (
        best_macro_f1
        +
        MIN_DELTA
    ):

        best_macro_f1 = (
            val_macro_f1
        )


        best_epoch = (
            epoch
        )


        best_state = copy.deepcopy(
            model.state_dict()
        )


        epochs_without_improvement = 0


        best_checkpoint = {
            "epoch":
                epoch,

            "model_state_dict":
                copy.deepcopy(
                    model.state_dict()
                ),

            "optimizer_state_dict":
                copy.deepcopy(
                    optimizer.state_dict()
                ),

            "best_macro_f1":
                best_macro_f1,

            "val_accuracy":
                val_acc,

            "val_weighted_f1":
                val_weighted_f1,

            "class_names":
                CLASS_NAMES,

            "class_weights":
                class_weights.tolist(),

            "seed":
                SEED,

            "experiment":
                "E10_YOLO11s_Matched_MobileNetV3Large_Controlled_E3YB_Protocol",

            "crop_detector":
                "YOLO11s",

            "training_protocol":
                "E3Y-B"
        }


        torch.save(
            best_checkpoint,
            RESULTS_DIR
            /
            "E10_MobileNetV3Large_best.pth"
        )


        print(
            f"*** NEW BEST MODEL — "
            f"Macro F1 = "
            f"{best_macro_f1:.4f}"
        )


    else:

        epochs_without_improvement += 1


        print(
            f"No improvement "
            f"("
            f"{epochs_without_improvement}"
            f"/{PATIENCE}"
            f")"
        )


    # -------------------------------------------------------------------------
    # EARLY STOPPING
    #
    # COPY E3Y-B:
    #
    # patience = 7
    # -------------------------------------------------------------------------

    if (
        epochs_without_improvement
        >=
        PATIENCE
    ):

        print()

        print(
            "Early stopping triggered."
        )

        break


training_time = (
    time.time()
    -
    training_start
)


# =============================================================================
# 23. RESTORE BEST MODEL
# =============================================================================

print()

print("=" * 90)
print("RESTORING BEST E10 MODEL")
print("=" * 90)


if best_state is None:

    raise RuntimeError(
        "No best model was saved."
    )


model.load_state_dict(
    best_state
)


print(
    f"Best epoch    : "
    f"{best_epoch}"
)

print(
    f"Best Macro F1 : "
    f"{best_macro_f1:.4f}"
)

print()


# =============================================================================
# 24. FINAL VALIDATION
# =============================================================================

print("=" * 90)
print("FINAL E10 VALIDATION")
print("=" * 90)


(
    final_loss,
    final_accuracy,
    final_macro_f1,
    final_weighted_f1,
    final_targets,
    final_predictions

) = evaluate(
    model,
    val_loader,
    criterion,
    DEVICE
)


print(
    f"Accuracy       : "
    f"{final_accuracy:.4f}"
)

print(
    f"Accuracy (%)   : "
    f"{final_accuracy * 100:.2f}%"
)

print(
    f"Macro F1       : "
    f"{final_macro_f1:.4f}"
)

print(
    f"Weighted F1    : "
    f"{final_weighted_f1:.4f}"
)

print(
    f"Validation loss: "
    f"{final_loss:.4f}"
)

print(
    f"Samples        : "
    f"{len(final_targets)}"
)

print()


# =============================================================================
# 25. CLASSIFICATION REPORT
# =============================================================================

print("=" * 90)
print("CLASSIFICATION REPORT")
print("=" * 90)


report_dict = classification_report(
    final_targets,
    final_predictions,
    labels=list(
        range(
            NUM_CLASSES
        )
    ),
    target_names=CLASS_NAMES,
    output_dict=True,
    zero_division=0
)


report_text = classification_report(
    final_targets,
    final_predictions,
    labels=list(
        range(
            NUM_CLASSES
        )
    ),
    target_names=CLASS_NAMES,
    zero_division=0
)


print(
    report_text
)


report_df = pd.DataFrame(
    report_dict
).transpose()


report_df.to_csv(
    RESULTS_DIR
    /
    "classification_report.csv"
)


with open(
    RESULTS_DIR
    /
    "classification_report.txt",
    "w",
    encoding="utf-8"
) as f:

    f.write(
        report_text
    )


# =============================================================================
# 26. CONFUSION MATRIX
# =============================================================================

cm = confusion_matrix(
    final_targets,
    final_predictions,
    labels=list(
        range(
            NUM_CLASSES
        )
    )
)


cm_df = pd.DataFrame(
    cm,
    index=CLASS_NAMES,
    columns=CLASS_NAMES
)


cm_df.to_csv(
    RESULTS_DIR
    /
    "confusion_matrix.csv"
)


print()

print("=" * 90)
print("CONFUSION MATRIX")
print("=" * 90)


print(
    cm_df
)


# =============================================================================
# 27. PER-CLASS RESULTS
# =============================================================================

per_class_rows = []


for i, class_name in enumerate(
    CLASS_NAMES
):

    tp = cm[
        i,
        i
    ]


    actual = cm[
        i,
        :
    ].sum()


    predicted = cm[
        :,
        i
    ].sum()


    incorrect = (
        actual
        -
        tp
    )


    recall = (
        tp
        /
        actual

        if actual > 0

        else 0
    )


    precision = (
        tp
        /
        predicted

        if predicted > 0

        else 0
    )


    if (
        precision
        +
        recall
    ) > 0:

        f1 = (
            2
            *
            precision
            *
            recall
            /
            (
                precision
                +
                recall
            )
        )

    else:

        f1 = 0


    per_class_rows.append(
        {
            "class":
                class_name,

            "actual":
                int(
                    actual
                ),

            "correct":
                int(
                    tp
                ),

            "incorrect":
                int(
                    incorrect
                ),

            "precision":
                precision,

            "recall":
                recall,

            "f1":
                f1
        }
    )


per_class_df = pd.DataFrame(
    per_class_rows
)


per_class_df.to_csv(
    RESULTS_DIR
    /
    "per_class_results.csv",
    index=False
)


print()

print("=" * 90)
print("PER-CLASS RESULTS")
print("=" * 90)


print(
    per_class_df.to_string(
        index=False
    )
)


# =============================================================================
# 28. PLASTIC-ONLY RESULTS
# =============================================================================
#
# COPY E3Y-B methodology.
#
# GT non_plastic samples are excluded.
#
# A plastic GT predicted as non_plastic still counts as an error.
#
# =============================================================================

plastic_indices = [

    CLASS_TO_IDX[
        "ecal"
    ],

    CLASS_TO_IDX[
        "hdpe"
    ],

    CLASS_TO_IDX[
        "mixed_plastic_rigid"
    ],

    CLASS_TO_IDX[
        "mixed_plastic_soft"
    ],

    CLASS_TO_IDX[
        "pet"
    ],

    CLASS_TO_IDX[
        "pet_oil"
    ]
]


plastic_mask = np.isin(
    np.array(
        final_targets
    ),
    plastic_indices
)


plastic_targets = np.array(
    final_targets
)[
    plastic_mask
]


plastic_predictions = np.array(
    final_predictions
)[
    plastic_mask
]


plastic_accuracy = accuracy_score(
    plastic_targets,
    plastic_predictions
)


plastic_macro_f1 = f1_score(
    plastic_targets,
    plastic_predictions,
    labels=plastic_indices,
    average="macro",
    zero_division=0
)


plastic_weighted_f1 = f1_score(
    plastic_targets,
    plastic_predictions,
    labels=plastic_indices,
    average="weighted",
    zero_division=0
)


print()

print("=" * 90)
print("PLASTIC-ONLY CLASSIFICATION")
print("=" * 90)


print(
    f"Plastic samples     : "
    f"{len(plastic_targets)}"
)

print(
    f"Plastic accuracy    : "
    f"{plastic_accuracy:.4f}"
)

print(
    f"Plastic accuracy    : "
    f"{plastic_accuracy * 100:.2f}%"
)

print(
    f"Plastic Macro F1    : "
    f"{plastic_macro_f1:.4f}"
)

print(
    f"Plastic Weighted F1 : "
    f"{plastic_weighted_f1:.4f}"
)


# =============================================================================
# 29. SAVE TRAINING HISTORY
# =============================================================================

history_df = pd.DataFrame(
    history
)


history_df.to_csv(
    RESULTS_DIR
    /
    "training_history.csv",
    index=False
)


with open(
    RESULTS_DIR
    /
    "training_history.json",
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        history,
        f,
        indent=4
    )


# =============================================================================
# 30. TRAINING CURVES — LOSS
# =============================================================================

plt.figure(
    figsize=(
        8,
        5
    )
)


plt.plot(
    history_df[
        "epoch"
    ],
    history_df[
        "train_loss"
    ],
    label="Train Loss"
)


plt.plot(
    history_df[
        "epoch"
    ],
    history_df[
        "val_loss"
    ],
    label="Validation Loss"
)


plt.xlabel(
    "Epoch"
)

plt.ylabel(
    "Loss"
)


plt.title(
    "E10 MobileNet-V3-Large Loss"
)


plt.legend()


plt.grid(
    True,
    alpha=0.3
)


plt.tight_layout()


plt.savefig(
    RESULTS_DIR
    /
    "loss_curve.png",
    dpi=300
)


plt.close()


# =============================================================================
# 31. TRAINING CURVES — MACRO F1
# =============================================================================

plt.figure(
    figsize=(
        8,
        5
    )
)


plt.plot(
    history_df[
        "epoch"
    ],
    history_df[
        "train_macro_f1"
    ],
    label="Train Macro F1"
)


plt.plot(
    history_df[
        "epoch"
    ],
    history_df[
        "val_macro_f1"
    ],
    label="Validation Macro F1"
)


plt.axvline(
    best_epoch,
    linestyle="--",
    label=f"Best Epoch ({best_epoch})"
)


plt.xlabel(
    "Epoch"
)

plt.ylabel(
    "Macro F1"
)


plt.title(
    "E10 MobileNet-V3-Large Macro F1"
)


plt.legend()


plt.grid(
    True,
    alpha=0.3
)


plt.tight_layout()


plt.savefig(
    RESULTS_DIR
    /
    "macro_f1_curve.png",
    dpi=300
)


plt.close()


# =============================================================================
# 32. TRAINING CURVES — ACCURACY
# =============================================================================

plt.figure(
    figsize=(
        8,
        5
    )
)


plt.plot(
    history_df[
        "epoch"
    ],
    history_df[
        "train_accuracy"
    ],
    label="Train Accuracy"
)


plt.plot(
    history_df[
        "epoch"
    ],
    history_df[
        "val_accuracy"
    ],
    label="Validation Accuracy"
)


plt.xlabel(
    "Epoch"
)

plt.ylabel(
    "Accuracy"
)


plt.title(
    "E10 MobileNet-V3-Large Accuracy"
)


plt.legend()


plt.grid(
    True,
    alpha=0.3
)


plt.tight_layout()


plt.savefig(
    RESULTS_DIR
    /
    "accuracy_curve.png",
    dpi=300
)


plt.close()


# =============================================================================
# 33. LEARNING-RATE CURVE
# =============================================================================

plt.figure(
    figsize=(
        8,
        5
    )
)


plt.plot(
    history_df[
        "epoch"
    ],
    history_df[
        "learning_rate"
    ],
    marker="o"
)


plt.xlabel(
    "Epoch"
)

plt.ylabel(
    "Learning Rate"
)


plt.title(
    "E10 Learning Rate — ReduceLROnPlateau"
)


plt.grid(
    True,
    alpha=0.3
)


plt.tight_layout()


plt.savefig(
    RESULTS_DIR
    /
    "learning_rate_curve.png",
    dpi=300
)


plt.close()


# =============================================================================
# 34. SAVE FINAL CHECKPOINT
# =============================================================================

final_checkpoint = {

    "experiment":
        "E10_YOLO11s_Matched_MobileNetV3Large_Controlled_E3YB_Protocol",

    "title":
        "E10 — YOLO11s-Matched Crops + MobileNetV3-Large Using the Controlled E3Y-B Training Protocol",

    "model":
        "MobileNetV3-Large",

    "crop_detector":
        "YOLO11s",

    "crop_dataset":
        "yolo11s_mobilenet_crops_E9",

    "training_protocol":
        "E3Y-B",

    "num_classes":
        NUM_CLASSES,

    "class_names":
        CLASS_NAMES,

    "class_weights":
        class_weights.tolist(),

    "best_epoch":
        best_epoch,

    "best_macro_f1":
        best_macro_f1,

    "final_accuracy":
        final_accuracy,

    "final_macro_f1":
        final_macro_f1,

    "final_weighted_f1":
        final_weighted_f1,

    "plastic_accuracy":
        plastic_accuracy,

    "plastic_macro_f1":
        plastic_macro_f1,

    "plastic_weighted_f1":
        plastic_weighted_f1,

    "model_state_dict":
        model.state_dict(),

    "seed":
        SEED,

    "batch_size":
        BATCH_SIZE,

    "learning_rate":
        LEARNING_RATE,

    "weight_decay":
        WEIGHT_DECAY,

    "scheduler":
        "ReduceLROnPlateau(mode=max,factor=0.5,patience=2)",

    "early_stopping_patience":
        PATIENCE,

    "min_delta":
        MIN_DELTA,

    "random_resized_crop":
        False
}


torch.save(
    final_checkpoint,
    RESULTS_DIR
    /
    "E10_MobileNetV3Large_final.pth"
)


# =============================================================================
# 35. SAVE EXPERIMENT SUMMARY
# =============================================================================

summary = {

    "experiment":
        "E10 — YOLO11s-Matched Crops + MobileNetV3-Large Using the Controlled E3Y-B Training Protocol",

    "purpose":
        (
            "Controlled comparison with E3Y-B using existing YOLO11s-matched "
            "E9 crops while reproducing the E3Y-B MobileNet training protocol."
        ),

    "crop_detector":
        "YOLO11s",

    "crop_dataset":
        "Existing E9 YOLO11s matched crops",

    "classifier":
        "MobileNetV3-Large",

    "training_protocol":
        "E3Y-B",

    "train_samples":
        len(
            train_df
        ),

    "validation_samples":
        len(
            val_df
        ),

    "best_epoch":
        best_epoch,

    "best_validation_macro_f1":
        best_macro_f1,

    "validation_accuracy":
        final_accuracy,

    "validation_macro_f1":
        final_macro_f1,

    "validation_weighted_f1":
        final_weighted_f1,

    "plastic_samples":
        len(
            plastic_targets
        ),

    "plastic_accuracy":
        plastic_accuracy,

    "plastic_macro_f1":
        plastic_macro_f1,

    "plastic_weighted_f1":
        plastic_weighted_f1,

    "epochs_completed":
        len(
            history
        ),

    "training_time_minutes":
        training_time
        /
        60,

    "batch_size":
        BATCH_SIZE,

    "epochs_requested":
        NUM_EPOCHS,

    "learning_rate":
        LEARNING_RATE,

    "weight_decay":
        WEIGHT_DECAY,

    "optimizer":
        "AdamW",

    "scheduler":
        {
            "name":
                "ReduceLROnPlateau",

            "mode":
                "max",

            "factor":
                0.5,

            "patience":
                2
        },

    "best_model_metric":
        "validation_macro_f1",

    "early_stopping_patience":
        PATIENCE,

    "min_delta":
        MIN_DELTA,

    "augmentation":
        {
            "resize":
                [
                    224,
                    224
                ],

            "random_horizontal_flip":
                0.5,

            "random_rotation_degrees":
                10,

            "brightness":
                0.15,

            "contrast":
                0.15,

            "saturation":
                0.10,

            "hue":
                0.02,

            "random_resized_crop":
                False
        },

    "class_balancing":
        True,

    "class_balancing_method":
        "inverse_frequency_class_weighted_cross_entropy",

    "class_weights":
        {
            CLASS_NAMES[i]:
                float(
                    class_weights[i]
                )

            for i in range(
                NUM_CLASSES
            )
        },

    "oversampling":
        False,

    "smote":
        False,

    "focal_loss":
        False,

    "class_specific_augmentation":
        False,

    "seed":
        SEED
}


with open(
    RESULTS_DIR
    /
    "E10_experiment_summary.json",
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        summary,
        f,
        indent=4
    )


# =============================================================================
# 36. FINAL OUTPUT
# =============================================================================

print()

print("=" * 90)

print(
    "E10 — CONTROLLED MOBILENET TRAINING COMPLETE"
)

print("=" * 90)

print()


print(
    f"Best epoch               : "
    f"{best_epoch}"
)


print(
    f"Best Validation Macro F1 : "
    f"{best_macro_f1:.4f}"
)


print(
    f"Validation Accuracy      : "
    f"{final_accuracy:.4f} "
    f"({final_accuracy * 100:.2f}%)"
)


print(
    f"Validation Macro F1      : "
    f"{final_macro_f1:.4f}"
)


print(
    f"Validation Weighted F1   : "
    f"{final_weighted_f1:.4f}"
)


print()


print(
    f"Plastic Accuracy         : "
    f"{plastic_accuracy:.4f} "
    f"({plastic_accuracy * 100:.2f}%)"
)


print(
    f"Plastic Macro F1         : "
    f"{plastic_macro_f1:.4f}"
)


print(
    f"Plastic Weighted F1      : "
    f"{plastic_weighted_f1:.4f}"
)


print()


print(
    "Class weights:"
)


for class_name, weight in zip(
    CLASS_NAMES,
    class_weights
):

    print(
        f"  "
        f"{class_name:<25} "
        f"{weight:.6f}"
    )


print()

print(
    "Best checkpoint:"
)

print(
    RESULTS_DIR
    /
    "E10_MobileNetV3Large_best.pth"
)

print()


print(
    "Final checkpoint:"
)

print(
    RESULTS_DIR
    /
    "E10_MobileNetV3Large_final.pth"
)

print()


print(
    "Results directory:"
)

print(
    RESULTS_DIR
)

print()

print("=" * 90)

E10 — YOLO11s-MATCHED CROPS + MOBILENET-V3-LARGE
CONTROLLED E3Y-B TRAINING PROTOCOL

Device         : cuda
GPU            : NVIDIA GeForce RTX 3050 Ti Laptop GPU
CUDA           : 12.6
GPU memory     : 4.00 GB
Classes        : 7
Batch size     : 64
Epochs         : 30
Learning rate  : 0.0001
Weight decay   : 0.0001
Early patience : 7
Seed           : 42

EXPERIMENT DEFINITION

E10 = EXISTING E9 YOLO11s-MATCHED CROPS + EXACT E3Y-B MOBILENET TRAINING RECIPE

Crop-producing detector : YOLO11s
Crop dataset            : Existing E9 crops
Classifier              : MobileNetV3-Large
Class balancing         : Class-weighted CrossEntropyLoss
Best model criterion    : Validation Macro-F1
Scheduler               : ReduceLROnPlateau

RandomResizedCrop       : NO
Oversampling            : NO
SMOTE                   : NO
Focal loss              : NO
Class-specific augment. : NO

CHECKING E9 CROP DATASET
C:\Users\varda\Documents\_My Computer\COMMON_space\Learning\Upgrad_EPGP_MS\upgrad_IIITB_MS_course\

In [ ]:
# E10 END-TO-END EVALUATION
# YOLO11s + MobileNetV3-Large
# Controlled E3Y-B Training Protocol
# =============================================================================
#
# PIPELINE
# --------
#
# Validation image
#     ->
# E5 YOLO11s detector
#     ->
# detections using:
#       imgsz   = 640
#       conf    = 0.001
#       NMS IoU = 0.60
#       max_det = 100
#     ->
# crop every valid YOLO detection
#     ->
# E10 MobileNetV3-Large
#     ->
# final class = MobileNet predicted class
#     ->
# final confidence =
#       YOLO detector confidence
#       *
#       MobileNet predicted-class probability
#     ->
# 7-class COCO evaluation
#
#
# IMPORTANT
# ---------
#
# This evaluation is intended to be directly comparable with:
#
#   E8
#   E9-C
#
# Everything remains identical except the MobileNet checkpoint.
#
#
# PRIMARY METRICS
# ---------------
#
#   mAP50-95
#   mAP50
#   mAP75
#   AR
#
#
# SECONDARY DIAGNOSTICS
# ---------------------
#
# Using one-to-one GT/prediction matching at IoU >= 0.50:
#
#   classification accuracy
#   macro-F1
#   weighted-F1
#   plastic-only accuracy
#   plastic-only macro-F1
#   plastic-only weighted-F1
#
# =============================================================================


# =============================================================================
# 1. IMPORTS
# =============================================================================

import json
import math
import time

from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd

import torch
import torch.nn as nn

from torchvision import models, transforms

from PIL import Image

from ultralytics import YOLO

from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix
)


# =============================================================================
# 2. CONFIGURATION
# =============================================================================

SEED = 42


# -----------------------------------------------------------------------------
# Main roots
# -----------------------------------------------------------------------------

DATASET_ROOT = Path(
    r"C:\Users\varda\Documents\_My Computer\COMMON_space\Learning\Upgrad_EPGP_MS"
    r"\upgrad_IIITB_MS_course\MS_LJMU_Material\Topic Data\SortWaste\dataset\dataset"
)


THESIS_CODE_ROOT = Path(
    r"C:\Users\varda\Documents\_My Computer\COMMON_space\Learning\Upgrad_EPGP_MS"
    r"\upgrad_IIITB_MS_course\MS_LJMU_Material\Thesis_Code"
)


# -----------------------------------------------------------------------------
# Validation images / COCO GT
# -----------------------------------------------------------------------------

VAL_IMAGES_DIR = (
    DATASET_ROOT
    / "splited_all_dataset_coco"
    / "val"
    / "images"
)


VAL_COCO_JSON = (
    DATASET_ROOT
    / "splited_all_dataset_coco"
    / "val"
    / "annotations"
    / "val_coco.json"
)


# -----------------------------------------------------------------------------
# E5 detector
#
# SAME YOLO11s detector used by E7, E8 and E9-C.
# -----------------------------------------------------------------------------

YOLO_CHECKPOINT = (
    THESIS_CODE_ROOT
    / "runs"
    / "sortwaste"
    / "E5_yolo11s_7class_aug_classbalance_640"
    / "weights"
    / "best.pt"
)


# -----------------------------------------------------------------------------
# E10 MobileNet checkpoint
#
# IMPORTANT:
# Use BEST checkpoint, not final checkpoint.
# -----------------------------------------------------------------------------

MOBILENET_CHECKPOINT = (
    DATASET_ROOT
    / "yolo11s_mobilenet_crops_E9"
    / "mobilenet_results"
    / "E10_controlled_E3YB_protocol"
    / "E10_MobileNetV3Large_best.pth"
)


# -----------------------------------------------------------------------------
# Output
# -----------------------------------------------------------------------------

RESULTS_DIR = (
    THESIS_CODE_ROOT
    / "runs"
    / "sortwaste"
    / "E10_yolo11s_mobilenet_endtoend"
)


RESULTS_DIR.mkdir(
    parents=True,
    exist_ok=True
)


# =============================================================================
# 3. INFERENCE SETTINGS
# =============================================================================
#
# SAME AS E8 / E9-C.
#
# =============================================================================

IMAGE_SIZE = 224

YOLO_IMGSZ = 640

YOLO_CONF = 0.001

YOLO_NMS_IOU = 0.60

YOLO_MAX_DET = 100

YOLO_BATCH = 1

MATCH_IOU_THRESHOLD = 0.50


# =============================================================================
# 4. 7-CLASS TAXONOMY
# =============================================================================

CLASS_NAMES = [
    "ecal",
    "hdpe",
    "mixed_plastic_rigid",
    "mixed_plastic_soft",
    "non_plastic",
    "pet",
    "pet_oil"
]

NUM_CLASSES = len(
    CLASS_NAMES
)


# -----------------------------------------------------------------------------
# 7-class internal zero-based IDs:
#
# 0 ecal
# 1 hdpe
# 2 mixed_plastic_rigid
# 3 mixed_plastic_soft
# 4 non_plastic
# 5 pet
# 6 pet_oil
#
# COCO category IDs used for evaluation will be:
#
# 1 ecal
# 2 hdpe
# 3 mixed_plastic_rigid
# 4 mixed_plastic_soft
# 5 non_plastic
# 6 pet
# 7 pet_oil
#
# -----------------------------------------------------------------------------

IDX_TO_COCO_CATEGORY = {
    0: 1,
    1: 2,
    2: 3,
    3: 4,
    4: 5,
    5: 6,
    6: 7
}


COCO_CATEGORY_TO_IDX = {
    v: k
    for k, v in IDX_TO_COCO_CATEGORY.items()
}


# =============================================================================
# 5. ORIGINAL SORTWASTE COCO -> 7-CLASS EVALUATION MAPPING
# =============================================================================
#
# Original COCO:
#
# 1 pet
# 2 hdpe
# 3 mixed_plastic_soft
# 4 ecal
# 5 metal
# 6 cardboard
# 7 mixed_plastic_rigid
# 8 pet_oil
#
#
# New evaluation COCO:
#
# 1 ecal
# 2 hdpe
# 3 mixed_plastic_rigid
# 4 mixed_plastic_soft
# 5 non_plastic
# 6 pet
# 7 pet_oil
#
# =============================================================================

ORIGINAL_COCO_TO_EVAL_COCO = {
    1: 6,   # pet
    2: 2,   # hdpe
    3: 4,   # mixed soft
    4: 1,   # ecal
    5: 5,   # metal -> non_plastic
    6: 5,   # cardboard -> non_plastic
    7: 3,   # mixed rigid
    8: 7    # pet_oil
}


# =============================================================================
# 6. REPRODUCIBILITY
# =============================================================================

np.random.seed(
    SEED
)

torch.manual_seed(
    SEED
)

if torch.cuda.is_available():

    torch.cuda.manual_seed(
        SEED
    )

    torch.cuda.manual_seed_all(
        SEED
    )


torch.backends.cudnn.deterministic = True

torch.backends.cudnn.benchmark = False


# =============================================================================
# 7. DEVICE
# =============================================================================

DEVICE = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)


print("=" * 100)
print("E10 END-TO-END — YOLO11s + MobileNetV3-Large")
print("CONTROLLED E3Y-B TRAINING PROTOCOL")
print("=" * 100)

print()

print(
    f"Device                : {DEVICE}"
)

if torch.cuda.is_available():

    print(
        f"GPU                   : "
        f"{torch.cuda.get_device_name(0)}"
    )

    print(
        f"CUDA                  : "
        f"{torch.version.cuda}"
    )


print(
    f"YOLO checkpoint       : {YOLO_CHECKPOINT}"
)

print(
    f"MobileNet checkpoint  : {MOBILENET_CHECKPOINT}"
)

print(
    f"Validation images     : {VAL_IMAGES_DIR}"
)

print(
    f"COCO GT               : {VAL_COCO_JSON}"
)

print(
    f"Output                 : {RESULTS_DIR}"
)

print()


# =============================================================================
# 8. PATH CHECKS
# =============================================================================

print("=" * 100)
print("CHECKING PATHS")
print("=" * 100)


required_paths = [
    VAL_IMAGES_DIR,
    VAL_COCO_JSON,
    YOLO_CHECKPOINT,
    MOBILENET_CHECKPOINT
]


for path in required_paths:

    print(
        f"{str(path):<125} "
        f"Exists: {path.exists()}"
    )

    if not path.exists():

        raise FileNotFoundError(
            f"Required path does not exist:\n"
            f"{path}"
        )


print()


# =============================================================================
# 9. LOAD ORIGINAL COCO GT
# =============================================================================

print("=" * 100)
print("LOADING ORIGINAL COCO GROUND TRUTH")
print("=" * 100)


with open(
    VAL_COCO_JSON,
    "r",
    encoding="utf-8"
) as f:

    original_coco = json.load(
        f
    )


print(
    f"Images      : "
    f"{len(original_coco['images'])}"
)

print(
    f"Annotations : "
    f"{len(original_coco['annotations'])}"
)

print()


# =============================================================================
# 10. BUILD REMAPPED 7-CLASS COCO GT
# =============================================================================

print("=" * 100)
print("BUILDING 7-CLASS COCO GROUND TRUTH")
print("=" * 100)


eval_categories = [
    {
        "id": 1,
        "name": "ecal"
    },
    {
        "id": 2,
        "name": "hdpe"
    },
    {
        "id": 3,
        "name": "mixed_plastic_rigid"
    },
    {
        "id": 4,
        "name": "mixed_plastic_soft"
    },
    {
        "id": 5,
        "name": "non_plastic"
    },
    {
        "id": 6,
        "name": "pet"
    },
    {
        "id": 7,
        "name": "pet_oil"
    }
]


remapped_annotations = []


for ann in original_coco[
    "annotations"
]:

    original_category = int(
        ann[
            "category_id"
        ]
    )


    new_category = (
        ORIGINAL_COCO_TO_EVAL_COCO[
            original_category
        ]
    )


    new_ann = dict(
        ann
    )


    new_ann[
        "category_id"
    ] = new_category


    # Ensure fields COCOeval expects are present.

    new_ann[
        "iscrowd"
    ] = int(
        new_ann.get(
            "iscrowd",
            0
        )
    )


    if (
        "area"
        not in new_ann
        or
        new_ann[
            "area"
        ] is None
    ):

        x, y, w, h = new_ann[
            "bbox"
        ]

        new_ann[
            "area"
        ] = float(
            max(
                0.0,
                w
            )
            *
            max(
                0.0,
                h
            )
        )


    remapped_annotations.append(
        new_ann
    )


remapped_coco = {

    "info":
        original_coco.get(
            "info",
            {}
        ),

    "licenses":
        original_coco.get(
            "licenses",
            []
        ),

    "images":
        original_coco[
            "images"
        ],

    "annotations":
        remapped_annotations,

    "categories":
        eval_categories
}


REMAPPED_GT_JSON = (
    RESULTS_DIR
    / "E10_val_gt_7class.json"
)


with open(
    REMAPPED_GT_JSON,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        remapped_coco,
        f
    )


print(
    f"Remapped GT saved: "
    f"{REMAPPED_GT_JSON}"
)

print(
    f"Images             : "
    f"{len(remapped_coco['images'])}"
)

print(
    f"Annotations        : "
    f"{len(remapped_coco['annotations'])}"
)

print()


# =============================================================================
# 11. IMAGE LOOKUP
# =============================================================================

image_id_to_info = {
    int(
        img[
            "id"
        ]
    ):
    img

    for img in remapped_coco[
        "images"
    ]
}


filename_to_image_id = {
    Path(
        img[
            "file_name"
        ]
    ).name:
    int(
        img[
            "id"
        ]
    )

    for img in remapped_coco[
        "images"
    ]
}


print(
    f"COCO image IDs loaded : "
    f"{len(image_id_to_info)}"
)

print()


# =============================================================================
# 12. LOAD YOLO11s
# =============================================================================

print("=" * 100)
print("LOADING E5 YOLO11s DETECTOR")
print("=" * 100)


yolo_model = YOLO(
    str(
        YOLO_CHECKPOINT
    )
)


print(
    "YOLO11s loaded."
)

print()


# =============================================================================
# 13. LOAD E10 MOBILENET
# =============================================================================

print("=" * 100)
print("LOADING E10 MOBILENET-V3-LARGE")
print("=" * 100)


mobilenet = models.mobilenet_v3_large(
    weights=None
)


in_features = (
    mobilenet
    .classifier[-1]
    .in_features
)


mobilenet.classifier[-1] = nn.Linear(
    in_features,
    NUM_CLASSES
)


checkpoint = torch.load(
    MOBILENET_CHECKPOINT,
    map_location=DEVICE
)


if (
    isinstance(
        checkpoint,
        dict
    )
    and
    "model_state_dict"
    in checkpoint
):

    model_state_dict = checkpoint[
        "model_state_dict"
    ]

else:

    model_state_dict = checkpoint


mobilenet.load_state_dict(
    model_state_dict
)


mobilenet = mobilenet.to(
    DEVICE
)


mobilenet.eval()


print(
    "E10 MobileNet loaded."
)

if isinstance(
    checkpoint,
    dict
):

    print(
        f"Checkpoint epoch      : "
        f"{checkpoint.get('epoch', 'N/A')}"
    )

    print(
        f"Best Macro F1         : "
        f"{checkpoint.get('best_macro_f1', 'N/A')}"
    )

    print(
        f"Checkpoint experiment : "
        f"{checkpoint.get('experiment', 'N/A')}"
    )


print()


# =============================================================================
# 14. MOBILENET VALIDATION TRANSFORM
# =============================================================================
#
# SAME deterministic validation preprocessing as E3Y-B / E10 training.
#
# =============================================================================

mobilenet_transform = transforms.Compose(
    [

        transforms.Resize(
            (
                IMAGE_SIZE,
                IMAGE_SIZE
            )
        ),

        transforms.ToTensor(),

        transforms.Normalize(
            mean=[
                0.485,
                0.456,
                0.406
            ],

            std=[
                0.229,
                0.224,
                0.225
            ]
        )
    ]
)


# =============================================================================
# 15. HELPERS
# =============================================================================

def xyxy_to_xywh(
    box
):

    x1, y1, x2, y2 = [
        float(
            v
        )
        for v in box
    ]


    return [
        x1,
        y1,
        max(
            0.0,
            x2 - x1
        ),
        max(
            0.0,
            y2 - y1
        )
    ]


def xywh_to_xyxy(
    box
):

    x, y, w, h = [
        float(
            v
        )
        for v in box
    ]


    return [
        x,
        y,
        x + w,
        y + h
    ]


def calculate_iou(
    box_a,
    box_b
):
    """
    Boxes are xyxy.
    """

    ax1, ay1, ax2, ay2 = box_a

    bx1, by1, bx2, by2 = box_b


    inter_x1 = max(
        ax1,
        bx1
    )

    inter_y1 = max(
        ay1,
        by1
    )

    inter_x2 = min(
        ax2,
        bx2
    )

    inter_y2 = min(
        ay2,
        by2
    )


    inter_w = max(
        0.0,
        inter_x2 - inter_x1
    )

    inter_h = max(
        0.0,
        inter_y2 - inter_y1
    )


    intersection = (
        inter_w
        *
        inter_h
    )


    area_a = max(
        0.0,
        ax2 - ax1
    ) * max(
        0.0,
        ay2 - ay1
    )


    area_b = max(
        0.0,
        bx2 - bx1
    ) * max(
        0.0,
        by2 - by1
    )


    union = (
        area_a
        +
        area_b
        -
        intersection
    )


    if union <= 0:

        return 0.0


    return (
        intersection
        /
        union
    )


# =============================================================================
# 16. PREPARE GT FOR DIAGNOSTIC MATCHING
# =============================================================================

gt_by_image = defaultdict(
    list
)


for ann in remapped_coco[
    "annotations"
]:

    image_id = int(
        ann[
            "image_id"
        ]
    )


    gt_box = xywh_to_xyxy(
        ann[
            "bbox"
        ]
    )


    gt_category_id = int(
        ann[
            "category_id"
        ]
    )


    gt_by_image[
        image_id
    ].append(
        {
            "ann_id":
                int(
                    ann[
                        "id"
                    ]
                ),

            "bbox_xyxy":
                gt_box,

            "category_id":
                gt_category_id
        }
    )


# =============================================================================
# 17. RUN END-TO-END INFERENCE
# =============================================================================

print("=" * 100)
print("RUNNING E10 END-TO-END INFERENCE")
print("=" * 100)


coco_predictions = []

diagnostic_predictions_by_image = defaultdict(
    list
)


total_yolo_detections = 0

total_valid_crops = 0

invalid_crops = 0

missing_image_ids = 0


start_time = time.time()


# -----------------------------------------------------------------------------
# Use COCO image list to preserve exact validation-set membership.
# -----------------------------------------------------------------------------

num_images = len(
    remapped_coco[
        "images"
    ]
)


for image_number, image_info in enumerate(
    remapped_coco[
        "images"
    ],
    start=1
):

    image_id = int(
        image_info[
            "id"
        ]
    )


    file_name = image_info[
        "file_name"
    ]


    image_path = (
        VAL_IMAGES_DIR
        /
        Path(
            file_name
        ).name
    )


    if not image_path.exists():

        raise FileNotFoundError(
            f"Validation image not found:\n"
            f"{image_path}"
        )


    # -------------------------------------------------------------------------
    # Load original image
    # -------------------------------------------------------------------------

    pil_image = Image.open(
        image_path
    ).convert(
        "RGB"
    )


    image_width, image_height = (
        pil_image.size
    )


    # -------------------------------------------------------------------------
    # YOLO inference
    # -------------------------------------------------------------------------

    yolo_results = yolo_model.predict(
        source=str(
            image_path
        ),
        imgsz=YOLO_IMGSZ,
        conf=YOLO_CONF,
        iou=YOLO_NMS_IOU,
        max_det=YOLO_MAX_DET,
        device=0 if torch.cuda.is_available() else "cpu",
        verbose=False
    )


    if len(
        yolo_results
    ) == 0:

        continue


    result = yolo_results[
        0
    ]


    boxes = result.boxes


    if boxes is None:

        continue


    if len(
        boxes
    ) == 0:

        continue


    xyxy_boxes = (
        boxes.xyxy
        .detach()
        .cpu()
        .numpy()
    )


    yolo_confidences = (
        boxes.conf
        .detach()
        .cpu()
        .numpy()
    )


    total_yolo_detections += len(
        xyxy_boxes
    )


    # -------------------------------------------------------------------------
    # Create valid crops and retain detector metadata.
    #
    # MobileNet is then applied as a batch for this image.
    # -------------------------------------------------------------------------

    crop_tensors = []

    detection_metadata = []


    for det_index in range(
        len(
            xyxy_boxes
        )
    ):

        x1, y1, x2, y2 = [
            float(
                v
            )
            for v in xyxy_boxes[
                det_index
            ]
        ]


        # Clamp to image boundaries.

        x1 = max(
            0.0,
            min(
                x1,
                float(
                    image_width
                )
            )
        )

        y1 = max(
            0.0,
            min(
                y1,
                float(
                    image_height
                )
            )
        )

        x2 = max(
            0.0,
            min(
                x2,
                float(
                    image_width
                )
            )
        )

        y2 = max(
            0.0,
            min(
                y2,
                float(
                    image_height
                )
            )
        )


        left = int(
            math.floor(
                x1
            )
        )

        top = int(
            math.floor(
                y1
            )
        )

        right = int(
            math.ceil(
                x2
            )
        )

        bottom = int(
            math.ceil(
                y2
            )
        )


        left = max(
            0,
            min(
                left,
                image_width
            )
        )

        top = max(
            0,
            min(
                top,
                image_height
            )
        )

        right = max(
            0,
            min(
                right,
                image_width
            )
        )

        bottom = max(
            0,
            min(
                bottom,
                image_height
            )
        )


        if (
            right <= left
            or
            bottom <= top
        ):

            invalid_crops += 1

            continue


        crop = pil_image.crop(
            (
                left,
                top,
                right,
                bottom
            )
        )


        if (
            crop.width <= 0
            or
            crop.height <= 0
        ):

            invalid_crops += 1

            continue


        tensor = mobilenet_transform(
            crop
        )


        crop_tensors.append(
            tensor
        )


        detection_metadata.append(
            {
                "bbox_xyxy":
                    [
                        x1,
                        y1,
                        x2,
                        y2
                    ],

                "yolo_conf":
                    float(
                        yolo_confidences[
                            det_index
                        ]
                    )
            }
        )


    if len(
        crop_tensors
    ) == 0:

        continue


    total_valid_crops += len(
        crop_tensors
    )


    # -------------------------------------------------------------------------
    # MobileNet classification
    # -------------------------------------------------------------------------

    crop_batch = torch.stack(
        crop_tensors
    ).to(
        DEVICE
    )


    with torch.no_grad():

        logits = mobilenet(
            crop_batch
        )


        probabilities = torch.softmax(
            logits,
            dim=1
        )


        predicted_indices = probabilities.argmax(
            dim=1
        )


        predicted_probabilities = probabilities[
            torch.arange(
                probabilities.size(
                    0
                ),
                device=DEVICE
            ),
            predicted_indices
        ]


    predicted_indices = (
        predicted_indices
        .detach()
        .cpu()
        .numpy()
    )


    predicted_probabilities = (
        predicted_probabilities
        .detach()
        .cpu()
        .numpy()
    )


    # -------------------------------------------------------------------------
    # Build COCO predictions
    # -------------------------------------------------------------------------

    for local_index, metadata in enumerate(
        detection_metadata
    ):

        mobile_idx = int(
            predicted_indices[
                local_index
            ]
        )


        mobile_prob = float(
            predicted_probabilities[
                local_index
            ]
        )


        yolo_conf = float(
            metadata[
                "yolo_conf"
            ]
        )


        final_confidence = (
            yolo_conf
            *
            mobile_prob
        )


        coco_category_id = (
            IDX_TO_COCO_CATEGORY[
                mobile_idx
            ]
        )


        bbox_xyxy = metadata[
            "bbox_xyxy"
        ]


        bbox_xywh = xyxy_to_xywh(
            bbox_xyxy
        )


        prediction = {
            "image_id":
                image_id,

            "category_id":
                coco_category_id,

            "bbox":
                [
                    float(
                        v
                    )
                    for v in bbox_xywh
                ],

            "score":
                float(
                    final_confidence
                )
        }


        coco_predictions.append(
            prediction
        )


        diagnostic_predictions_by_image[
            image_id
        ].append(
            {
                "bbox_xyxy":
                    bbox_xyxy,

                "category_id":
                    coco_category_id,

                "score":
                    final_confidence,

                "yolo_conf":
                    yolo_conf,

                "mobilenet_prob":
                    mobile_prob
            }
        )


    # -------------------------------------------------------------------------
    # Progress
    # -------------------------------------------------------------------------

    if (
        image_number % 50 == 0
        or
        image_number == num_images
    ):

        elapsed = (
            time.time()
            -
            start_time
        )


        print(
            f"Processed "
            f"{image_number:>4}/{num_images} images | "
            f"YOLO dets: {total_yolo_detections:>6} | "
            f"valid crops: {total_valid_crops:>6} | "
            f"COCO preds: {len(coco_predictions):>6} | "
            f"invalid: {invalid_crops:>3} | "
            f"time: {elapsed / 60:.1f} min"
        )


inference_time = (
    time.time()
    -
    start_time
)


print()

print("=" * 100)
print("INFERENCE COMPLETE")
print("=" * 100)


print(
    f"Images                : "
    f"{num_images}"
)

print(
    f"YOLO detections       : "
    f"{total_yolo_detections}"
)

print(
    f"MobileNet valid crops : "
    f"{total_valid_crops}"
)

print(
    f"COCO predictions      : "
    f"{len(coco_predictions)}"
)

print(
    f"Invalid crops         : "
    f"{invalid_crops}"
)

print(
    f"Missing image IDs     : "
    f"{missing_image_ids}"
)

print(
    f"Inference time        : "
    f"{inference_time / 60:.2f} min"
)

print()


# =============================================================================
# 18. SAVE RAW COCO PREDICTIONS
# =============================================================================

PREDICTIONS_JSON = (
    RESULTS_DIR
    / "E10_coco_predictions.json"
)


with open(
    PREDICTIONS_JSON,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        coco_predictions,
        f
    )


print(
    f"Predictions saved: "
    f"{PREDICTIONS_JSON}"
)

print()


# =============================================================================
# 19. COCO EVALUATION
# =============================================================================

print("=" * 100)
print("COCO EVALUATION")
print("=" * 100)


coco_gt = COCO(
    str(
        REMAPPED_GT_JSON
    )
)


if len(
    coco_predictions
) == 0:

    raise RuntimeError(
        "No COCO predictions were generated."
    )


coco_dt = coco_gt.loadRes(
    str(
        PREDICTIONS_JSON
    )
)


coco_eval = COCOeval(
    coco_gt,
    coco_dt,
    "bbox"
)


# Same maxDets design as prior standardized evaluation.

coco_eval.params.maxDets = [
    1,
    10,
    100
]


coco_eval.evaluate()

coco_eval.accumulate()

coco_eval.summarize()


# =============================================================================
# 20. PRIMARY METRICS
# =============================================================================

stats = coco_eval.stats


map_50_95 = float(
    stats[
        0
    ]
)

map_50 = float(
    stats[
        1
    ]
)

map_75 = float(
    stats[
        2
    ]
)


ar_1 = float(
    stats[
        6
    ]
)

ar_10 = float(
    stats[
        7
    ]
)

ar_100 = float(
    stats[
        8
    ]
)


print()

print("=" * 100)
print("E10 PRIMARY RESULTS")
print("=" * 100)


print(
    f"mAP50-95 : "
    f"{map_50_95:.6f} "
    f"({map_50_95 * 100:.2f}%)"
)

print(
    f"mAP50    : "
    f"{map_50:.6f} "
    f"({map_50 * 100:.2f}%)"
)

print(
    f"mAP75    : "
    f"{map_75:.6f} "
    f"({map_75 * 100:.2f}%)"
)

print(
    f"AR1      : "
    f"{ar_1:.6f}"
)

print(
    f"AR10     : "
    f"{ar_10:.6f}"
)

print(
    f"AR100    : "
    f"{ar_100:.6f}"
)

print()


# =============================================================================
# 21. PER-CLASS COCO AP
# =============================================================================
#
# COCO precision tensor:
#
# precision[
#     IoU,
#     recall,
#     class,
#     area,
#     maxDet
# ]
#
# =============================================================================

print("=" * 100)
print("PER-CLASS COCO AP")
print("=" * 100)


precision = (
    coco_eval.eval[
        "precision"
    ]
)


iou_thresholds = (
    coco_eval.params.iouThrs
)


# Area index 0 = all
# maxDet index 2 = maxDet 100

area_index = 0

maxdet_index = 2


iou50_index = int(
    np.argmin(
        np.abs(
            iou_thresholds
            -
            0.50
        )
    )
)


per_class_detection_rows = []


for class_index, class_name in enumerate(
    CLASS_NAMES
):

    # -------------------------------------------------------------------------
    # AP50-95
    # -------------------------------------------------------------------------

    class_precision_all = precision[
        :,
        :,
        class_index,
        area_index,
        maxdet_index
    ]


    valid_all = class_precision_all[
        class_precision_all
        >
        -1
    ]


    if len(
        valid_all
    ) > 0:

        class_ap = float(
            np.mean(
                valid_all
            )
        )

    else:

        class_ap = float(
            "nan"
        )


    # -------------------------------------------------------------------------
    # AP50
    # -------------------------------------------------------------------------

    class_precision_50 = precision[
        iou50_index,
        :,
        class_index,
        area_index,
        maxdet_index
    ]


    valid_50 = class_precision_50[
        class_precision_50
        >
        -1
    ]


    if len(
        valid_50
    ) > 0:

        class_ap50 = float(
            np.mean(
                valid_50
            )
        )

    else:

        class_ap50 = float(
            "nan"
        )


    per_class_detection_rows.append(
        {
            "class":
                class_name,

            "AP50":
                class_ap50,

            "AP50_95":
                class_ap
        }
    )


per_class_detection_df = pd.DataFrame(
    per_class_detection_rows
)


print(
    per_class_detection_df.to_string(
        index=False
    )
)


per_class_detection_df.to_csv(
    RESULTS_DIR
    / "E10_per_class_detection_AP.csv",
    index=False
)


print()


# =============================================================================
# 22. ONE-TO-ONE GT/PREDICTION MATCHING @ IoU >= 0.50
# =============================================================================
#
# SECONDARY DIAGNOSTIC ONLY.
#
# Matching is GEOMETRY ONLY.
#
# Class is NOT used to determine whether a prediction matches a GT.
#
# This avoids falsely inflating classification accuracy.
#
# Greedy strategy:
#
#   1. Generate all GT/pred pairs with IoU >= 0.50.
#   2. Sort pairs by IoU descending.
#   3. Match each GT and prediction at most once.
#
# =============================================================================

print("=" * 100)
print("DIAGNOSTIC MATCHING @ IoU >= 0.50")
print("=" * 100)


matched_true_indices = []

matched_pred_indices = []


match_rows = []


total_matches = 0


for image_id in image_id_to_info:

    gts = gt_by_image[
        image_id
    ]


    preds = diagnostic_predictions_by_image[
        image_id
    ]


    candidate_pairs = []


    for gt_index, gt in enumerate(
        gts
    ):

        for pred_index, pred in enumerate(
            preds
        ):

            iou = calculate_iou(
                gt[
                    "bbox_xyxy"
                ],
                pred[
                    "bbox_xyxy"
                ]
            )


            if iou >= MATCH_IOU_THRESHOLD:

                candidate_pairs.append(
                    (
                        iou,
                        gt_index,
                        pred_index
                    )
                )


    candidate_pairs.sort(
        key=lambda x: x[
            0
        ],
        reverse=True
    )


    used_gt = set()

    used_pred = set()


    for iou, gt_index, pred_index in candidate_pairs:

        if gt_index in used_gt:

            continue


        if pred_index in used_pred:

            continue


        used_gt.add(
            gt_index
        )

        used_pred.add(
            pred_index
        )


        gt = gts[
            gt_index
        ]


        pred = preds[
            pred_index
        ]


        gt_category = int(
            gt[
                "category_id"
            ]
        )


        pred_category = int(
            pred[
                "category_id"
            ]
        )


        gt_idx = (
            COCO_CATEGORY_TO_IDX[
                gt_category
            ]
        )


        pred_idx = (
            COCO_CATEGORY_TO_IDX[
                pred_category
            ]
        )


        matched_true_indices.append(
            gt_idx
        )


        matched_pred_indices.append(
            pred_idx
        )


        total_matches += 1


        match_rows.append(
            {
                "image_id":
                    image_id,

                "gt_annotation_id":
                    gt[
                        "ann_id"
                    ],

                "iou":
                    float(
                        iou
                    ),

                "gt_class":
                    CLASS_NAMES[
                        gt_idx
                    ],

                "predicted_class":
                    CLASS_NAMES[
                        pred_idx
                    ],

                "correct_class":
                    bool(
                        gt_idx
                        ==
                        pred_idx
                    ),

                "final_score":
                    float(
                        pred[
                            "score"
                        ]
                    ),

                "yolo_conf":
                    float(
                        pred[
                            "yolo_conf"
                        ]
                    ),

                "mobilenet_prob":
                    float(
                        pred[
                            "mobilenet_prob"
                        ]
                    )
            }
        )


matched_true_indices = np.array(
    matched_true_indices,
    dtype=np.int64
)


matched_pred_indices = np.array(
    matched_pred_indices,
    dtype=np.int64
)


matches_df = pd.DataFrame(
    match_rows
)


matches_df.to_csv(
    RESULTS_DIR
    / "E10_matched_predictions_iou50.csv",
    index=False
)


print(
    f"GT/pred matches : "
    f"{total_matches}"
)

print()


# =============================================================================
# 23. CLASSIFICATION DIAGNOSTICS
# =============================================================================

if total_matches == 0:

    raise RuntimeError(
        "No GT/prediction matches were found at IoU >= 0.50."
    )


matched_accuracy = accuracy_score(
    matched_true_indices,
    matched_pred_indices
)


matched_macro_f1 = f1_score(
    matched_true_indices,
    matched_pred_indices,
    labels=list(
        range(
            NUM_CLASSES
        )
    ),
    average="macro",
    zero_division=0
)


matched_weighted_f1 = f1_score(
    matched_true_indices,
    matched_pred_indices,
    labels=list(
        range(
            NUM_CLASSES
        )
    ),
    average="weighted",
    zero_division=0
)


correct_matches = int(
    np.sum(
        matched_true_indices
        ==
        matched_pred_indices
    )
)


print("=" * 100)
print("MATCHED CLASSIFICATION RESULTS")
print("=" * 100)


print(
    f"Matches             : "
    f"{total_matches}"
)

print(
    f"Correct classes     : "
    f"{correct_matches}"
)

print(
    f"Accuracy            : "
    f"{matched_accuracy:.6f} "
    f"({matched_accuracy * 100:.2f}%)"
)

print(
    f"Macro F1            : "
    f"{matched_macro_f1:.6f} "
    f"({matched_macro_f1 * 100:.2f}%)"
)

print(
    f"Weighted F1         : "
    f"{matched_weighted_f1:.6f} "
    f"({matched_weighted_f1 * 100:.2f}%)"
)

print()


# =============================================================================
# 24. CLASSIFICATION REPORT
# =============================================================================

matched_report_text = classification_report(
    matched_true_indices,
    matched_pred_indices,
    labels=list(
        range(
            NUM_CLASSES
        )
    ),
    target_names=CLASS_NAMES,
    zero_division=0
)


matched_report_dict = classification_report(
    matched_true_indices,
    matched_pred_indices,
    labels=list(
        range(
            NUM_CLASSES
        )
    ),
    target_names=CLASS_NAMES,
    output_dict=True,
    zero_division=0
)


print(
    matched_report_text
)


with open(
    RESULTS_DIR
    / "E10_matched_classification_report.txt",
    "w",
    encoding="utf-8"
) as f:

    f.write(
        matched_report_text
    )


pd.DataFrame(
    matched_report_dict
).transpose().to_csv(
    RESULTS_DIR
    / "E10_matched_classification_report.csv"
)


# =============================================================================
# 25. CONFUSION MATRIX
# =============================================================================

cm = confusion_matrix(
    matched_true_indices,
    matched_pred_indices,
    labels=list(
        range(
            NUM_CLASSES
        )
    )
)


cm_df = pd.DataFrame(
    cm,
    index=CLASS_NAMES,
    columns=CLASS_NAMES
)


cm_df.to_csv(
    RESULTS_DIR
    / "E10_matched_confusion_matrix.csv"
)


print("=" * 100)
print("MATCHED CONFUSION MATRIX")
print("=" * 100)


print(
    cm_df
)

print()


# =============================================================================
# 26. PLASTIC-ONLY DIAGNOSTICS
# =============================================================================
#
# Plastic GT classes:
#
# ecal
# hdpe
# mixed_plastic_rigid
# mixed_plastic_soft
# pet
# pet_oil
#
# non_plastic GT objects are excluded.
#
# A plastic GT predicted as non_plastic remains an ERROR.
#
# =============================================================================

plastic_indices = [
    0,  # ecal
    1,  # hdpe
    2,  # mixed rigid
    3,  # mixed soft
    5,  # pet
    6   # pet oil
]


plastic_mask = np.isin(
    matched_true_indices,
    plastic_indices
)


plastic_true = matched_true_indices[
    plastic_mask
]


plastic_pred = matched_pred_indices[
    plastic_mask
]


plastic_accuracy = accuracy_score(
    plastic_true,
    plastic_pred
)


plastic_macro_f1 = f1_score(
    plastic_true,
    plastic_pred,
    labels=plastic_indices,
    average="macro",
    zero_division=0
)


plastic_weighted_f1 = f1_score(
    plastic_true,
    plastic_pred,
    labels=plastic_indices,
    average="weighted",
    zero_division=0
)


print("=" * 100)
print("PLASTIC-ONLY MATCHED CLASSIFICATION")
print("=" * 100)


print(
    f"Plastic samples      : "
    f"{len(plastic_true)}"
)

print(
    f"Plastic accuracy     : "
    f"{plastic_accuracy:.6f} "
    f"({plastic_accuracy * 100:.2f}%)"
)

print(
    f"Plastic Macro F1     : "
    f"{plastic_macro_f1:.6f} "
    f"({plastic_macro_f1 * 100:.2f}%)"
)

print(
    f"Plastic Weighted F1  : "
    f"{plastic_weighted_f1:.6f} "
    f"({plastic_weighted_f1 * 100:.2f}%)"
)

print()


# =============================================================================
# 27. SAVE FINAL SUMMARY
# =============================================================================

summary = {

    "experiment":
        "E10 End-to-End — YOLO11s + MobileNetV3-Large Controlled E3Y-B Protocol",

    "detector":
        "E5 YOLO11s",

    "detector_checkpoint":
        str(
            YOLO_CHECKPOINT
        ),

    "mobilenet_checkpoint":
        str(
            MOBILENET_CHECKPOINT
        ),

    "mobilenet_training_protocol":
        "Controlled E3Y-B",

    "validation_images":
        num_images,

    "ground_truth_annotations":
        len(
            remapped_coco[
                "annotations"
            ]
        ),

    "yolo_detections":
        total_yolo_detections,

    "mobilenet_valid_crops":
        total_valid_crops,

    "coco_predictions":
        len(
            coco_predictions
        ),

    "invalid_crops":
        invalid_crops,

    "inference": {

        "imgsz":
            YOLO_IMGSZ,

        "conf":
            YOLO_CONF,

        "nms_iou":
            YOLO_NMS_IOU,

        "max_det":
            YOLO_MAX_DET,

        "final_score":
            "yolo_confidence * mobilenet_predicted_class_probability"
    },

    "primary_coco_metrics": {

        "mAP50_95":
            map_50_95,

        "mAP50":
            map_50,

        "mAP75":
            map_75,

        "AR1":
            ar_1,

        "AR10":
            ar_10,

        "AR100":
            ar_100
    },

    "matched_classification": {

        "iou_threshold":
            MATCH_IOU_THRESHOLD,

        "matches":
            total_matches,

        "correct":
            correct_matches,

        "accuracy":
            matched_accuracy,

        "macro_f1":
            matched_macro_f1,

        "weighted_f1":
            matched_weighted_f1
    },

    "plastic_only": {

        "samples":
            int(
                len(
                    plastic_true
                )
            ),

        "accuracy":
            plastic_accuracy,

        "macro_f1":
            plastic_macro_f1,

        "weighted_f1":
            plastic_weighted_f1
    }
}


SUMMARY_JSON = (
    RESULTS_DIR
    / "E10_endtoend_summary.json"
)


with open(
    SUMMARY_JSON,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        summary,
        f,
        indent=4
    )


# =============================================================================
# 28. SAVE COMPACT METRICS CSV
# =============================================================================

metrics_df = pd.DataFrame(
    [
        {
            "experiment":
                "E10",

            "mAP50_95":
                map_50_95,

            "mAP50":
                map_50,

            "mAP75":
                map_75,

            "AR1":
                ar_1,

            "AR10":
                ar_10,

            "AR100":
                ar_100,

            "matched_accuracy":
                matched_accuracy,

            "matched_macro_f1":
                matched_macro_f1,

            "matched_weighted_f1":
                matched_weighted_f1,

            "plastic_accuracy":
                plastic_accuracy,

            "plastic_macro_f1":
                plastic_macro_f1,

            "plastic_weighted_f1":
                plastic_weighted_f1
        }
    ]
)


metrics_df.to_csv(
    RESULTS_DIR
    / "E10_final_metrics.csv",
    index=False
)


# =============================================================================
# 29. FINAL OUTPUT
# =============================================================================

print()

print("=" * 100)
print("E10 END-TO-END EVALUATION COMPLETE")
print("=" * 100)

print()

print("PRIMARY COCO METRICS")
print("--------------------")

print(
    f"mAP50-95 : "
    f"{map_50_95:.6f} "
    f"({map_50_95 * 100:.2f}%)"
)

print(
    f"mAP50    : "
    f"{map_50:.6f} "
    f"({map_50 * 100:.2f}%)"
)

print(
    f"mAP75    : "
    f"{map_75:.6f} "
    f"({map_75 * 100:.2f}%)"
)

print(
    f"AR1      : "
    f"{ar_1:.6f}"
)

print(
    f"AR10     : "
    f"{ar_10:.6f}"
)

print(
    f"AR100    : "
    f"{ar_100:.6f}"
)

print()

print("MATCHED CLASSIFICATION")
print("----------------------")

print(
    f"Matches             : "
    f"{total_matches}"
)

print(
    f"Accuracy            : "
    f"{matched_accuracy * 100:.2f}%"
)

print(
    f"Macro F1            : "
    f"{matched_macro_f1 * 100:.2f}%"
)

print(
    f"Weighted F1         : "
    f"{matched_weighted_f1 * 100:.2f}%"
)

print()

print("PLASTIC-ONLY")
print("------------")

print(
    f"Samples             : "
    f"{len(plastic_true)}"
)

print(
    f"Accuracy            : "
    f"{plastic_accuracy * 100:.2f}%"
)

print(
    f"Macro F1            : "
    f"{plastic_macro_f1 * 100:.2f}%"
)

print(
    f"Weighted F1         : "
    f"{plastic_weighted_f1 * 100:.2f}%"
)

print()

print(
    "Per-class AP:"
)

print(
    per_class_detection_df.to_string(
        index=False
    )
)

print()

print(
    f"Results directory:\n"
    f"{RESULTS_DIR}"
)

print()

print("=" * 100)

E10 END-TO-END — YOLO11s + MobileNetV3-Large
CONTROLLED E3Y-B TRAINING PROTOCOL

Device                : cuda
GPU                   : NVIDIA GeForce RTX 3050 Ti Laptop GPU
CUDA                  : 12.6
YOLO checkpoint       : C:\Users\varda\Documents\_My Computer\COMMON_space\Learning\Upgrad_EPGP_MS\upgrad_IIITB_MS_course\MS_LJMU_Material\Thesis_Code\runs\sortwaste\E5_yolo11s_7class_aug_classbalance_640\weights\best.pt
MobileNet checkpoint  : C:\Users\varda\Documents\_My Computer\COMMON_space\Learning\Upgrad_EPGP_MS\upgrad_IIITB_MS_course\MS_LJMU_Material\Topic Data\SortWaste\dataset\dataset\yolo11s_mobilenet_crops_E9\mobilenet_results\E10_controlled_E3YB_protocol\E10_MobileNetV3Large_best.pth
Validation images     : C:\Users\varda\Documents\_My Computer\COMMON_space\Learning\Upgrad_EPGP_MS\upgrad_IIITB_MS_course\MS_LJMU_Material\Topic Data\SortWaste\dataset\dataset\splited_all_dataset_coco\val\images
COCO GT               : C:\Users\varda\Documents\_My Computer\COMMON_space\Learning\Up

# Model E11 : Investigation/evaluation tests

## E11 A : classification-oracle evaluation on your existing E8/E7 detections

same YOLO11s bounding boxes+same YOLO confidence+perfect GT class for geometrically matched detections

In [ ]:
# E11 - A — CLASSIFICATION-ORACLE ANALYSIS OF YOLO11s DETECTIONS
# =============================================================================
#
# PURPOSE
# -------
# Quantify how much of the remaining AP limitation is due to classification
# versus detection/localization.
#
#
# EXPERIMENT
# ----------
#
# Frozen E5 YOLO11s detector
#
# Validation inference:
#
#   imgsz   = 640
#   conf    = 0.001
#   NMS IoU = 0.60
#   max_det = 100
#
#
# BASELINE:
#   YOLO bbox
#   YOLO predicted class
#   YOLO confidence
#
#
# E11 CLASSIFICATION ORACLE:
#
#   1. Run YOLO normally.
#   2. Geometry-match predictions to GT one-to-one at IoU >= 0.50.
#   3. For every matched prediction:
#          predicted class <- GT class
#   4. Keep:
#          bounding box unchanged
#          YOLO confidence unchanged
#   5. Unmatched predictions remain exactly as YOLO predicted.
#   6. Evaluate using COCOeval.
#
#
# INTERPRETATION
# --------------
#
# If E11 AP is MUCH higher than E7:
#     classification is a major bottleneck.
#
# If E11 AP is only slightly higher than E7:
#     bounding-box localization / false positives / missed detections dominate.
#
#
# IMPORTANT
# ---------
#
# This is an ORACLE DIAGNOSTIC.
# It must NOT be reported as a deployable model result.
#
# =============================================================================


# =============================================================================
# 1. IMPORTS
# =============================================================================

import json
import time
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd

import torch

from ultralytics import YOLO

from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval


# =============================================================================
# 2. CONFIGURATION
# =============================================================================

SEED = 42


DATASET_ROOT = Path(
    r"C:\Users\varda\Documents\_My Computer\COMMON_space\Learning"
    r"\Upgrad_EPGP_MS\upgrad_IIITB_MS_course\MS_LJMU_Material"
    r"\Topic Data\SortWaste\dataset\dataset"
)


THESIS_CODE_ROOT = Path(
    r"C:\Users\varda\Documents\_My Computer\COMMON_space\Learning"
    r"\Upgrad_EPGP_MS\upgrad_IIITB_MS_course\MS_LJMU_Material"
    r"\Thesis_Code"
)


VAL_IMAGES_DIR = (
    DATASET_ROOT
    / "splited_all_dataset_coco"
    / "val"
    / "images"
)


VAL_COCO_JSON = (
    DATASET_ROOT
    / "splited_all_dataset_coco"
    / "val"
    / "annotations"
    / "val_coco.json"
)


YOLO_CHECKPOINT = (
    THESIS_CODE_ROOT
    / "runs"
    / "sortwaste"
    / "E5_yolo11s_7class_aug_classbalance_640"
    / "weights"
    / "best.pt"
)


RESULTS_DIR = (
    THESIS_CODE_ROOT
    / "runs"
    / "sortwaste"
    / "E11_classification_oracle"
)


RESULTS_DIR.mkdir(
    parents=True,
    exist_ok=True
)


# =============================================================================
# 3. SAME FINAL DETECTOR INFERENCE SETTINGS AS E7/E8/E10
# =============================================================================

YOLO_IMGSZ = 640
YOLO_CONF = 0.001
YOLO_NMS_IOU = 0.60
YOLO_MAX_DET = 100

MATCH_IOU_THRESHOLD = 0.50


# =============================================================================
# 4. 7-CLASS TAXONOMY
# =============================================================================

CLASS_NAMES = [
    "ecal",
    "hdpe",
    "mixed_plastic_rigid",
    "mixed_plastic_soft",
    "non_plastic",
    "pet",
    "pet_oil"
]


NUM_CLASSES = len(CLASS_NAMES)


# YOLO zero-based class -> evaluation COCO one-based category ID

YOLO_IDX_TO_EVAL_COCO = {
    0: 1,   # ecal
    1: 2,   # hdpe
    2: 3,   # mixed rigid
    3: 4,   # mixed soft
    4: 5,   # non plastic
    5: 6,   # pet
    6: 7    # pet oil
}


# Original SortWaste COCO:
#
# 1 pet
# 2 hdpe
# 3 mixed_plastic_soft
# 4 ecal
# 5 metal
# 6 cardboard
# 7 mixed_plastic_rigid
# 8 pet_oil
#
# Evaluation taxonomy:
#
# 1 ecal
# 2 hdpe
# 3 mixed_plastic_rigid
# 4 mixed_plastic_soft
# 5 non_plastic
# 6 pet
# 7 pet_oil

ORIGINAL_COCO_TO_EVAL_COCO = {
    1: 6,
    2: 2,
    3: 4,
    4: 1,
    5: 5,
    6: 5,
    7: 3,
    8: 7
}


# =============================================================================
# 5. REPRODUCIBILITY
# =============================================================================

np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():

    torch.cuda.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)


torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False


DEVICE = (
    0
    if torch.cuda.is_available()
    else "cpu"
)


# =============================================================================
# 6. INTRO
# =============================================================================

print("=" * 100)
print("E11 — CLASSIFICATION-ORACLE ANALYSIS OF YOLO11s DETECTIONS")
print("=" * 100)

print()

print(
    "Purpose : Estimate the maximum AP obtainable by fixing classification\n"
    "          while leaving YOLO11s bounding boxes and detector scores unchanged."
)

print()

print(f"YOLO checkpoint : {YOLO_CHECKPOINT}")
print(f"Validation data : {VAL_IMAGES_DIR}")
print(f"COCO GT         : {VAL_COCO_JSON}")
print(f"Results         : {RESULTS_DIR}")

print()

print(f"imgsz           : {YOLO_IMGSZ}")
print(f"confidence      : {YOLO_CONF}")
print(f"NMS IoU         : {YOLO_NMS_IOU}")
print(f"max_det         : {YOLO_MAX_DET}")
print(f"Oracle match IoU: {MATCH_IOU_THRESHOLD}")

print()


# =============================================================================
# 7. PATH CHECKS
# =============================================================================

print("=" * 100)
print("CHECKING PATHS")
print("=" * 100)


for path in [
    VAL_IMAGES_DIR,
    VAL_COCO_JSON,
    YOLO_CHECKPOINT
]:

    print(
        f"{str(path):<120} Exists: {path.exists()}"
    )

    if not path.exists():

        raise FileNotFoundError(
            f"Missing required path:\n{path}"
        )


print()


# =============================================================================
# 8. LOAD AND REMAP GT TO 7 CLASSES
# =============================================================================

print("=" * 100)
print("BUILDING 7-CLASS GROUND TRUTH")
print("=" * 100)


with open(
    VAL_COCO_JSON,
    "r",
    encoding="utf-8"
) as f:

    original_coco = json.load(f)


eval_categories = [
    {"id": 1, "name": "ecal"},
    {"id": 2, "name": "hdpe"},
    {"id": 3, "name": "mixed_plastic_rigid"},
    {"id": 4, "name": "mixed_plastic_soft"},
    {"id": 5, "name": "non_plastic"},
    {"id": 6, "name": "pet"},
    {"id": 7, "name": "pet_oil"}
]


remapped_annotations = []


for ann in original_coco["annotations"]:

    new_ann = dict(ann)

    new_ann["category_id"] = (
        ORIGINAL_COCO_TO_EVAL_COCO[
            int(ann["category_id"])
        ]
    )

    new_ann["iscrowd"] = int(
        new_ann.get(
            "iscrowd",
            0
        )
    )

    if (
        "area" not in new_ann
        or new_ann["area"] is None
    ):

        x, y, w, h = new_ann["bbox"]

        new_ann["area"] = float(
            max(w, 0.0)
            *
            max(h, 0.0)
        )

    remapped_annotations.append(
        new_ann
    )


remapped_coco = {
    "info": original_coco.get("info", {}),
    "licenses": original_coco.get("licenses", []),
    "images": original_coco["images"],
    "annotations": remapped_annotations,
    "categories": eval_categories
}


REMAPPED_GT_JSON = (
    RESULTS_DIR
    / "E11_val_gt_7class.json"
)


with open(
    REMAPPED_GT_JSON,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        remapped_coco,
        f
    )


print(
    f"Images      : {len(remapped_coco['images'])}"
)

print(
    f"Annotations : {len(remapped_coco['annotations'])}"
)

print(
    f"Saved GT    : {REMAPPED_GT_JSON}"
)

print()


# =============================================================================
# 9. HELPERS
# =============================================================================

def xywh_to_xyxy(box):

    x, y, w, h = [
        float(v)
        for v in box
    ]

    return [
        x,
        y,
        x + w,
        y + h
    ]


def xyxy_to_xywh(box):

    x1, y1, x2, y2 = [
        float(v)
        for v in box
    ]

    return [
        x1,
        y1,
        max(0.0, x2 - x1),
        max(0.0, y2 - y1)
    ]


def calculate_iou(
    box_a,
    box_b
):

    ax1, ay1, ax2, ay2 = box_a
    bx1, by1, bx2, by2 = box_b


    inter_x1 = max(ax1, bx1)
    inter_y1 = max(ay1, by1)

    inter_x2 = min(ax2, bx2)
    inter_y2 = min(ay2, by2)


    inter_w = max(
        0.0,
        inter_x2 - inter_x1
    )

    inter_h = max(
        0.0,
        inter_y2 - inter_y1
    )


    intersection = (
        inter_w
        *
        inter_h
    )


    area_a = (
        max(
            0.0,
            ax2 - ax1
        )
        *
        max(
            0.0,
            ay2 - ay1
        )
    )


    area_b = (
        max(
            0.0,
            bx2 - bx1
        )
        *
        max(
            0.0,
            by2 - by1
        )
    )


    union = (
        area_a
        +
        area_b
        -
        intersection
    )


    if union <= 0:

        return 0.0


    return (
        intersection
        /
        union
    )


# =============================================================================
# 10. BUILD GT LOOKUP FOR GEOMETRY MATCHING
# =============================================================================

gt_by_image = defaultdict(list)


for ann in remapped_coco["annotations"]:

    gt_by_image[
        int(ann["image_id"])
    ].append(
        {
            "ann_id":
                int(ann["id"]),

            "category_id":
                int(ann["category_id"]),

            "bbox_xyxy":
                xywh_to_xyxy(
                    ann["bbox"]
                )
        }
    )


# =============================================================================
# 11. LOAD YOLO
# =============================================================================

print("=" * 100)
print("LOADING E5 YOLO11s")
print("=" * 100)


model = YOLO(
    str(YOLO_CHECKPOINT)
)


print("YOLO11s loaded.")
print()


# =============================================================================
# 12. RUN DETECTOR
# =============================================================================

print("=" * 100)
print("RUNNING YOLO11s AND BUILDING E11 ORACLE PREDICTIONS")
print("=" * 100)


baseline_predictions = []
oracle_predictions = []

oracle_match_rows = []


total_detections = 0

total_geometry_matches = 0

total_class_corrections = 0

originally_correct_matches = 0


start_time = time.time()


num_images = len(
    remapped_coco["images"]
)


for image_number, image_info in enumerate(
    remapped_coco["images"],
    start=1
):

    image_id = int(
        image_info["id"]
    )


    image_path = (
        VAL_IMAGES_DIR
        /
        Path(
            image_info["file_name"]
        ).name
    )


    if not image_path.exists():

        raise FileNotFoundError(
            f"Validation image missing:\n"
            f"{image_path}"
        )


    # -------------------------------------------------------------------------
    # YOLO inference
    # -------------------------------------------------------------------------

    results = model.predict(
        source=str(image_path),
        imgsz=YOLO_IMGSZ,
        conf=YOLO_CONF,
        iou=YOLO_NMS_IOU,
        max_det=YOLO_MAX_DET,
        device=DEVICE,
        verbose=False
    )


    result = results[0]


    if (
        result.boxes is None
        or len(result.boxes) == 0
    ):

        continue


    boxes_xyxy = (
        result.boxes.xyxy
        .detach()
        .cpu()
        .numpy()
    )


    confidences = (
        result.boxes.conf
        .detach()
        .cpu()
        .numpy()
    )


    yolo_classes = (
        result.boxes.cls
        .detach()
        .cpu()
        .numpy()
        .astype(int)
    )


    total_detections += len(
        boxes_xyxy
    )


    # -------------------------------------------------------------------------
    # Store prediction objects for this image.
    # -------------------------------------------------------------------------

    image_preds = []


    for pred_index in range(
        len(boxes_xyxy)
    ):

        bbox_xyxy = [
            float(v)
            for v in boxes_xyxy[
                pred_index
            ]
        ]


        yolo_idx = int(
            yolo_classes[
                pred_index
            ]
        )


        eval_category = (
            YOLO_IDX_TO_EVAL_COCO[
                yolo_idx
            ]
        )


        score = float(
            confidences[
                pred_index
            ]
        )


        image_preds.append(
            {
                "pred_index":
                    pred_index,

                "bbox_xyxy":
                    bbox_xyxy,

                "original_category_id":
                    eval_category,

                "score":
                    score
            }
        )


        # Normal detector baseline prediction.

        baseline_predictions.append(
            {
                "image_id":
                    image_id,

                "category_id":
                    eval_category,

                "bbox":
                    xyxy_to_xywh(
                        bbox_xyxy
                    ),

                "score":
                    score
            }
        )


    # -------------------------------------------------------------------------
    # Geometry-only GT <-> prediction matching.
    #
    # IMPORTANT:
    # class is NOT considered during matching.
    # -------------------------------------------------------------------------

    gts = gt_by_image[
        image_id
    ]


    candidate_pairs = []


    for gt_index, gt in enumerate(
        gts
    ):

        for pred_index, pred in enumerate(
            image_preds
        ):

            iou = calculate_iou(
                gt["bbox_xyxy"],
                pred["bbox_xyxy"]
            )


            if iou >= MATCH_IOU_THRESHOLD:

                candidate_pairs.append(
                    (
                        iou,
                        gt_index,
                        pred_index
                    )
                )


    # Highest-IoU matches first.

    candidate_pairs.sort(
        key=lambda x: x[0],
        reverse=True
    )


    used_gt = set()
    used_pred = set()


    # Map prediction index -> GT category

    oracle_category_for_prediction = {}


    for iou, gt_index, pred_index in candidate_pairs:

        if gt_index in used_gt:

            continue


        if pred_index in used_pred:

            continue


        used_gt.add(gt_index)
        used_pred.add(pred_index)


        gt = gts[
            gt_index
        ]


        pred = image_preds[
            pred_index
        ]


        gt_category = int(
            gt["category_id"]
        )


        original_pred_category = int(
            pred["original_category_id"]
        )


        oracle_category_for_prediction[
            pred_index
        ] = gt_category


        total_geometry_matches += 1


        was_correct = (
            original_pred_category
            ==
            gt_category
        )


        if was_correct:

            originally_correct_matches += 1

        else:

            total_class_corrections += 1


        oracle_match_rows.append(
            {
                "image_id":
                    image_id,

                "gt_annotation_id":
                    int(
                        gt["ann_id"]
                    ),

                "prediction_index":
                    pred_index,

                "iou":
                    float(iou),

                "gt_category_id":
                    gt_category,

                "gt_class":
                    CLASS_NAMES[
                        gt_category - 1
                    ],

                "original_category_id":
                    original_pred_category,

                "original_class":
                    CLASS_NAMES[
                        original_pred_category - 1
                    ],

                "classification_was_correct":
                    bool(was_correct),

                "oracle_class":
                    CLASS_NAMES[
                        gt_category - 1
                    ],

                "score":
                    float(
                        pred["score"]
                    )
            }
        )


    # -------------------------------------------------------------------------
    # Build oracle predictions.
    #
    # Matched predictions:
    #     class <- GT class
    #
    # Unmatched predictions:
    #     retain original YOLO class
    #
    # bbox and score ALWAYS unchanged.
    # -------------------------------------------------------------------------

    for pred_index, pred in enumerate(
        image_preds
    ):

        if pred_index in oracle_category_for_prediction:

            oracle_category = (
                oracle_category_for_prediction[
                    pred_index
                ]
            )

        else:

            oracle_category = int(
                pred[
                    "original_category_id"
                ]
            )


        oracle_predictions.append(
            {
                "image_id":
                    image_id,

                "category_id":
                    oracle_category,

                "bbox":
                    xyxy_to_xywh(
                        pred[
                            "bbox_xyxy"
                        ]
                    ),

                "score":
                    float(
                        pred[
                            "score"
                        ]
                    )
            }
        )


    # -------------------------------------------------------------------------
    # Progress
    # -------------------------------------------------------------------------

    if (
        image_number % 50 == 0
        or image_number == num_images
    ):

        elapsed = (
            time.time()
            -
            start_time
        )


        print(
            f"Processed {image_number:>4}/{num_images} | "
            f"detections: {total_detections:>6} | "
            f"matches: {total_geometry_matches:>5} | "
            f"class corrections: {total_class_corrections:>5} | "
            f"time: {elapsed / 60:.1f} min"
        )


print()


# =============================================================================
# 13. SUMMARY OF ORACLE CLASS CORRECTIONS
# =============================================================================

print("=" * 100)
print("ORACLE MATCHING SUMMARY")
print("=" * 100)


print(
    f"Validation images             : "
    f"{num_images}"
)

print(
    f"GT objects                    : "
    f"{len(remapped_coco['annotations'])}"
)

print(
    f"YOLO detections               : "
    f"{total_detections}"
)

print(
    f"Geometry matches IoU >= .50   : "
    f"{total_geometry_matches}"
)

print(
    f"Originally correct classes    : "
    f"{originally_correct_matches}"
)

print(
    f"Classes corrected by oracle   : "
    f"{total_class_corrections}"
)


if total_geometry_matches > 0:

    original_matched_accuracy = (
        originally_correct_matches
        /
        total_geometry_matches
    )

else:

    original_matched_accuracy = float("nan")


print(
    f"Original matched class acc.   : "
    f"{original_matched_accuracy:.6f} "
    f"({original_matched_accuracy * 100:.2f}%)"
)

print()


# =============================================================================
# 14. SAVE PREDICTIONS
# =============================================================================

BASELINE_JSON = (
    RESULTS_DIR
    / "E11_yolo_baseline_predictions.json"
)


ORACLE_JSON = (
    RESULTS_DIR
    / "E11_classification_oracle_predictions.json"
)


with open(
    BASELINE_JSON,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        baseline_predictions,
        f
    )


with open(
    ORACLE_JSON,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        oracle_predictions,
        f
    )


oracle_matches_df = pd.DataFrame(
    oracle_match_rows
)


oracle_matches_df.to_csv(
    RESULTS_DIR
    / "E11_oracle_matches.csv",
    index=False
)


print(
    f"Baseline predictions : {BASELINE_JSON}"
)

print(
    f"Oracle predictions   : {ORACLE_JSON}"
)

print()


# =============================================================================
# 15. COCO EVALUATION FUNCTION
# =============================================================================

def evaluate_coco(
    gt_json,
    pred_json,
    label
):

    print("=" * 100)
    print(label)
    print("=" * 100)


    coco_gt = COCO(
        str(gt_json)
    )


    coco_dt = coco_gt.loadRes(
        str(pred_json)
    )


    evaluator = COCOeval(
        coco_gt,
        coco_dt,
        "bbox"
    )


    evaluator.params.maxDets = [
        1,
        10,
        100
    ]


    evaluator.evaluate()
    evaluator.accumulate()
    evaluator.summarize()


    stats = evaluator.stats


    results = {

        "mAP50_95":
            float(stats[0]),

        "mAP50":
            float(stats[1]),

        "mAP75":
            float(stats[2]),

        "AR1":
            float(stats[6]),

        "AR10":
            float(stats[7]),

        "AR100":
            float(stats[8])
    }


    # -------------------------------------------------------------------------
    # Per-class AP
    # -------------------------------------------------------------------------

    precision = evaluator.eval[
        "precision"
    ]


    iou_thresholds = (
        evaluator.params.iouThrs
    )


    iou50_idx = int(
        np.argmin(
            np.abs(
                iou_thresholds
                -
                0.50
            )
        )
    )


    rows = []


    for class_idx, class_name in enumerate(
        CLASS_NAMES
    ):

        all_iou_precision = precision[
            :,
            :,
            class_idx,
            0,
            2
        ]


        valid_all = all_iou_precision[
            all_iou_precision > -1
        ]


        ap = (
            float(
                np.mean(
                    valid_all
                )
            )
            if len(valid_all) > 0
            else float("nan")
        )


        iou50_precision = precision[
            iou50_idx,
            :,
            class_idx,
            0,
            2
        ]


        valid_50 = iou50_precision[
            iou50_precision > -1
        ]


        ap50 = (
            float(
                np.mean(
                    valid_50
                )
            )
            if len(valid_50) > 0
            else float("nan")
        )


        rows.append(
            {
                "class":
                    class_name,

                "AP50":
                    ap50,

                "AP50_95":
                    ap
            }
        )


    return (
        results,
        pd.DataFrame(rows)
    )


# =============================================================================
# 16. EVALUATE NORMAL YOLO BASELINE
# =============================================================================

baseline_metrics, baseline_per_class = evaluate_coco(
    REMAPPED_GT_JSON,
    BASELINE_JSON,
    "NORMAL YOLO11s BASELINE — SAME BOXES / ORIGINAL CLASSES"
)


baseline_per_class.to_csv(
    RESULTS_DIR
    / "E11_baseline_per_class_AP.csv",
    index=False
)


# =============================================================================
# 17. EVALUATE CLASSIFICATION ORACLE
# =============================================================================

oracle_metrics, oracle_per_class = evaluate_coco(
    REMAPPED_GT_JSON,
    ORACLE_JSON,
    "E11 CLASSIFICATION ORACLE — SAME BOXES / PERFECT MATCHED CLASSES"
)


oracle_per_class.to_csv(
    RESULTS_DIR
    / "E11_oracle_per_class_AP.csv",
    index=False
)


# =============================================================================
# 18. COMPARE BASELINE VS ORACLE
# =============================================================================

comparison = pd.DataFrame(
    [
        {
            "metric":
                "mAP50-95",

            "YOLO_baseline":
                baseline_metrics[
                    "mAP50_95"
                ],

            "E11_oracle":
                oracle_metrics[
                    "mAP50_95"
                ]
        },

        {
            "metric":
                "mAP50",

            "YOLO_baseline":
                baseline_metrics[
                    "mAP50"
                ],

            "E11_oracle":
                oracle_metrics[
                    "mAP50"
                ]
        },

        {
            "metric":
                "mAP75",

            "YOLO_baseline":
                baseline_metrics[
                    "mAP75"
                ],

            "E11_oracle":
                oracle_metrics[
                    "mAP75"
                ]
        },

        {
            "metric":
                "AR100",

            "YOLO_baseline":
                baseline_metrics[
                    "AR100"
                ],

            "E11_oracle":
                oracle_metrics[
                    "AR100"
                ]
        }
    ]
)


comparison[
    "absolute_gain"
] = (
    comparison[
        "E11_oracle"
    ]
    -
    comparison[
        "YOLO_baseline"
    ]
)


comparison[
    "gain_percentage_points"
] = (
    comparison[
        "absolute_gain"
    ]
    *
    100
)


comparison.to_csv(
    RESULTS_DIR
    / "E11_baseline_vs_classification_oracle.csv",
    index=False
)


# =============================================================================
# 19. PER-CLASS COMPARISON
# =============================================================================

per_class_comparison = baseline_per_class.merge(
    oracle_per_class,
    on="class",
    suffixes=(
        "_baseline",
        "_oracle"
    )
)


per_class_comparison[
    "AP50_gain"
] = (
    per_class_comparison[
        "AP50_oracle"
    ]
    -
    per_class_comparison[
        "AP50_baseline"
    ]
)


per_class_comparison[
    "AP50_95_gain"
] = (
    per_class_comparison[
        "AP50_95_oracle"
    ]
    -
    per_class_comparison[
        "AP50_95_baseline"
    ]
)


per_class_comparison.to_csv(
    RESULTS_DIR
    / "E11_per_class_oracle_gain.csv",
    index=False
)


# =============================================================================
# 20. SAVE SUMMARY JSON
# =============================================================================

summary = {

    "experiment":
        "E11 — Classification-Oracle Analysis of YOLO11s Detections",

    "purpose":
        (
            "Estimate classification upper-bound while keeping "
            "detector bounding boxes and confidence unchanged."
        ),

    "detector":
        "E5 YOLO11s",

    "inference": {

        "imgsz":
            YOLO_IMGSZ,

        "confidence_threshold":
            YOLO_CONF,

        "nms_iou":
            YOLO_NMS_IOU,

        "max_det":
            YOLO_MAX_DET,

        "oracle_match_iou":
            MATCH_IOU_THRESHOLD
    },

    "counts": {

        "images":
            num_images,

        "gt_objects":
            len(
                remapped_coco[
                    "annotations"
                ]
            ),

        "detections":
            total_detections,

        "geometry_matches":
            total_geometry_matches,

        "originally_correct_classes":
            originally_correct_matches,

        "oracle_class_corrections":
            total_class_corrections,

        "original_matched_classification_accuracy":
            original_matched_accuracy
    },

    "baseline":
        baseline_metrics,

    "classification_oracle":
        oracle_metrics,

    "oracle_gain": {

        key:
            float(
                oracle_metrics[key]
                -
                baseline_metrics[key]
            )

        for key in baseline_metrics
    }
}


SUMMARY_JSON = (
    RESULTS_DIR
    / "E11_summary.json"
)


with open(
    SUMMARY_JSON,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        summary,
        f,
        indent=4
    )


# =============================================================================
# 21. FINAL PRINT
# =============================================================================

print()

print("=" * 100)
print("E11 FINAL RESULTS")
print("=" * 100)

print()

print("NORMAL YOLO11s")
print("----------------")

print(
    f"mAP50-95 : "
    f"{baseline_metrics['mAP50_95']:.6f} "
    f"({baseline_metrics['mAP50_95'] * 100:.2f}%)"
)

print(
    f"mAP50    : "
    f"{baseline_metrics['mAP50']:.6f} "
    f"({baseline_metrics['mAP50'] * 100:.2f}%)"
)

print(
    f"mAP75    : "
    f"{baseline_metrics['mAP75']:.6f} "
    f"({baseline_metrics['mAP75'] * 100:.2f}%)"
)

print()

print("CLASSIFICATION ORACLE")
print("---------------------")

print(
    f"mAP50-95 : "
    f"{oracle_metrics['mAP50_95']:.6f} "
    f"({oracle_metrics['mAP50_95'] * 100:.2f}%)"
)

print(
    f"mAP50    : "
    f"{oracle_metrics['mAP50']:.6f} "
    f"({oracle_metrics['mAP50'] * 100:.2f}%)"
)

print(
    f"mAP75    : "
    f"{oracle_metrics['mAP75']:.6f} "
    f"({oracle_metrics['mAP75'] * 100:.2f}%)"
)

print()

print("ORACLE GAINS")
print("------------")

print(
    f"mAP50-95 gain : "
    f"{(oracle_metrics['mAP50_95'] - baseline_metrics['mAP50_95']) * 100:+.2f} percentage points"
)


print(
    f"mAP50 gain    : "
    f"{(oracle_metrics['mAP50'] - baseline_metrics['mAP50']) * 100:+.2f} percentage points"
)


print(
    f"mAP75 gain    : " 
    f"{(oracle_metrics['mAP75'] - baseline_metrics['mAP75']) * 100:+.2f} percentage points"
)


print()

print(
    f"Geometry matches       : "
    f"{total_geometry_matches}"
)

print(
    f"Class corrections      : "
    f"{total_class_corrections}"
)

print(
    f"Original matched acc.  : "
    f"{original_matched_accuracy * 100:.2f}%"
)

print()

print("PER-CLASS ORACLE GAIN")
print("---------------------")

print(
    per_class_comparison[
        [
            "class",
            "AP50_95_baseline",
            "AP50_95_oracle",
            "AP50_95_gain"
        ]
    ].to_string(
        index=False
    )
)

print()

print(
    f"Results directory:\n"
    f"{RESULTS_DIR}"
)

print()

print("=" * 100)
print("E11 CLASSIFICATION-ORACLE ANALYSIS COMPLETE")
print("=" * 100)

E11 — CLASSIFICATION-ORACLE ANALYSIS OF YOLO11s DETECTIONS

Purpose : Estimate the maximum AP obtainable by fixing classification
          while leaving YOLO11s bounding boxes and detector scores unchanged.

YOLO checkpoint : C:\Users\varda\Documents\_My Computer\COMMON_space\Learning\Upgrad_EPGP_MS\upgrad_IIITB_MS_course\MS_LJMU_Material\Thesis_Code\runs\sortwaste\E5_yolo11s_7class_aug_classbalance_640\weights\best.pt
Validation data : C:\Users\varda\Documents\_My Computer\COMMON_space\Learning\Upgrad_EPGP_MS\upgrad_IIITB_MS_course\MS_LJMU_Material\Topic Data\SortWaste\dataset\dataset\splited_all_dataset_coco\val\images
COCO GT         : C:\Users\varda\Documents\_My Computer\COMMON_space\Learning\Upgrad_EPGP_MS\upgrad_IIITB_MS_course\MS_LJMU_Material\Topic Data\SortWaste\dataset\dataset\splited_all_dataset_coco\val\annotations\val_coco.json
Results         : C:\Users\varda\Documents\_My Computer\COMMON_space\Learning\Upgrad_EPGP_MS\upgrad_IIITB_MS_course\MS_LJMU_Material\Thesis_Code\

## E11-B: Fusion-score ablation using the same YOLO11s boxes + MobileNet classes, testing different confidence rules

In [ ]:
# E11-B — FUSION-SCORE ABLATION YOLO11s + MobileNetV3-Large
# =============================================================================
#
# PURPOSE
# -------
# Determine whether the confidence fusion rule is limiting end-to-end AP.
#
# SAME:
#   - E5 YOLO11s detector
#   - validation images
#   - conf = 0.001
#   - NMS IoU = 0.60
#   - imgsz = 640
#   - max_det = 100
#   - MobileNet final class prediction
#   - bounding boxes unchanged
#
# ONLY VARIABLE:
#   confidence score assigned to final detection
#
#
# SCORE VARIANTS
# --------------
#
# A:
#   score = YOLO confidence
#
# B:
#   score = YOLO confidence * MobileNet probability
#   (current E8 rule)
#
# C:
#   score = MobileNet probability
#
# D:
#   score = sqrt(YOLO confidence * MobileNet probability)
#
#
# PRIMARY QUESTION:
#
# Does keeping YOLO's confidence ranking while replacing only the class
# outperform the current multiplication rule?
#
# =============================================================================


# =============================================================================
# 1. IMPORTS
# =============================================================================

import json
import math
import time

from pathlib import Path

import numpy as np
import pandas as pd

import torch
import torch.nn as nn

from torchvision import models, transforms
from PIL import Image

from ultralytics import YOLO

from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval


# =============================================================================
# 2. CONFIGURATION
# =============================================================================

SEED = 42


DATASET_ROOT = Path(
    r"C:\Users\varda\Documents\_My Computer\COMMON_space\Learning"
    r"\Upgrad_EPGP_MS\upgrad_IIITB_MS_course\MS_LJMU_Material"
    r"\Topic Data\SortWaste\dataset\dataset"
)


THESIS_CODE_ROOT = Path(
    r"C:\Users\varda\Documents\_My Computer\COMMON_space\Learning"
    r"\Upgrad_EPGP_MS\upgrad_IIITB_MS_course\MS_LJMU_Material"
    r"\Thesis_Code"
)


VAL_IMAGES_DIR = (
    DATASET_ROOT
    / "splited_all_dataset_coco"
    / "val"
    / "images"
)


VAL_COCO_JSON = (
    DATASET_ROOT
    / "splited_all_dataset_coco"
    / "val"
    / "annotations"
    / "val_coco.json"
)


# -----------------------------------------------------------------------------
# E5 YOLO11s detector
# -----------------------------------------------------------------------------

YOLO_CHECKPOINT = (
    THESIS_CODE_ROOT
    / "runs"
    / "sortwaste"
    / "E5_yolo11s_7class_aug_classbalance_640"
    / "weights"
    / "best.pt"
)


# -----------------------------------------------------------------------------
# IMPORTANT:
# Use E8's classifier = E3Y-B class-weighted MobileNet.
#
# E8 is currently the strongest two-stage pipeline.
# -----------------------------------------------------------------------------

MOBILENET_CHECKPOINT = (
    DATASET_ROOT
    / "yolo_mobilenet_crops_E3Y"
    / "mobilenet_results"
    / "E3Y_B_class_weighted"
    / "E3Y_B_MobileNetV3Large_best.pth"
)


RESULTS_DIR = (
    THESIS_CODE_ROOT
    / "runs"
    / "sortwaste"
    / "E12_fusion_score_ablation"
)


RESULTS_DIR.mkdir(
    parents=True,
    exist_ok=True
)


# =============================================================================
# 3. INFERENCE SETTINGS — SAME AS E8
# =============================================================================

IMAGE_SIZE = 224

YOLO_IMGSZ = 640
YOLO_CONF = 0.001
YOLO_NMS_IOU = 0.60
YOLO_MAX_DET = 100


# =============================================================================
# 4. CLASS TAXONOMY
# =============================================================================

CLASS_NAMES = [
    "ecal",
    "hdpe",
    "mixed_plastic_rigid",
    "mixed_plastic_soft",
    "non_plastic",
    "pet",
    "pet_oil"
]


NUM_CLASSES = len(CLASS_NAMES)


IDX_TO_COCO_CATEGORY = {
    0: 1,
    1: 2,
    2: 3,
    3: 4,
    4: 5,
    5: 6,
    6: 7
}


# Original SortWaste COCO -> 7-class evaluation taxonomy

ORIGINAL_COCO_TO_EVAL_COCO = {
    1: 6,   # pet
    2: 2,   # hdpe
    3: 4,   # mixed soft
    4: 1,   # ecal
    5: 5,   # metal -> nonplastic
    6: 5,   # cardboard -> nonplastic
    7: 3,   # mixed rigid
    8: 7    # pet oil
}


# =============================================================================
# 5. REPRODUCIBILITY
# =============================================================================

np.random.seed(SEED)
torch.manual_seed(SEED)


if torch.cuda.is_available():

    torch.cuda.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)


torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False


DEVICE = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)


YOLO_DEVICE = (
    0
    if torch.cuda.is_available()
    else "cpu"
)


# =============================================================================
# 6. INTRO
# =============================================================================

print("=" * 100)
print("E12 — YOLO11s + MOBILENET FUSION-SCORE ABLATION")
print("=" * 100)

print()

print(f"Device               : {DEVICE}")
print(f"YOLO checkpoint      : {YOLO_CHECKPOINT}")
print(f"MobileNet checkpoint : {MOBILENET_CHECKPOINT}")
print(f"Validation images    : {VAL_IMAGES_DIR}")
print(f"COCO GT              : {VAL_COCO_JSON}")
print(f"Output                : {RESULTS_DIR}")

print()


# =============================================================================
# 7. PATH CHECKS
# =============================================================================

for path in [
    VAL_IMAGES_DIR,
    VAL_COCO_JSON,
    YOLO_CHECKPOINT,
    MOBILENET_CHECKPOINT
]:

    print(
        f"{path} | Exists: {path.exists()}"
    )

    if not path.exists():

        raise FileNotFoundError(
            f"Required path missing:\n{path}"
        )


print()


# =============================================================================
# 8. LOAD + REMAP GT
# =============================================================================

print("=" * 100)
print("BUILDING 7-CLASS COCO GT")
print("=" * 100)


with open(
    VAL_COCO_JSON,
    "r",
    encoding="utf-8"
) as f:

    original_coco = json.load(f)


eval_categories = [
    {"id": 1, "name": "ecal"},
    {"id": 2, "name": "hdpe"},
    {"id": 3, "name": "mixed_plastic_rigid"},
    {"id": 4, "name": "mixed_plastic_soft"},
    {"id": 5, "name": "non_plastic"},
    {"id": 6, "name": "pet"},
    {"id": 7, "name": "pet_oil"}
]


remapped_annotations = []


for ann in original_coco["annotations"]:

    new_ann = dict(ann)

    new_ann["category_id"] = (
        ORIGINAL_COCO_TO_EVAL_COCO[
            int(ann["category_id"])
        ]
    )

    new_ann["iscrowd"] = int(
        new_ann.get("iscrowd", 0)
    )


    if (
        "area" not in new_ann
        or new_ann["area"] is None
    ):

        x, y, w, h = new_ann["bbox"]

        new_ann["area"] = float(
            max(0.0, w)
            *
            max(0.0, h)
        )


    remapped_annotations.append(
        new_ann
    )


remapped_coco = {
    "info": original_coco.get("info", {}),
    "licenses": original_coco.get("licenses", []),
    "images": original_coco["images"],
    "annotations": remapped_annotations,
    "categories": eval_categories
}


REMAPPED_GT_JSON = (
    RESULTS_DIR
    / "E12_val_gt_7class.json"
)


with open(
    REMAPPED_GT_JSON,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        remapped_coco,
        f
    )


print(
    f"Images      : {len(remapped_coco['images'])}"
)

print(
    f"Annotations : {len(remapped_coco['annotations'])}"
)

print()


# =============================================================================
# 9. HELPERS
# =============================================================================

def xyxy_to_xywh(box):

    x1, y1, x2, y2 = [
        float(v)
        for v in box
    ]

    return [
        x1,
        y1,
        max(0.0, x2 - x1),
        max(0.0, y2 - y1)
    ]


# =============================================================================
# 10. LOAD YOLO
# =============================================================================

print("=" * 100)
print("LOADING E5 YOLO11s")
print("=" * 100)


yolo_model = YOLO(
    str(YOLO_CHECKPOINT)
)


print("YOLO loaded.")
print()


# =============================================================================
# 11. LOAD E8 MOBILENET
# =============================================================================

print("=" * 100)
print("LOADING E8 / E3Y-B MOBILENET")
print("=" * 100)


mobilenet = models.mobilenet_v3_large(
    weights=None
)


in_features = (
    mobilenet
    .classifier[-1]
    .in_features
)


mobilenet.classifier[-1] = nn.Linear(
    in_features,
    NUM_CLASSES
)


checkpoint = torch.load(
    MOBILENET_CHECKPOINT,
    map_location=DEVICE
)


if (
    isinstance(checkpoint, dict)
    and "model_state_dict" in checkpoint
):

    state_dict = checkpoint[
        "model_state_dict"
    ]

else:

    state_dict = checkpoint


mobilenet.load_state_dict(
    state_dict
)


mobilenet = mobilenet.to(
    DEVICE
)


mobilenet.eval()


print("MobileNet loaded.")
print()


# =============================================================================
# 12. MOBILENET TRANSFORM
# =============================================================================

mobilenet_transform = transforms.Compose(
    [

        transforms.Resize(
            (
                IMAGE_SIZE,
                IMAGE_SIZE
            )
        ),

        transforms.ToTensor(),

        transforms.Normalize(
            mean=[
                0.485,
                0.456,
                0.406
            ],

            std=[
                0.229,
                0.224,
                0.225
            ]
        )
    ]
)


# =============================================================================
# 13. PREDICTION STORAGE
# =============================================================================

predictions = {

    # MobileNet class + YOLO confidence
    "A_yolo_conf":
        [],

    # MobileNet class + YOLO * MobileNet
    "B_product":
        [],

    # MobileNet class + MobileNet probability only
    "C_mobilenet_prob":
        [],

    # MobileNet class + sqrt(YOLO * MobileNet)
    "D_geometric_mean":
        []
}


# =============================================================================
# 14. RUN INFERENCE ONCE
# =============================================================================

print("=" * 100)
print("RUNNING SHARED YOLO + MOBILENET INFERENCE")
print("=" * 100)


total_detections = 0
valid_crops = 0
invalid_crops = 0


start_time = time.time()


num_images = len(
    remapped_coco["images"]
)


for image_number, image_info in enumerate(
    remapped_coco["images"],
    start=1
):

    image_id = int(
        image_info["id"]
    )


    image_path = (
        VAL_IMAGES_DIR
        /
        Path(
            image_info["file_name"]
        ).name
    )


    if not image_path.exists():

        raise FileNotFoundError(
            f"Image missing:\n{image_path}"
        )


    pil_image = Image.open(
        image_path
    ).convert(
        "RGB"
    )


    width, height = pil_image.size


    # -------------------------------------------------------------------------
    # YOLO
    # -------------------------------------------------------------------------

    results = yolo_model.predict(
        source=str(image_path),
        imgsz=YOLO_IMGSZ,
        conf=YOLO_CONF,
        iou=YOLO_NMS_IOU,
        max_det=YOLO_MAX_DET,
        device=YOLO_DEVICE,
        verbose=False
    )


    if len(results) == 0:

        continue


    result = results[0]


    if (
        result.boxes is None
        or len(result.boxes) == 0
    ):

        continue


    boxes_xyxy = (
        result.boxes.xyxy
        .detach()
        .cpu()
        .numpy()
    )


    yolo_scores = (
        result.boxes.conf
        .detach()
        .cpu()
        .numpy()
    )


    total_detections += len(
        boxes_xyxy
    )


    # -------------------------------------------------------------------------
    # BUILD CROPS
    # -------------------------------------------------------------------------

    crop_tensors = []
    metadata = []


    for i in range(
        len(boxes_xyxy)
    ):

        x1, y1, x2, y2 = [
            float(v)
            for v in boxes_xyxy[i]
        ]


        # Clamp bbox

        x1 = max(
            0.0,
            min(x1, float(width))
        )

        y1 = max(
            0.0,
            min(y1, float(height))
        )

        x2 = max(
            0.0,
            min(x2, float(width))
        )

        y2 = max(
            0.0,
            min(y2, float(height))
        )


        left = max(
            0,
            min(
                int(math.floor(x1)),
                width
            )
        )

        top = max(
            0,
            min(
                int(math.floor(y1)),
                height
            )
        )

        right = max(
            0,
            min(
                int(math.ceil(x2)),
                width
            )
        )

        bottom = max(
            0,
            min(
                int(math.ceil(y2)),
                height
            )
        )


        if (
            right <= left
            or bottom <= top
        ):

            invalid_crops += 1
            continue


        crop = pil_image.crop(
            (
                left,
                top,
                right,
                bottom
            )
        )


        if (
            crop.width <= 0
            or crop.height <= 0
        ):

            invalid_crops += 1
            continue


        crop_tensor = mobilenet_transform(
            crop
        )


        crop_tensors.append(
            crop_tensor
        )


        metadata.append(
            {
                "bbox_xyxy":
                    [
                        x1,
                        y1,
                        x2,
                        y2
                    ],

                "yolo_conf":
                    float(
                        yolo_scores[i]
                    )
            }
        )


    if len(crop_tensors) == 0:

        continue


    valid_crops += len(
        crop_tensors
    )


    # -------------------------------------------------------------------------
    # MOBILENET
    # -------------------------------------------------------------------------

    batch = torch.stack(
        crop_tensors
    ).to(
        DEVICE
    )


    with torch.no_grad():

        logits = mobilenet(
            batch
        )


        probs = torch.softmax(
            logits,
            dim=1
        )


        pred_classes = probs.argmax(
            dim=1
        )


        pred_probs = probs[
            torch.arange(
                probs.size(0),
                device=DEVICE
            ),
            pred_classes
        ]


    pred_classes = (
        pred_classes
        .detach()
        .cpu()
        .numpy()
    )


    pred_probs = (
        pred_probs
        .detach()
        .cpu()
        .numpy()
    )


    # -------------------------------------------------------------------------
    # BUILD FOUR SCORE VARIANTS
    # -------------------------------------------------------------------------

    for i, meta in enumerate(
        metadata
    ):

        class_idx = int(
            pred_classes[i]
        )


        category_id = (
            IDX_TO_COCO_CATEGORY[
                class_idx
            ]
        )


        yolo_conf = float(
            meta["yolo_conf"]
        )


        mobile_prob = float(
            pred_probs[i]
        )


        score_A = yolo_conf


        score_B = (
            yolo_conf
            *
            mobile_prob
        )


        score_C = mobile_prob


        score_D = math.sqrt(
            max(
                0.0,
                yolo_conf
                *
                mobile_prob
            )
        )


        bbox_xywh = xyxy_to_xywh(
            meta["bbox_xyxy"]
        )


        base_prediction = {
            "image_id":
                image_id,

            "category_id":
                category_id,

            "bbox":
                bbox_xywh
        }


        predictions[
            "A_yolo_conf"
        ].append(
            {
                **base_prediction,
                "score":
                    float(score_A)
            }
        )


        predictions[
            "B_product"
        ].append(
            {
                **base_prediction,
                "score":
                    float(score_B)
            }
        )


        predictions[
            "C_mobilenet_prob"
        ].append(
            {
                **base_prediction,
                "score":
                    float(score_C)
            }
        )


        predictions[
            "D_geometric_mean"
        ].append(
            {
                **base_prediction,
                "score":
                    float(score_D)
            }
        )


    # -------------------------------------------------------------------------
    # Progress
    # -------------------------------------------------------------------------

    if (
        image_number % 50 == 0
        or image_number == num_images
    ):

        elapsed = (
            time.time()
            -
            start_time
        )


        print(
            f"Processed "
            f"{image_number:>4}/{num_images} | "
            f"detections: {total_detections:>6} | "
            f"valid crops: {valid_crops:>6} | "
            f"invalid: {invalid_crops:>2} | "
            f"time: {elapsed / 60:.1f} min"
        )


print()


# =============================================================================
# 15. SANITY CHECK
# =============================================================================

print("=" * 100)
print("INFERENCE SUMMARY")
print("=" * 100)


print(
    f"Images          : {num_images}"
)

print(
    f"YOLO detections : {total_detections}"
)

print(
    f"Valid crops     : {valid_crops}"
)

print(
    f"Invalid crops   : {invalid_crops}"
)


for key, preds in predictions.items():

    print(
        f"{key:<25}: {len(preds)} predictions"
    )


print()


# =============================================================================
# 16. SAVE PREDICTION JSON FILES
# =============================================================================

prediction_files = {}


for variant_name, preds in predictions.items():

    output_json = (
        RESULTS_DIR
        /
        f"E12_{variant_name}.json"
    )


    with open(
        output_json,
        "w",
        encoding="utf-8"
    ) as f:

        json.dump(
            preds,
            f
        )


    prediction_files[
        variant_name
    ] = output_json


# =============================================================================
# 17. COCO EVALUATION FUNCTION
# =============================================================================

def evaluate_variant(
    variant_name,
    prediction_json
):

    print("=" * 100)
    print(
        f"EVALUATING {variant_name}"
    )
    print("=" * 100)


    coco_gt = COCO(
        str(
            REMAPPED_GT_JSON
        )
    )


    coco_dt = coco_gt.loadRes(
        str(
            prediction_json
        )
    )


    evaluator = COCOeval(
        coco_gt,
        coco_dt,
        "bbox"
    )


    evaluator.params.maxDets = [
        1,
        10,
        100
    ]


    evaluator.evaluate()
    evaluator.accumulate()
    evaluator.summarize()


    stats = evaluator.stats


    result = {

        "variant":
            variant_name,

        "mAP50_95":
            float(stats[0]),

        "mAP50":
            float(stats[1]),

        "mAP75":
            float(stats[2]),

        "AR1":
            float(stats[6]),

        "AR10":
            float(stats[7]),

        "AR100":
            float(stats[8])
    }


    # -------------------------------------------------------------------------
    # PER-CLASS AP
    # -------------------------------------------------------------------------

    precision = evaluator.eval[
        "precision"
    ]


    iou_thresholds = (
        evaluator.params.iouThrs
    )


    iou50_idx = int(
        np.argmin(
            np.abs(
                iou_thresholds
                -
                0.50
            )
        )
    )


    per_class_rows = []


    for class_idx, class_name in enumerate(
        CLASS_NAMES
    ):

        precision_all = precision[
            :,
            :,
            class_idx,
            0,
            2
        ]


        valid_all = precision_all[
            precision_all > -1
        ]


        class_ap = (
            float(
                np.mean(
                    valid_all
                )
            )
            if len(valid_all) > 0
            else float("nan")
        )


        precision50 = precision[
            iou50_idx,
            :,
            class_idx,
            0,
            2
        ]


        valid50 = precision50[
            precision50 > -1
        ]


        class_ap50 = (
            float(
                np.mean(
                    valid50
                )
            )
            if len(valid50) > 0
            else float("nan")
        )


        per_class_rows.append(
            {
                "variant":
                    variant_name,

                "class":
                    class_name,

                "AP50":
                    class_ap50,

                "AP50_95":
                    class_ap
            }
        )


    return (
        result,
        per_class_rows
    )


# =============================================================================
# 18. EVALUATE ALL VARIANTS
# =============================================================================

all_results = []

all_per_class_results = []


for variant_name in [
    "A_yolo_conf",
    "B_product",
    "C_mobilenet_prob",
    "D_geometric_mean"
]:

    result, per_class = evaluate_variant(
        variant_name,
        prediction_files[
            variant_name
        ]
    )


    all_results.append(
        result
    )


    all_per_class_results.extend(
        per_class
    )


# =============================================================================
# 19. RESULTS TABLE
# =============================================================================

results_df = pd.DataFrame(
    all_results
)


results_df = results_df.sort_values(
    by="mAP50_95",
    ascending=False
).reset_index(
    drop=True
)


results_df[
    "mAP50_95_pct"
] = (
    results_df[
        "mAP50_95"
    ]
    *
    100
)


results_df[
    "mAP50_pct"
] = (
    results_df[
        "mAP50"
    ]
    *
    100
)


results_df[
    "mAP75_pct"
] = (
    results_df[
        "mAP75"
    ]
    *
    100
)


RESULTS_CSV = (
    RESULTS_DIR
    / "E12_fusion_score_results.csv"
)


results_df.to_csv(
    RESULTS_CSV,
    index=False
)


per_class_df = pd.DataFrame(
    all_per_class_results
)


per_class_df.to_csv(
    RESULTS_DIR
    / "E12_per_class_results.csv",
    index=False
)


# =============================================================================
# 20. COMPARE AGAINST CURRENT E8 PRODUCT RULE
# =============================================================================

product_row = (
    results_df[
        results_df[
            "variant"
        ]
        ==
        "B_product"
    ]
    .iloc[0]
)


results_df[
    "gain_vs_product_pp"
] = (
    (
        results_df[
            "mAP50_95"
        ]
        -
        float(
            product_row[
                "mAP50_95"
            ]
        )
    )
    *
    100
)


# =============================================================================
# 21. PRINT FINAL RESULTS
# =============================================================================

print()

print("=" * 100)
print("E12 FINAL FUSION-SCORE RESULTS")
print("=" * 100)

print()


print(
    results_df[
        [
            "variant",
            "mAP50_95_pct",
            "mAP50_pct",
            "mAP75_pct",
            "AR100",
            "gain_vs_product_pp"
        ]
    ].to_string(
        index=False
    )
)


print()


best_row = results_df.iloc[0]


print("=" * 100)
print("BEST SCORE VARIANT")
print("=" * 100)

print(
    f"Variant    : "
    f"{best_row['variant']}"
)

print(
    f"mAP50-95   : "
    f"{best_row['mAP50_95']:.6f} "
    f"({best_row['mAP50_95'] * 100:.2f}%)"
)

print(
    f"mAP50      : "
    f"{best_row['mAP50']:.6f} "
    f"({best_row['mAP50'] * 100:.2f}%)"
)

print(
    f"mAP75      : "
    f"{best_row['mAP75']:.6f} "
    f"({best_row['mAP75'] * 100:.2f}%)"
)

print(
    f"Gain vs current product rule : "
    f"{best_row['gain_vs_product_pp']:+.2f} pp"
)

print()


# =============================================================================
# 22. PER-CLASS TABLE FOR EACH FUSION RULE
# =============================================================================

pivot_ap = per_class_df.pivot(
    index="class",
    columns="variant",
    values="AP50_95"
)


pivot_ap.to_csv(
    RESULTS_DIR
    / "E12_per_class_AP50_95_comparison.csv"
)


print("=" * 100)
print("PER-CLASS mAP50-95")
print("=" * 100)

print(
    pivot_ap.to_string()
)

print()


# =============================================================================
# 23. SAVE SUMMARY JSON
# =============================================================================

summary = {

    "experiment":
        "E12 — Fusion-Score Ablation",

    "detector":
        "E5 YOLO11s",

    "classifier":
        "E3Y-B MobileNetV3-Large",

    "same_as_E8_except":
        "final confidence scoring rule",

    "settings": {

        "imgsz":
            YOLO_IMGSZ,

        "conf":
            YOLO_CONF,

        "nms_iou":
            YOLO_NMS_IOU,

        "max_det":
            YOLO_MAX_DET
    },

    "counts": {

        "images":
            num_images,

        "detections":
            total_detections,

        "valid_crops":
            valid_crops,

        "invalid_crops":
            invalid_crops
    },

    "score_variants": {

        "A_yolo_conf":
            "YOLO confidence",

        "B_product":
            "YOLO confidence * MobileNet predicted probability",

        "C_mobilenet_prob":
            "MobileNet predicted probability",

        "D_geometric_mean":
            "sqrt(YOLO confidence * MobileNet predicted probability)"
    },

    "results":
        results_df.to_dict(
            orient="records"
        ),

    "best_variant":
        str(
            best_row[
                "variant"
            ]
        ),

    "best_mAP50_95":
        float(
            best_row[
                "mAP50_95"
            ]
        )
}


with open(
    RESULTS_DIR
    / "E12_summary.json",
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        summary,
        f,
        indent=4
    )


print(
    f"Results directory:\n{RESULTS_DIR}"
)

print()

print("=" * 100)
print("E12 FUSION-SCORE ABLATION COMPLETE")
print("=" * 100)

E12 — YOLO11s + MOBILENET FUSION-SCORE ABLATION

Device               : cuda
YOLO checkpoint      : C:\Users\varda\Documents\_My Computer\COMMON_space\Learning\Upgrad_EPGP_MS\upgrad_IIITB_MS_course\MS_LJMU_Material\Thesis_Code\runs\sortwaste\E5_yolo11s_7class_aug_classbalance_640\weights\best.pt
MobileNet checkpoint : C:\Users\varda\Documents\_My Computer\COMMON_space\Learning\Upgrad_EPGP_MS\upgrad_IIITB_MS_course\MS_LJMU_Material\Topic Data\SortWaste\dataset\dataset\yolo_mobilenet_crops_E3Y\mobilenet_results\E3Y_B_class_weighted\E3Y_B_MobileNetV3Large_best.pth
Validation images    : C:\Users\varda\Documents\_My Computer\COMMON_space\Learning\Upgrad_EPGP_MS\upgrad_IIITB_MS_course\MS_LJMU_Material\Topic Data\SortWaste\dataset\dataset\splited_all_dataset_coco\val\images
COCO GT              : C:\Users\varda\Documents\_My Computer\COMMON_space\Learning\Upgrad_EPGP_MS\upgrad_IIITB_MS_course\MS_LJMU_Material\Topic Data\SortWaste\dataset\dataset\splited_all_dataset_coco\val\annotations\val_c

# Model E12 — YOLO11m + 7-Class Taxonomy + Realistic Augmentation + Class-Aware Oversampling @640

In [ ]:
# E12 — YOLO11m + 7-Class Taxonomy + Realistic Augmentation + Class-Aware Oversampling @640
#
# LJMU Thesis — SortWaste
# ======================================================================
#
# PURPOSE
#
# Compare a higher-capacity YOLO11m detector against E5/E7 YOLO11s,
# while preserving the successful E5 training setup as closely as possible.
#
# PRIMARY CHANGE
#
#     E5  : YOLO11s @640
#     E12 : YOLO11m @640
#
# SAME:
#
#     - E5 7-class taxonomy
#     - E5 class-aware oversampled training dataset
#     - 640 resolution
#     - AdamW
#     - cosine LR schedule
#     - realistic augmentation
#     - 150 max epochs
#     - patience = 15
#     - seed = 42
#
# HARDWARE-DRIVEN CHANGE:
#
#     E5 batch = 4
#     E12 batch = 2
#
# If CUDA OOM occurs:
#
#     change batch = 1
#
# IMPORTANT:
#
#     This script DOES NOT rebuild the E5 dataset.
#     It directly reuses:
#
#     Thesis_Code\runs\sortwaste\E5_7class_dataset\E5_7class.yaml
#
# ======================================================================


import sys
import random
from pathlib import Path

import torch
from ultralytics import YOLO


# ======================================================================
# 1. GLOBAL SETTINGS
# ======================================================================

SEED = 42

random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)


# ======================================================================
# 2. THESIS OUTPUT ROOT
#    SAME ROOT USED IN E5
# ======================================================================

PROJECT_ROOT = Path(
    r"C:\Users\varda\Documents\_My Computer\COMMON_space\Learning"
    r"\Upgrad_EPGP_MS\upgrad_IIITB_MS_course"
    r"\MS_LJMU_Material\Thesis_Code"
    r"\runs\sortwaste"
)


# ======================================================================
# 3. EXISTING E5 DERIVED DATASET
# ======================================================================
#
# E5 already:
#
#   - converted 8 classes -> 7 classes
#   - merged metal + cardboard -> non_plastic
#   - oversampled non_plastic 2x
#   - oversampled pet_oil 3x
#   - preserved validation/test without oversampling
#
# DO NOT REBUILD IT.
#
# ======================================================================

E5_DATASET_ROOT = (
    PROJECT_ROOT
    / "E5_7class_dataset"
)


E5_DATA_YAML = (
    E5_DATASET_ROOT
    / "E5_7class.yaml"
)


# ======================================================================
# 4. E12 EXPERIMENT
# ======================================================================

EXPERIMENT_NAME = (
    "E12_yolo11m_7class_aug_classbalance_640"
)


RUN_DIR = (
    PROJECT_ROOT
    / EXPERIMENT_NAME
)


# ======================================================================
# 5. TRAINING SETTINGS
# ======================================================================

IMAGE_SIZE = 640

MAX_EPOCHS = 150

PATIENCE = 15

# Start with 2 on RTX 3050 Ti 4 GB.
#
# If CUDA OOM occurs:
#
#     change this to 1
#
BATCH_SIZE = 2

WORKERS = 0


# ======================================================================
# 6. ENVIRONMENT INFORMATION
# ======================================================================

print("=" * 90)
print("E12 — YOLO11m / 7 CLASS / AUGMENTATION / CLASS BALANCING")
print("=" * 90)

print(
    "Python          :",
    sys.version.split()[0]
)

print(
    "PyTorch         :",
    torch.__version__
)

print(
    "CUDA available  :",
    torch.cuda.is_available()
)


try:

    import ultralytics

    print(
        "Ultralytics    :",
        ultralytics.__version__
    )

except Exception:

    pass


if torch.cuda.is_available():

    DEVICE = 0

    print(
        "GPU             :",
        torch.cuda.get_device_name(0)
    )

    gpu_memory = (
        torch.cuda.get_device_properties(0).total_memory
        /
        1024 ** 3
    )

    print(
        "GPU memory      :",
        f"{gpu_memory:.2f} GB"
    )

else:

    DEVICE = "cpu"

    print(
        "\nWARNING: CUDA unavailable."
    )

    print(
        "YOLO11m training on CPU will be extremely slow."
    )


# ======================================================================
# 7. VERIFY EXISTING E5 DATASET
# ======================================================================

print(
    "\n"
    + "=" * 90
)

print(
    "EXISTING E5 DATASET CHECK"
)

print(
    "=" * 90
)


print(
    "Dataset root:"
)

print(
    E5_DATASET_ROOT
)


print(
    "\nDataset root exists:",
    E5_DATASET_ROOT.exists()
)


print(
    "\nDataset YAML:"
)

print(
    E5_DATA_YAML
)


print(
    "\nDataset YAML exists:",
    E5_DATA_YAML.exists()
)


if not E5_DATASET_ROOT.exists():

    raise FileNotFoundError(
        "\nExisting E5 dataset directory was not found:"
        f"\n{E5_DATASET_ROOT}"
        "\n\nRun/restore E5 dataset generation before E12."
    )


if not E5_DATA_YAML.exists():

    raise FileNotFoundError(
        "\nExisting E5 dataset YAML was not found:"
        f"\n{E5_DATA_YAML}"
    )


# ======================================================================
# 8. VERIFY E5 DATASET CONTENT
# ======================================================================

train_images_dir = (
    E5_DATASET_ROOT
    / "train"
    / "images"
)

train_labels_dir = (
    E5_DATASET_ROOT
    / "train"
    / "labels"
)

val_images_dir = (
    E5_DATASET_ROOT
    / "val"
    / "images"
)

val_labels_dir = (
    E5_DATASET_ROOT
    / "val"
    / "labels"
)


IMAGE_EXTENSIONS = {
    ".jpg",
    ".jpeg",
    ".png",
    ".bmp",
    ".tif",
    ".tiff",
    ".webp",
}


def count_images(directory):

    return len(
        [
            p

            for p in directory.iterdir()

            if (
                p.is_file()
                and
                p.suffix.lower()
                in IMAGE_EXTENSIONS
            )
        ]
    )


if not train_images_dir.exists():

    raise FileNotFoundError(
        f"\nMissing train image directory:\n{train_images_dir}"
    )


if not val_images_dir.exists():

    raise FileNotFoundError(
        f"\nMissing validation image directory:\n{val_images_dir}"
    )


train_image_count = count_images(
    train_images_dir
)

val_image_count = count_images(
    val_images_dir
)

train_label_count = len(
    list(
        train_labels_dir.glob(
            "*.txt"
        )
    )
)

val_label_count = len(
    list(
        val_labels_dir.glob(
            "*.txt"
        )
    )
)


print(
    "\n"
    + "=" * 90
)

print(
    "E5 DATASET COUNTS"
)

print(
    "=" * 90
)


print(
    "Train images :",
    train_image_count
)

print(
    "Train labels :",
    train_label_count
)

print(
    "Val images   :",
    val_image_count
)

print(
    "Val labels   :",
    val_label_count
)


# Expected E5:
#
# Train = 6575 effective oversampled images
# Val   = 780 untouched validation images
#

if train_image_count != train_label_count:

    raise RuntimeError(
        "\nTrain image/label count mismatch."
    )


if val_image_count != val_label_count:

    raise RuntimeError(
        "\nValidation image/label count mismatch."
    )


if train_image_count != 6575:

    print(
        "\nWARNING:"
        f"\nExpected 6575 E5 training images,"
        f" but found {train_image_count}."
    )


if val_image_count != 780:

    print(
        "\nWARNING:"
        f"\nExpected 780 E5 validation images,"
        f" but found {val_image_count}."
    )


# ======================================================================
# 9. CLEAR CUDA CACHE
# ======================================================================

if torch.cuda.is_available():

    torch.cuda.empty_cache()


# ======================================================================
# 10. LOAD PRETRAINED YOLO11m
# ======================================================================

print(
    "\n"
    + "=" * 90
)

print(
    "LOADING YOLO11m"
)

print(
    "=" * 90
)


model = YOLO(
    "yolo11m.pt"
)


print(
    "YOLO11m loaded successfully."
)


# ======================================================================
# 11. DISPLAY FINAL E12 CONFIGURATION
# ======================================================================

print(
    "\n"
    + "=" * 90
)

print(
    "FINAL E12 TRAINING CONFIGURATION"
)

print(
    "=" * 90
)


print(
    f"""
Experiment:
    E12

Model:
    YOLO11m

Reference experiment:
    E5 / E7 YOLO11s

Primary experimental change:
    YOLO11s -> YOLO11m

Taxonomy:
    7 classes

    0 = ecal
    1 = hdpe
    2 = mixed_plastic_rigid
    3 = mixed_plastic_soft
    4 = non_plastic
    5 = pet
    6 = pet_oil

Non-plastic:
    original metal + cardboard

Training dataset:
    EXISTING E5 class-aware oversampled dataset

Expected effective training images:
    6575

Validation images:
    780
    no oversampling

Input resolution:
    640 x 640

Optimizer:
    AdamW

Maximum epochs:
    {MAX_EPOCHS}

Early stopping:
    patience = {PATIENCE}

Batch:
    {BATCH_SIZE}

Augmentation:
    Mosaic             = 0.80
    MixUp              = 0.05
    Rotation           = +/- 5 degrees
    Translation        = 0.10
    Scale              = 0.30
    Horizontal flip    = 0.50
    Vertical flip      = 0
    HSV H              = 0.015
    HSV S              = 0.40
    HSV V              = 0.30
    Shear              = 0
    Perspective        = 0

Oversampling inherited from E5:
    non_plastic        = 2x targeted exposure
    pet_oil            = 3x targeted exposure

Final epochs:
    Mosaic disabled for final 10 epochs
"""
)


# ======================================================================
# 12. TRAIN E12
# ======================================================================

print(
    "\n"
    + "=" * 90
)

print(
    "STARTING E12 TRAINING"
)

print(
    "=" * 90
)


train_results = model.train(

    # --------------------------------------------------------------
    # Dataset
    # --------------------------------------------------------------

    data=str(
        E5_DATA_YAML
    ),

    # --------------------------------------------------------------
    # Resolution
    # SAME AS E5
    # --------------------------------------------------------------

    imgsz=640,

    # --------------------------------------------------------------
    # Training length
    # SAME AS E5
    # --------------------------------------------------------------

    epochs=150,

    patience=15,

    # --------------------------------------------------------------
    # Hardware
    #
    # E5 = batch 4
    # E12 = batch 2 because YOLO11m is larger
    # --------------------------------------------------------------

    batch=BATCH_SIZE,

    device=DEVICE,

    workers=0,

    cache=False,

    # --------------------------------------------------------------
    # Optimization
    # SAME AS E5
    # --------------------------------------------------------------

    optimizer="AdamW",

    cos_lr=True,

    amp=True,

    pretrained=True,

    # --------------------------------------------------------------
    # Realistic augmentation
    # EXACTLY SAME AS E5
    # --------------------------------------------------------------

    mosaic=0.80,

    mixup=0.05,

    degrees=5.0,

    translate=0.10,

    scale=0.30,

    flipud=0.0,

    fliplr=0.50,

    hsv_h=0.015,

    hsv_s=0.40,

    hsv_v=0.30,

    shear=0.0,

    perspective=0.0,

    close_mosaic=10,

    # --------------------------------------------------------------
    # Reproducibility
    # SAME AS E5
    # --------------------------------------------------------------

    seed=SEED,

    deterministic=True,

    # --------------------------------------------------------------
    # Validation
    # --------------------------------------------------------------

    val=True,

    # --------------------------------------------------------------
    # Output
    # --------------------------------------------------------------

    project=str(
        PROJECT_ROOT
    ),

    name=EXPERIMENT_NAME,

    exist_ok=False,

    plots=True,

    verbose=True,
)


# ======================================================================
# 13. LOCATE CHECKPOINTS
# ======================================================================

BEST_MODEL = (
    RUN_DIR
    / "weights"
    / "best.pt"
)


LAST_MODEL = (
    RUN_DIR
    / "weights"
    / "last.pt"
)


print(
    "\n"
    + "=" * 90
)

print(
    "E12 TRAINING COMPLETE"
)

print(
    "=" * 90
)




E12 — YOLO11m / 7 CLASS / AUGMENTATION / CLASS BALANCING
Python          : 3.11.15
PyTorch         : 2.13.0+cu126
CUDA available  : True
Ultralytics    : 8.4.126
GPU             : NVIDIA GeForce RTX 3050 Ti Laptop GPU
GPU memory      : 4.00 GB

EXISTING E5 DATASET CHECK
Dataset root:
C:\Users\varda\Documents\_My Computer\COMMON_space\Learning\Upgrad_EPGP_MS\upgrad_IIITB_MS_course\MS_LJMU_Material\Thesis_Code\runs\sortwaste\E5_7class_dataset

Dataset root exists: True

Dataset YAML:
C:\Users\varda\Documents\_My Computer\COMMON_space\Learning\Upgrad_EPGP_MS\upgrad_IIITB_MS_course\MS_LJMU_Material\Thesis_Code\runs\sortwaste\E5_7class_dataset\E5_7class.yaml

Dataset YAML exists: True

E5 DATASET COUNTS
Train images : 6575
Train labels : 6575
Val images   : 780
Val labels   : 780

LOADING YOLO11m
YOLO11m loaded successfully.

FINAL E12 TRAINING CONFIGURATION

Experiment:
    E12

Model:
    YOLO11m

Reference experiment:
    E5 / E7 YOLO11s

Primary experimental change:
    YOLO11s -> YOLO1

FileNotFoundError: 
Best E12 checkpoint not found:
C:\Users\varda\Documents\_My Computer\COMMON_space\Learning\Upgrad_EPGP_MS\upgrad_IIITB_MS_course\MS_LJMU_Material\Thesis_Code\runs\sortwaste\E12_yolo11m_7class_aug_classbalance_640\weights\best.pt

In [ ]:
## Final validation
print(
    "\nBest checkpoint:"
)

print(
    BEST_MODEL
)


print(
    "Exists:",
    BEST_MODEL.exists()
)


print(
    "\nLast checkpoint:"
)

print(
    LAST_MODEL
)


print(
    "Exists:",
    LAST_MODEL.exists()
)


if not BEST_MODEL.exists():

    raise FileNotFoundError(
        f"\nBest E12 checkpoint not found:\n{BEST_MODEL}"
    )



# ======================================================================
# 14. CLEAR CACHE BEFORE FINAL VALIDATION
# ======================================================================

if torch.cuda.is_available():

    torch.cuda.empty_cache()


# ======================================================================
# 15. FINAL VALIDATION
# ======================================================================
#
# IMPORTANT:
#
# E7 established NMS IoU = 0.60 as the best setting for E5.
#
# We therefore evaluate E12 with:
#
#     conf    = 0.001
#     NMS IoU = 0.60
#     max_det = 100
#
# This gives us a practical comparison against E7.
#
# ======================================================================

print(
    "\n"
    + "=" * 90
)

print(
    "E12 FINAL 7-CLASS VALIDATION"
)

print(
    "=" * 90
)


best_model = YOLO(
    str(
        BEST_MODEL
    )
)


val_results = best_model.val(

    data=str(
        E5_DATA_YAML
    ),

    split="val",

    imgsz=640,

    # Safer for YOLO11m on 4 GB VRAM.
    batch=BATCH_SIZE,

    device=DEVICE,

    workers=0,

    # Low confidence floor for AP calculation
    conf=0.001,

    # E7-selected NMS IoU
    iou=0.60,

    max_det=100,

    plots=True,

    project=str(
        PROJECT_ROOT
    ),

    name=(
        EXPERIMENT_NAME
        + "_final_val_nms060"
    ),

    exist_ok=True,
)


# ======================================================================
# 16. OVERALL RESULTS
# ======================================================================

print(
    "\n"
    + "=" * 90
)

print(
    "E12 OVERALL RESULTS — 7 CLASS"
)

print(
    "=" * 90
)


print(
    f"mAP50-95 : "
    f"{val_results.box.map:.6f} "
    f"= {val_results.box.map * 100:.2f}%"
)


print(
    f"mAP50    : "
    f"{val_results.box.map50:.6f} "
    f"= {val_results.box.map50 * 100:.2f}%"
)


print(
    f"mAP75    : "
    f"{val_results.box.map75:.6f} "
    f"= {val_results.box.map75 * 100:.2f}%"
)


print(
    f"Precision : "
    f"{val_results.box.mp:.6f} "
    f"= {val_results.box.mp * 100:.2f}%"
)


print(
    f"Recall    : "
    f"{val_results.box.mr:.6f} "
    f"= {val_results.box.mr * 100:.2f}%"
)


# ======================================================================
# 17. PER-CLASS RESULTS
# ======================================================================

print(
    "\n"
    + "=" * 90
)

print(
    "E12 PER-CLASS RESULTS"
)

print(
    "=" * 90
)


class_names = best_model.names


print(
    f"{'Class':25s}"
    f"{'Precision':>12s}"
    f"{'Recall':>12s}"
    f"{'AP50':>12s}"
    f"{'AP50-95':>12s}"
)


print(
    "-" * 73
)


for class_id in range(
    len(class_names)
):

    result = val_results.box.class_result(
        class_id
    )


    precision = result[0]

    recall = result[1]

    ap50 = result[2]

    ap5095 = result[3]


    print(
        f"{class_names[class_id]:25s}"
        f"{precision:12.4f}"
        f"{recall:12.4f}"
        f"{ap50:12.4f}"
        f"{ap5095:12.4f}"
    )


# ======================================================================
# 18. E7 REFERENCE
# ======================================================================

print(
    "\n"
    + "=" * 90
)

print(
    "REFERENCE — E7 YOLO11s @640"
)

print(
    "=" * 90
)


print(
    """
E7 / E5 best.pt with NMS IoU = 0.60

mAP50-95 = 0.418602 = 41.86%
mAP50    = 0.569952 = 57.00%
mAP75    = 0.473164 = 47.32%
Precision = 0.563418
Recall    = 0.569719

Per-class AP50-95:

ecal                  = 0.5563
hdpe                  = 0.5704
mixed_plastic_rigid   = 0.2954
mixed_plastic_soft    = 0.2654
non_plastic           = 0.1011
pet                   = 0.5768
pet_oil               = 0.5647
"""
)


# ======================================================================
# 19. PRINT E12 VS E7 DELTAS
# ======================================================================

E7_MAP = 0.418602
E7_MAP50 = 0.569952
E7_MAP75 = 0.473164


map_delta = (
    val_results.box.map
    -
    E7_MAP
)


map50_delta = (
    val_results.box.map50
    -
    E7_MAP50
)


map75_delta = (
    val_results.box.map75
    -
    E7_MAP75
)


print(
    "\n"
    + "=" * 90
)

print(
    "E12 VS E7"
)

print(
    "=" * 90
)


print(
    f"mAP50-95 delta : "
    f"{map_delta * 100:+.2f} percentage points"
)


print(
    f"mAP50 delta    : "
    f"{map50_delta * 100:+.2f} percentage points"
)


print(
    f"mAP75 delta    : "
    f"{map75_delta * 100:+.2f} percentage points"
)


# ======================================================================
# 20. SAVE SUMMARY
# ======================================================================

SUMMARY_FILE = (
    RUN_DIR
    / "E12_7class_final_summary.txt"
)


with open(
    SUMMARY_FILE,
    "w",
    encoding="utf-8"
) as f:

    f.write(
        "E12 — YOLO11m + 7-class taxonomy + realistic augmentation "
        "+ class-aware oversampling @640\n"
    )

    f.write(
        "=" * 80
        + "\n\n"
    )


    f.write(
        "Primary comparison:\n"
    )

    f.write(
        "E5/E7 YOLO11s @640 -> E12 YOLO11m @640\n\n"
    )


    f.write(
        f"Training dataset = {E5_DATA_YAML}\n"
    )

    f.write(
        f"Train images     = {train_image_count}\n"
    )

    f.write(
        f"Val images       = {val_image_count}\n\n"
    )


    f.write(
        f"Batch = {BATCH_SIZE}\n"
    )

    f.write(
        "imgsz = 640\n"
    )

    f.write(
        "NMS IoU final validation = 0.60\n\n"
    )


    f.write(
        f"mAP50-95 = {val_results.box.map:.6f}\n"
    )

    f.write(
        f"mAP50    = {val_results.box.map50:.6f}\n"
    )

    f.write(
        f"mAP75    = {val_results.box.map75:.6f}\n"
    )

    f.write(
        f"Precision = {val_results.box.mp:.6f}\n"
    )

    f.write(
        f"Recall    = {val_results.box.mr:.6f}\n\n"
    )


    f.write(
        "E12 vs E7\n"
    )

    f.write(
        "-" * 80
        + "\n"
    )


    f.write(
        f"mAP50-95 delta = {map_delta * 100:+.4f} pp\n"
    )

    f.write(
        f"mAP50 delta    = {map50_delta * 100:+.4f} pp\n"
    )

    f.write(
        f"mAP75 delta    = {map75_delta * 100:+.4f} pp\n\n"
    )


    f.write(
        "Per-class results\n"
    )

    f.write(
        "-" * 80
        + "\n"
    )


    for class_id in range(
        len(class_names)
    ):

        result = val_results.box.class_result(
            class_id
        )


        f.write(
            f"{class_names[class_id]} | "
            f"P={result[0]:.6f} | "
            f"R={result[1]:.6f} | "
            f"AP50={result[2]:.6f} | "
            f"AP50-95={result[3]:.6f}\n"
        )


# ======================================================================
# 21. COMPLETE
# ======================================================================

print(
    "\n"
    + "=" * 90
)

print(
    "E12 COMPLETE"
)

print(
    "=" * 90
)


print(
    "\nSummary:"
)

print(
    SUMMARY_FILE
)


print(
    "\nBest model:"
)

print(
    BEST_MODEL
)


print(
    "\nNext decision:"
)

print(
    """
If E12 meaningfully improves over E7:
    -> use E12 YOLO11m detections with the EXISTING E8 MobileNet
       without retraining MobileNet.

If E12 does not improve:
    -> stop detector training
    -> retain E8 as final proposed system
    -> proceed to untouched test evaluation.
"""
)


Best checkpoint:
C:\Users\varda\Documents\_My Computer\COMMON_space\Learning\Upgrad_EPGP_MS\upgrad_IIITB_MS_course\MS_LJMU_Material\Thesis_Code\runs\sortwaste\E12_yolo11m_7class_aug_classbalance_640\weights\best.pt
Exists: True

Last checkpoint:
C:\Users\varda\Documents\_My Computer\COMMON_space\Learning\Upgrad_EPGP_MS\upgrad_IIITB_MS_course\MS_LJMU_Material\Thesis_Code\runs\sortwaste\E12_yolo11m_7class_aug_classbalance_640\weights\last.pt
Exists: True

E12 FINAL 7-CLASS VALIDATION
Ultralytics 8.4.126  Python-3.11.15 torch-2.13.0+cu126 CUDA:0 (NVIDIA GeForce RTX 3050 Ti Laptop GPU, 4096MiB)
YOLO11m summary (fused): 126 layers, 20,035,429 parameters, 0 gradients, 67.8 GFLOPs
val: Fast image access  (ping: 0.10.0 ms, read: 684.474.0 MB/s, size: 2045.5 KB)
val: Scanning C:\Users\varda\Documents\_My Computer\COMMON_space\Learning\Upgrad_EPGP_MS\upgrad_IIITB_MS_course\MS_LJMU_Material\Thesis_Code\runs\sortwaste\E5_7class_dataset\val\labels.cache... 780 images, 0 backgrounds, 0 corrupt: 100

# Model E13 — E12 YOLO11m + Existing E8 MobileNet End-to-End

In [ ]:
# E13 — YOLO11m + MobileNetV3-Large, Class-Weighted - End-to-End 7-Class COCO Evaluation
#
# Detector:
#   E12 YOLO11m best.pt
#
# Classifier:
#   Existing E8 / E3Y-B class-weighted MobileNetV3-Large
#
# Pipeline:
#   YOLO11m detection
#       -> detector bbox crop
#       -> MobileNet classification
#       -> final class = MobileNet prediction
#       -> final score = YOLO confidence * MobileNet probability
#       -> explicit pycocotools COCOeval
#
# No retraining.
# ============================================================

import os
import json
import time
import random
from pathlib import Path

import numpy as np
from PIL import Image

import torch
import torch.nn as nn
from torchvision import models, transforms

from ultralytics import YOLO

from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval


# ============================================================
# 1. REPRODUCIBILITY
# ============================================================

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("=" * 90)
print("E13 — YOLO11m + MobileNetV3-Large, Class-Weighted")
print("=" * 90)
print("Device:", DEVICE)


# ============================================================
# 2. PATHS
# ============================================================

BASE = Path(
    r"C:\Users\varda\Documents\_My Computer\COMMON_space\Learning"
    r"\Upgrad_EPGP_MS\upgrad_IIITB_MS_course\MS_LJMU_Material"
)

THESIS_CODE = BASE / "Thesis_Code"

DATASET_ROOT = (
    BASE
    / "Topic Data"
    / "SortWaste"
    / "dataset"
    / "dataset"
)

COCO_VAL_ROOT = DATASET_ROOT / "splited_all_dataset_coco" / "val"

VAL_IMAGES = COCO_VAL_ROOT / "images"
VAL_COCO_JSON = COCO_VAL_ROOT / "annotations" / "val_coco.json"


# ----------------------------
# E12 detector
# ----------------------------

YOLO_CKPT = (
    THESIS_CODE
    / "runs"
    / "sortwaste"
    / "E12_yolo11m_7class_aug_classbalance_640"
    / "weights"
    / "best.pt"
)


# ----------------------------
# Existing E8 MobileNet
# This is the E3Y-B weighted classifier
# ----------------------------

MOBILENET_CKPT = (
    DATASET_ROOT
    / "yolo_mobilenet_crops_E3Y"
    / "mobilenet_results"
    / "E3Y_B_class_weighted"
    / "E3Y_B_MobileNetV3Large_best.pth"
)


# ----------------------------
# Output
# ----------------------------

OUTPUT_DIR = (
    THESIS_CODE
    / "runs"
    / "sortwaste"
    / "E13_yolo11m_mobilenet_endtoend"
)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

PRED_JSON = OUTPUT_DIR / "E13_predictions.json"
SUMMARY_TXT = OUTPUT_DIR / "E13_final_summary.txt"


print("\nPaths:")
print("YOLO checkpoint :", YOLO_CKPT)
print("MobileNet       :", MOBILENET_CKPT)
print("Val images      :", VAL_IMAGES)
print("COCO GT         :", VAL_COCO_JSON)
print("Output          :", OUTPUT_DIR)

assert YOLO_CKPT.exists(), f"Missing YOLO checkpoint: {YOLO_CKPT}"
assert MOBILENET_CKPT.exists(), f"Missing MobileNet checkpoint: {MOBILENET_CKPT}"
assert VAL_IMAGES.exists(), f"Missing val images: {VAL_IMAGES}"
assert VAL_COCO_JSON.exists(), f"Missing COCO annotation: {VAL_COCO_JSON}"


# ============================================================
# 3. 7-CLASS TAXONOMY
# ============================================================

CLASS_NAMES = [
    "ecal",
    "hdpe",
    "mixed_plastic_rigid",
    "mixed_plastic_soft",
    "non_plastic",
    "pet",
    "pet_oil",
]

NUM_CLASSES = len(CLASS_NAMES)

# MobileNet index -> evaluation category_id
# Our evaluation categories are 1-based:
#
# 1 ecal
# 2 hdpe
# 3 mixed_plastic_rigid
# 4 mixed_plastic_soft
# 5 non_plastic
# 6 pet
# 7 pet_oil

MOBILENET_IDX_TO_COCO_ID = {
    0: 1,
    1: 2,
    2: 3,
    3: 4,
    4: 5,
    5: 6,
    6: 7,
}


# Original SortWaste COCO category -> our 7-class category
ORIGINAL_COCO_TO_7CLASS = {
    1: 6,  # pet
    2: 2,  # hdpe
    3: 4,  # mixed_plastic_soft
    4: 1,  # ecal
    5: 5,  # metal -> non_plastic
    6: 5,  # cardboard -> non_plastic
    7: 3,  # mixed_plastic_rigid
    8: 7,  # pet_oil
}


# ============================================================
# 4. LOAD ORIGINAL COCO GT
# ============================================================

with open(VAL_COCO_JSON, "r", encoding="utf-8") as f:
    original_coco = json.load(f)

print("\nOriginal COCO:")
print("Images      :", len(original_coco["images"]))
print("Annotations :", len(original_coco["annotations"]))


# ============================================================
# 5. BUILD 7-CLASS COCO GROUND TRUTH
# ============================================================

categories_7 = [
    {"id": i + 1, "name": name}
    for i, name in enumerate(CLASS_NAMES)
]

annotations_7 = []

for ann in original_coco["annotations"]:

    old_category = int(ann["category_id"])
    new_category = ORIGINAL_COCO_TO_7CLASS[old_category]

    new_ann = ann.copy()
    new_ann["category_id"] = new_category

    # COCOeval bbox mode only;
    # segmentation may remain empty.
    annotations_7.append(new_ann)


coco_gt_dict = {
    "images": original_coco["images"],
    "annotations": annotations_7,
    "categories": categories_7,
}


GT_7CLASS_JSON = OUTPUT_DIR / "E13_val_gt_7class.json"

with open(GT_7CLASS_JSON, "w", encoding="utf-8") as f:
    json.dump(coco_gt_dict, f)

print("\n7-class COCO GT saved:")
print(GT_7CLASS_JSON)


# ============================================================
# 6. IMAGE LOOKUPS
# ============================================================

image_id_by_filename = {
    img["file_name"]: img["id"]
    for img in original_coco["images"]
}

image_info_by_id = {
    img["id"]: img
    for img in original_coco["images"]
}


# ============================================================
# 7. LOAD YOLO11m DETECTOR
# ============================================================

print("\nLoading E12 YOLO11m...")

yolo = YOLO(str(YOLO_CKPT))

print("YOLO loaded.")


# ============================================================
# 8. LOAD MOBILENETV3-LARGE
# ============================================================

print("\nLoading MobileNetV3-Large...")

mobilenet = models.mobilenet_v3_large(
    weights=None
)

# MobileNetV3 Large final layer:
# classifier[3]
in_features = mobilenet.classifier[3].in_features

mobilenet.classifier[3] = nn.Linear(
    in_features,
    NUM_CLASSES
)


checkpoint = torch.load(
    MOBILENET_CKPT,
    map_location=DEVICE
)

# Support both:
# 1. direct state_dict
# 2. checkpoint dict containing model_state_dict/state_dict
if isinstance(checkpoint, dict):

    if "model_state_dict" in checkpoint:
        state_dict = checkpoint["model_state_dict"]

    elif "state_dict" in checkpoint:
        state_dict = checkpoint["state_dict"]

    else:
        # checkpoint itself may already be the state dict
        state_dict = checkpoint

else:
    state_dict = checkpoint


# Remove "module." prefix if checkpoint came from DataParallel
clean_state_dict = {}

for key, value in state_dict.items():

    if key.startswith("module."):
        key = key[len("module."):]

    clean_state_dict[key] = value


mobilenet.load_state_dict(
    clean_state_dict,
    strict=True
)

mobilenet = mobilenet.to(DEVICE)
mobilenet.eval()

print("MobileNet loaded.")


# ============================================================
# 9. MOBILENET TRANSFORM
# Must match E3Y-B validation transform
# ============================================================

mobilenet_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    ),
])


# ============================================================
# 10. INFERENCE SETTINGS
# Same operating point as E7 / E8
# ============================================================

IMG_SIZE = 640
YOLO_CONF = 0.001
NMS_IOU = 0.60
MAX_DET = 100

print("\nInference settings:")
print("imgsz     :", IMG_SIZE)
print("conf      :", YOLO_CONF)
print("NMS IoU   :", NMS_IOU)
print("max_det   :", MAX_DET)


# ============================================================
# 11. RUN E13 END-TO-END INFERENCE
# ============================================================

print("\n" + "=" * 90)
print("RUNNING E13 INFERENCE")
print("=" * 90)

start_time = time.time()

predictions = []

total_yolo_detections = 0
total_valid_crops = 0
total_invalid_crops = 0


# Run YOLO over whole validation directory
results = yolo.predict(
    source=str(VAL_IMAGES),
    imgsz=IMG_SIZE,
    conf=YOLO_CONF,
    iou=NMS_IOU,
    max_det=MAX_DET,
    device=0 if torch.cuda.is_available() else "cpu",
    verbose=False,
    stream=True,
)


for image_index, result in enumerate(results, start=1):

    image_path = Path(result.path)
    filename = image_path.name

    if filename not in image_id_by_filename:
        raise KeyError(
            f"Filename not found in COCO JSON: {filename}"
        )

    image_id = image_id_by_filename[filename]

    image = Image.open(image_path).convert("RGB")

    width, height = image.size

    boxes = result.boxes

    if boxes is None or len(boxes) == 0:
        continue


    xyxy = boxes.xyxy.detach().cpu().numpy()
    yolo_confidences = boxes.conf.detach().cpu().numpy()

    total_yolo_detections += len(xyxy)


    # --------------------------------------------------------
    # Prepare all valid crops from this image
    # --------------------------------------------------------

    crop_tensors = []
    valid_detection_indices = []
    valid_boxes = []


    for det_idx, box in enumerate(xyxy):

        x1, y1, x2, y2 = map(float, box)

        # Clip coordinates
        x1 = max(0.0, min(x1, width))
        y1 = max(0.0, min(y1, height))
        x2 = max(0.0, min(x2, width))
        y2 = max(0.0, min(y2, height))

        # Invalid / zero-area crop
        if x2 <= x1 or y2 <= y1:

            total_invalid_crops += 1
            continue


        crop = image.crop(
            (
                int(round(x1)),
                int(round(y1)),
                int(round(x2)),
                int(round(y2)),
            )
        )

        if crop.width <= 0 or crop.height <= 0:

            total_invalid_crops += 1
            continue


        crop_tensor = mobilenet_transform(crop)

        crop_tensors.append(crop_tensor)
        valid_detection_indices.append(det_idx)
        valid_boxes.append((x1, y1, x2, y2))


    if not crop_tensors:
        continue


    total_valid_crops += len(crop_tensors)


    # --------------------------------------------------------
    # MobileNet batch inference
    # --------------------------------------------------------

    batch = torch.stack(crop_tensors).to(DEVICE)

    with torch.no_grad():

        logits = mobilenet(batch)

        probs = torch.softmax(
            logits,
            dim=1
        )

        predicted_probs, predicted_classes = torch.max(
            probs,
            dim=1
        )


    predicted_probs = predicted_probs.cpu().numpy()
    predicted_classes = predicted_classes.cpu().numpy()


    # --------------------------------------------------------
    # Convert to COCO predictions
    # --------------------------------------------------------

    for local_idx, det_idx in enumerate(valid_detection_indices):

        x1, y1, x2, y2 = valid_boxes[local_idx]

        bbox_w = x2 - x1
        bbox_h = y2 - y1

        mobilenet_idx = int(
            predicted_classes[local_idx]
        )

        final_category_id = MOBILENET_IDX_TO_COCO_ID[
            mobilenet_idx
        ]

        yolo_score = float(
            yolo_confidences[det_idx]
        )

        mobilenet_score = float(
            predicted_probs[local_idx]
        )

        # Same score fusion as E8 / E11-B best variant
        final_score = (
            yolo_score * mobilenet_score
        )


        predictions.append({
            "image_id": int(image_id),
            "category_id": int(final_category_id),
            "bbox": [
                float(x1),
                float(y1),
                float(bbox_w),
                float(bbox_h),
            ],
            "score": float(final_score),
        })


    if image_index % 100 == 0:

        elapsed = time.time() - start_time

        print(
            f"Processed {image_index}/780 images | "
            f"Predictions: {len(predictions)} | "
            f"Elapsed: {elapsed / 60:.2f} min"
        )


elapsed = time.time() - start_time


print("\nInference complete.")

print("YOLO detections :", total_yolo_detections)
print("Valid crops     :", total_valid_crops)
print("Invalid crops   :", total_invalid_crops)
print("Final preds     :", len(predictions))
print(f"Time            : {elapsed / 60:.2f} min")


# ============================================================
# 12. SAVE PREDICTIONS
# ============================================================

with open(PRED_JSON, "w", encoding="utf-8") as f:
    json.dump(predictions, f)

print("\nPredictions saved:")
print(PRED_JSON)


# ============================================================
# 13. COCOEVAL
# ============================================================

print("\n" + "=" * 90)
print("E13 EXPLICIT PYCOCOTOOLS COCOEVAL")
print("=" * 90)

coco_gt = COCO(str(GT_7CLASS_JSON))

if len(predictions) == 0:
    raise RuntimeError("No predictions produced.")

coco_dt = coco_gt.loadRes(str(PRED_JSON))

coco_eval = COCOeval(
    coco_gt,
    coco_dt,
    iouType="bbox"
)

coco_eval.params.maxDets = [
    1,
    10,
    100
]

coco_eval.evaluate()
coco_eval.accumulate()
coco_eval.summarize()


stats = coco_eval.stats

map_50_95 = float(stats[0])
map_50 = float(stats[1])
map_75 = float(stats[2])

ar_1 = float(stats[6])
ar_10 = float(stats[7])
ar_100 = float(stats[8])


# ============================================================
# 14. PER-CLASS AP
# ============================================================

precision = coco_eval.eval["precision"]

# Shape:
# [IoU, Recall, Category, Area, MaxDets]

iou_thresholds = coco_eval.params.iouThrs

iou50_idx = np.where(
    np.isclose(iou_thresholds, 0.50)
)[0][0]


per_class_results = {}


for class_index, class_name in enumerate(CLASS_NAMES):

    # AP50-95
    p = precision[
        :,
        :,
        class_index,
        0,
        -1
    ]

    valid = p[p > -1]

    class_ap = (
        float(np.mean(valid))
        if valid.size
        else float("nan")
    )


    # AP50
    p50 = precision[
        iou50_idx,
        :,
        class_index,
        0,
        -1
    ]

    valid50 = p50[p50 > -1]

    class_ap50 = (
        float(np.mean(valid50))
        if valid50.size
        else float("nan")
    )


    per_class_results[class_name] = {
        "AP50": class_ap50,
        "AP50-95": class_ap,
    }


# ============================================================
# 15. PRINT FINAL RESULTS
# ============================================================

print("\n" + "=" * 90)
print("E13 OVERALL RESULTS")
print("=" * 90)

print(
    f"mAP50-95 : {map_50_95:.6f} "
    f"= {map_50_95 * 100:.2f}%"
)

print(
    f"mAP50    : {map_50:.6f} "
    f"= {map_50 * 100:.2f}%"
)

print(
    f"mAP75    : {map_75:.6f} "
    f"= {map_75 * 100:.2f}%"
)

print(
    f"AR1      : {ar_1:.6f}"
)

print(
    f"AR10     : {ar_10:.6f}"
)

print(
    f"AR100    : {ar_100:.6f}"
)


print("\n" + "=" * 90)
print("E13 PER-CLASS RESULTS")
print("=" * 90)

print(
    f"{'Class':28s} "
    f"{'AP50':>12s} "
    f"{'AP50-95':>12s}"
)

print("-" * 55)

for name in CLASS_NAMES:

    vals = per_class_results[name]

    print(
        f"{name:28s} "
        f"{vals['AP50']:12.4f} "
        f"{vals['AP50-95']:12.4f}"
    )


# ============================================================
# 16. E8 REFERENCE
# ============================================================

E8_MAP = 0.420495
E8_MAP50 = 0.578265
E8_MAP75 = 0.470885


print("\n" + "=" * 90)
print("REFERENCE — E8 YOLO11s + MobileNet")
print("=" * 90)

print(
    f"E8 mAP50-95 : {E8_MAP:.6f} "
    f"= {E8_MAP * 100:.2f}%"
)

print(
    f"E8 mAP50    : {E8_MAP50:.6f} "
    f"= {E8_MAP50 * 100:.2f}%"
)

print(
    f"E8 mAP75    : {E8_MAP75:.6f} "
    f"= {E8_MAP75 * 100:.2f}%"
)


print("\n" + "=" * 90)
print("E13 VS E8")
print("=" * 90)

print(
    f"mAP50-95 delta : "
    f"{(map_50_95 - E8_MAP) * 100:+.2f} percentage points"
)

print(
    f"mAP50 delta    : "
    f"{(map_50 - E8_MAP50) * 100:+.2f} percentage points"
)

print(
    f"mAP75 delta    : "
    f"{(map_75 - E8_MAP75) * 100:+.2f} percentage points"
)


# ============================================================
# 17. SAVE SUMMARY
# ============================================================

with open(
    SUMMARY_TXT,
    "w",
    encoding="utf-8"
) as f:

    f.write(
        "E13 — YOLO11m + MobileNetV3-Large, "
        "Class-Weighted\n"
    )

    f.write("=" * 80 + "\n\n")

    f.write(f"YOLO checkpoint: {YOLO_CKPT}\n")
    f.write(f"MobileNet checkpoint: {MOBILENET_CKPT}\n\n")

    f.write(
        f"YOLO detections: {total_yolo_detections}\n"
    )

    f.write(
        f"Valid crops: {total_valid_crops}\n"
    )

    f.write(
        f"Invalid crops: {total_invalid_crops}\n"
    )

    f.write(
        f"Final predictions: {len(predictions)}\n\n"
    )

    f.write(
        f"mAP50-95: {map_50_95:.6f}\n"
    )

    f.write(
        f"mAP50: {map_50:.6f}\n"
    )

    f.write(
        f"mAP75: {map_75:.6f}\n"
    )

    f.write(
        f"AR1: {ar_1:.6f}\n"
    )

    f.write(
        f"AR10: {ar_10:.6f}\n"
    )

    f.write(
        f"AR100: {ar_100:.6f}\n\n"
    )

    f.write("Per-class:\n")

    for name in CLASS_NAMES:

        vals = per_class_results[name]

        f.write(
            f"{name}: "
            f"AP50={vals['AP50']:.6f}, "
            f"AP50-95={vals['AP50-95']:.6f}\n"
        )


print("\nSummary saved:")
print(SUMMARY_TXT)

print("\n" + "=" * 90)
print("E13 COMPLETE")
print("=" * 90)

E13 — YOLO11m + MobileNetV3-Large, Class-Weighted
Device: cuda

Paths:
YOLO checkpoint : C:\Users\varda\Documents\_My Computer\COMMON_space\Learning\Upgrad_EPGP_MS\upgrad_IIITB_MS_course\MS_LJMU_Material\Thesis_Code\runs\sortwaste\E12_yolo11m_7class_aug_classbalance_640\weights\best.pt
MobileNet       : C:\Users\varda\Documents\_My Computer\COMMON_space\Learning\Upgrad_EPGP_MS\upgrad_IIITB_MS_course\MS_LJMU_Material\Topic Data\SortWaste\dataset\dataset\yolo_mobilenet_crops_E3Y\mobilenet_results\E3Y_B_class_weighted\E3Y_B_MobileNetV3Large_best.pth
Val images      : C:\Users\varda\Documents\_My Computer\COMMON_space\Learning\Upgrad_EPGP_MS\upgrad_IIITB_MS_course\MS_LJMU_Material\Topic Data\SortWaste\dataset\dataset\splited_all_dataset_coco\val\images
COCO GT         : C:\Users\varda\Documents\_My Computer\COMMON_space\Learning\Upgrad_EPGP_MS\upgrad_IIITB_MS_course\MS_LJMU_Material\Topic Data\SortWaste\dataset\dataset\splited_all_dataset_coco\val\annotations\val_coco.json
Output          

# Model E14 — YOLO11m + MobileNet Confidence Fusion and Class-Gating Optimization = E13 + Fusion/Gating

In [ ]:
# E14 — Fusion and Gating Ablation - YOLO11m + MobileNetV3-Large
# ============================================================
# E14 — YOLO11m + MobileNet Confidence Fusion
#       and Class-Gating Optimization
#
# CORRECTED VERSION
#
# NO RETRAINING
# NO YOLO/MOBILENET INFERENCE REQUIRED IF CACHE EXISTS
#
# Detector:
#   E12 YOLO11m best.pt
#
# Classifier:
#   Existing E3Y-B / E8 class-weighted MobileNetV3-Large
#
# E14-A:
#   Always use MobileNet class
#   Sweep confidence fusion weights
#
# E14-B:
#   Gate MobileNet class replacement by MobileNet confidence
#   Product score uses MobileNet probability of FINAL class
#
# E14-C:
#   Best gating threshold + fusion-weight sweep
#   Fusion uses MobileNet probability of FINAL class
#
# Primary metric:
#   Explicit pycocotools COCO AP50-95
# ============================================================

import json
from pathlib import Path

import numpy as np
import pandas as pd

from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval


# ============================================================
# 1. PATHS
# ============================================================

BASE = Path(
    r"C:\Users\varda\Documents\_My Computer\COMMON_space\Learning"
    r"\Upgrad_EPGP_MS\upgrad_IIITB_MS_course\MS_LJMU_Material"
)

THESIS_CODE = BASE / "Thesis_Code"

DATASET_ROOT = (
    BASE
    / "Topic Data"
    / "SortWaste"
    / "dataset"
    / "dataset"
)


# Original E14 output directory
E14_DIR = (
    THESIS_CODE
    / "runs"
    / "sortwaste"
    / "E14_fusion_gating_ablation"
)


# Existing cache generated during E14
CACHE_JSON = (
    E14_DIR
    / "E14_cached_predictions.json"
)


# Existing 7-class validation GT generated during E14
GT_JSON = (
    E14_DIR
    / "E14_val_gt_7class.json"
)


# Corrected outputs
RESULTS_CSV = (
    E14_DIR
    / "E14_corrected_results.csv"
)

SUMMARY_TXT = (
    E14_DIR
    / "E14_corrected_summary.txt"
)


assert CACHE_JSON.exists(), f"Missing cache: {CACHE_JSON}"
assert GT_JSON.exists(), f"Missing GT: {GT_JSON}"


print("=" * 100)
print("E14 — YOLO11m + MobileNet Fusion and Gating Ablation")
print("CORRECTED FINAL-CLASS CONFIDENCE VERSION")
print("=" * 100)

print("\nCache:")
print(CACHE_JSON)

print("\nGround truth:")
print(GT_JSON)

print("\nOutput directory:")
print(E14_DIR)


# ============================================================
# 2. LOAD CACHE
# ============================================================

with open(
    CACHE_JSON,
    "r",
    encoding="utf-8"
) as f:

    cached = json.load(f)


print("\nCached detections:", len(cached))


# Validate that full MobileNet probability vector exists
for i, row in enumerate(cached[:10]):

    assert "mobilenet_probs" in row, (
        "Cache does not contain mobilenet_probs."
    )

    assert len(row["mobilenet_probs"]) == 7, (
        f"Unexpected probability count at row {i}"
    )


# ============================================================
# 3. LOAD COCO GT
# ============================================================

coco_gt = COCO(
    str(GT_JSON)
)


# ============================================================
# 4. COCO EVALUATION FUNCTION
# ============================================================

def evaluate_predictions(
    predictions,
    label
):

    if not predictions:
        raise RuntimeError(
            f"No predictions for {label}"
        )


    output_json = (
        E14_DIR
        / f"{label}.json"
    )


    with open(
        output_json,
        "w",
        encoding="utf-8"
    ) as f:

        json.dump(
            predictions,
            f
        )


    coco_dt = coco_gt.loadRes(
        str(output_json)
    )


    coco_eval = COCOeval(
        coco_gt,
        coco_dt,
        "bbox"
    )


    coco_eval.params.maxDets = [
        1,
        10,
        100
    ]


    coco_eval.evaluate()
    coco_eval.accumulate()
    coco_eval.summarize()


    stats = coco_eval.stats


    return {
        "experiment": label,
        "AP50-95": float(stats[0]),
        "AP50": float(stats[1]),
        "AP75": float(stats[2]),
        "AR100": float(stats[8]),
    }


# ============================================================
# 5. HELPER
# ============================================================

EPS = 1e-12


def make_prediction(
    row,
    final_class_idx,
    score
):

    return {
        "image_id": int(
            row["image_id"]
        ),

        "category_id": int(
            final_class_idx + 1
        ),

        "bbox": [
            float(v)
            for v in row["bbox"]
        ],

        "score": float(score),
    }


# ============================================================
# 6. STORE ALL RESULTS
# ============================================================

all_results = []


# ============================================================
# 7. E13 BASELINE REPRODUCTION
#
# Final class:
#   MobileNet top class
#
# Final score:
#   YOLO confidence * MobileNet top-class probability
#
# This should reproduce E13:
# AP ≈ 0.434479
# ============================================================

print("\n" + "=" * 100)
print("E13 BASELINE — PRODUCT FUSION")
print("=" * 100)


preds = []


for row in cached:

    final_idx = int(
        row["mobilenet_class_idx"]
    )

    mobile_prob = float(
        row["mobilenet_probs"][final_idx]
    )

    yolo_conf = float(
        row["yolo_conf"]
    )


    score = (
        yolo_conf
        *
        mobile_prob
    )


    preds.append(
        make_prediction(
            row,
            final_idx,
            score
        )
    )


result = evaluate_predictions(
    preds,
    "E13_baseline_product"
)

result["alpha"] = "product"
result["gate"] = "always_mobile"

all_results.append(
    result
)


# ============================================================
# 8. E14-A — FUSION WEIGHT SWEEP
#
# MobileNet ALWAYS supplies final class.
#
# Therefore:
#
# final_class = MobileNet top class
#
# score =
#     YOLO_conf ^ alpha
#     *
#     P_mobile(final_class) ^ (1-alpha)
#
# alpha = 0.50 is ranking-equivalent
# to product fusion.
# ============================================================

print("\n" + "=" * 100)
print("E14-A — FUSION WEIGHT SWEEP")
print("=" * 100)


ALPHAS = [
    0.00,
    0.25,
    0.40,
    0.50,
    0.60,
    0.70,
    0.80,
    0.90,
    1.00,
]


for alpha in ALPHAS:

    preds = []


    for row in cached:

        final_idx = int(
            row["mobilenet_class_idx"]
        )


        yolo_conf = max(
            float(
                row["yolo_conf"]
            ),
            EPS
        )


        mobile_prob = max(
            float(
                row["mobilenet_probs"][
                    final_idx
                ]
            ),
            EPS
        )


        score = (
            yolo_conf ** alpha
            *
            mobile_prob ** (
                1.0 - alpha
            )
        )


        preds.append(
            make_prediction(
                row,
                final_idx,
                score
            )
        )


    label = (
        f"E14A_fusion_alpha_{alpha:.2f}"
    )


    result = evaluate_predictions(
        preds,
        label
    )


    result["alpha"] = alpha
    result["gate"] = "always_mobile"


    all_results.append(
        result
    )


# ============================================================
# 9. E14-B — CORRECTED CLASS-GATING SWEEP
#
# Gate rule:
#
# if MobileNet top probability >= threshold:
#     final class = MobileNet top class
#
# else:
#     final class = YOLO class
#
#
# CORRECTION:
#
# MobileNet probability used in the score must
# correspond to FINAL CLASS.
#
# Therefore:
#
# mobile_prob =
#     MobileNet probability for final_class
#
#
# Product score:
#
# score =
#     YOLO confidence
#     *
#     P_mobile(final_class)
# ============================================================

print("\n" + "=" * 100)
print("E14-B — CORRECTED CLASS-GATING SWEEP")
print("=" * 100)


GATE_THRESHOLDS = [
    0.40,
    0.50,
    0.60,
    0.70,
    0.80,
    0.90,
    0.95,
]


gating_results = []


for threshold in GATE_THRESHOLDS:

    preds = []

    mobile_used = 0
    yolo_used = 0


    for row in cached:

        mn_top_prob = float(
            row["mobilenet_top_prob"]
        )


        # --------------------------------------------
        # Select final class
        # --------------------------------------------

        if mn_top_prob >= threshold:

            final_idx = int(
                row["mobilenet_class_idx"]
            )

            mobile_used += 1

        else:

            final_idx = int(
                row["yolo_class_idx"]
            )

            yolo_used += 1


        # --------------------------------------------
        # CORRECTED probability:
        # MobileNet probability of FINAL selected class
        # --------------------------------------------

        final_class_mobile_prob = max(
            float(
                row["mobilenet_probs"][
                    final_idx
                ]
            ),
            EPS
        )


        yolo_conf = max(
            float(
                row["yolo_conf"]
            ),
            EPS
        )


        # Corrected product score
        score = (
            yolo_conf
            *
            final_class_mobile_prob
        )


        preds.append(
            make_prediction(
                row,
                final_idx,
                score
            )
        )


    label = (
        f"E14B_gate_{threshold:.2f}"
    )


    result = evaluate_predictions(
        preds,
        label
    )


    result["alpha"] = "product"
    result["gate"] = threshold

    result[
        "mobilenet_class_count"
    ] = mobile_used

    result[
        "yolo_class_count"
    ] = yolo_used


    all_results.append(
        result
    )

    gating_results.append(
        result
    )


# ============================================================
# 10. FIND BEST CORRECTED GATE
# ============================================================

best_gate_result = max(
    gating_results,
    key=lambda x: x["AP50-95"]
)


BEST_GATE = float(
    best_gate_result["gate"]
)


print("\n" + "=" * 100)
print("BEST CORRECTED GATING THRESHOLD")
print("=" * 100)

print(
    "Threshold:",
    BEST_GATE
)

print(
    "AP50-95:",
    best_gate_result[
        "AP50-95"
    ]
)


# ============================================================
# 11. E14-C — CORRECTED COMBINED
#                GATING + FUSION
#
# Uses best corrected gating threshold.
#
# final class:
#     determined by gate
#
# MobileNet probability:
#     probability for FINAL class
#
# score:
#
# YOLO_conf ^ alpha
# *
# P_mobile(final_class) ^ (1-alpha)
# ============================================================

print("\n" + "=" * 100)
print("E14-C — CORRECTED GATING + FUSION")
print("=" * 100)


for alpha in ALPHAS:

    preds = []


    for row in cached:

        mn_top_prob = float(
            row["mobilenet_top_prob"]
        )


        # --------------------------------------------
        # Final class
        # --------------------------------------------

        if mn_top_prob >= BEST_GATE:

            final_idx = int(
                row[
                    "mobilenet_class_idx"
                ]
            )

        else:

            final_idx = int(
                row[
                    "yolo_class_idx"
                ]
            )


        # --------------------------------------------
        # Correct MobileNet probability
        # for FINAL class
        # --------------------------------------------

        final_class_mobile_prob = max(
            float(
                row["mobilenet_probs"][
                    final_idx
                ]
            ),
            EPS
        )


        yolo_conf = max(
            float(
                row["yolo_conf"]
            ),
            EPS
        )


        # --------------------------------------------
        # Weighted fusion
        # --------------------------------------------

        score = (
            yolo_conf ** alpha
            *
            final_class_mobile_prob
            ** (
                1.0 - alpha
            )
        )


        preds.append(
            make_prediction(
                row,
                final_idx,
                score
            )
        )


    label = (
        f"E14C_gate_{BEST_GATE:.2f}"
        f"_alpha_{alpha:.2f}"
    )


    result = evaluate_predictions(
        preds,
        label
    )


    result["alpha"] = alpha
    result["gate"] = BEST_GATE


    all_results.append(
        result
    )


# ============================================================
# 12. RESULTS DATAFRAME
# ============================================================

df = pd.DataFrame(
    all_results
)


df = df.sort_values(
    by="AP50-95",
    ascending=False
).reset_index(
    drop=True
)


df.to_csv(
    RESULTS_CSV,
    index=False
)


# ============================================================
# 13. FINAL RANKING
# ============================================================

print("\n" + "=" * 100)
print("E14 CORRECTED FINAL RANKING")
print("=" * 100)


display_cols = [
    "experiment",
    "AP50-95",
    "AP50",
    "AP75",
    "AR100",
    "alpha",
    "gate",
    "mobilenet_class_count",
    "yolo_class_count",
]


cols = [
    c
    for c in display_cols
    if c in df.columns
]


print(
    df[cols].to_string(
        index=False
    )
)


# ============================================================
# 14. BEST CONFIGURATION
# ============================================================

best = df.iloc[0]


best_ap = float(
    best["AP50-95"]
)


print("\n" + "=" * 100)
print("BEST CORRECTED E14 CONFIGURATION")
print("=" * 100)


print(
    "Experiment :",
    best["experiment"]
)


print(
    f"AP50-95    : "
    f"{best['AP50-95']:.6f} "
    f"= {best['AP50-95'] * 100:.2f}%"
)


print(
    f"AP50       : "
    f"{best['AP50']:.6f} "
    f"= {best['AP50'] * 100:.2f}%"
)


print(
    f"AP75       : "
    f"{best['AP75']:.6f} "
    f"= {best['AP75'] * 100:.2f}%"
)


print(
    f"AR100      : "
    f"{best['AR100']:.6f}"
)


# ============================================================
# 15. COMPARISON
# ============================================================

E13_AP = 0.434479

PREVIOUS_E14_BEST_AP = (
    0.445395
)

SORTWASTE_REFERENCE_AP = (
    0.451
)


print("\n" + "=" * 100)
print("COMPARISON")
print("=" * 100)


print(
    f"E13 baseline AP50-95       : "
    f"{E13_AP * 100:.2f}%"
)


print(
    f"Previous E14 provisional   : "
    f"{PREVIOUS_E14_BEST_AP * 100:.2f}%"
)


print(
    f"Corrected E14 best         : "
    f"{best_ap * 100:.2f}%"
)


print(
    f"Corrected E14 vs E13       : "
    f"{(best_ap - E13_AP) * 100:+.2f} pp"
)


print(
    f"Corrected vs provisional   : "
    f"{(best_ap - PREVIOUS_E14_BEST_AP) * 100:+.2f} pp"
)


print(
    f"SortWaste reference        : "
    f"{SORTWASTE_REFERENCE_AP * 100:.2f}%"
)


print(
    f"Distance to 45.10          : "
    f"{(best_ap - SORTWASTE_REFERENCE_AP) * 100:+.2f} pp"
)


# ============================================================
# 16. SAVE SUMMARY
# ============================================================

with open(
    SUMMARY_TXT,
    "w",
    encoding="utf-8"
) as f:

    f.write(
        "E14 — YOLO11m + MobileNet Confidence "
        "Fusion and Class-Gating Optimization\n"
    )

    f.write(
        "Corrected final-class confidence version\n"
    )

    f.write(
        "=" * 90
        + "\n\n"
    )


    f.write(
        "Correction:\n"
    )

    f.write(
        "MobileNet probability used for confidence "
        "fusion now always corresponds to the "
        "FINAL selected class.\n\n"
    )


    f.write(
        f"Cached detections: "
        f"{len(cached)}\n\n"
    )


    f.write(
        df.to_string(
            index=False
        )
    )


    f.write(
        "\n\nBEST CORRECTED CONFIGURATION:\n"
    )


    f.write(
        str(
            best.to_dict()
        )
    )


    f.write(
        "\n\n"
    )


    f.write(
        f"E13 AP50-95 = "
        f"{E13_AP:.6f}\n"
    )


    f.write(
        f"Previous E14 provisional = "
        f"{PREVIOUS_E14_BEST_AP:.6f}\n"
    )


    f.write(
        f"Corrected E14 best = "
        f"{best_ap:.6f}\n"
    )


    f.write(
        f"Corrected E14 vs E13 = "
        f"{best_ap - E13_AP:+.6f}\n"
    )


    f.write(
        f"Distance to SortWaste reference = "
        f"{best_ap - SORTWASTE_REFERENCE_AP:+.6f}\n"
    )


# ============================================================
# 17. FINISH
# ============================================================

print("\nResults CSV:")
print(
    RESULTS_CSV
)

print("\nSummary:")
print(
    SUMMARY_TXT
)

print("\n" + "=" * 100)
print("E14 CORRECTED RUN COMPLETE")
print("=" * 100)

E14 — YOLO11m + MobileNet Fusion and Gating Ablation
CORRECTED FINAL-CLASS CONFIDENCE VERSION

Cache:
C:\Users\varda\Documents\_My Computer\COMMON_space\Learning\Upgrad_EPGP_MS\upgrad_IIITB_MS_course\MS_LJMU_Material\Thesis_Code\runs\sortwaste\E14_fusion_gating_ablation\E14_cached_predictions.json

Ground truth:
C:\Users\varda\Documents\_My Computer\COMMON_space\Learning\Upgrad_EPGP_MS\upgrad_IIITB_MS_course\MS_LJMU_Material\Thesis_Code\runs\sortwaste\E14_fusion_gating_ablation\E14_val_gt_7class.json

Output directory:
C:\Users\varda\Documents\_My Computer\COMMON_space\Learning\Upgrad_EPGP_MS\upgrad_IIITB_MS_course\MS_LJMU_Material\Thesis_Code\runs\sortwaste\E14_fusion_gating_ablation

Cached detections: 55605
loading annotations into memory...
Done (t=0.04s)
creating index...
index created!

E13 BASELINE — PRODUCT FUSION
Loading and preparing results...
DONE (t=2.61s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=4.14s).
Accum

# Model E15 — YOLO11m + MobileNetV3-Large with Joint Class-Gating and Confidence-Fusion Optimization

In [ ]:
# E15 — Joint Gate + Fusion-Weight Grid Search
#
# Reuses E14 cached predictions.
#
# NO RETRAINING
# NO YOLO INFERENCE
# NO MOBILENET INFERENCE
#
# Goal:
#   Jointly optimize:
#       1. MobileNet class-gating threshold
#       2. YOLO/MobileNet confidence fusion alpha
#
# Correct scoring:
#
#   Final class:
#       MobileNet class if MobileNet top probability >= gate
#       otherwise YOLO class
#
#   MobileNet probability used for scoring:
#       probability of FINAL selected class
#
#   Final score:
#
#       YOLO_conf^alpha
#       *
#       P_mobile(final_class)^(1-alpha)
#
# Primary metric:
#   COCO AP50-95
# ============================================================

import json
from pathlib import Path

import pandas as pd

from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval


# ============================================================
# 1. PATHS
# ============================================================

BASE = Path(
    r"C:\Users\varda\Documents\_My Computer\COMMON_space\Learning"
    r"\Upgrad_EPGP_MS\upgrad_IIITB_MS_course\MS_LJMU_Material"
)

THESIS_CODE = BASE / "Thesis_Code"

E14_DIR = (
    THESIS_CODE
    / "runs"
    / "sortwaste"
    / "E14_fusion_gating_ablation"
)

E15_DIR = (
    THESIS_CODE
    / "runs"
    / "sortwaste"
    / "E15_joint_gate_fusion_grid"
)

E15_DIR.mkdir(
    parents=True,
    exist_ok=True
)


CACHE_JSON = (
    E14_DIR
    / "E14_cached_predictions.json"
)

GT_JSON = (
    E14_DIR
    / "E14_val_gt_7class.json"
)


RESULTS_CSV = (
    E15_DIR
    / "E15_results.csv"
)

SUMMARY_TXT = (
    E15_DIR
    / "E15_summary.txt"
)


assert CACHE_JSON.exists(), (
    f"Missing cache: {CACHE_JSON}"
)

assert GT_JSON.exists(), (
    f"Missing GT: {GT_JSON}"
)


# ============================================================
# 2. EXPERIMENT SETTINGS
# ============================================================

GATES = [
    0.90,
    0.92,
    0.94,
    0.95,
    0.96,
    0.97,
    0.98,
    0.99,
]


ALPHAS = [
    0.55,
    0.60,
    0.65,
    0.675,
    0.70,
    0.725,
    0.75,
    0.80,
    0.85,
]


EPS = 1e-12


E13_AP = 0.434479

E14_BEST_AP = 0.445840

SORTWASTE_REFERENCE = 0.451


# ============================================================
# 3. HEADER
# ============================================================

print("=" * 100)

print(
    "E15 — JOINT GATE + FUSION-WEIGHT GRID SEARCH"
)

print("=" * 100)

print("\nCache:")
print(CACHE_JSON)

print("\nGround truth:")
print(GT_JSON)

print("\nOutput:")
print(E15_DIR)

print(
    f"\nGate values: {GATES}"
)

print(
    f"Alpha values: {ALPHAS}"
)

print(
    f"\nTotal combinations: "
    f"{len(GATES) * len(ALPHAS)}"
)


# ============================================================
# 4. LOAD CACHE
# ============================================================

with open(
    CACHE_JSON,
    "r",
    encoding="utf-8"
) as f:

    cached = json.load(f)


print(
    "\nCached detections:",
    len(cached)
)


# Basic validation
for i, row in enumerate(
    cached[:20]
):

    assert (
        "mobilenet_probs"
        in row
    ), (
        "mobilenet_probs missing "
        f"from cache row {i}"
    )

    assert (
        len(
            row["mobilenet_probs"]
        )
        == 7
    ), (
        f"Unexpected class-probability "
        f"count at row {i}"
    )


# ============================================================
# 5. LOAD COCO GT
# ============================================================

coco_gt = COCO(
    str(GT_JSON)
)


# ============================================================
# 6. PREDICTION HELPER
# ============================================================

def make_prediction(
    row,
    final_idx,
    score
):

    return {
        "image_id": int(
            row["image_id"]
        ),

        "category_id": int(
            final_idx + 1
        ),

        "bbox": [
            float(v)
            for v
            in row["bbox"]
        ],

        "score": float(
            score
        ),
    }


# ============================================================
# 7. COCO EVALUATION
# ============================================================

def evaluate_predictions(
    predictions,
    label
):

    output_json = (
        E15_DIR
        / f"{label}.json"
    )


    with open(
        output_json,
        "w",
        encoding="utf-8"
    ) as f:

        json.dump(
            predictions,
            f
        )


    coco_dt = coco_gt.loadRes(
        str(output_json)
    )


    coco_eval = COCOeval(
        coco_gt,
        coco_dt,
        "bbox"
    )


    coco_eval.params.maxDets = [
        1,
        10,
        100
    ]


    coco_eval.evaluate()
    coco_eval.accumulate()
    coco_eval.summarize()


    stats = coco_eval.stats


    return {
        "experiment": label,

        "AP50-95": float(
            stats[0]
        ),

        "AP50": float(
            stats[1]
        ),

        "AP75": float(
            stats[2]
        ),

        "AR100": float(
            stats[8]
        ),
    }


# ============================================================
# 8. JOINT GRID SEARCH
# ============================================================

all_results = []


total_runs = (
    len(GATES)
    *
    len(ALPHAS)
)

run_number = 0


print("\n" + "=" * 100)

print(
    "E15 — RUNNING JOINT GRID"
)

print("=" * 100)


for gate in GATES:

    for alpha in ALPHAS:

        run_number += 1


        print(
            f"\n[{run_number}/{total_runs}] "
            f"Gate={gate:.3f}, "
            f"Alpha={alpha:.3f}"
        )


        preds = []

        mobile_used = 0
        yolo_used = 0


        for row in cached:

            mn_top_prob = float(
                row[
                    "mobilenet_top_prob"
                ]
            )


            # ----------------------------------------
            # Select final class
            # ----------------------------------------

            if (
                mn_top_prob
                >= gate
            ):

                final_idx = int(
                    row[
                        "mobilenet_class_idx"
                    ]
                )

                mobile_used += 1

            else:

                final_idx = int(
                    row[
                        "yolo_class_idx"
                    ]
                )

                yolo_used += 1


            # ----------------------------------------
            # Correct MobileNet probability:
            #
            # Probability of FINAL selected class
            # ----------------------------------------

            mobile_final_prob = max(
                float(
                    row[
                        "mobilenet_probs"
                    ][
                        final_idx
                    ]
                ),
                EPS
            )


            yolo_conf = max(
                float(
                    row[
                        "yolo_conf"
                    ]
                ),
                EPS
            )


            # ----------------------------------------
            # Joint fusion score
            # ----------------------------------------

            score = (
                yolo_conf ** alpha
                *
                mobile_final_prob
                ** (
                    1.0
                    - alpha
                )
            )


            preds.append(
                make_prediction(
                    row,
                    final_idx,
                    score
                )
            )


        label = (
            f"E15_gate_{gate:.3f}"
            f"_alpha_{alpha:.3f}"
        )


        result = (
            evaluate_predictions(
                preds,
                label
            )
        )


        result[
            "gate"
        ] = gate

        result[
            "alpha"
        ] = alpha

        result[
            "mobilenet_class_count"
        ] = mobile_used

        result[
            "yolo_class_count"
        ] = yolo_used

        result[
            "mobilenet_class_pct"
        ] = (
            mobile_used
            / len(cached)
            * 100
        )

        result[
            "yolo_class_pct"
        ] = (
            yolo_used
            / len(cached)
            * 100
        )


        all_results.append(
            result
        )


        print(
            f"AP50-95 = "
            f"{result['AP50-95'] * 100:.3f}%"
        )


# ============================================================
# 9. SORT RESULTS
# ============================================================

df = pd.DataFrame(
    all_results
)


df = df.sort_values(
    by=[
        "AP50-95",
        "AP50",
    ],
    ascending=[
        False,
        False,
    ]
).reset_index(
    drop=True
)


df.to_csv(
    RESULTS_CSV,
    index=False
)


# ============================================================
# 10. FINAL RANKING
# ============================================================

print("\n" + "=" * 100)

print(
    "E15 FINAL RANKING"
)

print("=" * 100)


print(
    df[
        [
            "experiment",
            "AP50-95",
            "AP50",
            "AP75",
            "AR100",
            "gate",
            "alpha",
            "mobilenet_class_count",
            "yolo_class_count",
            "mobilenet_class_pct",
            "yolo_class_pct",
        ]
    ]
    .head(20)
    .to_string(
        index=False
    )
)


# ============================================================
# 11. BEST RESULT
# ============================================================

best = df.iloc[0]


best_ap = float(
    best["AP50-95"]
)


print("\n" + "=" * 100)

print(
    "BEST E15 CONFIGURATION"
)

print("=" * 100)


print(
    "Experiment :",
    best["experiment"]
)


print(
    f"Gate       : "
    f"{best['gate']:.3f}"
)


print(
    f"Alpha      : "
    f"{best['alpha']:.3f}"
)


print(
    f"AP50-95    : "
    f"{best['AP50-95']:.6f} "
    f"= "
    f"{best['AP50-95'] * 100:.2f}%"
)


print(
    f"AP50       : "
    f"{best['AP50']:.6f} "
    f"= "
    f"{best['AP50'] * 100:.2f}%"
)


print(
    f"AP75       : "
    f"{best['AP75']:.6f} "
    f"= "
    f"{best['AP75'] * 100:.2f}%"
)


print(
    f"AR100      : "
    f"{best['AR100']:.6f}"
)


print(
    f"MobileNet class use: "
    f"{best['mobilenet_class_pct']:.2f}%"
)


print(
    f"YOLO class use     : "
    f"{best['yolo_class_pct']:.2f}%"
)


# ============================================================
# 12. COMPARISONS
# ============================================================

print("\n" + "=" * 100)

print(
    "COMPARISON"
)

print("=" * 100)


print(
    f"E13 AP50-95          : "
    f"{E13_AP * 100:.2f}%"
)


print(
    f"E14 corrected best   : "
    f"{E14_BEST_AP * 100:.2f}%"
)


print(
    f"E15 best             : "
    f"{best_ap * 100:.2f}%"
)


print(
    f"E15 vs E13           : "
    f"{(best_ap - E13_AP) * 100:+.2f} pp"
)


print(
    f"E15 vs E14           : "
    f"{(best_ap - E14_BEST_AP) * 100:+.2f} pp"
)


print(
    f"SortWaste reference  : "
    f"{SORTWASTE_REFERENCE * 100:.2f}%"
)


print(
    f"Distance to 45.10    : "
    f"{(best_ap - SORTWASTE_REFERENCE) * 100:+.2f} pp"
)


if (
    best_ap
    >= SORTWASTE_REFERENCE
):

    print(
        "\n*** VALIDATION AP HAS CROSSED "
        "THE 45.10 REFERENCE LEVEL ***"
    )

else:

    print(
        "\nRemaining validation gap: "
        f"{(SORTWASTE_REFERENCE - best_ap) * 100:.2f} pp"
    )


# ============================================================
# 13. SAVE SUMMARY
# ============================================================

with open(
    SUMMARY_TXT,
    "w",
    encoding="utf-8"
) as f:

    f.write(
        "E15 — Joint Gate + "
        "Fusion-Weight Grid Search\n"
    )

    f.write(
        "=" * 90
        + "\n\n"
    )


    f.write(
        f"Gate values: "
        f"{GATES}\n"
    )

    f.write(
        f"Alpha values: "
        f"{ALPHAS}\n"
    )

    f.write(
        f"Total combinations: "
        f"{total_runs}\n\n"
    )


    f.write(
        "Scoring equation:\n"
    )

    f.write(
        "score = "
        "YOLO_conf^alpha * "
        "P_mobile(final_class)"
        "^(1-alpha)\n\n"
    )


    f.write(
        "Top results:\n\n"
    )


    f.write(
        df.head(20)
        .to_string(
            index=False
        )
    )


    f.write(
        "\n\nBEST CONFIGURATION\n"
    )

    f.write(
        "-" * 60
        + "\n"
    )


    f.write(
        f"Experiment: "
        f"{best['experiment']}\n"
    )

    f.write(
        f"Gate: "
        f"{best['gate']:.3f}\n"
    )

    f.write(
        f"Alpha: "
        f"{best['alpha']:.3f}\n"
    )

    f.write(
        f"AP50-95: "
        f"{best_ap:.6f}\n"
    )

    f.write(
        f"AP50: "
        f"{best['AP50']:.6f}\n"
    )

    f.write(
        f"AP75: "
        f"{best['AP75']:.6f}\n"
    )

    f.write(
        f"AR100: "
        f"{best['AR100']:.6f}\n"
    )

    f.write(
        f"MobileNet class use: "
        f"{best['mobilenet_class_pct']:.2f}%\n"
    )

    f.write(
        f"YOLO class use: "
        f"{best['yolo_class_pct']:.2f}%\n\n"
    )


    f.write(
        f"E13 AP50-95: "
        f"{E13_AP:.6f}\n"
    )

    f.write(
        f"E14 corrected AP50-95: "
        f"{E14_BEST_AP:.6f}\n"
    )

    f.write(
        f"E15 best AP50-95: "
        f"{best_ap:.6f}\n"
    )

    f.write(
        f"E15 vs E14: "
        f"{best_ap - E14_BEST_AP:+.6f}\n"
    )

    f.write(
        f"Distance to 45.10: "
        f"{best_ap - SORTWASTE_REFERENCE:+.6f}\n"
    )


# ============================================================
# 14. FINISH
# ============================================================

print("\nResults CSV:")
print(
    RESULTS_CSV
)

print("\nSummary:")
print(
    SUMMARY_TXT
)

print("\n" + "=" * 100)

print(
    "E15 COMPLETE"
)

print("=" * 100)

E15 — JOINT GATE + FUSION-WEIGHT GRID SEARCH

Cache:
C:\Users\varda\Documents\_My Computer\COMMON_space\Learning\Upgrad_EPGP_MS\upgrad_IIITB_MS_course\MS_LJMU_Material\Thesis_Code\runs\sortwaste\E14_fusion_gating_ablation\E14_cached_predictions.json

Ground truth:
C:\Users\varda\Documents\_My Computer\COMMON_space\Learning\Upgrad_EPGP_MS\upgrad_IIITB_MS_course\MS_LJMU_Material\Thesis_Code\runs\sortwaste\E14_fusion_gating_ablation\E14_val_gt_7class.json

Output:
C:\Users\varda\Documents\_My Computer\COMMON_space\Learning\Upgrad_EPGP_MS\upgrad_IIITB_MS_course\MS_LJMU_Material\Thesis_Code\runs\sortwaste\E15_joint_gate_fusion_grid

Gate values: [0.9, 0.92, 0.94, 0.95, 0.96, 0.97, 0.98, 0.99]
Alpha values: [0.55, 0.6, 0.65, 0.675, 0.7, 0.725, 0.75, 0.8, 0.85]

Total combinations: 72

Cached detections: 55605
loading annotations into memory...
Done (t=0.06s)
creating index...
index created!

E15 — RUNNING JOINT GRID

[1/72] Gate=0.900, Alpha=0.550
Loading and preparing results...
DONE (t=0.2

# Model E16 — YOLO11m + MobileNetV3-Large with Class-Specific Gating and Confidence-Fusion Optimization

## E16-A — Class-Specific MobileNet Gating with Fixed Confidence Fusion

In [ ]:
# E16-A — Class-Specific MobileNet Gating
#         with Fixed Confidence Fusion
#
# Models:
#   Detector   : YOLO11m (E12)
#   Classifier : MobileNetV3-Large, class-weighted (E3Y-B)
#
# Reuses:
#   E14 cached YOLO11m + MobileNet predictions
#
# NO RETRAINING
# NO YOLO INFERENCE
# NO MOBILENET INFERENCE
#
# Starting point:
#   E15 best global gate = 0.98
#   E15 best alpha       = 0.70
#
# Goal:
#   Replace one global MobileNet gate with
#   class-specific MobileNet confidence thresholds.
#
# Final class:
#   if MobileNet top probability >= threshold for
#   MobileNet's predicted class:
#       use MobileNet class
#   else:
#       retain YOLO class
#
# Final score:
#
#   YOLO_conf^0.70 *
#   P_mobile(final_class)^0.30
#
# Primary metric:
#   COCO AP50-95
# ============================================================

import json
from pathlib import Path

import pandas as pd

from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval


# ============================================================
# 1. PATHS
# ============================================================

BASE = Path(
    r"C:\Users\varda\Documents\_My Computer\COMMON_space\Learning"
    r"\Upgrad_EPGP_MS\upgrad_IIITB_MS_course\MS_LJMU_Material"
)

THESIS_CODE = BASE / "Thesis_Code"

E14_DIR = (
    THESIS_CODE
    / "runs"
    / "sortwaste"
    / "E14_fusion_gating_ablation"
)

E16_DIR = (
    THESIS_CODE
    / "runs"
    / "sortwaste"
    / "E16A_class_specific_gating"
)

E16_DIR.mkdir(
    parents=True,
    exist_ok=True
)


CACHE_JSON = (
    E14_DIR
    / "E14_cached_predictions.json"
)

GT_JSON = (
    E14_DIR
    / "E14_val_gt_7class.json"
)


RESULTS_CSV = (
    E16_DIR
    / "E16A_class_gate_search.csv"
)

FINAL_JSON = (
    E16_DIR
    / "E16A_final_predictions.json"
)

SUMMARY_TXT = (
    E16_DIR
    / "E16A_summary.txt"
)


assert CACHE_JSON.exists(), (
    f"Missing cache: {CACHE_JSON}"
)

assert GT_JSON.exists(), (
    f"Missing GT: {GT_JSON}"
)


# ============================================================
# 2. CLASS DEFINITIONS
# ============================================================

CLASS_NAMES = [
    "ecal",
    "hdpe",
    "mixed_plastic_rigid",
    "mixed_plastic_soft",
    "non_plastic",
    "pet",
    "pet_oil",
]


# ============================================================
# 3. E16-A SETTINGS
# ============================================================

BASE_GATE = 0.98

ALPHA = 0.70

CANDIDATE_GATES = [
    0.94,
    0.96,
    0.98,
    0.99,
]


EPS = 1e-12


# Reference results
E13_AP = 0.434479

E14_AP = 0.445840

E15_AP = 0.448077

SORTWASTE_REFERENCE = 0.451


# ============================================================
# 4. HEADER
# ============================================================

print("=" * 100)

print(
    "E16-A — CLASS-SPECIFIC MOBILENET GATING "
    "WITH FIXED CONFIDENCE FUSION"
)

print("=" * 100)

print("\nCache:")
print(CACHE_JSON)

print("\nGround truth:")
print(GT_JSON)

print("\nOutput:")
print(E16_DIR)

print(
    f"\nBase global gate : {BASE_GATE}"
)

print(
    f"Fixed alpha      : {ALPHA}"
)

print(
    f"Candidate gates  : {CANDIDATE_GATES}"
)

print(
    f"Classes          : {CLASS_NAMES}"
)


# ============================================================
# 5. LOAD CACHED PREDICTIONS
# ============================================================

with open(
    CACHE_JSON,
    "r",
    encoding="utf-8"
) as f:

    cached = json.load(f)


print(
    "\nCached detections:",
    len(cached)
)


# Validate expected cache structure
required_keys = [
    "image_id",
    "bbox",
    "yolo_conf",
    "yolo_class_idx",
    "mobilenet_top_prob",
    "mobilenet_class_idx",
    "mobilenet_probs",
]


for i, row in enumerate(
    cached[:50]
):

    for key in required_keys:

        assert key in row, (
            f"Missing key '{key}' "
            f"in cache row {i}"
        )

    assert (
        len(row["mobilenet_probs"])
        == 7
    ), (
        f"Expected 7 MobileNet probabilities "
        f"in cache row {i}"
    )


# ============================================================
# 6. LOAD COCO GT
# ============================================================

coco_gt = COCO(
    str(GT_JSON)
)


# ============================================================
# 7. BUILD PREDICTIONS
# ============================================================

def build_predictions(
    class_gates
):

    """
    class_gates:
        dictionary:
        class_idx -> confidence threshold

    Gating is determined using the class predicted
    by MobileNet.

    If MobileNet confidence exceeds that class's
    threshold, MobileNet supplies final class.

    Otherwise YOLO supplies final class.
    """

    predictions = []

    mobile_used = 0
    yolo_used = 0


    # Count which system supplies the final class
    class_mobile_used = {
        i: 0
        for i in range(7)
    }

    class_yolo_used = {
        i: 0
        for i in range(7)
    }


    for row in cached:

        mn_idx = int(
            row["mobilenet_class_idx"]
        )

        mn_top_prob = float(
            row["mobilenet_top_prob"]
        )

        yolo_idx = int(
            row["yolo_class_idx"]
        )

        yolo_conf = max(
            float(
                row["yolo_conf"]
            ),
            EPS
        )


        # ----------------------------------------
        # Threshold associated with MobileNet's
        # predicted class
        # ----------------------------------------

        gate = float(
            class_gates[mn_idx]
        )


        # ----------------------------------------
        # Select final class
        # ----------------------------------------

        if mn_top_prob >= gate:

            final_idx = mn_idx

            mobile_used += 1

            class_mobile_used[
                mn_idx
            ] += 1

        else:

            final_idx = yolo_idx

            yolo_used += 1

            class_yolo_used[
                yolo_idx
            ] += 1


        # ----------------------------------------
        # IMPORTANT:
        #
        # Probability of the ACTUAL FINAL CLASS,
        # not always MobileNet top probability.
        # ----------------------------------------

        mobile_final_prob = max(
            float(
                row[
                    "mobilenet_probs"
                ][final_idx]
            ),
            EPS
        )


        # ----------------------------------------
        # Fixed E15 optimal confidence fusion
        # ----------------------------------------

        final_score = (
            yolo_conf ** ALPHA
            *
            mobile_final_prob
            ** (
                1.0 - ALPHA
            )
        )


        predictions.append(
            {
                "image_id": int(
                    row["image_id"]
                ),

                "category_id": int(
                    final_idx + 1
                ),

                "bbox": [
                    float(v)
                    for v
                    in row["bbox"]
                ],

                "score": float(
                    final_score
                ),
            }
        )


    return (
        predictions,
        mobile_used,
        yolo_used,
        class_mobile_used,
        class_yolo_used,
    )


# ============================================================
# 8. COCO EVALUATION
# ============================================================

def evaluate_predictions(
    predictions,
    output_json
):

    with open(
        output_json,
        "w",
        encoding="utf-8"
    ) as f:

        json.dump(
            predictions,
            f
        )


    coco_dt = coco_gt.loadRes(
        str(output_json)
    )


    coco_eval = COCOeval(
        coco_gt,
        coco_dt,
        "bbox"
    )


    coco_eval.params.maxDets = [
        1,
        10,
        100,
    ]


    coco_eval.evaluate()
    coco_eval.accumulate()
    coco_eval.summarize()


    stats = coco_eval.stats


    return {
        "AP50-95": float(
            stats[0]
        ),

        "AP50": float(
            stats[1]
        ),

        "AP75": float(
            stats[2]
        ),

        "AR100": float(
            stats[8]
        ),
    }


# ============================================================
# 9. BASELINE — REPRODUCE E15
# ============================================================

base_gates = {
    i: BASE_GATE
    for i in range(7)
}


print("\n" + "=" * 100)

print(
    "REPRODUCING E15 BASELINE"
)

print("=" * 100)


base_predictions, _, _, _, _ = (
    build_predictions(
        base_gates
    )
)


base_metrics = (
    evaluate_predictions(
        base_predictions,
        E16_DIR
        / "E16A_E15_baseline.json"
    )
)


print(
    f"\nReproduced E15 AP50-95: "
    f"{base_metrics['AP50-95']:.6f}"
)

print(
    f"Expected E15 AP50-95  : "
    f"{E15_AP:.6f}"
)


# ============================================================
# 10. CLASS-BY-CLASS GATE SEARCH
# ============================================================

all_results = []


print("\n" + "=" * 100)

print(
    "CLASS-SPECIFIC GATE SEARCH"
)

print("=" * 100)


for class_idx, class_name in enumerate(
    CLASS_NAMES
):

    print(
        "\n" + "-" * 100
    )

    print(
        f"Optimizing class: "
        f"{class_name} "
        f"(index {class_idx})"
    )

    print(
        "-" * 100
    )


    for gate in CANDIDATE_GATES:

        # All classes stay at E15 gate
        gates = {
            i: BASE_GATE
            for i in range(7)
        }

        # Change only current class
        gates[
            class_idx
        ] = gate


        predictions, mobile_used, yolo_used, _, _ = (
            build_predictions(
                gates
            )
        )


        label = (
            f"E16A_{class_name}"
            f"_gate_{gate:.3f}"
        )

        output_json = (
            E16_DIR
            / f"{label}.json"
        )


        metrics = (
            evaluate_predictions(
                predictions,
                output_json
            )
        )


        result = {
            "class_idx": class_idx,
            "class_name": class_name,
            "gate": gate,

            "AP50-95":
                metrics["AP50-95"],

            "AP50":
                metrics["AP50"],

            "AP75":
                metrics["AP75"],

            "AR100":
                metrics["AR100"],

            "mobile_used":
                mobile_used,

            "yolo_used":
                yolo_used,

            "delta_vs_E15_AP":
                (
                    metrics["AP50-95"]
                    - base_metrics["AP50-95"]
                ),
        }


        all_results.append(
            result
        )


        print(
            f"\n{class_name:25s}"
            f" gate={gate:.3f}"
            f" | AP={metrics['AP50-95'] * 100:.3f}%"
            f" | Δ={result['delta_vs_E15_AP'] * 100:+.3f} pp"
        )


# ============================================================
# 11. SAVE SEARCH RESULTS
# ============================================================

df = pd.DataFrame(
    all_results
)


df.to_csv(
    RESULTS_CSV,
    index=False
)


# ============================================================
# 12. SELECT BEST GATE FOR EACH CLASS
# ============================================================

best_class_gates = {}


print("\n" + "=" * 100)

print(
    "BEST GATE PER CLASS"
)

print("=" * 100)


for class_idx, class_name in enumerate(
    CLASS_NAMES
):

    class_df = (
        df[
            df["class_idx"]
            == class_idx
        ]
        .sort_values(
            by=[
                "AP50-95",
                "AP50",
            ],
            ascending=[
                False,
                False,
            ]
        )
    )


    best_row = (
        class_df.iloc[0]
    )


    best_gate = float(
        best_row["gate"]
    )


    best_class_gates[
        class_idx
    ] = best_gate


    print(
        f"{class_name:25s}"
        f" -> "
        f"{best_gate:.3f}"
        f" | AP="
        f"{best_row['AP50-95'] * 100:.3f}%"
        f" | Δ vs E15="
        f"{best_row['delta_vs_E15_AP'] * 100:+.3f} pp"
    )


# ============================================================
# 13. FINAL COMBINED CLASS-SPECIFIC CONFIGURATION
# ============================================================

print("\n" + "=" * 100)

print(
    "FINAL E16-A COMBINED CONFIGURATION"
)

print("=" * 100)


for idx, name in enumerate(
    CLASS_NAMES
):

    print(
        f"{name:25s}: "
        f"{best_class_gates[idx]:.3f}"
    )


(
    final_predictions,
    final_mobile_used,
    final_yolo_used,
    final_class_mobile,
    final_class_yolo,
) = build_predictions(
    best_class_gates
)


final_metrics = (
    evaluate_predictions(
        final_predictions,
        FINAL_JSON
    )
)


# ============================================================
# 14. FINAL RESULTS
# ============================================================

print("\n" + "=" * 100)

print(
    "E16-A FINAL RESULT"
)

print("=" * 100)


print(
    f"AP50-95 : "
    f"{final_metrics['AP50-95']:.6f} "
    f"= "
    f"{final_metrics['AP50-95'] * 100:.2f}%"
)

print(
    f"AP50     : "
    f"{final_metrics['AP50']:.6f} "
    f"= "
    f"{final_metrics['AP50'] * 100:.2f}%"
)

print(
    f"AP75     : "
    f"{final_metrics['AP75']:.6f} "
    f"= "
    f"{final_metrics['AP75'] * 100:.2f}%"
)

print(
    f"AR100    : "
    f"{final_metrics['AR100']:.6f}"
)


total_predictions = len(
    cached
)


print(
    f"\nMobileNet supplies class: "
    f"{final_mobile_used} "
    f"("
    f"{final_mobile_used / total_predictions * 100:.2f}%"
    f")"
)

print(
    f"YOLO retains class      : "
    f"{final_yolo_used} "
    f"("
    f"{final_yolo_used / total_predictions * 100:.2f}%"
    f")"
)


# ============================================================
# 15. COMPARISON
# ============================================================

print("\n" + "=" * 100)

print(
    "COMPARISON"
)

print("=" * 100)


print(
    f"E13 AP50-95 : "
    f"{E13_AP * 100:.2f}%"
)

print(
    f"E14 AP50-95 : "
    f"{E14_AP * 100:.2f}%"
)

print(
    f"E15 AP50-95 : "
    f"{E15_AP * 100:.2f}%"
)

print(
    f"E16-A       : "
    f"{final_metrics['AP50-95'] * 100:.2f}%"
)


print(
    f"\nE16-A vs E15: " f"{(final_metrics['AP50-95'] - E15_AP) * 100:+.3f} pp"
)


print(
    f"Distance to SortWaste 45.10: " f"{(final_metrics['AP50-95'] - SORTWASTE_REFERENCE) * 100:+.3f} pp"
)


# ============================================================
# 16. PER-CLASS SOURCE COUNTS
# ============================================================

print("\n" + "=" * 100)

print(
    "FINAL CLASS-SOURCE COUNTS"
)

print("=" * 100)


for idx, name in enumerate(
    CLASS_NAMES
):

    print(
        f"{name:25s}"
        f" MobileNet={final_class_mobile[idx]:6d}"
        f" | YOLO={final_class_yolo[idx]:6d}"
    )


# ============================================================
# 17. SAVE SUMMARY
# ============================================================

with open(
    SUMMARY_TXT,
    "w",
    encoding="utf-8"
) as f:

    f.write(
        "E16-A — Class-Specific MobileNet Gating "
        "with Fixed Confidence Fusion\n"
    )

    f.write(
        "=" * 90
        + "\n\n"
    )


    f.write(
        f"Base gate: {BASE_GATE}\n"
    )

    f.write(
        f"Fixed alpha: {ALPHA}\n"
    )

    f.write(
        f"Candidate gates: "
        f"{CANDIDATE_GATES}\n\n"
    )


    f.write(
        "Best class-specific gates:\n"
    )


    for idx, name in enumerate(
        CLASS_NAMES
    ):

        f.write(
            f"{name}: "
            f"{best_class_gates[idx]:.3f}\n"
        )


    f.write(
        "\nFinal metrics:\n"
    )

    f.write(
        f"AP50-95: "
        f"{final_metrics['AP50-95']:.6f}\n"
    )

    f.write(
        f"AP50: "
        f"{final_metrics['AP50']:.6f}\n"
    )

    f.write(
        f"AP75: "
        f"{final_metrics['AP75']:.6f}\n"
    )

    f.write(
        f"AR100: "
        f"{final_metrics['AR100']:.6f}\n\n"
    )


    f.write(
        f"MobileNet class use: "
        f"{final_mobile_used} "
        f"("
        f"{final_mobile_used / total_predictions * 100:.2f}%"
        f")\n"
    )

    f.write(
        f"YOLO class use: "
        f"{final_yolo_used} "
        f"("
        f"{final_yolo_used / total_predictions * 100:.2f}%"
        f")\n\n"
    )


    f.write(
        f"E15 AP50-95: "
        f"{E15_AP:.6f}\n"
    )

    f.write(
        f"E16-A AP50-95: "
        f"{final_metrics['AP50-95']:.6f}\n"
    )

    f.write(
        f"E16-A vs E15: " f"{(final_metrics['AP50-95']- E15_AP ):+.6f}\n"
    )

    f.write(
        f"Distance to 45.10: " f"{(final_metrics['AP50-95']- SORTWASTE_REFERENCE):+.6f}\n"
    )


# ============================================================
# 18. FINISH
# ============================================================

print("\nResults CSV:")
print(
    RESULTS_CSV
)

print("\nFinal predictions:")
print(
    FINAL_JSON
)

print("\nSummary:")
print(
    SUMMARY_TXT
)

print("\n" + "=" * 100)

print(
    "E16-A COMPLETE"
)

print("=" * 100)

E16-A — CLASS-SPECIFIC MOBILENET GATING WITH FIXED CONFIDENCE FUSION

Cache:
C:\Users\varda\Documents\_My Computer\COMMON_space\Learning\Upgrad_EPGP_MS\upgrad_IIITB_MS_course\MS_LJMU_Material\Thesis_Code\runs\sortwaste\E14_fusion_gating_ablation\E14_cached_predictions.json

Ground truth:
C:\Users\varda\Documents\_My Computer\COMMON_space\Learning\Upgrad_EPGP_MS\upgrad_IIITB_MS_course\MS_LJMU_Material\Thesis_Code\runs\sortwaste\E14_fusion_gating_ablation\E14_val_gt_7class.json

Output:
C:\Users\varda\Documents\_My Computer\COMMON_space\Learning\Upgrad_EPGP_MS\upgrad_IIITB_MS_course\MS_LJMU_Material\Thesis_Code\runs\sortwaste\E16A_class_specific_gating

Base global gate : 0.98
Fixed alpha      : 0.7
Candidate gates  : [0.94, 0.96, 0.98, 0.99]
Classes          : ['ecal', 'hdpe', 'mixed_plastic_rigid', 'mixed_plastic_soft', 'non_plastic', 'pet', 'pet_oil']

Cached detections: 55605
loading annotations into memory...
Done (t=0.11s)
creating index...
index created!

REPRODUCING E15 BASELINE


## E16-B — Class-Specific Gating with Local Confidence-Fusion Refinement

In [ ]:
# E16-B — Class-Specific Gating with Local
#         Confidence-Fusion Refinement
#
# Reuses:
#   E14 cached YOLO11m + MobileNet predictions
#   E16-A optimized class-specific gates
#
# NO RETRAINING
# NO YOLO INFERENCE
# NO MOBILENET INFERENCE
#
# Fixed class-specific gates from E16-A:
#   ecal                 0.99
#   hdpe                 0.98
#   mixed_plastic_rigid  0.98
#   mixed_plastic_soft   0.94
#   non_plastic          0.94
#   pet                  0.99
#   pet_oil              0.99
#
# Local alpha search:
#   [0.65, 0.675, 0.70, 0.725, 0.75]
#
# Final class:
#   if MobileNet top probability >= class-specific gate:
#       use MobileNet class
#   else:
#       retain YOLO class
#
# Final confidence:
#   score =
#       YOLO_conf ** alpha
#       *
#       P_mobile(final_class) ** (1-alpha)
#
# Primary metric:
#   COCO AP50-95
# ============================================================

import json
from pathlib import Path

import pandas as pd

from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval


# ============================================================
# 1. PATHS
# ============================================================

BASE = Path(
    r"C:\Users\varda\Documents\_My Computer\COMMON_space\Learning"
    r"\Upgrad_EPGP_MS\upgrad_IIITB_MS_course\MS_LJMU_Material"
)

THESIS_CODE = BASE / "Thesis_Code"

E14_DIR = (
    THESIS_CODE
    / "runs"
    / "sortwaste"
    / "E14_fusion_gating_ablation"
)

E16A_DIR = (
    THESIS_CODE
    / "runs"
    / "sortwaste"
    / "E16A_class_specific_gating"
)

E16B_DIR = (
    THESIS_CODE
    / "runs"
    / "sortwaste"
    / "E16B_local_fusion_refinement"
)

E16B_DIR.mkdir(
    parents=True,
    exist_ok=True
)


CACHE_JSON = (
    E14_DIR
    / "E14_cached_predictions.json"
)

GT_JSON = (
    E14_DIR
    / "E14_val_gt_7class.json"
)


RESULTS_CSV = (
    E16B_DIR
    / "E16B_alpha_search.csv"
)

FINAL_JSON = (
    E16B_DIR
    / "E16B_best_predictions.json"
)

SUMMARY_TXT = (
    E16B_DIR
    / "E16B_summary.txt"
)


assert CACHE_JSON.exists(), (
    f"Missing cache: {CACHE_JSON}"
)

assert GT_JSON.exists(), (
    f"Missing GT: {GT_JSON}"
)


# ============================================================
# 2. CLASS DEFINITIONS
# ============================================================

CLASS_NAMES = [
    "ecal",
    "hdpe",
    "mixed_plastic_rigid",
    "mixed_plastic_soft",
    "non_plastic",
    "pet",
    "pet_oil",
]


# ============================================================
# 3. FIXED E16-A CLASS-SPECIFIC GATES
# ============================================================

CLASS_GATES = {
    0: 0.99,  # ecal
    1: 0.98,  # hdpe
    2: 0.98,  # mixed_plastic_rigid
    3: 0.94,  # mixed_plastic_soft
    4: 0.94,  # non_plastic
    5: 0.99,  # pet
    6: 0.99,  # pet_oil
}


# ============================================================
# 4. LOCAL ALPHA SEARCH
# ============================================================

ALPHAS = [
    0.65,
    0.675,
    0.70,
    0.725,
    0.75,
]


EPS = 1e-12


# Reference values
E13_AP = 0.434479
E14_AP = 0.445840
E15_AP = 0.448077
E16A_AP = 0.449175

SORTWASTE_REFERENCE = 0.451


# ============================================================
# 5. HEADER
# ============================================================

print("=" * 100)

print(
    "E16-B — CLASS-SPECIFIC GATING WITH "
    "LOCAL CONFIDENCE-FUSION REFINEMENT"
)

print("=" * 100)


print("\nCache:")
print(CACHE_JSON)

print("\nGround truth:")
print(GT_JSON)

print("\nOutput:")
print(E16B_DIR)


print("\nFixed class-specific gates:")

for idx, name in enumerate(
    CLASS_NAMES
):

    print(
        f"{name:25s}: "
        f"{CLASS_GATES[idx]:.3f}"
    )


print(
    f"\nAlpha candidates: {ALPHAS}"
)


# ============================================================
# 6. LOAD CACHE
# ============================================================

with open(
    CACHE_JSON,
    "r",
    encoding="utf-8"
) as f:

    cached = json.load(f)


print(
    "\nCached detections:",
    len(cached)
)


# ============================================================
# 7. LOAD GT
# ============================================================

coco_gt = COCO(
    str(GT_JSON)
)


# ============================================================
# 8. BUILD PREDICTIONS
# ============================================================

def build_predictions(
    alpha
):

    predictions = []

    mobile_used = 0
    yolo_used = 0


    for row in cached:

        mn_idx = int(
            row["mobilenet_class_idx"]
        )

        yolo_idx = int(
            row["yolo_class_idx"]
        )

        mn_top_prob = float(
            row["mobilenet_top_prob"]
        )

        yolo_conf = max(
            float(
                row["yolo_conf"]
            ),
            EPS
        )


        # ----------------------------------------
        # Fixed E16-A class-specific gate
        # based on MobileNet predicted class
        # ----------------------------------------

        gate = float(
            CLASS_GATES[
                mn_idx
            ]
        )


        # ----------------------------------------
        # Final class
        # ----------------------------------------

        if mn_top_prob >= gate:

            final_idx = mn_idx
            mobile_used += 1

        else:

            final_idx = yolo_idx
            yolo_used += 1


        # ----------------------------------------
        # MobileNet probability corresponding to
        # the ACTUAL final class
        # ----------------------------------------

        mobile_final_prob = max(
            float(
                row[
                    "mobilenet_probs"
                ][final_idx]
            ),
            EPS
        )


        # ----------------------------------------
        # Weighted geometric confidence fusion
        # ----------------------------------------

        final_score = (
            yolo_conf ** alpha
            *
            mobile_final_prob
            ** (
                1.0 - alpha
            )
        )


        predictions.append(
            {
                "image_id": int(
                    row["image_id"]
                ),

                "category_id": int(
                    final_idx + 1
                ),

                "bbox": [
                    float(v)
                    for v in row["bbox"]
                ],

                "score": float(
                    final_score
                ),
            }
        )


    return (
        predictions,
        mobile_used,
        yolo_used,
    )


# ============================================================
# 9. COCO EVALUATION
# ============================================================

def evaluate_predictions(
    predictions,
    output_json
):

    with open(
        output_json,
        "w",
        encoding="utf-8"
    ) as f:

        json.dump(
            predictions,
            f
        )


    coco_dt = coco_gt.loadRes(
        str(output_json)
    )


    coco_eval = COCOeval(
        coco_gt,
        coco_dt,
        "bbox"
    )


    coco_eval.params.maxDets = [
        1,
        10,
        100,
    ]


    coco_eval.evaluate()
    coco_eval.accumulate()
    coco_eval.summarize()


    stats = coco_eval.stats


    return {
        "AP50-95": float(
            stats[0]
        ),

        "AP50": float(
            stats[1]
        ),

        "AP75": float(
            stats[2]
        ),

        "AR100": float(
            stats[8]
        ),
    }


# ============================================================
# 10. RUN LOCAL ALPHA SEARCH
# ============================================================

results = []


print("\n" + "=" * 100)

print(
    "LOCAL ALPHA SEARCH"
)

print("=" * 100)


for alpha in ALPHAS:

    print(
        "\n" + "-" * 100
    )

    print(
        f"Testing alpha = {alpha:.3f}"
    )

    print(
        "-" * 100
    )


    (
        predictions,
        mobile_used,
        yolo_used,
    ) = build_predictions(
        alpha
    )


    output_json = (
        E16B_DIR
        / f"E16B_alpha_{alpha:.3f}.json"
    )


    metrics = evaluate_predictions(
        predictions,
        output_json
    )


    result = {
        "alpha": alpha,

        "AP50-95":
            metrics["AP50-95"],

        "AP50":
            metrics["AP50"],

        "AP75":
            metrics["AP75"],

        "AR100":
            metrics["AR100"],

        "mobile_used":
            mobile_used,

        "yolo_used":
            yolo_used,

        "delta_vs_E16A":
            (
                metrics["AP50-95"]
                - E16A_AP
            ),

        "delta_vs_E15":
            (
                metrics["AP50-95"]
                - E15_AP
            ),
    }


    results.append(
        result
    )


    print(
        f"\nalpha={alpha:.3f}"
        f" | AP={metrics['AP50-95'] * 100:.4f}%"
        f" | AP50={metrics['AP50'] * 100:.4f}%"
        f" | AP75={metrics['AP75'] * 100:.4f}%"
        f" | AR100={metrics['AR100'] * 100:.4f}%"
        f" | Δ vs E16-A="
        f"{result['delta_vs_E16A'] * 100:+.4f} pp"
    )


# ============================================================
# 11. SAVE RESULTS TABLE
# ============================================================

df = pd.DataFrame(
    results
)


df.to_csv(
    RESULTS_CSV,
    index=False
)


# ============================================================
# 12. SELECT BEST CONFIGURATION
# ============================================================

best_row = (
    df.sort_values(
        by=[
            "AP50-95",
            "AP50",
            "AP75",
        ],
        ascending=[
            False,
            False,
            False,
        ]
    )
    .iloc[0]
)


BEST_ALPHA = float(
    best_row["alpha"]
)


print("\n" + "=" * 100)

print(
    "BEST E16-B CONFIGURATION"
)

print("=" * 100)


print(
    f"\nBest alpha: "
    f"{BEST_ALPHA:.3f}"
)

print(
    f"AP50-95 : "
    f"{best_row['AP50-95']:.6f} "
    f"= "
    f"{best_row['AP50-95'] * 100:.4f}%"
)

print(
    f"AP50     : "
    f"{best_row['AP50']:.6f} "
    f"= "
    f"{best_row['AP50'] * 100:.4f}%"
)

print(
    f"AP75     : "
    f"{best_row['AP75']:.6f} "
    f"= "
    f"{best_row['AP75'] * 100:.4f}%"
)

print(
    f"AR100    : "
    f"{best_row['AR100']:.6f} "
    f"= "
    f"{best_row['AR100'] * 100:.4f}%"
)


# ============================================================
# 13. SAVE BEST PREDICTIONS
# ============================================================

(
    best_predictions,
    best_mobile_used,
    best_yolo_used,
) = build_predictions(
    BEST_ALPHA
)


with open(
    FINAL_JSON,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        best_predictions,
        f
    )


# ============================================================
# 14. COMPARISON
# ============================================================

best_ap = float(
    best_row["AP50-95"]
)


print("\n" + "=" * 100)

print(
    "COMPARISON"
)

print("=" * 100)


print(
    f"E13 AP50-95 : "
    f"{E13_AP * 100:.4f}%"
)

print(
    f"E14 AP50-95 : "
    f"{E14_AP * 100:.4f}%"
)

print(
    f"E15 AP50-95 : "
    f"{E15_AP * 100:.4f}%"
)

print(
    f"E16-A       : "
    f"{E16A_AP * 100:.4f}%"
)

print(
    f"E16-B       : "
    f"{best_ap * 100:.4f}%"
)


print(
    f"\nE16-B vs E16-A: " f"{(best_ap- E16A_AP) * 100:+.4f} pp"
)


print(
    f"E16-B vs E15: " f"{(  best_ap - E15_AP) * 100:+.4f} pp"
)


print(
    f"Distance to SortWaste 45.10: " f"{(    best_ap    - SORTWASTE_REFERENCE) * 100:+.4f} pp"
)


# ============================================================
# 15. CLASS SOURCE USAGE
# ============================================================

total_predictions = len(
    cached
)


print("\n" + "=" * 100)

print(
    "CLASS-SOURCE USAGE"
)

print("=" * 100)


print(
    f"\nMobileNet supplies final class: "
    f"{best_mobile_used}"
    f" / {total_predictions}"
    f" = "
    f"{best_mobile_used / total_predictions * 100:.2f}%"
)

print(
    f"YOLO retains final class      : "
    f"{best_yolo_used}"
    f" / {total_predictions}"
    f" = "
    f"{best_yolo_used / total_predictions * 100:.2f}%"
)


# ============================================================
# 16. SAVE SUMMARY
# ============================================================

with open(
    SUMMARY_TXT,
    "w",
    encoding="utf-8"
) as f:

    f.write(
        "E16-B — Class-Specific Gating with "
        "Local Confidence-Fusion Refinement\n"
    )

    f.write(
        "=" * 90
        + "\n\n"
    )


    f.write(
        "Fixed class-specific gates:\n"
    )

    for idx, name in enumerate(
        CLASS_NAMES
    ):

        f.write(
            f"{name}: "
            f"{CLASS_GATES[idx]:.3f}\n"
        )


    f.write(
        "\nAlpha candidates:\n"
    )

    f.write(
        f"{ALPHAS}\n\n"
    )


    f.write(
        f"Best alpha: "
        f"{BEST_ALPHA:.3f}\n\n"
    )


    f.write(
        f"AP50-95: "
        f"{best_row['AP50-95']:.6f}\n"
    )

    f.write(
        f"AP50: "
        f"{best_row['AP50']:.6f}\n"
    )

    f.write(
        f"AP75: "
        f"{best_row['AP75']:.6f}\n"
    )

    f.write(
        f"AR100: "
        f"{best_row['AR100']:.6f}\n\n"
    )


    f.write(
        f"MobileNet class use: "
        f"{best_mobile_used}"
        f" / {total_predictions}"
        f" = "
        f"{best_mobile_used / total_predictions * 100:.2f}%\n"
    )

    f.write(
        f"YOLO class use: "
        f"{best_yolo_used}"
        f" / {total_predictions}"
        f" = "
        f"{best_yolo_used / total_predictions * 100:.2f}%\n\n"
    )


    f.write(
        f"E16-A AP50-95: "
        f"{E16A_AP:.6f}\n"
    )

    f.write(
        f"E16-B AP50-95: "
        f"{best_ap:.6f}\n"
    )

    f.write(
        f"E16-B vs E16-A: "
        f"{best_ap - E16A_AP:+.6f}\n"
    )

    f.write(
        f"Distance to SortWaste 0.451: "
        f"{best_ap - SORTWASTE_REFERENCE:+.6f}\n"
    )


# ============================================================
# 17. FINISH
# ============================================================

print("\nResults CSV:")
print(
    RESULTS_CSV
)

print("\nBest predictions:")
print(
    FINAL_JSON
)

print("\nSummary:")
print(
    SUMMARY_TXT
)

print("\n" + "=" * 100)

print(
    "E16-B COMPLETE"
)

print("=" * 100)

E16-B — CLASS-SPECIFIC GATING WITH LOCAL CONFIDENCE-FUSION REFINEMENT

Cache:
C:\Users\varda\Documents\_My Computer\COMMON_space\Learning\Upgrad_EPGP_MS\upgrad_IIITB_MS_course\MS_LJMU_Material\Thesis_Code\runs\sortwaste\E14_fusion_gating_ablation\E14_cached_predictions.json

Ground truth:
C:\Users\varda\Documents\_My Computer\COMMON_space\Learning\Upgrad_EPGP_MS\upgrad_IIITB_MS_course\MS_LJMU_Material\Thesis_Code\runs\sortwaste\E14_fusion_gating_ablation\E14_val_gt_7class.json

Output:
C:\Users\varda\Documents\_My Computer\COMMON_space\Learning\Upgrad_EPGP_MS\upgrad_IIITB_MS_course\MS_LJMU_Material\Thesis_Code\runs\sortwaste\E16B_local_fusion_refinement

Fixed class-specific gates:
ecal                     : 0.990
hdpe                     : 0.980
mixed_plastic_rigid      : 0.980
mixed_plastic_soft       : 0.940
non_plastic              : 0.940
pet                      : 0.990
pet_oil                  : 0.990

Alpha candidates: [0.65, 0.675, 0.7, 0.725, 0.75]

Cached detections: 55605
l

# Model E17 — YOLO11m @768 + 7-Class Taxonomy + Realistic Augmentation + Class-Aware Oversampling

In [ ]:
# E17 — YOLO11m @768 + 7-Class TaxonomY + Realistic Augmentation + Class-Aware Oversampling
#
# Purpose:
#   Resolution ablation against E12.
#
# E12:
#   YOLO11m @640
#
# E17:
#   YOLO11m @768
#
# Keep everything else as close as possible to E12.
# ============================================================

from pathlib import Path
from ultralytics import YOLO


# ============================================================
# 1. PATHS
# ============================================================

BASE = Path(
    r"C:\Users\varda\Documents\_My Computer\COMMON_space\Learning"
    r"\Upgrad_EPGP_MS\upgrad_IIITB_MS_course\MS_LJMU_Material"
)

THESIS_CODE = BASE / "Thesis_Code"

PROJECT_DIR = (
    THESIS_CODE
    / "runs"
    / "sortwaste"
)

# Reuse the exact same 7-class oversampled dataset used by E5/E12
DATA_YAML = (
    PROJECT_DIR
    / "E5_7class_dataset"
    / "E5_7class.yaml"
)

assert DATA_YAML.exists(), (
    f"Dataset YAML not found:\n{DATA_YAML}"
)


# ============================================================
# 2. EXPERIMENT SETTINGS
# ============================================================

EXPERIMENT_NAME = (
    "E17_yolo11m_7class_aug_classbalance_768"
)

IMAGE_SIZE = 768

EPOCHS = 150

PATIENCE = 15

# 4 GB RTX 3050 Ti:
# Start with batch=1 for safety at 768.
BATCH_SIZE = 1

WORKERS = 0

SEED = 42


# ============================================================
# 3. LOAD PRETRAINED YOLO11m
# ============================================================

model = YOLO(
    "yolo11m.pt"
)


# ============================================================
# 4. TRAIN
# ============================================================

results = model.train(

    # ----------------------------
    # Dataset
    # ----------------------------
    data=str(DATA_YAML),

    # ----------------------------
    # Resolution
    # ----------------------------
    imgsz=IMAGE_SIZE,

    # ----------------------------
    # Training duration
    # ----------------------------
    epochs=EPOCHS,
    patience=PATIENCE,

    # ----------------------------
    # Hardware
    # ----------------------------
    batch=BATCH_SIZE,
    workers=WORKERS,
    cache=False,

    # ----------------------------
    # Optimizer
    # Match E12
    # ----------------------------
    optimizer="AdamW",
    cos_lr=True,

    # ----------------------------
    # Pretraining / AMP
    # ----------------------------
    pretrained=True,
    amp=True,

    # ----------------------------
    # Realistic augmentation
    # SAME as E12
    # ----------------------------
    mosaic=0.80,
    mixup=0.05,

    degrees=5.0,
    translate=0.10,
    scale=0.30,

    shear=0.0,
    perspective=0.0,

    flipud=0.0,
    fliplr=0.50,

    hsv_h=0.015,
    hsv_s=0.40,
    hsv_v=0.30,

    close_mosaic=10,

    # ----------------------------
    # Reproducibility
    # ----------------------------
    seed=SEED,
    deterministic=True,

    # ----------------------------
    # Validation during training
    # ----------------------------
    val=True,

    # ----------------------------
    # Output
    # ----------------------------
    project=str(PROJECT_DIR),
    name=EXPERIMENT_NAME,
    exist_ok=False,

    # ----------------------------
    # Misc
    # ----------------------------
    verbose=True,
)


# ============================================================
# 5. BEST CHECKPOINT
# ============================================================

RUN_DIR = (
    PROJECT_DIR
    / EXPERIMENT_NAME
)

BEST_MODEL = (
    RUN_DIR
    / "weights"
    / "best.pt"
)

print("\n" + "=" * 100)
print("E17 TRAINING COMPLETE")
print("=" * 100)



New https://pypi.org/project/ultralytics/8.4.135 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.126  Python-3.11.15 torch-2.13.0+cu126 CUDA:0 (NVIDIA GeForce RTX 3050 Ti Laptop GPU, 4096MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=1, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=False, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=C:\Users\varda\Documents\_My Computer\COMMON_space\Learning\Upgrad_EPGP_MS\upgrad_IIITB_MS_course\MS_LJMU_Material\Thesis_Code\runs\sortwaste\E5_7class_dataset\E5_7class.yaml, degrees=5.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=150, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze

AssertionError: Best model not found:
C:\Users\varda\Documents\_My Computer\COMMON_space\Learning\Upgrad_EPGP_MS\upgrad_IIITB_MS_course\MS_LJMU_Material\Thesis_Code\runs\sortwaste\E17_yolo11m_7class_aug_classbalance_768\weights\best.pt

In [ ]:
# Final Validation
print("\nBest checkpoint:")
print(BEST_MODEL)


# ============================================================
# 6. FINAL VALIDATION
# ============================================================
#
# Match E12 final validation settings:
#
#   conf = 0.001
#   NMS IoU = 0.60
#   max_det = 100
#
# ============================================================

assert BEST_MODEL.exists(), (
    f"Best model not found:\n{BEST_MODEL}"
)


best_model = YOLO(
    str(BEST_MODEL)
)


print("\n" + "=" * 100)
print("E17 FINAL VALIDATION")
print("=" * 100)


val_results = best_model.val(

    data=str(DATA_YAML),

    imgsz=IMAGE_SIZE,

    batch=1,

    conf=0.001,

    iou=0.60,

    max_det=100,

    workers=WORKERS,

    plots=True,

    verbose=True,
)


# ============================================================
# 7. PRINT FINAL METRICS
# ============================================================

metrics = val_results.box


print("\n" + "=" * 100)
print("E17 FINAL RESULTS")
print("=" * 100)


print(
    f"\nmAP50-95 : "
    f"{metrics.map:.6f} "
    f"= {metrics.map * 100:.2f}%"
)

print(
    f"mAP50     : "
    f"{metrics.map50:.6f} "
    f"= {metrics.map50 * 100:.2f}%"
)

print(
    f"mAP75     : "
    f"{metrics.map75:.6f} "
    f"= {metrics.map75 * 100:.2f}%"
)


# ============================================================
# 8. PER-CLASS RESULTS
# ============================================================

print("\n" + "=" * 100)
print("PER-CLASS RESULTS")
print("=" * 100)


class_names = best_model.names


for class_id, class_name in class_names.items():

    ap50_95 = metrics.maps[class_id]

    print(
        f"{class_name:25s}"
        f" AP50-95={ap50_95:.6f}"
        f" ({ap50_95 * 100:.2f}%)"
    )


print("\n" + "=" * 100)
print("E17 COMPLETE")
print("=" * 100)


Best checkpoint:
C:\Users\varda\Documents\_My Computer\COMMON_space\Learning\Upgrad_EPGP_MS\upgrad_IIITB_MS_course\MS_LJMU_Material\Thesis_Code\runs\sortwaste\E17_yolo11m_7class_aug_classbalance_768\weights\best.pt

E17 FINAL VALIDATION
Ultralytics 8.4.126  Python-3.11.15 torch-2.13.0+cu126 CUDA:0 (NVIDIA GeForce RTX 3050 Ti Laptop GPU, 4096MiB)
YOLO11m summary (fused): 126 layers, 20,035,429 parameters, 0 gradients, 67.8 GFLOPs
val: Fast image access  (ping: 0.20.0 ms, read: 361.7296.7 MB/s, size: 2133.8 KB)
val: Scanning C:\Users\varda\Documents\_My Computer\COMMON_space\Learning\Upgrad_EPGP_MS\upgrad_IIITB_MS_course\MS_LJMU_Material\Thesis_Code\runs\sortwaste\E5_7class_dataset\val\labels.cache... 780 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 780/780  0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 780/780 14.6it/s 53.4s0.1ss
                   all        780      13065      0.604      0.612        0.6      0

# Model E18 — YOLO11m + ConvNeXt-Tiny Hard-Class Refinement Pipeline

## E18-A — YOLO11m-Matched Hard-Class Crop Dataset Generation

In [ ]:
# E18-A — YOLO11m-Matched Hard-Class Crop Dataset Generation
#
# Purpose:
#   Generate detector-derived crops for training/validation
#   of the E18 ConvNeXt-Tiny hard-class classifier.
#
# Detector:
#   E12 YOLO11m @640
#
# Matching:
#   Geometry-only one-to-one matching
#   Prediction ↔ GT using IoU >= 0.50
#
# Crop:
#   YOLO detector bounding box
#
# Target:
#   Matched ground-truth class
#
# Hard classes only:
#   mixed_plastic_rigid
#   mixed_plastic_soft
#   non_plastic
#   pet_oil
#
# Test set is NOT used.
# ============================================================

import json
import shutil
from collections import defaultdict, Counter
from pathlib import Path

from PIL import Image
from tqdm import tqdm
from ultralytics import YOLO


# ============================================================
# 1. PATHS
# ============================================================

BASE = Path(
    r"C:\Users\varda\Documents\_My Computer\COMMON_space\Learning"
    r"\Upgrad_EPGP_MS\upgrad_IIITB_MS_course\MS_LJMU_Material"
)

THESIS_CODE = BASE / "Thesis_Code"

DATASET_ROOT = (
    BASE
    / "Topic Data"
    / "SortWaste"
    / "dataset"
    / "dataset"
)

COCO_ROOT = (
    DATASET_ROOT
    / "splited_all_dataset_coco"
)

# E12 detector
YOLO11M_MODEL = (
    THESIS_CODE
    / "runs"
    / "sortwaste"
    / "E12_yolo11m_7class_aug_classbalance_640"
    / "weights"
    / "best.pt"
)

# Output crop dataset
OUTPUT_ROOT = (
    DATASET_ROOT
    / "yolo11m_convnext_hardclass_crops_E18"
)


assert YOLO11M_MODEL.exists(), (
    f"E12 model not found:\n{YOLO11M_MODEL}"
)

assert COCO_ROOT.exists(), (
    f"COCO dataset not found:\n{COCO_ROOT}"
)


# ============================================================
# 2. EXPERIMENT SETTINGS
# ============================================================

IMAGE_SIZE = 640

CONF_THRESHOLD = 0.001
NMS_IOU = 0.60
MAX_DET = 100

MATCH_IOU_THRESHOLD = 0.50

DEVICE = 0


# ============================================================
# 3. 7-CLASS EVALUATION TAXONOMY
# ============================================================

CLASS_NAMES = [
    "ecal",
    "hdpe",
    "mixed_plastic_rigid",
    "mixed_plastic_soft",
    "non_plastic",
    "pet",
    "pet_oil",
]


# Hard classes for E18
HARD_CLASSES = {
    2: "mixed_plastic_rigid",
    3: "mixed_plastic_soft",
    4: "non_plastic",
    6: "pet_oil",
}


# Original COCO category ID -> 7-class index
COCO_TO_7CLASS = {
    1: 5,  # pet
    2: 1,  # hdpe
    3: 3,  # mixed_plastic_soft
    4: 0,  # ecal
    5: 4,  # metal -> non_plastic
    6: 4,  # cardboard -> non_plastic
    7: 2,  # mixed_plastic_rigid
    8: 6,  # pet_oil
}


# ============================================================
# 4. IOU FUNCTION
# ============================================================

def bbox_iou_xyxy(box1, box2):
    """
    box format:
        [x1, y1, x2, y2]
    """

    x1 = max(box1[0], box2[0])
    y1 = max(box1[1], box2[1])
    x2 = min(box1[2], box2[2])
    y2 = min(box1[3], box2[3])

    inter_w = max(0.0, x2 - x1)
    inter_h = max(0.0, y2 - y1)

    intersection = inter_w * inter_h

    area1 = max(
        0.0,
        box1[2] - box1[0]
    ) * max(
        0.0,
        box1[3] - box1[1]
    )

    area2 = max(
        0.0,
        box2[2] - box2[0]
    ) * max(
        0.0,
        box2[3] - box2[1]
    )

    union = area1 + area2 - intersection

    if union <= 0:
        return 0.0

    return intersection / union


# ============================================================
# 5. COCO BBOX -> XYXY
# ============================================================

def coco_bbox_to_xyxy(bbox):
    """
    COCO bbox:
        [x, y, width, height]
    """

    x, y, w, h = bbox

    return [
        float(x),
        float(y),
        float(x + w),
        float(y + h),
    ]


# ============================================================
# 6. SAFE CROP
# ============================================================

def crop_detector_box(
    image,
    bbox_xyxy
):

    width, height = image.size

    x1, y1, x2, y2 = bbox_xyxy

    x1 = max(
        0,
        min(
            int(round(x1)),
            width - 1
        )
    )

    y1 = max(
        0,
        min(
            int(round(y1)),
            height - 1
        )
    )

    x2 = max(
        0,
        min(
            int(round(x2)),
            width
        )
    )

    y2 = max(
        0,
        min(
            int(round(y2)),
            height
        )
    )

    if x2 <= x1 or y2 <= y1:
        return None

    return image.crop(
        (
            x1,
            y1,
            x2,
            y2
        )
    )


# ============================================================
# 7. LOAD YOLO11m
# ============================================================

print("=" * 100)
print("E18-A — YOLO11m-MATCHED HARD-CLASS CROP DATASET GENERATION")
print("=" * 100)

print("\nDetector:")
print(YOLO11M_MODEL)

print("\nOutput:")
print(OUTPUT_ROOT)

print("\nHard classes:")

for idx, name in HARD_CLASSES.items():
    print(
        f"{idx}: {name}"
    )


model = YOLO(
    str(YOLO11M_MODEL)
)


# ============================================================
# 8. PROCESS ONE SPLIT
# ============================================================

def process_split(split_name):

    print("\n" + "=" * 100)
    print(f"PROCESSING SPLIT: {split_name.upper()}")
    print("=" * 100)

    split_root = (
        COCO_ROOT
        / split_name
    )

    image_dir = (
        split_root
        / "images"
    )

    annotation_file = (
        split_root
        / "annotations"
        / f"{split_name}_coco.json"
    )

    assert image_dir.exists(), (
        f"Image directory missing:\n{image_dir}"
    )

    assert annotation_file.exists(), (
        f"Annotation file missing:\n{annotation_file}"
    )


    # --------------------------------------------------------
    # Load COCO
    # --------------------------------------------------------

    with open(
        annotation_file,
        "r",
        encoding="utf-8"
    ) as f:

        coco = json.load(f)


    image_info = {
        int(img["id"]): img
        for img in coco["images"]
    }


    anns_by_image = defaultdict(list)

    for ann in coco["annotations"]:

        coco_category = int(
            ann["category_id"]
        )

        class_idx = COCO_TO_7CLASS[
            coco_category
        ]

        anns_by_image[
            int(
                ann["image_id"]
            )
        ].append(
            {
                "annotation_id":
                    int(
                        ann["id"]
                    ),

                "class_idx":
                    class_idx,

                "bbox":
                    coco_bbox_to_xyxy(
                        ann["bbox"]
                    ),
            }
        )


    # --------------------------------------------------------
    # Prepare output folders
    # --------------------------------------------------------

    split_output = (
        OUTPUT_ROOT
        / split_name
    )

    if split_output.exists():
        shutil.rmtree(
            split_output
        )

    split_output.mkdir(
        parents=True,
        exist_ok=True
    )


    for class_name in HARD_CLASSES.values():

        (
            split_output
            / class_name
        ).mkdir(
            parents=True,
            exist_ok=True
        )


    # --------------------------------------------------------
    # Statistics
    # --------------------------------------------------------

    total_gt = 0
    hard_gt = 0

    total_detections = 0
    total_matches = 0
    hard_matches = 0

    invalid_crops = 0

    saved_counts = Counter()


    image_ids = sorted(
        image_info.keys()
    )


    # --------------------------------------------------------
    # Loop over images
    # --------------------------------------------------------

    for image_id in tqdm(
        image_ids,
        desc=f"E18-A {split_name}"
    ):

        info = image_info[
            image_id
        ]

        image_path = (
            image_dir
            / info["file_name"]
        )

        if not image_path.exists():

            print(
                f"\nWARNING: missing image:"
                f"\n{image_path}"
            )

            continue


        gt_objects = anns_by_image.get(
            image_id,
            []
        )

        total_gt += len(
            gt_objects
        )

        hard_gt += sum(
            1
            for obj in gt_objects
            if obj["class_idx"] in HARD_CLASSES
        )


        # ----------------------------------------------------
        # YOLO inference
        # ----------------------------------------------------

        result = model.predict(

            source=str(
                image_path
            ),

            imgsz=IMAGE_SIZE,

            conf=CONF_THRESHOLD,

            iou=NMS_IOU,

            max_det=MAX_DET,

            device=DEVICE,

            verbose=False,
        )[0]


        pred_boxes = []


        if (
            result.boxes
            is not None
            and len(
                result.boxes
            ) > 0
        ):

            xyxy = (
                result.boxes.xyxy
                .detach()
                .cpu()
                .numpy()
            )

            confs = (
                result.boxes.conf
                .detach()
                .cpu()
                .numpy()
            )

            classes = (
                result.boxes.cls
                .detach()
                .cpu()
                .numpy()
                .astype(int)
            )


            for box, conf, cls_idx in zip(
                xyxy,
                confs,
                classes
            ):

                pred_boxes.append(
                    {
                        "bbox": [
                            float(v)
                            for v in box
                        ],

                        "confidence":
                            float(
                                conf
                            ),

                        "pred_class_idx":
                            int(
                                cls_idx
                            ),
                    }
                )


        total_detections += len(
            pred_boxes
        )


        # ----------------------------------------------------
        # Geometry-only one-to-one matching
        #
        # IMPORTANT:
        # Detector predicted class is NOT used for matching.
        # ----------------------------------------------------

        candidates = []


        for gt_idx, gt in enumerate(
            gt_objects
        ):

            for pred_idx, pred in enumerate(
                pred_boxes
            ):

                iou = bbox_iou_xyxy(
                    gt["bbox"],
                    pred["bbox"]
                )

                if iou >= MATCH_IOU_THRESHOLD:

                    candidates.append(
                        (
                            iou,
                            gt_idx,
                            pred_idx
                        )
                    )


        # Highest IoU first
        candidates.sort(
            key=lambda x: x[0],
            reverse=True
        )


        used_gt = set()
        used_pred = set()

        matches = []


        for iou, gt_idx, pred_idx in candidates:

            if gt_idx in used_gt:
                continue

            if pred_idx in used_pred:
                continue


            used_gt.add(
                gt_idx
            )

            used_pred.add(
                pred_idx
            )

            matches.append(
                (
                    gt_idx,
                    pred_idx,
                    iou
                )
            )


        total_matches += len(
            matches
        )


        # ----------------------------------------------------
        # Save hard-class detector crops
        # ----------------------------------------------------

        image = Image.open(
            image_path
        ).convert(
            "RGB"
        )


        for (
            gt_idx,
            pred_idx,
            match_iou
        ) in matches:

            gt = gt_objects[
                gt_idx
            ]

            pred = pred_boxes[
                pred_idx
            ]

            target_class_idx = int(
                gt["class_idx"]
            )


            # Only save hard classes
            if target_class_idx not in HARD_CLASSES:
                continue


            hard_matches += 1


            crop = crop_detector_box(
                image,
                pred["bbox"]
            )


            if crop is None:

                invalid_crops += 1
                continue


            class_name = HARD_CLASSES[
                target_class_idx
            ]


            output_dir = (
                split_output
                / class_name
            )


            crop_name = (
                f"{split_name}"
                f"_img{image_id}"
                f"_ann{gt['annotation_id']}"
                f"_iou{match_iou:.3f}"
                f".jpg"
            )


            crop_path = (
                output_dir
                / crop_name
            )


            crop.save(
                crop_path,
                quality=95
            )


            saved_counts[
                class_name
            ] += 1


        image.close()


    # ========================================================
    # SPLIT SUMMARY
    # ========================================================

    print("\n" + "=" * 100)
    print(
        f"{split_name.upper()} SUMMARY"
    )
    print("=" * 100)

    print(
        f"\nTotal GT objects       : "
        f"{total_gt}"
    )

    print(
        f"Hard-class GT objects  : "
        f"{hard_gt}"
    )

    print(
        f"YOLO detections        : "
        f"{total_detections}"
    )

    print(
        f"Total GT/pred matches  : "
        f"{total_matches}"
    )

    print(
        f"Hard-class matches     : "
        f"{hard_matches}"
    )

    print(
        f"Invalid crops          : "
        f"{invalid_crops}"
    )


    if hard_gt > 0:

        print(
            f"Hard-class coverage    : "
            f"{hard_matches / hard_gt * 100:.2f}%"
        )


    print(
        "\nSaved crops by class:"
    )


    for class_name in HARD_CLASSES.values():

        print(
            f"{class_name:25s}: "
            f"{saved_counts[class_name]}"
        )


    print(
        f"\nTotal crops saved: "
        f"{sum(saved_counts.values())}"
    )


    return {
        "split":
            split_name,

        "total_gt":
            total_gt,

        "hard_gt":
            hard_gt,

        "detections":
            total_detections,

        "total_matches":
            total_matches,

        "hard_matches":
            hard_matches,

        "invalid_crops":
            invalid_crops,

        "saved_counts":
            dict(
                saved_counts
            ),
    }


# ============================================================
# 9. GENERATE TRAIN + VALIDATION CROPS
# ============================================================

all_summaries = []


for split in [
    "train",
    "val"
]:

    summary = process_split(
        split
    )

    all_summaries.append(
        summary
    )


# ============================================================
# 10. SAVE SUMMARY JSON
# ============================================================

SUMMARY_JSON = (
    OUTPUT_ROOT
    / "E18A_crop_generation_summary.json"
)


with open(
    SUMMARY_JSON,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        all_summaries,
        f,
        indent=2
    )


# ============================================================
# 11. FINAL SUMMARY
# ============================================================

print("\n" + "=" * 100)
print("E18-A COMPLETE")
print("=" * 100)

print("\nCrop dataset:")
print(OUTPUT_ROOT)

print("\nSummary JSON:")
print(SUMMARY_JSON)

print(
    "\nGenerated splits:"
    "\n  train"
    "\n  val"
)

print(
    "\nTest set was NOT used."
)

E18-A — YOLO11m-MATCHED HARD-CLASS CROP DATASET GENERATION

Detector:
C:\Users\varda\Documents\_My Computer\COMMON_space\Learning\Upgrad_EPGP_MS\upgrad_IIITB_MS_course\MS_LJMU_Material\Thesis_Code\runs\sortwaste\E12_yolo11m_7class_aug_classbalance_640\weights\best.pt

Output:
C:\Users\varda\Documents\_My Computer\COMMON_space\Learning\Upgrad_EPGP_MS\upgrad_IIITB_MS_course\MS_LJMU_Material\Topic Data\SortWaste\dataset\dataset\yolo11m_convnext_hardclass_crops_E18

Hard classes:
2: mixed_plastic_rigid
3: mixed_plastic_soft
4: non_plastic
6: pet_oil

PROCESSING SPLIT: TRAIN


E18-A train: 100%|██████████| 3705/3705 [13:13<00:00,  4.67it/s]



TRAIN SUMMARY

Total GT objects       : 61842
Hard-class GT objects  : 19414
YOLO detections        : 231691
Total GT/pred matches  : 61674
Hard-class matches     : 19378
Invalid crops          : 0
Hard-class coverage    : 99.81%

Saved crops by class:
mixed_plastic_rigid      : 7052
mixed_plastic_soft       : 9057
non_plastic              : 2468
pet_oil                  : 801

Total crops saved: 19378

PROCESSING SPLIT: VAL


E18-A val: 100%|██████████| 780/780 [02:41<00:00,  4.83it/s]


VAL SUMMARY

Total GT objects       : 13065
Hard-class GT objects  : 3433
YOLO detections        : 55605
Total GT/pred matches  : 12697
Hard-class matches     : 3343
Invalid crops          : 0
Hard-class coverage    : 97.38%

Saved crops by class:
mixed_plastic_rigid      : 1084
mixed_plastic_soft       : 1411
non_plastic              : 682
pet_oil                  : 166

Total crops saved: 3343

E18-A COMPLETE

Crop dataset:
C:\Users\varda\Documents\_My Computer\COMMON_space\Learning\Upgrad_EPGP_MS\upgrad_IIITB_MS_course\MS_LJMU_Material\Topic Data\SortWaste\dataset\dataset\yolo11m_convnext_hardclass_crops_E18

Summary JSON:
C:\Users\varda\Documents\_My Computer\COMMON_space\Learning\Upgrad_EPGP_MS\upgrad_IIITB_MS_course\MS_LJMU_Material\Topic Data\SortWaste\dataset\dataset\yolo11m_convnext_hardclass_crops_E18\E18A_crop_generation_summary.json

Generated splits:
  train
  val

Test set was NOT used.


## E18-B — ConvNeXt-Tiny Training for Hard-Class Refinement Using YOLO11m-Matched Crops

In [ ]:
# E18-B — ConvNeXt-Tiny Training for Hard-Class Refinement Using YOLO11m-Matched Crops
#
# Experiment objective:
#   Train a stronger second-stage classifier specifically for
#   the hard classes identified by the previous experiments.
#
# Dataset:
#   E18-A YOLO11m-matched detector-box crops
#
# Hard classes:
#   0 = mixed_plastic_rigid
#   1 = mixed_plastic_soft
#   2 = non_plastic
#   3 = pet_oil
#
# Controlled training philosophy:
#   - ImageNet pretrained
#   - 224 x 224
#   - Same augmentation family as E3Y-B
#   - Class-weighted CrossEntropyLoss
#   - AdamW
#   - lr = 1e-4
#   - weight_decay = 1e-4
#   - ReduceLROnPlateau
#   - best model by validation Macro-F1
#   - early stopping patience = 7
#   - seed = 42
#
# No:
#   - oversampling
#   - SMOTE
#   - focal loss
#   - class-specific augmentation
#
# NOTE:
#   E16-A's gate/fusion rules are NOT applied during training.
#   They will be preserved during E18-C end-to-end evaluation.
# ============================================================


import os
import json
import random
import time
from pathlib import Path
from collections import Counter

import numpy as np

import torch
import torch.nn as nn

from torch.utils.data import DataLoader

from torchvision import datasets
from torchvision.transforms import v2
from torchvision.models import (
    convnext_tiny,
    ConvNeXt_Tiny_Weights,
)

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix,
)

from tqdm import tqdm


# ============================================================
# 1. PATHS
# ============================================================

BASE = Path(
    r"C:\Users\varda\Documents\_My Computer\COMMON_space\Learning"
    r"\Upgrad_EPGP_MS\upgrad_IIITB_MS_course\MS_LJMU_Material"
)

DATASET_ROOT = (
    BASE
    / "Topic Data"
    / "SortWaste"
    / "dataset"
    / "dataset"
)

THESIS_CODE = (
    BASE
    / "Thesis_Code"
)


# E18-A generated crops
CROP_ROOT = (
    DATASET_ROOT
    / "yolo11m_convnext_hardclass_crops_E18"
)

TRAIN_DIR = (
    CROP_ROOT
    / "train"
)

VAL_DIR = (
    CROP_ROOT
    / "val"
)


# E18-B outputs
OUTPUT_DIR = (
    THESIS_CODE
    / "runs"
    / "sortwaste"
    / "E18B_convnext_tiny_hardclass"
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)


BEST_MODEL_PATH = (
    OUTPUT_DIR
    / "E18B_ConvNeXtTiny_best.pth"
)

LAST_MODEL_PATH = (
    OUTPUT_DIR
    / "E18B_ConvNeXtTiny_last.pth"
)

HISTORY_PATH = (
    OUTPUT_DIR
    / "E18B_training_history.json"
)

SUMMARY_PATH = (
    OUTPUT_DIR
    / "E18B_summary.txt"
)


assert TRAIN_DIR.exists(), (
    f"Training crops not found:\n{TRAIN_DIR}"
)

assert VAL_DIR.exists(), (
    f"Validation crops not found:\n{VAL_DIR}"
)


# ============================================================
# 2. REPRODUCIBILITY
# ============================================================

SEED = 42


def seed_everything(seed):

    random.seed(seed)

    np.random.seed(seed)

    torch.manual_seed(seed)

    torch.cuda.manual_seed(seed)

    torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True

    torch.backends.cudnn.benchmark = False


seed_everything(SEED)


# ============================================================
# 3. DEVICE
# ============================================================

DEVICE = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)


print("=" * 100)
print("E18-B — ConvNeXt-Tiny Hard-Class Training")
print("=" * 100)

print(f"\nPyTorch : {torch.__version__}")
print(f"Device  : {DEVICE}")


if torch.cuda.is_available():

    print(
        f"GPU     : "
        f"{torch.cuda.get_device_name(0)}"
    )

    gpu_mem = (
        torch.cuda.get_device_properties(0)
        .total_memory
        / 1024**3
    )

    print(
        f"VRAM    : "
        f"{gpu_mem:.2f} GB"
    )


# ============================================================
# 4. HYPERPARAMETERS
# ============================================================

IMAGE_SIZE = 224

# ConvNeXt-Tiny is heavier than MobileNet.
# Start with 8 on 4 GB RTX 3050 Ti.
BATCH_SIZE = 8

NUM_EPOCHS = 30

LEARNING_RATE = 1e-4

WEIGHT_DECAY = 1e-4

NUM_WORKERS = 0

PATIENCE = 7

MIN_DELTA = 1e-4


# ============================================================
# 5. TRANSFORMS
#
# Deliberately follows the E3Y-B training augmentation family.
# No RandomResizedCrop.
# ============================================================

IMAGENET_MEAN = [
    0.485,
    0.456,
    0.406,
]

IMAGENET_STD = [
    0.229,
    0.224,
    0.225,
]


train_transform = v2.Compose(
    [
        v2.Resize(
            (
                IMAGE_SIZE,
                IMAGE_SIZE,
            )
        ),

        v2.RandomHorizontalFlip(
            p=0.5
        ),

        v2.RandomRotation(
            degrees=10
        ),

        v2.ColorJitter(
            brightness=0.15,
            contrast=0.15,
            saturation=0.10,
            hue=0.02,
        ),

        v2.ToImage(),

        v2.ToDtype(
            torch.float32,
            scale=True
        ),

        v2.Normalize(
            mean=IMAGENET_MEAN,
            std=IMAGENET_STD,
        ),
    ]
)


val_transform = v2.Compose(
    [
        v2.Resize(
            (
                IMAGE_SIZE,
                IMAGE_SIZE,
            )
        ),

        v2.ToImage(),

        v2.ToDtype(
            torch.float32,
            scale=True
        ),

        v2.Normalize(
            mean=IMAGENET_MEAN,
            std=IMAGENET_STD,
        ),
    ]
)


# ============================================================
# 6. DATASETS
# ============================================================

train_dataset = datasets.ImageFolder(
    root=TRAIN_DIR,
    transform=train_transform,
)

val_dataset = datasets.ImageFolder(
    root=VAL_DIR,
    transform=val_transform,
)


print("\n" + "=" * 100)
print("DATASET INFORMATION")
print("=" * 100)

print(
    f"\nTrain samples : "
    f"{len(train_dataset):,}"
)

print(
    f"Val samples   : "
    f"{len(val_dataset):,}"
)


print("\nImageFolder class mapping:")

for class_name, class_idx in (
    train_dataset.class_to_idx.items()
):

    print(
        f"{class_idx}: {class_name}"
    )


# ============================================================
# 7. EXPECTED HARD-CLASS MAPPING
# ============================================================

EXPECTED_CLASSES = [
    "mixed_plastic_rigid",
    "mixed_plastic_soft",
    "non_plastic",
    "pet_oil",
]


assert train_dataset.classes == EXPECTED_CLASSES, (
    "\nUnexpected train class order."
    f"\nExpected: {EXPECTED_CLASSES}"
    f"\nFound   : {train_dataset.classes}"
)


assert val_dataset.classes == EXPECTED_CLASSES, (
    "\nUnexpected validation class order."
    f"\nExpected: {EXPECTED_CLASSES}"
    f"\nFound   : {val_dataset.classes}"
)


assert (
    train_dataset.class_to_idx
    == val_dataset.class_to_idx
), (
    "Train/validation class mappings differ."
)


NUM_CLASSES = len(
    EXPECTED_CLASSES
)


CLASS_TO_IDX = (
    train_dataset.class_to_idx
)

IDX_TO_CLASS = {
    idx: name
    for name, idx in CLASS_TO_IDX.items()
}


# ============================================================
# 8. TRAINING CLASS DISTRIBUTION
# ============================================================

train_targets = [
    target
    for _, target in train_dataset.samples
]

train_counts = Counter(
    train_targets
)


print(
    "\nTraining class distribution:"
)

for idx in range(
    NUM_CLASSES
):

    print(
        f"{IDX_TO_CLASS[idx]:25s}: "
        f"{train_counts[idx]:,}"
    )


# ============================================================
# 9. CLASS WEIGHTS
#
# Same weighting formula used for E3Y-B:
#
#       N
# w = -------
#     K * n_c
#
# where:
#   N   = total training samples
#   K   = number of classes
#   n_c = samples belonging to class c
# ============================================================

total_train = len(
    train_dataset
)


class_weights = []


for class_idx in range(
    NUM_CLASSES
):

    class_count = (
        train_counts[class_idx]
    )

    weight = (
        total_train
        /
        (
            NUM_CLASSES
            * class_count
        )
    )

    class_weights.append(
        weight
    )


class_weights = torch.tensor(
    class_weights,
    dtype=torch.float32,
    device=DEVICE,
)


print(
    "\nClass weights:"
)

for idx, weight in enumerate(
    class_weights.detach().cpu().tolist()
):

    print(
        f"{IDX_TO_CLASS[idx]:25s}: "
        f"{weight:.6f}"
    )


# ============================================================
# 10. DATA LOADERS
# ============================================================

generator = torch.Generator()

generator.manual_seed(
    SEED
)


train_loader = DataLoader(
    train_dataset,

    batch_size=BATCH_SIZE,

    shuffle=True,

    num_workers=NUM_WORKERS,

    pin_memory=(
        DEVICE.type == "cuda"
    ),

    generator=generator,
)


val_loader = DataLoader(
    val_dataset,

    batch_size=BATCH_SIZE,

    shuffle=False,

    num_workers=NUM_WORKERS,

    pin_memory=(
        DEVICE.type == "cuda"
    ),
)


print(
    f"\nTrain batches : "
    f"{len(train_loader):,}"
)

print(
    f"Val batches   : "
    f"{len(val_loader):,}"
)


# ============================================================
# 11. LOAD PRETRAINED CONVNEXT-TINY
# ============================================================

weights = (
    ConvNeXt_Tiny_Weights.DEFAULT
)


model = convnext_tiny(
    weights=weights
)


# Torchvision ConvNeXt-Tiny classifier:
#
# Sequential(
#     LayerNorm2d(...)
#     Flatten(...)
#     Linear(768, 1000)
# )
#
# Replace only final Linear layer.

in_features = (
    model.classifier[2]
    .in_features
)


model.classifier[2] = nn.Linear(
    in_features,
    NUM_CLASSES,
)


model = model.to(
    DEVICE
)


print("\n" + "=" * 100)
print("MODEL")
print("=" * 100)

print(
    "\nModel              : "
    "ConvNeXt-Tiny"
)

print(
    "Pretrained weights : "
    "ImageNet DEFAULT"
)

print(
    f"Classifier input   : "
    f"{in_features}"
)

print(
    f"Output classes     : "
    f"{NUM_CLASSES}"
)


total_params = sum(
    p.numel()
    for p in model.parameters()
)


trainable_params = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)


print(
    f"Total parameters   : "
    f"{total_params:,}"
)

print(
    f"Trainable params   : "
    f"{trainable_params:,}"
)


# ============================================================
# 12. LOSS
# ============================================================

criterion = nn.CrossEntropyLoss(
    weight=class_weights
)


# ============================================================
# 13. OPTIMIZER
#
# Same optimizer family and hyperparameters as E3Y-B.
# ============================================================

optimizer = torch.optim.AdamW(
    model.parameters(),

    lr=LEARNING_RATE,

    weight_decay=WEIGHT_DECAY,
)


# ============================================================
# 14. LR SCHEDULER
#
# Same E3Y-B scheduler philosophy.
# Monitor validation Macro-F1.
# ============================================================

scheduler = (
    torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,

        mode="max",

        factor=0.5,

        patience=2,
    )
)


# ============================================================
# 15. AMP
# ============================================================

USE_AMP = (
    DEVICE.type == "cuda"
)


scaler = torch.amp.GradScaler(
    "cuda",
    enabled=USE_AMP,
)


print(
    f"\nAMP enabled : "
    f"{USE_AMP}"
)


# ============================================================
# 16. TRAIN ONE EPOCH
# ============================================================

def train_one_epoch(
    model,
    loader,
    optimizer,
    criterion,
):

    model.train()

    running_loss = 0.0

    all_targets = []
    all_preds = []


    progress = tqdm(
        loader,
        desc="Training",
        leave=False,
    )


    for images, targets in progress:

        images = images.to(
            DEVICE,
            non_blocking=True,
        )

        targets = targets.to(
            DEVICE,
            non_blocking=True,
        )


        optimizer.zero_grad(
            set_to_none=True
        )


        with torch.amp.autocast(
            device_type="cuda",
            enabled=USE_AMP,
        ):

            logits = model(
                images
            )

            loss = criterion(
                logits,
                targets
            )


        scaler.scale(
            loss
        ).backward()


        scaler.step(
            optimizer
        )


        scaler.update()


        running_loss += (
            loss.item()
            * images.size(0)
        )


        predictions = torch.argmax(
            logits,
            dim=1,
        )


        all_targets.extend(
            targets.detach()
            .cpu()
            .tolist()
        )

        all_preds.extend(
            predictions.detach()
            .cpu()
            .tolist()
        )


        progress.set_postfix(
            loss=f"{loss.item():.4f}"
        )


    epoch_loss = (
        running_loss
        / len(loader.dataset)
    )


    epoch_accuracy = accuracy_score(
        all_targets,
        all_preds,
    )


    epoch_macro_f1 = f1_score(
        all_targets,
        all_preds,
        average="macro",
        zero_division=0,
    )


    epoch_weighted_f1 = f1_score(
        all_targets,
        all_preds,
        average="weighted",
        zero_division=0,
    )


    return {
        "loss":
            epoch_loss,

        "accuracy":
            epoch_accuracy,

        "macro_f1":
            epoch_macro_f1,

        "weighted_f1":
            epoch_weighted_f1,
    }


# ============================================================
# 17. VALIDATION
# ============================================================

@torch.no_grad()
def validate(
    model,
    loader,
    criterion,
):

    model.eval()

    running_loss = 0.0

    all_targets = []
    all_preds = []


    progress = tqdm(
        loader,
        desc="Validation",
        leave=False,
    )


    for images, targets in progress:

        images = images.to(
            DEVICE,
            non_blocking=True,
        )

        targets = targets.to(
            DEVICE,
            non_blocking=True,
        )


        with torch.amp.autocast(
            device_type="cuda",
            enabled=USE_AMP,
        ):

            logits = model(
                images
            )

            loss = criterion(
                logits,
                targets
            )


        running_loss += (
            loss.item()
            * images.size(0)
        )


        predictions = torch.argmax(
            logits,
            dim=1,
        )


        all_targets.extend(
            targets.detach()
            .cpu()
            .tolist()
        )

        all_preds.extend(
            predictions.detach()
            .cpu()
            .tolist()
        )


    epoch_loss = (
        running_loss
        / len(loader.dataset)
    )


    epoch_accuracy = accuracy_score(
        all_targets,
        all_preds,
    )


    epoch_macro_f1 = f1_score(
        all_targets,
        all_preds,
        average="macro",
        zero_division=0,
    )


    epoch_weighted_f1 = f1_score(
        all_targets,
        all_preds,
        average="weighted",
        zero_division=0,
    )


    report = classification_report(
        all_targets,
        all_preds,

        labels=list(
            range(
                NUM_CLASSES
            )
        ),

        target_names=[
            IDX_TO_CLASS[i]
            for i in range(
                NUM_CLASSES
            )
        ],

        digits=4,

        zero_division=0,

        output_dict=True,
    )


    cm = confusion_matrix(
        all_targets,
        all_preds,

        labels=list(
            range(
                NUM_CLASSES
            )
        ),
    )


    return {
        "loss":
            epoch_loss,

        "accuracy":
            epoch_accuracy,

        "macro_f1":
            epoch_macro_f1,

        "weighted_f1":
            epoch_weighted_f1,

        "report":
            report,

        "confusion_matrix":
            cm.tolist(),

        "targets":
            all_targets,

        "predictions":
            all_preds,
    }


# ============================================================
# 18. TRAINING LOOP
# ============================================================

history = []


best_macro_f1 = -1.0

best_epoch = 0

epochs_without_improvement = 0


training_start = time.time()


for epoch in range(
    1,
    NUM_EPOCHS + 1
):

    print("\n" + "=" * 100)
    print(
        f"EPOCH "
        f"{epoch:02d}/{NUM_EPOCHS}"
    )
    print("=" * 100)


    epoch_start = time.time()


    train_metrics = train_one_epoch(
        model,
        train_loader,
        optimizer,
        criterion,
    )


    val_metrics = validate(
        model,
        val_loader,
        criterion,
    )


    # --------------------------------------------------------
    # Scheduler uses validation Macro-F1
    # --------------------------------------------------------

    scheduler.step(
        val_metrics[
            "macro_f1"
        ]
    )


    current_lr = (
        optimizer.param_groups[0][
            "lr"
        ]
    )


    epoch_seconds = (
        time.time()
        - epoch_start
    )


    # --------------------------------------------------------
    # Print epoch metrics
    # --------------------------------------------------------

    print(
        f"\nTrain loss       : "
        f"{train_metrics['loss']:.4f}"
    )

    print(
        f"Train accuracy   : "
        f"{train_metrics['accuracy'] * 100:.2f}%"
    )

    print(
        f"Train Macro-F1   : "
        f"{train_metrics['macro_f1']:.4f}"
    )

    print(
        f"Train Weighted-F1: "
        f"{train_metrics['weighted_f1']:.4f}"
    )


    print(
        f"\nVal loss         : "
        f"{val_metrics['loss']:.4f}"
    )

    print(
        f"Val accuracy     : "
        f"{val_metrics['accuracy'] * 100:.2f}%"
    )

    print(
        f"Val Macro-F1     : "
        f"{val_metrics['macro_f1']:.4f}"
    )

    print(
        f"Val Weighted-F1  : "
        f"{val_metrics['weighted_f1']:.4f}"
    )


    print(
        f"\nLearning rate    : "
        f"{current_lr:.8f}"
    )

    print(
        f"Epoch time       : "
        f"{epoch_seconds / 60:.2f} min"
    )


    # --------------------------------------------------------
    # Per-class F1
    # --------------------------------------------------------

    print(
        "\nValidation per-class F1:"
    )


    for class_idx in range(
        NUM_CLASSES
    ):

        class_name = (
            IDX_TO_CLASS[
                class_idx
            ]
        )

        f1 = (
            val_metrics[
                "report"
            ][class_name]["f1-score"]
        )

        support = int(
            val_metrics[
                "report"
            ][class_name]["support"]
        )


        print(
            f"{class_name:25s} "
            f"F1={f1:.4f} "
            f"support={support:,}"
        )


    # --------------------------------------------------------
    # Save history
    # --------------------------------------------------------

    history_entry = {

        "epoch":
            epoch,

        "lr":
            current_lr,

        "train_loss":
            train_metrics["loss"],

        "train_accuracy":
            train_metrics["accuracy"],

        "train_macro_f1":
            train_metrics["macro_f1"],

        "train_weighted_f1":
            train_metrics[
                "weighted_f1"
            ],

        "val_loss":
            val_metrics["loss"],

        "val_accuracy":
            val_metrics["accuracy"],

        "val_macro_f1":
            val_metrics["macro_f1"],

        "val_weighted_f1":
            val_metrics[
                "weighted_f1"
            ],

        "epoch_seconds":
            epoch_seconds,
    }


    history.append(
        history_entry
    )


    with open(
        HISTORY_PATH,
        "w",
        encoding="utf-8"
    ) as f:

        json.dump(
            history,
            f,
            indent=2
        )


    # --------------------------------------------------------
    # Best checkpoint based on validation Macro-F1
    #
    # Same model-selection metric as E3Y-B.
    # --------------------------------------------------------

    improvement = (
        val_metrics["macro_f1"]
        - best_macro_f1
    )


    if improvement > MIN_DELTA:

        best_macro_f1 = (
            val_metrics[
                "macro_f1"
            ]
        )

        best_epoch = epoch

        epochs_without_improvement = 0


        checkpoint = {

            "experiment":
                "E18-B",

            "model_name":
                "ConvNeXt-Tiny",

            "epoch":
                epoch,

            "model_state_dict":
                model.state_dict(),

            "optimizer_state_dict":
                optimizer.state_dict(),

            "val_macro_f1":
                val_metrics[
                    "macro_f1"
                ],

            "val_accuracy":
                val_metrics[
                    "accuracy"
                ],

            "val_weighted_f1":
                val_metrics[
                    "weighted_f1"
                ],

            "class_to_idx":
                CLASS_TO_IDX,

            "idx_to_class":
                IDX_TO_CLASS,

            "class_weights":
                class_weights.detach()
                .cpu()
                .tolist(),

            "image_size":
                IMAGE_SIZE,

            "hard_classes":
                EXPECTED_CLASSES,

            "seed":
                SEED,
        }


        torch.save(
            checkpoint,
            BEST_MODEL_PATH
        )


        print(
            "\n*** NEW BEST MODEL ***"
        )

        print(
            f"Epoch            : "
            f"{epoch}"
        )

        print(
            f"Val Macro-F1     : "
            f"{best_macro_f1:.4f}"
        )

        print(
            f"Saved            : "
            f"{BEST_MODEL_PATH}"
        )


    else:

        epochs_without_improvement += 1


        print(
            f"\nNo Macro-F1 improvement "
            f"for "
            f"{epochs_without_improvement}"
            f"/{PATIENCE} epochs."
        )


    # --------------------------------------------------------
    # Early stopping
    # --------------------------------------------------------

    if (
        epochs_without_improvement
        >= PATIENCE
    ):

        print(
            "\nEARLY STOPPING TRIGGERED"
        )

        print(
            f"Best epoch    : "
            f"{best_epoch}"
        )

        print(
            f"Best Macro-F1 : "
            f"{best_macro_f1:.4f}"
        )

        break


# ============================================================
# 19. SAVE LAST MODEL
# ============================================================

torch.save(
    {
        "experiment":
            "E18-B",

        "model_name":
            "ConvNeXt-Tiny",

        "epoch":
            epoch,

        "model_state_dict":
            model.state_dict(),

        "optimizer_state_dict":
            optimizer.state_dict(),

        "class_to_idx":
            CLASS_TO_IDX,

        "idx_to_class":
            IDX_TO_CLASS,

        "image_size":
            IMAGE_SIZE,

        "hard_classes":
            EXPECTED_CLASSES,
    },

    LAST_MODEL_PATH,
)


# ============================================================
# 20. LOAD BEST MODEL FOR FINAL VALIDATION
# ============================================================

print("\n" + "=" * 100)
print("FINAL VALIDATION USING BEST CHECKPOINT")
print("=" * 100)


checkpoint = torch.load(
    BEST_MODEL_PATH,
    map_location=DEVICE,
    weights_only=False,
)


model.load_state_dict(
    checkpoint[
        "model_state_dict"
    ]
)


final_val = validate(
    model,
    val_loader,
    criterion,
)


# ============================================================
# 21. FINAL CLASSIFICATION REPORT
# ============================================================

print(
    f"\nBest epoch       : "
    f"{checkpoint['epoch']}"
)

print(
    f"Validation loss  : "
    f"{final_val['loss']:.4f}"
)

print(
    f"Validation acc   : "
    f"{final_val['accuracy'] * 100:.2f}%"
)

print(
    f"Validation Macro-F1    : "
    f"{final_val['macro_f1']:.4f}"
)

print(
    f"Validation Weighted-F1 : "
    f"{final_val['weighted_f1']:.4f}"
)


print("\nClassification report:")


print(
    classification_report(

        final_val[
            "targets"
        ],

        final_val[
            "predictions"
        ],

        labels=list(
            range(
                NUM_CLASSES
            )
        ),

        target_names=[
            IDX_TO_CLASS[i]
            for i in range(
                NUM_CLASSES
            )
        ],

        digits=4,

        zero_division=0,
    )
)


print(
    "\nConfusion matrix:"
)

print(
    np.array(
        final_val[
            "confusion_matrix"
        ]
    )
)


# ============================================================
# 22. SAVE SUMMARY
# ============================================================

training_minutes = (
    time.time()
    - training_start
) / 60


summary_lines = [

    "=" * 100,

    "E18-B — ConvNeXt-Tiny Hard-Class Refinement",

    "=" * 100,

    "",

    f"Training crops: {TRAIN_DIR}",

    f"Validation crops: {VAL_DIR}",

    "",

    "Hard classes:",

]


for idx in range(
    NUM_CLASSES
):

    summary_lines.append(
        f"  {idx}: "
        f"{IDX_TO_CLASS[idx]}"
    )


summary_lines.extend(
    [
        "",

        f"Train samples: "
        f"{len(train_dataset)}",

        f"Validation samples: "
        f"{len(val_dataset)}",

        "",

        f"Image size: "
        f"{IMAGE_SIZE}",

        f"Batch size: "
        f"{BATCH_SIZE}",

        f"Initial learning rate: "
        f"{LEARNING_RATE}",

        f"Weight decay: "
        f"{WEIGHT_DECAY}",

        f"Max epochs: "
        f"{NUM_EPOCHS}",

        f"Early stopping patience: "
        f"{PATIENCE}",

        f"Seed: "
        f"{SEED}",

        "",

        "Loss: class-weighted CrossEntropyLoss",

        "Optimizer: AdamW",

        (
            "Scheduler: "
            "ReduceLROnPlateau "
            "(validation Macro-F1)"
        ),

        "",

        f"Best epoch: "
        f"{checkpoint['epoch']}",

        f"Best validation accuracy: "
        f"{final_val['accuracy']:.6f}",

        f"Best validation Macro-F1: "
        f"{final_val['macro_f1']:.6f}",

        f"Best validation Weighted-F1: "
        f"{final_val['weighted_f1']:.6f}",

        "",

        f"Training time: "
        f"{training_minutes:.2f} minutes",

        "",

        f"Best checkpoint: "
        f"{BEST_MODEL_PATH}",
    ]
)


with open(
    SUMMARY_PATH,
    "w",
    encoding="utf-8"
) as f:

    f.write(
        "\n".join(
            summary_lines
        )
    )


# ============================================================
# 23. COMPLETE
# ============================================================

print("\n" + "=" * 100)
print("E18-B COMPLETE")
print("=" * 100)

print(
    f"\nBest checkpoint:"
    f"\n{BEST_MODEL_PATH}"
)

print(
    f"\nTraining history:"
    f"\n{HISTORY_PATH}"
)

print(
    f"\nSummary:"
    f"\n{SUMMARY_PATH}"
)

E18-B — ConvNeXt-Tiny Hard-Class Training

PyTorch : 2.13.0+cu126
Device  : cuda
GPU     : NVIDIA GeForce RTX 3050 Ti Laptop GPU
VRAM    : 4.00 GB


KeyboardInterrupt: 

## E18-C — YOLO11m + ConvNeXt-Tiny Hard-Class Refinement with E16-A Gating and Confidence Fusion = YOLO11m + MobileNet + ConvNeXt specialist
The E18-C prediction pipeline is:

$$ \boxed{ \text{YOLO11m} \rightarrow \text{E16-A MobileNet gating} \rightarrow \text{ConvNeXt specialist for hard cases} \rightarrow \text{final predictions} \rightarrow \text{COCOeval} } $$

In [ ]:
# E18-C — YOLO11m + ConvNeXt-Tiny Hard-Class Refinement with E16-A Gating and Confidence Fusion
#
# Baseline reproduced:
#   E16-A
#   YOLO11m + MobileNetV3-Large
#   class-specific MobileNet gating
#   alpha = 0.70
#
# New E18-C component:
#   ConvNeXt-Tiny is used ONLY as a specialist when the
#   E16-A final class belongs to one of:
#
#       mixed_plastic_rigid
#       mixed_plastic_soft
#       non_plastic
#       pet_oil
#
# Easy classes remain controlled by the original E16-A:
#       ecal
#       hdpe
#       pet
#
# Final confidence:
#
#   score =
#       YOLO_confidence ^ 0.70
#       *
#       classifier_probability ^ 0.30
#
# Classifier probability:
#   - ConvNeXt probability if ConvNeXt supplies final class
#   - MobileNet probability otherwise
#
# Evaluation:
#   Explicit pycocotools.COCOeval
#   IoU = 0.50:0.95
#   maxDets = [1, 10, 100]
#
# IMPORTANT:
#   This is validation only.
#   Test set remains untouched.
# ============================================================


import json
import time
import random
from pathlib import Path
from collections import Counter

import numpy as np

from PIL import Image

import torch
import torch.nn as nn

from torchvision.transforms import v2
from torchvision.models import (
    mobilenet_v3_large,
    MobileNet_V3_Large_Weights,
    convnext_tiny,
    ConvNeXt_Tiny_Weights,
)

from ultralytics import YOLO

from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval


# ============================================================
# 1. REPRODUCIBILITY
# ============================================================

SEED = 42

random.seed(SEED)
np.random.seed(SEED)

torch.manual_seed(SEED)
torch.cuda.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False


# ============================================================
# 2. PATHS
# ============================================================

BASE = Path(
    r"C:\Users\varda\Documents\_My Computer\COMMON_space\Learning"
    r"\Upgrad_EPGP_MS\upgrad_IIITB_MS_course\MS_LJMU_Material"
)

THESIS_CODE = (
    BASE
    / "Thesis_Code"
)

DATASET_ROOT = (
    BASE
    / "Topic Data"
    / "SortWaste"
    / "dataset"
    / "dataset"
)

COCO_ROOT = (
    DATASET_ROOT
    / "splited_all_dataset_coco"
)


# ------------------------------------------------------------
# Validation data
# ------------------------------------------------------------

VAL_IMAGES = (
    COCO_ROOT
    / "val"
    / "images"
)

VAL_COCO = (
    COCO_ROOT
    / "val"
    / "annotations"
    / "val_coco.json"
)


# ------------------------------------------------------------
# E12 YOLO11m detector
# ------------------------------------------------------------

YOLO_MODEL_PATH = (
    THESIS_CODE
    / "runs"
    / "sortwaste"
    / "E12_yolo11m_7class_aug_classbalance_640"
    / "weights"
    / "best.pt"
)


# ------------------------------------------------------------
# Existing E3Y-B MobileNet used in E16-A
# ------------------------------------------------------------

MOBILENET_PATH = (
    DATASET_ROOT
    / "yolo_mobilenet_crops_E3Y"
    / "mobilenet_results"
    / "E3Y_B_class_weighted"
    / "E3Y_B_MobileNetV3Large_best.pth"
)


# ------------------------------------------------------------
# New E18-B ConvNeXt specialist
# ------------------------------------------------------------

CONVNEXT_PATH = (
    THESIS_CODE
    / "runs"
    / "sortwaste"
    / "E18B_convnext_tiny_hardclass"
    / "E18B_ConvNeXtTiny_best.pth"
)


# ------------------------------------------------------------
# Output
# ------------------------------------------------------------

OUTPUT_DIR = (
    THESIS_CODE
    / "runs"
    / "sortwaste"
    / "E18C_yolo11m_convnext_hardclass"
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)


GT_7CLASS_PATH = (
    OUTPUT_DIR
    / "E18C_val_gt_7class.json"
)

E16_PRED_PATH = (
    OUTPUT_DIR
    / "E18C_E16A_baseline_predictions.json"
)

E18_PRED_PATH = (
    OUTPUT_DIR
    / "E18C_predictions.json"
)

SUMMARY_PATH = (
    OUTPUT_DIR
    / "E18C_summary.txt"
)


# ============================================================
# 3. CHECK PATHS
# ============================================================

for path, label in [

    (VAL_IMAGES, "Validation images"),

    (VAL_COCO, "Validation COCO"),

    (YOLO_MODEL_PATH, "E12 YOLO11m"),

    (MOBILENET_PATH, "E3Y-B MobileNet"),

    (CONVNEXT_PATH, "E18-B ConvNeXt"),

]:

    assert path.exists(), (
        f"{label} not found:\n{path}"
    )


# ============================================================
# 4. DEVICE
# ============================================================

DEVICE = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)


print("=" * 100)
print(
    "E18-C — YOLO11m + ConvNeXt-Tiny Hard-Class Refinement"
)
print("=" * 100)

print(
    f"\nDevice : {DEVICE}"
)

if torch.cuda.is_available():

    print(
        "GPU    : "
        f"{torch.cuda.get_device_name(0)}"
    )


# ============================================================
# 5. DETECTOR SETTINGS
#
# Same settings used by E13-E16.
# ============================================================

YOLO_IMAGE_SIZE = 640

YOLO_CONF = 0.001

YOLO_NMS = 0.60

YOLO_MAX_DET = 100


# ============================================================
# 6. CLASSIFIER SETTINGS
# ============================================================

CLASSIFIER_SIZE = 224

ALPHA = 0.70


# ============================================================
# 7. 7-CLASS TAXONOMY
# ============================================================

CLASS_NAMES = [

    "ecal",

    "hdpe",

    "mixed_plastic_rigid",

    "mixed_plastic_soft",

    "non_plastic",

    "pet",

    "pet_oil",
]


NUM_CLASSES = len(
    CLASS_NAMES
)


# ------------------------------------------------------------
# Original COCO -> 7-class zero-based index
# ------------------------------------------------------------

COCO_TO_7CLASS = {

    1: 5,  # pet

    2: 1,  # hdpe

    3: 3,  # mixed soft

    4: 0,  # ecal

    5: 4,  # metal -> non_plastic

    6: 4,  # cardboard -> non_plastic

    7: 2,  # mixed rigid

    8: 6,  # pet_oil
}


# ============================================================
# 8. E16-A CLASS-SPECIFIC MOBILENET GATES
#
# Indexed by MobileNet predicted class.
# ============================================================

E16A_GATES = {

    0: 0.99,   # ecal

    1: 0.98,   # hdpe

    2: 0.98,   # mixed_plastic_rigid

    3: 0.94,   # mixed_plastic_soft

    4: 0.94,   # non_plastic

    5: 0.99,   # pet

    6: 0.99,   # pet_oil
}


# ============================================================
# 9. CONVNEXT HARD-CLASS MAPPING
#
# E18-B ImageFolder order:
#
# 0 mixed_plastic_rigid -> global 2
# 1 mixed_plastic_soft  -> global 3
# 2 non_plastic         -> global 4
# 3 pet_oil             -> global 6
# ============================================================

CONVNEXT_TO_GLOBAL = {

    0: 2,

    1: 3,

    2: 4,

    3: 6,
}


GLOBAL_TO_CONVNEXT = {

    2: 0,

    3: 1,

    4: 2,

    6: 3,
}


HARD_GLOBAL_CLASSES = set(
    GLOBAL_TO_CONVNEXT.keys()
)


# ============================================================
# 10. CONVNEXT SPECIALIST GATES
#
# For this first controlled E18-C experiment, use the same
# hard-class thresholds already established in E16-A.
#
# This avoids introducing another threshold-search experiment.
#
# IMPORTANT:
# These thresholds are indexed by CONVNEXT PREDICTED class
# after mapping it into the 7-class taxonomy.
# ============================================================

CONVNEXT_GATES = {

    2: 0.98,  # mixed rigid

    3: 0.94,  # mixed soft

    4: 0.94,  # nonplastic

    6: 0.99,  # pet oil
}


# ============================================================
# 11. IMAGE TRANSFORM
#
# Same deterministic validation preprocessing used for
# MobileNet and E18-B ConvNeXt.
# ============================================================

IMAGENET_MEAN = [
    0.485,
    0.456,
    0.406,
]

IMAGENET_STD = [
    0.229,
    0.224,
    0.225,
]


VAL_TRANSFORM = v2.Compose(
    [
        v2.Resize(
            (
                CLASSIFIER_SIZE,
                CLASSIFIER_SIZE
            )
        ),

        v2.ToImage(),

        v2.ToDtype(
            torch.float32,
            scale=True
        ),

        v2.Normalize(
            mean=IMAGENET_MEAN,
            std=IMAGENET_STD
        ),
    ]
)


# ============================================================
# 12. HELPER — EXTRACT STATE DICT
# ============================================================

def extract_state_dict(checkpoint):

    if not isinstance(
        checkpoint,
        dict
    ):
        return checkpoint


    for key in [

        "model_state_dict",

        "state_dict",

        "model",

    ]:

        if key in checkpoint:

            value = checkpoint[key]

            if isinstance(
                value,
                dict
            ):

                return value


    # May already be raw state_dict
    return checkpoint


# ============================================================
# 13. LOAD MOBILENET
# ============================================================

print(
    "\nLoading E3Y-B MobileNet..."
)


mobilenet = mobilenet_v3_large(
    weights=None
)


mobilenet.classifier[3] = nn.Linear(
    mobilenet.classifier[3].in_features,
    7
)


mobile_checkpoint = torch.load(
    MOBILENET_PATH,
    map_location=DEVICE,
    weights_only=False
)


mobile_state = extract_state_dict(
    mobile_checkpoint
)


mobilenet.load_state_dict(
    mobile_state
)


mobilenet = mobilenet.to(
    DEVICE
)

mobilenet.eval()


print(
    "MobileNet loaded."
)


# ============================================================
# 14. LOAD CONVNEXT-TINY
# ============================================================

print(
    "\nLoading E18-B ConvNeXt-Tiny..."
)


convnext = convnext_tiny(
    weights=None
)


convnext.classifier[2] = nn.Linear(
    convnext.classifier[2].in_features,
    4
)


conv_checkpoint = torch.load(
    CONVNEXT_PATH,
    map_location=DEVICE,
    weights_only=False
)


conv_state = extract_state_dict(
    conv_checkpoint
)


convnext.load_state_dict(
    conv_state
)


convnext = convnext.to(
    DEVICE
)

convnext.eval()


print(
    "ConvNeXt-Tiny loaded."
)


# ============================================================
# 15. LOAD YOLO11m
# ============================================================

print(
    "\nLoading E12 YOLO11m..."
)


yolo = YOLO(
    str(
        YOLO_MODEL_PATH
    )
)


print(
    "YOLO11m loaded."
)


# ============================================================
# 16. SAFE DETECTOR CROP
# ============================================================

def crop_box(
    image,
    bbox
):

    width, height = image.size

    x1, y1, x2, y2 = bbox


    x1 = max(
        0,
        min(
            int(round(x1)),
            width - 1
        )
    )

    y1 = max(
        0,
        min(
            int(round(y1)),
            height - 1
        )
    )

    x2 = max(
        0,
        min(
            int(round(x2)),
            width
        )
    )

    y2 = max(
        0,
        min(
            int(round(y2)),
            height
        )
    )


    if x2 <= x1 or y2 <= y1:

        return None


    return image.crop(
        (
            x1,
            y1,
            x2,
            y2
        )
    )


# ============================================================
# 17. CREATE 7-CLASS COCO GT
# ============================================================

def build_7class_ground_truth():

    with open(
        VAL_COCO,
        "r",
        encoding="utf-8"
    ) as f:

        original = json.load(f)


    gt = {

        "info":
            original.get(
                "info",
                {}
            ),

        "licenses":
            original.get(
                "licenses",
                []
            ),

        "images":
            original["images"],

        "categories": [
            {
                "id":
                    idx + 1,

                "name":
                    name,

                "supercategory":
                    "waste"
            }

            for idx, name in enumerate(
                CLASS_NAMES
            )
        ],

        "annotations":
            []
    }


    new_ann_id = 1


    for ann in original[
        "annotations"
    ]:

        original_cat = int(
            ann[
                "category_id"
            ]
        )

        global_idx = (
            COCO_TO_7CLASS[
                original_cat
            ]
        )


        new_ann = dict(
            ann
        )


        new_ann[
            "id"
        ] = new_ann_id


        new_ann[
            "category_id"
        ] = global_idx + 1


        # COCOeval requires these
        new_ann[
            "iscrowd"
        ] = int(
            ann.get(
                "iscrowd",
                0
            )
        )


        if "area" not in new_ann:

            _, _, w, h = (
                new_ann["bbox"]
            )

            new_ann[
                "area"
            ] = float(
                w * h
            )


        gt[
            "annotations"
        ].append(
            new_ann
        )


        new_ann_id += 1


    with open(
        GT_7CLASS_PATH,
        "w",
        encoding="utf-8"
    ) as f:

        json.dump(
            gt,
            f
        )


    print(
        "\n7-class GT created:"
    )

    print(
        GT_7CLASS_PATH
    )


build_7class_ground_truth()


# ============================================================
# 18. LOAD VALIDATION IMAGE METADATA
# ============================================================

with open(
    VAL_COCO,
    "r",
    encoding="utf-8"
) as f:

    coco_source = json.load(f)


image_records = sorted(
    coco_source["images"],
    key=lambda x: int(
        x["id"]
    )
)


print(
    f"\nValidation images : "
    f"{len(image_records)}"
)


# ============================================================
# 19. CLASSIFY ONE BATCH
# ============================================================

@torch.no_grad()
def classify_batch(
    model,
    tensors
):

    if len(tensors) == 0:

        return np.empty(
            (
                0,
                0
            )
        )


    batch = torch.stack(
        tensors
    ).to(
        DEVICE
    )


    with torch.amp.autocast(
        device_type="cuda",
        enabled=(
            DEVICE.type
            == "cuda"
        )
    ):

        logits = model(
            batch
        )


        probabilities = torch.softmax(
            logits,
            dim=1
        )


    return (
        probabilities
        .detach()
        .cpu()
        .numpy()
    )


# ============================================================
# 20. INFERENCE
# ============================================================

e16_predictions = []

e18_predictions = []


total_yolo_detections = 0

invalid_crops = 0

convnext_candidate_count = 0

convnext_accept_count = 0

convnext_reject_count = 0


e16_source_counts = Counter()

e18_source_counts = Counter()

convnext_predicted_counts = Counter()


start_time = time.time()


for image_number, image_info in enumerate(
    image_records,
    start=1
):

    image_id = int(
        image_info[
            "id"
        ]
    )

    image_path = (
        VAL_IMAGES
        / image_info[
            "file_name"
        ]
    )


    if image_number % 25 == 0:

        print(
            f"Processing "
            f"{image_number}/{len(image_records)}"
        )


    # --------------------------------------------------------
    # YOLO inference
    # --------------------------------------------------------

    result = yolo.predict(

        source=str(
            image_path
        ),

        imgsz=YOLO_IMAGE_SIZE,

        conf=YOLO_CONF,

        iou=YOLO_NMS,

        max_det=YOLO_MAX_DET,

        device=0
        if DEVICE.type == "cuda"
        else "cpu",

        verbose=False,

    )[0]


    if (
        result.boxes is None
        or len(result.boxes) == 0
    ):

        continue


    boxes = (
        result.boxes.xyxy
        .detach()
        .cpu()
        .numpy()
    )

    yolo_confs = (
        result.boxes.conf
        .detach()
        .cpu()
        .numpy()
    )

    yolo_classes = (
        result.boxes.cls
        .detach()
        .cpu()
        .numpy()
        .astype(int)
    )


    total_yolo_detections += len(
        boxes
    )


    # --------------------------------------------------------
    # Build valid crops
    # --------------------------------------------------------

    image = Image.open(
        image_path
    ).convert(
        "RGB"
    )


    valid_records = []

    crop_tensors = []


    for bbox, y_conf, y_cls in zip(
        boxes,
        yolo_confs,
        yolo_classes
    ):

        crop = crop_box(
            image,
            bbox
        )


        if crop is None:

            invalid_crops += 1

            continue


        crop_tensor = VAL_TRANSFORM(
            crop
        )


        valid_records.append(
            {
                "bbox_xyxy":
                    [
                        float(v)
                        for v in bbox
                    ],

                "yolo_conf":
                    float(
                        y_conf
                    ),

                "yolo_class":
                    int(
                        y_cls
                    ),
            }
        )


        crop_tensors.append(
            crop_tensor
        )


    image.close()


    if len(
        valid_records
    ) == 0:

        continue


    # --------------------------------------------------------
    # MobileNet — all valid YOLO crops
    # --------------------------------------------------------

    mobile_probs = classify_batch(
        mobilenet,
        crop_tensors
    )


    # ========================================================
    # STEP A — REPRODUCE E16-A DECISIONS
    # ========================================================

    e16_temp = []


    for record, mn_probs in zip(
        valid_records,
        mobile_probs
    ):

        yolo_idx = int(
            record[
                "yolo_class"
            ]
        )

        yolo_conf = float(
            record[
                "yolo_conf"
            ]
        )


        mn_idx = int(
            np.argmax(
                mn_probs
            )
        )

        mn_top_prob = float(
            mn_probs[
                mn_idx
            ]
        )


        mn_gate = (
            E16A_GATES[
                mn_idx
            ]
        )


        # ----------------------------------------------------
        # E16-A class gating
        # ----------------------------------------------------

        if (
            mn_top_prob
            >= mn_gate
        ):

            e16_final_idx = mn_idx

            e16_source = (
                "mobilenet"
            )

        else:

            e16_final_idx = yolo_idx

            e16_source = (
                "yolo"
            )


        e16_source_counts[
            e16_source
        ] += 1


        # ----------------------------------------------------
        # Correct E16-A score:
        # MobileNet probability for ACTUAL final class.
        # ----------------------------------------------------

        mn_prob_final = float(
            mn_probs[
                e16_final_idx
            ]
        )


        e16_score = (
            yolo_conf ** ALPHA
        ) * (
            max(
                mn_prob_final,
                1e-12
            )
            ** (
                1.0
                - ALPHA
            )
        )


        bbox = record[
            "bbox_xyxy"
        ]

        x1, y1, x2, y2 = bbox


        coco_bbox = [

            float(
                x1
            ),

            float(
                y1
            ),

            float(
                x2 - x1
            ),

            float(
                y2 - y1
            ),
        ]


        e16_predictions.append(
            {
                "image_id":
                    image_id,

                "category_id":
                    e16_final_idx + 1,

                "bbox":
                    coco_bbox,

                "score":
                    float(
                        e16_score
                    ),
            }
        )


        e16_temp.append(
            {
                "record":
                    record,

                "mn_probs":
                    mn_probs,

                "e16_final_idx":
                    e16_final_idx,

                "e16_score":
                    e16_score,

                "e16_source":
                    e16_source,
            }
        )


    # ========================================================
    # STEP B — IDENTIFY HARD-CLASS E16-A PREDICTIONS
    #
    # ConvNeXt is ONLY consulted when the E16-A final class
    # is already one of the four hard classes.
    # ========================================================

    specialist_positions = []

    specialist_tensors = []


    for position, item in enumerate(
        e16_temp
    ):

        if (
            item[
                "e16_final_idx"
            ]
            in HARD_GLOBAL_CLASSES
        ):

            specialist_positions.append(
                position
            )

            specialist_tensors.append(
                crop_tensors[
                    position
                ]
            )


    convnext_candidate_count += len(
        specialist_positions
    )


    # --------------------------------------------------------
    # Run ConvNeXt only on specialist candidates
    # --------------------------------------------------------

    if len(
        specialist_tensors
    ) > 0:

        conv_probs_batch = classify_batch(
            convnext,
            specialist_tensors
        )

    else:

        conv_probs_batch = []


    conv_result_by_position = {}


    for position, conv_probs in zip(
        specialist_positions,
        conv_probs_batch
    ):

        local_idx = int(
            np.argmax(
                conv_probs
            )
        )


        global_idx = (
            CONVNEXT_TO_GLOBAL[
                local_idx
            ]
        )


        top_probability = float(
            conv_probs[
                local_idx
            ]
        )


        conv_result_by_position[
            position
        ] = {

            "local_idx":
                local_idx,

            "global_idx":
                global_idx,

            "top_probability":
                top_probability,

            "probs":
                conv_probs,
        }


        convnext_predicted_counts[
            CLASS_NAMES[
                global_idx
            ]
        ] += 1


    # ========================================================
    # STEP C — E18-C SPECIALIST DECISION
    # ========================================================

    for position, item in enumerate(
        e16_temp
    ):

        record = item[
            "record"
        ]

        yolo_conf = float(
            record[
                "yolo_conf"
            ]
        )

        mn_probs = item[
            "mn_probs"
        ]


        # Start from E16-A decision
        final_idx = int(
            item[
                "e16_final_idx"
            ]
        )

        final_source = (
            item[
                "e16_source"
            ]
        )


        classifier_probability = float(
            mn_probs[
                final_idx
            ]
        )


        # ----------------------------------------------------
        # ConvNeXt specialist may refine ONLY hard E16 classes
        # ----------------------------------------------------

        if position in conv_result_by_position:

            conv_result = (
                conv_result_by_position[
                    position
                ]
            )


            conv_global_idx = int(
                conv_result[
                    "global_idx"
                ]
            )

            conv_top_prob = float(
                conv_result[
                    "top_probability"
                ]
            )


            specialist_gate = (
                CONVNEXT_GATES[
                    conv_global_idx
                ]
            )


            # ------------------------------------------------
            # Specialist accepted
            # ------------------------------------------------

            if (
                conv_top_prob
                >= specialist_gate
            ):

                final_idx = (
                    conv_global_idx
                )

                final_source = (
                    "convnext"
                )

                classifier_probability = (
                    conv_top_prob
                )


                convnext_accept_count += 1


            # ------------------------------------------------
            # Specialist rejected:
            # preserve the E16-A class AND E16-A MobileNet
            # confidence term.
            # ------------------------------------------------

            else:

                convnext_reject_count += 1


        e18_source_counts[
            final_source
        ] += 1


        # ----------------------------------------------------
        # E18 confidence fusion
        # ----------------------------------------------------

        final_score = (
            yolo_conf ** ALPHA
        ) * (
            max(
                classifier_probability,
                1e-12
            )
            ** (
                1.0
                - ALPHA
            )
        )


        bbox = record[
            "bbox_xyxy"
        ]

        x1, y1, x2, y2 = bbox


        coco_bbox = [

            float(
                x1
            ),

            float(
                y1
            ),

            float(
                x2 - x1
            ),

            float(
                y2 - y1
            ),
        ]


        e18_predictions.append(
            {
                "image_id":
                    image_id,

                "category_id":
                    final_idx + 1,

                "bbox":
                    coco_bbox,

                "score":
                    float(
                        final_score
                    ),
            }
        )


# ============================================================
# 21. SAVE PREDICTIONS
# ============================================================

with open(
    E16_PRED_PATH,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        e16_predictions,
        f
    )


with open(
    E18_PRED_PATH,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        e18_predictions,
        f
    )


inference_minutes = (
    time.time()
    - start_time
) / 60


print(
    "\nInference complete."
)

print(
    f"YOLO detections       : "
    f"{total_yolo_detections:,}"
)

print(
    f"Invalid crops         : "
    f"{invalid_crops:,}"
)

print(
    f"E16 predictions       : "
    f"{len(e16_predictions):,}"
)

print(
    f"E18 predictions       : "
    f"{len(e18_predictions):,}"
)

print(
    f"ConvNeXt candidates   : "
    f"{convnext_candidate_count:,}"
)

print(
    f"ConvNeXt accepted     : "
    f"{convnext_accept_count:,}"
)

print(
    f"ConvNeXt rejected     : "
    f"{convnext_reject_count:,}"
)

print(
    f"Inference time        : "
    f"{inference_minutes:.2f} min"
)


# ============================================================
# 22. COCO EVALUATION
# ============================================================

def run_coco_eval(
    prediction_path,
    experiment_name
):

    print(
        "\n"
        + "=" * 100
    )

    print(
        experiment_name
    )

    print(
        "=" * 100
    )


    coco_gt = COCO(
        str(
            GT_7CLASS_PATH
        )
    )


    with open(
        prediction_path,
        "r",
        encoding="utf-8"
    ) as f:

        predictions = json.load(f)


    coco_dt = coco_gt.loadRes(
        predictions
    )


    evaluator = COCOeval(
        coco_gt,
        coco_dt,
        "bbox"
    )


    evaluator.params.maxDets = [
        1,
        10,
        100
    ]


    evaluator.evaluate()

    evaluator.accumulate()

    evaluator.summarize()


    results = {

        "AP50_95":
            float(
                evaluator.stats[0]
            ),

        "AP50":
            float(
                evaluator.stats[1]
            ),

        "AP75":
            float(
                evaluator.stats[2]
            ),

        "AR1":
            float(
                evaluator.stats[6]
            ),

        "AR10":
            float(
                evaluator.stats[7]
            ),

        "AR100":
            float(
                evaluator.stats[8]
            ),
    }


    return (
        evaluator,
        results
    )


# ============================================================
# 23. E16-A BASELINE
# ============================================================

e16_eval, e16_metrics = run_coco_eval(

    E16_PRED_PATH,

    "E16-A BASELINE REPRODUCTION"
)


# ============================================================
# 24. E18-C
# ============================================================

e18_eval, e18_metrics = run_coco_eval(

    E18_PRED_PATH,

    "E18-C — CONVNEXT HARD-CLASS REFINEMENT"
)


# ============================================================
# 25. PER-CLASS AP FUNCTION
# ============================================================

def get_per_class_metrics(
    evaluator
):

    precision = (
        evaluator.eval[
            "precision"
        ]
    )

    iou_thresholds = (
        evaluator.params.iouThrs
    )


    ap50_index = int(
        np.where(
            np.isclose(
                iou_thresholds,
                0.50
            )
        )[0][0]
    )


    results = {}


    for class_idx, class_name in enumerate(
        CLASS_NAMES
    ):

        # ----------------------------------------------------
        # AP50:95
        #
        # dimensions:
        # [IoU, Recall, Class, Area, MaxDet]
        # ----------------------------------------------------

        values = precision[
            :,
            :,
            class_idx,
            0,
            -1
        ]


        values = values[
            values > -1
        ]


        if len(
            values
        ) > 0:

            ap = float(
                np.mean(
                    values
                )
            )

        else:

            ap = float(
                "nan"
            )


        # ----------------------------------------------------
        # AP50
        # ----------------------------------------------------

        values50 = precision[
            ap50_index,
            :,
            class_idx,
            0,
            -1
        ]


        values50 = values50[
            values50 > -1
        ]


        if len(
            values50
        ) > 0:

            ap50 = float(
                np.mean(
                    values50
                )
            )

        else:

            ap50 = float(
                "nan"
            )


        results[
            class_name
        ] = {

            "AP50_95":
                ap,

            "AP50":
                ap50
        }


    return results


# ============================================================
# 26. CLASS-WISE METRICS
# ============================================================

e16_class_metrics = get_per_class_metrics(
    e16_eval
)

e18_class_metrics = get_per_class_metrics(
    e18_eval
)


print(
    "\n"
    + "=" * 100
)

print(
    "CLASS-WISE COMPARISON"
)

print(
    "=" * 100
)


print(
    f"\n{'Class':25s}"
    f"{'E16 AP':>12s}"
    f"{'E18 AP':>12s}"
    f"{'Delta':>12s}"
    f"{'E16 AP50':>12s}"
    f"{'E18 AP50':>12s}"
)


for class_name in CLASS_NAMES:

    old_ap = (
        e16_class_metrics[
            class_name
        ][
            "AP50_95"
        ]
    )

    new_ap = (
        e18_class_metrics[
            class_name
        ][
            "AP50_95"
        ]
    )

    old_ap50 = (
        e16_class_metrics[
            class_name
        ][
            "AP50"
        ]
    )

    new_ap50 = (
        e18_class_metrics[
            class_name
        ][
            "AP50"
        ]
    )


    print(

        f"{class_name:25s}"

        f"{old_ap * 100:12.2f}"

        f"{new_ap * 100:12.2f}"

        f"{(new_ap-old_ap) * 100:12.2f}"

        f"{old_ap50 * 100:12.2f}"

        f"{new_ap50 * 100:12.2f}"
    )


# ============================================================
# 27. OVERALL COMPARISON
# ============================================================

print(
    "\n"
    + "=" * 100
)

print(
    "OVERALL RESULT"
)

print(
    "=" * 100
)


for metric in [

    "AP50_95",

    "AP50",

    "AP75",

    "AR100",

]:

    old_value = (
        e16_metrics[
            metric
        ]
    )

    new_value = (
        e18_metrics[
            metric
        ]
    )

    delta = (
        new_value
        - old_value
    )


    print(
        f"{metric:10s} | "
        f"E16-A = "
        f"{old_value * 100:7.3f}% | "
        f"E18-C = "
        f"{new_value * 100:7.3f}% | "
        f"Delta = "
        f"{delta * 100:+7.3f} pp"
    )


# ============================================================
# 28. SOURCE COUNTS
# ============================================================

print(
    "\n"
    + "=" * 100
)

print(
    "FINAL CLASS SOURCE COUNTS"
)

print(
    "=" * 100
)


print(
    "\nE16-A:"
)

for key, value in (
    e16_source_counts.items()
):

    print(
        f"{key:15s}: "
        f"{value:,}"
    )


print(
    "\nE18-C:"
)

for key, value in (
    e18_source_counts.items()
):

    print(
        f"{key:15s}: "
        f"{value:,}"
    )


print(
    "\nConvNeXt predicted hard classes:"
)

for class_name in [
    "mixed_plastic_rigid",
    "mixed_plastic_soft",
    "non_plastic",
    "pet_oil",
]:

    print(
        f"{class_name:25s}: "
        f"{convnext_predicted_counts[class_name]:,}"
    )


# ============================================================
# 29. SAVE SUMMARY
# ============================================================

summary = {

    "experiment":
        "E18-C",

    "title":
        (
            "YOLO11m + ConvNeXt-Tiny "
            "Hard-Class Refinement with "
            "E16-A Gating and Confidence Fusion"
        ),

    "alpha":
        ALPHA,

    "mobilenet_gates":
        {
            CLASS_NAMES[k]:
                v

            for k, v in (
                E16A_GATES.items()
            )
        },

    "convnext_gates":
        {
            CLASS_NAMES[k]:
                v

            for k, v in (
                CONVNEXT_GATES.items()
            )
        },

    "total_yolo_detections":
        total_yolo_detections,

    "invalid_crops":
        invalid_crops,

    "convnext_candidates":
        convnext_candidate_count,

    "convnext_accepted":
        convnext_accept_count,

    "convnext_rejected":
        convnext_reject_count,

    "e16_source_counts":
        dict(
            e16_source_counts
        ),

    "e18_source_counts":
        dict(
            e18_source_counts
        ),

    "e16_metrics":
        e16_metrics,

    "e18_metrics":
        e18_metrics,

    "e16_class_metrics":
        e16_class_metrics,

    "e18_class_metrics":
        e18_class_metrics,

    "delta_AP50_95":
        (
            e18_metrics[
                "AP50_95"
            ]
            -
            e16_metrics[
                "AP50_95"
            ]
        ),

    "delta_AP50":
        (
            e18_metrics[
                "AP50"
            ]
            -
            e16_metrics[
                "AP50"
            ]
        ),

    "delta_AP75":
        (
            e18_metrics[
                "AP75"
            ]
            -
            e16_metrics[
                "AP75"
            ]
        ),

    "inference_minutes":
        inference_minutes,
}


with open(
    SUMMARY_PATH,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        summary,
        f,
        indent=2
    )


# ============================================================
# 30. FINAL MESSAGE
# ============================================================

print(
    "\n"
    + "=" * 100
)

print(
    "E18-C COMPLETE"
)

print(
    "=" * 100
)


print(
    "\nE16-A AP50-95 : "
    f"{e16_metrics['AP50_95'] * 100:.4f}%"
)

print(
    "E18-C AP50-95 : "
    f"{e18_metrics['AP50_95'] * 100:.4f}%"
)

print(
    "Improvement   : "
    f"{( e18_metrics['AP50_95'] - e16_metrics['AP50_95'] ) * 100:+.4f} pp"
)


print(
    "\nSummary:"
)

print(
    SUMMARY_PATH
)

E18-C — YOLO11m + ConvNeXt-Tiny Hard-Class Refinement

Device : cuda
GPU    : NVIDIA GeForce RTX 3050 Ti Laptop GPU

Loading E3Y-B MobileNet...
MobileNet loaded.

Loading E18-B ConvNeXt-Tiny...
ConvNeXt-Tiny loaded.

Loading E12 YOLO11m...
YOLO11m loaded.

7-class GT created:
C:\Users\varda\Documents\_My Computer\COMMON_space\Learning\Upgrad_EPGP_MS\upgrad_IIITB_MS_course\MS_LJMU_Material\Thesis_Code\runs\sortwaste\E18C_yolo11m_convnext_hardclass\E18C_val_gt_7class.json

Validation images : 780
Processing 25/780
Processing 50/780
Processing 75/780
Processing 100/780
Processing 125/780
Processing 150/780
Processing 175/780
Processing 200/780
Processing 225/780
Processing 250/780
Processing 275/780
Processing 300/780
Processing 325/780
Processing 350/780
Processing 375/780
Processing 400/780
Processing 425/780
Processing 450/780
Processing 475/780
Processing 500/780
Processing 525/780
Processing 550/780
Processing 575/780
Processing 600/780
Processing 625/780
Processing 650/780
Processin

## E18-D — Class-Selective ConvNeXt Specialist Routing Ablation

In [ ]:
# E18-D — Class-Selective ConvNeXt Specialist Routing Ablation
#
# Purpose:
#   Determine whether restricting ConvNeXt intervention to
#   only the classes that benefited in E18-C improves
#   end-to-end COCO AP.
#
# No training.
#
# Detector:
#   E12 YOLO11m @640
#
# Base 7-class classifier:
#   E3Y-B MobileNetV3-Large
#
# Specialist:
#   E18-B ConvNeXt-Tiny
#
# E16-A MobileNet gates retained.
# Alpha retained at 0.70.
#
# Variants:
#
#   E18-C baseline:
#       mixed_rigid
#       mixed_soft
#       non_plastic
#       pet_oil
#
#   E18-D1:
#       non_plastic
#       pet_oil
#
#   E18-D2:
#       mixed_soft
#       non_plastic
#       pet_oil
#
# Evaluation:
#   explicit pycocotools.COCOeval
# ============================================================

import json
import random
import time
from pathlib import Path
from collections import Counter

import numpy as np
from PIL import Image

import torch
import torch.nn as nn

from torchvision.transforms import v2
from torchvision.models import (
    mobilenet_v3_large,
    convnext_tiny,
)

from ultralytics import YOLO

from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval


# ============================================================
# 1. REPRODUCIBILITY
# ============================================================

SEED = 42

random.seed(SEED)
np.random.seed(SEED)

torch.manual_seed(SEED)
torch.cuda.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False


# ============================================================
# 2. PATHS
# ============================================================

BASE = Path(
    r"C:\Users\varda\Documents\_My Computer\COMMON_space\Learning"
    r"\Upgrad_EPGP_MS\upgrad_IIITB_MS_course\MS_LJMU_Material"
)

THESIS_CODE = BASE / "Thesis_Code"

DATASET_ROOT = (
    BASE
    / "Topic Data"
    / "SortWaste"
    / "dataset"
    / "dataset"
)

COCO_ROOT = (
    DATASET_ROOT
    / "splited_all_dataset_coco"
)

VAL_IMAGES = (
    COCO_ROOT
    / "val"
    / "images"
)

VAL_COCO = (
    COCO_ROOT
    / "val"
    / "annotations"
    / "val_coco.json"
)


# ------------------------------------------------------------
# E12 YOLO11m
# ------------------------------------------------------------

YOLO_MODEL_PATH = (
    THESIS_CODE
    / "runs"
    / "sortwaste"
    / "E12_yolo11m_7class_aug_classbalance_640"
    / "weights"
    / "best.pt"
)


# ------------------------------------------------------------
# E3Y-B MobileNet
# ------------------------------------------------------------

MOBILENET_PATH = (
    DATASET_ROOT
    / "yolo_mobilenet_crops_E3Y"
    / "mobilenet_results"
    / "E3Y_B_class_weighted"
    / "E3Y_B_MobileNetV3Large_best.pth"
)


# ------------------------------------------------------------
# E18-B ConvNeXt
# ------------------------------------------------------------

CONVNEXT_PATH = (
    THESIS_CODE
    / "runs"
    / "sortwaste"
    / "E18B_convnext_tiny_hardclass"
    / "E18B_ConvNeXtTiny_best.pth"
)


# ------------------------------------------------------------
# Output
# ------------------------------------------------------------

OUTPUT_DIR = (
    THESIS_CODE
    / "runs"
    / "sortwaste"
    / "E18D_class_selective_convnext"
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)


GT_PATH = (
    OUTPUT_DIR
    / "E18D_val_gt_7class.json"
)

CACHE_PATH = (
    OUTPUT_DIR
    / "E18D_cached_predictions.json"
)

RESULTS_PATH = (
    OUTPUT_DIR
    / "E18D_results.json"
)

SUMMARY_PATH = (
    OUTPUT_DIR
    / "E18D_summary.txt"
)


# ============================================================
# 3. VALIDATE PATHS
# ============================================================

for path, label in [
    (VAL_IMAGES, "Validation images"),
    (VAL_COCO, "Validation COCO"),
    (YOLO_MODEL_PATH, "YOLO11m checkpoint"),
    (MOBILENET_PATH, "MobileNet checkpoint"),
    (CONVNEXT_PATH, "ConvNeXt checkpoint"),
]:

    assert path.exists(), (
        f"{label} not found:\n{path}"
    )


# ============================================================
# 4. DEVICE
# ============================================================

DEVICE = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

print("=" * 100)
print("E18-D — CLASS-SELECTIVE CONVNEXT SPECIALIST ROUTING ABLATION")
print("=" * 100)

print(f"\nDevice : {DEVICE}")

if torch.cuda.is_available():

    print(
        f"GPU    : "
        f"{torch.cuda.get_device_name(0)}"
    )


# ============================================================
# 5. SETTINGS
# ============================================================

YOLO_IMAGE_SIZE = 640

YOLO_CONF = 0.001

YOLO_NMS = 0.60

YOLO_MAX_DET = 100

CLASSIFIER_SIZE = 224

ALPHA = 0.70


# ============================================================
# 6. 7-CLASS TAXONOMY
# ============================================================

CLASS_NAMES = [
    "ecal",
    "hdpe",
    "mixed_plastic_rigid",
    "mixed_plastic_soft",
    "non_plastic",
    "pet",
    "pet_oil",
]


COCO_TO_7CLASS = {
    1: 5,  # pet
    2: 1,  # hdpe
    3: 3,  # mixed soft
    4: 0,  # ecal
    5: 4,  # metal -> nonplastic
    6: 4,  # cardboard -> nonplastic
    7: 2,  # mixed rigid
    8: 6,  # pet oil
}


# ============================================================
# 7. E16-A MOBILENET GATES
# ============================================================

E16A_GATES = {
    0: 0.99,  # ecal
    1: 0.98,  # hdpe
    2: 0.98,  # mixed rigid
    3: 0.94,  # mixed soft
    4: 0.94,  # nonplastic
    5: 0.99,  # pet
    6: 0.99,  # pet oil
}


# ============================================================
# 8. CONVNEXT CLASS MAPPING
#
# E18-B local indices:
#
# 0 = mixed rigid -> global 2
# 1 = mixed soft  -> global 3
# 2 = nonplastic  -> global 4
# 3 = pet oil     -> global 6
# ============================================================

CONVNEXT_TO_GLOBAL = {
    0: 2,
    1: 3,
    2: 4,
    3: 6,
}


# ============================================================
# 9. CONVNEXT CLASS-SPECIFIC GATES
# ============================================================

CONVNEXT_GATES = {
    2: 0.98,  # mixed rigid
    3: 0.94,  # mixed soft
    4: 0.94,  # nonplastic
    6: 0.99,  # pet oil
}


# ============================================================
# 10. ROUTING VARIANTS
# ============================================================

ROUTING_VARIANTS = {

    # Reproduces E18-C
    "E18-C_all_hard": {
        2,  # mixed rigid
        3,  # mixed soft
        4,  # nonplastic
        6,  # pet oil
    },

    # Best evidence-driven restricted specialist
    "E18-D1_nonplastic_petoil": {
        4,
        6,
    },

    # Keeps mixed soft because it was neutral/slightly positive
    "E18-D2_mixedsoft_nonplastic_petoil": {
        3,
        4,
        6,
    },
}


# ============================================================
# 11. VALIDATION TRANSFORM
# ============================================================

IMAGENET_MEAN = [
    0.485,
    0.456,
    0.406,
]

IMAGENET_STD = [
    0.229,
    0.224,
    0.225,
]


VAL_TRANSFORM = v2.Compose(
    [
        v2.Resize(
            (
                CLASSIFIER_SIZE,
                CLASSIFIER_SIZE
            )
        ),

        v2.ToImage(),

        v2.ToDtype(
            torch.float32,
            scale=True
        ),

        v2.Normalize(
            mean=IMAGENET_MEAN,
            std=IMAGENET_STD
        ),
    ]
)


# ============================================================
# 12. HELPER — STATE DICT
# ============================================================

def extract_state_dict(checkpoint):

    if not isinstance(
        checkpoint,
        dict
    ):
        return checkpoint

    for key in [
        "model_state_dict",
        "state_dict",
        "model",
    ]:

        if key in checkpoint:

            value = checkpoint[key]

            if isinstance(
                value,
                dict
            ):
                return value

    return checkpoint


# ============================================================
# 13. LOAD MOBILENET
# ============================================================

print("\nLoading MobileNet...")

mobilenet = mobilenet_v3_large(
    weights=None
)

mobilenet.classifier[3] = nn.Linear(
    mobilenet.classifier[3].in_features,
    7
)

checkpoint = torch.load(
    MOBILENET_PATH,
    map_location=DEVICE,
    weights_only=False
)

mobilenet.load_state_dict(
    extract_state_dict(
        checkpoint
    )
)

mobilenet = mobilenet.to(
    DEVICE
)

mobilenet.eval()

print("MobileNet loaded.")


# ============================================================
# 14. LOAD CONVNEXT
# ============================================================

print("\nLoading ConvNeXt...")

convnext = convnext_tiny(
    weights=None
)

convnext.classifier[2] = nn.Linear(
    convnext.classifier[2].in_features,
    4
)

checkpoint = torch.load(
    CONVNEXT_PATH,
    map_location=DEVICE,
    weights_only=False
)

convnext.load_state_dict(
    extract_state_dict(
        checkpoint
    )
)

convnext = convnext.to(
    DEVICE
)

convnext.eval()

print("ConvNeXt loaded.")


# ============================================================
# 15. LOAD YOLO
# ============================================================

print("\nLoading YOLO11m...")

yolo = YOLO(
    str(
        YOLO_MODEL_PATH
    )
)

print("YOLO11m loaded.")


# ============================================================
# 16. SAFE CROP
# ============================================================

def crop_box(
    image,
    bbox
):

    width, height = image.size

    x1, y1, x2, y2 = bbox

    x1 = max(
        0,
        min(
            int(round(x1)),
            width - 1
        )
    )

    y1 = max(
        0,
        min(
            int(round(y1)),
            height - 1
        )
    )

    x2 = max(
        0,
        min(
            int(round(x2)),
            width
        )
    )

    y2 = max(
        0,
        min(
            int(round(y2)),
            height
        )
    )

    if (
        x2 <= x1
        or y2 <= y1
    ):

        return None

    return image.crop(
        (
            x1,
            y1,
            x2,
            y2
        )
    )


# ============================================================
# 17. CLASSIFIER BATCH
# ============================================================

@torch.no_grad()
def classify_batch(
    model,
    tensors
):

    if len(tensors) == 0:

        return np.empty(
            (
                0,
                0
            )
        )

    batch = torch.stack(
        tensors
    ).to(
        DEVICE
    )

    with torch.amp.autocast(
        device_type="cuda",
        enabled=(
            DEVICE.type
            == "cuda"
        )
    ):

        logits = model(
            batch
        )

        probs = torch.softmax(
            logits,
            dim=1
        )

    return (
        probs
        .detach()
        .cpu()
        .numpy()
    )


# ============================================================
# 18. BUILD 7-CLASS GT
# ============================================================

with open(
    VAL_COCO,
    "r",
    encoding="utf-8"
) as f:

    original = json.load(f)


gt = {

    "info":
        original.get(
            "info",
            {}
        ),

    "licenses":
        original.get(
            "licenses",
            []
        ),

    "images":
        original["images"],

    "categories": [
        {
            "id":
                idx + 1,

            "name":
                name,

            "supercategory":
                "waste"
        }

        for idx, name in enumerate(
            CLASS_NAMES
        )
    ],

    "annotations":
        []
}


new_ann_id = 1


for ann in original[
    "annotations"
]:

    global_idx = (
        COCO_TO_7CLASS[
            int(
                ann[
                    "category_id"
                ]
            )
        ]
    )

    new_ann = dict(
        ann
    )

    new_ann["id"] = (
        new_ann_id
    )

    new_ann[
        "category_id"
    ] = global_idx + 1

    new_ann[
        "iscrowd"
    ] = int(
        ann.get(
            "iscrowd",
            0
        )
    )

    if "area" not in new_ann:

        _, _, w, h = (
            new_ann["bbox"]
        )

        new_ann[
            "area"
        ] = float(
            w * h
        )

    gt[
        "annotations"
    ].append(
        new_ann
    )

    new_ann_id += 1


with open(
    GT_PATH,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        gt,
        f
    )


# ============================================================
# 19. IMAGE RECORDS
# ============================================================

image_records = sorted(
    original[
        "images"
    ],
    key=lambda x: int(
        x["id"]
    )
)


# ============================================================
# 20. GENERATE CACHE
#
# Cache all information needed to test multiple routing rules
# without rerunning COCO inference again.
# ============================================================

cache = []

total_detections = 0

invalid_crops = 0

start_time = time.time()


for image_number, image_info in enumerate(
    image_records,
    start=1
):

    if image_number % 25 == 0:

        print(
            f"Processing "
            f"{image_number}/"
            f"{len(image_records)}"
        )

    image_id = int(
        image_info[
            "id"
        ]
    )

    image_path = (
        VAL_IMAGES
        / image_info[
            "file_name"
        ]
    )


    # --------------------------------------------------------
    # YOLO
    # --------------------------------------------------------

    result = yolo.predict(

        source=str(
            image_path
        ),

        imgsz=YOLO_IMAGE_SIZE,

        conf=YOLO_CONF,

        iou=YOLO_NMS,

        max_det=YOLO_MAX_DET,

        device=0
        if DEVICE.type == "cuda"
        else "cpu",

        verbose=False,

    )[0]


    if (
        result.boxes is None
        or len(
            result.boxes
        ) == 0
    ):

        continue


    boxes = (
        result.boxes.xyxy
        .detach()
        .cpu()
        .numpy()
    )

    yolo_confs = (
        result.boxes.conf
        .detach()
        .cpu()
        .numpy()
    )

    yolo_classes = (
        result.boxes.cls
        .detach()
        .cpu()
        .numpy()
        .astype(int)
    )


    total_detections += len(
        boxes
    )


    # --------------------------------------------------------
    # Prepare crops
    # --------------------------------------------------------

    image = Image.open(
        image_path
    ).convert(
        "RGB"
    )


    valid_records = []

    crop_tensors = []


    for bbox, conf, cls_idx in zip(
        boxes,
        yolo_confs,
        yolo_classes
    ):

        crop = crop_box(
            image,
            bbox
        )

        if crop is None:

            invalid_crops += 1

            continue

        tensor = VAL_TRANSFORM(
            crop
        )

        valid_records.append(
            {
                "image_id":
                    image_id,

                "bbox":
                    [
                        float(v)
                        for v in bbox
                    ],

                "yolo_conf":
                    float(
                        conf
                    ),

                "yolo_class":
                    int(
                        cls_idx
                    )
            }
        )

        crop_tensors.append(
            tensor
        )


    image.close()


    if len(
        valid_records
    ) == 0:

        continue


    # --------------------------------------------------------
    # MobileNet for all
    # --------------------------------------------------------

    mn_probs_batch = classify_batch(
        mobilenet,
        crop_tensors
    )


    # --------------------------------------------------------
    # ConvNeXt for all hard-candidate E16 predictions
    # --------------------------------------------------------

    temp = []

    specialist_positions = []

    specialist_tensors = []


    for position, (
        record,
        mn_probs
    ) in enumerate(
        zip(
            valid_records,
            mn_probs_batch
        )
    ):

        yolo_idx = (
            record[
                "yolo_class"
            ]
        )

        mn_idx = int(
            np.argmax(
                mn_probs
            )
        )

        mn_top_prob = float(
            mn_probs[
                mn_idx
            ]
        )


        if (
            mn_top_prob
            >= E16A_GATES[
                mn_idx
            ]
        ):

            e16_idx = mn_idx

        else:

            e16_idx = yolo_idx


        item = {

            "record":
                record,

            "mn_probs":
                mn_probs.tolist(),

            "e16_final_idx":
                int(
                    e16_idx
                ),
        }


        temp.append(
            item
        )


        if e16_idx in {
            2,
            3,
            4,
            6
        }:

            specialist_positions.append(
                position
            )

            specialist_tensors.append(
                crop_tensors[
                    position
                ]
            )


    # --------------------------------------------------------
    # ConvNeXt probabilities
    # --------------------------------------------------------

    if len(
        specialist_tensors
    ) > 0:

        conv_probs_batch = classify_batch(
            convnext,
            specialist_tensors
        )

    else:

        conv_probs_batch = []


    conv_by_position = {}


    for position, conv_probs in zip(
        specialist_positions,
        conv_probs_batch
    ):

        local_idx = int(
            np.argmax(
                conv_probs
            )
        )

        global_idx = (
            CONVNEXT_TO_GLOBAL[
                local_idx
            ]
        )

        conv_by_position[
            position
        ] = {

            "conv_probs":
                conv_probs.tolist(),

            "conv_local_idx":
                local_idx,

            "conv_global_idx":
                global_idx,

            "conv_top_prob":
                float(
                    conv_probs[
                        local_idx
                    ]
                ),
        }


    # --------------------------------------------------------
    # Save cache
    # --------------------------------------------------------

    for position, item in enumerate(
        temp
    ):

        entry = {

            "image_id":
                item[
                    "record"
                ][
                    "image_id"
                ],

            "bbox":
                item[
                    "record"
                ][
                    "bbox"
                ],

            "yolo_conf":
                item[
                    "record"
                ][
                    "yolo_conf"
                ],

            "yolo_class":
                item[
                    "record"
                ][
                    "yolo_class"
                ],

            "mn_probs":
                item[
                    "mn_probs"
                ],

            "e16_final_idx":
                item[
                    "e16_final_idx"
                ],

            "conv":
                None,
        }


        if position in conv_by_position:

            entry[
                "conv"
            ] = (
                conv_by_position[
                    position
                ]
            )


        cache.append(
            entry
        )


with open(
    CACHE_PATH,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        cache,
        f
    )


inference_minutes = (
    time.time()
    - start_time
) / 60


print("\nInference cache complete.")

print(
    f"YOLO detections : "
    f"{total_detections:,}"
)

print(
    f"Cached entries  : "
    f"{len(cache):,}"
)

print(
    f"Invalid crops   : "
    f"{invalid_crops:,}"
)

print(
    f"Time            : "
    f"{inference_minutes:.2f} min"
)


# ============================================================
# 21. BUILD PREDICTIONS FOR ONE ROUTING VARIANT
# ============================================================

def build_predictions(
    allowed_conv_classes
):

    predictions = []

    source_counts = Counter()

    conv_candidates = 0

    conv_accepted = 0

    conv_rejected = 0


    for item in cache:

        yolo_conf = float(
            item[
                "yolo_conf"
            ]
        )

        e16_idx = int(
            item[
                "e16_final_idx"
            ]
        )

        mn_probs = np.array(
            item[
                "mn_probs"
            ],
            dtype=np.float32
        )


        # ----------------------------------------------------
        # Determine E16 source for reporting
        # ----------------------------------------------------

        yolo_idx = int(
            item[
                "yolo_class"
            ]
        )

        mn_idx = int(
            np.argmax(
                mn_probs
            )
        )

        mn_top_prob = float(
            mn_probs[
                mn_idx
            ]
        )


        if (
            mn_top_prob
            >= E16A_GATES[
                mn_idx
            ]
        ):

            original_source = (
                "mobilenet"
            )

        else:

            original_source = (
                "yolo"
            )


        final_idx = e16_idx

        final_source = (
            original_source
        )

        classifier_prob = float(
            mn_probs[
                final_idx
            ]
        )


        # ----------------------------------------------------
        # Specialist candidate
        # ----------------------------------------------------

        conv_info = item[
            "conv"
        ]


        if conv_info is not None:

            conv_global_idx = int(
                conv_info[
                    "conv_global_idx"
                ]
            )

            conv_top_prob = float(
                conv_info[
                    "conv_top_prob"
                ]
            )


            # ConvNeXt only allowed if its predicted class is
            # included in this routing variant.
            if (
                conv_global_idx
                in allowed_conv_classes
            ):

                conv_candidates += 1


                if (
                    conv_top_prob
                    >= CONVNEXT_GATES[
                        conv_global_idx
                    ]
                ):

                    final_idx = (
                        conv_global_idx
                    )

                    final_source = (
                        "convnext"
                    )

                    classifier_prob = (
                        conv_top_prob
                    )

                    conv_accepted += 1


                else:

                    conv_rejected += 1


        source_counts[
            final_source
        ] += 1


        score = (
            yolo_conf ** ALPHA
        ) * (
            max(
                classifier_prob,
                1e-12
            )
            ** (
                1.0
                - ALPHA
            )
        )


        x1, y1, x2, y2 = (
            item[
                "bbox"
            ]
        )


        predictions.append(
            {
                "image_id":
                    int(
                        item[
                            "image_id"
                        ]
                    ),

                "category_id":
                    final_idx + 1,

                "bbox":
                    [
                        float(
                            x1
                        ),

                        float(
                            y1
                        ),

                        float(
                            x2 - x1
                        ),

                        float(
                            y2 - y1
                        ),
                    ],

                "score":
                    float(
                        score
                    ),
            }
        )


    return (
        predictions,
        source_counts,
        conv_candidates,
        conv_accepted,
        conv_rejected
    )


# ============================================================
# 22. COCO EVAL
# ============================================================

def evaluate_predictions(
    predictions,
    label
):

    print(
        "\n"
        + "=" * 100
    )

    print(
        label
    )

    print(
        "=" * 100
    )


    coco_gt = COCO(
        str(
            GT_PATH
        )
    )

    coco_dt = coco_gt.loadRes(
        predictions
    )

    evaluator = COCOeval(
        coco_gt,
        coco_dt,
        "bbox"
    )

    evaluator.params.maxDets = [
        1,
        10,
        100
    ]

    evaluator.evaluate()

    evaluator.accumulate()

    evaluator.summarize()


    metrics = {

        "AP50_95":
            float(
                evaluator.stats[0]
            ),

        "AP50":
            float(
                evaluator.stats[1]
            ),

        "AP75":
            float(
                evaluator.stats[2]
            ),

        "AR100":
            float(
                evaluator.stats[8]
            ),
    }


    return (
        evaluator,
        metrics
    )


# ============================================================
# 23. PER-CLASS AP
# ============================================================

def per_class_metrics(
    evaluator
):

    precision = (
        evaluator.eval[
            "precision"
        ]
    )

    iou_thresholds = (
        evaluator.params.iouThrs
    )

    ap50_index = int(
        np.where(
            np.isclose(
                iou_thresholds,
                0.50
            )
        )[0][0]
    )


    output = {}


    for class_idx, class_name in enumerate(
        CLASS_NAMES
    ):

        all_iou = precision[
            :,
            :,
            class_idx,
            0,
            -1
        ]

        all_iou = all_iou[
            all_iou > -1
        ]


        ap = (
            float(
                np.mean(
                    all_iou
                )
            )
            if len(
                all_iou
            ) > 0
            else float(
                "nan"
            )
        )


        at_50 = precision[
            ap50_index,
            :,
            class_idx,
            0,
            -1
        ]

        at_50 = at_50[
            at_50 > -1
        ]


        ap50 = (
            float(
                np.mean(
                    at_50
                )
            )
            if len(
                at_50
            ) > 0
            else float(
                "nan"
            )
        )


        output[
            class_name
        ] = {

            "AP50_95":
                ap,

            "AP50":
                ap50,
        }


    return output


# ============================================================
# 24. RUN ALL ROUTING VARIANTS
# ============================================================

all_results = {}


for variant_name, allowed_classes in (
    ROUTING_VARIANTS.items()
):

    print(
        "\n"
        + "#" * 100
    )

    print(
        f"RUNNING: "
        f"{variant_name}"
    )

    print(
        "Allowed ConvNeXt classes:"
    )

    for idx in sorted(
        allowed_classes
    ):

        print(
            f"  {CLASS_NAMES[idx]}"
        )


    (
        predictions,
        source_counts,
        candidate_count,
        accepted_count,
        rejected_count,

    ) = build_predictions(
        allowed_classes
    )


    evaluator, metrics = (
        evaluate_predictions(
            predictions,
            variant_name
        )
    )


    class_metrics = (
        per_class_metrics(
            evaluator
        )
    )


    all_results[
        variant_name
    ] = {

        "allowed_convnext_classes":
            [
                CLASS_NAMES[i]
                for i in sorted(
                    allowed_classes
                )
            ],

        "metrics":
            metrics,

        "class_metrics":
            class_metrics,

        "source_counts":
            dict(
                source_counts
            ),

        "convnext_candidates":
            candidate_count,

        "convnext_accepted":
            accepted_count,

        "convnext_rejected":
            rejected_count,
    }


# ============================================================
# 25. OVERALL COMPARISON
# ============================================================

print(
    "\n"
    + "=" * 100
)

print(
    "E18-D OVERALL COMPARISON"
)

print(
    "=" * 100
)


print(
    f"\n{'Variant':45s}"
    f"{'AP50-95':>12s}"
    f"{'AP50':>12s}"
    f"{'AP75':>12s}"
    f"{'AR100':>12s}"
)


for name, result in (
    all_results.items()
):

    m = result[
        "metrics"
    ]

    print(
        f"{name:45s}"
        f"{m['AP50_95'] * 100:12.3f}"
        f"{m['AP50'] * 100:12.3f}"
        f"{m['AP75'] * 100:12.3f}"
        f"{m['AR100'] * 100:12.3f}"
    )


# ============================================================
# 26. CLASS-WISE COMPARISON
# ============================================================

print(
    "\n"
    + "=" * 100
)

print(
    "CLASS-WISE AP50-95"
)

print(
    "=" * 100
)


header = (
    f"{'Class':25s}"
)

for variant_name in (
    ROUTING_VARIANTS
):

    header += (
        f"{variant_name:>35s}"
    )


print(
    "\n"
    + header
)


for class_name in (
    CLASS_NAMES
):

    row = (
        f"{class_name:25s}"
    )

    for variant_name in (
        ROUTING_VARIANTS
    ):

        value = (
            all_results[
                variant_name
            ][
                "class_metrics"
            ][
                class_name
            ][
                "AP50_95"
            ]
        )

        row += (
            f"{value * 100:35.2f}"
        )


    print(
        row
    )


# ============================================================
# 27. FIND BEST
# ============================================================

best_variant = max(

    all_results.items(),

    key=lambda item:
        item[1][
            "metrics"
        ][
            "AP50_95"
        ]
)


best_name = (
    best_variant[0]
)

best_metrics = (
    best_variant[1][
        "metrics"
    ]
)


baseline_ap = (
    all_results[
        "E18-C_all_hard"
    ][
        "metrics"
    ][
        "AP50_95"
    ]
)


gain = (
    best_metrics[
        "AP50_95"
    ]
    - baseline_ap
)


print(
    "\n"
    + "=" * 100
)

print(
    "BEST E18-D ROUTING"
)

print(
    "=" * 100
)


print(
    f"\nBest variant : "
    f"{best_name}"
)

print(
    f"AP50-95     : "
    f"{best_metrics['AP50_95'] * 100:.4f}%"
)

print(
    f"AP50        : "
    f"{best_metrics['AP50'] * 100:.4f}%"
)

print(
    f"AP75        : "
    f"{best_metrics['AP75'] * 100:.4f}%"
)

print(
    f"AR100       : "
    f"{best_metrics['AR100'] * 100:.4f}%"
)

print(
    f"Delta vs E18-C : "
    f"{gain * 100:+.4f} pp"
)


# ============================================================
# 28. SAVE RESULTS
# ============================================================

with open(
    RESULTS_PATH,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        all_results,
        f,
        indent=2
    )


summary_lines = [

    "=" * 100,

    "E18-D — Class-Selective ConvNeXt Specialist Routing Ablation",

    "=" * 100,

    "",

    f"Alpha: {ALPHA}",

    "",

]


for name, result in (
    all_results.items()
):

    m = result[
        "metrics"
    ]

    summary_lines.extend(
        [
            name,

            (
                "Allowed ConvNeXt classes: "
                + ", ".join(
                    result[
                        "allowed_convnext_classes"
                    ]
                )
            ),

            (
                f"AP50-95: "
                f"{m['AP50_95']:.6f}"
            ),

            (
                f"AP50: "
                f"{m['AP50']:.6f}"
            ),

            (
                f"AP75: "
                f"{m['AP75']:.6f}"
            ),

            (
                f"AR100: "
                f"{m['AR100']:.6f}"
            ),

            (
                "ConvNeXt accepted: "
                f"{result['convnext_accepted']}"
            ),

            "",
        ]
    )


summary_lines.extend(
    [
        "=" * 100,

        f"BEST VARIANT: "
        f"{best_name}",

        (
            f"BEST AP50-95: "
            f"{best_metrics['AP50_95']:.6f}"
        ),

        (
            f"GAIN VS E18-C: "
            f"{gain:.6f}"
        ),

        "=" * 100,
    ]
)


with open(
    SUMMARY_PATH,
    "w",
    encoding="utf-8"
) as f:

    f.write(
        "\n".join(
            summary_lines
        )
    )


print(
    "\nResults saved:"
)

print(
    RESULTS_PATH
)

print(
    "\nSummary saved:"
)

print(
    SUMMARY_PATH
)

print(
    "\nCache saved:"
)

print(
    CACHE_PATH
)

print(
    "\nE18-D COMPLETE."
)

E18-D — CLASS-SELECTIVE CONVNEXT SPECIALIST ROUTING ABLATION

Device : cuda
GPU    : NVIDIA GeForce RTX 3050 Ti Laptop GPU

Loading MobileNet...
MobileNet loaded.

Loading ConvNeXt...
ConvNeXt loaded.

Loading YOLO11m...
YOLO11m loaded.
Processing 25/780
Processing 50/780
Processing 75/780
Processing 100/780
Processing 125/780
Processing 150/780
Processing 175/780
Processing 200/780
Processing 225/780
Processing 250/780
Processing 275/780
Processing 300/780
Processing 325/780
Processing 350/780
Processing 375/780
Processing 400/780
Processing 425/780
Processing 450/780
Processing 475/780
Processing 500/780
Processing 525/780
Processing 550/780
Processing 575/780
Processing 600/780
Processing 625/780
Processing 650/780
Processing 675/780
Processing 700/780
Processing 725/780
Processing 750/780
Processing 775/780

Inference cache complete.
YOLO detections : 55,605
Cached entries  : 55,605
Invalid crops   : 0
Time            : 7.17 min

####################################################

## E18-E — ConvNeXt Class-Specific Gate Refinement

In [ ]:
# E18-E — ConvNeXt Class-Specific Gate Refinement
#
# Purpose:
#   Refine ConvNeXt confidence gates after E18-D showed that
#   the best specialist routing is:
#
#       mixed_plastic_soft
#       non_plastic
#       pet_oil
#
#   ConvNeXt remains DISABLED for mixed_plastic_rigid.
#
# No retraining.
# No YOLO inference.
# No MobileNet inference.
# No ConvNeXt inference.
#
# Reuses E18-D cached predictions.
#
# Fixed:
#   YOLO11m detector = E12
#   MobileNet routing = E16-A
#   ConvNeXt model = E18-B
#   ConvNeXt routing = E18-D2
#   alpha = 0.70
#
# Greedy gate optimization:
#
#   Step 1: tune Mixed Soft
#   Step 2: tune non-plastic using best Mixed Soft gate
#   Step 3: tune PET Oil using previous best gates
#
# Primary metric:
#   COCO AP50-95
#
# Tie-break:
#   AP50
# ============================================================


import json
from pathlib import Path
from collections import Counter

import numpy as np

from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval


# ============================================================
# 1. PATHS
# ============================================================

BASE = Path(
    r"C:\Users\varda\Documents\_My Computer\COMMON_space\Learning"
    r"\Upgrad_EPGP_MS\upgrad_IIITB_MS_course\MS_LJMU_Material"
)

THESIS_CODE = BASE / "Thesis_Code"


# ------------------------------------------------------------
# E18-D cache
# ------------------------------------------------------------

E18D_DIR = (
    THESIS_CODE
    / "runs"
    / "sortwaste"
    / "E18D_class_selective_convnext"
)

CACHE_PATH = (
    E18D_DIR
    / "E18D_cached_predictions.json"
)

GT_PATH = (
    E18D_DIR
    / "E18D_val_gt_7class.json"
)


# ------------------------------------------------------------
# E18-E output
# ------------------------------------------------------------

OUTPUT_DIR = (
    THESIS_CODE
    / "runs"
    / "sortwaste"
    / "E18E_convnext_gate_refinement"
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)


RESULTS_PATH = (
    OUTPUT_DIR
    / "E18E_gate_search_results.json"
)

SUMMARY_PATH = (
    OUTPUT_DIR
    / "E18E_summary.txt"
)

BEST_PREDICTIONS_PATH = (
    OUTPUT_DIR
    / "E18E_best_predictions.json"
)


assert CACHE_PATH.exists(), (
    f"E18-D cache not found:\n{CACHE_PATH}"
)

assert GT_PATH.exists(), (
    f"E18-D GT not found:\n{GT_PATH}"
)


# ============================================================
# 2. LOAD CACHE
# ============================================================

print("=" * 100)
print("E18-E — CONVNEXT CLASS-SPECIFIC GATE REFINEMENT")
print("=" * 100)


with open(
    CACHE_PATH,
    "r",
    encoding="utf-8"
) as f:

    cache = json.load(f)


print(
    f"\nCached detections : {len(cache):,}"
)


# ============================================================
# 3. CLASS TAXONOMY
# ============================================================

CLASS_NAMES = [
    "ecal",
    "hdpe",
    "mixed_plastic_rigid",
    "mixed_plastic_soft",
    "non_plastic",
    "pet",
    "pet_oil",
]


# Global indices:
#
# 0 ecal
# 1 hdpe
# 2 mixed rigid
# 3 mixed soft
# 4 nonplastic
# 5 pet
# 6 pet oil


MIXED_RIGID = 2
MIXED_SOFT = 3
NON_PLASTIC = 4
PET_OIL = 6


# ============================================================
# 4. FIXED E18-D2 ROUTING
# ============================================================

# Mixed Rigid intentionally excluded because E18-D showed
# that ConvNeXt routing hurt this class.

ALLOWED_CONVNEXT_CLASSES = {
    MIXED_SOFT,
    NON_PLASTIC,
    PET_OIL,
}


# ============================================================
# 5. FIXED CONFIDENCE FUSION
# ============================================================

ALPHA = 0.70


# ============================================================
# 6. E16-A MOBILENET GATES
#
# Needed only to determine whether the E16-A source was
# MobileNet or YOLO for reporting.
# ============================================================

E16A_GATES = {
    0: 0.99,  # ecal
    1: 0.98,  # hdpe
    2: 0.98,  # mixed rigid
    3: 0.94,  # mixed soft
    4: 0.94,  # nonplastic
    5: 0.99,  # pet
    6: 0.99,  # pet oil
}


# ============================================================
# 7. BASELINE E18-D2 CONVNEXT GATES
# ============================================================

BASE_GATES = {
    MIXED_SOFT: 0.94,
    NON_PLASTIC: 0.94,
    PET_OIL: 0.99,
}


# ============================================================
# 8. GATE SEARCH CANDIDATES
#
# Deliberately small search to avoid excessive validation
# optimization.
# ============================================================

GATE_CANDIDATES = {

    MIXED_SOFT: [
        0.90,
        0.92,
        0.94,
        0.96,
        0.98,
    ],

    NON_PLASTIC: [
        0.90,
        0.92,
        0.94,
        0.96,
        0.98,
    ],

    PET_OIL: [
        0.94,
        0.96,
        0.98,
        0.99,
    ],
}


# ============================================================
# 9. BUILD PREDICTIONS
# ============================================================

def build_predictions(
    conv_gates
):

    predictions = []

    source_counts = Counter()

    conv_candidates = Counter()

    conv_accepted = Counter()

    conv_rejected = Counter()


    for item in cache:

        yolo_conf = float(
            item["yolo_conf"]
        )

        yolo_idx = int(
            item["yolo_class"]
        )

        e16_idx = int(
            item["e16_final_idx"]
        )

        mn_probs = np.asarray(
            item["mn_probs"],
            dtype=np.float64
        )


        # ----------------------------------------------------
        # Determine original E16-A class source
        # ----------------------------------------------------

        mn_idx = int(
            np.argmax(
                mn_probs
            )
        )

        mn_top_prob = float(
            mn_probs[
                mn_idx
            ]
        )


        if (
            mn_top_prob
            >= E16A_GATES[
                mn_idx
            ]
        ):

            original_source = (
                "mobilenet"
            )

        else:

            original_source = (
                "yolo"
            )


        # ----------------------------------------------------
        # Begin with E16-A final result
        # ----------------------------------------------------

        final_idx = e16_idx

        final_source = (
            original_source
        )

        classifier_prob = float(
            mn_probs[
                final_idx
            ]
        )


        # ----------------------------------------------------
        # ConvNeXt specialist refinement
        # ----------------------------------------------------

        conv_info = item.get(
            "conv"
        )


        if conv_info is not None:

            conv_global_idx = int(
                conv_info[
                    "conv_global_idx"
                ]
            )

            conv_top_prob = float(
                conv_info[
                    "conv_top_prob"
                ]
            )


            # Only classes retained by E18-D2 are eligible.
            if (
                conv_global_idx
                in ALLOWED_CONVNEXT_CLASSES
            ):

                conv_candidates[
                    conv_global_idx
                ] += 1


                gate = float(
                    conv_gates[
                        conv_global_idx
                    ]
                )


                if (
                    conv_top_prob
                    >= gate
                ):

                    final_idx = (
                        conv_global_idx
                    )

                    final_source = (
                        "convnext"
                    )

                    classifier_prob = (
                        conv_top_prob
                    )

                    conv_accepted[
                        conv_global_idx
                    ] += 1


                else:

                    conv_rejected[
                        conv_global_idx
                    ] += 1


        source_counts[
            final_source
        ] += 1


        # ----------------------------------------------------
        # Final confidence score
        #
        # Same fusion used by E16/E18:
        #
        # S = YOLO_conf^0.70 × classifier_prob^0.30
        # ----------------------------------------------------

        classifier_prob = max(
            float(
                classifier_prob
            ),
            1e-12
        )


        score = (
            yolo_conf
            ** ALPHA
        ) * (
            classifier_prob
            ** (
                1.0
                - ALPHA
            )
        )


        x1, y1, x2, y2 = (
            item["bbox"]
        )


        predictions.append(
            {
                "image_id":
                    int(
                        item[
                            "image_id"
                        ]
                    ),

                "category_id":
                    int(
                        final_idx + 1
                    ),

                "bbox":
                    [
                        float(
                            x1
                        ),

                        float(
                            y1
                        ),

                        float(
                            x2 - x1
                        ),

                        float(
                            y2 - y1
                        ),
                    ],

                "score":
                    float(
                        score
                    ),
            }
        )


    return {
        "predictions":
            predictions,

        "source_counts":
            source_counts,

        "conv_candidates":
            conv_candidates,

        "conv_accepted":
            conv_accepted,

        "conv_rejected":
            conv_rejected,
    }


# ============================================================
# 10. COCO EVALUATION
# ============================================================

coco_gt = COCO(
    str(
        GT_PATH
    )
)


def evaluate_predictions(
    predictions,
    verbose=False
):

    coco_dt = coco_gt.loadRes(
        predictions
    )

    evaluator = COCOeval(
        coco_gt,
        coco_dt,
        "bbox"
    )

    evaluator.params.maxDets = [
        1,
        10,
        100
    ]


    if verbose:

        evaluator.evaluate()
        evaluator.accumulate()
        evaluator.summarize()

    else:

        # Suppress repetitive COCOeval printing,
        # but summarize() MUST still run because
        # it populates evaluator.stats.
        import contextlib
        import io

        with contextlib.redirect_stdout(
            io.StringIO()
        ):

            evaluator.evaluate()
            evaluator.accumulate()
            evaluator.summarize()


    metrics = {

        "AP50_95":
            float(
                evaluator.stats[0]
            ),

        "AP50":
            float(
                evaluator.stats[1]
            ),

        "AP75":
            float(
                evaluator.stats[2]
            ),

        "AR100":
            float(
                evaluator.stats[8]
            ),
    }


    return evaluator, metrics

# ============================================================
# 11. PER-CLASS METRICS
# ============================================================

def get_per_class_metrics(
    evaluator
):

    precision = (
        evaluator.eval[
            "precision"
        ]
    )

    iou_thresholds = (
        evaluator.params.iouThrs
    )


    ap50_idx = int(
        np.where(
            np.isclose(
                iou_thresholds,
                0.50
            )
        )[0][0]
    )


    output = {}


    for class_idx, class_name in enumerate(
        CLASS_NAMES
    ):

        # ----------------------------------------------------
        # AP50-95
        # ----------------------------------------------------

        values = precision[
            :,
            :,
            class_idx,
            0,
            -1
        ]

        values = values[
            values > -1
        ]


        ap = (
            float(
                np.mean(
                    values
                )
            )

            if values.size > 0

            else float(
                "nan"
            )
        )


        # ----------------------------------------------------
        # AP50
        # ----------------------------------------------------

        values50 = precision[
            ap50_idx,
            :,
            class_idx,
            0,
            -1
        ]

        values50 = values50[
            values50 > -1
        ]


        ap50 = (
            float(
                np.mean(
                    values50
                )
            )

            if values50.size > 0

            else float(
                "nan"
            )
        )


        output[
            class_name
        ] = {

            "AP50_95":
                ap,

            "AP50":
                ap50,
        }


    return output


# ============================================================
# 12. HELPER FOR ONE CONFIGURATION
# ============================================================

def run_configuration(
    gates,
    label,
    verbose=False
):

    built = build_predictions(
        gates
    )

    evaluator, metrics = (
        evaluate_predictions(
            built[
                "predictions"
            ],
            verbose=verbose
        )
    )


    class_metrics = (
        get_per_class_metrics(
            evaluator
        )
    )


    result = {

        "label":
            label,

        "gates": {
            CLASS_NAMES[k]:
                float(v)

            for k, v in (
                gates.items()
            )
        },

        "metrics":
            metrics,

        "class_metrics":
            class_metrics,

        "source_counts":
            {
                k:
                    int(v)

                for k, v in (
                    built[
                        "source_counts"
                    ].items()
                )
            },

        "convnext_candidates":
            {
                CLASS_NAMES[k]:
                    int(v)

                for k, v in (
                    built[
                        "conv_candidates"
                    ].items()
                )
            },

        "convnext_accepted":
            {
                CLASS_NAMES[k]:
                    int(v)

                for k, v in (
                    built[
                        "conv_accepted"
                    ].items()
                )
            },

        "convnext_rejected":
            {
                CLASS_NAMES[k]:
                    int(v)

                for k, v in (
                    built[
                        "conv_rejected"
                    ].items()
                )
            },
    }


    return (
        result,
        built[
            "predictions"
        ]
    )


# ============================================================
# 13. BASELINE — REPRODUCE E18-D2
# ============================================================

print(
    "\n"
    + "=" * 100
)

print(
    "BASELINE — REPRODUCING E18-D2"
)

print(
    "=" * 100
)


baseline_result, _ = (
    run_configuration(
        BASE_GATES.copy(),
        "E18-D2 baseline",
        verbose=True
    )
)


baseline_metrics = (
    baseline_result[
        "metrics"
    ]
)


print(
    "\nExpected approximately:"
)

print(
    "AP50-95 : 45.4729%"
)

print(
    "AP50    : 60.7331%"
)

print(
    "AP75    : 51.7112%"
)

print(
    "AR100   : 61.1162%"
)


print(
    "\nReproduced:"
)

print(
    f"AP50-95 : "
    f"{baseline_metrics['AP50_95'] * 100:.4f}%"
)

print(
    f"AP50    : "
    f"{baseline_metrics['AP50'] * 100:.4f}%"
)

print(
    f"AP75    : "
    f"{baseline_metrics['AP75'] * 100:.4f}%"
)

print(
    f"AR100   : "
    f"{baseline_metrics['AR100'] * 100:.4f}%"
)


# ============================================================
# 14. GREEDY GATE SEARCH
#
# Order:
#
#   1. Mixed Soft
#   2. non-plastic
#   3. PET Oil
#
# At each stage:
#
#   - all previously optimized gates stay fixed
#   - all future gates stay at baseline
#
# Selection:
#
#   highest AP50-95
#   then AP50 as tie-break
# ============================================================

current_gates = (
    BASE_GATES.copy()
)

search_history = []


SEARCH_ORDER = [
    MIXED_SOFT,
    NON_PLASTIC,
    PET_OIL,
]


for class_idx in (
    SEARCH_ORDER
):

    class_name = (
        CLASS_NAMES[
            class_idx
        ]
    )


    print(
        "\n"
        + "=" * 100
    )

    print(
        f"TUNING CONVNEXT GATE — "
        f"{class_name}"
    )

    print(
        "=" * 100
    )


    stage_results = []


    for candidate_gate in (
        GATE_CANDIDATES[
            class_idx
        ]
    ):

        test_gates = (
            current_gates.copy()
        )

        test_gates[
            class_idx
        ] = (
            candidate_gate
        )


        result, _ = (
            run_configuration(
                test_gates,
                (
                    f"{class_name}"
                    f"_gate_{candidate_gate:.3f}"
                ),
                verbose=False
            )
        )


        m = result[
            "metrics"
        ]


        accepted = (
            result[
                "convnext_accepted"
            ].get(
                class_name,
                0
            )
        )


        print(
            f"Gate {candidate_gate:.3f}"
            f" | AP50-95 "
            f"{m['AP50_95'] * 100:8.4f}%"
            f" | AP50 "
            f"{m['AP50'] * 100:8.4f}%"
            f" | AP75 "
            f"{m['AP75'] * 100:8.4f}%"
            f" | Accepted "
            f"{accepted:,}"
        )


        stage_results.append(
            result
        )


    # --------------------------------------------------------
    # Select best:
    #
    # 1. AP50-95
    # 2. AP50 tie-break
    # --------------------------------------------------------

    best_stage = max(

        stage_results,

        key=lambda x: (
            x[
                "metrics"
            ][
                "AP50_95"
            ],

            x[
                "metrics"
            ][
                "AP50"
            ],
        )
    )


    best_gate = (
        best_stage[
            "gates"
        ][
            class_name
        ]
    )


    current_gates[
        class_idx
    ] = (
        best_gate
    )


    print(
        "\nSelected gate:"
    )

    print(
        f"{class_name} = "
        f"{best_gate:.3f}"
    )

    print(
        f"Stage-best AP50-95 = "
        f"{best_stage['metrics']['AP50_95'] * 100:.4f}%"
    )


    search_history.append(
        {
            "class":
                class_name,

            "candidate_results":
                stage_results,

            "selected_gate":
                best_gate,

            "selected_metrics":
                best_stage[
                    "metrics"
                ],
        }
    )


# ============================================================
# 15. FINAL E18-E CONFIGURATION
# ============================================================

print(
    "\n"
    + "=" * 100
)

print(
    "FINAL E18-E CONFIGURATION"
)

print(
    "=" * 100
)


for class_idx in (
    SEARCH_ORDER
):

    print(
        f"{CLASS_NAMES[class_idx]:25s}"
        f": "
        f"{current_gates[class_idx]:.3f}"
    )


print(
    "\nMixed Rigid ConvNeXt routing:"
    " DISABLED"
)

print(
    f"Confidence fusion alpha: "
    f"{ALPHA:.2f}"
)


# ============================================================
# 16. FINAL EVALUATION
# ============================================================

final_result, final_predictions = (
    run_configuration(
        current_gates,
        "E18-E final",
        verbose=True
    )
)


final_metrics = (
    final_result[
        "metrics"
    ]
)


# ============================================================
# 17. DELTAS
# ============================================================

delta_ap = (
    final_metrics[
        "AP50_95"
    ]
    - baseline_metrics[
        "AP50_95"
    ]
)

delta_ap50 = (
    final_metrics[
        "AP50"
    ]
    - baseline_metrics[
        "AP50"
    ]
)

delta_ap75 = (
    final_metrics[
        "AP75"
    ]
    - baseline_metrics[
        "AP75"
    ]
)

delta_ar100 = (
    final_metrics[
        "AR100"
    ]
    - baseline_metrics[
        "AR100"
    ]
)


# ============================================================
# 18. FINAL COMPARISON
# ============================================================

print(
    "\n"
    + "=" * 100
)

print(
    "E18-E FINAL RESULT"
)

print(
    "=" * 100
)


print(
    f"\n{'Metric':12s}"
    f"{'E18-D2':>14s}"
    f"{'E18-E':>14s}"
    f"{'Delta':>14s}"
)


comparison = [

    (
        "AP50-95",
        baseline_metrics[
            "AP50_95"
        ],
        final_metrics[
            "AP50_95"
        ],
    ),

    (
        "AP50",
        baseline_metrics[
            "AP50"
        ],
        final_metrics[
            "AP50"
        ],
    ),

    (
        "AP75",
        baseline_metrics[
            "AP75"
        ],
        final_metrics[
            "AP75"
        ],
    ),

    (
        "AR100",
        baseline_metrics[
            "AR100"
        ],
        final_metrics[
            "AR100"
        ],
    ),
]


for metric_name, old, new in (
    comparison
):

    print(
        f"{metric_name:12s}"
        f"{old * 100:14.4f}"
        f"{new * 100:14.4f}"
        f"{(new-old) * 100:+14.4f}"
    )


# ============================================================
# 19. CLASS-WISE COMPARISON
# ============================================================

print(
    "\n"
    + "=" * 100
)

print(
    "CLASS-WISE AP50-95 COMPARISON"
)

print(
    "=" * 100
)


print(
    f"\n{'Class':28s}"
    f"{'E18-D2':>14s}"
    f"{'E18-E':>14s}"
    f"{'Delta':>14s}"
)


for class_name in (
    CLASS_NAMES
):

    old = (
        baseline_result[
            "class_metrics"
        ][
            class_name
        ][
            "AP50_95"
        ]
    )

    new = (
        final_result[
            "class_metrics"
        ][
            class_name
        ][
            "AP50_95"
        ]
    )


    print(
        f"{class_name:28s}"
        f"{old * 100:14.2f}"
        f"{new * 100:14.2f}"
        f"{(new-old) * 100:+14.2f}"
    )


# ============================================================
# 20. SOURCE COUNTS
# ============================================================

print(
    "\n"
    + "=" * 100
)

print(
    "FINAL CLASS SOURCE COUNTS"
)

print(
    "=" * 100
)


for source in [
    "yolo",
    "mobilenet",
    "convnext",
]:

    count = (
        final_result[
            "source_counts"
        ].get(
            source,
            0
        )
    )

    print(
        f"{source:15s}: "
        f"{count:,}"
    )


print(
    "\nConvNeXt accepted by class:"
)


for class_idx in (
    SEARCH_ORDER
):

    class_name = (
        CLASS_NAMES[
            class_idx
        ]
    )

    accepted = (
        final_result[
            "convnext_accepted"
        ].get(
            class_name,
            0
        )
    )

    print(
        f"{class_name:25s}: "
        f"{accepted:,}"
    )


# ============================================================
# 21. SAVE BEST PREDICTIONS
# ============================================================

with open(
    BEST_PREDICTIONS_PATH,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        final_predictions,
        f
    )


# ============================================================
# 22. SAVE STRUCTURED RESULTS
# ============================================================

output_data = {

    "experiment":
        (
            "E18-E — ConvNeXt "
            "Class-Specific Gate Refinement"
        ),

    "method":
        (
            "Greedy one-class-at-a-time "
            "ConvNeXt confidence-gate search"
        ),

    "routing": [
        "mixed_plastic_soft",
        "non_plastic",
        "pet_oil",
    ],

    "mixed_plastic_rigid_convnext":
        "disabled",

    "alpha":
        ALPHA,

    "baseline_gates": {
        CLASS_NAMES[k]:
            float(v)

        for k, v in (
            BASE_GATES.items()
        )
    },

    "final_gates": {
        CLASS_NAMES[k]:
            float(v)

        for k, v in (
            current_gates.items()
        )
    },

    "baseline_result":
        baseline_result,

    "search_history":
        search_history,

    "final_result":
        final_result,

    "delta_vs_E18_D2": {

        "AP50_95":
            float(
                delta_ap
            ),

        "AP50":
            float(
                delta_ap50
            ),

        "AP75":
            float(
                delta_ap75
            ),

        "AR100":
            float(
                delta_ar100
            ),
    },
}


with open(
    RESULTS_PATH,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        output_data,
        f,
        indent=2
    )


# ============================================================
# 23. SAVE SUMMARY
# ============================================================

summary_lines = [

    "=" * 100,

    "E18-E — ConvNeXt Class-Specific Gate Refinement",

    "=" * 100,

    "",

    "Routing:",
    "  mixed_plastic_soft",
    "  non_plastic",
    "  pet_oil",

    "",

    "ConvNeXt Mixed Rigid routing: DISABLED",

    "",

    f"Alpha: {ALPHA:.2f}",

    "",

    "Final ConvNeXt gates:",

]


for class_idx in (
    SEARCH_ORDER
):

    summary_lines.append(
        (
            f"  "
            f"{CLASS_NAMES[class_idx]}"
            f" = "
            f"{current_gates[class_idx]:.3f}"
        )
    )


summary_lines.extend(
    [

        "",

        "E18-D2 baseline:",

        (
            f"  AP50-95 = "
            f"{baseline_metrics['AP50_95']:.6f}"
        ),

        (
            f"  AP50    = "
            f"{baseline_metrics['AP50']:.6f}"
        ),

        (
            f"  AP75    = "
            f"{baseline_metrics['AP75']:.6f}"
        ),

        (
            f"  AR100   = "
            f"{baseline_metrics['AR100']:.6f}"
        ),

        "",

        "E18-E final:",

        (
            f"  AP50-95 = "
            f"{final_metrics['AP50_95']:.6f}"
        ),

        (
            f"  AP50    = "
            f"{final_metrics['AP50']:.6f}"
        ),

        (
            f"  AP75    = "
            f"{final_metrics['AP75']:.6f}"
        ),

        (
            f"  AR100   = "
            f"{final_metrics['AR100']:.6f}"
        ),

        "",

        (
            f"AP50-95 gain vs E18-D2 = "
            f"{delta_ap * 100:+.4f} pp"
        ),

        "",

        "=" * 100,
    ]
)


with open(
    SUMMARY_PATH,
    "w",
    encoding="utf-8"
) as f:

    f.write(
        "\n".join(
            summary_lines
        )
    )


# ============================================================
# 24. DONE
# ============================================================

print(
    "\n"
    + "=" * 100
)

print(
    "E18-E COMPLETE"
)

print(
    "=" * 100
)


print(
    "\nBest gates:"
)


for class_idx in (
    SEARCH_ORDER
):

    print(
        f"{CLASS_NAMES[class_idx]:25s}"
        f": "
        f"{current_gates[class_idx]:.3f}"
    )


print(
    f"\nE18-D2 AP50-95 : "
    f"{baseline_metrics['AP50_95'] * 100:.4f}%"
)

print(
    f"E18-E AP50-95  : "
    f"{final_metrics['AP50_95'] * 100:.4f}%"
)

print(
    f"Improvement    : "
    f"{delta_ap * 100:+.4f} pp"
)


print(
    "\nResults:"
)

print(
    RESULTS_PATH
)

print(
    "\nSummary:"
)

print(
    SUMMARY_PATH
)

print(
    "\nBest predictions:"
)

print(
    BEST_PREDICTIONS_PATH
)

E18-E — CONVNEXT CLASS-SPECIFIC GATE REFINEMENT

Cached detections : 55,605
loading annotations into memory...
Done (t=0.05s)
creating index...
index created!

BASELINE — REPRODUCING E18-D2
Loading and preparing results...
DONE (t=0.05s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=3.91s).
Accumulating evaluation results...
DONE (t=0.53s).
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.455
 Average Precision  (AP) @[ IoU=0.50      | area=   all | maxDets=100 ] = 0.607
 Average Precision  (AP) @[ IoU=0.75      | area=   all | maxDets=100 ] = 0.517
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= small | maxDets=100 ] = -1.000
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=medium | maxDets=100 ] = 0.152
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= large | maxDets=100 ] = 0.462
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets=  1 ] = 0.273
 Average Recall     (AR) @[ IoU=0.50:

# Model E19 — YOLO11m + ConvNeXt-Small Specialist Refinement

In [ ]:
# Part A — E19-A: Train ConvNeXt-Small
# ==========================================================================================
# E19-A — CONVNEXT-SMALL TRAINING FOR HARD-CLASS SPECIALIST REFINEMENT
# ==========================================================================================
#
# Controlled architecture-capacity experiment:
#
#   E18-B : ConvNeXt-Tiny
#   E19-A : ConvNeXt-Small
#
# Everything else is kept as close as possible to E18-B:
#
#   Dataset          : E18-A YOLO11m-matched hard-class crops
#   Classes          : 4
#   Image size       : 224
#   Loss             : Class-weighted CrossEntropyLoss
#   Optimizer        : AdamW
#   LR               : 1e-4
#   Weight decay     : 1e-4
#   Scheduler        : ReduceLROnPlateau
#   Epochs           : 30
#   Patience         : 7
#   Best checkpoint  : validation Macro-F1
#   Augmentation     : same controlled E3Y-B/E18-B protocol
#
# Only intentional architecture change:
#
#   ConvNeXt-Tiny -> ConvNeXt-Small
#
# Batch size is reduced from 8 -> 4 because of 4 GB GPU VRAM.
# ==========================================================================================


import os
import json
import random
import time
from pathlib import Path
from collections import Counter

import numpy as np
import torch
import torch.nn as nn

from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from torchvision.models import (
    convnext_small,
    ConvNeXt_Small_Weights,
)

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
)


# ==========================================================================================
# 1. REPRODUCIBILITY
# ==========================================================================================

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False


# ==========================================================================================
# 2. DEVICE
# ==========================================================================================

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("=" * 100)
print("E19-A — CONVNEXT-SMALL HARD-CLASS SPECIALIST TRAINING")
print("=" * 100)

print(f"\nPyTorch : {torch.__version__}")
print(f"Device  : {DEVICE}")

if DEVICE.type == "cuda":
    print(
        f"GPU     : "
        f"{torch.cuda.get_device_name(0)}"
    )


# ==========================================================================================
# 3. PATHS
# ==========================================================================================

BASE = Path(
    r"C:\Users\varda\Documents\_My Computer\COMMON_space\Learning"
    r"\Upgrad_EPGP_MS\upgrad_IIITB_MS_course\MS_LJMU_Material"
)

DATASET_ROOT = (
    BASE
    / "Topic Data"
    / "SortWaste"
    / "dataset"
    / "dataset"
)

E18_CROP_ROOT = (
    DATASET_ROOT
    / "yolo11m_convnext_hardclass_crops_E18"
)

TRAIN_DIR = (
    E18_CROP_ROOT
    / "train"
)

VAL_DIR = (
    E18_CROP_ROOT
    / "val"
)


OUTPUT_DIR = (
    BASE
    / "Thesis_Code"
    / "runs"
    / "sortwaste"
    / "E19A_convnext_small_hardclass"
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)


CHECKPOINT_PATH = (
    OUTPUT_DIR
    / "E19A_ConvNeXtSmall_best.pth"
)

HISTORY_PATH = (
    OUTPUT_DIR
    / "E19A_training_history.json"
)

SUMMARY_PATH = (
    OUTPUT_DIR
    / "E19A_summary.txt"
)


assert TRAIN_DIR.exists(), (
    f"Training crop directory not found:\n{TRAIN_DIR}"
)

assert VAL_DIR.exists(), (
    f"Validation crop directory not found:\n{VAL_DIR}"
)


print(f"\nTrain : {TRAIN_DIR}")
print(f"Val   : {VAL_DIR}")
print(f"Output: {OUTPUT_DIR}")


# ==========================================================================================
# 4. HYPERPARAMETERS
# ==========================================================================================

IMAGE_SIZE = 224

# ConvNeXt-Small is substantially heavier than Tiny.
# Use batch 4 first on the RTX 3050 Ti 4 GB.
BATCH_SIZE = 4

NUM_EPOCHS = 30

LEARNING_RATE = 1e-4

WEIGHT_DECAY = 1e-4

NUM_WORKERS = 0

PATIENCE = 7

MIN_DELTA = 1e-4

NUM_CLASSES = 4


# ==========================================================================================
# 5. EXPECTED CLASS ORDER
# ==========================================================================================

EXPECTED_CLASSES = [
    "mixed_plastic_rigid",
    "mixed_plastic_soft",
    "non_plastic",
    "pet_oil",
]


# ==========================================================================================
# 6. TRANSFORMS
#
# Same controlled protocol used for E18-B:
#
#   Resize
#   Horizontal flip
#   Rotation ±10°
#   Mild ColorJitter
#
# NO RandomResizedCrop.
# ==========================================================================================

IMAGENET_MEAN = [
    0.485,
    0.456,
    0.406,
]

IMAGENET_STD = [
    0.229,
    0.224,
    0.225,
]


train_transform = transforms.Compose(
    [
        transforms.Resize(
            (IMAGE_SIZE, IMAGE_SIZE)
        ),

        transforms.RandomHorizontalFlip(
            p=0.5
        ),

        transforms.RandomRotation(
            degrees=10
        ),

        transforms.ColorJitter(
            brightness=0.15,
            contrast=0.15,
            saturation=0.10,
            hue=0.02,
        ),

        transforms.ToTensor(),

        transforms.Normalize(
            mean=IMAGENET_MEAN,
            std=IMAGENET_STD,
        ),
    ]
)


val_transform = transforms.Compose(
    [
        transforms.Resize(
            (IMAGE_SIZE, IMAGE_SIZE)
        ),

        transforms.ToTensor(),

        transforms.Normalize(
            mean=IMAGENET_MEAN,
            std=IMAGENET_STD,
        ),
    ]
)


# ==========================================================================================
# 7. DATASETS
# ==========================================================================================

train_dataset = datasets.ImageFolder(
    TRAIN_DIR,
    transform=train_transform,
)

val_dataset = datasets.ImageFolder(
    VAL_DIR,
    transform=val_transform,
)


print("\nClass mapping:")
print(train_dataset.class_to_idx)


assert train_dataset.classes == EXPECTED_CLASSES, (
    "\nUnexpected training class order.\n"
    f"Expected: {EXPECTED_CLASSES}\n"
    f"Actual  : {train_dataset.classes}"
)

assert val_dataset.classes == EXPECTED_CLASSES, (
    "\nUnexpected validation class order.\n"
    f"Expected: {EXPECTED_CLASSES}\n"
    f"Actual  : {val_dataset.classes}"
)


print(
    f"\nTrain samples : "
    f"{len(train_dataset):,}"
)

print(
    f"Val samples   : "
    f"{len(val_dataset):,}"
)


# ==========================================================================================
# 8. CLASS DISTRIBUTION
# ==========================================================================================

train_targets = np.array(
    train_dataset.targets
)

train_counts = np.bincount(
    train_targets,
    minlength=NUM_CLASSES,
)


print("\nTraining class distribution:")

for idx, class_name in enumerate(
    EXPECTED_CLASSES
):
    print(
        f"{class_name:25s}: "
        f"{train_counts[idx]:,}"
    )


# ==========================================================================================
# 9. CLASS WEIGHTS
#
# Same formula used in E18-B:
#
#       weight_c = N / (K * n_c)
#
# ==========================================================================================

N = len(train_dataset)
K = NUM_CLASSES


class_weights = (
    N
    / (
        K
        * train_counts.astype(
            np.float64
        )
    )
)


class_weights_tensor = torch.tensor(
    class_weights,
    dtype=torch.float32,
    device=DEVICE,
)


print("\nClass weights:")

for idx, class_name in enumerate(
    EXPECTED_CLASSES
):
    print(
        f"{class_name:25s}: "
        f"{class_weights[idx]:.6f}"
    )


# ==========================================================================================
# 10. DATALOADERS
# ==========================================================================================

train_generator = torch.Generator()
train_generator.manual_seed(SEED)


train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=(
        DEVICE.type == "cuda"
    ),
    generator=train_generator,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=(
        DEVICE.type == "cuda"
    ),
)


print(
    f"\nTrain batches : "
    f"{len(train_loader):,}"
)

print(
    f"Val batches   : "
    f"{len(val_loader):,}"
)


# ==========================================================================================
# 11. LOAD PRETRAINED CONVNEXT-SMALL
# ==========================================================================================

print("\nLoading ImageNet pretrained ConvNeXt-Small...")


weights = (
    ConvNeXt_Small_Weights.DEFAULT
)

model = convnext_small(
    weights=weights
)


# Torchvision ConvNeXt classifier:
#
# classifier[0] = LayerNorm
# classifier[1] = Flatten
# classifier[2] = Linear
#
# Replace final Linear layer robustly.

in_features = (
    model.classifier[2].in_features
)

model.classifier[2] = nn.Linear(
    in_features,
    NUM_CLASSES,
)


model = model.to(
    DEVICE
)


total_params = sum(
    p.numel()
    for p in model.parameters()
)

trainable_params = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)


print("ConvNeXt-Small loaded.")

print(
    f"Total parameters     : "
    f"{total_params:,}"
)

print(
    f"Trainable parameters : "
    f"{trainable_params:,}"
)

print(
    f"Classifier input dim : "
    f"{in_features}"
)


# ==========================================================================================
# 12. LOSS
# ==========================================================================================

criterion = nn.CrossEntropyLoss(
    weight=class_weights_tensor
)


# ==========================================================================================
# 13. OPTIMIZER
# ==========================================================================================

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
)


# ==========================================================================================
# 14. LR SCHEDULER
#
# Same principle as E18-B:
# reduce LR when validation Macro-F1 stalls.
# ==========================================================================================

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="max",
    factor=0.5,
    patience=2,
)


# ==========================================================================================
# 15. AMP
# ==========================================================================================

USE_AMP = (
    DEVICE.type == "cuda"
)

if USE_AMP:
    scaler = torch.amp.GradScaler(
        "cuda"
    )
else:
    scaler = None


# ==========================================================================================
# 16. TRAIN ONE EPOCH
# ==========================================================================================

def train_one_epoch(
    model,
    loader,
    optimizer,
    criterion,
):

    model.train()

    running_loss = 0.0

    all_targets = []
    all_predictions = []


    for batch_idx, (
        images,
        targets,
    ) in enumerate(
        loader,
        start=1
    ):

        images = images.to(
            DEVICE,
            non_blocking=True
        )

        targets = targets.to(
            DEVICE,
            non_blocking=True
        )


        optimizer.zero_grad(
            set_to_none=True
        )


        if USE_AMP:

            with torch.amp.autocast(
                device_type="cuda",
                dtype=torch.float16,
            ):

                logits = model(
                    images
                )

                loss = criterion(
                    logits,
                    targets
                )


            scaler.scale(
                loss
            ).backward()

            scaler.step(
                optimizer
            )

            scaler.update()


        else:

            logits = model(
                images
            )

            loss = criterion(
                logits,
                targets
            )

            loss.backward()

            optimizer.step()


        running_loss += (
            loss.item()
            * images.size(0)
        )


        predictions = torch.argmax(
            logits,
            dim=1
        )


        all_targets.extend(
            targets.detach()
            .cpu()
            .tolist()
        )

        all_predictions.extend(
            predictions.detach()
            .cpu()
            .tolist()
        )


        if (
            batch_idx % 500 == 0
        ):

            print(
                f"    Batch "
                f"{batch_idx:,}/"
                f"{len(loader):,}"
            )


    epoch_loss = (
        running_loss
        / len(loader.dataset)
    )

    epoch_acc = accuracy_score(
        all_targets,
        all_predictions
    )

    epoch_macro_f1 = f1_score(
        all_targets,
        all_predictions,
        average="macro",
        zero_division=0,
    )


    return (
        epoch_loss,
        epoch_acc,
        epoch_macro_f1,
    )


# ==========================================================================================
# 17. VALIDATION
# ==========================================================================================

@torch.no_grad()
def validate(
    model,
    loader,
    criterion,
):

    model.eval()

    running_loss = 0.0

    all_targets = []
    all_predictions = []


    for images, targets in loader:

        images = images.to(
            DEVICE,
            non_blocking=True
        )

        targets = targets.to(
            DEVICE,
            non_blocking=True
        )


        if USE_AMP:

            with torch.amp.autocast(
                device_type="cuda",
                dtype=torch.float16,
            ):

                logits = model(
                    images
                )

                loss = criterion(
                    logits,
                    targets
                )

        else:

            logits = model(
                images
            )

            loss = criterion(
                logits,
                targets
            )


        running_loss += (
            loss.item()
            * images.size(0)
        )


        predictions = torch.argmax(
            logits,
            dim=1
        )


        all_targets.extend(
            targets.cpu().tolist()
        )

        all_predictions.extend(
            predictions.cpu().tolist()
        )


    val_loss = (
        running_loss
        / len(loader.dataset)
    )

    val_acc = accuracy_score(
        all_targets,
        all_predictions
    )

    macro_f1 = f1_score(
        all_targets,
        all_predictions,
        average="macro",
        zero_division=0,
    )

    weighted_f1 = f1_score(
        all_targets,
        all_predictions,
        average="weighted",
        zero_division=0,
    )


    return {
        "loss":
            float(val_loss),

        "accuracy":
            float(val_acc),

        "macro_f1":
            float(macro_f1),

        "weighted_f1":
            float(weighted_f1),

        "targets":
            all_targets,

        "predictions":
            all_predictions,
    }


# ==========================================================================================
# 18. TRAINING LOOP
# ==========================================================================================

best_macro_f1 = -1.0
best_epoch = 0

epochs_without_improvement = 0

history = []

training_start = time.time()


for epoch in range(
    1,
    NUM_EPOCHS + 1
):

    epoch_start = time.time()


    print(
        "\n"
        + "=" * 100
    )

    print(
        f"EPOCH "
        f"{epoch}/{NUM_EPOCHS}"
    )

    print(
        "=" * 100
    )


    # ------------------------------------------------------------------
    # Train
    # ------------------------------------------------------------------

    (
        train_loss,
        train_acc,
        train_macro_f1,
    ) = train_one_epoch(
        model,
        train_loader,
        optimizer,
        criterion,
    )


    # ------------------------------------------------------------------
    # Validate
    # ------------------------------------------------------------------

    val_result = validate(
        model,
        val_loader,
        criterion,
    )


    val_loss = (
        val_result[
            "loss"
        ]
    )

    val_acc = (
        val_result[
            "accuracy"
        ]
    )

    val_macro_f1 = (
        val_result[
            "macro_f1"
        ]
    )

    val_weighted_f1 = (
        val_result[
            "weighted_f1"
        ]
    )


    # ------------------------------------------------------------------
    # Scheduler
    # ------------------------------------------------------------------

    scheduler.step(
        val_macro_f1
    )


    current_lr = (
        optimizer.param_groups[0]["lr"]
    )


    epoch_minutes = (
        time.time()
        - epoch_start
    ) / 60.0


    print(
        f"\nTrain loss     : "
        f"{train_loss:.4f}"
    )

    print(
        f"Train accuracy : "
        f"{train_acc * 100:.2f}%"
    )

    print(
        f"Train Macro-F1 : "
        f"{train_macro_f1:.4f}"
    )

    print(
        f"\nVal loss       : "
        f"{val_loss:.4f}"
    )

    print(
        f"Val accuracy   : "
        f"{val_acc * 100:.2f}%"
    )

    print(
        f"Val Macro-F1   : "
        f"{val_macro_f1:.4f}"
    )

    print(
        f"Val Weighted-F1: "
        f"{val_weighted_f1:.4f}"
    )

    print(
        f"Learning rate  : "
        f"{current_lr:.8f}"
    )

    print(
        f"Epoch time     : "
        f"{epoch_minutes:.2f} min"
    )


    history.append(
        {
            "epoch":
                epoch,

            "train_loss":
                float(
                    train_loss
                ),

            "train_accuracy":
                float(
                    train_acc
                ),

            "train_macro_f1":
                float(
                    train_macro_f1
                ),

            "val_loss":
                float(
                    val_loss
                ),

            "val_accuracy":
                float(
                    val_acc
                ),

            "val_macro_f1":
                float(
                    val_macro_f1
                ),

            "val_weighted_f1":
                float(
                    val_weighted_f1
                ),

            "learning_rate":
                float(
                    current_lr
                ),
        }
    )


    # ------------------------------------------------------------------
    # Save best checkpoint based on validation Macro-F1
    # ------------------------------------------------------------------

    if (
        val_macro_f1
        > best_macro_f1
        + MIN_DELTA
    ):

        best_macro_f1 = (
            val_macro_f1
        )

        best_epoch = epoch

        epochs_without_improvement = 0


        torch.save(
            {
                "experiment":
                    "E19-A",

                "architecture":
                    "ConvNeXt-Small",

                "epoch":
                    epoch,

                "model_state_dict":
                    model.state_dict(),

                "optimizer_state_dict":
                    optimizer.state_dict(),

                "val_macro_f1":
                    val_macro_f1,

                "val_accuracy":
                    val_acc,

                "val_weighted_f1":
                    val_weighted_f1,

                "classes":
                    EXPECTED_CLASSES,

                "class_to_idx":
                    train_dataset.class_to_idx,

                "image_size":
                    IMAGE_SIZE,

                "class_weights":
                    class_weights.tolist(),

                "batch_size":
                    BATCH_SIZE,

                "learning_rate":
                    LEARNING_RATE,

                "weight_decay":
                    WEIGHT_DECAY,

                "seed":
                    SEED,
            },

            CHECKPOINT_PATH
        )


        print(
            "\n*** NEW BEST CHECKPOINT ***"
        )

        print(
            f"Best validation Macro-F1: "
            f"{best_macro_f1:.4f}"
        )

        print(
            f"Saved to:\n"
            f"{CHECKPOINT_PATH}"
        )


    else:

        epochs_without_improvement += 1

        print(
            f"\nNo Macro-F1 improvement "
            f"for "
            f"{epochs_without_improvement}/"
            f"{PATIENCE} epochs."
        )


    # ------------------------------------------------------------------
    # Save running history
    # ------------------------------------------------------------------

    with open(
        HISTORY_PATH,
        "w",
        encoding="utf-8"
    ) as f:

        json.dump(
            history,
            f,
            indent=2
        )


    # ------------------------------------------------------------------
    # Early stopping
    # ------------------------------------------------------------------

    if (
        epochs_without_improvement
        >= PATIENCE
    ):

        print(
            "\nEarly stopping triggered."
        )

        break


# ==========================================================================================
# 19. LOAD BEST CHECKPOINT
# ==========================================================================================

training_minutes = (
    time.time()
    - training_start
) / 60.0


print(
    "\n"
    + "=" * 100
)

print(
    "LOADING BEST E19-A CHECKPOINT"
)

print(
    "=" * 100
)


checkpoint = torch.load(
    CHECKPOINT_PATH,
    map_location=DEVICE,
    weights_only=False,
)


model.load_state_dict(
    checkpoint[
        "model_state_dict"
    ]
)


print(
    f"Best epoch    : "
    f"{checkpoint['epoch']}"
)

print(
    f"Best Macro-F1 : "
    f"{checkpoint['val_macro_f1']:.4f}"
)


# ==========================================================================================
# 20. FINAL STANDALONE VALIDATION
# ==========================================================================================

final_val = validate(
    model,
    val_loader,
    criterion,
)


print(
    "\n"
    + "=" * 100
)

print(
    "E19-A BEST CHECKPOINT — VALIDATION RESULTS"
)

print(
    "=" * 100
)


print(
    f"\nAccuracy    : "
    f"{final_val['accuracy'] * 100:.2f}%"
)

print(
    f"Macro-F1    : "
    f"{final_val['macro_f1']:.4f}"
)

print(
    f"Weighted-F1 : "
    f"{final_val['weighted_f1']:.4f}"
)


print(
    "\nClassification report:\n"
)


report = classification_report(
    final_val[
        "targets"
    ],
    final_val[
        "predictions"
    ],
    target_names=EXPECTED_CLASSES,
    digits=4,
    zero_division=0,
)

print(
    report
)


# ==========================================================================================
# 21. SAVE SUMMARY
# ==========================================================================================

summary = f"""
====================================================================================================
E19-A — CONVNEXT-SMALL HARD-CLASS SPECIALIST TRAINING
====================================================================================================

Architecture:
ConvNeXt-Small

Pretraining:
ImageNet DEFAULT weights

Hard classes:
{EXPECTED_CLASSES}

Dataset:
{E18_CROP_ROOT}

Image size:
{IMAGE_SIZE}

Batch size:
{BATCH_SIZE}

Class-weighted CrossEntropyLoss:
Yes

Optimizer:
AdamW

Learning rate:
{LEARNING_RATE}

Weight decay:
{WEIGHT_DECAY}

Best checkpoint criterion:
Validation Macro-F1

Best epoch:
{checkpoint['epoch']}

Validation accuracy:
{final_val['accuracy']:.6f}

Validation Macro-F1:
{final_val['macro_f1']:.6f}

Validation Weighted-F1:
{final_val['weighted_f1']:.6f}

Total training time:
{training_minutes:.2f} minutes

Checkpoint:
{CHECKPOINT_PATH}

Classification report:

{report}
"""


with open(
    SUMMARY_PATH,
    "w",
    encoding="utf-8"
) as f:

    f.write(
        summary
    )


print(
    "\n"
    + "=" * 100
)

print(
    "E19-A COMPLETE"
)

print(
    "=" * 100
)

print(
    f"\nCheckpoint:\n"
    f"{CHECKPOINT_PATH}"
)

print(
    f"\nSummary:\n"
    f"{SUMMARY_PATH}"
)

E19-A — CONVNEXT-SMALL HARD-CLASS SPECIALIST TRAINING

PyTorch : 2.13.0+cu126
Device  : cuda
GPU     : NVIDIA GeForce RTX 3050 Ti Laptop GPU

Train : C:\Users\varda\Documents\_My Computer\COMMON_space\Learning\Upgrad_EPGP_MS\upgrad_IIITB_MS_course\MS_LJMU_Material\Topic Data\SortWaste\dataset\dataset\yolo11m_convnext_hardclass_crops_E18\train
Val   : C:\Users\varda\Documents\_My Computer\COMMON_space\Learning\Upgrad_EPGP_MS\upgrad_IIITB_MS_course\MS_LJMU_Material\Topic Data\SortWaste\dataset\dataset\yolo11m_convnext_hardclass_crops_E18\val
Output: C:\Users\varda\Documents\_My Computer\COMMON_space\Learning\Upgrad_EPGP_MS\upgrad_IIITB_MS_course\MS_LJMU_Material\Thesis_Code\runs\sortwaste\E19A_convnext_small_hardclass

Class mapping:
{'mixed_plastic_rigid': 0, 'mixed_plastic_soft': 1, 'non_plastic': 2, 'pet_oil': 3}

Train samples : 19,378
Val samples   : 3,343

Training class distribution:
mixed_plastic_rigid      : 7,052
mixed_plastic_soft       : 9,057
non_plastic              : 2,468

100%|██████████| 192M/192M [00:08<00:00, 22.6MB/s] 


ConvNeXt-Small loaded.
Total parameters     : 49,457,764
Trainable parameters : 49,457,764
Classifier input dim : 768

EPOCH 1/30
    Batch 500/4,845
    Batch 1,000/4,845
    Batch 1,500/4,845
    Batch 2,000/4,845
    Batch 2,500/4,845
    Batch 3,000/4,845
    Batch 3,500/4,845
    Batch 4,000/4,845
    Batch 4,500/4,845

Train loss     : 0.4852
Train accuracy : 78.86%
Train Macro-F1 : 0.7871

Val loss       : 0.7598
Val accuracy   : 73.62%
Val Macro-F1   : 0.7697
Val Weighted-F1: 0.7349
Learning rate  : 0.00010000
Epoch time     : 21.65 min

*** NEW BEST CHECKPOINT ***
Best validation Macro-F1: 0.7697
Saved to:
C:\Users\varda\Documents\_My Computer\COMMON_space\Learning\Upgrad_EPGP_MS\upgrad_IIITB_MS_course\MS_LJMU_Material\Thesis_Code\runs\sortwaste\E19A_convnext_small_hardclass\E19A_ConvNeXtSmall_best.pth

EPOCH 2/30
    Batch 500/4,845
    Batch 1,000/4,845
    Batch 1,500/4,845
    Batch 2,000/4,845
    Batch 2,500/4,845
    Batch 3,000/4,845
    Batch 3,500/4,845
    Batch 4,0

In [ ]:
# E19-B — YOLO11m + MOBILENETV3-LARGE + CONVNEXT-SMALL - CLASS-SELECTIVE HARD-CLASS SPECIALIST REFINEMENT
# ==========================================================================================
#
# Comparison target:
#
#   E18-E:
#       YOLO11m
#       -> E16-A MobileNet gating
#       -> ConvNeXt-Tiny specialist
#       -> E18-D2 class-selective routing
#       -> E18-E specialist gates
#
#   E19-B:
#       Exact same pipeline,
#       BUT ConvNeXt-Tiny -> ConvNeXt-Small.
#
# No gate tuning is performed here.
#
# This is deliberately a clean capacity comparison.
# ==========================================================================================


import json
import time
from pathlib import Path
from collections import Counter

import numpy as np
from PIL import Image

import torch
import torch.nn as nn
from torchvision import transforms

from torchvision.models import (
    mobilenet_v3_large,
    MobileNet_V3_Large_Weights,
    convnext_small,
)

from ultralytics import YOLO

from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval


# ==========================================================================================
# 1. DEVICE
# ==========================================================================================

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)


print(
    "=" * 100
)

print(
    "E19-B — YOLO11m + MOBILENET + CONVNEXT-SMALL END-TO-END"
)

print(
    "=" * 100
)


print(
    f"\nDevice : {DEVICE}"
)

if DEVICE.type == "cuda":

    print(
        f"GPU    : "
        f"{torch.cuda.get_device_name(0)}"
    )


# ==========================================================================================
# 2. PATHS
# ==========================================================================================

BASE = Path(
    r"C:\Users\varda\Documents\_My Computer\COMMON_space\Learning"
    r"\Upgrad_EPGP_MS\upgrad_IIITB_MS_course\MS_LJMU_Material"
)


DATASET_ROOT = (
    BASE
    / "Topic Data"
    / "SortWaste"
    / "dataset"
    / "dataset"
)


VAL_ROOT = (
    DATASET_ROOT
    / "splited_all_dataset_coco"
    / "val"
)


VAL_IMAGES = (
    VAL_ROOT
    / "images"
)


VAL_COCO_JSON = (
    VAL_ROOT
    / "annotations"
    / "val_coco.json"
)


# ------------------------------------------------------------------
# E12 YOLO11m checkpoint
# ------------------------------------------------------------------

YOLO_PATH = (
    BASE
    / "Thesis_Code"
    / "runs"
    / "sortwaste"
    / "E12_yolo11m_7class_aug_classbalance_640"
    / "weights"
    / "best.pt"
)


# ------------------------------------------------------------------
# E3Y-B / E8 MobileNet checkpoint
# ------------------------------------------------------------------

MOBILENET_PATH = (
    DATASET_ROOT
    / "yolo_mobilenet_crops_E3Y"
    / "mobilenet_results"
    / "E3Y_B_class_weighted"
    / "E3Y_B_MobileNetV3Large_best.pth"
)


# ------------------------------------------------------------------
# E19-A ConvNeXt-Small checkpoint
# ------------------------------------------------------------------

CONVNEXT_PATH = (
    BASE
    / "Thesis_Code"
    / "runs"
    / "sortwaste"
    / "E19A_convnext_small_hardclass"
    / "E19A_ConvNeXtSmall_best.pth"
)


# ------------------------------------------------------------------
# Output
# ------------------------------------------------------------------

OUTPUT_DIR = (
    BASE
    / "Thesis_Code"
    / "runs"
    / "sortwaste"
    / "E19B_yolo11m_convnext_small_endtoend"
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)


GT_7CLASS_PATH = (
    OUTPUT_DIR
    / "E19B_val_gt_7class.json"
)

PREDICTIONS_PATH = (
    OUTPUT_DIR
    / "E19B_predictions.json"
)

RESULTS_PATH = (
    OUTPUT_DIR
    / "E19B_results.json"
)

SUMMARY_PATH = (
    OUTPUT_DIR
    / "E19B_summary.txt"
)


for required_path in [
    VAL_IMAGES,
    VAL_COCO_JSON,
    YOLO_PATH,
    MOBILENET_PATH,
    CONVNEXT_PATH,
]:

    assert required_path.exists(), (
        f"Missing:\n{required_path}"
    )


# ==========================================================================================
# 3. 7-CLASS TAXONOMY
# ==========================================================================================

CLASS_NAMES = [
    "ecal",
    "hdpe",
    "mixed_plastic_rigid",
    "mixed_plastic_soft",
    "non_plastic",
    "pet",
    "pet_oil",
]


# Global zero-based indices:
#
# 0 ecal
# 1 hdpe
# 2 mixed rigid
# 3 mixed soft
# 4 nonplastic
# 5 pet
# 6 pet oil

ECAL = 0
HDPE = 1
MIXED_RIGID = 2
MIXED_SOFT = 3
NON_PLASTIC = 4
PET = 5
PET_OIL = 6


# ==========================================================================================
# 4. ORIGINAL COCO -> 7-CLASS COCO
#
# Original COCO:
#
# 1 PET
# 2 HDPE
# 3 Mixed Soft
# 4 ECAL
# 5 Metal
# 6 Cardboard
# 7 Mixed Rigid
# 8 PET Oil
#
# New category IDs are 1..7.
# ==========================================================================================

ORIGINAL_COCO_TO_NEW = {
    1: 6,  # PET
    2: 2,  # HDPE
    3: 4,  # Mixed Soft
    4: 1,  # ECAL
    5: 5,  # Metal -> nonplastic
    6: 5,  # Cardboard -> nonplastic
    7: 3,  # Mixed Rigid
    8: 7,  # PET Oil
}


# ==========================================================================================
# 5. BUILD 7-CLASS COCO GT
# ==========================================================================================

print(
    "\nPreparing 7-class validation ground truth..."
)


with open(
    VAL_COCO_JSON,
    "r",
    encoding="utf-8"
) as f:

    original_gt = json.load(
        f
    )


gt_7class = {
    "images":
        original_gt[
            "images"
        ],

    "annotations":
        [],

    "categories":
        [
            {
                "id":
                    i + 1,

                "name":
                    class_name,
            }

            for i, class_name
            in enumerate(
                CLASS_NAMES
            )
        ],
}


for ann in original_gt[
    "annotations"
]:

    new_ann = ann.copy()

    new_ann[
        "category_id"
    ] = ORIGINAL_COCO_TO_NEW[
        int(
            ann[
                "category_id"
            ]
        )
    ]

    gt_7class[
        "annotations"
    ].append(
        new_ann
    )


with open(
    GT_7CLASS_PATH,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        gt_7class,
        f
    )


print(
    f"Images : "
    f"{len(gt_7class['images']):,}"
)

print(
    f"GT     : "
    f"{len(gt_7class['annotations']):,}"
)


# ==========================================================================================
# 6. IMAGE ID LOOKUP
# ==========================================================================================

filename_to_image_id = {

    item[
        "file_name"
    ]:
        int(
            item[
                "id"
            ]
        )

    for item in (
        gt_7class[
            "images"
        ]
    )
}


# ==========================================================================================
# 7. CLASSIFIER TRANSFORM
# ==========================================================================================

IMAGE_SIZE = 224


classifier_transform = transforms.Compose(
    [
        transforms.Resize(
            (
                IMAGE_SIZE,
                IMAGE_SIZE
            )
        ),

        transforms.ToTensor(),

        transforms.Normalize(
            mean=[
                0.485,
                0.456,
                0.406
            ],

            std=[
                0.229,
                0.224,
                0.225
            ],
        ),
    ]
)


# ==========================================================================================
# 8. LOAD MOBILENET
# ==========================================================================================

print(
    "\nLoading MobileNet..."
)


mobilenet = mobilenet_v3_large(
    weights=None
)


mn_in_features = (
    mobilenet
    .classifier[3]
    .in_features
)


mobilenet.classifier[3] = nn.Linear(
    mn_in_features,
    7
)


mn_checkpoint = torch.load(
    MOBILENET_PATH,
    map_location=DEVICE,
    weights_only=False,
)


# Support both direct state_dict and checkpoint dictionary.
if (
    isinstance(
        mn_checkpoint,
        dict
    )
    and
    "model_state_dict"
    in mn_checkpoint
):

    mn_state = (
        mn_checkpoint[
            "model_state_dict"
        ]
    )

else:

    mn_state = (
        mn_checkpoint
    )


mobilenet.load_state_dict(
    mn_state
)

mobilenet = mobilenet.to(
    DEVICE
)

mobilenet.eval()


print(
    "MobileNet loaded."
)


# ==========================================================================================
# 9. LOAD CONVNEXT-SMALL
# ==========================================================================================

print(
    "\nLoading ConvNeXt-Small..."
)


convnext = convnext_small(
    weights=None
)


conv_in_features = (
    convnext.classifier[2].in_features
)


convnext.classifier[2] = nn.Linear(
    conv_in_features,
    4
)


conv_checkpoint = torch.load(
    CONVNEXT_PATH,
    map_location=DEVICE,
    weights_only=False,
)


if (
    isinstance(
        conv_checkpoint,
        dict
    )
    and
    "model_state_dict"
    in conv_checkpoint
):

    conv_state = (
        conv_checkpoint[
            "model_state_dict"
        ]
    )

else:

    conv_state = (
        conv_checkpoint
    )


convnext.load_state_dict(
    conv_state
)

convnext = convnext.to(
    DEVICE
)

convnext.eval()


print(
    "ConvNeXt-Small loaded."
)


# ==========================================================================================
# 10. CONVNEXT LOCAL -> GLOBAL CLASS MAPPING
#
# ConvNeXt class order:
#
# 0 mixed_plastic_rigid
# 1 mixed_plastic_soft
# 2 non_plastic
# 3 pet_oil
# ==========================================================================================

CONV_LOCAL_TO_GLOBAL = {
    0: MIXED_RIGID,
    1: MIXED_SOFT,
    2: NON_PLASTIC,
    3: PET_OIL,
}


# ==========================================================================================
# 11. E16-A MOBILENET CLASS-SPECIFIC GATES
# ==========================================================================================

MOBILENET_GATES = {
    ECAL:
        0.99,

    HDPE:
        0.98,

    MIXED_RIGID:
        0.98,

    MIXED_SOFT:
        0.94,

    NON_PLASTIC:
        0.94,

    PET:
        0.99,

    PET_OIL:
        0.99,
}


# ==========================================================================================
# 12. E18-D2 ROUTING
#
# ConvNeXt is permitted to supply ONLY:
#
#   mixed soft
#   nonplastic
#   PET Oil
#
# Mixed Rigid refinement remains DISABLED.
# ==========================================================================================

ALLOWED_CONVNEXT_CLASSES = {
    MIXED_SOFT,
    NON_PLASTIC,
    PET_OIL,
}


# ==========================================================================================
# 13. FIXED E18-E CONVNEXT GATES
#
# Important:
#
# These are NOT tuned for ConvNeXt-Small.
#
# We intentionally use the E18-E Tiny gates unchanged so that
# E19 first measures the architecture effect cleanly.
# ==========================================================================================

CONVNEXT_GATES = {
    MIXED_SOFT:
        0.90,

    NON_PLASTIC:
        0.90,

    PET_OIL:
        0.94,
}


# ==========================================================================================
# 14. FIXED CONFIDENCE FUSION
# ==========================================================================================

ALPHA = 0.70


# ==========================================================================================
# 15. LOAD YOLO11m
# ==========================================================================================

print(
    "\nLoading YOLO11m..."
)


yolo = YOLO(
    str(
        YOLO_PATH
    )
)


print(
    "YOLO11m loaded."
)


# ==========================================================================================
# 16. INFERENCE PARAMETERS
# ==========================================================================================

YOLO_CONF = 0.001

YOLO_NMS_IOU = 0.60

YOLO_MAX_DET = 100

YOLO_IMGSZ = 640


# ==========================================================================================
# 17. PREDICTION COUNTERS
# ==========================================================================================

predictions = []

source_counts = Counter()

conv_proposed_counts = Counter()

conv_accepted_counts = Counter()

conv_rejected_counts = Counter()


total_yolo_detections = 0

total_valid_crops = 0

invalid_crops = 0

conv_candidates = 0


# ==========================================================================================
# 18. END-TO-END INFERENCE
# ==========================================================================================

image_records = sorted(
    gt_7class[
        "images"
    ],
    key=lambda x: int(
        x[
            "id"
        ]
    )
)


start_time = time.time()


for image_number, image_record in enumerate(
    image_records,
    start=1
):

    image_id = int(
        image_record[
            "id"
        ]
    )

    filename = (
        image_record[
            "file_name"
        ]
    )


    image_path = (
        VAL_IMAGES
        / filename
    )


    if not image_path.exists():

        raise FileNotFoundError(
            f"Missing validation image:\n"
            f"{image_path}"
        )


    pil_image = Image.open(
        image_path
    ).convert(
        "RGB"
    )


    image_width, image_height = (
        pil_image.size
    )


    # ------------------------------------------------------------------
    # YOLO detection
    # ------------------------------------------------------------------

    yolo_result = yolo.predict(
        source=str(
            image_path
        ),
        imgsz=YOLO_IMGSZ,
        conf=YOLO_CONF,
        iou=YOLO_NMS_IOU,
        max_det=YOLO_MAX_DET,
        verbose=False,
        device=0 if DEVICE.type == "cuda" else "cpu",
    )[0]


    boxes = (
        yolo_result.boxes
    )


    if boxes is None:

        continue


    xyxy_array = (
        boxes.xyxy
        .detach()
        .cpu()
        .numpy()
    )

    confidence_array = (
        boxes.conf
        .detach()
        .cpu()
        .numpy()
    )

    yolo_class_array = (
        boxes.cls
        .detach()
        .cpu()
        .numpy()
        .astype(int)
    )


    total_yolo_detections += (
        len(
            xyxy_array
        )
    )


    # ------------------------------------------------------------------
    # Process every YOLO detection
    # ------------------------------------------------------------------

    for (
        bbox_xyxy,
        yolo_conf,
        yolo_idx,
    ) in zip(
        xyxy_array,
        confidence_array,
        yolo_class_array,
    ):

        x1, y1, x2, y2 = [
            float(v)
            for v in bbox_xyxy
        ]


        # Clamp crop coordinates.
        crop_x1 = max(
            0,
            int(
                np.floor(
                    x1
                )
            )
        )

        crop_y1 = max(
            0,
            int(
                np.floor(
                    y1
                )
            )
        )

        crop_x2 = min(
            image_width,
            int(
                np.ceil(
                    x2
                )
            )
        )

        crop_y2 = min(
            image_height,
            int(
                np.ceil(
                    y2
                )
            )
        )


        if (
            crop_x2 <= crop_x1
            or
            crop_y2 <= crop_y1
        ):

            invalid_crops += 1
            continue


        crop = pil_image.crop(
            (
                crop_x1,
                crop_y1,
                crop_x2,
                crop_y2,
            )
        )


        input_tensor = (
            classifier_transform(
                crop
            )
            .unsqueeze(0)
            .to(
                DEVICE
            )
        )


        total_valid_crops += 1


        # ==================================================================
        # 18A. MOBILENET — E16-A
        # ==================================================================

        with torch.no_grad():

            if DEVICE.type == "cuda":

                with torch.amp.autocast(
                    device_type="cuda",
                    dtype=torch.float16,
                ):

                    mn_logits = mobilenet(
                        input_tensor
                    )

            else:

                mn_logits = mobilenet(
                    input_tensor
                )


        mn_probs = torch.softmax(
            mn_logits.float(),
            dim=1
        )[0]


        mn_top_prob, mn_top_idx = torch.max(
            mn_probs,
            dim=0
        )


        mn_top_prob = float(
            mn_top_prob.item()
        )

        mn_top_idx = int(
            mn_top_idx.item()
        )


        # --------------------------------------------------------------
        # E16-A class gating
        # --------------------------------------------------------------

        mn_gate = (
            MOBILENET_GATES[
                mn_top_idx
            ]
        )


        if (
            mn_top_prob
            >= mn_gate
        ):

            e16_final_idx = (
                mn_top_idx
            )

            e16_source = (
                "mobilenet"
            )

        else:

            e16_final_idx = int(
                yolo_idx
            )

            e16_source = (
                "yolo"
            )


        # Initial final class = E16-A decision.
        final_idx = (
            e16_final_idx
        )

        final_source = (
            e16_source
        )


        classifier_prob = float(
            mn_probs[
                final_idx
            ].item()
        )


        # ==================================================================
        # 18B. CONVNEXT-SMALL SPECIALIST
        #
        # Preserve E18 logic:
        # ConvNeXt is consulted only when the E16-A final class lies in
        # the original four hard-class set.
        # ==================================================================

        E16_HARD_CLASSES = {
            MIXED_RIGID,
            MIXED_SOFT,
            NON_PLASTIC,
            PET_OIL,
        }


        if (
            e16_final_idx
            in E16_HARD_CLASSES
        ):

            conv_candidates += 1


            with torch.no_grad():

                if DEVICE.type == "cuda":

                    with torch.amp.autocast(
                        device_type="cuda",
                        dtype=torch.float16,
                    ):

                        conv_logits = (
                            convnext(
                                input_tensor
                            )
                        )

                else:

                    conv_logits = (
                        convnext(
                            input_tensor
                        )
                    )


            conv_probs = torch.softmax(
                conv_logits.float(),
                dim=1
            )[0]


            (
                conv_top_prob,
                conv_local_idx,
            ) = torch.max(
                conv_probs,
                dim=0
            )


            conv_top_prob = float(
                conv_top_prob.item()
            )

            conv_local_idx = int(
                conv_local_idx.item()
            )


            conv_global_idx = (
                CONV_LOCAL_TO_GLOBAL[
                    conv_local_idx
                ]
            )


            conv_proposed_counts[
                CLASS_NAMES[
                    conv_global_idx
                ]
            ] += 1


            # --------------------------------------------------------------
            # E18-D2:
            #
            # Mixed Rigid ConvNeXt proposals are not allowed.
            # --------------------------------------------------------------

            if (
                conv_global_idx
                in ALLOWED_CONVNEXT_CLASSES
            ):

                conv_gate = (
                    CONVNEXT_GATES[
                        conv_global_idx
                    ]
                )


                if (
                    conv_top_prob
                    >= conv_gate
                ):

                    final_idx = (
                        conv_global_idx
                    )

                    final_source = (
                        "convnext"
                    )

                    classifier_prob = (
                        conv_top_prob
                    )


                    conv_accepted_counts[
                        CLASS_NAMES[
                            conv_global_idx
                        ]
                    ] += 1


                else:

                    conv_rejected_counts[
                        CLASS_NAMES[
                            conv_global_idx
                        ]
                    ] += 1


        # ==================================================================
        # 18C. FINAL CONFIDENCE FUSION
        #
        # S = C_YOLO^0.70 * P_classifier(final class)^0.30
        # ==================================================================

        classifier_prob = max(
            classifier_prob,
            1e-12
        )


        final_score = (
            float(
                yolo_conf
            )
            ** ALPHA
        ) * (
            classifier_prob
            ** (
                1.0
                - ALPHA
            )
        )


        source_counts[
            final_source
        ] += 1


        # ==================================================================
        # COCO prediction
        # ==================================================================

        bbox_width = (
            x2 - x1
        )

        bbox_height = (
            y2 - y1
        )


        predictions.append(
            {
                "image_id":
                    image_id,

                "category_id":
                    int(
                        final_idx + 1
                    ),

                "bbox":
                    [
                        float(
                            x1
                        ),

                        float(
                            y1
                        ),

                        float(
                            bbox_width
                        ),

                        float(
                            bbox_height
                        ),
                    ],

                "score":
                    float(
                        final_score
                    ),
            }
        )


    if (
        image_number % 25 == 0
    ):

        print(
            f"Processing "
            f"{image_number}/"
            f"{len(image_records)}"
        )


inference_minutes = (
    time.time()
    - start_time
) / 60.0


print(
    "\nInference complete."
)

print(
    f"YOLO detections       : "
    f"{total_yolo_detections:,}"
)

print(
    f"Valid crops           : "
    f"{total_valid_crops:,}"
)

print(
    f"Invalid crops         : "
    f"{invalid_crops:,}"
)

print(
    f"Final predictions     : "
    f"{len(predictions):,}"
)

print(
    f"ConvNeXt candidates   : "
    f"{conv_candidates:,}"
)

print(
    f"Inference time        : "
    f"{inference_minutes:.2f} min"
)


# ==========================================================================================
# 19. SAVE PREDICTIONS
# ==========================================================================================

with open(
    PREDICTIONS_PATH,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        predictions,
        f
    )


# ==========================================================================================
# 20. COCO EVALUATION
# ==========================================================================================

print(
    "\n"
    + "=" * 100
)

print(
    "E19-B COCO EVALUATION"
)

print(
    "=" * 100
)


coco_gt = COCO(
    str(
        GT_7CLASS_PATH
    )
)

coco_dt = coco_gt.loadRes(
    predictions
)


evaluator = COCOeval(
    coco_gt,
    coco_dt,
    "bbox"
)


evaluator.params.maxDets = [
    1,
    10,
    100
]


evaluator.evaluate()
evaluator.accumulate()
evaluator.summarize()


AP50_95 = float(
    evaluator.stats[0]
)

AP50 = float(
    evaluator.stats[1]
)

AP75 = float(
    evaluator.stats[2]
)

AR100 = float(
    evaluator.stats[8]
)


# ==========================================================================================
# 21. PER-CLASS AP
# ==========================================================================================

precision = (
    evaluator.eval[
        "precision"
    ]
)

iou_thresholds = (
    evaluator.params.iouThrs
)


ap50_index = int(
    np.where(
        np.isclose(
            iou_thresholds,
            0.50
        )
    )[0][0]
)


class_metrics = {}


print(
    "\n"
    + "=" * 100
)

print(
    "CLASS-WISE RESULTS"
)

print(
    "=" * 100
)


print(
    f"\n{'Class':28s}"
    f"{'AP50':>14s}"
    f"{'AP50-95':>14s}"
)


for class_idx, class_name in enumerate(
    CLASS_NAMES
):

    values = precision[
        :,
        :,
        class_idx,
        0,
        -1
    ]

    values = (
        values[
            values > -1
        ]
    )


    class_ap = (
        float(
            np.mean(
                values
            )
        )
        if values.size
        else float(
            "nan"
        )
    )


    values50 = precision[
        ap50_index,
        :,
        class_idx,
        0,
        -1
    ]

    values50 = (
        values50[
            values50 > -1
        ]
    )


    class_ap50 = (
        float(
            np.mean(
                values50
            )
        )
        if values50.size
        else float(
            "nan"
        )
    )


    class_metrics[
        class_name
    ] = {
        "AP50":
            class_ap50,

        "AP50_95":
            class_ap,
    }


    print(
        f"{class_name:28s}"
        f"{class_ap50 * 100:14.2f}"
        f"{class_ap * 100:14.2f}"
    )


# ==========================================================================================
# 22. COMPARE AGAINST E18-E
# ==========================================================================================

E18E_AP = 0.457263
E18E_AP50 = 0.611238
E18E_AP75 = 0.519444
E18E_AR100 = 0.614629


print(
    "\n"
    + "=" * 100
)

print(
    "E18-E CONVNEXT-TINY vs E19-B CONVNEXT-SMALL"
)

print(
    "=" * 100
)


print(
    f"\n{'Metric':12s}"
    f"{'E18-E Tiny':>16s}"
    f"{'E19-B Small':>16s}"
    f"{'Delta':>14s}"
)


comparison = [
    (
        "AP50-95",
        E18E_AP,
        AP50_95,
    ),

    (
        "AP50",
        E18E_AP50,
        AP50,
    ),

    (
        "AP75",
        E18E_AP75,
        AP75,
    ),

    (
        "AR100",
        E18E_AR100,
        AR100,
    ),
]


for metric, old, new in comparison:

    print(
        f"{metric:12s}"
        f"{old * 100:16.4f}"
        f"{new * 100:16.4f}"
        f"{(new-old) * 100:+14.4f}"
    )


# ==========================================================================================
# 23. SOURCE COUNTS
# ==========================================================================================

print(
    "\n"
    + "=" * 100
)

print(
    "FINAL CLASS SOURCE COUNTS"
)

print(
    "=" * 100
)


for source in [
    "yolo",
    "mobilenet",
    "convnext",
]:

    print(
        f"{source:15s}: "
        f"{source_counts.get(source, 0):,}"
    )


print(
    "\nConvNeXt proposed classes:"
)


for class_name in [
    "mixed_plastic_rigid",
    "mixed_plastic_soft",
    "non_plastic",
    "pet_oil",
]:

    print(
        f"{class_name:25s}: "
        f"{conv_proposed_counts.get(class_name, 0):,}"
    )


print(
    "\nConvNeXt accepted classes:"
)


for class_name in [
    "mixed_plastic_soft",
    "non_plastic",
    "pet_oil",
]:

    print(
        f"{class_name:25s}: "
        f"{conv_accepted_counts.get(class_name, 0):,}"
    )


# ==========================================================================================
# 24. SAVE RESULTS
# ==========================================================================================

results = {

    "experiment":
        "E19-B",

    "description":
        (
            "YOLO11m + MobileNetV3-Large "
            "+ ConvNeXt-Small class-selective "
            "hard-class refinement"
        ),

    "convnext_architecture":
        "ConvNeXt-Small",

    "convnext_checkpoint":
        str(
            CONVNEXT_PATH
        ),

    "routing_classes": [
        "mixed_plastic_soft",
        "non_plastic",
        "pet_oil",
    ],

    "mixed_plastic_rigid_routing":
        "disabled",

    "mobilenet_gates":
        {
            CLASS_NAMES[k]:
                float(v)
            for k, v
            in MOBILENET_GATES.items()
        },

    "convnext_gates":
        {
            CLASS_NAMES[k]:
                float(v)
            for k, v
            in CONVNEXT_GATES.items()
        },

    "alpha":
        ALPHA,

    "metrics": {
        "AP50_95":
            AP50_95,

        "AP50":
            AP50,

        "AP75":
            AP75,

        "AR100":
            AR100,
    },

    "delta_vs_E18E": {
        "AP50_95":
            AP50_95
            - E18E_AP,

        "AP50":
            AP50
            - E18E_AP50,

        "AP75":
            AP75
            - E18E_AP75,

        "AR100":
            AR100
            - E18E_AR100,
    },

    "class_metrics":
        class_metrics,

    "counts": {
        "yolo_detections":
            total_yolo_detections,

        "valid_crops":
            total_valid_crops,

        "invalid_crops":
            invalid_crops,

        "predictions":
            len(
                predictions
            ),

        "convnext_candidates":
            conv_candidates,

        "source_counts":
            dict(
                source_counts
            ),

        "convnext_proposed":
            dict(
                conv_proposed_counts
            ),

        "convnext_accepted":
            dict(
                conv_accepted_counts
            ),
    },

    "inference_minutes":
        inference_minutes,
}


with open(
    RESULTS_PATH,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        results,
        f,
        indent=2
    )


# ==========================================================================================
# 25. SAVE SUMMARY
# ==========================================================================================

summary = f"""
====================================================================================================
E19-B — YOLO11m + MOBILENETV3-LARGE + CONVNEXT-SMALL
====================================================================================================

ConvNeXt architecture:
ConvNeXt-Small

ConvNeXt specialist routing:
mixed_plastic_soft
non_plastic
pet_oil

Mixed Rigid ConvNeXt routing:
DISABLED

Fixed ConvNeXt gates:
mixed_plastic_soft = 0.90
non_plastic        = 0.90
pet_oil            = 0.94

Confidence-fusion alpha:
0.70

----------------------------------------------------------------------------------------------------
E19-B RESULTS
----------------------------------------------------------------------------------------------------

AP50-95 : {AP50_95 * 100:.4f}%
AP50    : {AP50 * 100:.4f}%
AP75    : {AP75 * 100:.4f}%
AR100   : {AR100 * 100:.4f}%

----------------------------------------------------------------------------------------------------
E18-E CONVNEXT-TINY REFERENCE
----------------------------------------------------------------------------------------------------

AP50-95 : {E18E_AP * 100:.4f}%
AP50    : {E18E_AP50 * 100:.4f}%
AP75    : {E18E_AP75 * 100:.4f}%
AR100   : {E18E_AR100 * 100:.4f}%

----------------------------------------------------------------------------------------------------
DELTA E19-B vs E18-E
----------------------------------------------------------------------------------------------------

AP50-95 : {(AP50_95 - E18E_AP) * 100:+.4f} pp
AP50    : {(AP50 - E18E_AP50) * 100:+.4f} pp
AP75    : {(AP75 - E18E_AP75) * 100:+.4f} pp
AR100   : {(AR100 - E18E_AR100) * 100:+.4f} pp

----------------------------------------------------------------------------------------------------
SOURCE COUNTS
----------------------------------------------------------------------------------------------------

YOLO      : {source_counts.get("yolo", 0):,}
MobileNet : {source_counts.get("mobilenet", 0):,}
ConvNeXt  : {source_counts.get("convnext", 0):,}

Inference time:
{inference_minutes:.2f} minutes

Checkpoint:
{CONVNEXT_PATH}
"""


with open(
    SUMMARY_PATH,
    "w",
    encoding="utf-8"
) as f:

    f.write(
        summary
    )


print(
    "\n"
    + "=" * 100
)

print(
    "E19-B COMPLETE"
)

print(
    "=" * 100
)


print(
    f"\nE18-E AP50-95 : "
    f"{E18E_AP * 100:.4f}%"
)

print(
    f"E19-B AP50-95 : "
    f"{AP50_95 * 100:.4f}%"
)

print(
    f"Delta          : "
    f"{(AP50_95 - E18E_AP) * 100:+.4f} pp"
)


print(
    f"\nResults:\n"
    f"{RESULTS_PATH}"
)

print(
    f"\nSummary:\n"
    f"{SUMMARY_PATH}"
)

print(
    f"\nPredictions:\n"
    f"{PREDICTIONS_PATH}"
)

E19-B — YOLO11m + MOBILENET + CONVNEXT-SMALL END-TO-END

Device : cuda
GPU    : NVIDIA GeForce RTX 3050 Ti Laptop GPU

Preparing 7-class validation ground truth...
Images : 780
GT     : 13,065

Loading MobileNet...
MobileNet loaded.

Loading ConvNeXt-Small...
ConvNeXt-Small loaded.

Loading YOLO11m...
YOLO11m loaded.
Processing 25/780
Processing 50/780
Processing 75/780
Processing 100/780
Processing 125/780
Processing 150/780
Processing 175/780
Processing 200/780
Processing 225/780
Processing 250/780
Processing 275/780
Processing 300/780
Processing 325/780
Processing 350/780
Processing 375/780
Processing 400/780
Processing 425/780
Processing 450/780
Processing 475/780
Processing 500/780
Processing 525/780
Processing 550/780
Processing 575/780
Processing 600/780
Processing 625/780
Processing 650/780
Processing 675/780
Processing 700/780
Processing 725/780
Processing 750/780
Processing 775/780

Inference complete.
YOLO detections       : 55,605
Valid crops           : 55,605
Invalid crop

## E19-C: CONVNEXT-SMALL CLASS-SPECIFIC GATE CALIBRATION

In [ ]:
# E19-C — CONVNEXT-SMALL CLASS-SPECIFIC GATE CALIBRATION
# ==========================================================================================
#
# Purpose:
#   Calibrate ConvNeXt-Small confidence gates after E19-B showed that
#   using the ConvNeXt-Tiny gates caused excessive specialist intervention.
#
# Fixed:
#   Detector                  : E12 YOLO11m
#   MobileNet                 : E3Y-B class-weighted MobileNetV3-Large
#   MobileNet routing         : E16-A class-specific gates
#   ConvNeXt architecture     : E19-A ConvNeXt-Small
#   ConvNeXt routing classes  : Mixed Soft, non-plastic, PET Oil
#   Mixed Rigid routing       : DISABLED
#   Confidence fusion alpha   : 0.70
#
# Tuned:
#   ConvNeXt-Small gates only
#
# Search method:
#   Greedy one-class-at-a-time optimization
#
# Primary metric:
#   COCO AP50-95
#
# Tie-break:
#   COCO AP50
#
# Baselines:
#   E19-B = 45.4097% AP50-95
#   E18-E = 45.7263% AP50-95
#
# IMPORTANT:
#   The networks are run only ONCE to build a probability cache.
#   All gate combinations are evaluated from the cache.
# ==========================================================================================


import json
import time
from pathlib import Path
from collections import Counter

import numpy as np
from PIL import Image

import torch
import torch.nn as nn
from torchvision import transforms

from torchvision.models import (
    mobilenet_v3_large,
    convnext_small,
)

from ultralytics import YOLO

from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval


# ==========================================================================================
# 1. DEVICE
# ==========================================================================================

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)


print("=" * 100)
print("E19-C — CONVNEXT-SMALL CLASS-SPECIFIC GATE CALIBRATION")
print("=" * 100)

print(f"\nDevice : {DEVICE}")

if DEVICE.type == "cuda":
    print(
        f"GPU    : "
        f"{torch.cuda.get_device_name(0)}"
    )


# ==========================================================================================
# 2. PATHS
# ==========================================================================================

BASE = Path(
    r"C:\Users\varda\Documents\_My Computer\COMMON_space\Learning"
    r"\Upgrad_EPGP_MS\upgrad_IIITB_MS_course\MS_LJMU_Material"
)

DATASET_ROOT = (
    BASE
    / "Topic Data"
    / "SortWaste"
    / "dataset"
    / "dataset"
)

THESIS_CODE = (
    BASE
    / "Thesis_Code"
)


VAL_ROOT = (
    DATASET_ROOT
    / "splited_all_dataset_coco"
    / "val"
)

VAL_IMAGES = (
    VAL_ROOT
    / "images"
)

VAL_COCO_JSON = (
    VAL_ROOT
    / "annotations"
    / "val_coco.json"
)


# ------------------------------------------------------------
# E12 YOLO11m
# ------------------------------------------------------------

YOLO_PATH = (
    THESIS_CODE
    / "runs"
    / "sortwaste"
    / "E12_yolo11m_7class_aug_classbalance_640"
    / "weights"
    / "best.pt"
)


# ------------------------------------------------------------
# E3Y-B MobileNet
# ------------------------------------------------------------

MOBILENET_PATH = (
    DATASET_ROOT
    / "yolo_mobilenet_crops_E3Y"
    / "mobilenet_results"
    / "E3Y_B_class_weighted"
    / "E3Y_B_MobileNetV3Large_best.pth"
)


# ------------------------------------------------------------
# E19-A ConvNeXt-Small
# ------------------------------------------------------------

CONVNEXT_PATH = (
    THESIS_CODE
    / "runs"
    / "sortwaste"
    / "E19A_convnext_small_hardclass"
    / "E19A_ConvNeXtSmall_best.pth"
)


# ------------------------------------------------------------
# E19-C output
# ------------------------------------------------------------

OUTPUT_DIR = (
    THESIS_CODE
    / "runs"
    / "sortwaste"
    / "E19C_convnext_small_gate_calibration"
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)


CACHE_PATH = (
    OUTPUT_DIR
    / "E19C_cached_predictions.json"
)

GT_7CLASS_PATH = (
    OUTPUT_DIR
    / "E19C_val_gt_7class.json"
)

RESULTS_PATH = (
    OUTPUT_DIR
    / "E19C_gate_search_results.json"
)

SUMMARY_PATH = (
    OUTPUT_DIR
    / "E19C_summary.txt"
)

BEST_PREDICTIONS_PATH = (
    OUTPUT_DIR
    / "E19C_best_predictions.json"
)


for path in [
    VAL_IMAGES,
    VAL_COCO_JSON,
    YOLO_PATH,
    MOBILENET_PATH,
    CONVNEXT_PATH,
]:

    assert path.exists(), (
        f"Missing required path:\n{path}"
    )


# ==========================================================================================
# 3. TAXONOMY
# ==========================================================================================

CLASS_NAMES = [
    "ecal",
    "hdpe",
    "mixed_plastic_rigid",
    "mixed_plastic_soft",
    "non_plastic",
    "pet",
    "pet_oil",
]


ECAL = 0
HDPE = 1
MIXED_RIGID = 2
MIXED_SOFT = 3
NON_PLASTIC = 4
PET = 5
PET_OIL = 6


# ==========================================================================================
# 4. ORIGINAL COCO -> 7-CLASS COCO
# ==========================================================================================

ORIGINAL_COCO_TO_NEW = {
    1: 6,  # PET
    2: 2,  # HDPE
    3: 4,  # Mixed Soft
    4: 1,  # ECAL
    5: 5,  # Metal -> nonplastic
    6: 5,  # Cardboard -> nonplastic
    7: 3,  # Mixed Rigid
    8: 7,  # PET Oil
}


# ==========================================================================================
# 5. BUILD 7-CLASS GT
# ==========================================================================================

with open(
    VAL_COCO_JSON,
    "r",
    encoding="utf-8"
) as f:

    original_gt = json.load(f)


gt_7class = {

    "images":
        original_gt[
            "images"
        ],

    "annotations":
        [],

    "categories":
        [
            {
                "id": i + 1,
                "name": name,
            }

            for i, name
            in enumerate(
                CLASS_NAMES
            )
        ],
}


for ann in original_gt[
    "annotations"
]:

    new_ann = ann.copy()

    new_ann[
        "category_id"
    ] = ORIGINAL_COCO_TO_NEW[
        int(
            ann[
                "category_id"
            ]
        )
    ]

    gt_7class[
        "annotations"
    ].append(
        new_ann
    )


with open(
    GT_7CLASS_PATH,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        gt_7class,
        f
    )


print(
    f"\nValidation images : "
    f"{len(gt_7class['images']):,}"
)

print(
    f"Validation GT     : "
    f"{len(gt_7class['annotations']):,}"
)


# ==========================================================================================
# 6. TRANSFORM
# ==========================================================================================

IMAGE_SIZE = 224


classifier_transform = transforms.Compose(
    [
        transforms.Resize(
            (
                IMAGE_SIZE,
                IMAGE_SIZE
            )
        ),

        transforms.ToTensor(),

        transforms.Normalize(
            mean=[
                0.485,
                0.456,
                0.406,
            ],

            std=[
                0.229,
                0.224,
                0.225,
            ],
        ),
    ]
)


# ==========================================================================================
# 7. MOBILE NET GATES — E16-A
# ==========================================================================================

MOBILENET_GATES = {
    ECAL: 0.99,
    HDPE: 0.98,
    MIXED_RIGID: 0.98,
    MIXED_SOFT: 0.94,
    NON_PLASTIC: 0.94,
    PET: 0.99,
    PET_OIL: 0.99,
}


# ==========================================================================================
# 8. CONVNEXT ROUTING
# ==========================================================================================

# E18-D2 / E18-E specialist classes only.

ALLOWED_CONVNEXT_CLASSES = {
    MIXED_SOFT,
    NON_PLASTIC,
    PET_OIL,
}


# ConvNeXt is still consulted when E16-A final class is one of
# the four original hard classes.

E16_HARD_CLASSES = {
    MIXED_RIGID,
    MIXED_SOFT,
    NON_PLASTIC,
    PET_OIL,
}


# ==========================================================================================
# 9. CONVNEXT LOCAL -> GLOBAL MAPPING
# ==========================================================================================

CONV_LOCAL_TO_GLOBAL = {
    0: MIXED_RIGID,
    1: MIXED_SOFT,
    2: NON_PLASTIC,
    3: PET_OIL,
}


# ==========================================================================================
# 10. FIXED FUSION
# ==========================================================================================

ALPHA = 0.70


# ==========================================================================================
# 11. GATE SEARCH
#
# Small was dramatically more confident than Tiny.
#
# Therefore search higher thresholds.
#
# We include E18-E's previous gates as references,
# but extend heavily toward 0.99.
# ==========================================================================================

BASE_GATES = {
    MIXED_SOFT: 0.90,
    NON_PLASTIC: 0.90,
    PET_OIL: 0.94,
}


GATE_CANDIDATES = {

    MIXED_SOFT: [
        0.90,
        0.92,
        0.94,
        0.96,
        0.97,
        0.98,
        0.99,
    ],

    NON_PLASTIC: [
        0.90,
        0.92,
        0.94,
        0.96,
        0.97,
        0.98,
        0.99,
    ],

    PET_OIL: [
        0.94,
        0.96,
        0.97,
        0.98,
        0.99,
        0.995,
    ],
}


SEARCH_ORDER = [
    MIXED_SOFT,
    NON_PLASTIC,
    PET_OIL,
]


# ==========================================================================================
# 12. YOLO SETTINGS
# ==========================================================================================

YOLO_CONF = 0.001
YOLO_NMS_IOU = 0.60
YOLO_MAX_DET = 100
YOLO_IMGSZ = 640


# ==========================================================================================
# 13. BUILD CACHE IF NEEDED
# ==========================================================================================

if CACHE_PATH.exists():

    print(
        "\nExisting E19-C cache found."
    )

    print(
        "Skipping neural-network inference."
    )

    with open(
        CACHE_PATH,
        "r",
        encoding="utf-8"
    ) as f:

        cache = json.load(f)


else:

    print(
        "\nNo E19-C cache found."
    )

    print(
        "Running YOLO + MobileNet + ConvNeXt-Small once..."
    )


    # --------------------------------------------------------------------------------------
    # Load MobileNet
    # --------------------------------------------------------------------------------------

    print(
        "\nLoading MobileNet..."
    )


    mobilenet = mobilenet_v3_large(
        weights=None
    )


    mn_in_features = (
        mobilenet
        .classifier[3]
        .in_features
    )


    mobilenet.classifier[3] = nn.Linear(
        mn_in_features,
        7
    )


    mn_checkpoint = torch.load(
        MOBILENET_PATH,
        map_location=DEVICE,
        weights_only=False,
    )


    if (
        isinstance(
            mn_checkpoint,
            dict
        )
        and
        "model_state_dict"
        in mn_checkpoint
    ):

        mn_state = (
            mn_checkpoint[
                "model_state_dict"
            ]
        )

    else:

        mn_state = (
            mn_checkpoint
        )


    mobilenet.load_state_dict(
        mn_state
    )

    mobilenet = mobilenet.to(
        DEVICE
    )

    mobilenet.eval()


    print(
        "MobileNet loaded."
    )


    # --------------------------------------------------------------------------------------
    # Load ConvNeXt-Small
    # --------------------------------------------------------------------------------------

    print(
        "\nLoading ConvNeXt-Small..."
    )


    convnext = convnext_small(
        weights=None
    )


    conv_in_features = (
        convnext
        .classifier[2]
        .in_features
    )


    convnext.classifier[2] = nn.Linear(
        conv_in_features,
        4
    )


    conv_checkpoint = torch.load(
        CONVNEXT_PATH,
        map_location=DEVICE,
        weights_only=False,
    )


    if (
        isinstance(
            conv_checkpoint,
            dict
        )
        and
        "model_state_dict"
        in conv_checkpoint
    ):

        conv_state = (
            conv_checkpoint[
                "model_state_dict"
            ]
        )

    else:

        conv_state = (
            conv_checkpoint
        )


    convnext.load_state_dict(
        conv_state
    )

    convnext = convnext.to(
        DEVICE
    )

    convnext.eval()


    print(
        "ConvNeXt-Small loaded."
    )


    # --------------------------------------------------------------------------------------
    # Load YOLO
    # --------------------------------------------------------------------------------------

    print(
        "\nLoading YOLO11m..."
    )


    yolo = YOLO(
        str(
            YOLO_PATH
        )
    )


    print(
        "YOLO11m loaded."
    )


    # --------------------------------------------------------------------------------------
    # Inference
    # --------------------------------------------------------------------------------------

    cache = []

    total_yolo = 0
    invalid_crops = 0
    conv_consulted = 0


    image_records = sorted(
        gt_7class[
            "images"
        ],

        key=lambda x: int(
            x[
                "id"
            ]
        )
    )


    inference_start = time.time()


    for image_number, image_record in enumerate(
        image_records,
        start=1,
    ):

        image_id = int(
            image_record[
                "id"
            ]
        )

        filename = (
            image_record[
                "file_name"
            ]
        )

        image_path = (
            VAL_IMAGES
            / filename
        )


        pil_image = Image.open(
            image_path
        ).convert(
            "RGB"
        )

        image_width, image_height = (
            pil_image.size
        )


        yolo_result = yolo.predict(
            source=str(
                image_path
            ),
            imgsz=YOLO_IMGSZ,
            conf=YOLO_CONF,
            iou=YOLO_NMS_IOU,
            max_det=YOLO_MAX_DET,
            verbose=False,
            device=0 if DEVICE.type == "cuda" else "cpu",
        )[0]


        boxes = (
            yolo_result.boxes
        )


        if boxes is None:
            continue


        xyxy_array = (
            boxes.xyxy
            .detach()
            .cpu()
            .numpy()
        )

        confidence_array = (
            boxes.conf
            .detach()
            .cpu()
            .numpy()
        )

        yolo_class_array = (
            boxes.cls
            .detach()
            .cpu()
            .numpy()
            .astype(int)
        )


        total_yolo += (
            len(
                xyxy_array
            )
        )


        for (
            bbox_xyxy,
            yolo_conf,
            yolo_idx,
        ) in zip(
            xyxy_array,
            confidence_array,
            yolo_class_array,
        ):

            x1, y1, x2, y2 = [
                float(v)
                for v in bbox_xyxy
            ]


            crop_x1 = max(
                0,
                int(
                    np.floor(
                        x1
                    )
                )
            )

            crop_y1 = max(
                0,
                int(
                    np.floor(
                        y1
                    )
                )
            )

            crop_x2 = min(
                image_width,
                int(
                    np.ceil(
                        x2
                    )
                )
            )

            crop_y2 = min(
                image_height,
                int(
                    np.ceil(
                        y2
                    )
                )
            )


            if (
                crop_x2 <= crop_x1
                or
                crop_y2 <= crop_y1
            ):

                invalid_crops += 1
                continue


            crop = pil_image.crop(
                (
                    crop_x1,
                    crop_y1,
                    crop_x2,
                    crop_y2,
                )
            )


            input_tensor = (
                classifier_transform(
                    crop
                )
                .unsqueeze(0)
                .to(
                    DEVICE
                )
            )


            # ==================================================================================
            # MobileNet
            # ==================================================================================

            with torch.no_grad():

                if DEVICE.type == "cuda":

                    with torch.amp.autocast(
                        device_type="cuda",
                        dtype=torch.float16,
                    ):

                        mn_logits = (
                            mobilenet(
                                input_tensor
                            )
                        )

                else:

                    mn_logits = (
                        mobilenet(
                            input_tensor
                        )
                    )


            mn_probs = torch.softmax(
                mn_logits.float(),
                dim=1
            )[0]


            mn_probs_np = (
                mn_probs
                .detach()
                .cpu()
                .numpy()
                .astype(
                    np.float32
                )
            )


            mn_top_idx = int(
                np.argmax(
                    mn_probs_np
                )
            )

            mn_top_prob = float(
                mn_probs_np[
                    mn_top_idx
                ]
            )


            # ==================================================================================
            # E16-A MobileNet gating
            # ==================================================================================

            if (
                mn_top_prob
                >= MOBILENET_GATES[
                    mn_top_idx
                ]
            ):

                e16_final_idx = (
                    mn_top_idx
                )

            else:

                e16_final_idx = int(
                    yolo_idx
                )


            # ==================================================================================
            # ConvNeXt-Small specialist prediction
            # ==================================================================================

            conv_info = None


            if (
                e16_final_idx
                in E16_HARD_CLASSES
            ):

                conv_consulted += 1


                with torch.no_grad():

                    if DEVICE.type == "cuda":

                        with torch.amp.autocast(
                            device_type="cuda",
                            dtype=torch.float16,
                        ):

                            conv_logits = (
                                convnext(
                                    input_tensor
                                )
                            )

                    else:

                        conv_logits = (
                            convnext(
                                input_tensor
                            )
                        )


                conv_probs = torch.softmax(
                    conv_logits.float(),
                    dim=1
                )[0]


                conv_probs_np = (
                    conv_probs
                    .detach()
                    .cpu()
                    .numpy()
                    .astype(
                        np.float32
                    )
                )


                conv_local_idx = int(
                    np.argmax(
                        conv_probs_np
                    )
                )


                conv_top_prob = float(
                    conv_probs_np[
                        conv_local_idx
                    ]
                )


                conv_global_idx = (
                    CONV_LOCAL_TO_GLOBAL[
                        conv_local_idx
                    ]
                )


                conv_info = {

                    "conv_probs":
                        conv_probs_np.tolist(),

                    "conv_local_idx":
                        conv_local_idx,

                    "conv_global_idx":
                        int(
                            conv_global_idx
                        ),

                    "conv_top_prob":
                        conv_top_prob,
                }


            cache.append(
                {
                    "image_id":
                        image_id,

                    "bbox":
                        [
                            x1,
                            y1,
                            x2,
                            y2,
                        ],

                    "yolo_conf":
                        float(
                            yolo_conf
                        ),

                    "yolo_class":
                        int(
                            yolo_idx
                        ),

                    "mn_probs":
                        mn_probs_np.tolist(),

                    "e16_final_idx":
                        int(
                            e16_final_idx
                        ),

                    "conv":
                        conv_info,
                }
            )


        if (
            image_number % 25 == 0
        ):

            print(
                f"Processing "
                f"{image_number}/"
                f"{len(image_records)}"
            )


    inference_minutes = (
        time.time()
        - inference_start
    ) / 60.0


    print(
        "\nInference cache complete."
    )

    print(
        f"YOLO detections     : "
        f"{total_yolo:,}"
    )

    print(
        f"Cached detections   : "
        f"{len(cache):,}"
    )

    print(
        f"Invalid crops       : "
        f"{invalid_crops:,}"
    )

    print(
        f"ConvNeXt consulted  : "
        f"{conv_consulted:,}"
    )

    print(
        f"Inference time      : "
        f"{inference_minutes:.2f} min"
    )


    with open(
        CACHE_PATH,
        "w",
        encoding="utf-8"
    ) as f:

        json.dump(
            cache,
            f
        )


print(
    f"\nCached detections : "
    f"{len(cache):,}"
)


# ==========================================================================================
# 14. LOAD COCO GT
# ==========================================================================================

coco_gt = COCO(
    str(
        GT_7CLASS_PATH
    )
)


# ==========================================================================================
# 15. BUILD FINAL PREDICTIONS FOR ONE GATE CONFIGURATION
# ==========================================================================================

def build_predictions(
    conv_gates
):

    predictions = []

    source_counts = Counter()

    conv_candidates = Counter()
    conv_accepted = Counter()
    conv_rejected = Counter()


    for item in cache:

        yolo_conf = float(
            item[
                "yolo_conf"
            ]
        )

        yolo_idx = int(
            item[
                "yolo_class"
            ]
        )

        e16_final_idx = int(
            item[
                "e16_final_idx"
            ]
        )


        mn_probs = np.asarray(
            item[
                "mn_probs"
            ],
            dtype=np.float64,
        )


        # ----------------------------------------------------------------------------------
        # Determine E16 source for reporting only
        # ----------------------------------------------------------------------------------

        mn_top_idx = int(
            np.argmax(
                mn_probs
            )
        )

        mn_top_prob = float(
            mn_probs[
                mn_top_idx
            ]
        )


        if (
            mn_top_prob
            >= MOBILENET_GATES[
                mn_top_idx
            ]
        ):

            original_source = (
                "mobilenet"
            )

        else:

            original_source = (
                "yolo"
            )


        # ----------------------------------------------------------------------------------
        # Start from E16-A
        # ----------------------------------------------------------------------------------

        final_idx = (
            e16_final_idx
        )

        final_source = (
            original_source
        )


        classifier_prob = float(
            mn_probs[
                final_idx
            ]
        )


        # ----------------------------------------------------------------------------------
        # ConvNeXt-Small
        # ----------------------------------------------------------------------------------

        conv_info = (
            item.get(
                "conv"
            )
        )


        if conv_info is not None:

            conv_global_idx = int(
                conv_info[
                    "conv_global_idx"
                ]
            )

            conv_top_prob = float(
                conv_info[
                    "conv_top_prob"
                ]
            )


            # Only E18-D2 retained specialist classes.
            if (
                conv_global_idx
                in ALLOWED_CONVNEXT_CLASSES
            ):

                conv_candidates[
                    conv_global_idx
                ] += 1


                gate = float(
                    conv_gates[
                        conv_global_idx
                    ]
                )


                if (
                    conv_top_prob
                    >= gate
                ):

                    final_idx = (
                        conv_global_idx
                    )

                    final_source = (
                        "convnext"
                    )

                    classifier_prob = (
                        conv_top_prob
                    )


                    conv_accepted[
                        conv_global_idx
                    ] += 1


                else:

                    conv_rejected[
                        conv_global_idx
                    ] += 1


        source_counts[
            final_source
        ] += 1


        # ----------------------------------------------------------------------------------
        # Final confidence score
        # ----------------------------------------------------------------------------------

        classifier_prob = max(
            classifier_prob,
            1e-12
        )


        score = (
            yolo_conf
            ** ALPHA
        ) * (
            classifier_prob
            ** (
                1.0
                - ALPHA
            )
        )


        x1, y1, x2, y2 = (
            item[
                "bbox"
            ]
        )


        predictions.append(
            {
                "image_id":
                    int(
                        item[
                            "image_id"
                        ]
                    ),

                "category_id":
                    int(
                        final_idx + 1
                    ),

                "bbox":
                    [
                        float(
                            x1
                        ),

                        float(
                            y1
                        ),

                        float(
                            x2 - x1
                        ),

                        float(
                            y2 - y1
                        ),
                    ],

                "score":
                    float(
                        score
                    ),
            }
        )


    return {

        "predictions":
            predictions,

        "source_counts":
            source_counts,

        "conv_candidates":
            conv_candidates,

        "conv_accepted":
            conv_accepted,

        "conv_rejected":
            conv_rejected,
    }


# ==========================================================================================
# 16. COCO EVALUATION
# ==========================================================================================

def evaluate_predictions(
    predictions,
    verbose=False
):

    coco_dt = coco_gt.loadRes(
        predictions
    )

    evaluator = COCOeval(
        coco_gt,
        coco_dt,
        "bbox"
    )

    evaluator.params.maxDets = [
        1,
        10,
        100
    ]


    if verbose:

        evaluator.evaluate()
        evaluator.accumulate()
        evaluator.summarize()


    else:

        import contextlib
        import io

        with contextlib.redirect_stdout(
            io.StringIO()
        ):

            evaluator.evaluate()
            evaluator.accumulate()
            evaluator.summarize()


    metrics = {

        "AP50_95":
            float(
                evaluator.stats[0]
            ),

        "AP50":
            float(
                evaluator.stats[1]
            ),

        "AP75":
            float(
                evaluator.stats[2]
            ),

        "AR100":
            float(
                evaluator.stats[8]
            ),
    }


    return evaluator, metrics


# ==========================================================================================
# 17. PER-CLASS METRICS
# ==========================================================================================

def get_per_class_metrics(
    evaluator
):

    precision = (
        evaluator.eval[
            "precision"
        ]
    )

    iou_thresholds = (
        evaluator.params.iouThrs
    )


    ap50_idx = int(
        np.where(
            np.isclose(
                iou_thresholds,
                0.50
            )
        )[0][0]
    )


    output = {}


    for class_idx, class_name in enumerate(
        CLASS_NAMES
    ):

        values = precision[
            :,
            :,
            class_idx,
            0,
            -1
        ]

        values = (
            values[
                values > -1
            ]
        )


        ap = (
            float(
                np.mean(
                    values
                )
            )

            if values.size > 0

            else float(
                "nan"
            )
        )


        values50 = precision[
            ap50_idx,
            :,
            class_idx,
            0,
            -1
        ]

        values50 = (
            values50[
                values50 > -1
            ]
        )


        ap50 = (
            float(
                np.mean(
                    values50
                )
            )

            if values50.size > 0

            else float(
                "nan"
            )
        )


        output[
            class_name
        ] = {

            "AP50_95":
                ap,

            "AP50":
                ap50,
        }


    return output


# ==========================================================================================
# 18. RUN ONE CONFIGURATION
# ==========================================================================================

def run_configuration(
    gates,
    label,
    verbose=False
):

    built = build_predictions(
        gates
    )


    evaluator, metrics = (
        evaluate_predictions(
            built[
                "predictions"
            ],
            verbose=verbose,
        )
    )


    class_metrics = (
        get_per_class_metrics(
            evaluator
        )
    )


    result = {

        "label":
            label,

        "gates": {
            CLASS_NAMES[k]:
                float(v)

            for k, v
            in gates.items()
        },

        "metrics":
            metrics,

        "class_metrics":
            class_metrics,

        "source_counts":
            {
                k:
                    int(v)

                for k, v
                in built[
                    "source_counts"
                ].items()
            },

        "convnext_candidates":
            {
                CLASS_NAMES[k]:
                    int(v)

                for k, v
                in built[
                    "conv_candidates"
                ].items()
            },

        "convnext_accepted":
            {
                CLASS_NAMES[k]:
                    int(v)

                for k, v
                in built[
                    "conv_accepted"
                ].items()
            },

        "convnext_rejected":
            {
                CLASS_NAMES[k]:
                    int(v)

                for k, v
                in built[
                    "conv_rejected"
                ].items()
            },
    }


    return (
        result,
        built[
            "predictions"
        ]
    )


# ==========================================================================================
# 19. BASELINE — REPRODUCE E19-B
# ==========================================================================================

print(
    "\n"
    + "=" * 100
)

print(
    "BASELINE — REPRODUCING E19-B"
)

print(
    "=" * 100
)


baseline_result, _ = (
    run_configuration(
        BASE_GATES.copy(),
        "E19-B baseline",
        verbose=True,
    )
)


baseline_metrics = (
    baseline_result[
        "metrics"
    ]
)


print(
    "\nExpected approximately:"
)

print(
    "AP50-95 : 45.4097%"
)

print(
    "AP50    : 60.6818%"
)

print(
    "AP75    : 51.6806%"
)

print(
    "AR100   : 60.8141%"
)


print(
    "\nReproduced:"
)

print(
    f"AP50-95 : "
    f"{baseline_metrics['AP50_95'] * 100:.4f}%"
)

print(
    f"AP50    : "
    f"{baseline_metrics['AP50'] * 100:.4f}%"
)

print(
    f"AP75    : "
    f"{baseline_metrics['AP75'] * 100:.4f}%"
)

print(
    f"AR100   : "
    f"{baseline_metrics['AR100'] * 100:.4f}%"
)


# ==========================================================================================
# 20. GREEDY GATE SEARCH
# ==========================================================================================

current_gates = (
    BASE_GATES.copy()
)

search_history = []


for class_idx in (
    SEARCH_ORDER
):

    class_name = (
        CLASS_NAMES[
            class_idx
        ]
    )


    print(
        "\n"
        + "=" * 100
    )

    print(
        f"TUNING CONVNEXT-SMALL GATE — "
        f"{class_name}"
    )

    print(
        "=" * 100
    )


    stage_results = []


    for candidate_gate in (
        GATE_CANDIDATES[
            class_idx
        ]
    ):

        test_gates = (
            current_gates.copy()
        )


        test_gates[
            class_idx
        ] = (
            candidate_gate
        )


        result, _ = (
            run_configuration(
                test_gates,
                (
                    f"{class_name}"
                    f"_gate_"
                    f"{candidate_gate:.3f}"
                ),
                verbose=False,
            )
        )


        metrics = (
            result[
                "metrics"
            ]
        )


        accepted = (
            result[
                "convnext_accepted"
            ].get(
                class_name,
                0,
            )
        )


        print(
            f"Gate {candidate_gate:.3f}"
            f" | AP50-95 "
            f"{metrics['AP50_95'] * 100:8.4f}%"
            f" | AP50 "
            f"{metrics['AP50'] * 100:8.4f}%"
            f" | AP75 "
            f"{metrics['AP75'] * 100:8.4f}%"
            f" | AR100 "
            f"{metrics['AR100'] * 100:8.4f}%"
            f" | Accepted "
            f"{accepted:,}"
        )


        stage_results.append(
            result
        )


    best_stage = max(
        stage_results,

        key=lambda x: (
            x[
                "metrics"
            ][
                "AP50_95"
            ],

            x[
                "metrics"
            ][
                "AP50"
            ],
        )
    )


    best_gate = float(
        best_stage[
            "gates"
        ][
            class_name
        ]
    )


    current_gates[
        class_idx
    ] = (
        best_gate
    )


    print(
        "\nSelected gate:"
    )

    print(
        f"{class_name} = "
        f"{best_gate:.3f}"
    )

    print(
        f"Stage-best AP50-95 = "
        f"{best_stage['metrics']['AP50_95'] * 100:.4f}%"
    )


    search_history.append(
        {
            "class":
                class_name,

            "selected_gate":
                best_gate,

            "selected_metrics":
                best_stage[
                    "metrics"
                ],

            "candidate_results":
                stage_results,
        }
    )


# ==========================================================================================
# 21. FINAL E19-C CONFIGURATION
# ==========================================================================================

print(
    "\n"
    + "=" * 100
)

print(
    "FINAL E19-C CONFIGURATION"
)

print(
    "=" * 100
)


for class_idx in (
    SEARCH_ORDER
):

    print(
        f"{CLASS_NAMES[class_idx]:25s}"
        f": "
        f"{current_gates[class_idx]:.3f}"
    )


print(
    "\nMixed Rigid ConvNeXt routing: DISABLED"
)

print(
    f"Confidence fusion alpha      : "
    f"{ALPHA:.2f}"
)


# ==========================================================================================
# 22. FINAL EVALUATION
# ==========================================================================================

final_result, final_predictions = (
    run_configuration(
        current_gates,
        "E19-C final",
        verbose=True,
    )
)


final_metrics = (
    final_result[
        "metrics"
    ]
)


# ==========================================================================================
# 23. REFERENCES
# ==========================================================================================

E19B_AP = 0.454097
E19B_AP50 = 0.606818
E19B_AP75 = 0.516806
E19B_AR100 = 0.608141


E18E_AP = 0.457263
E18E_AP50 = 0.611238
E18E_AP75 = 0.519444
E18E_AR100 = 0.614629


# ==========================================================================================
# 24. FINAL COMPARISON
# ==========================================================================================

print(
    "\n"
    + "=" * 100
)

print(
    "E19-C FINAL RESULT"
)

print(
    "=" * 100
)


print(
    f"\n{'Metric':12s}"
    f"{'E19-B':>14s}"
    f"{'E19-C':>14s}"
    f"{'Δ vs B':>14s}"
    f"{'E18-E':>14s}"
    f"{'Δ vs E18-E':>16s}"
)


rows = [
    (
        "AP50-95",
        E19B_AP,
        final_metrics[
            "AP50_95"
        ],
        E18E_AP,
    ),

    (
        "AP50",
        E19B_AP50,
        final_metrics[
            "AP50"
        ],
        E18E_AP50,
    ),

    (
        "AP75",
        E19B_AP75,
        final_metrics[
            "AP75"
        ],
        E18E_AP75,
    ),

    (
        "AR100",
        E19B_AR100,
        final_metrics[
            "AR100"
        ],
        E18E_AR100,
    ),
]


for (
    metric,
    e19b,
    e19c,
    e18e,
) in rows:

    print(
        f"{metric:12s}"
        f"{e19b * 100:14.4f}"
        f"{e19c * 100:14.4f}"
        f"{(e19c-e19b) * 100:+14.4f}"
        f"{e18e * 100:14.4f}"
        f"{(e19c-e18e) * 100:+16.4f}"
    )


# ==========================================================================================
# 25. CLASS-WISE RESULTS
# ==========================================================================================

print(
    "\n"
    + "=" * 100
)

print(
    "E19-C CLASS-WISE AP50-95"
)

print(
    "=" * 100
)


print(
    f"\n{'Class':28s}"
    f"{'AP50':>14s}"
    f"{'AP50-95':>14s}"
)


for class_name in (
    CLASS_NAMES
):

    values = (
        final_result[
            "class_metrics"
        ][
            class_name
        ]
    )


    print(
        f"{class_name:28s}"
        f"{values['AP50'] * 100:14.2f}"
        f"{values['AP50_95'] * 100:14.2f}"
    )


# ==========================================================================================
# 26. SOURCE COUNTS
# ==========================================================================================

print(
    "\n"
    + "=" * 100
)

print(
    "FINAL CLASS SOURCE COUNTS"
)

print(
    "=" * 100
)


for source in [
    "yolo",
    "mobilenet",
    "convnext",
]:

    print(
        f"{source:15s}: "
        f"{final_result['source_counts'].get(source, 0):,}"
    )


print(
    "\nConvNeXt-Small accepted by class:"
)


for class_idx in (
    SEARCH_ORDER
):

    class_name = (
        CLASS_NAMES[
            class_idx
        ]
    )


    print(
        f"{class_name:25s}: "
        f"{final_result['convnext_accepted'].get(class_name, 0):,}"
    )


# ==========================================================================================
# 27. SAVE BEST PREDICTIONS
# ==========================================================================================

with open(
    BEST_PREDICTIONS_PATH,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        final_predictions,
        f,
    )


# ==========================================================================================
# 28. SAVE RESULTS
# ==========================================================================================

output_data = {

    "experiment":
        (
            "E19-C — ConvNeXt-Small "
            "Class-Specific Gate Calibration"
        ),

    "architecture":
        "ConvNeXt-Small",

    "routing_classes": [
        "mixed_plastic_soft",
        "non_plastic",
        "pet_oil",
    ],

    "mixed_plastic_rigid_routing":
        "disabled",

    "alpha":
        ALPHA,

    "initial_gates": {
        CLASS_NAMES[k]:
            float(v)

        for k, v
        in BASE_GATES.items()
    },

    "final_gates": {
        CLASS_NAMES[k]:
            float(v)

        for k, v
        in current_gates.items()
    },

    "search_history":
        search_history,

    "baseline_result":
        baseline_result,

    "final_result":
        final_result,

    "reference_metrics": {

        "E19_B": {
            "AP50_95":
                E19B_AP,

            "AP50":
                E19B_AP50,

            "AP75":
                E19B_AP75,

            "AR100":
                E19B_AR100,
        },

        "E18_E": {
            "AP50_95":
                E18E_AP,

            "AP50":
                E18E_AP50,

            "AP75":
                E18E_AP75,

            "AR100":
                E18E_AR100,
        },
    },
}


with open(
    RESULTS_PATH,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        output_data,
        f,
        indent=2,
    )


# ==========================================================================================
# 29. SAVE SUMMARY
# ==========================================================================================

summary_lines = [

    "=" * 100,

    "E19-C — ConvNeXt-Small Class-Specific Gate Calibration",

    "=" * 100,

    "",

    "Architecture:",
    "ConvNeXt-Small",

    "",

    "Routing classes:",
    "  mixed_plastic_soft",
    "  non_plastic",
    "  pet_oil",

    "",

    "Mixed Rigid routing:",
    "DISABLED",

    "",

    f"Alpha: {ALPHA:.2f}",

    "",

    "Final gates:",
]


for class_idx in (
    SEARCH_ORDER
):

    summary_lines.append(
        (
            f"  "
            f"{CLASS_NAMES[class_idx]}"
            f" = "
            f"{current_gates[class_idx]:.3f}"
        )
    )


summary_lines.extend(
    [

        "",

        "E19-B:",
        f"  AP50-95 = {E19B_AP:.6f}",

        "",

        "E19-C:",
        (
            f"  AP50-95 = "
            f"{final_metrics['AP50_95']:.6f}"
        ),
        (
            f"  AP50    = "
            f"{final_metrics['AP50']:.6f}"
        ),
        (
            f"  AP75    = "
            f"{final_metrics['AP75']:.6f}"
        ),
        (
            f"  AR100   = "
            f"{final_metrics['AR100']:.6f}"
        ),

        "",

        "E18-E reference:",
        f"  AP50-95 = {E18E_AP:.6f}",

        "",

        (
            f"Gain vs E19-B = "
            f"{(final_metrics['AP50_95'] - E19B_AP) * 100:+.4f} pp"
        ),

        (
            f"Difference vs E18-E = "
            f"{(final_metrics['AP50_95'] - E18E_AP) * 100:+.4f} pp"
        ),

        "",

        "=" * 100,
    ]
)


with open(
    SUMMARY_PATH,
    "w",
    encoding="utf-8"
) as f:

    f.write(
        "\n".join(
            summary_lines
        )
    )


# ==========================================================================================
# 30. DONE
# ==========================================================================================

print(
    "\n"
    + "=" * 100
)

print(
    "E19-C COMPLETE"
)

print(
    "=" * 100
)


print(
    "\nBest ConvNeXt-Small gates:"
)


for class_idx in (
    SEARCH_ORDER
):

    print(
        f"{CLASS_NAMES[class_idx]:25s}"
        f": "
        f"{current_gates[class_idx]:.3f}"
    )


print(
    f"\nE19-B AP50-95 : "
    f"{E19B_AP * 100:.4f}%"
)

print(
    f"E19-C AP50-95 : "
    f"{final_metrics['AP50_95'] * 100:.4f}%"
)

print(
    f"E18-E AP50-95 : "
    f"{E18E_AP * 100:.4f}%"
)


print(
    f"\nGain vs E19-B : "
    f"{(final_metrics['AP50_95'] - E19B_AP) * 100:+.4f} pp"
)

print(
    f"vs E18-E      : "
    f"{(final_metrics['AP50_95'] - E18E_AP) * 100:+.4f} pp"
)


print(
    f"\nResults:\n"
    f"{RESULTS_PATH}"
)

print(
    f"\nSummary:\n"
    f"{SUMMARY_PATH}"
)

print(
    f"\nCache:\n"
    f"{CACHE_PATH}"
)

print(
    f"\nBest predictions:\n"
    f"{BEST_PREDICTIONS_PATH}"
)

E19-C — CONVNEXT-SMALL CLASS-SPECIFIC GATE CALIBRATION

Device : cuda
GPU    : NVIDIA GeForce RTX 3050 Ti Laptop GPU

Validation images : 780
Validation GT     : 13,065

No E19-C cache found.
Running YOLO + MobileNet + ConvNeXt-Small once...

Loading MobileNet...
MobileNet loaded.

Loading ConvNeXt-Small...
ConvNeXt-Small loaded.

Loading YOLO11m...
YOLO11m loaded.
Processing 25/780
Processing 50/780
Processing 75/780
Processing 100/780
Processing 125/780
Processing 150/780
Processing 175/780
Processing 200/780
Processing 225/780
Processing 250/780
Processing 275/780
Processing 300/780
Processing 325/780
Processing 350/780
Processing 375/780
Processing 400/780
Processing 425/780
Processing 450/780
Processing 475/780
Processing 500/780
Processing 525/780
Processing 550/780
Processing 575/780
Processing 600/780
Processing 625/780
Processing 650/780
Processing 675/780
Processing 700/780
Processing 725/780
Processing 750/780
Processing 775/780

Inference cache complete.
YOLO detections    

# TEST data evaluation

In [ ]:
# E19-C: YOLO11m + MobileNetV3-Large + ConvNeXt-Small - Frozen Class-Specific Gating + Confidence Fusion
# ==========================================================================================
#
# IMPORTANT:
#   This is FINAL TEST evaluation.
#
#   NO threshold tuning.
#   NO gate search.
#   NO alpha tuning.
#   NO retraining.
#
# Frozen validation-selected configuration:
#
#   YOLO11m:
#       imgsz = 640
#       conf  = 0.001
#       NMS   = 0.60
#       max_det = 100
#
#   MobileNet E16-A gates:
#       ECAL        0.99
#       HDPE        0.98
#       MixedRigid  0.98
#       MixedSoft   0.94
#       nonplastic  0.94
#       PET         0.99
#       PET Oil     0.99
#
#   ConvNeXt-Small routing:
#       Mixed Rigid : DISABLED
#       Mixed Soft  : ENABLED, gate 0.99
#       nonplastic  : ENABLED, gate 0.99
#       PET Oil     : ENABLED, gate 0.96
#
#   Confidence fusion:
#
#       S = C_YOLO^0.70 * P_classifier(final class)^0.30
#
# Primary metric:
#       COCO AP@[0.50:0.95]
#
# ==========================================================================================

import json
import time
from pathlib import Path
from collections import Counter

import numpy as np
from PIL import Image

import torch
import torch.nn as nn
from torchvision import transforms
from torchvision.models import mobilenet_v3_large, convnext_small

from ultralytics import YOLO

from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval


# ==========================================================================================
# 1. DEVICE
# ==========================================================================================

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("=" * 100)
print("FINAL TEST — E19-C YOLO11m + MobileNet + ConvNeXt-Small")
print("=" * 100)

print(f"\nDevice : {DEVICE}")

if DEVICE.type == "cuda":
    print(
        f"GPU    : {torch.cuda.get_device_name(0)}"
    )


# ==========================================================================================
# 2. PATHS
# ==========================================================================================

BASE = Path(
    r"C:\Users\varda\Documents\_My Computer\COMMON_space\Learning"
    r"\Upgrad_EPGP_MS\upgrad_IIITB_MS_course\MS_LJMU_Material"
)

DATASET_ROOT = (
    BASE
    / "Topic Data"
    / "SortWaste"
    / "dataset"
    / "dataset"
)

THESIS_CODE = (
    BASE
    / "Thesis_Code"
)


# ------------------------------------------------------------------------------------------
# TEST SPLIT
# ------------------------------------------------------------------------------------------

TEST_ROOT = (
    DATASET_ROOT
    / "splited_all_dataset_coco"
    / "test"
)

TEST_IMAGES = (
    TEST_ROOT
    / "images"
)

TEST_COCO_JSON = (
    TEST_ROOT
    / "annotations"
    / "test_coco.json"
)


# ------------------------------------------------------------------------------------------
# E12 YOLO11m
# ------------------------------------------------------------------------------------------

YOLO_PATH = (
    THESIS_CODE
    / "runs"
    / "sortwaste"
    / "E12_yolo11m_7class_aug_classbalance_640"
    / "weights"
    / "best.pt"
)


# ------------------------------------------------------------------------------------------
# E3Y-B MobileNet
# ------------------------------------------------------------------------------------------

MOBILENET_PATH = (
    DATASET_ROOT
    / "yolo_mobilenet_crops_E3Y"
    / "mobilenet_results"
    / "E3Y_B_class_weighted"
    / "E3Y_B_MobileNetV3Large_best.pth"
)


# ------------------------------------------------------------------------------------------
# E19-A ConvNeXt-Small
# ------------------------------------------------------------------------------------------

CONVNEXT_PATH = (
    THESIS_CODE
    / "runs"
    / "sortwaste"
    / "E19A_convnext_small_hardclass"
    / "E19A_ConvNeXtSmall_best.pth"
)


# ------------------------------------------------------------------------------------------
# OUTPUT
# ------------------------------------------------------------------------------------------

OUTPUT_DIR = (
    THESIS_CODE
    / "runs"
    / "sortwaste"
    / "FINAL_TEST_E19C"
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

GT_7CLASS_PATH = (
    OUTPUT_DIR
    / "E19C_test_gt_7class.json"
)

PREDICTIONS_PATH = (
    OUTPUT_DIR
    / "E19C_test_predictions.json"
)

RESULTS_PATH = (
    OUTPUT_DIR
    / "E19C_test_results.json"
)

SUMMARY_PATH = (
    OUTPUT_DIR
    / "E19C_test_summary.txt"
)


for path in [
    TEST_IMAGES,
    TEST_COCO_JSON,
    YOLO_PATH,
    MOBILENET_PATH,
    CONVNEXT_PATH,
]:

    assert path.exists(), (
        f"Missing required path:\n{path}"
    )


print(f"\nTest images : {TEST_IMAGES}")
print(f"Test COCO   : {TEST_COCO_JSON}")
print(f"Output      : {OUTPUT_DIR}")


# ==========================================================================================
# 3. TAXONOMY
# ==========================================================================================

CLASS_NAMES = [
    "ecal",
    "hdpe",
    "mixed_plastic_rigid",
    "mixed_plastic_soft",
    "non_plastic",
    "pet",
    "pet_oil",
]

ECAL = 0
HDPE = 1
MIXED_RIGID = 2
MIXED_SOFT = 3
NON_PLASTIC = 4
PET = 5
PET_OIL = 6


# ==========================================================================================
# 4. ORIGINAL COCO -> 7 CLASS
#
# Original:
# 1 PET
# 2 HDPE
# 3 Mixed Soft
# 4 ECAL
# 5 Metal
# 6 Cardboard
# 7 Mixed Rigid
# 8 PET Oil
#
# New one-based COCO category IDs:
# 1 ECAL
# 2 HDPE
# 3 Mixed Rigid
# 4 Mixed Soft
# 5 nonplastic
# 6 PET
# 7 PET Oil
# ==========================================================================================

ORIGINAL_COCO_TO_NEW = {
    1: 6,  # PET
    2: 2,  # HDPE
    3: 4,  # Mixed Soft
    4: 1,  # ECAL
    5: 5,  # Metal -> nonplastic
    6: 5,  # Cardboard -> nonplastic
    7: 3,  # Mixed Rigid
    8: 7,  # PET Oil
}


# ==========================================================================================
# 5. BUILD TEST GT — 7 CLASS
# ==========================================================================================

print("\nPreparing frozen 7-class test ground truth...")


with open(
    TEST_COCO_JSON,
    "r",
    encoding="utf-8"
) as f:

    original_gt = json.load(f)


gt_7class = {

    "images":
        original_gt["images"],

    "annotations":
        [],

    "categories":
        [
            {
                "id": i + 1,
                "name": name,
            }

            for i, name
            in enumerate(CLASS_NAMES)
        ],
}


for ann in original_gt["annotations"]:

    new_ann = ann.copy()

    new_ann["category_id"] = (
        ORIGINAL_COCO_TO_NEW[
            int(ann["category_id"])
        ]
    )

    gt_7class["annotations"].append(
        new_ann
    )


with open(
    GT_7CLASS_PATH,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        gt_7class,
        f
    )


print(
    f"Test images : "
    f"{len(gt_7class['images']):,}"
)

print(
    f"Test GT     : "
    f"{len(gt_7class['annotations']):,}"
)


# ==========================================================================================
# 6. TEST GT CLASS COUNTS
# ==========================================================================================

gt_class_counts = Counter()

for ann in gt_7class["annotations"]:

    gt_class_counts[
        int(ann["category_id"]) - 1
    ] += 1


print("\nTest GT distribution:")

for idx, class_name in enumerate(
    CLASS_NAMES
):

    print(
        f"{class_name:25s}: "
        f"{gt_class_counts[idx]:,}"
    )


# ==========================================================================================
# 7. CLASSIFIER TRANSFORM
# ==========================================================================================

IMAGE_SIZE = 224

classifier_transform = transforms.Compose(
    [
        transforms.Resize(
            (IMAGE_SIZE, IMAGE_SIZE)
        ),

        transforms.ToTensor(),

        transforms.Normalize(
            mean=[
                0.485,
                0.456,
                0.406
            ],

            std=[
                0.229,
                0.224,
                0.225
            ],
        ),
    ]
)


# ==========================================================================================
# 8. FROZEN MOBILE NET GATES — E16-A
# ==========================================================================================

MOBILENET_GATES = {
    ECAL: 0.99,
    HDPE: 0.98,
    MIXED_RIGID: 0.98,
    MIXED_SOFT: 0.94,
    NON_PLASTIC: 0.94,
    PET: 0.99,
    PET_OIL: 0.99,
}


# ==========================================================================================
# 9. FROZEN E19-C CONVNEXT-SMALL CONFIGURATION
# ==========================================================================================

E16_HARD_CLASSES = {
    MIXED_RIGID,
    MIXED_SOFT,
    NON_PLASTIC,
    PET_OIL,
}


# Mixed Rigid explicitly excluded.
ALLOWED_CONVNEXT_CLASSES = {
    MIXED_SOFT,
    NON_PLASTIC,
    PET_OIL,
}


CONVNEXT_GATES = {
    MIXED_SOFT: 0.99,
    NON_PLASTIC: 0.99,
    PET_OIL: 0.96,
}


# ConvNeXt four-class local order:
#
# 0 Mixed Rigid
# 1 Mixed Soft
# 2 nonplastic
# 3 PET Oil

CONV_LOCAL_TO_GLOBAL = {
    0: MIXED_RIGID,
    1: MIXED_SOFT,
    2: NON_PLASTIC,
    3: PET_OIL,
}


# ==========================================================================================
# 10. FROZEN CONFIDENCE FUSION
# ==========================================================================================

ALPHA = 0.70


# ==========================================================================================
# 11. FROZEN YOLO SETTINGS
# ==========================================================================================

YOLO_CONF = 0.001
YOLO_NMS_IOU = 0.60
YOLO_MAX_DET = 100
YOLO_IMGSZ = 640


# ==========================================================================================
# 12. LOAD MOBILE NET
# ==========================================================================================

print("\nLoading MobileNet...")


mobilenet = mobilenet_v3_large(
    weights=None
)

mn_in_features = (
    mobilenet.classifier[3].in_features
)

mobilenet.classifier[3] = nn.Linear(
    mn_in_features,
    7
)


mn_checkpoint = torch.load(
    MOBILENET_PATH,
    map_location=DEVICE,
    weights_only=False,
)


if (
    isinstance(mn_checkpoint, dict)
    and
    "model_state_dict" in mn_checkpoint
):

    mn_state = mn_checkpoint[
        "model_state_dict"
    ]

else:

    mn_state = mn_checkpoint


mobilenet.load_state_dict(
    mn_state
)

mobilenet = mobilenet.to(
    DEVICE
)

mobilenet.eval()

print("MobileNet loaded.")


# ==========================================================================================
# 13. LOAD CONVNEXT-SMALL
# ==========================================================================================

print("\nLoading ConvNeXt-Small...")


convnext = convnext_small(
    weights=None
)

conv_in_features = (
    convnext.classifier[2].in_features
)

convnext.classifier[2] = nn.Linear(
    conv_in_features,
    4
)


conv_checkpoint = torch.load(
    CONVNEXT_PATH,
    map_location=DEVICE,
    weights_only=False,
)


if (
    isinstance(conv_checkpoint, dict)
    and
    "model_state_dict" in conv_checkpoint
):

    conv_state = conv_checkpoint[
        "model_state_dict"
    ]

else:

    conv_state = conv_checkpoint


convnext.load_state_dict(
    conv_state
)

convnext = convnext.to(
    DEVICE
)

convnext.eval()

print("ConvNeXt-Small loaded.")


# ==========================================================================================
# 14. LOAD YOLO11m
# ==========================================================================================

print("\nLoading YOLO11m...")

yolo = YOLO(
    str(YOLO_PATH)
)

print("YOLO11m loaded.")


# ==========================================================================================
# 15. COUNTERS
# ==========================================================================================

predictions = []

source_counts = Counter()

conv_proposed_counts = Counter()
conv_accepted_counts = Counter()
conv_rejected_counts = Counter()

total_yolo_detections = 0
total_valid_crops = 0
invalid_crops = 0
conv_candidates = 0


# ==========================================================================================
# 16. TEST INFERENCE
# ==========================================================================================

image_records = sorted(
    gt_7class["images"],
    key=lambda x: int(x["id"])
)


start_time = time.time()


for image_number, image_record in enumerate(
    image_records,
    start=1,
):

    image_id = int(
        image_record["id"]
    )

    filename = image_record[
        "file_name"
    ]

    image_path = (
        TEST_IMAGES
        / filename
    )


    if not image_path.exists():

        raise FileNotFoundError(
            f"Missing test image:\n"
            f"{image_path}"
        )


    pil_image = Image.open(
        image_path
    ).convert(
        "RGB"
    )

    image_width, image_height = (
        pil_image.size
    )


    # --------------------------------------------------------------------------------------
    # YOLO11m detection
    # --------------------------------------------------------------------------------------

    yolo_result = yolo.predict(
        source=str(image_path),
        imgsz=YOLO_IMGSZ,
        conf=YOLO_CONF,
        iou=YOLO_NMS_IOU,
        max_det=YOLO_MAX_DET,
        verbose=False,
        device=(
            0
            if DEVICE.type == "cuda"
            else "cpu"
        ),
    )[0]


    boxes = yolo_result.boxes

    if boxes is None:
        continue


    xyxy_array = (
        boxes.xyxy
        .detach()
        .cpu()
        .numpy()
    )

    confidence_array = (
        boxes.conf
        .detach()
        .cpu()
        .numpy()
    )

    yolo_class_array = (
        boxes.cls
        .detach()
        .cpu()
        .numpy()
        .astype(int)
    )


    total_yolo_detections += (
        len(xyxy_array)
    )


    # --------------------------------------------------------------------------------------
    # Every detector prediction
    # --------------------------------------------------------------------------------------

    for (
        bbox_xyxy,
        yolo_conf,
        yolo_idx,
    ) in zip(
        xyxy_array,
        confidence_array,
        yolo_class_array,
    ):

        x1, y1, x2, y2 = [
            float(v)
            for v in bbox_xyxy
        ]


        crop_x1 = max(
            0,
            int(np.floor(x1))
        )

        crop_y1 = max(
            0,
            int(np.floor(y1))
        )

        crop_x2 = min(
            image_width,
            int(np.ceil(x2))
        )

        crop_y2 = min(
            image_height,
            int(np.ceil(y2))
        )


        if (
            crop_x2 <= crop_x1
            or
            crop_y2 <= crop_y1
        ):

            invalid_crops += 1
            continue


        crop = pil_image.crop(
            (
                crop_x1,
                crop_y1,
                crop_x2,
                crop_y2,
            )
        )


        input_tensor = (
            classifier_transform(
                crop
            )
            .unsqueeze(0)
            .to(DEVICE)
        )


        total_valid_crops += 1


        # ==================================================================================
        # A. MOBILENET
        # ==================================================================================

        with torch.no_grad():

            if DEVICE.type == "cuda":

                with torch.amp.autocast(
                    device_type="cuda",
                    dtype=torch.float16,
                ):

                    mn_logits = mobilenet(
                        input_tensor
                    )

            else:

                mn_logits = mobilenet(
                    input_tensor
                )


        mn_probs = torch.softmax(
            mn_logits.float(),
            dim=1
        )[0]


        mn_top_prob, mn_top_idx = (
            torch.max(
                mn_probs,
                dim=0
            )
        )


        mn_top_prob = float(
            mn_top_prob.item()
        )

        mn_top_idx = int(
            mn_top_idx.item()
        )


        # ==================================================================================
        # B. E16-A MOBILE NET CLASS GATING
        # ==================================================================================

        if (
            mn_top_prob
            >= MOBILENET_GATES[
                mn_top_idx
            ]
        ):

            e16_final_idx = (
                mn_top_idx
            )

            e16_source = (
                "mobilenet"
            )

        else:

            e16_final_idx = int(
                yolo_idx
            )

            e16_source = (
                "yolo"
            )


        final_idx = (
            e16_final_idx
        )

        final_source = (
            e16_source
        )


        # MobileNet probability for ACTUAL E16 final class.
        classifier_prob = float(
            mn_probs[
                final_idx
            ].item()
        )


        # ==================================================================================
        # C. CONVNEXT-SMALL SPECIALIST
        # ==================================================================================

        if (
            e16_final_idx
            in E16_HARD_CLASSES
        ):

            conv_candidates += 1


            with torch.no_grad():

                if DEVICE.type == "cuda":

                    with torch.amp.autocast(
                        device_type="cuda",
                        dtype=torch.float16,
                    ):

                        conv_logits = convnext(
                            input_tensor
                        )

                else:

                    conv_logits = convnext(
                        input_tensor
                    )


            conv_probs = torch.softmax(
                conv_logits.float(),
                dim=1
            )[0]


            (
                conv_top_prob,
                conv_local_idx,
            ) = torch.max(
                conv_probs,
                dim=0
            )


            conv_top_prob = float(
                conv_top_prob.item()
            )

            conv_local_idx = int(
                conv_local_idx.item()
            )


            conv_global_idx = (
                CONV_LOCAL_TO_GLOBAL[
                    conv_local_idx
                ]
            )


            conv_name = (
                CLASS_NAMES[
                    conv_global_idx
                ]
            )


            conv_proposed_counts[
                conv_name
            ] += 1


            # --------------------------------------------------------------------------------
            # Only allow E19-C selected specialist classes.
            # Mixed Rigid stays disabled.
            # --------------------------------------------------------------------------------

            if (
                conv_global_idx
                in ALLOWED_CONVNEXT_CLASSES
            ):

                conv_gate = (
                    CONVNEXT_GATES[
                        conv_global_idx
                    ]
                )


                if (
                    conv_top_prob
                    >= conv_gate
                ):

                    final_idx = (
                        conv_global_idx
                    )

                    final_source = (
                        "convnext"
                    )

                    classifier_prob = (
                        conv_top_prob
                    )


                    conv_accepted_counts[
                        conv_name
                    ] += 1


                else:

                    conv_rejected_counts[
                        conv_name
                    ] += 1


        # ==================================================================================
        # D. FINAL CONFIDENCE
        #
        # S = C_YOLO^0.70 * P_classifier(final class)^0.30
        # ==================================================================================

        classifier_prob = max(
            classifier_prob,
            1e-12
        )


        final_score = (
            float(yolo_conf)
            ** ALPHA
        ) * (
            classifier_prob
            ** (1.0 - ALPHA)
        )


        source_counts[
            final_source
        ] += 1


        # ==================================================================================
        # E. COCO FORMAT
        # ==================================================================================

        bbox_width = (
            x2 - x1
        )

        bbox_height = (
            y2 - y1
        )


        predictions.append(
            {
                "image_id":
                    image_id,

                "category_id":
                    int(
                        final_idx + 1
                    ),

                "bbox":
                    [
                        float(x1),
                        float(y1),
                        float(bbox_width),
                        float(bbox_height),
                    ],

                "score":
                    float(
                        final_score
                    ),
            }
        )


    if image_number % 25 == 0:

        print(
            f"Processing "
            f"{image_number}/"
            f"{len(image_records)}"
        )


inference_minutes = (
    time.time()
    - start_time
) / 60.0


print("\nInference complete.")

print(
    f"YOLO detections       : "
    f"{total_yolo_detections:,}"
)

print(
    f"Valid crops           : "
    f"{total_valid_crops:,}"
)

print(
    f"Invalid crops         : "
    f"{invalid_crops:,}"
)

print(
    f"Final predictions     : "
    f"{len(predictions):,}"
)

print(
    f"ConvNeXt candidates   : "
    f"{conv_candidates:,}"
)

print(
    f"Inference time        : "
    f"{inference_minutes:.2f} min"
)


# ==========================================================================================
# 17. SAVE RAW TEST PREDICTIONS
# ==========================================================================================

with open(
    PREDICTIONS_PATH,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        predictions,
        f
    )


# ==========================================================================================
# 18. COCO TEST EVALUATION
# ==========================================================================================

print("\n" + "=" * 100)
print("FINAL E19-C TEST — COCO EVALUATION")
print("=" * 100)


coco_gt = COCO(
    str(GT_7CLASS_PATH)
)

coco_dt = coco_gt.loadRes(
    predictions
)

evaluator = COCOeval(
    coco_gt,
    coco_dt,
    "bbox"
)

evaluator.params.maxDets = [
    1,
    10,
    100
]

evaluator.evaluate()
evaluator.accumulate()
evaluator.summarize()


AP50_95 = float(
    evaluator.stats[0]
)

AP50 = float(
    evaluator.stats[1]
)

AP75 = float(
    evaluator.stats[2]
)

AR1 = float(
    evaluator.stats[6]
)

AR10 = float(
    evaluator.stats[7]
)

AR100 = float(
    evaluator.stats[8]
)


# ==========================================================================================
# 19. CLASS-WISE AP50 AND AP50-95
# ==========================================================================================

precision = evaluator.eval[
    "precision"
]

iou_thresholds = (
    evaluator.params.iouThrs
)


ap50_index = int(
    np.where(
        np.isclose(
            iou_thresholds,
            0.50
        )
    )[0][0]
)


class_metrics = {}


print("\n" + "=" * 100)
print("FINAL E19-C TEST — CLASS-WISE RESULTS")
print("=" * 100)


print(
    f"\n{'Class':28s}"
    f"{'GT':>10s}"
    f"{'AP50':>14s}"
    f"{'AP50-95':>14s}"
)


for class_idx, class_name in enumerate(
    CLASS_NAMES
):

    # AP50-95
    values = precision[
        :,
        :,
        class_idx,
        0,
        -1
    ]

    values = values[
        values > -1
    ]

    class_ap = (
        float(np.mean(values))
        if values.size
        else float("nan")
    )


    # AP50
    values50 = precision[
        ap50_index,
        :,
        class_idx,
        0,
        -1
    ]

    values50 = values50[
        values50 > -1
    ]

    class_ap50 = (
        float(np.mean(values50))
        if values50.size
        else float("nan")
    )


    class_metrics[
        class_name
    ] = {

        "gt_count":
            int(
                gt_class_counts[
                    class_idx
                ]
            ),

        "AP50":
            class_ap50,

        "AP50_95":
            class_ap,
    }


    print(
        f"{class_name:28s}"
        f"{gt_class_counts[class_idx]:10,d}"
        f"{class_ap50 * 100:14.2f}"
        f"{class_ap * 100:14.2f}"
    )


# ==========================================================================================
# 20. PLASTIC-ONLY SUMMARY
#
# This is NOT another COCO evaluation.
# It is only a descriptive mean of the six unchanged plastic classes.
# ==========================================================================================

PLASTIC_CLASSES = [
    "ecal",
    "hdpe",
    "mixed_plastic_rigid",
    "mixed_plastic_soft",
    "pet",
    "pet_oil",
]


plastic_ap50_values = [
    class_metrics[name]["AP50"]
    for name in PLASTIC_CLASSES
]

plastic_ap_values = [
    class_metrics[name]["AP50_95"]
    for name in PLASTIC_CLASSES
]


PLASTIC_MEAN_AP50 = float(
    np.mean(
        plastic_ap50_values
    )
)

PLASTIC_MEAN_AP = float(
    np.mean(
        plastic_ap_values
    )
)


print("\n" + "=" * 100)
print("PLASTIC-CLASS DESCRIPTIVE SUMMARY")
print("=" * 100)

print(
    f"\nMean AP50 over six plastic classes    : "
    f"{PLASTIC_MEAN_AP50 * 100:.4f}%"
)

print(
    f"Mean AP50-95 over six plastic classes : "
    f"{PLASTIC_MEAN_AP * 100:.4f}%"
)


# ==========================================================================================
# 21. VALIDATION VS TEST
# ==========================================================================================

E19C_VAL_AP = 0.458788
E19C_VAL_AP50 = 0.612189
E19C_VAL_AP75 = 0.522138
E19C_VAL_AR100 = 0.613334


print("\n" + "=" * 100)
print("E19-C VALIDATION vs TEST")
print("=" * 100)


print(
    f"\n{'Metric':12s}"
    f"{'Validation':>16s}"
    f"{'Test':>16s}"
    f"{'Delta':>16s}"
)


comparison = [
    (
        "AP50-95",
        E19C_VAL_AP,
        AP50_95,
    ),

    (
        "AP50",
        E19C_VAL_AP50,
        AP50,
    ),

    (
        "AP75",
        E19C_VAL_AP75,
        AP75,
    ),

    (
        "AR100",
        E19C_VAL_AR100,
        AR100,
    ),
]


for metric, val_value, test_value in comparison:

    print(
        f"{metric:12s}"
        f"{val_value * 100:16.4f}"
        f"{test_value * 100:16.4f}"
        f"{(test_value-val_value) * 100:+16.4f}"
    )


# ==========================================================================================
# 22. SORTWASTE PAPER REFERENCE
#
# Paper YOLOv11 overall test:
# AP    = 45.1%
# AP50  = 56.7%
#
# IMPORTANT:
# Our taxonomy is 7 classes because Metal/Cardboard are merged.
# Therefore this overall comparison is informative, not perfectly task-identical.
# ==========================================================================================

PAPER_AP = 0.451
PAPER_AP50 = 0.567


print("\n" + "=" * 100)
print("SORTWASTE PAPER YOLOv11 TEST REFERENCE")
print("=" * 100)


print(
    f"\nPaper overall AP50-95 : "
    f"{PAPER_AP * 100:.2f}%"
)

print(
    f"E19-C test AP50-95    : "
    f"{AP50_95 * 100:.2f}%"
)

print(
    f"Difference             : "
    f"{(AP50_95 - PAPER_AP) * 100:+.2f} pp"
)


print(
    f"\nPaper overall AP50     : "
    f"{PAPER_AP50 * 100:.2f}%"
)

print(
    f"E19-C test AP50        : "
    f"{AP50 * 100:.2f}%"
)

print(
    f"Difference             : "
    f"{(AP50 - PAPER_AP50) * 100:+.2f} pp"
)


# ==========================================================================================
# 23. SOURCE COUNTS
# ==========================================================================================

print("\n" + "=" * 100)
print("FINAL CLASS SOURCE COUNTS")
print("=" * 100)


for source in [
    "yolo",
    "mobilenet",
    "convnext",
]:

    count = source_counts.get(
        source,
        0
    )

    percent = (
        count
        / len(predictions)
        * 100
        if predictions
        else 0
    )

    print(
        f"{source:15s}: "
        f"{count:8,d} "
        f"({percent:6.2f}%)"
    )


print(
    "\nConvNeXt proposed:"
)

for class_name in [
    "mixed_plastic_rigid",
    "mixed_plastic_soft",
    "non_plastic",
    "pet_oil",
]:

    print(
        f"{class_name:25s}: "
        f"{conv_proposed_counts.get(class_name, 0):,}"
    )


print(
    "\nConvNeXt accepted:"
)

for class_name in [
    "mixed_plastic_soft",
    "non_plastic",
    "pet_oil",
]:

    print(
        f"{class_name:25s}: "
        f"{conv_accepted_counts.get(class_name, 0):,}"
    )


# ==========================================================================================
# 24. SAVE RESULTS
# ==========================================================================================

results = {

    "experiment":
        "FINAL TEST — E19-C",

    "description":
        (
            "YOLO11m + MobileNetV3-Large + "
            "ConvNeXt-Small with frozen "
            "class-specific gating and "
            "confidence fusion"
        ),

    "split":
        "test",

    "test_images":
        len(
            gt_7class["images"]
        ),

    "test_gt":
        len(
            gt_7class["annotations"]
        ),

    "configuration": {

        "yolo_imgsz":
            YOLO_IMGSZ,

        "yolo_conf":
            YOLO_CONF,

        "yolo_nms":
            YOLO_NMS_IOU,

        "yolo_max_det":
            YOLO_MAX_DET,

        "alpha":
            ALPHA,

        "mobilenet_gates": {
            CLASS_NAMES[k]:
                float(v)

            for k, v
            in MOBILENET_GATES.items()
        },

        "convnext_allowed_classes":
            [
                "mixed_plastic_soft",
                "non_plastic",
                "pet_oil",
            ],

        "mixed_rigid_convnext":
            "disabled",

        "convnext_gates": {
            CLASS_NAMES[k]:
                float(v)

            for k, v
            in CONVNEXT_GATES.items()
        },
    },

    "metrics": {

        "AP50_95":
            AP50_95,

        "AP50":
            AP50,

        "AP75":
            AP75,

        "AR1":
            AR1,

        "AR10":
            AR10,

        "AR100":
            AR100,
    },

    "class_metrics":
        class_metrics,

    "plastic_descriptive_means": {

        "classes":
            PLASTIC_CLASSES,

        "mean_AP50":
            PLASTIC_MEAN_AP50,

        "mean_AP50_95":
            PLASTIC_MEAN_AP,
    },

    "validation_reference": {

        "AP50_95":
            E19C_VAL_AP,

        "AP50":
            E19C_VAL_AP50,

        "AP75":
            E19C_VAL_AP75,

        "AR100":
            E19C_VAL_AR100,
    },

    "sortwaste_paper_reference": {

        "YOLOv11_test_AP50_95":
            PAPER_AP,

        "YOLOv11_test_AP50":
            PAPER_AP50,
    },

    "counts": {

        "yolo_detections":
            total_yolo_detections,

        "valid_crops":
            total_valid_crops,

        "invalid_crops":
            invalid_crops,

        "predictions":
            len(predictions),

        "convnext_candidates":
            conv_candidates,

        "source_counts":
            dict(source_counts),

        "convnext_proposed":
            dict(
                conv_proposed_counts
            ),

        "convnext_accepted":
            dict(
                conv_accepted_counts
            ),
    },

    "inference_minutes":
        inference_minutes,
}


with open(
    RESULTS_PATH,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        results,
        f,
        indent=2
    )


# ==========================================================================================
# 25. SAVE SUMMARY
# ==========================================================================================

summary = f"""
====================================================================================================
FINAL TEST — E19-C
YOLO11m + MobileNetV3-Large + ConvNeXt-Small
====================================================================================================

Split:
TEST

Test images:
{len(gt_7class["images"]):,}

Test ground-truth objects:
{len(gt_7class["annotations"]):,}

----------------------------------------------------------------------------------------------------
FROZEN E19-C CONFIGURATION
----------------------------------------------------------------------------------------------------

YOLO:
imgsz   = {YOLO_IMGSZ}
conf    = {YOLO_CONF}
NMS     = {YOLO_NMS_IOU}
max_det = {YOLO_MAX_DET}

MobileNet:
E16-A class-specific gating

ConvNeXt-Small specialist routing:
Mixed Rigid = DISABLED
Mixed Soft  = gate 0.99
nonplastic  = gate 0.99
PET Oil     = gate 0.96

Confidence fusion:
alpha = {ALPHA}

S = C_YOLO^0.70 * P_classifier(final class)^0.30

----------------------------------------------------------------------------------------------------
FINAL TEST METRICS
----------------------------------------------------------------------------------------------------

AP50-95 = {AP50_95 * 100:.4f}%
AP50    = {AP50 * 100:.4f}%
AP75    = {AP75 * 100:.4f}%

AR1     = {AR1 * 100:.4f}%
AR10    = {AR10 * 100:.4f}%
AR100   = {AR100 * 100:.4f}%

----------------------------------------------------------------------------------------------------
PLASTIC-CLASS DESCRIPTIVE MEANS
----------------------------------------------------------------------------------------------------

Mean six-plastic AP50:
{PLASTIC_MEAN_AP50 * 100:.4f}%

Mean six-plastic AP50-95:
{PLASTIC_MEAN_AP * 100:.4f}%

----------------------------------------------------------------------------------------------------
VALIDATION -> TEST
----------------------------------------------------------------------------------------------------

Validation AP50-95:
{E19C_VAL_AP * 100:.4f}%

Test AP50-95:
{AP50_95 * 100:.4f}%

Difference:
{(AP50_95 - E19C_VAL_AP) * 100:+.4f} pp

----------------------------------------------------------------------------------------------------
SORTWASTE PAPER YOLOv11 TEST REFERENCE
----------------------------------------------------------------------------------------------------

Paper AP50-95:
{PAPER_AP * 100:.2f}%

E19-C AP50-95:
{AP50_95 * 100:.4f}%

Difference:
{(AP50_95 - PAPER_AP) * 100:+.4f} pp

Paper AP50:
{PAPER_AP50 * 100:.2f}%

E19-C AP50:
{AP50 * 100:.4f}%

Difference:
{(AP50 - PAPER_AP50) * 100:+.4f} pp

NOTE:
The SortWaste paper reports results on its original taxonomy.
E19-C uses the thesis 7-class taxonomy in which Metal and Cardboard
are merged into non_plastic. Therefore overall metrics are informative
but not perfectly taxonomy-identical.

----------------------------------------------------------------------------------------------------
INFERENCE
----------------------------------------------------------------------------------------------------

YOLO detections:
{total_yolo_detections:,}

Valid crops:
{total_valid_crops:,}

Invalid crops:
{invalid_crops:,}

Final predictions:
{len(predictions):,}

ConvNeXt candidates:
{conv_candidates:,}

YOLO final source:
{source_counts.get("yolo", 0):,}

MobileNet final source:
{source_counts.get("mobilenet", 0):,}

ConvNeXt final source:
{source_counts.get("convnext", 0):,}

Inference time:
{inference_minutes:.2f} minutes

====================================================================================================
"""


with open(
    SUMMARY_PATH,
    "w",
    encoding="utf-8"
) as f:

    f.write(summary)


# ==========================================================================================
# 26. COMPLETE
# ==========================================================================================

print("\n" + "=" * 100)
print("FINAL E19-C TEST COMPLETE")
print("=" * 100)


print(
    f"\nFINAL TEST AP50-95 : "
    f"{AP50_95 * 100:.4f}%"
)

print(
    f"FINAL TEST AP50    : "
    f"{AP50 * 100:.4f}%"
)

print(
    f"FINAL TEST AP75    : "
    f"{AP75 * 100:.4f}%"
)

print(
    f"FINAL TEST AR100   : "
    f"{AR100 * 100:.4f}%"
)


print(
    f"\nvs E19-C validation AP: "
    f"{(AP50_95-E19C_VAL_AP) * 100:+.4f} pp"
)

print(
    f"vs SortWaste YOLO AP:   "
    f"{(AP50_95-PAPER_AP) * 100:+.4f} pp"
)


print(
    f"\nResults:\n"
    f"{RESULTS_PATH}"
)

print(
    f"\nSummary:\n"
    f"{SUMMARY_PATH}"
)

print(
    f"\nPredictions:\n"
    f"{PREDICTIONS_PATH}"
)

FINAL TEST — E19-C YOLO11m + MobileNet + ConvNeXt-Small

Device : cuda
GPU    : NVIDIA GeForce RTX 3050 Ti Laptop GPU

Test images : C:\Users\varda\Documents\_My Computer\COMMON_space\Learning\Upgrad_EPGP_MS\upgrad_IIITB_MS_course\MS_LJMU_Material\Topic Data\SortWaste\dataset\dataset\splited_all_dataset_coco\test\images
Test COCO   : C:\Users\varda\Documents\_My Computer\COMMON_space\Learning\Upgrad_EPGP_MS\upgrad_IIITB_MS_course\MS_LJMU_Material\Topic Data\SortWaste\dataset\dataset\splited_all_dataset_coco\test\annotations\test_coco.json
Output      : C:\Users\varda\Documents\_My Computer\COMMON_space\Learning\Upgrad_EPGP_MS\upgrad_IIITB_MS_course\MS_LJMU_Material\Thesis_Code\runs\sortwaste\FINAL_TEST_E19C

Preparing frozen 7-class test ground truth...
Test images : 776
Test GT     : 12,618

Test GT distribution:
ecal                     : 3,026
hdpe                     : 3,269
mixed_plastic_rigid      : 1,230
mixed_plastic_soft       : 1,817
non_plastic              : 422
pet        

## Batch 1:
## E1 — YOLO11n Baseline
## E2 — Faster R-CNN ResNet50-FPN Baseline

In [ ]:
# ==================================================================================================
# FINAL TEST — BATCH 1
# E1: YOLO11n Baseline
# E2: Faster R-CNN ResNet50-FPN Baseline
#
# Evaluates each model in:
#   A) Native 8-class SortWaste taxonomy
#   B) Standardized 7-class taxonomy (Metal + Cardboard -> non_plastic)
#
# Primary evaluator: pycocotools.COCOeval
# TEST SET ONLY — NO TUNING
# ==================================================================================================

import os
import json
import time
from pathlib import Path
from collections import Counter

import torch
from PIL import Image
from tqdm import tqdm

from ultralytics import YOLO

import torchvision
from torchvision.transforms.functional import pil_to_tensor
from torchvision.models.detection import fasterrcnn_resnet50_fpn
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor

from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval


# ==================================================================================================
# 1. PATHS
# ==================================================================================================

BASE = Path(
    r"C:\Users\varda\Documents\_My Computer\COMMON_space\Learning"
    r"\Upgrad_EPGP_MS\upgrad_IIITB_MS_course\MS_LJMU_Material"
)

DATASET_ROOT = (
    BASE
    / "Topic Data"
    / "SortWaste"
    / "dataset"
    / "dataset"
)

TEST_ROOT = DATASET_ROOT / "splited_all_dataset_coco" / "test"
TEST_IMAGES = TEST_ROOT / "images"
TEST_COCO = TEST_ROOT / "annotations" / "test_coco.json"

THESIS_CODE = BASE / "Thesis_Code"

# E1 — authoritative YOLO11n baseline checkpoint
E1_WEIGHTS = (
    THESIS_CODE
    / "runs"
    / "detect"
    / "runs"
    / "sortwaste"
    / "yolo11n_baseline-2"
    / "weights"
    / "best.pt"
)

# E2 — Faster R-CNN baseline checkpoint
E2_WEIGHTS = (
    THESIS_CODE
    / "runs"
    / "sortwaste"
    / "faster_rcnn_baseline"
    / "best_model.pth"
)

OUT_ROOT = (
    THESIS_CODE
    / "runs"
    / "sortwaste"
    / "FINAL_TEST_BATCH1"
)

E1_OUT = OUT_ROOT / "E1_YOLO11n"
E2_OUT = OUT_ROOT / "E2_FasterRCNN"

E1_OUT.mkdir(parents=True, exist_ok=True)
E2_OUT.mkdir(parents=True, exist_ok=True)


# ==================================================================================================
# 2. DEVICE
# ==================================================================================================

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("=" * 100)
print("FINAL TEST — BATCH 1")
print("=" * 100)
print(f"PyTorch        : {torch.__version__}")
print(f"Torchvision    : {torchvision.__version__}")
print(f"Device         : {DEVICE}")

if torch.cuda.is_available():
    print(f"GPU            : {torch.cuda.get_device_name(0)}")

print()
print(f"Test images    : {TEST_IMAGES}")
print(f"Test COCO      : {TEST_COCO}")
print(f"E1 weights     : {E1_WEIGHTS}")
print(f"E2 weights     : {E2_WEIGHTS}")
print(f"Output         : {OUT_ROOT}")
print()

assert TEST_IMAGES.exists(), f"Missing: {TEST_IMAGES}"
assert TEST_COCO.exists(), f"Missing: {TEST_COCO}"
assert E1_WEIGHTS.exists(), f"Missing E1 checkpoint: {E1_WEIGHTS}"
assert E2_WEIGHTS.exists(), f"Missing E2 checkpoint: {E2_WEIGHTS}"


# ==================================================================================================
# 3. TAXONOMIES
# ==================================================================================================

# Original SortWaste COCO category IDs
ORIGINAL_NAMES = {
    1: "pet",
    2: "hdpe",
    3: "mixed_plastic_soft",
    4: "ecal",
    5: "metal",
    6: "cardboard",
    7: "mixed_plastic_rigid",
    8: "pet_oil",
}

# 7-class standardized taxonomy
CLASS7_NAMES = {
    1: "ecal",
    2: "hdpe",
    3: "mixed_plastic_rigid",
    4: "mixed_plastic_soft",
    5: "non_plastic",
    6: "pet",
    7: "pet_oil",
}

# Original COCO category ID -> standardized 7-class category ID
COCO8_TO_7 = {
    1: 6,  # pet
    2: 2,  # hdpe
    3: 4,  # mixed soft
    4: 1,  # ecal
    5: 5,  # metal -> non_plastic
    6: 5,  # cardboard -> non_plastic
    7: 3,  # mixed rigid
    8: 7,  # pet oil
}

# YOLO original class index -> original COCO category ID
YOLO_TO_COCO8 = {
    0: 1,  # pet
    1: 2,  # hdpe
    2: 3,  # mixed soft
    3: 4,  # ecal
    4: 5,  # metal
    5: 6,  # cardboard
    6: 7,  # mixed rigid
    7: 8,  # pet oil
}


# ==================================================================================================
# 4. LOAD ORIGINAL TEST COCO
# ==================================================================================================

with open(TEST_COCO, "r", encoding="utf-8") as f:
    gt8 = json.load(f)

print("=" * 100)
print("TEST DATASET")
print("=" * 100)

print(f"Images : {len(gt8['images']):,}")
print(f"GT     : {len(gt8['annotations']):,}")

count8 = Counter(a["category_id"] for a in gt8["annotations"])

print("\nNative 8-class GT distribution:")
for cid in sorted(ORIGINAL_NAMES):
    print(f"{ORIGINAL_NAMES[cid]:25s}: {count8[cid]:,}")


# ==================================================================================================
# 5. CREATE STANDARDIZED 7-CLASS TEST GT
# ==================================================================================================

gt7 = {
    "info": gt8.get("info", {}),
    "licenses": gt8.get("licenses", []),
    "images": gt8["images"],
    "annotations": [],
    "categories": [
        {"id": cid, "name": name}
        for cid, name in CLASS7_NAMES.items()
    ],
}

for ann in gt8["annotations"]:
    new_ann = ann.copy()
    new_ann["category_id"] = COCO8_TO_7[ann["category_id"]]
    gt7["annotations"].append(new_ann)

GT7_PATH = OUT_ROOT / "test_gt_7class.json"

with open(GT7_PATH, "w", encoding="utf-8") as f:
    json.dump(gt7, f)

count7 = Counter(a["category_id"] for a in gt7["annotations"])

print("\nStandardized 7-class GT distribution:")
for cid in sorted(CLASS7_NAMES):
    print(f"{CLASS7_NAMES[cid]:25s}: {count7[cid]:,}")

print()


# ==================================================================================================
# 6. HELPER — COCO EVALUATION
# ==================================================================================================

def evaluate_coco(gt_json_path, predictions, class_names, title):
    """
    Returns:
        overall dict
        per_class dict
    """

    print("\n" + "=" * 100)
    print(title)
    print("=" * 100)

    coco_gt = COCO(str(gt_json_path))

    if len(predictions) == 0:
        raise RuntimeError("No predictions were generated.")

    coco_dt = coco_gt.loadRes(predictions)

    evaluator = COCOeval(coco_gt, coco_dt, "bbox")
    evaluator.params.maxDets = [1, 10, 100]

    evaluator.evaluate()
    evaluator.accumulate()
    evaluator.summarize()

    overall = {
        "AP50-95": float(evaluator.stats[0]),
        "AP50": float(evaluator.stats[1]),
        "AP75": float(evaluator.stats[2]),
        "AR1": float(evaluator.stats[6]),
        "AR10": float(evaluator.stats[7]),
        "AR100": float(evaluator.stats[8]),
    }

    # ------------------------------------------------------------------------------------------------
    # Class-wise AP extraction
    # precision shape:
    # [IoU, Recall, Category, Area, MaxDets]
    # ------------------------------------------------------------------------------------------------

    precision = evaluator.eval["precision"]

    per_class = {}

    cat_ids = evaluator.params.catIds

    for k, cat_id in enumerate(cat_ids):

        name = class_names[cat_id]

        # AP50-95
        p_all = precision[:, :, k, 0, -1]
        valid_all = p_all[p_all > -1]
        ap = float(valid_all.mean()) if valid_all.size else float("nan")

        # AP50
        iou_50_idx = 0
        p50 = precision[iou_50_idx, :, k, 0, -1]
        valid50 = p50[p50 > -1]
        ap50 = float(valid50.mean()) if valid50.size else float("nan")

        per_class[name] = {
            "AP50": ap50,
            "AP50-95": ap,
        }

    print("\nCLASS-WISE RESULTS")
    print("-" * 75)
    print(f"{'Class':28s} {'AP50':>12s} {'AP50-95':>12s}")
    print("-" * 75)

    for name, values in per_class.items():
        print(
            f"{name:28s} "
            f"{values['AP50'] * 100:11.2f}% "
            f"{values['AP50-95'] * 100:11.2f}%"
        )

    return overall, per_class


# ==================================================================================================
# 7. HELPER — SAVE RESULTS
# ==================================================================================================

def save_results(
    output_dir,
    experiment_name,
    predictions8,
    predictions7,
    overall8,
    class8,
    overall7,
    class7,
    inference_time,
):

    output_dir.mkdir(parents=True, exist_ok=True)

    with open(output_dir / "predictions_8class.json", "w", encoding="utf-8") as f:
        json.dump(predictions8, f)

    with open(output_dir / "predictions_7class.json", "w", encoding="utf-8") as f:
        json.dump(predictions7, f)

    result = {
        "experiment": experiment_name,
        "inference_time_minutes": inference_time / 60.0,
        "native_8class": {
            "overall": overall8,
            "per_class": class8,
        },
        "remapped_7class": {
            "overall": overall7,
            "per_class": class7,
        },
    }

    with open(output_dir / "results.json", "w", encoding="utf-8") as f:
        json.dump(result, f, indent=2)

    summary_path = output_dir / "summary.txt"

    with open(summary_path, "w", encoding="utf-8") as f:

        f.write("=" * 90 + "\n")
        f.write(f"{experiment_name}\n")
        f.write("=" * 90 + "\n\n")

        f.write("NATIVE 8-CLASS TEST\n")
        f.write("-" * 60 + "\n")
        for key, val in overall8.items():
            f.write(f"{key:12s}: {val * 100:.4f}%\n")

        f.write("\nClass-wise:\n")
        for cls, vals in class8.items():
            f.write(
                f"{cls:25s} "
                f"AP50={vals['AP50'] * 100:.4f}%   "
                f"AP50-95={vals['AP50-95'] * 100:.4f}%\n"
            )

        f.write("\n\nSTANDARDIZED 7-CLASS TEST\n")
        f.write("-" * 60 + "\n")
        for key, val in overall7.items():
            f.write(f"{key:12s}: {val * 100:.4f}%\n")

        f.write("\nClass-wise:\n")
        for cls, vals in class7.items():
            f.write(
                f"{cls:25s} "
                f"AP50={vals['AP50'] * 100:.4f}%   "
                f"AP50-95={vals['AP50-95'] * 100:.4f}%\n"
            )

        f.write(f"\nInference time: {inference_time / 60:.2f} min\n")

    print(f"\nSaved: {summary_path}")


# ==================================================================================================
# 8. HELPER — REMAP 8-CLASS PREDICTIONS TO 7 CLASS
# ==================================================================================================

def remap_predictions_8_to_7(predictions8):

    predictions7 = []

    for p in predictions8:
        q = p.copy()
        q["category_id"] = COCO8_TO_7[p["category_id"]]
        predictions7.append(q)

    return predictions7


# ==================================================================================================
# 9. IMAGE ID LOOKUP
# ==================================================================================================

image_records = gt8["images"]

image_id_by_filename = {
    img["file_name"]: img["id"]
    for img in image_records
}

# Some COCO files may store subpaths. Also index basename for robustness.
image_id_by_basename = {
    Path(img["file_name"]).name: img["id"]
    for img in image_records
}


# ==================================================================================================
# 10. E1 — YOLO11n BASELINE
# ==================================================================================================

print("\n\n")
print("#" * 100)
print("E1 — YOLO11n BASELINE")
print("#" * 100)

torch.cuda.empty_cache()

e1_model = YOLO(str(E1_WEIGHTS))

e1_predictions8 = []

start = time.time()

for idx, img_info in enumerate(tqdm(image_records, desc="E1 YOLO11n")):

    image_path = TEST_IMAGES / Path(img_info["file_name"]).name
    image_id = img_info["id"]

    results = e1_model.predict(
        source=str(image_path),
        imgsz=640,
        conf=0.001,
        iou=0.70,
        max_det=100,
        device=0 if torch.cuda.is_available() else "cpu",
        verbose=False,
    )

    r = results[0]

    if r.boxes is None or len(r.boxes) == 0:
        continue

    xyxy = r.boxes.xyxy.detach().cpu().numpy()
    confs = r.boxes.conf.detach().cpu().numpy()
    clss = r.boxes.cls.detach().cpu().numpy().astype(int)

    for box, score, yolo_cls in zip(xyxy, confs, clss):

        if yolo_cls not in YOLO_TO_COCO8:
            continue

        x1, y1, x2, y2 = box

        w = max(0.0, x2 - x1)
        h = max(0.0, y2 - y1)

        if w <= 0 or h <= 0:
            continue

        e1_predictions8.append(
            {
                "image_id": int(image_id),
                "category_id": int(YOLO_TO_COCO8[yolo_cls]),
                "bbox": [
                    float(x1),
                    float(y1),
                    float(w),
                    float(h),
                ],
                "score": float(score),
            }
        )

e1_time = time.time() - start

print(f"\nE1 inference complete.")
print(f"Predictions : {len(e1_predictions8):,}")
print(f"Time        : {e1_time / 60:.2f} min")


# --------------------------------------------------------------------------------------------------
# E1 — 8 class evaluation
# --------------------------------------------------------------------------------------------------

e1_overall8, e1_class8 = evaluate_coco(
    TEST_COCO,
    e1_predictions8,
    ORIGINAL_NAMES,
    "E1 — YOLO11n BASELINE — NATIVE 8-CLASS TEST",
)


# --------------------------------------------------------------------------------------------------
# E1 — 7 class evaluation
# --------------------------------------------------------------------------------------------------

e1_predictions7 = remap_predictions_8_to_7(e1_predictions8)

e1_overall7, e1_class7 = evaluate_coco(
    GT7_PATH,
    e1_predictions7,
    CLASS7_NAMES,
    "E1 — YOLO11n BASELINE — STANDARDIZED 7-CLASS TEST",
)


save_results(
    E1_OUT,
    "E1 — YOLO11n Baseline",
    e1_predictions8,
    e1_predictions7,
    e1_overall8,
    e1_class8,
    e1_overall7,
    e1_class7,
    e1_time,
)

del e1_model
torch.cuda.empty_cache()


# ==================================================================================================
# 11. E2 — FASTER R-CNN RESNET50-FPN BASELINE
# ==================================================================================================

print("\n\n")
print("#" * 100)
print("E2 — FASTER R-CNN RESNET50-FPN BASELINE")
print("#" * 100)

torch.cuda.empty_cache()


# --------------------------------------------------------------------------------------------------
# Reconstruct baseline architecture
# 8 SortWaste classes + background = 9 classes
#
# Training baseline used 640-size images.
# --------------------------------------------------------------------------------------------------

e2_model = fasterrcnn_resnet50_fpn(
    weights=None,
    weights_backbone=None,
    num_classes=91,
    min_size=640,
    max_size=640,
)

in_features = e2_model.roi_heads.box_predictor.cls_score.in_features

e2_model.roi_heads.box_predictor = FastRCNNPredictor(
    in_features,
    9
)


# --------------------------------------------------------------------------------------------------
# Load checkpoint robustly
# --------------------------------------------------------------------------------------------------

checkpoint = torch.load(
    E2_WEIGHTS,
    map_location=DEVICE,
    weights_only=False,
)

if isinstance(checkpoint, dict):

    if "model_state_dict" in checkpoint:
        state_dict = checkpoint["model_state_dict"]

    elif "state_dict" in checkpoint:
        state_dict = checkpoint["state_dict"]

    elif "model" in checkpoint and isinstance(checkpoint["model"], dict):
        state_dict = checkpoint["model"]

    else:
        # checkpoint itself may be a plain state_dict
        state_dict = checkpoint

else:
    state_dict = checkpoint


# Remove "module." prefix if saved from DataParallel
clean_state_dict = {}

for key, value in state_dict.items():

    if key.startswith("module."):
        key = key[len("module."):]

    clean_state_dict[key] = value


e2_model.load_state_dict(clean_state_dict, strict=True)

# Evaluation settings — keep low-confidence detections for COCO ranking
e2_model.roi_heads.score_thresh = 0.001
e2_model.roi_heads.nms_thresh = 0.5
e2_model.roi_heads.detections_per_img = 100

e2_model.to(DEVICE)
e2_model.eval()

print("Faster R-CNN loaded successfully.")


# --------------------------------------------------------------------------------------------------
# Run inference
# --------------------------------------------------------------------------------------------------

e2_predictions8 = []

start = time.time()

with torch.no_grad():

    for idx, img_info in enumerate(tqdm(image_records, desc="E2 Faster R-CNN")):

        image_path = TEST_IMAGES / Path(img_info["file_name"]).name
        image_id = img_info["id"]

        image = Image.open(image_path).convert("RGB")

        tensor = pil_to_tensor(image).float() / 255.0
        tensor = tensor.to(DEVICE)

        outputs = e2_model([tensor])[0]

        boxes = outputs["boxes"].detach().cpu()
        scores = outputs["scores"].detach().cpu()
        labels = outputs["labels"].detach().cpu()

        for box, score, label in zip(boxes, scores, labels):

            label = int(label)

            # Faster R-CNN label 0 = background
            # labels 1..8 correspond to original COCO categories
            if label < 1 or label > 8:
                continue

            x1, y1, x2, y2 = box.tolist()

            w = max(0.0, x2 - x1)
            h = max(0.0, y2 - y1)

            if w <= 0 or h <= 0:
                continue

            e2_predictions8.append(
                {
                    "image_id": int(image_id),
                    "category_id": label,
                    "bbox": [
                        float(x1),
                        float(y1),
                        float(w),
                        float(h),
                    ],
                    "score": float(score),
                }
            )

e2_time = time.time() - start

print(f"\nE2 inference complete.")
print(f"Predictions : {len(e2_predictions8):,}")
print(f"Time        : {e2_time / 60:.2f} min")


# --------------------------------------------------------------------------------------------------
# E2 — Native 8 class evaluation
# --------------------------------------------------------------------------------------------------

e2_overall8, e2_class8 = evaluate_coco(
    TEST_COCO,
    e2_predictions8,
    ORIGINAL_NAMES,
    "E2 — FASTER R-CNN BASELINE — NATIVE 8-CLASS TEST",
)


# --------------------------------------------------------------------------------------------------
# E2 — standardized 7 class evaluation
# --------------------------------------------------------------------------------------------------

e2_predictions7 = remap_predictions_8_to_7(e2_predictions8)

e2_overall7, e2_class7 = evaluate_coco(
    GT7_PATH,
    e2_predictions7,
    CLASS7_NAMES,
    "E2 — FASTER R-CNN BASELINE — STANDARDIZED 7-CLASS TEST",
)


save_results(
    E2_OUT,
    "E2 — Faster R-CNN ResNet50-FPN Baseline",
    e2_predictions8,
    e2_predictions7,
    e2_overall8,
    e2_class8,
    e2_overall7,
    e2_class7,
    e2_time,
)

del e2_model
torch.cuda.empty_cache()


# ==================================================================================================
# 12. FINAL BATCH-1 SUMMARY
# ==================================================================================================

print("\n\n")
print("=" * 100)
print("FINAL TEST — BATCH 1 COMPLETE")
print("=" * 100)

print("\nNATIVE 8-CLASS RESULTS")
print("-" * 80)
print(
    f"{'Experiment':32s}"
    f"{'AP50-95':>12s}"
    f"{'AP50':>12s}"
    f"{'AP75':>12s}"
    f"{'AR100':>12s}"
)

print(
    f"{'E1 YOLO11n':32s}"
    f"{e1_overall8['AP50-95']*100:11.4f}%"
    f"{e1_overall8['AP50']*100:11.4f}%"
    f"{e1_overall8['AP75']*100:11.4f}%"
    f"{e1_overall8['AR100']*100:11.4f}%"
)

print(
    f"{'E2 Faster R-CNN':32s}"
    f"{e2_overall8['AP50-95']*100:11.4f}%"
    f"{e2_overall8['AP50']*100:11.4f}%"
    f"{e2_overall8['AP75']*100:11.4f}%"
    f"{e2_overall8['AR100']*100:11.4f}%"
)


print("\nSTANDARDIZED 7-CLASS RESULTS")
print("-" * 80)
print(
    f"{'Experiment':32s}"
    f"{'AP50-95':>12s}"
    f"{'AP50':>12s}"
    f"{'AP75':>12s}"
    f"{'AR100':>12s}"
)

print(
    f"{'E1 YOLO11n':32s}"
    f"{e1_overall7['AP50-95']*100:11.4f}%"
    f"{e1_overall7['AP50']*100:11.4f}%"
    f"{e1_overall7['AP75']*100:11.4f}%"
    f"{e1_overall7['AR100']*100:11.4f}%"
)

print(
    f"{'E2 Faster R-CNN':32s}"
    f"{e2_overall7['AP50-95']*100:11.4f}%"
    f"{e2_overall7['AP50']*100:11.4f}%"
    f"{e2_overall7['AP75']*100:11.4f}%"
    f"{e2_overall7['AR100']*100:11.4f}%"
)

print("\nSortWaste paper YOLOv11 reference:")
print("AP50-95 : 45.10%")
print("AP50     : 56.70%")

print("\nOutputs:")
print(E1_OUT)
print(E2_OUT)

print("\n" + "=" * 100)

FINAL TEST — BATCH 1
PyTorch        : 2.13.0+cu126
Torchvision    : 0.28.0+cu126
Device         : cuda
GPU            : NVIDIA GeForce RTX 3050 Ti Laptop GPU

Test images    : C:\Users\varda\Documents\_My Computer\COMMON_space\Learning\Upgrad_EPGP_MS\upgrad_IIITB_MS_course\MS_LJMU_Material\Topic Data\SortWaste\dataset\dataset\splited_all_dataset_coco\test\images
Test COCO      : C:\Users\varda\Documents\_My Computer\COMMON_space\Learning\Upgrad_EPGP_MS\upgrad_IIITB_MS_course\MS_LJMU_Material\Topic Data\SortWaste\dataset\dataset\splited_all_dataset_coco\test\annotations\test_coco.json
E1 weights     : C:\Users\varda\Documents\_My Computer\COMMON_space\Learning\Upgrad_EPGP_MS\upgrad_IIITB_MS_course\MS_LJMU_Material\Thesis_Code\runs\detect\runs\sortwaste\yolo11n_baseline-2\weights\best.pt
E2 weights     : C:\Users\varda\Documents\_My Computer\COMMON_space\Learning\Upgrad_EPGP_MS\upgrad_IIITB_MS_course\MS_LJMU_Material\Thesis_Code\runs\sortwaste\faster_rcnn_baseline\best_model.pth
Output  

E1 YOLO11n: 100%|██████████| 776/776 [01:24<00:00,  9.21it/s]



E1 inference complete.
Predictions : 63,597
Time        : 1.40 min

E1 — YOLO11n BASELINE — NATIVE 8-CLASS TEST
loading annotations into memory...
Done (t=0.08s)
creating index...
index created!
Loading and preparing results...
DONE (t=16.64s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=4.70s).
Accumulating evaluation results...
DONE (t=0.76s).
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.273
 Average Precision  (AP) @[ IoU=0.50      | area=   all | maxDets=100 ] = 0.386
 Average Precision  (AP) @[ IoU=0.75      | area=   all | maxDets=100 ] = 0.297
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= small | maxDets=100 ] = -1.000
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=medium | maxDets=100 ] = 0.074
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= large | maxDets=100 ] = 0.280
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets=  1 ] = 0.210
 Average Recall     (AR) @[ Io

E2 Faster R-CNN: 100%|██████████| 776/776 [05:36<00:00,  2.31it/s]



E2 inference complete.
Predictions : 62,891
Time        : 5.61 min

E2 — FASTER R-CNN BASELINE — NATIVE 8-CLASS TEST
loading annotations into memory...
Done (t=0.05s)
creating index...
index created!
Loading and preparing results...
DONE (t=0.06s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=3.90s).
Accumulating evaluation results...
DONE (t=0.66s).
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.360
 Average Precision  (AP) @[ IoU=0.50      | area=   all | maxDets=100 ] = 0.522
 Average Precision  (AP) @[ IoU=0.75      | area=   all | maxDets=100 ] = 0.414
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= small | maxDets=100 ] = -1.000
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=medium | maxDets=100 ] = 0.177
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= large | maxDets=100 ] = 0.367
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets=  1 ] = 0.282
 Average Recall     (AR) @

## Batch 2
| Experiment | Pipeline                                                  |
| ---------- | --------------------------------------------------------- |
| E3Y-A      | YOLO11n → MobileNetV3-Large, unweighted                   |
| E3Y-B      | YOLO11n → MobileNetV3-Large, class-weighted               |
| E3Y-P      | YOLO11n → MobileNetV3-Large, full-GT-crop trained variant |
| E4F-A      | Faster R-CNN → MobileNetV3-Large, class-weighted          |


In [ ]:
# FINAL TEST — BATCH 2 — CLEAN RERUN / OVERWRITE OLD PARTIAL RESULTS
#
# E3Y-A : YOLO11n -> MobileNetV3-Large, unweighted
# E3Y-B : YOLO11n -> MobileNetV3-Large, class-weighted
# E3Y-P : YOLO11n -> E3P MobileNetV3-Large
# E4F-A : Faster R-CNN -> MobileNetV3-Large, class-weighted
#
# IMPORTANT:
#   - Deletes previous FINAL_TEST_BATCH2 output folder first
#   - Re-runs all four experiments from scratch
#   - No test-set tuning
#   - Explicit pycocotools COCOeval
#   - Standardized 7-class taxonomy
# ==================================================================================================

import json
import time
import shutil
from pathlib import Path
from collections import Counter

import numpy as np
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F

from torchvision.models import mobilenet_v3_large
from torchvision.models.detection import fasterrcnn_resnet50_fpn
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torchvision.transforms import Compose, Resize, ToTensor, Normalize
from torchvision.transforms.functional import pil_to_tensor

from ultralytics import YOLO

from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval


# ==================================================================================================
# 1. PATHS
# ==================================================================================================

BASE = Path(
    r"C:\Users\varda\Documents\_My Computer\COMMON_space\Learning"
    r"\Upgrad_EPGP_MS\upgrad_IIITB_MS_course\MS_LJMU_Material"
)

DATASET_ROOT = (
    BASE
    / "Topic Data"
    / "SortWaste"
    / "dataset"
    / "dataset"
)

THESIS_CODE = BASE / "Thesis_Code"

TEST_ROOT = DATASET_ROOT / "splited_all_dataset_coco" / "test"
TEST_IMAGES = TEST_ROOT / "images"
TEST_COCO = TEST_ROOT / "annotations" / "test_coco.json"


# --------------------------------------------------------------------------------------------------
# Detector checkpoints
# --------------------------------------------------------------------------------------------------

E1_YOLO_WEIGHTS = (
    THESIS_CODE
    / "runs"
    / "detect"
    / "runs"
    / "sortwaste"
    / "yolo11n_baseline-2"
    / "weights"
    / "best.pt"
)

E2_FRCNN_WEIGHTS = (
    THESIS_CODE
    / "runs"
    / "sortwaste"
    / "faster_rcnn_baseline"
    / "best_model.pth"
)


# --------------------------------------------------------------------------------------------------
# MobileNet checkpoints — authoritative notebook-derived paths
# --------------------------------------------------------------------------------------------------

E3YA_WEIGHTS = (
    DATASET_ROOT
    / "yolo_mobilenet_crops_E3Y"
    / "mobilenet_results"
    / "E3Y_A_baseline"
    / "E3Y_MobileNetV3Large_best.pth"
)

E3YB_WEIGHTS = (
    DATASET_ROOT
    / "yolo_mobilenet_crops_E3Y"
    / "mobilenet_results"
    / "E3Y_B_class_weighted"
    / "E3Y_B_MobileNetV3Large_best.pth"
)

E3YP_WEIGHTS = (
    DATASET_ROOT
    / "cropped_data_plastic"
    / "mobilenet_v3_large_E3P_best.pth"
)

E4FA_WEIGHTS = (
    DATASET_ROOT
    / "faster_rcnn_mobilenet_crops_E4F"
    / "mobilenet_results"
    / "E4F-A_MobileNetV3Large"
    / "E4FA_MobileNetV3Large_best.pth"
)


# --------------------------------------------------------------------------------------------------
# Output — delete previous partial Batch 2 completely
# --------------------------------------------------------------------------------------------------

OUT_ROOT = (
    THESIS_CODE
    / "runs"
    / "sortwaste"
    / "FINAL_TEST_BATCH2"
)

if OUT_ROOT.exists():
    print(f"Deleting old partial Batch 2 results:\n{OUT_ROOT}")
    shutil.rmtree(OUT_ROOT)

OUT_ROOT.mkdir(parents=True, exist_ok=True)


# ==================================================================================================
# 2. DEVICE
# ==================================================================================================

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("=" * 100)
print("FINAL TEST — BATCH 2 — CLEAN RERUN")
print("=" * 100)

print(f"PyTorch        : {torch.__version__}")
print(f"Device         : {DEVICE}")

if torch.cuda.is_available():
    print(f"GPU            : {torch.cuda.get_device_name(0)}")

print()
print(f"Test images    : {TEST_IMAGES}")
print(f"Test COCO      : {TEST_COCO}")
print(f"E1 YOLO        : {E1_YOLO_WEIGHTS}")
print(f"E2 Faster R-CNN: {E2_FRCNN_WEIGHTS}")
print(f"E3Y-A          : {E3YA_WEIGHTS}")
print(f"E3Y-B          : {E3YB_WEIGHTS}")
print(f"E3Y-P          : {E3YP_WEIGHTS}")
print(f"E4F-A          : {E4FA_WEIGHTS}")
print(f"Output         : {OUT_ROOT}")


required_paths = [
    TEST_IMAGES,
    TEST_COCO,
    E1_YOLO_WEIGHTS,
    E2_FRCNN_WEIGHTS,
    E3YA_WEIGHTS,
    E3YB_WEIGHTS,
    E3YP_WEIGHTS,
    E4FA_WEIGHTS,
]

for p in required_paths:
    if not p.exists():
        raise FileNotFoundError(f"Missing required path:\n{p}")


# ==================================================================================================
# 3. STANDARDIZED 7-CLASS TAXONOMY
# ==================================================================================================

CLASS7_NAMES = {
    1: "ecal",
    2: "hdpe",
    3: "mixed_plastic_rigid",
    4: "mixed_plastic_soft",
    5: "non_plastic",
    6: "pet",
    7: "pet_oil",
}

COCO8_TO_7 = {
    1: 6,   # pet
    2: 2,   # hdpe
    3: 4,   # mixed soft
    4: 1,   # ecal
    5: 5,   # metal -> non_plastic
    6: 5,   # cardboard -> non_plastic
    7: 3,   # mixed rigid
    8: 7,   # pet oil
}


# ==================================================================================================
# 4. MOBILENET 7-CLASS MAPPING
#
# Authoritative 7-class folder/index order used by E3Y-A, E3Y-B, E3Y-P, E4F-A:
#
# 0 ecal
# 1 hdpe
# 2 mixed_plastic_rigid
# 3 mixed_plastic_soft
# 4 non_plastic
# 5 pet
# 6 pet_oil
# ==================================================================================================

MN7_IDX_TO_COCO7 = {
    0: 1,
    1: 2,
    2: 3,
    3: 4,
    4: 5,
    5: 6,
    6: 7,
}


# ==================================================================================================
# 5. CREATE STANDARDIZED 7-CLASS TEST GT
# ==================================================================================================

with open(TEST_COCO, "r", encoding="utf-8") as f:
    gt8 = json.load(f)

gt7 = {
    "info": gt8.get("info", {}),
    "licenses": gt8.get("licenses", []),
    "images": gt8["images"],
    "annotations": [],
    "categories": [
        {"id": cid, "name": name}
        for cid, name in CLASS7_NAMES.items()
    ],
}

for ann in gt8["annotations"]:
    new_ann = ann.copy()
    new_ann["category_id"] = COCO8_TO_7[int(ann["category_id"])]
    gt7["annotations"].append(new_ann)

GT7_PATH = OUT_ROOT / "test_gt_7class.json"

with open(GT7_PATH, "w", encoding="utf-8") as f:
    json.dump(gt7, f)

print()
print("=" * 100)
print("TEST DATASET")
print("=" * 100)
print(f"Images : {len(gt7['images']):,}")
print(f"GT     : {len(gt7['annotations']):,}")

counts = Counter(a["category_id"] for a in gt7["annotations"])

print("\nStandardized 7-class GT distribution:")
for cid, name in CLASS7_NAMES.items():
    print(f"{name:25s}: {counts[cid]:,}")


# ==================================================================================================
# 6. MOBILENET TRANSFORM
# ==================================================================================================

MN_TRANSFORM = Compose([
    Resize((224, 224)),
    ToTensor(),
    Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225],
    ),
])


# ==================================================================================================
# 7. CHECKPOINT HELPERS
# ==================================================================================================

def extract_state_dict(checkpoint):

    if not isinstance(checkpoint, dict):
        return checkpoint

    if "model_state_dict" in checkpoint:
        state = checkpoint["model_state_dict"]

    elif "state_dict" in checkpoint:
        state = checkpoint["state_dict"]

    elif "model" in checkpoint and isinstance(checkpoint["model"], dict):
        state = checkpoint["model"]

    else:
        state = checkpoint

    clean_state = {}

    for key, value in state.items():

        if key.startswith("module."):
            key = key[len("module."):]

        clean_state[key] = value

    return clean_state


def load_mobilenet(checkpoint_path, experiment_name):

    checkpoint = torch.load(
        checkpoint_path,
        map_location=DEVICE,
        weights_only=False,
    )

    state = extract_state_dict(checkpoint)

    final_weight_key = "classifier.3.weight"

    if final_weight_key not in state:
        raise RuntimeError(
            f"{experiment_name}: cannot find {final_weight_key}"
        )

    num_classes = int(state[final_weight_key].shape[0])

    print()
    print(f"{experiment_name}")
    print(f"Checkpoint     : {checkpoint_path}")
    print(f"Output classes : {num_classes}")

    # All four Batch-2 classifier checkpoints are confirmed 7-output models.
    if num_classes != 7:
        raise RuntimeError(
            f"{experiment_name}: expected 7 outputs, found {num_classes}"
        )

    model = mobilenet_v3_large(weights=None)

    in_features = model.classifier[3].in_features

    model.classifier[3] = nn.Linear(
        in_features,
        7,
    )

    model.load_state_dict(
        state,
        strict=True,
    )

    model.to(DEVICE)
    model.eval()

    return model


# ==================================================================================================
# 8. CLASSIFY DETECTOR CROPS
# ==================================================================================================

@torch.no_grad()
def classify_boxes(
    image,
    boxes,
    classifier,
    batch_size=64,
):

    W, H = image.size

    crops = []
    valid_positions = []

    for idx, box in enumerate(boxes):

        x1, y1, x2, y2 = [float(v) for v in box]

        x1 = max(0.0, min(x1, W - 1))
        y1 = max(0.0, min(y1, H - 1))
        x2 = max(0.0, min(x2, W))
        y2 = max(0.0, min(y2, H))

        if x2 <= x1 or y2 <= y1:
            continue

        crop = image.crop((x1, y1, x2, y2))

        crops.append(
            MN_TRANSFORM(crop)
        )

        valid_positions.append(idx)

    outputs = [None] * len(boxes)

    for start in range(0, len(crops), batch_size):

        batch = torch.stack(
            crops[start:start + batch_size]
        ).to(DEVICE)

        logits = classifier(batch)

        probs = F.softmax(
            logits,
            dim=1,
        )

        confs, preds = probs.max(dim=1)

        confs = confs.cpu().numpy()
        preds = preds.cpu().numpy()

        for local_idx, (pred_idx, prob) in enumerate(
            zip(preds, confs)
        ):

            original_idx = valid_positions[start + local_idx]

            category_id = MN7_IDX_TO_COCO7[int(pred_idx)]

            outputs[original_idx] = (
                int(category_id),
                float(prob),
            )

    return outputs


# ==================================================================================================
# 9. COCO EVALUATION
# ==================================================================================================

def evaluate_coco(predictions, title):

    print()
    print("=" * 100)
    print(title)
    print("=" * 100)

    coco_gt = COCO(str(GT7_PATH))
    coco_dt = coco_gt.loadRes(predictions)

    evaluator = COCOeval(
        coco_gt,
        coco_dt,
        "bbox",
    )

    evaluator.params.maxDets = [1, 10, 100]

    evaluator.evaluate()
    evaluator.accumulate()
    evaluator.summarize()

    overall = {
        "AP50-95": float(evaluator.stats[0]),
        "AP50": float(evaluator.stats[1]),
        "AP75": float(evaluator.stats[2]),
        "AR1": float(evaluator.stats[6]),
        "AR10": float(evaluator.stats[7]),
        "AR100": float(evaluator.stats[8]),
    }

    precision = evaluator.eval["precision"]

    per_class = {}

    for k, category_id in enumerate(evaluator.params.catIds):

        class_name = CLASS7_NAMES[category_id]

        vals = precision[:, :, k, 0, -1]
        valid = vals[vals > -1]

        ap5095 = (
            float(valid.mean())
            if valid.size
            else float("nan")
        )

        vals50 = precision[0, :, k, 0, -1]
        valid50 = vals50[vals50 > -1]

        ap50 = (
            float(valid50.mean())
            if valid50.size
            else float("nan")
        )

        per_class[class_name] = {
            "AP50": ap50,
            "AP50-95": ap5095,
        }

    print()
    print("CLASS-WISE RESULTS")
    print("-" * 76)
    print(
        f"{'Class':30s}"
        f"{'AP50':>15s}"
        f"{'AP50-95':>15s}"
    )
    print("-" * 76)

    for class_name, vals in per_class.items():

        print(
            f"{class_name:30s}"
            f"{vals['AP50']*100:14.2f}%"
            f"{vals['AP50-95']*100:14.2f}%"
        )

    return overall, per_class


# ==================================================================================================
# 10. SAVE RESULTS
# ==================================================================================================

def save_results(
    experiment_name,
    predictions,
    overall,
    per_class,
    inference_minutes,
    detector_boxes,
    invalid_crops,
    output_dir,
):

    output_dir.mkdir(
        parents=True,
        exist_ok=True,
    )

    with open(
        output_dir / "predictions.json",
        "w",
        encoding="utf-8",
    ) as f:
        json.dump(predictions, f)

    result_data = {
        "experiment": experiment_name,
        "taxonomy": "standardized 7-class",
        "inference_minutes": float(inference_minutes),
        "detector_boxes": int(detector_boxes),
        "final_predictions": int(len(predictions)),
        "invalid_crops": int(invalid_crops),
        "overall": overall,
        "per_class": per_class,
    }

    with open(
        output_dir / "results.json",
        "w",
        encoding="utf-8",
    ) as f:
        json.dump(
            result_data,
            f,
            indent=2,
        )

    with open(
        output_dir / "summary.txt",
        "w",
        encoding="utf-8",
    ) as f:

        f.write(experiment_name + "\n")
        f.write("=" * 90 + "\n\n")

        f.write(f"AP50-95 : {overall['AP50-95']*100:.4f}%\n")
        f.write(f"AP50    : {overall['AP50']*100:.4f}%\n")
        f.write(f"AP75    : {overall['AP75']*100:.4f}%\n")
        f.write(f"AR100   : {overall['AR100']*100:.4f}%\n")

        f.write("\nClass-wise:\n")

        for cls, vals in per_class.items():

            f.write(
                f"{cls:25s} "
                f"AP50={vals['AP50']*100:.4f}% "
                f"AP50-95={vals['AP50-95']*100:.4f}%\n"
            )

        f.write(f"\nDetector boxes : {detector_boxes:,}\n")
        f.write(f"Predictions    : {len(predictions):,}\n")
        f.write(f"Invalid crops  : {invalid_crops:,}\n")
        f.write(f"Inference time : {inference_minutes:.2f} min\n")


# ==================================================================================================
# 11. YOLO11n -> MOBILENET PIPELINE
# ==================================================================================================

def run_yolo_mobilenet(
    short_name,
    experiment_name,
    classifier_weights,
):

    print("\n\n" + "#" * 100)
    print(experiment_name)
    print("#" * 100)

    torch.cuda.empty_cache()

    classifier = load_mobilenet(
        classifier_weights,
        experiment_name,
    )

    detector = YOLO(
        str(E1_YOLO_WEIGHTS)
    )

    predictions = []

    detector_boxes = 0
    invalid_crops = 0

    start_time = time.time()

    total_images = len(gt7["images"])

    for image_number, image_info in enumerate(
        gt7["images"],
        start=1,
    ):

        image_id = int(image_info["id"])

        image_path = (
            TEST_IMAGES
            / Path(image_info["file_name"]).name
        )

        image = Image.open(
            image_path
        ).convert("RGB")

        result = detector.predict(
            source=str(image_path),
            imgsz=640,
            conf=0.001,
            iou=0.70,
            max_det=100,
            device=0 if torch.cuda.is_available() else "cpu",
            verbose=False,
        )[0]

        if result.boxes is None or len(result.boxes) == 0:
            continue

        boxes = (
            result.boxes.xyxy
            .detach()
            .cpu()
            .numpy()
        )

        det_scores = (
            result.boxes.conf
            .detach()
            .cpu()
            .numpy()
        )

        detector_boxes += len(boxes)

        classifications = classify_boxes(
            image,
            boxes,
            classifier,
            batch_size=64,
        )

        for box, det_conf, cls_result in zip(
            boxes,
            det_scores,
            classifications,
        ):

            if cls_result is None:
                invalid_crops += 1
                continue

            final_category_id, classifier_prob = cls_result

            x1, y1, x2, y2 = [float(v) for v in box]

            width = x2 - x1
            height = y2 - y1

            if width <= 0 or height <= 0:
                invalid_crops += 1
                continue

            final_score = (
                float(det_conf)
                * float(classifier_prob)
            )

            predictions.append({
                "image_id": image_id,
                "category_id": int(final_category_id),
                "bbox": [
                    float(x1),
                    float(y1),
                    float(width),
                    float(height),
                ],
                "score": float(final_score),
            })

        if image_number % 25 == 0 or image_number == total_images:

            print(
                f"{short_name}: "
                f"{image_number}/{total_images}"
            )

    elapsed = time.time() - start_time

    print()
    print(f"Detector boxes : {detector_boxes:,}")
    print(f"Predictions    : {len(predictions):,}")
    print(f"Invalid crops  : {invalid_crops:,}")
    print(f"Time           : {elapsed/60:.2f} min")

    overall, per_class = evaluate_coco(
        predictions,
        f"{experiment_name} — TEST",
    )

    save_results(
        experiment_name,
        predictions,
        overall,
        per_class,
        elapsed / 60,
        detector_boxes,
        invalid_crops,
        OUT_ROOT / short_name,
    )

    del classifier
    del detector

    torch.cuda.empty_cache()

    return {
        "overall": overall,
        "per_class": per_class,
    }


# ==================================================================================================
# 12. FASTER R-CNN LOADER
# ==================================================================================================

def load_faster_rcnn_detector():

    model = fasterrcnn_resnet50_fpn(
        weights=None,
        weights_backbone=None,
        num_classes=91,
        min_size=640,
        max_size=640,
    )

    in_features = (
        model.roi_heads
        .box_predictor
        .cls_score
        .in_features
    )

    model.roi_heads.box_predictor = FastRCNNPredictor(
        in_features,
        9,
    )

    checkpoint = torch.load(
        E2_FRCNN_WEIGHTS,
        map_location=DEVICE,
        weights_only=False,
    )

    state = extract_state_dict(
        checkpoint
    )

    model.load_state_dict(
        state,
        strict=True,
    )

    model.roi_heads.score_thresh = 0.001
    model.roi_heads.nms_thresh = 0.5
    model.roi_heads.detections_per_img = 100

    model.to(DEVICE)
    model.eval()

    return model


# ==================================================================================================
# 13. E4F-A
# ==================================================================================================

def run_e4fa():

    short_name = "E4F-A"

    experiment_name = (
        "E4F-A — Faster R-CNN + "
        "MobileNetV3-Large Class-Weighted"
    )

    print("\n\n" + "#" * 100)
    print(experiment_name)
    print("#" * 100)

    torch.cuda.empty_cache()

    detector = load_faster_rcnn_detector()

    classifier = load_mobilenet(
        E4FA_WEIGHTS,
        experiment_name,
    )

    predictions = []

    detector_boxes = 0
    invalid_crops = 0

    start_time = time.time()

    total_images = len(gt7["images"])

    with torch.no_grad():

        for image_number, image_info in enumerate(
            gt7["images"],
            start=1,
        ):

            image_id = int(image_info["id"])

            image_path = (
                TEST_IMAGES
                / Path(image_info["file_name"]).name
            )

            image = Image.open(
                image_path
            ).convert("RGB")

            tensor = (
                pil_to_tensor(image)
                .float()
                .div(255.0)
                .to(DEVICE)
            )

            output = detector([tensor])[0]

            boxes = (
                output["boxes"]
                .detach()
                .cpu()
                .numpy()
            )

            det_scores = (
                output["scores"]
                .detach()
                .cpu()
                .numpy()
            )

            detector_boxes += len(boxes)

            classifications = classify_boxes(
                image,
                boxes,
                classifier,
                batch_size=64,
            )

            for box, det_conf, cls_result in zip(
                boxes,
                det_scores,
                classifications,
            ):

                if cls_result is None:
                    invalid_crops += 1
                    continue

                final_category_id, classifier_prob = cls_result

                x1, y1, x2, y2 = [float(v) for v in box]

                width = x2 - x1
                height = y2 - y1

                if width <= 0 or height <= 0:
                    invalid_crops += 1
                    continue

                final_score = (
                    float(det_conf)
                    * float(classifier_prob)
                )

                predictions.append({
                    "image_id": image_id,
                    "category_id": int(final_category_id),
                    "bbox": [
                        float(x1),
                        float(y1),
                        float(width),
                        float(height),
                    ],
                    "score": float(final_score),
                })

            if image_number % 25 == 0 or image_number == total_images:

                print(
                    f"E4F-A: "
                    f"{image_number}/{total_images}"
                )

    elapsed = time.time() - start_time

    print()
    print(f"Detector boxes : {detector_boxes:,}")
    print(f"Predictions    : {len(predictions):,}")
    print(f"Invalid crops  : {invalid_crops:,}")
    print(f"Time           : {elapsed/60:.2f} min")

    overall, per_class = evaluate_coco(
        predictions,
        f"{experiment_name} — TEST",
    )

    save_results(
        experiment_name,
        predictions,
        overall,
        per_class,
        elapsed / 60,
        detector_boxes,
        invalid_crops,
        OUT_ROOT / short_name,
    )

    del detector
    del classifier

    torch.cuda.empty_cache()

    return {
        "overall": overall,
        "per_class": per_class,
    }


# ==================================================================================================
# 14. RUN ALL BATCH 2 EXPERIMENTS FROM SCRATCH
# ==================================================================================================

results = {}


results["E3Y-A"] = run_yolo_mobilenet(
    short_name="E3Y-A",
    experiment_name=(
        "E3Y-A — YOLO11n + "
        "MobileNetV3-Large Unweighted"
    ),
    classifier_weights=E3YA_WEIGHTS,
)


results["E3Y-B"] = run_yolo_mobilenet(
    short_name="E3Y-B",
    experiment_name=(
        "E3Y-B — YOLO11n + "
        "MobileNetV3-Large Class-Weighted"
    ),
    classifier_weights=E3YB_WEIGHTS,
)


results["E3Y-P"] = run_yolo_mobilenet(
    short_name="E3Y-P",
    experiment_name=(
        "E3Y-P — YOLO11n + "
        "E3P MobileNetV3-Large"
    ),
    classifier_weights=E3YP_WEIGHTS,
)


results["E4F-A"] = run_e4fa()


# ==================================================================================================
# 15. FINAL SUMMARY
# ==================================================================================================

print()
print()
print("=" * 100)
print("FINAL TEST — BATCH 2 COMPLETE")
print("=" * 100)

print(
    f"{'Experiment':15s}"
    f"{'AP50-95':>14s}"
    f"{'AP50':>14s}"
    f"{'AP75':>14s}"
    f"{'AR100':>14s}"
)

print("-" * 75)

for exp, result in results.items():

    o = result["overall"]

    print(
        f"{exp:15s}"
        f"{o['AP50-95']*100:13.4f}%"
        f"{o['AP50']*100:13.4f}%"
        f"{o['AP75']*100:13.4f}%"
        f"{o['AR100']*100:13.4f}%"
    )


# ==================================================================================================
# 16. CLASS-WISE AP50
# ==================================================================================================

print()
print("=" * 100)
print("CLASS-WISE TEST AP50")
print("=" * 100)

print(
    f"{'Class':28s}"
    + "".join(
        f"{exp:>14s}"
        for exp in results
    )
)

print(
    "-" * (
        28
        + 14 * len(results)
    )
)

for class_name in CLASS7_NAMES.values():

    row = f"{class_name:28s}"

    for exp in results:

        value = (
            results[exp]
            ["per_class"]
            [class_name]
            ["AP50"]
        )

        row += f"{value*100:13.2f}%"

    print(row)


# ==================================================================================================
# 17. CLASS-WISE AP50-95
# ==================================================================================================

print()
print("=" * 100)
print("CLASS-WISE TEST AP50-95")
print("=" * 100)

print(
    f"{'Class':28s}"
    + "".join(
        f"{exp:>14s}"
        for exp in results
    )
)

print(
    "-" * (
        28
        + 14 * len(results)
    )
)

for class_name in CLASS7_NAMES.values():

    row = f"{class_name:28s}"

    for exp in results:

        value = (
            results[exp]
            ["per_class"]
            [class_name]
            ["AP50-95"]
        )

        row += f"{value*100:13.2f}%"

    print(row)


print()
print("Outputs saved to:")
print(OUT_ROOT)

print()
print("=" * 100)
print("FINAL TEST — BATCH 2 COMPLETE")
print("=" * 100)

Deleting old partial Batch 2 results:
C:\Users\varda\Documents\_My Computer\COMMON_space\Learning\Upgrad_EPGP_MS\upgrad_IIITB_MS_course\MS_LJMU_Material\Thesis_Code\runs\sortwaste\FINAL_TEST_BATCH2
FINAL TEST — BATCH 2 — CLEAN RERUN
PyTorch        : 2.13.0+cu126
Device         : cuda
GPU            : NVIDIA GeForce RTX 3050 Ti Laptop GPU

Test images    : C:\Users\varda\Documents\_My Computer\COMMON_space\Learning\Upgrad_EPGP_MS\upgrad_IIITB_MS_course\MS_LJMU_Material\Topic Data\SortWaste\dataset\dataset\splited_all_dataset_coco\test\images
Test COCO      : C:\Users\varda\Documents\_My Computer\COMMON_space\Learning\Upgrad_EPGP_MS\upgrad_IIITB_MS_course\MS_LJMU_Material\Topic Data\SortWaste\dataset\dataset\splited_all_dataset_coco\test\annotations\test_coco.json
E1 YOLO        : C:\Users\varda\Documents\_My Computer\COMMON_space\Learning\Upgrad_EPGP_MS\upgrad_IIITB_MS_course\MS_LJMU_Material\Thesis_Code\runs\detect\runs\sortwaste\yolo11n_baseline-2\weights\best.pt
E2 Faster R-CNN: C:\U

## Batch 3
| Experiment | Checkpoint                 | Frozen TEST configuration         |
| ---------- | -------------------------- | --------------------------------- |
| **E5**     | E5 YOLO11s balanced        | imgsz 640, NMS IoU **0.70**       |
| **E6**     | E6 YOLO11s no oversampling | imgsz 640, NMS IoU **0.70**       |
| **E7**     | **same E5 checkpoint**     | imgsz 640, tuned NMS IoU **0.60** |


In [ ]:
# FINAL TEST — BATCH 3
#
# E5 : YOLO11s + 7-Class Taxonomy + Augmentation + Class-Aware Oversampling
# E6 : YOLO11s + 7-Class Taxonomy + Augmentation, No Oversampling
# E7 : E5 checkpoint + validation-selected NMS IoU = 0.60
#
# STANDARDIZED TEST EVALUATION:
#   - 7-class taxonomy
#   - Explicit pycocotools COCOeval
#   - AP@[0.50:0.95], AP50, AP75, AR100
#   - Class-wise AP50 and AP50-95
#
# IMPORTANT:
#   - TEST SET ONLY
#   - NO TEST-SET TUNING
#   - E7 reuses E5 checkpoint
# ==================================================================================================

import json
import time
import shutil
from pathlib import Path
from collections import Counter

import torch

from ultralytics import YOLO

from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval


# ==================================================================================================
# 1. PATHS
# ==================================================================================================

BASE = Path(
    r"C:\Users\varda\Documents\_My Computer\COMMON_space\Learning"
    r"\Upgrad_EPGP_MS\upgrad_IIITB_MS_course\MS_LJMU_Material"
)

DATASET_ROOT = (
    BASE
    / "Topic Data"
    / "SortWaste"
    / "dataset"
    / "dataset"
)

THESIS_CODE = BASE / "Thesis_Code"


# --------------------------------------------------------------------------------------------------
# Test data
# --------------------------------------------------------------------------------------------------

TEST_ROOT = (
    DATASET_ROOT
    / "splited_all_dataset_coco"
    / "test"
)

TEST_IMAGES = TEST_ROOT / "images"

TEST_COCO = (
    TEST_ROOT
    / "annotations"
    / "test_coco.json"
)


# --------------------------------------------------------------------------------------------------
# E5 checkpoint
# --------------------------------------------------------------------------------------------------

E5_WEIGHTS = (
    THESIS_CODE
    / "runs"
    / "sortwaste"
    / "E5_yolo11s_7class_aug_classbalance_640"
    / "weights"
    / "best.pt"
)


# --------------------------------------------------------------------------------------------------
# E6 checkpoint
# --------------------------------------------------------------------------------------------------

E6_WEIGHTS = (
    THESIS_CODE
    / "runs"
    / "sortwaste"
    / "E6_yolo11s_7class_aug_no_oversampling_640"
    / "weights"
    / "best.pt"
)


# --------------------------------------------------------------------------------------------------
# Output folder
# --------------------------------------------------------------------------------------------------

OUT_ROOT = (
    THESIS_CODE
    / "runs"
    / "sortwaste"
    / "FINAL_TEST_BATCH3"
)


# Clean old Batch 3 if it exists
if OUT_ROOT.exists():

    print(
        "Deleting old Batch 3 results:\n"
        f"{OUT_ROOT}"
    )

    shutil.rmtree(
        OUT_ROOT
    )


OUT_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)


# ==================================================================================================
# 2. DEVICE
# ==================================================================================================

DEVICE = (
    0
    if torch.cuda.is_available()
    else "cpu"
)


print("=" * 100)
print("FINAL TEST — BATCH 3")
print("=" * 100)

print(
    f"PyTorch        : "
    f"{torch.__version__}"
)

print(
    f"Device         : "
    f"{'cuda' if torch.cuda.is_available() else 'cpu'}"
)

if torch.cuda.is_available():

    print(
        f"GPU            : "
        f"{torch.cuda.get_device_name(0)}"
    )


print()
print(
    f"Test images    : "
    f"{TEST_IMAGES}"
)

print(
    f"Test COCO      : "
    f"{TEST_COCO}"
)

print(
    f"E5 weights     : "
    f"{E5_WEIGHTS}"
)

print(
    f"E6 weights     : "
    f"{E6_WEIGHTS}"
)

print(
    f"E7 weights     : "
    f"{E5_WEIGHTS}"
)

print(
    f"Output         : "
    f"{OUT_ROOT}"
)


# Verify paths
for p in [
    TEST_IMAGES,
    TEST_COCO,
    E5_WEIGHTS,
    E6_WEIGHTS,
]:

    if not p.exists():

        raise FileNotFoundError(
            f"Missing required path:\n{p}"
        )


# ==================================================================================================
# 3. STANDARDIZED 7-CLASS TAXONOMY
# ==================================================================================================

CLASS7_NAMES = {

    1: "ecal",

    2: "hdpe",

    3: "mixed_plastic_rigid",

    4: "mixed_plastic_soft",

    5: "non_plastic",

    6: "pet",

    7: "pet_oil",
}


# --------------------------------------------------------------------------------------------------
# Original COCO 8-class -> standardized 7-class
# --------------------------------------------------------------------------------------------------

COCO8_TO_7 = {

    1: 6,   # pet

    2: 2,   # hdpe

    3: 4,   # mixed soft

    4: 1,   # ecal

    5: 5,   # metal -> non_plastic

    6: 5,   # cardboard -> non_plastic

    7: 3,   # mixed rigid

    8: 7,   # pet oil
}


# --------------------------------------------------------------------------------------------------
# YOLO 7-class ID -> COCO standardized category ID
#
# YOLO model:
#   0 ecal
#   1 hdpe
#   2 mixed_plastic_rigid
#   3 mixed_plastic_soft
#   4 non_plastic
#   5 pet
#   6 pet_oil
#
# COCO evaluation categories are 1..7
# --------------------------------------------------------------------------------------------------

YOLO7_TO_COCO7 = {

    0: 1,

    1: 2,

    2: 3,

    3: 4,

    4: 5,

    5: 6,

    6: 7,
}


# ==================================================================================================
# 4. CREATE STANDARDIZED 7-CLASS TEST GT
# ==================================================================================================

with open(
    TEST_COCO,
    "r",
    encoding="utf-8",
) as f:

    gt8 = json.load(f)


gt7 = {

    "info":
        gt8.get(
            "info",
            {},
        ),

    "licenses":
        gt8.get(
            "licenses",
            [],
        ),

    "images":
        gt8[
            "images"
        ],

    "annotations":
        [],

    "categories": [

        {
            "id":
                category_id,

            "name":
                class_name,
        }

        for category_id, class_name
        in CLASS7_NAMES.items()
    ],
}


for annotation in gt8[
    "annotations"
]:

    new_annotation = (
        annotation.copy()
    )

    new_annotation[
        "category_id"
    ] = COCO8_TO_7[
        int(
            annotation[
                "category_id"
            ]
        )
    ]

    gt7[
        "annotations"
    ].append(
        new_annotation
    )


GT7_PATH = (
    OUT_ROOT
    / "test_gt_7class.json"
)


with open(
    GT7_PATH,
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        gt7,
        f,
    )


print()
print("=" * 100)
print("TEST DATASET")
print("=" * 100)

print(
    f"Images : "
    f"{len(gt7['images']):,}"
)

print(
    f"GT     : "
    f"{len(gt7['annotations']):,}"
)


gt_counts = Counter(

    annotation[
        "category_id"
    ]

    for annotation
    in gt7[
        "annotations"
    ]
)


print()
print(
    "Standardized 7-class "
    "GT distribution:"
)


for category_id, class_name in (
    CLASS7_NAMES.items()
):

    print(

        f"{class_name:25s}: "

        f"{gt_counts[category_id]:,}"
    )


# ==================================================================================================
# 5. EXPLICIT COCO EVALUATION
# ==================================================================================================

def evaluate_coco(
    predictions,
    title,
):

    print()
    print("=" * 100)
    print(title)
    print("=" * 100)


    if len(
        predictions
    ) == 0:

        raise RuntimeError(
            "No predictions generated."
        )


    coco_gt = COCO(
        str(
            GT7_PATH
        )
    )


    coco_dt = (
        coco_gt.loadRes(
            predictions
        )
    )


    evaluator = COCOeval(

        coco_gt,

        coco_dt,

        "bbox",
    )


    evaluator.params.maxDets = [

        1,

        10,

        100,
    ]


    evaluator.evaluate()

    evaluator.accumulate()

    evaluator.summarize()


    overall = {

        "AP50-95":
            float(
                evaluator.stats[
                    0
                ]
            ),

        "AP50":
            float(
                evaluator.stats[
                    1
                ]
            ),

        "AP75":
            float(
                evaluator.stats[
                    2
                ]
            ),

        "AR1":
            float(
                evaluator.stats[
                    6
                ]
            ),

        "AR10":
            float(
                evaluator.stats[
                    7
                ]
            ),

        "AR100":
            float(
                evaluator.stats[
                    8
                ]
            ),
    }


    # ----------------------------------------------------------------------------------------------
    # Class-wise AP
    # ----------------------------------------------------------------------------------------------

    precision = (
        evaluator.eval[
            "precision"
        ]
    )


    per_class = {}


    for k, category_id in enumerate(
        evaluator.params.catIds
    ):

        class_name = (
            CLASS7_NAMES[
                category_id
            ]
        )


        # AP@[0.50:0.95]
        values = precision[
            :,
            :,
            k,
            0,
            -1,
        ]

        valid = (
            values[
                values > -1
            ]
        )


        ap5095 = (

            float(
                valid.mean()
            )

            if valid.size

            else float(
                "nan"
            )
        )


        # AP50
        values50 = precision[
            0,
            :,
            k,
            0,
            -1,
        ]

        valid50 = (
            values50[
                values50 > -1
            ]
        )


        ap50 = (

            float(
                valid50.mean()
            )

            if valid50.size

            else float(
                "nan"
            )
        )


        per_class[
            class_name
        ] = {

            "AP50":
                ap50,

            "AP50-95":
                ap5095,
        }


    print()
    print(
        "CLASS-WISE RESULTS"
    )

    print(
        "-" * 76
    )

    print(

        f"{'Class':30s}"

        f"{'AP50':>15s}"

        f"{'AP50-95':>15s}"
    )

    print(
        "-" * 76
    )


    for class_name, values in (
        per_class.items()
    ):

        print(

            f"{class_name:30s}"

            f"{values['AP50']*100:14.2f}%"

            f"{values['AP50-95']*100:14.2f}%"
        )


    return (
        overall,
        per_class,
    )


# ==================================================================================================
# 6. SAVE RESULTS
# ==================================================================================================

def save_results(
    experiment_name,
    predictions,
    overall,
    per_class,
    inference_minutes,
    nms_iou,
    output_dir,
):

    output_dir.mkdir(
        parents=True,
        exist_ok=True,
    )


    # Predictions
    with open(
        output_dir
        / "predictions.json",
        "w",
        encoding="utf-8",
    ) as f:

        json.dump(
            predictions,
            f,
        )


    result_data = {

        "experiment":
            experiment_name,

        "taxonomy":
            "standardized 7-class",

        "imgsz":
            640,

        "confidence_threshold":
            0.001,

        "nms_iou":
            float(
                nms_iou
            ),

        "max_det":
            100,

        "inference_minutes":
            float(
                inference_minutes
            ),

        "predictions":
            int(
                len(
                    predictions
                )
            ),

        "overall":
            overall,

        "per_class":
            per_class,
    }


    with open(
        output_dir
        / "results.json",
        "w",
        encoding="utf-8",
    ) as f:

        json.dump(
            result_data,
            f,
            indent=2,
        )


    with open(
        output_dir
        / "summary.txt",
        "w",
        encoding="utf-8",
    ) as f:

        f.write(
            experiment_name
            + "\n"
        )

        f.write(
            "=" * 90
            + "\n\n"
        )


        f.write(
            f"AP50-95 : "
            f"{overall['AP50-95']*100:.4f}%\n"
        )

        f.write(
            f"AP50    : "
            f"{overall['AP50']*100:.4f}%\n"
        )

        f.write(
            f"AP75    : "
            f"{overall['AP75']*100:.4f}%\n"
        )

        f.write(
            f"AR100   : "
            f"{overall['AR100']*100:.4f}%\n"
        )

        f.write(
            f"NMS IoU : "
            f"{nms_iou:.2f}\n"
        )


        f.write(
            "\nClass-wise:\n"
        )


        for class_name, values in (
            per_class.items()
        ):

            f.write(

                f"{class_name:25s}"

                f" AP50="
                f"{values['AP50']*100:.4f}%"

                f" AP50-95="
                f"{values['AP50-95']*100:.4f}%\n"
            )


        f.write(
            f"\nPredictions    : "
            f"{len(predictions):,}\n"
        )

        f.write(
            f"Inference time : "
            f"{inference_minutes:.2f} min\n"
        )


# ==================================================================================================
# 7. RUN YOLO TEST EVALUATION
# ==================================================================================================

def run_yolo_test(
    short_name,
    experiment_name,
    weights,
    nms_iou,
):

    print()
    print()
    print("#" * 100)
    print(experiment_name)
    print("#" * 100)

    print(
        f"Checkpoint : "
        f"{weights}"
    )

    print(
        f"Image size : "
        f"640"
    )

    print(
        f"Conf       : "
        f"0.001"
    )

    print(
        f"NMS IoU    : "
        f"{nms_iou:.2f}"
    )

    print(
        f"Max det    : "
        f"100"
    )


    torch.cuda.empty_cache()


    model = YOLO(
        str(
            weights
        )
    )


    predictions = []


    start_time = (
        time.time()
    )


    total_images = len(
        gt7[
            "images"
        ]
    )


    for image_number, image_info in enumerate(
        gt7[
            "images"
        ],
        start=1,
    ):


        image_id = int(
            image_info[
                "id"
            ]
        )


        image_path = (

            TEST_IMAGES

            / Path(
                image_info[
                    "file_name"
                ]
            ).name
        )


        result = (
            model.predict(

                source=str(
                    image_path
                ),

                imgsz=640,

                conf=0.001,

                iou=nms_iou,

                max_det=100,

                device=DEVICE,

                verbose=False,

            )[0]
        )


        if (
            result.boxes is None
            or len(
                result.boxes
            ) == 0
        ):

            continue


        boxes = (

            result.boxes.xyxy

            .detach()

            .cpu()

            .numpy()
        )


        scores = (

            result.boxes.conf

            .detach()

            .cpu()

            .numpy()
        )


        classes = (

            result.boxes.cls

            .detach()

            .cpu()

            .numpy()
        )


        for box, score, yolo_class in zip(

            boxes,

            scores,

            classes,
        ):


            yolo_class = int(
                yolo_class
            )


            if (
                yolo_class
                not in YOLO7_TO_COCO7
            ):

                raise RuntimeError(

                    f"{short_name}: "
                    f"unexpected YOLO class ID "
                    f"{yolo_class}"
                )


            category_id = (
                YOLO7_TO_COCO7[
                    yolo_class
                ]
            )


            x1, y1, x2, y2 = [

                float(
                    v
                )

                for v
                in box
            ]


            width = (
                x2
                - x1
            )

            height = (
                y2
                - y1
            )


            if (
                width <= 0
                or height <= 0
            ):

                continue


            predictions.append({

                "image_id":
                    image_id,

                "category_id":
                    int(
                        category_id
                    ),

                "bbox": [

                    float(
                        x1
                    ),

                    float(
                        y1
                    ),

                    float(
                        width
                    ),

                    float(
                        height
                    ),
                ],

                "score":
                    float(
                        score
                    ),
            })


        if (

            image_number % 25 == 0

            or image_number
            == total_images

        ):

            print(

                f"{short_name}: "

                f"{image_number}/"
                f"{total_images}"
            )


    elapsed = (
        time.time()
        - start_time
    )


    print()
    print(
        f"Predictions : "
        f"{len(predictions):,}"
    )

    print(
        f"Time        : "
        f"{elapsed/60:.2f} min"
    )


    overall, per_class = (
        evaluate_coco(

            predictions,

            f"{experiment_name} — TEST",
        )
    )


    save_results(

        experiment_name,

        predictions,

        overall,

        per_class,

        elapsed / 60,

        nms_iou,

        OUT_ROOT
        / short_name,
    )


    del model

    torch.cuda.empty_cache()


    return {

        "overall":
            overall,

        "per_class":
            per_class,

        "time":
            elapsed / 60,
    }


# ==================================================================================================
# 8. RUN BATCH 3
# ==================================================================================================

results = {}


# --------------------------------------------------------------------------------------------------
# E5
#
# Original optimized YOLO11s balanced detector.
# Use NMS IoU 0.70.
# --------------------------------------------------------------------------------------------------

results[
    "E5"
] = run_yolo_test(

    short_name="E5",

    experiment_name=(
        "E5 — YOLO11s + Augmentation + "
        "Class-Aware Oversampling"
    ),

    weights=(
        E5_WEIGHTS
    ),

    nms_iou=0.70,
)


# --------------------------------------------------------------------------------------------------
# E6
#
# Controlled no-oversampling detector.
# --------------------------------------------------------------------------------------------------

results[
    "E6"
] = run_yolo_test(

    short_name="E6",

    experiment_name=(
        "E6 — YOLO11s + Augmentation — "
        "No Oversampling"
    ),

    weights=(
        E6_WEIGHTS
    ),

    nms_iou=0.70,
)


# --------------------------------------------------------------------------------------------------
# E7
#
# NO NEW CHECKPOINT.
#
# Reuses E5 detector with validation-selected NMS IoU = 0.60.
# --------------------------------------------------------------------------------------------------

results[
    "E7"
] = run_yolo_test(

    short_name="E7",

    experiment_name=(
        "E7 — E5 YOLO11s + "
        "NMS IoU Tuning"
    ),

    weights=(
        E5_WEIGHTS
    ),

    nms_iou=0.60,
)


# ==================================================================================================
# 9. FINAL OVERALL SUMMARY
# ==================================================================================================

print()
print()
print("=" * 100)
print("FINAL TEST — BATCH 3 COMPLETE")
print("=" * 100)


print()

print(
    f"{'Experiment':15s}"

    f"{'AP50-95':>14s}"

    f"{'AP50':>14s}"

    f"{'AP75':>14s}"

    f"{'AR100':>14s}"

    f"{'Time':>12s}"
)


print(
    "-" * 87
)


for experiment, result in (
    results.items()
):

    overall = (
        result[
            "overall"
        ]
    )


    print(

        f"{experiment:15s}"

        f"{overall['AP50-95']*100:13.4f}%"

        f"{overall['AP50']*100:13.4f}%"

        f"{overall['AP75']*100:13.4f}%"

        f"{overall['AR100']*100:13.4f}%"

        f"{result['time']:11.2f}m"
    )


# ==================================================================================================
# 10. CLASS-WISE AP50
# ==================================================================================================

print()
print("=" * 100)
print("CLASS-WISE TEST AP50")
print("=" * 100)


print(

    f"{'Class':28s}"

    + "".join(

        f"{experiment:>14s}"

        for experiment
        in results
    )
)


print(

    "-" * (

        28

        + 14
        * len(
            results
        )
    )
)


for class_name in (
    CLASS7_NAMES.values()
):

    row = (
        f"{class_name:28s}"
    )


    for experiment in results:

        value = (

            results[
                experiment
            ][
                "per_class"
            ][
                class_name
            ][
                "AP50"
            ]
        )


        row += (
            f"{value*100:13.2f}%"
        )


    print(
        row
    )


# ==================================================================================================
# 11. CLASS-WISE AP50-95
# ==================================================================================================

print()
print("=" * 100)
print("CLASS-WISE TEST AP50-95")
print("=" * 100)


print(

    f"{'Class':28s}"

    + "".join(

        f"{experiment:>14s}"

        for experiment
        in results
    )
)


print(

    "-" * (

        28

        + 14
        * len(
            results
        )
    )
)


for class_name in (
    CLASS7_NAMES.values()
):

    row = (
        f"{class_name:28s}"
    )


    for experiment in results:

        value = (

            results[
                experiment
            ][
                "per_class"
            ][
                class_name
            ][
                "AP50-95"
            ]
        )


        row += (
            f"{value*100:13.2f}%"
        )


    print(
        row
    )


print()
print(
    "Outputs saved to:"
)

print(
    OUT_ROOT
)

print()
print("=" * 100)
print("FINAL TEST — BATCH 3 COMPLETE")
print("=" * 100)

FINAL TEST — BATCH 3
PyTorch        : 2.13.0+cu126
Device         : cuda
GPU            : NVIDIA GeForce RTX 3050 Ti Laptop GPU

Test images    : C:\Users\varda\Documents\_My Computer\COMMON_space\Learning\Upgrad_EPGP_MS\upgrad_IIITB_MS_course\MS_LJMU_Material\Topic Data\SortWaste\dataset\dataset\splited_all_dataset_coco\test\images
Test COCO      : C:\Users\varda\Documents\_My Computer\COMMON_space\Learning\Upgrad_EPGP_MS\upgrad_IIITB_MS_course\MS_LJMU_Material\Topic Data\SortWaste\dataset\dataset\splited_all_dataset_coco\test\annotations\test_coco.json
E5 weights     : C:\Users\varda\Documents\_My Computer\COMMON_space\Learning\Upgrad_EPGP_MS\upgrad_IIITB_MS_course\MS_LJMU_Material\Thesis_Code\runs\sortwaste\E5_yolo11s_7class_aug_classbalance_640\weights\best.pt
E6 weights     : C:\Users\varda\Documents\_My Computer\COMMON_space\Learning\Upgrad_EPGP_MS\upgrad_IIITB_MS_course\MS_LJMU_Material\Thesis_Code\runs\sortwaste\E6_yolo11s_7class_aug_no_oversampling_640\weights\best.pt
E7 weigh

## Batch 4
| Experiment | Detector   | MobileNet                         | Frozen NMS |
| ---------- | ---------- | --------------------------------- | ---------: |
| **E8**     | E5 YOLO11s | E3Y-B weighted MobileNet          |   **0.60** |
| **E9-C**   | E5 YOLO11s | E9-B YOLO11s-matched MobileNet    |   **0.60** |
| **E10**    | E5 YOLO11s | E10 controlled-protocol MobileNet |   **0.60** |


In [ ]:
# FINAL TEST — BATCH 4
#
# E8   : E5 YOLO11s + E3Y-B MobileNet
# E9-C : E5 YOLO11s + E9-B MobileNet
# E10  : E5 YOLO11s + E10 MobileNet
#
# Frozen evaluation:
#   imgsz       = 640
#   conf        = 0.001
#   NMS IoU     = 0.60
#   max_det     = 100
#
# Final class:
#   MobileNet prediction
#
# Final score:
#   YOLO confidence × MobileNet probability(final class)
#
# Evaluation:
#   Explicit pycocotools COCOeval
#   Standardized 7-class taxonomy
#
# IMPORTANT:
#   NO TEST-SET TUNING
# ==================================================================================================

import json
import time
import shutil
from pathlib import Path
from collections import Counter

from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F

from torchvision.models import mobilenet_v3_large
from torchvision.transforms import Compose, Resize, ToTensor, Normalize

from ultralytics import YOLO

from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval


# ==================================================================================================
# 1. PATHS
# ==================================================================================================

BASE = Path(
    r"C:\Users\varda\Documents\_My Computer\COMMON_space\Learning"
    r"\Upgrad_EPGP_MS\upgrad_IIITB_MS_course\MS_LJMU_Material"
)

DATASET_ROOT = (
    BASE
    / "Topic Data"
    / "SortWaste"
    / "dataset"
    / "dataset"
)

THESIS_CODE = BASE / "Thesis_Code"


# --------------------------------------------------------------------------------------------------
# Test data
# --------------------------------------------------------------------------------------------------

TEST_ROOT = (
    DATASET_ROOT
    / "splited_all_dataset_coco"
    / "test"
)

TEST_IMAGES = TEST_ROOT / "images"

TEST_COCO = (
    TEST_ROOT
    / "annotations"
    / "test_coco.json"
)


# --------------------------------------------------------------------------------------------------
# E5 YOLO11s detector — reused by E8 / E9-C / E10
# --------------------------------------------------------------------------------------------------

E5_WEIGHTS = (
    THESIS_CODE
    / "runs"
    / "sortwaste"
    / "E5_yolo11s_7class_aug_classbalance_640"
    / "weights"
    / "best.pt"
)


# --------------------------------------------------------------------------------------------------
# E8 classifier = E3Y-B MobileNet
# --------------------------------------------------------------------------------------------------

E8_MN_WEIGHTS = (
    DATASET_ROOT
    / "yolo_mobilenet_crops_E3Y"
    / "mobilenet_results"
    / "E3Y_B_class_weighted"
    / "E3Y_B_MobileNetV3Large_best.pth"
)


# --------------------------------------------------------------------------------------------------
# E9-C classifier = E9-B MobileNet
# --------------------------------------------------------------------------------------------------

E9C_MN_WEIGHTS = (
    DATASET_ROOT
    / "yolo11s_mobilenet_crops_E9"
    / "mobilenet_results"
    / "E9_YOLO11s_class_weighted"
    / "E9_MobileNetV3Large_best.pth"
)


# --------------------------------------------------------------------------------------------------
# E10 classifier
# --------------------------------------------------------------------------------------------------

E10_MN_WEIGHTS = (
    DATASET_ROOT
    / "yolo11s_mobilenet_crops_E9"
    / "mobilenet_results"
    / "E10_controlled_E3YB_protocol"
    / "E10_MobileNetV3Large_best.pth"
)


# --------------------------------------------------------------------------------------------------
# Output
# --------------------------------------------------------------------------------------------------

OUT_ROOT = (
    THESIS_CODE
    / "runs"
    / "sortwaste"
    / "FINAL_TEST_BATCH4"
)

if OUT_ROOT.exists():

    print(
        "Deleting old Batch 4 results:\n"
        f"{OUT_ROOT}"
    )

    shutil.rmtree(
        OUT_ROOT
    )


OUT_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)


# ==================================================================================================
# 2. DEVICE
# ==================================================================================================

DEVICE = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

YOLO_DEVICE = (
    0
    if torch.cuda.is_available()
    else "cpu"
)


print("=" * 100)
print("FINAL TEST — BATCH 4")
print("=" * 100)

print(f"PyTorch        : {torch.__version__}")
print(f"Device         : {DEVICE}")

if torch.cuda.is_available():

    print(
        f"GPU            : "
        f"{torch.cuda.get_device_name(0)}"
    )


print()
print(f"Test images    : {TEST_IMAGES}")
print(f"Test COCO      : {TEST_COCO}")

print()
print(f"E5 detector    : {E5_WEIGHTS}")
print(f"E8 MobileNet   : {E8_MN_WEIGHTS}")
print(f"E9-C MobileNet : {E9C_MN_WEIGHTS}")
print(f"E10 MobileNet  : {E10_MN_WEIGHTS}")

print()
print(f"Output         : {OUT_ROOT}")


for p in [
    TEST_IMAGES,
    TEST_COCO,
    E5_WEIGHTS,
    E8_MN_WEIGHTS,
    E9C_MN_WEIGHTS,
    E10_MN_WEIGHTS,
]:

    if not p.exists():

        raise FileNotFoundError(
            f"Missing required path:\n{p}"
        )


# ==================================================================================================
# 3. STANDARDIZED 7-CLASS TAXONOMY
# ==================================================================================================

CLASS7_NAMES = {

    1: "ecal",

    2: "hdpe",

    3: "mixed_plastic_rigid",

    4: "mixed_plastic_soft",

    5: "non_plastic",

    6: "pet",

    7: "pet_oil",
}


COCO8_TO_7 = {

    1: 6,   # pet

    2: 2,   # hdpe

    3: 4,   # mixed soft

    4: 1,   # ecal

    5: 5,   # metal -> non_plastic

    6: 5,   # cardboard -> non_plastic

    7: 3,   # mixed rigid

    8: 7,   # pet oil
}


# MobileNet folder/index ordering:
#
# 0 ecal
# 1 hdpe
# 2 mixed_plastic_rigid
# 3 mixed_plastic_soft
# 4 non_plastic
# 5 pet
# 6 pet_oil

MN7_IDX_TO_COCO7 = {

    0: 1,

    1: 2,

    2: 3,

    3: 4,

    4: 5,

    5: 6,

    6: 7,
}


# ==================================================================================================
# 4. BUILD STANDARDIZED TEST GT
# ==================================================================================================

with open(
    TEST_COCO,
    "r",
    encoding="utf-8",
) as f:

    gt8 = json.load(f)


gt7 = {

    "info":
        gt8.get(
            "info",
            {},
        ),

    "licenses":
        gt8.get(
            "licenses",
            [],
        ),

    "images":
        gt8["images"],

    "annotations":
        [],

    "categories": [

        {
            "id": cid,
            "name": name,
        }

        for cid, name
        in CLASS7_NAMES.items()
    ],
}


for ann in gt8["annotations"]:

    new_ann = (
        ann.copy()
    )

    new_ann[
        "category_id"
    ] = COCO8_TO_7[
        int(
            ann[
                "category_id"
            ]
        )
    ]

    gt7[
        "annotations"
    ].append(
        new_ann
    )


GT7_PATH = (
    OUT_ROOT
    / "test_gt_7class.json"
)


with open(
    GT7_PATH,
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        gt7,
        f,
    )


print()
print("=" * 100)
print("TEST DATASET")
print("=" * 100)

print(
    f"Images : "
    f"{len(gt7['images']):,}"
)

print(
    f"GT     : "
    f"{len(gt7['annotations']):,}"
)


counts = Counter(
    ann["category_id"]
    for ann
    in gt7["annotations"]
)


print()
print(
    "Standardized 7-class "
    "GT distribution:"
)


for cid, name in (
    CLASS7_NAMES.items()
):

    print(
        f"{name:25s}: "
        f"{counts[cid]:,}"
    )


# ==================================================================================================
# 5. MOBILENET TRANSFORM
# ==================================================================================================

MN_TRANSFORM = Compose([

    Resize(
        (224, 224)
    ),

    ToTensor(),

    Normalize(
        mean=[
            0.485,
            0.456,
            0.406,
        ],

        std=[
            0.229,
            0.224,
            0.225,
        ],
    ),
])


# ==================================================================================================
# 6. CHECKPOINT HELPERS
# ==================================================================================================

def extract_state_dict(
    checkpoint,
):

    if not isinstance(
        checkpoint,
        dict,
    ):

        return checkpoint


    if "model_state_dict" in checkpoint:

        state = checkpoint[
            "model_state_dict"
        ]


    elif "state_dict" in checkpoint:

        state = checkpoint[
            "state_dict"
        ]


    elif (
        "model" in checkpoint
        and isinstance(
            checkpoint["model"],
            dict,
        )
    ):

        state = checkpoint[
            "model"
        ]


    else:

        state = checkpoint


    clean = {}


    for key, value in (
        state.items()
    ):

        if key.startswith(
            "module."
        ):

            key = key[
                len("module.") :
            ]


        clean[
            key
        ] = value


    return clean


# ==================================================================================================
# 7. LOAD MOBILENET
# ==================================================================================================

def load_mobilenet(
    checkpoint_path,
    experiment_name,
):

    checkpoint = torch.load(

        checkpoint_path,

        map_location=DEVICE,

        weights_only=False,
    )


    state = extract_state_dict(
        checkpoint
    )


    final_key = (
        "classifier.3.weight"
    )


    if final_key not in state:

        raise RuntimeError(

            f"{experiment_name}: "
            f"{final_key} not found."
        )


    num_classes = int(
        state[
            final_key
        ].shape[0]
    )


    print()
    print(
        experiment_name
    )

    print(
        f"Checkpoint     : "
        f"{checkpoint_path}"
    )

    print(
        f"Output classes : "
        f"{num_classes}"
    )


    if num_classes != 7:

        raise RuntimeError(

            f"{experiment_name}: "
            f"expected 7 outputs, "
            f"found {num_classes}"
        )


    model = (
        mobilenet_v3_large(
            weights=None
        )
    )


    in_features = (
        model.classifier[3]
        .in_features
    )


    model.classifier[3] = (
        nn.Linear(
            in_features,
            7,
        )
    )


    model.load_state_dict(

        state,

        strict=True,
    )


    model.to(
        DEVICE
    )


    model.eval()


    return model


# ==================================================================================================
# 8. CLASSIFY YOLO CROPS
# ==================================================================================================

@torch.no_grad()
def classify_boxes(
    image,
    boxes,
    classifier,
    batch_size=64,
):

    W, H = image.size


    crops = []

    valid_positions = []


    for index, box in enumerate(
        boxes
    ):

        x1, y1, x2, y2 = [

            float(v)

            for v
            in box
        ]


        x1 = max(
            0.0,
            min(
                x1,
                W - 1,
            ),
        )

        y1 = max(
            0.0,
            min(
                y1,
                H - 1,
            ),
        )

        x2 = max(
            0.0,
            min(
                x2,
                W,
            ),
        )

        y2 = max(
            0.0,
            min(
                y2,
                H,
            ),
        )


        if (
            x2 <= x1
            or y2 <= y1
        ):

            continue


        crop = image.crop(
            (
                x1,
                y1,
                x2,
                y2,
            )
        )


        crops.append(
            MN_TRANSFORM(
                crop
            )
        )


        valid_positions.append(
            index
        )


    outputs = [
        None
    ] * len(boxes)


    for start in range(
        0,
        len(crops),
        batch_size,
    ):

        batch = torch.stack(
            crops[
                start :
                start
                + batch_size
            ]
        ).to(
            DEVICE
        )


        logits = classifier(
            batch
        )


        probabilities = (
            F.softmax(
                logits,
                dim=1,
            )
        )


        confidence, predicted = (
            probabilities.max(
                dim=1
            )
        )


        confidence = (
            confidence
            .detach()
            .cpu()
            .numpy()
        )


        predicted = (
            predicted
            .detach()
            .cpu()
            .numpy()
        )


        for local_idx, (
            pred_idx,
            probability,
        ) in enumerate(
            zip(
                predicted,
                confidence,
            )
        ):

            original_idx = (
                valid_positions[
                    start
                    + local_idx
                ]
            )


            category_id = (
                MN7_IDX_TO_COCO7[
                    int(
                        pred_idx
                    )
                ]
            )


            outputs[
                original_idx
            ] = (

                int(
                    category_id
                ),

                float(
                    probability
                ),
            )


    return outputs


# ==================================================================================================
# 9. COCO EVALUATION
# ==================================================================================================

def evaluate_coco(
    predictions,
    title,
):

    print()
    print("=" * 100)
    print(title)
    print("=" * 100)


    if not predictions:

        raise RuntimeError(
            "No predictions generated."
        )


    coco_gt = COCO(
        str(
            GT7_PATH
        )
    )


    coco_dt = (
        coco_gt.loadRes(
            predictions
        )
    )


    evaluator = COCOeval(

        coco_gt,

        coco_dt,

        "bbox",
    )


    evaluator.params.maxDets = [

        1,

        10,

        100,
    ]


    evaluator.evaluate()

    evaluator.accumulate()

    evaluator.summarize()


    overall = {

        "AP50-95":
            float(
                evaluator.stats[
                    0
                ]
            ),

        "AP50":
            float(
                evaluator.stats[
                    1
                ]
            ),

        "AP75":
            float(
                evaluator.stats[
                    2
                ]
            ),

        "AR1":
            float(
                evaluator.stats[
                    6
                ]
            ),

        "AR10":
            float(
                evaluator.stats[
                    7
                ]
            ),

        "AR100":
            float(
                evaluator.stats[
                    8
                ]
            ),
    }


    precision = (
        evaluator.eval[
            "precision"
        ]
    )


    per_class = {}


    for k, category_id in enumerate(
        evaluator.params.catIds
    ):

        class_name = (
            CLASS7_NAMES[
                category_id
            ]
        )


        vals = precision[
            :,
            :,
            k,
            0,
            -1,
        ]


        valid = vals[
            vals > -1
        ]


        ap5095 = (

            float(
                valid.mean()
            )

            if valid.size

            else float(
                "nan"
            )
        )


        vals50 = precision[
            0,
            :,
            k,
            0,
            -1,
        ]


        valid50 = vals50[
            vals50 > -1
        ]


        ap50 = (

            float(
                valid50.mean()
            )

            if valid50.size

            else float(
                "nan"
            )
        )


        per_class[
            class_name
        ] = {

            "AP50":
                ap50,

            "AP50-95":
                ap5095,
        }


    print()
    print(
        "CLASS-WISE RESULTS"
    )


    print(
        "-" * 76
    )


    print(

        f"{'Class':30s}"

        f"{'AP50':>15s}"

        f"{'AP50-95':>15s}"
    )


    print(
        "-" * 76
    )


    for class_name, vals in (
        per_class.items()
    ):

        print(

            f"{class_name:30s}"

            f"{vals['AP50']*100:14.2f}%"

            f"{vals['AP50-95']*100:14.2f}%"
        )


    return (
        overall,
        per_class,
    )


# ==================================================================================================
# 10. SAVE RESULTS
# ==================================================================================================

def save_results(
    experiment_name,
    predictions,
    overall,
    per_class,
    time_minutes,
    detector_boxes,
    invalid_crops,
    output_dir,
):

    output_dir.mkdir(
        parents=True,
        exist_ok=True,
    )


    with open(
        output_dir
        / "predictions.json",
        "w",
        encoding="utf-8",
    ) as f:

        json.dump(
            predictions,
            f,
        )


    result = {

        "experiment":
            experiment_name,

        "taxonomy":
            "standardized 7-class",

        "imgsz":
            640,

        "conf":
            0.001,

        "nms_iou":
            0.60,

        "max_det":
            100,

        "score":
            "yolo_confidence * mobilenet_probability",

        "inference_minutes":
            float(
                time_minutes
            ),

        "detector_boxes":
            int(
                detector_boxes
            ),

        "final_predictions":
            int(
                len(
                    predictions
                )
            ),

        "invalid_crops":
            int(
                invalid_crops
            ),

        "overall":
            overall,

        "per_class":
            per_class,
    }


    with open(
        output_dir
        / "results.json",
        "w",
        encoding="utf-8",
    ) as f:

        json.dump(
            result,
            f,
            indent=2,
        )


    with open(
        output_dir
        / "summary.txt",
        "w",
        encoding="utf-8",
    ) as f:

        f.write(
            experiment_name
            + "\n"
        )

        f.write(
            "=" * 90
            + "\n\n"
        )


        f.write(
            f"AP50-95 : "
            f"{overall['AP50-95']*100:.4f}%\n"
        )

        f.write(
            f"AP50    : "
            f"{overall['AP50']*100:.4f}%\n"
        )

        f.write(
            f"AP75    : "
            f"{overall['AP75']*100:.4f}%\n"
        )

        f.write(
            f"AR100   : "
            f"{overall['AR100']*100:.4f}%\n"
        )


        f.write(
            "\nClass-wise:\n"
        )


        for class_name, vals in (
            per_class.items()
        ):

            f.write(

                f"{class_name:25s}"

                f" AP50="
                f"{vals['AP50']*100:.4f}%"

                f" AP50-95="
                f"{vals['AP50-95']*100:.4f}%\n"
            )


        f.write(
            f"\nDetector boxes : "
            f"{detector_boxes:,}\n"
        )

        f.write(
            f"Predictions    : "
            f"{len(predictions):,}\n"
        )

        f.write(
            f"Invalid crops  : "
            f"{invalid_crops:,}\n"
        )

        f.write(
            f"Inference time : "
            f"{time_minutes:.2f} min\n"
        )


# ==================================================================================================
# 11. RUN TWO-STAGE PIPELINE
# ==================================================================================================

def run_pipeline(
    short_name,
    experiment_name,
    classifier_weights,
):

    print()
    print()
    print("#" * 100)
    print(experiment_name)
    print("#" * 100)


    print(
        f"Detector       : "
        f"{E5_WEIGHTS}"
    )

    print(
        f"Classifier     : "
        f"{classifier_weights}"
    )

    print(
        f"Image size     : "
        f"640"
    )

    print(
        f"Conf           : "
        f"0.001"
    )

    print(
        f"NMS IoU        : "
        f"0.60"
    )

    print(
        f"Max det        : "
        f"100"
    )


    torch.cuda.empty_cache()


    detector = YOLO(
        str(
            E5_WEIGHTS
        )
    )


    classifier = load_mobilenet(

        classifier_weights,

        experiment_name,
    )


    predictions = []

    detector_boxes = 0

    invalid_crops = 0


    start_time = (
        time.time()
    )


    total_images = len(
        gt7[
            "images"
        ]
    )


    for image_number, image_info in enumerate(
        gt7[
            "images"
        ],
        start=1,
    ):


        image_id = int(
            image_info[
                "id"
            ]
        )


        image_path = (

            TEST_IMAGES

            / Path(
                image_info[
                    "file_name"
                ]
            ).name
        )


        image = Image.open(
            image_path
        ).convert(
            "RGB"
        )


        result = detector.predict(

            source=str(
                image_path
            ),

            imgsz=640,

            conf=0.001,

            iou=0.60,

            max_det=100,

            device=YOLO_DEVICE,

            verbose=False,

        )[0]


        if (
            result.boxes is None
            or len(
                result.boxes
            ) == 0
        ):

            continue


        boxes = (

            result.boxes.xyxy

            .detach()

            .cpu()

            .numpy()
        )


        yolo_confidence = (

            result.boxes.conf

            .detach()

            .cpu()

            .numpy()
        )


        detector_boxes += (
            len(
                boxes
            )
        )


        classifications = classify_boxes(

            image,

            boxes,

            classifier,

            batch_size=64,
        )


        for (
            box,
            detector_conf,
            classification,
        ) in zip(

            boxes,

            yolo_confidence,

            classifications,
        ):


            if classification is None:

                invalid_crops += 1

                continue


            (
                final_category_id,
                classifier_probability,
            ) = classification


            x1, y1, x2, y2 = [

                float(
                    v
                )

                for v
                in box
            ]


            width = (
                x2
                - x1
            )

            height = (
                y2
                - y1
            )


            if (
                width <= 0
                or height <= 0
            ):

                invalid_crops += 1

                continue


            # --------------------------------------------------------------------------------------
            # E8 / E9-C / E10 scoring
            # --------------------------------------------------------------------------------------

            final_score = (

                float(
                    detector_conf
                )

                *

                float(
                    classifier_probability
                )
            )


            predictions.append({

                "image_id":
                    image_id,

                "category_id":
                    int(
                        final_category_id
                    ),

                "bbox": [

                    float(
                        x1
                    ),

                    float(
                        y1
                    ),

                    float(
                        width
                    ),

                    float(
                        height
                    ),
                ],

                "score":
                    float(
                        final_score
                    ),
            })


        if (

            image_number
            % 25 == 0

            or image_number
            == total_images

        ):

            print(

                f"{short_name}: "

                f"{image_number}/"
                f"{total_images}"
            )


    elapsed = (

        time.time()

        - start_time
    )


    print()
    print(
        f"Detector boxes : "
        f"{detector_boxes:,}"
    )

    print(
        f"Predictions    : "
        f"{len(predictions):,}"
    )

    print(
        f"Invalid crops  : "
        f"{invalid_crops:,}"
    )

    print(
        f"Time           : "
        f"{elapsed/60:.2f} min"
    )


    overall, per_class = (
        evaluate_coco(

            predictions,

            f"{experiment_name} — TEST",
        )
    )


    save_results(

        experiment_name,

        predictions,

        overall,

        per_class,

        elapsed / 60,

        detector_boxes,

        invalid_crops,

        OUT_ROOT
        / short_name,
    )


    del detector
    del classifier


    torch.cuda.empty_cache()


    return {

        "overall":
            overall,

        "per_class":
            per_class,

        "time":
            elapsed / 60,
    }


# ==================================================================================================
# 12. RUN BATCH 4
# ==================================================================================================

results = {}


# --------------------------------------------------------------------------------------------------
# E8
# --------------------------------------------------------------------------------------------------

results[
    "E8"
] = run_pipeline(

    short_name="E8",

    experiment_name=(
        "E8 — YOLO11s + "
        "MobileNetV3-Large Class-Weighted"
    ),

    classifier_weights=(
        E8_MN_WEIGHTS
    ),
)


# --------------------------------------------------------------------------------------------------
# E9-C
# --------------------------------------------------------------------------------------------------

results[
    "E9-C"
] = run_pipeline(

    short_name="E9-C",

    experiment_name=(
        "E9-C — YOLO11s + "
        "YOLO11s-Matched MobileNetV3-Large"
    ),

    classifier_weights=(
        E9C_MN_WEIGHTS
    ),
)


# --------------------------------------------------------------------------------------------------
# E10
# --------------------------------------------------------------------------------------------------

results[
    "E10"
] = run_pipeline(

    short_name="E10",

    experiment_name=(
        "E10 — YOLO11s-Matched MobileNet "
        "Controlled E3Y-B Protocol"
    ),

    classifier_weights=(
        E10_MN_WEIGHTS
    ),
)


# ==================================================================================================
# 13. OVERALL SUMMARY
# ==================================================================================================

print()
print()
print("=" * 100)
print("FINAL TEST — BATCH 4 COMPLETE")
print("=" * 100)


print()

print(

    f"{'Experiment':15s}"

    f"{'AP50-95':>14s}"

    f"{'AP50':>14s}"

    f"{'AP75':>14s}"

    f"{'AR100':>14s}"

    f"{'Time':>12s}"
)


print(
    "-" * 87
)


for experiment, result in (
    results.items()
):

    overall = (
        result[
            "overall"
        ]
    )


    print(

        f"{experiment:15s}"

        f"{overall['AP50-95']*100:13.4f}%"

        f"{overall['AP50']*100:13.4f}%"

        f"{overall['AP75']*100:13.4f}%"

        f"{overall['AR100']*100:13.4f}%"

        f"{result['time']:11.2f}m"
    )


# ==================================================================================================
# 14. CLASS-WISE AP50
# ==================================================================================================

print()
print("=" * 100)
print("CLASS-WISE TEST AP50")
print("=" * 100)


print(

    f"{'Class':28s}"

    + "".join(

        f"{experiment:>14s}"

        for experiment
        in results
    )
)


print(

    "-" * (

        28

        + 14
        * len(
            results
        )
    )
)


for class_name in (
    CLASS7_NAMES.values()
):

    row = (
        f"{class_name:28s}"
    )


    for experiment in results:

        value = (

            results[
                experiment
            ][
                "per_class"
            ][
                class_name
            ][
                "AP50"
            ]
        )


        row += (
            f"{value*100:13.2f}%"
        )


    print(
        row
    )


# ==================================================================================================
# 15. CLASS-WISE AP50-95
# ==================================================================================================

print()
print("=" * 100)
print("CLASS-WISE TEST AP50-95")
print("=" * 100)


print(

    f"{'Class':28s}"

    + "".join(

        f"{experiment:>14s}"

        for experiment
        in results
    )
)


print(

    "-" * (

        28

        + 14
        * len(
            results
        )
    )
)


for class_name in (
    CLASS7_NAMES.values()
):

    row = (
        f"{class_name:28s}"
    )


    for experiment in results:

        value = (

            results[
                experiment
            ][
                "per_class"
            ][
                class_name
            ][
                "AP50-95"
            ]
        )


        row += (
            f"{value*100:13.2f}%"
        )


    print(
        row
    )


print()
print(
    "Outputs saved to:"
)

print(
    OUT_ROOT
)


print()
print("=" * 100)
print("FINAL TEST — BATCH 4 COMPLETE")
print("=" * 100)

FINAL TEST — BATCH 4
PyTorch        : 2.13.0+cu126
Device         : cuda
GPU            : NVIDIA GeForce RTX 3050 Ti Laptop GPU

Test images    : C:\Users\varda\Documents\_My Computer\COMMON_space\Learning\Upgrad_EPGP_MS\upgrad_IIITB_MS_course\MS_LJMU_Material\Topic Data\SortWaste\dataset\dataset\splited_all_dataset_coco\test\images
Test COCO      : C:\Users\varda\Documents\_My Computer\COMMON_space\Learning\Upgrad_EPGP_MS\upgrad_IIITB_MS_course\MS_LJMU_Material\Topic Data\SortWaste\dataset\dataset\splited_all_dataset_coco\test\annotations\test_coco.json

E5 detector    : C:\Users\varda\Documents\_My Computer\COMMON_space\Learning\Upgrad_EPGP_MS\upgrad_IIITB_MS_course\MS_LJMU_Material\Thesis_Code\runs\sortwaste\E5_yolo11s_7class_aug_classbalance_640\weights\best.pt
E8 MobileNet   : C:\Users\varda\Documents\_My Computer\COMMON_space\Learning\Upgrad_EPGP_MS\upgrad_IIITB_MS_course\MS_LJMU_Material\Topic Data\SortWaste\dataset\dataset\yolo_mobilenet_crops_E3Y\mobilenet_results\E3Y_B_class_

## Batch 5
| Experiment | Configuration                                                        |
| ---------- | -------------------------------------------------------------------- |
| **E12**    | YOLO11m @640 detector                                                |
| **E13**    | E12 + E3Y-B MobileNet, MobileNet always supplies class               |
| **E14-C**  | E12 + MobileNet, global gate **0.95**, fusion α = **0.70**           |
| **E15**    | E12 + MobileNet, optimized global gate **0.98**, fusion α = **0.70** |
| **E16**    | E12 + MobileNet, **class-specific gates**, fusion α = **0.70**       |


In [ ]:
# FINAL TEST — BATCH 5
#
# E12   : YOLO11m @640
# E13   : YOLO11m + MobileNetV3-Large
# E14-C : Global MobileNet gate = 0.95 + alpha = 0.70
# E15   : Global MobileNet gate = 0.98 + alpha = 0.70
# E16   : Class-specific MobileNet gates + alpha = 0.70
#
# IMPORTANT
# --------------------------------------------------------------------------------------------------
# - TEST SET ONLY
# - NO TEST-SET TUNING
# - All thresholds/gates/alpha selected previously on validation data
# - Explicit pycocotools COCOeval
# - Standardized 7-class taxonomy
#
# Frozen detector:
#   YOLO11m E12
#   imgsz  = 640
#   conf   = 0.001
#   NMS    = 0.60
#   maxdet = 100
#
# MobileNet:
#   E3Y-B class-weighted MobileNetV3-Large
#
# E13:
#   MobileNet ALWAYS supplies final class
#   score = YOLO_conf * P_MobileNet(predicted_class)
#
# E14-C:
#   MobileNet final class only if max MobileNet probability >= 0.95
#   otherwise keep YOLO class
#   score = YOLO_conf^0.70 * P_MobileNet(final_class)^0.30
#
# E15:
#   Same, but global gate = 0.98
#
# E16:
#   Gate selected according to MobileNet's proposed class:
#       ecal                 0.99
#       hdpe                 0.98
#       mixed_plastic_rigid  0.98
#       mixed_plastic_soft   0.94
#       non_plastic          0.94
#       pet                  0.99
#       pet_oil              0.99
#
#   score = YOLO_conf^0.70 * P_MobileNet(final_class)^0.30
# ==================================================================================================

import json
import time
import shutil
from pathlib import Path
from collections import Counter, defaultdict

from PIL import Image

import numpy as np

import torch
import torch.nn as nn
import torch.nn.functional as F

from torchvision.models import mobilenet_v3_large
from torchvision.transforms import (
    Compose,
    Resize,
    ToTensor,
    Normalize,
)

from ultralytics import YOLO

from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval


# ==================================================================================================
# 1. PATHS
# ==================================================================================================

BASE = Path(
    r"C:\Users\varda\Documents\_My Computer\COMMON_space\Learning"
    r"\Upgrad_EPGP_MS\upgrad_IIITB_MS_course\MS_LJMU_Material"
)

DATASET_ROOT = (
    BASE
    / "Topic Data"
    / "SortWaste"
    / "dataset"
    / "dataset"
)

THESIS_CODE = (
    BASE
    / "Thesis_Code"
)


# --------------------------------------------------------------------------------------------------
# TEST DATA
# --------------------------------------------------------------------------------------------------

TEST_ROOT = (
    DATASET_ROOT
    / "splited_all_dataset_coco"
    / "test"
)

TEST_IMAGES = (
    TEST_ROOT
    / "images"
)

TEST_COCO = (
    TEST_ROOT
    / "annotations"
    / "test_coco.json"
)


# --------------------------------------------------------------------------------------------------
# E12 YOLO11m checkpoint
# --------------------------------------------------------------------------------------------------

E12_WEIGHTS = (
    THESIS_CODE
    / "runs"
    / "sortwaste"
    / "E12_yolo11m_7class_aug_classbalance_640"
    / "weights"
    / "best.pt"
)


# --------------------------------------------------------------------------------------------------
# E3Y-B MobileNet checkpoint
#
# Reused by E13 / E14 / E15 / E16
# --------------------------------------------------------------------------------------------------

MOBILENET_WEIGHTS = (
    DATASET_ROOT
    / "yolo_mobilenet_crops_E3Y"
    / "mobilenet_results"
    / "E3Y_B_class_weighted"
    / "E3Y_B_MobileNetV3Large_best.pth"
)


# --------------------------------------------------------------------------------------------------
# OUTPUT
# --------------------------------------------------------------------------------------------------

OUT_ROOT = (
    THESIS_CODE
    / "runs"
    / "sortwaste"
    / "FINAL_TEST_BATCH5"
)


if OUT_ROOT.exists():

    print(
        "Deleting old Batch 5 results:\n"
        f"{OUT_ROOT}"
    )

    shutil.rmtree(
        OUT_ROOT
    )


OUT_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)


# ==================================================================================================
# 2. FROZEN PARAMETERS
# ==================================================================================================

IMAGE_SIZE = 640

CONF_THRESHOLD = 0.001

NMS_IOU = 0.60

MAX_DET = 100


# Confidence fusion
ALPHA = 0.70


# E14-C
E14_GATE = 0.95


# E15
E15_GATE = 0.98


# ==================================================================================================
# 3. CLASS-SPECIFIC E16 GATES
#
# Index corresponds directly to:
#
# 0 ecal
# 1 hdpe
# 2 mixed_plastic_rigid
# 3 mixed_plastic_soft
# 4 non_plastic
# 5 pet
# 6 pet_oil
# ==================================================================================================

E16_GATES = {

    0: 0.99,  # ecal

    1: 0.98,  # hdpe

    2: 0.98,  # mixed rigid

    3: 0.94,  # mixed soft

    4: 0.94,  # non plastic

    5: 0.99,  # pet

    6: 0.99,  # pet oil
}


# ==================================================================================================
# 4. DEVICE
# ==================================================================================================

DEVICE = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

YOLO_DEVICE = (
    0
    if torch.cuda.is_available()
    else "cpu"
)


print("=" * 100)
print("FINAL TEST — BATCH 5")
print("=" * 100)

print(
    f"PyTorch        : "
    f"{torch.__version__}"
)

print(
    f"Device         : "
    f"{DEVICE}"
)

if torch.cuda.is_available():

    print(
        f"GPU            : "
        f"{torch.cuda.get_device_name(0)}"
    )


print()

print(
    f"Test images    : "
    f"{TEST_IMAGES}"
)

print(
    f"Test COCO      : "
    f"{TEST_COCO}"
)

print()

print(
    f"E12 detector   : "
    f"{E12_WEIGHTS}"
)

print(
    f"MobileNet      : "
    f"{MOBILENET_WEIGHTS}"
)

print()

print(
    f"imgsz          : "
    f"{IMAGE_SIZE}"
)

print(
    f"conf           : "
    f"{CONF_THRESHOLD}"
)

print(
    f"NMS IoU        : "
    f"{NMS_IOU}"
)

print(
    f"max_det        : "
    f"{MAX_DET}"
)

print(
    f"alpha          : "
    f"{ALPHA}"
)

print(
    f"E14-C gate     : "
    f"{E14_GATE}"
)

print(
    f"E15 gate       : "
    f"{E15_GATE}"
)

print()

print(
    "E16 gates:"
)

for idx, gate in E16_GATES.items():

    print(
        f"  class {idx}: "
        f"{gate:.2f}"
    )


print()

print(
    f"Output         : "
    f"{OUT_ROOT}"
)


# --------------------------------------------------------------------------------------------------
# Verify paths
# --------------------------------------------------------------------------------------------------

for path in [

    TEST_IMAGES,

    TEST_COCO,

    E12_WEIGHTS,

    MOBILENET_WEIGHTS,

]:

    if not path.exists():

        raise FileNotFoundError(
            f"Required path missing:\n{path}"
        )


# ==================================================================================================
# 5. STANDARDIZED 7-CLASS TAXONOMY
# ==================================================================================================

CLASS7_NAMES = {

    1: "ecal",

    2: "hdpe",

    3: "mixed_plastic_rigid",

    4: "mixed_plastic_soft",

    5: "non_plastic",

    6: "pet",

    7: "pet_oil",
}


# --------------------------------------------------------------------------------------------------
# Original SortWaste COCO categories -> standardized 7 classes
# --------------------------------------------------------------------------------------------------

COCO8_TO_7 = {

    1: 6,  # pet

    2: 2,  # hdpe

    3: 4,  # mixed soft

    4: 1,  # ecal

    5: 5,  # metal -> nonplastic

    6: 5,  # cardboard -> nonplastic

    7: 3,  # mixed rigid

    8: 7,  # pet oil
}


# --------------------------------------------------------------------------------------------------
# YOLO/MobileNet index -> COCO evaluation category
#
# YOLO/MN index:
#
# 0 ecal
# 1 hdpe
# 2 mixed_plastic_rigid
# 3 mixed_plastic_soft
# 4 non_plastic
# 5 pet
# 6 pet_oil
# --------------------------------------------------------------------------------------------------

IDX_TO_COCO7 = {

    0: 1,

    1: 2,

    2: 3,

    3: 4,

    4: 5,

    5: 6,

    6: 7,
}


IDX_TO_NAME = {

    0: "ecal",

    1: "hdpe",

    2: "mixed_plastic_rigid",

    3: "mixed_plastic_soft",

    4: "non_plastic",

    5: "pet",

    6: "pet_oil",
}


# ==================================================================================================
# 6. BUILD STANDARDIZED TEST GT
# ==================================================================================================

with open(
    TEST_COCO,
    "r",
    encoding="utf-8",
) as f:

    gt8 = json.load(f)


gt7 = {

    "info":
        gt8.get(
            "info",
            {},
        ),

    "licenses":
        gt8.get(
            "licenses",
            [],
        ),

    "images":
        gt8[
            "images"
        ],

    "annotations":
        [],

    "categories": [

        {
            "id":
                category_id,

            "name":
                class_name,
        }

        for category_id, class_name
        in CLASS7_NAMES.items()
    ],
}


for annotation in (
    gt8["annotations"]
):

    new_annotation = (
        annotation.copy()
    )

    new_annotation[
        "category_id"
    ] = COCO8_TO_7[
        int(
            annotation[
                "category_id"
            ]
        )
    ]

    gt7[
        "annotations"
    ].append(
        new_annotation
    )


GT7_PATH = (
    OUT_ROOT
    / "test_gt_7class.json"
)


with open(
    GT7_PATH,
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        gt7,
        f,
    )


print()
print("=" * 100)
print("TEST DATASET")
print("=" * 100)

print(
    f"Images : "
    f"{len(gt7['images']):,}"
)

print(
    f"GT     : "
    f"{len(gt7['annotations']):,}"
)


gt_counts = Counter(

    annotation[
        "category_id"
    ]

    for annotation
    in gt7[
        "annotations"
    ]
)


print()
print(
    "Standardized 7-class "
    "GT distribution:"
)


for category_id, class_name in (
    CLASS7_NAMES.items()
):

    print(

        f"{class_name:25s}: "

        f"{gt_counts[category_id]:,}"
    )


# ==================================================================================================
# 7. MOBILENET TRANSFORM
# ==================================================================================================

MN_TRANSFORM = Compose([

    Resize(
        (224, 224)
    ),

    ToTensor(),

    Normalize(

        mean=[
            0.485,
            0.456,
            0.406,
        ],

        std=[
            0.229,
            0.224,
            0.225,
        ],
    ),
])


# ==================================================================================================
# 8. CHECKPOINT HELPER
# ==================================================================================================

def extract_state_dict(
    checkpoint,
):

    if not isinstance(
        checkpoint,
        dict,
    ):

        return checkpoint


    if (
        "model_state_dict"
        in checkpoint
    ):

        state = checkpoint[
            "model_state_dict"
        ]


    elif (
        "state_dict"
        in checkpoint
    ):

        state = checkpoint[
            "state_dict"
        ]


    elif (

        "model"
        in checkpoint

        and isinstance(
            checkpoint[
                "model"
            ],
            dict,
        )
    ):

        state = checkpoint[
            "model"
        ]


    else:

        state = checkpoint


    cleaned = {}


    for key, value in (
        state.items()
    ):

        if key.startswith(
            "module."
        ):

            key = key[
                len("module.") :
            ]


        cleaned[
            key
        ] = value


    return cleaned


# ==================================================================================================
# 9. LOAD MOBILENET
# ==================================================================================================

def load_mobilenet():

    checkpoint = torch.load(

        MOBILENET_WEIGHTS,

        map_location=DEVICE,

        weights_only=False,
    )


    state = extract_state_dict(
        checkpoint
    )


    final_key = (
        "classifier.3.weight"
    )


    if final_key not in state:

        raise RuntimeError(

            "Could not find "
            "'classifier.3.weight' "
            "in MobileNet checkpoint."
        )


    num_classes = int(

        state[
            final_key
        ].shape[
            0
        ]
    )


    print()
    print(
        "Loading MobileNet..."
    )

    print(
        f"Output classes : "
        f"{num_classes}"
    )


    if (
        num_classes
        != 7
    ):

        raise RuntimeError(

            f"Expected 7 MobileNet classes; "
            f"checkpoint contains "
            f"{num_classes}."
        )


    model = (
        mobilenet_v3_large(
            weights=None
        )
    )


    in_features = (

        model
        .classifier[
            3
        ]
        .in_features
    )


    model.classifier[
        3
    ] = nn.Linear(

        in_features,

        7,
    )


    model.load_state_dict(

        state,

        strict=True,
    )


    model.to(
        DEVICE
    )


    model.eval()


    return model


# ==================================================================================================
# 10. CLASSIFY CROPS AND RETURN ALL 7 PROBABILITIES
# ==================================================================================================

@torch.no_grad()
def classify_boxes_all_probs(
    image,
    boxes,
    classifier,
    batch_size=64,
):

    """
    Returns one entry per YOLO detection.

    Each valid entry contains the complete 7-class MobileNet
    probability vector.

    Invalid crops -> None.
    """

    W, H = image.size


    crops = []

    valid_positions = []


    for index, box in enumerate(
        boxes
    ):

        x1, y1, x2, y2 = [

            float(v)

            for v
            in box
        ]


        x1 = max(
            0.0,
            min(
                x1,
                W - 1,
            ),
        )


        y1 = max(
            0.0,
            min(
                y1,
                H - 1,
            ),
        )


        x2 = max(
            0.0,
            min(
                x2,
                W,
            ),
        )


        y2 = max(
            0.0,
            min(
                y2,
                H,
            ),
        )


        if (
            x2 <= x1
            or y2 <= y1
        ):

            continue


        crop = image.crop(

            (
                x1,
                y1,
                x2,
                y2,
            )
        )


        crops.append(

            MN_TRANSFORM(
                crop
            )
        )


        valid_positions.append(
            index
        )


    outputs = [

        None

        for _ in range(
            len(
                boxes
            )
        )
    ]


    for start in range(
        0,
        len(
            crops
        ),
        batch_size,
    ):

        batch = torch.stack(

            crops[
                start :
                start
                + batch_size
            ]

        ).to(
            DEVICE
        )


        logits = classifier(
            batch
        )


        probs = F.softmax(

            logits,

            dim=1,
        )


        probs = (

            probs

            .detach()

            .cpu()

            .numpy()
        )


        for local_index, probability_vector in enumerate(
            probs
        ):

            original_position = (

                valid_positions[
                    start
                    + local_index
                ]
            )


            outputs[
                original_position
            ] = probability_vector.astype(
                np.float32
            )


    return outputs


# ==================================================================================================
# 11. FUSION SCORE
# ==================================================================================================

def geometric_score(
    yolo_confidence,
    classifier_probability,
    alpha=ALPHA,
):

    """
    S =
        YOLO_confidence^alpha
        *
        classifier_probability^(1-alpha)
    """

    yolo_confidence = max(

        float(
            yolo_confidence
        ),

        1e-12,
    )


    classifier_probability = max(

        float(
            classifier_probability
        ),

        1e-12,
    )


    return (

        yolo_confidence
        ** alpha

        *

        classifier_probability
        ** (
            1.0
            - alpha
        )
    )


# ==================================================================================================
# 12. COCO EVALUATION
# ==================================================================================================

def evaluate_coco(
    predictions,
    title,
):

    print()
    print("=" * 100)
    print(title)
    print("=" * 100)


    if not predictions:

        raise RuntimeError(
            "No predictions generated."
        )


    coco_gt = COCO(
        str(
            GT7_PATH
        )
    )


    coco_dt = coco_gt.loadRes(
        predictions
    )


    evaluator = COCOeval(

        coco_gt,

        coco_dt,

        "bbox",
    )


    evaluator.params.maxDets = [

        1,

        10,

        100,
    ]


    evaluator.evaluate()

    evaluator.accumulate()

    evaluator.summarize()


    overall = {

        "AP50-95":
            float(
                evaluator.stats[
                    0
                ]
            ),

        "AP50":
            float(
                evaluator.stats[
                    1
                ]
            ),

        "AP75":
            float(
                evaluator.stats[
                    2
                ]
            ),

        "AR1":
            float(
                evaluator.stats[
                    6
                ]
            ),

        "AR10":
            float(
                evaluator.stats[
                    7
                ]
            ),

        "AR100":
            float(
                evaluator.stats[
                    8
                ]
            ),
    }


    precision = (

        evaluator.eval[
            "precision"
        ]
    )


    per_class = {}


    for k, category_id in enumerate(
        evaluator.params.catIds
    ):

        class_name = (
            CLASS7_NAMES[
                category_id
            ]
        )


        # AP@[.50:.95]
        values = precision[
            :,
            :,
            k,
            0,
            -1,
        ]


        valid = values[
            values > -1
        ]


        ap5095 = (

            float(
                valid.mean()
            )

            if valid.size

            else float(
                "nan"
            )
        )


        # AP50
        values50 = precision[
            0,
            :,
            k,
            0,
            -1,
        ]


        valid50 = values50[
            values50 > -1
        ]


        ap50 = (

            float(
                valid50.mean()
            )

            if valid50.size

            else float(
                "nan"
            )
        )


        per_class[
            class_name
        ] = {

            "AP50":
                ap50,

            "AP50-95":
                ap5095,
        }


    print()
    print(
        "CLASS-WISE RESULTS"
    )

    print(
        "-" * 76
    )


    print(

        f"{'Class':30s}"

        f"{'AP50':>15s}"

        f"{'AP50-95':>15s}"
    )


    print(
        "-" * 76
    )


    for class_name, values in (
        per_class.items()
    ):

        print(

            f"{class_name:30s}"

            f"{values['AP50']*100:14.2f}%"

            f"{values['AP50-95']*100:14.2f}%"
        )


    return (
        overall,
        per_class,
    )


# ==================================================================================================
# 13. SAVE RESULTS
# ==================================================================================================

def save_results(
    experiment_name,
    predictions,
    overall,
    per_class,
    output_dir,
    inference_minutes,
    extra_info=None,
):

    output_dir.mkdir(

        parents=True,

        exist_ok=True,
    )


    with open(

        output_dir
        / "predictions.json",

        "w",

        encoding="utf-8",

    ) as f:

        json.dump(
            predictions,
            f,
        )


    result = {

        "experiment":
            experiment_name,

        "taxonomy":
            "standardized 7-class",

        "imgsz":
            IMAGE_SIZE,

        "conf":
            CONF_THRESHOLD,

        "nms_iou":
            NMS_IOU,

        "max_det":
            MAX_DET,

        "inference_minutes":
            float(
                inference_minutes
            ),

        "num_predictions":
            int(
                len(
                    predictions
                )
            ),

        "overall":
            overall,

        "per_class":
            per_class,
    }


    if extra_info:

        result[
            "configuration"
        ] = extra_info


    with open(

        output_dir
        / "results.json",

        "w",

        encoding="utf-8",

    ) as f:

        json.dump(

            result,

            f,

            indent=2,
        )


    with open(

        output_dir
        / "summary.txt",

        "w",

        encoding="utf-8",

    ) as f:

        f.write(
            experiment_name
            + "\n"
        )

        f.write(
            "=" * 90
            + "\n\n"
        )


        f.write(

            f"AP50-95 : "
            f"{overall['AP50-95']*100:.4f}%\n"
        )


        f.write(

            f"AP50    : "
            f"{overall['AP50']*100:.4f}%\n"
        )


        f.write(

            f"AP75    : "
            f"{overall['AP75']*100:.4f}%\n"
        )


        f.write(

            f"AR100   : "
            f"{overall['AR100']*100:.4f}%\n"
        )


        f.write(
            "\nClass-wise:\n"
        )


        for class_name, values in (
            per_class.items()
        ):

            f.write(

                f"{class_name:25s}"

                f" AP50="
                f"{values['AP50']*100:.4f}%"

                f" AP50-95="
                f"{values['AP50-95']*100:.4f}%\n"
            )


# ==================================================================================================
# 14. LOAD MODELS
# ==================================================================================================

print()
print("=" * 100)
print("LOADING MODELS")
print("=" * 100)


detector = YOLO(
    str(
        E12_WEIGHTS
    )
)


classifier = load_mobilenet()


# ==================================================================================================
# 15. ONE INFERENCE PASS — GENERATE ALL FIVE EXPERIMENTS
# ==================================================================================================

prediction_sets = {

    "E12": [],

    "E13": [],

    "E14-C": [],

    "E15": [],

    "E16": [],
}


# Diagnostics
source_counts = {

    "E14-C": defaultdict(int),

    "E15": defaultdict(int),

    "E16": defaultdict(int),
}


detector_boxes = 0

valid_crops = 0

invalid_crops = 0


start_time = (
    time.time()
)


total_images = len(
    gt7[
        "images"
    ]
)


print()
print("=" * 100)
print("RUNNING SHARED E12 + MOBILENET INFERENCE")
print("=" * 100)


for image_number, image_info in enumerate(

    gt7[
        "images"
    ],

    start=1,

):

    image_id = int(
        image_info[
            "id"
        ]
    )


    image_path = (

        TEST_IMAGES

        / Path(
            image_info[
                "file_name"
            ]
        ).name
    )


    image = Image.open(
        image_path
    ).convert(
        "RGB"
    )


    # ----------------------------------------------------------------------------------------------
    # Frozen E12 detector
    # ----------------------------------------------------------------------------------------------

    result = detector.predict(

        source=str(
            image_path
        ),

        imgsz=IMAGE_SIZE,

        conf=CONF_THRESHOLD,

        iou=NMS_IOU,

        max_det=MAX_DET,

        device=YOLO_DEVICE,

        verbose=False,

    )[0]


    if (
        result.boxes is None
        or len(
            result.boxes
        ) == 0
    ):

        continue


    boxes = (

        result.boxes.xyxy

        .detach()

        .cpu()

        .numpy()
    )


    yolo_confidences = (

        result.boxes.conf

        .detach()

        .cpu()

        .numpy()
    )


    yolo_classes = (

        result.boxes.cls

        .detach()

        .cpu()

        .numpy()

        .astype(
            int
        )
    )


    detector_boxes += (
        len(
            boxes
        )
    )


    # ----------------------------------------------------------------------------------------------
    # Run MobileNet once for every detector crop
    # ----------------------------------------------------------------------------------------------

    mn_probabilities = (
        classify_boxes_all_probs(

            image,

            boxes,

            classifier,

            batch_size=64,
        )
    )


    for (
        box,
        yolo_conf,
        yolo_class,
        mn_probs,

    ) in zip(

        boxes,

        yolo_confidences,

        yolo_classes,

        mn_probabilities,

    ):


        if mn_probs is None:

            invalid_crops += 1

            continue


        valid_crops += 1


        yolo_class = int(
            yolo_class
        )


        if yolo_class not in (
            IDX_TO_COCO7
        ):

            raise RuntimeError(

                f"Unexpected YOLO class: "
                f"{yolo_class}"
            )


        # ------------------------------------------------------------------------------------------
        # Geometry
        # ------------------------------------------------------------------------------------------

        x1, y1, x2, y2 = [

            float(v)

            for v
            in box
        ]


        width = (
            x2 - x1
        )


        height = (
            y2 - y1
        )


        if (
            width <= 0
            or height <= 0
        ):

            invalid_crops += 1

            continue


        bbox = [

            float(
                x1
            ),

            float(
                y1
            ),

            float(
                width
            ),

            float(
                height
            ),
        ]


        # ------------------------------------------------------------------------------------------
        # MobileNet probabilities
        # ------------------------------------------------------------------------------------------

        mn_pred_class = int(

            np.argmax(
                mn_probs
            )
        )


        mn_pred_prob = float(

            mn_probs[
                mn_pred_class
            ]
        )


        # ==========================================================================================
        # E12 — detector only
        # ==========================================================================================

        prediction_sets[
            "E12"
        ].append({

            "image_id":
                image_id,

            "category_id":
                IDX_TO_COCO7[
                    yolo_class
                ],

            "bbox":
                bbox,

            "score":
                float(
                    yolo_conf
                ),
        })


        # ==========================================================================================
        # E13 — MobileNet ALWAYS supplies class
        #
        # score = YOLO confidence × MobileNet probability
        # ==========================================================================================

        e13_final_class = (
            mn_pred_class
        )


        e13_score = (

            float(
                yolo_conf
            )

            *

            float(
                mn_pred_prob
            )
        )


        prediction_sets[
            "E13"
        ].append({

            "image_id":
                image_id,

            "category_id":
                IDX_TO_COCO7[
                    e13_final_class
                ],

            "bbox":
                bbox,

            "score":
                float(
                    e13_score
                ),
        })


        # ==========================================================================================
        # E14-C
        #
        # Global MobileNet gate = 0.95
        # alpha = 0.70
        # ==========================================================================================

        if (
            mn_pred_prob
            >= E14_GATE
        ):

            e14_final_class = (
                mn_pred_class
            )

            source_counts[
                "E14-C"
            ][
                "MobileNet"
            ] += 1


        else:

            e14_final_class = (
                yolo_class
            )

            source_counts[
                "E14-C"
            ][
                "YOLO"
            ] += 1


        # IMPORTANT:
        # classifier probability must correspond to the ACTUAL final class.
        e14_classifier_prob = float(

            mn_probs[
                e14_final_class
            ]
        )


        e14_score = geometric_score(

            yolo_conf,

            e14_classifier_prob,

            alpha=ALPHA,
        )


        prediction_sets[
            "E14-C"
        ].append({

            "image_id":
                image_id,

            "category_id":
                IDX_TO_COCO7[
                    e14_final_class
                ],

            "bbox":
                bbox,

            "score":
                float(
                    e14_score
                ),
        })


        # ==========================================================================================
        # E15
        #
        # Global gate = 0.98
        # alpha = 0.70
        # ==========================================================================================

        if (
            mn_pred_prob
            >= E15_GATE
        ):

            e15_final_class = (
                mn_pred_class
            )

            source_counts[
                "E15"
            ][
                "MobileNet"
            ] += 1


        else:

            e15_final_class = (
                yolo_class
            )

            source_counts[
                "E15"
            ][
                "YOLO"
            ] += 1


        e15_classifier_prob = float(

            mn_probs[
                e15_final_class
            ]
        )


        e15_score = geometric_score(

            yolo_conf,

            e15_classifier_prob,

            alpha=ALPHA,
        )


        prediction_sets[
            "E15"
        ].append({

            "image_id":
                image_id,

            "category_id":
                IDX_TO_COCO7[
                    e15_final_class
                ],

            "bbox":
                bbox,

            "score":
                float(
                    e15_score
                ),
        })


        # ==========================================================================================
        # E16
        #
        # Class-specific gate indexed by MobileNet PROPOSED class.
        # ==========================================================================================

        e16_gate = E16_GATES[
            mn_pred_class
        ]


        if (
            mn_pred_prob
            >= e16_gate
        ):

            e16_final_class = (
                mn_pred_class
            )

            source_counts[
                "E16"
            ][
                "MobileNet"
            ] += 1


        else:

            e16_final_class = (
                yolo_class
            )

            source_counts[
                "E16"
            ][
                "YOLO"
            ] += 1


        e16_classifier_prob = float(

            mn_probs[
                e16_final_class
            ]
        )


        e16_score = geometric_score(

            yolo_conf,

            e16_classifier_prob,

            alpha=ALPHA,
        )


        prediction_sets[
            "E16"
        ].append({

            "image_id":
                image_id,

            "category_id":
                IDX_TO_COCO7[
                    e16_final_class
                ],

            "bbox":
                bbox,

            "score":
                float(
                    e16_score
                ),
        })


    # ----------------------------------------------------------------------------------------------
    # Progress
    # ----------------------------------------------------------------------------------------------

    if (

        image_number % 25 == 0

        or image_number
        == total_images

    ):

        print(

            f"Batch 5: "

            f"{image_number}/"
            f"{total_images}"
        )


elapsed = (
    time.time()
    - start_time
)


print()
print("=" * 100)
print("SHARED INFERENCE COMPLETE")
print("=" * 100)

print(
    f"Detector boxes : "
    f"{detector_boxes:,}"
)

print(
    f"Valid crops    : "
    f"{valid_crops:,}"
)

print(
    f"Invalid crops  : "
    f"{invalid_crops:,}"
)

print(
    f"Time           : "
    f"{elapsed/60:.2f} min"
)


for experiment in (
    prediction_sets
):

    print(

        f"{experiment:10s}: "
        f"{len(prediction_sets[experiment]):,} "
        f"predictions"
    )


print()
print(
    "Routing counts:"
)

for experiment in [
    "E14-C",
    "E15",
    "E16",
]:

    print(

        f"{experiment:10s} "

        f"YOLO="
        f"{source_counts[experiment]['YOLO']:,}   "

        f"MobileNet="
        f"{source_counts[experiment]['MobileNet']:,}"
    )


# ==================================================================================================
# 16. EVALUATE ALL FIVE EXPERIMENTS
# ==================================================================================================

experiment_names = {

    "E12":
        "E12 — YOLO11m + Augmentation + Class-Aware Oversampling",

    "E13":
        "E13 — YOLO11m + MobileNetV3-Large Class-Weighted",

    "E14-C":
        "E14-C — Confidence Fusion + Global MobileNet Class Gating",

    "E15":
        "E15 — Joint Class-Gating + Confidence-Fusion Optimization",

    "E16":
        "E16 — Class-Specific MobileNet Gating with Confidence Fusion",
}


extra_config = {

    "E12": {

        "final_class":
            "YOLO",

        "score":
            "YOLO confidence",
    },


    "E13": {

        "final_class":
            "MobileNet always",

        "score":
            "YOLO_conf * MobileNet_prob",
    },


    "E14-C": {

        "gate":
            E14_GATE,

        "alpha":
            ALPHA,

        "score":
            "YOLO_conf^0.70 * P_MobileNet(final_class)^0.30",
    },


    "E15": {

        "gate":
            E15_GATE,

        "alpha":
            ALPHA,

        "score":
            "YOLO_conf^0.70 * P_MobileNet(final_class)^0.30",
    },


    "E16": {

        "gates":
            {
                IDX_TO_NAME[k]:
                    v
                for k, v
                in E16_GATES.items()
            },

        "alpha":
            ALPHA,

        "score":
            "YOLO_conf^0.70 * P_MobileNet(final_class)^0.30",
    },
}


results = {}


for experiment in [

    "E12",

    "E13",

    "E14-C",

    "E15",

    "E16",

]:

    overall, per_class = (
        evaluate_coco(

            prediction_sets[
                experiment
            ],

            f"{experiment_names[experiment]} — TEST",
        )
    )


    results[
        experiment
    ] = {

        "overall":
            overall,

        "per_class":
            per_class,
    }


    save_results(

        experiment_names[
            experiment
        ],

        prediction_sets[
            experiment
        ],

        overall,

        per_class,

        OUT_ROOT
        / experiment,

        elapsed / 60,

        extra_info=
            extra_config[
                experiment
            ],
    )


# ==================================================================================================
# 17. FINAL OVERALL SUMMARY
# ==================================================================================================

print()
print()
print("=" * 100)
print("FINAL TEST — BATCH 5 COMPLETE")
print("=" * 100)


print()

print(

    f"{'Experiment':15s}"

    f"{'AP50-95':>14s}"

    f"{'AP50':>14s}"

    f"{'AP75':>14s}"

    f"{'AR100':>14s}"
)


print(
    "-" * 75
)


for experiment, result in (
    results.items()
):

    overall = (
        result[
            "overall"
        ]
    )


    print(

        f"{experiment:15s}"

        f"{overall['AP50-95']*100:13.4f}%"

        f"{overall['AP50']*100:13.4f}%"

        f"{overall['AP75']*100:13.4f}%"

        f"{overall['AR100']*100:13.4f}%"
    )


# ==================================================================================================
# 18. CLASS-WISE AP50
# ==================================================================================================

print()
print("=" * 100)
print("CLASS-WISE TEST AP50")
print("=" * 100)


print(

    f"{'Class':28s}"

    + "".join(

        f"{experiment:>12s}"

        for experiment
        in results
    )
)


print(

    "-" * (

        28
        + 12
        * len(
            results
        )
    )
)


for class_name in (
    CLASS7_NAMES.values()
):

    row = (
        f"{class_name:28s}"
    )


    for experiment in results:

        value = (

            results[
                experiment
            ][
                "per_class"
            ][
                class_name
            ][
                "AP50"
            ]
        )


        row += (
            f"{value*100:11.2f}%"
        )


    print(
        row
    )


# ==================================================================================================
# 19. CLASS-WISE AP50-95
# ==================================================================================================

print()
print("=" * 100)
print("CLASS-WISE TEST AP50-95")
print("=" * 100)


print(

    f"{'Class':28s}"

    + "".join(

        f"{experiment:>12s}"

        for experiment
        in results
    )
)


print(

    "-" * (

        28
        + 12
        * len(
            results
        )
    )
)


for class_name in (
    CLASS7_NAMES.values()
):

    row = (
        f"{class_name:28s}"
    )


    for experiment in results:

        value = (

            results[
                experiment
            ][
                "per_class"
            ][
                class_name
            ][
                "AP50-95"
            ]
        )


        row += (
            f"{value*100:11.2f}%"
        )


    print(
        row
    )


# ==================================================================================================
# 20. ROUTING SUMMARY
# ==================================================================================================

print()
print("=" * 100)
print("GATING / ROUTING SUMMARY")
print("=" * 100)


for experiment in [

    "E14-C",

    "E15",

    "E16",

]:

    total = (

        source_counts[
            experiment
        ][
            "YOLO"
        ]

        +

        source_counts[
            experiment
        ][
            "MobileNet"
        ]
    )


    yolo_count = (
        source_counts[
            experiment
        ][
            "YOLO"
        ]
    )


    mn_count = (
        source_counts[
            experiment
        ][
            "MobileNet"
        ]
    )


    print()

    print(
        experiment
    )


    print(

        f"YOLO final class      : "
        f"{yolo_count:,} "
        f"({100*yolo_count/total:.2f}%)"
    )


    print(

        f"MobileNet final class : "
        f"{mn_count:,} "
        f"({100*mn_count/total:.2f}%)"
    )


print()

print(
    "Outputs saved to:"
)

print(
    OUT_ROOT
)


print()
print("=" * 100)
print("FINAL TEST — BATCH 5 COMPLETE")
print("=" * 100)


# ==================================================================================================
# CLEANUP
# ==================================================================================================

del detector
del classifier

torch.cuda.empty_cache()

FINAL TEST — BATCH 5
PyTorch        : 2.13.0+cu126
Device         : cuda
GPU            : NVIDIA GeForce RTX 3050 Ti Laptop GPU

Test images    : C:\Users\varda\Documents\_My Computer\COMMON_space\Learning\Upgrad_EPGP_MS\upgrad_IIITB_MS_course\MS_LJMU_Material\Topic Data\SortWaste\dataset\dataset\splited_all_dataset_coco\test\images
Test COCO      : C:\Users\varda\Documents\_My Computer\COMMON_space\Learning\Upgrad_EPGP_MS\upgrad_IIITB_MS_course\MS_LJMU_Material\Topic Data\SortWaste\dataset\dataset\splited_all_dataset_coco\test\annotations\test_coco.json

E12 detector   : C:\Users\varda\Documents\_My Computer\COMMON_space\Learning\Upgrad_EPGP_MS\upgrad_IIITB_MS_course\MS_LJMU_Material\Thesis_Code\runs\sortwaste\E12_yolo11m_7class_aug_classbalance_640\weights\best.pt
MobileNet      : C:\Users\varda\Documents\_My Computer\COMMON_space\Learning\Upgrad_EPGP_MS\upgrad_IIITB_MS_course\MS_LJMU_Material\Topic Data\SortWaste\dataset\dataset\yolo_mobilenet_crops_E3Y\mobilenet_results\E3Y_B_class

## Batch 6
| Experiment | Configuration                                                                                               |
| ---------- | ----------------------------------------------------------------------------------------------------------- |
| **E17**    | YOLO11m @768 detector-only                                                                                  |
| **E18-E**  | E12 YOLO11m @640 + E3Y-B MobileNet + ConvNeXt-Tiny specialist                                               |
| **E19-B**  | E12 YOLO11m @640 + E3Y-B MobileNet + ConvNeXt-Small specialist, using the E18-E gates without recalibration |


In [ ]:
# FINAL TEST — BATCH 6
#
# E17   : YOLO11m @768 detector-only
# E18-E : E12 YOLO11m @640 + MobileNet E16 gating + ConvNeXt-Tiny specialist
# E19-B : E12 YOLO11m @640 + MobileNet E16 gating + ConvNeXt-Small specialist
#
# IMPORTANT
# --------------------------------------------------------------------------------------------------
# TEST SET ONLY
# NO TEST-SET TUNING
#
# E18-E / E19-B frozen configuration:
#
# Detector:
#   E12 YOLO11m @640
#   conf      = 0.001
#   NMS IoU   = 0.60
#   max_det   = 100
#
# MobileNet:
#   E3Y-B checkpoint
#
# MobileNet E16-A gates:
#   ecal                 0.99
#   hdpe                 0.98
#   mixed_plastic_rigid  0.98
#   mixed_plastic_soft   0.94
#   non_plastic          0.94
#   pet                  0.99
#   pet_oil              0.99
#
# Specialist routing:
#   mixed_plastic_rigid  DISABLED
#   mixed_plastic_soft   gate 0.90
#   non_plastic          gate 0.90
#   pet_oil              gate 0.94
#
# alpha = 0.70
#
# Final score:
#   YOLO_conf^0.70 * classifier_probability(final_class)^0.30
#
# For YOLO/MobileNet final decisions:
#   classifier_probability = MobileNet probability of actual final class
#
# For accepted ConvNeXt specialist decisions:
#   classifier_probability = ConvNeXt probability of accepted final class
#
# Explicit pycocotools COCOeval
# ==================================================================================================

import json
import time
import shutil
from pathlib import Path
from collections import Counter, defaultdict

import numpy as np
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F

from torchvision.models import (
    mobilenet_v3_large,
    convnext_tiny,
    convnext_small,
)

from torchvision.transforms import (
    Compose,
    Resize,
    ToTensor,
    Normalize,
)

from ultralytics import YOLO

from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval


# ==================================================================================================
# 1. PATHS
# ==================================================================================================

BASE = Path(
    r"C:\Users\varda\Documents\_My Computer\COMMON_space\Learning"
    r"\Upgrad_EPGP_MS\upgrad_IIITB_MS_course\MS_LJMU_Material"
)

DATASET_ROOT = (
    BASE
    / "Topic Data"
    / "SortWaste"
    / "dataset"
    / "dataset"
)

THESIS_CODE = (
    BASE
    / "Thesis_Code"
)


TEST_ROOT = (
    DATASET_ROOT
    / "splited_all_dataset_coco"
    / "test"
)

TEST_IMAGES = (
    TEST_ROOT
    / "images"
)

TEST_COCO = (
    TEST_ROOT
    / "annotations"
    / "test_coco.json"
)


# --------------------------------------------------------------------------------------------------
# E17
# --------------------------------------------------------------------------------------------------

E17_WEIGHTS = (
    THESIS_CODE
    / "runs"
    / "sortwaste"
    / "E17_yolo11m_7class_aug_classbalance_768"
    / "weights"
    / "best.pt"
)


# --------------------------------------------------------------------------------------------------
# E12 detector for E18/E19
# --------------------------------------------------------------------------------------------------

E12_WEIGHTS = (
    THESIS_CODE
    / "runs"
    / "sortwaste"
    / "E12_yolo11m_7class_aug_classbalance_640"
    / "weights"
    / "best.pt"
)


# --------------------------------------------------------------------------------------------------
# E3Y-B MobileNet
# --------------------------------------------------------------------------------------------------

MOBILENET_WEIGHTS = (
    DATASET_ROOT
    / "yolo_mobilenet_crops_E3Y"
    / "mobilenet_results"
    / "E3Y_B_class_weighted"
    / "E3Y_B_MobileNetV3Large_best.pth"
)


# --------------------------------------------------------------------------------------------------
# E18-B ConvNeXt-Tiny
# --------------------------------------------------------------------------------------------------

CONVNEXT_TINY_WEIGHTS = (
    THESIS_CODE
    / "runs"
    / "sortwaste"
    / "E18B_convnext_tiny_hardclass"
    / "E18B_ConvNeXtTiny_best.pth"
)


# --------------------------------------------------------------------------------------------------
# E19-A ConvNeXt-Small
# --------------------------------------------------------------------------------------------------

CONVNEXT_SMALL_WEIGHTS = (
    THESIS_CODE
    / "runs"
    / "sortwaste"
    / "E19A_convnext_small_hardclass"
    / "E19A_ConvNeXtSmall_best.pth"
)


# --------------------------------------------------------------------------------------------------
# Output
# --------------------------------------------------------------------------------------------------

OUT_ROOT = (
    THESIS_CODE
    / "runs"
    / "sortwaste"
    / "FINAL_TEST_BATCH6"
)


if OUT_ROOT.exists():

    print(
        "Deleting old Batch 6 results:\n"
        f"{OUT_ROOT}"
    )

    shutil.rmtree(
        OUT_ROOT
    )


OUT_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)


# ==================================================================================================
# 2. DEVICE
# ==================================================================================================

DEVICE = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

YOLO_DEVICE = (
    0
    if torch.cuda.is_available()
    else "cpu"
)


print("=" * 100)
print("FINAL TEST — BATCH 6")
print("=" * 100)

print(f"PyTorch          : {torch.__version__}")
print(f"Device           : {DEVICE}")

if torch.cuda.is_available():

    print(
        f"GPU              : "
        f"{torch.cuda.get_device_name(0)}"
    )


print()

print(f"Test images      : {TEST_IMAGES}")
print(f"Test COCO        : {TEST_COCO}")

print()

print(f"E17 detector     : {E17_WEIGHTS}")
print(f"E12 detector     : {E12_WEIGHTS}")
print(f"MobileNet        : {MOBILENET_WEIGHTS}")
print(f"ConvNeXt-Tiny    : {CONVNEXT_TINY_WEIGHTS}")
print(f"ConvNeXt-Small   : {CONVNEXT_SMALL_WEIGHTS}")

print()

print(f"Output           : {OUT_ROOT}")


for p in [

    TEST_IMAGES,
    TEST_COCO,

    E17_WEIGHTS,
    E12_WEIGHTS,

    MOBILENET_WEIGHTS,

    CONVNEXT_TINY_WEIGHTS,
    CONVNEXT_SMALL_WEIGHTS,

]:

    if not p.exists():

        raise FileNotFoundError(
            f"Missing required path:\n{p}"
        )


# ==================================================================================================
# 3. TAXONOMY
# ==================================================================================================

CLASS7_NAMES = {

    1: "ecal",

    2: "hdpe",

    3: "mixed_plastic_rigid",

    4: "mixed_plastic_soft",

    5: "non_plastic",

    6: "pet",

    7: "pet_oil",
}


IDX_TO_NAME = {

    0: "ecal",

    1: "hdpe",

    2: "mixed_plastic_rigid",

    3: "mixed_plastic_soft",

    4: "non_plastic",

    5: "pet",

    6: "pet_oil",
}


IDX_TO_COCO7 = {

    0: 1,

    1: 2,

    2: 3,

    3: 4,

    4: 5,

    5: 6,

    6: 7,
}


COCO8_TO_7 = {

    1: 6,   # PET

    2: 2,   # HDPE

    3: 4,   # mixed soft

    4: 1,   # ECAL

    5: 5,   # metal -> nonplastic

    6: 5,   # cardboard -> nonplastic

    7: 3,   # mixed rigid

    8: 7,   # PET Oil
}


# ==================================================================================================
# 4. HARD-CLASS SPECIALIST TAXONOMY
#
# ConvNeXt was trained on four classes:
#
# 0 = mixed_plastic_rigid
# 1 = mixed_plastic_soft
# 2 = non_plastic
# 3 = pet_oil
#
# Map local specialist class -> global 7-class model index
# ==================================================================================================

CONV_LOCAL_TO_GLOBAL = {

    0: 2,  # mixed rigid

    1: 3,  # mixed soft

    2: 4,  # nonplastic

    3: 6,  # pet oil
}


GLOBAL_TO_CONV_LOCAL = {

    2: 0,

    3: 1,

    4: 2,

    6: 3,
}


# ==================================================================================================
# 5. FROZEN PARAMETERS
# ==================================================================================================

# E17
E17_IMGSZ = 768

# E12 / E18 / E19
E12_IMGSZ = 640

CONF_THRESHOLD = 0.001

NMS_IOU = 0.60

MAX_DET = 100


# Confidence fusion
ALPHA = 0.70


# --------------------------------------------------------------------------------------------------
# E16-A MobileNet class-specific gates
# --------------------------------------------------------------------------------------------------

MN_GATES = {

    0: 0.99,  # ECAL

    1: 0.98,  # HDPE

    2: 0.98,  # Mixed Rigid

    3: 0.94,  # Mixed Soft

    4: 0.94,  # Nonplastic

    5: 0.99,  # PET

    6: 0.99,  # PET Oil
}


# --------------------------------------------------------------------------------------------------
# E18-E / E19-B specialist gates
#
# IMPORTANT:
# Mixed Rigid specialist routing is DISABLED.
# --------------------------------------------------------------------------------------------------

SPECIALIST_GATES = {

    3: 0.90,   # mixed soft

    4: 0.90,   # nonplastic

    6: 0.94,   # PET oil
}


SPECIALIST_ALLOWED_GLOBAL_CLASSES = set(
    SPECIALIST_GATES.keys()
)


# ==================================================================================================
# 6. BUILD 7-CLASS TEST GT
# ==================================================================================================

with open(
    TEST_COCO,
    "r",
    encoding="utf-8",
) as f:

    gt8 = json.load(f)


gt7 = {

    "info":
        gt8.get(
            "info",
            {},
        ),

    "licenses":
        gt8.get(
            "licenses",
            [],
        ),

    "images":
        gt8[
            "images"
        ],

    "annotations":
        [],

    "categories": [

        {
            "id":
                cid,

            "name":
                name,
        }

        for cid, name
        in CLASS7_NAMES.items()
    ],
}


for ann in gt8[
    "annotations"
]:

    new_ann = (
        ann.copy()
    )


    new_ann[
        "category_id"
    ] = COCO8_TO_7[
        int(
            ann[
                "category_id"
            ]
        )
    ]


    gt7[
        "annotations"
    ].append(
        new_ann
    )


GT7_PATH = (
    OUT_ROOT
    / "test_gt_7class.json"
)


with open(
    GT7_PATH,
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        gt7,
        f,
    )


print()
print("=" * 100)
print("TEST DATASET")
print("=" * 100)

print(
    f"Images : "
    f"{len(gt7['images']):,}"
)

print(
    f"GT     : "
    f"{len(gt7['annotations']):,}"
)


counts = Counter(

    x[
        "category_id"
    ]

    for x in
    gt7[
        "annotations"
    ]
)


print()

for cid, name in (
    CLASS7_NAMES.items()
):

    print(
        f"{name:25s}: "
        f"{counts[cid]:,}"
    )


# ==================================================================================================
# 7. TRANSFORMS
# ==================================================================================================

CLASSIFIER_TRANSFORM = Compose([

    Resize(
        (224, 224)
    ),

    ToTensor(),

    Normalize(

        mean=[
            0.485,
            0.456,
            0.406,
        ],

        std=[
            0.229,
            0.224,
            0.225,
        ],
    ),
])


# ==================================================================================================
# 8. CHECKPOINT HELPER
# ==================================================================================================

def extract_state_dict(
    checkpoint,
):

    if not isinstance(
        checkpoint,
        dict,
    ):

        return checkpoint


    if (
        "model_state_dict"
        in checkpoint
    ):

        state = checkpoint[
            "model_state_dict"
        ]


    elif (
        "state_dict"
        in checkpoint
    ):

        state = checkpoint[
            "state_dict"
        ]


    elif (

        "model"
        in checkpoint

        and isinstance(
            checkpoint[
                "model"
            ],
            dict,
        )
    ):

        state = checkpoint[
            "model"
        ]


    else:

        state = checkpoint


    cleaned = {}


    for key, value in (
        state.items()
    ):

        if key.startswith(
            "module."
        ):

            key = key[
                len("module.") :
            ]


        cleaned[
            key
        ] = value


    return cleaned


# ==================================================================================================
# 9. LOAD MOBILENET
# ==================================================================================================

def load_mobilenet():

    checkpoint = torch.load(

        MOBILENET_WEIGHTS,

        map_location=DEVICE,

        weights_only=False,
    )


    state = extract_state_dict(
        checkpoint
    )


    num_classes = int(

        state[
            "classifier.3.weight"
        ].shape[
            0
        ]
    )


    if num_classes != 7:

        raise RuntimeError(

            f"MobileNet expected 7 outputs; "
            f"found {num_classes}"
        )


    model = (
        mobilenet_v3_large(
            weights=None
        )
    )


    in_features = (

        model
        .classifier[
            3
        ]
        .in_features
    )


    model.classifier[
        3
    ] = nn.Linear(

        in_features,

        7,
    )


    model.load_state_dict(

        state,

        strict=True,
    )


    model.to(
        DEVICE
    )


    model.eval()


    return model


# ==================================================================================================
# 10. LOAD CONVNEXT SPECIALIST
# ==================================================================================================

def load_convnext(
    checkpoint_path,
    architecture,
):

    checkpoint = torch.load(

        checkpoint_path,

        map_location=DEVICE,

        weights_only=False,
    )


    state = extract_state_dict(
        checkpoint
    )


    # torchvision ConvNeXt classifier final layer:
    # classifier.2
    final_key = (
        "classifier.2.weight"
    )


    if final_key not in state:

        raise RuntimeError(

            f"Could not find "
            f"{final_key} in "
            f"{checkpoint_path}"
        )


    num_classes = int(

        state[
            final_key
        ].shape[
            0
        ]
    )


    if (
        num_classes
        != 4
    ):

        raise RuntimeError(

            f"ConvNeXt specialist expected "
            f"4 classes; found "
            f"{num_classes}"
        )


    if (
        architecture
        == "tiny"
    ):

        model = (
            convnext_tiny(
                weights=None
            )
        )


    elif (
        architecture
        == "small"
    ):

        model = (
            convnext_small(
                weights=None
            )
        )


    else:

        raise ValueError(
            architecture
        )


    in_features = (

        model
        .classifier[
            2
        ]
        .in_features
    )


    model.classifier[
        2
    ] = nn.Linear(

        in_features,

        4,
    )


    model.load_state_dict(

        state,

        strict=True,
    )


    model.to(
        DEVICE
    )


    model.eval()


    return model


# ==================================================================================================
# 11. CLASSIFIER PROBABILITY INFERENCE
# ==================================================================================================

@torch.no_grad()
def classify_crops_all_probs(
    image,
    boxes,
    model,
    batch_size,
):

    W, H = (
        image.size
    )


    crops = []

    valid_positions = []


    for index, box in enumerate(
        boxes
    ):

        x1, y1, x2, y2 = [

            float(
                v
            )

            for v
            in box
        ]


        x1 = max(
            0.0,
            min(
                x1,
                W - 1,
            ),
        )


        y1 = max(
            0.0,
            min(
                y1,
                H - 1,
            ),
        )


        x2 = max(
            0.0,
            min(
                x2,
                W,
            ),
        )


        y2 = max(
            0.0,
            min(
                y2,
                H,
            ),
        )


        if (
            x2 <= x1
            or y2 <= y1
        ):

            continue


        crop = image.crop(

            (
                x1,
                y1,
                x2,
                y2,
            )
        )


        crops.append(

            CLASSIFIER_TRANSFORM(
                crop
            )
        )


        valid_positions.append(
            index
        )


    outputs = [
        None
    ] * len(
        boxes
    )


    for start in range(
        0,
        len(
            crops
        ),
        batch_size,
    ):

        batch = torch.stack(

            crops[
                start :
                start
                + batch_size
            ]

        ).to(
            DEVICE
        )


        logits = model(
            batch
        )


        probs = F.softmax(

            logits,

            dim=1,

        ).detach().cpu().numpy()


        for local_idx, vector in enumerate(
            probs
        ):

            original_idx = (
                valid_positions[
                    start
                    + local_idx
                ]
            )


            outputs[
                original_idx
            ] = (
                vector.astype(
                    np.float32
                )
            )


    return outputs


# ==================================================================================================
# 12. FUSION SCORE
# ==================================================================================================

def fusion_score(
    yolo_conf,
    classifier_prob,
):

    yolo_conf = max(

        float(
            yolo_conf
        ),

        1e-12,
    )


    classifier_prob = max(

        float(
            classifier_prob
        ),

        1e-12,
    )


    return (

        yolo_conf
        ** ALPHA

        *

        classifier_prob
        ** (
            1.0
            - ALPHA
        )
    )


# ==================================================================================================
# 13. COCO EVALUATION
# ==================================================================================================

def evaluate_coco(
    predictions,
    title,
):

    print()
    print("=" * 100)
    print(title)
    print("=" * 100)


    coco_gt = COCO(
        str(
            GT7_PATH
        )
    )


    coco_dt = coco_gt.loadRes(
        predictions
    )


    evaluator = COCOeval(

        coco_gt,

        coco_dt,

        "bbox",
    )


    evaluator.params.maxDets = [

        1,

        10,

        100,
    ]


    evaluator.evaluate()

    evaluator.accumulate()

    evaluator.summarize()


    overall = {

        "AP50-95":
            float(
                evaluator.stats[
                    0
                ]
            ),

        "AP50":
            float(
                evaluator.stats[
                    1
                ]
            ),

        "AP75":
            float(
                evaluator.stats[
                    2
                ]
            ),

        "AR1":
            float(
                evaluator.stats[
                    6
                ]
            ),

        "AR10":
            float(
                evaluator.stats[
                    7
                ]
            ),

        "AR100":
            float(
                evaluator.stats[
                    8
                ]
            ),
    }


    precision = (
        evaluator.eval[
            "precision"
        ]
    )


    per_class = {}


    for k, category_id in enumerate(
        evaluator.params.catIds
    ):

        class_name = (
            CLASS7_NAMES[
                category_id
            ]
        )


        values = precision[
            :,
            :,
            k,
            0,
            -1,
        ]


        valid = (
            values[
                values > -1
            ]
        )


        ap = (

            float(
                valid.mean()
            )

            if valid.size

            else float(
                "nan"
            )
        )


        values50 = precision[
            0,
            :,
            k,
            0,
            -1,
        ]


        valid50 = (
            values50[
                values50 > -1
            ]
        )


        ap50 = (

            float(
                valid50.mean()
            )

            if valid50.size

            else float(
                "nan"
            )
        )


        per_class[
            class_name
        ] = {

            "AP50":
                ap50,

            "AP50-95":
                ap,
        }


    print()
    print("CLASS-WISE RESULTS")
    print("-" * 76)

    print(
        f"{'Class':30s}"
        f"{'AP50':>15s}"
        f"{'AP50-95':>15s}"
    )

    print("-" * 76)


    for class_name, vals in (
        per_class.items()
    ):

        print(

            f"{class_name:30s}"

            f"{vals['AP50']*100:14.2f}%"

            f"{vals['AP50-95']*100:14.2f}%"
        )


    return (
        overall,
        per_class,
    )


# ==================================================================================================
# 14. SAVE RESULTS
# ==================================================================================================

def save_results(
    experiment,
    predictions,
    overall,
    per_class,
    output_dir,
    inference_minutes,
    configuration,
):

    output_dir.mkdir(
        parents=True,
        exist_ok=True,
    )


    with open(
        output_dir
        / "predictions.json",
        "w",
        encoding="utf-8",
    ) as f:

        json.dump(
            predictions,
            f,
        )


    results = {

        "experiment":
            experiment,

        "taxonomy":
            "standardized 7-class",

        "inference_minutes":
            float(
                inference_minutes
            ),

        "overall":
            overall,

        "per_class":
            per_class,

        "configuration":
            configuration,
    }


    with open(
        output_dir
        / "results.json",
        "w",
        encoding="utf-8",
    ) as f:

        json.dump(
            results,
            f,
            indent=2,
        )


# ==================================================================================================
# 15. E17 — YOLO11m @768
# ==================================================================================================

print()
print()
print("#" * 100)
print("E17 — YOLO11m @768")
print("#" * 100)


e17_model = YOLO(
    str(
        E17_WEIGHTS
    )
)


e17_predictions = []


start_e17 = (
    time.time()
)


total_images = len(
    gt7[
        "images"
    ]
)


for image_number, image_info in enumerate(
    gt7[
        "images"
    ],
    start=1,
):

    image_path = (

        TEST_IMAGES

        / Path(
            image_info[
                "file_name"
            ]
        ).name
    )


    image_id = int(
        image_info[
            "id"
        ]
    )


    result = e17_model.predict(

        source=str(
            image_path
        ),

        imgsz=E17_IMGSZ,

        conf=CONF_THRESHOLD,

        iou=NMS_IOU,

        max_det=MAX_DET,

        device=YOLO_DEVICE,

        verbose=False,

    )[0]


    if (
        result.boxes is None
        or len(
            result.boxes
        ) == 0
    ):

        continue


    boxes = (
        result.boxes.xyxy
        .detach()
        .cpu()
        .numpy()
    )


    scores = (
        result.boxes.conf
        .detach()
        .cpu()
        .numpy()
    )


    classes = (
        result.boxes.cls
        .detach()
        .cpu()
        .numpy()
        .astype(
            int
        )
    )


    for box, score, cls in zip(
        boxes,
        scores,
        classes,
    ):

        x1, y1, x2, y2 = [

            float(
                v
            )

            for v
            in box
        ]


        w = (
            x2 - x1
        )

        h = (
            y2 - y1
        )


        if (
            w <= 0
            or h <= 0
        ):

            continue


        e17_predictions.append({

            "image_id":
                image_id,

            "category_id":
                IDX_TO_COCO7[
                    int(
                        cls
                    )
                ],

            "bbox": [

                x1,

                y1,

                w,

                h,
            ],

            "score":
                float(
                    score
                ),
        })


    if (

        image_number % 25 == 0

        or image_number
        == total_images

    ):

        print(

            f"E17: "
            f"{image_number}/"
            f"{total_images}"
        )


e17_time = (
    time.time()
    - start_e17
)


print()

print(
    f"E17 predictions : "
    f"{len(e17_predictions):,}"
)

print(
    f"E17 time        : "
    f"{e17_time/60:.2f} min"
)


del e17_model

torch.cuda.empty_cache()


# ==================================================================================================
# 16. LOAD SHARED E18/E19 MODELS
# ==================================================================================================

print()
print("=" * 100)
print("LOADING E18/E19 MODELS")
print("=" * 100)


detector = YOLO(
    str(
        E12_WEIGHTS
    )
)


mobilenet = (
    load_mobilenet()
)


conv_tiny = (
    load_convnext(

        CONVNEXT_TINY_WEIGHTS,

        "tiny",
    )
)


conv_small = (
    load_convnext(

        CONVNEXT_SMALL_WEIGHTS,

        "small",
    )
)


# ==================================================================================================
# 17. E18-E / E19-B SHARED INFERENCE
# ==================================================================================================

e18_predictions = []

e19_predictions = []


routing = {

    "E18-E":
        defaultdict(
            int
        ),

    "E19-B":
        defaultdict(
            int
        ),
}


conv_proposals = {

    "E18-E":
        defaultdict(
            int
        ),

    "E19-B":
        defaultdict(
            int
        ),
}


conv_accepts = {

    "E18-E":
        defaultdict(
            int
        ),

    "E19-B":
        defaultdict(
            int
        ),
}


start_shared = (
    time.time()
)


print()
print("=" * 100)
print("RUNNING E18-E / E19-B SHARED INFERENCE")
print("=" * 100)


for image_number, image_info in enumerate(
    gt7[
        "images"
    ],
    start=1,
):

    image_path = (

        TEST_IMAGES

        / Path(
            image_info[
                "file_name"
            ]
        ).name
    )


    image_id = int(
        image_info[
            "id"
        ]
    )


    image = Image.open(
        image_path
    ).convert(
        "RGB"
    )


    result = detector.predict(

        source=str(
            image_path
        ),

        imgsz=E12_IMGSZ,

        conf=CONF_THRESHOLD,

        iou=NMS_IOU,

        max_det=MAX_DET,

        device=YOLO_DEVICE,

        verbose=False,

    )[0]


    if (
        result.boxes is None
        or len(
            result.boxes
        ) == 0
    ):

        continue


    boxes = (
        result.boxes.xyxy
        .detach()
        .cpu()
        .numpy()
    )


    yolo_confidences = (
        result.boxes.conf
        .detach()
        .cpu()
        .numpy()
    )


    yolo_classes = (
        result.boxes.cls
        .detach()
        .cpu()
        .numpy()
        .astype(
            int
        )
    )


    # ----------------------------------------------------------------------------------------------
    # MobileNet probabilities for every YOLO crop
    # ----------------------------------------------------------------------------------------------

    mn_probs_all = (
        classify_crops_all_probs(

            image,

            boxes,

            mobilenet,

            batch_size=64,
        )
    )


    # ----------------------------------------------------------------------------------------------
    # First determine E16 final classes.
    #
    # Specialist is only consulted when the E16 FINAL CLASS is one of the hard classes.
    # ----------------------------------------------------------------------------------------------

    base_records = []


    specialist_candidate_positions = []


    for idx, (
        box,
        yolo_conf,
        yolo_class,
        mn_probs,
    ) in enumerate(
        zip(
            boxes,
            yolo_confidences,
            yolo_classes,
            mn_probs_all,
        )
    ):

        if (
            mn_probs
            is None
        ):

            base_records.append(
                None
            )

            continue


        yolo_class = int(
            yolo_class
        )


        mn_pred_class = int(
            np.argmax(
                mn_probs
            )
        )


        mn_pred_prob = float(
            mn_probs[
                mn_pred_class
            ]
        )


        mn_gate = (
            MN_GATES[
                mn_pred_class
            ]
        )


        # E16 base final class
        if (
            mn_pred_prob
            >= mn_gate
        ):

            base_final_class = (
                mn_pred_class
            )

            base_source = (
                "MobileNet"
            )


        else:

            base_final_class = (
                yolo_class
            )

            base_source = (
                "YOLO"
            )


        base_classifier_prob = float(
            mn_probs[
                base_final_class
            ]
        )


        x1, y1, x2, y2 = [

            float(
                v
            )

            for v
            in box
        ]


        w = (
            x2 - x1
        )

        h = (
            y2 - y1
        )


        if (
            w <= 0
            or h <= 0
        ):

            base_records.append(
                None
            )

            continue


        record = {

            "bbox": [
                x1,
                y1,
                w,
                h,
            ],

            "yolo_conf":
                float(
                    yolo_conf
                ),

            "base_final_class":
                int(
                    base_final_class
                ),

            "base_source":
                base_source,

            "base_classifier_prob":
                base_classifier_prob,
        }


        base_records.append(
            record
        )


        # Specialist only consulted if E16 final class is hard
        if (
            base_final_class
            in GLOBAL_TO_CONV_LOCAL
        ):

            specialist_candidate_positions.append(
                idx
            )


    # ----------------------------------------------------------------------------------------------
    # Run specialists only on candidate boxes
    # ----------------------------------------------------------------------------------------------

    if (
        specialist_candidate_positions
    ):

        specialist_boxes = np.array(
            [
                boxes[
                    idx
                ]

                for idx
                in specialist_candidate_positions
            ]
        )


        tiny_probs_subset = (
            classify_crops_all_probs(

                image,

                specialist_boxes,

                conv_tiny,

                batch_size=8,
            )
        )


        small_probs_subset = (
            classify_crops_all_probs(

                image,

                specialist_boxes,

                conv_small,

                batch_size=4,
            )
        )


        tiny_by_original = {}


        small_by_original = {}


        for local_idx, original_idx in enumerate(
            specialist_candidate_positions
        ):

            tiny_by_original[
                original_idx
            ] = tiny_probs_subset[
                local_idx
            ]


            small_by_original[
                original_idx
            ] = small_probs_subset[
                local_idx
            ]


    else:

        tiny_by_original = {}

        small_by_original = {}


    # ==============================================================================================
    # Construct E18-E / E19-B predictions
    # ==============================================================================================

    for idx, record in enumerate(
        base_records
    ):

        if (
            record
            is None
        ):

            continue


        # ------------------------------------------------------------------------------------------
        # Function-like repeated logic
        # ------------------------------------------------------------------------------------------

        for experiment, specialist_probs in [

            (
                "E18-E",
                tiny_by_original.get(
                    idx
                ),
            ),

            (
                "E19-B",
                small_by_original.get(
                    idx
                ),
            ),

        ]:

            final_class = (
                record[
                    "base_final_class"
                ]
            )


            final_source = (
                record[
                    "base_source"
                ]
            )


            classifier_prob = (
                record[
                    "base_classifier_prob"
                ]
            )


            # --------------------------------------------------------------------------------------
            # Specialist proposed result
            # --------------------------------------------------------------------------------------

            if (
                specialist_probs
                is not None
            ):

                specialist_local_class = int(
                    np.argmax(
                        specialist_probs
                    )
                )


                specialist_prob = float(
                    specialist_probs[
                        specialist_local_class
                    ]
                )


                specialist_global_class = (
                    CONV_LOCAL_TO_GLOBAL[
                        specialist_local_class
                    ]
                )


                conv_proposals[
                    experiment
                ][
                    IDX_TO_NAME[
                        specialist_global_class
                    ]
                ] += 1


                # ----------------------------------------------------------------------------------
                # E18-D2 / E18-E routing:
                #
                # Only MS + NP + PETOil proposals are allowed.
                # MR proposals are deliberately disabled.
                # ----------------------------------------------------------------------------------

                if (
                    specialist_global_class
                    in SPECIALIST_ALLOWED_GLOBAL_CLASSES
                ):

                    required_gate = (
                        SPECIALIST_GATES[
                            specialist_global_class
                        ]
                    )


                    if (
                        specialist_prob
                        >= required_gate
                    ):

                        final_class = (
                            specialist_global_class
                        )


                        final_source = (
                            "ConvNeXt"
                        )


                        classifier_prob = (
                            specialist_prob
                        )


                        conv_accepts[
                            experiment
                        ][
                            IDX_TO_NAME[
                                specialist_global_class
                            ]
                        ] += 1


            routing[
                experiment
            ][
                final_source
            ] += 1


            final_score = (
                fusion_score(

                    record[
                        "yolo_conf"
                    ],

                    classifier_prob,
                )
            )


            prediction = {

                "image_id":
                    image_id,

                "category_id":
                    IDX_TO_COCO7[
                        final_class
                    ],

                "bbox":
                    record[
                        "bbox"
                    ],

                "score":
                    float(
                        final_score
                    ),
            }


            if (
                experiment
                == "E18-E"
            ):

                e18_predictions.append(
                    prediction
                )


            else:

                e19_predictions.append(
                    prediction
                )


    if (

        image_number % 25 == 0

        or image_number
        == total_images

    ):

        print(

            f"E18/E19: "
            f"{image_number}/"
            f"{total_images}"
        )


shared_time = (
    time.time()
    - start_shared
)


print()
print("=" * 100)
print("E18/E19 SHARED INFERENCE COMPLETE")
print("=" * 100)

print(
    f"E18-E predictions : "
    f"{len(e18_predictions):,}"
)

print(
    f"E19-B predictions : "
    f"{len(e19_predictions):,}"
)

print(
    f"Shared time       : "
    f"{shared_time/60:.2f} min"
)


# ==================================================================================================
# 18. EVALUATE
# ==================================================================================================

results = {}


# --------------------------------------------------------------------------------------------------
# E17
# --------------------------------------------------------------------------------------------------

overall, per_class = (
    evaluate_coco(

        e17_predictions,

        "E17 — YOLO11m @768 — TEST",
    )
)


results[
    "E17"
] = {

    "overall":
        overall,

    "per_class":
        per_class,
}


save_results(

    "E17 — YOLO11m @768",

    e17_predictions,

    overall,

    per_class,

    OUT_ROOT
    / "E17",

    e17_time / 60,

    configuration={

        "imgsz":
            768,

        "conf":
            CONF_THRESHOLD,

        "nms_iou":
            NMS_IOU,

        "max_det":
            MAX_DET,
    },
)


# --------------------------------------------------------------------------------------------------
# E18-E
# --------------------------------------------------------------------------------------------------

overall, per_class = (
    evaluate_coco(

        e18_predictions,

        "E18-E — ConvNeXt-Tiny Class-Specific Gate Refinement — TEST",
    )
)


results[
    "E18-E"
] = {

    "overall":
        overall,

    "per_class":
        per_class,
}


save_results(

    "E18-E — ConvNeXt-Tiny Class-Specific Gate Refinement",

    e18_predictions,

    overall,

    per_class,

    OUT_ROOT
    / "E18-E",

    shared_time / 60,

    configuration={

        "detector":
            "E12 YOLO11m @640",

        "mobilenet":
            "E3Y-B",

        "specialist":
            "ConvNeXt-Tiny",

        "alpha":
            ALPHA,

        "specialist_gates":
            {
                "mixed_plastic_soft":
                    0.90,

                "non_plastic":
                    0.90,

                "pet_oil":
                    0.94,
            },

        "mixed_rigid_specialist":
            "disabled",
    },
)


# --------------------------------------------------------------------------------------------------
# E19-B
# --------------------------------------------------------------------------------------------------

overall, per_class = (
    evaluate_coco(

        e19_predictions,

        "E19-B — ConvNeXt-Small Pipeline with E18-E Gates — TEST",
    )
)


results[
    "E19-B"
] = {

    "overall":
        overall,

    "per_class":
        per_class,
}


save_results(

    "E19-B — YOLO11m + MobileNet + ConvNeXt-Small",

    e19_predictions,

    overall,

    per_class,

    OUT_ROOT
    / "E19-B",

    shared_time / 60,

    configuration={

        "detector":
            "E12 YOLO11m @640",

        "mobilenet":
            "E3Y-B",

        "specialist":
            "ConvNeXt-Small",

        "alpha":
            ALPHA,

        "specialist_gates":
            {
                "mixed_plastic_soft":
                    0.90,

                "non_plastic":
                    0.90,

                "pet_oil":
                    0.94,
            },

        "gate_source":
            "E18-E Tiny-derived gates; no Small recalibration",

        "mixed_rigid_specialist":
            "disabled",
    },
)


# ==================================================================================================
# 19. FINAL SUMMARY
# ==================================================================================================

print()
print()
print("=" * 100)
print("FINAL TEST — BATCH 6 COMPLETE")
print("=" * 100)


print()

print(

    f"{'Experiment':15s}"

    f"{'AP50-95':>14s}"

    f"{'AP50':>14s}"

    f"{'AP75':>14s}"

    f"{'AR100':>14s}"
)


print(
    "-" * 75
)


for experiment, result in (
    results.items()
):

    overall = (
        result[
            "overall"
        ]
    )


    print(

        f"{experiment:15s}"

        f"{overall['AP50-95']*100:13.4f}%"

        f"{overall['AP50']*100:13.4f}%"

        f"{overall['AP75']*100:13.4f}%"

        f"{overall['AR100']*100:13.4f}%"
    )


# ==================================================================================================
# 20. CLASS-WISE AP50
# ==================================================================================================

print()
print("=" * 100)
print("CLASS-WISE TEST AP50")
print("=" * 100)


print(

    f"{'Class':28s}"

    + "".join(

        f"{experiment:>14s}"

        for experiment
        in results
    )
)


print(

    "-" * (

        28

        + 14
        * len(
            results
        )
    )
)


for class_name in (
    CLASS7_NAMES.values()
):

    row = (
        f"{class_name:28s}"
    )


    for experiment in results:

        value = (

            results[
                experiment
            ][
                "per_class"
            ][
                class_name
            ][
                "AP50"
            ]
        )


        row += (
            f"{value*100:13.2f}%"
        )


    print(
        row
    )


# ==================================================================================================
# 21. CLASS-WISE AP50-95
# ==================================================================================================

print()
print("=" * 100)
print("CLASS-WISE TEST AP50-95")
print("=" * 100)


print(

    f"{'Class':28s}"

    + "".join(

        f"{experiment:>14s}"

        for experiment
        in results
    )
)


print(

    "-" * (

        28

        + 14
        * len(
            results
        )
    )
)


for class_name in (
    CLASS7_NAMES.values()
):

    row = (
        f"{class_name:28s}"
    )


    for experiment in results:

        value = (

            results[
                experiment
            ][
                "per_class"
            ][
                class_name
            ][
                "AP50-95"
            ]
        )


        row += (
            f"{value*100:13.2f}%"
        )


    print(
        row
    )


# ==================================================================================================
# 22. SPECIALIST ROUTING SUMMARY
# ==================================================================================================

print()
print("=" * 100)
print("SPECIALIST ROUTING SUMMARY")
print("=" * 100)


for experiment in [

    "E18-E",

    "E19-B",

]:

    print()
    print(
        experiment
    )


    total = sum(
        routing[
            experiment
        ].values()
    )


    for source in [

        "YOLO",

        "MobileNet",

        "ConvNeXt",

    ]:

        count = (
            routing[
                experiment
            ][
                source
            ]
        )


        percentage = (

            100.0
            * count
            / total

            if total

            else 0.0
        )


        print(

            f"{source:12s}: "
            f"{count:,} "
            f"({percentage:.2f}%)"
        )


    print()
    print(
        "ConvNeXt proposed:"
    )


    for cls in [

        "mixed_plastic_rigid",

        "mixed_plastic_soft",

        "non_plastic",

        "pet_oil",

    ]:

        print(

            f"  {cls:25s}: "
            f"{conv_proposals[experiment][cls]:,}"
        )


    print()
    print(
        "ConvNeXt accepted:"
    )


    for cls in [

        "mixed_plastic_soft",

        "non_plastic",

        "pet_oil",

    ]:

        print(

            f"  {cls:25s}: "
            f"{conv_accepts[experiment][cls]:,}"
        )


print()

print(
    "Outputs saved to:"
)

print(
    OUT_ROOT
)


print()
print("=" * 100)
print("FINAL TEST — BATCH 6 COMPLETE")
print("=" * 100)


# ==================================================================================================
# CLEANUP
# ==================================================================================================

del detector
del mobilenet
del conv_tiny
del conv_small

torch.cuda.empty_cache()

FINAL TEST — BATCH 6
PyTorch          : 2.13.0+cu126
Device           : cuda
GPU              : NVIDIA GeForce RTX 3050 Ti Laptop GPU

Test images      : C:\Users\varda\Documents\_My Computer\COMMON_space\Learning\Upgrad_EPGP_MS\upgrad_IIITB_MS_course\MS_LJMU_Material\Topic Data\SortWaste\dataset\dataset\splited_all_dataset_coco\test\images
Test COCO        : C:\Users\varda\Documents\_My Computer\COMMON_space\Learning\Upgrad_EPGP_MS\upgrad_IIITB_MS_course\MS_LJMU_Material\Topic Data\SortWaste\dataset\dataset\splited_all_dataset_coco\test\annotations\test_coco.json

E17 detector     : C:\Users\varda\Documents\_My Computer\COMMON_space\Learning\Upgrad_EPGP_MS\upgrad_IIITB_MS_course\MS_LJMU_Material\Thesis_Code\runs\sortwaste\E17_yolo11m_7class_aug_classbalance_768\weights\best.pt
E12 detector     : C:\Users\varda\Documents\_My Computer\COMMON_space\Learning\Upgrad_EPGP_MS\upgrad_IIITB_MS_course\MS_LJMU_Material\Thesis_Code\runs\sortwaste\E12_yolo11m_7class_aug_classbalance_640\weights\be

# Model E20 : Plastic-Focused Specialist Gate Refinement : E18-E as base and refining

## E20-A — Plastic-Focused Mixed Rigid / Mixed Soft Specialist Gate Search

In [ ]:
# E20-A — PLASTIC-FOCUSED MIXED RIGID / MIXED SOFT SPECIALIST GATE SEARCH
#
# PURPOSE
# --------------------------------------------------------------------------------------------------
# Refine E18-E using VALIDATION DATA ONLY.
#
# Objective:
#   Improve mean AP50 across the SIX plastic classes:
#
#       ECAL
#       HDPE
#       Mixed Rigid Plastic
#       Mixed Soft Plastic
#       PET
#       PET Oil
#
# IMPORTANT:
#   The model remains a FULL 7-CLASS task.
#   non_plastic remains part of:
#       - training history
#       - inference
#       - routing
#       - COCO evaluation
#
#   It is excluded ONLY from the six-plastic comparison metric.
#
# NO:
#   - retraining
#   - YOLO inference
#   - MobileNet inference
#   - ConvNeXt inference
#   - TEST-set use
#
# Reuses:
#   E18-D cached VALIDATION predictions.
#
#
# FROZEN COMPONENTS
# --------------------------------------------------------------------------------------------------
# Detector:
#   E12 YOLO11m @640
#
# MobileNet:
#   E3Y-B MobileNetV3-Large
#
# MobileNet routing:
#   E16-A class-specific gates
#
# ConvNeXt:
#   E18-B ConvNeXt-Tiny
#
# Alpha:
#   0.70
#
# Frozen ConvNeXt gates:
#   non_plastic = 0.90
#   PET Oil     = 0.94
#
#
# SEARCHED
# --------------------------------------------------------------------------------------------------
# Mixed Rigid ConvNeXt gate
# Mixed Soft  ConvNeXt gate
#
#
# SELECTION ORDER
# --------------------------------------------------------------------------------------------------
# 1. Highest six-plastic mean AP50
# 2. Highest mean AP50 of Mixed Rigid + Mixed Soft
# 3. Highest overall 7-class AP50-95
#
#
# SAFEGUARDS REPORTED
# --------------------------------------------------------------------------------------------------
# Overall 7-class AP50-95
# Overall 7-class AP50
# Overall AP75
# AR100
# non_plastic AP50
#
# ==================================================================================================

import json
from pathlib import Path
from collections import Counter

import numpy as np

from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval


# ==================================================================================================
# 1. PATHS
# ==================================================================================================

BASE = Path(
    r"C:\Users\varda\Documents\_My Computer\COMMON_space\Learning"
    r"\Upgrad_EPGP_MS\upgrad_IIITB_MS_course\MS_LJMU_Material"
)

THESIS_CODE = (
    BASE
    / "Thesis_Code"
)


E18D_DIR = (
    THESIS_CODE
    / "runs"
    / "sortwaste"
    / "E18D_class_selective_convnext"
)


CACHE_PATH = (
    E18D_DIR
    / "E18D_cached_predictions.json"
)


GT_PATH = (
    E18D_DIR
    / "E18D_val_gt_7class.json"
)


OUTPUT_DIR = (
    THESIS_CODE
    / "runs"
    / "sortwaste"
    / "E20A_plastic_focused_gate_search"
)


OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)


RESULTS_PATH = (
    OUTPUT_DIR
    / "E20A_grid_results.json"
)


BEST_PATH = (
    OUTPUT_DIR
    / "E20A_best_configuration.json"
)


SUMMARY_PATH = (
    OUTPUT_DIR
    / "E20A_summary.txt"
)


# ==================================================================================================
# 2. VALIDATE REQUIRED FILES
# ==================================================================================================

for path, label in [

    (
        CACHE_PATH,
        "E18-D cached predictions"
    ),

    (
        GT_PATH,
        "E18-D 7-class validation GT"
    ),

]:

    if not path.exists():

        raise FileNotFoundError(
            f"{label} not found:\n{path}"
        )


# ==================================================================================================
# 3. CLASS DEFINITIONS
# ==================================================================================================

CLASS_NAMES = [

    "ecal",

    "hdpe",

    "mixed_plastic_rigid",

    "mixed_plastic_soft",

    "non_plastic",

    "pet",

    "pet_oil",
]


ECAL = 0

HDPE = 1

MIXED_RIGID = 2

MIXED_SOFT = 3

NON_PLASTIC = 4

PET = 5

PET_OIL = 6


PLASTIC_CLASS_INDICES = [

    ECAL,

    HDPE,

    MIXED_RIGID,

    MIXED_SOFT,

    PET,

    PET_OIL,
]


# ==================================================================================================
# 4. E16-A MOBILENET GATES
#
# Required only to reconstruct the original source correctly
# for diagnostics.
#
# e16_final_idx is already cached and remains authoritative.
# ==================================================================================================

E16A_GATES = {

    0: 0.99,  # ECAL

    1: 0.98,  # HDPE

    2: 0.98,  # Mixed Rigid

    3: 0.94,  # Mixed Soft

    4: 0.94,  # non_plastic

    5: 0.99,  # PET

    6: 0.99,  # PET Oil
}


# ==================================================================================================
# 5. FIXED E18-E / E20 SETTINGS
# ==================================================================================================

ALPHA = 0.70


# Frozen from E18-E
NONPLASTIC_GATE = 0.90

PET_OIL_GATE = 0.94


# ==================================================================================================
# 6. SEARCH GRID
#
# None = ConvNeXt intervention disabled for that proposed class.
# ==================================================================================================

MIXED_RIGID_GATES = [

    None,

    0.90,

    0.92,

    0.94,

    0.96,

    0.97,

    0.98,

    0.985,

    0.99,

    0.995,
]


MIXED_SOFT_GATES = [

    None,

    0.85,

    0.88,

    0.90,

    0.92,

    0.94,

    0.96,

    0.98,

    0.99,
]


# ==================================================================================================
# 7. LOAD CACHE
# ==================================================================================================

print("=" * 110)

print(
    "E20-A — PLASTIC-FOCUSED MIXED RIGID / MIXED SOFT "
    "CONVNEXT GATE SEARCH"
)

print("=" * 110)


with open(
    CACHE_PATH,
    "r",
    encoding="utf-8"
) as f:

    cache = json.load(f)


print()

print(
    f"Cache entries      : "
    f"{len(cache):,}"
)

print(
    f"Cache              : "
    f"{CACHE_PATH}"
)

print(
    f"Validation GT      : "
    f"{GT_PATH}"
)

print(
    f"Output             : "
    f"{OUTPUT_DIR}"
)


print()

print(
    f"Alpha              : "
    f"{ALPHA:.2f}"
)

print(
    f"Non-plastic gate   : "
    f"{NONPLASTIC_GATE:.2f} [FROZEN]"
)

print(
    f"PET Oil gate       : "
    f"{PET_OIL_GATE:.2f} [FROZEN]"
)


print()

print(
    f"MR gate candidates : "
    f"{len(MIXED_RIGID_GATES)}"
)

print(
    f"MS gate candidates : "
    f"{len(MIXED_SOFT_GATES)}"
)

print(
    f"Grid combinations  : "
    f"{len(MIXED_RIGID_GATES) * len(MIXED_SOFT_GATES)}"
)


# ==================================================================================================
# 8. VALIDATE CACHE STRUCTURE
# ==================================================================================================

if len(cache) == 0:

    raise RuntimeError(
        "E18-D cache is empty."
    )


required_keys = {

    "image_id",

    "bbox",

    "yolo_conf",

    "yolo_class",

    "mn_probs",

    "e16_final_idx",

    "conv",
}


missing = (
    required_keys
    - set(
        cache[0].keys()
    )
)


if missing:

    raise RuntimeError(

        "E18-D cache is missing keys:\n"
        f"{sorted(missing)}"
    )


# ==================================================================================================
# 9. CACHE DIAGNOSTICS
# ==================================================================================================

num_conv_entries = sum(

    1

    for item
    in cache

    if item[
        "conv"
    ] is not None
)


print()

print(
    f"Entries with ConvNeXt probabilities : "
    f"{num_conv_entries:,}"
)

print(
    f"Entries without ConvNeXt            : "
    f"{len(cache) - num_conv_entries:,}"
)


# ==================================================================================================
# 10. GATE LABEL
# ==================================================================================================

def gate_label(
    gate
):

    if gate is None:

        return "OFF"

    return f"{gate:.3f}"


# ==================================================================================================
# 11. BUILD PREDICTIONS
# ==================================================================================================

def build_predictions(
    mixed_rigid_gate,
    mixed_soft_gate,
):

    """
    Rebuild one E20 candidate from the frozen E18-D validation cache.

    Base:
        E16-A final class.

    ConvNeXt may override only if:
        1. ConvNeXt exists for that cached crop
        2. ConvNeXt proposed class has routing enabled
        3. ConvNeXt confidence >= class-specific gate

    Frozen:
        non_plastic gate = 0.90
        PET Oil gate     = 0.94

    Tuned:
        Mixed Rigid
        Mixed Soft
    """

    predictions = []


    source_counts = Counter()

    specialist_proposed = Counter()

    specialist_accepted = Counter()

    specialist_rejected = Counter()


    for item in cache:

        # ------------------------------------------------------------------------------------------
        # Cached detector information
        # ------------------------------------------------------------------------------------------

        image_id = int(
            item[
                "image_id"
            ]
        )


        yolo_conf = float(
            item[
                "yolo_conf"
            ]
        )


        yolo_idx = int(
            item[
                "yolo_class"
            ]
        )


        e16_final_idx = int(
            item[
                "e16_final_idx"
            ]
        )


        mn_probs = np.asarray(

            item[
                "mn_probs"
            ],

            dtype=np.float32
        )


        # ------------------------------------------------------------------------------------------
        # Reconstruct true E16 source for diagnostics
        # ------------------------------------------------------------------------------------------

        mn_idx = int(
            np.argmax(
                mn_probs
            )
        )


        mn_top_prob = float(
            mn_probs[
                mn_idx
            ]
        )


        if (
            mn_top_prob
            >= E16A_GATES[
                mn_idx
            ]
        ):

            base_source = (
                "mobilenet"
            )

        else:

            base_source = (
                "yolo"
            )


        # ------------------------------------------------------------------------------------------
        # Base E16 prediction
        # ------------------------------------------------------------------------------------------

        final_idx = (
            e16_final_idx
        )


        final_source = (
            base_source
        )


        # MobileNet probability corresponding to ACTUAL E16 final class
        classifier_prob = float(
            mn_probs[
                final_idx
            ]
        )


        # ------------------------------------------------------------------------------------------
        # ConvNeXt specialist proposal
        # ------------------------------------------------------------------------------------------

        conv_info = (
            item[
                "conv"
            ]
        )


        if conv_info is not None:

            conv_global_idx = int(
                conv_info[
                    "conv_global_idx"
                ]
            )


            conv_top_prob = float(
                conv_info[
                    "conv_top_prob"
                ]
            )


            proposed_class_name = (
                CLASS_NAMES[
                    conv_global_idx
                ]
            )


            specialist_proposed[
                proposed_class_name
            ] += 1


            # --------------------------------------------------------------------------------------
            # Determine required gate according to specialist proposed class
            # --------------------------------------------------------------------------------------

            required_gate = None


            if (
                conv_global_idx
                == MIXED_RIGID
            ):

                required_gate = (
                    mixed_rigid_gate
                )


            elif (
                conv_global_idx
                == MIXED_SOFT
            ):

                required_gate = (
                    mixed_soft_gate
                )


            elif (
                conv_global_idx
                == NON_PLASTIC
            ):

                required_gate = (
                    NONPLASTIC_GATE
                )


            elif (
                conv_global_idx
                == PET_OIL
            ):

                required_gate = (
                    PET_OIL_GATE
                )


            # --------------------------------------------------------------------------------------
            # None means routing disabled for that class
            # --------------------------------------------------------------------------------------

            if required_gate is not None:

                if (
                    conv_top_prob
                    >= required_gate
                ):

                    final_idx = (
                        conv_global_idx
                    )


                    final_source = (
                        "convnext"
                    )


                    classifier_prob = (
                        conv_top_prob
                    )


                    specialist_accepted[
                        proposed_class_name
                    ] += 1


                else:

                    specialist_rejected[
                        proposed_class_name
                    ] += 1


        # ------------------------------------------------------------------------------------------
        # Final confidence fusion
        # ------------------------------------------------------------------------------------------

        classifier_prob = max(
            float(
                classifier_prob
            ),
            1e-12
        )


        yolo_conf_safe = max(
            yolo_conf,
            1e-12
        )


        score = (

            yolo_conf_safe
            ** ALPHA

        ) * (

            classifier_prob
            ** (
                1.0
                - ALPHA
            )
        )


        # ------------------------------------------------------------------------------------------
        # BBox xyxy -> COCO xywh
        # ------------------------------------------------------------------------------------------

        x1, y1, x2, y2 = [

            float(v)

            for v
            in item[
                "bbox"
            ]
        ]


        width = (
            x2 - x1
        )


        height = (
            y2 - y1
        )


        if (
            width <= 0
            or height <= 0
        ):

            continue


        predictions.append(
            {

                "image_id":
                    image_id,

                "category_id":
                    int(
                        final_idx
                        + 1
                    ),

                "bbox": [

                    x1,

                    y1,

                    width,

                    height,
                ],

                "score":
                    float(
                        score
                    ),
            }
        )


        source_counts[
            final_source
        ] += 1


    return {

        "predictions":
            predictions,

        "source_counts":
            dict(
                source_counts
            ),

        "specialist_proposed":
            dict(
                specialist_proposed
            ),

        "specialist_accepted":
            dict(
                specialist_accepted
            ),

        "specialist_rejected":
            dict(
                specialist_rejected
            ),
    }


# ==================================================================================================
# 12. LOAD COCO GT ONCE
# ==================================================================================================

coco_gt = COCO(
    str(
        GT_PATH
    )
)


# ==================================================================================================
# 13. SAFE MEAN HELPER
# ==================================================================================================

def valid_mean(
    values
):

    values = np.asarray(
        values
    )


    valid = values[
        values > -1
    ]


    if valid.size == 0:

        return float(
            "nan"
        )


    return float(
        np.mean(
            valid
        )
    )


# ==================================================================================================
# 14. COCO EVALUATION — QUIET GRID VERSION
#
# IMPORTANT:
#
# We deliberately DO NOT use evaluator.stats here.
#
# evaluator.stats is normally populated by evaluator.summarize().
# Running summarize() for 90 configurations would create enormous output.
#
# Instead we calculate all required metrics directly from:
#
#   evaluator.eval["precision"]
#   evaluator.eval["recall"]
#
# These are the same accumulated COCO evaluation tensors.
# ==================================================================================================

def evaluate_predictions(
    predictions
):

    if len(
        predictions
    ) == 0:

        raise RuntimeError(
            "No predictions supplied."
        )


    coco_dt = coco_gt.loadRes(
        predictions
    )


    evaluator = COCOeval(

        coco_gt,

        coco_dt,

        "bbox"
    )


    evaluator.params.maxDets = [

        1,

        10,

        100
    ]


    evaluator.evaluate()

    evaluator.accumulate()


    # ----------------------------------------------------------------------------------------------
    # Precision:
    #
    # [IoU, Recall, Class, Area, MaxDet]
    # ----------------------------------------------------------------------------------------------

    precision = (
        evaluator.eval[
            "precision"
        ]
    )


    # ----------------------------------------------------------------------------------------------
    # Recall:
    #
    # [IoU, Class, Area, MaxDet]
    # ----------------------------------------------------------------------------------------------

    recall = (
        evaluator.eval[
            "recall"
        ]
    )


    iou_thresholds = (
        evaluator.params.iouThrs
    )


    ap50_index = int(

        np.where(

            np.isclose(
                iou_thresholds,
                0.50
            )

        )[0][0]
    )


    ap75_index = int(

        np.where(

            np.isclose(
                iou_thresholds,
                0.75
            )

        )[0][0]
    )


    # ==============================================================================================
    # OVERALL COCO METRICS
    #
    # area index 0 = all
    # maxdet index -1 = 100
    # ==============================================================================================

    overall_ap5095 = valid_mean(

        precision[
            :,
            :,
            :,
            0,
            -1
        ]
    )


    overall_ap50 = valid_mean(

        precision[
            ap50_index,
            :,
            :,
            0,
            -1
        ]
    )


    overall_ap75 = valid_mean(

        precision[
            ap75_index,
            :,
            :,
            0,
            -1
        ]
    )


    overall_ar100 = valid_mean(

        recall[
            :,
            :,
            0,
            -1
        ]
    )


    overall = {

        "AP50_95":
            overall_ap5095,

        "AP50":
            overall_ap50,

        "AP75":
            overall_ap75,

        "AR100":
            overall_ar100,
    }


    # ==============================================================================================
    # CLASS-WISE METRICS
    # ==============================================================================================

    class_metrics = {}


    for class_idx, class_name in enumerate(
        CLASS_NAMES
    ):

        class_ap5095 = valid_mean(

            precision[
                :,
                :,
                class_idx,
                0,
                -1
            ]
        )


        class_ap50 = valid_mean(

            precision[
                ap50_index,
                :,
                class_idx,
                0,
                -1
            ]
        )


        class_metrics[
            class_name
        ] = {

            "AP50":
                class_ap50,

            "AP50_95":
                class_ap5095,
        }


    # ==============================================================================================
    # PRIMARY E20-A METRIC
    #
    # Mean AP50 across six plastic classes ONLY.
    #
    # non_plastic remains in the model and evaluation,
    # but is excluded from this specific benchmark average.
    # ==============================================================================================

    plastic_ap50_values = [

        class_metrics[
            CLASS_NAMES[
                idx
            ]
        ][
            "AP50"
        ]

        for idx
        in PLASTIC_CLASS_INDICES
    ]


    plastic_mean_ap50 = float(
        np.mean(
            plastic_ap50_values
        )
    )


    # ==============================================================================================
    # MR/MS TARGET METRIC
    # ==============================================================================================

    mr_ms_mean_ap50 = float(

        np.mean(
            [

                class_metrics[
                    "mixed_plastic_rigid"
                ][
                    "AP50"
                ],

                class_metrics[
                    "mixed_plastic_soft"
                ][
                    "AP50"
                ],
            ]
        )
    )


    return {

        "overall":
            overall,

        "class_metrics":
            class_metrics,

        "plastic_mean_AP50":
            plastic_mean_ap50,

        "mr_ms_mean_AP50":
            mr_ms_mean_ap50,

        "nonplastic_AP50":
            class_metrics[
                "non_plastic"
            ][
                "AP50"
            ],
    }


# ==================================================================================================
# 15. RECONSTRUCT EXACT E18-E BASELINE
#
# E18-E:
#
# MR          = OFF
# MS          = 0.90
# nonplastic  = 0.90
# PET Oil     = 0.94
# alpha       = 0.70
#
# IMPORTANT:
#
# This is NOT E20-A.
#
# This is simply the E18-E CONTROL rebuilt from the same cache
# so that E20-A and E18-E are evaluated through identical code.
# ==================================================================================================

print()

print("=" * 110)

print(
    "RECONSTRUCTING E18-E VALIDATION BASELINE"
)

print("=" * 110)


baseline_build = build_predictions(

    mixed_rigid_gate=None,

    mixed_soft_gate=0.90,
)


baseline_eval = evaluate_predictions(

    baseline_build[
        "predictions"
    ]
)


print()

print(
    f"E18-E reconstructed predictions : "
    f"{len(baseline_build['predictions']):,}"
)


print()

print(
    f"Six-plastic mean AP50           : "
    f"{baseline_eval['plastic_mean_AP50'] * 100:.4f}%"
)

print(
    f"Mixed Rigid AP50                : "
    f"{baseline_eval['class_metrics']['mixed_plastic_rigid']['AP50'] * 100:.4f}%"
)

print(
    f"Mixed Soft AP50                 : "
    f"{baseline_eval['class_metrics']['mixed_plastic_soft']['AP50'] * 100:.4f}%"
)

print(
    f"non_plastic AP50                : "
    f"{baseline_eval['nonplastic_AP50'] * 100:.4f}%"
)


print()

print(
    f"Overall 7-class AP50-95         : "
    f"{baseline_eval['overall']['AP50_95'] * 100:.4f}%"
)

print(
    f"Overall 7-class AP50            : "
    f"{baseline_eval['overall']['AP50'] * 100:.4f}%"
)

print(
    f"Overall AP75                    : "
    f"{baseline_eval['overall']['AP75'] * 100:.4f}%"
)

print(
    f"AR100                           : "
    f"{baseline_eval['overall']['AR100'] * 100:.4f}%"
)


print()

print(
    "E18-E reconstructed class-wise AP50:"
)


for class_name in (
    CLASS_NAMES
):

    print(

        f"{class_name:25s}: "

        f"{baseline_eval['class_metrics'][class_name]['AP50'] * 100:.2f}%"
    )


# ==================================================================================================
# 16. GRID SEARCH
# ==================================================================================================

all_results = []


total_combinations = (

    len(
        MIXED_RIGID_GATES
    )

    *

    len(
        MIXED_SOFT_GATES
    )
)


combination_number = 0


print()

print("=" * 110)

print(
    "RUNNING E20-A GRID SEARCH — VALIDATION ONLY"
)

print("=" * 110)


for mr_gate in (
    MIXED_RIGID_GATES
):

    for ms_gate in (
        MIXED_SOFT_GATES
    ):

        combination_number += 1


        build = build_predictions(

            mixed_rigid_gate=mr_gate,

            mixed_soft_gate=ms_gate,
        )


        evaluation = evaluate_predictions(

            build[
                "predictions"
            ]
        )


        result = {

            "mixed_rigid_gate":
                mr_gate,

            "mixed_soft_gate":
                ms_gate,

            "nonplastic_gate":
                NONPLASTIC_GATE,

            "pet_oil_gate":
                PET_OIL_GATE,

            "alpha":
                ALPHA,

            "plastic_mean_AP50":
                evaluation[
                    "plastic_mean_AP50"
                ],

            "mr_ms_mean_AP50":
                evaluation[
                    "mr_ms_mean_AP50"
                ],

            "nonplastic_AP50":
                evaluation[
                    "nonplastic_AP50"
                ],

            "overall":
                evaluation[
                    "overall"
                ],

            "class_metrics":
                evaluation[
                    "class_metrics"
                ],

            "source_counts":
                build[
                    "source_counts"
                ],

            "specialist_proposed":
                build[
                    "specialist_proposed"
                ],

            "specialist_accepted":
                build[
                    "specialist_accepted"
                ],

            "specialist_rejected":
                build[
                    "specialist_rejected"
                ],
        }


        all_results.append(
            result
        )


        print(

            f"[{combination_number:03d}/{total_combinations}] "

            f"MR={gate_label(mr_gate):>5s}  "

            f"MS={gate_label(ms_gate):>5s}  |  "

            f"Plastic AP50="
            f"{evaluation['plastic_mean_AP50'] * 100:7.3f}%  "

            f"MR="
            f"{evaluation['class_metrics']['mixed_plastic_rigid']['AP50'] * 100:6.2f}%  "

            f"MS="
            f"{evaluation['class_metrics']['mixed_plastic_soft']['AP50'] * 100:6.2f}%  "

            f"NP="
            f"{evaluation['nonplastic_AP50'] * 100:6.2f}%  "

            f"AP="
            f"{evaluation['overall']['AP50_95'] * 100:6.3f}%"
        )


# ==================================================================================================
# 17. RANK
#
# Primary:
#   Six-plastic mean AP50
#
# Secondary:
#   MR/MS mean AP50
#
# Tie-break:
#   Overall AP50-95
# ==================================================================================================

ranked_results = sorted(

    all_results,

    key=lambda r: (

        r[
            "plastic_mean_AP50"
        ],

        r[
            "mr_ms_mean_AP50"
        ],

        r[
            "overall"
        ][
            "AP50_95"
        ],

    ),

    reverse=True,
)


best = ranked_results[
    0
]


# ==================================================================================================
# 18. CALCULATE DELTAS VS E18-E
# ==================================================================================================

best_delta = {

    "plastic_mean_AP50_pp":

        (
            best[
                "plastic_mean_AP50"
            ]

            -

            baseline_eval[
                "plastic_mean_AP50"
            ]
        )

        * 100.0,


    "mixed_rigid_AP50_pp":

        (
            best[
                "class_metrics"
            ][
                "mixed_plastic_rigid"
            ][
                "AP50"
            ]

            -

            baseline_eval[
                "class_metrics"
            ][
                "mixed_plastic_rigid"
            ][
                "AP50"
            ]
        )

        * 100.0,


    "mixed_soft_AP50_pp":

        (
            best[
                "class_metrics"
            ][
                "mixed_plastic_soft"
            ][
                "AP50"
            ]

            -

            baseline_eval[
                "class_metrics"
            ][
                "mixed_plastic_soft"
            ][
                "AP50"
            ]
        )

        * 100.0,


    "nonplastic_AP50_pp":

        (
            best[
                "nonplastic_AP50"
            ]

            -

            baseline_eval[
                "nonplastic_AP50"
            ]
        )

        * 100.0,


    "overall_AP50_95_pp":

        (
            best[
                "overall"
            ][
                "AP50_95"
            ]

            -

            baseline_eval[
                "overall"
            ][
                "AP50_95"
            ]
        )

        * 100.0,


    "overall_AP50_pp":

        (
            best[
                "overall"
            ][
                "AP50"
            ]

            -

            baseline_eval[
                "overall"
            ][
                "AP50"
            ]
        )

        * 100.0,
}


best[
    "delta_vs_E18E"
] = best_delta


# ==================================================================================================
# 19. PRINT TOP 15
# ==================================================================================================

print()

print()

print("=" * 110)

print(
    "TOP 15 E20-A VALIDATION CONFIGURATIONS"
)

print("=" * 110)


print(
    f"\n"
    f"{'Rank':>4s} "
    f"{'MR Gate':>8s} "
    f"{'MS Gate':>8s} "
    f"{'Plastic AP50':>14s} "
    f"{'MR AP50':>10s} "
    f"{'MS AP50':>10s} "
    f"{'NP AP50':>10s} "
    f"{'AP50-95':>10s}"
)


print(
    "-" * 100
)


for rank, result in enumerate(
    ranked_results[:15],
    start=1
):

    mr_ap50 = (

        result[
            "class_metrics"
        ][
            "mixed_plastic_rigid"
        ][
            "AP50"
        ]

        * 100
    )


    ms_ap50 = (

        result[
            "class_metrics"
        ][
            "mixed_plastic_soft"
        ][
            "AP50"
        ]

        * 100
    )


    np_ap50 = (

        result[
            "nonplastic_AP50"
        ]

        * 100
    )


    print(

        f"{rank:4d} "

        f"{gate_label(result['mixed_rigid_gate']):>8s} "

        f"{gate_label(result['mixed_soft_gate']):>8s} "

        f"{result['plastic_mean_AP50'] * 100:13.4f}% "

        f"{mr_ap50:9.2f}% "

        f"{ms_ap50:9.2f}% "

        f"{np_ap50:9.2f}% "

        f"{result['overall']['AP50_95'] * 100:9.3f}%"
    )


# ==================================================================================================
# 20. BEST E20-A
# ==================================================================================================

print()

print("=" * 110)

print(
    "BEST E20-A CONFIGURATION — VALIDATION"
)

print("=" * 110)


print()

print(
    f"Mixed Rigid gate       : "
    f"{gate_label(best['mixed_rigid_gate'])}"
)

print(
    f"Mixed Soft gate        : "
    f"{gate_label(best['mixed_soft_gate'])}"
)

print(
    f"Non-plastic gate       : "
    f"{NONPLASTIC_GATE:.2f} [FROZEN]"
)

print(
    f"PET Oil gate           : "
    f"{PET_OIL_GATE:.2f} [FROZEN]"
)

print(
    f"Alpha                  : "
    f"{ALPHA:.2f}"
)


print()

print(
    f"Six-plastic mean AP50  : "
    f"{best['plastic_mean_AP50'] * 100:.4f}%"
)

print(
    f"MR + MS mean AP50      : "
    f"{best['mr_ms_mean_AP50'] * 100:.4f}%"
)


print()

print(
    f"Overall AP50-95        : "
    f"{best['overall']['AP50_95'] * 100:.4f}%"
)

print(
    f"Overall AP50           : "
    f"{best['overall']['AP50'] * 100:.4f}%"
)

print(
    f"Overall AP75           : "
    f"{best['overall']['AP75'] * 100:.4f}%"
)

print(
    f"AR100                  : "
    f"{best['overall']['AR100'] * 100:.4f}%"
)


print()

print(
    f"Non-plastic AP50       : "
    f"{best['nonplastic_AP50'] * 100:.4f}%"
)


# ==================================================================================================
# 21. BEST CLASS-WISE RESULTS
# ==================================================================================================

print()

print("=" * 110)

print(
    "BEST E20-A — CLASS-WISE VALIDATION RESULTS"
)

print("=" * 110)


print(
    f"\n"
    f"{'Class':28s}"
    f"{'AP50':>15s}"
    f"{'AP50-95':>15s}"
)


print(
    "-" * 58
)


for class_name in (
    CLASS_NAMES
):

    class_result = (

        best[
            "class_metrics"
        ][
            class_name
        ]
    )


    print(

        f"{class_name:28s}"

        f"{class_result['AP50'] * 100:14.2f}%"

        f"{class_result['AP50_95'] * 100:14.2f}%"
    )


# ==================================================================================================
# 22. DELTA VS E18-E
# ==================================================================================================

print()

print("=" * 110)

print(
    "BEST E20-A — DELTA VS RECONSTRUCTED E18-E"
)

print("=" * 110)


for metric, value in (
    best_delta.items()
):

    print(

        f"{metric:32s}: "
        f"{value:+.4f} pp"
    )


# ==================================================================================================
# 23. ROUTING
# ==================================================================================================

print()

print("=" * 110)

print(
    "BEST E20-A — ROUTING SUMMARY"
)

print("=" * 110)


total_sources = sum(

    best[
        "source_counts"
    ].values()
)


print()

print(
    "Final prediction source:"
)


for source in [

    "yolo",

    "mobilenet",

    "convnext",

]:

    count = int(

        best[
            "source_counts"
        ].get(
            source,
            0
        )
    )


    pct = (

        100.0
        * count
        / total_sources

        if total_sources > 0

        else 0.0
    )


    print(

        f"{source:15s}: "

        f"{count:,} "

        f"({pct:.2f}%)"
    )


print()

print(
    "ConvNeXt proposed / accepted:"
)


for class_name in [

    "mixed_plastic_rigid",

    "mixed_plastic_soft",

    "non_plastic",

    "pet_oil",

]:

    proposed = int(

        best[
            "specialist_proposed"
        ].get(
            class_name,
            0
        )
    )


    accepted = int(

        best[
            "specialist_accepted"
        ].get(
            class_name,
            0
        )
    )


    print(

        f"{class_name:25s}: "

        f"proposed={proposed:6,d}   "

        f"accepted={accepted:6,d}"
    )


# ==================================================================================================
# 24. SAVE GRID RESULTS
# ==================================================================================================

with open(
    RESULTS_PATH,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        ranked_results,
        f,
        indent=2
    )


# ==================================================================================================
# 25. SAVE BEST CONFIGURATION
# ==================================================================================================

baseline_output = {

    "experiment":
        "E18-E reconstructed baseline",

    "mixed_rigid_gate":
        None,

    "mixed_soft_gate":
        0.90,

    "nonplastic_gate":
        NONPLASTIC_GATE,

    "pet_oil_gate":
        PET_OIL_GATE,

    "alpha":
        ALPHA,

    "plastic_mean_AP50":
        baseline_eval[
            "plastic_mean_AP50"
        ],

    "mr_ms_mean_AP50":
        baseline_eval[
            "mr_ms_mean_AP50"
        ],

    "nonplastic_AP50":
        baseline_eval[
            "nonplastic_AP50"
        ],

    "overall":
        baseline_eval[
            "overall"
        ],

    "class_metrics":
        baseline_eval[
            "class_metrics"
        ],
}


best_output = {

    "experiment":
        "E20-A",

    "description":
        (
            "Plastic-focused Mixed Rigid / Mixed Soft "
            "ConvNeXt-Tiny specialist gate refinement"
        ),

    "selection_dataset":
        "validation",

    "test_set_used_for_selection":
        False,

    "selection_metric":
        "mean AP50 across six plastic classes",

    "selection_tiebreak_1":
        "mean AP50 of Mixed Rigid and Mixed Soft",

    "selection_tiebreak_2":
        "overall 7-class AP50-95",

    "six_plastic_classes": [

        "ecal",

        "hdpe",

        "mixed_plastic_rigid",

        "mixed_plastic_soft",

        "pet",

        "pet_oil",
    ],

    "seven_class_task": (
        CLASS_NAMES
    ),

    "baseline_E18E":
        baseline_output,

    "best_E20A":
        best,
}


with open(
    BEST_PATH,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        best_output,
        f,
        indent=2
    )


# ==================================================================================================
# 26. SAVE HUMAN-READABLE SUMMARY
# ==================================================================================================

summary_lines = [

    "=" * 110,

    "E20-A — Plastic-Focused Mixed Rigid / Mixed Soft ConvNeXt Gate Search",

    "=" * 110,

    "",

    "VALIDATION ONLY — TEST SET NOT USED FOR SELECTION",

    "",

    "Frozen:",

    "  Detector           : E12 YOLO11m @640",

    "  MobileNet          : E3Y-B",

    "  MobileNet routing  : E16-A",

    "  Specialist         : E18-B ConvNeXt-Tiny",

    f"  Alpha              : {ALPHA:.2f}",

    f"  Non-plastic gate   : {NONPLASTIC_GATE:.2f}",

    f"  PET Oil gate       : {PET_OIL_GATE:.2f}",

    "",

    "Primary objective:",

    "  Mean AP50 across ECAL, HDPE, Mixed Rigid, Mixed Soft, PET and PET Oil",

    "",

    "=" * 110,

    "RECONSTRUCTED E18-E BASELINE",

    "=" * 110,

    "",

    f"Plastic mean AP50 : {baseline_eval['plastic_mean_AP50'] * 100:.4f}%",

    f"MR AP50           : {baseline_eval['class_metrics']['mixed_plastic_rigid']['AP50'] * 100:.4f}%",

    f"MS AP50           : {baseline_eval['class_metrics']['mixed_plastic_soft']['AP50'] * 100:.4f}%",

    f"NP AP50           : {baseline_eval['nonplastic_AP50'] * 100:.4f}%",

    f"Overall AP50-95   : {baseline_eval['overall']['AP50_95'] * 100:.4f}%",

    f"Overall AP50      : {baseline_eval['overall']['AP50'] * 100:.4f}%",

    "",

    "=" * 110,

    "BEST E20-A",

    "=" * 110,

    "",

    f"Mixed Rigid gate  : {gate_label(best['mixed_rigid_gate'])}",

    f"Mixed Soft gate   : {gate_label(best['mixed_soft_gate'])}",

    f"Non-plastic gate  : {NONPLASTIC_GATE:.2f} [frozen]",

    f"PET Oil gate      : {PET_OIL_GATE:.2f} [frozen]",

    "",

    f"Plastic mean AP50 : {best['plastic_mean_AP50'] * 100:.4f}%",

    f"MR+MS mean AP50   : {best['mr_ms_mean_AP50'] * 100:.4f}%",

    f"Overall AP50-95   : {best['overall']['AP50_95'] * 100:.4f}%",

    f"Overall AP50      : {best['overall']['AP50'] * 100:.4f}%",

    f"Overall AP75      : {best['overall']['AP75'] * 100:.4f}%",

    f"AR100             : {best['overall']['AR100'] * 100:.4f}%",

    f"Non-plastic AP50  : {best['nonplastic_AP50'] * 100:.4f}%",

    "",

    "Delta vs E18-E:",

]


for metric, value in (
    best_delta.items()
):

    summary_lines.append(

        f"  {metric:30s}: "
        f"{value:+.4f} pp"
    )


summary_lines.extend(
    [

        "",

        "Best E20-A class-wise:",

    ]
)


for class_name in (
    CLASS_NAMES
):

    values = (

        best[
            "class_metrics"
        ][
            class_name
        ]
    )


    summary_lines.append(

        f"  {class_name:25s} "
        f"AP50={values['AP50'] * 100:.4f}%  "
        f"AP50-95={values['AP50_95'] * 100:.4f}%"
    )


summary_lines.extend(
    [

        "",

        "=" * 110,

    ]
)


with open(
    SUMMARY_PATH,
    "w",
    encoding="utf-8"
) as f:

    f.write(
        "\n".join(
            summary_lines
        )
    )


# ==================================================================================================
# 27. COMPLETE
# ==================================================================================================

print()

print("=" * 110)

print(
    "E20-A COMPLETE"
)

print("=" * 110)


print()

print(
    "Grid results:"
)

print(
    RESULTS_PATH
)


print()

print(
    "Best configuration:"
)

print(
    BEST_PATH
)


print()

print(
    "Summary:"
)

print(
    SUMMARY_PATH
)

E20-A — PLASTIC-FOCUSED MIXED RIGID / MIXED SOFT CONVNEXT GATE SEARCH

Cache entries      : 55,605
Cache              : C:\Users\varda\Documents\_My Computer\COMMON_space\Learning\Upgrad_EPGP_MS\upgrad_IIITB_MS_course\MS_LJMU_Material\Thesis_Code\runs\sortwaste\E18D_class_selective_convnext\E18D_cached_predictions.json
Validation GT      : C:\Users\varda\Documents\_My Computer\COMMON_space\Learning\Upgrad_EPGP_MS\upgrad_IIITB_MS_course\MS_LJMU_Material\Thesis_Code\runs\sortwaste\E18D_class_selective_convnext\E18D_val_gt_7class.json
Output             : C:\Users\varda\Documents\_My Computer\COMMON_space\Learning\Upgrad_EPGP_MS\upgrad_IIITB_MS_course\MS_LJMU_Material\Thesis_Code\runs\sortwaste\E20A_plastic_focused_gate_search

Alpha              : 0.70
Non-plastic gate   : 0.90 [FROZEN]
PET Oil gate       : 0.94 [FROZEN]

MR gate candidates : 10
MS gate candidates : 9
Grid combinations  : 90

Entries with ConvNeXt probabilities : 18,946
Entries without ConvNeXt            : 36,659
loadin

In [ ]:
## E20-A : Test evaluation
# ==========================================================================================
# FINAL HELD-OUT TEST — E20-A
#
# Plastic-focused Mixed Rigid / Mixed Soft ConvNeXt routing
#
# FROZEN from validation:
#   Mixed Rigid     : OFF
#   Mixed Soft      : 0.850
#   non_plastic     : 0.900
#   PET Oil         : 0.940
#   alpha           : 0.70
#
# Models:
#   YOLO11m         : E12
#   MobileNet       : E3Y-B
#   ConvNeXt-Tiny   : E18-B
#
# IMPORTANT:
#   - TEST SET ONLY
#   - NO GATE SEARCH
#   - NO THRESHOLD TUNING
#   - NO RETRAINING
# ==========================================================================================


import json
import time
from pathlib import Path
from collections import Counter

import numpy as np

from PIL import Image

import torch
import torch.nn as nn

from torchvision import transforms
from torchvision.models import (
    mobilenet_v3_large,
    convnext_tiny,
)

from ultralytics import YOLO

from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval


# ==========================================================================================
# 1. DEVICE
# ==========================================================================================

DEVICE = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

print("=" * 110)
print("FINAL HELD-OUT TEST — E20-A")
print("=" * 110)

print(f"\nPyTorch : {torch.__version__}")
print(f"Device  : {DEVICE}")

if torch.cuda.is_available():
    print(
        f"GPU     : "
        f"{torch.cuda.get_device_name(0)}"
    )


# ==========================================================================================
# 2. PATHS
# ==========================================================================================

BASE = Path(
    r"C:\Users\varda\Documents\_My Computer\COMMON_space\Learning"
    r"\Upgrad_EPGP_MS\upgrad_IIITB_MS_course\MS_LJMU_Material"
)

DATASET_ROOT = (
    BASE
    / "Topic Data"
    / "SortWaste"
    / "dataset"
    / "dataset"
)

THESIS_CODE = (
    BASE
    / "Thesis_Code"
)


# ------------------------------------------------------------------------------------------
# TEST DATA
# ------------------------------------------------------------------------------------------

TEST_ROOT = (
    DATASET_ROOT
    / "splited_all_dataset_coco"
    / "test"
)

TEST_IMAGES = (
    TEST_ROOT
    / "images"
)

TEST_COCO_JSON = (
    TEST_ROOT
    / "annotations"
    / "test_coco.json"
)


# ------------------------------------------------------------------------------------------
# E12 — YOLO11m
# ------------------------------------------------------------------------------------------

YOLO_PATH = (
    THESIS_CODE
    / "runs"
    / "sortwaste"
    / "E12_yolo11m_7class_aug_classbalance_640"
    / "weights"
    / "best.pt"
)


# ------------------------------------------------------------------------------------------
# E3Y-B — MobileNet
# ------------------------------------------------------------------------------------------

MOBILENET_PATH = (
    DATASET_ROOT
    / "yolo_mobilenet_crops_E3Y"
    / "mobilenet_results"
    / "E3Y_B_class_weighted"
    / "E3Y_B_MobileNetV3Large_best.pth"
)


# ------------------------------------------------------------------------------------------
# E18-B — ConvNeXt-Tiny
# ------------------------------------------------------------------------------------------

CONVNEXT_PATH = (
    THESIS_CODE
    / "runs"
    / "sortwaste"
    / "E18B_convnext_tiny_hardclass"
    / "E18B_ConvNeXtTiny_best.pth"
)


# ------------------------------------------------------------------------------------------
# E20-A TEST OUTPUT
# ------------------------------------------------------------------------------------------

OUTPUT_DIR = (
    THESIS_CODE
    / "runs"
    / "sortwaste"
    / "FINAL_TEST_E20A"
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)


GT_7CLASS_PATH = (
    OUTPUT_DIR
    / "E20A_test_gt_7class.json"
)

PREDICTIONS_PATH = (
    OUTPUT_DIR
    / "E20A_test_predictions.json"
)

RESULTS_PATH = (
    OUTPUT_DIR
    / "E20A_test_results.json"
)

SUMMARY_PATH = (
    OUTPUT_DIR
    / "E20A_test_summary.txt"
)


for path in [
    TEST_IMAGES,
    TEST_COCO_JSON,
    YOLO_PATH,
    MOBILENET_PATH,
    CONVNEXT_PATH,
]:

    assert path.exists(), (
        f"Missing required path:\n{path}"
    )


print(f"\nTest images : {TEST_IMAGES}")
print(f"Test COCO   : {TEST_COCO_JSON}")
print(f"YOLO        : {YOLO_PATH}")
print(f"MobileNet   : {MOBILENET_PATH}")
print(f"ConvNeXt    : {CONVNEXT_PATH}")
print(f"Output      : {OUTPUT_DIR}")


# ==========================================================================================
# 3. TAXONOMY
# ==========================================================================================

CLASS_NAMES = [
    "ecal",
    "hdpe",
    "mixed_plastic_rigid",
    "mixed_plastic_soft",
    "non_plastic",
    "pet",
    "pet_oil",
]

ECAL = 0
HDPE = 1
MIXED_RIGID = 2
MIXED_SOFT = 3
NON_PLASTIC = 4
PET = 5
PET_OIL = 6


# ==========================================================================================
# 4. ORIGINAL SORTWASTE COCO -> STANDARDIZED 7 CLASS
# ==========================================================================================

ORIGINAL_COCO_TO_NEW = {
    1: 6,  # PET
    2: 2,  # HDPE
    3: 4,  # Mixed Soft
    4: 1,  # ECAL
    5: 5,  # Metal -> non_plastic
    6: 5,  # Cardboard -> non_plastic
    7: 3,  # Mixed Rigid
    8: 7,  # PET Oil
}


# ==========================================================================================
# 5. CREATE FROZEN 7-CLASS TEST GROUND TRUTH
# ==========================================================================================

with open(
    TEST_COCO_JSON,
    "r",
    encoding="utf-8"
) as f:

    original_gt = json.load(f)


gt_7class = {

    "images":
        original_gt["images"],

    "annotations":
        [],

    "categories":
        [
            {
                "id": i + 1,
                "name": name,
            }

            for i, name
            in enumerate(CLASS_NAMES)
        ],
}


for ann in original_gt["annotations"]:

    new_ann = ann.copy()

    new_ann["category_id"] = (
        ORIGINAL_COCO_TO_NEW[
            int(ann["category_id"])
        ]
    )

    gt_7class[
        "annotations"
    ].append(
        new_ann
    )


with open(
    GT_7CLASS_PATH,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        gt_7class,
        f
    )


print(
    f"\nTest images : "
    f"{len(gt_7class['images']):,}"
)

print(
    f"Test GT     : "
    f"{len(gt_7class['annotations']):,}"
)


gt_class_counts = Counter()

for ann in gt_7class[
    "annotations"
]:

    gt_class_counts[
        int(ann["category_id"]) - 1
    ] += 1


print("\nTest GT distribution:")

for idx, class_name in enumerate(
    CLASS_NAMES
):

    print(
        f"{class_name:25s}: "
        f"{gt_class_counts[idx]:,}"
    )


# ==========================================================================================
# 6. CLASSIFIER TRANSFORM
# ==========================================================================================

IMAGE_SIZE = 224

classifier_transform = transforms.Compose(
    [
        transforms.Resize(
            (
                IMAGE_SIZE,
                IMAGE_SIZE,
            )
        ),

        transforms.ToTensor(),

        transforms.Normalize(
            mean=[
                0.485,
                0.456,
                0.406,
            ],

            std=[
                0.229,
                0.224,
                0.225,
            ],
        ),
    ]
)


# ==========================================================================================
# 7. FROZEN E16-A MOBILE NET GATES
# ==========================================================================================

MOBILENET_GATES = {
    ECAL: 0.99,
    HDPE: 0.98,
    MIXED_RIGID: 0.98,
    MIXED_SOFT: 0.94,
    NON_PLASTIC: 0.94,
    PET: 0.99,
    PET_OIL: 0.99,
}


# ==========================================================================================
# 8. FROZEN E20-A CONVNEXT CONFIGURATION
# ==========================================================================================

# Classes for which the ConvNeXt specialist was originally trained.
E16_HARD_CLASSES = {
    MIXED_RIGID,
    MIXED_SOFT,
    NON_PLASTIC,
    PET_OIL,
}


# E20-A validation result:
# Mixed Rigid must remain OFF.
ALLOWED_CONVNEXT_CLASSES = {
    MIXED_SOFT,
    NON_PLASTIC,
    PET_OIL,
}


CONVNEXT_GATES = {
    MIXED_SOFT: 0.850,
    NON_PLASTIC: 0.900,
    PET_OIL: 0.940,
}


# ConvNeXt local class order:
#
# 0 mixed_plastic_rigid
# 1 mixed_plastic_soft
# 2 non_plastic
# 3 pet_oil

CONV_LOCAL_TO_GLOBAL = {
    0: MIXED_RIGID,
    1: MIXED_SOFT,
    2: NON_PLASTIC,
    3: PET_OIL,
}


# ==========================================================================================
# 9. CONFIDENCE FUSION
# ==========================================================================================

ALPHA = 0.70


# ==========================================================================================
# 10. FROZEN YOLO SETTINGS
# ==========================================================================================

YOLO_CONF = 0.001
YOLO_NMS_IOU = 0.60
YOLO_MAX_DET = 100
YOLO_IMGSZ = 640


print("\nFrozen E20-A configuration:")
print(f"Mixed Rigid gate : OFF")
print(f"Mixed Soft gate  : {CONVNEXT_GATES[MIXED_SOFT]:.3f}")
print(f"Non-plastic gate : {CONVNEXT_GATES[NON_PLASTIC]:.3f}")
print(f"PET Oil gate     : {CONVNEXT_GATES[PET_OIL]:.3f}")
print(f"Alpha            : {ALPHA:.2f}")


# ==========================================================================================
# 11. LOAD MOBILENET
# ==========================================================================================

print("\nLoading E3Y-B MobileNet...")


mobilenet = mobilenet_v3_large(
    weights=None
)

mn_in_features = (
    mobilenet.classifier[3]
    .in_features
)

mobilenet.classifier[3] = nn.Linear(
    mn_in_features,
    7
)


mn_checkpoint = torch.load(
    MOBILENET_PATH,
    map_location=DEVICE,
    weights_only=False,
)


if (
    isinstance(mn_checkpoint, dict)
    and
    "model_state_dict"
    in mn_checkpoint
):

    mn_state = (
        mn_checkpoint[
            "model_state_dict"
        ]
    )

else:

    mn_state = (
        mn_checkpoint
    )


mobilenet.load_state_dict(
    mn_state
)

mobilenet = mobilenet.to(
    DEVICE
)

mobilenet.eval()

print("MobileNet loaded.")


# ==========================================================================================
# 12. LOAD E18-B CONVNEXT-TINY
# ==========================================================================================

print("\nLoading E18-B ConvNeXt-Tiny...")


convnext = convnext_tiny(
    weights=None
)

conv_in_features = (
    convnext.classifier[2]
    .in_features
)

convnext.classifier[2] = nn.Linear(
    conv_in_features,
    4
)


conv_checkpoint = torch.load(
    CONVNEXT_PATH,
    map_location=DEVICE,
    weights_only=False,
)


if (
    isinstance(conv_checkpoint, dict)
    and
    "model_state_dict"
    in conv_checkpoint
):

    conv_state = (
        conv_checkpoint[
            "model_state_dict"
        ]
    )

else:

    conv_state = (
        conv_checkpoint
    )


convnext.load_state_dict(
    conv_state
)

convnext = convnext.to(
    DEVICE
)

convnext.eval()

print("ConvNeXt-Tiny loaded.")


# ==========================================================================================
# 13. LOAD E12 YOLO11m
# ==========================================================================================

print("\nLoading E12 YOLO11m...")

yolo = YOLO(
    str(YOLO_PATH)
)

print("YOLO11m loaded.")


# ==========================================================================================
# 14. COUNTERS
# ==========================================================================================

predictions = []

source_counts = Counter()

conv_proposed_counts = Counter()
conv_accepted_counts = Counter()
conv_rejected_counts = Counter()

total_yolo_detections = 0
total_valid_crops = 0
invalid_crops = 0

conv_candidates = 0


# ==========================================================================================
# 15. TEST INFERENCE
# ==========================================================================================

image_records = sorted(
    gt_7class["images"],
    key=lambda x: int(
        x["id"]
    )
)


start_time = time.time()


for image_number, image_record in enumerate(
    image_records,
    start=1,
):

    image_id = int(
        image_record[
            "id"
        ]
    )

    filename = (
        image_record[
            "file_name"
        ]
    )

    image_path = (
        TEST_IMAGES
        / filename
    )


    if not image_path.exists():

        raise FileNotFoundError(
            f"Missing test image:\n"
            f"{image_path}"
        )


    pil_image = Image.open(
        image_path
    ).convert(
        "RGB"
    )

    image_width, image_height = (
        pil_image.size
    )


    # --------------------------------------------------------------------------------------
    # A. YOLO11m
    # --------------------------------------------------------------------------------------

    yolo_result = yolo.predict(
        source=str(
            image_path
        ),

        imgsz=YOLO_IMGSZ,

        conf=YOLO_CONF,

        iou=YOLO_NMS_IOU,

        max_det=YOLO_MAX_DET,

        verbose=False,

        device=(
            0
            if DEVICE.type == "cuda"
            else "cpu"
        ),
    )[0]


    boxes = (
        yolo_result.boxes
    )

    if boxes is None:
        continue


    xyxy_array = (
        boxes.xyxy
        .detach()
        .cpu()
        .numpy()
    )

    confidence_array = (
        boxes.conf
        .detach()
        .cpu()
        .numpy()
    )

    yolo_class_array = (
        boxes.cls
        .detach()
        .cpu()
        .numpy()
        .astype(int)
    )


    total_yolo_detections += (
        len(xyxy_array)
    )


    # --------------------------------------------------------------------------------------
    # Each detector prediction
    # --------------------------------------------------------------------------------------

    for (
        bbox_xyxy,
        yolo_conf,
        yolo_idx,
    ) in zip(
        xyxy_array,
        confidence_array,
        yolo_class_array,
    ):

        x1, y1, x2, y2 = [
            float(v)
            for v in bbox_xyxy
        ]


        crop_x1 = max(
            0,
            int(np.floor(x1))
        )

        crop_y1 = max(
            0,
            int(np.floor(y1))
        )

        crop_x2 = min(
            image_width,
            int(np.ceil(x2))
        )

        crop_y2 = min(
            image_height,
            int(np.ceil(y2))
        )


        if (
            crop_x2 <= crop_x1
            or
            crop_y2 <= crop_y1
        ):

            invalid_crops += 1
            continue


        crop = pil_image.crop(
            (
                crop_x1,
                crop_y1,
                crop_x2,
                crop_y2,
            )
        )


        input_tensor = (
            classifier_transform(
                crop
            )
            .unsqueeze(0)
            .to(DEVICE)
        )


        total_valid_crops += 1


        # ==================================================================================
        # B. MOBILENET
        # ==================================================================================

        with torch.no_grad():

            if DEVICE.type == "cuda":

                with torch.amp.autocast(
                    device_type="cuda",
                    dtype=torch.float16,
                ):

                    mn_logits = mobilenet(
                        input_tensor
                    )

            else:

                mn_logits = mobilenet(
                    input_tensor
                )


        mn_probs = torch.softmax(
            mn_logits.float(),
            dim=1
        )[0]


        mn_top_prob, mn_top_idx = (
            torch.max(
                mn_probs,
                dim=0
            )
        )


        mn_top_prob = float(
            mn_top_prob.item()
        )

        mn_top_idx = int(
            mn_top_idx.item()
        )


        # ==================================================================================
        # C. E16-A MOBILE NET GATING
        # ==================================================================================

        if (
            mn_top_prob
            >= MOBILENET_GATES[
                mn_top_idx
            ]
        ):

            e16_final_idx = (
                mn_top_idx
            )

            e16_source = (
                "mobilenet"
            )

        else:

            e16_final_idx = int(
                yolo_idx
            )

            e16_source = (
                "yolo"
            )


        final_idx = (
            e16_final_idx
        )

        final_source = (
            e16_source
        )


        classifier_prob = float(
            mn_probs[
                final_idx
            ].item()
        )


        # ==================================================================================
        # D. E18-B CONVNEXT-TINY SPECIALIST
        # ==================================================================================

        if (
            e16_final_idx
            in E16_HARD_CLASSES
        ):

            conv_candidates += 1


            with torch.no_grad():

                if DEVICE.type == "cuda":

                    with torch.amp.autocast(
                        device_type="cuda",
                        dtype=torch.float16,
                    ):

                        conv_logits = convnext(
                            input_tensor
                        )

                else:

                    conv_logits = convnext(
                        input_tensor
                    )


            conv_probs = torch.softmax(
                conv_logits.float(),
                dim=1
            )[0]


            conv_top_prob, conv_local_idx = (
                torch.max(
                    conv_probs,
                    dim=0
                )
            )


            conv_top_prob = float(
                conv_top_prob.item()
            )

            conv_local_idx = int(
                conv_local_idx.item()
            )


            conv_global_idx = (
                CONV_LOCAL_TO_GLOBAL[
                    conv_local_idx
                ]
            )


            conv_proposed_counts[
                conv_global_idx
            ] += 1


            # ------------------------------------------------------------------------------
            # E20-A selective routing
            #
            # Mixed Rigid is explicitly disabled.
            # ------------------------------------------------------------------------------

            if (
                conv_global_idx
                in ALLOWED_CONVNEXT_CLASSES
            ):

                gate = (
                    CONVNEXT_GATES[
                        conv_global_idx
                    ]
                )


                if (
                    conv_top_prob
                    >= gate
                ):

                    final_idx = (
                        conv_global_idx
                    )

                    final_source = (
                        "convnext"
                    )

                    classifier_prob = (
                        conv_top_prob
                    )


                    conv_accepted_counts[
                        conv_global_idx
                    ] += 1

                else:

                    conv_rejected_counts[
                        conv_global_idx
                    ] += 1


        source_counts[
            final_source
        ] += 1


        # ==================================================================================
        # E. FINAL CONFIDENCE
        #
        # Same fusion as E16/E18/E20 validation:
        #
        # score =
        # YOLO_conf ^ 0.70
        # *
        # classifier_prob ^ 0.30
        # ==================================================================================

        classifier_prob = max(
            classifier_prob,
            1e-12
        )


        final_score = (
            float(
                yolo_conf
            )
            ** ALPHA
        ) * (
            float(
                classifier_prob
            )
            ** (
                1.0 - ALPHA
            )
        )


        predictions.append(
            {
                "image_id":
                    image_id,

                "category_id":
                    int(
                        final_idx + 1
                    ),

                "bbox":
                    [
                        float(x1),
                        float(y1),
                        float(
                            x2 - x1
                        ),
                        float(
                            y2 - y1
                        ),
                    ],

                "score":
                    float(
                        final_score
                    ),
            }
        )


    if (
        image_number % 25 == 0
        or
        image_number == len(
            image_records
        )
    ):

        elapsed = (
            time.time()
            - start_time
        )

        print(
            f"[{image_number:4d}/"
            f"{len(image_records):4d}] "
            f"predictions={len(predictions):,} "
            f"time={elapsed/60:.2f} min"
        )


elapsed_seconds = (
    time.time()
    - start_time
)


print("\n" + "=" * 110)
print("E20-A TEST INFERENCE COMPLETE")
print("=" * 110)

print(
    f"\nYOLO detections : "
    f"{total_yolo_detections:,}"
)

print(
    f"Valid crops     : "
    f"{total_valid_crops:,}"
)

print(
    f"Invalid crops   : "
    f"{invalid_crops:,}"
)

print(
    f"Predictions     : "
    f"{len(predictions):,}"
)

print(
    f"Conv candidates : "
    f"{conv_candidates:,}"
)

print(
    f"Inference time  : "
    f"{elapsed_seconds/60:.2f} min"
)


# ==========================================================================================
# 16. SAVE PREDICTIONS
# ==========================================================================================

with open(
    PREDICTIONS_PATH,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        predictions,
        f
    )


# ==========================================================================================
# 17. COCO EVALUATION
# ==========================================================================================

print("\n" + "=" * 110)
print("E20-A — HELD-OUT TEST COCO EVALUATION")
print("=" * 110)


coco_gt = COCO(
    str(
        GT_7CLASS_PATH
    )
)

coco_dt = coco_gt.loadRes(
    predictions
)


coco_eval = COCOeval(
    coco_gt,
    coco_dt,
    "bbox"
)

coco_eval.params.maxDets = [
    1,
    10,
    100,
]

coco_eval.evaluate()
coco_eval.accumulate()
coco_eval.summarize()


overall_metrics = {
    "AP50-95":
        float(
            coco_eval.stats[0]
        ),

    "AP50":
        float(
            coco_eval.stats[1]
        ),

    "AP75":
        float(
            coco_eval.stats[2]
        ),

    "AR100":
        float(
            coco_eval.stats[8]
        ),
}


# ==========================================================================================
# 18. CLASS-WISE AP
# ==========================================================================================

precision = (
    coco_eval.eval[
        "precision"
    ]
)


per_class = {}


for class_idx, class_name in enumerate(
    CLASS_NAMES
):

    # COCO evaluator class index follows category list order.
    p_all = (
        precision[
            :,
            :,
            class_idx,
            0,
            -1,
        ]
    )

    valid_all = (
        p_all[
            p_all > -1
        ]
    )


    ap_50_95 = (
        float(
            valid_all.mean()
        )
        if valid_all.size
        else float("nan")
    )


    p50 = (
        precision[
            0,
            :,
            class_idx,
            0,
            -1,
        ]
    )

    valid50 = (
        p50[
            p50 > -1
        ]
    )


    ap50 = (
        float(
            valid50.mean()
        )
        if valid50.size
        else float("nan")
    )


    per_class[
        class_name
    ] = {
        "AP50":
            ap50,

        "AP50-95":
            ap_50_95,
    }


print("\n" + "=" * 110)
print("E20-A — CLASS-WISE HELD-OUT TEST RESULTS")
print("=" * 110)

print(
    f"\n{'Class':30s}"
    f"{'AP50':>15s}"
    f"{'AP50-95':>15s}"
)

print("-" * 60)


for class_name in CLASS_NAMES:

    values = (
        per_class[
            class_name
        ]
    )

    print(
        f"{class_name:30s}"
        f"{values['AP50']*100:14.2f}%"
        f"{values['AP50-95']*100:14.2f}%"
    )


# ==========================================================================================
# 19. SIX-PLASTIC MEANS
# ==========================================================================================

PLASTIC_CLASSES = [
    "ecal",
    "hdpe",
    "mixed_plastic_rigid",
    "mixed_plastic_soft",
    "pet",
    "pet_oil",
]


plastic_mean_ap50 = float(
    np.mean(
        [
            per_class[c][
                "AP50"
            ]
            for c
            in PLASTIC_CLASSES
        ]
    )
)


plastic_mean_ap = float(
    np.mean(
        [
            per_class[c][
                "AP50-95"
            ]
            for c
            in PLASTIC_CLASSES
        ]
    )
)


mr_ms_mean_ap50 = float(
    np.mean(
        [
            per_class[
                "mixed_plastic_rigid"
            ][
                "AP50"
            ],

            per_class[
                "mixed_plastic_soft"
            ][
                "AP50"
            ],
        ]
    )
)


print("\n" + "=" * 110)
print("E20-A — PLASTIC-FOCUSED TEST SUMMARY")
print("=" * 110)

print(
    f"\nSix-plastic mean AP50     : "
    f"{plastic_mean_ap50*100:.4f}%"
)

print(
    f"Six-plastic mean AP50-95  : "
    f"{plastic_mean_ap*100:.4f}%"
)

print(
    f"MR + MS mean AP50         : "
    f"{mr_ms_mean_ap50*100:.4f}%"
)

print(
    f"\nOverall AP50-95           : "
    f"{overall_metrics['AP50-95']*100:.4f}%"
)

print(
    f"Overall AP50              : "
    f"{overall_metrics['AP50']*100:.4f}%"
)

print(
    f"Overall AP75              : "
    f"{overall_metrics['AP75']*100:.4f}%"
)

print(
    f"AR100                     : "
    f"{overall_metrics['AR100']*100:.4f}%"
)


# ==========================================================================================
# 20. ROUTING SUMMARY
# ==========================================================================================

print("\n" + "=" * 110)
print("E20-A — TEST ROUTING SUMMARY")
print("=" * 110)


print("\nFinal prediction source:")

for source in [
    "yolo",
    "mobilenet",
    "convnext",
]:

    count = (
        source_counts[
            source
        ]
    )

    pct = (
        100.0
        * count
        / max(
            len(predictions),
            1
        )
    )

    print(
        f"{source:15s}: "
        f"{count:8,d} "
        f"({pct:6.2f}%)"
    )


print(
    "\nConvNeXt proposed / accepted:"
)


for idx in [
    MIXED_RIGID,
    MIXED_SOFT,
    NON_PLASTIC,
    PET_OIL,
]:

    print(
        f"{CLASS_NAMES[idx]:25s}: "
        f"proposed="
        f"{conv_proposed_counts[idx]:6,d}   "
        f"accepted="
        f"{conv_accepted_counts[idx]:6,d}"
    )


# ==========================================================================================
# 21. SAVE COMPLETE RESULT
# ==========================================================================================

result_payload = {

    "experiment":
        "E20-A",

    "split":
        "held-out TEST",

    "configuration": {

        "detector":
            "E12 YOLO11m 640",

        "mobilenet":
            "E3Y-B",

        "convnext":
            "E18-B ConvNeXt-Tiny",

        "mobilenet_gates":
            {
                CLASS_NAMES[k]:
                    float(v)

                for k, v
                in MOBILENET_GATES.items()
            },

        "convnext_gates":
            {
                "mixed_plastic_rigid":
                    "OFF",

                "mixed_plastic_soft":
                    0.850,

                "non_plastic":
                    0.900,

                "pet_oil":
                    0.940,
            },

        "alpha":
            ALPHA,
    },

    "overall":
        overall_metrics,

    "per_class":
        per_class,

    "plastic_mean_AP50":
        plastic_mean_ap50,

    "plastic_mean_AP50_95":
        plastic_mean_ap,

    "MR_MS_mean_AP50":
        mr_ms_mean_ap50,

    "source_counts":
        dict(
            source_counts
        ),

    "convnext_proposed":
        {
            CLASS_NAMES[k]:
                int(v)

            for k, v
            in conv_proposed_counts.items()
        },

    "convnext_accepted":
        {
            CLASS_NAMES[k]:
                int(v)

            for k, v
            in conv_accepted_counts.items()
        },

    "inference_minutes":
        elapsed_seconds / 60.0,
}


with open(
    RESULTS_PATH,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        result_payload,
        f,
        indent=2
    )


# ==========================================================================================
# 22. TEXT SUMMARY
# ==========================================================================================

with open(
    SUMMARY_PATH,
    "w",
    encoding="utf-8"
) as f:

    f.write(
        "E20-A — HELD-OUT TEST RESULTS\n"
    )

    f.write(
        "=" * 90
        + "\n\n"
    )

    f.write(
        "Frozen configuration\n"
    )

    f.write(
        "Mixed Rigid ConvNeXt : OFF\n"
    )

    f.write(
        "Mixed Soft gate      : 0.850\n"
    )

    f.write(
        "Non-plastic gate     : 0.900\n"
    )

    f.write(
        "PET Oil gate         : 0.940\n"
    )

    f.write(
        "Alpha                : 0.70\n\n"
    )


    for metric, value in (
        overall_metrics.items()
    ):

        f.write(
            f"{metric:15s}: "
            f"{value*100:.4f}%\n"
        )


    f.write(
        f"\nSix-plastic mean AP50    : "
        f"{plastic_mean_ap50*100:.4f}%\n"
    )

    f.write(
        f"Six-plastic mean AP50-95 : "
        f"{plastic_mean_ap*100:.4f}%\n"
    )

    f.write(
        f"MR + MS mean AP50        : "
        f"{mr_ms_mean_ap50*100:.4f}%\n"
    )


    f.write(
        "\nClass-wise results\n"
    )

    for class_name in (
        CLASS_NAMES
    ):

        values = (
            per_class[
                class_name
            ]
        )

        f.write(
            f"{class_name:25s} "
            f"AP50="
            f"{values['AP50']*100:.4f}%   "
            f"AP50-95="
            f"{values['AP50-95']*100:.4f}%\n"
        )


print("\n" + "=" * 110)
print("E20-A HELD-OUT TEST COMPLETE")
print("=" * 110)

print(
    f"\nPredictions : "
    f"{PREDICTIONS_PATH}"
)

print(
    f"Results     : "
    f"{RESULTS_PATH}"
)

print(
    f"Summary     : "
    f"{SUMMARY_PATH}"
)


FINAL HELD-OUT TEST — E20-A

PyTorch : 2.13.0+cu126
Device  : cuda
GPU     : NVIDIA GeForce RTX 3050 Ti Laptop GPU

Test images : C:\Users\varda\Documents\_My Computer\COMMON_space\Learning\Upgrad_EPGP_MS\upgrad_IIITB_MS_course\MS_LJMU_Material\Topic Data\SortWaste\dataset\dataset\splited_all_dataset_coco\test\images
Test COCO   : C:\Users\varda\Documents\_My Computer\COMMON_space\Learning\Upgrad_EPGP_MS\upgrad_IIITB_MS_course\MS_LJMU_Material\Topic Data\SortWaste\dataset\dataset\splited_all_dataset_coco\test\annotations\test_coco.json
YOLO        : C:\Users\varda\Documents\_My Computer\COMMON_space\Learning\Upgrad_EPGP_MS\upgrad_IIITB_MS_course\MS_LJMU_Material\Thesis_Code\runs\sortwaste\E12_yolo11m_7class_aug_classbalance_640\weights\best.pt
MobileNet   : C:\Users\varda\Documents\_My Computer\COMMON_space\Learning\Upgrad_EPGP_MS\upgrad_IIITB_MS_course\MS_LJMU_Material\Topic Data\SortWaste\dataset\dataset\yolo_mobilenet_crops_E3Y\mobilenet_results\E3Y_B_class_weighted\E3Y_B_MobileNetV

## E20-B — VALIDATION CONFUSION ANALYSIS FOR MIXED RIGID / MIXED SOFT


In [ ]:
# E20-B — VALIDATION CONFUSION ANALYSIS FOR MIXED RIGID / MIXED SOFT
#
# PURPOSE
# --------------------------------------------------------------------------------------------------
# Diagnose where Mixed Rigid and Mixed Soft errors come from under the frozen E20-A configuration.
#
# VALIDATION ONLY.
# NO TEST DATA.
# NO RETRAINING.
# NO YOLO/MobileNet/ConvNeXt inference.
#
# Reuses E18-D cached predictions.
#
# Frozen E20-A configuration:
#   Mixed Rigid specialist = OFF
#   Mixed Soft gate        = 0.85
#   non_plastic gate       = 0.90
#   PET Oil gate           = 0.94
#   alpha                  = 0.70
#
# Matching:
#   greedy one-to-one matching
#   IoU >= 0.50
#
# Outputs:
#   - full 7-class confusion matrix
#   - normalized confusion matrix
#   - MR/MS focused confusion
#   - missed GT counts
#   - unmatched prediction counts
# ==================================================================================================

import json
from pathlib import Path
from collections import Counter, defaultdict

import numpy as np
import pandas as pd


# ==================================================================================================
# 1. PATHS
# ==================================================================================================

BASE = Path(
    r"C:\Users\varda\Documents\_My Computer\COMMON_space\Learning"
    r"\Upgrad_EPGP_MS\upgrad_IIITB_MS_course\MS_LJMU_Material"
)

THESIS_CODE = BASE / "Thesis_Code"

E18D_DIR = (
    THESIS_CODE
    / "runs"
    / "sortwaste"
    / "E18D_class_selective_convnext"
)

CACHE_PATH = (
    E18D_DIR
    / "E18D_cached_predictions.json"
)

GT_PATH = (
    E18D_DIR
    / "E18D_val_gt_7class.json"
)

OUTPUT_DIR = (
    THESIS_CODE
    / "runs"
    / "sortwaste"
    / "E20B_confusion_analysis"
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)


# ==================================================================================================
# 2. CLASS DEFINITIONS
# ==================================================================================================

CLASS_NAMES = [
    "ecal",
    "hdpe",
    "mixed_plastic_rigid",
    "mixed_plastic_soft",
    "non_plastic",
    "pet",
    "pet_oil",
]

ECAL = 0
HDPE = 1
MR = 2
MS = 3
NP = 4
PET = 5
PET_OIL = 6


# ==================================================================================================
# 3. FROZEN E20-A SETTINGS
# ==================================================================================================

ALPHA = 0.70

MR_GATE = None
MS_GATE = 0.85
NP_GATE = 0.90
PETOIL_GATE = 0.94

IOU_THRESHOLD = 0.50


E16A_GATES = {
    0: 0.99,
    1: 0.98,
    2: 0.98,
    3: 0.94,
    4: 0.94,
    5: 0.99,
    6: 0.99,
}


# ==================================================================================================
# 4. LOAD CACHE + GT
# ==================================================================================================

with open(
    CACHE_PATH,
    "r",
    encoding="utf-8"
) as f:
    cache = json.load(f)

with open(
    GT_PATH,
    "r",
    encoding="utf-8"
) as f:
    gt = json.load(f)


print("=" * 110)
print("E20-B — VALIDATION CONFUSION ANALYSIS")
print("=" * 110)

print(f"\nCache entries : {len(cache):,}")
print(f"GT objects    : {len(gt['annotations']):,}")
print(f"IoU threshold : {IOU_THRESHOLD:.2f}")

print("\nFrozen E20-A:")
print("  Mixed Rigid : OFF")
print("  Mixed Soft  : 0.85")
print("  non_plastic : 0.90")
print("  PET Oil     : 0.94")
print("  alpha       : 0.70")


# ==================================================================================================
# 5. RECONSTRUCT E20-A PREDICTIONS FROM CACHE
# ==================================================================================================

predictions_by_image = defaultdict(list)


for item in cache:

    image_id = int(item["image_id"])

    yolo_idx = int(item["yolo_class"])
    yolo_conf = float(item["yolo_conf"])

    mn_probs = np.asarray(
        item["mn_probs"],
        dtype=np.float32
    )

    e16_final_idx = int(
        item["e16_final_idx"]
    )

    final_idx = e16_final_idx

    classifier_prob = float(
        mn_probs[final_idx]
    )

    final_source = "e16"


    conv = item["conv"]

    if conv is not None:

        conv_global_idx = int(
            conv["conv_global_idx"]
        )

        conv_top_prob = float(
            conv["conv_top_prob"]
        )

        gate = None

        if conv_global_idx == MR:
            gate = MR_GATE

        elif conv_global_idx == MS:
            gate = MS_GATE

        elif conv_global_idx == NP:
            gate = NP_GATE

        elif conv_global_idx == PET_OIL:
            gate = PETOIL_GATE


        if (
            gate is not None
            and conv_top_prob >= gate
        ):

            final_idx = conv_global_idx
            classifier_prob = conv_top_prob
            final_source = "convnext"


    classifier_prob = max(
        classifier_prob,
        1e-12
    )

    score = (
        max(yolo_conf, 1e-12) ** ALPHA
    ) * (
        classifier_prob ** (1.0 - ALPHA)
    )


    x1, y1, x2, y2 = [
        float(v)
        for v in item["bbox"]
    ]


    predictions_by_image[
        image_id
    ].append(
        {
            "class_idx": final_idx,
            "bbox_xyxy": [
                x1,
                y1,
                x2,
                y2
            ],
            "score": score,
            "source": final_source,
        }
    )


# ==================================================================================================
# 6. BUILD GT BY IMAGE
# ==================================================================================================

gt_by_image = defaultdict(list)

for ann in gt["annotations"]:

    image_id = int(
        ann["image_id"]
    )

    class_idx = int(
        ann["category_id"]
    ) - 1

    x, y, w, h = [
        float(v)
        for v in ann["bbox"]
    ]

    gt_by_image[
        image_id
    ].append(
        {
            "class_idx": class_idx,
            "bbox_xyxy": [
                x,
                y,
                x + w,
                y + h
            ]
        }
    )


# ==================================================================================================
# 7. IOU
# ==================================================================================================

def iou_xyxy(
    a,
    b
):

    ax1, ay1, ax2, ay2 = a
    bx1, by1, bx2, by2 = b

    ix1 = max(ax1, bx1)
    iy1 = max(ay1, by1)
    ix2 = min(ax2, bx2)
    iy2 = min(ay2, by2)

    iw = max(
        0.0,
        ix2 - ix1
    )

    ih = max(
        0.0,
        iy2 - iy1
    )

    intersection = (
        iw * ih
    )

    area_a = max(
        0.0,
        ax2 - ax1
    ) * max(
        0.0,
        ay2 - ay1
    )

    area_b = max(
        0.0,
        bx2 - bx1
    ) * max(
        0.0,
        by2 - by1
    )

    union = (
        area_a
        + area_b
        - intersection
    )

    if union <= 0:
        return 0.0

    return (
        intersection
        / union
    )


# ==================================================================================================
# 8. GREEDY MATCHING
#
# Important:
# We match spatially regardless of class.
# That lets us measure CLASS CONFUSION among detections that localized the object.
# ==================================================================================================

confusion = np.zeros(
    (
        len(CLASS_NAMES),
        len(CLASS_NAMES)
    ),
    dtype=np.int64
)

missed_gt = Counter()
false_positive = Counter()

matched_iou_values = []

matched_records = []


all_image_ids = sorted(
    set(gt_by_image.keys())
    | set(predictions_by_image.keys())
)


for image_id in all_image_ids:

    gts = gt_by_image.get(
        image_id,
        []
    )

    preds = predictions_by_image.get(
        image_id,
        []
    )


    # sort predictions by confidence descending
    preds = sorted(
        preds,
        key=lambda x: x["score"],
        reverse=True
    )


    candidate_pairs = []


    for gt_idx, gt_item in enumerate(
        gts
    ):

        for pred_idx, pred_item in enumerate(
            preds
        ):

            iou = iou_xyxy(
                gt_item["bbox_xyxy"],
                pred_item["bbox_xyxy"]
            )

            if iou >= IOU_THRESHOLD:

                candidate_pairs.append(
                    (
                        iou,
                        gt_idx,
                        pred_idx
                    )
                )


    # highest IoU first
    candidate_pairs.sort(
        reverse=True,
        key=lambda x: x[0]
    )


    matched_gt_indices = set()
    matched_pred_indices = set()


    for (
        iou,
        gt_idx,
        pred_idx
    ) in candidate_pairs:

        if gt_idx in matched_gt_indices:
            continue

        if pred_idx in matched_pred_indices:
            continue


        gt_item = gts[gt_idx]
        pred_item = preds[pred_idx]


        gt_class = int(
            gt_item["class_idx"]
        )

        pred_class = int(
            pred_item["class_idx"]
        )


        confusion[
            gt_class,
            pred_class
        ] += 1


        matched_gt_indices.add(
            gt_idx
        )

        matched_pred_indices.add(
            pred_idx
        )


        matched_iou_values.append(
            iou
        )


        matched_records.append(
            {
                "image_id": image_id,
                "gt_class": CLASS_NAMES[
                    gt_class
                ],
                "pred_class": CLASS_NAMES[
                    pred_class
                ],
                "iou": float(iou),
                "score": float(
                    pred_item["score"]
                ),
                "source": pred_item[
                    "source"
                ],
            }
        )


    # unmatched GT = missed localization/detection
    for gt_idx, gt_item in enumerate(
        gts
    ):

        if gt_idx not in matched_gt_indices:

            missed_gt[
                CLASS_NAMES[
                    gt_item[
                        "class_idx"
                    ]
                ]
            ] += 1


    # unmatched predictions = false positives
    for pred_idx, pred_item in enumerate(
        preds
    ):

        if pred_idx not in matched_pred_indices:

            false_positive[
                CLASS_NAMES[
                    pred_item[
                        "class_idx"
                    ]
                ]
            ] += 1


# ==================================================================================================
# 9. RAW CONFUSION MATRIX
# ==================================================================================================

df_confusion = pd.DataFrame(
    confusion,
    index=[
        f"GT: {name}"
        for name in CLASS_NAMES
    ],
    columns=[
        f"Pred: {name}"
        for name in CLASS_NAMES
    ]
)


print()
print("=" * 110)
print("RAW MATCHED-DETECTION CONFUSION MATRIX — IoU >= 0.50")
print("=" * 110)
print()
print(df_confusion.to_string())


# ==================================================================================================
# 10. ROW-NORMALIZED CONFUSION MATRIX
# ==================================================================================================

row_totals = confusion.sum(
    axis=1,
    keepdims=True
)


normalized = np.divide(
    confusion,
    row_totals,
    out=np.zeros_like(
        confusion,
        dtype=float
    ),
    where=(
        row_totals != 0
    )
)


df_normalized = pd.DataFrame(
    normalized * 100.0,
    index=[
        f"GT: {name}"
        for name in CLASS_NAMES
    ],
    columns=[
        f"Pred: {name}"
        for name in CLASS_NAMES
    ]
)


print()
print("=" * 110)
print("ROW-NORMALIZED MATCHED CONFUSION MATRIX (%)")
print("=" * 110)
print()
print(
    df_normalized
    .round(2)
    .to_string()
)


# ==================================================================================================
# 11. FOCUSED MR / MS CONFUSION
# ==================================================================================================

print()
print("=" * 110)
print("MIXED RIGID / MIXED SOFT ERROR ANALYSIS")
print("=" * 110)


for target_idx in [
    MR,
    MS
]:

    target_name = (
        CLASS_NAMES[
            target_idx
        ]
    )

    row = confusion[
        target_idx
    ]

    matched_total = int(
        row.sum()
    )

    correct = int(
        row[
            target_idx
        ]
    )

    errors = []

    for pred_idx, count in enumerate(
        row
    ):

        if pred_idx == target_idx:
            continue

        if count > 0:

            errors.append(
                (
                    int(count),
                    CLASS_NAMES[
                        pred_idx
                    ]
                )
            )


    errors.sort(
        reverse=True
    )


    print()
    print(
        f"{target_name.upper()}"
    )

    print(
        "-" * 70
    )

    print(
        f"Matched GT objects : "
        f"{matched_total:,}"
    )

    print(
        f"Correct class      : "
        f"{correct:,} "
        f"({100 * correct / matched_total:.2f}%)"
        if matched_total
        else "Correct class      : 0"
    )

    print(
        f"Missed GT          : "
        f"{missed_gt[target_name]:,}"
    )


    print(
        "\nWrong matched classifications:"
    )


    total_wrong = (
        matched_total
        - correct
    )


    for count, wrong_class in (
        errors
    ):

        pct_all = (
            100.0
            * count
            / matched_total
            if matched_total
            else 0.0
        )

        pct_wrong = (
            100.0
            * count
            / total_wrong
            if total_wrong
            else 0.0
        )


        print(
            f"  -> {wrong_class:24s}: "
            f"{count:5,d}   "
            f"{pct_all:6.2f}% of matched GT   "
            f"{pct_wrong:6.2f}% of class errors"
        )


# ==================================================================================================
# 12. DIRECT MR <-> MS CONFUSION
# ==================================================================================================

mr_to_ms = int(
    confusion[
        MR,
        MS
    ]
)

ms_to_mr = int(
    confusion[
        MS,
        MR
    ]
)


print()
print("=" * 110)
print("DIRECT MR <-> MS CONFUSION")
print("=" * 110)

print(
    f"Mixed Rigid -> Mixed Soft : "
    f"{mr_to_ms:,}"
)

print(
    f"Mixed Soft  -> Mixed Rigid: "
    f"{ms_to_mr:,}"
)

print(
    f"Total bidirectional       : "
    f"{mr_to_ms + ms_to_mr:,}"
)


# ==================================================================================================
# 13. MR/MS VS NON-PLASTIC
# ==================================================================================================

print()
print("=" * 110)
print("MR / MS <-> NON-PLASTIC CONFUSION")
print("=" * 110)


print(
    f"MR -> non_plastic : "
    f"{int(confusion[MR, NP]):,}"
)

print(
    f"MS -> non_plastic : "
    f"{int(confusion[MS, NP]):,}"
)

print(
    f"NP -> MR          : "
    f"{int(confusion[NP, MR]):,}"
)

print(
    f"NP -> MS          : "
    f"{int(confusion[NP, MS]):,}"
)


# ==================================================================================================
# 14. MISSED GT COUNTS
# ==================================================================================================

print()
print("=" * 110)
print("MISSED GT OBJECTS — NO PREDICTION MATCHED AT IoU >= 0.50")
print("=" * 110)


for class_name in CLASS_NAMES:

    print(
        f"{class_name:25s}: "
        f"{missed_gt[class_name]:,}"
    )


# ==================================================================================================
# 15. FALSE POSITIVE COUNTS
# ==================================================================================================

print()
print("=" * 110)
print("UNMATCHED PREDICTIONS / FALSE POSITIVES")
print("=" * 110)


for class_name in CLASS_NAMES:

    print(
        f"{class_name:25s}: "
        f"{false_positive[class_name]:,}"
    )


# ==================================================================================================
# 16. SOURCE ANALYSIS FOR MR / MS WRONG PREDICTIONS
# ==================================================================================================

source_error_counts = {

    "mixed_plastic_rigid":
        Counter(),

    "mixed_plastic_soft":
        Counter(),
}


for rec in matched_records:

    gt_class = (
        rec[
            "gt_class"
        ]
    )

    pred_class = (
        rec[
            "pred_class"
        ]
    )


    if (
        gt_class
        in source_error_counts
        and pred_class
        != gt_class
    ):

        source_error_counts[
            gt_class
        ][
            rec[
                "source"
            ]
        ] += 1


print()
print("=" * 110)
print("SOURCE OF MR / MS MISCLASSIFICATIONS")
print("=" * 110)


for class_name, counts in (
    source_error_counts.items()
):

    print()
    print(
        class_name
    )

    for source, count in (
        counts.most_common()
    ):

        print(
            f"  {source:15s}: "
            f"{count:,}"
        )


# ==================================================================================================
# 17. SAVE CSV / JSON
# ==================================================================================================

RAW_CSV = (
    OUTPUT_DIR
    / "E20B_confusion_raw.csv"
)

NORM_CSV = (
    OUTPUT_DIR
    / "E20B_confusion_normalized_percent.csv"
)

MATCHED_JSON = (
    OUTPUT_DIR
    / "E20B_matched_records.json"
)

SUMMARY_JSON = (
    OUTPUT_DIR
    / "E20B_confusion_summary.json"
)


df_confusion.to_csv(
    RAW_CSV
)

df_normalized.to_csv(
    NORM_CSV
)


with open(
    MATCHED_JSON,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        matched_records,
        f,
        indent=2
    )


summary = {

    "iou_threshold":
        IOU_THRESHOLD,

    "confusion_matrix":
        confusion.tolist(),

    "class_names":
        CLASS_NAMES,

    "missed_gt":
        dict(
            missed_gt
        ),

    "false_positive":
        dict(
            false_positive
        ),

    "mr_to_ms":
        mr_to_ms,

    "ms_to_mr":
        ms_to_mr,

    "mr_to_nonplastic":
        int(
            confusion[
                MR,
                NP
            ]
        ),

    "ms_to_nonplastic":
        int(
            confusion[
                MS,
                NP
            ]
        ),

    "nonplastic_to_mr":
        int(
            confusion[
                NP,
                MR
            ]
        ),

    "nonplastic_to_ms":
        int(
            confusion[
                NP,
                MS
            ]
        ),
}


with open(
    SUMMARY_JSON,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        summary,
        f,
        indent=2
    )


print()
print("=" * 110)
print("E20-B CONFUSION ANALYSIS COMPLETE")
print("=" * 110)

print()
print(
    f"Raw matrix       : "
    f"{RAW_CSV}"
)

print(
    f"Normalized matrix: "
    f"{NORM_CSV}"
)

print(
    f"Matched records  : "
    f"{MATCHED_JSON}"
)

print(
    f"Summary          : "
    f"{SUMMARY_JSON}"
)

E20-B — VALIDATION CONFUSION ANALYSIS

Cache entries : 55,605
GT objects    : 13,065
IoU threshold : 0.50

Frozen E20-A:
  Mixed Rigid : OFF
  Mixed Soft  : 0.85
  non_plastic : 0.90
  PET Oil     : 0.94
  alpha       : 0.70

RAW MATCHED-DETECTION CONFUSION MATRIX — IoU >= 0.50

                         Pred: ecal  Pred: hdpe  Pred: mixed_plastic_rigid  Pred: mixed_plastic_soft  Pred: non_plastic  Pred: pet  Pred: pet_oil
GT: ecal                       2231          96                         19                        92                 31         23              1
GT: hdpe                        271        4052                        102                       100                100        147             17
GT: mixed_plastic_rigid          41          79                        646                       217                 28         62             11
GT: mixed_plastic_soft           93         102                        148                       908                 32        126      

# Model E21: Mixed-Plastic Hard-Negative Specialist
| Experiment | Purpose                                              |
| ---------- | ---------------------------------------------------- |
| **E21-A**  | Train 6-class hard-negative ConvNeXt-Tiny specialist |
| **E21-B**  | Validation-only routing/trigger/gate search          |
| **E21-C**  | Frozen final E21 pipeline evaluated once on TEST     |


In [ ]:
# E21-A — MIXED-PLASTIC HARD-NEGATIVE SPECIALIST
#
# PURPOSE
# --------------------------------------------------------------------------------------------------
# Train a targeted ConvNeXt-Tiny specialist to improve discrimination of:
#
#   Mixed Rigid Plastic
#   Mixed Soft Plastic
#
# against the classes they are most often confused with:
#
#   HDPE
#   PET
#   ECAL
#   non_plastic
#
#
# IMPORTANT
# --------------------------------------------------------------------------------------------------
# - TRAIN + VALIDATION ONLY
# - TEST SET IS NOT USED
# - E3Y-B MobileNet is a 7-class model
# - Original crop folders are still 8 physical folders
# - cardboard + metal are merged into non_plastic for E21
#
#
# E21 SPECIALIST CLASSES
# --------------------------------------------------------------------------------------------------
# 0 mixed_plastic_rigid
# 1 mixed_plastic_soft
# 2 hdpe
# 3 pet
# 4 ecal
# 5 non_plastic
#
#
# HARD-NEGATIVE MINING
# --------------------------------------------------------------------------------------------------
# All MR and MS samples are kept.
#
# Negative classes are ranked using:
#
#     hardness = P(MR) + P(MS)
#
# from the existing E3Y-B 7-class MobileNet.
#
# This prioritizes negative crops that the current classifier already finds
# visually similar to Mixed Rigid / Mixed Soft.
#
#
# MODEL
# --------------------------------------------------------------------------------------------------
# ConvNeXt-Tiny, ImageNet pretrained
#
#
# SELECTION
# --------------------------------------------------------------------------------------------------
# Best validation Macro-F1
#
# ==================================================================================================

import json
import random
import time
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd

import torch
import torch.nn as nn

from torch.utils.data import (
    Dataset,
    DataLoader,
    WeightedRandomSampler,
)

from torchvision import models
from torchvision.models import ConvNeXt_Tiny_Weights
from torchvision.transforms import v2

from PIL import Image

from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    f1_score,
    accuracy_score,
)

from tqdm.auto import tqdm


# ==================================================================================================
# 1. REPRODUCIBILITY
# ==================================================================================================

SEED = 42

random.seed(SEED)
np.random.seed(SEED)

torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)


# ==================================================================================================
# 2. DEVICE
# ==================================================================================================

DEVICE = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)


print("=" * 110)
print("E21-A — MIXED-PLASTIC HARD-NEGATIVE SPECIALIST")
print("=" * 110)

print()
print(f"PyTorch : {torch.__version__}")
print(f"Device  : {DEVICE}")

if DEVICE.type == "cuda":

    print(
        f"GPU     : "
        f"{torch.cuda.get_device_name(0)}"
    )


# ==================================================================================================
# 3. PATHS
# ==================================================================================================

BASE = Path(
    r"C:\Users\varda\Documents\_My Computer\COMMON_space\Learning"
    r"\Upgrad_EPGP_MS\upgrad_IIITB_MS_course\MS_LJMU_Material"
)


DATASET_ROOT = (
    BASE
    / "Topic Data"
    / "SortWaste"
    / "dataset"
    / "dataset"
)


THESIS_CODE = (
    BASE
    / "Thesis_Code"
)


# --------------------------------------------------------------------------------------------------
# Existing GT crops
# --------------------------------------------------------------------------------------------------

CROP_ROOT = (
    DATASET_ROOT
    / "cropped_data"
)


TRAIN_CROP_ROOT = (
    CROP_ROOT
    / "train"
)


VAL_CROP_ROOT = (
    CROP_ROOT
    / "val"
)


# --------------------------------------------------------------------------------------------------
# Existing E3Y-B MobileNet — 7 CLASS
# --------------------------------------------------------------------------------------------------

MOBILENET_CKPT = (
    DATASET_ROOT
    / "yolo_mobilenet_crops_E3Y"
    / "mobilenet_results"
    / "E3Y_B_class_weighted"
    / "E3Y_B_MobileNetV3Large_best.pth"
)


# --------------------------------------------------------------------------------------------------
# E21-A output
# --------------------------------------------------------------------------------------------------

OUTPUT_DIR = (
    THESIS_CODE
    / "runs"
    / "sortwaste"
    / "E21A_mixedplastic_hardnegative_convnext"
)


OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)


BEST_MODEL_PATH = (
    OUTPUT_DIR
    / "E21A_ConvNeXtTiny_best.pth"
)


TRAIN_MANIFEST_PATH = (
    OUTPUT_DIR
    / "E21A_hard_negative_manifest_train.json"
)


VAL_MANIFEST_PATH = (
    OUTPUT_DIR
    / "E21A_validation_manifest.json"
)


HISTORY_PATH = (
    OUTPUT_DIR
    / "E21A_training_history.json"
)


RESULTS_PATH = (
    OUTPUT_DIR
    / "E21A_validation_results.json"
)


CONFUSION_CSV_PATH = (
    OUTPUT_DIR
    / "E21A_validation_confusion_matrix.csv"
)


# ==================================================================================================
# 4. VERIFY REQUIRED PATHS
# ==================================================================================================

required_paths = [

    TRAIN_CROP_ROOT,

    VAL_CROP_ROOT,

    MOBILENET_CKPT,
]


for path in required_paths:

    if not path.exists():

        raise FileNotFoundError(
            f"Required path not found:\n{path}"
        )


print()
print(f"Train crops : {TRAIN_CROP_ROOT}")
print(f"Val crops   : {VAL_CROP_ROOT}")
print(f"MobileNet   : {MOBILENET_CKPT}")
print(f"Output      : {OUTPUT_DIR}")


# ==================================================================================================
# 5. E3Y-B MOBILENET — CORRECT 7-CLASS MAPPING
#
# IMPORTANT:
#
# E3Y-B checkpoint output classes:
#
# 0 ecal
# 1 hdpe
# 2 mixed_plastic_rigid
# 3 mixed_plastic_soft
# 4 non_plastic
# 5 pet
# 6 pet_oil
#
# ==================================================================================================

MN_CLASS_NAMES = [

    "ecal",

    "hdpe",

    "mixed_plastic_rigid",

    "mixed_plastic_soft",

    "non_plastic",

    "pet",

    "pet_oil",
]


MN_NAME_TO_IDX = {

    name: idx

    for idx, name
    in enumerate(
        MN_CLASS_NAMES
    )
}


MN_MR = (
    MN_NAME_TO_IDX[
        "mixed_plastic_rigid"
    ]
)


MN_MS = (
    MN_NAME_TO_IDX[
        "mixed_plastic_soft"
    ]
)


print()
print("E3Y-B MobileNet mapping:")

for idx, name in enumerate(
    MN_CLASS_NAMES
):

    print(
        f"  {idx}: {name}"
    )


# ==================================================================================================
# 6. E21 SPECIALIST CLASS MAPPING
# ==================================================================================================

E21_CLASS_NAMES = [

    "mixed_plastic_rigid",

    "mixed_plastic_soft",

    "hdpe",

    "pet",

    "ecal",

    "non_plastic",
]


E21_NAME_TO_IDX = {

    name: idx

    for idx, name
    in enumerate(
        E21_CLASS_NAMES
    )
}


# ==================================================================================================
# 7. ORIGINAL CROP FOLDER -> E21 CLASS
#
# cropped_data still contains original folders:
#
# cardboard
# ecal
# hdpe
# metal
# mixed_plastic_rigid
# mixed_plastic_soft
# pet
# pet_oil
#
# PET Oil is excluded from E21 specialist.
#
# ==================================================================================================

SOURCE_TO_E21 = {

    "mixed_plastic_rigid":
        "mixed_plastic_rigid",

    "mixed_plastic_soft":
        "mixed_plastic_soft",

    "hdpe":
        "hdpe",

    "pet":
        "pet",

    "ecal":
        "ecal",

    "cardboard":
        "non_plastic",

    "metal":
        "non_plastic",
}


# ==================================================================================================
# 8. CONFIGURATION
# ==================================================================================================

IMAGE_SIZE = 224


# Hard-negative mining
BATCH_SIZE_MINE = 64


# ConvNeXt training
BATCH_SIZE_TRAIN = 24


# Windows + Jupyter safety
NUM_WORKERS = 0


EPOCHS = 15

LEARNING_RATE = 1e-4

WEIGHT_DECAY = 1e-4

PATIENCE = 4


# --------------------------------------------------------------------------------------------------
# Hard-negative sampling
#
# Each negative class is capped around 75% of mean MR/MS population.
# --------------------------------------------------------------------------------------------------

NEGATIVE_RATIO = 0.75


# ==================================================================================================
# 9. IMAGE EXTENSIONS
# ==================================================================================================

IMAGE_EXTENSIONS = {

    ".jpg",

    ".jpeg",

    ".png",

    ".bmp",

    ".webp",
}


# ==================================================================================================
# 10. IMAGE LISTING HELPER
# ==================================================================================================

def list_images(
    folder
):

    if not folder.exists():

        return []


    return sorted(
        [
            p

            for p in folder.rglob("*")

            if (
                p.is_file()
                and p.suffix.lower()
                in IMAGE_EXTENSIONS
            )
        ]
    )


# ==================================================================================================
# 11. COLLECT TRAIN FILES
# ==================================================================================================

train_source_files = {}


for source_class in (
    SOURCE_TO_E21.keys()
):

    folder = (
        TRAIN_CROP_ROOT
        / source_class
    )


    files = list_images(
        folder
    )


    train_source_files[
        source_class
    ] = files


print()
print("=" * 110)
print("AVAILABLE TRAIN CROPS")
print("=" * 110)


for class_name, files in (
    train_source_files.items()
):

    print(
        f"{class_name:25s}: "
        f"{len(files):,}"
    )


# ==================================================================================================
# 12. LOAD E3Y-B 7-CLASS MOBILENET
# ==================================================================================================

print()
print("=" * 110)
print("LOADING E3Y-B 7-CLASS MOBILENET")
print("=" * 110)


mobilenet = models.mobilenet_v3_large(
    weights=None
)


mobilenet.classifier[
    3
] = nn.Linear(

    mobilenet.classifier[
        3
    ].in_features,

    7
)


checkpoint = torch.load(
    MOBILENET_CKPT,
    map_location=DEVICE
)


# --------------------------------------------------------------------------------------------------
# Extract state dict safely
# --------------------------------------------------------------------------------------------------

if isinstance(
    checkpoint,
    dict
):

    if (
        "model_state_dict"
        in checkpoint
    ):

        state_dict = (
            checkpoint[
                "model_state_dict"
            ]
        )


    elif (
        "state_dict"
        in checkpoint
    ):

        state_dict = (
            checkpoint[
                "state_dict"
            ]
        )


    else:

        state_dict = (
            checkpoint
        )


else:

    state_dict = (
        checkpoint
    )


# --------------------------------------------------------------------------------------------------
# Remove DataParallel prefix if present
# --------------------------------------------------------------------------------------------------

clean_state_dict = {}


for key, value in (
    state_dict.items()
):

    if key.startswith(
        "module."
    ):

        clean_key = (
            key[
                len(
                    "module."
                ):
            ]
        )

    else:

        clean_key = (
            key
        )


    clean_state_dict[
        clean_key
    ] = value


# --------------------------------------------------------------------------------------------------
# Check classifier shape
# --------------------------------------------------------------------------------------------------

classifier_key = (
    "classifier.3.weight"
)


if (
    classifier_key
    not in clean_state_dict
):

    raise RuntimeError(
        "classifier.3.weight not found in E3Y-B checkpoint."
    )


checkpoint_num_classes = int(

    clean_state_dict[
        classifier_key
    ].shape[
        0
    ]
)


print(
    f"Checkpoint output classes : "
    f"{checkpoint_num_classes}"
)


if checkpoint_num_classes != 7:

    raise RuntimeError(

        f"E3Y-B checkpoint should contain 7 classes, "
        f"but checkpoint contains {checkpoint_num_classes}."
    )


mobilenet.load_state_dict(
    clean_state_dict,
    strict=True
)


mobilenet = mobilenet.to(
    DEVICE
)


mobilenet.eval()


print(
    "E3Y-B MobileNet loaded successfully."
)


# ==================================================================================================
# 13. HARD-NEGATIVE MINING TRANSFORM
# ==================================================================================================

mine_transform = v2.Compose(
    [

        v2.Resize(
            (
                IMAGE_SIZE,
                IMAGE_SIZE
            ),
            antialias=True
        ),


        v2.ToImage(),


        v2.ToDtype(
            torch.float32,
            scale=True
        ),


        v2.Normalize(
            mean=[
                0.485,
                0.456,
                0.406
            ],
            std=[
                0.229,
                0.224,
                0.225
            ]
        ),
    ]
)


# ==================================================================================================
# 14. MINING DATASET
# ==================================================================================================

class MiningDataset(
    Dataset
):

    def __init__(
        self,
        paths
    ):

        self.paths = (
            paths
        )


    def __len__(
        self
    ):

        return len(
            self.paths
        )


    def __getitem__(
        self,
        idx
    ):

        path = (
            self.paths[
                idx
            ]
        )


        with Image.open(
            path
        ) as img:

            image = (
                img.convert(
                    "RGB"
                )
            )


        image = mine_transform(
            image
        )


        return (

            image,

            str(
                path
            ),
        )


# ==================================================================================================
# 15. HARDNESS MINING
#
# hardness = P(MR) + P(MS)
# ==================================================================================================

@torch.inference_mode()
def mine_hardness(
    paths,
    description
):

    if len(
        paths
    ) == 0:

        return []


    dataset = MiningDataset(
        paths
    )


    loader = DataLoader(

        dataset,

        batch_size=BATCH_SIZE_MINE,

        shuffle=False,

        num_workers=NUM_WORKERS,

        pin_memory=(
            DEVICE.type
            == "cuda"
        )
    )


    records = []


    for (
        images,
        path_strings
    ) in tqdm(

        loader,

        desc=description
    ):

        images = images.to(
            DEVICE,
            non_blocking=True
        )


        logits = mobilenet(
            images
        )


        probs = torch.softmax(
            logits,
            dim=1
        )


        mr_probs = (
            probs[
                :,
                MN_MR
            ]
        )


        ms_probs = (
            probs[
                :,
                MN_MS
            ]
        )


        hardness = (

            mr_probs

            +

            ms_probs
        )


        for i in range(
            len(
                path_strings
            )
        ):

            records.append(
                {

                    "path":
                        path_strings[
                            i
                        ],

                    "mr_prob":
                        float(
                            mr_probs[
                                i
                            ].item()
                        ),

                    "ms_prob":
                        float(
                            ms_probs[
                                i
                            ].item()
                        ),

                    "hardness":
                        float(
                            hardness[
                                i
                            ].item()
                        ),
                }
            )


    return records


# ==================================================================================================
# 16. DETERMINE NEGATIVE TARGET SIZE
# ==================================================================================================

mr_count = len(
    train_source_files[
        "mixed_plastic_rigid"
    ]
)


ms_count = len(
    train_source_files[
        "mixed_plastic_soft"
    ]
)


target_reference = int(
    round(
        (
            mr_count
            +
            ms_count
        )
        /
        2.0
    )
)


negative_target = int(
    round(
        target_reference
        *
        NEGATIVE_RATIO
    )
)


print()
print("=" * 110)
print("E21-A HARD-NEGATIVE SAMPLING TARGETS")
print("=" * 110)


print(
    f"Mixed Rigid available : "
    f"{mr_count:,}"
)


print(
    f"Mixed Soft available  : "
    f"{ms_count:,}"
)


print(
    f"MR/MS mean count      : "
    f"{target_reference:,}"
)


print(
    f"Negative target/class : "
    f"{negative_target:,}"
)


# ==================================================================================================
# 17. BUILD TRAIN MANIFEST
# ==================================================================================================

train_manifest = []


# --------------------------------------------------------------------------------------------------
# Keep ALL Mixed Rigid
# --------------------------------------------------------------------------------------------------

for path in (
    train_source_files[
        "mixed_plastic_rigid"
    ]
):

    train_manifest.append(
        {

            "path":
                str(
                    path
                ),

            "class_name":
                "mixed_plastic_rigid",

            "class_idx":
                E21_NAME_TO_IDX[
                    "mixed_plastic_rigid"
                ],

            "selection":
                "all_target",

            "hardness":
                None,
        }
    )


# --------------------------------------------------------------------------------------------------
# Keep ALL Mixed Soft
# --------------------------------------------------------------------------------------------------

for path in (
    train_source_files[
        "mixed_plastic_soft"
    ]
):

    train_manifest.append(
        {

            "path":
                str(
                    path
                ),

            "class_name":
                "mixed_plastic_soft",

            "class_idx":
                E21_NAME_TO_IDX[
                    "mixed_plastic_soft"
                ],

            "selection":
                "all_target",

            "hardness":
                None,
        }
    )


# ==================================================================================================
# 18. MINE HDPE / PET / ECAL
# ==================================================================================================

for source_class in [

    "hdpe",

    "pet",

    "ecal",
]:

    paths = (
        train_source_files[
            source_class
        ]
    )


    print()
    print(
        "=" * 110
    )

    print(
        f"MINING HARD NEGATIVES — "
        f"{source_class.upper()}"
    )

    print(
        "=" * 110
    )


    records = mine_hardness(

        paths,

        description=(
            f"Mining {source_class}"
        )
    )


    records.sort(

        key=lambda x:
            x[
                "hardness"
            ],

        reverse=True
    )


    n_select = min(

        negative_target,

        len(
            records
        )
    )


    selected = (
        records[
            :n_select
        ]
    )


    for record in selected:

        train_manifest.append(
            {

                "path":
                    record[
                        "path"
                    ],

                "class_name":
                    source_class,

                "class_idx":
                    E21_NAME_TO_IDX[
                        source_class
                    ],

                "selection":
                    "hard_negative",

                "hardness":
                    record[
                        "hardness"
                    ],

                "mr_prob":
                    record[
                        "mr_prob"
                    ],

                "ms_prob":
                    record[
                        "ms_prob"
                    ],
            }
        )


    print(
        f"Selected : "
        f"{len(selected):,} "
        f"/ {len(records):,}"
    )


    if len(
        selected
    ) > 0:

        print(
            f"Selected hardness range : "
            f"{selected[-1]['hardness']:.4f} "
            f"to "
            f"{selected[0]['hardness']:.4f}"
        )


# ==================================================================================================
# 19. MINE NON-PLASTIC
#
# Merge metal + cardboard.
# ==================================================================================================

nonplastic_paths = (

    train_source_files[
        "metal"
    ]

    +

    train_source_files[
        "cardboard"
    ]
)


print()
print("=" * 110)
print("MINING HARD NEGATIVES — NON_PLASTIC")
print("=" * 110)


nonplastic_records = mine_hardness(

    nonplastic_paths,

    description=(
        "Mining non_plastic"
    )
)


nonplastic_records.sort(

    key=lambda x:
        x[
            "hardness"
        ],

    reverse=True
)


n_select_np = min(

    negative_target,

    len(
        nonplastic_records
    )
)


selected_nonplastic = (
    nonplastic_records[
        :n_select_np
    ]
)


for record in (
    selected_nonplastic
):

    train_manifest.append(
        {

            "path":
                record[
                    "path"
                ],

            "class_name":
                "non_plastic",

            "class_idx":
                E21_NAME_TO_IDX[
                    "non_plastic"
                ],

            "selection":
                "hard_negative",

            "hardness":
                record[
                    "hardness"
                ],

            "mr_prob":
                record[
                    "mr_prob"
                ],

            "ms_prob":
                record[
                    "ms_prob"
                ],
        }
    )


print(
    f"Selected : "
    f"{len(selected_nonplastic):,} "
    f"/ {len(nonplastic_records):,}"
)


# ==================================================================================================
# 20. SHUFFLE + SAVE TRAIN MANIFEST
# ==================================================================================================

random.shuffle(
    train_manifest
)


with open(
    TRAIN_MANIFEST_PATH,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        train_manifest,
        f,
        indent=2
    )


train_counts = Counter(

    item[
        "class_name"
    ]

    for item
    in train_manifest
)


print()
print("=" * 110)
print("FINAL E21-A TRAIN DISTRIBUTION")
print("=" * 110)


for class_name in (
    E21_CLASS_NAMES
):

    print(
        f"{class_name:25s}: "
        f"{train_counts[class_name]:,}"
    )


print()
print(
    f"Total train samples : "
    f"{len(train_manifest):,}"
)


# ==================================================================================================
# 21. BUILD VALIDATION MANIFEST
#
# VALIDATION MUST NOT BE HARD-NEGATIVE FILTERED.
#
# Use all relevant validation samples.
# ==================================================================================================

val_manifest = []


for (
    source_class,
    e21_class
) in SOURCE_TO_E21.items():

    folder = (
        VAL_CROP_ROOT
        / source_class
    )


    files = list_images(
        folder
    )


    for path in files:

        val_manifest.append(
            {

                "path":
                    str(
                        path
                    ),

                "class_name":
                    e21_class,

                "class_idx":
                    E21_NAME_TO_IDX[
                        e21_class
                    ],
            }
        )


with open(
    VAL_MANIFEST_PATH,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        val_manifest,
        f,
        indent=2
    )


val_counts = Counter(

    item[
        "class_name"
    ]

    for item
    in val_manifest
)


print()
print("=" * 110)
print("E21-A VALIDATION DISTRIBUTION")
print("=" * 110)


for class_name in (
    E21_CLASS_NAMES
):

    print(
        f"{class_name:25s}: "
        f"{val_counts[class_name]:,}"
    )


print()
print(
    f"Total validation samples : "
    f"{len(val_manifest):,}"
)


# ==================================================================================================
# 22. AUGMENTATION
# ==================================================================================================

train_transform = v2.Compose(
    [

        v2.RandomResizedCrop(
            size=(
                IMAGE_SIZE,
                IMAGE_SIZE
            ),
            scale=(
                0.85,
                1.0
            ),
            ratio=(
                0.90,
                1.10
            ),
            antialias=True
        ),


        v2.RandomHorizontalFlip(
            p=0.5
        ),


        v2.RandomRotation(
            degrees=10
        ),


        v2.ColorJitter(
            brightness=0.12,
            contrast=0.12,
            saturation=0.10,
            hue=0.02
        ),


        v2.ToImage(),


        v2.ToDtype(
            torch.float32,
            scale=True
        ),


        v2.Normalize(
            mean=[
                0.485,
                0.456,
                0.406
            ],
            std=[
                0.229,
                0.224,
                0.225
            ]
        ),
    ]
)


val_transform = v2.Compose(
    [

        v2.Resize(
            (
                IMAGE_SIZE,
                IMAGE_SIZE
            ),
            antialias=True
        ),


        v2.ToImage(),


        v2.ToDtype(
            torch.float32,
            scale=True
        ),


        v2.Normalize(
            mean=[
                0.485,
                0.456,
                0.406
            ],
            std=[
                0.229,
                0.224,
                0.225
            ]
        ),
    ]
)


# ==================================================================================================
# 23. MANIFEST DATASET
# ==================================================================================================

class ManifestDataset(
    Dataset
):

    def __init__(
        self,
        manifest,
        transform
    ):

        self.manifest = (
            manifest
        )

        self.transform = (
            transform
        )


    def __len__(
        self
    ):

        return len(
            self.manifest
        )


    def __getitem__(
        self,
        idx
    ):

        item = (
            self.manifest[
                idx
            ]
        )


        with Image.open(
            item[
                "path"
            ]
        ) as img:

            image = (
                img.convert(
                    "RGB"
                )
            )


        image = self.transform(
            image
        )


        label = int(
            item[
                "class_idx"
            ]
        )


        return (
            image,
            label
        )


# ==================================================================================================
# 24. DATASETS
# ==================================================================================================

train_dataset = ManifestDataset(

    train_manifest,

    train_transform
)


val_dataset = ManifestDataset(

    val_manifest,

    val_transform
)


# ==================================================================================================
# 25. BALANCED SAMPLER
#
# WeightedRandomSampler balances effective exposure across classes.
# ==================================================================================================

class_counts_array = np.asarray(
    [

        train_counts[
            class_name
        ]

        for class_name
        in E21_CLASS_NAMES
    ],
    dtype=np.float64
)


if np.any(
    class_counts_array == 0
):

    raise RuntimeError(
        "At least one E21 class contains zero training samples."
    )


class_sample_weights = (

    1.0

    /

    class_counts_array
)


sample_weights = np.asarray(
    [

        class_sample_weights[
            item[
                "class_idx"
            ]
        ]

        for item
        in train_manifest
    ],
    dtype=np.float64
)


sampler = WeightedRandomSampler(

    weights=torch.as_tensor(
        sample_weights,
        dtype=torch.double
    ),

    num_samples=len(
        train_manifest
    ),

    replacement=True
)


# ==================================================================================================
# 26. DATA LOADERS
# ==================================================================================================

train_loader = DataLoader(

    train_dataset,

    batch_size=BATCH_SIZE_TRAIN,

    sampler=sampler,

    num_workers=NUM_WORKERS,

    pin_memory=(
        DEVICE.type
        == "cuda"
    )
)


val_loader = DataLoader(

    val_dataset,

    batch_size=BATCH_SIZE_TRAIN,

    shuffle=False,

    num_workers=NUM_WORKERS,

    pin_memory=(
        DEVICE.type
        == "cuda"
    )
)


print()
print(
    f"Training batches   : "
    f"{len(train_loader):,}"
)


print(
    f"Validation batches : "
    f"{len(val_loader):,}"
)


# ==================================================================================================
# 27. BUILD CONVNEXT-TINY
# ==================================================================================================

print()
print("=" * 110)
print("BUILDING E21-A CONVNEXT-TINY")
print("=" * 110)


convnext_weights = (
    ConvNeXt_Tiny_Weights.IMAGENET1K_V1
)


model = models.convnext_tiny(
    weights=convnext_weights
)


in_features = (
    model.classifier[
        2
    ].in_features
)


model.classifier[
    2
] = nn.Linear(

    in_features,

    len(
        E21_CLASS_NAMES
    )
)


model = model.to(
    DEVICE
)


print(
    f"Output classes : "
    f"{len(E21_CLASS_NAMES)}"
)


print(
    f"Parameters     : "
    f"{sum(p.numel() for p in model.parameters()):,}"
)


# ==================================================================================================
# 28. LOSS WEIGHTS
#
# Sampler already balances class exposure.
#
# Therefore use only mild sqrt inverse-frequency weighting.
# ==================================================================================================

mean_count = float(
    np.mean(
        class_counts_array
    )
)


loss_weights = np.sqrt(

    mean_count

    /

    class_counts_array
)


loss_weights = (

    loss_weights

    /

    np.mean(
        loss_weights
    )
)


loss_weights_tensor = torch.tensor(

    loss_weights,

    dtype=torch.float32,

    device=DEVICE
)


print()
print("Loss weights:")


for (
    class_name,
    weight
) in zip(
    E21_CLASS_NAMES,
    loss_weights
):

    print(
        f"{class_name:25s}: "
        f"{weight:.4f}"
    )


criterion = nn.CrossEntropyLoss(
    weight=loss_weights_tensor
)


optimizer = torch.optim.AdamW(

    model.parameters(),

    lr=LEARNING_RATE,

    weight_decay=WEIGHT_DECAY
)


scheduler = (
    torch.optim.lr_scheduler.ReduceLROnPlateau(

        optimizer,

        mode="max",

        factor=0.5,

        patience=2
    )
)


# ==================================================================================================
# 29. TRAIN ONE EPOCH
# ==================================================================================================

def train_one_epoch(
    model,
    loader
):

    model.train()


    running_loss = 0.0


    all_true = []

    all_pred = []


    samples_seen = 0


    for (
        images,
        labels
    ) in tqdm(

        loader,

        desc="Train",

        leave=False
    ):

        images = images.to(
            DEVICE,
            non_blocking=True
        )


        labels = labels.to(
            DEVICE,
            non_blocking=True
        )


        optimizer.zero_grad(
            set_to_none=True
        )


        logits = model(
            images
        )


        loss = criterion(
            logits,
            labels
        )


        loss.backward()


        optimizer.step()


        batch_size = (
            images.size(
                0
            )
        )


        running_loss += (

            loss.item()

            *

            batch_size
        )


        samples_seen += (
            batch_size
        )


        predictions = torch.argmax(
            logits,
            dim=1
        )


        all_true.extend(

            labels.detach()
            .cpu()
            .numpy()
            .tolist()
        )


        all_pred.extend(

            predictions.detach()
            .cpu()
            .numpy()
            .tolist()
        )


    epoch_loss = (

        running_loss

        /

        samples_seen
    )


    epoch_acc = accuracy_score(
        all_true,
        all_pred
    )


    epoch_macro_f1 = f1_score(

        all_true,

        all_pred,

        average="macro",

        zero_division=0
    )


    return (

        epoch_loss,

        epoch_acc,

        epoch_macro_f1
    )


# ==================================================================================================
# 30. VALIDATION
# ==================================================================================================

@torch.inference_mode()
def validate(
    model,
    loader
):

    model.eval()


    running_loss = 0.0


    all_true = []

    all_pred = []

    all_probs = []


    samples_seen = 0


    for (
        images,
        labels
    ) in tqdm(

        loader,

        desc="Val",

        leave=False
    ):

        images = images.to(
            DEVICE,
            non_blocking=True
        )


        labels = labels.to(
            DEVICE,
            non_blocking=True
        )


        logits = model(
            images
        )


        loss = criterion(
            logits,
            labels
        )


        probs = torch.softmax(
            logits,
            dim=1
        )


        predictions = torch.argmax(
            probs,
            dim=1
        )


        batch_size = (
            images.size(
                0
            )
        )


        running_loss += (

            loss.item()

            *

            batch_size
        )


        samples_seen += (
            batch_size
        )


        all_true.extend(

            labels.detach()
            .cpu()
            .numpy()
            .tolist()
        )


        all_pred.extend(

            predictions.detach()
            .cpu()
            .numpy()
            .tolist()
        )


        all_probs.extend(

            probs.detach()
            .cpu()
            .numpy()
            .tolist()
        )


    val_loss = (

        running_loss

        /

        samples_seen
    )


    val_acc = accuracy_score(
        all_true,
        all_pred
    )


    val_macro_f1 = f1_score(

        all_true,

        all_pred,

        average="macro",

        zero_division=0
    )


    return {

        "loss":
            float(
                val_loss
            ),

        "accuracy":
            float(
                val_acc
            ),

        "macro_f1":
            float(
                val_macro_f1
            ),

        "y_true":
            all_true,

        "y_pred":
            all_pred,

        "probs":
            all_probs,
    }


# ==================================================================================================
# 31. TRAINING LOOP
# ==================================================================================================

history = []


best_macro_f1 = -1.0

best_epoch = -1


epochs_without_improvement = 0


print()
print("=" * 110)
print("TRAINING E21-A")
print("=" * 110)


for epoch in range(
    1,
    EPOCHS + 1
):

    epoch_start = (
        time.time()
    )


    (
        train_loss,
        train_acc,
        train_macro_f1
    ) = train_one_epoch(

        model,

        train_loader
    )


    val_result = validate(

        model,

        val_loader
    )


    val_macro_f1 = (
        val_result[
            "macro_f1"
        ]
    )


    scheduler.step(
        val_macro_f1
    )


    elapsed = (

        time.time()

        -

        epoch_start
    )


    current_lr = (

        optimizer.param_groups[
            0
        ][
            "lr"
        ]
    )


    record = {

        "epoch":
            epoch,

        "train_loss":
            float(
                train_loss
            ),

        "train_accuracy":
            float(
                train_acc
            ),

        "train_macro_f1":
            float(
                train_macro_f1
            ),

        "val_loss":
            float(
                val_result[
                    "loss"
                ]
            ),

        "val_accuracy":
            float(
                val_result[
                    "accuracy"
                ]
            ),

        "val_macro_f1":
            float(
                val_macro_f1
            ),

        "learning_rate":
            float(
                current_lr
            ),

        "seconds":
            float(
                elapsed
            ),
    }


    history.append(
        record
    )


    print()
    print(
        f"Epoch {epoch:02d}/{EPOCHS}"
    )


    print(
        f"  Train loss     : "
        f"{train_loss:.4f}"
    )


    print(
        f"  Train accuracy : "
        f"{train_acc * 100:.2f}%"
    )


    print(
        f"  Train macro-F1 : "
        f"{train_macro_f1:.4f}"
    )


    print(
        f"  Val loss       : "
        f"{val_result['loss']:.4f}"
    )


    print(
        f"  Val accuracy   : "
        f"{val_result['accuracy'] * 100:.2f}%"
    )


    print(
        f"  Val macro-F1   : "
        f"{val_macro_f1:.4f}"
    )


    print(
        f"  LR             : "
        f"{current_lr:.2e}"
    )


    print(
        f"  Time           : "
        f"{elapsed / 60:.2f} min"
    )


    # ----------------------------------------------------------------------------------------------
    # Save best
    # ----------------------------------------------------------------------------------------------

    if (
        val_macro_f1
        >
        best_macro_f1
    ):

        best_macro_f1 = (
            val_macro_f1
        )


        best_epoch = (
            epoch
        )


        epochs_without_improvement = 0


        torch.save(
            {

                "experiment":
                    "E21-A",

                "epoch":
                    epoch,

                "model_state_dict":
                    model.state_dict(),

                "class_names":
                    E21_CLASS_NAMES,

                "val_macro_f1":
                    float(
                        val_macro_f1
                    ),

                "image_size":
                    IMAGE_SIZE,

                "negative_ratio":
                    NEGATIVE_RATIO,

                "hardness_metric":
                    "P(MR) + P(MS) from E3Y-B MobileNet",
            },

            BEST_MODEL_PATH
        )


        print(
            "  *** BEST CHECKPOINT SAVED ***"
        )


    else:

        epochs_without_improvement += 1


    # ----------------------------------------------------------------------------------------------
    # Save history continuously
    # ----------------------------------------------------------------------------------------------

    with open(
        HISTORY_PATH,
        "w",
        encoding="utf-8"
    ) as f:

        json.dump(
            history,
            f,
            indent=2
        )


    # ----------------------------------------------------------------------------------------------
    # Early stopping
    # ----------------------------------------------------------------------------------------------

    if (
        epochs_without_improvement
        >=
        PATIENCE
    ):

        print()

        print(
            f"Early stopping: "
            f"{PATIENCE} epochs without "
            f"validation Macro-F1 improvement."
        )

        break


# ==================================================================================================
# 32. LOAD BEST E21-A CHECKPOINT
# ==================================================================================================

print()
print("=" * 110)
print("LOADING BEST E21-A CHECKPOINT")
print("=" * 110)


best_checkpoint = torch.load(
    BEST_MODEL_PATH,
    map_location=DEVICE
)


model.load_state_dict(
    best_checkpoint[
        "model_state_dict"
    ]
)


print(
    f"Best epoch    : "
    f"{best_checkpoint['epoch']}"
)


print(
    f"Best macro-F1 : "
    f"{best_checkpoint['val_macro_f1']:.4f}"
)


# ==================================================================================================
# 33. FINAL VALIDATION
# ==================================================================================================

final_val = validate(
    model,
    val_loader
)


y_true = np.asarray(
    final_val[
        "y_true"
    ]
)


y_pred = np.asarray(
    final_val[
        "y_pred"
    ]
)


print()
print("=" * 110)
print("E21-A — FINAL STANDALONE VALIDATION RESULTS")
print("=" * 110)


print()
print(
    f"Accuracy : "
    f"{final_val['accuracy'] * 100:.2f}%"
)


print(
    f"Macro-F1 : "
    f"{final_val['macro_f1']:.4f}"
)


# ==================================================================================================
# 34. CLASSIFICATION REPORT
# ==================================================================================================

labels = list(
    range(
        len(
            E21_CLASS_NAMES
        )
    )
)


report_dict = classification_report(

    y_true,

    y_pred,

    labels=labels,

    target_names=E21_CLASS_NAMES,

    output_dict=True,

    zero_division=0
)


report_text = classification_report(

    y_true,

    y_pred,

    labels=labels,

    target_names=E21_CLASS_NAMES,

    zero_division=0
)


print()
print(
    report_text
)


# ==================================================================================================
# 35. CONFUSION MATRIX
# ==================================================================================================

cm = confusion_matrix(

    y_true,

    y_pred,

    labels=labels
)


cm_df = pd.DataFrame(

    cm,

    index=[
        f"GT: {name}"
        for name
        in E21_CLASS_NAMES
    ],

    columns=[
        f"Pred: {name}"
        for name
        in E21_CLASS_NAMES
    ]
)


print()
print("=" * 110)
print("E21-A — STANDALONE VALIDATION CONFUSION MATRIX")
print("=" * 110)


print()
print(
    cm_df.to_string()
)


# ==================================================================================================
# 36. MR / MS SPECIALIST SUMMARY
# ==================================================================================================

mr_metrics = (
    report_dict[
        "mixed_plastic_rigid"
    ]
)


ms_metrics = (
    report_dict[
        "mixed_plastic_soft"
    ]
)


target_mean_f1 = (

    mr_metrics[
        "f1-score"
    ]

    +

    ms_metrics[
        "f1-score"
    ]

) / 2.0


target_mean_recall = (

    mr_metrics[
        "recall"
    ]

    +

    ms_metrics[
        "recall"
    ]

) / 2.0


print()
print("=" * 110)
print("E21-A — MR / MS SPECIALIST SUMMARY")
print("=" * 110)


print()
print(
    f"Mixed Rigid precision : "
    f"{mr_metrics['precision']:.4f}"
)


print(
    f"Mixed Rigid recall    : "
    f"{mr_metrics['recall']:.4f}"
)


print(
    f"Mixed Rigid F1        : "
    f"{mr_metrics['f1-score']:.4f}"
)


print()
print(
    f"Mixed Soft precision  : "
    f"{ms_metrics['precision']:.4f}"
)


print(
    f"Mixed Soft recall     : "
    f"{ms_metrics['recall']:.4f}"
)


print(
    f"Mixed Soft F1         : "
    f"{ms_metrics['f1-score']:.4f}"
)


print()
print(
    f"MR/MS mean F1         : "
    f"{target_mean_f1:.4f}"
)


print(
    f"MR/MS mean recall     : "
    f"{target_mean_recall:.4f}"
)


# ==================================================================================================
# 37. DIRECT MR <-> MS CONFUSION
# ==================================================================================================

MR_IDX = (
    E21_NAME_TO_IDX[
        "mixed_plastic_rigid"
    ]
)


MS_IDX = (
    E21_NAME_TO_IDX[
        "mixed_plastic_soft"
    ]
)


mr_to_ms = int(
    cm[
        MR_IDX,
        MS_IDX
    ]
)


ms_to_mr = int(
    cm[
        MS_IDX,
        MR_IDX
    ]
)


print()
print(
    f"MR -> MS errors : "
    f"{mr_to_ms:,}"
)


print(
    f"MS -> MR errors : "
    f"{ms_to_mr:,}"
)


# ==================================================================================================
# 38. SAVE RESULTS
# ==================================================================================================

results = {

    "experiment":
        "E21-A",

    "description":
        (
            "Six-class ConvNeXt-Tiny hard-negative specialist "
            "for Mixed Rigid / Mixed Soft discrimination"
        ),

    "test_set_used":
        False,

    "classes":
        E21_CLASS_NAMES,

    "hard_negative_score":
        "P(MR) + P(MS) from E3Y-B 7-class MobileNet",

    "negative_ratio":
        NEGATIVE_RATIO,

    "best_epoch":
        int(
            best_epoch
        ),

    "best_val_macro_f1":
        float(
            best_macro_f1
        ),

    "final_val_accuracy":
        float(
            final_val[
                "accuracy"
            ]
        ),

    "final_val_macro_f1":
        float(
            final_val[
                "macro_f1"
            ]
        ),

    "mixed_rigid_metrics":
        mr_metrics,

    "mixed_soft_metrics":
        ms_metrics,

    "mr_ms_mean_f1":
        float(
            target_mean_f1
        ),

    "mr_ms_mean_recall":
        float(
            target_mean_recall
        ),

    "mr_to_ms":
        int(
            mr_to_ms
        ),

    "ms_to_mr":
        int(
            ms_to_mr
        ),

    "classification_report":
        report_dict,

    "confusion_matrix":
        cm.tolist(),

    "train_distribution":
        dict(
            train_counts
        ),

    "validation_distribution":
        dict(
            val_counts
        ),
}


with open(
    RESULTS_PATH,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        results,
        f,
        indent=2
    )


cm_df.to_csv(
    CONFUSION_CSV_PATH
)


# ==================================================================================================
# 39. COMPLETE
# ==================================================================================================

print()
print("=" * 110)
print("E21-A COMPLETE")
print("=" * 110)


print()
print(
    f"Best checkpoint : "
    f"{BEST_MODEL_PATH}"
)


print(
    f"Train manifest  : "
    f"{TRAIN_MANIFEST_PATH}"
)


print(
    f"Val manifest    : "
    f"{VAL_MANIFEST_PATH}"
)


print(
    f"Training history: "
    f"{HISTORY_PATH}"
)


print(
    f"Results         : "
    f"{RESULTS_PATH}"
)


print(
    f"Confusion CSV   : "
    f"{CONFUSION_CSV_PATH}"
)

E21-A — MIXED-PLASTIC HARD-NEGATIVE SPECIALIST

PyTorch : 2.13.0+cu126
Device  : cuda
GPU     : NVIDIA GeForce RTX 3050 Ti Laptop GPU

Train crops : C:\Users\varda\Documents\_My Computer\COMMON_space\Learning\Upgrad_EPGP_MS\upgrad_IIITB_MS_course\MS_LJMU_Material\Topic Data\SortWaste\dataset\dataset\cropped_data\train
Val crops   : C:\Users\varda\Documents\_My Computer\COMMON_space\Learning\Upgrad_EPGP_MS\upgrad_IIITB_MS_course\MS_LJMU_Material\Topic Data\SortWaste\dataset\dataset\cropped_data\val
MobileNet   : C:\Users\varda\Documents\_My Computer\COMMON_space\Learning\Upgrad_EPGP_MS\upgrad_IIITB_MS_course\MS_LJMU_Material\Topic Data\SortWaste\dataset\dataset\yolo_mobilenet_crops_E3Y\mobilenet_results\E3Y_B_class_weighted\E3Y_B_MobileNetV3Large_best.pth
Output      : C:\Users\varda\Documents\_My Computer\COMMON_space\Learning\Upgrad_EPGP_MS\upgrad_IIITB_MS_course\MS_LJMU_Material\Thesis_Code\runs\sortwaste\E21A_mixedplastic_hardnegative_convnext

E3Y-B MobileNet mapping:
  0: ecal
  1

Mining hdpe: 100%|██████████| 263/263 [01:03<00:00,  4.15it/s]


Selected : 6,054 / 16,803
Selected hardness range : 0.0000 to 1.0000

MINING HARD NEGATIVES — PET


Mining pet: 100%|██████████| 188/188 [00:48<00:00,  3.88it/s]


Selected : 6,054 / 11,976
Selected hardness range : 0.0000 to 0.9942

MINING HARD NEGATIVES — ECAL


...

Mining non_plastic: 100%|██████████| 39/39 [00:09<00:00,  3.95it/s]


Selected : 2,469 / 2,469

FINAL E21-A TRAIN DISTRIBUTION
mixed_plastic_rigid      : 7,066
mixed_plastic_soft       : 9,077
hdpe                     : 6,054
pet                      : 6,054
ecal                     : 6,054
non_plastic              : 2,469

Total train samples : 36,774

E21-A VALIDATION DISTRIBUTION
mixed_plastic_rigid      : 1,120
mixed_plastic_soft       : 1,443
hdpe                     : 4,972
pet                      : 2,108
ecal                     : 2,552
non_plastic              : 702

Total validation samples : 12,897

Training batches   : 1,533
Validation batches : 538

BUILDING E21-A CONVNEXT-TINY
Output classes : 6
Parameters     : 27,824,742

Loss weights:
mixed_plastic_rigid      : 0.8804
mixed_plastic_soft       : 0.7768
hdpe                     : 0.9511
pet                      : 0.9511
ecal                     : 0.9511
non_plastic              : 1.4894

TRAINING E21-A



Epoch 01/15
  Train loss     : 0.4996
  Train accuracy : 80.99%
  Train macro-F1 : 0.8088
  Val loss       : 0.7749
  Val accuracy   : 76.65%
  Val macro-F1   : 0.7157
  LR             : 1.00e-04
  Time           : 198.22 min
  *** BEST CHECKPOINT SAVED ***



Epoch 02/15
  Train loss     : 0.1580
  Train accuracy : 94.37%
  Train macro-F1 : 0.9434
  Val loss       : 1.0315
  Val accuracy   : 72.97%
  Val macro-F1   : 0.6833
  LR             : 1.00e-04
  Time           : 201.27 min


KeyboardInterrupt: 

In [ ]:
# E21-A — STANDALONE VALIDATION OF SAVED BEST CHECKPOINT
#
# Loads:
#   E21A_ConvNeXtTiny_best.pth
#
# Evaluates:
#   full 6-class validation set
#
# Reports:
#   accuracy
#   macro-F1
#   per-class precision / recall / F1
#   confusion matrix
#   MR -> MS
#   MS -> MR
#
# TEST SET IS NOT USED.
# ==================================================================================================

import json
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd

import torch
import torch.nn as nn

from torch.utils.data import (
    Dataset,
    DataLoader,
)

from torchvision import models
from torchvision.transforms import v2

from PIL import Image

from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score,
    f1_score,
)

from tqdm.auto import tqdm


# ==================================================================================================
# 1. DEVICE
# ==================================================================================================

DEVICE = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

print("=" * 110)
print("E21-A — SAVED CHECKPOINT VALIDATION")
print("=" * 110)

print()
print(f"PyTorch : {torch.__version__}")
print(f"Device  : {DEVICE}")

if DEVICE.type == "cuda":
    print(
        f"GPU     : "
        f"{torch.cuda.get_device_name(0)}"
    )


# ==================================================================================================
# 2. PATHS
# ==================================================================================================

BASE = Path(
    r"C:\Users\varda\Documents\_My Computer\COMMON_space\Learning"
    r"\Upgrad_EPGP_MS\upgrad_IIITB_MS_course\MS_LJMU_Material"
)

THESIS_CODE = (
    BASE
    / "Thesis_Code"
)

OUTPUT_DIR = (
    THESIS_CODE
    / "runs"
    / "sortwaste"
    / "E21A_mixedplastic_hardnegative_convnext"
)

BEST_MODEL_PATH = (
    OUTPUT_DIR
    / "E21A_ConvNeXtTiny_best.pth"
)

VAL_MANIFEST_PATH = (
    OUTPUT_DIR
    / "E21A_validation_manifest.json"
)

VALIDATION_RESULTS_PATH = (
    OUTPUT_DIR
    / "E21A_saved_checkpoint_validation_results.json"
)

CONFUSION_CSV_PATH = (
    OUTPUT_DIR
    / "E21A_saved_checkpoint_validation_confusion.csv"
)


# ==================================================================================================
# 3. VERIFY FILES
# ==================================================================================================

for path in [
    BEST_MODEL_PATH,
    VAL_MANIFEST_PATH,
]:

    if not path.exists():
        raise FileNotFoundError(
            f"Required file not found:\n{path}"
        )


print()
print(
    f"Checkpoint : "
    f"{BEST_MODEL_PATH}"
)

print(
    f"Val manifest: "
    f"{VAL_MANIFEST_PATH}"
)


# ==================================================================================================
# 4. LOAD CHECKPOINT
# ==================================================================================================

checkpoint = torch.load(
    BEST_MODEL_PATH,
    map_location=DEVICE
)


print()
print("=" * 110)
print("CHECKPOINT INFORMATION")
print("=" * 110)

print(
    f"Experiment   : "
    f"{checkpoint.get('experiment', 'unknown')}"
)

print(
    f"Saved epoch  : "
    f"{checkpoint.get('epoch', 'unknown')}"
)

print(
    f"Saved val F1 : "
    f"{checkpoint.get('val_macro_f1', 'unknown')}"
)


# ==================================================================================================
# 5. CLASS MAPPING
# ==================================================================================================

E21_CLASS_NAMES = checkpoint.get(
    "class_names",
    [
        "mixed_plastic_rigid",
        "mixed_plastic_soft",
        "hdpe",
        "pet",
        "ecal",
        "non_plastic",
    ]
)

NUM_CLASSES = len(
    E21_CLASS_NAMES
)

E21_NAME_TO_IDX = {
    name: idx
    for idx, name
    in enumerate(
        E21_CLASS_NAMES
    )
}


print()
print("Class mapping:")

for idx, name in enumerate(
    E21_CLASS_NAMES
):
    print(
        f"  {idx}: {name}"
    )


# ==================================================================================================
# 6. LOAD VALIDATION MANIFEST
# ==================================================================================================

with open(
    VAL_MANIFEST_PATH,
    "r",
    encoding="utf-8"
) as f:

    val_manifest = json.load(f)


val_counts = Counter(
    item["class_name"]
    for item in val_manifest
)


print()
print("=" * 110)
print("VALIDATION DISTRIBUTION")
print("=" * 110)

for class_name in E21_CLASS_NAMES:

    print(
        f"{class_name:25s}: "
        f"{val_counts[class_name]:,}"
    )

print(
    f"\nTotal validation samples: "
    f"{len(val_manifest):,}"
)


# ==================================================================================================
# 7. VALIDATION TRANSFORM
# ==================================================================================================

IMAGE_SIZE = int(
    checkpoint.get(
        "image_size",
        224
    )
)


val_transform = v2.Compose(
    [
        v2.Resize(
            (
                IMAGE_SIZE,
                IMAGE_SIZE
            ),
            antialias=True
        ),

        v2.ToImage(),

        v2.ToDtype(
            torch.float32,
            scale=True
        ),

        v2.Normalize(
            mean=[
                0.485,
                0.456,
                0.406
            ],
            std=[
                0.229,
                0.224,
                0.225
            ]
        ),
    ]
)


# ==================================================================================================
# 8. DATASET
# ==================================================================================================

class ManifestDataset(
    Dataset
):

    def __init__(
        self,
        manifest,
        transform
    ):

        self.manifest = manifest
        self.transform = transform


    def __len__(
        self
    ):

        return len(
            self.manifest
        )


    def __getitem__(
        self,
        idx
    ):

        item = (
            self.manifest[
                idx
            ]
        )


        with Image.open(
            item[
                "path"
            ]
        ) as img:

            image = img.convert(
                "RGB"
            )


        image = self.transform(
            image
        )


        label = int(
            item[
                "class_idx"
            ]
        )


        return (
            image,
            label
        )


val_dataset = ManifestDataset(
    val_manifest,
    val_transform
)


# ==================================================================================================
# 9. DATA LOADER
# ==================================================================================================

BATCH_SIZE = 32

NUM_WORKERS = 0


val_loader = DataLoader(

    val_dataset,

    batch_size=BATCH_SIZE,

    shuffle=False,

    num_workers=NUM_WORKERS,

    pin_memory=(
        DEVICE.type
        == "cuda"
    )
)


print()
print(
    f"Validation batches: "
    f"{len(val_loader):,}"
)


# ==================================================================================================
# 10. BUILD CONVNEXT-TINY
# ==================================================================================================

print()
print("=" * 110)
print("LOADING E21-A CONVNEXT-TINY")
print("=" * 110)


model = models.convnext_tiny(
    weights=None
)


in_features = (
    model.classifier[
        2
    ].in_features
)


model.classifier[
    2
] = nn.Linear(
    in_features,
    NUM_CLASSES
)


state_dict = checkpoint[
    "model_state_dict"
]


model.load_state_dict(
    state_dict,
    strict=True
)


model = model.to(
    DEVICE
)

model.eval()


print(
    "E21-A checkpoint loaded successfully."
)


# ==================================================================================================
# 11. RUN VALIDATION
# ==================================================================================================

all_true = []

all_pred = []

all_probs = []


print()
print("=" * 110)
print("RUNNING VALIDATION")
print("=" * 110)


with torch.inference_mode():

    for images, labels in tqdm(
        val_loader,
        desc="Validation"
    ):

        images = images.to(
            DEVICE,
            non_blocking=True
        )


        logits = model(
            images
        )


        probs = torch.softmax(
            logits,
            dim=1
        )


        preds = torch.argmax(
            probs,
            dim=1
        )


        all_true.extend(
            labels.numpy()
            .tolist()
        )


        all_pred.extend(
            preds.cpu()
            .numpy()
            .tolist()
        )


        all_probs.extend(
            probs.cpu()
            .numpy()
            .tolist()
        )


y_true = np.asarray(
    all_true
)

y_pred = np.asarray(
    all_pred
)


# ==================================================================================================
# 12. OVERALL METRICS
# ==================================================================================================

accuracy = accuracy_score(
    y_true,
    y_pred
)


macro_f1 = f1_score(
    y_true,
    y_pred,
    average="macro",
    zero_division=0
)


print()
print("=" * 110)
print("E21-A — FINAL STANDALONE VALIDATION RESULTS")
print("=" * 110)

print()
print(
    f"Accuracy : "
    f"{accuracy * 100:.2f}%"
)

print(
    f"Macro-F1 : "
    f"{macro_f1:.4f}"
)


# ==================================================================================================
# 13. CLASSIFICATION REPORT
# ==================================================================================================

labels = list(
    range(
        NUM_CLASSES
    )
)


report_dict = classification_report(

    y_true,

    y_pred,

    labels=labels,

    target_names=E21_CLASS_NAMES,

    output_dict=True,

    zero_division=0
)


report_text = classification_report(

    y_true,

    y_pred,

    labels=labels,

    target_names=E21_CLASS_NAMES,

    zero_division=0
)


print()
print("=" * 110)
print("CLASSIFICATION REPORT")
print("=" * 110)

print()
print(
    report_text
)


# ==================================================================================================
# 14. CONFUSION MATRIX
# ==================================================================================================

cm = confusion_matrix(

    y_true,

    y_pred,

    labels=labels
)


cm_df = pd.DataFrame(

    cm,

    index=[
        f"GT: {name}"
        for name
        in E21_CLASS_NAMES
    ],

    columns=[
        f"Pred: {name}"
        for name
        in E21_CLASS_NAMES
    ]
)


print()
print("=" * 110)
print("E21-A — VALIDATION CONFUSION MATRIX")
print("=" * 110)

print()
print(
    cm_df.to_string()
)


# ==================================================================================================
# 15. ROW-NORMALIZED CONFUSION MATRIX
# ==================================================================================================

row_totals = cm.sum(
    axis=1,
    keepdims=True
)


cm_normalized = np.divide(

    cm,

    row_totals,

    out=np.zeros_like(
        cm,
        dtype=float
    ),

    where=(
        row_totals != 0
    )
)


cm_normalized_df = pd.DataFrame(

    cm_normalized * 100.0,

    index=[
        f"GT: {name}"
        for name
        in E21_CLASS_NAMES
    ],

    columns=[
        f"Pred: {name}"
        for name
        in E21_CLASS_NAMES
    ]
)


print()
print("=" * 110)
print("ROW-NORMALIZED CONFUSION MATRIX (%)")
print("=" * 110)

print()
print(
    cm_normalized_df
    .round(2)
    .to_string()
)


# ==================================================================================================
# 16. MR / MS SUMMARY
# ==================================================================================================

MR_IDX = E21_NAME_TO_IDX[
    "mixed_plastic_rigid"
]

MS_IDX = E21_NAME_TO_IDX[
    "mixed_plastic_soft"
]


mr_metrics = report_dict[
    "mixed_plastic_rigid"
]

ms_metrics = report_dict[
    "mixed_plastic_soft"
]


mr_ms_mean_f1 = (

    mr_metrics[
        "f1-score"
    ]

    +

    ms_metrics[
        "f1-score"
    ]

) / 2.0


mr_ms_mean_recall = (

    mr_metrics[
        "recall"
    ]

    +

    ms_metrics[
        "recall"
    ]

) / 2.0


mr_to_ms = int(
    cm[
        MR_IDX,
        MS_IDX
    ]
)


ms_to_mr = int(
    cm[
        MS_IDX,
        MR_IDX
    ]
)


print()
print("=" * 110)
print("E21-A — MR / MS SPECIALIST SUMMARY")
print("=" * 110)


print()
print(
    f"Mixed Rigid precision : "
    f"{mr_metrics['precision']:.4f}"
)

print(
    f"Mixed Rigid recall    : "
    f"{mr_metrics['recall']:.4f}"
)

print(
    f"Mixed Rigid F1        : "
    f"{mr_metrics['f1-score']:.4f}"
)


print()
print(
    f"Mixed Soft precision  : "
    f"{ms_metrics['precision']:.4f}"
)

print(
    f"Mixed Soft recall     : "
    f"{ms_metrics['recall']:.4f}"
)

print(
    f"Mixed Soft F1         : "
    f"{ms_metrics['f1-score']:.4f}"
)


print()
print(
    f"MR/MS mean F1         : "
    f"{mr_ms_mean_f1:.4f}"
)

print(
    f"MR/MS mean recall     : "
    f"{mr_ms_mean_recall:.4f}"
)


print()
print(
    f"MR -> MS errors       : "
    f"{mr_to_ms:,}"
)

print(
    f"MS -> MR errors       : "
    f"{ms_to_mr:,}"
)


# ==================================================================================================
# 17. SAVE RESULTS
# ==================================================================================================

results = {

    "experiment":
        "E21-A standalone checkpoint validation",

    "checkpoint_epoch":
        checkpoint.get(
            "epoch"
        ),

    "test_set_used":
        False,

    "accuracy":
        float(
            accuracy
        ),

    "macro_f1":
        float(
            macro_f1
        ),

    "mr_metrics":
        mr_metrics,

    "ms_metrics":
        ms_metrics,

    "mr_ms_mean_f1":
        float(
            mr_ms_mean_f1
        ),

    "mr_ms_mean_recall":
        float(
            mr_ms_mean_recall
        ),

    "mr_to_ms":
        int(
            mr_to_ms
        ),

    "ms_to_mr":
        int(
            ms_to_mr
        ),

    "classification_report":
        report_dict,

    "confusion_matrix":
        cm.tolist(),

    "class_names":
        E21_CLASS_NAMES,
}


with open(
    VALIDATION_RESULTS_PATH,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        results,
        f,
        indent=2
    )


cm_df.to_csv(
    CONFUSION_CSV_PATH
)


cm_normalized_df.to_csv(
    OUTPUT_DIR
    / "E21A_saved_checkpoint_confusion_normalized.csv"
)


print()
print("=" * 110)
print("E21-A CHECKPOINT VALIDATION COMPLETE")
print("=" * 110)

print()
print(
    f"Results       : "
    f"{VALIDATION_RESULTS_PATH}"
)

print(
    f"Confusion CSV : "
    f"{CONFUSION_CSV_PATH}"
)

E21-A — SAVED CHECKPOINT VALIDATION

PyTorch : 2.13.0+cu126
Device  : cuda
GPU     : NVIDIA GeForce RTX 3050 Ti Laptop GPU

Checkpoint : C:\Users\varda\Documents\_My Computer\COMMON_space\Learning\Upgrad_EPGP_MS\upgrad_IIITB_MS_course\MS_LJMU_Material\Thesis_Code\runs\sortwaste\E21A_mixedplastic_hardnegative_convnext\E21A_ConvNeXtTiny_best.pth
Val manifest: C:\Users\varda\Documents\_My Computer\COMMON_space\Learning\Upgrad_EPGP_MS\upgrad_IIITB_MS_course\MS_LJMU_Material\Thesis_Code\runs\sortwaste\E21A_mixedplastic_hardnegative_convnext\E21A_validation_manifest.json

CHECKPOINT INFORMATION
Experiment   : E21-A
Saved epoch  : 1
Saved val F1 : 0.7156981344186176

Class mapping:
  0: mixed_plastic_rigid
  1: mixed_plastic_soft
  2: hdpe
  3: pet
  4: ecal
  5: non_plastic

VALIDATION DISTRIBUTION
mixed_plastic_rigid      : 1,120
mixed_plastic_soft       : 1,443
hdpe                     : 4,972
pet                      : 2,108
ecal                     : 2,552
non_plastic              : 702


Validation: 100%|██████████| 404/404 [04:26<00:00,  1.51it/s]


E21-A — FINAL STANDALONE VALIDATION RESULTS

Accuracy : 76.65%
Macro-F1 : 0.7157

CLASSIFICATION REPORT

                     precision    recall  f1-score   support

mixed_plastic_rigid       0.56      0.70      0.62      1120
 mixed_plastic_soft       0.55      0.75      0.63      1443
               hdpe       0.92      0.73      0.82      4972
                pet       0.89      0.85      0.87      2108
               ecal       0.75      0.88      0.81      2552
        non_plastic       0.61      0.50      0.55       702

           accuracy                           0.77     12897
          macro avg       0.71      0.73      0.72     12897
       weighted avg       0.79      0.77      0.77     12897


E21-A — VALIDATION CONFUSION MATRIX

                         Pred: mixed_plastic_rigid  Pred: mixed_plastic_soft  Pred: hdpe  Pred: pet  Pred: ecal  Pred: non_plastic
GT: mixed_plastic_rigid                        784                       214          80         23           5 

In [ ]:
# E21-B — End-to-End Validation Integration
# ==================================================================================================
# E21-B — END-TO-END VALIDATION INTEGRATION OF E21-A HARD-NEGATIVE SPECIALIST
#
# PURPOSE
# --------------------------------------------------------------------------------------------------
# Integrate the trained E21-A 6-class hard-negative ConvNeXt-Tiny specialist into
# the frozen E20-A validation pipeline.
#
# VALIDATION ONLY.
# TEST SET MUST NOT BE USED HERE.
#
#
# FROZEN E20-A BASE
# --------------------------------------------------------------------------------------------------
# Detector                  : E12 YOLO11m @640
# MobileNet                 : E3Y-B
# E16-A MobileNet routing   : frozen
# E18-B ConvNeXt-Tiny       : frozen
# E20-A gates:
#       Mixed Rigid         : OFF
#       Mixed Soft          : 0.85
#       non_plastic         : 0.90
#       PET Oil             : 0.94
# alpha                     : 0.70
#
#
# NEW COMPONENT
# --------------------------------------------------------------------------------------------------
# E21-A ConvNeXt-Tiny hard-negative specialist:
#
#   0 mixed_plastic_rigid
#   1 mixed_plastic_soft
#   2 hdpe
#   3 pet
#   4 ecal
#   5 non_plastic
#
#
# IMPORTANT ROUTING RULE
# --------------------------------------------------------------------------------------------------
# E21-A may ONLY override E20-A when E21-A proposes:
#
#   mixed_plastic_rigid
#   mixed_plastic_soft
#
# If E21-A proposes:
#
#   HDPE
#   PET
#   ECAL
#   non_plastic
#
# the original E20-A prediction is retained.
#
#
# SEARCH
# --------------------------------------------------------------------------------------------------
# MR confidence threshold
# MS confidence threshold
#
# Primary objective:
#   six-plastic mean AP50
#
# Secondary:
#   mean(MR AP50, MS AP50)
#
# Tie-break:
#   overall 7-class AP50-95
#
#
# DESIGN
# --------------------------------------------------------------------------------------------------
# Stage 1:
#   Run E21-A ONCE on all cached E18-D detector crops.
#
# Stage 2:
#   Save E21 probabilities into a new validation cache.
#
# Stage 3:
#   Perform gate search entirely from cache.
#
# ==================================================================================================

import json
import time
from pathlib import Path
from collections import Counter, defaultdict

import numpy as np

import torch
import torch.nn as nn

from torchvision import models
from torchvision.transforms import v2

from PIL import Image

from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval

from tqdm.auto import tqdm


# ==================================================================================================
# 1. DEVICE
# ==================================================================================================

DEVICE = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)


print("=" * 115)
print("E21-B — END-TO-END VALIDATION INTEGRATION")
print("=" * 115)

print()
print(f"PyTorch : {torch.__version__}")
print(f"Device  : {DEVICE}")

if DEVICE.type == "cuda":

    print(
        f"GPU     : "
        f"{torch.cuda.get_device_name(0)}"
    )


# ==================================================================================================
# 2. PATHS
# ==================================================================================================

BASE = Path(
    r"C:\Users\varda\Documents\_My Computer\COMMON_space\Learning"
    r"\Upgrad_EPGP_MS\upgrad_IIITB_MS_course\MS_LJMU_Material"
)


DATASET_ROOT = (
    BASE
    / "Topic Data"
    / "SortWaste"
    / "dataset"
    / "dataset"
)


THESIS_CODE = (
    BASE
    / "Thesis_Code"
)


# --------------------------------------------------------------------------------------------------
# Validation images
# --------------------------------------------------------------------------------------------------

VAL_IMAGES_DIR = (
    DATASET_ROOT
    / "splited_all_dataset_coco"
    / "val"
    / "images"
)


# --------------------------------------------------------------------------------------------------
# E18-D cache + generated 7-class validation GT
# --------------------------------------------------------------------------------------------------

E18D_DIR = (
    THESIS_CODE
    / "runs"
    / "sortwaste"
    / "E18D_class_selective_convnext"
)


E18D_CACHE_PATH = (
    E18D_DIR
    / "E18D_cached_predictions.json"
)


GT_PATH = (
    E18D_DIR
    / "E18D_val_gt_7class.json"
)


# --------------------------------------------------------------------------------------------------
# E21-A checkpoint
# --------------------------------------------------------------------------------------------------

E21A_DIR = (
    THESIS_CODE
    / "runs"
    / "sortwaste"
    / "E21A_mixedplastic_hardnegative_convnext"
)


E21A_CKPT = (
    E21A_DIR
    / "E21A_ConvNeXtTiny_best.pth"
)


# --------------------------------------------------------------------------------------------------
# E21-B output
# --------------------------------------------------------------------------------------------------

OUTPUT_DIR = (
    THESIS_CODE
    / "runs"
    / "sortwaste"
    / "E21B_end_to_end_validation"
)


OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)


E21_CACHE_PATH = (
    OUTPUT_DIR
    / "E21B_cached_predictions_with_E21A.json"
)


GRID_RESULTS_PATH = (
    OUTPUT_DIR
    / "E21B_grid_results.json"
)


BEST_CONFIG_PATH = (
    OUTPUT_DIR
    / "E21B_best_configuration.json"
)


BEST_PREDICTIONS_PATH = (
    OUTPUT_DIR
    / "E21B_best_predictions.json"
)


SUMMARY_PATH = (
    OUTPUT_DIR
    / "E21B_summary.txt"
)


# ==================================================================================================
# 3. VERIFY REQUIRED FILES
# ==================================================================================================

required_paths = [

    VAL_IMAGES_DIR,

    E18D_CACHE_PATH,

    GT_PATH,

    E21A_CKPT,
]


for path in required_paths:

    if not path.exists():

        raise FileNotFoundError(
            f"Required path not found:\n{path}"
        )


print()
print(f"Validation images : {VAL_IMAGES_DIR}")
print(f"E18-D cache       : {E18D_CACHE_PATH}")
print(f"Validation GT     : {GT_PATH}")
print(f"E21-A checkpoint  : {E21A_CKPT}")
print(f"Output            : {OUTPUT_DIR}")


# ==================================================================================================
# 4. 7-CLASS GLOBAL MODEL MAPPING
# ==================================================================================================

CLASS_NAMES = [

    "ecal",

    "hdpe",

    "mixed_plastic_rigid",

    "mixed_plastic_soft",

    "non_plastic",

    "pet",

    "pet_oil",
]


ECAL = 0
HDPE = 1
MR = 2
MS = 3
NP = 4
PET = 5
PET_OIL = 6


PLASTIC_CLASS_INDICES = [

    ECAL,

    HDPE,

    MR,

    MS,

    PET,

    PET_OIL,
]


# ==================================================================================================
# 5. E21-A LOCAL MAPPING
# ==================================================================================================

E21_CLASS_NAMES = [

    "mixed_plastic_rigid",

    "mixed_plastic_soft",

    "hdpe",

    "pet",

    "ecal",

    "non_plastic",
]


E21_MR = 0
E21_MS = 1
E21_HDPE = 2
E21_PET = 3
E21_ECAL = 4
E21_NP = 5


E21_TO_GLOBAL = {

    E21_MR:
        MR,

    E21_MS:
        MS,

    E21_HDPE:
        HDPE,

    E21_PET:
        PET,

    E21_ECAL:
        ECAL,

    E21_NP:
        NP,
}


# ==================================================================================================
# 6. FROZEN E20-A CONFIGURATION
# ==================================================================================================

ALPHA = 0.70


E20_MR_GATE = None

E20_MS_GATE = 0.85

E20_NP_GATE = 0.90

E20_PETOIL_GATE = 0.94


# ==================================================================================================
# 7. E21-B SEARCH GRID
#
# None = E21 override disabled for that class
# ==================================================================================================

MR_GATES = [

    None,

    0.80,

    0.85,

    0.88,

    0.90,

    0.92,

    0.94,

    0.96,

    0.97,

    0.98,

    0.99,
]


MS_GATES = [

    None,

    0.80,

    0.85,

    0.88,

    0.90,

    0.92,

    0.94,

    0.96,

    0.97,

    0.98,

    0.99,
]


# ==================================================================================================
# 8. LOAD E18-D CACHE
# ==================================================================================================

with open(
    E18D_CACHE_PATH,
    "r",
    encoding="utf-8"
) as f:

    base_cache = json.load(f)


print()
print(
    f"E18-D cached detections : "
    f"{len(base_cache):,}"
)


# ==================================================================================================
# 9. LOAD VALIDATION GT
# ==================================================================================================

with open(
    GT_PATH,
    "r",
    encoding="utf-8"
) as f:

    gt_json = json.load(f)


image_id_to_filename = {

    int(image[
        "id"
    ]):
        image[
            "file_name"
        ]

    for image in gt_json[
        "images"
    ]
}


print(
    f"Validation images in GT : "
    f"{len(image_id_to_filename):,}"
)


# ==================================================================================================
# 10. LOAD E21-A CHECKPOINT
# ==================================================================================================

print()
print("=" * 115)
print("LOADING E21-A SPECIALIST")
print("=" * 115)


checkpoint = torch.load(
    E21A_CKPT,
    map_location=DEVICE
)


checkpoint_classes = checkpoint.get(
    "class_names",
    E21_CLASS_NAMES
)


if checkpoint_classes != E21_CLASS_NAMES:

    print()
    print(
        "Checkpoint class names:"
    )

    print(
        checkpoint_classes
    )

    raise RuntimeError(
        "E21-A checkpoint class mapping does not match expected mapping."
    )


model = models.convnext_tiny(
    weights=None
)


in_features = (
    model.classifier[
        2
    ].in_features
)


model.classifier[
    2
] = nn.Linear(
    in_features,
    6
)


model.load_state_dict(
    checkpoint[
        "model_state_dict"
    ],
    strict=True
)


model = model.to(
    DEVICE
)


model.eval()


print(
    f"E21-A epoch : "
    f"{checkpoint.get('epoch')}"
)

print(
    f"E21-A val F1: "
    f"{checkpoint.get('val_macro_f1')}"
)

print(
    "E21-A loaded successfully."
)


# ==================================================================================================
# 11. E21-A TRANSFORM
# ==================================================================================================

IMAGE_SIZE = int(
    checkpoint.get(
        "image_size",
        224
    )
)


e21_transform = v2.Compose(
    [

        v2.Resize(
            (
                IMAGE_SIZE,
                IMAGE_SIZE
            ),
            antialias=True
        ),

        v2.ToImage(),

        v2.ToDtype(
            torch.float32,
            scale=True
        ),

        v2.Normalize(
            mean=[
                0.485,
                0.456,
                0.406
            ],
            std=[
                0.229,
                0.224,
                0.225
            ]
        ),
    ]
)


# ==================================================================================================
# 12. GENERATE E21-A CACHE
#
# If already created, skip expensive inference.
# ==================================================================================================

if E21_CACHE_PATH.exists():

    print()
    print("=" * 115)
    print("FOUND EXISTING E21-B CACHE — SKIPPING E21-A INFERENCE")
    print("=" * 115)


    with open(
        E21_CACHE_PATH,
        "r",
        encoding="utf-8"
    ) as f:

        cache = json.load(f)


    print(
        f"Loaded cached entries : "
        f"{len(cache):,}"
    )


else:

    print()
    print("=" * 115)
    print("RUNNING E21-A ON E20-A VALIDATION DETECTOR CROPS")
    print("=" * 115)


    # ----------------------------------------------------------------------------------------------
    # Group cache entries by image so each validation image is loaded only once
    # ----------------------------------------------------------------------------------------------

    entries_by_image = defaultdict(
        list
    )


    for idx, item in enumerate(
        base_cache
    ):

        entries_by_image[
            int(
                item[
                    "image_id"
                ]
            )
        ].append(
            (
                idx,
                item
            )
        )


    # Copy records
    cache = [

        dict(
            item
        )

        for item
        in base_cache
    ]


    # Initialize
    for item in cache:

        item[
            "e21"
        ] = None


    start_time = (
        time.time()
    )


    processed = 0

    invalid_crops = 0


    with torch.inference_mode():

        for image_number, (
            image_id,
            image_entries
        ) in enumerate(

            tqdm(
                list(
                    entries_by_image.items()
                ),
                desc="E21-A validation inference"
            ),

            start=1
        ):

            filename = (
                image_id_to_filename[
                    image_id
                ]
            )


            image_path = (
                VAL_IMAGES_DIR
                / filename
            )


            if not image_path.exists():

                raise FileNotFoundError(
                    f"Validation image not found:\n{image_path}"
                )


            with Image.open(
                image_path
            ) as img:

                full_image = img.convert(
                    "RGB"
                )


            image_width, image_height = (
                full_image.size
            )


            # --------------------------------------------------------------------------------------
            # Build all valid crops for this image
            # --------------------------------------------------------------------------------------

            crop_tensors = []

            crop_meta = []


            for cache_index, item in (
                image_entries
            ):

                x1, y1, x2, y2 = [

                    float(v)

                    for v
                    in item[
                        "bbox"
                    ]
                ]


                # Clamp to image boundaries
                x1 = max(
                    0.0,
                    min(
                        x1,
                        image_width
                    )
                )

                y1 = max(
                    0.0,
                    min(
                        y1,
                        image_height
                    )
                )

                x2 = max(
                    0.0,
                    min(
                        x2,
                        image_width
                    )
                )

                y2 = max(
                    0.0,
                    min(
                        y2,
                        image_height
                    )
                )


                left = int(
                    np.floor(
                        x1
                    )
                )

                top = int(
                    np.floor(
                        y1
                    )
                )

                right = int(
                    np.ceil(
                        x2
                    )
                )

                bottom = int(
                    np.ceil(
                        y2
                    )
                )


                if (
                    right <= left
                    or bottom <= top
                ):

                    invalid_crops += 1

                    continue


                crop = full_image.crop(
                    (
                        left,
                        top,
                        right,
                        bottom
                    )
                )


                tensor = e21_transform(
                    crop
                )


                crop_tensors.append(
                    tensor
                )


                crop_meta.append(
                    cache_index
                )


            # --------------------------------------------------------------------------------------
            # Batch per image to avoid excessive GPU memory
            # --------------------------------------------------------------------------------------

            if len(
                crop_tensors
            ) == 0:

                continue


            batch_size = 32


            for start in range(
                0,
                len(
                    crop_tensors
                ),
                batch_size
            ):

                batch_tensors = torch.stack(

                    crop_tensors[
                        start:
                        start + batch_size
                    ]
                ).to(
                    DEVICE,
                    non_blocking=True
                )


                logits = model(
                    batch_tensors
                )


                probs = torch.softmax(
                    logits,
                    dim=1
                )


                top_probs, top_indices = torch.max(
                    probs,
                    dim=1
                )


                probs_np = (
                    probs.cpu()
                    .numpy()
                )


                top_probs_np = (
                    top_probs.cpu()
                    .numpy()
                )


                top_indices_np = (
                    top_indices.cpu()
                    .numpy()
                )


                meta_slice = crop_meta[
                    start:
                    start + batch_size
                ]


                for local_idx, cache_index in enumerate(
                    meta_slice
                ):

                    local_class_idx = int(
                        top_indices_np[
                            local_idx
                        ]
                    )


                    global_class_idx = int(
                        E21_TO_GLOBAL[
                            local_class_idx
                        ]
                    )


                    cache[
                        cache_index
                    ][
                        "e21"
                    ] = {

                        "probs":
                            [
                                float(x)
                                for x
                                in probs_np[
                                    local_idx
                                ]
                            ],

                        "local_idx":
                            local_class_idx,

                        "global_idx":
                            global_class_idx,

                        "top_prob":
                            float(
                                top_probs_np[
                                    local_idx
                                ]
                            ),
                    }


                    processed += 1


    elapsed = (
        time.time()
        - start_time
    )


    print()
    print(
        f"E21 crops processed : "
        f"{processed:,}"
    )

    print(
        f"Invalid crops       : "
        f"{invalid_crops:,}"
    )

    print(
        f"Inference time      : "
        f"{elapsed / 60:.2f} min"
    )


    with open(
        E21_CACHE_PATH,
        "w",
        encoding="utf-8"
    ) as f:

        json.dump(
            cache,
            f
        )


    print()
    print(
        f"E21 cache saved:\n"
        f"{E21_CACHE_PATH}"
    )


# ==================================================================================================
# 13. CACHE SANITY
# ==================================================================================================

num_e21 = sum(

    1

    for item
    in cache

    if item.get(
        "e21"
    ) is not None
)


print()
print(
    f"Cache entries with E21 output : "
    f"{num_e21:,} / {len(cache):,}"
)


# ==================================================================================================
# 14. RECONSTRUCT FROZEN E20-A BASE
# ==================================================================================================

def reconstruct_e20a(
    item
):

    """
    Returns:
        final_idx
        classifier_prob
        source
    """

    e16_final_idx = int(
        item[
            "e16_final_idx"
        ]
    )


    mn_probs = np.asarray(
        item[
            "mn_probs"
        ],
        dtype=np.float32
    )


    final_idx = (
        e16_final_idx
    )


    classifier_prob = float(
        mn_probs[
            final_idx
        ]
    )


    source = (
        "e16"
    )


    conv = (
        item[
            "conv"
        ]
    )


    if conv is not None:

        conv_global_idx = int(
            conv[
                "conv_global_idx"
            ]
        )


        conv_top_prob = float(
            conv[
                "conv_top_prob"
            ]
        )


        required_gate = None


        if conv_global_idx == MR:

            required_gate = (
                E20_MR_GATE
            )


        elif conv_global_idx == MS:

            required_gate = (
                E20_MS_GATE
            )


        elif conv_global_idx == NP:

            required_gate = (
                E20_NP_GATE
            )


        elif conv_global_idx == PET_OIL:

            required_gate = (
                E20_PETOIL_GATE
            )


        if (
            required_gate
            is not None
            and conv_top_prob
            >= required_gate
        ):

            final_idx = (
                conv_global_idx
            )


            classifier_prob = (
                conv_top_prob
            )


            source = (
                "e18_convnext"
            )


    return (

        final_idx,

        classifier_prob,

        source
    )


# ==================================================================================================
# 15. BUILD E21-B PREDICTIONS FOR ONE THRESHOLD PAIR
# ==================================================================================================

def build_predictions(
    mr_gate,
    ms_gate
):

    predictions = []


    source_counts = Counter()

    e21_proposals = Counter()

    e21_accepted = Counter()


    for item in cache:

        (
            final_idx,
            classifier_prob,
            source
        ) = reconstruct_e20a(
            item
        )


        # ------------------------------------------------------------------------------------------
        # E21-A candidate
        # ------------------------------------------------------------------------------------------

        e21 = (
            item.get(
                "e21"
            )
        )


        if e21 is not None:

            e21_global_idx = int(
                e21[
                    "global_idx"
                ]
            )


            e21_top_prob = float(
                e21[
                    "top_prob"
                ]
            )


            e21_proposals[
                CLASS_NAMES[
                    e21_global_idx
                ]
            ] += 1


            required_gate = None


            # E21 only allowed to override with MR/MS
            if (
                e21_global_idx
                == MR
            ):

                required_gate = (
                    mr_gate
                )


            elif (
                e21_global_idx
                == MS
            ):

                required_gate = (
                    ms_gate
                )


            # HDPE/PET/ECAL/NP proposals do nothing
            if (
                required_gate
                is not None
                and e21_top_prob
                >= required_gate
            ):

                final_idx = (
                    e21_global_idx
                )


                classifier_prob = (
                    e21_top_prob
                )


                source = (
                    "e21"
                )


                e21_accepted[
                    CLASS_NAMES[
                        final_idx
                    ]
                ] += 1


        # ------------------------------------------------------------------------------------------
        # Score
        # ------------------------------------------------------------------------------------------

        yolo_conf = max(
            float(
                item[
                    "yolo_conf"
                ]
            ),
            1e-12
        )


        classifier_prob = max(
            float(
                classifier_prob
            ),
            1e-12
        )


        score = (

            yolo_conf
            ** ALPHA

        ) * (

            classifier_prob
            ** (
                1.0
                - ALPHA
            )
        )


        x1, y1, x2, y2 = [

            float(v)

            for v
            in item[
                "bbox"
            ]
        ]


        width = (
            x2 - x1
        )

        height = (
            y2 - y1
        )


        if (
            width <= 0
            or height <= 0
        ):

            continue


        predictions.append(
            {

                "image_id":
                    int(
                        item[
                            "image_id"
                        ]
                    ),

                "category_id":
                    int(
                        final_idx
                        + 1
                    ),

                "bbox": [

                    x1,

                    y1,

                    width,

                    height,
                ],

                "score":
                    float(
                        score
                    ),
            }
        )


        source_counts[
            source
        ] += 1


    return {

        "predictions":
            predictions,

        "source_counts":
            dict(
                source_counts
            ),

        "e21_proposals":
            dict(
                e21_proposals
            ),

        "e21_accepted":
            dict(
                e21_accepted
            ),
    }


# ==================================================================================================
# 16. COCO EVALUATOR
# ==================================================================================================

coco_gt = COCO(
    str(
        GT_PATH
    )
)


def valid_mean(
    values
):

    values = np.asarray(
        values
    )


    valid = values[
        values > -1
    ]


    if valid.size == 0:

        return float(
            "nan"
        )


    return float(
        np.mean(
            valid
        )
    )


def evaluate_predictions(
    predictions
):

    coco_dt = coco_gt.loadRes(
        predictions
    )


    evaluator = COCOeval(

        coco_gt,

        coco_dt,

        "bbox"
    )


    evaluator.params.maxDets = [

        1,

        10,

        100,
    ]


    evaluator.evaluate()

    evaluator.accumulate()


    precision = (
        evaluator.eval[
            "precision"
        ]
    )


    recall = (
        evaluator.eval[
            "recall"
        ]
    )


    iou_thresholds = (
        evaluator.params.iouThrs
    )


    idx50 = int(

        np.where(
            np.isclose(
                iou_thresholds,
                0.50
            )
        )[0][0]
    )


    idx75 = int(

        np.where(
            np.isclose(
                iou_thresholds,
                0.75
            )
        )[0][0]
    )


    overall = {

        "AP50_95":
            valid_mean(
                precision[
                    :,
                    :,
                    :,
                    0,
                    -1
                ]
            ),

        "AP50":
            valid_mean(
                precision[
                    idx50,
                    :,
                    :,
                    0,
                    -1
                ]
            ),

        "AP75":
            valid_mean(
                precision[
                    idx75,
                    :,
                    :,
                    0,
                    -1
                ]
            ),

        "AR100":
            valid_mean(
                recall[
                    :,
                    :,
                    0,
                    -1
                ]
            ),
    }


    class_metrics = {}


    for class_idx, class_name in enumerate(
        CLASS_NAMES
    ):

        class_metrics[
            class_name
        ] = {

            "AP50":
                valid_mean(
                    precision[
                        idx50,
                        :,
                        class_idx,
                        0,
                        -1
                    ]
                ),

            "AP50_95":
                valid_mean(
                    precision[
                        :,
                        :,
                        class_idx,
                        0,
                        -1
                    ]
                ),
        }


    plastic_mean_ap50 = float(

        np.mean(
            [

                class_metrics[
                    CLASS_NAMES[
                        idx
                    ]
                ][
                    "AP50"
                ]

                for idx
                in PLASTIC_CLASS_INDICES
            ]
        )
    )


    mr_ms_mean_ap50 = float(

        np.mean(
            [

                class_metrics[
                    "mixed_plastic_rigid"
                ][
                    "AP50"
                ],

                class_metrics[
                    "mixed_plastic_soft"
                ][
                    "AP50"
                ],
            ]
        )
    )


    return {

        "overall":
            overall,

        "class_metrics":
            class_metrics,

        "plastic_mean_AP50":
            plastic_mean_ap50,

        "mr_ms_mean_AP50":
            mr_ms_mean_ap50,

        "nonplastic_AP50":
            class_metrics[
                "non_plastic"
            ][
                "AP50"
            ],
    }


# ==================================================================================================
# 17. RECONSTRUCT E20-A BASELINE
#
# E21 disabled:
# MR = OFF
# MS = OFF
# ==================================================================================================

print()
print("=" * 115)
print("RECONSTRUCTING E20-A VALIDATION BASELINE")
print("=" * 115)


baseline_build = build_predictions(

    mr_gate=None,

    ms_gate=None
)


baseline_eval = evaluate_predictions(

    baseline_build[
        "predictions"
    ]
)


print()
print(
    f"Six-plastic mean AP50 : "
    f"{baseline_eval['plastic_mean_AP50'] * 100:.4f}%"
)


print(
    f"MR AP50               : "
    f"{baseline_eval['class_metrics']['mixed_plastic_rigid']['AP50'] * 100:.4f}%"
)


print(
    f"MS AP50               : "
    f"{baseline_eval['class_metrics']['mixed_plastic_soft']['AP50'] * 100:.4f}%"
)


print(
    f"Overall AP50-95       : "
    f"{baseline_eval['overall']['AP50_95'] * 100:.4f}%"
)


print(
    f"Overall AP50          : "
    f"{baseline_eval['overall']['AP50'] * 100:.4f}%"
)


print(
    f"non_plastic AP50      : "
    f"{baseline_eval['nonplastic_AP50'] * 100:.4f}%"
)


# ==================================================================================================
# 18. GRID SEARCH
# ==================================================================================================

results = []


total_combinations = (

    len(
        MR_GATES
    )

    *

    len(
        MS_GATES
    )
)


counter = 0


def gate_label(
    gate
):

    if gate is None:

        return "OFF"

    return f"{gate:.2f}"


print()
print("=" * 115)
print("E21-B MR/MS OVERRIDE GATE SEARCH — VALIDATION ONLY")
print("=" * 115)


for mr_gate in MR_GATES:

    for ms_gate in MS_GATES:

        counter += 1


        built = build_predictions(

            mr_gate=mr_gate,

            ms_gate=ms_gate
        )


        evaluation = evaluate_predictions(

            built[
                "predictions"
            ]
        )


        result = {

            "mr_gate":
                mr_gate,

            "ms_gate":
                ms_gate,

            "plastic_mean_AP50":
                evaluation[
                    "plastic_mean_AP50"
                ],

            "mr_ms_mean_AP50":
                evaluation[
                    "mr_ms_mean_AP50"
                ],

            "nonplastic_AP50":
                evaluation[
                    "nonplastic_AP50"
                ],

            "overall":
                evaluation[
                    "overall"
                ],

            "class_metrics":
                evaluation[
                    "class_metrics"
                ],

            "source_counts":
                built[
                    "source_counts"
                ],

            "e21_proposals":
                built[
                    "e21_proposals"
                ],

            "e21_accepted":
                built[
                    "e21_accepted"
                ],
        }


        results.append(
            result
        )


        print(

            f"[{counter:03d}/{total_combinations}] "

            f"MR={gate_label(mr_gate):>4s} "

            f"MS={gate_label(ms_gate):>4s} | "

            f"Plastic="
            f"{evaluation['plastic_mean_AP50'] * 100:7.3f}% | "

            f"MR="
            f"{evaluation['class_metrics']['mixed_plastic_rigid']['AP50'] * 100:6.2f}% | "

            f"MS="
            f"{evaluation['class_metrics']['mixed_plastic_soft']['AP50'] * 100:6.2f}% | "

            f"AP="
            f"{evaluation['overall']['AP50_95'] * 100:6.3f}%"
        )


# ==================================================================================================
# 19. RANK
# ==================================================================================================

ranked = sorted(

    results,

    key=lambda x: (

        x[
            "plastic_mean_AP50"
        ],

        x[
            "mr_ms_mean_AP50"
        ],

        x[
            "overall"
        ][
            "AP50_95"
        ],
    ),

    reverse=True
)


best = (
    ranked[
        0
    ]
)


# ==================================================================================================
# 20. DELTA VS E20-A
# ==================================================================================================

delta = {

    "plastic_mean_AP50_pp":

        (
            best[
                "plastic_mean_AP50"
            ]

            -

            baseline_eval[
                "plastic_mean_AP50"
            ]
        )
        * 100.0,


    "MR_AP50_pp":

        (
            best[
                "class_metrics"
            ][
                "mixed_plastic_rigid"
            ][
                "AP50"
            ]

            -

            baseline_eval[
                "class_metrics"
            ][
                "mixed_plastic_rigid"
            ][
                "AP50"
            ]
        )
        * 100.0,


    "MS_AP50_pp":

        (
            best[
                "class_metrics"
            ][
                "mixed_plastic_soft"
            ][
                "AP50"
            ]

            -

            baseline_eval[
                "class_metrics"
            ][
                "mixed_plastic_soft"
            ][
                "AP50"
            ]
        )
        * 100.0,


    "overall_AP50_95_pp":

        (
            best[
                "overall"
            ][
                "AP50_95"
            ]

            -

            baseline_eval[
                "overall"
            ][
                "AP50_95"
            ]
        )
        * 100.0,


    "overall_AP50_pp":

        (
            best[
                "overall"
            ][
                "AP50"
            ]

            -

            baseline_eval[
                "overall"
            ][
                "AP50"
            ]
        )
        * 100.0,


    "nonplastic_AP50_pp":

        (
            best[
                "nonplastic_AP50"
            ]

            -

            baseline_eval[
                "nonplastic_AP50"
            ]
        )
        * 100.0,
}


best[
    "delta_vs_E20A"
] = delta


# ==================================================================================================
# 21. PRINT TOP 15
# ==================================================================================================

print()
print()
print("=" * 115)
print("TOP 15 E21-B VALIDATION CONFIGURATIONS")
print("=" * 115)


print(
    f"\n"
    f"{'Rank':>4s} "
    f"{'MR':>6s} "
    f"{'MS':>6s} "
    f"{'Plastic AP50':>14s} "
    f"{'MR AP50':>10s} "
    f"{'MS AP50':>10s} "
    f"{'NP AP50':>10s} "
    f"{'AP50-95':>10s}"
)


print(
    "-" * 90
)


for rank, result in enumerate(
    ranked[:15],
    start=1
):

    print(

        f"{rank:4d} "

        f"{gate_label(result['mr_gate']):>6s} "

        f"{gate_label(result['ms_gate']):>6s} "

        f"{result['plastic_mean_AP50'] * 100:13.4f}% "

        f"{result['class_metrics']['mixed_plastic_rigid']['AP50'] * 100:9.2f}% "

        f"{result['class_metrics']['mixed_plastic_soft']['AP50'] * 100:9.2f}% "

        f"{result['nonplastic_AP50'] * 100:9.2f}% "

        f"{result['overall']['AP50_95'] * 100:9.3f}%"
    )


# ==================================================================================================
# 22. BEST CONFIGURATION
# ==================================================================================================

print()
print("=" * 115)
print("BEST E21-B CONFIGURATION — VALIDATION")
print("=" * 115)


print()
print(
    f"E21 MR gate            : "
    f"{gate_label(best['mr_gate'])}"
)


print(
    f"E21 MS gate            : "
    f"{gate_label(best['ms_gate'])}"
)


print()
print(
    f"Six-plastic mean AP50  : "
    f"{best['plastic_mean_AP50'] * 100:.4f}%"
)


print(
    f"MR AP50                : "
    f"{best['class_metrics']['mixed_plastic_rigid']['AP50'] * 100:.4f}%"
)


print(
    f"MS AP50                : "
    f"{best['class_metrics']['mixed_plastic_soft']['AP50'] * 100:.4f}%"
)


print(
    f"MR/MS mean AP50        : "
    f"{best['mr_ms_mean_AP50'] * 100:.4f}%"
)


print()
print(
    f"Overall AP50-95        : "
    f"{best['overall']['AP50_95'] * 100:.4f}%"
)


print(
    f"Overall AP50           : "
    f"{best['overall']['AP50'] * 100:.4f}%"
)


print(
    f"Overall AP75           : "
    f"{best['overall']['AP75'] * 100:.4f}%"
)


print(
    f"AR100                  : "
    f"{best['overall']['AR100'] * 100:.4f}%"
)


print(
    f"non_plastic AP50       : "
    f"{best['nonplastic_AP50'] * 100:.4f}%"
)


# ==================================================================================================
# 23. DELTAS
# ==================================================================================================

print()
print("=" * 115)
print("DELTA VS FROZEN E20-A VALIDATION")
print("=" * 115)


for name, value in delta.items():

    print(
        f"{name:30s}: "
        f"{value:+.4f} pp"
    )


# ==================================================================================================
# 24. CLASS-WISE
# ==================================================================================================

print()
print("=" * 115)
print("BEST E21-B — CLASS-WISE VALIDATION")
print("=" * 115)


print(
    f"\n"
    f"{'Class':28s}"
    f"{'AP50':>14s}"
    f"{'AP50-95':>14s}"
)


print(
    "-" * 56
)


for class_name in CLASS_NAMES:

    cm = (
        best[
            "class_metrics"
        ][
            class_name
        ]
    )


    print(

        f"{class_name:28s}"

        f"{cm['AP50'] * 100:13.2f}%"

        f"{cm['AP50_95'] * 100:13.2f}%"
    )


# ==================================================================================================
# 25. ROUTING
# ==================================================================================================

print()
print("=" * 115)
print("BEST E21-B ROUTING")
print("=" * 115)


print()
print(
    "Prediction source:"
)


total_predictions = sum(
    best[
        "source_counts"
    ].values()
)


for source, count in (
    best[
        "source_counts"
    ].items()
):

    pct = (
        100.0
        * count
        / total_predictions
    )


    print(
        f"{source:20s}: "
        f"{count:7,d} "
        f"({pct:6.2f}%)"
    )


print()
print(
    "E21 proposals / accepted:"
)


for class_name in [
    "mixed_plastic_rigid",
    "mixed_plastic_soft",
]:

    proposed = best[
        "e21_proposals"
    ].get(
        class_name,
        0
    )


    accepted = best[
        "e21_accepted"
    ].get(
        class_name,
        0
    )


    print(

        f"{class_name:25s}: "

        f"proposed={proposed:6,d}  "

        f"accepted={accepted:6,d}"
    )


# ==================================================================================================
# 26. SAVE GRID
# ==================================================================================================

with open(
    GRID_RESULTS_PATH,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        ranked,
        f,
        indent=2
    )


# ==================================================================================================
# 27. SAVE BEST PREDICTIONS
# ==================================================================================================

best_build = build_predictions(

    mr_gate=best[
        "mr_gate"
    ],

    ms_gate=best[
        "ms_gate"
    ]
)


with open(
    BEST_PREDICTIONS_PATH,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        best_build[
            "predictions"
        ],
        f
    )


# ==================================================================================================
# 28. SAVE BEST CONFIGURATION
# ==================================================================================================

best_output = {

    "experiment":
        "E21-B",

    "selection_dataset":
        "validation",

    "test_set_used_for_selection":
        False,

    "base_pipeline":
        "Frozen E20-A",

    "E21A_checkpoint":
        str(
            E21A_CKPT
        ),

    "selection_metric":
        "six-plastic mean AP50",

    "secondary_metric":
        "MR/MS mean AP50",

    "tie_break":
        "overall AP50-95",

    "baseline_E20A":
        baseline_eval,

    "best":
        best,
}


with open(
    BEST_CONFIG_PATH,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        best_output,
        f,
        indent=2
    )


# ==================================================================================================
# 29. SUMMARY
# ==================================================================================================

summary_lines = [

    "=" * 115,

    "E21-B — End-to-End Validation Integration",

    "=" * 115,

    "",

    "VALIDATION ONLY — TEST NOT USED",

    "",

    "Base:",
    "  Frozen E20-A",

    "",

    "E21-A:",
    f"  checkpoint = {E21A_CKPT}",

    "",

    "Best gates:",

    f"  MR = {gate_label(best['mr_gate'])}",

    f"  MS = {gate_label(best['ms_gate'])}",

    "",

    "Baseline E20-A:",

    f"  Six-plastic mean AP50 = {baseline_eval['plastic_mean_AP50'] * 100:.4f}%",

    f"  MR AP50               = {baseline_eval['class_metrics']['mixed_plastic_rigid']['AP50'] * 100:.4f}%",

    f"  MS AP50               = {baseline_eval['class_metrics']['mixed_plastic_soft']['AP50'] * 100:.4f}%",

    "",

    "Best E21-B:",

    f"  Six-plastic mean AP50 = {best['plastic_mean_AP50'] * 100:.4f}%",

    f"  MR AP50               = {best['class_metrics']['mixed_plastic_rigid']['AP50'] * 100:.4f}%",

    f"  MS AP50               = {best['class_metrics']['mixed_plastic_soft']['AP50'] * 100:.4f}%",

    f"  Overall AP50-95       = {best['overall']['AP50_95'] * 100:.4f}%",

    f"  Overall AP50          = {best['overall']['AP50'] * 100:.4f}%",

    f"  non_plastic AP50      = {best['nonplastic_AP50'] * 100:.4f}%",

    "",

    "Delta vs E20-A:",
]


for name, value in (
    delta.items()
):

    summary_lines.append(
        f"  {name:30s}: {value:+.4f} pp"
    )


with open(
    SUMMARY_PATH,
    "w",
    encoding="utf-8"
) as f:

    f.write(
        "\n".join(
            summary_lines
        )
    )


# ==================================================================================================
# 30. COMPLETE
# ==================================================================================================

print()
print("=" * 115)
print("E21-B COMPLETE")
print("=" * 115)


print()
print(
    f"E21 cache       : "
    f"{E21_CACHE_PATH}"
)


print(
    f"Grid results    : "
    f"{GRID_RESULTS_PATH}"
)


print(
    f"Best config     : "
    f"{BEST_CONFIG_PATH}"
)


print(
    f"Best predictions: "
    f"{BEST_PREDICTIONS_PATH}"
)


print(
    f"Summary         : "
    f"{SUMMARY_PATH}"
)

E21-B — END-TO-END VALIDATION INTEGRATION

PyTorch : 2.13.0+cu126
Device  : cuda
GPU     : NVIDIA GeForce RTX 3050 Ti Laptop GPU

Validation images : C:\Users\varda\Documents\_My Computer\COMMON_space\Learning\Upgrad_EPGP_MS\upgrad_IIITB_MS_course\MS_LJMU_Material\Topic Data\SortWaste\dataset\dataset\splited_all_dataset_coco\val\images
E18-D cache       : C:\Users\varda\Documents\_My Computer\COMMON_space\Learning\Upgrad_EPGP_MS\upgrad_IIITB_MS_course\MS_LJMU_Material\Thesis_Code\runs\sortwaste\E18D_class_selective_convnext\E18D_cached_predictions.json
Validation GT     : C:\Users\varda\Documents\_My Computer\COMMON_space\Learning\Upgrad_EPGP_MS\upgrad_IIITB_MS_course\MS_LJMU_Material\Thesis_Code\runs\sortwaste\E18D_class_selective_convnext\E18D_val_gt_7class.json
E21-A checkpoint  : C:\Users\varda\Documents\_My Computer\COMMON_space\Learning\Upgrad_EPGP_MS\upgrad_IIITB_MS_course\MS_LJMU_Material\Thesis_Code\runs\sortwaste\E21A_mixedplastic_hardnegative_convnext\E21A_ConvNeXtTiny_best.

E21-A validation inference: 100%|██████████| 780/780 [28:04<00:00,  2.16s/it] 



E21 crops processed : 55,605
Invalid crops       : 0
Inference time      : 28.07 min

E21 cache saved:
C:\Users\varda\Documents\_My Computer\COMMON_space\Learning\Upgrad_EPGP_MS\upgrad_IIITB_MS_course\MS_LJMU_Material\Thesis_Code\runs\sortwaste\E21B_end_to_end_validation\E21B_cached_predictions_with_E21A.json

Cache entries with E21 output : 55,605 / 55,605
loading annotations into memory...
Done (t=0.07s)
creating index...
index created!

RECONSTRUCTING E20-A VALIDATION BASELINE
Loading and preparing results...
DONE (t=0.11s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=8.03s).
Accumulating evaluation results...
DONE (t=1.04s).

Six-plastic mean AP50 : 68.0766%
MR AP50               : 49.6173%
MS AP50               : 42.8212%
Overall AP50-95       : 45.7559%
Overall AP50          : 61.1717%
non_plastic AP50      : 19.7421%

E21-B MR/MS OVERRIDE GATE SEARCH — VALIDATION ONLY
Loading and preparing results...
DONE (t=0.11s)
cre

In [ ]:
# E21-C — Frozen E21-B configuration evaluated once on the held-out TEST set.
# ==================================================================================================
# E21-C — FINAL HELD-OUT TEST
#
# PURPOSE
# --------------------------------------------------------------------------------------------------
# Evaluate the validation-selected E21-B configuration ONCE on the held-out TEST set.
#
# NO TEST-SET TUNING.
#
#
# FROZEN BASE PIPELINE
# --------------------------------------------------------------------------------------------------
# Detector                  : E12 YOLO11m @640
# MobileNet                 : E3Y-B
# E16-A MobileNet routing   : frozen
# E18-B ConvNeXt-Tiny       : frozen
#
# E20-A gates:
#   Mixed Rigid             : OFF
#   Mixed Soft              : 0.85
#   non_plastic             : 0.90
#   PET Oil                 : 0.94
#
# alpha                     : 0.70
#
#
# E21-A SPECIALIST
# --------------------------------------------------------------------------------------------------
# ConvNeXt-Tiny hard-negative specialist
#
# Classes:
#   0 mixed_plastic_rigid
#   1 mixed_plastic_soft
#   2 hdpe
#   3 pet
#   4 ecal
#   5 non_plastic
#
#
# FROZEN E21-B VALIDATION-SELECTED GATES
# --------------------------------------------------------------------------------------------------
# MR override gate = 0.97
# MS override gate = 0.99
#
# E21 may override ONLY when predicting MR or MS.
#
# ==================================================================================================

import json
import time
from pathlib import Path
from collections import Counter

import numpy as np

import torch
import torch.nn as nn

from torchvision import models
from torchvision.transforms import v2

from PIL import Image

from ultralytics import YOLO

from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval


# ==================================================================================================
# 1. DEVICE
# ==================================================================================================

DEVICE = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)


print("=" * 115)
print("FINAL HELD-OUT TEST — E21-C")
print("=" * 115)

print()
print(f"PyTorch : {torch.__version__}")
print(f"Device  : {DEVICE}")

if DEVICE.type == "cuda":
    print(
        f"GPU     : "
        f"{torch.cuda.get_device_name(0)}"
    )


# ==================================================================================================
# 2. PATHS
# ==================================================================================================

BASE = Path(
    r"C:\Users\varda\Documents\_My Computer\COMMON_space\Learning"
    r"\Upgrad_EPGP_MS\upgrad_IIITB_MS_course\MS_LJMU_Material"
)

DATASET_ROOT = (
    BASE
    / "Topic Data"
    / "SortWaste"
    / "dataset"
    / "dataset"
)

THESIS_CODE = (
    BASE
    / "Thesis_Code"
)


# --------------------------------------------------------------------------------------------------
# Test data
# --------------------------------------------------------------------------------------------------

TEST_IMAGES_DIR = (
    DATASET_ROOT
    / "splited_all_dataset_coco"
    / "test"
    / "images"
)

TEST_COCO_PATH = (
    DATASET_ROOT
    / "splited_all_dataset_coco"
    / "test"
    / "annotations"
    / "test_coco.json"
)


# --------------------------------------------------------------------------------------------------
# E12 YOLO
# --------------------------------------------------------------------------------------------------

YOLO_CKPT = (
    THESIS_CODE
    / "runs"
    / "sortwaste"
    / "E12_yolo11m_7class_aug_classbalance_640"
    / "weights"
    / "best.pt"
)


# --------------------------------------------------------------------------------------------------
# E3Y-B MobileNet
# --------------------------------------------------------------------------------------------------

MOBILENET_CKPT = (
    DATASET_ROOT
    / "yolo_mobilenet_crops_E3Y"
    / "mobilenet_results"
    / "E3Y_B_class_weighted"
    / "E3Y_B_MobileNetV3Large_best.pth"
)


# --------------------------------------------------------------------------------------------------
# E18-B ConvNeXt-Tiny
# --------------------------------------------------------------------------------------------------

E18B_CKPT = (
    THESIS_CODE
    / "runs"
    / "sortwaste"
    / "E18B_convnext_tiny_hardclass"
    / "E18B_ConvNeXtTiny_best.pth"
)


# --------------------------------------------------------------------------------------------------
# E21-A ConvNeXt-Tiny
# --------------------------------------------------------------------------------------------------

E21A_CKPT = (
    THESIS_CODE
    / "runs"
    / "sortwaste"
    / "E21A_mixedplastic_hardnegative_convnext"
    / "E21A_ConvNeXtTiny_best.pth"
)


# --------------------------------------------------------------------------------------------------
# Output
# --------------------------------------------------------------------------------------------------

OUTPUT_DIR = (
    THESIS_CODE
    / "runs"
    / "sortwaste"
    / "FINAL_TEST_E21C"
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)


PREDICTIONS_PATH = (
    OUTPUT_DIR
    / "E21C_test_predictions.json"
)

RESULTS_PATH = (
    OUTPUT_DIR
    / "E21C_test_results.json"
)

SUMMARY_PATH = (
    OUTPUT_DIR
    / "E21C_test_summary.txt"
)


# ==================================================================================================
# 3. VERIFY PATHS
# ==================================================================================================

for path, label in [

    (TEST_IMAGES_DIR, "Test images"),
    (TEST_COCO_PATH, "Test COCO"),
    (YOLO_CKPT, "YOLO"),
    (MOBILENET_CKPT, "MobileNet"),
    (E18B_CKPT, "E18-B ConvNeXt"),
    (E21A_CKPT, "E21-A ConvNeXt"),

]:

    if not path.exists():

        raise FileNotFoundError(
            f"{label} not found:\n{path}"
        )


print()
print(f"Test images : {TEST_IMAGES_DIR}")
print(f"Test COCO   : {TEST_COCO_PATH}")
print(f"YOLO        : {YOLO_CKPT}")
print(f"MobileNet   : {MOBILENET_CKPT}")
print(f"ConvNeXt-18 : {E18B_CKPT}")
print(f"ConvNeXt-21 : {E21A_CKPT}")
print(f"Output      : {OUTPUT_DIR}")


# ==================================================================================================
# 4. GLOBAL 7-CLASS MAPPING
# ==================================================================================================

CLASS_NAMES = [

    "ecal",
    "hdpe",
    "mixed_plastic_rigid",
    "mixed_plastic_soft",
    "non_plastic",
    "pet",
    "pet_oil",
]

ECAL = 0
HDPE = 1
MR = 2
MS = 3
NP = 4
PET = 5
PET_OIL = 6


PLASTIC_CLASS_INDICES = [

    ECAL,
    HDPE,
    MR,
    MS,
    PET,
    PET_OIL,
]


# ==================================================================================================
# 5. FROZEN SETTINGS
# ==================================================================================================

YOLO_IMGSZ = 640

YOLO_CONF = 0.001

YOLO_NMS_IOU = 0.60

YOLO_MAXDET = 100


ALPHA = 0.70


# E16-A MobileNet gates
MN_GATES = {

    ECAL: 0.99,

    HDPE: 0.98,

    MR: 0.98,

    MS: 0.94,

    NP: 0.94,

    PET: 0.99,

    PET_OIL: 0.99,
}


# E20-A / E18 specialist gates
E18_MR_GATE = None

E18_MS_GATE = 0.85

E18_NP_GATE = 0.90

E18_PETOIL_GATE = 0.94


# E21-B validation-selected gates
E21_MR_GATE = 0.97

E21_MS_GATE = 0.99


# ==================================================================================================
# 6. E18-B CLASS MAPPING
#
# E18-B specialist classes:
#
# 0 MR
# 1 MS
# 2 NP
# 3 PET Oil
# ==================================================================================================

E18_TO_GLOBAL = {

    0: MR,

    1: MS,

    2: NP,

    3: PET_OIL,
}


# ==================================================================================================
# 7. E21-A CLASS MAPPING
# ==================================================================================================

E21_CLASS_NAMES = [

    "mixed_plastic_rigid",

    "mixed_plastic_soft",

    "hdpe",

    "pet",

    "ecal",

    "non_plastic",
]


E21_TO_GLOBAL = {

    0: MR,

    1: MS,

    2: HDPE,

    3: PET,

    4: ECAL,

    5: NP,
}


# ==================================================================================================
# 8. LOAD TEST COCO
# ==================================================================================================

with open(
    TEST_COCO_PATH,
    "r",
    encoding="utf-8"
) as f:

    test_coco_json = json.load(f)


image_records = sorted(

    test_coco_json[
        "images"
    ],

    key=lambda x: int(
        x[
            "id"
        ]
    )
)


print()
print(
    f"Test images : "
    f"{len(image_records):,}"
)

print(
    f"Test GT     : "
    f"{len(test_coco_json['annotations']):,}"
)


# ==================================================================================================
# 9. BUILD 7-CLASS TEST GT
#
# Original COCO:
#
# 1 PET
# 2 HDPE
# 3 Mixed Soft
# 4 ECAL
# 5 Metal
# 6 Cardboard
# 7 Mixed Rigid
# 8 PET Oil
#
# New model:
#
# 1 ECAL
# 2 HDPE
# 3 Mixed Rigid
# 4 Mixed Soft
# 5 non_plastic
# 6 PET
# 7 PET Oil
# ==================================================================================================

ORIGINAL_TO_MODEL = {

    1: PET,

    2: HDPE,

    3: MS,

    4: ECAL,

    5: NP,

    6: NP,

    7: MR,

    8: PET_OIL,
}


test_gt_7class = {

    "images":
        test_coco_json[
            "images"
        ],

    "annotations":
        [],

    "categories":
        [
            {
                "id": i + 1,
                "name": name
            }

            for i, name
            in enumerate(
                CLASS_NAMES
            )
        ],
}


gt_distribution = Counter()


for ann in test_coco_json[
    "annotations"
]:

    original_category = int(
        ann[
            "category_id"
        ]
    )


    model_idx = (
        ORIGINAL_TO_MODEL[
            original_category
        ]
    )


    new_ann = dict(
        ann
    )


    new_ann[
        "category_id"
    ] = (
        model_idx
        + 1
    )


    test_gt_7class[
        "annotations"
    ].append(
        new_ann
    )


    gt_distribution[
        CLASS_NAMES[
            model_idx
        ]
    ] += 1


TEST_GT_7_PATH = (
    OUTPUT_DIR
    / "E21C_test_gt_7class.json"
)


with open(
    TEST_GT_7_PATH,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        test_gt_7class,
        f
    )


print()
print("Test GT distribution:")

for class_name in CLASS_NAMES:

    print(
        f"{class_name:25s}: "
        f"{gt_distribution[class_name]:,}"
    )


# ==================================================================================================
# 10. TRANSFORMS
# ==================================================================================================

classifier_transform = v2.Compose(
    [

        v2.Resize(
            (
                224,
                224
            ),
            antialias=True
        ),

        v2.ToImage(),

        v2.ToDtype(
            torch.float32,
            scale=True
        ),

        v2.Normalize(
            mean=[
                0.485,
                0.456,
                0.406
            ],
            std=[
                0.229,
                0.224,
                0.225
            ]
        ),
    ]
)


# ==================================================================================================
# 11. LOAD MOBILENET
# ==================================================================================================

print()
print("Loading E3Y-B MobileNet...")


mobilenet = models.mobilenet_v3_large(
    weights=None
)


mobilenet.classifier[
    3
] = nn.Linear(

    mobilenet.classifier[
        3
    ].in_features,

    7
)


checkpoint = torch.load(
    MOBILENET_CKPT,
    map_location=DEVICE
)


if (
    isinstance(
        checkpoint,
        dict
    )
    and "model_state_dict"
    in checkpoint
):

    state_dict = checkpoint[
        "model_state_dict"
    ]


elif (
    isinstance(
        checkpoint,
        dict
    )
    and "state_dict"
    in checkpoint
):

    state_dict = checkpoint[
        "state_dict"
    ]


else:

    state_dict = checkpoint


clean_state = {}


for key, value in (
    state_dict.items()
):

    clean_key = (
        key[7:]
        if key.startswith(
            "module."
        )
        else key
    )


    clean_state[
        clean_key
    ] = value


mobilenet.load_state_dict(
    clean_state,
    strict=True
)


mobilenet = mobilenet.to(
    DEVICE
)


mobilenet.eval()


print(
    "MobileNet loaded."
)


# ==================================================================================================
# 12. LOAD E18-B CONVNEXT
# ==================================================================================================

print()
print(
    "Loading E18-B ConvNeXt-Tiny..."
)


e18_model = models.convnext_tiny(
    weights=None
)


e18_model.classifier[
    2
] = nn.Linear(

    e18_model.classifier[
        2
    ].in_features,

    4
)


e18_checkpoint = torch.load(
    E18B_CKPT,
    map_location=DEVICE
)


if (
    isinstance(
        e18_checkpoint,
        dict
    )
    and "model_state_dict"
    in e18_checkpoint
):

    e18_state = (
        e18_checkpoint[
            "model_state_dict"
        ]
    )

else:

    e18_state = (
        e18_checkpoint
    )


e18_model.load_state_dict(
    e18_state,
    strict=True
)


e18_model = e18_model.to(
    DEVICE
)


e18_model.eval()


print(
    "E18-B ConvNeXt loaded."
)


# ==================================================================================================
# 13. LOAD E21-A CONVNEXT
# ==================================================================================================

print()
print(
    "Loading E21-A ConvNeXt-Tiny..."
)


e21_checkpoint = torch.load(
    E21A_CKPT,
    map_location=DEVICE
)


e21_model = models.convnext_tiny(
    weights=None
)


e21_model.classifier[
    2
] = nn.Linear(

    e21_model.classifier[
        2
    ].in_features,

    6
)


e21_model.load_state_dict(
    e21_checkpoint[
        "model_state_dict"
    ],
    strict=True
)


e21_model = e21_model.to(
    DEVICE
)


e21_model.eval()


print(
    "E21-A ConvNeXt loaded."
)


# ==================================================================================================
# 14. LOAD YOLO
# ==================================================================================================

print()
print(
    "Loading E12 YOLO11m..."
)


yolo = YOLO(
    str(
        YOLO_CKPT
    )
)


print(
    "YOLO11m loaded."
)


# ==================================================================================================
# 15. PRINT FROZEN CONFIG
# ==================================================================================================

print()
print("=" * 115)
print("FROZEN E21-C CONFIGURATION")
print("=" * 115)

print()
print(
    f"E20 Mixed Rigid gate : OFF"
)

print(
    f"E20 Mixed Soft gate  : "
    f"{E18_MS_GATE:.3f}"
)

print(
    f"E20 non-plastic gate : "
    f"{E18_NP_GATE:.3f}"
)

print(
    f"E20 PET Oil gate     : "
    f"{E18_PETOIL_GATE:.3f}"
)

print(
    f"E21 Mixed Rigid gate : "
    f"{E21_MR_GATE:.3f}"
)

print(
    f"E21 Mixed Soft gate  : "
    f"{E21_MS_GATE:.3f}"
)

print(
    f"Alpha                : "
    f"{ALPHA:.2f}"
)


# ==================================================================================================
# 16. INFERENCE
# ==================================================================================================

predictions = []


source_counts = Counter()


e18_proposed = Counter()
e18_accepted = Counter()


e21_proposed = Counter()
e21_accepted = Counter()


num_yolo_detections = 0
num_valid_crops = 0
num_invalid_crops = 0


start_time = (
    time.time()
)


with torch.inference_mode():

    for image_number, image_info in enumerate(
        image_records,
        start=1
    ):

        image_id = int(
            image_info[
                "id"
            ]
        )


        filename = (
            image_info[
                "file_name"
            ]
        )


        image_path = (
            TEST_IMAGES_DIR
            / filename
        )


        with Image.open(
            image_path
        ) as img:

            full_image = img.convert(
                "RGB"
            )


        image_width, image_height = (
            full_image.size
        )


        # ------------------------------------------------------------------------------------------
        # YOLO
        # ------------------------------------------------------------------------------------------

        results = yolo.predict(

            source=str(
                image_path
            ),

            imgsz=YOLO_IMGSZ,

            conf=YOLO_CONF,

            iou=YOLO_NMS_IOU,

            max_det=YOLO_MAXDET,

            verbose=False,

            device=0
            if DEVICE.type == "cuda"
            else "cpu"
        )


        boxes = (
            results[
                0
            ].boxes
        )


        if boxes is None:

            continue


        num_yolo_detections += (
            len(
                boxes
            )
        )


        # ------------------------------------------------------------------------------------------
        # Process each detection
        # ------------------------------------------------------------------------------------------

        for box in boxes:

            xyxy = (
                box.xyxy[
                    0
                ]
                .detach()
                .cpu()
                .numpy()
            )


            x1, y1, x2, y2 = [
                float(v)
                for v in xyxy
            ]


            yolo_conf = float(
                box.conf[
                    0
                ].item()
            )


            yolo_idx = int(
                box.cls[
                    0
                ].item()
            )


            # --------------------------------------------------------------------------------------
            # Clamp crop
            # --------------------------------------------------------------------------------------

            left = int(
                max(
                    0,
                    np.floor(
                        x1
                    )
                )
            )

            top = int(
                max(
                    0,
                    np.floor(
                        y1
                    )
                )
            )

            right = int(
                min(
                    image_width,
                    np.ceil(
                        x2
                    )
                )
            )

            bottom = int(
                min(
                    image_height,
                    np.ceil(
                        y2
                    )
                )
            )


            if (
                right <= left
                or bottom <= top
            ):

                num_invalid_crops += 1

                continue


            crop = full_image.crop(
                (
                    left,
                    top,
                    right,
                    bottom
                )
            )


            tensor = classifier_transform(
                crop
            ).unsqueeze(
                0
            ).to(
                DEVICE
            )


            num_valid_crops += 1


            # ======================================================================================
            # STAGE 1 — E3Y-B MOBILENET + E16-A ROUTING
            # ======================================================================================

            mn_logits = mobilenet(
                tensor
            )


            mn_probs = torch.softmax(
                mn_logits,
                dim=1
            )[0]


            mn_top_prob, mn_idx_tensor = torch.max(
                mn_probs,
                dim=0
            )


            mn_idx = int(
                mn_idx_tensor.item()
            )


            mn_top_prob_value = float(
                mn_top_prob.item()
            )


            if (
                mn_top_prob_value
                >= MN_GATES[
                    mn_idx
                ]
            ):

                final_idx = (
                    mn_idx
                )

                classifier_prob = (
                    mn_top_prob_value
                )

                source = (
                    "mobilenet"
                )


            else:

                final_idx = (
                    yolo_idx
                )


                classifier_prob = float(
                    mn_probs[
                        final_idx
                    ].item()
                )


                source = (
                    "yolo"
                )


            # ======================================================================================
            # STAGE 2 — E18-B HARD-CLASS SPECIALIST
            #
            # Run only if E16 final class is:
            # MR / MS / NP / PET Oil
            # ======================================================================================

            if final_idx in [
                MR,
                MS,
                NP,
                PET_OIL,
            ]:

                e18_logits = e18_model(
                    tensor
                )


                e18_probs = torch.softmax(
                    e18_logits,
                    dim=1
                )[0]


                e18_top_prob, e18_local_idx_tensor = torch.max(
                    e18_probs,
                    dim=0
                )


                e18_local_idx = int(
                    e18_local_idx_tensor.item()
                )


                e18_global_idx = int(
                    E18_TO_GLOBAL[
                        e18_local_idx
                    ]
                )


                e18_top_prob_value = float(
                    e18_top_prob.item()
                )


                e18_proposed[
                    CLASS_NAMES[
                        e18_global_idx
                    ]
                ] += 1


                required_gate = None


                if e18_global_idx == MR:

                    required_gate = (
                        E18_MR_GATE
                    )


                elif e18_global_idx == MS:

                    required_gate = (
                        E18_MS_GATE
                    )


                elif e18_global_idx == NP:

                    required_gate = (
                        E18_NP_GATE
                    )


                elif e18_global_idx == PET_OIL:

                    required_gate = (
                        E18_PETOIL_GATE
                    )


                if (
                    required_gate
                    is not None
                    and e18_top_prob_value
                    >= required_gate
                ):

                    final_idx = (
                        e18_global_idx
                    )


                    classifier_prob = (
                        e18_top_prob_value
                    )


                    source = (
                        "e18_convnext"
                    )


                    e18_accepted[
                        CLASS_NAMES[
                            e18_global_idx
                        ]
                    ] += 1


            # ======================================================================================
            # STAGE 3 — E21-A HARD-NEGATIVE SPECIALIST
            #
            # Run on ALL detector crops.
            #
            # Only allowed to override if prediction = MR or MS.
            # ======================================================================================

            e21_logits = e21_model(
                tensor
            )


            e21_probs = torch.softmax(
                e21_logits,
                dim=1
            )[0]


            e21_top_prob, e21_local_idx_tensor = torch.max(
                e21_probs,
                dim=0
            )


            e21_local_idx = int(
                e21_local_idx_tensor.item()
            )


            e21_global_idx = int(
                E21_TO_GLOBAL[
                    e21_local_idx
                ]
            )


            e21_top_prob_value = float(
                e21_top_prob.item()
            )


            e21_proposed[
                CLASS_NAMES[
                    e21_global_idx
                ]
            ] += 1


            # --------------------------------------------------------------------------------------
            # Validation-selected MR/MS overrides only
            # --------------------------------------------------------------------------------------

            if (
                e21_global_idx
                == MR
                and e21_top_prob_value
                >= E21_MR_GATE
            ):

                final_idx = (
                    MR
                )


                classifier_prob = (
                    e21_top_prob_value
                )


                source = (
                    "e21"
                )


                e21_accepted[
                    "mixed_plastic_rigid"
                ] += 1


            elif (
                e21_global_idx
                == MS
                and e21_top_prob_value
                >= E21_MS_GATE
            ):

                final_idx = (
                    MS
                )


                classifier_prob = (
                    e21_top_prob_value
                )


                source = (
                    "e21"
                )


                e21_accepted[
                    "mixed_plastic_soft"
                ] += 1


            # ======================================================================================
            # FINAL SCORE
            # ======================================================================================

            classifier_prob = max(
                classifier_prob,
                1e-12
            )


            score = (

                max(
                    yolo_conf,
                    1e-12
                )
                ** ALPHA

            ) * (

                classifier_prob
                ** (
                    1.0
                    - ALPHA
                )
            )


            width = (
                x2 - x1
            )

            height = (
                y2 - y1
            )


            predictions.append(
                {

                    "image_id":
                        image_id,

                    "category_id":
                        int(
                            final_idx
                            + 1
                        ),

                    "bbox": [

                        x1,

                        y1,

                        width,

                        height,
                    ],

                    "score":
                        float(
                            score
                        ),
                }
            )


            source_counts[
                source
            ] += 1


        # ------------------------------------------------------------------------------------------
        # Progress
        # ------------------------------------------------------------------------------------------

        if (
            image_number % 25 == 0
            or image_number
            == len(
                image_records
            )
        ):

            elapsed = (
                time.time()
                - start_time
            )


            print(

                f"[{image_number:4d}/"
                f"{len(image_records):4d}] "

                f"predictions="
                f"{len(predictions):,} "

                f"time="
                f"{elapsed / 60:.2f} min"
            )


# ==================================================================================================
# 17. INFERENCE SUMMARY
# ==================================================================================================

inference_time = (
    time.time()
    - start_time
)


print()
print("=" * 115)
print("E21-C TEST INFERENCE COMPLETE")
print("=" * 115)


print()
print(
    f"YOLO detections : "
    f"{num_yolo_detections:,}"
)

print(
    f"Valid crops     : "
    f"{num_valid_crops:,}"
)

print(
    f"Invalid crops   : "
    f"{num_invalid_crops:,}"
)

print(
    f"Predictions     : "
    f"{len(predictions):,}"
)

print(
    f"Inference time  : "
    f"{inference_time / 60:.2f} min"
)


# ==================================================================================================
# 18. SAVE PREDICTIONS
# ==================================================================================================

with open(
    PREDICTIONS_PATH,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        predictions,
        f
    )


# ==================================================================================================
# 19. COCO EVALUATION
# ==================================================================================================

print()
print("=" * 115)
print("E21-C — HELD-OUT TEST COCO EVALUATION")
print("=" * 115)


coco_gt = COCO(
    str(
        TEST_GT_7_PATH
    )
)


coco_dt = coco_gt.loadRes(
    predictions
)


evaluator = COCOeval(

    coco_gt,

    coco_dt,

    "bbox"
)


evaluator.params.maxDets = [

    1,

    10,

    100,
]


evaluator.evaluate()

evaluator.accumulate()

evaluator.summarize()


overall_ap = float(
    evaluator.stats[
        0
    ]
)

overall_ap50 = float(
    evaluator.stats[
        1
    ]
)

overall_ap75 = float(
    evaluator.stats[
        2
    ]
)

ar100 = float(
    evaluator.stats[
        8
    ]
)


# ==================================================================================================
# 20. CLASS-WISE AP50 + AP50-95
# ==================================================================================================

precision = (
    evaluator.eval[
        "precision"
    ]
)


iou_thresholds = (
    evaluator.params.iouThrs
)


idx50 = int(

    np.where(
        np.isclose(
            iou_thresholds,
            0.50
        )
    )[0][0]
)


def valid_mean(
    values
):

    values = np.asarray(
        values
    )


    valid = (
        values[
            values > -1
        ]
    )


    if valid.size == 0:

        return float(
            "nan"
        )


    return float(
        np.mean(
            valid
        )
    )


class_metrics = {}


for class_idx, class_name in enumerate(
    CLASS_NAMES
):

    ap50 = valid_mean(

        precision[
            idx50,
            :,
            class_idx,
            0,
            -1
        ]
    )


    ap5095 = valid_mean(

        precision[
            :,
            :,
            class_idx,
            0,
            -1
        ]
    )


    class_metrics[
        class_name
    ] = {

        "AP50":
            ap50,

        "AP50_95":
            ap5095,
    }


print()
print("=" * 115)
print("E21-C — CLASS-WISE HELD-OUT TEST RESULTS")
print("=" * 115)


print(
    f"\n"
    f"{'Class':32s}"
    f"{'AP50':>14s}"
    f"{'AP50-95':>16s}"
)


print(
    "-" * 64
)


for class_name in CLASS_NAMES:

    metrics = (
        class_metrics[
            class_name
        ]
    )


    print(

        f"{class_name:32s}"

        f"{metrics['AP50'] * 100:13.2f}%"

        f"{metrics['AP50_95'] * 100:15.2f}%"
    )


# ==================================================================================================
# 21. PLASTIC-FOCUSED SUMMARY
# ==================================================================================================

six_plastic_mean_ap50 = float(

    np.mean(
        [

            class_metrics[
                CLASS_NAMES[
                    idx
                ]
            ][
                "AP50"
            ]

            for idx
            in PLASTIC_CLASS_INDICES
        ]
    )
)


six_plastic_mean_ap = float(

    np.mean(
        [

            class_metrics[
                CLASS_NAMES[
                    idx
                ]
            ][
                "AP50_95"
            ]

            for idx
            in PLASTIC_CLASS_INDICES
        ]
    )
)


mr_ms_mean_ap50 = float(

    np.mean(
        [

            class_metrics[
                "mixed_plastic_rigid"
            ][
                "AP50"
            ],

            class_metrics[
                "mixed_plastic_soft"
            ][
                "AP50"
            ],
        ]
    )
)


print()
print("=" * 115)
print("E21-C — PLASTIC-FOCUSED TEST SUMMARY")
print("=" * 115)


print()
print(
    f"Six-plastic mean AP50     : "
    f"{six_plastic_mean_ap50 * 100:.4f}%"
)


print(
    f"Six-plastic mean AP50-95  : "
    f"{six_plastic_mean_ap * 100:.4f}%"
)


print(
    f"MR + MS mean AP50         : "
    f"{mr_ms_mean_ap50 * 100:.4f}%"
)


print()
print(
    f"Overall AP50-95           : "
    f"{overall_ap * 100:.4f}%"
)


print(
    f"Overall AP50              : "
    f"{overall_ap50 * 100:.4f}%"
)


print(
    f"Overall AP75              : "
    f"{overall_ap75 * 100:.4f}%"
)


print(
    f"AR100                     : "
    f"{ar100 * 100:.4f}%"
)


# ==================================================================================================
# 22. ROUTING SUMMARY
# ==================================================================================================

print()
print("=" * 115)
print("E21-C — TEST ROUTING SUMMARY")
print("=" * 115)


total_sources = sum(
    source_counts.values()
)


print()
print(
    "Final prediction source:"
)


for source in [

    "yolo",

    "mobilenet",

    "e18_convnext",

    "e21",
]:

    count = int(
        source_counts.get(
            source,
            0
        )
    )


    pct = (

        100.0
        * count
        / total_sources

        if total_sources > 0

        else 0.0
    )


    print(

        f"{source:18s}: "
        f"{count:7,d} "
        f"({pct:6.2f}%)"
    )


print()
print(
    "E18 proposals / accepted:"
)


for class_name in [

    "mixed_plastic_rigid",

    "mixed_plastic_soft",

    "non_plastic",

    "pet_oil",
]:

    print(

        f"{class_name:25s}: "

        f"proposed="
        f"{e18_proposed.get(class_name, 0):6,d} "

        f"accepted="
        f"{e18_accepted.get(class_name, 0):6,d}"
    )


print()
print(
    "E21 proposals / accepted:"
)


for class_name in [

    "mixed_plastic_rigid",

    "mixed_plastic_soft",
]:

    print(

        f"{class_name:25s}: "

        f"proposed="
        f"{e21_proposed.get(class_name, 0):6,d} "

        f"accepted="
        f"{e21_accepted.get(class_name, 0):6,d}"
    )


# ==================================================================================================
# 23. SAVE RESULTS
# ==================================================================================================

results = {

    "experiment":
        "E21-C",

    "dataset":
        "held-out test",

    "test_set_used_for_tuning":
        False,

    "frozen_validation_selected_configuration":
        {

            "E20_MR_gate":
                None,

            "E20_MS_gate":
                E18_MS_GATE,

            "E20_nonplastic_gate":
                E18_NP_GATE,

            "E20_pet_oil_gate":
                E18_PETOIL_GATE,

            "E21_MR_gate":
                E21_MR_GATE,

            "E21_MS_gate":
                E21_MS_GATE,

            "alpha":
                ALPHA,
        },

    "overall":
        {

            "AP50_95":
                overall_ap,

            "AP50":
                overall_ap50,

            "AP75":
                overall_ap75,

            "AR100":
                ar100,
        },

    "plastic_summary":
        {

            "six_plastic_mean_AP50":
                six_plastic_mean_ap50,

            "six_plastic_mean_AP50_95":
                six_plastic_mean_ap,

            "MR_MS_mean_AP50":
                mr_ms_mean_ap50,
        },

    "class_metrics":
        class_metrics,

    "source_counts":
        dict(
            source_counts
        ),

    "e18_proposed":
        dict(
            e18_proposed
        ),

    "e18_accepted":
        dict(
            e18_accepted
        ),

    "e21_proposed":
        dict(
            e21_proposed
        ),

    "e21_accepted":
        dict(
            e21_accepted
        ),

    "inference_time_minutes":
        inference_time
        / 60.0,
}


with open(
    RESULTS_PATH,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        results,
        f,
        indent=2
    )


# ==================================================================================================
# 24. SAVE SUMMARY
# ==================================================================================================

summary_lines = [

    "=" * 115,

    "E21-C — FINAL HELD-OUT TEST",

    "=" * 115,

    "",

    "Frozen configuration:",

    "  E20 MR gate        : OFF",

    f"  E20 MS gate        : {E18_MS_GATE:.3f}",

    f"  E20 NP gate        : {E18_NP_GATE:.3f}",

    f"  E20 PET Oil gate   : {E18_PETOIL_GATE:.3f}",

    f"  E21 MR gate        : {E21_MR_GATE:.3f}",

    f"  E21 MS gate        : {E21_MS_GATE:.3f}",

    f"  alpha              : {ALPHA:.2f}",

    "",

    "Held-out TEST results:",

    f"  Six-plastic mean AP50    : {six_plastic_mean_ap50 * 100:.4f}%",

    f"  Six-plastic mean AP50-95 : {six_plastic_mean_ap * 100:.4f}%",

    f"  MR/MS mean AP50          : {mr_ms_mean_ap50 * 100:.4f}%",

    f"  Overall AP50-95          : {overall_ap * 100:.4f}%",

    f"  Overall AP50             : {overall_ap50 * 100:.4f}%",

    f"  Overall AP75             : {overall_ap75 * 100:.4f}%",

    f"  AR100                    : {ar100 * 100:.4f}%",

    "",

    "Class-wise:",

]


for class_name in CLASS_NAMES:

    metrics = (
        class_metrics[
            class_name
        ]
    )


    summary_lines.append(

        f"  {class_name:25s} "
        f"AP50={metrics['AP50'] * 100:.4f}% "
        f"AP50-95={metrics['AP50_95'] * 100:.4f}%"
    )


summary_lines.extend(
    [

        "",

        "=" * 115,
    ]
)


with open(
    SUMMARY_PATH,
    "w",
    encoding="utf-8"
) as f:

    f.write(
        "\n".join(
            summary_lines
        )
    )


# ==================================================================================================
# 25. COMPLETE
# ==================================================================================================

print()
print("=" * 115)
print("E21-C HELD-OUT TEST COMPLETE")
print("=" * 115)


print()
print(
    f"Predictions : "
    f"{PREDICTIONS_PATH}"
)

print(
    f"Results     : "
    f"{RESULTS_PATH}"
)

print(
    f"Summary     : "
    f"{SUMMARY_PATH}"
)

FINAL HELD-OUT TEST — E21-C

PyTorch : 2.13.0+cu126
Device  : cuda
GPU     : NVIDIA GeForce RTX 3050 Ti Laptop GPU

Test images : C:\Users\varda\Documents\_My Computer\COMMON_space\Learning\Upgrad_EPGP_MS\upgrad_IIITB_MS_course\MS_LJMU_Material\Topic Data\SortWaste\dataset\dataset\splited_all_dataset_coco\test\images
Test COCO   : C:\Users\varda\Documents\_My Computer\COMMON_space\Learning\Upgrad_EPGP_MS\upgrad_IIITB_MS_course\MS_LJMU_Material\Topic Data\SortWaste\dataset\dataset\splited_all_dataset_coco\test\annotations\test_coco.json
YOLO        : C:\Users\varda\Documents\_My Computer\COMMON_space\Learning\Upgrad_EPGP_MS\upgrad_IIITB_MS_course\MS_LJMU_Material\Thesis_Code\runs\sortwaste\E12_yolo11m_7class_aug_classbalance_640\weights\best.pt
MobileNet   : C:\Users\varda\Documents\_My Computer\COMMON_space\Learning\Upgrad_EPGP_MS\upgrad_IIITB_MS_course\MS_LJMU_Material\Topic Data\SortWaste\dataset\dataset\yolo_mobilenet_crops_E3Y\mobilenet_results\E3Y_B_class_weighted\E3Y_B_MobileNetV

In [ ]:
# E21-C — MR / MS VALIDATION ERROR ANALYSIS
#
# Purpose:
#   Determine whether Mixed Rigid / Mixed Soft errors are caused mainly by:
#       1. localization failure
#       2. classification failure
#       3. MR <-> MS confusion
#       4. confusion with other plastic classes
#
# VALIDATION ONLY.
# DO NOT USE TEST PREDICTIONS FOR THIS ANALYSIS.
# ==========================================================================================

import json
from pathlib import Path
from collections import Counter, defaultdict

import numpy as np
import pandas as pd


# ==========================================================================================
# 1. PATHS
# ==========================================================================================

BASE = Path(
    r"C:\Users\varda\Documents\_My Computer\COMMON_space\Learning"
    r"\Upgrad_EPGP_MS\upgrad_IIITB_MS_course\MS_LJMU_Material"
)

DATASET_ROOT = (
    BASE
    / "Topic Data"
    / "SortWaste"
    / "dataset"
    / "dataset"
)

THESIS_CODE = (
    BASE
    / "Thesis_Code"
)


# ------------------------------------------------------------------------------------------
# ORIGINAL VALIDATION COCO GT
# ------------------------------------------------------------------------------------------

VAL_GT_ORIGINAL = (
    DATASET_ROOT
    / "splited_all_dataset_coco"
    / "val"
    / "annotations"
    / "val_coco.json"
)


# ------------------------------------------------------------------------------------------
# E21-C VALIDATION PREDICTIONS
#
# Replace this with the exact validation prediction JSON created by your
# E21-C validation run.
# ------------------------------------------------------------------------------------------

E21C_VAL_PREDICTIONS = Path(
    r"C:\Users\varda\Documents\_My Computer\COMMON_space\Learning"
    r"\Upgrad_EPGP_MS\upgrad_IIITB_MS_course\MS_LJMU_Material"
    r"\Thesis_Code\runs\sortwaste\E21B_end_to_end_validation"
    r"\E21B_best_predictions.json"
)

# ------------------------------------------------------------------------------------------
# OUTPUT
# ------------------------------------------------------------------------------------------

OUTPUT_DIR = (
    THESIS_CODE
    / "runs"
    / "sortwaste"
    / "E21C_MR_MS_error_analysis"
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)


# ==========================================================================================
# 2. STANDARDIZED 7-CLASS TAXONOMY
# ==========================================================================================

CLASS_NAMES = [
    "ecal",
    "hdpe",
    "mixed_plastic_rigid",
    "mixed_plastic_soft",
    "non_plastic",
    "pet",
    "pet_oil",
]

ECAL = 0
HDPE = 1
MR = 2
MS = 3
NON_PLASTIC = 4
PET = 5
PET_OIL = 6


# Original SortWaste COCO ID -> standardized 7-class zero-based index
ORIGINAL_TO_7 = {
    1: PET,
    2: HDPE,
    3: MS,
    4: ECAL,
    5: NON_PLASTIC,    # metal
    6: NON_PLASTIC,    # cardboard
    7: MR,
    8: PET_OIL,
}


# ==========================================================================================
# 3. CHECK PATHS
# ==========================================================================================

assert VAL_GT_ORIGINAL.exists(), (
    f"Validation GT not found:\n{VAL_GT_ORIGINAL}"
)

assert E21C_VAL_PREDICTIONS.exists(), (
    "\nE21-C validation predictions JSON not found.\n\n"
    f"Current path:\n{E21C_VAL_PREDICTIONS}\n\n"
    "Set E21C_VAL_PREDICTIONS to the exact JSON from your E21-C "
    "VALIDATION run — not FINAL_TEST_E21C."
)


# ==========================================================================================
# 4. LOAD DATA
# ==========================================================================================

with open(
    VAL_GT_ORIGINAL,
    "r",
    encoding="utf-8"
) as f:
    gt_data = json.load(f)


with open(
    E21C_VAL_PREDICTIONS,
    "r",
    encoding="utf-8"
) as f:
    predictions = json.load(f)


print("=" * 110)
print("E21-C — MR / MS VALIDATION ERROR ANALYSIS")
print("=" * 110)

print(f"\nValidation GT objects : {len(gt_data['annotations']):,}")
print(f"Predictions           : {len(predictions):,}")


# ==========================================================================================
# 5. IMAGE METADATA
# ==========================================================================================

image_info = {
    int(img["id"]): img
    for img in gt_data["images"]
}


# ==========================================================================================
# 6. CONVERT GT INTO STANDARDIZED 7 CLASS
# ==========================================================================================

gt_by_image = defaultdict(list)

for ann in gt_data["annotations"]:

    original_cat = int(
        ann["category_id"]
    )

    cls = ORIGINAL_TO_7[
        original_cat
    ]

    x, y, w, h = map(
        float,
        ann["bbox"]
    )

    gt_by_image[
        int(ann["image_id"])
    ].append(
        {
            "annotation_id":
                int(ann["id"]),

            "class_idx":
                cls,

            "class_name":
                CLASS_NAMES[cls],

            "bbox_xywh":
                [x, y, w, h],

            "bbox_xyxy":
                [
                    x,
                    y,
                    x + w,
                    y + h,
                ],
        }
    )


# ==========================================================================================
# 7. LOAD PREDICTIONS BY IMAGE
#
# Assumes E21-C predictions use standardized category IDs:
#     category_id = 1..7
# ==========================================================================================

pred_by_image = defaultdict(list)

for pred_idx, pred in enumerate(
    predictions
):

    cls = int(
        pred["category_id"]
    ) - 1

    x, y, w, h = map(
        float,
        pred["bbox"]
    )

    pred_by_image[
        int(pred["image_id"])
    ].append(
        {
            "prediction_idx":
                pred_idx,

            "class_idx":
                cls,

            "class_name":
                CLASS_NAMES[cls],

            "score":
                float(
                    pred["score"]
                ),

            "bbox_xywh":
                [x, y, w, h],

            "bbox_xyxy":
                [
                    x,
                    y,
                    x + w,
                    y + h,
                ],
        }
    )


# ==========================================================================================
# 8. IOU
# ==========================================================================================

def calculate_iou(box_a, box_b):

    ax1, ay1, ax2, ay2 = box_a
    bx1, by1, bx2, by2 = box_b

    ix1 = max(ax1, bx1)
    iy1 = max(ay1, by1)
    ix2 = min(ax2, bx2)
    iy2 = min(ay2, by2)

    iw = max(
        0.0,
        ix2 - ix1
    )

    ih = max(
        0.0,
        iy2 - iy1
    )

    intersection = (
        iw * ih
    )

    area_a = max(
        0.0,
        ax2 - ax1
    ) * max(
        0.0,
        ay2 - ay1
    )

    area_b = max(
        0.0,
        bx2 - bx1
    ) * max(
        0.0,
        by2 - by1
    )

    union = (
        area_a
        + area_b
        - intersection
    )

    if union <= 0:
        return 0.0

    return (
        intersection
        / union
    )


# ==========================================================================================
# 9. ONE-TO-ONE CLASS-AGNOSTIC MATCHING
#
# Important:
#   Matching is done WITHOUT using the class label.
#
# Therefore:
#   IoU >= 0.50 + wrong label = classification error
#   no matched prediction >= 0.50 = localization/detection error
# ==========================================================================================

IOU_THRESHOLD = 0.50

analysis_rows = []


for image_id, gts in gt_by_image.items():

    preds = pred_by_image.get(
        image_id,
        []
    )

    if not gts:
        continue


    # ----------------------------------------------------------------------
    # Build every GT / prediction IoU candidate
    # ----------------------------------------------------------------------

    candidates = []

    for gt_idx, gt in enumerate(
        gts
    ):

        for pred_idx, pred in enumerate(
            preds
        ):

            iou = calculate_iou(
                gt["bbox_xyxy"],
                pred["bbox_xyxy"],
            )

            if iou >= IOU_THRESHOLD:

                candidates.append(
                    (
                        iou,
                        gt_idx,
                        pred_idx,
                    )
                )


    # ----------------------------------------------------------------------
    # Highest IoU first -> one-to-one greedy matching
    # ----------------------------------------------------------------------

    candidates.sort(
        key=lambda x: x[0],
        reverse=True
    )

    matched_gt = set()
    matched_pred = set()

    matches = {}


    for (
        iou,
        gt_idx,
        pred_idx,
    ) in candidates:

        if gt_idx in matched_gt:
            continue

        if pred_idx in matched_pred:
            continue

        matched_gt.add(
            gt_idx
        )

        matched_pred.add(
            pred_idx
        )

        matches[
            gt_idx
        ] = (
            pred_idx,
            iou,
        )


    # ----------------------------------------------------------------------
    # We care specifically about MR and MS GT objects
    # ----------------------------------------------------------------------

    for gt_idx, gt in enumerate(
        gts
    ):

        true_cls = (
            gt["class_idx"]
        )

        if true_cls not in {
            MR,
            MS,
        }:
            continue


        row = {
            "image_id":
                image_id,

            "file_name":
                image_info[
                    image_id
                ]["file_name"],

            "annotation_id":
                gt["annotation_id"],

            "true_class":
                gt["class_name"],

            "matched":
                False,

            "iou":
                0.0,

            "pred_class":
                "NO_MATCH",

            "prediction_score":
                np.nan,

            "error_type":
                "localization_or_detection",
        }


        # ------------------------------------------------------------------
        # Proper match at IoU >= 0.50 exists
        # ------------------------------------------------------------------

        if gt_idx in matches:

            pred_idx, iou = (
                matches[
                    gt_idx
                ]
            )

            pred = (
                preds[
                    pred_idx
                ]
            )

            pred_cls = (
                pred[
                    "class_idx"
                ]
            )


            row[
                "matched"
            ] = True

            row[
                "iou"
            ] = iou

            row[
                "pred_class"
            ] = pred[
                "class_name"
            ]

            row[
                "prediction_score"
            ] = pred[
                "score"
            ]


            if pred_cls == true_cls:

                row[
                    "error_type"
                ] = (
                    "correct"
                )

            elif (
                true_cls == MR
                and
                pred_cls == MS
            ):

                row[
                    "error_type"
                ] = (
                    "MR_to_MS"
                )

            elif (
                true_cls == MS
                and
                pred_cls == MR
            ):

                row[
                    "error_type"
                ] = (
                    "MS_to_MR"
                )

            else:

                row[
                    "error_type"
                ] = (
                    "other_classification"
                )


        # ------------------------------------------------------------------
        # If no >=0.50 match, also save the best raw IoU for diagnosis
        # ------------------------------------------------------------------

        else:

            best_iou = 0.0
            best_pred = None

            for pred in preds:

                iou = calculate_iou(
                    gt["bbox_xyxy"],
                    pred["bbox_xyxy"],
                )

                if iou > best_iou:

                    best_iou = iou
                    best_pred = pred


            row[
                "best_iou_below_threshold"
            ] = best_iou


            if best_pred is not None:

                row[
                    "closest_pred_class"
                ] = (
                    best_pred[
                        "class_name"
                    ]
                )

                row[
                    "closest_pred_score"
                ] = (
                    best_pred[
                        "score"
                    ]
                )


        analysis_rows.append(
            row
        )


# ==========================================================================================
# 10. DATAFRAME
# ==========================================================================================

df = pd.DataFrame(
    analysis_rows
)

assert len(df) > 0, (
    "No MR/MS GT objects were found."
)


# ==========================================================================================
# 11. PER-CLASS ERROR DECOMPOSITION
# ==========================================================================================

print("\n" + "=" * 110)
print("MR / MS ERROR DECOMPOSITION")
print("=" * 110)


summary_rows = []


for class_name in [
    "mixed_plastic_rigid",
    "mixed_plastic_soft",
]:

    subset = df[
        df["true_class"]
        == class_name
    ]

    total = len(
        subset
    )

    correct = (
        subset[
            "error_type"
        ]
        == "correct"
    ).sum()

    localization = (
        subset[
            "error_type"
        ]
        == "localization_or_detection"
    ).sum()

    mr_ms_confusion = (
        subset[
            "error_type"
        ].isin(
            [
                "MR_to_MS",
                "MS_to_MR",
            ]
        )
    ).sum()

    other_classification = (
        subset[
            "error_type"
        ]
        == "other_classification"
    ).sum()

    classification_total = (
        mr_ms_confusion
        + other_classification
    )


    print(
        f"\n{class_name.upper()}"
    )

    print(
        f"GT objects                   : "
        f"{total:,}"
    )

    print(
        f"Correct @ IoU>=0.50          : "
        f"{correct:,} "
        f"({100*correct/total:.2f}%)"
    )

    print(
        f"Classification errors        : "
        f"{classification_total:,} "
        f"({100*classification_total/total:.2f}%)"
    )

    print(
        f"  MR<->MS confusion          : "
        f"{mr_ms_confusion:,} "
        f"({100*mr_ms_confusion/total:.2f}%)"
    )

    print(
        f"  Other-class confusion      : "
        f"{other_classification:,} "
        f"({100*other_classification/total:.2f}%)"
    )

    print(
        f"Localization/detection errors: "
        f"{localization:,} "
        f"({100*localization/total:.2f}%)"
    )


    summary_rows.append(
        {
            "class":
                class_name,

            "gt_objects":
                total,

            "correct":
                correct,

            "correct_pct":
                100 * correct / total,

            "classification_errors":
                classification_total,

            "classification_error_pct":
                100 * classification_total / total,

            "MR_MS_confusions":
                mr_ms_confusion,

            "MR_MS_confusion_pct":
                100 * mr_ms_confusion / total,

            "other_classification":
                other_classification,

            "localization_detection":
                localization,

            "localization_detection_pct":
                100 * localization / total,
        }
    )


# ==========================================================================================
# 12. CONFUSION BREAKDOWN
# ==========================================================================================

print("\n" + "=" * 110)
print("CLASSIFICATION CONFUSION — ONLY OBJECTS LOCALIZED AT IoU >= 0.50")
print("=" * 110)


for class_name in [
    "mixed_plastic_rigid",
    "mixed_plastic_soft",
]:

    subset = df[
        (
            df["true_class"]
            == class_name
        )
        &
        (
            df["matched"]
            == True
        )
    ]


    counts = (
        subset[
            "pred_class"
        ]
        .value_counts()
    )


    print(
        f"\nTrue class: {class_name}"
    )

    for pred_class, count in (
        counts.items()
    ):

        pct = (
            100.0
            * count
            / len(subset)
        )

        print(
            f"{pred_class:25s}: "
            f"{count:5,d} "
            f"({pct:6.2f}%)"
        )


# ==========================================================================================
# 13. DIRECT MR <-> MS CONFUSION
# ==========================================================================================

mr_to_ms = df[
    df["error_type"]
    == "MR_to_MS"
]

ms_to_mr = df[
    df["error_type"]
    == "MS_to_MR"
]


print("\n" + "=" * 110)
print("DIRECT MR <-> MS CONFUSION")
print("=" * 110)

print(
    f"\nMR -> MS : "
    f"{len(mr_to_ms):,}"
)

print(
    f"MS -> MR : "
    f"{len(ms_to_mr):,}"
)

print(
    f"Total    : "
    f"{len(mr_to_ms) + len(ms_to_mr):,}"
)


# ==========================================================================================
# 14. CLASSIFICATION-FIXABLE VS LOCALIZATION-LIMITED
# ==========================================================================================

classification_errors = df[
    df["error_type"].isin(
        [
            "MR_to_MS",
            "MS_to_MR",
            "other_classification",
        ]
    )
]

localization_errors = df[
    df["error_type"]
    == "localization_or_detection"
]


fixable = len(
    classification_errors
)

localization_limited = len(
    localization_errors
)

total_errors = (
    fixable
    + localization_limited
)


print("\n" + "=" * 110)
print("ERROR OPPORTUNITY")
print("=" * 110)


print(
    f"\nClassification-fixable errors : "
    f"{fixable:,}"
)

print(
    f"Localization-limited errors   : "
    f"{localization_limited:,}"
)


if total_errors > 0:

    print(
        f"\nFraction potentially fixable "
        f"by better classifier:"
        f"\n    "
        f"{100*fixable/total_errors:.2f}%"
    )

    print(
        f"\nFraction requiring better "
        f"detection/localization:"
        f"\n    "
        f"{100*localization_limited/total_errors:.2f}%"
    )


# ==========================================================================================
# 15. IOU DISTRIBUTION
# ==========================================================================================

matched_df = df[
    df["matched"]
    == True
]


print("\n" + "=" * 110)
print("MATCHED-BOX IoU DISTRIBUTION")
print("=" * 110)


for class_name in [
    "mixed_plastic_rigid",
    "mixed_plastic_soft",
]:

    values = matched_df[
        matched_df[
            "true_class"
        ]
        == class_name
    ]["iou"].values


    if len(values) == 0:
        continue


    print(
        f"\n{class_name}"
    )

    print(
        f"Mean IoU   : "
        f"{np.mean(values):.4f}"
    )

    print(
        f"Median IoU : "
        f"{np.median(values):.4f}"
    )

    print(
        f"IoU < 0.60 : "
        f"{100*np.mean(values < 0.60):.2f}%"
    )

    print(
        f"IoU >=0.75 : "
        f"{100*np.mean(values >= 0.75):.2f}%"
    )


# ==========================================================================================
# 16. LOW-IoU NEAR MISSES
# ==========================================================================================

localization_df = df[
    df["error_type"]
    == "localization_or_detection"
].copy()


if (
    "best_iou_below_threshold"
    in localization_df.columns
):

    near_miss = localization_df[
        localization_df[
            "best_iou_below_threshold"
        ]
        >= 0.40
    ]

    severe_miss = localization_df[
        localization_df[
            "best_iou_below_threshold"
        ]
        < 0.40
    ]


    print("\n" + "=" * 110)
    print("LOCALIZATION ERROR SEVERITY")
    print("=" * 110)

    print(
        f"\nNear misses "
        f"(best IoU 0.40-0.49): "
        f"{len(near_miss):,}"
    )

    print(
        f"Severe localization/misses "
        f"(best IoU <0.40): "
        f"{len(severe_miss):,}"
    )


# ==========================================================================================
# 17. SAVE ANALYSIS FILES
# ==========================================================================================

detail_csv = (
    OUTPUT_DIR
    / "E21C_MR_MS_error_details.csv"
)

summary_csv = (
    OUTPUT_DIR
    / "E21C_MR_MS_error_summary.csv"
)

confusion_csv = (
    OUTPUT_DIR
    / "E21C_MR_MS_confusion_matrix.csv"
)


df.to_csv(
    detail_csv,
    index=False
)


pd.DataFrame(
    summary_rows
).to_csv(
    summary_csv,
    index=False
)


# Confusion matrix for matched MR/MS GT
confusion = pd.crosstab(
    matched_df[
        "true_class"
    ],
    matched_df[
        "pred_class"
    ],
)


confusion.to_csv(
    confusion_csv
)


print("\n" + "=" * 110)
print("ANALYSIS COMPLETE")
print("=" * 110)

print(
    f"\nDetailed errors : "
    f"{detail_csv}"
)

print(
    f"Summary         : "
    f"{summary_csv}"
)

print(
    f"Confusion matrix: "
    f"{confusion_csv}"
)

E21-C — MR / MS VALIDATION ERROR ANALYSIS

Validation GT objects : 13,065
Predictions           : 55,605

MR / MS ERROR DECOMPOSITION

MIXED_PLASTIC_RIGID
GT objects                   : 1,120
Correct @ IoU>=0.50          : 698 (62.32%)
Classification errors        : 386 (34.46%)
  MR<->MS confusion          : 191 (17.05%)
  Other-class confusion      : 195 (17.41%)
Localization/detection errors: 36 (3.21%)

MIXED_PLASTIC_SOFT
GT objects                   : 1,443
Correct @ IoU>=0.50          : 924 (64.03%)
Classification errors        : 487 (33.75%)
  MR<->MS confusion          : 150 (10.40%)
  Other-class confusion      : 337 (23.35%)
Localization/detection errors: 32 (2.22%)

CLASSIFICATION CONFUSION — ONLY OBJECTS LOCALIZED AT IoU >= 0.50

True class: mixed_plastic_rigid
mixed_plastic_rigid      :   698 ( 64.39%)
mixed_plastic_soft       :   191 ( 17.62%)
hdpe                     :    72 (  6.64%)
pet                      :    50 (  4.61%)
ecal                     :    34 (  3.14%)
n

In [ ]:
# E21 ROUTING-OPPORTUNITY ANALYSIS
#
# PURPOSE
# --------------------------------------------------------------------------------------------------
# For GT Mixed Rigid / Mixed Soft objects:
#
# 1. Match GT object to final E21-B/E21-C validation detection at IoU >= 0.50
# 2. Check whether final pipeline classification is correct
# 3. Check what E21-A specialist predicted for the SAME detection
#
# This separates:
#
#   A) E21 specialist limitation:
#      final wrong + E21 wrong
#
#   B) Routing/gating missed opportunity:
#      final wrong + E21 predicted GT correctly
#
#   C) Harmful E21 override:
#      upstream/pre-E21 correct but final became wrong
#
# VALIDATION ONLY — TEST NOT USED
# ==================================================================================================

import json
from pathlib import Path
from collections import Counter, defaultdict

import numpy as np
import pandas as pd


# ==================================================================================================
# 1. PATHS
# ==================================================================================================

BASE = Path(
    r"C:\Users\varda\Documents\_My Computer\COMMON_space\Learning"
    r"\Upgrad_EPGP_MS\upgrad_IIITB_MS_course\MS_LJMU_Material"
)

DATASET_ROOT = (
    BASE
    / "Topic Data"
    / "SortWaste"
    / "dataset"
    / "dataset"
)

THESIS_CODE = BASE / "Thesis_Code"


VAL_GT_ORIGINAL = (
    DATASET_ROOT
    / "splited_all_dataset_coco"
    / "val"
    / "annotations"
    / "val_coco.json"
)


E21B_DIR = (
    THESIS_CODE
    / "runs"
    / "sortwaste"
    / "E21B_end_to_end_validation"
)


CACHE_PATH = (
    E21B_DIR
    / "E21B_cached_predictions_with_E21A.json"
)


FINAL_PRED_PATH = (
    E21B_DIR
    / "E21B_best_predictions.json"
)


OUTPUT_DIR = (
    THESIS_CODE
    / "runs"
    / "sortwaste"
    / "E21C_MR_MS_routing_analysis"
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)


DETAILS_PATH = (
    OUTPUT_DIR
    / "E21C_MR_MS_E21_routing_details.csv"
)


SUMMARY_PATH = (
    OUTPUT_DIR
    / "E21C_MR_MS_E21_routing_summary.csv"
)


# ==================================================================================================
# 2. VERIFY
# ==================================================================================================

for path in [
    VAL_GT_ORIGINAL,
    CACHE_PATH,
    FINAL_PRED_PATH,
]:
    assert path.exists(), f"Missing:\n{path}"


# ==================================================================================================
# 3. CLASS MAPPINGS
# ==================================================================================================

# Standardized global 7-class mapping:
#
# 0 ecal
# 1 hdpe
# 2 mixed_plastic_rigid
# 3 mixed_plastic_soft
# 4 non_plastic
# 5 pet
# 6 pet_oil

CLASS_NAMES = [
    "ecal",
    "hdpe",
    "mixed_plastic_rigid",
    "mixed_plastic_soft",
    "non_plastic",
    "pet",
    "pet_oil",
]


MR_IDX = 2
MS_IDX = 3


# Original SortWaste COCO category -> standardized 7-class index
COCO_TO_GLOBAL = {
    1: 5,   # pet
    2: 1,   # hdpe
    3: 3,   # mixed_plastic_soft
    4: 0,   # ecal
    5: 4,   # metal -> non_plastic
    6: 4,   # cardboard -> non_plastic
    7: 2,   # mixed_plastic_rigid
    8: 6,   # pet_oil
}


# ==================================================================================================
# 4. LOAD FILES
# ==================================================================================================

with open(
    VAL_GT_ORIGINAL,
    "r",
    encoding="utf-8"
) as f:
    gt_coco = json.load(f)


with open(
    CACHE_PATH,
    "r",
    encoding="utf-8"
) as f:
    cache = json.load(f)


with open(
    FINAL_PRED_PATH,
    "r",
    encoding="utf-8"
) as f:
    final_preds = json.load(f)


print("=" * 110)
print("E21-C — MR/MS E21 ROUTING-OPPORTUNITY ANALYSIS")
print("=" * 110)

print()
print(f"GT annotations       : {len(gt_coco['annotations']):,}")
print(f"Cached predictions   : {len(cache):,}")
print(f"Final predictions    : {len(final_preds):,}")


assert len(cache) == len(final_preds), (
    "Cache and final prediction counts differ."
)


# ==================================================================================================
# 5. VERIFY CACHE AND FINAL PREDICTIONS ARE ALIGNED
# ==================================================================================================

alignment_failures = 0


for i, (c, p) in enumerate(
    zip(cache, final_preds)
):

    if int(c["image_id"]) != int(p["image_id"]):
        alignment_failures += 1
        continue

    # Cache bbox is XYXY
    cx1, cy1, cx2, cy2 = c["bbox"]

    # Final bbox is XYWH
    px, py, pw, ph = p["bbox"]

    px2 = px + pw
    py2 = py + ph

    if not np.allclose(
        [cx1, cy1, cx2, cy2],
        [px, py, px2, py2],
        atol=1e-3
    ):
        alignment_failures += 1


print()
print(
    f"Cache/final alignment failures : "
    f"{alignment_failures}"
)


if alignment_failures > 0:
    raise RuntimeError(
        "Cache and final predictions do not align safely."
    )


# ==================================================================================================
# 6. BBOX HELPERS
# ==================================================================================================

def xywh_to_xyxy(bbox):

    x, y, w, h = bbox

    return [
        float(x),
        float(y),
        float(x + w),
        float(y + h),
    ]


def iou_xyxy(a, b):

    ax1, ay1, ax2, ay2 = a
    bx1, by1, bx2, by2 = b

    ix1 = max(ax1, bx1)
    iy1 = max(ay1, by1)
    ix2 = min(ax2, bx2)
    iy2 = min(ay2, by2)

    iw = max(0.0, ix2 - ix1)
    ih = max(0.0, iy2 - iy1)

    inter = iw * ih

    area_a = max(0.0, ax2 - ax1) * max(
        0.0,
        ay2 - ay1
    )

    area_b = max(0.0, bx2 - bx1) * max(
        0.0,
        by2 - by1
    )

    union = area_a + area_b - inter

    if union <= 0:
        return 0.0

    return inter / union


# ==================================================================================================
# 7. ORGANIZE GT BY IMAGE
# ==================================================================================================

gt_by_image = defaultdict(list)


for ann in gt_coco["annotations"]:

    original_category = int(
        ann["category_id"]
    )

    global_idx = COCO_TO_GLOBAL[
        original_category
    ]


    if global_idx not in [
        MR_IDX,
        MS_IDX,
    ]:
        continue


    gt_by_image[
        int(ann["image_id"])
    ].append(
        {
            "ann_id": int(
                ann["id"]
            ),

            "gt_idx": global_idx,

            "bbox_xyxy": xywh_to_xyxy(
                ann["bbox"]
            ),
        }
    )


# ==================================================================================================
# 8. ORGANIZE DETECTIONS BY IMAGE
# ==================================================================================================

pred_by_image = defaultdict(list)


for idx, (cached, final) in enumerate(
    zip(cache, final_preds)
):

    # category_id in final COCO result = 1..7
    final_idx = int(
        final["category_id"]
    ) - 1


    record = {
        "pred_index":
            idx,

        "bbox_xyxy":
            [
                float(x)
                for x in cached["bbox"]
            ],

        "final_idx":
            final_idx,

        "final_score":
            float(
                final["score"]
            ),

        "yolo_idx":
            int(
                cached["yolo_class"]
            ),

        "pre_e21_idx":
            int(
                cached["e16_final_idx"]
            ),

        "e21":
            cached.get(
                "e21"
            ),
    }


    pred_by_image[
        int(cached["image_id"])
    ].append(
        record
    )


# ==================================================================================================
# 9. ONE-TO-ONE GREEDY MATCHING
#
# Class agnostic, because we are diagnosing classification.
# ==================================================================================================

matches = []


for image_id, gt_items in (
    gt_by_image.items()
):

    pred_items = (
        pred_by_image.get(
            image_id,
            []
        )
    )


    candidate_pairs = []


    for gi, gt in enumerate(
        gt_items
    ):

        for pi, pred in enumerate(
            pred_items
        ):

            iou = iou_xyxy(
                gt["bbox_xyxy"],
                pred["bbox_xyxy"]
            )

            if iou >= 0.50:

                candidate_pairs.append(
                    (
                        iou,
                        gi,
                        pi
                    )
                )


    candidate_pairs.sort(
        reverse=True,
        key=lambda x: x[0]
    )


    used_gt = set()
    used_pred = set()


    for iou, gi, pi in (
        candidate_pairs
    ):

        if gi in used_gt:
            continue

        if pi in used_pred:
            continue


        used_gt.add(
            gi
        )

        used_pred.add(
            pi
        )


        gt = gt_items[gi]
        pred = pred_items[pi]


        matches.append(
            {
                "image_id":
                    image_id,

                "ann_id":
                    gt["ann_id"],

                "gt_idx":
                    gt["gt_idx"],

                "gt_class":
                    CLASS_NAMES[
                        gt["gt_idx"]
                    ],

                "iou":
                    float(iou),

                **pred,
            }
        )


# ==================================================================================================
# 10. ANALYSE E21 BEHAVIOUR
# ==================================================================================================

rows = []


for item in matches:

    gt_idx = item["gt_idx"]

    final_idx = item["final_idx"]

    pre_idx = item["pre_e21_idx"]

    e21 = item["e21"]


    final_correct = (
        final_idx == gt_idx
    )


    pre_correct = (
        pre_idx == gt_idx
    )


    e21_available = (
        e21 is not None
    )


    if e21_available:

        e21_idx = int(
            e21["global_idx"]
        )

        e21_prob = float(
            e21["top_prob"]
        )

        e21_correct = (
            e21_idx == gt_idx
        )

    else:

        e21_idx = None
        e21_prob = None
        e21_correct = False


    # ----------------------------------------------------------------------------------------------
    # Diagnostic category
    # ----------------------------------------------------------------------------------------------

    if final_correct:

        if (
            not pre_correct
            and e21_correct
        ):

            diagnosis = (
                "E21_fixed_error"
            )

        else:

            diagnosis = (
                "final_correct"
            )


    else:

        if e21_correct:

            diagnosis = (
                "routing_missed_opportunity"
            )

        else:

            diagnosis = (
                "E21_specialist_wrong"
            )


    # Did routing potentially damage a correct pre-E21 result?
    harmful_override = (
        pre_correct
        and
        not final_correct
    )


    rows.append(
        {
            "image_id":
                item["image_id"],

            "ann_id":
                item["ann_id"],

            "gt_class":
                item["gt_class"],

            "iou":
                item["iou"],

            "pre_e21_class":
                CLASS_NAMES[
                    pre_idx
                ],

            "pre_e21_correct":
                pre_correct,

            "e21_available":
                e21_available,

            "e21_class":
                (
                    CLASS_NAMES[
                        e21_idx
                    ]
                    if e21_idx
                    is not None
                    else None
                ),

            "e21_top_prob":
                e21_prob,

            "e21_correct":
                e21_correct,

            "final_class":
                CLASS_NAMES[
                    final_idx
                ],

            "final_correct":
                final_correct,

            "diagnosis":
                diagnosis,

            "harmful_override":
                harmful_override,

            "yolo_class":
                CLASS_NAMES[
                    item["yolo_idx"]
                ],
        }
    )


df = pd.DataFrame(
    rows
)


# ==================================================================================================
# 11. PRINT RESULTS
# ==================================================================================================

print()
print("=" * 110)
print("MATCHED MR/MS OBJECTS")
print("=" * 110)

print()
print(
    f"Total matched MR/MS objects : "
    f"{len(df):,}"
)


for target_class in [
    "mixed_plastic_rigid",
    "mixed_plastic_soft",
]:

    sub = df[
        df["gt_class"]
        ==
        target_class
    ]


    wrong = sub[
        ~sub["final_correct"]
    ]


    routing_missed = wrong[
        wrong["e21_correct"]
    ]


    specialist_wrong = wrong[
        ~wrong["e21_correct"]
    ]


    e21_fixed = sub[
        sub["diagnosis"]
        ==
        "E21_fixed_error"
    ]


    harmful = sub[
        sub["harmful_override"]
    ]


    print()
    print("-" * 110)
    print(target_class.upper())
    print("-" * 110)

    print(
        f"Matched GT objects                  : "
        f"{len(sub):,}"
    )

    print(
        f"Final correct                       : "
        f"{sub['final_correct'].sum():,}"
    )

    print(
        f"Final wrong                         : "
        f"{len(wrong):,}"
    )

    print()

    print(
        f"E21 FIXED upstream mistake          : "
        f"{len(e21_fixed):,}"
    )

    print(
        f"ROUTING MISSED OPPORTUNITY          : "
        f"{len(routing_missed):,}"
    )

    print(
        f"E21 SPECIALIST ALSO WRONG           : "
        f"{len(specialist_wrong):,}"
    )

    print(
        f"Potential harmful override          : "
        f"{len(harmful):,}"
    )


    if len(wrong) > 0:

        print()
        print(
            "Among final classification errors:"
        )

        print(
            f"  E21 knew correct GT class         : "
            f"{len(routing_missed) / len(wrong) * 100:.2f}%"
        )

        print(
            f"  E21 itself was wrong              : "
            f"{len(specialist_wrong) / len(wrong) * 100:.2f}%"
        )


# ==================================================================================================
# 12. E21 MISSED OPPORTUNITY BY FINAL WRONG CLASS
# ==================================================================================================

print()
print("=" * 110)
print("ROUTING MISSED OPPORTUNITIES — BY FINAL WRONG CLASS")
print("=" * 110)


missed = df[
    (
        ~df["final_correct"]
    )
    &
    (
        df["e21_correct"]
    )
]


if len(missed) == 0:

    print()
    print(
        "No routing missed opportunities found."
    )

else:

    table = pd.crosstab(

        missed[
            "gt_class"
        ],

        missed[
            "final_class"
        ],
    )

    print()
    print(
        table.to_string()
    )


# ==================================================================================================
# 13. E21 SPECIALIST ERRORS
# ==================================================================================================

print()
print("=" * 110)
print("WHEN FINAL IS WRONG — WHAT DID E21-A PREDICT?")
print("=" * 110)


wrong_df = df[
    ~df["final_correct"]
]


for target_class in [
    "mixed_plastic_rigid",
    "mixed_plastic_soft",
]:

    sub = wrong_df[
        wrong_df["gt_class"]
        ==
        target_class
    ]


    counts = (
        sub["e21_class"]
        .value_counts(
            dropna=False
        )
    )


    print()
    print(
        f"GT = {target_class}"
    )

    print(
        counts.to_string()
    )


# ==================================================================================================
# 14. CONFIDENCE OF CORRECT E21 PREDICTIONS THAT WERE REJECTED
# ==================================================================================================

print()
print("=" * 110)
print("E21 CORRECT-BUT-REJECTED CONFIDENCE")
print("=" * 110)


if len(missed) > 0:

    for target_class in [
        "mixed_plastic_rigid",
        "mixed_plastic_soft",
    ]:

        sub = missed[
            missed["gt_class"]
            ==
            target_class
        ]


        print()
        print(
            target_class
        )


        if len(sub) == 0:

            print(
                "No missed opportunities."
            )

            continue


        probs = sub[
            "e21_top_prob"
        ].dropna()


        print(
            f"Count        : "
            f"{len(probs):,}"
        )

        print(
            f"Mean prob    : "
            f"{probs.mean():.4f}"
        )

        print(
            f"Median prob  : "
            f"{probs.median():.4f}"
        )

        print(
            f">= 0.90      : "
            f"{(probs >= 0.90).mean() * 100:.2f}%"
        )

        print(
            f">= 0.95      : "
            f"{(probs >= 0.95).mean() * 100:.2f}%"
        )

        print(
            f">= 0.97      : "
            f"{(probs >= 0.97).mean() * 100:.2f}%"
        )

        print(
            f">= 0.99      : "
            f"{(probs >= 0.99).mean() * 100:.2f}%"
        )


# ==================================================================================================
# 15. SAVE
# ==================================================================================================

df.to_csv(
    DETAILS_PATH,
    index=False
)


summary_records = []


for target_class in [
    "mixed_plastic_rigid",
    "mixed_plastic_soft",
]:

    sub = df[
        df["gt_class"]
        ==
        target_class
    ]


    wrong = sub[
        ~sub["final_correct"]
    ]


    summary_records.append(
        {
            "class":
                target_class,

            "matched":
                len(sub),

            "final_correct":
                int(
                    sub[
                        "final_correct"
                    ].sum()
                ),

            "final_wrong":
                len(wrong),

            "e21_fixed_error":
                int(
                    (
                        sub[
                            "diagnosis"
                        ]
                        ==
                        "E21_fixed_error"
                    ).sum()
                ),

            "routing_missed_opportunity":
                int(
                    (
                        sub[
                            "diagnosis"
                        ]
                        ==
                        "routing_missed_opportunity"
                    ).sum()
                ),

            "e21_specialist_wrong":
                int(
                    (
                        sub[
                            "diagnosis"
                        ]
                        ==
                        "E21_specialist_wrong"
                    ).sum()
                ),

            "harmful_override":
                int(
                    sub[
                        "harmful_override"
                    ].sum()
                ),
        }
    )


summary_df = pd.DataFrame(
    summary_records
)


summary_df.to_csv(
    SUMMARY_PATH,
    index=False
)


print()
print("=" * 110)
print("ANALYSIS COMPLETE")
print("=" * 110)

print()
print(
    f"Details : {DETAILS_PATH}"
)

print(
    f"Summary : {SUMMARY_PATH}"
)

E21-C — MR/MS E21 ROUTING-OPPORTUNITY ANALYSIS

GT annotations       : 13,065
Cached predictions   : 55,605
Final predictions    : 55,605

Cache/final alignment failures : 0

MATCHED MR/MS OBJECTS

Total matched MR/MS objects : 2,495

--------------------------------------------------------------------------------------------------------------
MIXED_PLASTIC_RIGID
--------------------------------------------------------------------------------------------------------------
Matched GT objects                  : 1,084
Final correct                       : 698
Final wrong                         : 386

E21 FIXED upstream mistake          : 52
ROUTING MISSED OPPORTUNITY          : 171
E21 SPECIALIST ALSO WRONG           : 215
Potential harmful override          : 9

Among final classification errors:
  E21 knew correct GT class         : 44.30%
  E21 itself was wrong              : 55.70%

------------------------------------------------------------------------------------------------------

# Model E22-A — class-conditional E21-A routing refinement.
Frozen upstream pipeline
YOLO → MobileNet/E16 → E18/E20
                ↓
        pre-E21-A class
                ↓
             E21-A

E21-A predicts MR:
    only consider override if pre-E21-A class is:
    MS / HDPE / PET

E21-A predicts MS:
    only consider override if pre-E21-A class is:
    MR / HDPE / PET / ECAL


In [ ]:
# E22-A — CLASS-CONDITIONAL E21-A ROUTING REFINEMENT
#
# VALIDATION ONLY
# NO RETRAINING
# NO TEST ACCESS
#
# --------------------------------------------------------------------------------------------------
# PURPOSE
# --------------------------------------------------------------------------------------------------
#
# E21-B currently uses:
#
#     E21-A MR override gate = 0.97
#     E21-A MS override gate = 0.99
#
# regardless of the pre-E21-A class.
#
# E21 routing-opportunity analysis showed that many remaining MR/MS errors were cases where
# E21-A actually predicted the correct class, but with confidence below those strict gates.
#
# E22-A therefore:
#
#     1. freezes the complete upstream E20-A pipeline
#     2. uses the already-cached E21-A probabilities
#     3. restricts E21-A corrections to known confusion routes
#     4. searches lower confidence thresholds on VALIDATION only
#
#
# CLASS-CONDITIONAL ROUTING
# --------------------------------------------------------------------------------------------------
#
# If E21-A predicts MR:
#
#     allow override ONLY if pre-E21-A prediction is:
#
#         MS
#         HDPE
#         PET
#
#
# If E21-A predicts MS:
#
#     allow override ONLY if pre-E21-A prediction is:
#
#         MR
#         HDPE
#         PET
#         ECAL
#
#
# PRIMARY SELECTION METRIC
# --------------------------------------------------------------------------------------------------
#
#     Six-plastic mean AP50
#
# SECONDARY
#
#     MR/MS mean AP50
#
# TIE-BREAK
#
#     Overall AP50-95
#
#
# SAFEGUARDS REPORTED
# --------------------------------------------------------------------------------------------------
#
#     Overall AP50
#     ECAL AP50
#     HDPE AP50
#     PET AP50
#     non_plastic AP50
#
#
# EXISTING E21-B VALIDATION BASELINE
# --------------------------------------------------------------------------------------------------
#
#     E21-A MR gate = 0.97
#     E21-A MS gate = 0.99
#     unconditional MR/MS override
#
# ==================================================================================================


import json
from pathlib import Path
from collections import Counter

import numpy as np

from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval


# ==================================================================================================
# 1. PATHS
# ==================================================================================================

BASE = Path(
    r"C:\Users\varda\Documents\_My Computer\COMMON_space\Learning"
    r"\Upgrad_EPGP_MS\upgrad_IIITB_MS_course\MS_LJMU_Material"
)


THESIS_CODE = (
    BASE
    / "Thesis_Code"
)


# --------------------------------------------------------------------------------------------------
# Existing E18-D / E20 validation GT
# --------------------------------------------------------------------------------------------------

E18D_DIR = (
    THESIS_CODE
    / "runs"
    / "sortwaste"
    / "E18D_class_selective_convnext"
)


GT_PATH = (
    E18D_DIR
    / "E18D_val_gt_7class.json"
)


# --------------------------------------------------------------------------------------------------
# Existing E21-B cache
#
# IMPORTANT:
# This already contains E21-A probabilities for all 55,605 validation detections.
# No neural-network inference is required for E22.
# --------------------------------------------------------------------------------------------------

E21B_DIR = (
    THESIS_CODE
    / "runs"
    / "sortwaste"
    / "E21B_end_to_end_validation"
)


CACHE_PATH = (
    E21B_DIR
    / "E21B_cached_predictions_with_E21A.json"
)


# --------------------------------------------------------------------------------------------------
# E22 output
# --------------------------------------------------------------------------------------------------

OUTPUT_DIR = (
    THESIS_CODE
    / "runs"
    / "sortwaste"
    / "E22A_class_conditional_E21_routing"
)


OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)


GRID_RESULTS_PATH = (
    OUTPUT_DIR
    / "E22A_grid_results.json"
)


BEST_CONFIG_PATH = (
    OUTPUT_DIR
    / "E22A_best_configuration.json"
)


BEST_PREDICTIONS_PATH = (
    OUTPUT_DIR
    / "E22A_best_predictions.json"
)


SUMMARY_PATH = (
    OUTPUT_DIR
    / "E22A_summary.txt"
)


# ==================================================================================================
# 2. VERIFY FILES
# ==================================================================================================

for path in [

    GT_PATH,
    CACHE_PATH,

]:

    if not path.exists():

        raise FileNotFoundError(
            f"Required file not found:\n{path}"
        )


print("=" * 115)
print("E22-A — CLASS-CONDITIONAL E21-A ROUTING REFINEMENT")
print("=" * 115)

print()
print(f"Validation GT : {GT_PATH}")
print(f"E21-B cache   : {CACHE_PATH}")
print(f"Output        : {OUTPUT_DIR}")


# ==================================================================================================
# 3. GLOBAL 7-CLASS MAPPING
# ==================================================================================================

CLASS_NAMES = [

    "ecal",
    "hdpe",
    "mixed_plastic_rigid",
    "mixed_plastic_soft",
    "non_plastic",
    "pet",
    "pet_oil",
]


ECAL = 0
HDPE = 1
MR = 2
MS = 3
NP = 4
PET = 5
PET_OIL = 6


PLASTIC_CLASS_INDICES = [

    ECAL,
    HDPE,
    MR,
    MS,
    PET,
    PET_OIL,
]


# ==================================================================================================
# 4. FROZEN UPSTREAM E20-A SETTINGS
#
# These reconstruct the pipeline prediction immediately BEFORE E21-A.
# ==================================================================================================

ALPHA = 0.70


# E18 / E20 specialist gates
E20_MR_GATE = None

E20_MS_GATE = 0.85

E20_NP_GATE = 0.90

E20_PETOIL_GATE = 0.94


# ==================================================================================================
# 5. EXISTING E21-B CONFIGURATION
#
# Used as the current baseline we want E22-A to beat.
# ==================================================================================================

E21B_MR_GATE = 0.97

E21B_MS_GATE = 0.99


# ==================================================================================================
# 6. E22 CLASS-CONDITIONAL ROUTES
# ==================================================================================================

# If E21-A proposes MR, only allow correction from these current classes.
MR_ALLOWED_FROM = {

    MS,
    HDPE,
    PET,
}


# If E21-A proposes MS, only allow correction from these current classes.
MS_ALLOWED_FROM = {

    MR,
    HDPE,
    PET,
    ECAL,
}


print()
print("=" * 115)
print("E22-A ALLOWED ROUTES")
print("=" * 115)

print()

print(
    "E21-A -> Mixed Rigid allowed from:"
)

for idx in sorted(
    MR_ALLOWED_FROM
):

    print(
        f"  {CLASS_NAMES[idx]}"
    )


print()
print(
    "E21-A -> Mixed Soft allowed from:"
)

for idx in sorted(
    MS_ALLOWED_FROM
):

    print(
        f"  {CLASS_NAMES[idx]}"
    )


# ==================================================================================================
# 7. SEARCH GRID
#
# OFF is included so validation is allowed to decide that one specialist route
# should not be used at all.
# ==================================================================================================

MR_GATES = [

    None,

    0.60,
    0.65,
    0.70,
    0.75,
    0.80,
    0.85,
    0.90,
    0.92,
    0.94,
    0.95,
]


MS_GATES = [

    None,

    0.60,
    0.65,
    0.70,
    0.75,
    0.80,
    0.85,
    0.90,
    0.92,
    0.94,
    0.95,
]


# ==================================================================================================
# 8. LOAD CACHE
# ==================================================================================================

with open(
    CACHE_PATH,
    "r",
    encoding="utf-8"
) as f:

    cache = json.load(f)


print()
print(
    f"Cached predictions : "
    f"{len(cache):,}"
)


num_e21 = sum(

    1

    for item in cache

    if item.get(
        "e21"
    ) is not None
)


print(
    f"With E21-A output  : "
    f"{num_e21:,}"
)


if num_e21 != len(
    cache
):

    raise RuntimeError(
        "Not all cached predictions contain E21-A probabilities."
    )


# ==================================================================================================
# 9. LOAD COCO GT
# ==================================================================================================

coco_gt = COCO(
    str(
        GT_PATH
    )
)


# ==================================================================================================
# 10. HELPER — RECONSTRUCT PRE-E21-A PIPELINE
#
# This gives the prediction immediately after:
#
#     YOLO
#     MobileNet/E16
#     E18-B/E20
#
# but BEFORE E21-A is allowed to modify it.
# ==================================================================================================

def reconstruct_pre_e21(
    item
):

    e16_final_idx = int(
        item[
            "e16_final_idx"
        ]
    )


    mn_probs = np.asarray(
        item[
            "mn_probs"
        ],
        dtype=np.float32
    )


    final_idx = (
        e16_final_idx
    )


    classifier_prob = float(
        mn_probs[
            final_idx
        ]
    )


    source = (
        "upstream"
    )


    conv = item.get(
        "conv"
    )


    if conv is not None:

        conv_global_idx = int(
            conv[
                "conv_global_idx"
            ]
        )


        conv_top_prob = float(
            conv[
                "conv_top_prob"
            ]
        )


        required_gate = None


        if conv_global_idx == MR:

            required_gate = (
                E20_MR_GATE
            )


        elif conv_global_idx == MS:

            required_gate = (
                E20_MS_GATE
            )


        elif conv_global_idx == NP:

            required_gate = (
                E20_NP_GATE
            )


        elif conv_global_idx == PET_OIL:

            required_gate = (
                E20_PETOIL_GATE
            )


        if (
            required_gate is not None
            and conv_top_prob
            >= required_gate
        ):

            final_idx = (
                conv_global_idx
            )


            classifier_prob = (
                conv_top_prob
            )


            source = (
                "e18"
            )


    return (

        final_idx,
        classifier_prob,
        source,
    )


# ==================================================================================================
# 11. COCO EVALUATION HELPERS
# ==================================================================================================

def valid_mean(
    values
):

    values = np.asarray(
        values
    )


    valid = values[
        values > -1
    ]


    if valid.size == 0:

        return float(
            "nan"
        )


    return float(
        np.mean(
            valid
        )
    )


def evaluate_predictions(
    predictions
):

    coco_dt = coco_gt.loadRes(
        predictions
    )


    evaluator = COCOeval(

        coco_gt,
        coco_dt,
        "bbox"
    )


    evaluator.params.maxDets = [

        1,
        10,
        100,
    ]


    evaluator.evaluate()
    evaluator.accumulate()


    precision = (
        evaluator.eval[
            "precision"
        ]
    )


    recall = (
        evaluator.eval[
            "recall"
        ]
    )


    iou_thresholds = (
        evaluator.params.iouThrs
    )


    idx50 = int(

        np.where(
            np.isclose(
                iou_thresholds,
                0.50
            )
        )[0][0]
    )


    idx75 = int(

        np.where(
            np.isclose(
                iou_thresholds,
                0.75
            )
        )[0][0]
    )


    overall = {

        "AP50_95":
            valid_mean(
                precision[
                    :,
                    :,
                    :,
                    0,
                    -1
                ]
            ),

        "AP50":
            valid_mean(
                precision[
                    idx50,
                    :,
                    :,
                    0,
                    -1
                ]
            ),

        "AP75":
            valid_mean(
                precision[
                    idx75,
                    :,
                    :,
                    0,
                    -1
                ]
            ),

        "AR100":
            valid_mean(
                recall[
                    :,
                    :,
                    0,
                    -1
                ]
            ),
    }


    class_metrics = {}


    for class_idx, class_name in enumerate(
        CLASS_NAMES
    ):

        class_metrics[
            class_name
        ] = {

            "AP50":
                valid_mean(
                    precision[
                        idx50,
                        :,
                        class_idx,
                        0,
                        -1
                    ]
                ),

            "AP50_95":
                valid_mean(
                    precision[
                        :,
                        :,
                        class_idx,
                        0,
                        -1
                    ]
                ),
        }


    six_plastic_mean_ap50 = float(

        np.mean(
            [

                class_metrics[
                    CLASS_NAMES[
                        idx
                    ]
                ][
                    "AP50"
                ]

                for idx
                in PLASTIC_CLASS_INDICES
            ]
        )
    )


    mr_ms_mean_ap50 = float(

        np.mean(
            [

                class_metrics[
                    "mixed_plastic_rigid"
                ][
                    "AP50"
                ],

                class_metrics[
                    "mixed_plastic_soft"
                ][
                    "AP50"
                ],
            ]
        )
    )


    return {

        "overall":
            overall,

        "class_metrics":
            class_metrics,

        "six_plastic_mean_AP50":
            six_plastic_mean_ap50,

        "MR_MS_mean_AP50":
            mr_ms_mean_ap50,

        "nonplastic_AP50":
            class_metrics[
                "non_plastic"
            ][
                "AP50"
            ],
    }


# ==================================================================================================
# 12. BUILD PREDICTIONS — EXISTING E21-B
#
# Existing E21-B:
#
#     MR gate 0.97
#     MS gate 0.99
#
# No class-conditional restriction.
# ==================================================================================================

def build_existing_e21b():

    predictions = []

    source_counts = Counter()

    accepted = Counter()


    for item in cache:

        (
            final_idx,
            classifier_prob,
            source
        ) = reconstruct_pre_e21(
            item
        )


        e21 = item[
            "e21"
        ]


        e21_global_idx = int(
            e21[
                "global_idx"
            ]
        )


        e21_top_prob = float(
            e21[
                "top_prob"
            ]
        )


        if (
            e21_global_idx == MR
            and e21_top_prob
            >= E21B_MR_GATE
        ):

            final_idx = MR

            classifier_prob = (
                e21_top_prob
            )

            source = "e21"


            accepted[
                "mixed_plastic_rigid"
            ] += 1


        elif (
            e21_global_idx == MS
            and e21_top_prob
            >= E21B_MS_GATE
        ):

            final_idx = MS

            classifier_prob = (
                e21_top_prob
            )

            source = "e21"


            accepted[
                "mixed_plastic_soft"
            ] += 1


        yolo_conf = max(
            float(
                item[
                    "yolo_conf"
                ]
            ),
            1e-12
        )


        classifier_prob = max(
            float(
                classifier_prob
            ),
            1e-12
        )


        score = (

            yolo_conf
            ** ALPHA

        ) * (

            classifier_prob
            ** (
                1.0
                - ALPHA
            )
        )


        x1, y1, x2, y2 = [

            float(v)

            for v in item[
                "bbox"
            ]
        ]


        width = (
            x2 - x1
        )

        height = (
            y2 - y1
        )


        if (
            width <= 0
            or height <= 0
        ):

            continue


        predictions.append(
            {

                "image_id":
                    int(
                        item[
                            "image_id"
                        ]
                    ),

                "category_id":
                    int(
                        final_idx
                        + 1
                    ),

                "bbox": [

                    x1,
                    y1,
                    width,
                    height,
                ],

                "score":
                    float(
                        score
                    ),
            }
        )


        source_counts[
            source
        ] += 1


    return {

        "predictions":
            predictions,

        "source_counts":
            dict(
                source_counts
            ),

        "accepted":
            dict(
                accepted
            ),
    }


# ==================================================================================================
# 13. BUILD E22 CLASS-CONDITIONAL PREDICTIONS
# ==================================================================================================

def build_e22_predictions(
    mr_gate,
    ms_gate
):

    predictions = []

    source_counts = Counter()

    route_proposed = Counter()

    route_accepted = Counter()


    for item in cache:

        (
            pre_e21_idx,
            classifier_prob,
            source
        ) = reconstruct_pre_e21(
            item
        )


        # Keep copy because routing condition is based on the
        # prediction BEFORE E21-A acts.
        final_idx = (
            pre_e21_idx
        )


        e21 = item[
            "e21"
        ]


        e21_global_idx = int(
            e21[
                "global_idx"
            ]
        )


        e21_top_prob = float(
            e21[
                "top_prob"
            ]
        )


        # ==========================================================================================
        # E21-A -> MIXED RIGID
        # ==========================================================================================

        if e21_global_idx == MR:

            route_name = (
                f"{CLASS_NAMES[pre_e21_idx]}"
                f"->mixed_plastic_rigid"
            )


            route_proposed[
                route_name
            ] += 1


            if (

                mr_gate is not None

                and

                pre_e21_idx
                in MR_ALLOWED_FROM

                and

                e21_top_prob
                >= mr_gate

            ):

                final_idx = MR

                classifier_prob = (
                    e21_top_prob
                )

                source = (
                    "e21_conditional"
                )


                route_accepted[
                    route_name
                ] += 1


        # ==========================================================================================
        # E21-A -> MIXED SOFT
        # ==========================================================================================

        elif e21_global_idx == MS:

            route_name = (
                f"{CLASS_NAMES[pre_e21_idx]}"
                f"->mixed_plastic_soft"
            )


            route_proposed[
                route_name
            ] += 1


            if (

                ms_gate is not None

                and

                pre_e21_idx
                in MS_ALLOWED_FROM

                and

                e21_top_prob
                >= ms_gate

            ):

                final_idx = MS

                classifier_prob = (
                    e21_top_prob
                )

                source = (
                    "e21_conditional"
                )


                route_accepted[
                    route_name
                ] += 1


        # ==========================================================================================
        # FINAL SCORE
        # ==========================================================================================

        yolo_conf = max(
            float(
                item[
                    "yolo_conf"
                ]
            ),
            1e-12
        )


        classifier_prob = max(
            float(
                classifier_prob
            ),
            1e-12
        )


        score = (

            yolo_conf
            ** ALPHA

        ) * (

            classifier_prob
            ** (
                1.0
                - ALPHA
            )
        )


        x1, y1, x2, y2 = [

            float(v)

            for v in item[
                "bbox"
            ]
        ]


        width = (
            x2 - x1
        )

        height = (
            y2 - y1
        )


        if (
            width <= 0
            or height <= 0
        ):

            continue


        predictions.append(
            {

                "image_id":
                    int(
                        item[
                            "image_id"
                        ]
                    ),

                "category_id":
                    int(
                        final_idx
                        + 1
                    ),

                "bbox": [

                    x1,
                    y1,
                    width,
                    height,
                ],

                "score":
                    float(
                        score
                    ),
            }
        )


        source_counts[
            source
        ] += 1


    return {

        "predictions":
            predictions,

        "source_counts":
            dict(
                source_counts
            ),

        "route_proposed":
            dict(
                route_proposed
            ),

        "route_accepted":
            dict(
                route_accepted
            ),
    }


# ==================================================================================================
# 14. EVALUATE EXISTING E21-B BASELINE
# ==================================================================================================

print()
print("=" * 115)
print("RECONSTRUCTING EXISTING E21-B VALIDATION BASELINE")
print("=" * 115)


baseline_build = (
    build_existing_e21b()
)


baseline_eval = evaluate_predictions(
    baseline_build[
        "predictions"
    ]
)


print()
print(
    f"Six-plastic mean AP50 : "
    f"{baseline_eval['six_plastic_mean_AP50'] * 100:.4f}%"
)


print(
    f"MR AP50               : "
    f"{baseline_eval['class_metrics']['mixed_plastic_rigid']['AP50'] * 100:.4f}%"
)


print(
    f"MS AP50               : "
    f"{baseline_eval['class_metrics']['mixed_plastic_soft']['AP50'] * 100:.4f}%"
)


print(
    f"MR/MS mean AP50       : "
    f"{baseline_eval['MR_MS_mean_AP50'] * 100:.4f}%"
)


print(
    f"Overall AP50-95       : "
    f"{baseline_eval['overall']['AP50_95'] * 100:.4f}%"
)


print(
    f"Overall AP50          : "
    f"{baseline_eval['overall']['AP50'] * 100:.4f}%"
)


print(
    f"non_plastic AP50      : "
    f"{baseline_eval['nonplastic_AP50'] * 100:.4f}%"
)


# ==================================================================================================
# 15. GRID SEARCH
# ==================================================================================================

def gate_label(
    value
):

    if value is None:

        return "OFF"


    return f"{value:.2f}"


results = []


total_combinations = (

    len(
        MR_GATES
    )

    *

    len(
        MS_GATES
    )
)


counter = 0


print()
print("=" * 115)
print("E22-A CLASS-CONDITIONAL GATE SEARCH — VALIDATION ONLY")
print("=" * 115)


for mr_gate in MR_GATES:

    for ms_gate in MS_GATES:

        counter += 1


        built = build_e22_predictions(

            mr_gate=mr_gate,

            ms_gate=ms_gate
        )


        evaluation = evaluate_predictions(

            built[
                "predictions"
            ]
        )


        result = {

            "mr_gate":
                mr_gate,

            "ms_gate":
                ms_gate,

            "six_plastic_mean_AP50":
                evaluation[
                    "six_plastic_mean_AP50"
                ],

            "MR_MS_mean_AP50":
                evaluation[
                    "MR_MS_mean_AP50"
                ],

            "overall":
                evaluation[
                    "overall"
                ],

            "class_metrics":
                evaluation[
                    "class_metrics"
                ],

            "nonplastic_AP50":
                evaluation[
                    "nonplastic_AP50"
                ],

            "source_counts":
                built[
                    "source_counts"
                ],

            "route_proposed":
                built[
                    "route_proposed"
                ],

            "route_accepted":
                built[
                    "route_accepted"
                ],
        }


        results.append(
            result
        )


        print(

            f"[{counter:03d}/{total_combinations}] "

            f"MR={gate_label(mr_gate):>4s} "

            f"MS={gate_label(ms_gate):>4s} | "

            f"Plastic="
            f"{evaluation['six_plastic_mean_AP50'] * 100:7.3f}% | "

            f"MR="
            f"{evaluation['class_metrics']['mixed_plastic_rigid']['AP50'] * 100:6.2f}% | "

            f"MS="
            f"{evaluation['class_metrics']['mixed_plastic_soft']['AP50'] * 100:6.2f}% | "

            f"AP="
            f"{evaluation['overall']['AP50_95'] * 100:6.3f}%"
        )


# ==================================================================================================
# 16. RANK CONFIGURATIONS
# ==================================================================================================

ranked = sorted(

    results,

    key=lambda x: (

        x[
            "six_plastic_mean_AP50"
        ],

        x[
            "MR_MS_mean_AP50"
        ],

        x[
            "overall"
        ][
            "AP50_95"
        ],
    ),

    reverse=True
)


best = (
    ranked[
        0
    ]
)


# ==================================================================================================
# 17. CALCULATE DELTAS VS EXISTING E21-B
# ==================================================================================================

delta = {

    "six_plastic_mean_AP50_pp":

        (
            best[
                "six_plastic_mean_AP50"
            ]

            -

            baseline_eval[
                "six_plastic_mean_AP50"
            ]
        )
        * 100.0,


    "MR_AP50_pp":

        (
            best[
                "class_metrics"
            ][
                "mixed_plastic_rigid"
            ][
                "AP50"
            ]

            -

            baseline_eval[
                "class_metrics"
            ][
                "mixed_plastic_rigid"
            ][
                "AP50"
            ]
        )
        * 100.0,


    "MS_AP50_pp":

        (
            best[
                "class_metrics"
            ][
                "mixed_plastic_soft"
            ][
                "AP50"
            ]

            -

            baseline_eval[
                "class_metrics"
            ][
                "mixed_plastic_soft"
            ][
                "AP50"
            ]
        )
        * 100.0,


    "MR_MS_mean_AP50_pp":

        (
            best[
                "MR_MS_mean_AP50"
            ]

            -

            baseline_eval[
                "MR_MS_mean_AP50"
            ]
        )
        * 100.0,


    "overall_AP50_95_pp":

        (
            best[
                "overall"
            ][
                "AP50_95"
            ]

            -

            baseline_eval[
                "overall"
            ][
                "AP50_95"
            ]
        )
        * 100.0,


    "overall_AP50_pp":

        (
            best[
                "overall"
            ][
                "AP50"
            ]

            -

            baseline_eval[
                "overall"
            ][
                "AP50"
            ]
        )
        * 100.0,


    "ECAL_AP50_pp":

        (
            best[
                "class_metrics"
            ][
                "ecal"
            ][
                "AP50"
            ]

            -

            baseline_eval[
                "class_metrics"
            ][
                "ecal"
            ][
                "AP50"
            ]
        )
        * 100.0,


    "HDPE_AP50_pp":

        (
            best[
                "class_metrics"
            ][
                "hdpe"
            ][
                "AP50"
            ]

            -

            baseline_eval[
                "class_metrics"
            ][
                "hdpe"
            ][
                "AP50"
            ]
        )
        * 100.0,


    "PET_AP50_pp":

        (
            best[
                "class_metrics"
            ][
                "pet"
            ][
                "AP50"
            ]

            -

            baseline_eval[
                "class_metrics"
            ][
                "pet"
            ][
                "AP50"
            ]
        )
        * 100.0,


    "nonplastic_AP50_pp":

        (
            best[
                "nonplastic_AP50"
            ]

            -

            baseline_eval[
                "nonplastic_AP50"
            ]
        )
        * 100.0,
}


best[
    "delta_vs_E21B"
] = delta


# ==================================================================================================
# 18. PRINT TOP 20
# ==================================================================================================

print()
print()
print("=" * 115)
print("TOP 20 E22-A VALIDATION CONFIGURATIONS")
print("=" * 115)


print(
    f"\n"
    f"{'Rank':>4s} "
    f"{'MR':>6s} "
    f"{'MS':>6s} "
    f"{'Plastic AP50':>14s} "
    f"{'MR AP50':>10s} "
    f"{'MS AP50':>10s} "
    f"{'MR/MS':>10s} "
    f"{'AP50-95':>10s}"
)


print(
    "-" * 92
)


for rank, result in enumerate(
    ranked[
        :20
    ],
    start=1
):

    print(

        f"{rank:4d} "

        f"{gate_label(result['mr_gate']):>6s} "

        f"{gate_label(result['ms_gate']):>6s} "

        f"{result['six_plastic_mean_AP50'] * 100:13.4f}% "

        f"{result['class_metrics']['mixed_plastic_rigid']['AP50'] * 100:9.2f}% "

        f"{result['class_metrics']['mixed_plastic_soft']['AP50'] * 100:9.2f}% "

        f"{result['MR_MS_mean_AP50'] * 100:9.2f}% "

        f"{result['overall']['AP50_95'] * 100:9.3f}%"
    )


# ==================================================================================================
# 19. BEST CONFIGURATION
# ==================================================================================================

print()
print("=" * 115)
print("BEST E22-A CONFIGURATION — VALIDATION")
print("=" * 115)


print()
print(
    f"E21-A MR conditional gate : "
    f"{gate_label(best['mr_gate'])}"
)


print(
    f"E21-A MS conditional gate : "
    f"{gate_label(best['ms_gate'])}"
)


print()
print(
    f"Six-plastic mean AP50     : "
    f"{best['six_plastic_mean_AP50'] * 100:.4f}%"
)


print(
    f"MR AP50                   : "
    f"{best['class_metrics']['mixed_plastic_rigid']['AP50'] * 100:.4f}%"
)


print(
    f"MS AP50                   : "
    f"{best['class_metrics']['mixed_plastic_soft']['AP50'] * 100:.4f}%"
)


print(
    f"MR/MS mean AP50           : "
    f"{best['MR_MS_mean_AP50'] * 100:.4f}%"
)


print()
print(
    f"Overall AP50-95           : "
    f"{best['overall']['AP50_95'] * 100:.4f}%"
)


print(
    f"Overall AP50              : "
    f"{best['overall']['AP50'] * 100:.4f}%"
)


print(
    f"Overall AP75              : "
    f"{best['overall']['AP75'] * 100:.4f}%"
)


print(
    f"AR100                     : "
    f"{best['overall']['AR100'] * 100:.4f}%"
)


print(
    f"non_plastic AP50          : "
    f"{best['nonplastic_AP50'] * 100:.4f}%"
)


# ==================================================================================================
# 20. DELTA VS E21-B
# ==================================================================================================

print()
print("=" * 115)
print("DELTA VS EXISTING E21-B VALIDATION")
print("=" * 115)


for metric_name, value in (
    delta.items()
):

    print(
        f"{metric_name:32s}: "
        f"{value:+.4f} pp"
    )


# ==================================================================================================
# 21. CLASS-WISE RESULTS
# ==================================================================================================

print()
print("=" * 115)
print("BEST E22-A — CLASS-WISE VALIDATION")
print("=" * 115)


print(
    f"\n"
    f"{'Class':28s}"
    f"{'AP50':>14s}"
    f"{'AP50-95':>14s}"
)


print(
    "-" * 56
)


for class_name in CLASS_NAMES:

    metrics = (
        best[
            "class_metrics"
        ][
            class_name
        ]
    )


    print(

        f"{class_name:28s}"

        f"{metrics['AP50'] * 100:13.2f}%"

        f"{metrics['AP50_95'] * 100:13.2f}%"
    )


# ==================================================================================================
# 22. ROUTING DETAILS
# ==================================================================================================

print()
print("=" * 115)
print("BEST E22-A — CLASS-CONDITIONAL ROUTING")
print("=" * 115)


print()
print(
    "Accepted E21-A routes:"
)


accepted_routes = (
    best[
        "route_accepted"
    ]
)


if len(
    accepted_routes
) == 0:

    print(
        "  No E21-A conditional overrides accepted."
    )


else:

    for route_name, count in sorted(

        accepted_routes.items(),

        key=lambda x:
            x[
                1
            ],

        reverse=True
    ):

        print(
            f"{route_name:45s}: "
            f"{count:6,d}"
        )


print()
print(
    "Final prediction sources:"
)


total_sources = sum(
    best[
        "source_counts"
    ].values()
)


for source, count in (
    best[
        "source_counts"
    ].items()
):

    percentage = (

        100.0
        * count
        / total_sources
    )


    print(

        f"{source:24s}: "

        f"{count:7,d} "

        f"({percentage:6.2f}%)"
    )


# ==================================================================================================
# 23. DECISION CHECK
#
# Do not automatically promote E22-A merely because one metric moved.
#
# The primary condition is improvement in six-plastic mean AP50.
# ==================================================================================================

improves_primary = (

    best[
        "six_plastic_mean_AP50"
    ]

    >

    baseline_eval[
        "six_plastic_mean_AP50"
    ]
)


print()
print("=" * 115)
print("E22-A VALIDATION DECISION")
print("=" * 115)


if improves_primary:

    print()
    print(
        "E22-A improves the PRIMARY validation metric."
    )

    print(
        "Candidate for frozen held-out TEST evaluation."
    )


else:

    print()
    print(
        "E22-A does NOT improve the PRIMARY validation metric."
    )

    print(
        "Keep E21-B as the final model."
    )


# ==================================================================================================
# 24. SAVE GRID RESULTS
# ==================================================================================================

with open(
    GRID_RESULTS_PATH,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        ranked,
        f,
        indent=2
    )


# ==================================================================================================
# 25. SAVE BEST PREDICTIONS
# ==================================================================================================

best_build = build_e22_predictions(

    mr_gate=best[
        "mr_gate"
    ],

    ms_gate=best[
        "ms_gate"
    ]
)


with open(
    BEST_PREDICTIONS_PATH,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        best_build[
            "predictions"
        ],
        f
    )


# ==================================================================================================
# 26. SAVE CONFIGURATION
# ==================================================================================================

best_output = {

    "experiment":
        "E22-A",

    "description":
        (
            "Validation-selected class-conditional "
            "E21-A MR/MS routing refinement"
        ),

    "selection_dataset":
        "validation",

    "test_used":
        False,

    "upstream_pipeline":
        "Frozen pre-E21-A E21-B pipeline",

    "MR_allowed_from":
        [
            CLASS_NAMES[
                idx
            ]
            for idx in sorted(
                MR_ALLOWED_FROM
            )
        ],

    "MS_allowed_from":
        [
            CLASS_NAMES[
                idx
            ]
            for idx in sorted(
                MS_ALLOWED_FROM
            )
        ],

    "selection":
        {

            "primary":
                "six-plastic mean AP50",

            "secondary":
                "MR/MS mean AP50",

            "tie_break":
                "overall AP50-95",
        },

    "existing_E21B_baseline":
        baseline_eval,

    "best_E22A":
        best,

    "promote_to_test":
        bool(
            improves_primary
        ),
}


with open(
    BEST_CONFIG_PATH,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        best_output,
        f,
        indent=2
    )


# ==================================================================================================
# 27. SAVE SUMMARY
# ==================================================================================================

summary_lines = [

    "=" * 115,

    "E22-A — CLASS-CONDITIONAL E21-A ROUTING REFINEMENT",

    "=" * 115,

    "",

    "VALIDATION ONLY",

    "TEST SET NOT USED",

    "",

    "Allowed E21-A -> MR routes:",

    "  MS -> MR",

    "  HDPE -> MR",

    "  PET -> MR",

    "",

    "Allowed E21-A -> MS routes:",

    "  MR -> MS",

    "  HDPE -> MS",

    "  PET -> MS",

    "  ECAL -> MS",

    "",

    "Existing E21-B validation:",

    f"  Six-plastic mean AP50 = "
    f"{baseline_eval['six_plastic_mean_AP50'] * 100:.4f}%",

    f"  MR AP50               = "
    f"{baseline_eval['class_metrics']['mixed_plastic_rigid']['AP50'] * 100:.4f}%",

    f"  MS AP50               = "
    f"{baseline_eval['class_metrics']['mixed_plastic_soft']['AP50'] * 100:.4f}%",

    "",

    "Best E22-A:",

    f"  MR gate                = "
    f"{gate_label(best['mr_gate'])}",

    f"  MS gate                = "
    f"{gate_label(best['ms_gate'])}",

    f"  Six-plastic mean AP50 = "
    f"{best['six_plastic_mean_AP50'] * 100:.4f}%",

    f"  MR AP50               = "
    f"{best['class_metrics']['mixed_plastic_rigid']['AP50'] * 100:.4f}%",

    f"  MS AP50               = "
    f"{best['class_metrics']['mixed_plastic_soft']['AP50'] * 100:.4f}%",

    f"  Overall AP50-95       = "
    f"{best['overall']['AP50_95'] * 100:.4f}%",

    "",

    "Delta vs E21-B:",
]


for metric_name, value in (
    delta.items()
):

    summary_lines.append(

        f"  {metric_name:32s}: "
        f"{value:+.4f} pp"
    )


summary_lines.extend(
    [

        "",

        (
            "PROMOTE TO TEST"
            if improves_primary
            else
            "DO NOT PROMOTE — KEEP E21-B"
        ),
    ]
)


with open(
    SUMMARY_PATH,
    "w",
    encoding="utf-8"
) as f:

    f.write(
        "\n".join(
            summary_lines
        )
    )


# ==================================================================================================
# 28. COMPLETE
# ==================================================================================================

print()
print("=" * 115)
print("E22-A COMPLETE")
print("=" * 115)


print()
print(
    f"Grid results     : "
    f"{GRID_RESULTS_PATH}"
)


print(
    f"Best config      : "
    f"{BEST_CONFIG_PATH}"
)


print(
    f"Best predictions : "
    f"{BEST_PREDICTIONS_PATH}"
)


print(
    f"Summary          : "
    f"{SUMMARY_PATH}"
)

E22-A — CLASS-CONDITIONAL E21-A ROUTING REFINEMENT

Validation GT : C:\Users\varda\Documents\_My Computer\COMMON_space\Learning\Upgrad_EPGP_MS\upgrad_IIITB_MS_course\MS_LJMU_Material\Thesis_Code\runs\sortwaste\E18D_class_selective_convnext\E18D_val_gt_7class.json
E21-B cache   : C:\Users\varda\Documents\_My Computer\COMMON_space\Learning\Upgrad_EPGP_MS\upgrad_IIITB_MS_course\MS_LJMU_Material\Thesis_Code\runs\sortwaste\E21B_end_to_end_validation\E21B_cached_predictions_with_E21A.json
Output        : C:\Users\varda\Documents\_My Computer\COMMON_space\Learning\Upgrad_EPGP_MS\upgrad_IIITB_MS_course\MS_LJMU_Material\Thesis_Code\runs\sortwaste\E22A_class_conditional_E21_routing

E22-A ALLOWED ROUTES

E21-A -> Mixed Rigid allowed from:
  hdpe
  mixed_plastic_soft
  pet

E21-A -> Mixed Soft allowed from:
  ecal
  hdpe
  mixed_plastic_rigid
  pet

Cached predictions : 55,605
With E21-A output  : 55,605
loading annotations into memory...
Done (t=0.05s)
creating index...
index created!

RECONSTRU

# Model E23 - Binary MR/MS Hard-Pair

## E23-A — Binary MR/MS Hard-Pair ConvNeXt-Tiny Specialist

In [ ]:
# E23-A — BINARY MIXED-RIGID VS MIXED-SOFT HARD-PAIR SPECIALIST
#
# TRAIN + VALIDATION ONLY
# TEST IS NOT USED
#
# --------------------------------------------------------------------------------------------------
# GOAL
# --------------------------------------------------------------------------------------------------
#
# Train a dedicated binary classifier whose only task is:
#
#       Mixed Rigid Plastic  vs  Mixed Soft Plastic
#
#
# CLASS MAPPING
# --------------------------------------------------------------------------------------------------
#
#       0 = mixed_plastic_rigid
#       1 = mixed_plastic_soft
#
#
# HARD-PAIR SAMPLING
# --------------------------------------------------------------------------------------------------
#
# E3Y-B MobileNet is used ONLY on TRAIN crops to estimate difficulty.
#
# For GT Mixed Rigid:
#
#       hardness = P(Mixed Soft)
#
# For GT Mixed Soft:
#
#       hardness = P(Mixed Rigid)
#
# Sampling weight:
#
#       class_balance_weight * (1 + HARDNESS_BOOST * hardness)
#
# Therefore crops that the existing classifier finds confusing are sampled more frequently.
#
#
# MODEL
# --------------------------------------------------------------------------------------------------
#
#       ConvNeXt-Tiny
#       ImageNet pretrained
#       2-class output
#
#
# MODEL SELECTION
# --------------------------------------------------------------------------------------------------
#
#       Validation Macro-F1
#
#
# E23-B, later:
#       integrate this binary specialist on top of the frozen E20-A pipeline
#
# ==================================================================================================


import json
import random
import time
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd

import torch
import torch.nn as nn

from torch.utils.data import (
    Dataset,
    DataLoader,
    WeightedRandomSampler,
)

from torchvision import models
from torchvision.models import ConvNeXt_Tiny_Weights
from torchvision.transforms import v2

from PIL import Image

from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score,
    f1_score,
    balanced_accuracy_score,
)

from tqdm.auto import tqdm


# ==================================================================================================
# 1. REPRODUCIBILITY
# ==================================================================================================

SEED = 42

random.seed(SEED)
np.random.seed(SEED)

torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)


# ==================================================================================================
# 2. DEVICE
# ==================================================================================================

DEVICE = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)


print("=" * 110)
print("E23-A — BINARY MR/MS HARD-PAIR SPECIALIST")
print("=" * 110)

print()
print(f"PyTorch : {torch.__version__}")
print(f"Device  : {DEVICE}")

if DEVICE.type == "cuda":

    print(
        f"GPU     : "
        f"{torch.cuda.get_device_name(0)}"
    )


# ==================================================================================================
# 3. PATHS
# ==================================================================================================

BASE = Path(
    r"C:\Users\varda\Documents\_My Computer\COMMON_space\Learning"
    r"\Upgrad_EPGP_MS\upgrad_IIITB_MS_course\MS_LJMU_Material"
)


DATASET_ROOT = (
    BASE
    / "Topic Data"
    / "SortWaste"
    / "dataset"
    / "dataset"
)


THESIS_CODE = (
    BASE
    / "Thesis_Code"
)


# --------------------------------------------------------------------------------------------------
# Existing GT crop folders
# --------------------------------------------------------------------------------------------------

TRAIN_CROP_ROOT = (
    DATASET_ROOT
    / "cropped_data"
    / "train"
)


VAL_CROP_ROOT = (
    DATASET_ROOT
    / "cropped_data"
    / "val"
)


# --------------------------------------------------------------------------------------------------
# Existing E3Y-B MobileNet
# 7-class model
# --------------------------------------------------------------------------------------------------

MOBILENET_CKPT = (
    DATASET_ROOT
    / "yolo_mobilenet_crops_E3Y"
    / "mobilenet_results"
    / "E3Y_B_class_weighted"
    / "E3Y_B_MobileNetV3Large_best.pth"
)


# --------------------------------------------------------------------------------------------------
# E23-A output
# --------------------------------------------------------------------------------------------------

OUTPUT_DIR = (
    THESIS_CODE
    / "runs"
    / "sortwaste"
    / "E23A_binary_MR_MS_hardpair_convnext"
)


OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)


BEST_MODEL_PATH = (
    OUTPUT_DIR
    / "E23A_ConvNeXtTiny_MR_MS_best.pth"
)


HARDNESS_CACHE_PATH = (
    OUTPUT_DIR
    / "E23A_train_hardness_cache.json"
)


TRAIN_MANIFEST_PATH = (
    OUTPUT_DIR
    / "E23A_train_manifest.json"
)


VAL_MANIFEST_PATH = (
    OUTPUT_DIR
    / "E23A_val_manifest.json"
)


HISTORY_PATH = (
    OUTPUT_DIR
    / "E23A_training_history.json"
)


RESULTS_PATH = (
    OUTPUT_DIR
    / "E23A_validation_results.json"
)


CONFUSION_PATH = (
    OUTPUT_DIR
    / "E23A_validation_confusion_matrix.csv"
)


NORMALIZED_CONFUSION_PATH = (
    OUTPUT_DIR
    / "E23A_validation_confusion_normalized.csv"
)


# ==================================================================================================
# 4. VERIFY PATHS
# ==================================================================================================

for path in [

    TRAIN_CROP_ROOT,
    VAL_CROP_ROOT,
    MOBILENET_CKPT,

]:

    if not path.exists():

        raise FileNotFoundError(
            f"Required path not found:\n{path}"
        )


print()
print(f"Train crops : {TRAIN_CROP_ROOT}")
print(f"Val crops   : {VAL_CROP_ROOT}")
print(f"MobileNet   : {MOBILENET_CKPT}")
print(f"Output      : {OUTPUT_DIR}")


# ==================================================================================================
# 5. BINARY CLASS MAPPING
# ==================================================================================================

E23_CLASS_NAMES = [

    "mixed_plastic_rigid",

    "mixed_plastic_soft",
]


E23_NAME_TO_IDX = {

    name: idx

    for idx, name
    in enumerate(
        E23_CLASS_NAMES
    )
}


MR_BINARY = 0
MS_BINARY = 1


# ==================================================================================================
# 6. EXISTING E3Y-B MOBILENET MAPPING
#
# 0 = ecal
# 1 = hdpe
# 2 = mixed_plastic_rigid
# 3 = mixed_plastic_soft
# 4 = non_plastic
# 5 = pet
# 6 = pet_oil
# ==================================================================================================

MN_CLASS_NAMES = [

    "ecal",
    "hdpe",
    "mixed_plastic_rigid",
    "mixed_plastic_soft",
    "non_plastic",
    "pet",
    "pet_oil",
]


MN_MR = 2

MN_MS = 3


# ==================================================================================================
# 7. CONFIG
# ==================================================================================================

IMAGE_SIZE = 224


# MobileNet mining
MINE_BATCH_SIZE = 64


# ConvNeXt training
TRAIN_BATCH_SIZE = 24


# User's Windows/Jupyter environment
NUM_WORKERS = 0


# Binary dataset is much smaller than E21-A.
# Start conservatively because previous ConvNeXt training overfit quickly.
MAX_EPOCHS = 8


EARLY_STOP_PATIENCE = 2


LEARNING_RATE = 1e-4

WEIGHT_DECAY = 1e-4


# --------------------------------------------------------------------------------------------------
# Hard-pair boost
#
# Easy sample:
#       multiplier approximately 1
#
# Fully opposite-looking sample:
#       multiplier approximately 3
#
# No validation tuning is involved in this value.
# --------------------------------------------------------------------------------------------------

HARDNESS_BOOST = 2.0


# ==================================================================================================
# 8. IMAGE LIST HELPER
# ==================================================================================================

IMAGE_EXTENSIONS = {

    ".jpg",
    ".jpeg",
    ".png",
    ".bmp",
    ".webp",
}


def list_images(
    folder
):

    return sorted(
        [

            p

            for p in folder.rglob("*")

            if (
                p.is_file()
                and p.suffix.lower()
                in IMAGE_EXTENSIONS
            )
        ]
    )


# ==================================================================================================
# 9. COLLECT MR/MS TRAIN + VALIDATION FILES
# ==================================================================================================

train_files = {}

val_files = {}


for class_name in (
    E23_CLASS_NAMES
):

    train_files[
        class_name
    ] = list_images(

        TRAIN_CROP_ROOT
        / class_name
    )


    val_files[
        class_name
    ] = list_images(

        VAL_CROP_ROOT
        / class_name
    )


print()
print("=" * 110)
print("MR/MS DATASET")
print("=" * 110)


for class_name in E23_CLASS_NAMES:

    print(
        f"{class_name:25s} | "
        f"train={len(train_files[class_name]):,} | "
        f"val={len(val_files[class_name]):,}"
    )


train_total = sum(
    len(v)
    for v in train_files.values()
)


val_total = sum(
    len(v)
    for v in val_files.values()
)


print()
print(
    f"Total train : "
    f"{train_total:,}"
)

print(
    f"Total val   : "
    f"{val_total:,}"
)


# ==================================================================================================
# 10. LOAD E3Y-B MOBILENET
#
# Used ONLY for TRAIN hard-pair scoring.
# ==================================================================================================

print()
print("=" * 110)
print("LOADING E3Y-B MOBILENET FOR TRAIN HARD-PAIR MINING")
print("=" * 110)


mobilenet = models.mobilenet_v3_large(
    weights=None
)


mobilenet.classifier[
    3
] = nn.Linear(

    mobilenet.classifier[
        3
    ].in_features,

    7
)


checkpoint = torch.load(
    MOBILENET_CKPT,
    map_location=DEVICE
)


if (
    isinstance(
        checkpoint,
        dict
    )
    and "model_state_dict"
    in checkpoint
):

    state_dict = (
        checkpoint[
            "model_state_dict"
        ]
    )


elif (
    isinstance(
        checkpoint,
        dict
    )
    and "state_dict"
    in checkpoint
):

    state_dict = (
        checkpoint[
            "state_dict"
        ]
    )


else:

    state_dict = (
        checkpoint
    )


clean_state = {}


for key, value in (
    state_dict.items()
):

    clean_key = (

        key[7:]

        if key.startswith(
            "module."
        )

        else key
    )


    clean_state[
        clean_key
    ] = value


checkpoint_classes = int(

    clean_state[
        "classifier.3.weight"
    ].shape[
        0
    ]
)


print(
    f"Checkpoint output classes : "
    f"{checkpoint_classes}"
)


if checkpoint_classes != 7:

    raise RuntimeError(

        f"Expected E3Y-B MobileNet with 7 classes, "
        f"found {checkpoint_classes}."
    )


mobilenet.load_state_dict(
    clean_state,
    strict=True
)


mobilenet = mobilenet.to(
    DEVICE
)


mobilenet.eval()


print(
    "E3Y-B MobileNet loaded."
)


# ==================================================================================================
# 11. HARDNESS-MINING TRANSFORM
# ==================================================================================================

mine_transform = v2.Compose(
    [

        v2.Resize(
            (
                IMAGE_SIZE,
                IMAGE_SIZE
            ),
            antialias=True
        ),

        v2.ToImage(),

        v2.ToDtype(
            torch.float32,
            scale=True
        ),

        v2.Normalize(
            mean=[
                0.485,
                0.456,
                0.406
            ],
            std=[
                0.229,
                0.224,
                0.225
            ]
        ),
    ]
)


# ==================================================================================================
# 12. MINING DATASET
# ==================================================================================================

class MiningDataset(
    Dataset
):

    def __init__(
        self,
        records
    ):

        self.records = (
            records
        )


    def __len__(
        self
    ):

        return len(
            self.records
        )


    def __getitem__(
        self,
        idx
    ):

        record = (
            self.records[
                idx
            ]
        )


        path = Path(
            record[
                "path"
            ]
        )


        with Image.open(
            path
        ) as img:

            image = img.convert(
                "RGB"
            )


        image = mine_transform(
            image
        )


        return (

            image,

            int(
                record[
                    "class_idx"
                ]
            ),

            str(
                path
            ),
        )


# ==================================================================================================
# 13. BASIC TRAIN MANIFEST
# ==================================================================================================

base_train_manifest = []


for class_name in E23_CLASS_NAMES:

    class_idx = (
        E23_NAME_TO_IDX[
            class_name
        ]
    )


    for path in train_files[
        class_name
    ]:

        base_train_manifest.append(
            {

                "path":
                    str(
                        path
                    ),

                "class_name":
                    class_name,

                "class_idx":
                    class_idx,
            }
        )


# ==================================================================================================
# 14. MINE TRAIN HARDNESS OR LOAD EXISTING CACHE
# ==================================================================================================

if HARDNESS_CACHE_PATH.exists():

    print()
    print("=" * 110)
    print("FOUND EXISTING TRAIN HARDNESS CACHE")
    print("=" * 110)


    with open(
        HARDNESS_CACHE_PATH,
        "r",
        encoding="utf-8"
    ) as f:

        train_manifest = json.load(f)


    print(
        f"Loaded records : "
        f"{len(train_manifest):,}"
    )


else:

    print()
    print("=" * 110)
    print("MINING MR/MS TRAIN HARD PAIRS")
    print("=" * 110)


    mining_dataset = MiningDataset(
        base_train_manifest
    )


    mining_loader = DataLoader(

        mining_dataset,

        batch_size=MINE_BATCH_SIZE,

        shuffle=False,

        num_workers=NUM_WORKERS,

        pin_memory=(
            DEVICE.type
            == "cuda"
        )
    )


    train_manifest = []


    with torch.inference_mode():

        for (
            images,
            labels,
            paths
        ) in tqdm(

            mining_loader,

            desc="Mining MR/MS hardness"
        ):

            images = images.to(
                DEVICE,
                non_blocking=True
            )


            logits = mobilenet(
                images
            )


            probs = torch.softmax(
                logits,
                dim=1
            )


            mr_probs = (
                probs[
                    :,
                    MN_MR
                ]
            )


            ms_probs = (
                probs[
                    :,
                    MN_MS
                ]
            )


            labels_np = (
                labels
                .numpy()
            )


            for i in range(
                len(
                    paths
                )
            ):

                binary_gt = int(
                    labels_np[
                        i
                    ]
                )


                mr_prob = float(
                    mr_probs[
                        i
                    ].item()
                )


                ms_prob = float(
                    ms_probs[
                        i
                    ].item()
                )


                # ----------------------------------------------------------------------------------
                # Hardness = probability assigned to the opposite mixed-plastic class
                # ----------------------------------------------------------------------------------

                if binary_gt == MR_BINARY:

                    hardness = (
                        ms_prob
                    )


                else:

                    hardness = (
                        mr_prob
                    )


                train_manifest.append(
                    {

                        "path":
                            paths[
                                i
                            ],

                        "class_name":
                            E23_CLASS_NAMES[
                                binary_gt
                            ],

                        "class_idx":
                            binary_gt,

                        "mn_mr_prob":
                            mr_prob,

                        "mn_ms_prob":
                            ms_prob,

                        "hardness":
                            float(
                                hardness
                            ),
                    }
                )


    with open(
        HARDNESS_CACHE_PATH,
        "w",
        encoding="utf-8"
    ) as f:

        json.dump(
            train_manifest,
            f
        )


    print()
    print(
        f"Hardness cache saved : "
        f"{HARDNESS_CACHE_PATH}"
    )


# ==================================================================================================
# 15. HARDNESS SUMMARY
# ==================================================================================================

print()
print("=" * 110)
print("TRAIN HARD-PAIR SUMMARY")
print("=" * 110)


for class_idx, class_name in enumerate(
    E23_CLASS_NAMES
):

    hardness_values = np.asarray(
        [

            item[
                "hardness"
            ]

            for item in train_manifest

            if item[
                "class_idx"
            ] == class_idx
        ],
        dtype=np.float32
    )


    print()
    print(
        class_name
    )


    print(
        f"Count              : "
        f"{len(hardness_values):,}"
    )


    print(
        f"Mean opposite prob : "
        f"{hardness_values.mean():.4f}"
    )


    print(
        f"Median             : "
        f"{np.median(hardness_values):.4f}"
    )


    print(
        f">= 0.25            : "
        f"{np.mean(hardness_values >= 0.25) * 100:.2f}%"
    )


    print(
        f">= 0.50            : "
        f"{np.mean(hardness_values >= 0.50) * 100:.2f}%"
    )


    print(
        f">= 0.75            : "
        f"{np.mean(hardness_values >= 0.75) * 100:.2f}%"
    )


# ==================================================================================================
# 16. SAVE TRAIN MANIFEST
# ==================================================================================================

with open(
    TRAIN_MANIFEST_PATH,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        train_manifest,
        f,
        indent=2
    )


# ==================================================================================================
# 17. BUILD VALIDATION MANIFEST
#
# IMPORTANT:
# Validation is not mined, filtered or oversampled.
# ==================================================================================================

val_manifest = []


for class_name in E23_CLASS_NAMES:

    class_idx = (
        E23_NAME_TO_IDX[
            class_name
        ]
    )


    for path in val_files[
        class_name
    ]:

        val_manifest.append(
            {

                "path":
                    str(
                        path
                    ),

                "class_name":
                    class_name,

                "class_idx":
                    class_idx,
            }
        )


with open(
    VAL_MANIFEST_PATH,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        val_manifest,
        f,
        indent=2
    )


# ==================================================================================================
# 18. TRANSFORMS
#
# Mild/realistic augmentation.
# ==================================================================================================

train_transform = v2.Compose(
    [

        v2.RandomResizedCrop(
            size=(
                IMAGE_SIZE,
                IMAGE_SIZE
            ),
            scale=(
                0.88,
                1.0
            ),
            ratio=(
                0.92,
                1.08
            ),
            antialias=True
        ),

        v2.RandomHorizontalFlip(
            p=0.5
        ),

        v2.RandomRotation(
            degrees=8
        ),

        v2.ColorJitter(
            brightness=0.10,
            contrast=0.10,
            saturation=0.08,
            hue=0.015
        ),

        v2.ToImage(),

        v2.ToDtype(
            torch.float32,
            scale=True
        ),

        v2.Normalize(
            mean=[
                0.485,
                0.456,
                0.406
            ],
            std=[
                0.229,
                0.224,
                0.225
            ]
        ),
    ]
)


val_transform = v2.Compose(
    [

        v2.Resize(
            (
                IMAGE_SIZE,
                IMAGE_SIZE
            ),
            antialias=True
        ),

        v2.ToImage(),

        v2.ToDtype(
            torch.float32,
            scale=True
        ),

        v2.Normalize(
            mean=[
                0.485,
                0.456,
                0.406
            ],
            std=[
                0.229,
                0.224,
                0.225
            ]
        ),
    ]
)


# ==================================================================================================
# 19. DATASET
# ==================================================================================================

class ManifestDataset(
    Dataset
):

    def __init__(
        self,
        manifest,
        transform
    ):

        self.manifest = (
            manifest
        )

        self.transform = (
            transform
        )


    def __len__(
        self
    ):

        return len(
            self.manifest
        )


    def __getitem__(
        self,
        idx
    ):

        item = (
            self.manifest[
                idx
            ]
        )


        with Image.open(
            item[
                "path"
            ]
        ) as img:

            image = img.convert(
                "RGB"
            )


        image = self.transform(
            image
        )


        label = int(
            item[
                "class_idx"
            ]
        )


        return (
            image,
            label
        )


train_dataset = ManifestDataset(
    train_manifest,
    train_transform
)


val_dataset = ManifestDataset(
    val_manifest,
    val_transform
)


# ==================================================================================================
# 20. HARD-PAIR + CLASS-BALANCED SAMPLER
# ==================================================================================================

train_counts = Counter(

    item[
        "class_idx"
    ]

    for item in train_manifest
)


print()
print("=" * 110)
print("E23-A TRAIN DISTRIBUTION")
print("=" * 110)


for class_idx, class_name in enumerate(
    E23_CLASS_NAMES
):

    print(
        f"{class_name:25s}: "
        f"{train_counts[class_idx]:,}"
    )


# --------------------------------------------------------------------------------------------------
# Base inverse-frequency balancing
# --------------------------------------------------------------------------------------------------

class_balance_weights = {

    class_idx:

        len(
            train_manifest
        )
        /
        (
            len(
                E23_CLASS_NAMES
            )
            *
            train_counts[
                class_idx
            ]
        )

    for class_idx in range(
        len(
            E23_CLASS_NAMES
        )
    )
}


# --------------------------------------------------------------------------------------------------
# Final sampling weight
#
# balance factor * hard-pair multiplier
# --------------------------------------------------------------------------------------------------

sample_weights = []


for item in train_manifest:

    class_idx = int(
        item[
            "class_idx"
        ]
    )


    hardness = float(
        item[
            "hardness"
        ]
    )


    hard_multiplier = (

        1.0

        +

        HARDNESS_BOOST
        * hardness
    )


    final_weight = (

        class_balance_weights[
            class_idx
        ]

        *

        hard_multiplier
    )


    sample_weights.append(
        final_weight
    )


sample_weights = torch.tensor(
    sample_weights,
    dtype=torch.double
)


sampler = WeightedRandomSampler(

    weights=sample_weights,

    num_samples=len(
        train_manifest
    ),

    replacement=True
)


print()
print(
    f"Hardness boost : "
    f"{HARDNESS_BOOST:.2f}"
)


# ==================================================================================================
# 21. DATA LOADERS
# ==================================================================================================

train_loader = DataLoader(

    train_dataset,

    batch_size=TRAIN_BATCH_SIZE,

    sampler=sampler,

    num_workers=NUM_WORKERS,

    pin_memory=(
        DEVICE.type
        == "cuda"
    )
)


val_loader = DataLoader(

    val_dataset,

    batch_size=TRAIN_BATCH_SIZE,

    shuffle=False,

    num_workers=NUM_WORKERS,

    pin_memory=(
        DEVICE.type
        == "cuda"
    )
)


print()
print(
    f"Training batches   : "
    f"{len(train_loader):,}"
)


print(
    f"Validation batches : "
    f"{len(val_loader):,}"
)


# ==================================================================================================
# 22. BUILD CONVNEXT-TINY
# ==================================================================================================

print()
print("=" * 110)
print("BUILDING E23-A CONVNEXT-TINY")
print("=" * 110)


weights = (
    ConvNeXt_Tiny_Weights.IMAGENET1K_V1
)


model = models.convnext_tiny(
    weights=weights
)


in_features = (
    model.classifier[
        2
    ].in_features
)


model.classifier[
    2
] = nn.Linear(
    in_features,
    2
)


model = model.to(
    DEVICE
)


print(
    f"Output classes : 2"
)


print(
    f"Parameters     : "
    f"{sum(p.numel() for p in model.parameters()):,}"
)


# ==================================================================================================
# 23. LOSS / OPTIMIZER
#
# Sampler already handles class balance.
# Do NOT stack another class-weighted CE loss here.
# ==================================================================================================

criterion = nn.CrossEntropyLoss()


optimizer = torch.optim.AdamW(

    model.parameters(),

    lr=LEARNING_RATE,

    weight_decay=WEIGHT_DECAY
)


scheduler = (
    torch.optim.lr_scheduler.ReduceLROnPlateau(

        optimizer,

        mode="max",

        factor=0.5,

        patience=1
    )
)


# ==================================================================================================
# 24. TRAIN ONE EPOCH
# ==================================================================================================

def train_one_epoch():

    model.train()


    running_loss = 0.0

    total_samples = 0


    y_true = []

    y_pred = []


    for (
        images,
        labels
    ) in tqdm(

        train_loader,

        desc="Train",

        leave=False
    ):

        images = images.to(
            DEVICE,
            non_blocking=True
        )


        labels = labels.to(
            DEVICE,
            non_blocking=True
        )


        optimizer.zero_grad(
            set_to_none=True
        )


        logits = model(
            images
        )


        loss = criterion(
            logits,
            labels
        )


        loss.backward()


        optimizer.step()


        batch_size = (
            images.size(
                0
            )
        )


        running_loss += (

            loss.item()
            *
            batch_size
        )


        total_samples += (
            batch_size
        )


        predictions = torch.argmax(
            logits,
            dim=1
        )


        y_true.extend(
            labels.detach()
            .cpu()
            .numpy()
            .tolist()
        )


        y_pred.extend(
            predictions.detach()
            .cpu()
            .numpy()
            .tolist()
        )


    return {

        "loss":

            running_loss
            /
            total_samples,

        "accuracy":

            accuracy_score(
                y_true,
                y_pred
            ),

        "macro_f1":

            f1_score(
                y_true,
                y_pred,
                average="macro",
                zero_division=0
            ),

        "balanced_accuracy":

            balanced_accuracy_score(
                y_true,
                y_pred
            ),
    }


# ==================================================================================================
# 25. VALIDATE
# ==================================================================================================

@torch.inference_mode()
def validate():

    model.eval()


    running_loss = 0.0

    total_samples = 0


    y_true = []

    y_pred = []

    all_probs = []


    for (
        images,
        labels
    ) in tqdm(

        val_loader,

        desc="Val",

        leave=False
    ):

        images = images.to(
            DEVICE,
            non_blocking=True
        )


        labels = labels.to(
            DEVICE,
            non_blocking=True
        )


        logits = model(
            images
        )


        loss = criterion(
            logits,
            labels
        )


        probabilities = torch.softmax(
            logits,
            dim=1
        )


        predictions = torch.argmax(
            probabilities,
            dim=1
        )


        batch_size = (
            images.size(
                0
            )
        )


        running_loss += (

            loss.item()
            *
            batch_size
        )


        total_samples += (
            batch_size
        )


        y_true.extend(
            labels.detach()
            .cpu()
            .numpy()
            .tolist()
        )


        y_pred.extend(
            predictions.detach()
            .cpu()
            .numpy()
            .tolist()
        )


        all_probs.extend(
            probabilities.detach()
            .cpu()
            .numpy()
            .tolist()
        )


    return {

        "loss":

            running_loss
            /
            total_samples,

        "accuracy":

            accuracy_score(
                y_true,
                y_pred
            ),

        "macro_f1":

            f1_score(
                y_true,
                y_pred,
                average="macro",
                zero_division=0
            ),

        "balanced_accuracy":

            balanced_accuracy_score(
                y_true,
                y_pred
            ),

        "y_true":
            y_true,

        "y_pred":
            y_pred,

        "probs":
            all_probs,
    }


# ==================================================================================================
# 26. TRAINING LOOP
# ==================================================================================================

history = []


best_macro_f1 = -1.0

best_epoch = None


epochs_without_improvement = 0


print()
print("=" * 110)
print("TRAINING E23-A")
print("=" * 110)


for epoch in range(
    1,
    MAX_EPOCHS + 1
):

    start_time = (
        time.time()
    )


    train_result = (
        train_one_epoch()
    )


    val_result = (
        validate()
    )


    scheduler.step(
        val_result[
            "macro_f1"
        ]
    )


    current_lr = float(
        optimizer.param_groups[
            0
        ][
            "lr"
        ]
    )


    elapsed_minutes = (

        time.time()
        -
        start_time

    ) / 60.0


    record = {

        "epoch":
            epoch,

        "train_loss":
            float(
                train_result[
                    "loss"
                ]
            ),

        "train_accuracy":
            float(
                train_result[
                    "accuracy"
                ]
            ),

        "train_macro_f1":
            float(
                train_result[
                    "macro_f1"
                ]
            ),

        "train_balanced_accuracy":
            float(
                train_result[
                    "balanced_accuracy"
                ]
            ),

        "val_loss":
            float(
                val_result[
                    "loss"
                ]
            ),

        "val_accuracy":
            float(
                val_result[
                    "accuracy"
                ]
            ),

        "val_macro_f1":
            float(
                val_result[
                    "macro_f1"
                ]
            ),

        "val_balanced_accuracy":
            float(
                val_result[
                    "balanced_accuracy"
                ]
            ),

        "learning_rate":
            current_lr,

        "minutes":
            elapsed_minutes,
    }


    history.append(
        record
    )


    print()
    print(
        f"Epoch {epoch:02d}/{MAX_EPOCHS}"
    )


    print(
        f"  Train loss         : "
        f"{train_result['loss']:.4f}"
    )


    print(
        f"  Train accuracy     : "
        f"{train_result['accuracy'] * 100:.2f}%"
    )


    print(
        f"  Train macro-F1     : "
        f"{train_result['macro_f1']:.4f}"
    )


    print(
        f"  Val loss           : "
        f"{val_result['loss']:.4f}"
    )


    print(
        f"  Val accuracy       : "
        f"{val_result['accuracy'] * 100:.2f}%"
    )


    print(
        f"  Val balanced acc   : "
        f"{val_result['balanced_accuracy'] * 100:.2f}%"
    )


    print(
        f"  Val macro-F1       : "
        f"{val_result['macro_f1']:.4f}"
    )


    print(
        f"  LR                 : "
        f"{current_lr:.2e}"
    )


    print(
        f"  Time               : "
        f"{elapsed_minutes:.2f} min"
    )


    # ----------------------------------------------------------------------------------------------
    # Save best checkpoint
    # ----------------------------------------------------------------------------------------------

    if (
        val_result[
            "macro_f1"
        ]
        >
        best_macro_f1
    ):

        best_macro_f1 = float(
            val_result[
                "macro_f1"
            ]
        )


        best_epoch = (
            epoch
        )


        epochs_without_improvement = 0


        torch.save(
            {

                "experiment":
                    "E23-A",

                "description":
                    (
                        "Binary Mixed Rigid vs Mixed Soft "
                        "hard-pair ConvNeXt-Tiny specialist"
                    ),

                "epoch":
                    epoch,

                "model_state_dict":
                    model.state_dict(),

                "class_names":
                    E23_CLASS_NAMES,

                "image_size":
                    IMAGE_SIZE,

                "hardness_boost":
                    HARDNESS_BOOST,

                "hardness_source":
                    (
                        "E3Y-B MobileNet opposite mixed-plastic probability"
                    ),

                "val_macro_f1":
                    float(
                        val_result[
                            "macro_f1"
                        ]
                    ),

                "val_balanced_accuracy":
                    float(
                        val_result[
                            "balanced_accuracy"
                        ]
                    ),
            },

            BEST_MODEL_PATH
        )


        print(
            "  *** BEST CHECKPOINT SAVED ***"
        )


    else:

        epochs_without_improvement += 1


    # ----------------------------------------------------------------------------------------------
    # Save history after every epoch
    # ----------------------------------------------------------------------------------------------

    with open(
        HISTORY_PATH,
        "w",
        encoding="utf-8"
    ) as f:

        json.dump(
            history,
            f,
            indent=2
        )


    # ----------------------------------------------------------------------------------------------
    # Early stop
    # ----------------------------------------------------------------------------------------------

    if (
        epochs_without_improvement
        >= EARLY_STOP_PATIENCE
    ):

        print()
        print(
            f"Early stopping after "
            f"{EARLY_STOP_PATIENCE} epochs "
            f"without validation Macro-F1 improvement."
        )

        break


# ==================================================================================================
# 27. LOAD BEST CHECKPOINT
# ==================================================================================================

print()
print("=" * 110)
print("LOADING BEST E23-A CHECKPOINT")
print("=" * 110)


best_checkpoint = torch.load(
    BEST_MODEL_PATH,
    map_location=DEVICE
)


model.load_state_dict(
    best_checkpoint[
        "model_state_dict"
    ],
    strict=True
)


print(
    f"Best epoch        : "
    f"{best_checkpoint['epoch']}"
)


print(
    f"Best val Macro-F1 : "
    f"{best_checkpoint['val_macro_f1']:.4f}"
)


print(
    f"Best balanced acc : "
    f"{best_checkpoint['val_balanced_accuracy'] * 100:.2f}%"
)


# ==================================================================================================
# 28. FINAL VALIDATION
# ==================================================================================================

final_val = validate()


y_true = np.asarray(
    final_val[
        "y_true"
    ]
)


y_pred = np.asarray(
    final_val[
        "y_pred"
    ]
)


print()
print("=" * 110)
print("E23-A — FINAL STANDALONE VALIDATION RESULTS")
print("=" * 110)


print()
print(
    f"Accuracy          : "
    f"{final_val['accuracy'] * 100:.2f}%"
)


print(
    f"Balanced accuracy : "
    f"{final_val['balanced_accuracy'] * 100:.2f}%"
)


print(
    f"Macro-F1          : "
    f"{final_val['macro_f1']:.4f}"
)


# ==================================================================================================
# 29. CLASSIFICATION REPORT
# ==================================================================================================

report_dict = classification_report(

    y_true,
    y_pred,

    labels=[
        MR_BINARY,
        MS_BINARY
    ],

    target_names=
        E23_CLASS_NAMES,

    output_dict=True,

    zero_division=0
)


report_text = classification_report(

    y_true,
    y_pred,

    labels=[
        MR_BINARY,
        MS_BINARY
    ],

    target_names=
        E23_CLASS_NAMES,

    zero_division=0
)


print()
print("=" * 110)
print("CLASSIFICATION REPORT")
print("=" * 110)

print()
print(
    report_text
)


# ==================================================================================================
# 30. CONFUSION MATRIX
# ==================================================================================================

cm = confusion_matrix(

    y_true,
    y_pred,

    labels=[
        MR_BINARY,
        MS_BINARY
    ]
)


cm_df = pd.DataFrame(

    cm,

    index=[
        "GT: mixed_plastic_rigid",
        "GT: mixed_plastic_soft",
    ],

    columns=[
        "Pred: mixed_plastic_rigid",
        "Pred: mixed_plastic_soft",
    ]
)


print()
print("=" * 110)
print("E23-A — VALIDATION CONFUSION MATRIX")
print("=" * 110)

print()
print(
    cm_df.to_string()
)


# ==================================================================================================
# 31. NORMALIZED CONFUSION
# ==================================================================================================

row_totals = cm.sum(
    axis=1,
    keepdims=True
)


cm_normalized = np.divide(

    cm,

    row_totals,

    out=np.zeros_like(
        cm,
        dtype=float
    ),

    where=(
        row_totals != 0
    )
)


cm_normalized_pct = (

    cm_normalized
    *
    100.0
)


cm_normalized_df = pd.DataFrame(

    cm_normalized_pct,

    index=[
        "GT: mixed_plastic_rigid",
        "GT: mixed_plastic_soft",
    ],

    columns=[
        "Pred: mixed_plastic_rigid",
        "Pred: mixed_plastic_soft",
    ]
)


print()
print("=" * 110)
print("ROW-NORMALIZED CONFUSION (%)")
print("=" * 110)

print()
print(
    cm_normalized_df
    .round(2)
    .to_string()
)


# ==================================================================================================
# 32. KEY BINARY SUMMARY
# ==================================================================================================

mr_metrics = (
    report_dict[
        "mixed_plastic_rigid"
    ]
)


ms_metrics = (
    report_dict[
        "mixed_plastic_soft"
    ]
)


mr_to_ms = int(
    cm[
        MR_BINARY,
        MS_BINARY
    ]
)


ms_to_mr = int(
    cm[
        MS_BINARY,
        MR_BINARY
    ]
)


print()
print("=" * 110)
print("E23-A — MR/MS SPECIALIST SUMMARY")
print("=" * 110)


print()
print(
    f"MR precision       : "
    f"{mr_metrics['precision']:.4f}"
)


print(
    f"MR recall          : "
    f"{mr_metrics['recall']:.4f}"
)


print(
    f"MR F1              : "
    f"{mr_metrics['f1-score']:.4f}"
)


print()
print(
    f"MS precision       : "
    f"{ms_metrics['precision']:.4f}"
)


print(
    f"MS recall          : "
    f"{ms_metrics['recall']:.4f}"
)


print(
    f"MS F1              : "
    f"{ms_metrics['f1-score']:.4f}"
)


print()
print(
    f"MR -> MS errors    : "
    f"{mr_to_ms:,}"
)


print(
    f"MS -> MR errors    : "
    f"{ms_to_mr:,}"
)


# ==================================================================================================
# 33. SAVE RESULTS
# ==================================================================================================

results = {

    "experiment":
        "E23-A",

    "description":
        (
            "Binary Mixed Rigid vs Mixed Soft "
            "hard-pair ConvNeXt-Tiny specialist"
        ),

    "test_used":
        False,

    "classes":
        E23_CLASS_NAMES,

    "best_epoch":
        int(
            best_checkpoint[
                "epoch"
            ]
        ),

    "hardness_boost":
        HARDNESS_BOOST,

    "accuracy":
        float(
            final_val[
                "accuracy"
            ]
        ),

    "balanced_accuracy":
        float(
            final_val[
                "balanced_accuracy"
            ]
        ),

    "macro_f1":
        float(
            final_val[
                "macro_f1"
            ]
        ),

    "mixed_rigid":
        mr_metrics,

    "mixed_soft":
        ms_metrics,

    "MR_to_MS_errors":
        mr_to_ms,

    "MS_to_MR_errors":
        ms_to_mr,

    "confusion_matrix":
        cm.tolist(),

    "classification_report":
        report_dict,
}


with open(
    RESULTS_PATH,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        results,
        f,
        indent=2
    )


cm_df.to_csv(
    CONFUSION_PATH
)


cm_normalized_df.to_csv(
    NORMALIZED_CONFUSION_PATH
)


# ==================================================================================================
# 34. COMPLETE
# ==================================================================================================

print()
print("=" * 110)
print("E23-A COMPLETE")
print("=" * 110)


print()
print(
    f"Best checkpoint : "
    f"{BEST_MODEL_PATH}"
)


print(
    f"Hardness cache  : "
    f"{HARDNESS_CACHE_PATH}"
)


print(
    f"Train manifest  : "
    f"{TRAIN_MANIFEST_PATH}"
)


print(
    f"Val manifest    : "
    f"{VAL_MANIFEST_PATH}"
)


print(
    f"History         : "
    f"{HISTORY_PATH}"
)


print(
    f"Results         : "
    f"{RESULTS_PATH}"
)

E23-A — BINARY MR/MS HARD-PAIR SPECIALIST

PyTorch : 2.13.0+cu126
Device  : cuda
GPU     : NVIDIA GeForce RTX 3050 Ti Laptop GPU

Train crops : C:\Users\varda\Documents\_My Computer\COMMON_space\Learning\Upgrad_EPGP_MS\upgrad_IIITB_MS_course\MS_LJMU_Material\Topic Data\SortWaste\dataset\dataset\cropped_data\train
Val crops   : C:\Users\varda\Documents\_My Computer\COMMON_space\Learning\Upgrad_EPGP_MS\upgrad_IIITB_MS_course\MS_LJMU_Material\Topic Data\SortWaste\dataset\dataset\cropped_data\val
MobileNet   : C:\Users\varda\Documents\_My Computer\COMMON_space\Learning\Upgrad_EPGP_MS\upgrad_IIITB_MS_course\MS_LJMU_Material\Topic Data\SortWaste\dataset\dataset\yolo_mobilenet_crops_E3Y\mobilenet_results\E3Y_B_class_weighted\E3Y_B_MobileNetV3Large_best.pth
Output      : C:\Users\varda\Documents\_My Computer\COMMON_space\Learning\Upgrad_EPGP_MS\upgrad_IIITB_MS_course\MS_LJMU_Material\Thesis_Code\runs\sortwaste\E23A_binary_MR_MS_hardpair_convnext

MR/MS DATASET
mixed_plastic_rigid       | train

Mining MR/MS hardness: 100%|██████████| 253/253 [01:22<00:00,  3.06it/s]



Hardness cache saved : C:\Users\varda\Documents\_My Computer\COMMON_space\Learning\Upgrad_EPGP_MS\upgrad_IIITB_MS_course\MS_LJMU_Material\Thesis_Code\runs\sortwaste\E23A_binary_MR_MS_hardpair_convnext\E23A_train_hardness_cache.json

TRAIN HARD-PAIR SUMMARY

mixed_plastic_rigid
Count              : 7,066
Mean opposite prob : 0.0085
Median             : 0.0000
>= 0.25            : 0.86%
>= 0.50            : 0.38%
>= 0.75            : 0.13%

mixed_plastic_soft
Count              : 9,077
Mean opposite prob : 0.0098
Median             : 0.0000
>= 0.25            : 1.20%
>= 0.50            : 0.55%
>= 0.75            : 0.26%

E23-A TRAIN DISTRIBUTION
mixed_plastic_rigid      : 7,066
mixed_plastic_soft       : 9,077

Hardness boost : 2.00

Training batches   : 673
Validation batches : 107

BUILDING E23-A CONVNEXT-TINY
Output classes : 2
Parameters     : 27,821,666

TRAINING E23-A



Epoch 01/8
  Train loss         : 0.2540
  Train accuracy     : 88.61%
  Train macro-F1     : 0.8861
  Val loss           : 0.7172
  Val accuracy       : 76.63%
  Val balanced acc   : 76.24%
  Val macro-F1       : 0.7624
  LR                 : 1.00e-04
  Time               : 99.71 min
  *** BEST CHECKPOINT SAVED ***



Epoch 02/8
  Train loss         : 0.0798
  Train accuracy     : 96.97%
  Train macro-F1     : 0.9697
  Val loss           : 1.1167
  Val accuracy       : 73.23%
  Val balanced acc   : 74.91%
  Val macro-F1       : 0.7319
  LR                 : 1.00e-04
  Time               : 98.60 min



Epoch 03/8
  Train loss         : 0.0529
  Train accuracy     : 97.99%
  Train macro-F1     : 0.9799
  Val loss           : 0.8318
  Val accuracy       : 77.37%
  Val balanced acc   : 76.99%
  Val macro-F1       : 0.7700
  LR                 : 1.00e-04
  Time               : 107.13 min
  *** BEST CHECKPOINT SAVED ***



Epoch 04/8
  Train loss         : 0.0382
  Train accuracy     : 98.65%
  Train macro-F1     : 0.9865
  Val loss           : 1.0979
  Val accuracy       : 76.00%
  Val balanced acc   : 76.77%
  Val macro-F1       : 0.7597
  LR                 : 1.00e-04
  Time               : 98.87 min


KeyboardInterrupt: 

## E23-B: = E20-A + frozen E23-A binary MR/MS specialist, evaluated and tuned on validation only.
E20-A predicts MR
    ↓
E23-A predicts MS with sufficient confidence
    ↓
allow MR → MS

E20-A predicts MS
    ↓
E23-A predicts MR with sufficient confidence
    ↓
allow MS → MR

In [29]:
# E23-B — E20-A + BINARY MR/MS SPECIALIST
#
# VALIDATION ONLY
# NO TEST ACCESS
# NO RETRAINING
#
# --------------------------------------------------------------------------------------------------
# BASE PIPELINE
# --------------------------------------------------------------------------------------------------
#
# Frozen E20-A:
#
#   E12 YOLO11m @640
#       ↓
#   E3Y-B MobileNet / E16
#       ↓
#   E18-B ConvNeXt-Tiny
#       ↓
#   E20-A gates
#
#       MR      = OFF
#       MS      = 0.85
#       NP      = 0.90
#       PET Oil = 0.94
#       alpha   = 0.70
#
#
# NEW E23-A STAGE
# --------------------------------------------------------------------------------------------------
#
# Binary classifier:
#
#       0 = Mixed Rigid
#       1 = Mixed Soft
#
#
# E23-A is invoked ONLY when the current E20-A prediction is:
#
#       Mixed Rigid
#       or
#       Mixed Soft
#
#
# Allowed changes:
#
#       E20-A MR + E23-A MS  -> optionally MR -> MS
#       E20-A MS + E23-A MR  -> optionally MS -> MR
#
#
# No other classes can be changed.
#
#
# IMPORTANT:
# --------------------------------------------------------------------------------------------------
#
# The ORIGINAL E20-A detection score is preserved after an MR/MS flip.
#
# Therefore this experiment isolates:
#
#       "Does E23-A improve the MR/MS class decision?"
#
# rather than mixing classification improvement with score recalibration.
#
#
# SELECTION
# --------------------------------------------------------------------------------------------------
#
# Primary:
#       six-plastic mean AP50
#
# Secondary:
#       MR/MS mean AP50
#
# Tie-break:
#       overall AP50-95
#
# ==================================================================================================


import json
import time
from pathlib import Path
from collections import Counter

import numpy as np

import torch
import torch.nn as nn

from torch.utils.data import Dataset, DataLoader

from torchvision import models
from torchvision.transforms import v2

from PIL import Image

from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval

from tqdm.auto import tqdm


# ==================================================================================================
# 1. DEVICE
# ==================================================================================================

DEVICE = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)


print("=" * 115)
print("E23-B — E20-A + BINARY MR/MS SPECIALIST")
print("=" * 115)

print()
print(f"PyTorch : {torch.__version__}")
print(f"Device  : {DEVICE}")

if DEVICE.type == "cuda":

    print(
        f"GPU     : "
        f"{torch.cuda.get_device_name(0)}"
    )


# ==================================================================================================
# 2. PATHS
# ==================================================================================================

BASE = Path(
    r"C:\Users\varda\Documents\_My Computer\COMMON_space\Learning"
    r"\Upgrad_EPGP_MS\upgrad_IIITB_MS_course\MS_LJMU_Material"
)


DATASET_ROOT = (
    BASE
    / "Topic Data"
    / "SortWaste"
    / "dataset"
    / "dataset"
)


THESIS_CODE = (
    BASE
    / "Thesis_Code"
)


VAL_IMAGES = (
    DATASET_ROOT
    / "splited_all_dataset_coco"
    / "val"
    / "images"
)


# --------------------------------------------------------------------------------------------------
# Existing E18-D cache contains:
#
# YOLO confidence
# MobileNet probabilities
# E16 final class
# E18 ConvNeXt probabilities/results
# bbox
# image_id
# --------------------------------------------------------------------------------------------------

E18D_DIR = (
    THESIS_CODE
    / "runs"
    / "sortwaste"
    / "E18D_class_selective_convnext"
)


BASE_CACHE_PATH = (
    E18D_DIR
    / "E18D_cached_predictions.json"
)


GT_PATH = (
    E18D_DIR
    / "E18D_val_gt_7class.json"
)


# --------------------------------------------------------------------------------------------------
# Frozen E23-A checkpoint
# --------------------------------------------------------------------------------------------------

E23A_DIR = (
    THESIS_CODE
    / "runs"
    / "sortwaste"
    / "E23A_binary_MR_MS_hardpair_convnext"
)


E23A_CKPT = (
    E23A_DIR
    / "E23A_ConvNeXtTiny_MR_MS_best.pth"
)


# --------------------------------------------------------------------------------------------------
# E23-B output
# --------------------------------------------------------------------------------------------------

OUTPUT_DIR = (
    THESIS_CODE
    / "runs"
    / "sortwaste"
    / "E23B_E20A_binary_MRMS_routing"
)


OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)


E23_CACHE_PATH = (
    OUTPUT_DIR
    / "E23B_cached_predictions_with_E23A.json"
)


GRID_RESULTS_PATH = (
    OUTPUT_DIR
    / "E23B_grid_results.json"
)


BEST_CONFIG_PATH = (
    OUTPUT_DIR
    / "E23B_best_configuration.json"
)


BEST_PREDICTIONS_PATH = (
    OUTPUT_DIR
    / "E23B_best_predictions.json"
)


SUMMARY_PATH = (
    OUTPUT_DIR
    / "E23B_summary.txt"
)


# ==================================================================================================
# 3. VERIFY INPUTS
# ==================================================================================================

for path in [

    VAL_IMAGES,
    BASE_CACHE_PATH,
    GT_PATH,
    E23A_CKPT,

]:

    if not path.exists():

        raise FileNotFoundError(
            f"Required path not found:\n{path}"
        )


print()
print(f"Validation images : {VAL_IMAGES}")
print(f"E18-D cache       : {BASE_CACHE_PATH}")
print(f"Validation GT     : {GT_PATH}")
print(f"E23-A checkpoint  : {E23A_CKPT}")
print(f"Output            : {OUTPUT_DIR}")


# ==================================================================================================
# 4. 7-CLASS MAPPING
# ==================================================================================================

CLASS_NAMES = [

    "ecal",
    "hdpe",
    "mixed_plastic_rigid",
    "mixed_plastic_soft",
    "non_plastic",
    "pet",
    "pet_oil",
]


ECAL = 0
HDPE = 1
MR = 2
MS = 3
NP = 4
PET = 5
PET_OIL = 6


PLASTIC_CLASS_INDICES = [

    ECAL,
    HDPE,
    MR,
    MS,
    PET,
    PET_OIL,
]


# ==================================================================================================
# 5. FROZEN E20-A SETTINGS
# ==================================================================================================

ALPHA = 0.70


E20_MR_GATE = None

E20_MS_GATE = 0.85

E20_NP_GATE = 0.90

E20_PETOIL_GATE = 0.94


# ==================================================================================================
# 6. E23-A BINARY MAPPING
# ==================================================================================================

E23_MR = 0

E23_MS = 1


E23_CLASS_NAMES = [

    "mixed_plastic_rigid",
    "mixed_plastic_soft",
]


# ==================================================================================================
# 7. E23-B GATE SEARCH
#
# MR gate:
#       confidence required to flip current MS -> MR
#
# MS gate:
#       confidence required to flip current MR -> MS
# ==================================================================================================

MR_GATES = [

    None,
    0.50,
    0.60,
    0.65,
    0.70,
    0.75,
    0.80,
    0.85,
    0.90,
    0.92,
    0.94,
    0.95,
    0.97,
    0.99,
]


MS_GATES = [

    None,
    0.50,
    0.60,
    0.65,
    0.70,
    0.75,
    0.80,
    0.85,
    0.90,
    0.92,
    0.94,
    0.95,
    0.97,
    0.99,
]


# ==================================================================================================
# 8. LOAD BASE CACHE
# ==================================================================================================

with open(
    BASE_CACHE_PATH,
    "r",
    encoding="utf-8"
) as f:

    base_cache = json.load(f)


print()
print(
    f"Cached validation detections : "
    f"{len(base_cache):,}"
)


# ==================================================================================================
# 9. LOAD GT + IMAGE LOOKUP
# ==================================================================================================

with open(
    GT_PATH,
    "r",
    encoding="utf-8"
) as f:

    gt_json = json.load(f)


image_lookup = {

    int(img["id"]):
        img["file_name"]

    for img in gt_json[
        "images"
    ]
}


print(
    f"Validation images in GT      : "
    f"{len(image_lookup):,}"
)


coco_gt = COCO(
    str(
        GT_PATH
    )
)


# ==================================================================================================
# 10. RECONSTRUCT FROZEN E20-A
# ==================================================================================================

def reconstruct_e20a(
    item
):

    final_idx = int(
        item[
            "e16_final_idx"
        ]
    )


    mn_probs = np.asarray(
        item[
            "mn_probs"
        ],
        dtype=np.float32
    )


    classifier_prob = float(
        mn_probs[
            final_idx
        ]
    )


    source = "e16"


    conv = item.get(
        "conv"
    )


    if conv is not None:

        conv_global_idx = int(
            conv[
                "conv_global_idx"
            ]
        )


        conv_top_prob = float(
            conv[
                "conv_top_prob"
            ]
        )


        required_gate = None


        if conv_global_idx == MR:

            required_gate = (
                E20_MR_GATE
            )


        elif conv_global_idx == MS:

            required_gate = (
                E20_MS_GATE
            )


        elif conv_global_idx == NP:

            required_gate = (
                E20_NP_GATE
            )


        elif conv_global_idx == PET_OIL:

            required_gate = (
                E20_PETOIL_GATE
            )


        if (
            required_gate is not None
            and
            conv_top_prob >= required_gate
        ):

            final_idx = (
                conv_global_idx
            )


            classifier_prob = (
                conv_top_prob
            )


            source = "e18"


    yolo_conf = max(
        float(
            item[
                "yolo_conf"
            ]
        ),
        1e-12
    )


    classifier_prob = max(
        float(
            classifier_prob
        ),
        1e-12
    )


    # Frozen E20-A score
    score = (

        yolo_conf
        ** ALPHA

    ) * (

        classifier_prob
        ** (
            1.0
            - ALPHA
        )
    )


    return {

        "class_idx":
            int(
                final_idx
            ),

        "score":
            float(
                score
            ),

        "source":
            source,
    }


# ==================================================================================================
# 11. IDENTIFY ONLY MR/MS CANDIDATES
# ==================================================================================================

candidate_indices = []


e20_class_counts = Counter()


for idx, item in enumerate(
    base_cache
):

    e20 = reconstruct_e20a(
        item
    )


    item[
        "_e20_class_idx"
    ] = int(
        e20[
            "class_idx"
        ]
    )


    item[
        "_e20_score"
    ] = float(
        e20[
            "score"
        ]
    )


    item[
        "_e20_source"
    ] = (
        e20[
            "source"
        ]
    )


    e20_class_counts[
        int(
            e20[
                "class_idx"
            ]
        )
    ] += 1


    if e20[
        "class_idx"
    ] in {

        MR,
        MS,

    }:

        candidate_indices.append(
            idx
        )


print()
print("=" * 115)
print("FROZEN E20-A PREDICTION DISTRIBUTION")
print("=" * 115)


for class_idx, class_name in enumerate(
    CLASS_NAMES
):

    print(
        f"{class_name:28s}: "
        f"{e20_class_counts[class_idx]:7,d}"
    )


print()
print(
    f"E23-A MR/MS candidates : "
    f"{len(candidate_indices):,}"
)


print(
    f"Candidate fraction      : "
    f"{100 * len(candidate_indices) / len(base_cache):.2f}%"
)


# ==================================================================================================
# 12. LOAD E23-A MODEL
# ==================================================================================================

print()
print("=" * 115)
print("LOADING FROZEN E23-A")
print("=" * 115)


e23_model = models.convnext_tiny(
    weights=None
)


in_features = (
    e23_model.classifier[
        2
    ].in_features
)


e23_model.classifier[
    2
] = nn.Linear(
    in_features,
    2
)


checkpoint = torch.load(
    E23A_CKPT,
    map_location=DEVICE
)


if (
    isinstance(
        checkpoint,
        dict
    )
    and
    "model_state_dict"
    in checkpoint
):

    state_dict = (
        checkpoint[
            "model_state_dict"
        ]
    )


else:

    state_dict = (
        checkpoint
    )


clean_state = {}


for key, value in (
    state_dict.items()
):

    clean_key = (

        key[7:]

        if key.startswith(
            "module."
        )

        else key
    )


    clean_state[
        clean_key
    ] = value


output_classes = int(

    clean_state[
        "classifier.2.weight"
    ].shape[
        0
    ]
)


print(
    f"Checkpoint output classes : "
    f"{output_classes}"
)


if output_classes != 2:

    raise RuntimeError(

        f"E23-A must have 2 output classes. "
        f"Found {output_classes}."
    )


e23_model.load_state_dict(
    clean_state,
    strict=True
)


e23_model = e23_model.to(
    DEVICE
)


e23_model.eval()


if isinstance(
    checkpoint,
    dict
):

    print(
        f"Checkpoint epoch          : "
        f"{checkpoint.get('epoch', 'unknown')}"
    )


    print(
        f"Checkpoint val macro-F1   : "
        f"{checkpoint.get('val_macro_f1', 'unknown')}"
    )


print(
    "E23-A loaded."
)


# ==================================================================================================
# 13. E23-A VALIDATION TRANSFORM
# ==================================================================================================

IMAGE_SIZE = 224


e23_transform = v2.Compose(
    [

        v2.Resize(
            (
                IMAGE_SIZE,
                IMAGE_SIZE
            ),
            antialias=True
        ),

        v2.ToImage(),

        v2.ToDtype(
            torch.float32,
            scale=True
        ),

        v2.Normalize(
            mean=[
                0.485,
                0.456,
                0.406
            ],
            std=[
                0.229,
                0.224,
                0.225
            ]
        ),
    ]
)


# ==================================================================================================
# 14. DETECTION-CROP DATASET
# ==================================================================================================

class CandidateCropDataset(
    Dataset
):

    def __init__(
        self,
        cache,
        candidate_indices,
        image_lookup,
        image_root
    ):

        self.cache = (
            cache
        )


        self.candidate_indices = (
            candidate_indices
        )


        self.image_lookup = (
            image_lookup
        )


        self.image_root = Path(
            image_root
        )


    def __len__(
        self
    ):

        return len(
            self.candidate_indices
        )


    def __getitem__(
        self,
        dataset_idx
    ):

        cache_idx = int(
            self.candidate_indices[
                dataset_idx
            ]
        )


        item = (
            self.cache[
                cache_idx
            ]
        )


        image_id = int(
            item[
                "image_id"
            ]
        )


        filename = (
            self.image_lookup[
                image_id
            ]
        )


        image_path = (
            self.image_root
            / filename
        )


        with Image.open(
            image_path
        ) as img:

            image = img.convert(
                "RGB"
            )


            width, height = (
                image.size
            )


            x1, y1, x2, y2 = [

                float(v)

                for v in item[
                    "bbox"
                ]
            ]


            # Clamp bbox
            x1 = max(
                0.0,
                min(
                    x1,
                    width - 1
                )
            )

            y1 = max(
                0.0,
                min(
                    y1,
                    height - 1
                )
            )

            x2 = max(
                x1 + 1.0,
                min(
                    x2,
                    width
                )
            )

            y2 = max(
                y1 + 1.0,
                min(
                    y2,
                    height
                )
            )


            crop = image.crop(
                (
                    int(
                        np.floor(
                            x1
                        )
                    ),
                    int(
                        np.floor(
                            y1
                        )
                    ),
                    int(
                        np.ceil(
                            x2
                        )
                    ),
                    int(
                        np.ceil(
                            y2
                        )
                    ),
                )
            )


        crop = e23_transform(
            crop
        )


        return (

            crop,

            cache_idx,
        )


# ==================================================================================================
# 15. E23-A INFERENCE OR LOAD CACHE
# ==================================================================================================

if E23_CACHE_PATH.exists():

    print()
    print("=" * 115)
    print("FOUND EXISTING E23-B CACHE")
    print("=" * 115)


    with open(
        E23_CACHE_PATH,
        "r",
        encoding="utf-8"
    ) as f:

        cache = json.load(f)


    print(
        f"Loaded predictions : "
        f"{len(cache):,}"
    )


else:

    print()
    print("=" * 115)
    print("RUNNING E23-A ON E20-A MR/MS CANDIDATES")
    print("=" * 115)


    cache = base_cache


    for item in cache:

        item[
            "e23"
        ] = None


    candidate_dataset = CandidateCropDataset(

        cache=cache,

        candidate_indices=candidate_indices,

        image_lookup=image_lookup,

        image_root=VAL_IMAGES
    )


    candidate_loader = DataLoader(

        candidate_dataset,

        batch_size=32,

        shuffle=False,

        num_workers=0,

        pin_memory=(
            DEVICE.type
            == "cuda"
        )
    )


    start_time = (
        time.time()
    )


    processed = 0


    with torch.inference_mode():

        for (
            images,
            cache_indices
        ) in tqdm(

            candidate_loader,

            desc="E23-A MR/MS inference"
        ):

            images = images.to(
                DEVICE,
                non_blocking=True
            )


            logits = e23_model(
                images
            )


            probs = torch.softmax(
                logits,
                dim=1
            )


            top_probs, local_indices = torch.max(
                probs,
                dim=1
            )


            probs_np = (
                probs
                .detach()
                .cpu()
                .numpy()
            )


            local_np = (
                local_indices
                .detach()
                .cpu()
                .numpy()
            )


            top_np = (
                top_probs
                .detach()
                .cpu()
                .numpy()
            )


            cache_idx_np = (
                cache_indices
                .numpy()
            )


            for i in range(
                len(
                    cache_idx_np
                )
            ):

                cache_idx = int(
                    cache_idx_np[
                        i
                    ]
                )


                local_idx = int(
                    local_np[
                        i
                    ]
                )


                global_idx = (

                    MR

                    if local_idx
                    == E23_MR

                    else MS
                )


                cache[
                    cache_idx
                ][
                    "e23"
                ] = {

                    "probs":
                        probs_np[
                            i
                        ].tolist(),

                    "local_idx":
                        local_idx,

                    "global_idx":
                        int(
                            global_idx
                        ),

                    "top_prob":
                        float(
                            top_np[
                                i
                            ]
                        ),
                }


                processed += 1


    elapsed = (

        time.time()
        -
        start_time

    ) / 60.0


    print()
    print(
        f"E23-A candidate inference complete."
    )


    print(
        f"Candidates processed : "
        f"{processed:,}"
    )


    print(
        f"Inference time       : "
        f"{elapsed:.2f} min"
    )


    with open(
        E23_CACHE_PATH,
        "w",
        encoding="utf-8"
    ) as f:

        json.dump(
            cache,
            f
        )


    print(
        f"Cache saved          : "
        f"{E23_CACHE_PATH}"
    )


# ==================================================================================================
# 16. VERIFY E23 CACHE
# ==================================================================================================

num_e23 = sum(

    1

    for item in cache

    if item.get(
        "e23"
    ) is not None
)


print()
print(
    f"E23-A outputs cached : "
    f"{num_e23:,}"
)


# ==================================================================================================
# 17. EVALUATION HELPERS
# ==================================================================================================

def valid_mean(
    values
):

    values = np.asarray(
        values
    )


    valid = values[
        values > -1
    ]


    if valid.size == 0:

        return float(
            "nan"
        )


    return float(
        np.mean(
            valid
        )
    )


def evaluate_predictions(
    predictions
):

    coco_dt = coco_gt.loadRes(
        predictions
    )


    evaluator = COCOeval(

        coco_gt,
        coco_dt,
        "bbox"
    )


    evaluator.params.maxDets = [

        1,
        10,
        100,
    ]


    evaluator.evaluate()
    evaluator.accumulate()


    precision = (
        evaluator.eval[
            "precision"
        ]
    )


    recall = (
        evaluator.eval[
            "recall"
        ]
    )


    iou_thresholds = (
        evaluator.params.iouThrs
    )


    idx50 = int(

        np.where(
            np.isclose(
                iou_thresholds,
                0.50
            )
        )[0][0]
    )


    idx75 = int(

        np.where(
            np.isclose(
                iou_thresholds,
                0.75
            )
        )[0][0]
    )


    overall = {

        "AP50_95":
            valid_mean(
                precision[
                    :,
                    :,
                    :,
                    0,
                    -1
                ]
            ),

        "AP50":
            valid_mean(
                precision[
                    idx50,
                    :,
                    :,
                    0,
                    -1
                ]
            ),

        "AP75":
            valid_mean(
                precision[
                    idx75,
                    :,
                    :,
                    0,
                    -1
                ]
            ),

        "AR100":
            valid_mean(
                recall[
                    :,
                    :,
                    0,
                    -1
                ]
            ),
    }


    class_metrics = {}


    for class_idx, class_name in enumerate(
        CLASS_NAMES
    ):

        class_metrics[
            class_name
        ] = {

            "AP50":
                valid_mean(
                    precision[
                        idx50,
                        :,
                        class_idx,
                        0,
                        -1
                    ]
                ),

            "AP50_95":
                valid_mean(
                    precision[
                        :,
                        :,
                        class_idx,
                        0,
                        -1
                    ]
                ),
        }


    six_plastic_mean_ap50 = float(

        np.mean(
            [

                class_metrics[
                    CLASS_NAMES[
                        idx
                    ]
                ][
                    "AP50"
                ]

                for idx
                in PLASTIC_CLASS_INDICES
            ]
        )
    )


    mr_ms_mean_ap50 = float(

        np.mean(
            [

                class_metrics[
                    "mixed_plastic_rigid"
                ][
                    "AP50"
                ],

                class_metrics[
                    "mixed_plastic_soft"
                ][
                    "AP50"
                ],
            ]
        )
    )


    return {

        "overall":
            overall,

        "class_metrics":
            class_metrics,

        "six_plastic_mean_AP50":
            six_plastic_mean_ap50,

        "MR_MS_mean_AP50":
            mr_ms_mean_ap50,
    }


# ==================================================================================================
# 18. BUILD E20-A / E23-B PREDICTIONS
# ==================================================================================================

def build_predictions(
    mr_gate=None,
    ms_gate=None
):

    predictions = []


    flip_counts = Counter()


    proposal_counts = Counter()


    for item in cache:

        pre_class = int(
            item[
                "_e20_class_idx"
            ]
        )


        final_class = (
            pre_class
        )


        # IMPORTANT:
        # Preserve frozen E20-A score
        final_score = float(
            item[
                "_e20_score"
            ]
        )


        e23 = item.get(
            "e23"
        )


        if (
            e23 is not None
            and
            pre_class in {
                MR,
                MS,
            }
        ):

            e23_class = int(
                e23[
                    "global_idx"
                ]
            )


            e23_prob = float(
                e23[
                    "top_prob"
                ]
            )


            # --------------------------------------------------------------------------------------
            # Current E20-A = MS
            # E23-A says MR
            # --------------------------------------------------------------------------------------

            if (
                pre_class == MS
                and
                e23_class == MR
            ):

                proposal_counts[
                    "MS->MR"
                ] += 1


                if (
                    mr_gate is not None
                    and
                    e23_prob >= mr_gate
                ):

                    final_class = MR


                    flip_counts[
                        "MS->MR"
                    ] += 1


            # --------------------------------------------------------------------------------------
            # Current E20-A = MR
            # E23-A says MS
            # --------------------------------------------------------------------------------------

            elif (
                pre_class == MR
                and
                e23_class == MS
            ):

                proposal_counts[
                    "MR->MS"
                ] += 1


                if (
                    ms_gate is not None
                    and
                    e23_prob >= ms_gate
                ):

                    final_class = MS


                    flip_counts[
                        "MR->MS"
                    ] += 1


        x1, y1, x2, y2 = [

            float(v)

            for v in item[
                "bbox"
            ]
        ]


        width = (
            x2 - x1
        )

        height = (
            y2 - y1
        )


        if (
            width <= 0
            or
            height <= 0
        ):

            continue


        predictions.append(
            {

                "image_id":
                    int(
                        item[
                            "image_id"
                        ]
                    ),

                "category_id":
                    int(
                        final_class
                        + 1
                    ),

                "bbox": [

                    x1,
                    y1,
                    width,
                    height,
                ],

                "score":
                    float(
                        final_score
                    ),
            }
        )


    return {

        "predictions":
            predictions,

        "proposal_counts":
            dict(
                proposal_counts
            ),

        "flip_counts":
            dict(
                flip_counts
            ),
    }


# ==================================================================================================
# 19. FROZEN E20-A BASELINE
# ==================================================================================================

print()
print("=" * 115)
print("FROZEN E20-A VALIDATION BASELINE")
print("=" * 115)


baseline_build = build_predictions(

    mr_gate=None,

    ms_gate=None
)


baseline_eval = evaluate_predictions(

    baseline_build[
        "predictions"
    ]
)


print()
print(
    f"Six-plastic mean AP50 : "
    f"{baseline_eval['six_plastic_mean_AP50'] * 100:.4f}%"
)


print(
    f"MR AP50               : "
    f"{baseline_eval['class_metrics']['mixed_plastic_rigid']['AP50'] * 100:.4f}%"
)


print(
    f"MS AP50               : "
    f"{baseline_eval['class_metrics']['mixed_plastic_soft']['AP50'] * 100:.4f}%"
)


print(
    f"MR/MS mean AP50       : "
    f"{baseline_eval['MR_MS_mean_AP50'] * 100:.4f}%"
)


print(
    f"Overall AP50-95       : "
    f"{baseline_eval['overall']['AP50_95'] * 100:.4f}%"
)


print(
    f"Overall AP50          : "
    f"{baseline_eval['overall']['AP50'] * 100:.4f}%"
)


# ==================================================================================================
# 20. GRID SEARCH
# ==================================================================================================

def gate_label(
    gate
):

    if gate is None:

        return "OFF"


    return f"{gate:.2f}"


results = []


total_combinations = (

    len(
        MR_GATES
    )

    *

    len(
        MS_GATES
    )
)


counter = 0


print()
print("=" * 115)
print("E23-B MR/MS BINARY ROUTING SEARCH — VALIDATION ONLY")
print("=" * 115)


for mr_gate in MR_GATES:

    for ms_gate in MS_GATES:

        counter += 1


        built = build_predictions(

            mr_gate=mr_gate,

            ms_gate=ms_gate
        )


        evaluation = evaluate_predictions(

            built[
                "predictions"
            ]
        )


        result = {

            "mr_gate":
                mr_gate,

            "ms_gate":
                ms_gate,

            "six_plastic_mean_AP50":
                evaluation[
                    "six_plastic_mean_AP50"
                ],

            "MR_MS_mean_AP50":
                evaluation[
                    "MR_MS_mean_AP50"
                ],

            "overall":
                evaluation[
                    "overall"
                ],

            "class_metrics":
                evaluation[
                    "class_metrics"
                ],

            "proposal_counts":
                built[
                    "proposal_counts"
                ],

            "flip_counts":
                built[
                    "flip_counts"
                ],
        }


        results.append(
            result
        )


        print(

            f"[{counter:03d}/{total_combinations}] "

            f"MR={gate_label(mr_gate):>4s} "

            f"MS={gate_label(ms_gate):>4s} | "

            f"Plastic="
            f"{evaluation['six_plastic_mean_AP50'] * 100:7.3f}% | "

            f"MR="
            f"{evaluation['class_metrics']['mixed_plastic_rigid']['AP50'] * 100:6.2f}% | "

            f"MS="
            f"{evaluation['class_metrics']['mixed_plastic_soft']['AP50'] * 100:6.2f}% | "

            f"MR/MS="
            f"{evaluation['MR_MS_mean_AP50'] * 100:6.2f}% | "

            f"AP="
            f"{evaluation['overall']['AP50_95'] * 100:6.3f}%"
        )


# ==================================================================================================
# 21. RANK
# ==================================================================================================

ranked = sorted(

    results,

    key=lambda x: (

        x[
            "six_plastic_mean_AP50"
        ],

        x[
            "MR_MS_mean_AP50"
        ],

        x[
            "overall"
        ][
            "AP50_95"
        ],
    ),

    reverse=True
)


best = (
    ranked[
        0
    ]
)


# ==================================================================================================
# 22. TOP 20
# ==================================================================================================

print()
print()
print("=" * 115)
print("TOP 20 E23-B VALIDATION CONFIGURATIONS")
print("=" * 115)


print(
    f"\n"
    f"{'Rank':>4s} "
    f"{'MR':>6s} "
    f"{'MS':>6s} "
    f"{'Plastic AP50':>14s} "
    f"{'MR AP50':>10s} "
    f"{'MS AP50':>10s} "
    f"{'MR/MS':>10s} "
    f"{'AP50-95':>10s}"
)


print(
    "-" * 92
)


for rank, result in enumerate(
    ranked[
        :20
    ],
    start=1
):

    print(

        f"{rank:4d} "

        f"{gate_label(result['mr_gate']):>6s} "

        f"{gate_label(result['ms_gate']):>6s} "

        f"{result['six_plastic_mean_AP50'] * 100:13.4f}% "

        f"{result['class_metrics']['mixed_plastic_rigid']['AP50'] * 100:9.2f}% "

        f"{result['class_metrics']['mixed_plastic_soft']['AP50'] * 100:9.2f}% "

        f"{result['MR_MS_mean_AP50'] * 100:9.2f}% "

        f"{result['overall']['AP50_95'] * 100:9.3f}%"
    )


# ==================================================================================================
# 23. DELTAS
# ==================================================================================================

delta = {

    "six_plastic_mean_AP50_pp":

        (
            best[
                "six_plastic_mean_AP50"
            ]

            -

            baseline_eval[
                "six_plastic_mean_AP50"
            ]
        )
        * 100.0,


    "MR_AP50_pp":

        (
            best[
                "class_metrics"
            ][
                "mixed_plastic_rigid"
            ][
                "AP50"
            ]

            -

            baseline_eval[
                "class_metrics"
            ][
                "mixed_plastic_rigid"
            ][
                "AP50"
            ]
        )
        * 100.0,


    "MS_AP50_pp":

        (
            best[
                "class_metrics"
            ][
                "mixed_plastic_soft"
            ][
                "AP50"
            ]

            -

            baseline_eval[
                "class_metrics"
            ][
                "mixed_plastic_soft"
            ][
                "AP50"
            ]
        )
        * 100.0,


    "MR_MS_mean_AP50_pp":

        (
            best[
                "MR_MS_mean_AP50"
            ]

            -

            baseline_eval[
                "MR_MS_mean_AP50"
            ]
        )
        * 100.0,


    "overall_AP50_95_pp":

        (
            best[
                "overall"
            ][
                "AP50_95"
            ]

            -

            baseline_eval[
                "overall"
            ][
                "AP50_95"
            ]
        )
        * 100.0,


    "overall_AP50_pp":

        (
            best[
                "overall"
            ][
                "AP50"
            ]

            -

            baseline_eval[
                "overall"
            ][
                "AP50"
            ]
        )
        * 100.0,
}


# ==================================================================================================
# 24. BEST RESULT
# ==================================================================================================

print()
print("=" * 115)
print("BEST E23-B CONFIGURATION — VALIDATION")
print("=" * 115)


print()
print(
    f"MS -> MR gate : "
    f"{gate_label(best['mr_gate'])}"
)


print(
    f"MR -> MS gate : "
    f"{gate_label(best['ms_gate'])}"
)


print()
print(
    f"Six-plastic mean AP50 : "
    f"{best['six_plastic_mean_AP50'] * 100:.4f}%"
)


print(
    f"MR AP50               : "
    f"{best['class_metrics']['mixed_plastic_rigid']['AP50'] * 100:.4f}%"
)


print(
    f"MS AP50               : "
    f"{best['class_metrics']['mixed_plastic_soft']['AP50'] * 100:.4f}%"
)


print(
    f"MR/MS mean AP50       : "
    f"{best['MR_MS_mean_AP50'] * 100:.4f}%"
)


print()
print(
    f"Overall AP50-95       : "
    f"{best['overall']['AP50_95'] * 100:.4f}%"
)


print(
    f"Overall AP50          : "
    f"{best['overall']['AP50'] * 100:.4f}%"
)


print(
    f"Overall AP75          : "
    f"{best['overall']['AP75'] * 100:.4f}%"
)


print(
    f"AR100                 : "
    f"{best['overall']['AR100'] * 100:.4f}%"
)


# ==================================================================================================
# 25. CLASS-WISE RESULTS
# ==================================================================================================

print()
print("=" * 115)
print("BEST E23-B — CLASS-WISE VALIDATION")
print("=" * 115)


print(
    f"\n"
    f"{'Class':28s}"
    f"{'AP50':>14s}"
    f"{'AP50-95':>14s}"
)


print(
    "-" * 56
)


for class_name in CLASS_NAMES:

    metrics = (
        best[
            "class_metrics"
        ][
            class_name
        ]
    )


    print(

        f"{class_name:28s}"

        f"{metrics['AP50'] * 100:13.2f}%"

        f"{metrics['AP50_95'] * 100:13.2f}%"
    )


# ==================================================================================================
# 26. ROUTING COUNTS
# ==================================================================================================

print()
print("=" * 115)
print("BEST E23-B — ROUTING")
print("=" * 115)


print()
print(
    "E23-A disagreement proposals:"
)


for route, count in sorted(
    best[
        "proposal_counts"
    ].items()
):

    print(
        f"{route:12s}: "
        f"{count:6,d}"
    )


print()
print(
    "Accepted flips:"
)


for route, count in sorted(
    best[
        "flip_counts"
    ].items()
):

    print(
        f"{route:12s}: "
        f"{count:6,d}"
    )


# ==================================================================================================
# 27. DELTA VS E20-A
# ==================================================================================================

print()
print("=" * 115)
print("DELTA VS FROZEN E20-A VALIDATION")
print("=" * 115)


for metric_name, value in (
    delta.items()
):

    print(
        f"{metric_name:32s}: "
        f"{value:+.4f} pp"
    )


# ==================================================================================================
# 28. DECISION
# ==================================================================================================

improves_primary = (

    best[
        "six_plastic_mean_AP50"
    ]

    >

    baseline_eval[
        "six_plastic_mean_AP50"
    ]
)


improves_mrms = (

    best[
        "MR_MS_mean_AP50"
    ]

    >

    baseline_eval[
        "MR_MS_mean_AP50"
    ]
)


print()
print("=" * 115)
print("E23-B VALIDATION DECISION")
print("=" * 115)


if (
    improves_primary
    and
    improves_mrms
):

    print()
    print(
        "E23-B improves BOTH:"
    )

    print(
        "  - six-plastic mean AP50"
    )

    print(
        "  - MR/MS mean AP50"
    )

    print()
    print(
        "E23-B is a strong candidate for frozen TEST evaluation."
    )


elif improves_primary:

    print()
    print(
        "E23-B improves the PRIMARY six-plastic metric,"
    )

    print(
        "but does not improve MR/MS mean AP50."
    )

    print(
        "Inspect class-wise trade-offs before promotion."
    )


elif improves_mrms:

    print()
    print(
        "E23-B improves MR/MS mean AP50,"
    )

    print(
        "but not the primary six-plastic mean AP50."
    )

    print(
        "Do NOT automatically promote to TEST."
    )


else:

    print()
    print(
        "E23-B does not improve the target metrics."
    )

    print(
        "Do not promote to TEST."
    )


# ==================================================================================================
# 29. SAVE BEST PREDICTIONS
# ==================================================================================================

best_build = build_predictions(

    mr_gate=best[
        "mr_gate"
    ],

    ms_gate=best[
        "ms_gate"
    ]
)


with open(
    BEST_PREDICTIONS_PATH,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        best_build[
            "predictions"
        ],
        f
    )


# ==================================================================================================
# 30. SAVE GRID
# ==================================================================================================

with open(
    GRID_RESULTS_PATH,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        ranked,
        f,
        indent=2
    )


# ==================================================================================================
# 31. SAVE CONFIG
# ==================================================================================================

output_config = {

    "experiment":
        "E23-B",

    "description":
        (
            "Frozen E20-A pipeline plus "
            "E23-A binary MR/MS specialist"
        ),

    "dataset":
        "validation",

    "test_used":
        False,

    "base_pipeline":
        "E20-A",

    "specialist":
        "E23-A",

    "routing":
        {

            "allowed":
                [
                    "MS->MR",
                    "MR->MS",
                ],

            "other_classes_untouched":
                True,

            "preserve_E20A_detection_score":
                True,
        },

    "selection":
        {

            "primary":
                "six-plastic mean AP50",

            "secondary":
                "MR/MS mean AP50",

            "tie_break":
                "overall AP50-95",
        },

    "E20A_baseline":
        baseline_eval,

    "best_E23B":
        best,

    "delta_vs_E20A":
        delta,

    "promote_to_test":
        bool(
            improves_primary
            and
            improves_mrms
        ),
}


with open(
    BEST_CONFIG_PATH,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        output_config,
        f,
        indent=2
    )


# ==================================================================================================
# 32. SAVE SUMMARY
# ==================================================================================================

summary_lines = [

    "=" * 115,

    "E23-B — E20-A + BINARY MR/MS SPECIALIST",

    "=" * 115,

    "",

    "VALIDATION ONLY",

    "TEST NOT USED",

    "",

    "Frozen base: E20-A",

    "New specialist: E23-A",

    "",

    "Allowed routing:",

    "  E20-A MS -> E23-A MR",

    "  E20-A MR -> E23-A MS",

    "",

    "Best configuration:",

    f"  MS -> MR gate = "
    f"{gate_label(best['mr_gate'])}",

    f"  MR -> MS gate = "
    f"{gate_label(best['ms_gate'])}",

    "",

    f"Six-plastic mean AP50 = "
    f"{best['six_plastic_mean_AP50'] * 100:.4f}%",

    f"MR AP50               = "
    f"{best['class_metrics']['mixed_plastic_rigid']['AP50'] * 100:.4f}%",

    f"MS AP50               = "
    f"{best['class_metrics']['mixed_plastic_soft']['AP50'] * 100:.4f}%",

    f"MR/MS mean AP50       = "
    f"{best['MR_MS_mean_AP50'] * 100:.4f}%",

    f"Overall AP50-95       = "
    f"{best['overall']['AP50_95'] * 100:.4f}%",

    "",

    "Delta vs E20-A:",
]


for metric_name, value in (
    delta.items()
):

    summary_lines.append(

        f"  {metric_name:32s}: "
        f"{value:+.4f} pp"
    )


with open(
    SUMMARY_PATH,
    "w",
    encoding="utf-8"
) as f:

    f.write(
        "\n".join(
            summary_lines
        )
    )


# ==================================================================================================
# 33. COMPLETE
# ==================================================================================================

print()
print("=" * 115)
print("E23-B COMPLETE")
print("=" * 115)


print()
print(
    f"E23 cache        : "
    f"{E23_CACHE_PATH}"
)


print(
    f"Grid results     : "
    f"{GRID_RESULTS_PATH}"
)


print(
    f"Best config      : "
    f"{BEST_CONFIG_PATH}"
)


print(
    f"Best predictions : "
    f"{BEST_PREDICTIONS_PATH}"
)


print(
    f"Summary          : "
    f"{SUMMARY_PATH}"
)

E23-B — E20-A + BINARY MR/MS SPECIALIST

PyTorch : 2.13.0+cu126
Device  : cuda
GPU     : NVIDIA GeForce RTX 3050 Ti Laptop GPU

Validation images : C:\Users\varda\Documents\_My Computer\COMMON_space\Learning\Upgrad_EPGP_MS\upgrad_IIITB_MS_course\MS_LJMU_Material\Topic Data\SortWaste\dataset\dataset\splited_all_dataset_coco\val\images
E18-D cache       : C:\Users\varda\Documents\_My Computer\COMMON_space\Learning\Upgrad_EPGP_MS\upgrad_IIITB_MS_course\MS_LJMU_Material\Thesis_Code\runs\sortwaste\E18D_class_selective_convnext\E18D_cached_predictions.json
Validation GT     : C:\Users\varda\Documents\_My Computer\COMMON_space\Learning\Upgrad_EPGP_MS\upgrad_IIITB_MS_course\MS_LJMU_Material\Thesis_Code\runs\sortwaste\E18D_class_selective_convnext\E18D_val_gt_7class.json
E23-A checkpoint  : C:\Users\varda\Documents\_My Computer\COMMON_space\Learning\Upgrad_EPGP_MS\upgrad_IIITB_MS_course\MS_LJMU_Material\Thesis_Code\runs\sortwaste\E23A_binary_MR_MS_hardpair_convnext\E23A_ConvNeXtTiny_MR_MS_best.

E23-A MR/MS inference: 100%|██████████| 460/460 [22:28<00:00,  2.93s/it]



E23-A candidate inference complete.
Candidates processed : 14,690
Inference time       : 22.48 min
Cache saved          : C:\Users\varda\Documents\_My Computer\COMMON_space\Learning\Upgrad_EPGP_MS\upgrad_IIITB_MS_course\MS_LJMU_Material\Thesis_Code\runs\sortwaste\E23B_E20A_binary_MRMS_routing\E23B_cached_predictions_with_E23A.json

E23-A outputs cached : 14,690

FROZEN E20-A VALIDATION BASELINE
Loading and preparing results...
DONE (t=0.07s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=9.60s).
Accumulating evaluation results...
DONE (t=0.53s).

Six-plastic mean AP50 : 68.0766%
MR AP50               : 49.6173%
MS AP50               : 42.8212%
MR/MS mean AP50       : 46.2193%
Overall AP50-95       : 45.7559%
Overall AP50          : 61.1717%

E23-B MR/MS BINARY ROUTING SEARCH — VALIDATION ONLY
Loading and preparing results...
DONE (t=0.07s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type

## E23-C : E21-B + Binary MR/MS Specialist    
E21-B predicts MS + E23-A predicts MR → possible MS → MR
E21-B predicts MR + E23-A predicts MS → possible MR → MS

In [31]:
# E23-C — E21-B + BINARY MR/MS SPECIALIST
#
# VALIDATION ONLY
# NO TEST ACCESS
# NO RETRAINING
#
# --------------------------------------------------------------------------------------------------
# FROZEN BASE
# --------------------------------------------------------------------------------------------------
#
# E21-B:
#
#   E12 YOLO11m @640
#       ↓
#   E3Y-B MobileNet / E16
#       ↓
#   E18-B / E20-A
#       ↓
#   E21-A hard-negative specialist
#
#
# Frozen E20-A:
#
#   MR      = OFF
#   MS      = 0.85
#   NP      = 0.90
#   PET Oil = 0.94
#
#
# Frozen E21-A gates:
#
#   MR = 0.97
#   MS = 0.99
#
# alpha = 0.70
#
#
# --------------------------------------------------------------------------------------------------
# E23-A
# --------------------------------------------------------------------------------------------------
#
# Binary:
#
#   0 = Mixed Rigid
#   1 = Mixed Soft
#
#
# E23-A is invoked ONLY if final frozen E21-B class is:
#
#   MR
#   MS
#
#
# Allowed:
#
#   E21-B MS + E23-A MR → MS -> MR
#   E21-B MR + E23-A MS → MR -> MS
#
#
# E23-A does NOT change:
#
#   ECAL
#   HDPE
#   non_plastic
#   PET
#   PET Oil
#
#
# IMPORTANT:
#
# Preserve the frozen E21-B detection score after any E23-A MR/MS flip.
#
#
# SELECTION:
#
# Primary   = six-plastic mean AP50
# Secondary = MR/MS mean AP50
# Tie-break = overall AP50-95
#
# ==================================================================================================


import json
import time
from pathlib import Path
from collections import Counter

import numpy as np

import torch
import torch.nn as nn

from torch.utils.data import Dataset, DataLoader

from torchvision import models
from torchvision.transforms import v2

from PIL import Image

from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval

from tqdm.auto import tqdm


# ==================================================================================================
# 1. DEVICE
# ==================================================================================================

DEVICE = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)


print("=" * 118)
print("E23-C — E21-B + BINARY MR/MS SPECIALIST")
print("=" * 118)

print()
print(f"PyTorch : {torch.__version__}")
print(f"Device  : {DEVICE}")

if DEVICE.type == "cuda":
    print(
        f"GPU     : "
        f"{torch.cuda.get_device_name(0)}"
    )


# ==================================================================================================
# 2. PATHS
# ==================================================================================================

BASE = Path(
    r"C:\Users\varda\Documents\_My Computer\COMMON_space\Learning"
    r"\Upgrad_EPGP_MS\upgrad_IIITB_MS_course\MS_LJMU_Material"
)


DATASET_ROOT = (
    BASE
    / "Topic Data"
    / "SortWaste"
    / "dataset"
    / "dataset"
)


THESIS_CODE = (
    BASE
    / "Thesis_Code"
)


VAL_IMAGES = (
    DATASET_ROOT
    / "splited_all_dataset_coco"
    / "val"
    / "images"
)


# --------------------------------------------------------------------------------------------------
# E21-B validation cache
#
# Contains:
#
#   YOLO confidence
#   bbox
#   MobileNet probabilities
#   E16 final prediction
#   E18-B results
#   E21-A results
#
# --------------------------------------------------------------------------------------------------

E21B_DIR = (
    THESIS_CODE
    / "runs"
    / "sortwaste"
    / "E21B_end_to_end_validation"
)


E21B_CACHE_PATH = (
    E21B_DIR
    / "E21B_cached_predictions_with_E21A.json"
)


# --------------------------------------------------------------------------------------------------
# GT
# --------------------------------------------------------------------------------------------------

E18D_DIR = (
    THESIS_CODE
    / "runs"
    / "sortwaste"
    / "E18D_class_selective_convnext"
)


GT_PATH = (
    E18D_DIR
    / "E18D_val_gt_7class.json"
)


# --------------------------------------------------------------------------------------------------
# E23-A
# --------------------------------------------------------------------------------------------------

E23A_DIR = (
    THESIS_CODE
    / "runs"
    / "sortwaste"
    / "E23A_binary_MR_MS_hardpair_convnext"
)


E23A_CKPT = (
    E23A_DIR
    / "E23A_ConvNeXtTiny_MR_MS_best.pth"
)


# --------------------------------------------------------------------------------------------------
# E23-C outputs
# --------------------------------------------------------------------------------------------------

OUTPUT_DIR = (
    THESIS_CODE
    / "runs"
    / "sortwaste"
    / "E23C_E21B_binary_MRMS_routing"
)


OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)


E23_CACHE_PATH = (
    OUTPUT_DIR
    / "E23C_cached_predictions_with_E23A.json"
)


GRID_RESULTS_PATH = (
    OUTPUT_DIR
    / "E23C_grid_results.json"
)


BEST_CONFIG_PATH = (
    OUTPUT_DIR
    / "E23C_best_configuration.json"
)


BEST_PREDICTIONS_PATH = (
    OUTPUT_DIR
    / "E23C_best_predictions.json"
)


SUMMARY_PATH = (
    OUTPUT_DIR
    / "E23C_summary.txt"
)


# ==================================================================================================
# 3. VERIFY
# ==================================================================================================

for path in [
    VAL_IMAGES,
    E21B_CACHE_PATH,
    GT_PATH,
    E23A_CKPT,
]:

    if not path.exists():

        raise FileNotFoundError(
            f"Required path not found:\n{path}"
        )


print()
print(f"Validation images : {VAL_IMAGES}")
print(f"E21-B cache       : {E21B_CACHE_PATH}")
print(f"Validation GT     : {GT_PATH}")
print(f"E23-A checkpoint  : {E23A_CKPT}")
print(f"Output            : {OUTPUT_DIR}")


# ==================================================================================================
# 4. GLOBAL CLASS MAPPING
# ==================================================================================================

CLASS_NAMES = [
    "ecal",
    "hdpe",
    "mixed_plastic_rigid",
    "mixed_plastic_soft",
    "non_plastic",
    "pet",
    "pet_oil",
]


ECAL = 0
HDPE = 1
MR = 2
MS = 3
NP = 4
PET = 5
PET_OIL = 6


PLASTIC_INDICES = [
    ECAL,
    HDPE,
    MR,
    MS,
    PET,
    PET_OIL,
]


# ==================================================================================================
# 5. FROZEN E20-A CONFIGURATION
# ==================================================================================================

ALPHA = 0.70


E20_MR_GATE = None
E20_MS_GATE = 0.85
E20_NP_GATE = 0.90
E20_PETOIL_GATE = 0.94


# ==================================================================================================
# 6. FROZEN E21-B CONFIGURATION
#
# E21-A output classes use global 7-class indices.
# ==================================================================================================

E21_MR_GATE = 0.97
E21_MS_GATE = 0.99


# ==================================================================================================
# 7. E23-A BINARY MAPPING
# ==================================================================================================

E23_MR = 0
E23_MS = 1


# ==================================================================================================
# 8. E23-C SEARCH GRID
#
# mr_gate = confidence required for E21-B MS -> MR
#
# ms_gate = confidence required for E21-B MR -> MS
# ==================================================================================================

MR_GATES = [
    None,
    0.50,
    0.60,
    0.65,
    0.70,
    0.75,
    0.80,
    0.85,
    0.90,
    0.92,
    0.94,
    0.95,
    0.97,
    0.99,
]


MS_GATES = [
    None,
    0.50,
    0.60,
    0.65,
    0.70,
    0.75,
    0.80,
    0.85,
    0.90,
    0.92,
    0.94,
    0.95,
    0.97,
    0.99,
]


# ==================================================================================================
# 9. LOAD E21-B CACHE
# ==================================================================================================

with open(
    E21B_CACHE_PATH,
    "r",
    encoding="utf-8"
) as f:

    base_cache = json.load(f)


print()
print(
    f"Cached validation detections : "
    f"{len(base_cache):,}"
)


num_e21 = sum(
    1
    for item in base_cache
    if item.get("e21") is not None
)


print(
    f"Entries with E21-A output    : "
    f"{num_e21:,}"
)


if num_e21 != len(base_cache):

    raise RuntimeError(
        "Expected E21-A output for every cached validation detection."
    )


# ==================================================================================================
# 10. LOAD GT
# ==================================================================================================

with open(
    GT_PATH,
    "r",
    encoding="utf-8"
) as f:

    gt_json = json.load(f)


image_lookup = {
    int(img["id"]):
        img["file_name"]

    for img in gt_json["images"]
}


coco_gt = COCO(
    str(GT_PATH)
)


print(
    f"Validation GT images         : "
    f"{len(image_lookup):,}"
)


# ==================================================================================================
# 11. RECONSTRUCT E20-A
# ==================================================================================================

def reconstruct_e20a(
    item
):

    final_idx = int(
        item["e16_final_idx"]
    )


    mn_probs = np.asarray(
        item["mn_probs"],
        dtype=np.float32
    )


    classifier_prob = float(
        mn_probs[
            final_idx
        ]
    )


    source = "e16"


    conv = item.get(
        "conv"
    )


    if conv is not None:

        conv_global_idx = int(
            conv[
                "conv_global_idx"
            ]
        )


        conv_top_prob = float(
            conv[
                "conv_top_prob"
            ]
        )


        required_gate = None


        if conv_global_idx == MR:
            required_gate = E20_MR_GATE

        elif conv_global_idx == MS:
            required_gate = E20_MS_GATE

        elif conv_global_idx == NP:
            required_gate = E20_NP_GATE

        elif conv_global_idx == PET_OIL:
            required_gate = E20_PETOIL_GATE


        if (
            required_gate is not None
            and
            conv_top_prob >= required_gate
        ):

            final_idx = (
                conv_global_idx
            )


            classifier_prob = (
                conv_top_prob
            )


            source = "e18"


    return (
        int(final_idx),
        float(classifier_prob),
        source,
    )


# ==================================================================================================
# 12. RECONSTRUCT FROZEN E21-B
# ==================================================================================================

def reconstruct_e21b(
    item
):

    (
        final_idx,
        classifier_prob,
        source
    ) = reconstruct_e20a(
        item
    )


    e21 = item[
        "e21"
    ]


    e21_global_idx = int(
        e21[
            "global_idx"
        ]
    )


    e21_top_prob = float(
        e21[
            "top_prob"
        ]
    )


    # ------------------------------------------------------------------------------------------------
    # E21-A -> MR
    # ------------------------------------------------------------------------------------------------

    if (
        e21_global_idx == MR
        and
        e21_top_prob >= E21_MR_GATE
    ):

        final_idx = MR

        classifier_prob = (
            e21_top_prob
        )

        source = "e21"


    # ------------------------------------------------------------------------------------------------
    # E21-A -> MS
    # ------------------------------------------------------------------------------------------------

    elif (
        e21_global_idx == MS
        and
        e21_top_prob >= E21_MS_GATE
    ):

        final_idx = MS

        classifier_prob = (
            e21_top_prob
        )

        source = "e21"


    # ------------------------------------------------------------------------------------------------
    # E21-B score
    # ------------------------------------------------------------------------------------------------

    yolo_conf = max(
        float(
            item[
                "yolo_conf"
            ]
        ),
        1e-12
    )


    classifier_prob = max(
        float(
            classifier_prob
        ),
        1e-12
    )


    score = (

        yolo_conf
        ** ALPHA

    ) * (

        classifier_prob
        ** (
            1.0
            - ALPHA
        )
    )


    return {
        "class_idx":
            int(
                final_idx
            ),

        "score":
            float(
                score
            ),

        "source":
            source,
    }


# ==================================================================================================
# 13. RECONSTRUCT ALL FROZEN E21-B PREDICTIONS
# ==================================================================================================

candidate_indices = []


e21b_class_counts = Counter()


for idx, item in enumerate(
    base_cache
):

    e21b = reconstruct_e21b(
        item
    )


    item[
        "_e21b_class_idx"
    ] = int(
        e21b[
            "class_idx"
        ]
    )


    item[
        "_e21b_score"
    ] = float(
        e21b[
            "score"
        ]
    )


    item[
        "_e21b_source"
    ] = (
        e21b[
            "source"
        ]
    )


    e21b_class_counts[
        e21b[
            "class_idx"
        ]
    ] += 1


    if e21b[
        "class_idx"
    ] in {
        MR,
        MS,
    }:

        candidate_indices.append(
            idx
        )


print()
print("=" * 118)
print("FROZEN E21-B PREDICTION DISTRIBUTION")
print("=" * 118)


for class_idx, class_name in enumerate(
    CLASS_NAMES
):

    print(
        f"{class_name:28s}: "
        f"{e21b_class_counts[class_idx]:7,d}"
    )


print()
print(
    f"E23-A MR/MS candidates : "
    f"{len(candidate_indices):,}"
)


print(
    f"Candidate fraction      : "
    f"{100 * len(candidate_indices) / len(base_cache):.2f}%"
)


# ==================================================================================================
# 14. LOAD E23-A
# ==================================================================================================

print()
print("=" * 118)
print("LOADING FROZEN E23-A")
print("=" * 118)


e23_model = models.convnext_tiny(
    weights=None
)


in_features = (
    e23_model.classifier[
        2
    ].in_features
)


e23_model.classifier[
    2
] = nn.Linear(
    in_features,
    2
)


checkpoint = torch.load(
    E23A_CKPT,
    map_location=DEVICE
)


if (
    isinstance(
        checkpoint,
        dict
    )
    and
    "model_state_dict"
    in checkpoint
):

    state_dict = (
        checkpoint[
            "model_state_dict"
        ]
    )

else:

    state_dict = checkpoint


clean_state = {}


for key, value in state_dict.items():

    clean_key = (

        key[7:]

        if key.startswith(
            "module."
        )

        else key
    )


    clean_state[
        clean_key
    ] = value


output_classes = int(
    clean_state[
        "classifier.2.weight"
    ].shape[
        0
    ]
)


print(
    f"Checkpoint output classes : "
    f"{output_classes}"
)


if output_classes != 2:

    raise RuntimeError(
        f"E23-A must have two outputs. Found {output_classes}."
    )


e23_model.load_state_dict(
    clean_state,
    strict=True
)


e23_model = e23_model.to(
    DEVICE
)


e23_model.eval()


if isinstance(
    checkpoint,
    dict
):

    print(
        f"Checkpoint epoch          : "
        f"{checkpoint.get('epoch', 'unknown')}"
    )


    print(
        f"Checkpoint val macro-F1   : "
        f"{checkpoint.get('val_macro_f1', 'unknown')}"
    )


print(
    "E23-A loaded."
)


# ==================================================================================================
# 15. TRANSFORM
# ==================================================================================================

IMAGE_SIZE = 224


e23_transform = v2.Compose(
    [

        v2.Resize(
            (
                IMAGE_SIZE,
                IMAGE_SIZE
            ),
            antialias=True
        ),

        v2.ToImage(),

        v2.ToDtype(
            torch.float32,
            scale=True
        ),

        v2.Normalize(
            mean=[
                0.485,
                0.456,
                0.406
            ],
            std=[
                0.229,
                0.224,
                0.225
            ]
        ),
    ]
)


# ==================================================================================================
# 16. CANDIDATE DATASET
# ==================================================================================================

class CandidateCropDataset(
    Dataset
):

    def __init__(
        self,
        cache,
        candidate_indices,
        image_lookup,
        image_root
    ):

        self.cache = cache

        self.candidate_indices = (
            candidate_indices
        )

        self.image_lookup = (
            image_lookup
        )

        self.image_root = Path(
            image_root
        )


    def __len__(
        self
    ):

        return len(
            self.candidate_indices
        )


    def __getitem__(
        self,
        dataset_idx
    ):

        cache_idx = int(
            self.candidate_indices[
                dataset_idx
            ]
        )


        item = self.cache[
            cache_idx
        ]


        image_id = int(
            item[
                "image_id"
            ]
        )


        filename = self.image_lookup[
            image_id
        ]


        image_path = (
            self.image_root
            / filename
        )


        with Image.open(
            image_path
        ) as img:

            image = img.convert(
                "RGB"
            )


            width, height = image.size


            x1, y1, x2, y2 = [
                float(v)
                for v in item[
                    "bbox"
                ]
            ]


            x1 = max(
                0.0,
                min(
                    x1,
                    width - 1
                )
            )


            y1 = max(
                0.0,
                min(
                    y1,
                    height - 1
                )
            )


            x2 = max(
                x1 + 1.0,
                min(
                    x2,
                    width
                )
            )


            y2 = max(
                y1 + 1.0,
                min(
                    y2,
                    height
                )
            )


            crop = image.crop(
                (
                    int(
                        np.floor(x1)
                    ),

                    int(
                        np.floor(y1)
                    ),

                    int(
                        np.ceil(x2)
                    ),

                    int(
                        np.ceil(y2)
                    ),
                )
            )


        crop = e23_transform(
            crop
        )


        return (
            crop,
            cache_idx,
        )


# ==================================================================================================
# 17. RUN E23-A OR LOAD CACHE
# ==================================================================================================

if E23_CACHE_PATH.exists():

    print()
    print("=" * 118)
    print("FOUND EXISTING E23-C CACHE")
    print("=" * 118)


    with open(
        E23_CACHE_PATH,
        "r",
        encoding="utf-8"
    ) as f:

        cache = json.load(f)


    print(
        f"Loaded entries : "
        f"{len(cache):,}"
    )


else:

    print()
    print("=" * 118)
    print("RUNNING E23-A ON FROZEN E21-B MR/MS CANDIDATES")
    print("=" * 118)


    cache = base_cache


    for item in cache:
        item["e23"] = None


    candidate_dataset = CandidateCropDataset(
        cache=cache,
        candidate_indices=candidate_indices,
        image_lookup=image_lookup,
        image_root=VAL_IMAGES
    )


    candidate_loader = DataLoader(
        candidate_dataset,
        batch_size=32,
        shuffle=False,
        num_workers=0,
        pin_memory=(
            DEVICE.type
            == "cuda"
        )
    )


    start_time = time.time()


    processed = 0


    with torch.inference_mode():

        for (
            images,
            cache_indices
        ) in tqdm(
            candidate_loader,
            desc="E23-A on E21-B MR/MS"
        ):

            images = images.to(
                DEVICE,
                non_blocking=True
            )


            logits = e23_model(
                images
            )


            probs = torch.softmax(
                logits,
                dim=1
            )


            top_probs, local_indices = torch.max(
                probs,
                dim=1
            )


            probs_np = (
                probs
                .detach()
                .cpu()
                .numpy()
            )


            local_np = (
                local_indices
                .detach()
                .cpu()
                .numpy()
            )


            top_np = (
                top_probs
                .detach()
                .cpu()
                .numpy()
            )


            cache_indices_np = (
                cache_indices
                .numpy()
            )


            for i in range(
                len(
                    cache_indices_np
                )
            ):

                cache_idx = int(
                    cache_indices_np[i]
                )


                local_idx = int(
                    local_np[i]
                )


                global_idx = (
                    MR
                    if local_idx == E23_MR
                    else MS
                )


                cache[
                    cache_idx
                ][
                    "e23"
                ] = {

                    "probs":
                        probs_np[
                            i
                        ].tolist(),

                    "local_idx":
                        local_idx,

                    "global_idx":
                        int(
                            global_idx
                        ),

                    "top_prob":
                        float(
                            top_np[
                                i
                            ]
                        ),
                }


                processed += 1


    inference_minutes = (
        time.time()
        -
        start_time
    ) / 60.0


    print()
    print(
        f"E23-A outputs created : "
        f"{processed:,}"
    )


    print(
        f"Inference time        : "
        f"{inference_minutes:.2f} min"
    )


    with open(
        E23_CACHE_PATH,
        "w",
        encoding="utf-8"
    ) as f:

        json.dump(
            cache,
            f
        )


    print(
        f"Cache saved           : "
        f"{E23_CACHE_PATH}"
    )


# ==================================================================================================
# 18. VERIFY CACHE
# ==================================================================================================

num_e23 = sum(
    1
    for item in cache
    if item.get("e23") is not None
)


print()
print(
    f"E23-A cached outputs : "
    f"{num_e23:,}"
)


if num_e23 != len(
    candidate_indices
):

    raise RuntimeError(

        f"Expected {len(candidate_indices):,} "
        f"E23-A candidate outputs but found {num_e23:,}."
    )


# ==================================================================================================
# 19. EVALUATION HELPERS
# ==================================================================================================

def valid_mean(
    values
):

    values = np.asarray(
        values
    )


    valid = values[
        values > -1
    ]


    if valid.size == 0:
        return float("nan")


    return float(
        np.mean(
            valid
        )
    )


def evaluate_predictions(
    predictions
):

    coco_dt = coco_gt.loadRes(
        predictions
    )


    evaluator = COCOeval(
        coco_gt,
        coco_dt,
        "bbox"
    )


    evaluator.params.maxDets = [
        1,
        10,
        100,
    ]


    evaluator.evaluate()
    evaluator.accumulate()


    precision = (
        evaluator.eval[
            "precision"
        ]
    )


    recall = (
        evaluator.eval[
            "recall"
        ]
    )


    iou_thresholds = (
        evaluator.params.iouThrs
    )


    idx50 = int(
        np.where(
            np.isclose(
                iou_thresholds,
                0.50
            )
        )[0][0]
    )


    idx75 = int(
        np.where(
            np.isclose(
                iou_thresholds,
                0.75
            )
        )[0][0]
    )


    overall = {

        "AP50_95":
            valid_mean(
                precision[
                    :,
                    :,
                    :,
                    0,
                    -1
                ]
            ),

        "AP50":
            valid_mean(
                precision[
                    idx50,
                    :,
                    :,
                    0,
                    -1
                ]
            ),

        "AP75":
            valid_mean(
                precision[
                    idx75,
                    :,
                    :,
                    0,
                    -1
                ]
            ),

        "AR100":
            valid_mean(
                recall[
                    :,
                    :,
                    0,
                    -1
                ]
            ),
    }


    class_metrics = {}


    for class_idx, class_name in enumerate(
        CLASS_NAMES
    ):

        class_metrics[
            class_name
        ] = {

            "AP50":
                valid_mean(
                    precision[
                        idx50,
                        :,
                        class_idx,
                        0,
                        -1
                    ]
                ),

            "AP50_95":
                valid_mean(
                    precision[
                        :,
                        :,
                        class_idx,
                        0,
                        -1
                    ]
                ),
        }


    six_plastic_mean_ap50 = float(
        np.mean(
            [
                class_metrics[
                    CLASS_NAMES[idx]
                ][
                    "AP50"
                ]

                for idx in PLASTIC_INDICES
            ]
        )
    )


    mr_ms_mean_ap50 = float(
        np.mean(
            [
                class_metrics[
                    "mixed_plastic_rigid"
                ][
                    "AP50"
                ],

                class_metrics[
                    "mixed_plastic_soft"
                ][
                    "AP50"
                ],
            ]
        )
    )


    return {

        "overall":
            overall,

        "class_metrics":
            class_metrics,

        "six_plastic_mean_AP50":
            six_plastic_mean_ap50,

        "MR_MS_mean_AP50":
            mr_ms_mean_ap50,
    }


# ==================================================================================================
# 20. BUILD PREDICTIONS
# ==================================================================================================

def build_predictions(
    mr_gate=None,
    ms_gate=None
):

    predictions = []


    proposals = Counter()

    accepted = Counter()


    final_sources = Counter()


    for item in cache:

        pre_class = int(
            item[
                "_e21b_class_idx"
            ]
        )


        final_class = (
            pre_class
        )


        final_score = float(
            item[
                "_e21b_score"
            ]
        )


        source = (
            item[
                "_e21b_source"
            ]
        )


        e23 = item.get(
            "e23"
        )


        if (
            e23 is not None
            and
            pre_class in {
                MR,
                MS,
            }
        ):

            e23_class = int(
                e23[
                    "global_idx"
                ]
            )


            e23_prob = float(
                e23[
                    "top_prob"
                ]
            )


            # --------------------------------------------------------------------------------------
            # Frozen E21-B = MS
            # E23-A proposes MR
            # --------------------------------------------------------------------------------------

            if (
                pre_class == MS
                and
                e23_class == MR
            ):

                proposals[
                    "MS->MR"
                ] += 1


                if (
                    mr_gate is not None
                    and
                    e23_prob >= mr_gate
                ):

                    final_class = MR

                    source = "e23_binary"


                    accepted[
                        "MS->MR"
                    ] += 1


            # --------------------------------------------------------------------------------------
            # Frozen E21-B = MR
            # E23-A proposes MS
            # --------------------------------------------------------------------------------------

            elif (
                pre_class == MR
                and
                e23_class == MS
            ):

                proposals[
                    "MR->MS"
                ] += 1


                if (
                    ms_gate is not None
                    and
                    e23_prob >= ms_gate
                ):

                    final_class = MS

                    source = "e23_binary"


                    accepted[
                        "MR->MS"
                    ] += 1


        x1, y1, x2, y2 = [
            float(v)
            for v in item[
                "bbox"
            ]
        ]


        width = (
            x2 - x1
        )


        height = (
            y2 - y1
        )


        if (
            width <= 0
            or
            height <= 0
        ):
            continue


        predictions.append(
            {

                "image_id":
                    int(
                        item[
                            "image_id"
                        ]
                    ),

                "category_id":
                    int(
                        final_class
                        + 1
                    ),

                "bbox": [
                    x1,
                    y1,
                    width,
                    height,
                ],

                # Preserve E21-B score.
                "score":
                    float(
                        final_score
                    ),
            }
        )


        final_sources[
            source
        ] += 1


    return {

        "predictions":
            predictions,

        "proposal_counts":
            dict(
                proposals
            ),

        "accepted_counts":
            dict(
                accepted
            ),

        "source_counts":
            dict(
                final_sources
            ),
    }


# ==================================================================================================
# 21. FROZEN E21-B BASELINE
# ==================================================================================================

print()
print("=" * 118)
print("FROZEN E21-B VALIDATION BASELINE")
print("=" * 118)


baseline_build = build_predictions(
    mr_gate=None,
    ms_gate=None
)


baseline_eval = evaluate_predictions(
    baseline_build[
        "predictions"
    ]
)


print()
print(
    f"Six-plastic mean AP50 : "
    f"{baseline_eval['six_plastic_mean_AP50'] * 100:.4f}%"
)


print(
    f"MR AP50               : "
    f"{baseline_eval['class_metrics']['mixed_plastic_rigid']['AP50'] * 100:.4f}%"
)


print(
    f"MS AP50               : "
    f"{baseline_eval['class_metrics']['mixed_plastic_soft']['AP50'] * 100:.4f}%"
)


print(
    f"MR/MS mean AP50       : "
    f"{baseline_eval['MR_MS_mean_AP50'] * 100:.4f}%"
)


print(
    f"Overall AP50-95       : "
    f"{baseline_eval['overall']['AP50_95'] * 100:.4f}%"
)


print(
    f"Overall AP50          : "
    f"{baseline_eval['overall']['AP50'] * 100:.4f}%"
)


# ==================================================================================================
# 22. SANITY CHECK
#
# Previous validated E21-B reconstruction:
#
# six-plastic mean AP50 ~= 68.3369%
# MR                  ~= 51.1137%
# MS                  ~= 43.1823%
# AP50-95             ~= 45.9515%
# ==================================================================================================

print()
print(
    "Reference E21-B validation:"
)

print(
    "  Plastic AP50 ≈ 68.3369%"
)

print(
    "  MR AP50      ≈ 51.1137%"
)

print(
    "  MS AP50      ≈ 43.1823%"
)

print(
    "  AP50-95      ≈ 45.9515%"
)


# ==================================================================================================
# 23. GRID SEARCH
# ==================================================================================================

def gate_label(
    gate
):

    if gate is None:
        return "OFF"


    return f"{gate:.2f}"


results = []


total_combinations = (
    len(MR_GATES)
    *
    len(MS_GATES)
)


counter = 0


print()
print("=" * 118)
print("E23-C MR/MS ROUTING SEARCH — VALIDATION ONLY")
print("=" * 118)


for mr_gate in MR_GATES:

    for ms_gate in MS_GATES:

        counter += 1


        built = build_predictions(
            mr_gate=mr_gate,
            ms_gate=ms_gate
        )


        evaluation = evaluate_predictions(
            built[
                "predictions"
            ]
        )


        result = {

            "mr_gate":
                mr_gate,

            "ms_gate":
                ms_gate,

            "six_plastic_mean_AP50":
                evaluation[
                    "six_plastic_mean_AP50"
                ],

            "MR_MS_mean_AP50":
                evaluation[
                    "MR_MS_mean_AP50"
                ],

            "overall":
                evaluation[
                    "overall"
                ],

            "class_metrics":
                evaluation[
                    "class_metrics"
                ],

            "proposal_counts":
                built[
                    "proposal_counts"
                ],

            "accepted_counts":
                built[
                    "accepted_counts"
                ],

            "source_counts":
                built[
                    "source_counts"
                ],
        }


        results.append(
            result
        )


        print(

            f"[{counter:03d}/{total_combinations}] "

            f"MR={gate_label(mr_gate):>4s} "

            f"MS={gate_label(ms_gate):>4s} | "

            f"Plastic="
            f"{evaluation['six_plastic_mean_AP50'] * 100:7.3f}% | "

            f"MR="
            f"{evaluation['class_metrics']['mixed_plastic_rigid']['AP50'] * 100:6.2f}% | "

            f"MS="
            f"{evaluation['class_metrics']['mixed_plastic_soft']['AP50'] * 100:6.2f}% | "

            f"MR/MS="
            f"{evaluation['MR_MS_mean_AP50'] * 100:6.2f}% | "

            f"AP="
            f"{evaluation['overall']['AP50_95'] * 100:6.3f}%"
        )


# ==================================================================================================
# 24. RANK
# ==================================================================================================

ranked = sorted(

    results,

    key=lambda x: (

        x[
            "six_plastic_mean_AP50"
        ],

        x[
            "MR_MS_mean_AP50"
        ],

        x[
            "overall"
        ][
            "AP50_95"
        ],
    ),

    reverse=True
)


best = ranked[
    0
]


# ==================================================================================================
# 25. TOP 20
# ==================================================================================================

print()
print()
print("=" * 118)
print("TOP 20 E23-C VALIDATION CONFIGURATIONS")
print("=" * 118)


print(
    f"\n"
    f"{'Rank':>4s} "
    f"{'MR':>6s} "
    f"{'MS':>6s} "
    f"{'Plastic AP50':>14s} "
    f"{'MR AP50':>10s} "
    f"{'MS AP50':>10s} "
    f"{'MR/MS':>10s} "
    f"{'AP50-95':>10s}"
)


print(
    "-" * 92
)


for rank, result in enumerate(
    ranked[:20],
    start=1
):

    print(

        f"{rank:4d} "

        f"{gate_label(result['mr_gate']):>6s} "

        f"{gate_label(result['ms_gate']):>6s} "

        f"{result['six_plastic_mean_AP50'] * 100:13.4f}% "

        f"{result['class_metrics']['mixed_plastic_rigid']['AP50'] * 100:9.2f}% "

        f"{result['class_metrics']['mixed_plastic_soft']['AP50'] * 100:9.2f}% "

        f"{result['MR_MS_mean_AP50'] * 100:9.2f}% "

        f"{result['overall']['AP50_95'] * 100:9.3f}%"
    )


# ==================================================================================================
# 26. DELTAS
# ==================================================================================================

delta = {

    "six_plastic_mean_AP50_pp":

        (
            best[
                "six_plastic_mean_AP50"
            ]
            -
            baseline_eval[
                "six_plastic_mean_AP50"
            ]
        )
        * 100.0,


    "MR_AP50_pp":

        (
            best[
                "class_metrics"
            ][
                "mixed_plastic_rigid"
            ][
                "AP50"
            ]
            -
            baseline_eval[
                "class_metrics"
            ][
                "mixed_plastic_rigid"
            ][
                "AP50"
            ]
        )
        * 100.0,


    "MS_AP50_pp":

        (
            best[
                "class_metrics"
            ][
                "mixed_plastic_soft"
            ][
                "AP50"
            ]
            -
            baseline_eval[
                "class_metrics"
            ][
                "mixed_plastic_soft"
            ][
                "AP50"
            ]
        )
        * 100.0,


    "MR_MS_mean_AP50_pp":

        (
            best[
                "MR_MS_mean_AP50"
            ]
            -
            baseline_eval[
                "MR_MS_mean_AP50"
            ]
        )
        * 100.0,


    "overall_AP50_95_pp":

        (
            best[
                "overall"
            ][
                "AP50_95"
            ]
            -
            baseline_eval[
                "overall"
            ][
                "AP50_95"
            ]
        )
        * 100.0,


    "overall_AP50_pp":

        (
            best[
                "overall"
            ][
                "AP50"
            ]
            -
            baseline_eval[
                "overall"
            ][
                "AP50"
            ]
        )
        * 100.0,
}


# ==================================================================================================
# 27. BEST
# ==================================================================================================

print()
print("=" * 118)
print("BEST E23-C CONFIGURATION — VALIDATION")
print("=" * 118)


print()
print(
    f"E21-B MS -> MR gate : "
    f"{gate_label(best['mr_gate'])}"
)


print(
    f"E21-B MR -> MS gate : "
    f"{gate_label(best['ms_gate'])}"
)


print()
print(
    f"Six-plastic mean AP50 : "
    f"{best['six_plastic_mean_AP50'] * 100:.4f}%"
)


print(
    f"MR AP50               : "
    f"{best['class_metrics']['mixed_plastic_rigid']['AP50'] * 100:.4f}%"
)


print(
    f"MS AP50               : "
    f"{best['class_metrics']['mixed_plastic_soft']['AP50'] * 100:.4f}%"
)


print(
    f"MR/MS mean AP50       : "
    f"{best['MR_MS_mean_AP50'] * 100:.4f}%"
)


print()
print(
    f"Overall AP50-95       : "
    f"{best['overall']['AP50_95'] * 100:.4f}%"
)


print(
    f"Overall AP50          : "
    f"{best['overall']['AP50'] * 100:.4f}%"
)


print(
    f"Overall AP75          : "
    f"{best['overall']['AP75'] * 100:.4f}%"
)


print(
    f"AR100                 : "
    f"{best['overall']['AR100'] * 100:.4f}%"
)


# ==================================================================================================
# 28. CLASS-WISE
# ==================================================================================================

print()
print("=" * 118)
print("BEST E23-C — CLASS-WISE VALIDATION")
print("=" * 118)


print(
    f"\n"
    f"{'Class':28s}"
    f"{'AP50':>14s}"
    f"{'AP50-95':>14s}"
)


print(
    "-" * 56
)


for class_name in CLASS_NAMES:

    metrics = best[
        "class_metrics"
    ][
        class_name
    ]


    print(
        f"{class_name:28s}"
        f"{metrics['AP50'] * 100:13.2f}%"
        f"{metrics['AP50_95'] * 100:13.2f}%"
    )


# ==================================================================================================
# 29. ROUTING
# ==================================================================================================

print()
print("=" * 118)
print("BEST E23-C — ROUTING")
print("=" * 118)


print()
print(
    "E23-A disagreement proposals:"
)


for route, count in sorted(
    best[
        "proposal_counts"
    ].items()
):

    print(
        f"{route:12s}: "
        f"{count:6,d}"
    )


print()
print(
    "Accepted flips:"
)


for route, count in sorted(
    best[
        "accepted_counts"
    ].items()
):

    print(
        f"{route:12s}: "
        f"{count:6,d}"
    )


print()
print(
    "Final sources:"
)


total_sources = sum(
    best[
        "source_counts"
    ].values()
)


for source, count in best[
    "source_counts"
].items():

    pct = (
        100.0
        * count
        / total_sources
    )


    print(
        f"{source:18s}: "
        f"{count:7,d} "
        f"({pct:6.2f}%)"
    )


# ==================================================================================================
# 30. DELTA
# ==================================================================================================

print()
print("=" * 118)
print("DELTA VS FROZEN E21-B VALIDATION")
print("=" * 118)


for metric_name, value in delta.items():

    print(
        f"{metric_name:32s}: "
        f"{value:+.4f} pp"
    )


# ==================================================================================================
# 31. DECISION
# ==================================================================================================

improves_primary = (
    best[
        "six_plastic_mean_AP50"
    ]
    >
    baseline_eval[
        "six_plastic_mean_AP50"
    ]
)


improves_mrms = (
    best[
        "MR_MS_mean_AP50"
    ]
    >
    baseline_eval[
        "MR_MS_mean_AP50"
    ]
)


print()
print("=" * 118)
print("E23-C VALIDATION DECISION")
print("=" * 118)


if (
    improves_primary
    and
    improves_mrms
):

    print()
    print(
        "E23-C improves BOTH:"
    )

    print(
        "  - six-plastic mean AP50"
    )

    print(
        "  - MR/MS mean AP50"
    )

    print()
    print(
        "E23-C is eligible for frozen TEST consideration."
    )


elif improves_primary:

    print()
    print(
        "E23-C improves six-plastic mean AP50,"
    )

    print(
        "but not MR/MS mean AP50."
    )


elif improves_mrms:

    print()
    print(
        "E23-C improves MR/MS mean AP50,"
    )

    print(
        "but not six-plastic mean AP50."
    )


else:

    print()
    print(
        "E23-C does not improve the target metrics."
    )

    print(
        "Do not promote it to TEST."
    )


# ==================================================================================================
# 32. SAVE
# ==================================================================================================

best_build = build_predictions(
    mr_gate=best[
        "mr_gate"
    ],
    ms_gate=best[
        "ms_gate"
    ]
)


with open(
    BEST_PREDICTIONS_PATH,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        best_build[
            "predictions"
        ],
        f
    )


with open(
    GRID_RESULTS_PATH,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        ranked,
        f,
        indent=2
    )


output_config = {

    "experiment":
        "E23-C",

    "description":
        (
            "Frozen E21-B pipeline plus "
            "E23-A binary MR/MS specialist"
        ),

    "dataset":
        "validation",

    "test_used":
        False,

    "base_pipeline":
        "E21-B",

    "specialist":
        "E23-A",

    "routing":
        {
            "allowed":
                [
                    "MS->MR",
                    "MR->MS",
                ],

            "other_classes_untouched":
                True,

            "preserve_E21B_detection_score":
                True,
        },

    "selection":
        {
            "primary":
                "six-plastic mean AP50",

            "secondary":
                "MR/MS mean AP50",

            "tie_break":
                "overall AP50-95",
        },

    "E21B_baseline":
        baseline_eval,

    "best_E23C":
        best,

    "delta_vs_E21B":
        delta,

    "promote_to_test":
        bool(
            improves_primary
            and
            improves_mrms
        ),
}


with open(
    BEST_CONFIG_PATH,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        output_config,
        f,
        indent=2
    )


summary_lines = [

    "=" * 118,

    "E23-C — E21-B + BINARY MR/MS SPECIALIST",

    "=" * 118,

    "",

    "VALIDATION ONLY",

    "TEST NOT USED",

    "",

    "Frozen base: E21-B",

    "New specialist: E23-A",

    "",

    f"MS -> MR gate = "
    f"{gate_label(best['mr_gate'])}",

    f"MR -> MS gate = "
    f"{gate_label(best['ms_gate'])}",

    "",

    f"Six-plastic mean AP50 = "
    f"{best['six_plastic_mean_AP50'] * 100:.4f}%",

    f"MR AP50               = "
    f"{best['class_metrics']['mixed_plastic_rigid']['AP50'] * 100:.4f}%",

    f"MS AP50               = "
    f"{best['class_metrics']['mixed_plastic_soft']['AP50'] * 100:.4f}%",

    f"MR/MS mean AP50       = "
    f"{best['MR_MS_mean_AP50'] * 100:.4f}%",

    f"Overall AP50-95       = "
    f"{best['overall']['AP50_95'] * 100:.4f}%",

    "",

    "Delta vs E21-B:",
]


for metric_name, value in delta.items():

    summary_lines.append(
        f"  {metric_name:32s}: "
        f"{value:+.4f} pp"
    )


with open(
    SUMMARY_PATH,
    "w",
    encoding="utf-8"
) as f:

    f.write(
        "\n".join(
            summary_lines
        )
    )


# ==================================================================================================
# 33. COMPLETE
# ==================================================================================================

print()
print("=" * 118)
print("E23-C COMPLETE")
print("=" * 118)


print()
print(
    f"E23 cache        : "
    f"{E23_CACHE_PATH}"
)


print(
    f"Grid results     : "
    f"{GRID_RESULTS_PATH}"
)


print(
    f"Best config      : "
    f"{BEST_CONFIG_PATH}"
)


print(
    f"Best predictions : "
    f"{BEST_PREDICTIONS_PATH}"
)


print(
    f"Summary          : "
    f"{SUMMARY_PATH}"
)

E23-C — E21-B + BINARY MR/MS SPECIALIST

PyTorch : 2.13.0+cu126
Device  : cuda
GPU     : NVIDIA GeForce RTX 3050 Ti Laptop GPU

Validation images : C:\Users\varda\Documents\_My Computer\COMMON_space\Learning\Upgrad_EPGP_MS\upgrad_IIITB_MS_course\MS_LJMU_Material\Topic Data\SortWaste\dataset\dataset\splited_all_dataset_coco\val\images
E21-B cache       : C:\Users\varda\Documents\_My Computer\COMMON_space\Learning\Upgrad_EPGP_MS\upgrad_IIITB_MS_course\MS_LJMU_Material\Thesis_Code\runs\sortwaste\E21B_end_to_end_validation\E21B_cached_predictions_with_E21A.json
Validation GT     : C:\Users\varda\Documents\_My Computer\COMMON_space\Learning\Upgrad_EPGP_MS\upgrad_IIITB_MS_course\MS_LJMU_Material\Thesis_Code\runs\sortwaste\E18D_class_selective_convnext\E18D_val_gt_7class.json
E23-A checkpoint  : C:\Users\varda\Documents\_My Computer\COMMON_space\Learning\Upgrad_EPGP_MS\upgrad_IIITB_MS_course\MS_LJMU_Material\Thesis_Code\runs\sortwaste\E23A_binary_MR_MS_hardpair_convnext\E23A_ConvNeXtTiny_MR_M

E23-A on E21-B MR/MS: 100%|██████████| 467/467 [28:52<00:00,  3.71s/it]



E23-A outputs created : 14,943
Inference time        : 28.87 min
Cache saved           : C:\Users\varda\Documents\_My Computer\COMMON_space\Learning\Upgrad_EPGP_MS\upgrad_IIITB_MS_course\MS_LJMU_Material\Thesis_Code\runs\sortwaste\E23C_E21B_binary_MRMS_routing\E23C_cached_predictions_with_E23A.json

E23-A cached outputs : 14,943

FROZEN E21-B VALIDATION BASELINE
Loading and preparing results...
DONE (t=0.07s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=4.28s).
Accumulating evaluation results...
DONE (t=0.54s).

Six-plastic mean AP50 : 68.3369%
MR AP50               : 51.1137%
MS AP50               : 43.1823%
MR/MS mean AP50       : 47.1480%
Overall AP50-95       : 45.9515%
Overall AP50          : 61.3954%

Reference E21-B validation:
  Plastic AP50 ≈ 68.3369%
  MR AP50      ≈ 51.1137%
  MS AP50      ≈ 43.1823%
  AP50-95      ≈ 45.9515%

E23-C MR/MS ROUTING SEARCH — VALIDATION ONLY
Loading and preparing results...
DONE (t=0.0

## E23-D — Confidence-Contrastive Mixed Rigid/Mixed Soft Routing with Frozen E20-A and the E23-A Binary ConvNeXt-Tiny Specialist
E20-A prediction
    ↓
MR / MS ?
    ↓ yes
E23-A prediction
    ↓
Disagrees with E20-A ?
    ↓ yes
E23 confidence >= T_high ?
    ↓ yes
E20 confidence <= T_low ?
    ↓ yes
Override MR ↔ MS
    ↓
Final E23-D prediction

In [32]:
# E23-D — CONFIDENCE-CONTRASTIVE MR/MS ROUTING
#
# VALIDATION ONLY
# NO TEST ACCESS
# NO RETRAINING
#
# --------------------------------------------------------------------------------------------------
# BASE:
#   Frozen E20-A
#
# SPECIALIST:
#   E23-A binary MR/MS classifier
#
# DIFFERENCE FROM E23-B:
#
#   E23-B used only:
#
#       E23-A confidence >= threshold
#
#   E23-D requires BOTH:
#
#       E23-A confidence >= high threshold
#       AND
#       E20-A current-class confidence <= uncertainty threshold
#
#
# Allowed:
#
#   E20-A MS -> E23-A MR
#   E20-A MR -> E23-A MS
#
# All other classes untouched.
#
# Detection score is preserved from E20-A.
#
# Selection:
#
#   Primary   = six-plastic mean AP50
#   Secondary = MR/MS mean AP50
#   Tie-break = overall AP50-95
#
# ==================================================================================================


import json
from pathlib import Path
from collections import Counter

import numpy as np

from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval


# ==================================================================================================
# 1. PATHS
# ==================================================================================================

BASE = Path(
    r"C:\Users\varda\Documents\_My Computer\COMMON_space\Learning"
    r"\Upgrad_EPGP_MS\upgrad_IIITB_MS_course\MS_LJMU_Material"
)


THESIS_CODE = (
    BASE
    / "Thesis_Code"
)


# --------------------------------------------------------------------------------------------------
# Existing E23-B cache:
#
# Contains original E18-D entries plus:
#
#   _e20_class_idx
#   _e20_score
#   _e20_source
#   e23
#
# --------------------------------------------------------------------------------------------------

E23B_DIR = (
    THESIS_CODE
    / "runs"
    / "sortwaste"
    / "E23B_E20A_binary_MRMS_routing"
)


CACHE_PATH = (
    E23B_DIR
    / "E23B_cached_predictions_with_E23A.json"
)


# --------------------------------------------------------------------------------------------------
# Validation GT
# --------------------------------------------------------------------------------------------------

GT_PATH = (
    THESIS_CODE
    / "runs"
    / "sortwaste"
    / "E18D_class_selective_convnext"
    / "E18D_val_gt_7class.json"
)


# --------------------------------------------------------------------------------------------------
# Outputs
# --------------------------------------------------------------------------------------------------

OUTPUT_DIR = (
    THESIS_CODE
    / "runs"
    / "sortwaste"
    / "E23D_confidence_contrastive_MRMS"
)


OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)


GRID_RESULTS_PATH = (
    OUTPUT_DIR
    / "E23D_grid_results.json"
)


BEST_CONFIG_PATH = (
    OUTPUT_DIR
    / "E23D_best_configuration.json"
)


BEST_PREDICTIONS_PATH = (
    OUTPUT_DIR
    / "E23D_best_predictions.json"
)


SUMMARY_PATH = (
    OUTPUT_DIR
    / "E23D_summary.txt"
)


# ==================================================================================================
# 2. VERIFY
# ==================================================================================================

for path in [
    CACHE_PATH,
    GT_PATH,
]:

    if not path.exists():

        raise FileNotFoundError(
            f"Required path not found:\n{path}"
        )


print("=" * 120)
print("E23-D — CONFIDENCE-CONTRASTIVE MR/MS ROUTING")
print("=" * 120)

print()
print(f"Cache      : {CACHE_PATH}")
print(f"Val GT     : {GT_PATH}")
print(f"Output     : {OUTPUT_DIR}")


# ==================================================================================================
# 3. CLASSES
# ==================================================================================================

CLASS_NAMES = [
    "ecal",
    "hdpe",
    "mixed_plastic_rigid",
    "mixed_plastic_soft",
    "non_plastic",
    "pet",
    "pet_oil",
]


ECAL = 0
HDPE = 1
MR = 2
MS = 3
NP = 4
PET = 5
PET_OIL = 6


PLASTIC_INDICES = [
    ECAL,
    HDPE,
    MR,
    MS,
    PET,
    PET_OIL,
]


# ==================================================================================================
# 4. LOAD CACHE
# ==================================================================================================

with open(
    CACHE_PATH,
    "r",
    encoding="utf-8"
) as f:

    cache = json.load(f)


print()
print(
    f"Cached detections : "
    f"{len(cache):,}"
)


num_e23 = sum(
    1
    for item in cache
    if item.get("e23") is not None
)


print(
    f"E23-A outputs     : "
    f"{num_e23:,}"
)


# ==================================================================================================
# 5. VERIFY REQUIRED FIELDS
# ==================================================================================================

required_fields = [
    "_e20_class_idx",
    "_e20_score",
]


for idx, item in enumerate(cache[:10]):

    for field in required_fields:

        if field not in item:

            raise RuntimeError(
                f"Cache missing field '{field}' "
                f"at entry {idx}."
            )


# ==================================================================================================
# 6. LOAD GT
# ==================================================================================================

coco_gt = COCO(
    str(
        GT_PATH
    )
)


# ==================================================================================================
# 7. CONFIDENCE EXTRACTION
#
# IMPORTANT:
#
# We need a confidence representing how strongly E20-A supports
# its CURRENT class before E23-A.
#
# Rules:
#
#   If E20-A source = e18:
#       use accepted ConvNeXt top probability
#
#   Else:
#       use MobileNet probability corresponding to the
#       current E20-A class.
#
# This gives the "current classifier confidence" that E23-A
# must overcome.
# ==================================================================================================

def get_e20_class_confidence(
    item
):

    current_class = int(
        item[
            "_e20_class_idx"
        ]
    )


    source = item.get(
        "_e20_source",
        "e16"
    )


    # --------------------------------------------------------------------------------------------------
    # If final E20-A class came from E18 ConvNeXt
    # --------------------------------------------------------------------------------------------------

    if source == "e18":

        conv = item.get(
            "conv"
        )


        if conv is not None:

            conv_global_idx = int(
                conv[
                    "conv_global_idx"
                ]
            )


            conv_top_prob = float(
                conv[
                    "conv_top_prob"
                ]
            )


            if (
                conv_global_idx
                ==
                current_class
            ):

                return (
                    conv_top_prob
                )


    # --------------------------------------------------------------------------------------------------
    # Otherwise use MobileNet probability for current class
    # --------------------------------------------------------------------------------------------------

    mn_probs = np.asarray(
        item[
            "mn_probs"
        ],
        dtype=np.float32
    )


    return float(
        mn_probs[
            current_class
        ]
    )


# ==================================================================================================
# 8. PRECOMPUTE E20 CONFIDENCE
# ==================================================================================================

e20_confidences = []


for item in cache:

    conf = get_e20_class_confidence(
        item
    )


    item[
        "_e20_class_conf"
    ] = float(
        conf
    )


    e20_confidences.append(
        conf
    )


e20_confidences = np.asarray(
    e20_confidences,
    dtype=np.float32
)


print()
print("=" * 120)
print("E20-A CURRENT-CLASS CONFIDENCE SUMMARY")
print("=" * 120)


mr_confs = []
ms_confs = []


for item in cache:

    cls = int(
        item[
            "_e20_class_idx"
        ]
    )


    if cls == MR:

        mr_confs.append(
            item[
                "_e20_class_conf"
            ]
        )


    elif cls == MS:

        ms_confs.append(
            item[
                "_e20_class_conf"
            ]
        )


for name, values in [
    ("MR", mr_confs),
    ("MS", ms_confs),
]:

    arr = np.asarray(
        values,
        dtype=np.float32
    )


    print()
    print(name)

    print(
        f"Count   : {len(arr):,}"
    )

    print(
        f"Mean    : {np.mean(arr):.4f}"
    )

    print(
        f"Median  : {np.median(arr):.4f}"
    )

    print(
        f"P25     : {np.quantile(arr, 0.25):.4f}"
    )

    print(
        f"P75     : {np.quantile(arr, 0.75):.4f}"
    )


# ==================================================================================================
# 9. SEARCH GRIDS
#
# E23 minimum confidence:
#
#   very high because E23-B showed low thresholds were harmful
#
# E20 maximum confidence:
#
#   E23-A is allowed only when current E20-A confidence
#   is sufficiently weak.
# ==================================================================================================

E23_THRESHOLDS = [
    0.90,
    0.92,
    0.94,
    0.95,
    0.96,
    0.97,
    0.98,
    0.985,
    0.99,
    0.992,
    0.995,
    0.997,
    0.999,
]


E20_MAX_THRESHOLDS = [
    0.30,
    0.40,
    0.50,
    0.60,
    0.70,
    0.75,
    0.80,
    0.85,
    0.90,
    0.95,
]


# ==================================================================================================
# 10. EVALUATION HELPERS
# ==================================================================================================

def valid_mean(
    values
):

    values = np.asarray(
        values
    )


    valid = values[
        values > -1
    ]


    if valid.size == 0:

        return float(
            "nan"
        )


    return float(
        np.mean(
            valid
        )
    )


def evaluate_predictions(
    predictions
):

    coco_dt = coco_gt.loadRes(
        predictions
    )


    evaluator = COCOeval(
        coco_gt,
        coco_dt,
        "bbox"
    )


    evaluator.params.maxDets = [
        1,
        10,
        100,
    ]


    evaluator.evaluate()
    evaluator.accumulate()


    precision = (
        evaluator.eval[
            "precision"
        ]
    )


    recall = (
        evaluator.eval[
            "recall"
        ]
    )


    iou_thresholds = (
        evaluator.params.iouThrs
    )


    idx50 = int(
        np.where(
            np.isclose(
                iou_thresholds,
                0.50
            )
        )[0][0]
    )


    idx75 = int(
        np.where(
            np.isclose(
                iou_thresholds,
                0.75
            )
        )[0][0]
    )


    overall = {

        "AP50_95":
            valid_mean(
                precision[
                    :,
                    :,
                    :,
                    0,
                    -1
                ]
            ),

        "AP50":
            valid_mean(
                precision[
                    idx50,
                    :,
                    :,
                    0,
                    -1
                ]
            ),

        "AP75":
            valid_mean(
                precision[
                    idx75,
                    :,
                    :,
                    0,
                    -1
                ]
            ),

        "AR100":
            valid_mean(
                recall[
                    :,
                    :,
                    0,
                    -1
                ]
            ),
    }


    class_metrics = {}


    for class_idx, class_name in enumerate(
        CLASS_NAMES
    ):

        class_metrics[
            class_name
        ] = {

            "AP50":
                valid_mean(
                    precision[
                        idx50,
                        :,
                        class_idx,
                        0,
                        -1
                    ]
                ),

            "AP50_95":
                valid_mean(
                    precision[
                        :,
                        :,
                        class_idx,
                        0,
                        -1
                    ]
                ),
        }


    six_plastic = float(
        np.mean(
            [
                class_metrics[
                    CLASS_NAMES[idx]
                ][
                    "AP50"
                ]

                for idx in PLASTIC_INDICES
            ]
        )
    )


    mr_ms_mean = float(
        np.mean(
            [
                class_metrics[
                    "mixed_plastic_rigid"
                ][
                    "AP50"
                ],

                class_metrics[
                    "mixed_plastic_soft"
                ][
                    "AP50"
                ],
            ]
        )
    )


    return {

        "overall":
            overall,

        "class_metrics":
            class_metrics,

        "six_plastic_mean_AP50":
            six_plastic,

        "MR_MS_mean_AP50":
            mr_ms_mean,
    }


# ==================================================================================================
# 11. BUILD PREDICTIONS
#
# Direction controls:
#
#   MS -> MR:
#       e23_mr_min
#       e20_ms_max
#
#   MR -> MS:
#       e23_ms_min
#       e20_mr_max
#
# Passing None means OFF.
# ==================================================================================================

def build_predictions(
    e23_mr_min=None,
    e20_ms_max=None,
    e23_ms_min=None,
    e20_mr_max=None
):

    predictions = []


    proposals = Counter()

    accepted = Counter()


    accepted_e23_conf = {
        "MS->MR": [],
        "MR->MS": [],
    }


    accepted_e20_conf = {
        "MS->MR": [],
        "MR->MS": [],
    }


    for item in cache:

        pre_class = int(
            item[
                "_e20_class_idx"
            ]
        )


        final_class = (
            pre_class
        )


        final_score = float(
            item[
                "_e20_score"
            ]
        )


        e20_conf = float(
            item[
                "_e20_class_conf"
            ]
        )


        e23 = item.get(
            "e23"
        )


        if (
            e23 is not None
            and
            pre_class in {
                MR,
                MS,
            }
        ):

            e23_class = int(
                e23[
                    "global_idx"
                ]
            )


            e23_prob = float(
                e23[
                    "top_prob"
                ]
            )


            # ======================================================================================
            # E20-A MS -> E23-A MR
            # ======================================================================================

            if (
                pre_class == MS
                and
                e23_class == MR
            ):

                proposals[
                    "MS->MR"
                ] += 1


                if (
                    e23_mr_min is not None
                    and
                    e20_ms_max is not None
                    and
                    e23_prob >= e23_mr_min
                    and
                    e20_conf <= e20_ms_max
                ):

                    final_class = MR


                    accepted[
                        "MS->MR"
                    ] += 1


                    accepted_e23_conf[
                        "MS->MR"
                    ].append(
                        e23_prob
                    )


                    accepted_e20_conf[
                        "MS->MR"
                    ].append(
                        e20_conf
                    )


            # ======================================================================================
            # E20-A MR -> E23-A MS
            # ======================================================================================

            elif (
                pre_class == MR
                and
                e23_class == MS
            ):

                proposals[
                    "MR->MS"
                ] += 1


                if (
                    e23_ms_min is not None
                    and
                    e20_mr_max is not None
                    and
                    e23_prob >= e23_ms_min
                    and
                    e20_conf <= e20_mr_max
                ):

                    final_class = MS


                    accepted[
                        "MR->MS"
                    ] += 1


                    accepted_e23_conf[
                        "MR->MS"
                    ].append(
                        e23_prob
                    )


                    accepted_e20_conf[
                        "MR->MS"
                    ].append(
                        e20_conf
                    )


        x1, y1, x2, y2 = [
            float(v)
            for v in item[
                "bbox"
            ]
        ]


        width = (
            x2 - x1
        )


        height = (
            y2 - y1
        )


        if (
            width <= 0
            or
            height <= 0
        ):

            continue


        predictions.append(
            {

                "image_id":
                    int(
                        item[
                            "image_id"
                        ]
                    ),

                "category_id":
                    int(
                        final_class
                        + 1
                    ),

                "bbox": [
                    x1,
                    y1,
                    width,
                    height,
                ],

                "score":
                    final_score,
            }
        )


    return {

        "predictions":
            predictions,

        "proposal_counts":
            dict(
                proposals
            ),

        "accepted_counts":
            dict(
                accepted
            ),

        "accepted_e23_conf":
            accepted_e23_conf,

        "accepted_e20_conf":
            accepted_e20_conf,
    }


# ==================================================================================================
# 12. BASELINE
# ==================================================================================================

print()
print("=" * 120)
print("FROZEN E20-A BASELINE")
print("=" * 120)


baseline_build = build_predictions()


baseline_eval = evaluate_predictions(
    baseline_build[
        "predictions"
    ]
)


print()
print(
    f"Six-plastic mean AP50 : "
    f"{baseline_eval['six_plastic_mean_AP50'] * 100:.4f}%"
)


print(
    f"MR AP50               : "
    f"{baseline_eval['class_metrics']['mixed_plastic_rigid']['AP50'] * 100:.4f}%"
)


print(
    f"MS AP50               : "
    f"{baseline_eval['class_metrics']['mixed_plastic_soft']['AP50'] * 100:.4f}%"
)


print(
    f"MR/MS mean AP50       : "
    f"{baseline_eval['MR_MS_mean_AP50'] * 100:.4f}%"
)


print(
    f"Overall AP50-95       : "
    f"{baseline_eval['overall']['AP50_95'] * 100:.4f}%"
)


# ==================================================================================================
# 13. SEARCH EACH DIRECTION INDEPENDENTLY FIRST
#
# This is important:
#
# We first identify whether:
#
#   MS -> MR helps
#   MR -> MS helps
#
# independently.
#
# Then combine the best candidates.
# ==================================================================================================

direction_results = []


print()
print("=" * 120)
print("STAGE 1 — INDEPENDENT DIRECTION SEARCH")
print("=" * 120)


# --------------------------------------------------------------------------------------------------
# MS -> MR
# --------------------------------------------------------------------------------------------------

for e23_min in E23_THRESHOLDS:

    for e20_max in E20_MAX_THRESHOLDS:

        built = build_predictions(
            e23_mr_min=e23_min,
            e20_ms_max=e20_max,

            e23_ms_min=None,
            e20_mr_max=None
        )


        evaluation = evaluate_predictions(
            built[
                "predictions"
            ]
        )


        result = {

            "direction":
                "MS->MR",

            "e23_min":
                e23_min,

            "e20_max":
                e20_max,

            "six_plastic_mean_AP50":
                evaluation[
                    "six_plastic_mean_AP50"
                ],

            "MR_MS_mean_AP50":
                evaluation[
                    "MR_MS_mean_AP50"
                ],

            "overall_AP50_95":
                evaluation[
                    "overall"
                ][
                    "AP50_95"
                ],

            "MR_AP50":
                evaluation[
                    "class_metrics"
                ][
                    "mixed_plastic_rigid"
                ][
                    "AP50"
                ],

            "MS_AP50":
                evaluation[
                    "class_metrics"
                ][
                    "mixed_plastic_soft"
                ][
                    "AP50"
                ],

            "accepted":
                built[
                    "accepted_counts"
                ].get(
                    "MS->MR",
                    0
                ),
        }


        direction_results.append(
            result
        )


# --------------------------------------------------------------------------------------------------
# MR -> MS
# --------------------------------------------------------------------------------------------------

for e23_min in E23_THRESHOLDS:

    for e20_max in E20_MAX_THRESHOLDS:

        built = build_predictions(
            e23_mr_min=None,
            e20_ms_max=None,

            e23_ms_min=e23_min,
            e20_mr_max=e20_max
        )


        evaluation = evaluate_predictions(
            built[
                "predictions"
            ]
        )


        result = {

            "direction":
                "MR->MS",

            "e23_min":
                e23_min,

            "e20_max":
                e20_max,

            "six_plastic_mean_AP50":
                evaluation[
                    "six_plastic_mean_AP50"
                ],

            "MR_MS_mean_AP50":
                evaluation[
                    "MR_MS_mean_AP50"
                ],

            "overall_AP50_95":
                evaluation[
                    "overall"
                ][
                    "AP50_95"
                ],

            "MR_AP50":
                evaluation[
                    "class_metrics"
                ][
                    "mixed_plastic_rigid"
                ][
                    "AP50"
                ],

            "MS_AP50":
                evaluation[
                    "class_metrics"
                ][
                    "mixed_plastic_soft"
                ][
                    "AP50"
                ],

            "accepted":
                built[
                    "accepted_counts"
                ].get(
                    "MR->MS",
                    0
                ),
        }


        direction_results.append(
            result
        )


# ==================================================================================================
# 14. RANK DIRECTION RESULTS
# ==================================================================================================

ranked_direction = sorted(

    direction_results,

    key=lambda x: (
        x[
            "six_plastic_mean_AP50"
        ],

        x[
            "MR_MS_mean_AP50"
        ],

        x[
            "overall_AP50_95"
        ],

        -x[
            "accepted"
        ],
    ),

    reverse=True
)


print()
print("=" * 120)
print("TOP 20 INDEPENDENT DIRECTION CONFIGURATIONS")
print("=" * 120)


print(
    f"\n"
    f"{'Rank':>4s} "
    f"{'Direction':>9s} "
    f"{'E23min':>8s} "
    f"{'E20max':>8s} "
    f"{'Plastic':>10s} "
    f"{'MR':>8s} "
    f"{'MS':>8s} "
    f"{'MR/MS':>9s} "
    f"{'AP':>8s} "
    f"{'Accepted':>9s}"
)


print(
    "-" * 100
)


for rank, result in enumerate(
    ranked_direction[:20],
    start=1
):

    print(

        f"{rank:4d} "

        f"{result['direction']:>9s} "

        f"{result['e23_min']:8.3f} "

        f"{result['e20_max']:8.2f} "

        f"{result['six_plastic_mean_AP50'] * 100:9.4f}% "

        f"{result['MR_AP50'] * 100:7.2f}% "

        f"{result['MS_AP50'] * 100:7.2f}% "

        f"{result['MR_MS_mean_AP50'] * 100:8.2f}% "

        f"{result['overall_AP50_95'] * 100:7.3f}% "

        f"{result['accepted']:9,d}"
    )


# ==================================================================================================
# 15. TAKE TOP CANDIDATES FROM EACH DIRECTION
#
# Keep top 10 unique threshold pairs per direction.
# ==================================================================================================

top_mr = [
    x
    for x in ranked_direction
    if x[
        "direction"
    ] == "MS->MR"
][:10]


top_ms = [
    x
    for x in ranked_direction
    if x[
        "direction"
    ] == "MR->MS"
][:10]


# Add OFF as candidate for each direction
mr_candidates = [
    None
] + top_mr


ms_candidates = [
    None
] + top_ms


# ==================================================================================================
# 16. COMBINED SEARCH
# ==================================================================================================

print()
print("=" * 120)
print("STAGE 2 — COMBINED SELECTIVE ROUTING SEARCH")
print("=" * 120)


combined_results = []


for mr_candidate in mr_candidates:

    for ms_candidate in ms_candidates:

        # --------------------------------------------------------------------------------------------------
        # MS -> MR parameters
        # --------------------------------------------------------------------------------------------------

        if mr_candidate is None:

            mr_e23 = None
            mr_e20 = None

        else:

            mr_e23 = float(
                mr_candidate[
                    "e23_min"
                ]
            )

            mr_e20 = float(
                mr_candidate[
                    "e20_max"
                ]
            )


        # --------------------------------------------------------------------------------------------------
        # MR -> MS parameters
        # --------------------------------------------------------------------------------------------------

        if ms_candidate is None:

            ms_e23 = None
            ms_e20 = None

        else:

            ms_e23 = float(
                ms_candidate[
                    "e23_min"
                ]
            )

            ms_e20 = float(
                ms_candidate[
                    "e20_max"
                ]
            )


        built = build_predictions(

            e23_mr_min=mr_e23,
            e20_ms_max=mr_e20,

            e23_ms_min=ms_e23,
            e20_mr_max=ms_e20
        )


        evaluation = evaluate_predictions(
            built[
                "predictions"
            ]
        )


        accepted_total = sum(
            built[
                "accepted_counts"
            ].values()
        )


        result = {

            "MS_to_MR":
                None
                if mr_candidate is None
                else {
                    "e23_min":
                        mr_e23,

                    "e20_max":
                        mr_e20,
                },


            "MR_to_MS":
                None
                if ms_candidate is None
                else {
                    "e23_min":
                        ms_e23,

                    "e20_max":
                        ms_e20,
                },


            "six_plastic_mean_AP50":
                evaluation[
                    "six_plastic_mean_AP50"
                ],

            "MR_MS_mean_AP50":
                evaluation[
                    "MR_MS_mean_AP50"
                ],

            "overall":
                evaluation[
                    "overall"
                ],

            "class_metrics":
                evaluation[
                    "class_metrics"
                ],

            "proposal_counts":
                built[
                    "proposal_counts"
                ],

            "accepted_counts":
                built[
                    "accepted_counts"
                ],

            "accepted_total":
                accepted_total,
        }


        combined_results.append(
            result
        )


# ==================================================================================================
# 17. RANK COMBINED
#
# Fourth tie-break:
# fewer overrides preferred.
# ==================================================================================================

ranked_combined = sorted(

    combined_results,

    key=lambda x: (

        x[
            "six_plastic_mean_AP50"
        ],

        x[
            "MR_MS_mean_AP50"
        ],

        x[
            "overall"
        ][
            "AP50_95"
        ],

        -x[
            "accepted_total"
        ],
    ),

    reverse=True
)


best = ranked_combined[
    0
]


# ==================================================================================================
# 18. TOP 20
# ==================================================================================================

def config_label(
    config
):

    if config is None:

        return "OFF"


    return (
        f"E23>={config['e23_min']:.3f},"
        f"E20<={config['e20_max']:.2f}"
    )


print()
print()
print("=" * 120)
print("TOP 20 E23-D VALIDATION CONFIGURATIONS")
print("=" * 120)


for rank, result in enumerate(
    ranked_combined[:20],
    start=1
):

    print()
    print(
        f"Rank {rank}"
    )

    print(
        f"  MS -> MR : "
        f"{config_label(result['MS_to_MR'])}"
    )

    print(
        f"  MR -> MS : "
        f"{config_label(result['MR_to_MS'])}"
    )

    print(
        f"  Plastic AP50 : "
        f"{result['six_plastic_mean_AP50'] * 100:.4f}%"
    )

    print(
        f"  MR AP50      : "
        f"{result['class_metrics']['mixed_plastic_rigid']['AP50'] * 100:.4f}%"
    )

    print(
        f"  MS AP50      : "
        f"{result['class_metrics']['mixed_plastic_soft']['AP50'] * 100:.4f}%"
    )

    print(
        f"  MR/MS mean   : "
        f"{result['MR_MS_mean_AP50'] * 100:.4f}%"
    )

    print(
        f"  AP50-95      : "
        f"{result['overall']['AP50_95'] * 100:.4f}%"
    )

    print(
        f"  Accepted     : "
        f"{result['accepted_total']:,}"
    )


# ==================================================================================================
# 19. DELTA VS BASELINE
# ==================================================================================================

delta = {

    "six_plastic_mean_AP50_pp":

        (
            best[
                "six_plastic_mean_AP50"
            ]
            -
            baseline_eval[
                "six_plastic_mean_AP50"
            ]
        )
        * 100.0,


    "MR_AP50_pp":

        (
            best[
                "class_metrics"
            ][
                "mixed_plastic_rigid"
            ][
                "AP50"
            ]
            -
            baseline_eval[
                "class_metrics"
            ][
                "mixed_plastic_rigid"
            ][
                "AP50"
            ]
        )
        * 100.0,


    "MS_AP50_pp":

        (
            best[
                "class_metrics"
            ][
                "mixed_plastic_soft"
            ][
                "AP50"
            ]
            -
            baseline_eval[
                "class_metrics"
            ][
                "mixed_plastic_soft"
            ][
                "AP50"
            ]
        )
        * 100.0,


    "MR_MS_mean_AP50_pp":

        (
            best[
                "MR_MS_mean_AP50"
            ]
            -
            baseline_eval[
                "MR_MS_mean_AP50"
            ]
        )
        * 100.0,


    "overall_AP50_95_pp":

        (
            best[
                "overall"
            ][
                "AP50_95"
            ]
            -
            baseline_eval[
                "overall"
            ][
                "AP50_95"
            ]
        )
        * 100.0,
}


# ==================================================================================================
# 20. BEST CONFIGURATION
# ==================================================================================================

print()
print("=" * 120)
print("BEST E23-D CONFIGURATION — VALIDATION")
print("=" * 120)


print()
print(
    f"MS -> MR : "
    f"{config_label(best['MS_to_MR'])}"
)


print(
    f"MR -> MS : "
    f"{config_label(best['MR_to_MS'])}"
)


print()
print(
    f"Six-plastic mean AP50 : "
    f"{best['six_plastic_mean_AP50'] * 100:.4f}%"
)


print(
    f"MR AP50               : "
    f"{best['class_metrics']['mixed_plastic_rigid']['AP50'] * 100:.4f}%"
)


print(
    f"MS AP50               : "
    f"{best['class_metrics']['mixed_plastic_soft']['AP50'] * 100:.4f}%"
)


print(
    f"MR/MS mean AP50       : "
    f"{best['MR_MS_mean_AP50'] * 100:.4f}%"
)


print()
print(
    f"Overall AP50-95       : "
    f"{best['overall']['AP50_95'] * 100:.4f}%"
)


print(
    f"Overall AP50          : "
    f"{best['overall']['AP50'] * 100:.4f}%"
)


print(
    f"Overall AP75          : "
    f"{best['overall']['AP75'] * 100:.4f}%"
)


print(
    f"AR100                 : "
    f"{best['overall']['AR100'] * 100:.4f}%"
)


print()
print(
    "Accepted routes:"
)


for route in [
    "MS->MR",
    "MR->MS",
]:

    print(
        f"{route:12s}: "
        f"{best['accepted_counts'].get(route, 0):,}"
    )


# ==================================================================================================
# 21. DELTA
# ==================================================================================================

print()
print("=" * 120)
print("DELTA VS FROZEN E20-A VALIDATION")
print("=" * 120)


for metric_name, value in delta.items():

    print(
        f"{metric_name:32s}: "
        f"{value:+.4f} pp"
    )


# ==================================================================================================
# 22. DECISION
# ==================================================================================================

improves_primary = (
    best[
        "six_plastic_mean_AP50"
    ]
    >
    baseline_eval[
        "six_plastic_mean_AP50"
    ]
)


improves_mrms = (
    best[
        "MR_MS_mean_AP50"
    ]
    >
    baseline_eval[
        "MR_MS_mean_AP50"
    ]
)


print()
print("=" * 120)
print("E23-D VALIDATION DECISION")
print("=" * 120)


if (
    improves_primary
    and
    improves_mrms
):

    print()
    print(
        "E23-D improves BOTH target metrics."
    )

    print(
        "Eligible for frozen TEST consideration."
    )


elif improves_primary:

    print()
    print(
        "E23-D improves six-plastic AP50,"
    )

    print(
        "but not MR/MS mean AP50."
    )


elif improves_mrms:

    print()
    print(
        "E23-D improves MR/MS mean AP50,"
    )

    print(
        "but not six-plastic AP50."
    )


else:

    print()
    print(
        "E23-D does not improve the target metrics."
    )

    print(
        "Do not promote to TEST."
    )


# ==================================================================================================
# 23. SAVE BEST PREDICTIONS
# ==================================================================================================

best_ms_to_mr = best[
    "MS_to_MR"
]


best_mr_to_ms = best[
    "MR_to_MS"
]


best_build = build_predictions(

    e23_mr_min=(
        None
        if best_ms_to_mr is None
        else best_ms_to_mr[
            "e23_min"
        ]
    ),

    e20_ms_max=(
        None
        if best_ms_to_mr is None
        else best_ms_to_mr[
            "e20_max"
        ]
    ),

    e23_ms_min=(
        None
        if best_mr_to_ms is None
        else best_mr_to_ms[
            "e23_min"
        ]
    ),

    e20_mr_max=(
        None
        if best_mr_to_ms is None
        else best_mr_to_ms[
            "e20_max"
        ]
    )
)


with open(
    BEST_PREDICTIONS_PATH,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        best_build[
            "predictions"
        ],
        f
    )


# ==================================================================================================
# 24. SAVE RESULTS
# ==================================================================================================

with open(
    GRID_RESULTS_PATH,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        {
            "direction_results":
                ranked_direction,

            "combined_results":
                ranked_combined,
        },
        f,
        indent=2
    )


output_config = {

    "experiment":
        "E23-D",

    "dataset":
        "validation",

    "test_used":
        False,

    "base_pipeline":
        "E20-A",

    "specialist":
        "E23-A",

    "routing_rule":
        (
            "E23-A override requires high E23 confidence "
            "AND low E20-A current-class confidence"
        ),

    "best_configuration":
        best,

    "baseline":
        baseline_eval,

    "delta_vs_E20A":
        delta,

    "promote_to_test":
        bool(
            improves_primary
            and
            improves_mrms
        ),
}


with open(
    BEST_CONFIG_PATH,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        output_config,
        f,
        indent=2
    )


# ==================================================================================================
# 25. SUMMARY
# ==================================================================================================

summary_lines = [

    "=" * 120,

    "E23-D — CONFIDENCE-CONTRASTIVE MR/MS ROUTING",

    "=" * 120,

    "",

    "VALIDATION ONLY",

    "TEST NOT USED",

    "",

    f"MS -> MR : "
    f"{config_label(best['MS_to_MR'])}",

    f"MR -> MS : "
    f"{config_label(best['MR_to_MS'])}",

    "",

    f"Six-plastic mean AP50 : "
    f"{best['six_plastic_mean_AP50'] * 100:.4f}%",

    f"MR AP50               : "
    f"{best['class_metrics']['mixed_plastic_rigid']['AP50'] * 100:.4f}%",

    f"MS AP50               : "
    f"{best['class_metrics']['mixed_plastic_soft']['AP50'] * 100:.4f}%",

    f"MR/MS mean AP50       : "
    f"{best['MR_MS_mean_AP50'] * 100:.4f}%",

    f"Overall AP50-95       : "
    f"{best['overall']['AP50_95'] * 100:.4f}%",

    "",

    "Accepted routes:",

    f"  MS -> MR : "
    f"{best['accepted_counts'].get('MS->MR', 0):,}",

    f"  MR -> MS : "
    f"{best['accepted_counts'].get('MR->MS', 0):,}",

    "",

    "Delta vs E20-A:",
]


for metric_name, value in delta.items():

    summary_lines.append(
        f"  {metric_name:32s}: "
        f"{value:+.4f} pp"
    )


with open(
    SUMMARY_PATH,
    "w",
    encoding="utf-8"
) as f:

    f.write(
        "\n".join(
            summary_lines
        )
    )


# ==================================================================================================
# 26. COMPLETE
# ==================================================================================================

print()
print("=" * 120)
print("E23-D COMPLETE")
print("=" * 120)


print()
print(
    f"Grid results     : "
    f"{GRID_RESULTS_PATH}"
)


print(
    f"Best config      : "
    f"{BEST_CONFIG_PATH}"
)


print(
    f"Best predictions : "
    f"{BEST_PREDICTIONS_PATH}"
)


print(
    f"Summary          : "
    f"{SUMMARY_PATH}"
)

E23-D — CONFIDENCE-CONTRASTIVE MR/MS ROUTING

Cache      : C:\Users\varda\Documents\_My Computer\COMMON_space\Learning\Upgrad_EPGP_MS\upgrad_IIITB_MS_course\MS_LJMU_Material\Thesis_Code\runs\sortwaste\E23B_E20A_binary_MRMS_routing\E23B_cached_predictions_with_E23A.json
Val GT     : C:\Users\varda\Documents\_My Computer\COMMON_space\Learning\Upgrad_EPGP_MS\upgrad_IIITB_MS_course\MS_LJMU_Material\Thesis_Code\runs\sortwaste\E18D_class_selective_convnext\E18D_val_gt_7class.json
Output     : C:\Users\varda\Documents\_My Computer\COMMON_space\Learning\Upgrad_EPGP_MS\upgrad_IIITB_MS_course\MS_LJMU_Material\Thesis_Code\runs\sortwaste\E23D_confidence_contrastive_MRMS

Cached detections : 55,605
E23-A outputs     : 14,690
loading annotations into memory...
Done (t=0.05s)
creating index...
index created!

E20-A CURRENT-CLASS CONFIDENCE SUMMARY

MR
Count   : 5,171
Mean    : 0.5325
Median  : 0.5733
P25     : 0.0672
P75     : 0.9904

MS
Count   : 9,519
Mean    : 0.5867
Median  : 0.8677
P25     : 0.0

In [38]:
### E23-D — Frozen Held-Out Test Evaluation of Confidence-Contrastive MR/MS Routing
# ==================================================================================================
# E23-D — FINAL CORRECTED FROZEN HELD-OUT TEST EVALUATION
#
# Uses:
#
#   VERIFIED E20-A TEST predictions:
#       FINAL_TEST_E20A\E20A_test_predictions.json
#
#   VERIFIED E12 TEST detector predictions:
#       FINAL_TEST_BATCH5\E12\predictions.json
#
# Why E12 is needed:
#
#   E20-A score was:
#
#       final_score =
#           yolo_conf ** 0.70
#           *
#           classifier_prob ** 0.30
#
# Therefore:
#
#       classifier_prob =
#           (final_score / yolo_conf**0.70) ** (1/0.30)
#
# This recovers the EXACT classifier confidence used by E20-A,
# regardless of whether that confidence came from MobileNet or E18-B ConvNeXt.
#
# We therefore DO NOT rerun:
#
#   - YOLO
#   - MobileNet
#   - E18-B ConvNeXt
#
# We run ONLY:
#
#   - E23-A binary MR/MS ConvNeXt-Tiny
#
# and ONLY on final E20-A MR/MS detections.
#
#
# FROZEN E23-D VALIDATION RULE
# --------------------------------------------------------------------------------------------------
#
# MS -> MR:
#     E23 P(MR) >= 0.920
#     AND
#     E20 current-class confidence <= 0.90
#
# MR -> MS:
#     E23 P(MS) >= 0.997
#     AND
#     E20 current-class confidence <= 0.95
#
# IMPORTANT:
#
#   - thresholds were selected on VALIDATION
#   - absolutely NO threshold search on TEST
#   - E23-D changes CLASS ONLY
#   - E20-A bbox and detection score are preserved exactly
#
# ==================================================================================================

import json
import time
from pathlib import Path
from collections import defaultdict, Counter

import numpy as np

from PIL import Image

import torch
import torch.nn as nn

from torchvision import transforms
from torchvision.models import convnext_tiny

from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval


# ==================================================================================================
# 1. DEVICE
# ==================================================================================================

DEVICE = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

print("=" * 120)
print("E23-D — FINAL CORRECTED FROZEN HELD-OUT TEST")
print("=" * 120)

print()
print(f"PyTorch : {torch.__version__}")
print(f"Device  : {DEVICE}")

if torch.cuda.is_available():
    print(
        f"GPU     : "
        f"{torch.cuda.get_device_name(0)}"
    )


# ==================================================================================================
# 2. PATHS
# ==================================================================================================

BASE = Path(
    r"C:\Users\varda\Documents\_My Computer\COMMON_space\Learning"
    r"\Upgrad_EPGP_MS\upgrad_IIITB_MS_course\MS_LJMU_Material"
)

DATASET_ROOT = (
    BASE
    / "Topic Data"
    / "SortWaste"
    / "dataset"
    / "dataset"
)

THESIS_CODE = (
    BASE
    / "Thesis_Code"
)

RUNS_ROOT = (
    THESIS_CODE
    / "runs"
    / "sortwaste"
)


# --------------------------------------------------------------------------------------------------
# TEST images
# --------------------------------------------------------------------------------------------------

TEST_IMAGES = (
    DATASET_ROOT
    / "splited_all_dataset_coco"
    / "test"
    / "images"
)


# --------------------------------------------------------------------------------------------------
# VERIFIED E20-A TEST outputs
# --------------------------------------------------------------------------------------------------

E20_DIR = (
    RUNS_ROOT
    / "FINAL_TEST_E20A"
)

E20_PRED_PATH = (
    E20_DIR
    / "E20A_test_predictions.json"
)

E20_GT_PATH = (
    E20_DIR
    / "E20A_test_gt_7class.json"
)


# --------------------------------------------------------------------------------------------------
# VERIFIED standalone E12 detector predictions
# --------------------------------------------------------------------------------------------------

E12_PRED_PATH = (
    RUNS_ROOT
    / "FINAL_TEST_BATCH5"
    / "E12"
    / "predictions.json"
)


# --------------------------------------------------------------------------------------------------
# E23-A binary specialist
# --------------------------------------------------------------------------------------------------

E23_CKPT = (
    RUNS_ROOT
    / "E23A_binary_MR_MS_hardpair_convnext"
    / "E23A_ConvNeXtTiny_MR_MS_best.pth"
)


# --------------------------------------------------------------------------------------------------
# Frozen E23-D validation configuration
# --------------------------------------------------------------------------------------------------

E23D_CONFIG_PATH = (
    RUNS_ROOT
    / "E23D_confidence_contrastive_MRMS"
    / "E23D_best_configuration.json"
)


# --------------------------------------------------------------------------------------------------
# Output
# --------------------------------------------------------------------------------------------------

OUTPUT_DIR = (
    RUNS_ROOT
    / "FINAL_TEST_E23D_CORRECTED"
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)


MATCH_CACHE_PATH = (
    OUTPUT_DIR
    / "E23D_E20A_E12_matched_cache.json"
)

E23_CACHE_PATH = (
    OUTPUT_DIR
    / "E23D_E23A_MRMS_test_cache.json"
)

FINAL_PRED_PATH = (
    OUTPUT_DIR
    / "E23D_test_predictions.json"
)

RESULTS_PATH = (
    OUTPUT_DIR
    / "E23D_test_results.json"
)

SUMMARY_PATH = (
    OUTPUT_DIR
    / "E23D_test_summary.txt"
)


for p in [
    TEST_IMAGES,
    E20_PRED_PATH,
    E20_GT_PATH,
    E12_PRED_PATH,
    E23_CKPT,
    E23D_CONFIG_PATH,
]:

    if not p.exists():

        raise FileNotFoundError(
            f"Required file/path not found:\n{p}"
        )


# ==================================================================================================
# 3. TAXONOMY
# ==================================================================================================

CLASS_NAMES = [
    "ecal",
    "hdpe",
    "mixed_plastic_rigid",
    "mixed_plastic_soft",
    "non_plastic",
    "pet",
    "pet_oil",
]

ECAL = 0
HDPE = 1
MR = 2
MS = 3
NP = 4
PET = 5
PET_OIL = 6


PLASTIC_INDICES = [
    ECAL,
    HDPE,
    MR,
    MS,
    PET,
    PET_OIL,
]


# ==================================================================================================
# 4. LOAD FROZEN E23-D CONFIGURATION
# ==================================================================================================

with open(
    E23D_CONFIG_PATH,
    "r",
    encoding="utf-8"
) as f:

    config = json.load(f)


if config.get(
    "test_used"
) is not False:

    raise RuntimeError(
        "E23-D configuration was not marked validation-only."
    )


best_config = (
    config[
        "best_configuration"
    ]
)


MS_TO_MR_E23_MIN = float(
    best_config[
        "MS_to_MR"
    ][
        "e23_min"
    ]
)

MS_TO_MR_E20_MAX = float(
    best_config[
        "MS_to_MR"
    ][
        "e20_max"
    ]
)

MR_TO_MS_E23_MIN = float(
    best_config[
        "MR_to_MS"
    ][
        "e23_min"
    ]
)

MR_TO_MS_E20_MAX = float(
    best_config[
        "MR_to_MS"
    ][
        "e20_max"
    ]
)


print()
print("=" * 120)
print("FROZEN E23-D THRESHOLDS")
print("=" * 120)

print()
print(
    f"MS -> MR : "
    f"E23 >= {MS_TO_MR_E23_MIN:.3f}, "
    f"E20 <= {MS_TO_MR_E20_MAX:.2f}"
)

print(
    f"MR -> MS : "
    f"E23 >= {MR_TO_MS_E23_MIN:.3f}, "
    f"E20 <= {MR_TO_MS_E20_MAX:.2f}"
)


# Hard safety check

if not (
    np.isclose(
        MS_TO_MR_E23_MIN,
        0.920
    )
    and
    np.isclose(
        MS_TO_MR_E20_MAX,
        0.90
    )
    and
    np.isclose(
        MR_TO_MS_E23_MIN,
        0.997
    )
    and
    np.isclose(
        MR_TO_MS_E20_MAX,
        0.95
    )
):

    raise RuntimeError(
        "Frozen thresholds do not match validated E23-D configuration."
    )


# ==================================================================================================
# 5. LOAD E20-A + E12 PREDICTIONS
# ==================================================================================================

with open(
    E20_PRED_PATH,
    "r",
    encoding="utf-8"
) as f:

    e20_predictions = json.load(f)


with open(
    E12_PRED_PATH,
    "r",
    encoding="utf-8"
) as f:

    e12_predictions = json.load(f)


print()
print("=" * 120)
print("LOADED FROZEN TEST PREDICTIONS")
print("=" * 120)

print()
print(
    f"E20-A predictions : "
    f"{len(e20_predictions):,}"
)

print(
    f"E12 predictions   : "
    f"{len(e12_predictions):,}"
)


if len(
    e20_predictions
) != 57487:

    raise RuntimeError(
        "E20-A prediction count is not the verified 57,487."
    )


if len(
    e12_predictions
) != 57487:

    raise RuntimeError(
        "E12 prediction count differs from 57,487. "
        "Do not continue."
    )


# ==================================================================================================
# 6. COCO EVALUATION HELPERS
# ==================================================================================================

coco_gt = COCO(
    str(
        E20_GT_PATH
    )
)


def valid_mean(
    values
):

    values = np.asarray(
        values
    )

    valid = values[
        values > -1
    ]

    if valid.size == 0:

        return float(
            "nan"
        )

    return float(
        valid.mean()
    )


def evaluate_predictions(
    predictions
):

    coco_dt = coco_gt.loadRes(
        predictions
    )

    evaluator = COCOeval(
        coco_gt,
        coco_dt,
        "bbox"
    )

    evaluator.params.maxDets = [
        1,
        10,
        100,
    ]

    evaluator.evaluate()
    evaluator.accumulate()

    precision = (
        evaluator.eval[
            "precision"
        ]
    )

    recall = (
        evaluator.eval[
            "recall"
        ]
    )

    ious = (
        evaluator.params.iouThrs
    )


    idx50 = int(
        np.where(
            np.isclose(
                ious,
                0.50
            )
        )[0][0]
    )

    idx75 = int(
        np.where(
            np.isclose(
                ious,
                0.75
            )
        )[0][0]
    )


    overall_ap = valid_mean(
        precision[
            :,
            :,
            :,
            0,
            -1
        ]
    )

    overall_ap50 = valid_mean(
        precision[
            idx50,
            :,
            :,
            0,
            -1
        ]
    )

    overall_ap75 = valid_mean(
        precision[
            idx75,
            :,
            :,
            0,
            -1
        ]
    )

    ar100 = valid_mean(
        recall[
            :,
            :,
            0,
            -1
        ]
    )


    class_metrics = {}


    for idx, name in enumerate(
        CLASS_NAMES
    ):

        class_ap50 = valid_mean(
            precision[
                idx50,
                :,
                idx,
                0,
                -1
            ]
        )

        class_ap = valid_mean(
            precision[
                :,
                :,
                idx,
                0,
                -1
            ]
        )


        class_metrics[
            name
        ] = {
            "AP50":
                class_ap50,

            "AP50_95":
                class_ap,
        }


    six_plastic_ap50 = float(
        np.mean(
            [
                class_metrics[
                    CLASS_NAMES[i]
                ][
                    "AP50"
                ]

                for i
                in PLASTIC_INDICES
            ]
        )
    )


    six_plastic_ap = float(
        np.mean(
            [
                class_metrics[
                    CLASS_NAMES[i]
                ][
                    "AP50_95"
                ]

                for i
                in PLASTIC_INDICES
            ]
        )
    )


    mr_ms_mean = float(
        np.mean(
            [
                class_metrics[
                    "mixed_plastic_rigid"
                ][
                    "AP50"
                ],

                class_metrics[
                    "mixed_plastic_soft"
                ][
                    "AP50"
                ],
            ]
        )
    )


    return {
        "AP50_95":
            overall_ap,

        "AP50":
            overall_ap50,

        "AP75":
            overall_ap75,

        "AR100":
            ar100,

        "six_plastic_AP50":
            six_plastic_ap50,

        "six_plastic_AP50_95":
            six_plastic_ap,

        "MR_MS_mean_AP50":
            mr_ms_mean,

        "class_metrics":
            class_metrics,
    }


# ==================================================================================================
# 7. VERIFY SAVED E20-A BASELINE FIRST
#
# This is instant compared with inference.
# ==================================================================================================

print()
print("=" * 120)
print("VERIFYING SAVED E20-A TEST BASELINE")
print("=" * 120)


e20_metrics = evaluate_predictions(
    e20_predictions
)


print()
print(
    f"AP50-95        : "
    f"{e20_metrics['AP50_95'] * 100:.4f}%"
)

print(
    f"AP50           : "
    f"{e20_metrics['AP50'] * 100:.4f}%"
)

print(
    f"Plastic AP50   : "
    f"{e20_metrics['six_plastic_AP50'] * 100:.4f}%"
)

print(
    f"MR AP50        : "
    f"{e20_metrics['class_metrics']['mixed_plastic_rigid']['AP50'] * 100:.4f}%"
)

print(
    f"MS AP50        : "
    f"{e20_metrics['class_metrics']['mixed_plastic_soft']['AP50'] * 100:.4f}%"
)


# Verified E20-A references

REF_E20 = {
    "AP50_95": 0.462612,
    "AP50": 0.623271,
    "six_plastic": 0.694353,
}


if (
    abs(
        e20_metrics[
            "AP50_95"
        ]
        -
        REF_E20[
            "AP50_95"
        ]
    )
    > 0.001
    or
    abs(
        e20_metrics[
            "AP50"
        ]
        -
        REF_E20[
            "AP50"
        ]
    )
    > 0.001
    or
    abs(
        e20_metrics[
            "six_plastic_AP50"
        ]
        -
        REF_E20[
            "six_plastic"
        ]
    )
    > 0.001
):

    raise RuntimeError(
        "\nSaved E20-A predictions do not reproduce "
        "the verified E20-A TEST baseline.\n"
        "E23-A inference has NOT started."
    )


print()
print(
    "E20-A baseline verification PASSED."
)


# ==================================================================================================
# 8. MATCH E20-A DETECTIONS TO E12 DETECTIONS
#
# Matching key:
#   image_id + bbox
#
# Both files contain exactly the same detector geometry,
# but their list ordering can differ.
# ==================================================================================================

print()
print("=" * 120)
print("MATCHING E20-A TO E12 DETECTIONS")
print("=" * 120)


# Index E12 by image_id

e12_by_image = defaultdict(
    list
)


for idx, pred in enumerate(
    e12_predictions
):

    e12_by_image[
        int(
            pred[
                "image_id"
            ]
        )
    ].append(
        (
            idx,
            pred
        )
    )


used_e12 = set()

matched_records = []

unmatched = []


for e20_idx, p20 in enumerate(
    e20_predictions
):

    image_id = int(
        p20[
            "image_id"
        ]
    )

    bbox20 = np.asarray(
        p20[
            "bbox"
        ],
        dtype=np.float64
    )


    best_candidate = None
    best_error = float(
        "inf"
    )


    for e12_idx, p12 in e12_by_image[
        image_id
    ]:

        if e12_idx in used_e12:
            continue


        bbox12 = np.asarray(
            p12[
                "bbox"
            ],
            dtype=np.float64
        )


        error = float(
            np.max(
                np.abs(
                    bbox20
                    -
                    bbox12
                )
            )
        )


        if error < best_error:

            best_error = error

            best_candidate = (
                e12_idx,
                p12
            )


    # same detector boxes should be extremely close

    if (
        best_candidate is None
        or
        best_error > 1e-3
    ):

        unmatched.append(
            {
                "e20_idx":
                    e20_idx,

                "image_id":
                    image_id,

                "best_error":
                    best_error,
            }
        )

        continue


    e12_idx, p12 = (
        best_candidate
    )

    used_e12.add(
        e12_idx
    )


    matched_records.append(
        {
            "e20_index":
                e20_idx,

            "image_id":
                image_id,

            "category_id":
                int(
                    p20[
                        "category_id"
                    ]
                ),

            "bbox":
                [
                    float(v)
                    for v
                    in p20[
                        "bbox"
                    ]
                ],

            "e20_score":
                float(
                    p20[
                        "score"
                    ]
                ),

            "yolo_conf":
                float(
                    p12[
                        "score"
                    ]
                ),

            "bbox_error":
                best_error,
        }
    )


print()
print(
    f"Matched   : "
    f"{len(matched_records):,}"
)

print(
    f"Unmatched : "
    f"{len(unmatched):,}"
)

print(
    f"Unique E12 used : "
    f"{len(used_e12):,}"
)


if len(
    matched_records
) != len(
    e20_predictions
):

    raise RuntimeError(
        "\nNot every E20-A detection matched an E12 detector box.\n"
        "E23-A inference has NOT started."
    )


# ==================================================================================================
# 9. RECOVER EXACT E20-A CLASSIFIER CONFIDENCE
#
# E20 score =
#       yolo_conf^0.70
#       *
#       classifier_prob^0.30
#
# classifier_prob =
#       (E20_score / yolo_conf^0.70)^(1/0.30)
#
# ==================================================================================================

ALPHA = 0.70


invalid_recovered_conf = []


for record in matched_records:

    e20_score = max(
        float(
            record[
                "e20_score"
            ]
        ),
        1e-12
    )

    yolo_conf = max(
        float(
            record[
                "yolo_conf"
            ]
        ),
        1e-12
    )


    recovered_conf = (
        e20_score
        /
        (
            yolo_conf
            ** ALPHA
        )
    ) ** (
        1.0
        /
        (
            1.0
            -
            ALPHA
        )
    )


    record[
        "e20_class_conf"
    ] = float(
        recovered_conf
    )


    # tiny floating point overshoot is okay

    if (
        recovered_conf < -1e-5
        or
        recovered_conf > 1.005
    ):

        invalid_recovered_conf.append(
            record
        )


if invalid_recovered_conf:

    print()
    print(
        "Examples of invalid recovered confidence:"
    )

    for x in invalid_recovered_conf[
        :5
    ]:

        print(
            x
        )


    raise RuntimeError(
        "\nRecovered E20 classifier confidence is invalid.\n"
        "This means E12 score is not the corresponding raw YOLO confidence.\n"
        "E23-A inference has NOT started."
    )


# Clamp only floating-point edge noise.

for record in matched_records:

    record[
        "e20_class_conf"
    ] = float(
        np.clip(
            record[
                "e20_class_conf"
            ],
            0.0,
            1.0
        )
    )


conf_array = np.asarray(
    [
        r[
            "e20_class_conf"
        ]
        for r
        in matched_records
    ]
)


print()
print("=" * 120)
print("RECOVERED E20-A CLASSIFIER CONFIDENCE")
print("=" * 120)

print()
print(
    f"Min    : {conf_array.min():.6f}"
)

print(
    f"Mean   : {conf_array.mean():.6f}"
)

print(
    f"Median : {np.median(conf_array):.6f}"
)

print(
    f"Max    : {conf_array.max():.6f}"
)


# Save matching/recovered information immediately.

with open(
    MATCH_CACHE_PATH,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        matched_records,
        f
    )


# ==================================================================================================
# 10. SELECT ONLY FINAL E20-A MR/MS DETECTIONS
#
# COCO category IDs:
#
# MR = class index 2 -> category_id 3
# MS = class index 3 -> category_id 4
# ==================================================================================================

MR_CATEGORY_ID = 3
MS_CATEGORY_ID = 4


candidate_records = [
    r
    for r in matched_records
    if r[
        "category_id"
    ] in {
        MR_CATEGORY_ID,
        MS_CATEGORY_ID,
    }
]


candidate_indices = [
    r[
        "e20_index"
    ]
    for r
    in candidate_records
]


mr_count = sum(
    1
    for r
    in candidate_records
    if r[
        "category_id"
    ]
    ==
    MR_CATEGORY_ID
)

ms_count = sum(
    1
    for r
    in candidate_records
    if r[
        "category_id"
    ]
    ==
    MS_CATEGORY_ID
)


print()
print("=" * 120)
print("E23-A TEST CANDIDATES")
print("=" * 120)

print()
print(
    f"MR candidates : {mr_count:,}"
)

print(
    f"MS candidates : {ms_count:,}"
)

print(
    f"Total         : {len(candidate_records):,}"
)

print(
    f"Fraction      : "
    f"{100 * len(candidate_records) / len(e20_predictions):.2f}%"
)


# ==================================================================================================
# 11. LOAD TEST IMAGE LOOKUP
# ==================================================================================================

with open(
    E20_GT_PATH,
    "r",
    encoding="utf-8"
) as f:

    gt_json = json.load(f)


image_lookup = {
    int(
        x[
            "id"
        ]
    ):
        x[
            "file_name"
        ]

    for x
    in gt_json[
        "images"
    ]
}


# ==================================================================================================
# 12. E23-A TRANSFORM
# ==================================================================================================

IMAGE_SIZE = 224


classifier_transform = transforms.Compose(
    [
        transforms.Resize(
            (
                IMAGE_SIZE,
                IMAGE_SIZE,
            )
        ),

        transforms.ToTensor(),

        transforms.Normalize(
            mean=[
                0.485,
                0.456,
                0.406,
            ],

            std=[
                0.229,
                0.224,
                0.225,
            ],
        ),
    ]
)


# ==================================================================================================
# 13. LOAD E23-A
# ==================================================================================================

print()
print("=" * 120)
print("LOADING E23-A BINARY CONVNEXT-TINY")
print("=" * 120)


e23_model = convnext_tiny(
    weights=None
)

in_features = (
    e23_model
    .classifier[2]
    .in_features
)

e23_model.classifier[2] = nn.Linear(
    in_features,
    2
)


checkpoint = torch.load(
    E23_CKPT,
    map_location=DEVICE,
    weights_only=False
)


if (
    isinstance(
        checkpoint,
        dict
    )
    and
    "model_state_dict"
    in checkpoint
):

    state_dict = (
        checkpoint[
            "model_state_dict"
        ]
    )

else:

    state_dict = (
        checkpoint
    )


e23_model.load_state_dict(
    state_dict
)

e23_model = e23_model.to(
    DEVICE
)

e23_model.eval()


print(
    "E23-A loaded."
)


# ==================================================================================================
# 14. RUN OR REUSE E23-A TEST CACHE
# ==================================================================================================

if E23_CACHE_PATH.exists():

    print()
    print(
        "Existing E23-A TEST cache found."
    )

    print(
        "Reusing cached specialist outputs."
    )


    with open(
        E23_CACHE_PATH,
        "r",
        encoding="utf-8"
    ) as f:

        e23_cache = json.load(f)


else:

    print()
    print("=" * 120)
    print("RUNNING E23-A ON FROZEN E20-A MR/MS DETECTIONS")
    print("=" * 120)


    BATCH_SIZE = 32

    e23_cache = {}

    start_time = time.time()


    @torch.inference_mode()
    def run_batch(
        tensors
    ):

        inputs = torch.stack(
            tensors
        ).to(
            DEVICE
        )


        if DEVICE.type == "cuda":

            with torch.amp.autocast(
                device_type="cuda",
                dtype=torch.float16,
            ):

                logits = e23_model(
                    inputs
                )

        else:

            logits = e23_model(
                inputs
            )


        probs = torch.softmax(
            logits.float(),
            dim=1
        )


        return (
            probs
            .detach()
            .cpu()
            .numpy()
        )


    # Group detections by image so each test image is opened once.

    candidates_by_image = defaultdict(
        list
    )


    for r in candidate_records:

        candidates_by_image[
            int(
                r[
                    "image_id"
                ]
            )
        ].append(
            r
        )


    batch_tensors = []

    batch_records = []

    processed = 0


    def flush_batch():

        nonlocal_batch = None


    # explicit helper without nonlocal complications
    def process_current_batch(
        batch_tensors,
        batch_records,
    ):

        if len(
            batch_tensors
        ) == 0:

            return


        probs_batch = run_batch(
            batch_tensors
        )


        for r, probs in zip(
            batch_records,
            probs_batch
        ):

            mr_prob = float(
                probs[0]
            )

            ms_prob = float(
                probs[1]
            )


            if mr_prob >= ms_prob:

                global_idx = MR

                top_prob = mr_prob

            else:

                global_idx = MS

                top_prob = ms_prob


            e23_cache[
                str(
                    r[
                        "e20_index"
                    ]
                )
            ] = {
                "pred_global_idx":
                    int(
                        global_idx
                    ),

                "top_prob":
                    float(
                        top_prob
                    ),

                "mr_prob":
                    mr_prob,

                "ms_prob":
                    ms_prob,
            }


    image_ids_sorted = sorted(
        candidates_by_image.keys()
    )


    for image_number, image_id in enumerate(
        image_ids_sorted,
        start=1
    ):

        image_path = (
            TEST_IMAGES
            / image_lookup[
                image_id
            ]
        )


        with Image.open(
            image_path
        ) as pil_image:

            pil_image = pil_image.convert(
                "RGB"
            )

            image_width, image_height = (
                pil_image.size
            )


            for r in candidates_by_image[
                image_id
            ]:

                x, y, w, h = [
                    float(v)
                    for v
                    in r[
                        "bbox"
                    ]
                ]


                x1 = max(
                    0,
                    int(
                        np.floor(
                            x
                        )
                    )
                )

                y1 = max(
                    0,
                    int(
                        np.floor(
                            y
                        )
                    )
                )

                x2 = min(
                    image_width,
                    int(
                        np.ceil(
                            x + w
                        )
                    )
                )

                y2 = min(
                    image_height,
                    int(
                        np.ceil(
                            y + h
                        )
                    )
                )


                if (
                    x2 <= x1
                    or
                    y2 <= y1
                ):

                    raise RuntimeError(
                        f"Invalid crop for E20 index "
                        f"{r['e20_index']}"
                    )


                crop = pil_image.crop(
                    (
                        x1,
                        y1,
                        x2,
                        y2,
                    )
                )


                batch_tensors.append(
                    classifier_transform(
                        crop
                    )
                )

                batch_records.append(
                    r
                )


                if len(
                    batch_tensors
                ) >= BATCH_SIZE:

                    process_current_batch(
                        batch_tensors,
                        batch_records,
                    )

                    processed += len(
                        batch_tensors
                    )

                    batch_tensors = []
                    batch_records = []


        if (
            image_number % 50 == 0
            or
            image_number
            ==
            len(
                image_ids_sorted
            )
        ):

            elapsed = (
                time.time()
                -
                start_time
            ) / 60.0


            print(
                f"[{image_number:4d}/"
                f"{len(image_ids_sorted):4d}] "
                f"E23 outputs="
                f"{len(e23_cache):,}/"
                f"{len(candidate_records):,} "
                f"time={elapsed:.2f} min"
            )


    # remaining partial batch

    if len(
        batch_tensors
    ) > 0:

        process_current_batch(
            batch_tensors,
            batch_records,
        )


    if len(
        e23_cache
    ) != len(
        candidate_records
    ):

        raise RuntimeError(
            "E23-A output count does not match MR/MS candidates."
        )


    with open(
        E23_CACHE_PATH,
        "w",
        encoding="utf-8"
    ) as f:

        json.dump(
            e23_cache,
            f
        )


    elapsed = (
        time.time()
        -
        start_time
    ) / 60.0


    print()
    print(
        f"E23-A inference complete: "
        f"{elapsed:.2f} min"
    )

    print(
        f"Cache saved: "
        f"{E23_CACHE_PATH}"
    )


# ==================================================================================================
# 15. APPLY FROZEN E23-D
# ==================================================================================================

print()
print("=" * 120)
print("APPLYING FROZEN E23-D ROUTING")
print("=" * 120)


final_predictions = [
    dict(p)
    for p
    in e20_predictions
]


proposals = Counter()

accepted = Counter()


record_lookup = {
    int(
        r[
            "e20_index"
        ]
    ):
        r

    for r
    in matched_records
}


for e20_idx in candidate_indices:

    original = e20_predictions[
        e20_idx
    ]


    current_idx = (
        int(
            original[
                "category_id"
            ]
        )
        - 1
    )


    e20_conf = float(
        record_lookup[
            e20_idx
        ][
            "e20_class_conf"
        ]
    )


    e23 = e23_cache[
        str(
            e20_idx
        )
    ]


    e23_idx = int(
        e23[
            "pred_global_idx"
        ]
    )

    e23_prob = float(
        e23[
            "top_prob"
        ]
    )


    # ----------------------------------------------------------------------------------------------
    # MS -> MR
    # ----------------------------------------------------------------------------------------------

    if (
        current_idx == MS
        and
        e23_idx == MR
    ):

        proposals[
            "MS->MR"
        ] += 1


        if (
            e23_prob
            >=
            MS_TO_MR_E23_MIN
            and
            e20_conf
            <=
            MS_TO_MR_E20_MAX
        ):

            final_predictions[
                e20_idx
            ][
                "category_id"
            ] = (
                MR + 1
            )


            accepted[
                "MS->MR"
            ] += 1


    # ----------------------------------------------------------------------------------------------
    # MR -> MS
    # ----------------------------------------------------------------------------------------------

    elif (
        current_idx == MR
        and
        e23_idx == MS
    ):

        proposals[
            "MR->MS"
        ] += 1


        if (
            e23_prob
            >=
            MR_TO_MS_E23_MIN
            and
            e20_conf
            <=
            MR_TO_MS_E20_MAX
        ):

            final_predictions[
                e20_idx
            ][
                "category_id"
            ] = (
                MS + 1
            )


            accepted[
                "MR->MS"
            ] += 1


# IMPORTANT:
# bbox and score are untouched.


print()
print(
    "E23-A disagreement proposals:"
)

print(
    f"MS -> MR : "
    f"{proposals['MS->MR']:,}"
)

print(
    f"MR -> MS : "
    f"{proposals['MR->MS']:,}"
)


print()
print(
    "Accepted E23-D overrides:"
)

print(
    f"MS -> MR : "
    f"{accepted['MS->MR']:,}"
)

print(
    f"MR -> MS : "
    f"{accepted['MR->MS']:,}"
)


# ==================================================================================================
# 16. STRUCTURAL SAFETY CHECK
#
# E23-D MUST change only MR/MS category IDs.
#
# bbox and score MUST remain bit-for-bit equivalent.
# ==================================================================================================

changed_classes = 0


for before, after in zip(
    e20_predictions,
    final_predictions,
):

    if before[
        "image_id"
    ] != after[
        "image_id"
    ]:

        raise RuntimeError(
            "Image ID changed unexpectedly."
        )


    if before[
        "bbox"
    ] != after[
        "bbox"
    ]:

        raise RuntimeError(
            "Bounding box changed unexpectedly."
        )


    if before[
        "score"
    ] != after[
        "score"
    ]:

        raise RuntimeError(
            "Detection score changed unexpectedly."
        )


    if (
        before[
            "category_id"
        ]
        !=
        after[
            "category_id"
        ]
    ):

        changed_classes += 1


print()
print(
    f"Total class changes: "
    f"{changed_classes:,}"
)


if changed_classes != (
    accepted[
        "MS->MR"
    ]
    +
    accepted[
        "MR->MS"
    ]
):

    raise RuntimeError(
        "Changed class count does not equal accepted E23-D routes."
    )


# ==================================================================================================
# 17. SAVE FINAL PREDICTIONS
# ==================================================================================================

with open(
    FINAL_PRED_PATH,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        final_predictions,
        f
    )


# ==================================================================================================
# 18. FINAL COCO TEST EVALUATION
# ==================================================================================================

print()
print("=" * 120)
print("E23-D — FROZEN HELD-OUT TEST COCO EVALUATION")
print("=" * 120)


e23d_metrics = evaluate_predictions(
    final_predictions
)


print()
print("=" * 120)
print("E23-D — HELD-OUT TEST RESULTS")
print("=" * 120)

print()
print(
    f"AP50-95                : "
    f"{e23d_metrics['AP50_95'] * 100:.4f}%"
)

print(
    f"AP50                   : "
    f"{e23d_metrics['AP50'] * 100:.4f}%"
)

print(
    f"AP75                   : "
    f"{e23d_metrics['AP75'] * 100:.4f}%"
)

print(
    f"AR100                  : "
    f"{e23d_metrics['AR100'] * 100:.4f}%"
)

print()
print(
    f"Six-plastic mean AP50 : "
    f"{e23d_metrics['six_plastic_AP50'] * 100:.4f}%"
)

print(
    f"Six-plastic AP50-95   : "
    f"{e23d_metrics['six_plastic_AP50_95'] * 100:.4f}%"
)

print(
    f"MR/MS mean AP50       : "
    f"{e23d_metrics['MR_MS_mean_AP50'] * 100:.4f}%"
)


# ==================================================================================================
# 19. CLASS-WISE RESULTS
# ==================================================================================================

print()
print("=" * 120)
print("CLASS-WISE TEST RESULTS")
print("=" * 120)

print(
    f"\n"
    f"{'Class':30s} "
    f"{'AP50':>12s} "
    f"{'AP50-95':>12s}"
)

print(
    "-" * 58
)


for name in CLASS_NAMES:

    values = (
        e23d_metrics[
            "class_metrics"
        ][
            name
        ]
    )


    print(
        f"{name:30s} "
        f"{values['AP50'] * 100:11.2f}% "
        f"{values['AP50_95'] * 100:11.2f}%"
    )


# ==================================================================================================
# 20. CRITICAL SAFETY CHECK:
#
# E23-D does not touch ECAL, HDPE, NP, PET, PET Oil.
#
# Therefore those metrics should remain exactly E20-A.
# ==================================================================================================

print()
print("=" * 120)
print("UNCHANGED-CLASS SANITY CHECK")
print("=" * 120)


for name in [
    "ecal",
    "hdpe",
    "non_plastic",
    "pet",
    "pet_oil",
]:

    before_ap50 = (
        e20_metrics[
            "class_metrics"
        ][
            name
        ][
            "AP50"
        ]
    )

    after_ap50 = (
        e23d_metrics[
            "class_metrics"
        ][
            name
        ][
            "AP50"
        ]
    )


    delta = (
        after_ap50
        -
        before_ap50
    ) * 100


    print(
        f"{name:25s}: "
        f"{delta:+.6f} pp"
    )


# ==================================================================================================
# 21. DELTA VS E20-A TEST
# ==================================================================================================

print()
print("=" * 120)
print("DELTA VS VERIFIED E20-A TEST")
print("=" * 120)


delta_e20 = {
    "AP50_95_pp":
        (
            e23d_metrics[
                "AP50_95"
            ]
            -
            e20_metrics[
                "AP50_95"
            ]
        )
        * 100,

    "AP50_pp":
        (
            e23d_metrics[
                "AP50"
            ]
            -
            e20_metrics[
                "AP50"
            ]
        )
        * 100,

    "six_plastic_AP50_pp":
        (
            e23d_metrics[
                "six_plastic_AP50"
            ]
            -
            e20_metrics[
                "six_plastic_AP50"
            ]
        )
        * 100,

    "MR_AP50_pp":
        (
            e23d_metrics[
                "class_metrics"
            ][
                "mixed_plastic_rigid"
            ][
                "AP50"
            ]
            -
            e20_metrics[
                "class_metrics"
            ][
                "mixed_plastic_rigid"
            ][
                "AP50"
            ]
        )
        * 100,

    "MS_AP50_pp":
        (
            e23d_metrics[
                "class_metrics"
            ][
                "mixed_plastic_soft"
            ][
                "AP50"
            ]
            -
            e20_metrics[
                "class_metrics"
            ][
                "mixed_plastic_soft"
            ][
                "AP50"
            ]
        )
        * 100,

    "MR_MS_mean_AP50_pp":
        (
            e23d_metrics[
                "MR_MS_mean_AP50"
            ]
            -
            e20_metrics[
                "MR_MS_mean_AP50"
            ]
        )
        * 100,
}


for key, value in (
    delta_e20.items()
):

    print(
        f"{key:30s}: "
        f"{value:+.4f} pp"
    )


# ==================================================================================================
# 22. COMPARE WITH E21-B + SORTWASTE
# ==================================================================================================

E21B_TEST = {
    "AP50_95": 46.2881,
    "AP50": 62.3518,
    "six_plastic_AP50": 69.4212,
    "MR": 48.70,
    "MS": 42.53,
    "MR_MS": 45.6119,
}

SORTWASTE_SIX_PLASTIC_AP50 = 69.38


print()
print("=" * 120)
print("FINAL COMPARISON")
print("=" * 120)

print(
    f"\n"
    f"{'System':12s}"
    f"{'AP50-95':>12s}"
    f"{'AP50':>12s}"
    f"{'Plastic':>12s}"
    f"{'MR':>10s}"
    f"{'MS':>10s}"
    f"{'MR/MS':>12s}"
)

print(
    "-" * 80
)


print(
    f"{'E20-A':12s}"
    f"{e20_metrics['AP50_95'] * 100:11.4f}%"
    f"{e20_metrics['AP50'] * 100:11.4f}%"
    f"{e20_metrics['six_plastic_AP50'] * 100:11.4f}%"
    f"{e20_metrics['class_metrics']['mixed_plastic_rigid']['AP50'] * 100:9.2f}%"
    f"{e20_metrics['class_metrics']['mixed_plastic_soft']['AP50'] * 100:9.2f}%"
    f"{e20_metrics['MR_MS_mean_AP50'] * 100:11.4f}%"
)


print(
    f"{'E21-B':12s}"
    f"{E21B_TEST['AP50_95']:11.4f}%"
    f"{E21B_TEST['AP50']:11.4f}%"
    f"{E21B_TEST['six_plastic_AP50']:11.4f}%"
    f"{E21B_TEST['MR']:9.2f}%"
    f"{E21B_TEST['MS']:9.2f}%"
    f"{E21B_TEST['MR_MS']:11.4f}%"
)


print(
    f"{'E23-D':12s}"
    f"{e23d_metrics['AP50_95'] * 100:11.4f}%"
    f"{e23d_metrics['AP50'] * 100:11.4f}%"
    f"{e23d_metrics['six_plastic_AP50'] * 100:11.4f}%"
    f"{e23d_metrics['class_metrics']['mixed_plastic_rigid']['AP50'] * 100:9.2f}%"
    f"{e23d_metrics['class_metrics']['mixed_plastic_soft']['AP50'] * 100:9.2f}%"
    f"{e23d_metrics['MR_MS_mean_AP50'] * 100:11.4f}%"
)


sortwaste_delta = (
    e23d_metrics[
        "six_plastic_AP50"
    ]
    * 100
    -
    SORTWASTE_SIX_PLASTIC_AP50
)


print()
print("=" * 120)
print("ALIGNED SORTWASTE SIX-PLASTIC AP50")
print("=" * 120)

print()
print(
    f"SortWaste : "
    f"{SORTWASTE_SIX_PLASTIC_AP50:.4f}%"
)

print(
    f"E23-D     : "
    f"{e23d_metrics['six_plastic_AP50'] * 100:.4f}%"
)

print(
    f"Difference: "
    f"{sortwaste_delta:+.4f} pp"
)


# ==================================================================================================
# 23. SAVE RESULTS
# ==================================================================================================

result_payload = {
    "experiment":
        "E23-D",

    "split":
        "held-out TEST",

    "valid_test_evaluation":
        True,

    "base_predictions":
        str(
            E20_PRED_PATH
        ),

    "detector_predictions":
        str(
            E12_PRED_PATH
        ),

    "method":
        (
            "Frozen E20-A predictions with algebraically recovered "
            "E20 classifier confidence and E23-A confidence-contrastive "
            "MR/MS routing."
        ),

    "frozen_thresholds": {
        "MS_to_MR": {
            "e23_min":
                MS_TO_MR_E23_MIN,

            "e20_max":
                MS_TO_MR_E20_MAX,
        },

        "MR_to_MS": {
            "e23_min":
                MR_TO_MS_E23_MIN,

            "e20_max":
                MR_TO_MS_E20_MAX,
        },
    },

    "E20A_baseline":
        e20_metrics,

    "E23D":
        e23d_metrics,

    "routing": {
        "proposals":
            dict(
                proposals
            ),

        "accepted":
            dict(
                accepted
            ),

        "total_changed":
            int(
                changed_classes
            ),
    },

    "delta_vs_E20A_pp":
        delta_e20,

    "sortwaste_six_plastic_AP50":
        SORTWASTE_SIX_PLASTIC_AP50,

    "delta_vs_sortwaste_pp":
        sortwaste_delta,
}


with open(
    RESULTS_PATH,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        result_payload,
        f,
        indent=2
    )


# ==================================================================================================
# 24. SAVE SUMMARY
# ==================================================================================================

summary_lines = [
    "E23-D — FROZEN HELD-OUT TEST",
    "=" * 100,
    "",
    "Base: verified E20-A TEST predictions",
    "",
    (
        f"MS -> MR: "
        f"E23 >= {MS_TO_MR_E23_MIN:.3f}, "
        f"E20 <= {MS_TO_MR_E20_MAX:.2f}"
    ),
    (
        f"MR -> MS: "
        f"E23 >= {MR_TO_MS_E23_MIN:.3f}, "
        f"E20 <= {MR_TO_MS_E20_MAX:.2f}"
    ),
    "",
    f"AP50-95               : {e23d_metrics['AP50_95'] * 100:.4f}%",
    f"AP50                  : {e23d_metrics['AP50'] * 100:.4f}%",
    f"AP75                  : {e23d_metrics['AP75'] * 100:.4f}%",
    f"AR100                 : {e23d_metrics['AR100'] * 100:.4f}%",
    f"Six-plastic mean AP50: {e23d_metrics['six_plastic_AP50'] * 100:.4f}%",
    f"MR/MS mean AP50      : {e23d_metrics['MR_MS_mean_AP50'] * 100:.4f}%",
    "",
    f"MS -> MR accepted: {accepted['MS->MR']:,}",
    f"MR -> MS accepted: {accepted['MR->MS']:,}",
    "",
    f"SortWaste six-plastic AP50: {SORTWASTE_SIX_PLASTIC_AP50:.4f}%",
    f"E23-D six-plastic AP50    : {e23d_metrics['six_plastic_AP50'] * 100:.4f}%",
    f"Difference                : {sortwaste_delta:+.4f} pp",
]


with open(
    SUMMARY_PATH,
    "w",
    encoding="utf-8"
) as f:

    f.write(
        "\n".join(
            summary_lines
        )
    )


# ==================================================================================================
# 25. COMPLETE
# ==================================================================================================

print()
print("=" * 120)
print("E23-D CORRECTED HELD-OUT TEST COMPLETE")
print("=" * 120)

print()
print(
    f"Matched cache : "
    f"{MATCH_CACHE_PATH}"
)

print(
    f"E23-A cache   : "
    f"{E23_CACHE_PATH}"
)

print(
    f"Predictions   : "
    f"{FINAL_PRED_PATH}"
)

print(
    f"Results       : "
    f"{RESULTS_PATH}"
)

print(
    f"Summary       : "
    f"{SUMMARY_PATH}"
)

E23-D — FINAL CORRECTED FROZEN HELD-OUT TEST

PyTorch : 2.13.0+cu126
Device  : cuda
GPU     : NVIDIA GeForce RTX 3050 Ti Laptop GPU

FROZEN E23-D THRESHOLDS

MS -> MR : E23 >= 0.920, E20 <= 0.90
MR -> MS : E23 >= 0.997, E20 <= 0.95

LOADED FROZEN TEST PREDICTIONS

E20-A predictions : 57,487
E12 predictions   : 57,487
loading annotations into memory...
Done (t=0.04s)
creating index...
index created!

VERIFYING SAVED E20-A TEST BASELINE
Loading and preparing results...
DONE (t=0.07s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=5.69s).
Accumulating evaluation results...
DONE (t=0.54s).

AP50-95        : 46.2612%
AP50           : 62.3271%
Plastic AP50   : 69.4353%
MR AP50        : 48.9530%
MS AP50        : 42.0863%

E20-A baseline verification PASSED.

MATCHING E20-A TO E12 DETECTIONS

Matched   : 57,487
Unmatched : 0
Unique E12 used : 57,487

RECOVERED E20-A CLASSIFIER CONFIDENCE

Min    : 0.000000
Mean   : 0.604343
Median : 0.8

# Model E24

## E24-A — Validation-Only Confidence Recalibration of E23-D MR/MS Overrides

In [40]:
# E24-A — VALIDATION-ONLY CONFIDENCE RECALIBRATION OF E23-D MR/MS OVERRIDES
#
# PURPOSE
# --------------------------------------------------------------------------------------------------
# E23-D changes some E20-A Mixed Soft <-> Mixed Rigid labels but preserves the original
# E20-A detection score.
#
# Hypothesis:
#   After an accepted E23-A class override, the original E20-A score may not be optimally
#   calibrated for the NEW class.
#
# E24-A therefore keeps:
#   - E20-A predictions frozen
#   - E23-D routing thresholds frozen
#   - E23-D accepted/rejected decisions frozen
#   - all bounding boxes frozen
#   - all non-overridden scores frozen
#
# It searches ONLY the confidence assigned to accepted E23-D overrides:
#
#       new_score = E20_score ** beta * E23_confidence ** (1 - beta)
#
# beta = 1.00 reproduces E23-D exactly.
#
# DATA:
#   VALIDATION ONLY
#
# NO:
#   - training
#   - YOLO inference
#   - MobileNet inference
#   - ConvNeXt inference
#   - TEST-set access
#
# PRIMARY SELECTION:
#   Six-plastic mean AP50
#
# SECONDARY:
#   MR/MS mean AP50
#
# TIE-BREAK:
#   Overall 7-class AP50-95
# ==================================================================================================

import json
from pathlib import Path
from collections import Counter

import numpy as np

from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval


# ==================================================================================================
# 1. PATHS
# ==================================================================================================

BASE = Path(
    r"C:\Users\varda\Documents\_My Computer\COMMON_space\Learning"
    r"\Upgrad_EPGP_MS\upgrad_IIITB_MS_course\MS_LJMU_Material"
)

THESIS_CODE = (
    BASE
    / "Thesis_Code"
)

RUNS_ROOT = (
    THESIS_CODE
    / "runs"
    / "sortwaste"
)


CACHE_PATH = (
    RUNS_ROOT
    / "E23B_E20A_binary_MRMS_routing"
    / "E23B_cached_predictions_with_E23A.json"
)


E23D_CONFIG_PATH = (
    RUNS_ROOT
    / "E23D_confidence_contrastive_MRMS"
    / "E23D_best_configuration.json"
)


GT_PATH = (
    RUNS_ROOT
    / "E18D_class_selective_convnext"
    / "E18D_val_gt_7class.json"
)


OUTPUT_DIR = (
    RUNS_ROOT
    / "E24A_E23D_override_confidence_recalibration"
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)


GRID_PATH = (
    OUTPUT_DIR
    / "E24A_grid_results.json"
)

BEST_CONFIG_PATH = (
    OUTPUT_DIR
    / "E24A_best_configuration.json"
)

BEST_PRED_PATH = (
    OUTPUT_DIR
    / "E24A_best_predictions.json"
)

SUMMARY_PATH = (
    OUTPUT_DIR
    / "E24A_summary.txt"
)


for path, label in [
    (CACHE_PATH, "E23-B/E23-D validation cache"),
    (E23D_CONFIG_PATH, "E23-D frozen configuration"),
    (GT_PATH, "7-class validation GT"),
]:

    if not path.exists():

        raise FileNotFoundError(
            f"{label} not found:\n{path}"
        )


# ==================================================================================================
# 2. CLASS DEFINITIONS
# ==================================================================================================

CLASS_NAMES = [
    "ecal",
    "hdpe",
    "mixed_plastic_rigid",
    "mixed_plastic_soft",
    "non_plastic",
    "pet",
    "pet_oil",
]

ECAL = 0
HDPE = 1
MR = 2
MS = 3
NON_PLASTIC = 4
PET = 5
PET_OIL = 6


PLASTIC_CLASS_INDICES = [
    ECAL,
    HDPE,
    MR,
    MS,
    PET,
    PET_OIL,
]


# ==================================================================================================
# 3. LOAD CACHE + E23-D CONFIG
# ==================================================================================================

with open(
    CACHE_PATH,
    "r",
    encoding="utf-8"
) as f:

    cache = json.load(f)


with open(
    E23D_CONFIG_PATH,
    "r",
    encoding="utf-8"
) as f:

    e23d_config = json.load(f)


print("=" * 115)
print("E24-A — VALIDATION-ONLY CONFIDENCE RECALIBRATION OF E23-D MR/MS OVERRIDES")
print("=" * 115)

print()
print(
    f"Cache entries : "
    f"{len(cache):,}"
)

print(
    f"Cache         : "
    f"{CACHE_PATH}"
)

print(
    f"Validation GT : "
    f"{GT_PATH}"
)

print(
    f"Output        : "
    f"{OUTPUT_DIR}"
)


if len(cache) != 55605:

    raise RuntimeError(
        f"Expected 55,605 validation detections, "
        f"found {len(cache):,}."
    )


# ==================================================================================================
# 4. LOAD FROZEN E23-D THRESHOLDS
# ==================================================================================================

if e23d_config.get(
    "test_used"
) is not False:

    raise RuntimeError(
        "E23-D configuration is not marked validation-only."
    )


best_e23d = (
    e23d_config[
        "best_configuration"
    ]
)


MS_TO_MR_E23_MIN = float(
    best_e23d[
        "MS_to_MR"
    ][
        "e23_min"
    ]
)

MS_TO_MR_E20_MAX = float(
    best_e23d[
        "MS_to_MR"
    ][
        "e20_max"
    ]
)

MR_TO_MS_E23_MIN = float(
    best_e23d[
        "MR_to_MS"
    ][
        "e23_min"
    ]
)

MR_TO_MS_E20_MAX = float(
    best_e23d[
        "MR_to_MS"
    ][
        "e20_max"
    ]
)


print()
print("Frozen E23-D routing:")

print(
    f"  MS -> MR : "
    f"E23 >= {MS_TO_MR_E23_MIN:.3f}, "
    f"E20 <= {MS_TO_MR_E20_MAX:.2f}"
)

print(
    f"  MR -> MS : "
    f"E23 >= {MR_TO_MS_E23_MIN:.3f}, "
    f"E20 <= {MR_TO_MS_E20_MAX:.2f}"
)


# Hard fail if anything has changed.

if not (
    np.isclose(
        MS_TO_MR_E23_MIN,
        0.920
    )
    and
    np.isclose(
        MS_TO_MR_E20_MAX,
        0.90
    )
    and
    np.isclose(
        MR_TO_MS_E23_MIN,
        0.997
    )
    and
    np.isclose(
        MR_TO_MS_E20_MAX,
        0.95
    )
):

    raise RuntimeError(
        "Frozen E23-D thresholds do not match the validated configuration."
    )


# ==================================================================================================
# 5. E23 CACHE PARSER
#
# Supports likely historical field naming variants.
# ==================================================================================================

def extract_e23(
    item
):

    info = item.get(
        "e23"
    )


    if info is None:

        return None


    if not isinstance(
        info,
        dict
    ):

        raise RuntimeError(
            f"Unexpected E23 structure:\n{info}"
        )


    # ----------------------------------------------------------------------------------------------
    # First recover confidence/probabilities if available.
    # ----------------------------------------------------------------------------------------------

    probs = None


    for key in [
        "probs",
        "probabilities",
        "e23_probs",
    ]:

        if key in info:

            probs = np.asarray(
                info[
                    key
                ],
                dtype=np.float64
            )

            break


    # ----------------------------------------------------------------------------------------------
    # Predicted class
    # ----------------------------------------------------------------------------------------------

    pred = None


    global_keys = [
        "pred_global_idx",
        "global_idx",
        "e23_global_idx",
        "e23_pred_global_idx",
        "predicted_global_idx",
    ]


    for key in global_keys:

        if key in info:

            pred = int(
                info[
                    key
                ]
            )

            break


    # Binary local classes:
    # 0 = MR
    # 1 = MS

    if pred is None:

        for key in [
            "pred_local_idx",
            "local_idx",
            "e23_local_idx",
            "pred_idx",
            "class_idx",
        ]:

            if key in info:

                local = int(
                    info[
                        key
                    ]
                )

                if local == 0:

                    pred = MR

                elif local == 1:

                    pred = MS

                elif local in {
                    MR,
                    MS,
                }:

                    pred = local

                else:

                    raise RuntimeError(
                        f"Unexpected E23 class index: {local}"
                    )

                break


    if pred is None and probs is not None:

        local = int(
            np.argmax(
                probs
            )
        )

        pred = (
            MR
            if local == 0
            else MS
        )


    # ----------------------------------------------------------------------------------------------
    # Top confidence
    # ----------------------------------------------------------------------------------------------

    top_prob = None


    for key in [
        "top_prob",
        "confidence",
        "prob",
        "e23_top_prob",
        "pred_prob",
    ]:

        if key in info:

            top_prob = float(
                info[
                    key
                ]
            )

            break


    if top_prob is None and probs is not None:

        top_prob = float(
            np.max(
                probs
            )
        )


    # Sometimes MR/MS probs may be stored separately.

    if top_prob is None:

        mr_prob = None
        ms_prob = None


        for key in [
            "mr_prob",
            "p_mr",
        ]:

            if key in info:

                mr_prob = float(
                    info[
                        key
                    ]
                )


        for key in [
            "ms_prob",
            "p_ms",
        ]:

            if key in info:

                ms_prob = float(
                    info[
                        key
                    ]
                )


        if (
            mr_prob is not None
            and
            ms_prob is not None
        ):

            if mr_prob >= ms_prob:

                pred = MR
                top_prob = mr_prob

            else:

                pred = MS
                top_prob = ms_prob


    if (
        pred is None
        or
        top_prob is None
    ):

        raise RuntimeError(
            "Unable to decode E23 cache entry:\n"
            f"{info}"
        )


    if pred not in {
        MR,
        MS,
    }:

        raise RuntimeError(
            f"E23 predicted invalid global class {pred}"
        )


    return {
        "pred_idx":
            int(
                pred
            ),

        "top_prob":
            float(
                top_prob
            ),
    }


# Show the first E23 entry just as a diagnostic.

first_e23 = next(
    (
        item[
            "e23"
        ]

        for item
        in cache

        if item.get(
            "e23"
        )
        is not None
    ),
    None
)


print()
print(
    "Example E23 cache entry:"
)

print(
    first_e23
)


# ==================================================================================================
# 6. EXACT E20-A CURRENT-CLASS CONFIDENCE
#
# This reproduces the confidence definition used by E23-D validation:
#
# If E20-A final class came from E18 and ConvNeXt proposed that exact class:
#     use ConvNeXt top probability
#
# Otherwise:
#     use MobileNet probability of current E20-A class
#
# ==================================================================================================

def get_e20_current_class_confidence(
    item
):

    current_idx = int(
        item[
            "_e20_class_idx"
        ]
    )


    source = str(
        item.get(
            "_e20_source",
            ""
        )
    ).lower()


    conv = item.get(
        "conv"
    )


    if (
        source
        in {
            "e18",
            "convnext",
        }
        and
        conv is not None
    ):

        conv_global_idx = int(
            conv[
                "conv_global_idx"
            ]
        )


        if conv_global_idx == current_idx:

            return float(
                conv[
                    "conv_top_prob"
                ]
            )


    mn_probs = np.asarray(
        item[
            "mn_probs"
        ],
        dtype=np.float64
    )


    return float(
        mn_probs[
            current_idx
        ]
    )


# ==================================================================================================
# 7. BUILD FROZEN E23-D ROUTING ONCE
#
# We record accepted E23-D overrides.
# E24-A is NOT allowed to modify these decisions.
# ==================================================================================================

routing_records = []

proposal_counts = Counter()
accepted_counts = Counter()


for cache_idx, item in enumerate(
    cache
):

    current_idx = int(
        item[
            "_e20_class_idx"
        ]
    )


    final_idx = (
        current_idx
    )


    e20_score = float(
        item[
            "_e20_score"
        ]
    )


    e20_conf = (
        get_e20_current_class_confidence(
            item
        )
    )


    e23_info = extract_e23(
        item
    )


    accepted_direction = None

    e23_top_prob = None


    if (
        current_idx in {
            MR,
            MS,
        }
        and
        e23_info is not None
    ):

        e23_idx = int(
            e23_info[
                "pred_idx"
            ]
        )

        e23_top_prob = float(
            e23_info[
                "top_prob"
            ]
        )


        # ------------------------------------------------------------------------------------------
        # MS -> MR
        # ------------------------------------------------------------------------------------------

        if (
            current_idx == MS
            and
            e23_idx == MR
        ):

            proposal_counts[
                "MS->MR"
            ] += 1


            if (
                e23_top_prob
                >=
                MS_TO_MR_E23_MIN
                and
                e20_conf
                <=
                MS_TO_MR_E20_MAX
            ):

                final_idx = (
                    MR
                )

                accepted_direction = (
                    "MS->MR"
                )

                accepted_counts[
                    "MS->MR"
                ] += 1


        # ------------------------------------------------------------------------------------------
        # MR -> MS
        # ------------------------------------------------------------------------------------------

        elif (
            current_idx == MR
            and
            e23_idx == MS
        ):

            proposal_counts[
                "MR->MS"
            ] += 1


            if (
                e23_top_prob
                >=
                MR_TO_MS_E23_MIN
                and
                e20_conf
                <=
                MR_TO_MS_E20_MAX
            ):

                final_idx = (
                    MS
                )

                accepted_direction = (
                    "MR->MS"
                )

                accepted_counts[
                    "MR->MS"
                ] += 1


    routing_records.append(
        {
            "cache_index":
                cache_idx,

            "original_idx":
                current_idx,

            "final_idx":
                final_idx,

            "e20_score":
                e20_score,

            "e20_conf":
                float(
                    e20_conf
                ),

            "e23_top_prob":
                (
                    float(
                        e23_top_prob
                    )
                    if e23_top_prob is not None
                    else None
                ),

            "accepted_direction":
                accepted_direction,
        }
    )


print()
print("=" * 115)
print("FROZEN E23-D ROUTING RECONSTRUCTION")
print("=" * 115)

print()

print(
    f"MS -> MR proposals : "
    f"{proposal_counts['MS->MR']:,}"
)

print(
    f"MR -> MS proposals : "
    f"{proposal_counts['MR->MS']:,}"
)

print()

print(
    f"MS -> MR accepted  : "
    f"{accepted_counts['MS->MR']:,}"
)

print(
    f"MR -> MS accepted  : "
    f"{accepted_counts['MR->MS']:,}"
)

print(
    f"Total accepted     : "
    f"{sum(accepted_counts.values()):,}"
)


# Must reproduce validated E23-D routing exactly.

if (
    accepted_counts[
        "MS->MR"
    ]
    != 606
    or
    accepted_counts[
        "MR->MS"
    ]
    != 196
):

    raise RuntimeError(
        "\nE23-D accepted routing does not reproduce "
        "the validated 606 MS->MR + 196 MR->MS.\n"
        "E24-A search has NOT started."
    )


# ==================================================================================================
# 8. BUILD COCO PREDICTIONS
#
# bbox in cache is XYXY.
# ==================================================================================================

def build_predictions(
    beta
):

    """
    beta = 1.0:
        exact E23-D score preservation.

    beta < 1:
        accepted E23-D override score becomes:

            E20_score^beta * E23_conf^(1-beta)

    Everything else remains untouched.
    """

    predictions = []


    for item, routing in zip(
        cache,
        routing_records
    ):

        x1, y1, x2, y2 = [
            float(v)

            for v
            in item[
                "bbox"
            ]
        ]


        width = (
            x2 - x1
        )

        height = (
            y2 - y1
        )


        if (
            width <= 0
            or
            height <= 0
        ):

            continue


        score = float(
            routing[
                "e20_score"
            ]
        )


        # ------------------------------------------------------------------------------------------
        # E24-A changes score ONLY for accepted E23-D overrides.
        # ------------------------------------------------------------------------------------------

        if (
            routing[
                "accepted_direction"
            ]
            is not None
        ):

            e23_prob = float(
                routing[
                    "e23_top_prob"
                ]
            )


            score = (
                max(
                    score,
                    1e-12
                )
                ** beta
            ) * (
                max(
                    e23_prob,
                    1e-12
                )
                ** (
                    1.0
                    -
                    beta
                )
            )


        predictions.append(
            {
                "image_id":
                    int(
                        item[
                            "image_id"
                        ]
                    ),

                "category_id":
                    int(
                        routing[
                            "final_idx"
                        ]
                        + 1
                    ),

                "bbox": [
                    x1,
                    y1,
                    width,
                    height,
                ],

                "score":
                    float(
                        score
                    ),
            }
        )


    return predictions


# ==================================================================================================
# 9. COCO EVALUATION
# ==================================================================================================

coco_gt = COCO(
    str(
        GT_PATH
    )
)


def valid_mean(
    values
):

    values = np.asarray(
        values
    )


    valid = values[
        values > -1
    ]


    if valid.size == 0:

        return float(
            "nan"
        )


    return float(
        valid.mean()
    )


def evaluate_predictions(
    predictions
):

    coco_dt = coco_gt.loadRes(
        predictions
    )


    evaluator = COCOeval(
        coco_gt,
        coco_dt,
        "bbox"
    )


    evaluator.params.maxDets = [
        1,
        10,
        100,
    ]


    evaluator.evaluate()
    evaluator.accumulate()


    precision = (
        evaluator.eval[
            "precision"
        ]
    )

    recall = (
        evaluator.eval[
            "recall"
        ]
    )


    ious = (
        evaluator.params.iouThrs
    )


    idx50 = int(
        np.where(
            np.isclose(
                ious,
                0.50
            )
        )[0][0]
    )


    idx75 = int(
        np.where(
            np.isclose(
                ious,
                0.75
            )
        )[0][0]
    )


    overall = {
        "AP50_95":
            valid_mean(
                precision[
                    :,
                    :,
                    :,
                    0,
                    -1
                ]
            ),

        "AP50":
            valid_mean(
                precision[
                    idx50,
                    :,
                    :,
                    0,
                    -1
                ]
            ),

        "AP75":
            valid_mean(
                precision[
                    idx75,
                    :,
                    :,
                    0,
                    -1
                ]
            ),

        "AR100":
            valid_mean(
                recall[
                    :,
                    :,
                    0,
                    -1
                ]
            ),
    }


    class_metrics = {}


    for class_idx, class_name in enumerate(
        CLASS_NAMES
    ):

        class_metrics[
            class_name
        ] = {
            "AP50":
                valid_mean(
                    precision[
                        idx50,
                        :,
                        class_idx,
                        0,
                        -1
                    ]
                ),

            "AP50_95":
                valid_mean(
                    precision[
                        :,
                        :,
                        class_idx,
                        0,
                        -1
                    ]
                ),
        }


    six_plastic_mean_ap50 = float(
        np.mean(
            [
                class_metrics[
                    CLASS_NAMES[
                        idx
                    ]
                ][
                    "AP50"
                ]

                for idx
                in PLASTIC_CLASS_INDICES
            ]
        )
    )


    mr_ms_mean_ap50 = float(
        np.mean(
            [
                class_metrics[
                    "mixed_plastic_rigid"
                ][
                    "AP50"
                ],

                class_metrics[
                    "mixed_plastic_soft"
                ][
                    "AP50"
                ],
            ]
        )
    )


    return {
        "overall":
            overall,

        "class_metrics":
            class_metrics,

        "six_plastic_mean_AP50":
            six_plastic_mean_ap50,

        "MR_MS_mean_AP50":
            mr_ms_mean_ap50,
    }


# ==================================================================================================
# 10. REPRODUCE E23-D EXACTLY FIRST
#
# beta = 1.0 => score remains E20-A score.
# ==================================================================================================

print()
print("=" * 115)
print("RECONSTRUCTING FROZEN E23-D VALIDATION BASELINE")
print("=" * 115)


baseline_predictions = build_predictions(
    beta=1.0
)


baseline_eval = evaluate_predictions(
    baseline_predictions
)


print()
print(
    f"Predictions             : "
    f"{len(baseline_predictions):,}"
)

print(
    f"Six-plastic mean AP50  : "
    f"{baseline_eval['six_plastic_mean_AP50'] * 100:.4f}%"
)

print(
    f"MR AP50                : "
    f"{baseline_eval['class_metrics']['mixed_plastic_rigid']['AP50'] * 100:.4f}%"
)

print(
    f"MS AP50                : "
    f"{baseline_eval['class_metrics']['mixed_plastic_soft']['AP50'] * 100:.4f}%"
)

print(
    f"MR/MS mean AP50        : "
    f"{baseline_eval['MR_MS_mean_AP50'] * 100:.4f}%"
)

print(
    f"Overall AP50-95        : "
    f"{baseline_eval['overall']['AP50_95'] * 100:.4f}%"
)

print(
    f"Overall AP50           : "
    f"{baseline_eval['overall']['AP50'] * 100:.4f}%"
)


# Exact E23-D validation references.

REF_PLASTIC = (
    float(
        best_e23d[
            "six_plastic_mean_AP50"
        ]
    )
)

REF_MRMS = (
    float(
        best_e23d[
            "MR_MS_mean_AP50"
        ]
    )
)

REF_AP = (
    float(
        best_e23d[
            "overall"
        ][
            "AP50_95"
        ]
    )
)


if (
    abs(
        baseline_eval[
            "six_plastic_mean_AP50"
        ]
        -
        REF_PLASTIC
    )
    > 1e-6
    or
    abs(
        baseline_eval[
            "MR_MS_mean_AP50"
        ]
        -
        REF_MRMS
    )
    > 1e-6
    or
    abs(
        baseline_eval[
            "overall"
        ][
            "AP50_95"
        ]
        -
        REF_AP
    )
    > 1e-6
):

    raise RuntimeError(
        "\nFrozen E23-D baseline was NOT reproduced exactly.\n"
        "E24-A search has NOT started."
    )


print()
print(
    "E23-D validation reproduction PASSED."
)


# ==================================================================================================
# 11. E24-A BETA GRID
#
# beta = 1.00 is the untouched E23-D baseline.
#
# Lower beta gives increasing weight to E23 specialist confidence.
# ==================================================================================================

BETAS = [
    round(
        x,
        2
    )

    for x in np.arange(
        0.00,
        1.0001,
        0.05
    )
]


print()
print("=" * 115)
print("RUNNING E24-A VALIDATION GRID SEARCH")
print("=" * 115)

print()
print(
    f"Beta candidates : "
    f"{len(BETAS)}"
)

print(
    "Score rule      : "
    "E20_score^beta × E23_conf^(1-beta)"
)

print(
    "beta=1.00       : "
    "exact E23-D baseline"
)


all_results = []


for experiment_number, beta in enumerate(
    BETAS,
    start=1
):

    predictions = build_predictions(
        beta=beta
    )


    evaluation = evaluate_predictions(
        predictions
    )


    result = {
        "beta":
            float(
                beta
            ),

        "six_plastic_mean_AP50":
            evaluation[
                "six_plastic_mean_AP50"
            ],

        "MR_MS_mean_AP50":
            evaluation[
                "MR_MS_mean_AP50"
            ],

        "overall":
            evaluation[
                "overall"
            ],

        "class_metrics":
            evaluation[
                "class_metrics"
            ],
    }


    all_results.append(
        result
    )


    print(
        f"[{experiment_number:02d}/{len(BETAS):02d}] "
        f"beta={beta:4.2f} | "
        f"Plastic="
        f"{evaluation['six_plastic_mean_AP50'] * 100:7.4f}%  "
        f"MR="
        f"{evaluation['class_metrics']['mixed_plastic_rigid']['AP50'] * 100:7.3f}%  "
        f"MS="
        f"{evaluation['class_metrics']['mixed_plastic_soft']['AP50'] * 100:7.3f}%  "
        f"MR/MS="
        f"{evaluation['MR_MS_mean_AP50'] * 100:7.3f}%  "
        f"AP="
        f"{evaluation['overall']['AP50_95'] * 100:7.4f}%"
    )


# ==================================================================================================
# 12. RANK
#
# Same thesis-oriented selection hierarchy:
#
# 1. six-plastic AP50
# 2. MR/MS mean AP50
# 3. overall AP50-95
# ==================================================================================================

ranked = sorted(
    all_results,

    key=lambda x: (
        x[
            "six_plastic_mean_AP50"
        ],

        x[
            "MR_MS_mean_AP50"
        ],

        x[
            "overall"
        ][
            "AP50_95"
        ],
    ),

    reverse=True,
)


best = ranked[
    0
]


best_beta = float(
    best[
        "beta"
    ]
)


# ==================================================================================================
# 13. DELTA VS E23-D
# ==================================================================================================

delta_vs_e23d = {
    "six_plastic_AP50_pp":
        (
            best[
                "six_plastic_mean_AP50"
            ]
            -
            baseline_eval[
                "six_plastic_mean_AP50"
            ]
        )
        * 100,

    "MR_AP50_pp":
        (
            best[
                "class_metrics"
            ][
                "mixed_plastic_rigid"
            ][
                "AP50"
            ]
            -
            baseline_eval[
                "class_metrics"
            ][
                "mixed_plastic_rigid"
            ][
                "AP50"
            ]
        )
        * 100,

    "MS_AP50_pp":
        (
            best[
                "class_metrics"
            ][
                "mixed_plastic_soft"
            ][
                "AP50"
            ]
            -
            baseline_eval[
                "class_metrics"
            ][
                "mixed_plastic_soft"
            ][
                "AP50"
            ]
        )
        * 100,

    "MR_MS_mean_AP50_pp":
        (
            best[
                "MR_MS_mean_AP50"
            ]
            -
            baseline_eval[
                "MR_MS_mean_AP50"
            ]
        )
        * 100,

    "overall_AP50_95_pp":
        (
            best[
                "overall"
            ][
                "AP50_95"
            ]
            -
            baseline_eval[
                "overall"
            ][
                "AP50_95"
            ]
        )
        * 100,

    "overall_AP50_pp":
        (
            best[
                "overall"
            ][
                "AP50"
            ]
            -
            baseline_eval[
                "overall"
            ][
                "AP50"
            ]
        )
        * 100,
}


# ==================================================================================================
# 14. PRINT TOP RESULTS
# ==================================================================================================

print()
print("=" * 115)
print("TOP E24-A VALIDATION RESULTS")
print("=" * 115)

print(
    f"\n"
    f"{'Rank':>4s} "
    f"{'Beta':>7s} "
    f"{'Plastic AP50':>14s} "
    f"{'MR AP50':>11s} "
    f"{'MS AP50':>11s} "
    f"{'MR/MS':>11s} "
    f"{'AP50-95':>11s}"
)

print(
    "-" * 80
)


for rank, result in enumerate(
    ranked[:10],
    start=1
):

    print(
        f"{rank:4d} "
        f"{result['beta']:7.2f} "
        f"{result['six_plastic_mean_AP50'] * 100:13.4f}% "
        f"{result['class_metrics']['mixed_plastic_rigid']['AP50'] * 100:10.3f}% "
        f"{result['class_metrics']['mixed_plastic_soft']['AP50'] * 100:10.3f}% "
        f"{result['MR_MS_mean_AP50'] * 100:10.3f}% "
        f"{result['overall']['AP50_95'] * 100:10.4f}%"
    )


# ==================================================================================================
# 15. FINAL DECISION
# ==================================================================================================

print()
print("=" * 115)
print("BEST E24-A CONFIGURATION — VALIDATION ONLY")
print("=" * 115)

print()
print(
    f"Best beta               : "
    f"{best_beta:.2f}"
)

print(
    f"Six-plastic mean AP50  : "
    f"{best['six_plastic_mean_AP50'] * 100:.4f}%"
)

print(
    f"MR AP50                : "
    f"{best['class_metrics']['mixed_plastic_rigid']['AP50'] * 100:.4f}%"
)

print(
    f"MS AP50                : "
    f"{best['class_metrics']['mixed_plastic_soft']['AP50'] * 100:.4f}%"
)

print(
    f"MR/MS mean AP50        : "
    f"{best['MR_MS_mean_AP50'] * 100:.4f}%"
)

print(
    f"Overall AP50-95        : "
    f"{best['overall']['AP50_95'] * 100:.4f}%"
)

print(
    f"Overall AP50           : "
    f"{best['overall']['AP50'] * 100:.4f}%"
)


print()
print("Delta vs frozen E23-D:")

for metric, value in (
    delta_vs_e23d.items()
):

    print(
        f"{metric:28s}: "
        f"{value:+.4f} pp"
    )


# Promotion rule:
#
# Must improve six-plastic AP50.
#
# If beta == 1.0, E24-A found no score recalibration better than E23-D.

PROMOTE_TO_TEST = (
    best[
        "six_plastic_mean_AP50"
    ]
    >
    baseline_eval[
        "six_plastic_mean_AP50"
    ]
    +
    1e-12
)


print()
print("=" * 115)

if PROMOTE_TO_TEST:

    print(
        "DECISION: E24-A IMPROVES VALIDATION — ELIGIBLE FOR ONE FROZEN TEST EVALUATION."
    )

else:

    print(
        "DECISION: E24-A DOES NOT IMPROVE VALIDATION — REJECT; DO NOT TEST."
    )

print("=" * 115)


# ==================================================================================================
# 16. SAVE OUTPUTS
# ==================================================================================================

best_predictions = build_predictions(
    beta=best_beta
)


with open(
    GRID_PATH,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        ranked,
        f,
        indent=2
    )


best_payload = {
    "experiment":
        "E24-A",

    "description":
        (
            "Validation-only confidence recalibration "
            "of accepted E23-D MR/MS overrides"
        ),

    "dataset":
        "validation",

    "test_used":
        False,

    "base_pipeline":
        "E23-D",

    "routing_changed":
        False,

    "routing_frozen": {
        "MS_to_MR": {
            "e23_min":
                MS_TO_MR_E23_MIN,

            "e20_max":
                MS_TO_MR_E20_MAX,
        },

        "MR_to_MS": {
            "e23_min":
                MR_TO_MS_E23_MIN,

            "e20_max":
                MR_TO_MS_E20_MAX,
        },
    },

    "accepted_counts":
        dict(
            accepted_counts
        ),

    "score_rule":
        "E20_score^beta * E23_confidence^(1-beta)",

    "baseline_E23D":
        baseline_eval,

    "best_beta":
        best_beta,

    "best":
        best,

    "delta_vs_E23D_pp":
        delta_vs_e23d,

    "promote_to_test":
        bool(
            PROMOTE_TO_TEST
        ),
}


with open(
    BEST_CONFIG_PATH,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        best_payload,
        f,
        indent=2
    )


with open(
    BEST_PRED_PATH,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        best_predictions,
        f
    )


summary_lines = [
    "E24-A — Validation-Only Confidence Recalibration of E23-D MR/MS Overrides",
    "=" * 105,
    "",
    "VALIDATION ONLY — TEST NOT USED",
    "",
    "Frozen routing:",
    (
        f"  MS -> MR: "
        f"E23 >= {MS_TO_MR_E23_MIN:.3f}, "
        f"E20 <= {MS_TO_MR_E20_MAX:.2f}"
    ),
    (
        f"  MR -> MS: "
        f"E23 >= {MR_TO_MS_E23_MIN:.3f}, "
        f"E20 <= {MR_TO_MS_E20_MAX:.2f}"
    ),
    "",
    "Score rule:",
    "  new_score = E20_score^beta * E23_confidence^(1-beta)",
    "",
    f"Best beta               : {best_beta:.2f}",
    f"Six-plastic mean AP50  : {best['six_plastic_mean_AP50'] * 100:.4f}%",
    f"MR AP50                : {best['class_metrics']['mixed_plastic_rigid']['AP50'] * 100:.4f}%",
    f"MS AP50                : {best['class_metrics']['mixed_plastic_soft']['AP50'] * 100:.4f}%",
    f"MR/MS mean AP50        : {best['MR_MS_mean_AP50'] * 100:.4f}%",
    f"Overall AP50-95        : {best['overall']['AP50_95'] * 100:.4f}%",
    f"Overall AP50           : {best['overall']['AP50'] * 100:.4f}%",
    "",
    f"Promote to TEST        : {PROMOTE_TO_TEST}",
    "",
    "Delta vs E23-D:",
]


for metric, value in (
    delta_vs_e23d.items()
):

    summary_lines.append(
        f"  {metric:28s}: "
        f"{value:+.4f} pp"
    )


with open(
    SUMMARY_PATH,
    "w",
    encoding="utf-8"
) as f:

    f.write(
        "\n".join(
            summary_lines
        )
    )


# ==================================================================================================
# 17. COMPLETE
# ==================================================================================================

print()
print("=" * 115)
print("E24-A COMPLETE")
print("=" * 115)

print()
print(
    f"Grid results : "
    f"{GRID_PATH}"
)

print(
    f"Best config  : "
    f"{BEST_CONFIG_PATH}"
)

print(
    f"Predictions  : "
    f"{BEST_PRED_PATH}"
)

print(
    f"Summary      : "
    f"{SUMMARY_PATH}"
)

E24-A — VALIDATION-ONLY CONFIDENCE RECALIBRATION OF E23-D MR/MS OVERRIDES

Cache entries : 55,605
Cache         : C:\Users\varda\Documents\_My Computer\COMMON_space\Learning\Upgrad_EPGP_MS\upgrad_IIITB_MS_course\MS_LJMU_Material\Thesis_Code\runs\sortwaste\E23B_E20A_binary_MRMS_routing\E23B_cached_predictions_with_E23A.json
Validation GT : C:\Users\varda\Documents\_My Computer\COMMON_space\Learning\Upgrad_EPGP_MS\upgrad_IIITB_MS_course\MS_LJMU_Material\Thesis_Code\runs\sortwaste\E18D_class_selective_convnext\E18D_val_gt_7class.json
Output        : C:\Users\varda\Documents\_My Computer\COMMON_space\Learning\Upgrad_EPGP_MS\upgrad_IIITB_MS_course\MS_LJMU_Material\Thesis_Code\runs\sortwaste\E24A_E23D_override_confidence_recalibration

Frozen E23-D routing:
  MS -> MR : E23 >= 0.920, E20 <= 0.90
  MR -> MS : E23 >= 0.997, E20 <= 0.95

Example E23 cache entry:
{'probs': [0.9999901056289673, 9.90843000181485e-06], 'local_idx': 0, 'global_idx': 2, 'top_prob': 0.9999901056289673}

FROZEN E23-D R

In [41]:
## E24-A — Frozen Held-Out Test Evaluation of Confidence-Recalibrated E23-D MR/MS Overrides
# ==================================================================================================
# E24-A — FROZEN HELD-OUT TEST EVALUATION OF CONFIDENCE-RECALIBRATED
#         E23-D MR/MS OVERRIDES
#
# PURPOSE
# --------------------------------------------------------------------------------------------------
# Validation-selected E24-A configuration:
#
#     beta = 0.35
#
# Frozen E23-D routing:
#
#     MS -> MR:
#         E23 >= 0.920
#         E20 <= 0.90
#
#     MR -> MS:
#         E23 >= 0.997
#         E20 <= 0.95
#
# E24-A changes ONLY the confidence score of ACCEPTED E23-D overrides:
#
#     new_score =
#         E20_score ** beta
#         *
#         E23_confidence ** (1 - beta)
#
# All of the following remain unchanged:
#
#     - E20-A detector boxes
#     - E20-A image IDs
#     - non-routed class predictions
#     - E23-D routing decisions
#     - routing thresholds
#     - E23-A outputs
#
# TEST is used ONLY ONCE for frozen evaluation.
#
# NO:
#     - threshold search
#     - beta search
#     - YOLO inference
#     - MobileNet inference
#     - E18-B inference
#     - E23-A inference
#     - retraining
#
# ==================================================================================================

import json
from pathlib import Path
from collections import Counter

import numpy as np

from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval


# ==================================================================================================
# 1. PATHS
# ==================================================================================================

BASE = Path(
    r"C:\Users\varda\Documents\_My Computer\COMMON_space\Learning"
    r"\Upgrad_EPGP_MS\upgrad_IIITB_MS_course\MS_LJMU_Material"
)

THESIS_CODE = (
    BASE
    / "Thesis_Code"
)

RUNS_ROOT = (
    THESIS_CODE
    / "runs"
    / "sortwaste"
)


# --------------------------------------------------------------------------------------------------
# Verified E20-A TEST predictions
# --------------------------------------------------------------------------------------------------

E20_PRED_PATH = (
    RUNS_ROOT
    / "FINAL_TEST_E20A"
    / "E20A_test_predictions.json"
)


E20_GT_PATH = (
    RUNS_ROOT
    / "FINAL_TEST_E20A"
    / "E20A_test_gt_7class.json"
)


# --------------------------------------------------------------------------------------------------
# Corrected E23-D TEST caches
# --------------------------------------------------------------------------------------------------

E23D_TEST_DIR = (
    RUNS_ROOT
    / "FINAL_TEST_E23D_CORRECTED"
)


MATCH_CACHE_PATH = (
    E23D_TEST_DIR
    / "E23D_E20A_E12_matched_cache.json"
)


E23_CACHE_PATH = (
    E23D_TEST_DIR
    / "E23D_E23A_MRMS_test_cache.json"
)


# --------------------------------------------------------------------------------------------------
# E23-D frozen validation config
# --------------------------------------------------------------------------------------------------

E23D_CONFIG_PATH = (
    RUNS_ROOT
    / "E23D_confidence_contrastive_MRMS"
    / "E23D_best_configuration.json"
)


# --------------------------------------------------------------------------------------------------
# E24-A validation-selected config
# --------------------------------------------------------------------------------------------------

E24A_CONFIG_PATH = (
    RUNS_ROOT
    / "E24A_E23D_override_confidence_recalibration"
    / "E24A_best_configuration.json"
)


# --------------------------------------------------------------------------------------------------
# E24-A TEST output
# --------------------------------------------------------------------------------------------------

OUTPUT_DIR = (
    RUNS_ROOT
    / "FINAL_TEST_E24A"
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)


PREDICTIONS_PATH = (
    OUTPUT_DIR
    / "E24A_test_predictions.json"
)


RESULTS_PATH = (
    OUTPUT_DIR
    / "E24A_test_results.json"
)


SUMMARY_PATH = (
    OUTPUT_DIR
    / "E24A_test_summary.txt"
)


for path, label in [

    (
        E20_PRED_PATH,
        "Verified E20-A TEST predictions"
    ),

    (
        E20_GT_PATH,
        "7-class TEST GT"
    ),

    (
        MATCH_CACHE_PATH,
        "Corrected E23-D E20/E12 matched cache"
    ),

    (
        E23_CACHE_PATH,
        "Corrected E23-D E23-A TEST cache"
    ),

    (
        E23D_CONFIG_PATH,
        "Frozen E23-D validation configuration"
    ),

    (
        E24A_CONFIG_PATH,
        "Validation-selected E24-A configuration"
    ),
]:

    if not path.exists():

        raise FileNotFoundError(
            f"{label} not found:\n{path}"
        )


# ==================================================================================================
# 2. TAXONOMY
# ==================================================================================================

CLASS_NAMES = [
    "ecal",
    "hdpe",
    "mixed_plastic_rigid",
    "mixed_plastic_soft",
    "non_plastic",
    "pet",
    "pet_oil",
]


ECAL = 0
HDPE = 1
MR = 2
MS = 3
NON_PLASTIC = 4
PET = 5
PET_OIL = 6


PLASTIC_INDICES = [
    ECAL,
    HDPE,
    MR,
    MS,
    PET,
    PET_OIL,
]


MR_CATEGORY_ID = (
    MR + 1
)

MS_CATEGORY_ID = (
    MS + 1
)


# ==================================================================================================
# 3. LOAD FILES
# ==================================================================================================

print("=" * 120)
print("E24-A — FROZEN HELD-OUT TEST EVALUATION")
print("=" * 120)


with open(
    E20_PRED_PATH,
    "r",
    encoding="utf-8"
) as f:

    e20_predictions = json.load(f)


with open(
    MATCH_CACHE_PATH,
    "r",
    encoding="utf-8"
) as f:

    matched_cache = json.load(f)


with open(
    E23_CACHE_PATH,
    "r",
    encoding="utf-8"
) as f:

    e23_cache = json.load(f)


with open(
    E23D_CONFIG_PATH,
    "r",
    encoding="utf-8"
) as f:

    e23d_config = json.load(f)


with open(
    E24A_CONFIG_PATH,
    "r",
    encoding="utf-8"
) as f:

    e24a_config = json.load(f)


print()
print(
    f"E20-A predictions : "
    f"{len(e20_predictions):,}"
)

print(
    f"Matched cache     : "
    f"{len(matched_cache):,}"
)

print(
    f"E23-A cache       : "
    f"{len(e23_cache):,}"
)


# ==================================================================================================
# 4. HARD SAFETY CHECKS
# ==================================================================================================

if len(
    e20_predictions
) != 57487:

    raise RuntimeError(
        "Expected verified E20-A TEST prediction count = 57,487."
    )


if len(
    matched_cache
) != 57487:

    raise RuntimeError(
        "Matched cache does not contain 57,487 detections."
    )


# E23-A was run only on final E20 MR/MS candidates.

EXPECTED_E23_CANDIDATES = 17193


if len(
    e23_cache
) != EXPECTED_E23_CANDIDATES:

    raise RuntimeError(
        f"Expected E23-A TEST cache with "
        f"{EXPECTED_E23_CANDIDATES:,} entries, "
        f"found {len(e23_cache):,}."
    )


# ==================================================================================================
# 5. LOAD FROZEN E23-D THRESHOLDS
# ==================================================================================================

if e23d_config.get(
    "test_used"
) is not False:

    raise RuntimeError(
        "E23-D configuration is not validation-only."
    )


e23d_best = (
    e23d_config[
        "best_configuration"
    ]
)


MS_TO_MR_E23_MIN = float(
    e23d_best[
        "MS_to_MR"
    ][
        "e23_min"
    ]
)


MS_TO_MR_E20_MAX = float(
    e23d_best[
        "MS_to_MR"
    ][
        "e20_max"
    ]
)


MR_TO_MS_E23_MIN = float(
    e23d_best[
        "MR_to_MS"
    ][
        "e23_min"
    ]
)


MR_TO_MS_E20_MAX = float(
    e23d_best[
        "MR_to_MS"
    ][
        "e20_max"
    ]
)


if not (

    np.isclose(
        MS_TO_MR_E23_MIN,
        0.920
    )

    and

    np.isclose(
        MS_TO_MR_E20_MAX,
        0.90
    )

    and

    np.isclose(
        MR_TO_MS_E23_MIN,
        0.997
    )

    and

    np.isclose(
        MR_TO_MS_E20_MAX,
        0.95
    )
):

    raise RuntimeError(
        "Frozen E23-D thresholds do not match validation selection."
    )


# ==================================================================================================
# 6. LOAD FROZEN E24-A BETA
# ==================================================================================================

if e24a_config.get(
    "test_used"
) is not False:

    raise RuntimeError(
        "E24-A configuration is not validation-only."
    )


if e24a_config.get(
    "promote_to_test"
) is not True:

    raise RuntimeError(
        "E24-A was not marked eligible for TEST."
    )


BETA = float(
    e24a_config[
        "best_beta"
    ]
)


print()
print("=" * 120)
print("FROZEN CONFIGURATION")
print("=" * 120)

print()
print(
    f"MS -> MR : "
    f"E23 >= {MS_TO_MR_E23_MIN:.3f}, "
    f"E20 <= {MS_TO_MR_E20_MAX:.2f}"
)

print(
    f"MR -> MS : "
    f"E23 >= {MR_TO_MS_E23_MIN:.3f}, "
    f"E20 <= {MR_TO_MS_E20_MAX:.2f}"
)

print()
print(
    f"E24-A beta : "
    f"{BETA:.2f}"
)


if not np.isclose(
    BETA,
    0.35
):

    raise RuntimeError(
        f"Expected validation-selected beta = 0.35, "
        f"found {BETA}."
    )


# ==================================================================================================
# 7. BUILD LOOKUP FOR RECOVERED E20 CONFIDENCE
# ==================================================================================================

record_lookup = {}


for record in matched_cache:

    idx = int(
        record[
            "e20_index"
        ]
    )


    if idx in record_lookup:

        raise RuntimeError(
            f"Duplicate E20 index in matched cache: {idx}"
        )


    record_lookup[
        idx
    ] = record


if len(
    record_lookup
) != len(
    e20_predictions
):

    raise RuntimeError(
        "Matched-cache lookup size differs from E20 prediction count."
    )


# ==================================================================================================
# 8. COCO EVALUATION HELPERS
# ==================================================================================================

coco_gt = COCO(
    str(
        E20_GT_PATH
    )
)


def valid_mean(
    values
):

    values = np.asarray(
        values
    )


    valid = values[
        values > -1
    ]


    if valid.size == 0:

        return float(
            "nan"
        )


    return float(
        valid.mean()
    )


def evaluate_predictions(
    predictions
):

    coco_dt = coco_gt.loadRes(
        predictions
    )


    evaluator = COCOeval(
        coco_gt,
        coco_dt,
        "bbox"
    )


    evaluator.params.maxDets = [
        1,
        10,
        100,
    ]


    evaluator.evaluate()
    evaluator.accumulate()


    precision = (
        evaluator.eval[
            "precision"
        ]
    )


    recall = (
        evaluator.eval[
            "recall"
        ]
    )


    ious = (
        evaluator.params.iouThrs
    )


    idx50 = int(
        np.where(
            np.isclose(
                ious,
                0.50
            )
        )[0][0]
    )


    idx75 = int(
        np.where(
            np.isclose(
                ious,
                0.75
            )
        )[0][0]
    )


    overall = {

        "AP50_95":

            valid_mean(
                precision[
                    :,
                    :,
                    :,
                    0,
                    -1
                ]
            ),


        "AP50":

            valid_mean(
                precision[
                    idx50,
                    :,
                    :,
                    0,
                    -1
                ]
            ),


        "AP75":

            valid_mean(
                precision[
                    idx75,
                    :,
                    :,
                    0,
                    -1
                ]
            ),


        "AR100":

            valid_mean(
                recall[
                    :,
                    :,
                    0,
                    -1
                ]
            ),
    }


    class_metrics = {}


    for idx, name in enumerate(
        CLASS_NAMES
    ):

        class_metrics[
            name
        ] = {

            "AP50":

                valid_mean(
                    precision[
                        idx50,
                        :,
                        idx,
                        0,
                        -1
                    ]
                ),


            "AP50_95":

                valid_mean(
                    precision[
                        :,
                        :,
                        idx,
                        0,
                        -1
                    ]
                ),
        }


    six_plastic_ap50 = float(
        np.mean(
            [
                class_metrics[
                    CLASS_NAMES[
                        idx
                    ]
                ][
                    "AP50"
                ]

                for idx
                in PLASTIC_INDICES
            ]
        )
    )


    six_plastic_ap = float(
        np.mean(
            [
                class_metrics[
                    CLASS_NAMES[
                        idx
                    ]
                ][
                    "AP50_95"
                ]

                for idx
                in PLASTIC_INDICES
            ]
        )
    )


    mr_ms_mean_ap50 = float(
        np.mean(
            [
                class_metrics[
                    "mixed_plastic_rigid"
                ][
                    "AP50"
                ],

                class_metrics[
                    "mixed_plastic_soft"
                ][
                    "AP50"
                ],
            ]
        )
    )


    return {

        "overall":
            overall,

        "class_metrics":
            class_metrics,

        "six_plastic_mean_AP50":
            six_plastic_ap50,

        "six_plastic_mean_AP50_95":
            six_plastic_ap,

        "MR_MS_mean_AP50":
            mr_ms_mean_ap50,
    }


# ==================================================================================================
# 9. VERIFY ORIGINAL E20-A TEST BASELINE
# ==================================================================================================

print()
print("=" * 120)
print("VERIFYING E20-A TEST BASELINE")
print("=" * 120)


e20_eval = evaluate_predictions(
    e20_predictions
)


print()
print(
    f"AP50-95       : "
    f"{e20_eval['overall']['AP50_95'] * 100:.4f}%"
)

print(
    f"AP50          : "
    f"{e20_eval['overall']['AP50'] * 100:.4f}%"
)

print(
    f"Plastic AP50  : "
    f"{e20_eval['six_plastic_mean_AP50'] * 100:.4f}%"
)


if not (

    abs(
        e20_eval[
            "overall"
        ][
            "AP50_95"
        ]
        -
        0.462612
    )
    < 0.001

    and

    abs(
        e20_eval[
            "overall"
        ][
            "AP50"
        ]
        -
        0.623271
    )
    < 0.001

    and

    abs(
        e20_eval[
            "six_plastic_mean_AP50"
        ]
        -
        0.694353
    )
    < 0.001
):

    raise RuntimeError(
        "Verified E20-A TEST baseline was not reproduced."
    )


print()
print(
    "E20-A baseline verification PASSED."
)


# ==================================================================================================
# 10. RECONSTRUCT FROZEN E23-D TEST EXACTLY
#
# Scores remain original E20 scores.
# ==================================================================================================

e23d_predictions = [
    dict(
        p
    )

    for p
    in e20_predictions
]


proposal_counts = Counter()
accepted_counts = Counter()


accepted_records = []


for e20_idx, original in enumerate(
    e20_predictions
):

    current_idx = (
        int(
            original[
                "category_id"
            ]
        )
        - 1
    )


    if current_idx not in {
        MR,
        MS,
    }:

        continue


    cache_key = str(
        e20_idx
    )


    if cache_key not in e23_cache:

        raise RuntimeError(
            f"Missing E23-A TEST cache for E20 index {e20_idx}"
        )


    e23 = (
        e23_cache[
            cache_key
        ]
    )


    e23_idx = int(
        e23[
            "pred_global_idx"
        ]
    )


    e23_prob = float(
        e23[
            "top_prob"
        ]
    )


    e20_conf = float(
        record_lookup[
            e20_idx
        ][
            "e20_class_conf"
        ]
    )


    accepted_direction = None


    # ----------------------------------------------------------------------------------------------
    # MS -> MR
    # ----------------------------------------------------------------------------------------------

    if (
        current_idx == MS
        and
        e23_idx == MR
    ):

        proposal_counts[
            "MS->MR"
        ] += 1


        if (
            e23_prob
            >=
            MS_TO_MR_E23_MIN
            and
            e20_conf
            <=
            MS_TO_MR_E20_MAX
        ):

            e23d_predictions[
                e20_idx
            ][
                "category_id"
            ] = (
                MR + 1
            )


            accepted_counts[
                "MS->MR"
            ] += 1


            accepted_direction = (
                "MS->MR"
            )


    # ----------------------------------------------------------------------------------------------
    # MR -> MS
    # ----------------------------------------------------------------------------------------------

    elif (
        current_idx == MR
        and
        e23_idx == MS
    ):

        proposal_counts[
            "MR->MS"
        ] += 1


        if (
            e23_prob
            >=
            MR_TO_MS_E23_MIN
            and
            e20_conf
            <=
            MR_TO_MS_E20_MAX
        ):

            e23d_predictions[
                e20_idx
            ][
                "category_id"
            ] = (
                MS + 1
            )


            accepted_counts[
                "MR->MS"
            ] += 1


            accepted_direction = (
                "MR->MS"
            )


    if accepted_direction is not None:

        accepted_records.append(
            {
                "e20_index":
                    int(
                        e20_idx
                    ),

                "direction":
                    accepted_direction,

                "e20_score":
                    float(
                        original[
                            "score"
                        ]
                    ),

                "e23_prob":
                    e23_prob,

                "e20_conf":
                    e20_conf,
            }
        )


print()
print("=" * 120)
print("RECONSTRUCTED E23-D TEST ROUTING")
print("=" * 120)

print()
print(
    f"MS -> MR proposals : "
    f"{proposal_counts['MS->MR']:,}"
)

print(
    f"MR -> MS proposals : "
    f"{proposal_counts['MR->MS']:,}"
)

print()
print(
    f"MS -> MR accepted  : "
    f"{accepted_counts['MS->MR']:,}"
)

print(
    f"MR -> MS accepted  : "
    f"{accepted_counts['MR->MS']:,}"
)

print(
    f"Total accepted     : "
    f"{len(accepted_records):,}"
)


# These are the verified corrected E23-D TEST routing counts.

if not (

    accepted_counts[
        "MS->MR"
    ]
    == 691

    and

    accepted_counts[
        "MR->MS"
    ]
    == 311

    and

    len(
        accepted_records
    )
    == 1002
):

    raise RuntimeError(
        "E23-D TEST routing does not reproduce "
        "the verified 691 + 311 = 1,002 overrides."
    )


# ==================================================================================================
# 11. VERIFY RECONSTRUCTED E23-D TEST METRICS
# ==================================================================================================

print()
print("=" * 120)
print("VERIFYING RECONSTRUCTED E23-D TEST BASELINE")
print("=" * 120)


e23d_eval = evaluate_predictions(
    e23d_predictions
)


print()
print(
    f"AP50-95               : "
    f"{e23d_eval['overall']['AP50_95'] * 100:.4f}%"
)

print(
    f"AP50                  : "
    f"{e23d_eval['overall']['AP50'] * 100:.4f}%"
)

print(
    f"AP75                  : "
    f"{e23d_eval['overall']['AP75'] * 100:.4f}%"
)

print(
    f"AR100                 : "
    f"{e23d_eval['overall']['AR100'] * 100:.4f}%"
)

print(
    f"Six-plastic mean AP50: "
    f"{e23d_eval['six_plastic_mean_AP50'] * 100:.4f}%"
)

print(
    f"MR AP50               : "
    f"{e23d_eval['class_metrics']['mixed_plastic_rigid']['AP50'] * 100:.4f}%"
)

print(
    f"MS AP50               : "
    f"{e23d_eval['class_metrics']['mixed_plastic_soft']['AP50'] * 100:.4f}%"
)


# Exact corrected E23-D reference.

if not (

    abs(
        e23d_eval[
            "overall"
        ][
            "AP50_95"
        ]
        -
        0.462851
    )
    < 1e-5

    and

    abs(
        e23d_eval[
            "overall"
        ][
            "AP50"
        ]
        -
        0.623548
    )
    < 1e-5

    and

    abs(
        e23d_eval[
            "six_plastic_mean_AP50"
        ]
        -
        0.694677
    )
    < 1e-5
):

    raise RuntimeError(
        "\nCorrected E23-D TEST result was NOT reproduced.\n"
        "E24-A TEST scoring has NOT been applied."
    )


print()
print(
    "Corrected E23-D TEST reproduction PASSED."
)


# ==================================================================================================
# 12. BUILD E24-A
#
# IMPORTANT:
#
# Start from exact E23-D class decisions.
#
# Change score ONLY on the 1,002 accepted overrides:
#
#     E20_score ^ beta
#     *
#     E23_prob ^ (1-beta)
#
# ==================================================================================================

print()
print("=" * 120)
print("APPLYING FROZEN E24-A SCORE RECALIBRATION")
print("=" * 120)


e24a_predictions = [
    dict(
        p
    )

    for p
    in e23d_predictions
]


old_scores = []
new_scores = []


for record in accepted_records:

    idx = int(
        record[
            "e20_index"
        ]
    )


    e20_score = max(
        float(
            record[
                "e20_score"
            ]
        ),
        1e-12
    )


    e23_prob = max(
        float(
            record[
                "e23_prob"
            ]
        ),
        1e-12
    )


    new_score = (
        e20_score
        ** BETA
    ) * (
        e23_prob
        ** (
            1.0
            -
            BETA
        )
    )


    old_scores.append(
        e20_score
    )

    new_scores.append(
        new_score
    )


    e24a_predictions[
        idx
    ][
        "score"
    ] = float(
        new_score
    )


print()
print(
    f"Recalibrated predictions : "
    f"{len(accepted_records):,}"
)

print(
    f"Frozen beta              : "
    f"{BETA:.2f}"
)


print()
print(
    f"Mean original score      : "
    f"{np.mean(old_scores):.6f}"
)

print(
    f"Mean recalibrated score  : "
    f"{np.mean(new_scores):.6f}"
)


# ==================================================================================================
# 13. STRUCTURAL SAFETY CHECK
#
# E24-A MUST NOT alter:
#     image_id
#     bbox
#     category_id relative to E23-D
#
# It may change score ONLY for the accepted 1,002 overrides.
# ==================================================================================================

score_changes = 0
class_changes_vs_e23d = 0
bbox_changes = 0
image_changes = 0


accepted_indices = {
    int(
        x[
            "e20_index"
        ]
    )

    for x
    in accepted_records
}


for idx, (
    before,
    after
) in enumerate(
    zip(
        e23d_predictions,
        e24a_predictions
    )
):

    if (
        before[
            "image_id"
        ]
        !=
        after[
            "image_id"
        ]
    ):

        image_changes += 1


    if (
        before[
            "bbox"
        ]
        !=
        after[
            "bbox"
        ]
    ):

        bbox_changes += 1


    if (
        before[
            "category_id"
        ]
        !=
        after[
            "category_id"
        ]
    ):

        class_changes_vs_e23d += 1


    if not np.isclose(
        float(
            before[
                "score"
            ]
        ),
        float(
            after[
                "score"
            ]
        ),
        atol=0,
        rtol=0,
    ):

        score_changes += 1


        if idx not in accepted_indices:

            raise RuntimeError(
                f"Score changed for a non-accepted detection: "
                f"E20 index {idx}"
            )


print()
print("=" * 120)
print("E24-A STRUCTURAL SAFETY CHECK")
print("=" * 120)

print()
print(
    f"Image-ID changes vs E23-D : "
    f"{image_changes}"
)

print(
    f"BBox changes vs E23-D     : "
    f"{bbox_changes}"
)

print(
    f"Class changes vs E23-D    : "
    f"{class_changes_vs_e23d}"
)

print(
    f"Score changes vs E23-D    : "
    f"{score_changes:,}"
)


if (
    image_changes != 0
    or
    bbox_changes != 0
    or
    class_changes_vs_e23d != 0
):

    raise RuntimeError(
        "E24-A altered something other than confidence scores."
    )


if score_changes != len(
    accepted_records
):

    raise RuntimeError(
        "Number of changed scores does not equal accepted E23-D overrides."
    )


# ==================================================================================================
# 14. HELD-OUT TEST EVALUATION
# ==================================================================================================

print()
print("=" * 120)
print("E24-A — FROZEN HELD-OUT TEST COCO EVALUATION")
print("=" * 120)


e24a_eval = evaluate_predictions(
    e24a_predictions
)


# ==================================================================================================
# 15. RESULTS
# ==================================================================================================

print()
print("=" * 120)
print("E24-A — HELD-OUT TEST RESULTS")
print("=" * 120)


print()
print(
    f"AP50-95                : "
    f"{e24a_eval['overall']['AP50_95'] * 100:.4f}%"
)

print(
    f"AP50                   : "
    f"{e24a_eval['overall']['AP50'] * 100:.4f}%"
)

print(
    f"AP75                   : "
    f"{e24a_eval['overall']['AP75'] * 100:.4f}%"
)

print(
    f"AR100                  : "
    f"{e24a_eval['overall']['AR100'] * 100:.4f}%"
)


print()
print(
    f"Six-plastic mean AP50 : "
    f"{e24a_eval['six_plastic_mean_AP50'] * 100:.4f}%"
)

print(
    f"Six-plastic AP50-95   : "
    f"{e24a_eval['six_plastic_mean_AP50_95'] * 100:.4f}%"
)

print(
    f"MR/MS mean AP50       : "
    f"{e24a_eval['MR_MS_mean_AP50'] * 100:.4f}%"
)


# ==================================================================================================
# 16. CLASS-WISE RESULTS
# ==================================================================================================

print()
print("=" * 120)
print("E24-A — CLASS-WISE TEST RESULTS")
print("=" * 120)


print(
    f"\n"
    f"{'Class':30s}"
    f"{'AP50':>15s}"
    f"{'AP50-95':>15s}"
)

print(
    "-" * 60
)


for class_name in CLASS_NAMES:

    values = (
        e24a_eval[
            "class_metrics"
        ][
            class_name
        ]
    )


    print(
        f"{class_name:30s}"
        f"{values['AP50'] * 100:14.2f}%"
        f"{values['AP50_95'] * 100:14.2f}%"
    )


# ==================================================================================================
# 17. DELTA VS E23-D
# ==================================================================================================

delta_vs_e23d = {

    "AP50_95_pp":

        (
            e24a_eval[
                "overall"
            ][
                "AP50_95"
            ]

            -

            e23d_eval[
                "overall"
            ][
                "AP50_95"
            ]
        )
        * 100,


    "AP50_pp":

        (
            e24a_eval[
                "overall"
            ][
                "AP50"
            ]

            -

            e23d_eval[
                "overall"
            ][
                "AP50"
            ]
        )
        * 100,


    "AP75_pp":

        (
            e24a_eval[
                "overall"
            ][
                "AP75"
            ]

            -

            e23d_eval[
                "overall"
            ][
                "AP75"
            ]
        )
        * 100,


    "AR100_pp":

        (
            e24a_eval[
                "overall"
            ][
                "AR100"
            ]

            -

            e23d_eval[
                "overall"
            ][
                "AR100"
            ]
        )
        * 100,


    "six_plastic_AP50_pp":

        (
            e24a_eval[
                "six_plastic_mean_AP50"
            ]

            -

            e23d_eval[
                "six_plastic_mean_AP50"
            ]
        )
        * 100,


    "MR_AP50_pp":

        (
            e24a_eval[
                "class_metrics"
            ][
                "mixed_plastic_rigid"
            ][
                "AP50"
            ]

            -

            e23d_eval[
                "class_metrics"
            ][
                "mixed_plastic_rigid"
            ][
                "AP50"
            ]
        )
        * 100,


    "MS_AP50_pp":

        (
            e24a_eval[
                "class_metrics"
            ][
                "mixed_plastic_soft"
            ][
                "AP50"
            ]

            -

            e23d_eval[
                "class_metrics"
            ][
                "mixed_plastic_soft"
            ][
                "AP50"
            ]
        )
        * 100,


    "MR_MS_mean_AP50_pp":

        (
            e24a_eval[
                "MR_MS_mean_AP50"
            ]

            -

            e23d_eval[
                "MR_MS_mean_AP50"
            ]
        )
        * 100,
}


print()
print("=" * 120)
print("DELTA VS CORRECTED E23-D TEST")
print("=" * 120)

print()


for metric, value in (
    delta_vs_e23d.items()
):

    print(
        f"{metric:30s}: "
        f"{value:+.4f} pp"
    )


# ==================================================================================================
# 18. UNCHANGED-CLASS CHECK
#
# E24-A modifies scores only for MR/MS-routed detections.
#
# ECAL, HDPE, NP, PET, PET Oil must remain identical.
# ==================================================================================================

print()
print("=" * 120)
print("UNCHANGED-CLASS SANITY CHECK")
print("=" * 120)

print()


for class_name in [
    "ecal",
    "hdpe",
    "non_plastic",
    "pet",
    "pet_oil",
]:

    before = (
        e23d_eval[
            "class_metrics"
        ][
            class_name
        ][
            "AP50"
        ]
    )


    after = (
        e24a_eval[
            "class_metrics"
        ][
            class_name
        ][
            "AP50"
        ]
    )


    delta = (
        after
        -
        before
    ) * 100


    print(
        f"{class_name:25s}: "
        f"{delta:+.6f} pp"
    )


# ==================================================================================================
# 19. COMPARISON TABLE
# ==================================================================================================

SORTWASTE_PLASTIC_AP50 = (
    69.3800
)


E21B = {
    "AP50_95": 46.2881,
    "AP50": 62.3518,
    "plastic": 69.4212,
    "MR": 48.70,
    "MS": 42.53,
    "MRMS": 45.6119,
}


print()
print("=" * 120)
print("FINAL COMPARISON")
print("=" * 120)


print(
    f"\n"
    f"{'System':12s}"
    f"{'AP50-95':>12s}"
    f"{'AP50':>12s}"
    f"{'Plastic':>12s}"
    f"{'MR':>10s}"
    f"{'MS':>10s}"
    f"{'MR/MS':>12s}"
)


print(
    "-" * 80
)


print(
    f"{'E20-A':12s}"
    f"{e20_eval['overall']['AP50_95'] * 100:11.4f}%"
    f"{e20_eval['overall']['AP50'] * 100:11.4f}%"
    f"{e20_eval['six_plastic_mean_AP50'] * 100:11.4f}%"
    f"{e20_eval['class_metrics']['mixed_plastic_rigid']['AP50'] * 100:9.2f}%"
    f"{e20_eval['class_metrics']['mixed_plastic_soft']['AP50'] * 100:9.2f}%"
    f"{e20_eval['MR_MS_mean_AP50'] * 100:11.4f}%"
)


print(
    f"{'E21-B':12s}"
    f"{E21B['AP50_95']:11.4f}%"
    f"{E21B['AP50']:11.4f}%"
    f"{E21B['plastic']:11.4f}%"
    f"{E21B['MR']:9.2f}%"
    f"{E21B['MS']:9.2f}%"
    f"{E21B['MRMS']:11.4f}%"
)


print(
    f"{'E23-D':12s}"
    f"{e23d_eval['overall']['AP50_95'] * 100:11.4f}%"
    f"{e23d_eval['overall']['AP50'] * 100:11.4f}%"
    f"{e23d_eval['six_plastic_mean_AP50'] * 100:11.4f}%"
    f"{e23d_eval['class_metrics']['mixed_plastic_rigid']['AP50'] * 100:9.2f}%"
    f"{e23d_eval['class_metrics']['mixed_plastic_soft']['AP50'] * 100:9.2f}%"
    f"{e23d_eval['MR_MS_mean_AP50'] * 100:11.4f}%"
)


print(
    f"{'E24-A':12s}"
    f"{e24a_eval['overall']['AP50_95'] * 100:11.4f}%"
    f"{e24a_eval['overall']['AP50'] * 100:11.4f}%"
    f"{e24a_eval['six_plastic_mean_AP50'] * 100:11.4f}%"
    f"{e24a_eval['class_metrics']['mixed_plastic_rigid']['AP50'] * 100:9.2f}%"
    f"{e24a_eval['class_metrics']['mixed_plastic_soft']['AP50'] * 100:9.2f}%"
    f"{e24a_eval['MR_MS_mean_AP50'] * 100:11.4f}%"
)


# ==================================================================================================
# 20. SORTWASTE COMPARISON
# ==================================================================================================

sortwaste_delta = (
    e24a_eval[
        "six_plastic_mean_AP50"
    ]
    * 100
    -
    SORTWASTE_PLASTIC_AP50
)


gap_to_70 = (
    70.0
    -
    e24a_eval[
        "six_plastic_mean_AP50"
    ]
    * 100
)


print()
print("=" * 120)
print("ALIGNED SIX-PLASTIC AP50 COMPARISON")
print("=" * 120)

print()
print(
    f"SortWaste YOLOv11 : "
    f"{SORTWASTE_PLASTIC_AP50:.4f}%"
)

print(
    f"E24-A             : "
    f"{e24a_eval['six_plastic_mean_AP50'] * 100:.4f}%"
)

print(
    f"Difference        : "
    f"{sortwaste_delta:+.4f} pp"
)

print()
print(
    f"Gap to 70.0000%   : "
    f"{gap_to_70:+.4f} pp"
)


# ==================================================================================================
# 21. SAVE PREDICTIONS
# ==================================================================================================

with open(
    PREDICTIONS_PATH,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        e24a_predictions,
        f
    )


# ==================================================================================================
# 22. SAVE RESULTS
# ==================================================================================================

result_payload = {

    "experiment":
        "E24-A",

    "split":
        "held-out TEST",

    "test_used_for_selection":
        False,

    "base_pipeline":
        "E23-D",

    "validation_selected_beta":
        BETA,

    "score_rule":
        "E20_score^beta * E23_confidence^(1-beta)",

    "routing": {

        "MS_to_MR": {
            "e23_min":
                MS_TO_MR_E23_MIN,

            "e20_max":
                MS_TO_MR_E20_MAX,

            "accepted":
                int(
                    accepted_counts[
                        "MS->MR"
                    ]
                ),
        },

        "MR_to_MS": {
            "e23_min":
                MR_TO_MS_E23_MIN,

            "e20_max":
                MR_TO_MS_E20_MAX,

            "accepted":
                int(
                    accepted_counts[
                        "MR->MS"
                    ]
                ),
        },
    },

    "E20A_baseline":
        e20_eval,

    "E23D_baseline":
        e23d_eval,

    "E24A":
        e24a_eval,

    "delta_vs_E23D_pp":
        delta_vs_e23d,

    "sortwaste_six_plastic_AP50":
        SORTWASTE_PLASTIC_AP50,

    "delta_vs_sortwaste_pp":
        sortwaste_delta,

    "gap_to_70_pp":
        gap_to_70,
}


with open(
    RESULTS_PATH,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        result_payload,
        f,
        indent=2
    )


# ==================================================================================================
# 23. SAVE SUMMARY
# ==================================================================================================

summary_lines = [

    "E24-A — Frozen Held-Out Test Evaluation",
    "=" * 105,

    "",

    "VALIDATION-SELECTED CONFIGURATION",

    f"Beta                  : {BETA:.2f}",

    (
        f"MS -> MR             : "
        f"E23 >= {MS_TO_MR_E23_MIN:.3f}, "
        f"E20 <= {MS_TO_MR_E20_MAX:.2f}"
    ),

    (
        f"MR -> MS             : "
        f"E23 >= {MR_TO_MS_E23_MIN:.3f}, "
        f"E20 <= {MR_TO_MS_E20_MAX:.2f}"
    ),

    "",

    f"Accepted MS -> MR    : {accepted_counts['MS->MR']:,}",

    f"Accepted MR -> MS    : {accepted_counts['MR->MS']:,}",

    "",

    "E24-A TEST RESULTS",

    f"AP50-95              : {e24a_eval['overall']['AP50_95'] * 100:.4f}%",

    f"AP50                 : {e24a_eval['overall']['AP50'] * 100:.4f}%",

    f"AP75                 : {e24a_eval['overall']['AP75'] * 100:.4f}%",

    f"AR100                : {e24a_eval['overall']['AR100'] * 100:.4f}%",

    f"Six-plastic AP50     : {e24a_eval['six_plastic_mean_AP50'] * 100:.4f}%",

    f"Six-plastic AP50-95  : {e24a_eval['six_plastic_mean_AP50_95'] * 100:.4f}%",

    f"MR/MS mean AP50      : {e24a_eval['MR_MS_mean_AP50'] * 100:.4f}%",

    "",

    f"SortWaste plastic AP50 : {SORTWASTE_PLASTIC_AP50:.4f}%",

    f"Difference             : {sortwaste_delta:+.4f} pp",

    f"Gap to 70%             : {gap_to_70:+.4f} pp",

    "",

    "Delta vs E23-D:",
]


for metric, value in (
    delta_vs_e23d.items()
):

    summary_lines.append(
        f"  {metric:28s}: "
        f"{value:+.4f} pp"
    )


with open(
    SUMMARY_PATH,
    "w",
    encoding="utf-8"
) as f:

    f.write(
        "\n".join(
            summary_lines
        )
    )


# ==================================================================================================
# 24. COMPLETE
# ==================================================================================================

print()
print("=" * 120)
print("E24-A FROZEN HELD-OUT TEST COMPLETE")
print("=" * 120)

print()
print(
    f"Predictions : "
    f"{PREDICTIONS_PATH}"
)

print(
    f"Results     : "
    f"{RESULTS_PATH}"
)

print(
    f"Summary     : "
    f"{SUMMARY_PATH}"
)

E24-A — FROZEN HELD-OUT TEST EVALUATION

E20-A predictions : 57,487
Matched cache     : 57,487
E23-A cache       : 17,193

FROZEN CONFIGURATION

MS -> MR : E23 >= 0.920, E20 <= 0.90
MR -> MS : E23 >= 0.997, E20 <= 0.95

E24-A beta : 0.35
loading annotations into memory...
Done (t=0.05s)
creating index...
index created!

VERIFYING E20-A TEST BASELINE
Loading and preparing results...
DONE (t=0.05s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=3.84s).
Accumulating evaluation results...
DONE (t=0.58s).

AP50-95       : 46.2612%
AP50          : 62.3271%
Plastic AP50  : 69.4353%

E20-A baseline verification PASSED.

RECONSTRUCTED E23-D TEST ROUTING

MS -> MR proposals : 2,166
MR -> MS proposals : 2,407

MS -> MR accepted  : 691
MR -> MS accepted  : 311
Total accepted     : 1,002

VERIFYING RECONSTRUCTED E23-D TEST BASELINE
Loading and preparing results...
DONE (t=0.03s)
creating index...
index created!
Running per image evaluation..

## E24-B — Validation-Only Class-Specific Confidence Fusion for Mixed Rigid and Mixed Soft

In [42]:
# E24-B — VALIDATION-ONLY CLASS-SPECIFIC MR/MS CONFIDENCE FUSION
#
# PURPOSE
# --------------------------------------------------------------------------------------------------
# Starting point:
#   Frozen E23-D routing.
#
# E23-D class decisions remain EXACTLY unchanged.
#
# Search ONLY separate confidence-fusion weights for final:
#
#   Mixed Rigid:
#       score = YOLO_conf ^ alpha_MR
#               *
#               E20_classifier_component ^ (1 - alpha_MR)
#
#   Mixed Soft:
#       score = YOLO_conf ^ alpha_MS
#               *
#               E20_classifier_component ^ (1 - alpha_MS)
#
# All other classes:
#       original E20/E23-D score remains untouched.
#
#
# IMPORTANT
# --------------------------------------------------------------------------------------------------
# Original E20-A score:
#
#       E20_score =
#           YOLO_conf ^ 0.70
#           *
#           classifier_prob ^ 0.30
#
# Therefore classifier_prob can be recovered EXACTLY:
#
#       classifier_prob =
#           (E20_score / YOLO_conf^0.70)^(1/0.30)
#
#
# alpha_MR = 0.70 and alpha_MS = 0.70
# MUST reproduce frozen E23-D validation exactly.
#
#
# VALIDATION ONLY
# NO TEST
# NO MODEL INFERENCE
# NO RETRAINING
# NO ROUTING SEARCH
# ==================================================================================================

import json
from pathlib import Path
from collections import Counter

import numpy as np

from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval


# ==================================================================================================
# 1. PATHS
# ==================================================================================================

BASE = Path(
    r"C:\Users\varda\Documents\_My Computer\COMMON_space\Learning"
    r"\Upgrad_EPGP_MS\upgrad_IIITB_MS_course\MS_LJMU_Material"
)

THESIS_CODE = (
    BASE
    / "Thesis_Code"
)

RUNS_ROOT = (
    THESIS_CODE
    / "runs"
    / "sortwaste"
)


CACHE_PATH = (
    RUNS_ROOT
    / "E23B_E20A_binary_MRMS_routing"
    / "E23B_cached_predictions_with_E23A.json"
)


E23D_CONFIG_PATH = (
    RUNS_ROOT
    / "E23D_confidence_contrastive_MRMS"
    / "E23D_best_configuration.json"
)


GT_PATH = (
    RUNS_ROOT
    / "E18D_class_selective_convnext"
    / "E18D_val_gt_7class.json"
)


OUTPUT_DIR = (
    RUNS_ROOT
    / "E24B_class_specific_MRMS_fusion"
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)


GRID_PATH = (
    OUTPUT_DIR
    / "E24B_grid_results.json"
)

BEST_CONFIG_PATH = (
    OUTPUT_DIR
    / "E24B_best_configuration.json"
)

BEST_PRED_PATH = (
    OUTPUT_DIR
    / "E24B_best_predictions.json"
)

SUMMARY_PATH = (
    OUTPUT_DIR
    / "E24B_summary.txt"
)


for path in [
    CACHE_PATH,
    E23D_CONFIG_PATH,
    GT_PATH,
]:

    if not path.exists():

        raise FileNotFoundError(
            f"Required file not found:\n{path}"
        )


# ==================================================================================================
# 2. CLASSES
# ==================================================================================================

CLASS_NAMES = [
    "ecal",
    "hdpe",
    "mixed_plastic_rigid",
    "mixed_plastic_soft",
    "non_plastic",
    "pet",
    "pet_oil",
]


ECAL = 0
HDPE = 1
MR = 2
MS = 3
NON_PLASTIC = 4
PET = 5
PET_OIL = 6


PLASTIC_INDICES = [
    ECAL,
    HDPE,
    MR,
    MS,
    PET,
    PET_OIL,
]


# ==================================================================================================
# 3. LOAD INPUT
# ==================================================================================================

with open(
    CACHE_PATH,
    "r",
    encoding="utf-8"
) as f:

    cache = json.load(f)


with open(
    E23D_CONFIG_PATH,
    "r",
    encoding="utf-8"
) as f:

    e23d_config = json.load(f)


print("=" * 115)
print("E24-B — VALIDATION-ONLY CLASS-SPECIFIC MR/MS CONFIDENCE FUSION")
print("=" * 115)

print()
print(
    f"Cache entries : "
    f"{len(cache):,}"
)

print(
    f"Validation GT : "
    f"{GT_PATH}"
)

print(
    f"Output        : "
    f"{OUTPUT_DIR}"
)


if len(cache) != 55605:

    raise RuntimeError(
        "Expected 55,605 validation detections."
    )


# ==================================================================================================
# 4. FROZEN E23-D CONFIG
# ==================================================================================================

if e23d_config.get(
    "test_used"
) is not False:

    raise RuntimeError(
        "E23-D configuration is not validation-only."
    )


best_e23d = (
    e23d_config[
        "best_configuration"
    ]
)


MS_TO_MR_E23_MIN = float(
    best_e23d[
        "MS_to_MR"
    ][
        "e23_min"
    ]
)

MS_TO_MR_E20_MAX = float(
    best_e23d[
        "MS_to_MR"
    ][
        "e20_max"
    ]
)

MR_TO_MS_E23_MIN = float(
    best_e23d[
        "MR_to_MS"
    ][
        "e23_min"
    ]
)

MR_TO_MS_E20_MAX = float(
    best_e23d[
        "MR_to_MS"
    ][
        "e20_max"
    ]
)


if not (
    np.isclose(
        MS_TO_MR_E23_MIN,
        0.920
    )
    and
    np.isclose(
        MS_TO_MR_E20_MAX,
        0.90
    )
    and
    np.isclose(
        MR_TO_MS_E23_MIN,
        0.997
    )
    and
    np.isclose(
        MR_TO_MS_E20_MAX,
        0.95
    )
):

    raise RuntimeError(
        "Frozen E23-D thresholds do not match."
    )


print()
print(
    f"MS -> MR : "
    f"E23 >= {MS_TO_MR_E23_MIN:.3f}, "
    f"E20 <= {MS_TO_MR_E20_MAX:.2f}"
)

print(
    f"MR -> MS : "
    f"E23 >= {MR_TO_MS_E23_MIN:.3f}, "
    f"E20 <= {MR_TO_MS_E20_MAX:.2f}"
)


# ==================================================================================================
# 5. E23 PARSER
# ==================================================================================================

def extract_e23(
    item
):

    info = item.get(
        "e23"
    )


    if info is None:

        return None


    # Your cache format:
    #
    # {
    #   'probs': [...],
    #   'local_idx': ...,
    #   'global_idx': ...,
    #   'top_prob': ...
    # }

    if "global_idx" in info:

        pred_idx = int(
            info[
                "global_idx"
            ]
        )

    elif "local_idx" in info:

        local_idx = int(
            info[
                "local_idx"
            ]
        )

        pred_idx = (
            MR
            if local_idx == 0
            else MS
        )

    else:

        probs = np.asarray(
            info[
                "probs"
            ],
            dtype=np.float64
        )

        pred_idx = (
            MR
            if int(
                np.argmax(
                    probs
                )
            )
            == 0
            else MS
        )


    if "top_prob" in info:

        top_prob = float(
            info[
                "top_prob"
            ]
        )

    else:

        top_prob = float(
            np.max(
                np.asarray(
                    info[
                        "probs"
                    ],
                    dtype=np.float64
                )
            )
        )


    return {
        "pred_idx":
            pred_idx,

        "top_prob":
            top_prob,
    }


# ==================================================================================================
# 6. E20 CURRENT-CLASS CONFIDENCE FOR E23-D ROUTING
# ==================================================================================================

def get_e20_current_class_confidence(
    item
):

    current_idx = int(
        item[
            "_e20_class_idx"
        ]
    )


    source = str(
        item.get(
            "_e20_source",
            ""
        )
    ).lower()


    conv = item.get(
        "conv"
    )


    if (
        source in {
            "e18",
            "convnext",
        }
        and
        conv is not None
        and
        int(
            conv[
                "conv_global_idx"
            ]
        )
        ==
        current_idx
    ):

        return float(
            conv[
                "conv_top_prob"
            ]
        )


    mn_probs = np.asarray(
        item[
            "mn_probs"
        ],
        dtype=np.float64
    )


    return float(
        mn_probs[
            current_idx
        ]
    )


# ==================================================================================================
# 7. RECONSTRUCT FROZEN E23-D ROUTING
# ==================================================================================================

routing_records = []

proposal_counts = Counter()
accepted_counts = Counter()


for idx, item in enumerate(
    cache
):

    original_idx = int(
        item[
            "_e20_class_idx"
        ]
    )


    final_idx = (
        original_idx
    )


    e20_conf = (
        get_e20_current_class_confidence(
            item
        )
    )


    e23 = extract_e23(
        item
    )


    accepted_direction = None


    if (
        original_idx in {
            MR,
            MS,
        }
        and
        e23 is not None
    ):

        e23_idx = int(
            e23[
                "pred_idx"
            ]
        )

        e23_prob = float(
            e23[
                "top_prob"
            ]
        )


        # MS -> MR
        if (
            original_idx == MS
            and
            e23_idx == MR
        ):

            proposal_counts[
                "MS->MR"
            ] += 1


            if (
                e23_prob
                >=
                MS_TO_MR_E23_MIN
                and
                e20_conf
                <=
                MS_TO_MR_E20_MAX
            ):

                final_idx = (
                    MR
                )

                accepted_direction = (
                    "MS->MR"
                )

                accepted_counts[
                    "MS->MR"
                ] += 1


        # MR -> MS
        elif (
            original_idx == MR
            and
            e23_idx == MS
        ):

            proposal_counts[
                "MR->MS"
            ] += 1


            if (
                e23_prob
                >=
                MR_TO_MS_E23_MIN
                and
                e20_conf
                <=
                MR_TO_MS_E20_MAX
            ):

                final_idx = (
                    MS
                )

                accepted_direction = (
                    "MR->MS"
                )

                accepted_counts[
                    "MR->MS"
                ] += 1


    # ----------------------------------------------------------------------------------------------
    # Recover exact classifier component used in E20 score.
    #
    # E20_score = yolo^0.70 * classifier^0.30
    # ----------------------------------------------------------------------------------------------

    e20_score = max(
        float(
            item[
                "_e20_score"
            ]
        ),
        1e-12
    )


    yolo_conf = max(
        float(
            item[
                "yolo_conf"
            ]
        ),
        1e-12
    )


    classifier_component = (
        e20_score
        /
        (
            yolo_conf
            ** 0.70
        )
    ) ** (
        1.0
        /
        0.30
    )


    classifier_component = float(
        np.clip(
            classifier_component,
            0.0,
            1.0
        )
    )


    routing_records.append(
        {
            "cache_index":
                idx,

            "original_idx":
                original_idx,

            "final_idx":
                final_idx,

            "e20_score":
                e20_score,

            "yolo_conf":
                yolo_conf,

            "classifier_component":
                classifier_component,

            "accepted_direction":
                accepted_direction,
        }
    )


print()
print("=" * 115)
print("FROZEN E23-D ROUTING")
print("=" * 115)

print()
print(
    f"MS -> MR proposals : "
    f"{proposal_counts['MS->MR']:,}"
)

print(
    f"MR -> MS proposals : "
    f"{proposal_counts['MR->MS']:,}"
)

print()
print(
    f"MS -> MR accepted  : "
    f"{accepted_counts['MS->MR']:,}"
)

print(
    f"MR -> MS accepted  : "
    f"{accepted_counts['MR->MS']:,}"
)


if not (
    accepted_counts[
        "MS->MR"
    ]
    == 606
    and
    accepted_counts[
        "MR->MS"
    ]
    == 196
):

    raise RuntimeError(
        "E23-D routing reconstruction failed."
    )


# ==================================================================================================
# 8. BUILD PREDICTIONS
# ==================================================================================================

def build_predictions(
    alpha_mr,
    alpha_ms
):

    predictions = []


    for item, route in zip(
        cache,
        routing_records
    ):

        final_idx = int(
            route[
                "final_idx"
            ]
        )


        # Original E20/E23-D score.
        score = float(
            route[
                "e20_score"
            ]
        )


        # ------------------------------------------------------------------------------------------
        # E24-B changes confidence ONLY for detections whose FINAL class is MR or MS.
        # ------------------------------------------------------------------------------------------

        if final_idx == MR:

            alpha = float(
                alpha_mr
            )


            score = (
                route[
                    "yolo_conf"
                ]
                ** alpha
            ) * (
                max(
                    route[
                        "classifier_component"
                    ],
                    1e-12
                )
                ** (
                    1.0
                    -
                    alpha
                )
            )


        elif final_idx == MS:

            alpha = float(
                alpha_ms
            )


            score = (
                route[
                    "yolo_conf"
                ]
                ** alpha
            ) * (
                max(
                    route[
                        "classifier_component"
                    ],
                    1e-12
                )
                ** (
                    1.0
                    -
                    alpha
                )
            )


        x1, y1, x2, y2 = [
            float(v)

            for v
            in item[
                "bbox"
            ]
        ]


        width = (
            x2 - x1
        )

        height = (
            y2 - y1
        )


        if (
            width <= 0
            or
            height <= 0
        ):

            continue


        predictions.append(
            {
                "image_id":
                    int(
                        item[
                            "image_id"
                        ]
                    ),

                "category_id":
                    int(
                        final_idx
                        + 1
                    ),

                "bbox": [
                    x1,
                    y1,
                    width,
                    height,
                ],

                "score":
                    float(
                        score
                    ),
            }
        )


    return predictions


# ==================================================================================================
# 9. COCO EVALUATION
# ==================================================================================================

coco_gt = COCO(
    str(
        GT_PATH
    )
)


def valid_mean(
    values
):

    values = np.asarray(
        values
    )


    valid = values[
        values > -1
    ]


    if valid.size == 0:

        return float(
            "nan"
        )


    return float(
        valid.mean()
    )


def evaluate_predictions(
    predictions
):

    coco_dt = coco_gt.loadRes(
        predictions
    )


    evaluator = COCOeval(
        coco_gt,
        coco_dt,
        "bbox"
    )


    evaluator.params.maxDets = [
        1,
        10,
        100,
    ]


    evaluator.evaluate()
    evaluator.accumulate()


    precision = (
        evaluator.eval[
            "precision"
        ]
    )

    recall = (
        evaluator.eval[
            "recall"
        ]
    )


    ious = (
        evaluator.params.iouThrs
    )


    idx50 = int(
        np.where(
            np.isclose(
                ious,
                0.50
            )
        )[0][0]
    )


    idx75 = int(
        np.where(
            np.isclose(
                ious,
                0.75
            )
        )[0][0]
    )


    overall = {

        "AP50_95":

            valid_mean(
                precision[
                    :,
                    :,
                    :,
                    0,
                    -1
                ]
            ),


        "AP50":

            valid_mean(
                precision[
                    idx50,
                    :,
                    :,
                    0,
                    -1
                ]
            ),


        "AP75":

            valid_mean(
                precision[
                    idx75,
                    :,
                    :,
                    0,
                    -1
                ]
            ),


        "AR100":

            valid_mean(
                recall[
                    :,
                    :,
                    0,
                    -1
                ]
            ),
    }


    class_metrics = {}


    for class_idx, class_name in enumerate(
        CLASS_NAMES
    ):

        class_metrics[
            class_name
        ] = {

            "AP50":

                valid_mean(
                    precision[
                        idx50,
                        :,
                        class_idx,
                        0,
                        -1
                    ]
                ),


            "AP50_95":

                valid_mean(
                    precision[
                        :,
                        :,
                        class_idx,
                        0,
                        -1
                    ]
                ),
        }


    six_plastic = float(
        np.mean(
            [
                class_metrics[
                    CLASS_NAMES[
                        idx
                    ]
                ][
                    "AP50"
                ]

                for idx
                in PLASTIC_INDICES
            ]
        )
    )


    mr_ms = float(
        np.mean(
            [
                class_metrics[
                    "mixed_plastic_rigid"
                ][
                    "AP50"
                ],

                class_metrics[
                    "mixed_plastic_soft"
                ][
                    "AP50"
                ],
            ]
        )
    )


    return {

        "overall":
            overall,

        "class_metrics":
            class_metrics,

        "six_plastic_mean_AP50":
            six_plastic,

        "MR_MS_mean_AP50":
            mr_ms,
    }


# ==================================================================================================
# 10. BASELINE CHECK — αMR = αMS = 0.70
# ==================================================================================================

print()
print("=" * 115)
print("RECONSTRUCTING FROZEN E23-D BASELINE")
print("=" * 115)


baseline_predictions = build_predictions(
    alpha_mr=0.70,
    alpha_ms=0.70,
)


baseline_eval = evaluate_predictions(
    baseline_predictions
)


print()
print(
    f"Six-plastic AP50 : "
    f"{baseline_eval['six_plastic_mean_AP50'] * 100:.4f}%"
)

print(
    f"MR AP50          : "
    f"{baseline_eval['class_metrics']['mixed_plastic_rigid']['AP50'] * 100:.4f}%"
)

print(
    f"MS AP50          : "
    f"{baseline_eval['class_metrics']['mixed_plastic_soft']['AP50'] * 100:.4f}%"
)

print(
    f"MR/MS AP50       : "
    f"{baseline_eval['MR_MS_mean_AP50'] * 100:.4f}%"
)

print(
    f"AP50-95          : "
    f"{baseline_eval['overall']['AP50_95'] * 100:.4f}%"
)


REF_PLASTIC = float(
    best_e23d[
        "six_plastic_mean_AP50"
    ]
)

REF_MRMS = float(
    best_e23d[
        "MR_MS_mean_AP50"
    ]
)

REF_AP = float(
    best_e23d[
        "overall"
    ][
        "AP50_95"
    ]
)


if not (
    abs(
        baseline_eval[
            "six_plastic_mean_AP50"
        ]
        -
        REF_PLASTIC
    )
    < 1e-6

    and

    abs(
        baseline_eval[
            "MR_MS_mean_AP50"
        ]
        -
        REF_MRMS
    )
    < 1e-6

    and

    abs(
        baseline_eval[
            "overall"
        ][
            "AP50_95"
        ]
        -
        REF_AP
    )
    < 1e-6
):

    raise RuntimeError(
        "\nE23-D baseline reproduction FAILED.\n"
        "E24-B search has NOT started."
    )


print()
print(
    "E23-D validation reproduction PASSED."
)


# ==================================================================================================
# 11. ALPHA GRID
# ==================================================================================================

ALPHAS = [
    round(
        x,
        2
    )

    for x in np.arange(
        0.40,
        0.9001,
        0.05
    )
]


TOTAL = (
    len(
        ALPHAS
    )
    *
    len(
        ALPHAS
    )
)


print()
print("=" * 115)
print("RUNNING E24-B VALIDATION GRID SEARCH")
print("=" * 115)

print()
print(
    f"MR alpha candidates : "
    f"{ALPHAS}"
)

print(
    f"MS alpha candidates : "
    f"{ALPHAS}"
)

print(
    f"Total combinations  : "
    f"{TOTAL}"
)


all_results = []

counter = 0


for alpha_mr in ALPHAS:

    for alpha_ms in ALPHAS:

        counter += 1


        predictions = build_predictions(
            alpha_mr=alpha_mr,
            alpha_ms=alpha_ms,
        )


        evaluation = evaluate_predictions(
            predictions
        )


        result = {

            "alpha_MR":
                float(
                    alpha_mr
                ),

            "alpha_MS":
                float(
                    alpha_ms
                ),

            "six_plastic_mean_AP50":
                evaluation[
                    "six_plastic_mean_AP50"
                ],

            "MR_MS_mean_AP50":
                evaluation[
                    "MR_MS_mean_AP50"
                ],

            "overall":
                evaluation[
                    "overall"
                ],

            "class_metrics":
                evaluation[
                    "class_metrics"
                ],
        }


        all_results.append(
            result
        )


        print(
            f"[{counter:03d}/{TOTAL:03d}] "
            f"MR α={alpha_mr:.2f} "
            f"MS α={alpha_ms:.2f} | "
            f"Plastic="
            f"{evaluation['six_plastic_mean_AP50']*100:7.4f}% "
            f"MR="
            f"{evaluation['class_metrics']['mixed_plastic_rigid']['AP50']*100:7.3f}% "
            f"MS="
            f"{evaluation['class_metrics']['mixed_plastic_soft']['AP50']*100:7.3f}% "
            f"MR/MS="
            f"{evaluation['MR_MS_mean_AP50']*100:7.3f}% "
            f"AP="
            f"{evaluation['overall']['AP50_95']*100:7.4f}%"
        )


# ==================================================================================================
# 12. RANK
# ==================================================================================================

ranked = sorted(

    all_results,

    key=lambda r: (

        r[
            "six_plastic_mean_AP50"
        ],

        r[
            "MR_MS_mean_AP50"
        ],

        r[
            "overall"
        ][
            "AP50_95"
        ],
    ),

    reverse=True,
)


best = ranked[
    0
]


# ==================================================================================================
# 13. DELTAS
# ==================================================================================================

delta = {

    "six_plastic_AP50_pp":

        (
            best[
                "six_plastic_mean_AP50"
            ]
            -
            baseline_eval[
                "six_plastic_mean_AP50"
            ]
        )
        * 100,


    "MR_AP50_pp":

        (
            best[
                "class_metrics"
            ][
                "mixed_plastic_rigid"
            ][
                "AP50"
            ]
            -
            baseline_eval[
                "class_metrics"
            ][
                "mixed_plastic_rigid"
            ][
                "AP50"
            ]
        )
        * 100,


    "MS_AP50_pp":

        (
            best[
                "class_metrics"
            ][
                "mixed_plastic_soft"
            ][
                "AP50"
            ]
            -
            baseline_eval[
                "class_metrics"
            ][
                "mixed_plastic_soft"
            ][
                "AP50"
            ]
        )
        * 100,


    "MR_MS_mean_AP50_pp":

        (
            best[
                "MR_MS_mean_AP50"
            ]
            -
            baseline_eval[
                "MR_MS_mean_AP50"
            ]
        )
        * 100,


    "overall_AP50_95_pp":

        (
            best[
                "overall"
            ][
                "AP50_95"
            ]
            -
            baseline_eval[
                "overall"
            ][
                "AP50_95"
            ]
        )
        * 100,
}


# ==================================================================================================
# 14. TOP RESULTS
# ==================================================================================================

print()
print("=" * 115)
print("TOP E24-B VALIDATION RESULTS")
print("=" * 115)

print(
    f"\n"
    f"{'Rank':>4s} "
    f"{'αMR':>6s} "
    f"{'αMS':>6s} "
    f"{'Plastic':>11s} "
    f"{'MR':>10s} "
    f"{'MS':>10s} "
    f"{'MR/MS':>10s} "
    f"{'AP50-95':>11s}"
)

print(
    "-" * 80
)


for rank, result in enumerate(
    ranked[:15],
    start=1
):

    print(
        f"{rank:4d} "
        f"{result['alpha_MR']:6.2f} "
        f"{result['alpha_MS']:6.2f} "
        f"{result['six_plastic_mean_AP50']*100:10.4f}% "
        f"{result['class_metrics']['mixed_plastic_rigid']['AP50']*100:9.3f}% "
        f"{result['class_metrics']['mixed_plastic_soft']['AP50']*100:9.3f}% "
        f"{result['MR_MS_mean_AP50']*100:9.3f}% "
        f"{result['overall']['AP50_95']*100:10.4f}%"
    )


# ==================================================================================================
# 15. BEST CONFIGURATION
# ==================================================================================================

print()
print("=" * 115)
print("BEST E24-B CONFIGURATION — VALIDATION ONLY")
print("=" * 115)

print()
print(
    f"MR alpha               : "
    f"{best['alpha_MR']:.2f}"
)

print(
    f"MS alpha               : "
    f"{best['alpha_MS']:.2f}"
)

print()
print(
    f"Six-plastic mean AP50 : "
    f"{best['six_plastic_mean_AP50']*100:.4f}%"
)

print(
    f"MR AP50               : "
    f"{best['class_metrics']['mixed_plastic_rigid']['AP50']*100:.4f}%"
)

print(
    f"MS AP50               : "
    f"{best['class_metrics']['mixed_plastic_soft']['AP50']*100:.4f}%"
)

print(
    f"MR/MS mean AP50       : "
    f"{best['MR_MS_mean_AP50']*100:.4f}%"
)

print(
    f"Overall AP50-95       : "
    f"{best['overall']['AP50_95']*100:.4f}%"
)

print(
    f"Overall AP50          : "
    f"{best['overall']['AP50']*100:.4f}%"
)


print()
print(
    "Delta vs E23-D:"
)


for metric, value in delta.items():

    print(
        f"{metric:28s}: "
        f"{value:+.4f} pp"
    )


PROMOTE_TO_TEST = (
    best[
        "six_plastic_mean_AP50"
    ]
    >
    baseline_eval[
        "six_plastic_mean_AP50"
    ]
    +
    1e-12
)


print()
print("=" * 115)

if PROMOTE_TO_TEST:

    print(
        "DECISION: E24-B IMPROVES VALIDATION — ELIGIBLE FOR ONE FROZEN TEST EVALUATION."
    )

else:

    print(
        "DECISION: E24-B DOES NOT IMPROVE VALIDATION — REJECT; DO NOT TEST."
    )

print("=" * 115)


# ==================================================================================================
# 16. SAVE
# ==================================================================================================

best_predictions = build_predictions(
    alpha_mr=float(
        best[
            "alpha_MR"
        ]
    ),

    alpha_ms=float(
        best[
            "alpha_MS"
        ]
    ),
)


with open(
    GRID_PATH,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        ranked,
        f,
        indent=2
    )


payload = {

    "experiment":
        "E24-B",

    "description":
        (
            "Validation-only class-specific YOLO/classifier "
            "confidence fusion for frozen E23-D MR/MS predictions"
        ),

    "dataset":
        "validation",

    "test_used":
        False,

    "base_pipeline":
        "E23-D",

    "routing_changed":
        False,

    "frozen_E23D_routing": {

        "MS_to_MR": {
            "e23_min":
                MS_TO_MR_E23_MIN,

            "e20_max":
                MS_TO_MR_E20_MAX,
        },

        "MR_to_MS": {
            "e23_min":
                MR_TO_MS_E23_MIN,

            "e20_max":
                MR_TO_MS_E20_MAX,
        },
    },

    "baseline_alpha":
        0.70,

    "score_rule":
        "YOLO_conf^alpha_class * recovered_E20_classifier_component^(1-alpha_class)",

    "baseline_E23D":
        baseline_eval,

    "best":
        best,

    "delta_vs_E23D_pp":
        delta,

    "promote_to_test":
        bool(
            PROMOTE_TO_TEST
        ),
}


with open(
    BEST_CONFIG_PATH,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        payload,
        f,
        indent=2
    )


with open(
    BEST_PRED_PATH,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        best_predictions,
        f
    )


summary_lines = [

    "E24-B — Validation-Only Class-Specific MR/MS Confidence Fusion",

    "=" * 100,

    "",

    "VALIDATION ONLY — TEST NOT USED",

    "",

    f"Best MR alpha          : {best['alpha_MR']:.2f}",

    f"Best MS alpha          : {best['alpha_MS']:.2f}",

    "",

    f"Six-plastic mean AP50 : {best['six_plastic_mean_AP50']*100:.4f}%",

    f"MR AP50               : {best['class_metrics']['mixed_plastic_rigid']['AP50']*100:.4f}%",

    f"MS AP50               : {best['class_metrics']['mixed_plastic_soft']['AP50']*100:.4f}%",

    f"MR/MS mean AP50       : {best['MR_MS_mean_AP50']*100:.4f}%",

    f"Overall AP50-95       : {best['overall']['AP50_95']*100:.4f}%",

    "",

    f"Promote to TEST       : {PROMOTE_TO_TEST}",

    "",

    "Delta vs E23-D:",
]


for metric, value in delta.items():

    summary_lines.append(
        f"  {metric:28s}: "
        f"{value:+.4f} pp"
    )


with open(
    SUMMARY_PATH,
    "w",
    encoding="utf-8"
) as f:

    f.write(
        "\n".join(
            summary_lines
        )
    )


print()
print("=" * 115)
print("E24-B COMPLETE")
print("=" * 115)

print()
print(
    f"Grid results : "
    f"{GRID_PATH}"
)

print(
    f"Best config  : "
    f"{BEST_CONFIG_PATH}"
)

print(
    f"Predictions  : "
    f"{BEST_PRED_PATH}"
)

print(
    f"Summary      : "
    f"{SUMMARY_PATH}"
)

E24-B — VALIDATION-ONLY CLASS-SPECIFIC MR/MS CONFIDENCE FUSION

Cache entries : 55,605
Validation GT : C:\Users\varda\Documents\_My Computer\COMMON_space\Learning\Upgrad_EPGP_MS\upgrad_IIITB_MS_course\MS_LJMU_Material\Thesis_Code\runs\sortwaste\E18D_class_selective_convnext\E18D_val_gt_7class.json
Output        : C:\Users\varda\Documents\_My Computer\COMMON_space\Learning\Upgrad_EPGP_MS\upgrad_IIITB_MS_course\MS_LJMU_Material\Thesis_Code\runs\sortwaste\E24B_class_specific_MRMS_fusion

MS -> MR : E23 >= 0.920, E20 <= 0.90
MR -> MS : E23 >= 0.997, E20 <= 0.95

FROZEN E23-D ROUTING

MS -> MR proposals : 1,783
MR -> MS proposals : 2,100

MS -> MR accepted  : 606
MR -> MS accepted  : 196
loading annotations into memory...
Done (t=0.05s)
creating index...
index created!

RECONSTRUCTING FROZEN E23-D BASELINE
Loading and preparing results...
DONE (t=0.08s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=3.94s).
Accumulating evaluation re

In [43]:
## E24-B — Frozen Held-Out Test Evaluation of Class-Specific MR/MS Confidence Fusion
# ==================================================================================================
# E24-B — FROZEN HELD-OUT TEST EVALUATION
#         CLASS-SPECIFIC MR/MS CONFIDENCE FUSION
#
# VALIDATION-SELECTED CONFIGURATION
# --------------------------------------------------------------------------------------------------
#
#   Mixed Rigid alpha = 0.85
#   Mixed Soft  alpha = 0.50
#
# Base pipeline:
#   Frozen E23-D
#
# Frozen E23-D routing:
#
#   MS -> MR:
#       E23 >= 0.920
#       E20 <= 0.90
#
#   MR -> MS:
#       E23 >= 0.997
#       E20 <= 0.95
#
#
# E24-B score:
#
#   final MR:
#       score =
#           YOLO_conf ^ 0.85
#           *
#           recovered_E20_classifier_component ^ 0.15
#
#   final MS:
#       score =
#           YOLO_conf ^ 0.50
#           *
#           recovered_E20_classifier_component ^ 0.50
#
# Other final classes:
#       original E20/E23-D score unchanged
#
#
# IMPORTANT
# --------------------------------------------------------------------------------------------------
# The classifier component is recovered EXACTLY from the original E20-A score:
#
#   E20_score =
#       YOLO_conf ^ 0.70
#       *
#       classifier_component ^ 0.30
#
# therefore:
#
#   classifier_component =
#       (E20_score / YOLO_conf^0.70) ^ (1/0.30)
#
#
# NO:
#   - model inference
#   - routing search
#   - alpha search
#   - threshold tuning
#   - retraining
#
# TEST = frozen evaluation only.
# ==================================================================================================

import json
from pathlib import Path
from collections import Counter

import numpy as np

from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval


# ==================================================================================================
# 1. PATHS
# ==================================================================================================

BASE = Path(
    r"C:\Users\varda\Documents\_My Computer\COMMON_space\Learning"
    r"\Upgrad_EPGP_MS\upgrad_IIITB_MS_course\MS_LJMU_Material"
)

THESIS_CODE = (
    BASE
    / "Thesis_Code"
)

RUNS_ROOT = (
    THESIS_CODE
    / "runs"
    / "sortwaste"
)


# --------------------------------------------------------------------------------------------------
# Verified E20-A TEST predictions
# --------------------------------------------------------------------------------------------------

E20_PRED_PATH = (
    RUNS_ROOT
    / "FINAL_TEST_E20A"
    / "E20A_test_predictions.json"
)


E20_GT_PATH = (
    RUNS_ROOT
    / "FINAL_TEST_E20A"
    / "E20A_test_gt_7class.json"
)


# --------------------------------------------------------------------------------------------------
# Corrected E23-D TEST caches
# --------------------------------------------------------------------------------------------------

E23D_TEST_DIR = (
    RUNS_ROOT
    / "FINAL_TEST_E23D_CORRECTED"
)


MATCH_CACHE_PATH = (
    E23D_TEST_DIR
    / "E23D_E20A_E12_matched_cache.json"
)


E23_CACHE_PATH = (
    E23D_TEST_DIR
    / "E23D_E23A_MRMS_test_cache.json"
)


# --------------------------------------------------------------------------------------------------
# Frozen E23-D validation config
# --------------------------------------------------------------------------------------------------

E23D_CONFIG_PATH = (
    RUNS_ROOT
    / "E23D_confidence_contrastive_MRMS"
    / "E23D_best_configuration.json"
)


# --------------------------------------------------------------------------------------------------
# Validation-selected E24-B configuration
# --------------------------------------------------------------------------------------------------

E24B_CONFIG_PATH = (
    RUNS_ROOT
    / "E24B_class_specific_MRMS_fusion"
    / "E24B_best_configuration.json"
)


# --------------------------------------------------------------------------------------------------
# Output
# --------------------------------------------------------------------------------------------------

OUTPUT_DIR = (
    RUNS_ROOT
    / "FINAL_TEST_E24B"
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)


PREDICTIONS_PATH = (
    OUTPUT_DIR
    / "E24B_test_predictions.json"
)


RESULTS_PATH = (
    OUTPUT_DIR
    / "E24B_test_results.json"
)


SUMMARY_PATH = (
    OUTPUT_DIR
    / "E24B_test_summary.txt"
)


for path, label in [

    (
        E20_PRED_PATH,
        "E20-A TEST predictions"
    ),

    (
        E20_GT_PATH,
        "TEST GT"
    ),

    (
        MATCH_CACHE_PATH,
        "E20/E12 matched TEST cache"
    ),

    (
        E23_CACHE_PATH,
        "E23-A MR/MS TEST cache"
    ),

    (
        E23D_CONFIG_PATH,
        "E23-D frozen validation config"
    ),

    (
        E24B_CONFIG_PATH,
        "E24-B validation-selected config"
    ),
]:

    if not path.exists():

        raise FileNotFoundError(
            f"{label} not found:\n{path}"
        )


# ==================================================================================================
# 2. CLASSES
# ==================================================================================================

CLASS_NAMES = [
    "ecal",
    "hdpe",
    "mixed_plastic_rigid",
    "mixed_plastic_soft",
    "non_plastic",
    "pet",
    "pet_oil",
]


ECAL = 0
HDPE = 1
MR = 2
MS = 3
NON_PLASTIC = 4
PET = 5
PET_OIL = 6


PLASTIC_INDICES = [
    ECAL,
    HDPE,
    MR,
    MS,
    PET,
    PET_OIL,
]


# ==================================================================================================
# 3. LOAD INPUTS
# ==================================================================================================

print("=" * 120)
print("E24-B — FROZEN HELD-OUT TEST EVALUATION")
print("=" * 120)


with open(
    E20_PRED_PATH,
    "r",
    encoding="utf-8"
) as f:

    e20_predictions = json.load(f)


with open(
    MATCH_CACHE_PATH,
    "r",
    encoding="utf-8"
) as f:

    matched_cache = json.load(f)


with open(
    E23_CACHE_PATH,
    "r",
    encoding="utf-8"
) as f:

    e23_cache = json.load(f)


with open(
    E23D_CONFIG_PATH,
    "r",
    encoding="utf-8"
) as f:

    e23d_config = json.load(f)


with open(
    E24B_CONFIG_PATH,
    "r",
    encoding="utf-8"
) as f:

    e24b_config = json.load(f)


print()
print(
    f"E20-A predictions : "
    f"{len(e20_predictions):,}"
)

print(
    f"Matched cache     : "
    f"{len(matched_cache):,}"
)

print(
    f"E23-A cache       : "
    f"{len(e23_cache):,}"
)


# ==================================================================================================
# 4. BASIC SAFETY CHECKS
# ==================================================================================================

if len(
    e20_predictions
) != 57487:

    raise RuntimeError(
        "Expected 57,487 E20-A TEST predictions."
    )


if len(
    matched_cache
) != 57487:

    raise RuntimeError(
        "Expected matched TEST cache with 57,487 detections."
    )


if len(
    e23_cache
) != 17193:

    raise RuntimeError(
        "Expected E23-A TEST cache with 17,193 MR/MS candidates."
    )


# ==================================================================================================
# 5. LOAD FROZEN E23-D ROUTING
# ==================================================================================================

if e23d_config.get(
    "test_used"
) is not False:

    raise RuntimeError(
        "E23-D configuration was not validation-only."
    )


e23d_best = (
    e23d_config[
        "best_configuration"
    ]
)


MS_TO_MR_E23_MIN = float(
    e23d_best[
        "MS_to_MR"
    ][
        "e23_min"
    ]
)


MS_TO_MR_E20_MAX = float(
    e23d_best[
        "MS_to_MR"
    ][
        "e20_max"
    ]
)


MR_TO_MS_E23_MIN = float(
    e23d_best[
        "MR_to_MS"
    ][
        "e23_min"
    ]
)


MR_TO_MS_E20_MAX = float(
    e23d_best[
        "MR_to_MS"
    ][
        "e20_max"
    ]
)


if not (
    np.isclose(
        MS_TO_MR_E23_MIN,
        0.920
    )
    and
    np.isclose(
        MS_TO_MR_E20_MAX,
        0.90
    )
    and
    np.isclose(
        MR_TO_MS_E23_MIN,
        0.997
    )
    and
    np.isclose(
        MR_TO_MS_E20_MAX,
        0.95
    )
):

    raise RuntimeError(
        "Frozen E23-D routing does not match validation configuration."
    )


# ==================================================================================================
# 6. LOAD FROZEN E24-B ALPHAS
# ==================================================================================================

if e24b_config.get(
    "test_used"
) is not False:

    raise RuntimeError(
        "E24-B configuration is not validation-only."
    )


if e24b_config.get(
    "promote_to_test"
) is not True:

    raise RuntimeError(
        "E24-B was not marked eligible for TEST."
    )


best_e24b = (
    e24b_config[
        "best"
    ]
)


ALPHA_MR = float(
    best_e24b[
        "alpha_MR"
    ]
)


ALPHA_MS = float(
    best_e24b[
        "alpha_MS"
    ]
)


if not (
    np.isclose(
        ALPHA_MR,
        0.85
    )
    and
    np.isclose(
        ALPHA_MS,
        0.50
    )
):

    raise RuntimeError(
        f"Expected validation-selected "
        f"alpha_MR=0.85 and alpha_MS=0.50, "
        f"found {ALPHA_MR}, {ALPHA_MS}."
    )


print()
print("=" * 120)
print("FROZEN CONFIGURATION")
print("=" * 120)

print()
print(
    f"MS -> MR : "
    f"E23 >= {MS_TO_MR_E23_MIN:.3f}, "
    f"E20 <= {MS_TO_MR_E20_MAX:.2f}"
)

print(
    f"MR -> MS : "
    f"E23 >= {MR_TO_MS_E23_MIN:.3f}, "
    f"E20 <= {MR_TO_MS_E20_MAX:.2f}"
)

print()
print(
    f"MR alpha : "
    f"{ALPHA_MR:.2f}"
)

print(
    f"MS alpha : "
    f"{ALPHA_MS:.2f}"
)


# ==================================================================================================
# 7. MATCHED CACHE LOOKUP
# ==================================================================================================

record_lookup = {}


for record in matched_cache:

    idx = int(
        record[
            "e20_index"
        ]
    )


    if idx in record_lookup:

        raise RuntimeError(
            f"Duplicate E20 index: {idx}"
        )


    record_lookup[
        idx
    ] = record


if len(
    record_lookup
) != 57487:

    raise RuntimeError(
        "Matched-cache lookup incomplete."
    )


# ==================================================================================================
# 8. COCO EVALUATION
# ==================================================================================================

coco_gt = COCO(
    str(
        E20_GT_PATH
    )
)


def valid_mean(
    values
):

    values = np.asarray(
        values
    )


    valid = values[
        values > -1
    ]


    if valid.size == 0:

        return float(
            "nan"
        )


    return float(
        valid.mean()
    )


def evaluate_predictions(
    predictions
):

    coco_dt = coco_gt.loadRes(
        predictions
    )


    evaluator = COCOeval(
        coco_gt,
        coco_dt,
        "bbox"
    )


    evaluator.params.maxDets = [
        1,
        10,
        100,
    ]


    evaluator.evaluate()
    evaluator.accumulate()


    precision = (
        evaluator.eval[
            "precision"
        ]
    )


    recall = (
        evaluator.eval[
            "recall"
        ]
    )


    ious = (
        evaluator.params.iouThrs
    )


    idx50 = int(
        np.where(
            np.isclose(
                ious,
                0.50
            )
        )[0][0]
    )


    idx75 = int(
        np.where(
            np.isclose(
                ious,
                0.75
            )
        )[0][0]
    )


    overall = {

        "AP50_95":

            valid_mean(
                precision[
                    :,
                    :,
                    :,
                    0,
                    -1
                ]
            ),


        "AP50":

            valid_mean(
                precision[
                    idx50,
                    :,
                    :,
                    0,
                    -1
                ]
            ),


        "AP75":

            valid_mean(
                precision[
                    idx75,
                    :,
                    :,
                    0,
                    -1
                ]
            ),


        "AR100":

            valid_mean(
                recall[
                    :,
                    :,
                    0,
                    -1
                ]
            ),
    }


    class_metrics = {}


    for class_idx, class_name in enumerate(
        CLASS_NAMES
    ):

        class_metrics[
            class_name
        ] = {

            "AP50":

                valid_mean(
                    precision[
                        idx50,
                        :,
                        class_idx,
                        0,
                        -1
                    ]
                ),


            "AP50_95":

                valid_mean(
                    precision[
                        :,
                        :,
                        class_idx,
                        0,
                        -1
                    ]
                ),
        }


    six_plastic = float(
        np.mean(
            [
                class_metrics[
                    CLASS_NAMES[
                        idx
                    ]
                ][
                    "AP50"
                ]

                for idx
                in PLASTIC_INDICES
            ]
        )
    )


    six_plastic_ap = float(
        np.mean(
            [
                class_metrics[
                    CLASS_NAMES[
                        idx
                    ]
                ][
                    "AP50_95"
                ]

                for idx
                in PLASTIC_INDICES
            ]
        )
    )


    mr_ms = float(
        np.mean(
            [
                class_metrics[
                    "mixed_plastic_rigid"
                ][
                    "AP50"
                ],

                class_metrics[
                    "mixed_plastic_soft"
                ][
                    "AP50"
                ],
            ]
        )
    )


    return {

        "overall":
            overall,

        "class_metrics":
            class_metrics,

        "six_plastic_mean_AP50":
            six_plastic,

        "six_plastic_mean_AP50_95":
            six_plastic_ap,

        "MR_MS_mean_AP50":
            mr_ms,
    }


# ==================================================================================================
# 9. VERIFY E20-A BASELINE
# ==================================================================================================

print()
print("=" * 120)
print("VERIFYING E20-A TEST BASELINE")
print("=" * 120)


e20_eval = evaluate_predictions(
    e20_predictions
)


print()
print(
    f"AP50-95      : "
    f"{e20_eval['overall']['AP50_95']*100:.4f}%"
)

print(
    f"AP50         : "
    f"{e20_eval['overall']['AP50']*100:.4f}%"
)

print(
    f"Plastic AP50 : "
    f"{e20_eval['six_plastic_mean_AP50']*100:.4f}%"
)


if not (
    abs(
        e20_eval[
            "overall"
        ][
            "AP50_95"
        ]
        -
        0.462612
    )
    < 0.001

    and

    abs(
        e20_eval[
            "overall"
        ][
            "AP50"
        ]
        -
        0.623271
    )
    < 0.001

    and

    abs(
        e20_eval[
            "six_plastic_mean_AP50"
        ]
        -
        0.694353
    )
    < 0.001
):

    raise RuntimeError(
        "E20-A TEST baseline verification failed."
    )


print()
print(
    "E20-A baseline verification PASSED."
)


# ==================================================================================================
# 10. RECONSTRUCT E23-D TEST
# ==================================================================================================

e23d_predictions = [
    dict(
        p
    )

    for p
    in e20_predictions
]


proposal_counts = Counter()
accepted_counts = Counter()


for idx, original in enumerate(
    e20_predictions
):

    current_idx = (
        int(
            original[
                "category_id"
            ]
        )
        - 1
    )


    if current_idx not in {
        MR,
        MS,
    }:

        continue


    key = str(
        idx
    )


    if key not in e23_cache:

        raise RuntimeError(
            f"Missing E23 cache for E20 detection {idx}"
        )


    e23 = (
        e23_cache[
            key
        ]
    )


    e23_idx = int(
        e23[
            "pred_global_idx"
        ]
    )


    e23_prob = float(
        e23[
            "top_prob"
        ]
    )


    e20_conf = float(
        record_lookup[
            idx
        ][
            "e20_class_conf"
        ]
    )


    # ----------------------------------------------------------------------------------------------
    # MS -> MR
    # ----------------------------------------------------------------------------------------------

    if (
        current_idx == MS
        and
        e23_idx == MR
    ):

        proposal_counts[
            "MS->MR"
        ] += 1


        if (
            e23_prob
            >=
            MS_TO_MR_E23_MIN
            and
            e20_conf
            <=
            MS_TO_MR_E20_MAX
        ):

            e23d_predictions[
                idx
            ][
                "category_id"
            ] = (
                MR + 1
            )


            accepted_counts[
                "MS->MR"
            ] += 1


    # ----------------------------------------------------------------------------------------------
    # MR -> MS
    # ----------------------------------------------------------------------------------------------

    elif (
        current_idx == MR
        and
        e23_idx == MS
    ):

        proposal_counts[
            "MR->MS"
        ] += 1


        if (
            e23_prob
            >=
            MR_TO_MS_E23_MIN
            and
            e20_conf
            <=
            MR_TO_MS_E20_MAX
        ):

            e23d_predictions[
                idx
            ][
                "category_id"
            ] = (
                MS + 1
            )


            accepted_counts[
                "MR->MS"
            ] += 1


print()
print("=" * 120)
print("RECONSTRUCTED E23-D TEST ROUTING")
print("=" * 120)

print()
print(
    f"MS -> MR proposals : "
    f"{proposal_counts['MS->MR']:,}"
)

print(
    f"MR -> MS proposals : "
    f"{proposal_counts['MR->MS']:,}"
)

print()
print(
    f"MS -> MR accepted  : "
    f"{accepted_counts['MS->MR']:,}"
)

print(
    f"MR -> MS accepted  : "
    f"{accepted_counts['MR->MS']:,}"
)


if not (
    accepted_counts[
        "MS->MR"
    ]
    == 691

    and

    accepted_counts[
        "MR->MS"
    ]
    == 311
):

    raise RuntimeError(
        "Corrected E23-D TEST routing was not reproduced."
    )


# ==================================================================================================
# 11. VERIFY E23-D BASELINE
# ==================================================================================================

print()
print("=" * 120)
print("VERIFYING CORRECTED E23-D TEST BASELINE")
print("=" * 120)


e23d_eval = evaluate_predictions(
    e23d_predictions
)


print()
print(
    f"AP50-95               : "
    f"{e23d_eval['overall']['AP50_95']*100:.4f}%"
)

print(
    f"AP50                  : "
    f"{e23d_eval['overall']['AP50']*100:.4f}%"
)

print(
    f"Six-plastic mean AP50: "
    f"{e23d_eval['six_plastic_mean_AP50']*100:.4f}%"
)

print(
    f"MR AP50               : "
    f"{e23d_eval['class_metrics']['mixed_plastic_rigid']['AP50']*100:.4f}%"
)

print(
    f"MS AP50               : "
    f"{e23d_eval['class_metrics']['mixed_plastic_soft']['AP50']*100:.4f}%"
)


if not (
    abs(
        e23d_eval[
            "overall"
        ][
            "AP50_95"
        ]
        -
        0.462851
    )
    < 1e-5

    and

    abs(
        e23d_eval[
            "overall"
        ][
            "AP50"
        ]
        -
        0.623548
    )
    < 1e-5

    and

    abs(
        e23d_eval[
            "six_plastic_mean_AP50"
        ]
        -
        0.694677
    )
    < 1e-5
):

    raise RuntimeError(
        "\nCorrected E23-D TEST baseline was NOT reproduced.\n"
        "E24-B scoring has NOT been applied."
    )


print()
print(
    "Corrected E23-D TEST reproduction PASSED."
)


# ==================================================================================================
# 12. BUILD E24-B TEST PREDICTIONS
#
# IMPORTANT:
#
# E24-B validation rescored ALL detections whose FINAL E23-D class was MR or MS.
#
# Therefore TEST must do EXACTLY the same thing.
#
# ==================================================================================================

print()
print("=" * 120)
print("APPLYING FROZEN E24-B CLASS-SPECIFIC FUSION")
print("=" * 120)


e24b_predictions = [
    dict(
        p
    )

    for p
    in e23d_predictions
]


mr_rescored = 0
ms_rescored = 0


classifier_components = []


for idx, prediction in enumerate(
    e23d_predictions
):

    final_idx = (
        int(
            prediction[
                "category_id"
            ]
        )
        - 1
    )


    # Only final MR/MS are changed.

    if final_idx not in {
        MR,
        MS,
    }:

        continue


    record = (
        record_lookup[
            idx
        ]
    )


    # ----------------------------------------------------------------------------------------------
    # Original E20-A quantities
    # ----------------------------------------------------------------------------------------------

    e20_score = max(
        float(
            e20_predictions[
                idx
            ][
                "score"
            ]
        ),
        1e-12
    )


    yolo_conf = max(
        float(
            record[
                "yolo_conf"
            ]
        ),
        1e-12
    )


    # ----------------------------------------------------------------------------------------------
    # Recover classifier component exactly from original E20 score:
    #
    #   E20_score = YOLO^0.70 * classifier^0.30
    # ----------------------------------------------------------------------------------------------

    classifier_component = (
        e20_score
        /
        (
            yolo_conf
            ** 0.70
        )
    ) ** (
        1.0
        /
        0.30
    )


    classifier_component = float(
        np.clip(
            classifier_component,
            0.0,
            1.0
        )
    )


    classifier_components.append(
        classifier_component
    )


    # ----------------------------------------------------------------------------------------------
    # Final class-specific fusion
    # ----------------------------------------------------------------------------------------------

    if final_idx == MR:

        alpha = (
            ALPHA_MR
        )

        mr_rescored += 1


    else:

        alpha = (
            ALPHA_MS
        )

        ms_rescored += 1


    new_score = (
        yolo_conf
        ** alpha
    ) * (
        max(
            classifier_component,
            1e-12
        )
        ** (
            1.0
            -
            alpha
        )
    )


    e24b_predictions[
        idx
    ][
        "score"
    ] = float(
        new_score
    )


print()
print(
    f"Final MR rescored : "
    f"{mr_rescored:,}"
)

print(
    f"Final MS rescored : "
    f"{ms_rescored:,}"
)

print(
    f"Total rescored    : "
    f"{mr_rescored + ms_rescored:,}"
)


print()
print(
    f"Recovered classifier component min    : "
    f"{np.min(classifier_components):.6f}"
)

print(
    f"Recovered classifier component mean   : "
    f"{np.mean(classifier_components):.6f}"
)

print(
    f"Recovered classifier component median : "
    f"{np.median(classifier_components):.6f}"
)

print(
    f"Recovered classifier component max    : "
    f"{np.max(classifier_components):.6f}"
)


# ==================================================================================================
# 13. STRUCTURAL SAFETY CHECK
# ==================================================================================================

image_changes = 0
bbox_changes = 0
class_changes = 0

score_changes = 0

illegal_score_changes = 0


for idx, (
    before,
    after
) in enumerate(
    zip(
        e23d_predictions,
        e24b_predictions
    )
):

    if before[
        "image_id"
    ] != after[
        "image_id"
    ]:

        image_changes += 1


    if before[
        "bbox"
    ] != after[
        "bbox"
    ]:

        bbox_changes += 1


    if before[
        "category_id"
    ] != after[
        "category_id"
    ]:

        class_changes += 1


    score_changed = (
        float(
            before[
                "score"
            ]
        )
        !=
        float(
            after[
                "score"
            ]
        )
    )


    if score_changed:

        score_changes += 1


        final_idx = (
            int(
                after[
                    "category_id"
                ]
            )
            - 1
        )


        if final_idx not in {
            MR,
            MS,
        }:

            illegal_score_changes += 1


print()
print("=" * 120)
print("E24-B STRUCTURAL SAFETY CHECK")
print("=" * 120)

print()
print(
    f"Image-ID changes vs E23-D : "
    f"{image_changes}"
)

print(
    f"BBox changes vs E23-D     : "
    f"{bbox_changes}"
)

print(
    f"Class changes vs E23-D    : "
    f"{class_changes}"
)

print(
    f"Score changes vs E23-D    : "
    f"{score_changes:,}"
)

print(
    f"Illegal score changes     : "
    f"{illegal_score_changes}"
)


if (
    image_changes != 0
    or
    bbox_changes != 0
    or
    class_changes != 0
    or
    illegal_score_changes != 0
):

    raise RuntimeError(
        "E24-B altered something outside final MR/MS confidence scores."
    )


# ==================================================================================================
# 14. E24-B TEST EVALUATION
# ==================================================================================================

print()
print("=" * 120)
print("E24-B — FROZEN HELD-OUT TEST COCO EVALUATION")
print("=" * 120)


e24b_eval = evaluate_predictions(
    e24b_predictions
)


# ==================================================================================================
# 15. RESULTS
# ==================================================================================================

print()
print("=" * 120)
print("E24-B — HELD-OUT TEST RESULTS")
print("=" * 120)


print()
print(
    f"AP50-95                : "
    f"{e24b_eval['overall']['AP50_95']*100:.4f}%"
)

print(
    f"AP50                   : "
    f"{e24b_eval['overall']['AP50']*100:.4f}%"
)

print(
    f"AP75                   : "
    f"{e24b_eval['overall']['AP75']*100:.4f}%"
)

print(
    f"AR100                  : "
    f"{e24b_eval['overall']['AR100']*100:.4f}%"
)


print()
print(
    f"Six-plastic mean AP50 : "
    f"{e24b_eval['six_plastic_mean_AP50']*100:.4f}%"
)

print(
    f"Six-plastic AP50-95   : "
    f"{e24b_eval['six_plastic_mean_AP50_95']*100:.4f}%"
)

print(
    f"MR/MS mean AP50       : "
    f"{e24b_eval['MR_MS_mean_AP50']*100:.4f}%"
)


# ==================================================================================================
# 16. CLASS-WISE RESULTS
# ==================================================================================================

print()
print("=" * 120)
print("E24-B — CLASS-WISE TEST RESULTS")
print("=" * 120)


print(
    f"\n"
    f"{'Class':30s}"
    f"{'AP50':>15s}"
    f"{'AP50-95':>15s}"
)

print(
    "-" * 60
)


for class_name in CLASS_NAMES:

    values = (
        e24b_eval[
            "class_metrics"
        ][
            class_name
        ]
    )


    print(
        f"{class_name:30s}"
        f"{values['AP50']*100:14.2f}%"
        f"{values['AP50_95']*100:14.2f}%"
    )


# ==================================================================================================
# 17. DELTA VS E23-D
# ==================================================================================================

delta_vs_e23d = {

    "AP50_95_pp":

        (
            e24b_eval[
                "overall"
            ][
                "AP50_95"
            ]

            -

            e23d_eval[
                "overall"
            ][
                "AP50_95"
            ]
        )
        * 100,


    "AP50_pp":

        (
            e24b_eval[
                "overall"
            ][
                "AP50"
            ]

            -

            e23d_eval[
                "overall"
            ][
                "AP50"
            ]
        )
        * 100,


    "AP75_pp":

        (
            e24b_eval[
                "overall"
            ][
                "AP75"
            ]

            -

            e23d_eval[
                "overall"
            ][
                "AP75"
            ]
        )
        * 100,


    "AR100_pp":

        (
            e24b_eval[
                "overall"
            ][
                "AR100"
            ]

            -

            e23d_eval[
                "overall"
            ][
                "AR100"
            ]
        )
        * 100,


    "six_plastic_AP50_pp":

        (
            e24b_eval[
                "six_plastic_mean_AP50"
            ]

            -

            e23d_eval[
                "six_plastic_mean_AP50"
            ]
        )
        * 100,


    "MR_AP50_pp":

        (
            e24b_eval[
                "class_metrics"
            ][
                "mixed_plastic_rigid"
            ][
                "AP50"
            ]

            -

            e23d_eval[
                "class_metrics"
            ][
                "mixed_plastic_rigid"
            ][
                "AP50"
            ]
        )
        * 100,


    "MS_AP50_pp":

        (
            e24b_eval[
                "class_metrics"
            ][
                "mixed_plastic_soft"
            ][
                "AP50"
            ]

            -

            e23d_eval[
                "class_metrics"
            ][
                "mixed_plastic_soft"
            ][
                "AP50"
            ]
        )
        * 100,


    "MR_MS_mean_AP50_pp":

        (
            e24b_eval[
                "MR_MS_mean_AP50"
            ]

            -

            e23d_eval[
                "MR_MS_mean_AP50"
            ]
        )
        * 100,
}


print()
print("=" * 120)
print("DELTA VS CORRECTED E23-D TEST")
print("=" * 120)

print()


for metric, value in (
    delta_vs_e23d.items()
):

    print(
        f"{metric:30s}: "
        f"{value:+.4f} pp"
    )


# ==================================================================================================
# 18. UNCHANGED-CLASS SANITY CHECK
# ==================================================================================================

print()
print("=" * 120)
print("UNCHANGED-CLASS SANITY CHECK")
print("=" * 120)

print()


for class_name in [
    "ecal",
    "hdpe",
    "non_plastic",
    "pet",
    "pet_oil",
]:

    before = (
        e23d_eval[
            "class_metrics"
        ][
            class_name
        ][
            "AP50"
        ]
    )


    after = (
        e24b_eval[
            "class_metrics"
        ][
            class_name
        ][
            "AP50"
        ]
    )


    delta = (
        after
        -
        before
    ) * 100


    print(
        f"{class_name:25s}: "
        f"{delta:+.6f} pp"
    )


# ==================================================================================================
# 19. FINAL COMPARISON
# ==================================================================================================

SORTWASTE_PLASTIC_AP50 = (
    69.3800
)


E21B = {
    "AP50_95": 46.2881,
    "AP50": 62.3518,
    "plastic": 69.4212,
    "MR": 48.70,
    "MS": 42.53,
    "MRMS": 45.6119,
}


print()
print("=" * 120)
print("FINAL COMPARISON")
print("=" * 120)


print(
    f"\n"
    f"{'System':12s}"
    f"{'AP50-95':>12s}"
    f"{'AP50':>12s}"
    f"{'Plastic':>12s}"
    f"{'MR':>10s}"
    f"{'MS':>10s}"
    f"{'MR/MS':>12s}"
)


print(
    "-" * 80
)


print(
    f"{'E20-A':12s}"
    f"{e20_eval['overall']['AP50_95']*100:11.4f}%"
    f"{e20_eval['overall']['AP50']*100:11.4f}%"
    f"{e20_eval['six_plastic_mean_AP50']*100:11.4f}%"
    f"{e20_eval['class_metrics']['mixed_plastic_rigid']['AP50']*100:9.2f}%"
    f"{e20_eval['class_metrics']['mixed_plastic_soft']['AP50']*100:9.2f}%"
    f"{e20_eval['MR_MS_mean_AP50']*100:11.4f}%"
)


print(
    f"{'E21-B':12s}"
    f"{E21B['AP50_95']:11.4f}%"
    f"{E21B['AP50']:11.4f}%"
    f"{E21B['plastic']:11.4f}%"
    f"{E21B['MR']:9.2f}%"
    f"{E21B['MS']:9.2f}%"
    f"{E21B['MRMS']:11.4f}%"
)


print(
    f"{'E23-D':12s}"
    f"{e23d_eval['overall']['AP50_95']*100:11.4f}%"
    f"{e23d_eval['overall']['AP50']*100:11.4f}%"
    f"{e23d_eval['six_plastic_mean_AP50']*100:11.4f}%"
    f"{e23d_eval['class_metrics']['mixed_plastic_rigid']['AP50']*100:9.2f}%"
    f"{e23d_eval['class_metrics']['mixed_plastic_soft']['AP50']*100:9.2f}%"
    f"{e23d_eval['MR_MS_mean_AP50']*100:11.4f}%"
)


print(
    f"{'E24-B':12s}"
    f"{e24b_eval['overall']['AP50_95']*100:11.4f}%"
    f"{e24b_eval['overall']['AP50']*100:11.4f}%"
    f"{e24b_eval['six_plastic_mean_AP50']*100:11.4f}%"
    f"{e24b_eval['class_metrics']['mixed_plastic_rigid']['AP50']*100:9.2f}%"
    f"{e24b_eval['class_metrics']['mixed_plastic_soft']['AP50']*100:9.2f}%"
    f"{e24b_eval['MR_MS_mean_AP50']*100:11.4f}%"
)


# ==================================================================================================
# 20. SORTWASTE / 70% COMPARISON
# ==================================================================================================

plastic_test_score = (
    e24b_eval[
        "six_plastic_mean_AP50"
    ]
    * 100
)


sortwaste_delta = (
    plastic_test_score
    -
    SORTWASTE_PLASTIC_AP50
)


gap_to_70 = (
    70.0
    -
    plastic_test_score
)


print()
print("=" * 120)
print("ALIGNED SIX-PLASTIC AP50 COMPARISON")
print("=" * 120)

print()
print(
    f"SortWaste YOLOv11 : "
    f"{SORTWASTE_PLASTIC_AP50:.4f}%"
)

print(
    f"E24-B             : "
    f"{plastic_test_score:.4f}%"
)

print(
    f"Difference        : "
    f"{sortwaste_delta:+.4f} pp"
)

print()
print(
    f"Gap to 70.0000%   : "
    f"{gap_to_70:+.4f} pp"
)


# ==================================================================================================
# 21. SAVE PREDICTIONS
# ==================================================================================================

with open(
    PREDICTIONS_PATH,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        e24b_predictions,
        f
    )


# ==================================================================================================
# 22. SAVE RESULTS
# ==================================================================================================

result_payload = {

    "experiment":
        "E24-B",

    "split":
        "held-out TEST",

    "test_used_for_selection":
        False,

    "base_pipeline":
        "E23-D",

    "validation_selected_configuration": {

        "alpha_MR":
            ALPHA_MR,

        "alpha_MS":
            ALPHA_MS,
    },

    "score_rule":
        (
            "YOLO_conf^alpha_class * "
            "recovered_E20_classifier_component^(1-alpha_class)"
        ),

    "routing_changed":
        False,

    "routing": {

        "MS_to_MR": {

            "e23_min":
                MS_TO_MR_E23_MIN,

            "e20_max":
                MS_TO_MR_E20_MAX,

            "accepted":
                int(
                    accepted_counts[
                        "MS->MR"
                    ]
                ),
        },


        "MR_to_MS": {

            "e23_min":
                MR_TO_MS_E23_MIN,

            "e20_max":
                MR_TO_MS_E20_MAX,

            "accepted":
                int(
                    accepted_counts[
                        "MR->MS"
                    ]
                ),
        },
    },

    "rescored_counts": {

        "mixed_plastic_rigid":
            int(
                mr_rescored
            ),

        "mixed_plastic_soft":
            int(
                ms_rescored
            ),
    },

    "E20A":
        e20_eval,

    "E23D":
        e23d_eval,

    "E24B":
        e24b_eval,

    "delta_vs_E23D_pp":
        delta_vs_e23d,

    "sortwaste_six_plastic_AP50":
        SORTWASTE_PLASTIC_AP50,

    "delta_vs_sortwaste_pp":
        sortwaste_delta,

    "gap_to_70_pp":
        gap_to_70,
}


with open(
    RESULTS_PATH,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        result_payload,
        f,
        indent=2
    )


# ==================================================================================================
# 23. SAVE SUMMARY
# ==================================================================================================

summary_lines = [

    "E24-B — Frozen Held-Out TEST Evaluation",

    "=" * 105,

    "",

    "VALIDATION-SELECTED CONFIGURATION",

    f"MR alpha               : {ALPHA_MR:.2f}",

    f"MS alpha               : {ALPHA_MS:.2f}",

    "",

    (
        f"MS -> MR              : "
        f"E23 >= {MS_TO_MR_E23_MIN:.3f}, "
        f"E20 <= {MS_TO_MR_E20_MAX:.2f}"
    ),

    (
        f"MR -> MS              : "
        f"E23 >= {MR_TO_MS_E23_MIN:.3f}, "
        f"E20 <= {MR_TO_MS_E20_MAX:.2f}"
    ),

    "",

    f"Final MR rescored      : {mr_rescored:,}",

    f"Final MS rescored      : {ms_rescored:,}",

    "",

    "TEST RESULTS",

    f"AP50-95               : {e24b_eval['overall']['AP50_95']*100:.4f}%",

    f"AP50                  : {e24b_eval['overall']['AP50']*100:.4f}%",

    f"AP75                  : {e24b_eval['overall']['AP75']*100:.4f}%",

    f"AR100                 : {e24b_eval['overall']['AR100']*100:.4f}%",

    f"Six-plastic mean AP50 : {e24b_eval['six_plastic_mean_AP50']*100:.4f}%",

    f"Six-plastic AP50-95   : {e24b_eval['six_plastic_mean_AP50_95']*100:.4f}%",

    f"MR/MS mean AP50       : {e24b_eval['MR_MS_mean_AP50']*100:.4f}%",

    "",

    f"SortWaste six-plastic : {SORTWASTE_PLASTIC_AP50:.4f}%",

    f"Difference             : {sortwaste_delta:+.4f} pp",

    f"Gap to 70%             : {gap_to_70:+.4f} pp",

    "",

    "Delta vs E23-D:",
]


for metric, value in (
    delta_vs_e23d.items()
):

    summary_lines.append(
        f"  {metric:28s}: "
        f"{value:+.4f} pp"
    )


with open(
    SUMMARY_PATH,
    "w",
    encoding="utf-8"
) as f:

    f.write(
        "\n".join(
            summary_lines
        )
    )


# ==================================================================================================
# 24. COMPLETE
# ==================================================================================================

print()
print("=" * 120)
print("E24-B FROZEN HELD-OUT TEST COMPLETE")
print("=" * 120)

print()
print(
    f"Predictions : "
    f"{PREDICTIONS_PATH}"
)

print(
    f"Results     : "
    f"{RESULTS_PATH}"
)

print(
    f"Summary     : "
    f"{SUMMARY_PATH}"
)

E24-B — FROZEN HELD-OUT TEST EVALUATION

E20-A predictions : 57,487
Matched cache     : 57,487
E23-A cache       : 17,193

FROZEN CONFIGURATION

MS -> MR : E23 >= 0.920, E20 <= 0.90
MR -> MS : E23 >= 0.997, E20 <= 0.95

MR alpha : 0.85
MS alpha : 0.50
loading annotations into memory...
Done (t=0.04s)
creating index...
index created!

VERIFYING E20-A TEST BASELINE
Loading and preparing results...
DONE (t=0.08s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=3.95s).
Accumulating evaluation results...
DONE (t=0.60s).

AP50-95      : 46.2612%
AP50         : 62.3271%
Plastic AP50 : 69.4353%

E20-A baseline verification PASSED.

RECONSTRUCTED E23-D TEST ROUTING

MS -> MR proposals : 2,166
MR -> MS proposals : 2,407

MS -> MR accepted  : 691
MR -> MS accepted  : 311

VERIFYING CORRECTED E23-D TEST BASELINE
Loading and preparing results...
DONE (t=0.06s)
creating index...
index created!
Running per image evaluation...
Evaluate annotatio

## E24-C — Combined Validation-Selected Confidence Fusion and Specialist Override Recalibration

In [44]:
# E24-C — COMBINED VALIDATION-SELECTED MR/MS CONFIDENCE FUSION
#         + E23-D OVERRIDE RECALIBRATION
#
# FROZEN FROM VALIDATION
# --------------------------------------------------------------------------------------------------
#
# E23-D routing:
#   MS -> MR : E23 >= 0.920 AND E20 <= 0.90
#   MR -> MS : E23 >= 0.997 AND E20 <= 0.95
#
# E24-A:
#   beta = 0.35
#
# E24-B:
#   alpha_MR = 0.85
#   alpha_MS = 0.50
#
#
# SCORE LOGIC
# --------------------------------------------------------------------------------------------------
#
# Step 1 — for every FINAL MR/MS detection:
#
#   class_score =
#       YOLO_conf ^ alpha_class
#       *
#       recovered_E20_classifier_component ^ (1-alpha_class)
#
#
# Step 2 — ONLY for accepted E23-D class overrides:
#
#   final_score =
#       class_score ^ beta
#       *
#       E23_confidence ^ (1-beta)
#
#
# Non-MR/MS detections:
#   original E20/E23-D score unchanged.
#
#
# VALIDATION ONLY
# NO SEARCH
# NO MODEL INFERENCE
# NO TEST ACCESS
# ==================================================================================================

import json
from pathlib import Path
from collections import Counter

import numpy as np

from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval


# ==================================================================================================
# 1. PATHS
# ==================================================================================================

BASE = Path(
    r"C:\Users\varda\Documents\_My Computer\COMMON_space\Learning"
    r"\Upgrad_EPGP_MS\upgrad_IIITB_MS_course\MS_LJMU_Material"
)

RUNS_ROOT = (
    BASE
    / "Thesis_Code"
    / "runs"
    / "sortwaste"
)


CACHE_PATH = (
    RUNS_ROOT
    / "E23B_E20A_binary_MRMS_routing"
    / "E23B_cached_predictions_with_E23A.json"
)

GT_PATH = (
    RUNS_ROOT
    / "E18D_class_selective_convnext"
    / "E18D_val_gt_7class.json"
)

E23D_CONFIG_PATH = (
    RUNS_ROOT
    / "E23D_confidence_contrastive_MRMS"
    / "E23D_best_configuration.json"
)

E24A_CONFIG_PATH = (
    RUNS_ROOT
    / "E24A_E23D_override_confidence_recalibration"
    / "E24A_best_configuration.json"
)

E24B_CONFIG_PATH = (
    RUNS_ROOT
    / "E24B_class_specific_MRMS_fusion"
    / "E24B_best_configuration.json"
)


OUTPUT_DIR = (
    RUNS_ROOT
    / "E24C_combined_MRMS_confidence"
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)


PRED_PATH = (
    OUTPUT_DIR
    / "E24C_validation_predictions.json"
)

RESULTS_PATH = (
    OUTPUT_DIR
    / "E24C_validation_results.json"
)

SUMMARY_PATH = (
    OUTPUT_DIR
    / "E24C_summary.txt"
)


for path in [
    CACHE_PATH,
    GT_PATH,
    E23D_CONFIG_PATH,
    E24A_CONFIG_PATH,
    E24B_CONFIG_PATH,
]:

    if not path.exists():

        raise FileNotFoundError(
            f"Required file not found:\n{path}"
        )


# ==================================================================================================
# 2. CLASSES
# ==================================================================================================

CLASS_NAMES = [
    "ecal",
    "hdpe",
    "mixed_plastic_rigid",
    "mixed_plastic_soft",
    "non_plastic",
    "pet",
    "pet_oil",
]

ECAL = 0
HDPE = 1
MR = 2
MS = 3
NON_PLASTIC = 4
PET = 5
PET_OIL = 6


PLASTIC_INDICES = [
    ECAL,
    HDPE,
    MR,
    MS,
    PET,
    PET_OIL,
]


# ==================================================================================================
# 3. LOAD INPUT
# ==================================================================================================

with open(
    CACHE_PATH,
    "r",
    encoding="utf-8"
) as f:

    cache = json.load(f)


with open(
    E23D_CONFIG_PATH,
    "r",
    encoding="utf-8"
) as f:

    e23d_config = json.load(f)


with open(
    E24A_CONFIG_PATH,
    "r",
    encoding="utf-8"
) as f:

    e24a_config = json.load(f)


with open(
    E24B_CONFIG_PATH,
    "r",
    encoding="utf-8"
) as f:

    e24b_config = json.load(f)


print("=" * 120)
print("E24-C — COMBINED VALIDATION-SELECTED MR/MS CONFIDENCE FUSION")
print("=" * 120)

print()
print(
    f"Cache entries : "
    f"{len(cache):,}"
)


if len(cache) != 55605:

    raise RuntimeError(
        "Expected 55,605 validation detections."
    )


# ==================================================================================================
# 4. LOAD FROZEN PARAMETERS
# ==================================================================================================

e23d_best = (
    e23d_config[
        "best_configuration"
    ]
)


MS_TO_MR_E23_MIN = float(
    e23d_best[
        "MS_to_MR"
    ][
        "e23_min"
    ]
)

MS_TO_MR_E20_MAX = float(
    e23d_best[
        "MS_to_MR"
    ][
        "e20_max"
    ]
)

MR_TO_MS_E23_MIN = float(
    e23d_best[
        "MR_to_MS"
    ][
        "e23_min"
    ]
)

MR_TO_MS_E20_MAX = float(
    e23d_best[
        "MR_to_MS"
    ][
        "e20_max"
    ]
)


BETA = float(
    e24a_config[
        "best_beta"
    ]
)


e24b_best = (
    e24b_config[
        "best"
    ]
)


ALPHA_MR = float(
    e24b_best[
        "alpha_MR"
    ]
)

ALPHA_MS = float(
    e24b_best[
        "alpha_MS"
    ]
)


# Hard checks

if not (
    np.isclose(MS_TO_MR_E23_MIN, 0.920)
    and
    np.isclose(MS_TO_MR_E20_MAX, 0.90)
    and
    np.isclose(MR_TO_MS_E23_MIN, 0.997)
    and
    np.isclose(MR_TO_MS_E20_MAX, 0.95)
    and
    np.isclose(BETA, 0.35)
    and
    np.isclose(ALPHA_MR, 0.85)
    and
    np.isclose(ALPHA_MS, 0.50)
):

    raise RuntimeError(
        "Frozen validation-selected E24-C inputs do not match expected values."
    )


print()
print("Frozen configuration:")

print(
    f"  MS -> MR : "
    f"E23 >= {MS_TO_MR_E23_MIN:.3f}, "
    f"E20 <= {MS_TO_MR_E20_MAX:.2f}"
)

print(
    f"  MR -> MS : "
    f"E23 >= {MR_TO_MS_E23_MIN:.3f}, "
    f"E20 <= {MR_TO_MS_E20_MAX:.2f}"
)

print(
    f"  beta      : "
    f"{BETA:.2f}"
)

print(
    f"  alpha MR  : "
    f"{ALPHA_MR:.2f}"
)

print(
    f"  alpha MS  : "
    f"{ALPHA_MS:.2f}"
)


# ==================================================================================================
# 5. HELPERS
# ==================================================================================================

def extract_e23(
    item
):

    info = item.get(
        "e23"
    )

    if info is None:

        return None


    if "global_idx" in info:

        pred_idx = int(
            info[
                "global_idx"
            ]
        )

    else:

        local_idx = int(
            info[
                "local_idx"
            ]
        )

        pred_idx = (
            MR
            if local_idx == 0
            else MS
        )


    if "top_prob" in info:

        top_prob = float(
            info[
                "top_prob"
            ]
        )

    else:

        top_prob = float(
            np.max(
                np.asarray(
                    info[
                        "probs"
                    ],
                    dtype=np.float64
                )
            )
        )


    return {
        "pred_idx":
            pred_idx,

        "top_prob":
            top_prob,
    }


def get_e20_current_class_confidence(
    item
):

    current_idx = int(
        item[
            "_e20_class_idx"
        ]
    )


    source = str(
        item.get(
            "_e20_source",
            ""
        )
    ).lower()


    conv = item.get(
        "conv"
    )


    if (
        source in {
            "e18",
            "convnext",
        }
        and
        conv is not None
        and
        int(
            conv[
                "conv_global_idx"
            ]
        )
        ==
        current_idx
    ):

        return float(
            conv[
                "conv_top_prob"
            ]
        )


    mn_probs = np.asarray(
        item[
            "mn_probs"
        ],
        dtype=np.float64
    )


    return float(
        mn_probs[
            current_idx
        ]
    )


# ==================================================================================================
# 6. RECONSTRUCT E23-D + RECOVER E20 CLASSIFIER COMPONENT
# ==================================================================================================

records = []

proposal_counts = Counter()
accepted_counts = Counter()


for idx, item in enumerate(
    cache
):

    original_idx = int(
        item[
            "_e20_class_idx"
        ]
    )


    final_idx = (
        original_idx
    )


    e20_score = max(
        float(
            item[
                "_e20_score"
            ]
        ),
        1e-12
    )


    yolo_conf = max(
        float(
            item[
                "yolo_conf"
            ]
        ),
        1e-12
    )


    # ----------------------------------------------------------------------------------------------
    # Recover exact original E20 classifier component.
    # ----------------------------------------------------------------------------------------------

    classifier_component = (
        e20_score
        /
        (
            yolo_conf
            ** 0.70
        )
    ) ** (
        1.0
        /
        0.30
    )


    classifier_component = float(
        np.clip(
            classifier_component,
            0.0,
            1.0
        )
    )


    e20_current_conf = (
        get_e20_current_class_confidence(
            item
        )
    )


    e23 = extract_e23(
        item
    )


    accepted_direction = None
    e23_prob = None


    if (
        original_idx in {
            MR,
            MS,
        }
        and
        e23 is not None
    ):

        e23_idx = int(
            e23[
                "pred_idx"
            ]
        )


        e23_prob = float(
            e23[
                "top_prob"
            ]
        )


        # MS -> MR

        if (
            original_idx == MS
            and
            e23_idx == MR
        ):

            proposal_counts[
                "MS->MR"
            ] += 1


            if (
                e23_prob
                >=
                MS_TO_MR_E23_MIN

                and

                e20_current_conf
                <=
                MS_TO_MR_E20_MAX
            ):

                final_idx = (
                    MR
                )

                accepted_direction = (
                    "MS->MR"
                )

                accepted_counts[
                    "MS->MR"
                ] += 1


        # MR -> MS

        elif (
            original_idx == MR
            and
            e23_idx == MS
        ):

            proposal_counts[
                "MR->MS"
            ] += 1


            if (
                e23_prob
                >=
                MR_TO_MS_E23_MIN

                and

                e20_current_conf
                <=
                MR_TO_MS_E20_MAX
            ):

                final_idx = (
                    MS
                )

                accepted_direction = (
                    "MR->MS"
                )

                accepted_counts[
                    "MR->MS"
                ] += 1


    records.append(
        {
            "final_idx":
                final_idx,

            "original_idx":
                original_idx,

            "e20_score":
                e20_score,

            "yolo_conf":
                yolo_conf,

            "classifier_component":
                classifier_component,

            "accepted_direction":
                accepted_direction,

            "e23_prob":
                e23_prob,
        }
    )


print()
print("=" * 120)
print("FROZEN E23-D ROUTING RECONSTRUCTION")
print("=" * 120)

print()
print(
    f"MS -> MR accepted : "
    f"{accepted_counts['MS->MR']:,}"
)

print(
    f"MR -> MS accepted : "
    f"{accepted_counts['MR->MS']:,}"
)


if not (
    accepted_counts[
        "MS->MR"
    ]
    == 606
    and
    accepted_counts[
        "MR->MS"
    ]
    == 196
):

    raise RuntimeError(
        "E23-D validation routing reconstruction failed."
    )


# ==================================================================================================
# 7. BUILD PREDICTIONS
# ==================================================================================================

def build_predictions(
    mode
):

    """
    mode:
        'E23D'
        'E24A'
        'E24B'
        'E24C'
    """

    predictions = []


    for item, r in zip(
        cache,
        records
    ):

        final_idx = int(
            r[
                "final_idx"
            ]
        )


        score = float(
            r[
                "e20_score"
            ]
        )


        # ==========================================================================================
        # E24-B base class-specific score
        # ==========================================================================================

        class_specific_score = (
            score
        )


        if final_idx == MR:

            class_specific_score = (
                r[
                    "yolo_conf"
                ]
                ** ALPHA_MR
            ) * (
                max(
                    r[
                        "classifier_component"
                    ],
                    1e-12
                )
                ** (
                    1.0
                    -
                    ALPHA_MR
                )
            )


        elif final_idx == MS:

            class_specific_score = (
                r[
                    "yolo_conf"
                ]
                ** ALPHA_MS
            ) * (
                max(
                    r[
                        "classifier_component"
                    ],
                    1e-12
                )
                ** (
                    1.0
                    -
                    ALPHA_MS
                )
            )


        # ==========================================================================================
        # Select experiment score
        # ==========================================================================================

        if mode == "E23D":

            score = float(
                r[
                    "e20_score"
                ]
            )


        elif mode == "E24A":

            score = float(
                r[
                    "e20_score"
                ]
            )


            if (
                r[
                    "accepted_direction"
                ]
                is not None
            ):

                score = (
                    max(
                        score,
                        1e-12
                    )
                    ** BETA
                ) * (
                    max(
                        float(
                            r[
                                "e23_prob"
                            ]
                        ),
                        1e-12
                    )
                    ** (
                        1.0
                        -
                        BETA
                    )
                )


        elif mode == "E24B":

            score = float(
                class_specific_score
            )


        elif mode == "E24C":

            score = float(
                class_specific_score
            )


            # --------------------------------------------------------------------------------------
            # Combine E24-B base score with E24-A specialist confidence,
            # but only for E23-D accepted overrides.
            # --------------------------------------------------------------------------------------

            if (
                r[
                    "accepted_direction"
                ]
                is not None
            ):

                score = (
                    max(
                        score,
                        1e-12
                    )
                    ** BETA
                ) * (
                    max(
                        float(
                            r[
                                "e23_prob"
                            ]
                        ),
                        1e-12
                    )
                    ** (
                        1.0
                        -
                        BETA
                    )
                )


        else:

            raise ValueError(
                f"Unknown mode: {mode}"
            )


        x1, y1, x2, y2 = [
            float(v)

            for v
            in item[
                "bbox"
            ]
        ]


        width = (
            x2
            -
            x1
        )

        height = (
            y2
            -
            y1
        )


        if (
            width <= 0
            or
            height <= 0
        ):

            continue


        predictions.append(
            {
                "image_id":
                    int(
                        item[
                            "image_id"
                        ]
                    ),

                "category_id":
                    int(
                        final_idx
                        + 1
                    ),

                "bbox": [
                    x1,
                    y1,
                    width,
                    height,
                ],

                "score":
                    float(
                        score
                    ),
            }
        )


    return predictions


# ==================================================================================================
# 8. EVALUATION
# ==================================================================================================

coco_gt = COCO(
    str(
        GT_PATH
    )
)


def valid_mean(
    values
):

    values = np.asarray(
        values
    )


    valid = values[
        values > -1
    ]


    if valid.size == 0:

        return float(
            "nan"
        )


    return float(
        valid.mean()
    )


def evaluate_predictions(
    predictions
):

    coco_dt = coco_gt.loadRes(
        predictions
    )


    evaluator = COCOeval(
        coco_gt,
        coco_dt,
        "bbox"
    )


    evaluator.params.maxDets = [
        1,
        10,
        100,
    ]


    evaluator.evaluate()
    evaluator.accumulate()


    precision = (
        evaluator.eval[
            "precision"
        ]
    )


    recall = (
        evaluator.eval[
            "recall"
        ]
    )


    ious = (
        evaluator.params.iouThrs
    )


    idx50 = int(
        np.where(
            np.isclose(
                ious,
                0.50
            )
        )[0][0]
    )


    idx75 = int(
        np.where(
            np.isclose(
                ious,
                0.75
            )
        )[0][0]
    )


    overall = {

        "AP50_95":
            valid_mean(
                precision[
                    :,
                    :,
                    :,
                    0,
                    -1
                ]
            ),

        "AP50":
            valid_mean(
                precision[
                    idx50,
                    :,
                    :,
                    0,
                    -1
                ]
            ),

        "AP75":
            valid_mean(
                precision[
                    idx75,
                    :,
                    :,
                    0,
                    -1
                ]
            ),

        "AR100":
            valid_mean(
                recall[
                    :,
                    :,
                    0,
                    -1
                ]
            ),
    }


    class_metrics = {}


    for idx, name in enumerate(
        CLASS_NAMES
    ):

        class_metrics[
            name
        ] = {

            "AP50":
                valid_mean(
                    precision[
                        idx50,
                        :,
                        idx,
                        0,
                        -1
                    ]
                ),

            "AP50_95":
                valid_mean(
                    precision[
                        :,
                        :,
                        idx,
                        0,
                        -1
                    ]
                ),
        }


    plastic = float(
        np.mean(
            [
                class_metrics[
                    CLASS_NAMES[
                        idx
                    ]
                ][
                    "AP50"
                ]

                for idx
                in PLASTIC_INDICES
            ]
        )
    )


    mr_ms = float(
        np.mean(
            [
                class_metrics[
                    "mixed_plastic_rigid"
                ][
                    "AP50"
                ],

                class_metrics[
                    "mixed_plastic_soft"
                ][
                    "AP50"
                ],
            ]
        )
    )


    return {

        "overall":
            overall,

        "class_metrics":
            class_metrics,

        "six_plastic_mean_AP50":
            plastic,

        "MR_MS_mean_AP50":
            mr_ms,
    }


# ==================================================================================================
# 9. EVALUATE ALL FOUR FOR DIRECT VALIDATION COMPARISON
# ==================================================================================================

results = {}


for mode in [
    "E23D",
    "E24A",
    "E24B",
    "E24C",
]:

    print()
    print("=" * 120)
    print(
        f"EVALUATING {mode}"
    )
    print("=" * 120)


    preds = build_predictions(
        mode
    )


    evaluation = evaluate_predictions(
        preds
    )


    results[
        mode
    ] = evaluation


    print()
    print(
        f"Six-plastic AP50 : "
        f"{evaluation['six_plastic_mean_AP50'] * 100:.4f}%"
    )

    print(
        f"MR AP50          : "
        f"{evaluation['class_metrics']['mixed_plastic_rigid']['AP50'] * 100:.4f}%"
    )

    print(
        f"MS AP50          : "
        f"{evaluation['class_metrics']['mixed_plastic_soft']['AP50'] * 100:.4f}%"
    )

    print(
        f"MR/MS mean AP50  : "
        f"{evaluation['MR_MS_mean_AP50'] * 100:.4f}%"
    )

    print(
        f"AP50-95          : "
        f"{evaluation['overall']['AP50_95'] * 100:.4f}%"
    )

    print(
        f"AP50             : "
        f"{evaluation['overall']['AP50'] * 100:.4f}%"
    )


# ==================================================================================================
# 10. FAIL-FAST REFERENCES
# ==================================================================================================

if abs(
    results[
        "E23D"
    ][
        "six_plastic_mean_AP50"
    ]
    -
    0.6835713966347757
) > 1e-6:

    raise RuntimeError(
        "E23-D validation reference not reproduced."
    )


# E24-A reference

if abs(
    results[
        "E24A"
    ][
        "six_plastic_mean_AP50"
    ]
    -
    0.684891
) > 0.00005:

    raise RuntimeError(
        "E24-A validation result was not reproduced."
    )


# E24-B reference

if abs(
    results[
        "E24B"
    ][
        "six_plastic_mean_AP50"
    ]
    -
    0.684103
) > 0.00005:

    raise RuntimeError(
        "E24-B validation result was not reproduced."
    )


print()
print(
    "E23-D / E24-A / E24-B validation reproduction PASSED."
)


# ==================================================================================================
# 11. E24-C DELTAS
# ==================================================================================================

e24c = (
    results[
        "E24C"
    ]
)


e23d = (
    results[
        "E23D"
    ]
)


delta = {

    "six_plastic_AP50_pp":

        (
            e24c[
                "six_plastic_mean_AP50"
            ]
            -
            e23d[
                "six_plastic_mean_AP50"
            ]
        )
        * 100,


    "MR_AP50_pp":

        (
            e24c[
                "class_metrics"
            ][
                "mixed_plastic_rigid"
            ][
                "AP50"
            ]
            -
            e23d[
                "class_metrics"
            ][
                "mixed_plastic_rigid"
            ][
                "AP50"
            ]
        )
        * 100,


    "MS_AP50_pp":

        (
            e24c[
                "class_metrics"
            ][
                "mixed_plastic_soft"
            ][
                "AP50"
            ]
            -
            e23d[
                "class_metrics"
            ][
                "mixed_plastic_soft"
            ][
                "AP50"
            ]
        )
        * 100,


    "MR_MS_mean_AP50_pp":

        (
            e24c[
                "MR_MS_mean_AP50"
            ]
            -
            e23d[
                "MR_MS_mean_AP50"
            ]
        )
        * 100,


    "AP50_95_pp":

        (
            e24c[
                "overall"
            ][
                "AP50_95"
            ]
            -
            e23d[
                "overall"
            ][
                "AP50_95"
            ]
        )
        * 100,


    "AP50_pp":

        (
            e24c[
                "overall"
            ][
                "AP50"
            ]
            -
            e23d[
                "overall"
            ][
                "AP50"
            ]
        )
        * 100,
}


# ==================================================================================================
# 12. FINAL TABLE
# ==================================================================================================

print()
print("=" * 120)
print("E24 FAMILY — VALIDATION COMPARISON")
print("=" * 120)


print(
    f"\n"
    f"{'System':10s}"
    f"{'Plastic':>12s}"
    f"{'MR':>10s}"
    f"{'MS':>10s}"
    f"{'MR/MS':>12s}"
    f"{'AP50-95':>12s}"
    f"{'AP50':>12s}"
)


print(
    "-" * 80
)


for mode in [
    "E23D",
    "E24A",
    "E24B",
    "E24C",
]:

    r = (
        results[
            mode
        ]
    )


    print(
        f"{mode:10s}"
        f"{r['six_plastic_mean_AP50']*100:11.4f}%"
        f"{r['class_metrics']['mixed_plastic_rigid']['AP50']*100:9.3f}%"
        f"{r['class_metrics']['mixed_plastic_soft']['AP50']*100:9.3f}%"
        f"{r['MR_MS_mean_AP50']*100:11.4f}%"
        f"{r['overall']['AP50_95']*100:11.4f}%"
        f"{r['overall']['AP50']*100:11.4f}%"
    )


print()
print("=" * 120)
print("E24-C DELTA VS E23-D")
print("=" * 120)

print()


for metric, value in (
    delta.items()
):

    print(
        f"{metric:28s}: "
        f"{value:+.4f} pp"
    )


# ==================================================================================================
# 13. DECISION
# ==================================================================================================

PROMOTE = (
    e24c[
        "six_plastic_mean_AP50"
    ]
    >
    e23d[
        "six_plastic_mean_AP50"
    ]
    +
    1e-12
)


print()
print("=" * 120)

if PROMOTE:

    print(
        "E24-C IMPROVES VALIDATION."
    )

else:

    print(
        "E24-C DOES NOT IMPROVE VALIDATION — REJECT."
    )

print("=" * 120)


# ==================================================================================================
# 14. SAVE
# ==================================================================================================

e24c_predictions = build_predictions(
    "E24C"
)


with open(
    PRED_PATH,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        e24c_predictions,
        f
    )


payload = {

    "experiment":
        "E24-C",

    "dataset":
        "validation",

    "test_used":
        False,

    "description":
        (
            "Combined validation-selected E24-A specialist override "
            "recalibration and E24-B class-specific MR/MS fusion"
        ),

    "base_pipeline":
        "E23-D",

    "frozen_configuration": {

        "beta":
            BETA,

        "alpha_MR":
            ALPHA_MR,

        "alpha_MS":
            ALPHA_MS,

        "MS_to_MR": {
            "e23_min":
                MS_TO_MR_E23_MIN,

            "e20_max":
                MS_TO_MR_E20_MAX,
        },

        "MR_to_MS": {
            "e23_min":
                MR_TO_MS_E23_MIN,

            "e20_max":
                MR_TO_MS_E20_MAX,
        },
    },

    "results":
        results,

    "delta_vs_E23D_pp":
        delta,

    "promote":
        bool(
            PROMOTE
        ),
}


with open(
    RESULTS_PATH,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        payload,
        f,
        indent=2
    )


summary_lines = [

    "E24-C — Combined Validation-Selected MR/MS Confidence Fusion",

    "=" * 105,

    "",

    "VALIDATION ONLY",

    "",

    f"beta     : {BETA:.2f}",

    f"alpha MR : {ALPHA_MR:.2f}",

    f"alpha MS : {ALPHA_MS:.2f}",

    "",

    f"Six-plastic AP50 : {e24c['six_plastic_mean_AP50']*100:.4f}%",

    f"MR AP50          : {e24c['class_metrics']['mixed_plastic_rigid']['AP50']*100:.4f}%",

    f"MS AP50          : {e24c['class_metrics']['mixed_plastic_soft']['AP50']*100:.4f}%",

    f"MR/MS mean AP50  : {e24c['MR_MS_mean_AP50']*100:.4f}%",

    f"AP50-95          : {e24c['overall']['AP50_95']*100:.4f}%",

    f"AP50             : {e24c['overall']['AP50']*100:.4f}%",

    "",

    f"Promote          : {PROMOTE}",

]


with open(
    SUMMARY_PATH,
    "w",
    encoding="utf-8"
) as f:

    f.write(
        "\n".join(
            summary_lines
        )
    )


print()
print("=" * 120)
print("E24-C COMPLETE")
print("=" * 120)

print()
print(
    f"Predictions : "
    f"{PRED_PATH}"
)

print(
    f"Results     : "
    f"{RESULTS_PATH}"
)

print(
    f"Summary     : "
    f"{SUMMARY_PATH}"
)

E24-C — COMBINED VALIDATION-SELECTED MR/MS CONFIDENCE FUSION

Cache entries : 55,605

Frozen configuration:
  MS -> MR : E23 >= 0.920, E20 <= 0.90
  MR -> MS : E23 >= 0.997, E20 <= 0.95
  beta      : 0.35
  alpha MR  : 0.85
  alpha MS  : 0.50

FROZEN E23-D ROUTING RECONSTRUCTION

MS -> MR accepted : 606
MR -> MS accepted : 196
loading annotations into memory...
Done (t=0.04s)
creating index...
index created!

EVALUATING E23D
Loading and preparing results...
DONE (t=0.07s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=3.93s).
Accumulating evaluation results...
DONE (t=0.50s).

Six-plastic AP50 : 68.3571%
MR AP50          : 51.7173%
MS AP50          : 42.4042%
MR/MS mean AP50  : 47.0608%
AP50-95          : 45.9348%
AP50             : 61.4121%

EVALUATING E24A
Loading and preparing results...
DONE (t=0.06s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=3.83s).
Accumulating

In [45]:
## E24-C — Frozen Held-Out Test Evaluation of Combined MR/MS Class-Specific Fusion and Specialist Override Recalibration
# ==================================================================================================
# E24-C — FROZEN HELD-OUT TEST EVALUATION
#         COMBINED MR/MS CLASS-SPECIFIC FUSION + SPECIALIST OVERRIDE RECALIBRATION
#
# VALIDATION-SELECTED CONFIGURATION
# --------------------------------------------------------------------------------------------------
#
# E23-D routing:
#   MS -> MR : E23 >= 0.920 AND E20 <= 0.90
#   MR -> MS : E23 >= 0.997 AND E20 <= 0.95
#
# E24-A:
#   beta = 0.35
#
# E24-B:
#   alpha_MR = 0.85
#   alpha_MS = 0.50
#
#
# SCORE LOGIC
# --------------------------------------------------------------------------------------------------
#
# STEP 1:
# For every FINAL MR/MS detection:
#
#   class_score =
#       YOLO_conf ^ alpha_class
#       *
#       recovered_E20_classifier_component ^ (1 - alpha_class)
#
#
# STEP 2:
# ONLY for detections actually overridden by E23-D:
#
#   final_score =
#       class_score ^ beta
#       *
#       E23_confidence ^ (1 - beta)
#
#
# Other final classes:
#   original E20/E23-D score unchanged
#
#
# NO:
#   - YOLO inference
#   - MobileNet inference
#   - ConvNeXt inference
#   - E23-A inference
#   - alpha search
#   - beta search
#   - routing search
#   - threshold tuning
#
# TEST = frozen evaluation only.
# ==================================================================================================

import json
from pathlib import Path
from collections import Counter

import numpy as np

from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval


# ==================================================================================================
# 1. PATHS
# ==================================================================================================

BASE = Path(
    r"C:\Users\varda\Documents\_My Computer\COMMON_space\Learning"
    r"\Upgrad_EPGP_MS\upgrad_IIITB_MS_course\MS_LJMU_Material"
)

RUNS_ROOT = (
    BASE
    / "Thesis_Code"
    / "runs"
    / "sortwaste"
)


E20_PRED_PATH = (
    RUNS_ROOT
    / "FINAL_TEST_E20A"
    / "E20A_test_predictions.json"
)


E20_GT_PATH = (
    RUNS_ROOT
    / "FINAL_TEST_E20A"
    / "E20A_test_gt_7class.json"
)


MATCH_CACHE_PATH = (
    RUNS_ROOT
    / "FINAL_TEST_E23D_CORRECTED"
    / "E23D_E20A_E12_matched_cache.json"
)


E23_CACHE_PATH = (
    RUNS_ROOT
    / "FINAL_TEST_E23D_CORRECTED"
    / "E23D_E23A_MRMS_test_cache.json"
)


E23D_CONFIG_PATH = (
    RUNS_ROOT
    / "E23D_confidence_contrastive_MRMS"
    / "E23D_best_configuration.json"
)


E24A_CONFIG_PATH = (
    RUNS_ROOT
    / "E24A_E23D_override_confidence_recalibration"
    / "E24A_best_configuration.json"
)


E24B_CONFIG_PATH = (
    RUNS_ROOT
    / "E24B_class_specific_MRMS_fusion"
    / "E24B_best_configuration.json"
)


OUTPUT_DIR = (
    RUNS_ROOT
    / "FINAL_TEST_E24C"
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)


PREDICTIONS_PATH = (
    OUTPUT_DIR
    / "E24C_test_predictions.json"
)


RESULTS_PATH = (
    OUTPUT_DIR
    / "E24C_test_results.json"
)


SUMMARY_PATH = (
    OUTPUT_DIR
    / "E24C_test_summary.txt"
)


for path, label in [

    (E20_PRED_PATH, "E20-A TEST predictions"),
    (E20_GT_PATH, "TEST GT"),
    (MATCH_CACHE_PATH, "E20/E12 matched TEST cache"),
    (E23_CACHE_PATH, "E23-A TEST cache"),
    (E23D_CONFIG_PATH, "E23-D validation config"),
    (E24A_CONFIG_PATH, "E24-A validation config"),
    (E24B_CONFIG_PATH, "E24-B validation config"),

]:

    if not path.exists():

        raise FileNotFoundError(
            f"{label} not found:\n{path}"
        )


# ==================================================================================================
# 2. CLASSES
# ==================================================================================================

CLASS_NAMES = [
    "ecal",
    "hdpe",
    "mixed_plastic_rigid",
    "mixed_plastic_soft",
    "non_plastic",
    "pet",
    "pet_oil",
]


ECAL = 0
HDPE = 1
MR = 2
MS = 3
NON_PLASTIC = 4
PET = 5
PET_OIL = 6


PLASTIC_INDICES = [
    ECAL,
    HDPE,
    MR,
    MS,
    PET,
    PET_OIL,
]


# ==================================================================================================
# 3. LOAD FILES
# ==================================================================================================

print("=" * 120)
print("E24-C — FROZEN HELD-OUT TEST EVALUATION")
print("=" * 120)


with open(
    E20_PRED_PATH,
    "r",
    encoding="utf-8"
) as f:

    e20_predictions = json.load(f)


with open(
    MATCH_CACHE_PATH,
    "r",
    encoding="utf-8"
) as f:

    matched_cache = json.load(f)


with open(
    E23_CACHE_PATH,
    "r",
    encoding="utf-8"
) as f:

    e23_cache = json.load(f)


with open(
    E23D_CONFIG_PATH,
    "r",
    encoding="utf-8"
) as f:

    e23d_config = json.load(f)


with open(
    E24A_CONFIG_PATH,
    "r",
    encoding="utf-8"
) as f:

    e24a_config = json.load(f)


with open(
    E24B_CONFIG_PATH,
    "r",
    encoding="utf-8"
) as f:

    e24b_config = json.load(f)


print()
print(
    f"E20-A predictions : {len(e20_predictions):,}"
)

print(
    f"Matched cache     : {len(matched_cache):,}"
)

print(
    f"E23-A cache       : {len(e23_cache):,}"
)


# ==================================================================================================
# 4. SAFETY CHECKS
# ==================================================================================================

if len(e20_predictions) != 57487:

    raise RuntimeError(
        "Expected 57,487 E20-A TEST predictions."
    )


if len(matched_cache) != 57487:

    raise RuntimeError(
        "Expected 57,487 matched-cache entries."
    )


if len(e23_cache) != 17193:

    raise RuntimeError(
        "Expected 17,193 E23-A TEST entries."
    )


# ==================================================================================================
# 5. LOAD FROZEN PARAMETERS
# ==================================================================================================

e23d_best = (
    e23d_config[
        "best_configuration"
    ]
)


MS_TO_MR_E23_MIN = float(
    e23d_best[
        "MS_to_MR"
    ][
        "e23_min"
    ]
)

MS_TO_MR_E20_MAX = float(
    e23d_best[
        "MS_to_MR"
    ][
        "e20_max"
    ]
)

MR_TO_MS_E23_MIN = float(
    e23d_best[
        "MR_to_MS"
    ][
        "e23_min"
    ]
)

MR_TO_MS_E20_MAX = float(
    e23d_best[
        "MR_to_MS"
    ][
        "e20_max"
    ]
)


BETA = float(
    e24a_config[
        "best_beta"
    ]
)


e24b_best = (
    e24b_config[
        "best"
    ]
)


ALPHA_MR = float(
    e24b_best[
        "alpha_MR"
    ]
)

ALPHA_MS = float(
    e24b_best[
        "alpha_MS"
    ]
)


if not (
    np.isclose(MS_TO_MR_E23_MIN, 0.920)
    and
    np.isclose(MS_TO_MR_E20_MAX, 0.90)
    and
    np.isclose(MR_TO_MS_E23_MIN, 0.997)
    and
    np.isclose(MR_TO_MS_E20_MAX, 0.95)
    and
    np.isclose(BETA, 0.35)
    and
    np.isclose(ALPHA_MR, 0.85)
    and
    np.isclose(ALPHA_MS, 0.50)
):

    raise RuntimeError(
        "Frozen E24-C configuration does not match validation-selected parameters."
    )


print()
print("=" * 120)
print("FROZEN CONFIGURATION")
print("=" * 120)

print()
print(
    f"MS -> MR : E23 >= {MS_TO_MR_E23_MIN:.3f}, "
    f"E20 <= {MS_TO_MR_E20_MAX:.2f}"
)

print(
    f"MR -> MS : E23 >= {MR_TO_MS_E23_MIN:.3f}, "
    f"E20 <= {MR_TO_MS_E20_MAX:.2f}"
)

print(
    f"beta      : {BETA:.2f}"
)

print(
    f"alpha MR  : {ALPHA_MR:.2f}"
)

print(
    f"alpha MS  : {ALPHA_MS:.2f}"
)


# ==================================================================================================
# 6. MATCH CACHE LOOKUP
# ==================================================================================================

record_lookup = {}


for record in matched_cache:

    idx = int(
        record[
            "e20_index"
        ]
    )


    if idx in record_lookup:

        raise RuntimeError(
            f"Duplicate E20 index: {idx}"
        )


    record_lookup[
        idx
    ] = record


if len(record_lookup) != 57487:

    raise RuntimeError(
        "Matched-cache lookup incomplete."
    )


# ==================================================================================================
# 7. COCO EVALUATION HELPERS
# ==================================================================================================

coco_gt = COCO(
    str(
        E20_GT_PATH
    )
)


def valid_mean(
    values
):

    values = np.asarray(
        values
    )


    valid = values[
        values > -1
    ]


    if valid.size == 0:

        return float(
            "nan"
        )


    return float(
        valid.mean()
    )


def evaluate_predictions(
    predictions
):

    coco_dt = coco_gt.loadRes(
        predictions
    )


    evaluator = COCOeval(
        coco_gt,
        coco_dt,
        "bbox"
    )


    evaluator.params.maxDets = [
        1,
        10,
        100,
    ]


    evaluator.evaluate()
    evaluator.accumulate()


    precision = (
        evaluator.eval[
            "precision"
        ]
    )

    recall = (
        evaluator.eval[
            "recall"
        ]
    )


    ious = (
        evaluator.params.iouThrs
    )


    idx50 = int(
        np.where(
            np.isclose(
                ious,
                0.50
            )
        )[0][0]
    )


    idx75 = int(
        np.where(
            np.isclose(
                ious,
                0.75
            )
        )[0][0]
    )


    overall = {

        "AP50_95":
            valid_mean(
                precision[
                    :,
                    :,
                    :,
                    0,
                    -1
                ]
            ),

        "AP50":
            valid_mean(
                precision[
                    idx50,
                    :,
                    :,
                    0,
                    -1
                ]
            ),

        "AP75":
            valid_mean(
                precision[
                    idx75,
                    :,
                    :,
                    0,
                    -1
                ]
            ),

        "AR100":
            valid_mean(
                recall[
                    :,
                    :,
                    0,
                    -1
                ]
            ),
    }


    class_metrics = {}


    for idx, name in enumerate(
        CLASS_NAMES
    ):

        class_metrics[
            name
        ] = {

            "AP50":
                valid_mean(
                    precision[
                        idx50,
                        :,
                        idx,
                        0,
                        -1
                    ]
                ),

            "AP50_95":
                valid_mean(
                    precision[
                        :,
                        :,
                        idx,
                        0,
                        -1
                    ]
                ),
        }


    six_plastic = float(
        np.mean(
            [
                class_metrics[
                    CLASS_NAMES[
                        idx
                    ]
                ][
                    "AP50"
                ]

                for idx
                in PLASTIC_INDICES
            ]
        )
    )


    six_plastic_ap = float(
        np.mean(
            [
                class_metrics[
                    CLASS_NAMES[
                        idx
                    ]
                ][
                    "AP50_95"
                ]

                for idx
                in PLASTIC_INDICES
            ]
        )
    )


    mr_ms = float(
        np.mean(
            [
                class_metrics[
                    "mixed_plastic_rigid"
                ][
                    "AP50"
                ],

                class_metrics[
                    "mixed_plastic_soft"
                ][
                    "AP50"
                ],
            ]
        )
    )


    return {

        "overall":
            overall,

        "class_metrics":
            class_metrics,

        "six_plastic_mean_AP50":
            six_plastic,

        "six_plastic_mean_AP50_95":
            six_plastic_ap,

        "MR_MS_mean_AP50":
            mr_ms,
    }


# ==================================================================================================
# 8. VERIFY E20-A
# ==================================================================================================

print()
print("=" * 120)
print("VERIFYING E20-A TEST BASELINE")
print("=" * 120)


e20_eval = evaluate_predictions(
    e20_predictions
)


print()
print(
    f"AP50-95      : {e20_eval['overall']['AP50_95']*100:.4f}%"
)

print(
    f"AP50         : {e20_eval['overall']['AP50']*100:.4f}%"
)

print(
    f"Plastic AP50 : {e20_eval['six_plastic_mean_AP50']*100:.4f}%"
)


if not (
    abs(
        e20_eval[
            "overall"
        ][
            "AP50_95"
        ]
        -
        0.462612
    )
    < 0.001

    and

    abs(
        e20_eval[
            "overall"
        ][
            "AP50"
        ]
        -
        0.623271
    )
    < 0.001

    and

    abs(
        e20_eval[
            "six_plastic_mean_AP50"
        ]
        -
        0.694353
    )
    < 0.001
):

    raise RuntimeError(
        "E20-A TEST baseline verification failed."
    )


print()
print(
    "E20-A baseline verification PASSED."
)


# ==================================================================================================
# 9. RECONSTRUCT E23-D ROUTING
# ==================================================================================================

e23d_predictions = [
    dict(
        p
    )

    for p
    in e20_predictions
]


accepted_records = {}
proposal_counts = Counter()
accepted_counts = Counter()


for idx, original in enumerate(
    e20_predictions
):

    current_idx = (
        int(
            original[
                "category_id"
            ]
        )
        - 1
    )


    if current_idx not in {
        MR,
        MS,
    }:

        continue


    key = str(
        idx
    )


    if key not in e23_cache:

        raise RuntimeError(
            f"Missing E23-A TEST cache for E20 index {idx}"
        )


    e23 = (
        e23_cache[
            key
        ]
    )


    e23_idx = int(
        e23[
            "pred_global_idx"
        ]
    )


    e23_prob = float(
        e23[
            "top_prob"
        ]
    )


    e20_conf = float(
        record_lookup[
            idx
        ][
            "e20_class_conf"
        ]
    )


    accepted_direction = None


    # ----------------------------------------------------------------------------------------------
    # MS -> MR
    # ----------------------------------------------------------------------------------------------

    if (
        current_idx == MS
        and
        e23_idx == MR
    ):

        proposal_counts[
            "MS->MR"
        ] += 1


        if (
            e23_prob >= MS_TO_MR_E23_MIN
            and
            e20_conf <= MS_TO_MR_E20_MAX
        ):

            e23d_predictions[
                idx
            ][
                "category_id"
            ] = (
                MR + 1
            )


            accepted_counts[
                "MS->MR"
            ] += 1


            accepted_direction = (
                "MS->MR"
            )


    # ----------------------------------------------------------------------------------------------
    # MR -> MS
    # ----------------------------------------------------------------------------------------------

    elif (
        current_idx == MR
        and
        e23_idx == MS
    ):

        proposal_counts[
            "MR->MS"
        ] += 1


        if (
            e23_prob >= MR_TO_MS_E23_MIN
            and
            e20_conf <= MR_TO_MS_E20_MAX
        ):

            e23d_predictions[
                idx
            ][
                "category_id"
            ] = (
                MS + 1
            )


            accepted_counts[
                "MR->MS"
            ] += 1


            accepted_direction = (
                "MR->MS"
            )


    if accepted_direction is not None:

        accepted_records[
            idx
        ] = {

            "direction":
                accepted_direction,

            "e23_prob":
                e23_prob,
        }


print()
print("=" * 120)
print("RECONSTRUCTED E23-D TEST ROUTING")
print("=" * 120)

print()
print(
    f"MS -> MR proposals : {proposal_counts['MS->MR']:,}"
)

print(
    f"MR -> MS proposals : {proposal_counts['MR->MS']:,}"
)

print()
print(
    f"MS -> MR accepted  : {accepted_counts['MS->MR']:,}"
)

print(
    f"MR -> MS accepted  : {accepted_counts['MR->MS']:,}"
)

print(
    f"Total accepted     : {len(accepted_records):,}"
)


if not (
    accepted_counts[
        "MS->MR"
    ]
    == 691
    and
    accepted_counts[
        "MR->MS"
    ]
    == 311
    and
    len(accepted_records)
    == 1002
):

    raise RuntimeError(
        "Corrected E23-D TEST routing was not reproduced."
    )


# ==================================================================================================
# 10. VERIFY E23-D BASELINE
# ==================================================================================================

print()
print("=" * 120)
print("VERIFYING CORRECTED E23-D TEST BASELINE")
print("=" * 120)


e23d_eval = evaluate_predictions(
    e23d_predictions
)


print()
print(
    f"AP50-95               : {e23d_eval['overall']['AP50_95']*100:.4f}%"
)

print(
    f"AP50                  : {e23d_eval['overall']['AP50']*100:.4f}%"
)

print(
    f"AP75                  : {e23d_eval['overall']['AP75']*100:.4f}%"
)

print(
    f"AR100                 : {e23d_eval['overall']['AR100']*100:.4f}%"
)

print(
    f"Six-plastic mean AP50: {e23d_eval['six_plastic_mean_AP50']*100:.4f}%"
)

print(
    f"MR AP50               : {e23d_eval['class_metrics']['mixed_plastic_rigid']['AP50']*100:.4f}%"
)

print(
    f"MS AP50               : {e23d_eval['class_metrics']['mixed_plastic_soft']['AP50']*100:.4f}%"
)


if not (
    abs(
        e23d_eval[
            "overall"
        ][
            "AP50_95"
        ]
        -
        0.462851
    )
    < 1e-5

    and

    abs(
        e23d_eval[
            "overall"
        ][
            "AP50"
        ]
        -
        0.623548
    )
    < 1e-5

    and

    abs(
        e23d_eval[
            "six_plastic_mean_AP50"
        ]
        -
        0.694677
    )
    < 1e-5
):

    raise RuntimeError(
        "Corrected E23-D TEST baseline not reproduced."
    )


print()
print(
    "Corrected E23-D TEST reproduction PASSED."
)


# ==================================================================================================
# 11. BUILD E24-C
# ==================================================================================================

print()
print("=" * 120)
print("APPLYING FROZEN E24-C SCORING")
print("=" * 120)


e24c_predictions = [
    dict(
        p
    )

    for p
    in e23d_predictions
]


mr_rescored = 0
ms_rescored = 0
override_recalibrated = 0


classifier_components = []


for idx, prediction in enumerate(
    e23d_predictions
):

    final_idx = (
        int(
            prediction[
                "category_id"
            ]
        )
        - 1
    )


    # ----------------------------------------------------------------------------------------------
    # Only final MR/MS get E24-B-style class-specific fusion.
    # ----------------------------------------------------------------------------------------------

    if final_idx not in {
        MR,
        MS,
    }:

        continue


    record = (
        record_lookup[
            idx
        ]
    )


    e20_score = max(
        float(
            e20_predictions[
                idx
            ][
                "score"
            ]
        ),
        1e-12
    )


    yolo_conf = max(
        float(
            record[
                "yolo_conf"
            ]
        ),
        1e-12
    )


    # ----------------------------------------------------------------------------------------------
    # Recover exact E20 classifier component.
    # ----------------------------------------------------------------------------------------------

    classifier_component = (
        e20_score
        /
        (
            yolo_conf
            ** 0.70
        )
    ) ** (
        1.0
        /
        0.30
    )


    classifier_component = float(
        np.clip(
            classifier_component,
            0.0,
            1.0
        )
    )


    classifier_components.append(
        classifier_component
    )


    # ----------------------------------------------------------------------------------------------
    # Class-specific fusion
    # ----------------------------------------------------------------------------------------------

    if final_idx == MR:

        alpha = (
            ALPHA_MR
        )

        mr_rescored += 1


    else:

        alpha = (
            ALPHA_MS
        )

        ms_rescored += 1


    class_specific_score = (
        yolo_conf
        ** alpha
    ) * (
        max(
            classifier_component,
            1e-12
        )
        ** (
            1.0
            -
            alpha
        )
    )


    final_score = float(
        class_specific_score
    )


    # ----------------------------------------------------------------------------------------------
    # If E23-D actually changed the class, apply specialist recalibration.
    # ----------------------------------------------------------------------------------------------

    if idx in accepted_records:

        e23_prob = max(
            float(
                accepted_records[
                    idx
                ][
                    "e23_prob"
                ]
            ),
            1e-12
        )


        final_score = (
            max(
                class_specific_score,
                1e-12
            )
            ** BETA
        ) * (
            e23_prob
            ** (
                1.0
                -
                BETA
            )
        )


        override_recalibrated += 1


    e24c_predictions[
        idx
    ][
        "score"
    ] = float(
        final_score
    )


print()
print(
    f"Final MR rescored          : {mr_rescored:,}"
)

print(
    f"Final MS rescored          : {ms_rescored:,}"
)

print(
    f"Total MR/MS rescored       : {mr_rescored + ms_rescored:,}"
)

print(
    f"Override recalibrated      : {override_recalibrated:,}"
)


print()
print(
    f"Classifier component min   : {np.min(classifier_components):.6f}"
)

print(
    f"Classifier component mean  : {np.mean(classifier_components):.6f}"
)

print(
    f"Classifier component median: {np.median(classifier_components):.6f}"
)

print(
    f"Classifier component max   : {np.max(classifier_components):.6f}"
)


if (
    mr_rescored
    +
    ms_rescored
) != 17193:

    raise RuntimeError(
        "Expected exactly 17,193 final MR/MS detections to be rescored."
    )


if override_recalibrated != 1002:

    raise RuntimeError(
        "Expected exactly 1,002 accepted E23-D overrides to receive specialist recalibration."
    )


# ==================================================================================================
# 12. STRUCTURAL SAFETY CHECK
# ==================================================================================================

image_changes = 0
bbox_changes = 0
class_changes = 0
score_changes = 0
illegal_score_changes = 0


for idx, (
    before,
    after
) in enumerate(
    zip(
        e23d_predictions,
        e24c_predictions
    )
):

    if before[
        "image_id"
    ] != after[
        "image_id"
    ]:

        image_changes += 1


    if before[
        "bbox"
    ] != after[
        "bbox"
    ]:

        bbox_changes += 1


    if before[
        "category_id"
    ] != after[
        "category_id"
    ]:

        class_changes += 1


    if float(
        before[
            "score"
        ]
    ) != float(
        after[
            "score"
        ]
    ):

        score_changes += 1


        final_idx = (
            int(
                after[
                    "category_id"
                ]
            )
            -
            1
        )


        if final_idx not in {
            MR,
            MS,
        }:

            illegal_score_changes += 1


print()
print("=" * 120)
print("E24-C STRUCTURAL SAFETY CHECK")
print("=" * 120)

print()
print(
    f"Image-ID changes vs E23-D : {image_changes}"
)

print(
    f"BBox changes vs E23-D     : {bbox_changes}"
)

print(
    f"Class changes vs E23-D    : {class_changes}"
)

print(
    f"Score changes vs E23-D    : {score_changes:,}"
)

print(
    f"Illegal score changes     : {illegal_score_changes}"
)


if (
    image_changes != 0
    or
    bbox_changes != 0
    or
    class_changes != 0
    or
    illegal_score_changes != 0
):

    raise RuntimeError(
        "E24-C altered something outside MR/MS confidence scoring."
    )


# ==================================================================================================
# 13. TEST EVALUATION
# ==================================================================================================

print()
print("=" * 120)
print("E24-C — FROZEN HELD-OUT TEST COCO EVALUATION")
print("=" * 120)


e24c_eval = evaluate_predictions(
    e24c_predictions
)


# ==================================================================================================
# 14. RESULTS
# ==================================================================================================

print()
print("=" * 120)
print("E24-C — HELD-OUT TEST RESULTS")
print("=" * 120)


print()
print(
    f"AP50-95                : {e24c_eval['overall']['AP50_95']*100:.4f}%"
)

print(
    f"AP50                   : {e24c_eval['overall']['AP50']*100:.4f}%"
)

print(
    f"AP75                   : {e24c_eval['overall']['AP75']*100:.4f}%"
)

print(
    f"AR100                  : {e24c_eval['overall']['AR100']*100:.4f}%"
)


print()
print(
    f"Six-plastic mean AP50 : {e24c_eval['six_plastic_mean_AP50']*100:.4f}%"
)

print(
    f"Six-plastic AP50-95   : {e24c_eval['six_plastic_mean_AP50_95']*100:.4f}%"
)

print(
    f"MR/MS mean AP50       : {e24c_eval['MR_MS_mean_AP50']*100:.4f}%"
)


# ==================================================================================================
# 15. CLASS-WISE
# ==================================================================================================

print()
print("=" * 120)
print("E24-C — CLASS-WISE TEST RESULTS")
print("=" * 120)


print(
    f"\n"
    f"{'Class':30s}"
    f"{'AP50':>15s}"
    f"{'AP50-95':>15s}"
)

print(
    "-" * 60
)


for class_name in CLASS_NAMES:

    values = (
        e24c_eval[
            "class_metrics"
        ][
            class_name
        ]
    )


    print(
        f"{class_name:30s}"
        f"{values['AP50']*100:14.2f}%"
        f"{values['AP50_95']*100:14.2f}%"
    )


# ==================================================================================================
# 16. DELTA VS E23-D
# ==================================================================================================

delta_vs_e23d = {

    "AP50_95_pp":
        (
            e24c_eval[
                "overall"
            ][
                "AP50_95"
            ]
            -
            e23d_eval[
                "overall"
            ][
                "AP50_95"
            ]
        )
        * 100,


    "AP50_pp":
        (
            e24c_eval[
                "overall"
            ][
                "AP50"
            ]
            -
            e23d_eval[
                "overall"
            ][
                "AP50"
            ]
        )
        * 100,


    "AP75_pp":
        (
            e24c_eval[
                "overall"
            ][
                "AP75"
            ]
            -
            e23d_eval[
                "overall"
            ][
                "AP75"
            ]
        )
        * 100,


    "AR100_pp":
        (
            e24c_eval[
                "overall"
            ][
                "AR100"
            ]
            -
            e23d_eval[
                "overall"
            ][
                "AR100"
            ]
        )
        * 100,


    "six_plastic_AP50_pp":
        (
            e24c_eval[
                "six_plastic_mean_AP50"
            ]
            -
            e23d_eval[
                "six_plastic_mean_AP50"
            ]
        )
        * 100,


    "MR_AP50_pp":
        (
            e24c_eval[
                "class_metrics"
            ][
                "mixed_plastic_rigid"
            ][
                "AP50"
            ]
            -
            e23d_eval[
                "class_metrics"
            ][
                "mixed_plastic_rigid"
            ][
                "AP50"
            ]
        )
        * 100,


    "MS_AP50_pp":
        (
            e24c_eval[
                "class_metrics"
            ][
                "mixed_plastic_soft"
            ][
                "AP50"
            ]
            -
            e23d_eval[
                "class_metrics"
            ][
                "mixed_plastic_soft"
            ][
                "AP50"
            ]
        )
        * 100,


    "MR_MS_mean_AP50_pp":
        (
            e24c_eval[
                "MR_MS_mean_AP50"
            ]
            -
            e23d_eval[
                "MR_MS_mean_AP50"
            ]
        )
        * 100,
}


print()
print("=" * 120)
print("DELTA VS CORRECTED E23-D TEST")
print("=" * 120)

print()


for metric, value in delta_vs_e23d.items():

    print(
        f"{metric:30s}: {value:+.4f} pp"
    )


# ==================================================================================================
# 17. UNCHANGED-CLASS SANITY CHECK
# ==================================================================================================

print()
print("=" * 120)
print("UNCHANGED-CLASS SANITY CHECK")
print("=" * 120)

print()


for class_name in [
    "ecal",
    "hdpe",
    "non_plastic",
    "pet",
    "pet_oil",
]:

    before = (
        e23d_eval[
            "class_metrics"
        ][
            class_name
        ][
            "AP50"
        ]
    )


    after = (
        e24c_eval[
            "class_metrics"
        ][
            class_name
        ][
            "AP50"
        ]
    )


    delta = (
        after
        -
        before
    ) * 100


    print(
        f"{class_name:25s}: {delta:+.6f} pp"
    )


# ==================================================================================================
# 18. COMPARISON
# ==================================================================================================

SORTWASTE_PLASTIC_AP50 = (
    69.3800
)


E21B = {
    "AP50_95": 46.2881,
    "AP50": 62.3518,
    "plastic": 69.4212,
    "MR": 48.70,
    "MS": 42.53,
    "MRMS": 45.6119,
}


print()
print("=" * 120)
print("FINAL COMPARISON")
print("=" * 120)


print(
    f"\n"
    f"{'System':12s}"
    f"{'AP50-95':>12s}"
    f"{'AP50':>12s}"
    f"{'Plastic':>12s}"
    f"{'MR':>10s}"
    f"{'MS':>10s}"
    f"{'MR/MS':>12s}"
)


print(
    "-" * 80
)


print(
    f"{'E20-A':12s}"
    f"{e20_eval['overall']['AP50_95']*100:11.4f}%"
    f"{e20_eval['overall']['AP50']*100:11.4f}%"
    f"{e20_eval['six_plastic_mean_AP50']*100:11.4f}%"
    f"{e20_eval['class_metrics']['mixed_plastic_rigid']['AP50']*100:9.2f}%"
    f"{e20_eval['class_metrics']['mixed_plastic_soft']['AP50']*100:9.2f}%"
    f"{e20_eval['MR_MS_mean_AP50']*100:11.4f}%"
)


print(
    f"{'E21-B':12s}"
    f"{E21B['AP50_95']:11.4f}%"
    f"{E21B['AP50']:11.4f}%"
    f"{E21B['plastic']:11.4f}%"
    f"{E21B['MR']:9.2f}%"
    f"{E21B['MS']:9.2f}%"
    f"{E21B['MRMS']:11.4f}%"
)


print(
    f"{'E23-D':12s}"
    f"{e23d_eval['overall']['AP50_95']*100:11.4f}%"
    f"{e23d_eval['overall']['AP50']*100:11.4f}%"
    f"{e23d_eval['six_plastic_mean_AP50']*100:11.4f}%"
    f"{e23d_eval['class_metrics']['mixed_plastic_rigid']['AP50']*100:9.2f}%"
    f"{e23d_eval['class_metrics']['mixed_plastic_soft']['AP50']*100:9.2f}%"
    f"{e23d_eval['MR_MS_mean_AP50']*100:11.4f}%"
)


print(
    f"{'E24-C':12s}"
    f"{e24c_eval['overall']['AP50_95']*100:11.4f}%"
    f"{e24c_eval['overall']['AP50']*100:11.4f}%"
    f"{e24c_eval['six_plastic_mean_AP50']*100:11.4f}%"
    f"{e24c_eval['class_metrics']['mixed_plastic_rigid']['AP50']*100:9.2f}%"
    f"{e24c_eval['class_metrics']['mixed_plastic_soft']['AP50']*100:9.2f}%"
    f"{e24c_eval['MR_MS_mean_AP50']*100:11.4f}%"
)


# ==================================================================================================
# 19. SORTWASTE + GAP TO 70
# ==================================================================================================

plastic_score = (
    e24c_eval[
        "six_plastic_mean_AP50"
    ]
    * 100
)


sortwaste_delta = (
    plastic_score
    -
    SORTWASTE_PLASTIC_AP50
)


gap_to_70 = (
    70.0
    -
    plastic_score
)


print()
print("=" * 120)
print("ALIGNED SIX-PLASTIC AP50 COMPARISON")
print("=" * 120)

print()
print(
    f"SortWaste YOLOv11 : {SORTWASTE_PLASTIC_AP50:.4f}%"
)

print(
    f"E24-C             : {plastic_score:.4f}%"
)

print(
    f"Difference        : {sortwaste_delta:+.4f} pp"
)

print()
print(
    f"Gap to 70.0000%   : {gap_to_70:+.4f} pp"
)


# ==================================================================================================
# 20. SAVE
# ==================================================================================================

with open(
    PREDICTIONS_PATH,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        e24c_predictions,
        f
    )


result_payload = {

    "experiment":
        "E24-C",

    "split":
        "held-out TEST",

    "test_used_for_parameter_selection":
        False,

    "interpretation_note":
        (
            "E24-C combines E24-A and E24-B validation-selected parameters. "
            "Because E24-A and E24-B TEST outcomes were already observed before "
            "this combined TEST evaluation, E24-C should be described as a post-hoc "
            "combined ablation rather than a fully independent final-model selection."
        ),

    "base_pipeline":
        "E23-D",

    "frozen_configuration": {

        "beta":
            BETA,

        "alpha_MR":
            ALPHA_MR,

        "alpha_MS":
            ALPHA_MS,

        "MS_to_MR": {

            "e23_min":
                MS_TO_MR_E23_MIN,

            "e20_max":
                MS_TO_MR_E20_MAX,
        },

        "MR_to_MS": {

            "e23_min":
                MR_TO_MS_E23_MIN,

            "e20_max":
                MR_TO_MS_E20_MAX,
        },
    },

    "routing_counts": {

        "MS_to_MR":
            int(
                accepted_counts[
                    "MS->MR"
                ]
            ),

        "MR_to_MS":
            int(
                accepted_counts[
                    "MR->MS"
                ]
            ),
    },

    "rescored_counts": {

        "final_MR":
            int(
                mr_rescored
            ),

        "final_MS":
            int(
                ms_rescored
            ),

        "override_recalibrated":
            int(
                override_recalibrated
            ),
    },

    "E20A":
        e20_eval,

    "E23D":
        e23d_eval,

    "E24C":
        e24c_eval,

    "delta_vs_E23D_pp":
        delta_vs_e23d,

    "sortwaste_six_plastic_AP50":
        SORTWASTE_PLASTIC_AP50,

    "delta_vs_sortwaste_pp":
        sortwaste_delta,

    "gap_to_70_pp":
        gap_to_70,
}


with open(
    RESULTS_PATH,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        result_payload,
        f,
        indent=2
    )


summary_lines = [

    "E24-C — Frozen Held-Out TEST Evaluation",

    "=" * 105,

    "",

    "FROZEN CONFIGURATION",

    f"beta                  : {BETA:.2f}",

    f"alpha MR              : {ALPHA_MR:.2f}",

    f"alpha MS              : {ALPHA_MS:.2f}",

    "",

    f"MS -> MR accepted     : {accepted_counts['MS->MR']:,}",

    f"MR -> MS accepted     : {accepted_counts['MR->MS']:,}",

    "",

    f"Final MR rescored     : {mr_rescored:,}",

    f"Final MS rescored     : {ms_rescored:,}",

    f"Overrides recalibrated: {override_recalibrated:,}",

    "",

    "TEST RESULTS",

    f"AP50-95               : {e24c_eval['overall']['AP50_95']*100:.4f}%",

    f"AP50                  : {e24c_eval['overall']['AP50']*100:.4f}%",

    f"AP75                  : {e24c_eval['overall']['AP75']*100:.4f}%",

    f"AR100                 : {e24c_eval['overall']['AR100']*100:.4f}%",

    f"Six-plastic mean AP50 : {e24c_eval['six_plastic_mean_AP50']*100:.4f}%",

    f"Six-plastic AP50-95   : {e24c_eval['six_plastic_mean_AP50_95']*100:.4f}%",

    f"MR/MS mean AP50       : {e24c_eval['MR_MS_mean_AP50']*100:.4f}%",

    "",

    f"SortWaste six-plastic : {SORTWASTE_PLASTIC_AP50:.4f}%",

    f"Difference             : {sortwaste_delta:+.4f} pp",

    f"Gap to 70%             : {gap_to_70:+.4f} pp",

    "",

    "Delta vs E23-D:",
]


for metric, value in delta_vs_e23d.items():

    summary_lines.append(
        f"  {metric:28s}: {value:+.4f} pp"
    )


with open(
    SUMMARY_PATH,
    "w",
    encoding="utf-8"
) as f:

    f.write(
        "\n".join(
            summary_lines
        )
    )


print()
print("=" * 120)
print("E24-C FROZEN HELD-OUT TEST COMPLETE")
print("=" * 120)

print()
print(
    f"Predictions : {PREDICTIONS_PATH}"
)

print(
    f"Results     : {RESULTS_PATH}"
)

print(
    f"Summary     : {SUMMARY_PATH}"
)

E24-C — FROZEN HELD-OUT TEST EVALUATION

E20-A predictions : 57,487
Matched cache     : 57,487
E23-A cache       : 17,193

FROZEN CONFIGURATION

MS -> MR : E23 >= 0.920, E20 <= 0.90
MR -> MS : E23 >= 0.997, E20 <= 0.95
beta      : 0.35
alpha MR  : 0.85
alpha MS  : 0.50
loading annotations into memory...
Done (t=0.04s)
creating index...
index created!

VERIFYING E20-A TEST BASELINE
Loading and preparing results...
DONE (t=0.06s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=3.89s).
Accumulating evaluation results...
DONE (t=2.72s).

AP50-95      : 46.2612%
AP50         : 62.3271%
Plastic AP50 : 69.4353%

E20-A baseline verification PASSED.

RECONSTRUCTED E23-D TEST ROUTING

MS -> MR proposals : 2,166
MR -> MS proposals : 2,407

MS -> MR accepted  : 691
MR -> MS accepted  : 311
Total accepted     : 1,002

VERIFYING CORRECTED E23-D TEST BASELINE
Loading and preparing results...
DONE (t=0.04s)
creating index...
index created!
Runni

# Model E25

## E25-A — Multi-Resolution Ensemble of YOLO11m-640 (E12) and YOLO11m-768 (E17)

In [46]:
# E25-A — MULTI-RESOLUTION OUTPUT ENSEMBLE OF E20-A AND YOLO11m-768 (E17)
#
# VALIDATION ONLY
#
# GOAL
# --------------------------------------------------------------------------------------------------
# Improve E20-A using complementary detections/localisation from the existing E17 YOLO11m @ 768.
#
# Why this version?
# --------------------------------------------------------------------------------------------------
# E20-A already contains the E12 detector + downstream classification pipeline.
# Rather than rerun E20-A from scratch, we reconstruct its exact validation predictions from the
# existing E23-B cache and fuse them with E17 predictions.
#
# This therefore requires:
#   - NO detector training
#   - NO MobileNet inference
#   - NO E18 ConvNeXt inference
#   - ONLY ONE E17 validation inference (cached)
#
# FUSION
# --------------------------------------------------------------------------------------------------
# Conservative class-wise Weighted Box Fusion (WBF-style).
#
# Small validation search:
#   IoU thresholds : 0.50, 0.55, 0.60, 0.65
#   E20:E17 weights: 1.00:1.00, 1.25:1.00, 1.50:1.00
#
# Total = 12 fusion configurations.
#
# Selection:
#   1. six-plastic mean AP50
#   2. overall AP50-95
#   3. MR/MS mean AP50
#
# TEST IS NOT USED.
# ==================================================================================================

import json
import math
from pathlib import Path
from collections import defaultdict

import numpy as np

from ultralytics import YOLO

from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval


# ==================================================================================================
# 1. PATHS
# ==================================================================================================

BASE = Path(
    r"C:\Users\varda\Documents\_My Computer\COMMON_space\Learning"
    r"\Upgrad_EPGP_MS\upgrad_IIITB_MS_course\MS_LJMU_Material"
)

DATASET_ROOT = (
    BASE
    / "Topic Data"
    / "SortWaste"
    / "dataset"
    / "dataset"
)

RUNS_ROOT = (
    BASE
    / "Thesis_Code"
    / "runs"
    / "sortwaste"
)


# --------------------------------------------------------------------------------------------------
# E20-A validation reconstruction source
# --------------------------------------------------------------------------------------------------

E20_CACHE_PATH = (
    RUNS_ROOT
    / "E23B_E20A_binary_MRMS_routing"
    / "E23B_cached_predictions_with_E23A.json"
)


# --------------------------------------------------------------------------------------------------
# Exact 7-class validation ground truth already used in E20/E23 experiments
# --------------------------------------------------------------------------------------------------

GT_PATH = (
    RUNS_ROOT
    / "E18D_class_selective_convnext"
    / "E18D_val_gt_7class.json"
)


# --------------------------------------------------------------------------------------------------
# Validation images
# --------------------------------------------------------------------------------------------------

VAL_IMAGES = (
    DATASET_ROOT
    / "splited_all_dataset_coco"
    / "val"
    / "images"
)


# --------------------------------------------------------------------------------------------------
# E17 checkpoint
# --------------------------------------------------------------------------------------------------

E17_CKPT = (
    RUNS_ROOT
    / "E17_yolo11m_7class_aug_classbalance_768"
    / "weights"
    / "best.pt"
)


# --------------------------------------------------------------------------------------------------
# Output
# --------------------------------------------------------------------------------------------------

OUTPUT_DIR = (
    RUNS_ROOT
    / "E25A_E20A_E17_multiresolution_ensemble"
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)


E20_PRED_PATH = (
    OUTPUT_DIR
    / "E25A_E20A_validation_predictions.json"
)

E17_CACHE_PATH = (
    OUTPUT_DIR
    / "E25A_E17_validation_predictions.json"
)

GRID_PATH = (
    OUTPUT_DIR
    / "E25A_fusion_grid_results.json"
)

BEST_CONFIG_PATH = (
    OUTPUT_DIR
    / "E25A_best_configuration.json"
)

BEST_PRED_PATH = (
    OUTPUT_DIR
    / "E25A_best_validation_predictions.json"
)

SUMMARY_PATH = (
    OUTPUT_DIR
    / "E25A_summary.txt"
)


for path, label in [
    (E20_CACHE_PATH, "E20-A validation cache"),
    (GT_PATH, "7-class validation ground truth"),
    (VAL_IMAGES, "validation image directory"),
    (E17_CKPT, "E17 checkpoint"),
]:

    if not path.exists():

        raise FileNotFoundError(
            f"{label} not found:\n{path}"
        )


# ==================================================================================================
# 2. TAXONOMY
# ==================================================================================================

CLASS_NAMES = [
    "ecal",
    "hdpe",
    "mixed_plastic_rigid",
    "mixed_plastic_soft",
    "non_plastic",
    "pet",
    "pet_oil",
]


ECAL = 0
HDPE = 1
MR = 2
MS = 3
NON_PLASTIC = 4
PET = 5
PET_OIL = 6


PLASTIC_INDICES = [
    ECAL,
    HDPE,
    MR,
    MS,
    PET,
    PET_OIL,
]


# ==================================================================================================
# 3. LOAD GT + CREATE IMAGE-ID LOOKUP
# ==================================================================================================

with open(
    GT_PATH,
    "r",
    encoding="utf-8"
) as f:

    gt_json = json.load(f)


images = (
    gt_json[
        "images"
    ]
)


print("=" * 120)
print("E25-A — E20-A + E17 MULTI-RESOLUTION OUTPUT ENSEMBLE")
print("=" * 120)

print()
print(
    f"Validation GT images : "
    f"{len(images):,}"
)


# Map both exact normalized filename and basename.

exact_name_to_id = {}
basename_to_id = {}

duplicate_basenames = set()


for img in images:

    image_id = int(
        img[
            "id"
        ]
    )

    file_name = str(
        img[
            "file_name"
        ]
    ).replace(
        "\\",
        "/"
    )


    exact_name_to_id[
        file_name
    ] = image_id


    basename = Path(
        file_name
    ).name


    if (
        basename in basename_to_id
        and
        basename_to_id[
            basename
        ]
        !=
        image_id
    ):

        duplicate_basenames.add(
            basename
        )


    basename_to_id[
        basename
    ] = image_id


if duplicate_basenames:

    raise RuntimeError(
        "Duplicate validation image basenames found. "
        "Cannot safely map E17 results to COCO image IDs."
    )


# ==================================================================================================
# 4. RECONSTRUCT EXACT E20-A VALIDATION PREDICTIONS
# ==================================================================================================

with open(
    E20_CACHE_PATH,
    "r",
    encoding="utf-8"
) as f:

    e20_cache = json.load(f)


print()
print(
    f"E20 cache detections : "
    f"{len(e20_cache):,}"
)


if len(
    e20_cache
) != 55605:

    raise RuntimeError(
        f"Expected 55,605 E20-A validation detections, "
        f"found {len(e20_cache):,}."
    )


e20_predictions = []


for item in e20_cache:

    x1, y1, x2, y2 = [
        float(v)

        for v
        in item[
            "bbox"
        ]
    ]


    width = (
        x2
        -
        x1
    )

    height = (
        y2
        -
        y1
    )


    if (
        width <= 0
        or
        height <= 0
    ):

        continue


    e20_predictions.append(
        {
            "image_id":
                int(
                    item[
                        "image_id"
                    ]
                ),

            "category_id":
                int(
                    item[
                        "_e20_class_idx"
                    ]
                    + 1
                ),

            "bbox": [
                x1,
                y1,
                width,
                height,
            ],

            "score":
                float(
                    item[
                        "_e20_score"
                    ]
                ),
        }
    )


print(
    f"E20 predictions built : "
    f"{len(e20_predictions):,}"
)


if len(
    e20_predictions
) != 55605:

    raise RuntimeError(
        "E20-A reconstruction lost detections."
    )


with open(
    E20_PRED_PATH,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        e20_predictions,
        f
    )


# ==================================================================================================
# 5. COCO EVALUATION HELPER
# ==================================================================================================

coco_gt = COCO(
    str(
        GT_PATH
    )
)


def valid_mean(
    values
):

    values = np.asarray(
        values
    )


    valid = values[
        values > -1
    ]


    if valid.size == 0:

        return float(
            "nan"
        )


    return float(
        valid.mean()
    )


def evaluate_predictions(
    predictions
):

    if len(
        predictions
    ) == 0:

        raise RuntimeError(
            "Cannot evaluate empty prediction list."
        )


    coco_dt = coco_gt.loadRes(
        predictions
    )


    evaluator = COCOeval(
        coco_gt,
        coco_dt,
        "bbox"
    )


    evaluator.params.maxDets = [
        1,
        10,
        100,
    ]


    evaluator.evaluate()
    evaluator.accumulate()


    precision = (
        evaluator.eval[
            "precision"
        ]
    )

    recall = (
        evaluator.eval[
            "recall"
        ]
    )


    ious = (
        evaluator.params.iouThrs
    )


    idx50 = int(
        np.where(
            np.isclose(
                ious,
                0.50
            )
        )[0][0]
    )


    idx75 = int(
        np.where(
            np.isclose(
                ious,
                0.75
            )
        )[0][0]
    )


    overall = {

        "AP50_95":
            valid_mean(
                precision[
                    :,
                    :,
                    :,
                    0,
                    -1
                ]
            ),

        "AP50":
            valid_mean(
                precision[
                    idx50,
                    :,
                    :,
                    0,
                    -1
                ]
            ),

        "AP75":
            valid_mean(
                precision[
                    idx75,
                    :,
                    :,
                    0,
                    -1
                ]
            ),

        "AR100":
            valid_mean(
                recall[
                    :,
                    :,
                    0,
                    -1
                ]
            ),
    }


    class_metrics = {}


    for class_idx, class_name in enumerate(
        CLASS_NAMES
    ):

        class_metrics[
            class_name
        ] = {

            "AP50":
                valid_mean(
                    precision[
                        idx50,
                        :,
                        class_idx,
                        0,
                        -1
                    ]
                ),

            "AP50_95":
                valid_mean(
                    precision[
                        :,
                        :,
                        class_idx,
                        0,
                        -1
                    ]
                ),
        }


    plastic = float(
        np.mean(
            [
                class_metrics[
                    CLASS_NAMES[
                        idx
                    ]
                ][
                    "AP50"
                ]

                for idx
                in PLASTIC_INDICES
            ]
        )
    )


    mr_ms = float(
        np.mean(
            [
                class_metrics[
                    "mixed_plastic_rigid"
                ][
                    "AP50"
                ],

                class_metrics[
                    "mixed_plastic_soft"
                ][
                    "AP50"
                ],
            ]
        )
    )


    return {

        "overall":
            overall,

        "class_metrics":
            class_metrics,

        "six_plastic_mean_AP50":
            plastic,

        "MR_MS_mean_AP50":
            mr_ms,
    }


# ==================================================================================================
# 6. VERIFY E20-A VALIDATION BASELINE FIRST
# ==================================================================================================

print()
print("=" * 120)
print("VERIFYING E20-A VALIDATION BASELINE")
print("=" * 120)


e20_eval = evaluate_predictions(
    e20_predictions
)


print()
print(
    f"AP50-95               : "
    f"{e20_eval['overall']['AP50_95'] * 100:.4f}%"
)

print(
    f"AP50                  : "
    f"{e20_eval['overall']['AP50'] * 100:.4f}%"
)

print(
    f"Six-plastic mean AP50: "
    f"{e20_eval['six_plastic_mean_AP50'] * 100:.4f}%"
)

print(
    f"MR AP50               : "
    f"{e20_eval['class_metrics']['mixed_plastic_rigid']['AP50'] * 100:.4f}%"
)

print(
    f"MS AP50               : "
    f"{e20_eval['class_metrics']['mixed_plastic_soft']['AP50'] * 100:.4f}%"
)


# Known E20-A validation references.

if not (
    abs(
        e20_eval[
            "six_plastic_mean_AP50"
        ]
        -
        0.6807664598358752
    )
    < 1e-6

    and

    abs(
        e20_eval[
            "overall"
        ][
            "AP50_95"
        ]
        -
        0.4575590834744165
    )
    < 1e-6
):

    raise RuntimeError(
        "\nE20-A VALIDATION BASELINE DID NOT REPRODUCE.\n"
        "E17 inference has NOT started.\n"
        "Do not continue until this is resolved."
    )


print()
print(
    "E20-A validation reproduction PASSED."
)


# ==================================================================================================
# 7. LOAD AND VERIFY E17 CHECKPOINT
# ==================================================================================================

model = YOLO(
    str(
        E17_CKPT
    )
)


model_names_raw = (
    model.names
)


if isinstance(
    model_names_raw,
    dict
):

    model_names = [
        str(
            model_names_raw[
                idx
            ]
        ).lower()

        for idx
        in sorted(
            model_names_raw.keys()
        )
    ]

else:

    model_names = [
        str(
            x
        ).lower()

        for x
        in model_names_raw
    ]


print()
print("=" * 120)
print("E17 CHECKPOINT VERIFICATION")
print("=" * 120)

print()
print(
    f"Checkpoint : "
    f"{E17_CKPT}"
)

print(
    f"Classes    : "
    f"{model_names}"
)


if model_names != CLASS_NAMES:

    raise RuntimeError(
        "\nE17 class ordering does NOT match expected 7-class taxonomy.\n"
        f"Expected: {CLASS_NAMES}\n"
        f"Found   : {model_names}\n"
        "Inference has NOT started."
    )


print()
print(
    "E17 taxonomy verification PASSED."
)


# ==================================================================================================
# 8. E17 INFERENCE — RUN ONCE, THEN CACHE
#
# E17 trained/inferred at 768.
#
# We use the same low-confidence/high-recall settings underlying the E12/E20 pipeline:
#
#   conf    = 0.001
#   iou     = 0.60
#   max_det = 100
#
# ==================================================================================================

if E17_CACHE_PATH.exists():

    print()
    print("=" * 120)
    print("USING EXISTING E17 VALIDATION CACHE")
    print("=" * 120)

    print()
    print(
        f"Cache : "
        f"{E17_CACHE_PATH}"
    )


    with open(
        E17_CACHE_PATH,
        "r",
        encoding="utf-8"
    ) as f:

        e17_predictions = json.load(f)


else:

    print()
    print("=" * 120)
    print("RUNNING E17 VALIDATION INFERENCE @ 768")
    print("=" * 120)

    print()
    print(
        "This is the only GPU inference required by E25-A."
    )


    e17_predictions = []


    result_stream = model.predict(

        source=str(
            VAL_IMAGES
        ),

        imgsz=768,

        conf=0.001,

        iou=0.60,

        max_det=100,

        device=0,

        stream=True,

        verbose=False,

        save=False,

    )


    processed_images = 0


    for result in result_stream:

        processed_images += 1


        image_path = Path(
            str(
                result.path
            )
        )


        basename = (
            image_path.name
        )


        if basename not in basename_to_id:

            raise RuntimeError(
                f"Could not map E17 result image to COCO ID:\n"
                f"{image_path}"
            )


        image_id = int(
            basename_to_id[
                basename
            ]
        )


        boxes = (
            result.boxes
        )


        if boxes is None:

            continue


        xyxy = (
            boxes.xyxy
            .detach()
            .cpu()
            .numpy()
        )


        confs = (
            boxes.conf
            .detach()
            .cpu()
            .numpy()
        )


        classes = (
            boxes.cls
            .detach()
            .cpu()
            .numpy()
            .astype(
                int
            )
        )


        for box, conf, cls_idx in zip(
            xyxy,
            confs,
            classes
        ):

            x1, y1, x2, y2 = [
                float(v)

                for v
                in box
            ]


            width = (
                x2 - x1
            )

            height = (
                y2 - y1
            )


            if (
                width <= 0
                or
                height <= 0
            ):

                continue


            if cls_idx < 0 or cls_idx >= 7:

                raise RuntimeError(
                    f"E17 produced invalid class index: {cls_idx}"
                )


            e17_predictions.append(
                {
                    "image_id":
                        image_id,

                    "category_id":
                        int(
                            cls_idx
                            + 1
                        ),

                    "bbox": [
                        x1,
                        y1,
                        width,
                        height,
                    ],

                    "score":
                        float(
                            conf
                        ),
                }
            )


    print()
    print(
        f"Images processed : "
        f"{processed_images:,}"
    )

    print(
        f"E17 detections   : "
        f"{len(e17_predictions):,}"
    )


    if processed_images != len(
        images
    ):

        raise RuntimeError(
            f"E17 processed {processed_images:,} images, "
            f"but GT contains {len(images):,} images."
        )


    with open(
        E17_CACHE_PATH,
        "w",
        encoding="utf-8"
    ) as f:

        json.dump(
            e17_predictions,
            f
        )


    print()
    print(
        f"E17 inference cache saved:\n"
        f"{E17_CACHE_PATH}"
    )


# ==================================================================================================
# 9. EVALUATE E17 ALONE
# ==================================================================================================

print()
print("=" * 120)
print("E17 STANDALONE VALIDATION RESULT")
print("=" * 120)


e17_eval = evaluate_predictions(
    e17_predictions
)


print()
print(
    f"Detections             : "
    f"{len(e17_predictions):,}"
)

print(
    f"AP50-95               : "
    f"{e17_eval['overall']['AP50_95'] * 100:.4f}%"
)

print(
    f"AP50                  : "
    f"{e17_eval['overall']['AP50'] * 100:.4f}%"
)

print(
    f"Six-plastic mean AP50: "
    f"{e17_eval['six_plastic_mean_AP50'] * 100:.4f}%"
)

print(
    f"MR AP50               : "
    f"{e17_eval['class_metrics']['mixed_plastic_rigid']['AP50'] * 100:.4f}%"
)

print(
    f"MS AP50               : "
    f"{e17_eval['class_metrics']['mixed_plastic_soft']['AP50'] * 100:.4f}%"
)


# ==================================================================================================
# 10. INTERNAL REPRESENTATION
# ==================================================================================================

def xywh_to_xyxy(
    bbox
):

    x, y, w, h = [
        float(v)

        for v
        in bbox
    ]


    return np.asarray(
        [
            x,
            y,
            x + w,
            y + h,
        ],
        dtype=np.float64
    )


def xyxy_to_xywh(
    box
):

    x1, y1, x2, y2 = [
        float(v)

        for v
        in box
    ]


    return [
        x1,
        y1,
        x2 - x1,
        y2 - y1,
    ]


def box_iou(
    box_a,
    box_b
):

    x1 = max(
        box_a[0],
        box_b[0]
    )

    y1 = max(
        box_a[1],
        box_b[1]
    )

    x2 = min(
        box_a[2],
        box_b[2]
    )

    y2 = min(
        box_a[3],
        box_b[3]
    )


    iw = max(
        0.0,
        x2 - x1
    )

    ih = max(
        0.0,
        y2 - y1
    )


    intersection = (
        iw
        *
        ih
    )


    area_a = max(
        0.0,
        box_a[2] - box_a[0]
    ) * max(
        0.0,
        box_a[3] - box_a[1]
    )


    area_b = max(
        0.0,
        box_b[2] - box_b[0]
    ) * max(
        0.0,
        box_b[3] - box_b[1]
    )


    union = (
        area_a
        +
        area_b
        -
        intersection
    )


    if union <= 0:

        return 0.0


    return float(
        intersection
        /
        union
    )


# ==================================================================================================
# 11. GROUP PREDICTIONS BY IMAGE + CLASS
# ==================================================================================================

def group_predictions(
    predictions,
    source_name
):

    grouped = defaultdict(
        list
    )


    for p in predictions:

        class_idx = (
            int(
                p[
                    "category_id"
                ]
            )
            -
            1
        )


        grouped[
            (
                int(
                    p[
                        "image_id"
                    ]
                ),
                class_idx,
            )
        ].append(
            {
                "box":
                    xywh_to_xyxy(
                        p[
                            "bbox"
                        ]
                    ),

                "score":
                    float(
                        p[
                            "score"
                        ]
                    ),

                "source":
                    source_name,
            }
        )


    return grouped


e20_grouped = group_predictions(
    e20_predictions,
    "E20"
)

e17_grouped = group_predictions(
    e17_predictions,
    "E17"
)


# ==================================================================================================
# 12. CONSERVATIVE WBF-STYLE FUSION
#
# Important behaviour:
#
# - Fusion occurs only within the SAME image and SAME class.
# - Coordinates use confidence × model-weight weighted averaging.
# - Confidence is the model-weight-weighted average of cluster confidence.
# - A detection seen by only one model is retained.
#
# This avoids throwing away complementary detections, while overlapping detections can improve
# localisation.
# ==================================================================================================

def fuse_cluster(
    detections,
    model_weights
):

    coordinate_weights = np.asarray(
        [
            max(
                d[
                    "score"
                ],
                1e-12
            )
            *
            model_weights[
                d[
                    "source"
                ]
            ]

            for d
            in detections
        ],
        dtype=np.float64
    )


    boxes = np.asarray(
        [
            d[
                "box"
            ]

            for d
            in detections
        ],
        dtype=np.float64
    )


    fused_box = np.average(
        boxes,
        axis=0,
        weights=coordinate_weights
    )


    # ----------------------------------------------------------------------------------------------
    # Confidence weighted by detector importance.
    #
    # We normalise only by weights of detections that actually contributed.
    # Single-model detections therefore remain viable instead of being automatically halved.
    # ----------------------------------------------------------------------------------------------

    score_numerator = sum(
        model_weights[
            d[
                "source"
            ]
        ]
        *
        d[
            "score"
        ]

        for d
        in detections
    )


    score_denominator = sum(
        model_weights[
            d[
                "source"
            ]
        ]

        for d
        in detections
    )


    fused_score = (
        score_numerator
        /
        score_denominator
    )


    return (
        fused_box,
        float(
            fused_score
        )
    )


def wbf_group(
    detections,
    iou_threshold,
    model_weights
):

    if not detections:

        return []


    # High weighted-confidence detections seed clusters first.

    ordered = sorted(

        detections,

        key=lambda d: (
            d[
                "score"
            ]
            *
            model_weights[
                d[
                    "source"
                ]
            ]
        ),

        reverse=True,
    )


    clusters = []


    for detection in ordered:

        best_cluster_idx = None
        best_iou = -1.0


        for cluster_idx, cluster in enumerate(
            clusters
        ):

            fused_box, _ = fuse_cluster(
                cluster,
                model_weights
            )


            iou = box_iou(
                detection[
                    "box"
                ],
                fused_box
            )


            if (
                iou >= iou_threshold
                and
                iou > best_iou
            ):

                best_iou = (
                    iou
                )

                best_cluster_idx = (
                    cluster_idx
                )


        if best_cluster_idx is None:

            clusters.append(
                [
                    detection
                ]
            )

        else:

            clusters[
                best_cluster_idx
            ].append(
                detection
            )


    fused = []


    for cluster in clusters:

        fused_box, fused_score = fuse_cluster(
            cluster,
            model_weights
        )


        fused.append(
            (
                fused_box,
                fused_score,
                len(
                    cluster
                ),
            )
        )


    return fused


def build_fused_predictions(
    iou_threshold,
    e20_weight,
    e17_weight
):

    model_weights = {
        "E20":
            float(
                e20_weight
            ),

        "E17":
            float(
                e17_weight
            ),
    }


    all_keys = (
        set(
            e20_grouped.keys()
        )
        |
        set(
            e17_grouped.keys()
        )
    )


    fused_predictions = []


    for (
        image_id,
        class_idx
    ) in all_keys:

        detections = (
            e20_grouped.get(
                (
                    image_id,
                    class_idx,
                ),
                []
            )
            +
            e17_grouped.get(
                (
                    image_id,
                    class_idx,
                ),
                []
            )
        )


        fused = wbf_group(
            detections=detections,
            iou_threshold=iou_threshold,
            model_weights=model_weights,
        )


        for (
            box,
            score,
            cluster_size
        ) in fused:

            bbox = xyxy_to_xywh(
                box
            )


            if (
                bbox[2] <= 0
                or
                bbox[3] <= 0
            ):

                continue


            fused_predictions.append(
                {
                    "image_id":
                        int(
                            image_id
                        ),

                    "category_id":
                        int(
                            class_idx
                            + 1
                        ),

                    "bbox":
                        [
                            float(v)

                            for v
                            in bbox
                        ],

                    "score":
                        float(
                            score
                        ),
                }
            )


    return fused_predictions


# ==================================================================================================
# 13. SMALL VALIDATION GRID
# ==================================================================================================

IOU_THRESHOLDS = [
    0.50,
    0.55,
    0.60,
    0.65,
]


WEIGHT_CONFIGS = [
    (
        1.00,
        1.00,
    ),

    (
        1.25,
        1.00,
    ),

    (
        1.50,
        1.00,
    ),
]


TOTAL_CONFIGS = (
    len(
        IOU_THRESHOLDS
    )
    *
    len(
        WEIGHT_CONFIGS
    )
)


print()
print("=" * 120)
print("E25-A — SMALL VALIDATION FUSION SEARCH")
print("=" * 120)

print()
print(
    f"IoU thresholds : "
    f"{IOU_THRESHOLDS}"
)

print(
    f"Weight configs : "
    f"{WEIGHT_CONFIGS}"
)

print(
    f"Total configs  : "
    f"{TOTAL_CONFIGS}"
)


grid_results = []

counter = 0


for iou_threshold in IOU_THRESHOLDS:

    for (
        e20_weight,
        e17_weight
    ) in WEIGHT_CONFIGS:

        counter += 1


        fused_predictions = build_fused_predictions(

            iou_threshold=
                iou_threshold,

            e20_weight=
                e20_weight,

            e17_weight=
                e17_weight,
        )


        evaluation = evaluate_predictions(
            fused_predictions
        )


        result = {

            "fusion_iou":
                float(
                    iou_threshold
                ),

            "E20_weight":
                float(
                    e20_weight
                ),

            "E17_weight":
                float(
                    e17_weight
                ),

            "num_predictions":
                int(
                    len(
                        fused_predictions
                    )
                ),

            "six_plastic_mean_AP50":
                evaluation[
                    "six_plastic_mean_AP50"
                ],

            "MR_MS_mean_AP50":
                evaluation[
                    "MR_MS_mean_AP50"
                ],

            "overall":
                evaluation[
                    "overall"
                ],

            "class_metrics":
                evaluation[
                    "class_metrics"
                ],
        }


        grid_results.append(
            result
        )


        print(
            f"[{counter:02d}/{TOTAL_CONFIGS:02d}] "
            f"IoU={iou_threshold:.2f} "
            f"W(E20:E17)={e20_weight:.2f}:{e17_weight:.2f} | "
            f"N={len(fused_predictions):,} | "
            f"Plastic="
            f"{evaluation['six_plastic_mean_AP50']*100:7.4f}% | "
            f"AP="
            f"{evaluation['overall']['AP50_95']*100:7.4f}% | "
            f"MR="
            f"{evaluation['class_metrics']['mixed_plastic_rigid']['AP50']*100:7.3f}% | "
            f"MS="
            f"{evaluation['class_metrics']['mixed_plastic_soft']['AP50']*100:7.3f}%"
        )


# ==================================================================================================
# 14. RANK RESULTS
#
# Primary:
#   six-plastic mean AP50
#
# Secondary:
#   overall AP50-95
#
# Tertiary:
#   MR/MS mean AP50
# ==================================================================================================

ranked = sorted(

    grid_results,

    key=lambda r: (

        r[
            "six_plastic_mean_AP50"
        ],

        r[
            "overall"
        ][
            "AP50_95"
        ],

        r[
            "MR_MS_mean_AP50"
        ],
    ),

    reverse=True,
)


best = (
    ranked[
        0
    ]
)


# ==================================================================================================
# 15. PRINT TOP RESULTS
# ==================================================================================================

print()
print("=" * 120)
print("TOP E25-A VALIDATION RESULTS")
print("=" * 120)

print(
    f"\n"
    f"{'Rank':>4s} "
    f"{'IoU':>6s} "
    f"{'E20W':>7s} "
    f"{'E17W':>7s} "
    f"{'Plastic':>12s} "
    f"{'AP50-95':>12s} "
    f"{'AP50':>11s} "
    f"{'MR':>10s} "
    f"{'MS':>10s}"
)

print(
    "-" * 95
)


for rank, result in enumerate(
    ranked,
    start=1
):

    print(
        f"{rank:4d} "
        f"{result['fusion_iou']:6.2f} "
        f"{result['E20_weight']:7.2f} "
        f"{result['E17_weight']:7.2f} "
        f"{result['six_plastic_mean_AP50']*100:11.4f}% "
        f"{result['overall']['AP50_95']*100:11.4f}% "
        f"{result['overall']['AP50']*100:10.4f}% "
        f"{result['class_metrics']['mixed_plastic_rigid']['AP50']*100:9.3f}% "
        f"{result['class_metrics']['mixed_plastic_soft']['AP50']*100:9.3f}%"
    )


# ==================================================================================================
# 16. BEST RESULT + DELTA VS E20-A
# ==================================================================================================

delta_vs_e20 = {

    "six_plastic_AP50_pp":

        (
            best[
                "six_plastic_mean_AP50"
            ]
            -
            e20_eval[
                "six_plastic_mean_AP50"
            ]
        )
        *
        100,


    "AP50_95_pp":

        (
            best[
                "overall"
            ][
                "AP50_95"
            ]
            -
            e20_eval[
                "overall"
            ][
                "AP50_95"
            ]
        )
        *
        100,


    "AP50_pp":

        (
            best[
                "overall"
            ][
                "AP50"
            ]
            -
            e20_eval[
                "overall"
            ][
                "AP50"
            ]
        )
        *
        100,


    "MR_AP50_pp":

        (
            best[
                "class_metrics"
            ][
                "mixed_plastic_rigid"
            ][
                "AP50"
            ]
            -
            e20_eval[
                "class_metrics"
            ][
                "mixed_plastic_rigid"
            ][
                "AP50"
            ]
        )
        *
        100,


    "MS_AP50_pp":

        (
            best[
                "class_metrics"
            ][
                "mixed_plastic_soft"
            ][
                "AP50"
            ]
            -
            e20_eval[
                "class_metrics"
            ][
                "mixed_plastic_soft"
            ][
                "AP50"
            ]
        )
        *
        100,
}


print()
print("=" * 120)
print("BEST E25-A CONFIGURATION — VALIDATION ONLY")
print("=" * 120)

print()
print(
    f"Fusion IoU             : "
    f"{best['fusion_iou']:.2f}"
)

print(
    f"E20 weight             : "
    f"{best['E20_weight']:.2f}"
)

print(
    f"E17 weight             : "
    f"{best['E17_weight']:.2f}"
)

print()
print(
    f"Six-plastic mean AP50 : "
    f"{best['six_plastic_mean_AP50']*100:.4f}%"
)

print(
    f"Overall AP50-95       : "
    f"{best['overall']['AP50_95']*100:.4f}%"
)

print(
    f"Overall AP50          : "
    f"{best['overall']['AP50']*100:.4f}%"
)

print(
    f"MR AP50               : "
    f"{best['class_metrics']['mixed_plastic_rigid']['AP50']*100:.4f}%"
)

print(
    f"MS AP50               : "
    f"{best['class_metrics']['mixed_plastic_soft']['AP50']*100:.4f}%"
)


print()
print(
    "Delta vs E20-A:"
)


for metric, value in delta_vs_e20.items():

    print(
        f"{metric:28s}: "
        f"{value:+.4f} pp"
    )


# ==================================================================================================
# 17. PROMOTION RULE
#
# We have already seen E24 validation gains fail to generalize.
#
# Therefore be stricter here:
#
# Promote only if:
#   six-plastic AP50 improvement >= +0.20 pp
#
# A +0.20 pp gate is still modest but avoids wasting a TEST run on another microscopic VAL gain.
# ==================================================================================================

PROMOTE_TO_TEST = (
    delta_vs_e20[
        "six_plastic_AP50_pp"
    ]
    >=
    0.20
)


print()
print("=" * 120)

if PROMOTE_TO_TEST:

    print(
        "DECISION: E25-A SHOWS SUFFICIENT VALIDATION GAIN — ELIGIBLE FOR FROZEN TEST."
    )

else:

    print(
        "DECISION: E25-A VALIDATION GAIN IS TOO SMALL — DO NOT TEST; KEEP E23-D."
    )

print("=" * 120)


# ==================================================================================================
# 18. SAVE BEST PREDICTIONS
# ==================================================================================================

best_predictions = build_fused_predictions(

    iou_threshold=
        float(
            best[
                "fusion_iou"
            ]
        ),

    e20_weight=
        float(
            best[
                "E20_weight"
            ]
        ),

    e17_weight=
        float(
            best[
                "E17_weight"
            ]
        ),
)


with open(
    BEST_PRED_PATH,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        best_predictions,
        f
    )


with open(
    GRID_PATH,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        ranked,
        f,
        indent=2
    )


# ==================================================================================================
# 19. SAVE CONFIG
# ==================================================================================================

config_payload = {

    "experiment":
        "E25-A",

    "description":
        (
            "Validation-only multi-resolution output ensemble "
            "of E20-A and E17 YOLO11m-768"
        ),

    "dataset":
        "validation",

    "test_used":
        False,

    "E17_inference": {

        "imgsz":
            768,

        "conf":
            0.001,

        "iou":
            0.60,

        "max_det":
            100,
    },

    "E20A_baseline":
        e20_eval,

    "E17_standalone":
        e17_eval,

    "best":
        best,

    "delta_vs_E20A_pp":
        delta_vs_e20,

    "promotion_threshold_plastic_AP50_pp":
        0.20,

    "promote_to_test":
        bool(
            PROMOTE_TO_TEST
        ),
}


with open(
    BEST_CONFIG_PATH,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        config_payload,
        f,
        indent=2
    )


# ==================================================================================================
# 20. SUMMARY
# ==================================================================================================

summary_lines = [

    "E25-A — E20-A + E17 Multi-Resolution Output Ensemble",

    "=" * 105,

    "",

    "VALIDATION ONLY — TEST NOT USED",

    "",

    "E20-A baseline:",

    f"  Six-plastic AP50 : {e20_eval['six_plastic_mean_AP50']*100:.4f}%",

    f"  AP50-95          : {e20_eval['overall']['AP50_95']*100:.4f}%",

    "",

    "E17 standalone:",

    f"  Six-plastic AP50 : {e17_eval['six_plastic_mean_AP50']*100:.4f}%",

    f"  AP50-95          : {e17_eval['overall']['AP50_95']*100:.4f}%",

    "",

    "Best E25-A:",

    f"  Fusion IoU       : {best['fusion_iou']:.2f}",

    f"  E20 weight       : {best['E20_weight']:.2f}",

    f"  E17 weight       : {best['E17_weight']:.2f}",

    f"  Six-plastic AP50 : {best['six_plastic_mean_AP50']*100:.4f}%",

    f"  AP50-95          : {best['overall']['AP50_95']*100:.4f}%",

    f"  AP50             : {best['overall']['AP50']*100:.4f}%",

    f"  MR AP50          : {best['class_metrics']['mixed_plastic_rigid']['AP50']*100:.4f}%",

    f"  MS AP50          : {best['class_metrics']['mixed_plastic_soft']['AP50']*100:.4f}%",

    "",

    f"Promote to TEST    : {PROMOTE_TO_TEST}",

    "",

    "Delta vs E20-A:",
]


for metric, value in delta_vs_e20.items():

    summary_lines.append(
        f"  {metric:28s}: "
        f"{value:+.4f} pp"
    )


with open(
    SUMMARY_PATH,
    "w",
    encoding="utf-8"
) as f:

    f.write(
        "\n".join(
            summary_lines
        )
    )


# ==================================================================================================
# 21. COMPLETE
# ==================================================================================================

print()
print("=" * 120)
print("E25-A COMPLETE")
print("=" * 120)

print()
print(
    f"E17 cache     : "
    f"{E17_CACHE_PATH}"
)

print(
    f"Grid results  : "
    f"{GRID_PATH}"
)

print(
    f"Best config   : "
    f"{BEST_CONFIG_PATH}"
)

print(
    f"Predictions   : "
    f"{BEST_PRED_PATH}"
)

print(
    f"Summary       : "
    f"{SUMMARY_PATH}"
)

E25-A — E20-A + E17 MULTI-RESOLUTION OUTPUT ENSEMBLE

Validation GT images : 780

E20 cache detections : 55,605
E20 predictions built : 55,605
loading annotations into memory...
Done (t=0.04s)
creating index...
index created!

VERIFYING E20-A VALIDATION BASELINE
Loading and preparing results...
DONE (t=0.07s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=4.04s).
Accumulating evaluation results...
DONE (t=0.54s).

AP50-95               : 45.7559%
AP50                  : 61.1717%
Six-plastic mean AP50: 68.0766%
MR AP50               : 49.6173%
MS AP50               : 42.8212%

E20-A validation reproduction PASSED.

E17 CHECKPOINT VERIFICATION

Checkpoint : C:\Users\varda\Documents\_My Computer\COMMON_space\Learning\Upgrad_EPGP_MS\upgrad_IIITB_MS_course\MS_LJMU_Material\Thesis_Code\runs\sortwaste\E17_yolo11m_7class_aug_classbalance_768\weights\best.pt
Classes    : ['ecal', 'hdpe', 'mixed_plastic_rigid', 'mixed_plastic_soft', 'non_

## E25-B — Test-Time Augmented E20-A Pipeline Using YOLO11m-640

In [47]:
# E25-B — VALIDATION-ONLY TEST-TIME AUGMENTED E20-A PIPELINE
#
# PIPELINE
# --------------------------------------------------------------------------------------------------
#
# E12 YOLO11m @ 640
#     augment=True
#     conf=0.001
#     iou=0.60
#     max_det=100
#
# -> crop
# -> E3Y-B MobileNet
# -> E16-A confidence gating
# -> E18-B ConvNeXt hard-class routing
# -> E20-A confidence fusion
#
#
# IMPORTANT
# --------------------------------------------------------------------------------------------------
# This is a DIRECT apples-to-apples comparison with E20-A.
#
# Only detector inference changes:
#
#     normal E12 inference
#           ->
#     E12 Test-Time Augmented inference
#
# Everything downstream remains frozen.
#
# VALIDATION ONLY.
# TEST IS NOT USED.
# ==================================================================================================

import json
from pathlib import Path
from collections import Counter

import numpy as np
from PIL import Image

import torch
import torch.nn as nn
import torchvision.transforms as T
from torchvision.models import (
    mobilenet_v3_large,
    MobileNet_V3_Large_Weights,
    convnext_tiny,
)

from ultralytics import YOLO

from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval


# ==================================================================================================
# 1. PATHS
# ==================================================================================================

BASE = Path(
    r"C:\Users\varda\Documents\_My Computer\COMMON_space\Learning"
    r"\Upgrad_EPGP_MS\upgrad_IIITB_MS_course\MS_LJMU_Material"
)

DATASET_ROOT = (
    BASE
    / "Topic Data"
    / "SortWaste"
    / "dataset"
    / "dataset"
)

RUNS_ROOT = (
    BASE
    / "Thesis_Code"
    / "runs"
    / "sortwaste"
)


VAL_IMAGES = (
    DATASET_ROOT
    / "splited_all_dataset_coco"
    / "val"
    / "images"
)


GT_PATH = (
    RUNS_ROOT
    / "E18D_class_selective_convnext"
    / "E18D_val_gt_7class.json"
)


# --------------------------------------------------------------------------------------------------
# E12 detector
# --------------------------------------------------------------------------------------------------

E12_CKPT = (
    RUNS_ROOT
    / "E12_yolo11m_7class_aug_classbalance_640"
    / "weights"
    / "best.pt"
)


# --------------------------------------------------------------------------------------------------
# E3Y-B MobileNet
# --------------------------------------------------------------------------------------------------

MN_CKPT = (
    DATASET_ROOT
    / "yolo_mobilenet_crops_E3Y"
    / "mobilenet_results"
    / "E3Y_B_class_weighted"
    / "E3Y_B_MobileNetV3Large_best.pth"
)


# --------------------------------------------------------------------------------------------------
# E18-B ConvNeXt Tiny
# --------------------------------------------------------------------------------------------------

CONV_CKPT = (
    RUNS_ROOT
    / "E18B_convnext_tiny_hardclass"
    / "E18B_ConvNeXtTiny_best.pth"
)


OUTPUT_DIR = (
    RUNS_ROOT
    / "E25B_E20A_TTA_validation"
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)


RAW_YOLO_CACHE = (
    OUTPUT_DIR
    / "E25B_TTA_yolo_predictions.json"
)

FINAL_PRED_PATH = (
    OUTPUT_DIR
    / "E25B_validation_predictions.json"
)

RESULTS_PATH = (
    OUTPUT_DIR
    / "E25B_validation_results.json"
)

SUMMARY_PATH = (
    OUTPUT_DIR
    / "E25B_summary.txt"
)


for path, label in [

    (VAL_IMAGES, "validation image directory"),
    (GT_PATH, "validation GT"),
    (E12_CKPT, "E12 YOLO checkpoint"),
    (MN_CKPT, "E3Y-B MobileNet checkpoint"),
    (CONV_CKPT, "E18-B ConvNeXt checkpoint"),

]:

    if not path.exists():

        raise FileNotFoundError(
            f"{label} not found:\n{path}"
        )


# ==================================================================================================
# 2. TAXONOMY
# ==================================================================================================

CLASS_NAMES = [
    "ecal",
    "hdpe",
    "mixed_plastic_rigid",
    "mixed_plastic_soft",
    "non_plastic",
    "pet",
    "pet_oil",
]


ECAL = 0
HDPE = 1
MR = 2
MS = 3
NP = 4
PET = 5
PET_OIL = 6


PLASTIC_INDICES = [
    ECAL,
    HDPE,
    MR,
    MS,
    PET,
    PET_OIL,
]


# ==================================================================================================
# 3. FROZEN E16-A / E20-A SETTINGS
# ==================================================================================================

# MobileNet acceptance thresholds.

E16_GATES = {

    ECAL: 0.99,
    HDPE: 0.98,
    MR: 0.98,
    MS: 0.94,
    NP: 0.94,
    PET: 0.99,
    PET_OIL: 0.99,
}


# E18-B hard classes.

HARD_CLASSES = {
    MR,
    MS,
    NP,
    PET_OIL,
}


# E20-A ConvNeXt override gates.

CONV_GATES = {

    # MR OFF
    MR: None,

    MS: 0.850,

    NP: 0.900,

    PET_OIL: 0.940,
}


ALPHA = (
    0.70
)


# ==================================================================================================
# 4. DEVICE
# ==================================================================================================

DEVICE = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)


print("=" * 120)
print("E25-B — TEST-TIME AUGMENTED E20-A PIPELINE — VALIDATION")
print("=" * 120)

print()
print(
    f"Device : {DEVICE}"
)


if DEVICE.type != "cuda":

    print(
        "WARNING: CUDA is not available. "
        "This run will be much slower."
    )


# ==================================================================================================
# 5. LOAD GT / IMAGE LOOKUP
# ==================================================================================================

with open(
    GT_PATH,
    "r",
    encoding="utf-8"
) as f:

    gt_json = json.load(f)


image_records = (
    gt_json[
        "images"
    ]
)


basename_to_id = {}
id_to_filename = {}


for record in image_records:

    image_id = int(
        record[
            "id"
        ]
    )

    basename = Path(
        record[
            "file_name"
        ]
    ).name


    if basename in basename_to_id:

        raise RuntimeError(
            f"Duplicate validation basename: {basename}"
        )


    basename_to_id[
        basename
    ] = image_id


    id_to_filename[
        image_id
    ] = basename


print()
print(
    f"Validation images : {len(image_records):,}"
)


if len(
    image_records
) != 780:

    raise RuntimeError(
        "Expected 780 validation images."
    )


# ==================================================================================================
# 6. LOAD E12 YOLO + VERIFY TAXONOMY
# ==================================================================================================

yolo = YOLO(
    str(
        E12_CKPT
    )
)


if isinstance(
    yolo.names,
    dict
):

    yolo_names = [
        str(
            yolo.names[
                i
            ]
        ).lower()

        for i
        in sorted(
            yolo.names.keys()
        )
    ]

else:

    yolo_names = [
        str(x).lower()
        for x
        in yolo.names
    ]


print()
print(
    f"E12 classes : {yolo_names}"
)


if yolo_names != CLASS_NAMES:

    raise RuntimeError(
        "\nE12 taxonomy mismatch.\n"
        f"Expected: {CLASS_NAMES}\n"
        f"Found   : {yolo_names}"
    )


print(
    "E12 taxonomy verification PASSED."
)


# ==================================================================================================
# 7. LOAD MOBILENET
# ==================================================================================================

mn_model = mobilenet_v3_large(
    weights=None
)


mn_model.classifier[
    3
] = nn.Linear(
    mn_model.classifier[
        3
    ].in_features,
    7
)


mn_state = torch.load(
    MN_CKPT,
    map_location=DEVICE,
    weights_only=False
)


# Support checkpoint dictionaries.

if isinstance(
    mn_state,
    dict
) and "model_state_dict" in mn_state:

    mn_state = (
        mn_state[
            "model_state_dict"
        ]
    )


mn_model.load_state_dict(
    mn_state
)


mn_model = (
    mn_model
    .to(
        DEVICE
    )
    .eval()
)


# ==================================================================================================
# 8. LOAD CONVNEXT
# ==================================================================================================

conv_model = convnext_tiny(
    weights=None
)


# E18-B trained on 4 hard classes:
#
# 0 MR
# 1 MS
# 2 NP
# 3 PET Oil

conv_model.classifier[
    2
] = nn.Linear(
    conv_model.classifier[
        2
    ].in_features,
    4
)


conv_state = torch.load(
    CONV_CKPT,
    map_location=DEVICE,
    weights_only=False
)


if isinstance(
    conv_state,
    dict
) and "model_state_dict" in conv_state:

    conv_state = (
        conv_state[
            "model_state_dict"
        ]
    )


conv_model.load_state_dict(
    conv_state
)


conv_model = (
    conv_model
    .to(
        DEVICE
    )
    .eval()
)


CONV_LOCAL_TO_GLOBAL = {

    0: MR,
    1: MS,
    2: NP,
    3: PET_OIL,
}


# ==================================================================================================
# 9. TRANSFORMS
# ==================================================================================================

imagenet_mean = [
    0.485,
    0.456,
    0.406,
]

imagenet_std = [
    0.229,
    0.224,
    0.225,
]


crop_transform = T.Compose(
    [

        T.Resize(
            (
                224,
                224,
            )
        ),

        T.ToTensor(),

        T.Normalize(
            mean=imagenet_mean,
            std=imagenet_std,
        ),
    ]
)


# ==================================================================================================
# 10. COCO EVALUATION
# ==================================================================================================

coco_gt = COCO(
    str(
        GT_PATH
    )
)


def valid_mean(
    values
):

    values = np.asarray(
        values
    )


    valid = values[
        values > -1
    ]


    if valid.size == 0:

        return float(
            "nan"
        )


    return float(
        valid.mean()
    )


def evaluate_predictions(
    predictions
):

    coco_dt = coco_gt.loadRes(
        predictions
    )


    evaluator = COCOeval(
        coco_gt,
        coco_dt,
        "bbox"
    )


    evaluator.params.maxDets = [
        1,
        10,
        100,
    ]


    evaluator.evaluate()
    evaluator.accumulate()


    precision = (
        evaluator.eval[
            "precision"
        ]
    )

    recall = (
        evaluator.eval[
            "recall"
        ]
    )


    ious = (
        evaluator.params.iouThrs
    )


    idx50 = int(
        np.where(
            np.isclose(
                ious,
                0.50
            )
        )[0][0]
    )


    idx75 = int(
        np.where(
            np.isclose(
                ious,
                0.75
            )
        )[0][0]
    )


    overall = {

        "AP50_95":
            valid_mean(
                precision[
                    :,
                    :,
                    :,
                    0,
                    -1
                ]
            ),

        "AP50":
            valid_mean(
                precision[
                    idx50,
                    :,
                    :,
                    0,
                    -1
                ]
            ),

        "AP75":
            valid_mean(
                precision[
                    idx75,
                    :,
                    :,
                    0,
                    -1
                ]
            ),

        "AR100":
            valid_mean(
                recall[
                    :,
                    :,
                    0,
                    -1
                ]
            ),
    }


    class_metrics = {}


    for class_idx, class_name in enumerate(
        CLASS_NAMES
    ):

        class_metrics[
            class_name
        ] = {

            "AP50":
                valid_mean(
                    precision[
                        idx50,
                        :,
                        class_idx,
                        0,
                        -1
                    ]
                ),

            "AP50_95":
                valid_mean(
                    precision[
                        :,
                        :,
                        class_idx,
                        0,
                        -1
                    ]
                ),
        }


    six_plastic = float(
        np.mean(
            [

                class_metrics[
                    CLASS_NAMES[
                        idx
                    ]
                ][
                    "AP50"
                ]

                for idx
                in PLASTIC_INDICES
            ]
        )
    )


    mr_ms = float(
        np.mean(
            [

                class_metrics[
                    "mixed_plastic_rigid"
                ][
                    "AP50"
                ],

                class_metrics[
                    "mixed_plastic_soft"
                ][
                    "AP50"
                ],
            ]
        )
    )


    return {

        "overall":
            overall,

        "class_metrics":
            class_metrics,

        "six_plastic_mean_AP50":
            six_plastic,

        "MR_MS_mean_AP50":
            mr_ms,
    }


# ==================================================================================================
# 11. RUN / CACHE TTA YOLO
# ==================================================================================================

if RAW_YOLO_CACHE.exists():

    print()
    print("=" * 120)
    print("USING EXISTING E25-B TTA YOLO CACHE")
    print("=" * 120)


    with open(
        RAW_YOLO_CACHE,
        "r",
        encoding="utf-8"
    ) as f:

        raw_yolo_predictions = json.load(f)


else:

    print()
    print("=" * 120)
    print("RUNNING E12 TTA DETECTOR INFERENCE")
    print("=" * 120)

    print()
    print(
        "augment=True"
    )


    raw_yolo_predictions = []


    results = yolo.predict(

        source=str(
            VAL_IMAGES
        ),

        imgsz=640,

        conf=0.001,

        iou=0.60,

        max_det=100,

        augment=True,

        device=0,

        stream=True,

        verbose=False,

        save=False,
    )


    processed_images = 0


    for result in results:

        processed_images += 1


        basename = Path(
            result.path
        ).name


        if basename not in basename_to_id:

            raise RuntimeError(
                f"Unable to map YOLO result to validation GT:\n{basename}"
            )


        image_id = (
            basename_to_id[
                basename
            ]
        )


        boxes = result.boxes


        if boxes is None:

            continue


        xyxy = (
            boxes.xyxy
            .detach()
            .cpu()
            .numpy()
        )


        confs = (
            boxes.conf
            .detach()
            .cpu()
            .numpy()
        )


        classes = (
            boxes.cls
            .detach()
            .cpu()
            .numpy()
            .astype(
                int
            )
        )


        for box, conf, cls_idx in zip(
            xyxy,
            confs,
            classes,
        ):

            x1, y1, x2, y2 = [
                float(v)
                for v
                in box
            ]


            if (
                x2 <= x1
                or
                y2 <= y1
            ):

                continue


            raw_yolo_predictions.append(
                {

                    "image_id":
                        int(
                            image_id
                        ),

                    "bbox_xyxy": [
                        x1,
                        y1,
                        x2,
                        y2,
                    ],

                    "yolo_conf":
                        float(
                            conf
                        ),

                    "yolo_class":
                        int(
                            cls_idx
                        ),
                }
            )


    print()
    print(
        f"Images processed : {processed_images:,}"
    )

    print(
        f"TTA detections   : {len(raw_yolo_predictions):,}"
    )


    if processed_images != 780:

        raise RuntimeError(
            f"Expected 780 images, processed {processed_images}."
        )


    with open(
        RAW_YOLO_CACHE,
        "w",
        encoding="utf-8"
    ) as f:

        json.dump(
            raw_yolo_predictions,
            f
        )


    print(
        f"TTA detector cache saved:\n{RAW_YOLO_CACHE}"
    )


# ==================================================================================================
# 12. PROCESS TTA DETECTIONS THROUGH E20-A
# ==================================================================================================

print()
print("=" * 120)
print("RUNNING FROZEN E20-A CLASSIFICATION PIPELINE ON TTA DETECTIONS")
print("=" * 120)


predictions = []

routing_counts = Counter()

invalid_crops = 0


current_image_id = None
current_image = None


with torch.inference_mode():

    for det_idx, det in enumerate(
        raw_yolo_predictions,
        start=1
    ):

        image_id = int(
            det[
                "image_id"
            ]
        )


        if image_id != current_image_id:

            filename = (
                id_to_filename[
                    image_id
                ]
            )


            image_path = (
                VAL_IMAGES
                /
                filename
            )


            if not image_path.exists():

                raise FileNotFoundError(
                    f"Validation image not found:\n{image_path}"
                )


            current_image = Image.open(
                image_path
            ).convert(
                "RGB"
            )


            current_image_id = (
                image_id
            )


        x1, y1, x2, y2 = [
            float(v)

            for v
            in det[
                "bbox_xyxy"
            ]
        ]


        # Clip crop.

        W, H = (
            current_image.size
        )


        cx1 = max(
            0,
            int(
                np.floor(
                    x1
                )
            )
        )

        cy1 = max(
            0,
            int(
                np.floor(
                    y1
                )
            )
        )

        cx2 = min(
            W,
            int(
                np.ceil(
                    x2
                )
            )
        )

        cy2 = min(
            H,
            int(
                np.ceil(
                    y2
                )
            )
        )


        if (
            cx2 <= cx1
            or
            cy2 <= cy1
        ):

            invalid_crops += 1
            continue


        crop = current_image.crop(
            (
                cx1,
                cy1,
                cx2,
                cy2,
            )
        )


        tensor = (
            crop_transform(
                crop
            )
            .unsqueeze(
                0
            )
            .to(
                DEVICE
            )
        )


        # ==========================================================================================
        # MobileNet
        # ==========================================================================================

        mn_logits = (
            mn_model(
                tensor
            )
        )


        mn_probs = (
            torch.softmax(
                mn_logits,
                dim=1
            )
            [0]
        )


        mn_prob, mn_idx = torch.max(
            mn_probs,
            dim=0
        )


        mn_idx = int(
            mn_idx.item()
        )

        mn_prob = float(
            mn_prob.item()
        )


        # ==========================================================================================
        # E16-A gating
        #
        # If MobileNet is sufficiently confident:
        #     use MobileNet class.
        #
        # Otherwise:
        #     preserve YOLO class.
        # ==========================================================================================

        yolo_class = int(
            det[
                "yolo_class"
            ]
        )


        final_idx = (
            yolo_class
        )

        classifier_prob = float(
            mn_probs[
                yolo_class
            ].item()
        )


        route_source = (
            "yolo"
        )


        gate = (
            E16_GATES[
                mn_idx
            ]
        )


        if mn_prob >= gate:

            final_idx = (
                mn_idx
            )

            classifier_prob = (
                mn_prob
            )

            route_source = (
                "mobilenet"
            )


        # ==========================================================================================
        # E18-B ConvNeXt
        # ==========================================================================================

        if final_idx in HARD_CLASSES:

            conv_logits = (
                conv_model(
                    tensor
                )
            )


            conv_probs = (
                torch.softmax(
                    conv_logits,
                    dim=1
                )
                [0]
            )


            conv_prob, conv_local_idx = torch.max(
                conv_probs,
                dim=0
            )


            conv_prob = float(
                conv_prob.item()
            )

            conv_local_idx = int(
                conv_local_idx.item()
            )


            conv_global_idx = (
                CONV_LOCAL_TO_GLOBAL[
                    conv_local_idx
                ]
            )


            conv_gate = (
                CONV_GATES[
                    final_idx
                ]
            )


            # MR gate = OFF.

            if (
                conv_gate is not None

                and

                conv_prob >= conv_gate
            ):

                final_idx = (
                    conv_global_idx
                )

                classifier_prob = (
                    conv_prob
                )

                route_source = (
                    "convnext"
                )


        # ==========================================================================================
        # E20-A confidence fusion
        # ==========================================================================================

        yolo_conf = max(
            float(
                det[
                    "yolo_conf"
                ]
            ),
            1e-12,
        )


        classifier_prob = max(
            float(
                classifier_prob
            ),
            1e-12,
        )


        score = (
            yolo_conf
            ** ALPHA
        ) * (
            classifier_prob
            ** (
                1.0
                -
                ALPHA
            )
        )


        predictions.append(
            {

                "image_id":
                    image_id,

                "category_id":
                    int(
                        final_idx
                        + 1
                    ),

                "bbox": [
                    x1,
                    y1,
                    x2 - x1,
                    y2 - y1,
                ],

                "score":
                    float(
                        score
                    ),
            }
        )


        routing_counts[
            route_source
        ] += 1


        if det_idx % 5000 == 0:

            print(
                f"Processed "
                f"{det_idx:,}/"
                f"{len(raw_yolo_predictions):,}"
            )


# ==================================================================================================
# 13. SUMMARY BEFORE EVALUATION
# ==================================================================================================

print()
print("=" * 120)
print("E25-B PIPELINE PROCESSING COMPLETE")
print("=" * 120)

print()
print(
    f"TTA detector detections : "
    f"{len(raw_yolo_predictions):,}"
)

print(
    f"Valid final predictions : "
    f"{len(predictions):,}"
)

print(
    f"Invalid crops           : "
    f"{invalid_crops:,}"
)


print()
print(
    "Routing:"
)


for source in [
    "yolo",
    "mobilenet",
    "convnext",
]:

    count = (
        routing_counts[
            source
        ]
    )


    pct = (
        count
        /
        max(
            len(
                predictions
            ),
            1
        )
        *
        100
    )


    print(
        f"  {source:10s}: "
        f"{count:7,d} "
        f"({pct:6.2f}%)"
    )


# ==================================================================================================
# 14. EVALUATE
# ==================================================================================================

print()
print("=" * 120)
print("E25-B — VALIDATION COCO EVALUATION")
print("=" * 120)


evaluation = evaluate_predictions(
    predictions
)


# ==================================================================================================
# 15. RESULTS
# ==================================================================================================

print()
print("=" * 120)
print("E25-B — VALIDATION RESULTS")
print("=" * 120)


print()
print(
    f"AP50-95               : "
    f"{evaluation['overall']['AP50_95']*100:.4f}%"
)

print(
    f"AP50                  : "
    f"{evaluation['overall']['AP50']*100:.4f}%"
)

print(
    f"AP75                  : "
    f"{evaluation['overall']['AP75']*100:.4f}%"
)

print(
    f"AR100                 : "
    f"{evaluation['overall']['AR100']*100:.4f}%"
)


print()
print(
    f"Six-plastic mean AP50: "
    f"{evaluation['six_plastic_mean_AP50']*100:.4f}%"
)

print(
    f"MR/MS mean AP50      : "
    f"{evaluation['MR_MS_mean_AP50']*100:.4f}%"
)


print()
print(
    f"{'Class':30s}"
    f"{'AP50':>15s}"
    f"{'AP50-95':>15s}"
)


print(
    "-" * 60
)


for class_name in CLASS_NAMES:

    metrics = (
        evaluation[
            "class_metrics"
        ][
            class_name
        ]
    )


    print(
        f"{class_name:30s}"
        f"{metrics['AP50']*100:14.2f}%"
        f"{metrics['AP50_95']*100:14.2f}%"
    )


# ==================================================================================================
# 16. DELTA VS VERIFIED E20-A VALIDATION
# ==================================================================================================

E20_BASELINE = {

    "plastic":
        0.6807664598358752,

    "AP":
        0.4575590834744165,

    "AP50":
        0.6117171119521958,

    "MR":
        0.4961733577935696,

    "MS":
        0.4282122331870161,
}


delta = {

    "six_plastic_AP50_pp":

        (
            evaluation[
                "six_plastic_mean_AP50"
            ]
            -
            E20_BASELINE[
                "plastic"
            ]
        )
        *
        100,


    "AP50_95_pp":

        (
            evaluation[
                "overall"
            ][
                "AP50_95"
            ]
            -
            E20_BASELINE[
                "AP"
            ]
        )
        *
        100,


    "AP50_pp":

        (
            evaluation[
                "overall"
            ][
                "AP50"
            ]
            -
            E20_BASELINE[
                "AP50"
            ]
        )
        *
        100,


    "MR_AP50_pp":

        (
            evaluation[
                "class_metrics"
            ][
                "mixed_plastic_rigid"
            ][
                "AP50"
            ]
            -
            E20_BASELINE[
                "MR"
            ]
        )
        *
        100,


    "MS_AP50_pp":

        (
            evaluation[
                "class_metrics"
            ][
                "mixed_plastic_soft"
            ][
                "AP50"
            ]
            -
            E20_BASELINE[
                "MS"
            ]
        )
        *
        100,
}


print()
print("=" * 120)
print("DELTA VS E20-A VALIDATION")
print("=" * 120)

print()


for metric, value in delta.items():

    print(
        f"{metric:28s}: "
        f"{value:+.4f} pp"
    )


# ==================================================================================================
# 17. DECISION
#
# Strict requirement after E24 experience:
#
# At least +0.25 pp six-plastic AP50 on validation.
# ==================================================================================================

PROMOTE_TO_TEST = (
    delta[
        "six_plastic_AP50_pp"
    ]
    >=
    0.25
)


print()
print("=" * 120)

if PROMOTE_TO_TEST:

    print(
        "DECISION: E25-B SHOWS A STRONG ENOUGH VALIDATION GAIN — ELIGIBLE FOR ONE FROZEN TEST."
    )

else:

    print(
        "DECISION: E25-B DOES NOT SHOW ENOUGH VALIDATION GAIN — REJECT; DO NOT TEST."
    )

print("=" * 120)


# ==================================================================================================
# 18. SAVE
# ==================================================================================================

with open(
    FINAL_PRED_PATH,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        predictions,
        f
    )


result_payload = {

    "experiment":
        "E25-B",

    "dataset":
        "validation",

    "test_used":
        False,

    "description":
        (
            "E20-A pipeline with YOLO11m-640 "
            "test-time augmented detector inference"
        ),

    "detector": {

        "checkpoint":
            str(
                E12_CKPT
            ),

        "imgsz":
            640,

        "augment":
            True,

        "conf":
            0.001,

        "iou":
            0.60,

        "max_det":
            100,
    },

    "baseline_E20A":
        E20_BASELINE,

    "routing_counts":
        dict(
            routing_counts
        ),

    "results":
        evaluation,

    "delta_vs_E20A_pp":
        delta,

    "promotion_threshold_pp":
        0.25,

    "promote_to_test":
        bool(
            PROMOTE_TO_TEST
        ),
}


with open(
    RESULTS_PATH,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        result_payload,
        f,
        indent=2
    )


summary_lines = [

    "E25-B — TTA E20-A Pipeline — Validation",

    "=" * 100,

    "",

    "VALIDATION ONLY — TEST NOT USED",

    "",

    f"TTA detections         : {len(raw_yolo_predictions):,}",

    f"Final predictions      : {len(predictions):,}",

    "",

    f"AP50-95               : {evaluation['overall']['AP50_95']*100:.4f}%",

    f"AP50                  : {evaluation['overall']['AP50']*100:.4f}%",

    f"Six-plastic mean AP50 : {evaluation['six_plastic_mean_AP50']*100:.4f}%",

    f"MR AP50               : {evaluation['class_metrics']['mixed_plastic_rigid']['AP50']*100:.4f}%",

    f"MS AP50               : {evaluation['class_metrics']['mixed_plastic_soft']['AP50']*100:.4f}%",

    "",

    f"Promote to TEST       : {PROMOTE_TO_TEST}",

    "",

    "Delta vs E20-A:",
]


for metric, value in delta.items():

    summary_lines.append(
        f"  {metric:28s}: "
        f"{value:+.4f} pp"
    )


with open(
    SUMMARY_PATH,
    "w",
    encoding="utf-8"
) as f:

    f.write(
        "\n".join(
            summary_lines
        )
    )


print()
print("=" * 120)
print("E25-B COMPLETE")
print("=" * 120)

print()
print(
    f"YOLO TTA cache : {RAW_YOLO_CACHE}"
)

print(
    f"Predictions    : {FINAL_PRED_PATH}"
)

print(
    f"Results        : {RESULTS_PATH}"
)

print(
    f"Summary        : {SUMMARY_PATH}"
)

E25-B — TEST-TIME AUGMENTED E20-A PIPELINE — VALIDATION

Device : cuda

Validation images : 780

E12 classes : ['ecal', 'hdpe', 'mixed_plastic_rigid', 'mixed_plastic_soft', 'non_plastic', 'pet', 'pet_oil']
E12 taxonomy verification PASSED.
loading annotations into memory...
Done (t=0.05s)
creating index...
index created!

RUNNING E12 TTA DETECTOR INFERENCE

augment=True

Images processed : 780
TTA detections   : 70,206
TTA detector cache saved:
C:\Users\varda\Documents\_My Computer\COMMON_space\Learning\Upgrad_EPGP_MS\upgrad_IIITB_MS_course\MS_LJMU_Material\Thesis_Code\runs\sortwaste\E25B_E20A_TTA_validation\E25B_TTA_yolo_predictions.json

RUNNING FROZEN E20-A CLASSIFICATION PIPELINE ON TTA DETECTIONS
Processed 5,000/70,206
Processed 10,000/70,206
Processed 15,000/70,206
Processed 20,000/70,206
Processed 25,000/70,206
Processed 30,000/70,206
Processed 35,000/70,206
Processed 40,000/70,206
Processed 45,000/70,206
Processed 50,000/70,206
Processed 55,000/70,206
Processed 60,000/70,206
Pr